In [51]:
# ============================================================
# MODULE 0 — ENVIRONMENT, REPRODUCIBILITY & PROJECT CONFIG
# TA-MoE Cardiovascular Disease-Status Classification
# Kaggle GPU: T4 x 2
# ============================================================

import os
import sys
import gc
import json
import random
import platform
import warnings
import subprocess
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0.1 Global experiment configuration
# ------------------------------------------------------------

PROJECT_NAME = "TA_MoE_CVD_Rebuild"
SEED = 42

TARGET = "HeartDiseaseorAttack"

TEST_SIZE = 0.20
N_SPLITS = 5

EPS = 1e-7

# IMPORTANT:
# Holdout must NEVER be used for:
# - model selection
# - hyperparameter selection
# - calibration selection
# - expert selection
# - gate selection
# - threshold selection
# - uncertainty tuning
# - fairness threshold tuning
#
# All development decisions must use DEV / OOF only.

CONFIG = {
    "project_name": PROJECT_NAME,
    "seed": SEED,
    "target": TARGET,
    "test_size": TEST_SIZE,
    "n_splits": N_SPLITS,
    "primary_metric": "PR_AUC",
    "secondary_metrics": [
        "ROC_AUC",
        "Brier",
        "NLL",
        "ECE",
        "Precision",
        "Recall",
        "F1",
        "MCC",
        "Balanced_Accuracy"
    ],
    "task": "cardiovascular_disease_status_classification",
    "holdout_policy": "final_evaluation_only",
}

# ------------------------------------------------------------
# 0.2 Reproducibility
# ------------------------------------------------------------

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# 0.3 PyTorch / GPU configuration
# ------------------------------------------------------------

try:
    import torch

    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # More deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    TORCH_AVAILABLE = True

except Exception as e:
    TORCH_AVAILABLE = False
    print("PyTorch initialization warning:", e)

# ------------------------------------------------------------
# 0.4 Output directories
# ------------------------------------------------------------

ROOT_DIR = Path("/kaggle/working") / PROJECT_NAME

DATA_DIR = ROOT_DIR / "data"
SPLIT_DIR = ROOT_DIR / "splits"
PRED_DIR = ROOT_DIR / "predictions"
MODEL_DIR = ROOT_DIR / "models"
TAB_DIR = ROOT_DIR / "tables"
FIG_DIR = ROOT_DIR / "figures"
LOG_DIR = ROOT_DIR / "logs"
CONFIG_DIR = ROOT_DIR / "configs"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"

for d in [
    ROOT_DIR,
    DATA_DIR,
    SPLIT_DIR,
    PRED_DIR,
    MODEL_DIR,
    TAB_DIR,
    FIG_DIR,
    LOG_DIR,
    CONFIG_DIR,
    CHECKPOINT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 0.5 Helpers
# ------------------------------------------------------------

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def convert(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, Path):
            return str(o)
        return o

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=convert)


def save_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)


# ------------------------------------------------------------
# 0.6 Hardware information
# ------------------------------------------------------------

print("=" * 90)
print("ENVIRONMENT")
print("=" * 90)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU cores:", os.cpu_count())

if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("CUDA version:", torch.version.cuda)
        print("GPU count:", torch.cuda.device_count())

        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(
                f"GPU {i}: {props.name} | "
                f"VRAM={props.total_memory / 1024**3:.2f} GB"
            )

        DEVICE = torch.device("cuda:0")

    else:
        DEVICE = torch.device("cpu")
else:
    DEVICE = "cpu"

print("Primary device:", DEVICE)

# ------------------------------------------------------------
# 0.7 Save initial configuration
# ------------------------------------------------------------

environment_info = {
    "timestamp": datetime.now().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "torch_available": TORCH_AVAILABLE,
}

if TORCH_AVAILABLE:
    environment_info.update({
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_count": torch.cuda.device_count()
        if torch.cuda.is_available() else 0,
        "gpu_names": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ] if torch.cuda.is_available() else [],
    })

save_json(CONFIG, CONFIG_DIR / "global_config.json")
save_json(environment_info, LOG_DIR / "environment.json")

print("\nProject root:")
print(ROOT_DIR)

print("\nMODULE 0 COMPLETE.")

# ============================================================
# MODULE 1 — DATA DISCOVERY, LOADING & AUDIT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1.1 Search Kaggle input for the correct BRFSS file
# ------------------------------------------------------------

candidate_csvs = list(Path("/kaggle/input").rglob("*.csv"))

print(f"CSV files detected: {len(candidate_csvs)}")

matching_files = []

for path in candidate_csvs:
    try:
        header_df = pd.read_csv(path, nrows=3)

        if TARGET in header_df.columns:
            matching_files.append(path)

    except Exception:
        pass

print("\nCSV files containing target:", TARGET)

for i, path in enumerate(matching_files):
    print(f"[{i}] {path}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        f"No CSV containing target '{TARGET}' was found under /kaggle/input.\n"
        "Attach the BRFSS 2015 dataset to this Kaggle notebook first."
    )

# ------------------------------------------------------------
# 1.2 Select dataset
# ------------------------------------------------------------

# If only one matching file exists, use it automatically.
# If several exist, prefer the file whose full row count is closest
# to the expected BRFSS dataset (~253,680 samples).

if len(matching_files) == 1:
    DATA_PATH = matching_files[0]

else:
    file_sizes = []

    for path in matching_files:
        try:
            temp = pd.read_csv(path, usecols=[TARGET])
            file_sizes.append((path, len(temp)))
        except Exception:
            continue

    print("\nMatching file sizes:")
    for p, n in file_sizes:
        print(f"{n:,} rows | {p}")

    EXPECTED_N = 253_680

    DATA_PATH = min(
        file_sizes,
        key=lambda x: abs(x[1] - EXPECTED_N)
    )[0]

print("\nSelected dataset:")
print(DATA_PATH)

# ------------------------------------------------------------
# 1.3 Load full dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH).copy()

# remove accidental whitespace
df.columns = [str(c).strip() for c in df.columns]

# persistent row identifier
df.insert(
    0,
    "orig_row_id",
    np.arange(len(df), dtype=np.int64)
)

print("\nDataset loaded.")
print("Shape including row ID:", df.shape)

# ------------------------------------------------------------
# 1.4 Target validation
# ------------------------------------------------------------

if TARGET not in df.columns:
    raise ValueError(
        f"Target '{TARGET}' not found.\n"
        f"Columns:\n{list(df.columns)}"
    )

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

n_missing_target = int(df[TARGET].isna().sum())

if n_missing_target > 0:
    raise ValueError(
        f"Target contains {n_missing_target} missing/non-numeric values. "
        "Do not silently remove them before inspection."
    )

target_values = sorted(df[TARGET].unique().tolist())

if not set(target_values).issubset({0, 1, 0.0, 1.0}):
    raise ValueError(
        f"Unexpected target values: {target_values}"
    )

df[TARGET] = df[TARGET].astype(np.int8)

# ------------------------------------------------------------
# 1.5 Basic audit
# ------------------------------------------------------------

FEATURE_COLS = [
    c for c in df.columns
    if c not in ["orig_row_id", TARGET]
]

n_rows = len(df)
n_features = len(FEATURE_COLS)

n_positive = int(df[TARGET].sum())
n_negative = int(n_rows - n_positive)
positive_rate = float(df[TARGET].mean())

duplicate_feature_rows = int(
    df.drop(columns=["orig_row_id"]).duplicated().sum()
)

missing_total = int(
    df[FEATURE_COLS].isna().sum().sum()
)

audit_summary = pd.DataFrame([{
    "n_samples": n_rows,
    "n_features": n_features,
    "n_columns_total_without_row_id": len(df.columns) - 1,
    "target": TARGET,
    "positive_count": n_positive,
    "negative_count": n_negative,
    "positive_rate": positive_rate,
    "negative_rate": 1 - positive_rate,
    "duplicate_rows": duplicate_feature_rows,
    "missing_feature_values": missing_total,
}])

print("\n" + "=" * 90)
print("DATASET AUDIT")
print("=" * 90)

display(audit_summary)

print("\nColumns:")
print(FEATURE_COLS)

# ------------------------------------------------------------
# 1.6 Per-column audit
# ------------------------------------------------------------

column_rows = []

for col in FEATURE_COLS + [TARGET]:

    s = df[col]

    column_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "n_unique": int(s.nunique(dropna=True)),
        "n_missing": int(s.isna().sum()),
        "min": float(s.min()) if pd.api.types.is_numeric_dtype(s) else None,
        "max": float(s.max()) if pd.api.types.is_numeric_dtype(s) else None,
        "mean": float(s.mean()) if pd.api.types.is_numeric_dtype(s) else None,
    })

column_audit = pd.DataFrame(column_rows)

display(column_audit)

# ------------------------------------------------------------
# 1.7 Save audit outputs
# ------------------------------------------------------------

save_csv(
    audit_summary,
    TAB_DIR / "module1_dataset_audit.csv"
)

save_csv(
    column_audit,
    TAB_DIR / "module1_column_audit.csv"
)

save_json(
    {
        "data_path": str(DATA_PATH),
        "n_samples": n_rows,
        "n_features": n_features,
        "feature_cols": FEATURE_COLS,
        "target": TARGET,
        "positive_count": n_positive,
        "negative_count": n_negative,
        "positive_rate": positive_rate,
        "duplicate_rows": duplicate_feature_rows,
        "missing_feature_values": missing_total,
    },
    LOG_DIR / "module1_dataset_manifest.json"
)

print("\nMODULE 1 COMPLETE.")

# ============================================================
# MODULE 2 — FEATURE SCHEMA & DATA INTEGRITY
# ============================================================

# ------------------------------------------------------------
# 2.1 Known BRFSS semantic feature groups
# ------------------------------------------------------------

BINARY_COLS = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]

ORDINAL_COLS = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]

COUNT_COLS = [
    "MentHlth",
    "PhysHlth",
]

CONTINUOUS_COLS = [
    "BMI",
]

MULTICLASS_COLS = [
    "Diabetes",
]

# ------------------------------------------------------------
# 2.2 Retain only columns actually present
# ------------------------------------------------------------

BINARY_COLS = [c for c in BINARY_COLS if c in FEATURE_COLS]
ORDINAL_COLS = [c for c in ORDINAL_COLS if c in FEATURE_COLS]
COUNT_COLS = [c for c in COUNT_COLS if c in FEATURE_COLS]
CONTINUOUS_COLS = [c for c in CONTINUOUS_COLS if c in FEATURE_COLS]
MULTICLASS_COLS = [c for c in MULTICLASS_COLS if c in FEATURE_COLS]

known = set(
    BINARY_COLS
    + ORDINAL_COLS
    + COUNT_COLS
    + CONTINUOUS_COLS
    + MULTICLASS_COLS
)

OTHER_COLS = [
    c for c in FEATURE_COLS
    if c not in known
]

# ------------------------------------------------------------
# 2.3 Inspect Diabetes coding
# ------------------------------------------------------------

if "Diabetes" in df.columns:
    print(
        "Diabetes unique values:",
        sorted(df["Diabetes"].dropna().unique().tolist())
    )

# ------------------------------------------------------------
# 2.4 Schema table
# ------------------------------------------------------------

schema_rows = []

groups = {
    "binary": BINARY_COLS,
    "ordinal": ORDINAL_COLS,
    "count": COUNT_COLS,
    "continuous": CONTINUOUS_COLS,
    "multiclass_ordinal": MULTICLASS_COLS,
    "other": OTHER_COLS,
}

for group_name, cols in groups.items():
    for col in cols:
        schema_rows.append({
            "feature": col,
            "group": group_name,
            "n_unique": int(df[col].nunique()),
            "min": float(df[col].min()),
            "max": float(df[col].max()),
        })

feature_schema = pd.DataFrame(schema_rows)

print("\n" + "=" * 90)
print("FEATURE SCHEMA")
print("=" * 90)

display(feature_schema)

print("\nFeature group counts:")
display(
    feature_schema
    .groupby("group")
    .size()
    .rename("n_features")
    .reset_index()
)

# ------------------------------------------------------------
# 2.5 Binary integrity checks
# ------------------------------------------------------------

binary_issues = {}

for col in BINARY_COLS:

    vals = set(df[col].dropna().unique().tolist())

    if not vals.issubset({0, 1, 0.0, 1.0}):
        binary_issues[col] = sorted(vals)

if binary_issues:
    print("\nWARNING — unexpected binary coding:")
    print(binary_issues)
else:
    print("\nAll binary columns contain only {0,1}.")

# ------------------------------------------------------------
# 2.6 Expected range checks
# ------------------------------------------------------------

EXPECTED_RANGES = {
    "GenHlth": (1, 5),
    "MentHlth": (0, 30),
    "PhysHlth": (0, 30),
    "Age": (1, 13),
    "Education": (1, 6),
    "Income": (1, 8),
}

range_issues = []

for col, (low, high) in EXPECTED_RANGES.items():

    if col not in df.columns:
        continue

    observed_min = df[col].min()
    observed_max = df[col].max()

    if observed_min < low or observed_max > high:

        range_issues.append({
            "feature": col,
            "expected_min": low,
            "expected_max": high,
            "observed_min": observed_min,
            "observed_max": observed_max,
        })

if range_issues:
    print("\nWARNING — range issues:")
    display(pd.DataFrame(range_issues))
else:
    print("Known ordinal/count ranges are valid.")

# ------------------------------------------------------------
# 2.7 Subset expert features
# ------------------------------------------------------------

SUBSET_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "DiffWalk",
    "GenHlth",
    "Age",
    "Income",
    "Sex",
]

SUBSET_FEATURES = [
    c for c in SUBSET_FEATURES
    if c in FEATURE_COLS
]

print("\nLogReg_Subset candidate features:")
print(SUBSET_FEATURES)

# ------------------------------------------------------------
# 2.8 Save schema
# ------------------------------------------------------------

save_csv(
    feature_schema,
    TAB_DIR / "module2_feature_schema.csv"
)

schema_manifest = {
    "feature_cols": FEATURE_COLS,
    "binary_cols": BINARY_COLS,
    "ordinal_cols": ORDINAL_COLS,
    "count_cols": COUNT_COLS,
    "continuous_cols": CONTINUOUS_COLS,
    "multiclass_cols": MULTICLASS_COLS,
    "other_cols": OTHER_COLS,
    "subset_features": SUBSET_FEATURES,
}

save_json(
    schema_manifest,
    LOG_DIR / "module2_feature_schema.json"
)

print("\nMODULE 2 COMPLETE.")

# ============================================================
# MODULE 3 — LOCKED DEV/HOLDOUT SPLIT + FIXED CV FOLDS
# ============================================================

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

# ------------------------------------------------------------
# 3.1 X / y
# ------------------------------------------------------------

X = df[FEATURE_COLS].copy()
y = df[TARGET].astype(np.int8).copy()

row_ids = df["orig_row_id"].to_numpy()

# ------------------------------------------------------------
# 3.2 Fixed stratified 80/20 split
# ------------------------------------------------------------

all_indices = np.arange(len(df))

dev_idx, holdout_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y
)

dev_idx = np.asarray(dev_idx)
holdout_idx = np.asarray(holdout_idx)

# Stable ordering makes later prediction banks easier to compare
dev_idx = np.sort(dev_idx)
holdout_idx = np.sort(holdout_idx)

X_dev = X.iloc[dev_idx].reset_index(drop=True)
y_dev = y.iloc[dev_idx].reset_index(drop=True)

X_holdout = X.iloc[holdout_idx].reset_index(drop=True)
y_holdout = y.iloc[holdout_idx].reset_index(drop=True)

dev_row_ids = row_ids[dev_idx]
holdout_row_ids = row_ids[holdout_idx]

# ------------------------------------------------------------
# 3.3 Split summary
# ------------------------------------------------------------

split_summary = pd.DataFrame([
    {
        "split": "FULL",
        "n": len(y),
        "positive": int(y.sum()),
        "negative": int(len(y) - y.sum()),
        "prevalence": float(y.mean()),
    },
    {
        "split": "DEV",
        "n": len(y_dev),
        "positive": int(y_dev.sum()),
        "negative": int(len(y_dev) - y_dev.sum()),
        "prevalence": float(y_dev.mean()),
    },
    {
        "split": "HOLDOUT",
        "n": len(y_holdout),
        "positive": int(y_holdout.sum()),
        "negative": int(len(y_holdout) - y_holdout.sum()),
        "prevalence": float(y_holdout.mean()),
    },
])

print("=" * 90)
print("LOCKED DATA SPLIT")
print("=" * 90)

display(split_summary)

# ------------------------------------------------------------
# 3.4 Integrity assertions
# ------------------------------------------------------------

assert len(set(dev_idx).intersection(set(holdout_idx))) == 0
assert len(dev_idx) + len(holdout_idx) == len(df)

assert len(np.unique(dev_row_ids)) == len(dev_row_ids)
assert len(np.unique(holdout_row_ids)) == len(holdout_row_ids)

print("DEV/HOLDOUT overlap: 0")
print("Split integrity: PASSED")

# ------------------------------------------------------------
# 3.5 Create fixed DEV CV assignments
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_assignment = np.full(
    len(X_dev),
    -1,
    dtype=np.int8
)

fold_rows = []

for fold, (train_local_idx, valid_local_idx) in enumerate(
    skf.split(X_dev, y_dev),
    start=0
):

    fold_assignment[valid_local_idx] = fold

    fold_rows.append({
        "fold": fold,
        "train_n": len(train_local_idx),
        "valid_n": len(valid_local_idx),
        "train_positive": int(y_dev.iloc[train_local_idx].sum()),
        "valid_positive": int(y_dev.iloc[valid_local_idx].sum()),
        "train_prevalence": float(
            y_dev.iloc[train_local_idx].mean()
        ),
        "valid_prevalence": float(
            y_dev.iloc[valid_local_idx].mean()
        ),
    })

assert np.all(fold_assignment >= 0)

fold_summary = pd.DataFrame(fold_rows)

print("\n" + "=" * 90)
print("FIXED 5-FOLD DEV ASSIGNMENTS")
print("=" * 90)

display(fold_summary)

# ------------------------------------------------------------
# 3.6 Persistent split registry
# ------------------------------------------------------------

dev_registry = pd.DataFrame({
    "dev_position": np.arange(len(X_dev)),
    "orig_row_id": dev_row_ids,
    "target": y_dev.to_numpy(),
    "fold": fold_assignment,
})

holdout_registry = pd.DataFrame({
    "holdout_position": np.arange(len(X_holdout)),
    "orig_row_id": holdout_row_ids,
    "target": y_holdout.to_numpy(),
})

save_csv(
    dev_registry,
    SPLIT_DIR / "dev_fold_registry.csv"
)

save_csv(
    holdout_registry,
    SPLIT_DIR / "holdout_registry.csv"
)

np.save(
    SPLIT_DIR / "dev_original_indices.npy",
    dev_idx
)

np.save(
    SPLIT_DIR / "holdout_original_indices.npy",
    holdout_idx
)

np.save(
    SPLIT_DIR / "dev_fold_assignment.npy",
    fold_assignment
)

save_csv(
    split_summary,
    TAB_DIR / "module3_split_summary.csv"
)

save_csv(
    fold_summary,
    TAB_DIR / "module3_fold_summary.csv"
)

# ------------------------------------------------------------
# 3.7 Split manifest
# ------------------------------------------------------------

split_manifest = {
    "seed": SEED,
    "test_size": TEST_SIZE,
    "n_splits": N_SPLITS,

    "full_n": len(y),
    "dev_n": len(y_dev),
    "holdout_n": len(y_holdout),

    "full_prevalence": float(y.mean()),
    "dev_prevalence": float(y_dev.mean()),
    "holdout_prevalence": float(y_holdout.mean()),

    "split_locked": True,
    "holdout_policy": "final_evaluation_only",
}

save_json(
    split_manifest,
    LOG_DIR / "module3_split_manifest.json"
)

print("\nSaved:")
print(SPLIT_DIR / "dev_fold_registry.csv")
print(SPLIT_DIR / "holdout_registry.csv")
print(SPLIT_DIR / "dev_fold_assignment.npy")

print("\nMODULE 3 COMPLETE.")
print("\n*** SPLIT IS NOW LOCKED. DO NOT CHANGE SEED OR HOLDOUT. ***")

ENVIRONMENT
Python: 3.13.15
Platform: Linux-6.18.48+-x86_64-with-glibc2.39
CPU cores: 4
PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 2
GPU 0: Tesla T4 | VRAM=14.56 GB
GPU 1: Tesla T4 | VRAM=14.56 GB
Primary device: cuda:0

Project root:
/kaggle/working/TA_MoE_CVD_Rebuild

MODULE 0 COMPLETE.
CSV files detected: 2

CSV files containing target: HeartDiseaseorAttack
[0] /kaggle/input/datasets/alexteboul/heart-disease-health-indicators-dataset/heart_disease_health_indicators_BRFSS2015.csv

Selected dataset:
/kaggle/input/datasets/alexteboul/heart-disease-health-indicators-dataset/heart_disease_health_indicators_BRFSS2015.csv

Dataset loaded.
Shape including row ID: (253680, 23)

DATASET AUDIT


,n_samples,n_features,n_columns_total_without_row_id,target,positive_count,negative_count,positive_rate,negative_rate,duplicate_rows,missing_feature_values
0,253680,21,22,HeartDiseaseorAttack,23893,229787,0.094186,0.905814,23899,0



Columns:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


,column,dtype,n_unique,n_missing,min,max,mean
0,HighBP,float64,2,0,0.0,1.0,0.429001
1,HighChol,float64,2,0,0.0,1.0,0.424121
2,CholCheck,float64,2,0,0.0,1.0,0.962670
3,BMI,float64,84,0,12.0,98.0,28.382364
4,Smoker,float64,2,0,0.0,1.0,0.443169
5,Stroke,float64,2,0,0.0,1.0,0.040571
6,Diabetes,float64,3,0,0.0,2.0,0.296921
7,PhysActivity,float64,2,0,0.0,1.0,0.756544
8,Fruits,float64,2,0,0.0,1.0,0.634256
9,Veggies,float64,2,0,0.0,1.0,0.811420



MODULE 1 COMPLETE.
Diabetes unique values: [0.0, 1.0, 2.0]

FEATURE SCHEMA


...


All binary columns contain only {0,1}.
Known ordinal/count ranges are valid.

LogReg_Subset candidate features:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'Diabetes', 'PhysActivity', 'DiffWalk', 'GenHlth', 'Age', 'Income', 'Sex']

MODULE 2 COMPLETE.
LOCKED DATA SPLIT


,split,n,positive,negative,prevalence
0,FULL,253680,23893,229787,0.094186
1,DEV,202944,19114,183830,0.094184
2,HOLDOUT,50736,4779,45957,0.094193


DEV/HOLDOUT overlap: 0
Split integrity: PASSED

FIXED 5-FOLD DEV ASSIGNMENTS


,fold,train_n,valid_n,train_positive,valid_positive,train_prevalence,valid_prevalence
0,0,162355,40589,15291,3823,0.094183,0.094188
1,1,162355,40589,15291,3823,0.094183,0.094188
2,2,162355,40589,15291,3823,0.094183,0.094188
3,3,162355,40589,15291,3823,0.094183,0.094188
4,4,162356,40588,15292,3822,0.094188,0.094166



Saved:
/kaggle/working/TA_MoE_CVD_Rebuild/splits/dev_fold_registry.csv
/kaggle/working/TA_MoE_CVD_Rebuild/splits/holdout_registry.csv
/kaggle/working/TA_MoE_CVD_Rebuild/splits/dev_fold_assignment.npy

MODULE 3 COMPLETE.

*** SPLIT IS NOW LOCKED. DO NOT CHANGE SEED OR HOLDOUT. ***


In [2]:
# ============================================================
# MODULE 4 — LEAKAGE-SAFE PREPROCESSING INFRASTRUCTURE
# ============================================================
#
# Goals:
# 1. Define preprocessing separately for model families.
# 2. Fit preprocessing ONLY inside each CV training fold.
# 3. Preserve raw feature representation for tree models.
# 4. Produce standardized representation for linear / dense NN.
# 5. Prepare semantic groups for embedding-based models
#    (DeepFM / AutoInt) without transforming them prematurely.
#
# IMPORTANT:
# No preprocessor is fitted globally in this module.
# ============================================================

import copy
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# ------------------------------------------------------------
# 4.1 Safety checks
# ------------------------------------------------------------

required_vars = [
    "X_dev",
    "y_dev",
    "X_holdout",
    "y_holdout",
    "FEATURE_COLS",
    "BINARY_COLS",
    "ORDINAL_COLS",
    "COUNT_COLS",
    "CONTINUOUS_COLS",
    "MULTICLASS_COLS",
    "fold_assignment",
    "N_SPLITS",
    "SEED",
    "TAB_DIR",
    "LOG_DIR",
    "save_json",
    "save_csv",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise ValueError(
        f"Module 4 missing required objects from Modules 0–3: {missing}"
    )

# ------------------------------------------------------------
# 4.2 Class-imbalance statistics
# ------------------------------------------------------------

N_POS_DEV = int(y_dev.sum())
N_NEG_DEV = int(len(y_dev) - N_POS_DEV)

NEG_POS_RATIO = N_NEG_DEV / N_POS_DEV

print("=" * 90)
print("CLASS IMBALANCE")
print("=" * 90)

print(f"DEV positives      : {N_POS_DEV:,}")
print(f"DEV negatives      : {N_NEG_DEV:,}")
print(f"Positive prevalence: {y_dev.mean():.6f}")
print(f"Negative/positive ratio: {NEG_POS_RATIO:.6f}")

# Used later for models such as XGBoost.
SCALE_POS_WEIGHT = float(NEG_POS_RATIO)

# ------------------------------------------------------------
# 4.3 Feature-role definitions
# ------------------------------------------------------------

# Raw binary variables
LINEAR_BINARY_COLS = BINARY_COLS.copy()

# Numeric/ordered variables to standardize for linear / dense NN
LINEAR_SCALE_COLS = (
    ORDINAL_COLS
    + COUNT_COLS
    + CONTINUOUS_COLS
    + MULTICLASS_COLS
)

# Check complete feature coverage
covered_features = set(
    LINEAR_BINARY_COLS + LINEAR_SCALE_COLS
)

uncovered_features = [
    c for c in FEATURE_COLS
    if c not in covered_features
]

if uncovered_features:
    print(
        "WARNING: Features currently not assigned "
        f"to a preprocessing group: {uncovered_features}"
    )
else:
    print("\nAll predictors assigned to preprocessing groups.")

# ------------------------------------------------------------
# 4.4 Linear-model preprocessing
# ------------------------------------------------------------

def build_linear_preprocessor():
    """
    For:
        - Logistic Regression
        - ElasticNet Logistic Regression
        - LogReg_Subset
        - possible linear stackers

    Binary variables remain 0/1.
    Ordered/count/continuous variables are standardized.

    Every preprocessing object must be fitted inside a fold.
    """

    binary_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
    ])

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "binary",
                binary_pipe,
                LINEAR_BINARY_COLS
            ),
            (
                "scaled",
                numeric_pipe,
                LINEAR_SCALE_COLS
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# ------------------------------------------------------------
# 4.5 Dense-neural-network preprocessing
# ------------------------------------------------------------

def build_dense_nn_preprocessor():
    """
    For:
        - MLP
        - MLP-Focal
        - MLP-FGSM
        - MLP-Focal-FGSM

    Binary variables remain 0/1.
    Numeric / ordinal variables are standardized.
    """

    binary_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
    ])

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "binary",
                binary_pipe,
                LINEAR_BINARY_COLS
            ),
            (
                "scaled",
                numeric_pipe,
                LINEAR_SCALE_COLS
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# ------------------------------------------------------------
# 4.6 Tree-model preprocessing
# ------------------------------------------------------------

def build_tree_preprocessor():
    """
    For:
        - Random Forest
        - Extra Trees
        - HistGradientBoosting
        - XGBoost
        - LightGBM
        - CatBoost
        - AdaBoost if used

    Preserve original numeric scale and ordinal coding.
    Only impute if necessary.

    Current BRFSS data contains no missing values,
    but keeping an imputer makes the pipeline robust.
    """

    raw_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "raw",
                raw_pipe,
                FEATURE_COLS
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# ------------------------------------------------------------
# 4.7 Subset linear preprocessing
# ------------------------------------------------------------

def build_subset_linear_preprocessor():

    subset_binary = [
        c for c in SUBSET_FEATURES
        if c in BINARY_COLS
    ]

    subset_scale = [
        c for c in SUBSET_FEATURES
        if c not in subset_binary
    ]

    binary_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
    ])

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "binary",
                binary_pipe,
                subset_binary
            ),
            (
                "scaled",
                numeric_pipe,
                subset_scale
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# ------------------------------------------------------------
# 4.8 Deep tabular schema
# ------------------------------------------------------------
#
# DeepFM and AutoInt will NOT use the same representation
# as ordinary MLPs.
#
# Categorical-like variables receive embeddings.
# Continuous variables remain dense numeric inputs.
#
# We only define the schema here.
# Actual encoders are built inside their training modules.
# ------------------------------------------------------------

DEEP_CATEGORICAL_COLS = (
    BINARY_COLS
    + ORDINAL_COLS
    + MULTICLASS_COLS
)

DEEP_CONTINUOUS_COLS = (
    COUNT_COLS
    + CONTINUOUS_COLS
)

# Preserve FEATURE_COLS order where possible
DEEP_CATEGORICAL_COLS = [
    c for c in FEATURE_COLS
    if c in DEEP_CATEGORICAL_COLS
]

DEEP_CONTINUOUS_COLS = [
    c for c in FEATURE_COLS
    if c in DEEP_CONTINUOUS_COLS
]

# Cardinalities for embeddings
DEEP_CARDINALITIES = {}

for col in DEEP_CATEGORICAL_COLS:

    values = sorted(
        pd.Series(X_dev[col])
        .dropna()
        .unique()
        .tolist()
    )

    DEEP_CARDINALITIES[col] = {
        "n_unique": len(values),
        "values": values,
    }

print("\n" + "=" * 90)
print("DEEP TABULAR SCHEMA")
print("=" * 90)

print("Categorical/embedding features:")
print(DEEP_CATEGORICAL_COLS)

print("\nContinuous features:")
print(DEEP_CONTINUOUS_COLS)

print("\nEmbedding cardinalities:")

for c, info in DEEP_CARDINALITIES.items():
    print(
        f"{c:<20} "
        f"n_unique={info['n_unique']:<3} "
        f"values={info['values']}"
    )


# ------------------------------------------------------------
# 4.9 Fixed fold accessor
# ------------------------------------------------------------

def get_fold_indices(fold_id):
    """
    Returns LOCAL DEV indices.

    These indices refer to X_dev / y_dev,
    not the original dataframe.
    """

    if fold_id not in range(N_SPLITS):
        raise ValueError(
            f"fold_id must be 0 to {N_SPLITS - 1}"
        )

    valid_idx = np.where(
        fold_assignment == fold_id
    )[0]

    train_idx = np.where(
        fold_assignment != fold_id
    )[0]

    return train_idx, valid_idx


# ------------------------------------------------------------
# 4.10 Generic fold-preparation helper
# ------------------------------------------------------------

def prepare_fold_data(
    fold_id,
    preprocessor_type="linear",
    subset=False,
):
    """
    Fits preprocessing ONLY on the training portion
    of the requested DEV fold.

    Returns:
        fitted preprocessor
        X_train_transformed
        X_valid_transformed
        y_train
        y_valid
        train_idx
        valid_idx
    """

    train_idx, valid_idx = get_fold_indices(fold_id)

    if subset:

        cols = SUBSET_FEATURES

        X_tr = X_dev.iloc[
            train_idx
        ][cols].copy()

        X_va = X_dev.iloc[
            valid_idx
        ][cols].copy()

        preprocessor = (
            build_subset_linear_preprocessor()
        )

    else:

        X_tr = X_dev.iloc[
            train_idx
        ].copy()

        X_va = X_dev.iloc[
            valid_idx
        ].copy()

        if preprocessor_type == "linear":

            preprocessor = (
                build_linear_preprocessor()
            )

        elif preprocessor_type == "dense_nn":

            preprocessor = (
                build_dense_nn_preprocessor()
            )

        elif preprocessor_type == "tree":

            preprocessor = (
                build_tree_preprocessor()
            )

        else:
            raise ValueError(
                "preprocessor_type must be one of: "
                "'linear', 'dense_nn', 'tree'"
            )

    y_tr = y_dev.iloc[
        train_idx
    ].to_numpy(dtype=np.int64)

    y_va = y_dev.iloc[
        valid_idx
    ].to_numpy(dtype=np.int64)

    # CRITICAL:
    # fit on training fold ONLY
    X_tr_t = preprocessor.fit_transform(X_tr)

    # validation transformed using training-fitted transformer
    X_va_t = preprocessor.transform(X_va)

    return {
        "preprocessor": preprocessor,

        "X_train": np.asarray(
            X_tr_t,
            dtype=np.float32
        ),

        "X_valid": np.asarray(
            X_va_t,
            dtype=np.float32
        ),

        "y_train": y_tr,
        "y_valid": y_va,

        "train_idx": train_idx,
        "valid_idx": valid_idx,
    }


# ------------------------------------------------------------
# 4.11 Fold-0 infrastructure test
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("PREPROCESSING SANITY TEST — FOLD 0")
print("=" * 90)

linear_test = prepare_fold_data(
    fold_id=0,
    preprocessor_type="linear",
)

tree_test = prepare_fold_data(
    fold_id=0,
    preprocessor_type="tree",
)

subset_test = prepare_fold_data(
    fold_id=0,
    subset=True,
)

print("\nLinear representation:")
print(
    "Train:",
    linear_test["X_train"].shape,
    "| Valid:",
    linear_test["X_valid"].shape
)

print("\nTree representation:")
print(
    "Train:",
    tree_test["X_train"].shape,
    "| Valid:",
    tree_test["X_valid"].shape
)

print("\nSubset representation:")
print(
    "Train:",
    subset_test["X_train"].shape,
    "| Valid:",
    subset_test["X_valid"].shape
)

# ------------------------------------------------------------
# 4.12 Integrity assertions
# ------------------------------------------------------------

assert (
    linear_test["X_train"].shape[0]
    == len(linear_test["y_train"])
)

assert (
    linear_test["X_valid"].shape[0]
    == len(linear_test["y_valid"])
)

assert (
    tree_test["X_train"].shape[1]
    == len(FEATURE_COLS)
)

assert (
    subset_test["X_train"].shape[1]
    == len(SUBSET_FEATURES)
)

assert np.isfinite(
    linear_test["X_train"]
).all()

assert np.isfinite(
    linear_test["X_valid"]
).all()

assert np.isfinite(
    tree_test["X_train"]
).all()

assert np.isfinite(
    tree_test["X_valid"]
).all()

print("\nAll preprocessing integrity checks PASSED.")

# ------------------------------------------------------------
# 4.13 Feature-order manifests
# ------------------------------------------------------------

try:

    linear_feature_order = (
        linear_test["preprocessor"]
        .get_feature_names_out()
        .tolist()
    )

except Exception:

    linear_feature_order = (
        LINEAR_BINARY_COLS
        + LINEAR_SCALE_COLS
    )

try:

    tree_feature_order = (
        tree_test["preprocessor"]
        .get_feature_names_out()
        .tolist()
    )

except Exception:

    tree_feature_order = (
        FEATURE_COLS.copy()
    )

# ------------------------------------------------------------
# 4.14 Save preprocessing manifest
# ------------------------------------------------------------

preprocessing_manifest = {

    "linear_binary_cols":
        LINEAR_BINARY_COLS,

    "linear_scaled_cols":
        LINEAR_SCALE_COLS,

    "tree_cols":
        FEATURE_COLS,

    "subset_features":
        SUBSET_FEATURES,

    "deep_categorical_cols":
        DEEP_CATEGORICAL_COLS,

    "deep_continuous_cols":
        DEEP_CONTINUOUS_COLS,

    "deep_cardinalities":
        DEEP_CARDINALITIES,

    "linear_output_order":
        linear_feature_order,

    "tree_output_order":
        tree_feature_order,

    "scale_pos_weight":
        SCALE_POS_WEIGHT,
}

save_json(
    preprocessing_manifest,
    LOG_DIR / "module4_preprocessing_manifest.json"
)

# ------------------------------------------------------------
# 4.15 Save small summary table
# ------------------------------------------------------------

preprocessing_summary = pd.DataFrame([
    {
        "representation": "linear",
        "n_output_features":
            linear_test["X_train"].shape[1],
        "scaled": True,
        "intended_models":
            "LogReg, ElasticNet, linear stackers",
    },

    {
        "representation": "tree",
        "n_output_features":
            tree_test["X_train"].shape[1],
        "scaled": False,
        "intended_models":
            "RF, ET, HistGB, XGB, LGBM, CatBoost",
    },

    {
        "representation": "dense_nn",
        "n_output_features":
            linear_test["X_train"].shape[1],
        "scaled": True,
        "intended_models":
            "MLP, Focal MLP, FGSM MLP",
    },

    {
        "representation": "deep_tabular",
        "n_output_features":
            len(
                DEEP_CATEGORICAL_COLS
                + DEEP_CONTINUOUS_COLS
            ),
        "scaled": "continuous only",
        "intended_models":
            "DeepFM, AutoInt",
    },

    {
        "representation": "subset_linear",
        "n_output_features":
            subset_test["X_train"].shape[1],
        "scaled": True,
        "intended_models":
            "LogReg_Subset",
    },
])

display(preprocessing_summary)

save_csv(
    preprocessing_summary,
    TAB_DIR / "module4_preprocessing_summary.csv"
)

# ------------------------------------------------------------
# 4.16 Cleanup temporary transformed arrays
# ------------------------------------------------------------

del linear_test
del tree_test
del subset_test

gc.collect()

print("\nMODULE 4 COMPLETE.")

CLASS IMBALANCE
DEV positives      : 19,114
DEV negatives      : 183,830
Positive prevalence: 0.094184
Negative/positive ratio: 9.617558

All predictors assigned to preprocessing groups.

DEEP TABULAR SCHEMA
Categorical/embedding features:
['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

Continuous features:
['BMI', 'MentHlth', 'PhysHlth']

Embedding cardinalities:
HighBP               n_unique=2   values=[0.0, 1.0]
HighChol             n_unique=2   values=[0.0, 1.0]
CholCheck            n_unique=2   values=[0.0, 1.0]
Smoker               n_unique=2   values=[0.0, 1.0]
Stroke               n_unique=2   values=[0.0, 1.0]
Diabetes             n_unique=3   values=[0.0, 1.0, 2.0]
PhysActivity         n_unique=2   values=[0.0, 1.0]
Fruits               n_unique=2   values=[0.0, 1.0]
Veggies              n_unique=2   values=

,representation,n_output_features,scaled,intended_models
0,linear,21,True,"LogReg, ElasticNet, linear stackers"
1,tree,21,False,"RF, ET, HistGB, XGB, LGBM, CatBoost"
2,dense_nn,21,True,"MLP, Focal MLP, FGSM MLP"
3,deep_tabular,21,continuous only,"DeepFM, AutoInt"
4,subset_linear,12,True,LogReg_Subset



MODULE 4 COMPLETE.


In [3]:
# ============================================================
# MODULE 5 — UNIVERSAL METRICS, THRESHOLDS & EXPERIMENT ENGINE
# ============================================================
#
# Purpose:
# - one consistent metric implementation for every model
# - threshold selection using DEV/OOF only
# - ECE calculation
# - confusion-matrix-derived metrics
# - OOF/HOLDOUT prediction storage helpers
# - fold-level logging
# - experiment registry
#
# No model is trained in this module.
# ============================================================

import os
import gc
import json
import time
import math
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# 5.1 Safety checks
# ------------------------------------------------------------

required_vars = [
    "SEED",
    "EPS",
    "TARGET",
    "X_dev",
    "y_dev",
    "X_holdout",
    "y_holdout",
    "fold_assignment",
    "N_SPLITS",
    "PRED_DIR",
    "TAB_DIR",
    "LOG_DIR",
    "CONFIG_DIR",
    "save_json",
    "save_csv",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise ValueError(
        f"Module 5 missing required objects: {missing}"
    )

# ------------------------------------------------------------
# 5.2 Probability utilities
# ------------------------------------------------------------

def clip_probs(p, eps=EPS):
    """
    Clips probabilities to avoid:
    - log(0)
    - log(1)
    - numerical problems in NLL / logit transforms
    """
    p = np.asarray(p, dtype=float).reshape(-1)
    return np.clip(p, eps, 1.0 - eps)


def safe_logit(p, eps=EPS):
    p = clip_probs(p, eps)
    return np.log(p / (1.0 - p))


def sigmoid(x):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


# ------------------------------------------------------------
# 5.3 Expected Calibration Error
# ------------------------------------------------------------

def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=15,
    strategy="uniform"
):
    """
    Binary ECE.

    strategy:
        'uniform'  -> equal-width probability bins
        'quantile' -> approximately equal-sized bins
    """

    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = clip_probs(y_prob)

    if len(y_true) != len(y_prob):
        raise ValueError("y_true and y_prob length mismatch.")

    if strategy == "uniform":

        edges = np.linspace(
            0.0,
            1.0,
            n_bins + 1
        )

    elif strategy == "quantile":

        edges = np.quantile(
            y_prob,
            np.linspace(
                0.0,
                1.0,
                n_bins + 1
            )
        )

        edges = np.unique(edges)

        if len(edges) < 2:
            return 0.0

    else:
        raise ValueError(
            "strategy must be 'uniform' or 'quantile'"
        )

    ece = 0.0
    n = len(y_true)

    for i in range(len(edges) - 1):

        left = edges[i]
        right = edges[i + 1]

        if i == len(edges) - 2:
            mask = (
                (y_prob >= left)
                & (y_prob <= right)
            )
        else:
            mask = (
                (y_prob >= left)
                & (y_prob < right)
            )

        if mask.sum() == 0:
            continue

        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()

        ece += (
            mask.sum() / n
        ) * abs(
            bin_acc - bin_conf
        )

    return float(ece)


# ------------------------------------------------------------
# 5.4 Calibration table helper
# ------------------------------------------------------------

def calibration_table(
    y_true,
    y_prob,
    n_bins=15
):
    """
    Returns bin-level calibration diagnostics.
    Useful later for calibration plots.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    rows = []

    for i in range(n_bins):

        left = edges[i]
        right = edges[i + 1]

        if i == n_bins - 1:

            mask = (
                (y_prob >= left)
                & (y_prob <= right)
            )

        else:

            mask = (
                (y_prob >= left)
                & (y_prob < right)
            )

        n_bin = int(mask.sum())

        if n_bin == 0:
            continue

        rows.append({
            "bin": i,
            "lower": left,
            "upper": right,
            "n": n_bin,
            "mean_predicted_probability":
                float(y_prob[mask].mean()),
            "observed_positive_rate":
                float(y_true[mask].mean()),
            "absolute_gap":
                float(
                    abs(
                        y_prob[mask].mean()
                        - y_true[mask].mean()
                    )
                ),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 5.5 Confusion matrix metrics
# ------------------------------------------------------------

def confusion_metrics(
    y_true,
    y_prob,
    threshold
):
    """
    Produces threshold-dependent metrics
    and exact confusion counts.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else 0.0
    )

    f1 = (
        2.0 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "Threshold": float(threshold),

        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),

        "Precision": float(precision),
        "Recall": float(recall),
        "Specificity": float(specificity),
        "NPV": float(npv),
        "F1": float(f1),

        "MCC": float(
            matthews_corrcoef(
                y_true,
                y_pred
            )
        ),

        "Balanced_Accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "Accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "Predicted_Positive_Rate": float(
            y_pred.mean()
        ),
    }


# ------------------------------------------------------------
# 5.6 Threshold-independent metrics
# ------------------------------------------------------------

def probability_metrics(
    y_true,
    y_prob,
    ece_bins=15
):
    """
    Ranking + probability-quality metrics.

    These do NOT depend on the classification threshold.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)

    return {
        "PR_AUC": float(
            average_precision_score(
                y_true,
                y_prob
            )
        ),

        "ROC_AUC": float(
            roc_auc_score(
                y_true,
                y_prob
            )
        ),

        "Brier": float(
            brier_score_loss(
                y_true,
                y_prob
            )
        ),

        "NLL": float(
            log_loss(
                y_true,
                y_prob,
                labels=[0, 1]
            )
        ),

        "ECE": float(
            expected_calibration_error(
                y_true,
                y_prob,
                n_bins=ece_bins,
                strategy="uniform"
            )
        ),

        "Mean_Probability": float(
            y_prob.mean()
        ),

        "Observed_Prevalence": float(
            y_true.mean()
        ),
    }


# ------------------------------------------------------------
# 5.7 Complete evaluator
# ------------------------------------------------------------

def evaluate_binary_predictions(
    y_true,
    y_prob,
    threshold=0.5,
    ece_bins=15
):
    """
    Single authoritative evaluator.

    Every model in this project should
    eventually pass through this function.
    """

    metrics = probability_metrics(
        y_true,
        y_prob,
        ece_bins=ece_bins
    )

    metrics.update(
        confusion_metrics(
            y_true,
            y_prob,
            threshold
        )
    )

    return metrics


# ------------------------------------------------------------
# 5.8 Threshold sweep
# ------------------------------------------------------------

def threshold_sweep(
    y_true,
    y_prob,
    thresholds=None
):
    """
    Sweeps classification thresholds.

    IMPORTANT:
    PR-AUC / ROC-AUC are NOT included here
    because they do not change with threshold.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)

    if thresholds is None:
        thresholds = np.arange(
            0.01,
            0.501,
            0.005
        )

    rows = []

    for t in thresholds:

        m = confusion_metrics(
            y_true,
            y_prob,
            threshold=float(t)
        )

        rows.append(m)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 5.9 Threshold selectors
# ------------------------------------------------------------

def select_best_threshold(
    y_true,
    y_prob,
    objective="F1",
    thresholds=None,
    min_precision=None,
    min_recall=None,
):
    """
    DEV/OOF-only threshold selector.

    Supported objective:
        F1
        MCC
        Balanced_Accuracy
        Recall
        Precision
    """

    table = threshold_sweep(
        y_true,
        y_prob,
        thresholds=thresholds
    )

    filtered = table.copy()

    if min_precision is not None:

        filtered = filtered[
            filtered["Precision"]
            >= min_precision
        ]

    if min_recall is not None:

        filtered = filtered[
            filtered["Recall"]
            >= min_recall
        ]

    if len(filtered) == 0:

        raise ValueError(
            "No threshold satisfies the requested "
            "precision/recall constraint."
        )

    if objective not in filtered.columns:

        raise ValueError(
            f"Unknown threshold objective: {objective}"
        )

    # Stable deterministic tie-breaking:
    # 1. maximize objective
    # 2. maximize MCC
    # 3. maximize Balanced Accuracy
    # 4. prefer larger threshold
    ranked = filtered.sort_values(
        by=[
            objective,
            "MCC",
            "Balanced_Accuracy",
            "Threshold"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )

    best = ranked.iloc[0].to_dict()

    return (
        float(best["Threshold"]),
        best,
        table
    )


# ------------------------------------------------------------
# 5.10 Common threshold selection package
# ------------------------------------------------------------

def select_standard_thresholds(
    y_true,
    y_prob
):
    """
    Generates several candidate operating points.

    No clinical claim is attached to any of them.
    They are purely statistical operating points.
    """

    results = {}

    for objective in [
        "F1",
        "MCC",
        "Balanced_Accuracy"
    ]:

        t, best, _ = select_best_threshold(
            y_true,
            y_prob,
            objective=objective
        )

        results[objective] = {
            "threshold": t,
            "metrics": best
        }

    # Recall-constrained examples.
    # These are exploratory DEV operating points,
    # not "clinical thresholds".
    for target_recall in [
        0.60,
        0.70,
        0.80
    ]:

        key = (
            f"RecallAtLeast_"
            f"{int(target_recall * 100)}"
        )

        try:

            t, best, _ = select_best_threshold(
                y_true,
                y_prob,
                objective="Precision",
                min_recall=target_recall
            )

            results[key] = {
                "threshold": t,
                "metrics": best
            }

        except ValueError:

            results[key] = None

    return results


# ------------------------------------------------------------
# 5.11 Fold-wise metric evaluator
# ------------------------------------------------------------

def evaluate_by_fold(
    y_true,
    y_prob,
    fold_ids,
    threshold=None
):
    """
    Evaluates the same OOF vector separately
    on each fixed DEV fold.

    If threshold is None:
        threshold is selected separately inside
        each fold ONLY for descriptive diagnostics.

    Do NOT use these fold-specific thresholds
    as final operating thresholds.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)
    fold_ids = np.asarray(fold_ids)

    rows = []

    for fold in sorted(
        np.unique(fold_ids)
    ):

        mask = fold_ids == fold

        y_f = y_true[mask]
        p_f = y_prob[mask]

        if threshold is None:

            fold_threshold, _, _ = (
                select_best_threshold(
                    y_f,
                    p_f,
                    objective="F1"
                )
            )

        else:

            fold_threshold = threshold

        metrics = evaluate_binary_predictions(
            y_f,
            p_f,
            threshold=fold_threshold
        )

        metrics["Fold"] = int(fold)
        metrics["N"] = int(mask.sum())

        rows.append(metrics)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 5.12 Prediction-bank initialization
# ------------------------------------------------------------

OOF_BANK = pd.DataFrame({
    "dev_position":
        np.arange(len(y_dev)),
    "target":
        y_dev.to_numpy(dtype=np.int8),
    "fold":
        fold_assignment,
})

HOLDOUT_BANK = pd.DataFrame({
    "holdout_position":
        np.arange(len(y_holdout)),
    "target":
        y_holdout.to_numpy(dtype=np.int8),
})

print("=" * 90)
print("PREDICTION BANK INITIALIZED")
print("=" * 90)

print("OOF bank shape     :", OOF_BANK.shape)
print("HOLDOUT bank shape :", HOLDOUT_BANK.shape)


# ------------------------------------------------------------
# 5.13 Add model to prediction bank
# ------------------------------------------------------------

def add_predictions_to_bank(
    model_name,
    oof_pred,
    holdout_pred=None,
    overwrite=False
):
    """
    Adds model probabilities to the global
    OOF/HOLDOUT prediction banks.

    The holdout prediction is stored but should
    not be used for development decisions.
    """

    global OOF_BANK
    global HOLDOUT_BANK

    oof_pred = clip_probs(oof_pred)

    if len(oof_pred) != len(OOF_BANK):
        raise ValueError(
            f"{model_name}: OOF length mismatch."
        )

    if (
        model_name in OOF_BANK.columns
        and not overwrite
    ):
        raise ValueError(
            f"{model_name} already exists in OOF bank."
        )

    OOF_BANK[model_name] = oof_pred

    if holdout_pred is not None:

        holdout_pred = clip_probs(
            holdout_pred
        )

        if (
            len(holdout_pred)
            != len(HOLDOUT_BANK)
        ):
            raise ValueError(
                f"{model_name}: HOLDOUT length mismatch."
            )

        if (
            model_name in HOLDOUT_BANK.columns
            and not overwrite
        ):
            raise ValueError(
                f"{model_name} already exists "
                "in HOLDOUT bank."
            )

        HOLDOUT_BANK[
            model_name
        ] = holdout_pred

    # persist immediately
    save_csv(
        OOF_BANK,
        PRED_DIR / "oof_prediction_bank.csv"
    )

    save_csv(
        HOLDOUT_BANK,
        PRED_DIR / "holdout_prediction_bank.csv"
    )


# ------------------------------------------------------------
# 5.14 Experiment registry
# ------------------------------------------------------------

EXPERIMENT_REGISTRY_PATH = (
    LOG_DIR
    / "experiment_registry.csv"
)

EXPERIMENT_COLUMNS = [
    "experiment_id",
    "timestamp",
    "model_name",
    "family",
    "status",
    "runtime_seconds",
    "OOF_PR_AUC",
    "OOF_ROC_AUC",
    "OOF_Brier",
    "OOF_NLL",
    "OOF_ECE",
    "OOF_threshold",
    "OOF_F1",
    "OOF_MCC",
    "OOF_Recall",
    "OOF_Precision",
    "notes",
]


def load_experiment_registry():

    if EXPERIMENT_REGISTRY_PATH.exists():

        return pd.read_csv(
            EXPERIMENT_REGISTRY_PATH
        )

    return pd.DataFrame(
        columns=EXPERIMENT_COLUMNS
    )


def register_experiment(
    experiment_id,
    model_name,
    family,
    status,
    runtime_seconds=None,
    oof_metrics=None,
    notes=""
):
    """
    Append/update experiment registry.
    """

    registry = load_experiment_registry()

    row = {
        "experiment_id":
            experiment_id,

        "timestamp":
            datetime.now().isoformat(),

        "model_name":
            model_name,

        "family":
            family,

        "status":
            status,

        "runtime_seconds":
            runtime_seconds,

        "OOF_PR_AUC":
            None,

        "OOF_ROC_AUC":
            None,

        "OOF_Brier":
            None,

        "OOF_NLL":
            None,

        "OOF_ECE":
            None,

        "OOF_threshold":
            None,

        "OOF_F1":
            None,

        "OOF_MCC":
            None,

        "OOF_Recall":
            None,

        "OOF_Precision":
            None,

        "notes":
            notes,
    }

    if oof_metrics is not None:

        row.update({
            "OOF_PR_AUC":
                oof_metrics.get(
                    "PR_AUC"
                ),

            "OOF_ROC_AUC":
                oof_metrics.get(
                    "ROC_AUC"
                ),

            "OOF_Brier":
                oof_metrics.get(
                    "Brier"
                ),

            "OOF_NLL":
                oof_metrics.get(
                    "NLL"
                ),

            "OOF_ECE":
                oof_metrics.get(
                    "ECE"
                ),

            "OOF_threshold":
                oof_metrics.get(
                    "Threshold"
                ),

            "OOF_F1":
                oof_metrics.get(
                    "F1"
                ),

            "OOF_MCC":
                oof_metrics.get(
                    "MCC"
                ),

            "OOF_Recall":
                oof_metrics.get(
                    "Recall"
                ),

            "OOF_Precision":
                oof_metrics.get(
                    "Precision"
                ),
        })

    # Remove old copy of same experiment ID
    registry = registry[
        registry["experiment_id"]
        != experiment_id
    ]

    registry = pd.concat(
        [
            registry,
            pd.DataFrame([row])
        ],
        ignore_index=True
    )

    registry.to_csv(
        EXPERIMENT_REGISTRY_PATH,
        index=False
    )


# ------------------------------------------------------------
# 5.15 Model result package
# ------------------------------------------------------------

def build_model_result_package(
    model_name,
    family,
    oof_pred,
    threshold_objective="F1"
):
    """
    Creates all OOF evaluation outputs for
    one finished candidate model.

    HOLDOUT is intentionally absent here.
    """

    oof_pred = clip_probs(oof_pred)

    # threshold selected only from OOF
    threshold, threshold_metrics, sweep = (
        select_best_threshold(
            y_dev,
            oof_pred,
            objective=threshold_objective
        )
    )

    overall_metrics = (
        evaluate_binary_predictions(
            y_dev,
            oof_pred,
            threshold=threshold
        )
    )

    fold_metrics = evaluate_by_fold(
        y_dev,
        oof_pred,
        fold_assignment,
        threshold=threshold
    )

    return {
        "model_name":
            model_name,

        "family":
            family,

        "threshold":
            threshold,

        "metrics":
            overall_metrics,

        "fold_metrics":
            fold_metrics,

        "threshold_sweep":
            sweep,
    }


# ------------------------------------------------------------
# 5.16 Save model result package
# ------------------------------------------------------------

def save_model_result_package(
    package
):

    model_name = package[
        "model_name"
    ]

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
    )

    model_result_dir = (
        TAB_DIR
        / "model_results"
        / safe_name
    )

    model_result_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    save_json(
        package["metrics"],
        model_result_dir
        / "overall_metrics.json"
    )

    save_csv(
        package["fold_metrics"],
        model_result_dir
        / "fold_metrics.csv"
    )

    save_csv(
        package["threshold_sweep"],
        model_result_dir
        / "threshold_sweep.csv"
    )

    save_json(
        {
            "model_name":
                package["model_name"],

            "family":
                package["family"],

            "selected_threshold":
                package["threshold"],
        },
        model_result_dir
        / "model_summary.json"
    )


# ------------------------------------------------------------
# 5.17 Metric consistency checker
# ------------------------------------------------------------

def metric_consistency_check(metrics, tol=1e-10):
    """
    Checks whether reported F1 agrees exactly
    with reported precision and recall.
    """

    p = metrics["Precision"]
    r = metrics["Recall"]
    f1 = metrics["F1"]

    expected_f1 = (
        2 * p * r / (p + r)
        if (p + r) > 0
        else 0.0
    )

    difference = abs(
        f1 - expected_f1
    )

    return {
        "reported_F1": f1,
        "recomputed_F1": expected_f1,
        "absolute_difference": difference,
        "consistent": bool(
            difference <= tol
        )
    }


# ------------------------------------------------------------
# 5.18 Sanity test using dummy probabilities
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("METRIC ENGINE SANITY TEST")
print("=" * 90)

rng = np.random.default_rng(SEED)

dummy_prob = (
    0.05
    + 0.45 * y_dev.to_numpy()
    + rng.normal(
        0,
        0.08,
        size=len(y_dev)
    )
)

dummy_prob = clip_probs(
    dummy_prob
)

dummy_threshold, _, _ = (
    select_best_threshold(
        y_dev,
        dummy_prob,
        objective="F1"
    )
)

dummy_metrics = (
    evaluate_binary_predictions(
        y_dev,
        dummy_prob,
        threshold=dummy_threshold
    )
)

consistency = (
    metric_consistency_check(
        dummy_metrics
    )
)

print(
    "Dummy best F1 threshold:",
    dummy_threshold
)

print("\nDummy metrics:")

for key in [
    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "Balanced_Accuracy"
]:
    print(
        f"{key:<20}: "
        f"{dummy_metrics[key]:.6f}"
    )

print("\nF1 consistency:")
print(consistency)

assert consistency[
    "consistent"
]

# ------------------------------------------------------------
# 5.19 Save evaluation configuration
# ------------------------------------------------------------

evaluation_config = {
    "ece_bins": 15,

    "threshold_grid": {
        "start": 0.01,
        "stop": 0.50,
        "step": 0.005,
    },

    "primary_metric":
        "PR_AUC",

    "threshold_objectives": [
        "F1",
        "MCC",
        "Balanced_Accuracy",
    ],

    "threshold_selection_source":
        "DEV_OOF_ONLY",

    "holdout_threshold_tuning":
        False,
}

save_json(
    evaluation_config,
    CONFIG_DIR
    / "module5_evaluation_config.json"
)

print("\nMODULE 5 COMPLETE.")

PREDICTION BANK INITIALIZED
OOF bank shape     : (202944, 3)
HOLDOUT bank shape : (50736, 2)

METRIC ENGINE SANITY TEST
Dummy best F1 threshold: 0.30999999999999994

Dummy metrics:
PR_AUC              : 0.999643
ROC_AUC             : 0.999962
Brier               : 0.031244
NLL                 : 0.127448
ECE                 : 0.101955
Precision           : 0.994126
Recall              : 0.991629
F1                  : 0.992876
MCC                 : 0.992137
Balanced_Accuracy   : 0.995510

F1 consistency:
{'reported_F1': 0.9928758512310111, 'recomputed_F1': 0.9928758512310111, 'absolute_difference': 0.0, 'consistent': True}

MODULE 5 COMPLETE.


In [4]:
# ============================================================
# MODULE 6 — CLASSICAL + BOOSTING CANDIDATE MODEL BANK
# ============================================================


import gc
import time
import warnings
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 6.1 Safety checks
# ------------------------------------------------------------

required_vars = [
    "SEED",
    "N_SPLITS",
    "X_dev",
    "y_dev",
    "X_holdout",
    "y_holdout",
    "fold_assignment",
    "FEATURE_COLS",
    "SCALE_POS_WEIGHT",
    "build_linear_preprocessor",
    "build_tree_preprocessor",
    "build_subset_linear_preprocessor",
    "build_model_result_package",
    "save_model_result_package",
    "add_predictions_to_bank",
    "register_experiment",
    "evaluate_binary_predictions",
    "select_best_threshold",
    "PRED_DIR",
    "TAB_DIR",
    "LOG_DIR",
    "MODEL_DIR",
]

missing = [
    v for v in required_vars
    if v not in globals()
]

if missing:
    raise ValueError(
        f"Module 6 missing required objects: {missing}"
    )

# ------------------------------------------------------------
# 6.2 Optional package version check
# ------------------------------------------------------------

import sklearn
import xgboost
import lightgbm
import catboost

print("=" * 90)
print("PACKAGE VERSIONS")
print("=" * 90)

print("scikit-learn :", sklearn.__version__)
print("xgboost      :", xgboost.__version__)
print("lightgbm     :", lightgbm.__version__)
print("catboost     :", catboost.__version__)

# ------------------------------------------------------------
# 6.3 Model factories
# ------------------------------------------------------------
#
# These are reconstruction / strong baseline configurations.
# We are NOT doing aggressive hyperparameter optimization yet.
#
# Later architecture search should not be contaminated by
# repeatedly tuning models against HOLDOUT.
# ------------------------------------------------------------

def get_module6_model_factories():

    factories = {}

    # --------------------------------------------------------
    # Linear
    # --------------------------------------------------------

    factories["LogReg_L2"] = {
        "family": "linear",
        "preprocessor": "linear",
        "factory": lambda: LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
        ),
    }

    factories["LogReg_ElasticNet"] = {
        "family": "linear",
        "preprocessor": "linear",
        "factory": lambda: LogisticRegression(
            penalty="elasticnet",
            C=0.5,
            l1_ratio=0.5,
            solver="saga",
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1,
        ),
    }

    # --------------------------------------------------------
    # Bagging / trees
    # --------------------------------------------------------

    factories["RandomForest"] = {
        "family": "tree_bagging",
        "preprocessor": "tree",
        "factory": lambda: RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED,
        ),
    }

    factories["ExtraTrees"] = {
        "family": "tree_bagging",
        "preprocessor": "tree",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED,
        ),
    }

    # --------------------------------------------------------
    # HistGradientBoosting
    # --------------------------------------------------------
    #
    # This was one of the strongest models in our previous
    # experiments, so keep a strong but not over-tuned setup.
    # --------------------------------------------------------

    factories["HistGradientBoosting"] = {
        "family": "gradient_boosting",
        "preprocessor": "tree",
        "factory": lambda: HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=400,
            min_samples_leaf=40,
            l2_regularization=1.0,
            random_state=SEED,
        ),
    }

    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    factories["XGBoost"] = {
        "family": "gradient_boosting",
        "preprocessor": "tree",
        "factory": lambda: XGBClassifier(
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            scale_pos_weight=SCALE_POS_WEIGHT,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            device="cuda",
            random_state=SEED,
            n_jobs=-1,
        ),
    }

    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    factories["LightGBM"] = {
        "family": "gradient_boosting",
        "preprocessor": "tree",
        "factory": lambda: LGBMClassifier(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    }

    # --------------------------------------------------------
    # CatBoost
    # --------------------------------------------------------
    #
    # Keep manuscript configuration close to the reconstruction
    # target rather than replacing it immediately.
    # --------------------------------------------------------

    factories["CatBoost"] = {
        "family": "gradient_boosting",
        "preprocessor": "tree",
        "factory": lambda: CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            bagging_temperature=0.5,
            loss_function="Logloss",
            eval_metric="Logloss",
            auto_class_weights="Balanced",
            task_type="GPU",
            devices="0",
            verbose=False,
            random_seed=SEED,
        ),
    }

    return factories


MODEL_FACTORIES = get_module6_model_factories()

print("\nModule 6 model bank:")
for i, (name, cfg) in enumerate(
    MODEL_FACTORIES.items(),
    start=1
):
    print(
        f"{i:02d}. {name:<25} "
        f"| family={cfg['family']}"
    )

# ------------------------------------------------------------
# 6.4 Preprocessor factory
# ------------------------------------------------------------

def make_preprocessor(preprocessor_name):

    if preprocessor_name == "linear":
        return build_linear_preprocessor()

    elif preprocessor_name == "tree":
        return build_tree_preprocessor()

    else:
        raise ValueError(
            f"Unknown preprocessor: {preprocessor_name}"
        )


# ------------------------------------------------------------
# 6.5 Safe positive-probability predictor
# ------------------------------------------------------------

def predict_positive_probability(
    model,
    X
):
    """
    Returns P(y=1).

    All Module 6 models expose predict_proba.
    """

    if not hasattr(model, "predict_proba"):
        raise AttributeError(
            f"{type(model).__name__} "
            "does not provide predict_proba."
        )

    p = model.predict_proba(X)

    if p.ndim != 2 or p.shape[1] != 2:
        raise ValueError(
            f"Unexpected predict_proba shape: {p.shape}"
        )

    return clip_probs(
        p[:, 1]
    )


# ------------------------------------------------------------
# 6.6 Generic CV runner
# ------------------------------------------------------------

def run_sklearn_candidate_cv(
    model_name,
    model_config
):
    """
    Train one candidate model with the FIXED 5-fold DEV split.

    Produces:
        OOF predictions
        averaged HOLDOUT predictions
        fold-level diagnostics

    HOLDOUT predictions are never used to choose anything.
    """

    family = model_config["family"]
    preprocessor_name = (
        model_config["preprocessor"]
    )

    experiment_id = (
        f"M6_{model_name}"
    )

    print("\n" + "#" * 100)
    print(
        f"STARTING MODEL: {model_name}"
    )
    print(
        f"Family: {family}"
    )
    print("#" * 100)

    start_time = time.time()

    register_experiment(
        experiment_id=experiment_id,
        model_name=model_name,
        family=family,
        status="running",
        notes="Module 6 fixed-fold candidate model"
    )

    # --------------------------------------------------------
    # Prediction buffers
    # --------------------------------------------------------

    oof_pred = np.zeros(
        len(y_dev),
        dtype=np.float64
    )

    holdout_fold_preds = []

    fold_rows = []

    # --------------------------------------------------------
    # CV
    # --------------------------------------------------------

    for fold in range(N_SPLITS):

        fold_start = time.time()

        train_idx = np.where(
            fold_assignment != fold
        )[0]

        valid_idx = np.where(
            fold_assignment == fold
        )[0]

        X_train_raw = (
            X_dev.iloc[
                train_idx
            ].copy()
        )

        X_valid_raw = (
            X_dev.iloc[
                valid_idx
            ].copy()
        )

        y_train = (
            y_dev.iloc[
                train_idx
            ].to_numpy(dtype=np.int64)
        )

        y_valid = (
            y_dev.iloc[
                valid_idx
            ].to_numpy(dtype=np.int64)
        )

        # ----------------------------------------------------
        # Fit preprocessing on training fold ONLY
        # ----------------------------------------------------

        preprocessor = (
            make_preprocessor(
                preprocessor_name
            )
        )

        X_train = (
            preprocessor.fit_transform(
                X_train_raw
            )
        )

        X_valid = (
            preprocessor.transform(
                X_valid_raw
            )
        )

        X_hold = (
            preprocessor.transform(
                X_holdout
            )
        )

        X_train = np.asarray(
            X_train,
            dtype=np.float32
        )

        X_valid = np.asarray(
            X_valid,
            dtype=np.float32
        )

        X_hold = np.asarray(
            X_hold,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model = (
            model_config["factory"]()
        )

        model.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # Predict validation and holdout
        # ----------------------------------------------------

        p_valid = (
            predict_positive_probability(
                model,
                X_valid
            )
        )

        p_hold = (
            predict_positive_probability(
                model,
                X_hold
            )
        )

        oof_pred[
            valid_idx
        ] = p_valid

        holdout_fold_preds.append(
            p_hold
        )

        # ----------------------------------------------------
        # Fold diagnostics
        # ----------------------------------------------------

        fold_threshold, _, _ = (
            select_best_threshold(
                y_valid,
                p_valid,
                objective="F1"
            )
        )

        fold_metrics = (
            evaluate_binary_predictions(
                y_valid,
                p_valid,
                threshold=fold_threshold
            )
        )

        fold_runtime = (
            time.time()
            - fold_start
        )

        fold_row = {
            "Model":
                model_name,

            "Fold":
                fold,

            "Train_N":
                len(train_idx),

            "Valid_N":
                len(valid_idx),

            "Threshold_F1":
                fold_threshold,

            "Runtime_seconds":
                fold_runtime,
        }

        fold_row.update(
            fold_metrics
        )

        fold_rows.append(
            fold_row
        )

        print(
            f"Fold {fold + 1}/{N_SPLITS} "
            f"| PR={fold_metrics['PR_AUC']:.6f} "
            f"| ROC={fold_metrics['ROC_AUC']:.6f} "
            f"| Brier={fold_metrics['Brier']:.6f} "
            f"| ECE={fold_metrics['ECE']:.6f} "
            f"| F1={fold_metrics['F1']:.6f} "
            f"| {fold_runtime:.1f}s"
        )

        # ----------------------------------------------------
        # Cleanup fold memory
        # ----------------------------------------------------

        del model
        del preprocessor
        del X_train
        del X_valid
        del X_hold
        del X_train_raw
        del X_valid_raw

        gc.collect()

        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    # --------------------------------------------------------
    # Average HOLDOUT predictions across fold models
    # --------------------------------------------------------

    holdout_pred = np.mean(
        np.vstack(
            holdout_fold_preds
        ),
        axis=0
    )

    holdout_pred = clip_probs(
        holdout_pred
    )

    # --------------------------------------------------------
    # Verify complete OOF coverage
    # --------------------------------------------------------

    if not np.isfinite(
        oof_pred
    ).all():
        raise ValueError(
            f"{model_name}: "
            "OOF contains non-finite values."
        )

    if np.all(
        oof_pred == 0
    ):
        raise ValueError(
            f"{model_name}: "
            "OOF prediction vector appears empty."
        )

    # --------------------------------------------------------
    # OOF model result package
    # --------------------------------------------------------

    package = (
        build_model_result_package(
            model_name=model_name,
            family=family,
            oof_pred=oof_pred,
            threshold_objective="F1"
        )
    )

    save_model_result_package(
        package
    )

    # --------------------------------------------------------
    # Add predictions to persistent banks
    # --------------------------------------------------------

    add_predictions_to_bank(
        model_name=model_name,
        oof_pred=oof_pred,
        holdout_pred=holdout_pred,
        overwrite=True
    )

    # --------------------------------------------------------
    # Save standalone prediction files
    # --------------------------------------------------------

    np.save(
        PRED_DIR
        / f"{model_name}_OOF.npy",
        oof_pred
    )

    np.save(
        PRED_DIR
        / f"{model_name}_HOLDOUT.npy",
        holdout_pred
    )

    fold_df = pd.DataFrame(
        fold_rows
    )

    save_csv(
        fold_df,
        TAB_DIR
        / "module6_fold_metrics"
        / f"{model_name}.csv"
    )

    runtime = (
        time.time()
        - start_time
    )

    # --------------------------------------------------------
    # Register finished experiment
    # --------------------------------------------------------

    register_experiment(
        experiment_id=experiment_id,
        model_name=model_name,
        family=family,
        status="complete",
        runtime_seconds=runtime,
        oof_metrics=package["metrics"],
        notes=(
            "5-fold fixed OOF; "
            "holdout predictions stored but "
            "not used for selection"
        )
    )

    print("\nOOF SUMMARY")
    print(
        f"PR-AUC : "
        f"{package['metrics']['PR_AUC']:.6f}"
    )
    print(
        f"ROC-AUC: "
        f"{package['metrics']['ROC_AUC']:.6f}"
    )
    print(
        f"Brier  : "
        f"{package['metrics']['Brier']:.6f}"
    )
    print(
        f"NLL    : "
        f"{package['metrics']['NLL']:.6f}"
    )
    print(
        f"ECE    : "
        f"{package['metrics']['ECE']:.6f}"
    )
    print(
        f"F1     : "
        f"{package['metrics']['F1']:.6f}"
    )
    print(
        f"MCC    : "
        f"{package['metrics']['MCC']:.6f}"
    )
    print(
        f"Threshold: "
        f"{package['threshold']:.6f}"
    )
    print(
        f"Runtime: {runtime / 60:.2f} min"
    )

    return {
        "model_name":
            model_name,

        "oof_pred":
            oof_pred,

        "holdout_pred":
            holdout_pred,

        "metrics":
            package["metrics"],

        "threshold":
            package["threshold"],

        "fold_metrics":
            fold_df,

        "runtime_seconds":
            runtime,
    }


# ------------------------------------------------------------
# 6.7 Runner with resume support
# ------------------------------------------------------------

MODULE6_RESULTS = {}

for model_name, model_config in MODEL_FACTORIES.items():

    oof_file = (
        PRED_DIR
        / f"{model_name}_OOF.npy"
    )

    hold_file = (
        PRED_DIR
        / f"{model_name}_HOLDOUT.npy"
    )

    if (
        oof_file.exists()
        and hold_file.exists()
    ):

        print(
            f"\n[RESUME] Existing predictions found "
            f"for {model_name}. Loading instead of retraining."
        )

        oof_pred = np.load(
            oof_file
        )

        hold_pred = np.load(
            hold_file
        )

        package = (
            build_model_result_package(
                model_name=model_name,
                family=model_config["family"],
                oof_pred=oof_pred,
                threshold_objective="F1"
            )
        )

        add_predictions_to_bank(
            model_name=model_name,
            oof_pred=oof_pred,
            holdout_pred=hold_pred,
            overwrite=True
        )

        MODULE6_RESULTS[
            model_name
        ] = {
            "model_name":
                model_name,

            "oof_pred":
                oof_pred,

            "holdout_pred":
                hold_pred,

            "metrics":
                package["metrics"],

            "threshold":
                package["threshold"],
        }

        continue

    MODULE6_RESULTS[
        model_name
    ] = run_sklearn_candidate_cv(
        model_name=model_name,
        model_config=model_config
    )

# ------------------------------------------------------------
# 6.8 Build Module 6 OOF comparison table
# ------------------------------------------------------------

comparison_rows = []

for model_name, result in (
    MODULE6_RESULTS.items()
):

    m = result["metrics"]

    comparison_rows.append({
        "Model":
            model_name,

        "PR_AUC":
            m["PR_AUC"],

        "ROC_AUC":
            m["ROC_AUC"],

        "Brier":
            m["Brier"],

        "NLL":
            m["NLL"],

        "ECE":
            m["ECE"],

        "Precision":
            m["Precision"],

        "Recall":
            m["Recall"],

        "F1":
            m["F1"],

        "MCC":
            m["MCC"],

        "Balanced_Accuracy":
            m["Balanced_Accuracy"],

        "Threshold":
            m["Threshold"],
    })

module6_comparison = (
    pd.DataFrame(
        comparison_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("MODULE 6 — OOF MODEL COMPARISON")
print("=" * 100)

display(
    module6_comparison
)

save_csv(
    module6_comparison,
    TAB_DIR
    / "module6_oof_model_comparison.csv"
)

# ------------------------------------------------------------
# 6.9 Fold stability table
# ------------------------------------------------------------

stability_rows = []

for model_name, result in (
    MODULE6_RESULTS.items()
):

    if "fold_metrics" not in result:
        continue

    fdf = result[
        "fold_metrics"
    ]

    stability_rows.append({
        "Model":
            model_name,

        "PR_AUC_mean":
            fdf["PR_AUC"].mean(),

        "PR_AUC_std":
            fdf["PR_AUC"].std(),

        "ROC_AUC_mean":
            fdf["ROC_AUC"].mean(),

        "ROC_AUC_std":
            fdf["ROC_AUC"].std(),

        "Brier_mean":
            fdf["Brier"].mean(),

        "Brier_std":
            fdf["Brier"].std(),

        "ECE_mean":
            fdf["ECE"].mean(),

        "ECE_std":
            fdf["ECE"].std(),
    })

module6_stability = (
    pd.DataFrame(
        stability_rows
    )
    .sort_values(
        "PR_AUC_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("MODULE 6 — FOLD STABILITY")
print("=" * 100)

display(
    module6_stability
)

save_csv(
    module6_stability,
    TAB_DIR
    / "module6_fold_stability.csv"
)

# ------------------------------------------------------------
# 6.10 Final integrity check
# ------------------------------------------------------------

expected_models = list(
    MODEL_FACTORIES.keys()
)

missing_oof = [
    m for m in expected_models
    if m not in OOF_BANK.columns
]

missing_hold = [
    m for m in expected_models
    if m not in HOLDOUT_BANK.columns
]

if missing_oof:
    raise ValueError(
        f"Missing OOF models: {missing_oof}"
    )

if missing_hold:
    raise ValueError(
        f"Missing HOLDOUT models: {missing_hold}"
    )

print("\nPrediction bank integrity: PASSED")
print(
    "OOF bank:",
    OOF_BANK.shape
)
print(
    "HOLDOUT bank:",
    HOLDOUT_BANK.shape
)

print("\nMODULE 6 COMPLETE.")

PACKAGE VERSIONS
scikit-learn : 1.6.1
xgboost      : 3.4.1
lightgbm     : 4.6.0
catboost     : 1.2.10

Module 6 model bank:
01. LogReg_L2                 | family=linear
02. LogReg_ElasticNet         | family=linear
03. RandomForest              | family=tree_bagging
04. ExtraTrees                | family=tree_bagging
05. HistGradientBoosting      | family=gradient_boosting
06. XGBoost                   | family=gradient_boosting
07. LightGBM                  | family=gradient_boosting
08. CatBoost                  | family=gradient_boosting

####################################################################################################
STARTING MODEL: LogReg_L2
Family: linear
####################################################################################################
Fold 1/5 | PR=0.370689 | ROC=0.848325 | Brier=0.165928 | ECE=0.258900 | F1=0.378059 | 1.9s
Fold 2/5 | PR=0.373908 | ROC=0.847781 | Brier=0.166037 | ECE=0.259131 | F1=0.372919 | 1.7s
Fold 3/5 | PR=0.370587 | R

,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Balanced_Accuracy,Threshold
0,CatBoost,0.377535,0.849216,0.161784,0.475348,0.243946,0.242962,0.804593,0.373222,0.342935,0.771962,0.500
1,HistGradientBoosting,0.377012,0.849628,0.069876,0.236194,0.002378,0.348113,0.532332,0.420950,0.356610,0.714341,0.215
2,LightGBM,0.372779,0.846576,0.157540,0.463084,0.233591,0.245828,0.791514,0.375144,0.342517,0.769515,0.500
3,LogReg_ElasticNet,0.371730,0.847375,0.164999,0.495340,0.256907,0.246427,0.792822,0.375988,0.343676,0.770368,0.500
4,LogReg_L2,0.371729,0.847376,0.165008,0.495369,0.256929,0.246410,0.792770,0.375963,0.343639,0.770339,0.500
5,XGBoost,0.364962,0.842457,0.153124,0.452376,0.223666,0.250136,0.771058,0.377733,0.341505,0.765358,0.500
6,ExtraTrees,0.337513,0.833682,0.106182,0.335571,0.141975,0.309558,0.544732,0.394775,0.328657,0.709201,0.490
7,RandomForest,0.337500,0.835684,0.084384,0.278252,0.080645,0.302199,0.586010,0.398761,0.336646,0.722658,0.350



MODULE 6 — FOLD STABILITY


,Model,PR_AUC_mean,PR_AUC_std,ROC_AUC_mean,ROC_AUC_std,Brier_mean,Brier_std,ECE_mean,ECE_std
0,CatBoost,0.378003,0.006957,0.849237,0.001945,0.161784,0.001225,0.243946,0.002300
1,HistGradientBoosting,0.377609,0.006088,0.849650,0.001061,0.069876,0.000254,0.003578,0.000672
2,LightGBM,0.373329,0.007186,0.846605,0.001623,0.157540,0.001347,0.233591,0.002431
3,LogReg_L2,0.372174,0.005704,0.847394,0.001318,0.165008,0.001251,0.256929,0.002074
4,LogReg_ElasticNet,0.372174,0.005706,0.847394,0.001318,0.164999,0.001251,0.256907,0.002073
5,XGBoost,0.365437,0.007091,0.842491,0.002282,0.153124,0.000980,0.223666,0.002017
6,RandomForest,0.337925,0.003774,0.835707,0.001854,0.084384,0.000470,0.080649,0.001424
7,ExtraTrees,0.337869,0.004939,0.833707,0.001629,0.106182,0.000767,0.141975,0.002003



Prediction bank integrity: PASSED
OOF bank: (202944, 11)
HOLDOUT bank: (50736, 10)

MODULE 6 COMPLETE.


In [5]:
# ============================================================
# MODULE 5 PATCH — FULL THRESHOLD RANGE
# ============================================================

def threshold_sweep(
    y_true,
    y_prob,
    thresholds=None
):
    """
    Sweeps the full probability range.

    Ranking/calibration metrics are threshold-independent
    and therefore are not computed here.
    """

    y_true = np.asarray(y_true).astype(int)
    y_prob = clip_probs(y_prob)

    if thresholds is None:
        thresholds = np.arange(
            0.005,
            0.996,
            0.005
        )

    rows = []

    for t in thresholds:
        rows.append(
            confusion_metrics(
                y_true,
                y_prob,
                threshold=float(t)
            )
        )

    return pd.DataFrame(rows)


# update saved evaluation configuration
evaluation_config["threshold_grid"] = {
    "start": 0.005,
    "stop": 0.995,
    "step": 0.005,
}

save_json(
    evaluation_config,
    CONFIG_DIR / "module5_evaluation_config.json"
)

print("Threshold grid patched: 0.005 → 0.995")

# ============================================================
# MODULE 6 PATCH — RECOMPUTE THRESHOLD METRICS
# NO MODEL RETRAINING
# ============================================================

comparison_rows = []
patched_results = {}

for model_name, model_config in MODEL_FACTORIES.items():

    oof_file = PRED_DIR / f"{model_name}_OOF.npy"

    if not oof_file.exists():
        raise FileNotFoundError(oof_file)

    oof_pred = np.load(oof_file)

    package = build_model_result_package(
        model_name=model_name,
        family=model_config["family"],
        oof_pred=oof_pred,
        threshold_objective="F1"
    )

    save_model_result_package(package)

    patched_results[model_name] = package

    m = package["metrics"]

    comparison_rows.append({
        "Model": model_name,
        "PR_AUC": m["PR_AUC"],
        "ROC_AUC": m["ROC_AUC"],
        "Brier": m["Brier"],
        "NLL": m["NLL"],
        "ECE": m["ECE"],
        "Precision": m["Precision"],
        "Recall": m["Recall"],
        "F1": m["F1"],
        "MCC": m["MCC"],
        "Balanced_Accuracy": m["Balanced_Accuracy"],
        "Threshold": m["Threshold"],
    })

module6_comparison_patched = (
    pd.DataFrame(comparison_rows)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

display(module6_comparison_patched)

save_csv(
    module6_comparison_patched,
    TAB_DIR / "module6_oof_model_comparison_patched.csv"
)

print("MODULE 6 threshold-dependent metrics recomputed.")

Threshold grid patched: 0.005 → 0.995


,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Balanced_Accuracy,Threshold
0,CatBoost,0.377535,0.849216,0.161784,0.475348,0.243946,0.328529,0.581982,0.419980,0.359039,0.729151,0.690
1,HistGradientBoosting,0.377012,0.849628,0.069876,0.236194,0.002378,0.348113,0.532332,0.420950,0.356610,0.714341,0.215
2,LightGBM,0.372779,0.846576,0.157540,0.463084,0.233591,0.345198,0.530240,0.418162,0.353438,0.712830,0.715
3,LogReg_ElasticNet,0.371730,0.847375,0.164999,0.495340,0.256907,0.329699,0.566339,0.416771,0.354242,0.723310,0.690
4,LogReg_L2,0.371729,0.847376,0.165008,0.495369,0.256929,0.329730,0.566548,0.416853,0.354347,0.723401,0.690
5,XGBoost,0.364962,0.842457,0.153124,0.452376,0.223666,0.333603,0.538767,0.412060,0.347180,0.713433,0.695
6,ExtraTrees,0.337513,0.833682,0.106182,0.335571,0.141975,0.317276,0.522392,0.394781,0.327216,0.702756,0.505
7,RandomForest,0.337500,0.835684,0.084384,0.278252,0.080645,0.302199,0.586010,0.398761,0.336646,0.722658,0.350


MODULE 6 threshold-dependent metrics recomputed.


In [6]:
# ============================================================
# MODULE 7A — UNIVERSAL PYTORCH TRAINING INFRASTRUCTURE
# ============================================================
#
# Supports:
# - BCE MLP
# - Focal-loss MLP
# - FGSM MLP
# - Focal + FGSM MLP
# - later DeepFM / AutoInt
#
# Features:
# - fixed 5-fold OOF
# - mixed precision on T4
# - early stopping using validation PR-AUC
# - reproducible DataLoader
# - GPU checkpointing
# - fold-wise prediction
# - constrained FGSM
#
# ============================================================

import gc
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd

from pathlib import Path

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    average_precision_score
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 7A.1 Safety
# ------------------------------------------------------------

required_vars = [
    "SEED",
    "N_SPLITS",
    "SCALE_POS_WEIGHT",
    "X_dev",
    "y_dev",
    "X_holdout",
    "y_holdout",
    "fold_assignment",
    "build_dense_nn_preprocessor",
    "build_model_result_package",
    "save_model_result_package",
    "add_predictions_to_bank",
    "register_experiment",
    "select_best_threshold",
    "evaluate_binary_predictions",
    "PRED_DIR",
    "TAB_DIR",
    "LOG_DIR",
    "CHECKPOINT_DIR",
]

missing = [
    v for v in required_vars
    if v not in globals()
]

if missing:
    raise ValueError(
        f"Module 7A missing: {missing}"
    )

# ------------------------------------------------------------
# 7A.2 Device
# ------------------------------------------------------------

TORCH_DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    torch.cuda.is_available()
)

print("=" * 90)
print("PYTORCH TRAINING CONFIGURATION")
print("=" * 90)

print("Device :", TORCH_DEVICE)
print("AMP    :", USE_AMP)

if torch.cuda.is_available():
    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )

# ------------------------------------------------------------
# 7A.3 Reproducibility helper
# ------------------------------------------------------------

def seed_everything(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # reproducibility > a tiny speed gain
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()

# ------------------------------------------------------------
# 7A.4 DataLoader worker seeding
# ------------------------------------------------------------

def seed_worker(worker_id):

    worker_seed = (
        SEED + worker_id
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator():

    g = torch.Generator()
    g.manual_seed(SEED)

    return g

# ------------------------------------------------------------
# 7A.5 Dataset
# ------------------------------------------------------------

class TabularTensorDataset(Dataset):

    def __init__(self, X, y=None):

        self.X = torch.as_tensor(
            X,
            dtype=torch.float32
        )

        if y is None:
            self.y = None
        else:
            self.y = torch.as_tensor(
                y,
                dtype=torch.float32
            )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        if self.y is None:
            return self.X[idx]

        return (
            self.X[idx],
            self.y[idx]
        )

# ------------------------------------------------------------
# 7A.6 Focal loss
# ------------------------------------------------------------

class BinaryFocalLoss(nn.Module):

    def __init__(
        self,
        alpha=0.50,
        gamma=2.0
    ):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(
        self,
        logits,
        targets
    ):

        logits = logits.reshape(-1)
        targets = targets.reshape(-1)

        probs = torch.sigmoid(logits)

        pt = torch.where(
            targets == 1,
            probs,
            1.0 - probs
        )

        alpha_t = torch.where(
            targets == 1,
            torch.full_like(
                targets,
                self.alpha
            ),
            torch.full_like(
                targets,
                1.0 - self.alpha
            )
        )

        loss = (
            -alpha_t
            * ((1.0 - pt) ** self.gamma)
            * torch.log(
                pt.clamp(min=1e-7)
            )
        )

        return loss.mean()

# ------------------------------------------------------------
# 7A.7 MLP architecture
# ------------------------------------------------------------

class TabularMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims=(128, 64),
        dropout=0.30
    ):

        super().__init__()

        layers = []

        prev = input_dim

        for h in hidden_dims:

            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])

            prev = h

        layers.append(
            nn.Linear(prev, 1)
        )

        self.net = nn.Sequential(
            *layers
        )

    def forward(self, x):

        return self.net(x).squeeze(1)

# ------------------------------------------------------------
# 7A.8 Loss factory
# ------------------------------------------------------------

def make_loss(
    loss_name,
    device
):

    if loss_name == "weighted_bce":

        pos_weight = torch.tensor(
            [SCALE_POS_WEIGHT],
            dtype=torch.float32,
            device=device
        )

        return nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

    elif loss_name == "bce":

        return nn.BCEWithLogitsLoss()

    elif loss_name == "focal":

        return BinaryFocalLoss(
            alpha=0.50,
            gamma=2.0
        )

    else:

        raise ValueError(
            f"Unknown loss: {loss_name}"
        )

# ------------------------------------------------------------
# 7A.9 Prediction helper
# ------------------------------------------------------------

@torch.no_grad()
def torch_predict_proba(
    model,
    X,
    batch_size=4096
):

    dataset = TabularTensorDataset(
        X,
        y=None
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    model.eval()

    preds = []

    for xb in loader:

        xb = xb.to(
            TORCH_DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(xb)

        p = torch.sigmoid(
            logits.float()
        )

        preds.append(
            p.cpu().numpy()
        )

    return clip_probs(
        np.concatenate(preds)
    )

# ------------------------------------------------------------
# 7A.10 FGSM perturbation
# ------------------------------------------------------------

def constrained_fgsm(
    model,
    xb,
    yb,
    loss_fn,
    epsilon,
    attack_mask,
    lower_bound,
    upper_bound
):
    """
    Constraint-aware adversarial perturbation.

    IMPORTANT:
    Binary variables are NOT perturbed.

    Only standardized non-binary features
    receive FGSM perturbations.
    """

    x_adv = (
        xb.detach()
        .clone()
        .requires_grad_(True)
    )

    # use float32 here for stable gradients
    logits = model(x_adv)

    loss = loss_fn(
        logits,
        yb
    )

    model.zero_grad(
        set_to_none=True
    )

    loss.backward()

    grad_sign = (
        x_adv.grad
        .detach()
        .sign()
    )

    perturbation = (
        epsilon
        * grad_sign
        * attack_mask
    )

    x_adv = (
        x_adv.detach()
        + perturbation
    )

    # keep every dimension within values observed
    # in the current training fold
    x_adv = torch.maximum(
        x_adv,
        lower_bound
    )

    x_adv = torch.minimum(
        x_adv,
        upper_bound
    )

    return x_adv.detach()

# ------------------------------------------------------------
# 7A.11 Main neural training function
# ------------------------------------------------------------

def train_mlp_fold(
    model_name,
    fold_id,
    config
):

    seed_everything(
        SEED + fold_id
    )

    train_idx = np.where(
        fold_assignment != fold_id
    )[0]

    valid_idx = np.where(
        fold_assignment == fold_id
    )[0]

    # --------------------------------------------------------
    # Fold-safe preprocessing
    # --------------------------------------------------------

    prep = (
        build_dense_nn_preprocessor()
    )

    X_train = prep.fit_transform(
        X_dev.iloc[train_idx]
    )

    X_valid = prep.transform(
        X_dev.iloc[valid_idx]
    )

    X_hold = prep.transform(
        X_holdout
    )

    X_train = np.asarray(
        X_train,
        dtype=np.float32
    )

    X_valid = np.asarray(
        X_valid,
        dtype=np.float32
    )

    X_hold = np.asarray(
        X_hold,
        dtype=np.float32
    )

    y_train = (
        y_dev.iloc[train_idx]
        .to_numpy(dtype=np.float32)
    )

    y_valid = (
        y_dev.iloc[valid_idx]
        .to_numpy(dtype=np.float32)
    )

    # --------------------------------------------------------
    # Dataset / loaders
    # --------------------------------------------------------

    train_ds = TabularTensorDataset(
        X_train,
        y_train
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=make_generator(),
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = TabularMLP(
        input_dim=X_train.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(TORCH_DEVICE)

    loss_fn = make_loss(
        config["loss"],
        TORCH_DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    # --------------------------------------------------------
    # FGSM masks / legal bounds
    # --------------------------------------------------------

    n_binary = len(
        LINEAR_BINARY_COLS
    )

    attack_mask_np = np.zeros(
        X_train.shape[1],
        dtype=np.float32
    )

    # Only non-binary dimensions
    attack_mask_np[
        n_binary:
    ] = 1.0

    attack_mask = torch.tensor(
        attack_mask_np,
        dtype=torch.float32,
        device=TORCH_DEVICE
    ).view(1, -1)

    lower_bound = torch.tensor(
        X_train.min(axis=0),
        dtype=torch.float32,
        device=TORCH_DEVICE
    ).view(1, -1)

    upper_bound = torch.tensor(
        X_train.max(axis=0),
        dtype=torch.float32,
        device=TORCH_DEVICE
    ).view(1, -1)

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    best_pr = -np.inf
    best_epoch = -1
    best_state = None

    patience_counter = 0

    history = []

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    max_epochs = config[
        "max_epochs"
    ]

    patience = config[
        "patience"
    ]

    # --------------------------------------------------------
    # Epochs
    # --------------------------------------------------------

    for epoch in range(
        1,
        max_epochs + 1
    ):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for xb, yb in train_loader:

            xb = xb.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            # -----------------------------------------------
            # Standard update
            # -----------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = model(xb)

                clean_loss = loss_fn(
                    logits,
                    yb
                )

            # -----------------------------------------------
            # Adversarial training if requested
            # -----------------------------------------------

            if config["fgsm"]:

                # Generate adversarial examples
                # in full precision.
                with torch.autocast(
                    device_type="cuda",
                    enabled=False
                ):

                    x_adv = constrained_fgsm(
                        model=model,
                        xb=xb.float(),
                        yb=yb.float(),
                        loss_fn=loss_fn,
                        epsilon=config[
                            "epsilon"
                        ],
                        attack_mask=attack_mask,
                        lower_bound=lower_bound,
                        upper_bound=upper_bound,
                    )

                # reset gradients produced while constructing attack
                optimizer.zero_grad(
                    set_to_none=True
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=USE_AMP
                ):

                    logits_clean = model(xb)
                    logits_adv = model(x_adv)

                    loss_clean = loss_fn(
                        logits_clean,
                        yb
                    )

                    loss_adv = loss_fn(
                        logits_adv,
                        yb
                    )

                    loss = (
                        config[
                            "clean_weight"
                        ]
                        * loss_clean
                        +
                        config[
                            "adv_weight"
                        ]
                        * loss_adv
                    )

            else:

                loss = clean_loss

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

            batch_n = len(xb)

            running_loss += (
                float(loss.detach().cpu())
                * batch_n
            )

            n_seen += batch_n

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        p_valid = torch_predict_proba(
            model,
            X_valid,
            batch_size=4096
        )

        valid_pr = (
            average_precision_score(
                y_valid.astype(int),
                p_valid
            )
        )

        epoch_loss = (
            running_loss
            / max(n_seen, 1)
        )

        history.append({
            "epoch": epoch,
            "train_loss": epoch_loss,
            "valid_PR_AUC": valid_pr,
        })

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if valid_pr > (
            best_pr
            + config["min_delta"]
        ):

            best_pr = valid_pr
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            epoch == 1
            or epoch % 5 == 0
            or patience_counter >= patience
        ):

            print(
                f"    epoch={epoch:02d} "
                f"loss={epoch_loss:.5f} "
                f"val_PR={valid_pr:.6f} "
                f"best={best_pr:.6f}"
            )

        if patience_counter >= patience:
            break

    # --------------------------------------------------------
    # Restore best checkpoint
    # --------------------------------------------------------

    if best_state is None:
        raise RuntimeError(
            "No valid checkpoint created."
        )

    model.load_state_dict(
        best_state
    )

    # --------------------------------------------------------
    # Final fold predictions
    # --------------------------------------------------------

    p_valid = torch_predict_proba(
        model,
        X_valid
    )

    p_hold = torch_predict_proba(
        model,
        X_hold
    )

    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    ckpt_dir = (
        CHECKPOINT_DIR
        / model_name
    )

    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    torch.save(
        {
            "state_dict":
                best_state,

            "input_dim":
                X_train.shape[1],

            "config":
                config,

            "fold":
                fold_id,

            "best_epoch":
                best_epoch,

            "best_valid_PR_AUC":
                best_pr,
        },
        ckpt_dir
        / f"fold_{fold_id}.pt"
    )

    history_df = pd.DataFrame(
        history
    )

    save_csv(
        history_df,
        TAB_DIR
        / "module7_training_history"
        / model_name
        / f"fold_{fold_id}.csv"
    )

    result = {
        "valid_idx":
            valid_idx,

        "p_valid":
            p_valid,

        "p_hold":
            p_hold,

        "best_epoch":
            best_epoch,

        "best_PR":
            best_pr,

        "history":
            history_df,
    }

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del model
    del prep
    del X_train
    del X_valid
    del X_hold
    del train_ds
    del train_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


print("\nMODULE 7A COMPLETE.")

# ============================================================
# MODULE 7B — MLP SPECIALIST MODEL BANK
# ============================================================

import time
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 7B.1 Configurations
# ------------------------------------------------------------

MLP_CONFIGS = {

    "MLP_BCE": {
        "family":
            "deep_mlp",

        "hidden_dims":
            (128, 64),

        "dropout":
            0.30,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "batch_size":
            512,

        "max_epochs":
            35,

        "patience":
            6,

        "min_delta":
            1e-4,

        "loss":
            "weighted_bce",

        "fgsm":
            False,

        "epsilon":
            0.0,

        "clean_weight":
            1.0,

        "adv_weight":
            0.0,
    },


    "MLP_Focal": {
        "family":
            "deep_mlp",

        "hidden_dims":
            (64, 32),

        "dropout":
            0.30,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "batch_size":
            512,

        "max_epochs":
            35,

        "patience":
            6,

        "min_delta":
            1e-4,

        "loss":
            "focal",

        "fgsm":
            False,

        "epsilon":
            0.0,

        "clean_weight":
            1.0,

        "adv_weight":
            0.0,
    },


    "MLP_FGSM": {
        "family":
            "deep_mlp",

        "hidden_dims":
            (128, 64),

        "dropout":
            0.30,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "batch_size":
            512,

        "max_epochs":
            35,

        "patience":
            6,

        "min_delta":
            1e-4,

        "loss":
            "weighted_bce",

        "fgsm":
            True,

        # manuscript-aligned strength
        "epsilon":
            0.05,

        # conservative adversarial mixture
        "clean_weight":
            0.50,

        "adv_weight":
            0.50,
    },


    "MLP_Focal_FGSM": {
        "family":
            "deep_mlp",

        "hidden_dims":
            (128, 64),

        "dropout":
            0.30,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "batch_size":
            512,

        "max_epochs":
            35,

        "patience":
            6,

        "min_delta":
            1e-4,

        "loss":
            "focal",

        "fgsm":
            True,

        "epsilon":
            0.05,

        "clean_weight":
            0.50,

        "adv_weight":
            0.50,
    },
}

# ------------------------------------------------------------
# 7B.2 Generic model runner
# ------------------------------------------------------------

def run_mlp_candidate(
    model_name,
    config
):

    print("\n" + "#" * 100)
    print(
        f"MODEL: {model_name}"
    )
    print("#" * 100)

    start = time.time()

    experiment_id = (
        f"M7_{model_name}"
    )

    register_experiment(
        experiment_id=
            experiment_id,

        model_name=
            model_name,

        family=
            config["family"],

        status=
            "running",

        notes=
            "Module 7 neural candidate"
    )

    oof_pred = np.zeros(
        len(y_dev),
        dtype=np.float64
    )

    hold_preds = []

    fold_rows = []

    for fold in range(
        N_SPLITS
    ):

        print(
            f"\n  Fold {fold + 1}/{N_SPLITS}"
        )

        fold_start = time.time()

        fold_result = train_mlp_fold(
            model_name=
                model_name,

            fold_id=
                fold,

            config=
                config,
        )

        valid_idx = (
            fold_result[
                "valid_idx"
            ]
        )

        p_valid = (
            fold_result[
                "p_valid"
            ]
        )

        p_hold = (
            fold_result[
                "p_hold"
            ]
        )

        oof_pred[
            valid_idx
        ] = p_valid

        hold_preds.append(
            p_hold
        )

        # descriptive fold threshold
        fold_threshold, _, _ = (
            select_best_threshold(
                y_dev.iloc[
                    valid_idx
                ],
                p_valid,
                objective="F1"
            )
        )

        fold_metrics = (
            evaluate_binary_predictions(
                y_dev.iloc[
                    valid_idx
                ],
                p_valid,
                threshold=
                    fold_threshold,
            )
        )

        fold_runtime = (
            time.time()
            - fold_start
        )

        row = {
            "Model":
                model_name,

            "Fold":
                fold,

            "Best_Epoch":
                fold_result[
                    "best_epoch"
                ],

            "Best_Validation_PR":
                fold_result[
                    "best_PR"
                ],

            "Runtime_seconds":
                fold_runtime,
        }

        row.update(
            fold_metrics
        )

        fold_rows.append(
            row
        )

        print(
            f"  Fold result | "
            f"PR={fold_metrics['PR_AUC']:.6f} "
            f"ROC={fold_metrics['ROC_AUC']:.6f} "
            f"Brier={fold_metrics['Brier']:.6f} "
            f"F1={fold_metrics['F1']:.6f}"
        )

    # --------------------------------------------------------
    # Aggregate holdout prediction
    # --------------------------------------------------------

    holdout_pred = np.mean(
        np.vstack(
            hold_preds
        ),
        axis=0
    )

    holdout_pred = clip_probs(
        holdout_pred
    )

    # --------------------------------------------------------
    # OOF package
    # --------------------------------------------------------

    package = (
        build_model_result_package(
            model_name=
                model_name,

            family=
                config["family"],

            oof_pred=
                oof_pred,

            threshold_objective=
                "F1",
        )
    )

    save_model_result_package(
        package
    )

    add_predictions_to_bank(
        model_name=
            model_name,

        oof_pred=
            oof_pred,

        holdout_pred=
            holdout_pred,

        overwrite=True,
    )

    np.save(
        PRED_DIR
        / f"{model_name}_OOF.npy",
        oof_pred
    )

    np.save(
        PRED_DIR
        / f"{model_name}_HOLDOUT.npy",
        holdout_pred
    )

    fold_df = pd.DataFrame(
        fold_rows
    )

    save_csv(
        fold_df,
        TAB_DIR
        / "module7_fold_metrics"
        / f"{model_name}.csv"
    )

    runtime = (
        time.time()
        - start
    )

    register_experiment(
        experiment_id=
            experiment_id,

        model_name=
            model_name,

        family=
            config["family"],

        status=
            "complete",

        runtime_seconds=
            runtime,

        oof_metrics=
            package["metrics"],

        notes=(
            "Fixed-fold OOF neural model; "
            "mixed precision; "
            "early stop by PR-AUC"
        ),
    )

    print("\nOOF SUMMARY")

    for metric in [
        "PR_AUC",
        "ROC_AUC",
        "Brier",
        "NLL",
        "ECE",
        "Precision",
        "Recall",
        "F1",
        "MCC",
    ]:

        print(
            f"{metric:<12}: "
            f"{package['metrics'][metric]:.6f}"
        )

    print(
        f"Threshold   : "
        f"{package['threshold']:.6f}"
    )

    print(
        f"Runtime     : "
        f"{runtime / 60:.2f} min"
    )

    return {
        "model_name":
            model_name,

        "oof_pred":
            oof_pred,

        "holdout_pred":
            holdout_pred,

        "metrics":
            package["metrics"],

        "threshold":
            package["threshold"],

        "fold_metrics":
            fold_df,
    }

# ------------------------------------------------------------
# 7B.3 Train / resume
# ------------------------------------------------------------

MODULE7_MLP_RESULTS = {}

for model_name, config in (
    MLP_CONFIGS.items()
):

    oof_file = (
        PRED_DIR
        / f"{model_name}_OOF.npy"
    )

    hold_file = (
        PRED_DIR
        / f"{model_name}_HOLDOUT.npy"
    )

    if (
        oof_file.exists()
        and hold_file.exists()
    ):

        print(
            f"\n[RESUME] Loading {model_name}"
        )

        oof_pred = np.load(
            oof_file
        )

        hold_pred = np.load(
            hold_file
        )

        package = (
            build_model_result_package(
                model_name=
                    model_name,

                family=
                    config["family"],

                oof_pred=
                    oof_pred,

                threshold_objective=
                    "F1",
            )
        )

        add_predictions_to_bank(
            model_name=
                model_name,

            oof_pred=
                oof_pred,

            holdout_pred=
                hold_pred,

            overwrite=True,
        )

        MODULE7_MLP_RESULTS[
            model_name
        ] = {
            "oof_pred":
                oof_pred,

            "holdout_pred":
                hold_pred,

            "metrics":
                package[
                    "metrics"
                ],

            "threshold":
                package[
                    "threshold"
                ],
        }

    else:

        MODULE7_MLP_RESULTS[
            model_name
        ] = (
            run_mlp_candidate(
                model_name,
                config
            )
        )

# ------------------------------------------------------------
# 7B.4 Comparison
# ------------------------------------------------------------

rows = []

for model_name, result in (
    MODULE7_MLP_RESULTS.items()
):

    m = result["metrics"]

    rows.append({
        "Model":
            model_name,

        "PR_AUC":
            m["PR_AUC"],

        "ROC_AUC":
            m["ROC_AUC"],

        "Brier":
            m["Brier"],

        "NLL":
            m["NLL"],

        "ECE":
            m["ECE"],

        "Precision":
            m["Precision"],

        "Recall":
            m["Recall"],

        "F1":
            m["F1"],

        "MCC":
            m["MCC"],

        "Threshold":
            m["Threshold"],
    })

module7_mlp_comparison = (
    pd.DataFrame(rows)
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("MODULE 7B — MLP OOF COMPARISON")
print("=" * 100)

display(
    module7_mlp_comparison
)

save_csv(
    module7_mlp_comparison,
    TAB_DIR
    / "module7_mlp_oof_comparison.csv"
)

print(
    "\nOOF bank shape:",
    OOF_BANK.shape
)

print(
    "HOLDOUT bank shape:",
    HOLDOUT_BANK.shape
)

print("\nMODULE 7B COMPLETE.")

PYTORCH TRAINING CONFIGURATION
Device : cuda:0
AMP    : True
GPU    : Tesla T4

MODULE 7A COMPLETE.

####################################################################################################
MODEL: MLP_BCE
####################################################################################################

  Fold 1/5
    epoch=01 loss=0.91401 val_PR=0.370996 best=0.370996
    epoch=05 loss=0.87158 val_PR=0.376490 best=0.376490
    epoch=10 loss=0.86734 val_PR=0.374995 best=0.377852
    epoch=13 loss=0.86631 val_PR=0.376491 best=0.377852
  Fold result | PR=0.377852 ROC=0.851880 Brier=0.166999 F1=0.421675

  Fold 2/5
    epoch=01 loss=0.91460 val_PR=0.374642 best=0.374642
    epoch=05 loss=0.87261 val_PR=0.376036 best=0.376401
    epoch=10 loss=0.86555 val_PR=0.378598 best=0.378598
    epoch=15 loss=0.86415 val_PR=0.378470 best=0.379544
    epoch=19 loss=0.86159 val_PR=0.378947 best=0.379544
  Fold result | PR=0.379544 ROC=0.850180 Brier=0.167434 F1=0.422454

  Fold 3/5
    ep

,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,MLP_BCE,0.376499,0.850047,0.168696,0.489990,0.253454,0.336808,0.559538,0.420500,0.357756,0.715
1,MLP_Focal,0.375743,0.849344,0.097017,0.344413,0.155949,0.344920,0.537303,0.420135,0.355960,0.390
2,MLP_Focal_FGSM,0.374405,0.849344,0.096750,0.343746,0.155133,0.331728,0.569216,0.419171,0.357066,0.380
3,MLP_FGSM,0.372766,0.848833,0.177083,0.512347,0.269478,0.335053,0.557288,0.418497,0.355420,0.720



OOF bank shape: (202944, 15)
HOLDOUT bank shape: (50736, 14)

MODULE 7B COMPLETE.


In [7]:
# ============================================================
# MODULE 7C — DEEPFM + AUTOINT
# ============================================================

import gc
import copy
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 7C.1 Schema helpers
# ------------------------------------------------------------

CAT_COLS = DEEP_CATEGORICAL_COLS.copy()
CONT_COLS_DEEP = DEEP_CONTINUOUS_COLS.copy()

CAT_VALUE_MAPS = {}

for col in CAT_COLS:
    vals = sorted(X_dev[col].dropna().unique().tolist())
    CAT_VALUE_MAPS[col] = {
        v: i for i, v in enumerate(vals)
    }

CAT_CARDINALITIES = [
    len(CAT_VALUE_MAPS[c])
    for c in CAT_COLS
]

print("Categorical features:", len(CAT_COLS))
print("Continuous features :", len(CONT_COLS_DEEP))
print("Cardinalities       :", CAT_CARDINALITIES)

# ------------------------------------------------------------
# 7C.2 Fold-safe encoder
# ------------------------------------------------------------

def prepare_deep_tabular_fold(fold_id):

    train_idx = np.where(
        fold_assignment != fold_id
    )[0]

    valid_idx = np.where(
        fold_assignment == fold_id
    )[0]

    Xtr = X_dev.iloc[train_idx].copy()
    Xva = X_dev.iloc[valid_idx].copy()
    Xho = X_holdout.copy()

    # categorical -> integer index
    def encode_cat(df_):

        arrs = []

        for c in CAT_COLS:

            mapper = CAT_VALUE_MAPS[c]

            encoded = (
                df_[c]
                .map(mapper)
                .fillna(0)
                .astype(np.int64)
                .to_numpy()
            )

            arrs.append(encoded)

        return np.column_stack(arrs)

    Xtr_cat = encode_cat(Xtr)
    Xva_cat = encode_cat(Xva)
    Xho_cat = encode_cat(Xho)

    # continuous -> fold-safe standardization
    train_mean = (
        Xtr[CONT_COLS_DEEP]
        .mean()
    )

    train_std = (
        Xtr[CONT_COLS_DEEP]
        .std()
        .replace(0, 1)
    )

    def encode_cont(df_):

        z = (
            df_[CONT_COLS_DEEP]
            - train_mean
        ) / train_std

        return z.to_numpy(
            dtype=np.float32
        )

    Xtr_cont = encode_cont(Xtr)
    Xva_cont = encode_cont(Xva)
    Xho_cont = encode_cont(Xho)

    ytr = y_dev.iloc[
        train_idx
    ].to_numpy(dtype=np.float32)

    yva = y_dev.iloc[
        valid_idx
    ].to_numpy(dtype=np.float32)

    return {
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "Xtr_cat": Xtr_cat,
        "Xva_cat": Xva_cat,
        "Xho_cat": Xho_cat,
        "Xtr_cont": Xtr_cont,
        "Xva_cont": Xva_cont,
        "Xho_cont": Xho_cont,
        "ytr": ytr,
        "yva": yva,
    }

# ------------------------------------------------------------
# 7C.3 Dataset
# ------------------------------------------------------------

class DeepTabularDataset(Dataset):

    def __init__(
        self,
        X_cat,
        X_cont,
        y=None
    ):

        self.X_cat = torch.as_tensor(
            X_cat,
            dtype=torch.long
        )

        self.X_cont = torch.as_tensor(
            X_cont,
            dtype=torch.float32
        )

        self.y = None if y is None else torch.as_tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.X_cat)

    def __getitem__(self, idx):

        if self.y is None:
            return (
                self.X_cat[idx],
                self.X_cont[idx]
            )

        return (
            self.X_cat[idx],
            self.X_cont[idx],
            self.y[idx]
        )

# ------------------------------------------------------------
# 7C.4 Shared embedding block
# ------------------------------------------------------------

class FeatureEmbeddings(nn.Module):

    def __init__(
        self,
        cardinalities,
        embedding_dim=16
    ):

        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(
                cardinality,
                embedding_dim
            )
            for cardinality in cardinalities
        ])

    def forward(self, x_cat):

        embs = []

        for j, emb in enumerate(
            self.embeddings
        ):
            embs.append(
                emb(x_cat[:, j])
            )

        return torch.stack(
            embs,
            dim=1
        )

# ------------------------------------------------------------
# 7C.5 DeepFM
# ------------------------------------------------------------

class DeepFMModel(nn.Module):

    def __init__(
        self,
        cardinalities,
        n_cont,
        embedding_dim=16,
        hidden_dims=(128, 64),
        dropout=0.2
    ):

        super().__init__()

        self.embedding = FeatureEmbeddings(
            cardinalities,
            embedding_dim
        )

        self.n_fields = len(
            cardinalities
        )

        deep_input_dim = (
            self.n_fields
            * embedding_dim
            + n_cont
        )

        layers = []

        prev = deep_input_dim

        for h in hidden_dims:

            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])

            prev = h

        self.deep = nn.Sequential(
            *layers
        )

        self.deep_out = nn.Linear(
            prev,
            1
        )

        self.linear_cat = nn.ModuleList([
            nn.Embedding(card, 1)
            for card in cardinalities
        ])

        self.linear_cont = nn.Linear(
            n_cont,
            1
        )

    def forward(
        self,
        x_cat,
        x_cont
    ):

        emb = self.embedding(
            x_cat
        )

        # FM second-order interaction
        sum_emb = emb.sum(dim=1)

        sum_square = (
            sum_emb ** 2
        ).sum(
            dim=1,
            keepdim=True
        )

        square_sum = (
            emb ** 2
        ).sum(
            dim=1
        ).sum(
            dim=1,
            keepdim=True
        )

        fm_second = 0.5 * (
            sum_square - square_sum
        )

        linear_cat = torch.stack([
            layer(
                x_cat[:, j]
            ).squeeze(-1)
            for j, layer in enumerate(
                self.linear_cat
            )
        ], dim=1).sum(
            dim=1,
            keepdim=True
        )

        linear_cont = self.linear_cont(
            x_cont
        )

        deep_input = torch.cat(
            [
                emb.flatten(1),
                x_cont
            ],
            dim=1
        )

        deep_out = self.deep_out(
            self.deep(
                deep_input
            )
        )

        logits = (
            linear_cat
            + linear_cont
            + fm_second
            + deep_out
        )

        return logits.squeeze(1)

# ------------------------------------------------------------
# 7C.6 AutoInt
# ------------------------------------------------------------

class AutoIntModel(nn.Module):

    def __init__(
        self,
        cardinalities,
        n_cont,
        embedding_dim=16,
        num_heads=2,
        num_layers=2,
        hidden_dims=(128, 64),
        dropout=0.20
    ):

        super().__init__()

        self.embedding = FeatureEmbeddings(
            cardinalities,
            embedding_dim
        )

        self.n_cat = len(
            cardinalities
        )

        self.cont_proj = nn.ModuleList([
            nn.Linear(
                1,
                embedding_dim
            )
            for _ in range(n_cont)
        ])

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embedding_dim,
                nhead=num_heads,
                dim_feedforward=
                    embedding_dim * 4,
                dropout=dropout,
                batch_first=True,
                activation="relu",
            )
        )

        self.attention = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        n_fields = (
            self.n_cat
            + n_cont
        )

        flattened_dim = (
            n_fields
            * embedding_dim
        )

        layers = []

        prev = flattened_dim

        for h in hidden_dims:

            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])

            prev = h

        self.deep = nn.Sequential(
            *layers
        )

        self.out = nn.Linear(
            prev,
            1
        )

    def forward(
        self,
        x_cat,
        x_cont
    ):

        cat_emb = self.embedding(
            x_cat
        )

        cont_emb = []

        for j, layer in enumerate(
            self.cont_proj
        ):

            v = x_cont[
                :, j
            ].unsqueeze(1)

            cont_emb.append(
                layer(v)
            )

        cont_emb = torch.stack(
            cont_emb,
            dim=1
        )

        tokens = torch.cat(
            [
                cat_emb,
                cont_emb
            ],
            dim=1
        )

        att = self.attention(
            tokens
        )

        x = att.flatten(1)

        x = self.deep(x)

        return self.out(
            x
        ).squeeze(1)

# ------------------------------------------------------------
# 7C.7 Deep prediction helper
# ------------------------------------------------------------

@torch.no_grad()
def deep_predict_proba(
    model,
    X_cat,
    X_cont,
    batch_size=4096
):

    ds = DeepTabularDataset(
        X_cat,
        X_cont
    )

    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model.eval()

    preds = []

    for xcat, xcont in dl:

        xcat = xcat.to(
            TORCH_DEVICE,
            non_blocking=True
        )

        xcont = xcont.to(
            TORCH_DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(
                xcat,
                xcont
            )

        p = torch.sigmoid(
            logits.float()
        )

        preds.append(
            p.cpu().numpy()
        )

    return clip_probs(
        np.concatenate(preds)
    )

# ------------------------------------------------------------
# 7C.8 Train one fold
# ------------------------------------------------------------

def train_deep_tabular_fold(
    model_name,
    fold_id,
    config
):

    seed_everything(
        SEED + fold_id
    )

    data = prepare_deep_tabular_fold(
        fold_id
    )

    train_ds = DeepTabularDataset(
        data["Xtr_cat"],
        data["Xtr_cont"],
        data["ytr"]
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config[
            "batch_size"
        ],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(),
    )

    if model_name == "DeepFM":

        model = DeepFMModel(
            cardinalities=
                CAT_CARDINALITIES,
            n_cont=
                len(CONT_COLS_DEEP),
            embedding_dim=
                config["embedding_dim"],
            hidden_dims=
                config["hidden_dims"],
            dropout=
                config["dropout"],
        )

    elif model_name == "AutoInt":

        model = AutoIntModel(
            cardinalities=
                CAT_CARDINALITIES,
            n_cont=
                len(CONT_COLS_DEEP),
            embedding_dim=
                config["embedding_dim"],
            num_heads=
                config["num_heads"],
            num_layers=
                config["num_layers"],
            hidden_dims=
                config["hidden_dims"],
            dropout=
                config["dropout"],
        )

    else:

        raise ValueError(
            model_name
        )

    model = model.to(
        TORCH_DEVICE
    )

    pos_weight = torch.tensor(
        [SCALE_POS_WEIGHT],
        dtype=torch.float32,
        device=TORCH_DEVICE
    )

    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ]
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = -1
    patience_counter = 0

    history = []

    for epoch in range(
        1,
        config["max_epochs"] + 1
    ):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for xcat, xcont, yb in (
            train_loader
        ):

            xcat = xcat.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            xcont = xcont.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = model(
                    xcat,
                    xcont
                )

                loss = loss_fn(
                    logits,
                    yb
                )

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

            n = len(yb)

            running_loss += (
                float(
                    loss.detach().cpu()
                )
                * n
            )

            n_seen += n

        p_valid = (
            deep_predict_proba(
                model,
                data["Xva_cat"],
                data["Xva_cont"]
            )
        )

        valid_pr = (
            average_precision_score(
                data["yva"].astype(int),
                p_valid
            )
        )

        epoch_loss = (
            running_loss
            / n_seen
        )

        history.append({
            "epoch":
                epoch,
            "train_loss":
                epoch_loss,
            "valid_PR_AUC":
                valid_pr,
        })

        if valid_pr > (
            best_pr
            + config["min_delta"]
        ):

            best_pr = valid_pr
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            epoch == 1
            or epoch % 5 == 0
            or patience_counter
                >= config["patience"]
        ):

            print(
                f"    epoch={epoch:02d} "
                f"loss={epoch_loss:.5f} "
                f"val_PR={valid_pr:.6f} "
                f"best={best_pr:.6f}"
            )

        if (
            patience_counter
            >= config["patience"]
        ):
            break

    model.load_state_dict(
        best_state
    )

    p_valid = deep_predict_proba(
        model,
        data["Xva_cat"],
        data["Xva_cont"]
    )

    p_hold = deep_predict_proba(
        model,
        data["Xho_cat"],
        data["Xho_cont"]
    )

    return {
        "valid_idx":
            data["valid_idx"],
        "p_valid":
            p_valid,
        "p_hold":
            p_hold,
        "best_epoch":
            best_epoch,
        "best_PR":
            best_pr,
        "history":
            pd.DataFrame(history),
    }

# ------------------------------------------------------------
# 7C.9 Configs
# ------------------------------------------------------------

DEEP_CONFIGS = {

    "DeepFM": {
        "family":
            "deep_tabular",
        "embedding_dim":
            16,
        "hidden_dims":
            (128, 64),
        "dropout":
            0.20,
        "learning_rate":
            1e-3,
        "weight_decay":
            1e-5,
        "batch_size":
            1024,
        "max_epochs":
            30,
        "patience":
            5,
        "min_delta":
            1e-4,
    },

    "AutoInt": {
        "family":
            "deep_tabular",
        "embedding_dim":
            16,
        "num_heads":
            2,
        "num_layers":
            2,
        "hidden_dims":
            (128, 64),
        "dropout":
            0.20,
        "learning_rate":
            1e-3,
        "weight_decay":
            1e-5,
        "batch_size":
            1024,
        "max_epochs":
            30,
        "patience":
            5,
        "min_delta":
            1e-4,
    },
}

# ------------------------------------------------------------
# 7C.10 Full CV runner
# ------------------------------------------------------------

def run_deep_candidate(
    model_name,
    config
):

    print("\n" + "#" * 100)
    print(
        f"MODEL: {model_name}"
    )
    print("#" * 100)

    oof_pred = np.zeros(
        len(y_dev),
        dtype=np.float64
    )

    hold_preds = []
    fold_rows = []

    start = time.time()

    for fold in range(
        N_SPLITS
    ):

        print(
            f"\nFold {fold + 1}/{N_SPLITS}"
        )

        fold_start = time.time()

        res = (
            train_deep_tabular_fold(
                model_name,
                fold,
                config
            )
        )

        idx = res["valid_idx"]

        oof_pred[
            idx
        ] = res["p_valid"]

        hold_preds.append(
            res["p_hold"]
        )

        t, _, _ = (
            select_best_threshold(
                y_dev.iloc[idx],
                res["p_valid"],
                objective="F1"
            )
        )

        m = (
            evaluate_binary_predictions(
                y_dev.iloc[idx],
                res["p_valid"],
                threshold=t
            )
        )

        row = {
            "Model":
                model_name,
            "Fold":
                fold,
            "Best_Epoch":
                res["best_epoch"],
            "Best_Validation_PR":
                res["best_PR"],
            "Runtime_seconds":
                time.time()
                - fold_start,
        }

        row.update(m)
        fold_rows.append(row)

        print(
            f"  PR={m['PR_AUC']:.6f} "
            f"ROC={m['ROC_AUC']:.6f} "
            f"Brier={m['Brier']:.6f} "
            f"F1={m['F1']:.6f}"
        )

        gc.collect()
        torch.cuda.empty_cache()

    holdout_pred = clip_probs(
        np.mean(
            np.vstack(hold_preds),
            axis=0
        )
    )

    package = (
        build_model_result_package(
            model_name=model_name,
            family=config["family"],
            oof_pred=oof_pred,
            threshold_objective="F1"
        )
    )

    save_model_result_package(
        package
    )

    add_predictions_to_bank(
        model_name=model_name,
        oof_pred=oof_pred,
        holdout_pred=holdout_pred,
        overwrite=True
    )

    np.save(
        PRED_DIR
        / f"{model_name}_OOF.npy",
        oof_pred
    )

    np.save(
        PRED_DIR
        / f"{model_name}_HOLDOUT.npy",
        holdout_pred
    )

    fold_df = pd.DataFrame(
        fold_rows
    )

    save_csv(
        fold_df,
        TAB_DIR
        / "module7_deep_fold_metrics"
        / f"{model_name}.csv"
    )

    runtime = (
        time.time()
        - start
    )

    register_experiment(
        experiment_id=
            f"M7_{model_name}",
        model_name=
            model_name,
        family=
            config["family"],
        status=
            "complete",
        runtime_seconds=
            runtime,
        oof_metrics=
            package["metrics"],
        notes=
            "Custom PyTorch deep-tabular model"
    )

    print("\nOOF SUMMARY")

    for metric in [
        "PR_AUC",
        "ROC_AUC",
        "Brier",
        "NLL",
        "ECE",
        "Precision",
        "Recall",
        "F1",
        "MCC",
    ]:
        print(
            f"{metric:<12}: "
            f"{package['metrics'][metric]:.6f}"
        )

    print(
        "Threshold   :",
        package["threshold"]
    )

    return {
        "oof_pred":
            oof_pred,
        "holdout_pred":
            holdout_pred,
        "metrics":
            package["metrics"],
        "threshold":
            package["threshold"],
        "fold_metrics":
            fold_df,
    }

# ------------------------------------------------------------
# 7C.11 Train
# ------------------------------------------------------------

MODULE7_DEEP_RESULTS = {}

for model_name, config in (
    DEEP_CONFIGS.items()
):

    MODULE7_DEEP_RESULTS[
        model_name
    ] = run_deep_candidate(
        model_name,
        config
    )

# ------------------------------------------------------------
# 7C.12 Comparison
# ------------------------------------------------------------

rows = []

for model_name, result in (
    MODULE7_DEEP_RESULTS.items()
):

    m = result["metrics"]

    rows.append({
        "Model":
            model_name,
        "PR_AUC":
            m["PR_AUC"],
        "ROC_AUC":
            m["ROC_AUC"],
        "Brier":
            m["Brier"],
        "NLL":
            m["NLL"],
        "ECE":
            m["ECE"],
        "Precision":
            m["Precision"],
        "Recall":
            m["Recall"],
        "F1":
            m["F1"],
        "MCC":
            m["MCC"],
        "Threshold":
            m["Threshold"],
    })

module7_deep_comparison = (
    pd.DataFrame(rows)
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    module7_deep_comparison
)

save_csv(
    module7_deep_comparison,
    TAB_DIR
    / "module7_deep_oof_comparison.csv"
)

print("\nMODULE 7C COMPLETE.")

Categorical features: 18
Continuous features : 3
Cardinalities       : [2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 5, 2, 2, 13, 6, 8]

####################################################################################################
MODEL: DeepFM
####################################################################################################

Fold 1/5
    epoch=01 loss=12.01041 val_PR=0.207266 best=0.207266
    epoch=05 loss=3.67609 val_PR=0.249259 best=0.250290
    epoch=10 loss=2.23688 val_PR=0.269898 best=0.269898
    epoch=15 loss=1.66235 val_PR=0.263920 best=0.274049
    epoch=20 loss=1.36102 val_PR=0.282046 best=0.282046
    epoch=25 loss=1.18331 val_PR=0.288262 best=0.294125
    epoch=30 loss=1.08247 val_PR=0.302570 best=0.302570
  PR=0.302570 ROC=0.825997 Brier=0.176111 F1=0.386716

Fold 2/5
    epoch=01 loss=11.65577 val_PR=0.197051 best=0.197051
    epoch=05 loss=3.68761 val_PR=0.250248 best=0.250248
    epoch=10 loss=2.26112 val_PR=0.266099 best=0.268986
    epoch=15 loss=1.

,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,AutoInt,0.376859,0.849007,0.166093,0.483137,0.245961,0.354659,0.512766,0.419303,0.354068,0.735
1,DeepFM,0.286682,0.807714,0.181195,0.637504,0.214694,0.276926,0.573454,0.373490,0.307962,0.720



MODULE 7C COMPLETE.


In [8]:
# ============================================================
# MODULE 7C PATCH — STABLE DEEPFM v2
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
import time
import copy

# ------------------------------------------------------------
# 1. Stable embedding block
# ------------------------------------------------------------

class StableFeatureEmbeddings(nn.Module):

    def __init__(
        self,
        cardinalities,
        embedding_dim=8,
        init_std=1e-3,
    ):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality in cardinalities
        ])

        # IMPORTANT:
        # very small initialization for FM stability
        for emb in self.embeddings:
            nn.init.normal_(
                emb.weight,
                mean=0.0,
                std=init_std
            )

    def forward(self, x_cat):

        return torch.stack(
            [
                emb(x_cat[:, j])
                for j, emb in enumerate(self.embeddings)
            ],
            dim=1
        )


# ------------------------------------------------------------
# 2. Stable DeepFM
# ------------------------------------------------------------

class StableDeepFM(nn.Module):

    def __init__(
        self,
        cardinalities,
        n_cont,
        embedding_dim=8,
        hidden_dims=(256, 128),
        dropout=0.10,
        init_std=1e-3,
    ):
        super().__init__()

        self.embedding = StableFeatureEmbeddings(
            cardinalities,
            embedding_dim=embedding_dim,
            init_std=init_std,
        )

        self.n_fields = len(cardinalities)

        # -----------------------------------------------
        # First-order sparse terms
        # -----------------------------------------------

        self.linear_cat = nn.ModuleList([
            nn.Embedding(card, 1)
            for card in cardinalities
        ])

        for emb in self.linear_cat:
            nn.init.zeros_(emb.weight)

        # -----------------------------------------------
        # First-order continuous terms
        # -----------------------------------------------

        self.linear_cont = nn.Linear(
            n_cont,
            1
        )

        nn.init.zeros_(
            self.linear_cont.weight
        )

        nn.init.zeros_(
            self.linear_cont.bias
        )

        # -----------------------------------------------
        # Deep branch
        # -----------------------------------------------

        deep_input_dim = (
            self.n_fields * embedding_dim
            + n_cont
        )

        layers = []

        prev = deep_input_dim

        for h in hidden_dims:

            linear = nn.Linear(
                prev,
                h
            )

            nn.init.xavier_uniform_(
                linear.weight
            )

            nn.init.zeros_(
                linear.bias
            )

            layers.extend([
                linear,
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])

            prev = h

        self.deep = nn.Sequential(
            *layers
        )

        self.deep_out = nn.Linear(
            prev,
            1
        )

        nn.init.xavier_uniform_(
            self.deep_out.weight
        )

        nn.init.zeros_(
            self.deep_out.bias
        )

        # Learnable scaling prevents FM branch
        # dominating early training
        self.fm_scale = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(
        self,
        x_cat,
        x_cont
    ):

        # [B, fields, embedding]
        emb = self.embedding(
            x_cat
        )

        # -----------------------------------------------
        # Standard FM second-order interaction
        # -----------------------------------------------

        summed = emb.sum(
            dim=1
        )

        square_of_sum = (
            summed.pow(2)
            .sum(
                dim=1,
                keepdim=True
            )
        )

        sum_of_square = (
            emb.pow(2)
            .sum(
                dim=1
            )
            .sum(
                dim=1,
                keepdim=True
            )
        )

        fm_second = (
            0.5
            * (
                square_of_sum
                - sum_of_square
            )
        )

        # -----------------------------------------------
        # Sparse first-order terms
        # -----------------------------------------------

        linear_cat_terms = []

        for j, layer in enumerate(
            self.linear_cat
        ):

            linear_cat_terms.append(
                layer(
                    x_cat[:, j]
                )
            )

        linear_cat = torch.stack(
            linear_cat_terms,
            dim=1
        ).sum(
            dim=1
        )

        # -----------------------------------------------
        # Continuous first-order term
        # -----------------------------------------------

        linear_cont = (
            self.linear_cont(
                x_cont
            )
        )

        # -----------------------------------------------
        # Deep branch
        # -----------------------------------------------

        deep_input = torch.cat(
            [
                emb.flatten(1),
                x_cont
            ],
            dim=1
        )

        deep_repr = self.deep(
            deep_input
        )

        deep_out = self.deep_out(
            deep_repr
        )

        logits = (
            linear_cat
            + linear_cont
            + self.fm_scale
            * fm_second
            + deep_out
        )

        return logits.squeeze(1)


# ------------------------------------------------------------
# 3. Dedicated fold trainer
# ------------------------------------------------------------

def train_deepfm_v2_fold(
    fold_id,
    config
):

    seed_everything(
        SEED + fold_id
    )

    data = prepare_deep_tabular_fold(
        fold_id
    )

    train_ds = DeepTabularDataset(
        data["Xtr_cat"],
        data["Xtr_cont"],
        data["ytr"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(),
    )

    model = StableDeepFM(
        cardinalities=
            CAT_CARDINALITIES,

        n_cont=
            len(CONT_COLS_DEEP),

        embedding_dim=
            config["embedding_dim"],

        hidden_dims=
            config["hidden_dims"],

        dropout=
            config["dropout"],

        init_std=
            config["init_std"],
    ).to(TORCH_DEVICE)

    # -----------------------------------------------
    # Balanced loss
    # -----------------------------------------------

    pos_weight = torch.tensor(
        [SCALE_POS_WEIGHT],
        dtype=torch.float32,
        device=TORCH_DEVICE
    )

    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_pr = -np.inf
    best_epoch = -1
    best_state = None

    patience_count = 0
    history = []

    for epoch in range(
        1,
        config["max_epochs"] + 1
    ):

        model.train()

        total_loss = 0.0
        total_n = 0

        for xcat, xcont, yb in train_loader:

            xcat = xcat.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            xcont = xcont.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                TORCH_DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = model(
                    xcat,
                    xcont
                )

                loss = loss_fn(
                    logits,
                    yb
                )

            scaler.scale(
                loss
            ).backward()

            # extra stability
            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            n = len(yb)

            total_loss += (
                float(
                    loss.detach().cpu()
                )
                * n
            )

            total_n += n

        # -----------------------------------------------
        # validation PR-AUC
        # -----------------------------------------------

        p_valid = deep_predict_proba(
            model,
            data["Xva_cat"],
            data["Xva_cont"],
            batch_size=4096
        )

        valid_pr = (
            average_precision_score(
                data["yva"].astype(int),
                p_valid
            )
        )

        epoch_loss = (
            total_loss
            / total_n
        )

        history.append({
            "epoch": epoch,
            "train_loss": epoch_loss,
            "valid_PR_AUC": valid_pr,
            "fm_scale":
                float(
                    model.fm_scale
                    .detach()
                    .cpu()
                ),
        })

        if valid_pr > (
            best_pr
            + config["min_delta"]
        ):

            best_pr = valid_pr
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_count = 0

        else:

            patience_count += 1

        if (
            epoch == 1
            or epoch % 3 == 0
            or patience_count
                >= config["patience"]
        ):

            print(
                f"    epoch={epoch:02d} "
                f"loss={epoch_loss:.5f} "
                f"val_PR={valid_pr:.6f} "
                f"best={best_pr:.6f} "
                f"fm_scale="
                f"{float(model.fm_scale.detach().cpu()):.4f}"
            )

        if (
            patience_count
            >= config["patience"]
        ):
            break

    if best_state is None:
        raise RuntimeError(
            "DeepFM v2 failed to create checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    p_valid = deep_predict_proba(
        model,
        data["Xva_cat"],
        data["Xva_cont"]
    )

    p_hold = deep_predict_proba(
        model,
        data["Xho_cat"],
        data["Xho_cont"]
    )

    return {
        "valid_idx":
            data["valid_idx"],

        "p_valid":
            p_valid,

        "p_hold":
            p_hold,

        "best_epoch":
            best_epoch,

        "best_PR":
            best_pr,

        "history":
            pd.DataFrame(history),
    }


# ------------------------------------------------------------
# 4. Configuration close to old successful DeepFM
# ------------------------------------------------------------

DEEPFM_V2_CONFIG = {

    "embedding_dim": 8,

    "hidden_dims":
        (256, 128),

    "dropout":
        0.10,

    "init_std":
        1e-3,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-6,

    "batch_size":
        1024,

    "max_epochs":
        25,

    "patience":
        5,

    "min_delta":
        1e-4,
}


# ------------------------------------------------------------
# 5. Full OOF training
# ------------------------------------------------------------

model_name = "DeepFM_v2"

oof_pred = np.zeros(
    len(y_dev),
    dtype=np.float64
)

hold_preds = []

fold_rows = []

start = time.time()

for fold in range(N_SPLITS):

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"DeepFM_v2 Fold "
        f"{fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 80
    )

    fold_start = time.time()

    res = train_deepfm_v2_fold(
        fold,
        DEEPFM_V2_CONFIG
    )

    idx = res["valid_idx"]

    oof_pred[idx] = (
        res["p_valid"]
    )

    hold_preds.append(
        res["p_hold"]
    )

    threshold, _, _ = (
        select_best_threshold(
            y_dev.iloc[idx],
            res["p_valid"],
            objective="F1"
        )
    )

    m = evaluate_binary_predictions(
        y_dev.iloc[idx],
        res["p_valid"],
        threshold=threshold
    )

    row = {
        "Model":
            model_name,

        "Fold":
            fold,

        "Best_Epoch":
            res["best_epoch"],

        "Best_Validation_PR":
            res["best_PR"],

        "Runtime_seconds":
            time.time()
            - fold_start,
    }

    row.update(m)

    fold_rows.append(row)

    print(
        f"Fold PR={m['PR_AUC']:.6f} "
        f"ROC={m['ROC_AUC']:.6f} "
        f"Brier={m['Brier']:.6f} "
        f"F1={m['F1']:.6f}"
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 6. Aggregate
# ------------------------------------------------------------

holdout_pred = clip_probs(
    np.mean(
        np.vstack(hold_preds),
        axis=0
    )
)

package = build_model_result_package(
    model_name=model_name,
    family="deep_tabular",
    oof_pred=oof_pred,
    threshold_objective="F1"
)

save_model_result_package(
    package
)

add_predictions_to_bank(
    model_name=model_name,
    oof_pred=oof_pred,
    holdout_pred=holdout_pred,
    overwrite=True
)

np.save(
    PRED_DIR
    / "DeepFM_v2_OOF.npy",
    oof_pred
)

np.save(
    PRED_DIR
    / "DeepFM_v2_HOLDOUT.npy",
    holdout_pred
)

deepfm_v2_folds = pd.DataFrame(
    fold_rows
)

save_csv(
    deepfm_v2_folds,
    TAB_DIR
    / "module7_deep_fold_metrics"
    / "DeepFM_v2.csv"
)


# ------------------------------------------------------------
# 7. Result
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "DeepFM_v2 OOF RESULT"
)

print(
    "=" * 100
)

for metric in [
    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",
    "Precision",
    "Recall",
    "F1",
    "MCC",
]:

    print(
        f"{metric:<12}: "
        f"{package['metrics'][metric]:.6f}"
    )

print(
    f"Threshold   : "
    f"{package['threshold']:.6f}"
)

print(
    f"Runtime     : "
    f"{(time.time() - start)/60:.2f} min"
)


DeepFM_v2 Fold 1/5
    epoch=01 loss=1.01109 val_PR=0.360082 best=0.360082 fm_scale=0.0574
    epoch=03 loss=0.89777 val_PR=0.369948 best=0.369948 fm_scale=0.0094
    epoch=06 loss=0.87890 val_PR=0.366658 best=0.369948 fm_scale=-0.0671
    epoch=09 loss=0.87075 val_PR=0.376583 best=0.376583 fm_scale=-0.1388
    epoch=12 loss=0.86429 val_PR=0.373570 best=0.376583 fm_scale=-0.1898
    epoch=14 loss=0.85969 val_PR=0.372727 best=0.376583 fm_scale=-0.2252
Fold PR=0.376583 ROC=0.851736 Brier=0.176607 F1=0.420798

DeepFM_v2 Fold 2/5
    epoch=01 loss=1.01335 val_PR=0.368555 best=0.368555 fm_scale=0.0755
    epoch=03 loss=0.89427 val_PR=0.372880 best=0.372880 fm_scale=0.0850
    epoch=06 loss=0.87718 val_PR=0.371238 best=0.372880 fm_scale=0.0757
    epoch=09 loss=0.86887 val_PR=0.373257 best=0.376323 fm_scale=0.0589
    epoch=12 loss=0.86125 val_PR=0.368047 best=0.376323 fm_scale=0.0448
    epoch=13 loss=0.85777 val_PR=0.373171 best=0.376323 fm_scale=0.0389
Fold PR=0.376323 ROC=0.848885 Brier

In [9]:
# ============================================================
# MODULE 7D — TABNET + LOGREG_SUBSET
# ============================================================
#
# Final candidate-generation module before:
# - complete leaderboard
# - calibration
# - diversity analysis
# - expert specialization
#
# ============================================================

import gc
import time
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# ============================================================
# PART A — LOGISTIC REGRESSION SUBSET
# ============================================================

print("=" * 100)
print("PART A — LOGREG_SUBSET")
print("=" * 100)

MODEL_NAME = "LogReg_Subset"

oof_pred = np.zeros(
    len(y_dev),
    dtype=np.float64
)

hold_preds = []
fold_rows = []

start = time.time()

for fold in range(N_SPLITS):

    fold_start = time.time()

    train_idx = np.where(
        fold_assignment != fold
    )[0]

    valid_idx = np.where(
        fold_assignment == fold
    )[0]

    # --------------------------------------------------------
    # Fold-safe subset preprocessing
    # --------------------------------------------------------

    prep = (
        build_subset_linear_preprocessor()
    )

    Xtr = prep.fit_transform(
        X_dev.iloc[
            train_idx
        ][SUBSET_FEATURES]
    )

    Xva = prep.transform(
        X_dev.iloc[
            valid_idx
        ][SUBSET_FEATURES]
    )

    Xho = prep.transform(
        X_holdout[
            SUBSET_FEATURES
        ]
    )

    ytr = (
        y_dev.iloc[
            train_idx
        ]
        .to_numpy(dtype=np.int64)
    )

    yva = (
        y_dev.iloc[
            valid_idx
        ]
        .to_numpy(dtype=np.int64)
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = LogisticRegression(
        penalty="elasticnet",
        C=0.5,
        l1_ratio=0.5,
        solver="saga",
        class_weight="balanced",
        max_iter=3000,
        random_state=SEED,
        n_jobs=-1,
    )

    model.fit(
        Xtr,
        ytr
    )

    p_valid = clip_probs(
        model.predict_proba(
            Xva
        )[:, 1]
    )

    p_hold = clip_probs(
        model.predict_proba(
            Xho
        )[:, 1]
    )

    oof_pred[
        valid_idx
    ] = p_valid

    hold_preds.append(
        p_hold
    )

    threshold, _, _ = (
        select_best_threshold(
            yva,
            p_valid,
            objective="F1"
        )
    )

    metrics = (
        evaluate_binary_predictions(
            yva,
            p_valid,
            threshold=threshold
        )
    )

    row = {
        "Model":
            MODEL_NAME,

        "Fold":
            fold,

        "Runtime_seconds":
            time.time()
            - fold_start,
    }

    row.update(
        metrics
    )

    fold_rows.append(
        row
    )

    print(
        f"Fold {fold + 1}/{N_SPLITS} "
        f"| PR={metrics['PR_AUC']:.6f} "
        f"| ROC={metrics['ROC_AUC']:.6f} "
        f"| F1={metrics['F1']:.6f}"
    )

    del model, prep, Xtr, Xva, Xho
    gc.collect()


# ------------------------------------------------------------
# Aggregate LogReg_Subset
# ------------------------------------------------------------

holdout_pred = clip_probs(
    np.mean(
        np.vstack(hold_preds),
        axis=0
    )
)

package_subset = (
    build_model_result_package(
        model_name=MODEL_NAME,
        family="linear",
        oof_pred=oof_pred,
        threshold_objective="F1"
    )
)

save_model_result_package(
    package_subset
)

add_predictions_to_bank(
    model_name=MODEL_NAME,
    oof_pred=oof_pred,
    holdout_pred=holdout_pred,
    overwrite=True
)

np.save(
    PRED_DIR
    / "LogReg_Subset_OOF.npy",
    oof_pred
)

np.save(
    PRED_DIR
    / "LogReg_Subset_HOLDOUT.npy",
    holdout_pred
)

subset_fold_df = pd.DataFrame(
    fold_rows
)

save_csv(
    subset_fold_df,
    TAB_DIR
    / "module7_subset_fold_metrics.csv"
)

register_experiment(
    experiment_id=
        "M7_LogReg_Subset",

    model_name=
        MODEL_NAME,

    family=
        "linear",

    status=
        "complete",

    runtime_seconds=
        time.time() - start,

    oof_metrics=
        package_subset["metrics"],

    notes=
        "12-feature subset elastic-net logistic expert"
)

print("\nLogReg_Subset OOF:")

for metric in [
    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",
    "Precision",
    "Recall",
    "F1",
    "MCC",
]:

    print(
        f"{metric:<12}: "
        f"{package_subset['metrics'][metric]:.6f}"
    )


# ============================================================
# PART B — TABNET
# ============================================================

print("\n" + "=" * 100)
print("PART B — TABNET")
print("=" * 100)

# ------------------------------------------------------------
# Dependency check
# ------------------------------------------------------------

if importlib.util.find_spec(
    "pytorch_tabnet"
) is None:

    print(
        "pytorch_tabnet not found. "
        "Attempting installation..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pytorch-tabnet"
    ])

from pytorch_tabnet.tab_model import (
    TabNetClassifier
)

from pytorch_tabnet.metrics import (
    Metric
)


# ------------------------------------------------------------
# Custom PR-AUC metric for early stopping
# ------------------------------------------------------------

class PRAUCMetric(Metric):

    def __init__(self):

        self._name = "pr_auc"
        self._maximize = True

    def __call__(
        self,
        y_true,
        y_score
    ):

        # TabNet may return two-column probabilities
        if (
            y_score.ndim == 2
            and y_score.shape[1] == 2
        ):

            score = (
                y_score[:, 1]
            )

        else:

            score = (
                y_score.reshape(-1)
            )

        return (
            average_precision_score(
                y_true,
                score
            )
        )


# ------------------------------------------------------------
# Prepare semantic indices
# ------------------------------------------------------------

TABNET_FEATURE_ORDER = (
    DEEP_CATEGORICAL_COLS
    + DEEP_CONTINUOUS_COLS
)

TABNET_CAT_IDXS = list(
    range(
        len(
            DEEP_CATEGORICAL_COLS
        )
    )
)

TABNET_CAT_DIMS = [
    DEEP_CARDINALITIES[
        c
    ]["n_unique"]
    for c in DEEP_CATEGORICAL_COLS
]

print(
    "TabNet feature count:",
    len(TABNET_FEATURE_ORDER)
)

print(
    "Categorical indices:",
    TABNET_CAT_IDXS
)

print(
    "Categorical dimensions:",
    TABNET_CAT_DIMS
)


# ------------------------------------------------------------
# TabNet fold-preparation helper
# ------------------------------------------------------------

def prepare_tabnet_fold(
    fold_id
):

    train_idx = np.where(
        fold_assignment != fold_id
    )[0]

    valid_idx = np.where(
        fold_assignment == fold_id
    )[0]

    Xtr_df = (
        X_dev.iloc[
            train_idx
        ][TABNET_FEATURE_ORDER]
        .copy()
    )

    Xva_df = (
        X_dev.iloc[
            valid_idx
        ][TABNET_FEATURE_ORDER]
        .copy()
    )

    Xho_df = (
        X_holdout[
            TABNET_FEATURE_ORDER
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Convert categorical values into 0-based indices
    # --------------------------------------------------------

    for col in (
        DEEP_CATEGORICAL_COLS
    ):

        values = sorted(
            Xtr_df[
                col
            ]
            .dropna()
            .unique()
            .tolist()
        )

        mapper = {
            v: i
            for i, v
            in enumerate(values)
        }

        Xtr_df[col] = (
            Xtr_df[col]
            .map(mapper)
            .fillna(0)
            .astype(int)
        )

        Xva_df[col] = (
            Xva_df[col]
            .map(mapper)
            .fillna(0)
            .astype(int)
        )

        Xho_df[col] = (
            Xho_df[col]
            .map(mapper)
            .fillna(0)
            .astype(int)
        )

    # --------------------------------------------------------
    # Fold-safe scaling of dense features
    # --------------------------------------------------------

    mean_ = (
        Xtr_df[
            DEEP_CONTINUOUS_COLS
        ]
        .mean()
    )

    std_ = (
        Xtr_df[
            DEEP_CONTINUOUS_COLS
        ]
        .std()
        .replace(
            0,
            1
        )
    )

    for frame in [
        Xtr_df,
        Xva_df,
        Xho_df,
    ]:

        frame[
            DEEP_CONTINUOUS_COLS
        ] = (
            frame[
                DEEP_CONTINUOUS_COLS
            ]
            - mean_
        ) / std_

    return {
        "train_idx":
            train_idx,

        "valid_idx":
            valid_idx,

        "Xtr":
            Xtr_df.to_numpy(
                dtype=np.float32
            ),

        "Xva":
            Xva_df.to_numpy(
                dtype=np.float32
            ),

        "Xho":
            Xho_df.to_numpy(
                dtype=np.float32
            ),

        "ytr":
            y_dev.iloc[
                train_idx
            ]
            .to_numpy(
                dtype=np.int64
            ),

        "yva":
            y_dev.iloc[
                valid_idx
            ]
            .to_numpy(
                dtype=np.int64
            ),
    }


# ------------------------------------------------------------
# TabNet config
# ------------------------------------------------------------

TABNET_CONFIG = {

    "n_d":
        24,

    "n_a":
        24,

    "n_steps":
        5,

    "gamma":
        1.5,

    "lambda_sparse":
        1e-4,

    "learning_rate":
        2e-2,

    "weight_decay":
        1e-5,

    "batch_size":
        4096,

    "virtual_batch_size":
        512,

    "max_epochs":
        100,

    "patience":
        12,
}


# ------------------------------------------------------------
# Full TabNet CV
# ------------------------------------------------------------

MODEL_NAME = "TabNet"

tabnet_oof = np.zeros(
    len(y_dev),
    dtype=np.float64
)

tabnet_hold_preds = []
tabnet_fold_rows = []

tabnet_start = time.time()

for fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"TabNet Fold "
        f"{fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 80
    )

    fold_start = time.time()

    data = prepare_tabnet_fold(
        fold
    )

    model = TabNetClassifier(

        n_d=
            TABNET_CONFIG["n_d"],

        n_a=
            TABNET_CONFIG["n_a"],

        n_steps=
            TABNET_CONFIG["n_steps"],

        gamma=
            TABNET_CONFIG["gamma"],

        lambda_sparse=
            TABNET_CONFIG[
                "lambda_sparse"
            ],

        cat_idxs=
            TABNET_CAT_IDXS,

        cat_dims=
            TABNET_CAT_DIMS,

        cat_emb_dim=
            4,

        optimizer_fn=
            torch.optim.AdamW,

        optimizer_params={
            "lr":
                TABNET_CONFIG[
                    "learning_rate"
                ],

            "weight_decay":
                TABNET_CONFIG[
                    "weight_decay"
                ],
        },

        mask_type=
            "entmax",

        seed=
            SEED + fold,

        verbose=
            0,

        device_name=
            "cuda",
    )

    model.fit(

        X_train=
            data["Xtr"],

        y_train=
            data["ytr"],

        eval_set=[
            (
                data["Xva"],
                data["yva"]
            )
        ],

        eval_name=[
            "valid"
        ],

        eval_metric=[
            PRAUCMetric
        ],

        weights=
            1,

        max_epochs=
            TABNET_CONFIG[
                "max_epochs"
            ],

        patience=
            TABNET_CONFIG[
                "patience"
            ],

        batch_size=
            TABNET_CONFIG[
                "batch_size"
            ],

        virtual_batch_size=
            TABNET_CONFIG[
                "virtual_batch_size"
            ],

        num_workers=
            2,

        drop_last=
            False,
    )

    p_valid = clip_probs(
        model.predict_proba(
            data["Xva"]
        )[:, 1]
    )

    p_hold = clip_probs(
        model.predict_proba(
            data["Xho"]
        )[:, 1]
    )

    idx = data[
        "valid_idx"
    ]

    tabnet_oof[
        idx
    ] = p_valid

    tabnet_hold_preds.append(
        p_hold
    )

    threshold, _, _ = (
        select_best_threshold(
            data["yva"],
            p_valid,
            objective="F1"
        )
    )

    metrics = (
        evaluate_binary_predictions(
            data["yva"],
            p_valid,
            threshold=threshold
        )
    )

    row = {
        "Model":
            MODEL_NAME,

        "Fold":
            fold,

        "Runtime_seconds":
            time.time()
            - fold_start,
    }

    row.update(
        metrics
    )

    tabnet_fold_rows.append(
        row
    )

    print(
        f"PR={metrics['PR_AUC']:.6f} "
        f"ROC={metrics['ROC_AUC']:.6f} "
        f"Brier={metrics['Brier']:.6f} "
        f"ECE={metrics['ECE']:.6f} "
        f"F1={metrics['F1']:.6f}"
    )

    del model
    del data

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Aggregate TabNet
# ------------------------------------------------------------

tabnet_holdout = clip_probs(
    np.mean(
        np.vstack(
            tabnet_hold_preds
        ),
        axis=0
    )
)

package_tabnet = (
    build_model_result_package(
        model_name=
            "TabNet",

        family=
            "deep_tabular",

        oof_pred=
            tabnet_oof,

        threshold_objective=
            "F1"
    )
)

save_model_result_package(
    package_tabnet
)

add_predictions_to_bank(
    model_name=
        "TabNet",

    oof_pred=
        tabnet_oof,

    holdout_pred=
        tabnet_holdout,

    overwrite=
        True
)

np.save(
    PRED_DIR
    / "TabNet_OOF.npy",
    tabnet_oof
)

np.save(
    PRED_DIR
    / "TabNet_HOLDOUT.npy",
    tabnet_holdout
)

tabnet_fold_df = (
    pd.DataFrame(
        tabnet_fold_rows
    )
)

save_csv(
    tabnet_fold_df,
    TAB_DIR
    / "module7_tabnet_fold_metrics.csv"
)

register_experiment(

    experiment_id=
        "M7_TabNet",

    model_name=
        "TabNet",

    family=
        "deep_tabular",

    status=
        "complete",

    runtime_seconds=
        time.time()
        - tabnet_start,

    oof_metrics=
        package_tabnet[
            "metrics"
        ],

    notes=
        "TabNet with fold-safe categorical/dense preprocessing"
)

print(
    "\n"
    + "=" * 100
)

print(
    "TABNET OOF RESULT"
)

print(
    "=" * 100
)

for metric in [
    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",
    "Precision",
    "Recall",
    "F1",
    "MCC",
]:

    print(
        f"{metric:<12}: "
        f"{package_tabnet['metrics'][metric]:.6f}"
    )


# ============================================================
# PART C — QUICK CANDIDATE SUMMARY
# ============================================================

summary_rows = []

for model_name in [
    "LogReg_Subset",
    "TabNet"
]:

    pred = np.load(
        PRED_DIR
        / f"{model_name}_OOF.npy"
    )

    package = (
        build_model_result_package(
            model_name=
                model_name,

            family=
                (
                    "linear"
                    if model_name
                    == "LogReg_Subset"
                    else "deep_tabular"
                ),

            oof_pred=
                pred,

            threshold_objective=
                "F1"
        )
    )

    m = package[
        "metrics"
    ]

    summary_rows.append({
        "Model":
            model_name,

        "PR_AUC":
            m["PR_AUC"],

        "ROC_AUC":
            m["ROC_AUC"],

        "Brier":
            m["Brier"],

        "NLL":
            m["NLL"],

        "ECE":
            m["ECE"],

        "Precision":
            m["Precision"],

        "Recall":
            m["Recall"],

        "F1":
            m["F1"],

        "MCC":
            m["MCC"],

        "Threshold":
            m["Threshold"],
    })

module7d_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n"
    + "=" * 100
)

print(
    "MODULE 7D SUMMARY"
)

print(
    "=" * 100
)

display(
    module7d_summary
)

print(
    "\nOOF bank:",
    OOF_BANK.shape
)

print(
    "HOLDOUT bank:",
    HOLDOUT_BANK.shape
)

print(
    "\nMODULE 7D COMPLETE."
)

PART A — LOGREG_SUBSET
Fold 1/5 | PR=0.369463 | ROC=0.846045 | F1=0.411172
Fold 2/5 | PR=0.365416 | ROC=0.844806 | F1=0.413008
Fold 3/5 | PR=0.366753 | ROC=0.843478 | F1=0.412358
Fold 4/5 | PR=0.361207 | ROC=0.843261 | F1=0.417445
Fold 5/5 | PR=0.375727 | ROC=0.847038 | F1=0.413633

LogReg_Subset OOF:
PR_AUC      : 0.367312
ROC_AUC     : 0.844914
Brier       : 0.166113
NLL         : 0.498671
ECE         : 0.259250
Precision   : 0.323162
Recall      : 0.570681
F1          : 0.412650
MCC         : 0.350156

PART B — TABNET
pytorch_tabnet not found. Attempting installation...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 572.8 kB/s eta 0:00:00
TabNet feature count: 21
Categorical indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Categorical dimensions: [2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 5, 2, 2, 13, 6, 8]

TabNet Fold 1/5

Early stopping occurred at epoch 26 with best_epoch = 14 and best_valid_pr_auc = 0.36667
PR=0.366675 ROC=0.847609 Brier=0.168716 ECE=

,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,LogReg_Subset,0.367312,0.844914,0.166113,0.498671,0.259250,0.323162,0.570681,0.412650,0.350156,0.685
1,TabNet,0.362890,0.844509,0.166975,0.488919,0.249691,0.338197,0.533536,0.413981,0.348995,0.725



OOF bank: (202944, 20)
HOLDOUT bank: (50736, 19)

MODULE 7D COMPLETE.


In [10]:
# ============================================================
# MODULE 8 — COMPLETE LEADERBOARD, DIVERSITY & SPECIALIZATION
# ============================================================
#
# IMPORTANT:
# - Uses DEV OOF predictions only.
# - HOLDOUT is not inspected.
# - Broken original DeepFM is excluded.
# - No final ensemble is selected here.
# - Raw-probability averaging is intentionally postponed until
#   cross-fitted calibration.
#
# ============================================================

import itertools
import numpy as np
import pandas as pd

from scipy.stats import rankdata, spearmanr

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# ------------------------------------------------------------
# 8.1 Canonical candidate model list
# ------------------------------------------------------------

CANDIDATE_MODELS = [

    # linear
    "LogReg_L2",
    "LogReg_ElasticNet",
    "LogReg_Subset",

    # bagging
    "RandomForest",
    "ExtraTrees",

    # boosting
    "HistGradientBoosting",
    "XGBoost",
    "LightGBM",
    "CatBoost",

    # neural
    "MLP_BCE",
    "MLP_Focal",
    "MLP_FGSM",
    "MLP_Focal_FGSM",

    # deep tabular
    "AutoInt",
    "DeepFM_v2",
    "TabNet",
]

# deliberately excluded:
EXCLUDED_MODELS = {
    "DeepFM":
        "Broken initial custom implementation; replaced by DeepFM_v2"
}

missing_models = [
    m for m in CANDIDATE_MODELS
    if m not in OOF_BANK.columns
]

if missing_models:
    raise ValueError(
        f"Missing candidate predictions: {missing_models}"
    )

print("=" * 100)
print("CANONICAL CANDIDATE BANK")
print("=" * 100)

print(
    f"Number of valid candidates: "
    f"{len(CANDIDATE_MODELS)}"
)

for i, m in enumerate(
    CANDIDATE_MODELS,
    start=1
):
    print(f"{i:02d}. {m}")

print("\nExcluded:")
for k, v in EXCLUDED_MODELS.items():
    print(f"- {k}: {v}")


# ------------------------------------------------------------
# 8.2 Helper: model family
# ------------------------------------------------------------

MODEL_FAMILIES = {

    "LogReg_L2":
        "linear",

    "LogReg_ElasticNet":
        "linear",

    "LogReg_Subset":
        "linear",

    "RandomForest":
        "tree_bagging",

    "ExtraTrees":
        "tree_bagging",

    "HistGradientBoosting":
        "gradient_boosting",

    "XGBoost":
        "gradient_boosting",

    "LightGBM":
        "gradient_boosting",

    "CatBoost":
        "gradient_boosting",

    "MLP_BCE":
        "deep_mlp",

    "MLP_Focal":
        "deep_mlp",

    "MLP_FGSM":
        "deep_mlp",

    "MLP_Focal_FGSM":
        "deep_mlp",

    "AutoInt":
        "deep_tabular",

    "DeepFM_v2":
        "deep_tabular",

    "TabNet":
        "deep_tabular",
}


# ------------------------------------------------------------
# 8.3 Complete OOF leaderboard
# ------------------------------------------------------------

leaderboard_rows = []

MODEL_THRESHOLDS = {}

for model_name in CANDIDATE_MODELS:

    p = OOF_BANK[
        model_name
    ].to_numpy()

    package = (
        build_model_result_package(
            model_name=model_name,
            family=MODEL_FAMILIES[
                model_name
            ],
            oof_pred=p,
            threshold_objective="F1"
        )
    )

    m = package["metrics"]

    MODEL_THRESHOLDS[
        model_name
    ] = package[
        "threshold"
    ]

    # -----------------------------------------------
    # fold PR / ROC stability
    # -----------------------------------------------

    fold_pr = []
    fold_roc = []

    for fold in range(N_SPLITS):

        mask = (
            fold_assignment
            == fold
        )

        fold_pr.append(
            average_precision_score(
                y_dev.to_numpy()[mask],
                p[mask]
            )
        )

        fold_roc.append(
            roc_auc_score(
                y_dev.to_numpy()[mask],
                p[mask]
            )
        )

    leaderboard_rows.append({

        "Model":
            model_name,

        "Family":
            MODEL_FAMILIES[
                model_name
            ],

        "PR_AUC":
            m["PR_AUC"],

        "PR_AUC_mean_fold":
            np.mean(fold_pr),

        "PR_AUC_std":
            np.std(
                fold_pr,
                ddof=1
            ),

        "ROC_AUC":
            m["ROC_AUC"],

        "ROC_AUC_std":
            np.std(
                fold_roc,
                ddof=1
            ),

        "Brier":
            m["Brier"],

        "NLL":
            m["NLL"],

        "ECE":
            m["ECE"],

        "Precision":
            m["Precision"],

        "Recall":
            m["Recall"],

        "F1":
            m["F1"],

        "MCC":
            m["MCC"],

        "Threshold":
            package[
                "threshold"
            ],
    })


complete_leaderboard = (
    pd.DataFrame(
        leaderboard_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 100
)

print(
    "COMPLETE RAW OOF LEADERBOARD"
)

print(
    "=" * 100
)

display(
    complete_leaderboard
)

save_csv(
    complete_leaderboard,
    TAB_DIR
    / "module8_complete_raw_leaderboard.csv"
)


# ------------------------------------------------------------
# 8.4 Prediction matrices
# ------------------------------------------------------------

P_raw = OOF_BANK[
    CANDIDATE_MODELS
].copy()

y_array = (
    y_dev
    .to_numpy(dtype=int)
)

# Rank normalization:
#
# each model becomes approximately Uniform(0,1)
# according to its own OOF ranking.
#
# This removes probability-scale differences caused by
# class weighting while retaining ordering information.
# ------------------------------------------------------------

P_rank = pd.DataFrame(
    index=P_raw.index
)

for model_name in CANDIDATE_MODELS:

    p = P_raw[
        model_name
    ].to_numpy()

    ranks = rankdata(
        p,
        method="average"
    )

    P_rank[
        model_name
    ] = (
        ranks
        / (len(ranks) + 1.0)
    )


# ------------------------------------------------------------
# 8.5 Pearson correlation — raw probabilities
# ------------------------------------------------------------

raw_corr = (
    P_raw.corr(
        method="pearson"
    )
)

print(
    "\n"
    + "=" * 100
)

print(
    "RAW PROBABILITY CORRELATION"
)

print(
    "=" * 100
)

display(
    raw_corr.round(3)
)

save_csv(
    raw_corr.reset_index(),
    TAB_DIR
    / "module8_raw_probability_correlation.csv"
)


# ------------------------------------------------------------
# 8.6 Spearman / rank correlation
# ------------------------------------------------------------

rank_corr = (
    P_raw.corr(
        method="spearman"
    )
)

print(
    "\n"
    + "=" * 100
)

print(
    "RANK CORRELATION"
)

print(
    "=" * 100
)

display(
    rank_corr.round(3)
)

save_csv(
    rank_corr.reset_index(),
    TAB_DIR
    / "module8_rank_correlation.csv"
)


# ------------------------------------------------------------
# 8.7 Residual / error correlation
# ------------------------------------------------------------
#
# Absolute probability error:
#
#     |y - p|
#
# Raw calibration affects this, so interpret together with
# rank-based diagnostics.
#
# ------------------------------------------------------------

error_matrix = pd.DataFrame(
    index=P_raw.index
)

for model_name in CANDIDATE_MODELS:

    error_matrix[
        model_name
    ] = np.abs(
        y_array
        - P_raw[
            model_name
        ].to_numpy()
    )

error_corr = (
    error_matrix.corr()
)

print(
    "\n"
    + "=" * 100
)

print(
    "ABSOLUTE ERROR CORRELATION"
)

print(
    "=" * 100
)

display(
    error_corr.round(3)
)

save_csv(
    error_corr.reset_index(),
    TAB_DIR
    / "module8_absolute_error_correlation.csv"
)


# ------------------------------------------------------------
# 8.8 Threshold-based disagreement
# ------------------------------------------------------------
#
# Uses each model's own DEV-OOF F1-optimal threshold.
#
# This is only a diversity diagnostic.
# It is not the final clinical/decision threshold.
# ------------------------------------------------------------

binary_predictions = {}

for model_name in CANDIDATE_MODELS:

    threshold = (
        MODEL_THRESHOLDS[
            model_name
        ]
    )

    binary_predictions[
        model_name
    ] = (
        P_raw[
            model_name
        ].to_numpy()
        >= threshold
    ).astype(int)


pair_rows = []

for m1, m2 in itertools.combinations(
    CANDIDATE_MODELS,
    2
):

    p1 = P_raw[
        m1
    ].to_numpy()

    p2 = P_raw[
        m2
    ].to_numpy()

    r1 = P_rank[
        m1
    ].to_numpy()

    r2 = P_rank[
        m2
    ].to_numpy()

    pred1 = (
        binary_predictions[m1]
    )

    pred2 = (
        binary_predictions[m2]
    )

    disagreement = np.mean(
        pred1 != pred2
    )

    pos_mask = (
        y_array == 1
    )

    neg_mask = (
        y_array == 0
    )

    pos_disagreement = np.mean(
        pred1[pos_mask]
        != pred2[pos_mask]
    )

    neg_disagreement = np.mean(
        pred1[neg_mask]
        != pred2[neg_mask]
    )

    pair_rows.append({

        "Model_A":
            m1,

        "Model_B":
            m2,

        "Pearson_raw":
            np.corrcoef(
                p1,
                p2
            )[0, 1],

        "Spearman":
            spearmanr(
                p1,
                p2
            ).statistic,

        "Pearson_rank":
            np.corrcoef(
                r1,
                r2
            )[0, 1],

        "Decision_disagreement":
            disagreement,

        "Positive_disagreement":
            pos_disagreement,

        "Negative_disagreement":
            neg_disagreement,
    })


pairwise_diversity = (
    pd.DataFrame(
        pair_rows
    )
)

print(
    "\n"
    + "=" * 100
)

print(
    "MOST DIVERSE MODEL PAIRS"
)

print(
    "=" * 100
)

display(
    pairwise_diversity
    .sort_values(
        [
            "Spearman",
            "Positive_disagreement"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(30)
)

save_csv(
    pairwise_diversity,
    TAB_DIR
    / "module8_pairwise_diversity.csv"
)


# ------------------------------------------------------------
# 8.9 Top-tier subset
# ------------------------------------------------------------
#
# Keep broad enough not to prematurely discard diversity.
# ------------------------------------------------------------

TOP_PR_FLOOR = 0.370

TOP_MODELS = (
    complete_leaderboard
    .loc[
        complete_leaderboard[
            "PR_AUC"
        ] >= TOP_PR_FLOOR,
        "Model"
    ]
    .tolist()
)

print(
    "\nModels with PR-AUC >=",
    TOP_PR_FLOOR
)

print(
    TOP_MODELS
)


# ------------------------------------------------------------
# 8.10 Consensus difficulty score
# ------------------------------------------------------------
#
# We cannot average raw probabilities yet because probability
# scales differ substantially.
#
# Instead we use rank-normalized predictions.
#
# Positive case:
#   difficult if consensus rank is low.
#
# Negative case:
#   difficult if consensus rank is high.
#
# ------------------------------------------------------------

consensus_rank = (
    P_rank[
        TOP_MODELS
    ]
    .mean(axis=1)
    .to_numpy()
)

difficulty = np.where(
    y_array == 1,
    1.0 - consensus_rank,
    consensus_rank
)

difficulty_df = pd.DataFrame({

    "target":
        y_array,

    "consensus_rank":
        consensus_rank,

    "difficulty":
        difficulty,
})

# ------------------------------------------------------------
# 8.11 Define hard regions
# ------------------------------------------------------------

HARD_QUANTILE = 0.80

hard_cut = np.quantile(
    difficulty,
    HARD_QUANTILE
)

hard_mask = (
    difficulty >= hard_cut
)

hard_pos_mask = (
    hard_mask
    & (y_array == 1)
)

hard_neg_mask = (
    hard_mask
    & (y_array == 0)
)

print(
    "\nHard-region threshold:",
    hard_cut
)

print(
    "Hard overall N:",
    int(
        hard_mask.sum()
    )
)

print(
    "Hard positives:",
    int(
        hard_pos_mask.sum()
    )
)

print(
    "Hard negatives:",
    int(
        hard_neg_mask.sum()
    )
)


# ------------------------------------------------------------
# 8.12 Hard-region ranking diagnostics
# ------------------------------------------------------------
#
# PR-AUC on a positive-only subset is undefined.
# Therefore:
#
# - full hard region -> PR-AUC / ROC-AUC
# - hard positives -> mean rank / recovery rates
# ------------------------------------------------------------

hard_rows = []

for model_name in CANDIDATE_MODELS:

    raw_p = P_raw[
        model_name
    ].to_numpy()

    rank_p = P_rank[
        model_name
    ].to_numpy()

    row = {

        "Model":
            model_name,

        "Global_PR_AUC":
            average_precision_score(
                y_array,
                raw_p
            ),

        "Hard_PR_AUC":
            average_precision_score(
                y_array[
                    hard_mask
                ],
                raw_p[
                    hard_mask
                ]
            ),

        "Hard_ROC_AUC":
            roc_auc_score(
                y_array[
                    hard_mask
                ],
                raw_p[
                    hard_mask
                ]
            ),

        "Hard_Positive_Mean_Rank":
            rank_p[
                hard_pos_mask
            ].mean(),

        "Hard_Negative_Mean_Rank":
            rank_p[
                hard_neg_mask
            ].mean(),
    }

    # -----------------------------------------------
    # Positive recovery in top X% model-specific rank
    # -----------------------------------------------

    for frac in [
        0.10,
        0.20,
        0.30
    ]:

        threshold_rank = (
            1.0 - frac
        )

        recovered = np.mean(
            rank_p[
                hard_pos_mask
            ]
            >= threshold_rank
        )

        row[
            f"HardPositive_Recovery_Top{int(frac*100)}pct"
        ] = recovered

    hard_rows.append(row)


hard_region_table = (
    pd.DataFrame(
        hard_rows
    )
    .sort_values(
        "Hard_PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 100
)

print(
    "HARD-REGION SPECIALIZATION"
)

print(
    "=" * 100
)

display(
    hard_region_table
)

save_csv(
    hard_region_table,
    TAB_DIR
    / "module8_hard_region_specialization.csv"
)


# ------------------------------------------------------------
# 8.13 Anchor-specific rescue analysis
# ------------------------------------------------------------
#
# Use strongest raw PR model as provisional anchor.
#
# A positive is considered missed by anchor when it falls
# below anchor's own F1-optimal threshold.
#
# Then determine which other models recover it.
# ------------------------------------------------------------

ANCHOR_MODEL = (
    complete_leaderboard
    .iloc[0]["Model"]
)

anchor_threshold = (
    MODEL_THRESHOLDS[
        ANCHOR_MODEL
    ]
)

anchor_pred = (
    P_raw[
        ANCHOR_MODEL
    ].to_numpy()
    >= anchor_threshold
).astype(int)

anchor_fn_mask = (
    (y_array == 1)
    & (anchor_pred == 0)
)

anchor_fp_mask = (
    (y_array == 0)
    & (anchor_pred == 1)
)

print(
    "\nProvisional anchor:",
    ANCHOR_MODEL
)

print(
    "Anchor FN count:",
    int(
        anchor_fn_mask.sum()
    )
)

print(
    "Anchor FP count:",
    int(
        anchor_fp_mask.sum()
    )
)


rescue_rows = []

for model_name in CANDIDATE_MODELS:

    if model_name == ANCHOR_MODEL:
        continue

    pred = (
        binary_predictions[
            model_name
        ]
    )

    # -----------------------------------------------
    # FN rescue
    # -----------------------------------------------

    fn_rescued = (
        anchor_fn_mask
        & (pred == 1)
    )

    fn_rescue_rate = (
        fn_rescued.sum()
        / max(
            anchor_fn_mask.sum(),
            1
        )
    )

    # -----------------------------------------------
    # FP correction
    # -----------------------------------------------

    fp_corrected = (
        anchor_fp_mask
        & (pred == 0)
    )

    fp_correction_rate = (
        fp_corrected.sum()
        / max(
            anchor_fp_mask.sum(),
            1
        )
    )

    rescue_rows.append({

        "Anchor":
            ANCHOR_MODEL,

        "Candidate":
            model_name,

        "Anchor_FN":
            int(
                anchor_fn_mask.sum()
            ),

        "FN_rescued":
            int(
                fn_rescued.sum()
            ),

        "FN_rescue_rate":
            fn_rescue_rate,

        "Anchor_FP":
            int(
                anchor_fp_mask.sum()
            ),

        "FP_corrected":
            int(
                fp_corrected.sum()
            ),

        "FP_correction_rate":
            fp_correction_rate,

        "Spearman_with_anchor":
            rank_corr.loc[
                ANCHOR_MODEL,
                model_name
            ],
    })


rescue_table = (
    pd.DataFrame(
        rescue_rows
    )
    .sort_values(
        [
            "FN_rescue_rate",
            "FP_correction_rate"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 100
)

print(
    "ANCHOR ERROR-RESCUE ANALYSIS"
)

print(
    "=" * 100
)

display(
    rescue_table
)

save_csv(
    rescue_table,
    TAB_DIR
    / "module8_anchor_rescue_analysis.csv"
)


# ------------------------------------------------------------
# 8.14 Diversity summary by model
# ------------------------------------------------------------

diversity_rows = []

for model_name in CANDIDATE_MODELS:

    others = [
        m
        for m in CANDIDATE_MODELS
        if m != model_name
    ]

    avg_spearman = (
        rank_corr.loc[
            model_name,
            others
        ].mean()
    )

    pair_mask = (
        (
            pairwise_diversity[
                "Model_A"
            ] == model_name
        )
        |
        (
            pairwise_diversity[
                "Model_B"
            ] == model_name
        )
    )

    model_pairs = (
        pairwise_diversity[
            pair_mask
        ]
    )

    diversity_rows.append({

        "Model":
            model_name,

        "PR_AUC":
            float(
                complete_leaderboard
                .loc[
                    complete_leaderboard[
                        "Model"
                    ] == model_name,
                    "PR_AUC"
                ]
                .iloc[0]
            ),

        "Avg_Spearman_to_Others":
            avg_spearman,

        "Avg_Decision_Disagreement":
            model_pairs[
                "Decision_disagreement"
            ].mean(),

        "Avg_Positive_Disagreement":
            model_pairs[
                "Positive_disagreement"
            ].mean(),

        "Avg_Negative_Disagreement":
            model_pairs[
                "Negative_disagreement"
            ].mean(),
    })


diversity_summary = (
    pd.DataFrame(
        diversity_rows
    )
)

# simple descriptive diversity quantity;
# NOT a final expert-selection score
diversity_summary[
    "Rank_Diversity"
] = (
    1.0
    - diversity_summary[
        "Avg_Spearman_to_Others"
    ]
)

diversity_summary = (
    diversity_summary
    .sort_values(
        [
            "PR_AUC",
            "Rank_Diversity"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 100
)

print(
    "MODEL-LEVEL DIVERSITY SUMMARY"
)

print(
    "=" * 100
)

display(
    diversity_summary
)

save_csv(
    diversity_summary,
    TAB_DIR
    / "module8_model_diversity_summary.csv"
)


# ------------------------------------------------------------
# 8.15 Save canonical model set
# ------------------------------------------------------------

module8_manifest = {

    "candidate_models":
        CANDIDATE_MODELS,

    "excluded_models":
        EXCLUDED_MODELS,

    "top_pr_floor":
        TOP_PR_FLOOR,

    "top_models":
        TOP_MODELS,

    "provisional_anchor":
        ANCHOR_MODEL,

    "hard_region_quantile":
        HARD_QUANTILE,

    "hard_region_threshold":
        float(hard_cut),

    "analysis_source":
        "DEV_OOF_ONLY",

    "holdout_used":
        False,
}

save_json(
    module8_manifest,
    LOG_DIR
    / "module8_candidate_analysis_manifest.json"
)

print(
    "\nMODULE 8 COMPLETE."
)

CANONICAL CANDIDATE BANK
Number of valid candidates: 16
01. LogReg_L2
02. LogReg_ElasticNet
03. LogReg_Subset
04. RandomForest
05. ExtraTrees
06. HistGradientBoosting
07. XGBoost
08. LightGBM
09. CatBoost
10. MLP_BCE
11. MLP_Focal
12. MLP_FGSM
13. MLP_Focal_FGSM
14. AutoInt
15. DeepFM_v2
16. TabNet

Excluded:
- DeepFM: Broken initial custom implementation; replaced by DeepFM_v2

COMPLETE RAW OOF LEADERBOARD


,Model,Family,PR_AUC,PR_AUC_mean_fold,PR_AUC_std,ROC_AUC,ROC_AUC_std,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,CatBoost,gradient_boosting,0.377535,0.378003,0.006957,0.849216,0.001945,0.161784,0.475348,0.243946,0.328529,0.581982,0.419980,0.359039,0.690
1,HistGradientBoosting,gradient_boosting,0.377012,0.377609,0.006088,0.849628,0.001061,0.069876,0.236194,0.002378,0.348113,0.532332,0.420950,0.356610,0.215
2,AutoInt,deep_tabular,0.376859,0.378427,0.005208,0.849007,0.001612,0.166093,0.483137,0.245961,0.354659,0.512766,0.419303,0.354068,0.735
3,MLP_BCE,deep_mlp,0.376499,0.378556,0.006305,0.850047,0.001623,0.168696,0.489990,0.253454,0.336808,0.559538,0.420500,0.357756,0.715
4,MLP_Focal,deep_mlp,0.375743,0.377652,0.006117,0.849344,0.001626,0.097017,0.344413,0.155949,0.344920,0.537303,0.420135,0.355960,0.390
5,MLP_Focal_FGSM,deep_mlp,0.374405,0.376584,0.006669,0.849344,0.001365,0.096750,0.343746,0.155133,0.331728,0.569216,0.419171,0.357066,0.380
6,DeepFM_v2,deep_tabular,0.374324,0.375801,0.006070,0.848231,0.002420,0.171079,0.497199,0.256397,0.341065,0.542325,0.418769,0.354739,0.730
7,LightGBM,gradient_boosting,0.372779,0.373329,0.007186,0.846576,0.001623,0.157540,0.463084,0.233591,0.345198,0.530240,0.418162,0.353438,0.715
8,MLP_FGSM,deep_mlp,0.372766,0.377176,0.006231,0.848833,0.001576,0.177083,0.512347,0.269478,0.335053,0.557288,0.418497,0.355420,0.720
9,LogReg_ElasticNet,linear,0.371730,0.372174,0.005706,0.847375,0.001318,0.164999,0.495340,0.256907,0.329699,0.566339,0.416771,0.354242,0.690



RAW PROBABILITY CORRELATION


,LogReg_L2,LogReg_ElasticNet,LogReg_Subset,RandomForest,ExtraTrees,HistGradientBoosting,XGBoost,LightGBM,CatBoost,MLP_BCE,MLP_Focal,MLP_FGSM,MLP_Focal_FGSM,AutoInt,DeepFM_v2,TabNet
LogReg_L2,1.000,1.000,0.991,0.900,0.915,0.897,0.960,0.974,0.983,0.983,0.980,0.980,0.976,0.981,0.978,0.969
LogReg_ElasticNet,1.000,1.000,0.991,0.900,0.915,0.897,0.960,0.974,0.983,0.983,0.980,0.980,0.976,0.981,0.978,0.969
LogReg_Subset,0.991,0.991,1.000,0.895,0.909,0.890,0.953,0.965,0.975,0.974,0.972,0.971,0.967,0.972,0.970,0.960
RandomForest,0.900,0.900,0.895,1.000,0.972,0.910,0.916,0.914,0.911,0.899,0.888,0.890,0.889,0.900,0.897,0.891
ExtraTrees,0.915,0.915,0.909,0.972,1.000,0.886,0.928,0.929,0.927,0.919,0.909,0.914,0.913,0.917,0.915,0.909
HistGradientBoosting,0.897,0.897,0.890,0.910,0.886,1.000,0.888,0.893,0.892,0.876,0.866,0.864,0.868,0.882,0.874,0.873
XGBoost,0.960,0.960,0.953,0.916,0.928,0.888,1.000,0.988,0.984,0.967,0.955,0.961,0.954,0.964,0.966,0.952
LightGBM,0.974,0.974,0.965,0.914,0.929,0.893,0.988,1.000,0.991,0.980,0.969,0.975,0.968,0.978,0.978,0.966
CatBoost,0.983,0.983,0.975,0.911,0.927,0.892,0.984,0.991,1.000,0.989,0.980,0.984,0.978,0.987,0.986,0.973
MLP_BCE,0.983,0.983,0.974,0.899,0.919,0.876,0.967,0.980,0.989,1.000,0.987,0.993,0.984,0.988,0.988,0.974



RANK CORRELATION


...


Provisional anchor: CatBoost
Anchor FN count: 7990
Anchor FP count: 22736

ANCHOR ERROR-RESCUE ANALYSIS


,Anchor,Candidate,Anchor_FN,FN_rescued,FN_rescue_rate,Anchor_FP,FP_corrected,FP_correction_rate,Spearman_with_anchor
0,CatBoost,RandomForest,7990,1209,0.151314,22736,3853,0.169467,0.941947
1,CatBoost,ExtraTrees,7990,947,0.118523,22736,6475,0.284791,0.943878
2,CatBoost,LogReg_Subset,7990,740,0.092616,22736,3493,0.153633,0.974532
3,CatBoost,LogReg_L2,7990,580,0.072591,22736,3198,0.140658,0.983391
4,CatBoost,LogReg_ElasticNet,7990,578,0.072340,22736,3198,0.140658,0.983394
5,CatBoost,MLP_Focal_FGSM,7990,537,0.067209,22736,2935,0.129090,0.987217
6,CatBoost,TabNet,7990,511,0.063955,22736,5057,0.222423,0.970193
7,CatBoost,MLP_FGSM,7990,492,0.061577,22736,3621,0.159263,0.986135
8,CatBoost,MLP_BCE,7990,377,0.047184,22736,3035,0.133489,0.988730
9,CatBoost,DeepFM_v2,7990,367,0.045932,22736,4079,0.179407,0.985767



MODEL-LEVEL DIVERSITY SUMMARY


,Model,PR_AUC,Avg_Spearman_to_Others,Avg_Decision_Disagreement,Avg_Positive_Disagreement,Avg_Negative_Disagreement,Rank_Diversity
0,CatBoost,0.377535,0.978533,0.039227,0.085292,0.034437,0.021467
1,HistGradientBoosting,0.377012,0.976673,0.039851,0.087098,0.034939,0.023327
2,AutoInt,0.376859,0.972586,0.041906,0.092267,0.036670,0.027414
3,MLP_BCE,0.376499,0.977594,0.036576,0.080109,0.032050,0.022406
4,MLP_Focal,0.375743,0.977316,0.037700,0.082348,0.033058,0.022684
5,MLP_Focal_FGSM,0.374405,0.977101,0.040090,0.086603,0.035254,0.022899
6,DeepFM_v2,0.374324,0.973423,0.043046,0.095107,0.037633,0.026577
7,LightGBM,0.372779,0.971539,0.040535,0.090510,0.035338,0.028461
8,MLP_FGSM,0.372766,0.975803,0.040284,0.087489,0.035376,0.024197
9,LogReg_ElasticNet,0.371730,0.973849,0.040985,0.088515,0.036043,0.026151



MODULE 8 COMPLETE.


In [11]:
# ============================================================
# MODULE 9 — CROSS-FITTED CALIBRATION BANK
# ============================================================
#
# Methods:
#   1. Raw / Identity
#   2. Platt scaling on logit(p)
#   3. Isotonic regression
#
# Selection:
#   - DEV OOF only
#   - preserve ranking performance
#   - minimize Brier, then NLL, then ECE
#
# HOLDOUT:
#   - never used for calibrator selection
#   - transformed only after method is frozen
# ============================================================

import gc
import json
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

# ------------------------------------------------------------
# 9.1 Canonical candidate bank
# ------------------------------------------------------------

CALIBRATION_MODELS = CANDIDATE_MODELS.copy()

assert "DeepFM" not in CALIBRATION_MODELS
assert "DeepFM_v2" in CALIBRATION_MODELS

missing_oof = [
    m for m in CALIBRATION_MODELS
    if m not in OOF_BANK.columns
]

missing_hold = [
    m for m in CALIBRATION_MODELS
    if m not in HOLDOUT_BANK.columns
]

if missing_oof:
    raise ValueError(
        f"Missing OOF predictions: {missing_oof}"
    )

if missing_hold:
    raise ValueError(
        f"Missing HOLDOUT predictions: {missing_hold}"
    )

print("=" * 100)
print("MODULE 9 — CROSS-FITTED CALIBRATION")
print("=" * 100)

print(
    "Models:",
    len(CALIBRATION_MODELS)
)

# ------------------------------------------------------------
# 9.2 Platt calibrator
# ------------------------------------------------------------

class PlattCalibrator:

    def __init__(self):

        self.model = LogisticRegression(
            penalty="l2",
            C=1e6,
            solver="lbfgs",
            max_iter=2000,
            random_state=SEED,
        )

    def fit(
        self,
        p,
        y
    ):

        x = safe_logit(
            p
        ).reshape(-1, 1)

        self.model.fit(
            x,
            np.asarray(y).astype(int)
        )

        return self

    def predict(self, p):

        x = safe_logit(
            p
        ).reshape(-1, 1)

        return clip_probs(
            self.model.predict_proba(
                x
            )[:, 1]
        )


# ------------------------------------------------------------
# 9.3 Isotonic calibrator
# ------------------------------------------------------------

class IsotonicCalibrator:

    def __init__(self):

        self.model = IsotonicRegression(
            y_min=EPS,
            y_max=1.0 - EPS,
            increasing=True,
            out_of_bounds="clip",
        )

    def fit(
        self,
        p,
        y
    ):

        self.model.fit(
            np.asarray(p),
            np.asarray(y).astype(int)
        )

        return self

    def predict(self, p):

        return clip_probs(
            self.model.predict(
                np.asarray(p)
            )
        )


# ------------------------------------------------------------
# 9.4 Factory
# ------------------------------------------------------------

def make_calibrator(
    method
):

    if method == "platt":
        return PlattCalibrator()

    if method == "isotonic":
        return IsotonicCalibrator()

    raise ValueError(
        f"Unknown calibrator: {method}"
    )


# ------------------------------------------------------------
# 9.5 Cross-fitted calibration
# ------------------------------------------------------------

def cross_fitted_calibration(
    raw_oof,
    method
):
    """
    Calibration fold k:
        FIT calibrator on OOF rows from folds != k
        APPLY calibrator to OOF rows from fold == k

    The resulting vector is itself cross-fitted.
    """

    raw_oof = clip_probs(
        raw_oof
    )

    calibrated = np.zeros(
        len(raw_oof),
        dtype=np.float64
    )

    for fold in range(
        N_SPLITS
    ):

        cal_train_mask = (
            fold_assignment != fold
        )

        cal_valid_mask = (
            fold_assignment == fold
        )

        calibrator = make_calibrator(
            method
        )

        calibrator.fit(
            raw_oof[
                cal_train_mask
            ],
            y_dev.to_numpy()[
                cal_train_mask
            ]
        )

        calibrated[
            cal_valid_mask
        ] = calibrator.predict(
            raw_oof[
                cal_valid_mask
            ]
        )

    if not np.isfinite(
        calibrated
    ).all():

        raise ValueError(
            f"{method}: non-finite calibrated probabilities."
        )

    return clip_probs(
        calibrated
    )


# ------------------------------------------------------------
# 9.6 Full DEV calibrator for frozen HOLDOUT
# ------------------------------------------------------------

def calibrate_holdout_after_selection(
    raw_oof,
    raw_holdout,
    method
):

    if method == "raw":

        return clip_probs(
            raw_holdout
        )

    calibrator = make_calibrator(
        method
    )

    calibrator.fit(
        raw_oof,
        y_dev.to_numpy()
    )

    return calibrator.predict(
        raw_holdout
    )


# ------------------------------------------------------------
# 9.7 Calibration metrics
# ------------------------------------------------------------

def calibration_candidate_metrics(
    model_name,
    method,
    probabilities
):

    probability_result = (
        probability_metrics(
            y_dev,
            probabilities,
            ece_bins=15
        )
    )

    return {

        "Model":
            model_name,

        "Method":
            method,

        "PR_AUC":
            probability_result[
                "PR_AUC"
            ],

        "ROC_AUC":
            probability_result[
                "ROC_AUC"
            ],

        "Brier":
            probability_result[
                "Brier"
            ],

        "NLL":
            probability_result[
                "NLL"
            ],

        "ECE":
            probability_result[
                "ECE"
            ],

        "Mean_Probability":
            probability_result[
                "Mean_Probability"
            ],

        "Observed_Prevalence":
            probability_result[
                "Observed_Prevalence"
            ],
    }


# ------------------------------------------------------------
# 9.8 Build all calibration candidates
# ------------------------------------------------------------

CALIBRATION_CANDIDATE_PREDS = {}

calibration_rows = []

for model_name in (
    CALIBRATION_MODELS
):

    print(
        f"\nCalibrating {model_name}"
    )

    raw_oof = clip_probs(
        OOF_BANK[
            model_name
        ].to_numpy()
    )

    # --------------------------------------------------------
    # Raw candidate
    # --------------------------------------------------------

    CALIBRATION_CANDIDATE_PREDS[
        (
            model_name,
            "raw"
        )
    ] = raw_oof.copy()

    calibration_rows.append(
        calibration_candidate_metrics(
            model_name,
            "raw",
            raw_oof
        )
    )

    # --------------------------------------------------------
    # Platt
    # --------------------------------------------------------

    platt_oof = (
        cross_fitted_calibration(
            raw_oof,
            method="platt"
        )
    )

    CALIBRATION_CANDIDATE_PREDS[
        (
            model_name,
            "platt"
        )
    ] = platt_oof

    calibration_rows.append(
        calibration_candidate_metrics(
            model_name,
            "platt",
            platt_oof
        )
    )

    # --------------------------------------------------------
    # Isotonic
    # --------------------------------------------------------

    isotonic_oof = (
        cross_fitted_calibration(
            raw_oof,
            method="isotonic"
        )
    )

    CALIBRATION_CANDIDATE_PREDS[
        (
            model_name,
            "isotonic"
        )
    ] = isotonic_oof

    calibration_rows.append(
        calibration_candidate_metrics(
            model_name,
            "isotonic",
            isotonic_oof
        )
    )


calibration_comparison = (
    pd.DataFrame(
        calibration_rows
    )
)

# ------------------------------------------------------------
# 9.9 Ranking preservation
# ------------------------------------------------------------

raw_pr_map = (
    calibration_comparison[
        calibration_comparison[
            "Method"
        ] == "raw"
    ]
    .set_index("Model")[
        "PR_AUC"
    ]
    .to_dict()
)

calibration_comparison[
    "PR_Delta_vs_Raw"
] = calibration_comparison.apply(
    lambda r:
        r["PR_AUC"]
        - raw_pr_map[
            r["Model"]
        ],
    axis=1
)

# ------------------------------------------------------------
# 9.10 Calibration method selection
# ------------------------------------------------------------
#
# Requirements:
#
# 1. do not sacrifice > 0.0005 PR-AUC
# 2. choose minimum cross-fitted Brier
# 3. NLL tie-breaker
# 4. ECE tie-breaker
#
# Including RAW means an already well-calibrated model
# such as HistGB does not have to be recalibrated.
# ------------------------------------------------------------

MAX_PR_DROP = 0.0005

selected_rows = []

SELECTED_CALIBRATION = {}

for model_name in (
    CALIBRATION_MODELS
):

    sub = (
        calibration_comparison[
            calibration_comparison[
                "Model"
            ] == model_name
        ]
        .copy()
    )

    raw_pr = raw_pr_map[
        model_name
    ]

    eligible = sub[
        sub["PR_AUC"]
        >= raw_pr - MAX_PR_DROP
    ].copy()

    if len(eligible) == 0:

        eligible = sub[
            sub["Method"]
            == "raw"
        ].copy()

    eligible = (
        eligible
        .sort_values(
            [
                "Brier",
                "NLL",
                "ECE"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
    )

    chosen = (
        eligible.iloc[0]
    )

    method = chosen[
        "Method"
    ]

    SELECTED_CALIBRATION[
        model_name
    ] = method

    selected_rows.append(
        chosen.to_dict()
    )


selected_calibration_table = (
    pd.DataFrame(
        selected_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9.11 Build selected calibrated OOF bank
# ------------------------------------------------------------

CAL_OOF_BANK = pd.DataFrame({

    "dev_position":
        np.arange(
            len(y_dev)
        ),

    "target":
        y_dev.to_numpy(
            dtype=np.int8
        ),

    "fold":
        fold_assignment,
})


CAL_HOLDOUT_BANK = pd.DataFrame({

    "holdout_position":
        np.arange(
            len(y_holdout)
        ),

    "target":
        y_holdout.to_numpy(
            dtype=np.int8
        ),
})


for model_name in (
    CALIBRATION_MODELS
):

    selected_method = (
        SELECTED_CALIBRATION[
            model_name
        ]
    )

    calibrated_oof = (
        CALIBRATION_CANDIDATE_PREDS[
            (
                model_name,
                selected_method
            )
        ]
    )

    CAL_OOF_BANK[
        model_name
    ] = calibrated_oof

    raw_oof = (
        OOF_BANK[
            model_name
        ].to_numpy()
    )

    raw_holdout = (
        HOLDOUT_BANK[
            model_name
        ].to_numpy()
    )

    calibrated_holdout = (
        calibrate_holdout_after_selection(
            raw_oof,
            raw_holdout,
            selected_method
        )
    )

    CAL_HOLDOUT_BANK[
        model_name
    ] = calibrated_holdout


# ------------------------------------------------------------
# 9.12 Save banks
# ------------------------------------------------------------

save_csv(
    CAL_OOF_BANK,
    PRED_DIR
    / "calibrated_oof_prediction_bank.csv"
)

save_csv(
    CAL_HOLDOUT_BANK,
    PRED_DIR
    / "calibrated_holdout_prediction_bank.csv"
)

save_csv(
    calibration_comparison,
    TAB_DIR
    / "module9_all_calibration_candidates.csv"
)

save_csv(
    selected_calibration_table,
    TAB_DIR
    / "module9_selected_calibration.csv"
)


# ------------------------------------------------------------
# 9.13 Display all methods
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "CALIBRATION METHOD COMPARISON"
)

print(
    "=" * 110
)

display(
    calibration_comparison
    .sort_values(
        [
            "Model",
            "Brier"
        ]
    )
)


# ------------------------------------------------------------
# 9.14 Display selected methods
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "SELECTED CALIBRATION METHOD PER MODEL"
)

print(
    "=" * 110
)

display(
    selected_calibration_table[
        [
            "Model",
            "Method",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "PR_Delta_vs_Raw",
            "Mean_Probability",
        ]
    ]
)


# ------------------------------------------------------------
# 9.15 Before-vs-after table
# ------------------------------------------------------------

before_after_rows = []

for model_name in (
    CALIBRATION_MODELS
):

    raw_row = (
        calibration_comparison[
            (
                calibration_comparison[
                    "Model"
                ] == model_name
            )
            &
            (
                calibration_comparison[
                    "Method"
                ] == "raw"
            )
        ]
        .iloc[0]
    )

    chosen = (
        selected_calibration_table[
            selected_calibration_table[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )

    before_after_rows.append({

        "Model":
            model_name,

        "Calibration":
            chosen["Method"],

        "PR_raw":
            raw_row["PR_AUC"],

        "PR_cal":
            chosen["PR_AUC"],

        "Brier_raw":
            raw_row["Brier"],

        "Brier_cal":
            chosen["Brier"],

        "Brier_gain":
            raw_row["Brier"]
            - chosen["Brier"],

        "NLL_raw":
            raw_row["NLL"],

        "NLL_cal":
            chosen["NLL"],

        "NLL_gain":
            raw_row["NLL"]
            - chosen["NLL"],

        "ECE_raw":
            raw_row["ECE"],

        "ECE_cal":
            chosen["ECE"],

        "ECE_gain":
            raw_row["ECE"]
            - chosen["ECE"],
    })


calibration_gain_table = (
    pd.DataFrame(
        before_after_rows
    )
    .sort_values(
        "Brier_cal"
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 110
)

print(
    "RAW → SELECTED CALIBRATION IMPROVEMENT"
)

print(
    "=" * 110
)

display(
    calibration_gain_table
)


save_csv(
    calibration_gain_table,
    TAB_DIR
    / "module9_calibration_gain_table.csv"
)


# ------------------------------------------------------------
# 9.16 Sanity checks
# ------------------------------------------------------------

for model_name in (
    CALIBRATION_MODELS
):

    p = CAL_OOF_BANK[
        model_name
    ].to_numpy()

    assert np.isfinite(
        p
    ).all()

    assert (
        p.min() > 0
    )

    assert (
        p.max() < 1
    )


print(
    "\nCalibrated OOF bank:",
    CAL_OOF_BANK.shape
)

print(
    "Calibrated HOLDOUT bank:",
    CAL_HOLDOUT_BANK.shape
)

print(
    "\nObserved DEV prevalence:",
    float(
        y_dev.mean()
    )
)

print(
    "\nMean calibrated probabilities:"
)

for m in (
    CALIBRATION_MODELS
):

    print(
        f"{m:<24} "
        f"{CAL_OOF_BANK[m].mean():.6f} "
        f"| {SELECTED_CALIBRATION[m]}"
    )


# ------------------------------------------------------------
# 9.17 Manifest
# ------------------------------------------------------------

module9_manifest = {

    "methods": [
        "raw",
        "platt",
        "isotonic"
    ],

    "selection_source":
        "cross_fitted_DEV_OOF",

    "selection_primary":
        "Brier",

    "selection_tiebreakers": [
        "NLL",
        "ECE"
    ],

    "maximum_allowed_PR_drop":
        MAX_PR_DROP,

    "selected_calibration":
        SELECTED_CALIBRATION,

    "holdout_used_for_selection":
        False,
}

save_json(
    module9_manifest,
    LOG_DIR
    / "module9_calibration_manifest.json"
)

print(
    "\nMODULE 9 COMPLETE."
)

MODULE 9 — CROSS-FITTED CALIBRATION
Models: 16

Calibrating LogReg_L2

Calibrating LogReg_ElasticNet

Calibrating LogReg_Subset

Calibrating RandomForest

Calibrating ExtraTrees

Calibrating HistGradientBoosting

Calibrating XGBoost

Calibrating LightGBM

Calibrating CatBoost

Calibrating MLP_BCE

Calibrating MLP_Focal

Calibrating MLP_FGSM

Calibrating MLP_Focal_FGSM

Calibrating AutoInt

Calibrating DeepFM_v2

Calibrating TabNet

CALIBRATION METHOD COMPARISON


,Model,Method,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_Probability,Observed_Prevalence,PR_Delta_vs_Raw
40,AutoInt,platt,0.375956,0.848609,0.069977,0.236752,0.002604,0.094288,0.094184,-0.000903
41,AutoInt,isotonic,0.372207,0.847996,0.069984,0.236940,0.001129,0.094329,0.094184,-0.004652
39,AutoInt,raw,0.376859,0.849007,0.166093,0.483137,0.245961,0.340145,0.094184,0.000000
25,CatBoost,platt,0.377444,0.849204,0.069874,0.236331,0.001670,0.094167,0.094184,-0.000092
26,CatBoost,isotonic,0.373689,0.848555,0.069900,0.236614,0.000825,0.094193,0.094184,-0.003846
24,CatBoost,raw,0.377535,0.849216,0.161784,0.475348,0.243946,0.338129,0.094184,0.000000
43,DeepFM_v2,platt,0.373844,0.848080,0.070080,0.236999,0.001611,0.094199,0.094184,-0.000480
44,DeepFM_v2,isotonic,0.369788,0.847516,0.070134,0.237409,0.001269,0.094246,0.094184,-0.004536
42,DeepFM_v2,raw,0.374324,0.848231,0.171079,0.497199,0.256397,0.350580,0.094184,0.000000
14,ExtraTrees,isotonic,0.334188,0.833096,0.072261,0.245019,0.000583,0.094176,0.094184,-0.003325



SELECTED CALIBRATION METHOD PER MODEL


,Model,Method,PR_AUC,ROC_AUC,Brier,NLL,ECE,PR_Delta_vs_Raw,Mean_Probability
0,CatBoost,platt,0.377444,0.849204,0.069874,0.236331,0.001670,-0.000092,0.094167
1,HistGradientBoosting,raw,0.377012,0.849628,0.069876,0.236194,0.002378,0.000000,0.094016
2,AutoInt,raw,0.376859,0.849007,0.166093,0.483137,0.245961,0.000000,0.340145
3,MLP_BCE,raw,0.376499,0.850047,0.168696,0.489990,0.253454,0.000000,0.347637
4,MLP_Focal,platt,0.375245,0.849199,0.070078,0.236858,0.006574,-0.000498,0.094208
5,MLP_Focal_FGSM,raw,0.374405,0.849344,0.096750,0.343746,0.155133,0.000000,0.247402
6,DeepFM_v2,platt,0.373844,0.848080,0.070080,0.236999,0.001611,-0.000480,0.094199
7,MLP_FGSM,raw,0.372766,0.848833,0.177083,0.512347,0.269478,0.000000,0.363662
8,LightGBM,platt,0.372688,0.846566,0.070177,0.237764,0.001273,-0.000091,0.094161
9,LogReg_ElasticNet,platt,0.371663,0.847367,0.070429,0.238359,0.009141,-0.000066,0.094147



RAW → SELECTED CALIBRATION IMPROVEMENT


,Model,Calibration,PR_raw,PR_cal,Brier_raw,Brier_cal,Brier_gain,NLL_raw,NLL_cal,NLL_gain,ECE_raw,ECE_cal,ECE_gain
0,CatBoost,platt,0.377535,0.377444,0.161784,0.069874,0.091909,0.475348,0.236331,0.239017,0.243946,0.001670,0.242276
1,HistGradientBoosting,raw,0.377012,0.377012,0.069876,0.069876,0.000000,0.236194,0.236194,0.000000,0.002378,0.002378,0.000000
2,MLP_Focal,platt,0.375743,0.375245,0.097017,0.070078,0.026940,0.344413,0.236858,0.107555,0.155949,0.006574,0.149375
3,DeepFM_v2,platt,0.374324,0.373844,0.171079,0.070080,0.100999,0.497199,0.236999,0.260200,0.256397,0.001611,0.254786
4,LightGBM,platt,0.372779,0.372688,0.157540,0.070177,0.087363,0.463084,0.237764,0.225319,0.233591,0.001273,0.232318
5,LogReg_ElasticNet,platt,0.371730,0.371663,0.164999,0.070429,0.094570,0.495340,0.238359,0.256981,0.256907,0.009141,0.247766
6,LogReg_L2,platt,0.371729,0.371662,0.165008,0.070429,0.094579,0.495369,0.238360,0.257010,0.256929,0.009126,0.247803
7,XGBoost,platt,0.364962,0.364888,0.153124,0.070664,0.082460,0.452376,0.239992,0.212385,0.223666,0.001207,0.222459
8,LogReg_Subset,platt,0.367312,0.367217,0.166113,0.070730,0.095384,0.498671,0.239676,0.258996,0.259250,0.009298,0.249952
9,RandomForest,platt,0.337500,0.337483,0.084384,0.072111,0.012273,0.278252,0.244271,0.033981,0.080645,0.004285,0.076360



Calibrated OOF bank: (202944, 19)
Calibrated HOLDOUT bank: (50736, 18)

Observed DEV prevalence: 0.09418361715547147

Mean calibrated probabilities:
LogReg_L2                0.094147 | platt
LogReg_ElasticNet        0.094147 | platt
LogReg_Subset            0.094156 | platt
RandomForest             0.094151 | platt
ExtraTrees               0.094143 | platt
HistGradientBoosting     0.094016 | raw
XGBoost                  0.094158 | platt
LightGBM                 0.094161 | platt
CatBoost                 0.094167 | platt
MLP_BCE                  0.347637 | raw
MLP_Focal                0.094208 | platt
MLP_FGSM                 0.363662 | raw
MLP_Focal_FGSM           0.247402 | raw
AutoInt                  0.340145 | raw
DeepFM_v2                0.094199 | platt
TabNet                   0.343875 | raw

MODULE 9 COMPLETE.


In [12]:
# ============================================================
# MODULE 9 PATCH — CALIBRATION-FIRST SELECTION POLICY
# ============================================================

# ------------------------------------------------------------
# 1. Calibration quality criteria
# ------------------------------------------------------------

RAW_BRIER_OK = 0.075
RAW_ECE_OK = 0.020
RAW_MEAN_PREV_GAP_OK = 0.010

# Cross-fitted calibration can slightly alter pooled ranking
MAX_CAL_PR_DROP = 0.003

DEV_PREVALENCE = float(y_dev.mean())

SELECTED_CALIBRATION_V2 = {}
selection_rows_v2 = []

for model_name in CALIBRATION_MODELS:

    sub = (
        calibration_comparison[
            calibration_comparison["Model"] == model_name
        ]
        .copy()
    )

    raw_row = (
        sub[
            sub["Method"] == "raw"
        ]
        .iloc[0]
    )

    raw_pr = float(
        raw_row["PR_AUC"]
    )

    raw_well_calibrated = (
        raw_row["Brier"] <= RAW_BRIER_OK
        and raw_row["ECE"] <= RAW_ECE_OK
        and abs(
            raw_row["Mean_Probability"]
            - DEV_PREVALENCE
        ) <= RAW_MEAN_PREV_GAP_OK
    )

    # --------------------------------------------------------
    # CASE A:
    # Raw model is already probabilistically trustworthy.
    # Raw remains eligible.
    # --------------------------------------------------------

    if raw_well_calibrated:

        eligible = sub[
            sub["PR_AUC"]
            >= raw_pr - MAX_CAL_PR_DROP
        ].copy()

    # --------------------------------------------------------
    # CASE B:
    # Raw model is clearly miscalibrated.
    # Calibration becomes mandatory.
    # --------------------------------------------------------

    else:

        eligible = sub[
            sub["Method"].isin(
                ["platt", "isotonic"]
            )
        ].copy()

        eligible = eligible[
            eligible["PR_AUC"]
            >= raw_pr - MAX_CAL_PR_DROP
        ]

        # Safety fallback:
        # if isotonic / Platt both move pooled ranking too much,
        # prefer Platt rather than returning to a grossly
        # miscalibrated raw probability scale.
        if len(eligible) == 0:

            eligible = sub[
                sub["Method"] == "platt"
            ].copy()

    # --------------------------------------------------------
    # Select on probability quality
    # --------------------------------------------------------

    eligible = (
        eligible
        .sort_values(
            [
                "Brier",
                "NLL",
                "ECE"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
    )

    chosen = (
        eligible.iloc[0]
    )

    method = chosen["Method"]

    SELECTED_CALIBRATION_V2[
        model_name
    ] = method

    row = chosen.to_dict()

    row["Raw_Well_Calibrated"] = (
        raw_well_calibrated
    )

    row["Raw_PR_AUC"] = raw_pr

    row["CrossFit_PR_Shift"] = (
        chosen["PR_AUC"] - raw_pr
    )

    selection_rows_v2.append(
        row
    )


selected_calibration_v2 = (
    pd.DataFrame(
        selection_rows_v2
    )
    .sort_values(
        "Raw_PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "=" * 110
)

print(
    "REVISED CALIBRATION SELECTION"
)

print(
    "=" * 110
)

display(
    selected_calibration_v2[
        [
            "Model",
            "Method",
            "Raw_Well_Calibrated",
            "Raw_PR_AUC",
            "PR_AUC",
            "CrossFit_PR_Shift",
            "Brier",
            "NLL",
            "ECE",
            "Mean_Probability",
        ]
    ]
)


# ============================================================
# 2. Rebuild calibrated OOF/HOLDOUT banks
# ============================================================

CAL_OOF_BANK = pd.DataFrame({

    "dev_position":
        np.arange(len(y_dev)),

    "target":
        y_dev.to_numpy(
            dtype=np.int8
        ),

    "fold":
        fold_assignment,
})


CAL_HOLDOUT_BANK = pd.DataFrame({

    "holdout_position":
        np.arange(
            len(y_holdout)
        ),

    "target":
        y_holdout.to_numpy(
            dtype=np.int8
        ),
})


for model_name in CALIBRATION_MODELS:

    selected_method = (
        SELECTED_CALIBRATION_V2[
            model_name
        ]
    )

    # cross-fitted OOF probability
    calibrated_oof = (
        CALIBRATION_CANDIDATE_PREDS[
            (
                model_name,
                selected_method
            )
        ]
    )

    CAL_OOF_BANK[
        model_name
    ] = calibrated_oof

    # frozen HOLDOUT calibration:
    # calibrator fitted on complete DEV OOF predictions only
    raw_oof = (
        OOF_BANK[
            model_name
        ].to_numpy()
    )

    raw_holdout = (
        HOLDOUT_BANK[
            model_name
        ].to_numpy()
    )

    calibrated_holdout = (
        calibrate_holdout_after_selection(
            raw_oof,
            raw_holdout,
            selected_method
        )
    )

    CAL_HOLDOUT_BANK[
        model_name
    ] = calibrated_holdout


# ------------------------------------------------------------
# 3. Save corrected probability banks
# ------------------------------------------------------------

save_csv(
    CAL_OOF_BANK,
    PRED_DIR
    / "calibrated_oof_prediction_bank.csv"
)

save_csv(
    CAL_HOLDOUT_BANK,
    PRED_DIR
    / "calibrated_holdout_prediction_bank.csv"
)

save_csv(
    selected_calibration_v2,
    TAB_DIR
    / "module9_selected_calibration_v2.csv"
)


# ============================================================
# 4. Build final calibration audit
# ============================================================

audit_rows = []

for model_name in CALIBRATION_MODELS:

    p_raw = (
        OOF_BANK[
            model_name
        ].to_numpy()
    )

    p_cal = (
        CAL_OOF_BANK[
            model_name
        ].to_numpy()
    )

    raw_m = probability_metrics(
        y_dev,
        p_raw
    )

    cal_m = probability_metrics(
        y_dev,
        p_cal
    )

    audit_rows.append({

        "Model":
            model_name,

        "Method":
            SELECTED_CALIBRATION_V2[
                model_name
            ],

        # ranking authority
        "Raw_PR_AUC":
            raw_m["PR_AUC"],

        "Raw_ROC_AUC":
            raw_m["ROC_AUC"],

        # cross-fit calibrated diagnostics
        "CrossFit_Cal_PR_AUC":
            cal_m["PR_AUC"],

        "CrossFit_Cal_ROC_AUC":
            cal_m["ROC_AUC"],

        "Brier_raw":
            raw_m["Brier"],

        "Brier_cal":
            cal_m["Brier"],

        "NLL_raw":
            raw_m["NLL"],

        "NLL_cal":
            cal_m["NLL"],

        "ECE_raw":
            raw_m["ECE"],

        "ECE_cal":
            cal_m["ECE"],

        "Mean_raw":
            float(
                np.mean(p_raw)
            ),

        "Mean_cal":
            float(
                np.mean(p_cal)
            ),

        "DEV_Prevalence":
            DEV_PREVALENCE,
    })


calibration_audit_v2 = (
    pd.DataFrame(
        audit_rows
    )
    .sort_values(
        "Raw_PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n"
    + "=" * 120
)

print(
    "FINAL CALIBRATION AUDIT"
)

print(
    "=" * 120
)

display(
    calibration_audit_v2
)

save_csv(
    calibration_audit_v2,
    TAB_DIR
    / "module9_final_calibration_audit.csv"
)


# ============================================================
# 5. Probability-scale sanity check
# ============================================================

print(
    "\nObserved DEV prevalence:",
    f"{DEV_PREVALENCE:.6f}"
)

print(
    "\nFinal calibrated means:"
)

for model_name in CALIBRATION_MODELS:

    mean_p = float(
        CAL_OOF_BANK[
            model_name
        ].mean()
    )

    print(
        f"{model_name:<24} "
        f"{mean_p:.6f} "
        f"| "
        f"{SELECTED_CALIBRATION_V2[model_name]}"
    )


# ------------------------------------------------------------
# 6. Strong sanity warning
# ------------------------------------------------------------

bad_scale_models = []

for model_name in CALIBRATION_MODELS:

    gap = abs(
        CAL_OOF_BANK[
            model_name
        ].mean()
        - DEV_PREVALENCE
    )

    if gap > 0.02:

        bad_scale_models.append(
            (
                model_name,
                float(gap)
            )
        )

if bad_scale_models:

    print(
        "\nWARNING: Remaining large "
        "mean-probability gaps:"
    )

    print(
        bad_scale_models
    )

else:

    print(
        "\nProbability-scale sanity check: PASSED"
    )


# ------------------------------------------------------------
# 7. Update manifest
# ------------------------------------------------------------

module9_manifest_v2 = {

    "raw_bank_role":
        "authoritative ranking performance",

    "calibrated_bank_role":
        (
            "probability fusion, uncertainty, "
            "gating and decision analysis"
        ),

    "calibration_methods":
        [
            "raw",
            "platt",
            "isotonic"
        ],

    "raw_well_calibrated_criteria": {

        "max_brier":
            RAW_BRIER_OK,

        "max_ece":
            RAW_ECE_OK,

        "max_mean_prevalence_gap":
            RAW_MEAN_PREV_GAP_OK,
    },

    "maximum_crossfit_PR_shift":
        MAX_CAL_PR_DROP,

    "selected_methods":
        SELECTED_CALIBRATION_V2,

    "holdout_used_for_selection":
        False,
}

save_json(
    module9_manifest_v2,
    LOG_DIR
    / "module9_calibration_manifest_v2.json"
)

print(
    "\nMODULE 9 PATCH COMPLETE."
)

REVISED CALIBRATION SELECTION


,Model,Method,Raw_Well_Calibrated,Raw_PR_AUC,PR_AUC,CrossFit_PR_Shift,Brier,NLL,ECE,Mean_Probability
0,CatBoost,platt,False,0.377535,0.377444,-0.000092,0.069874,0.236331,0.001670,0.094167
1,HistGradientBoosting,raw,True,0.377012,0.377012,0.000000,0.069876,0.236194,0.002378,0.094016
2,AutoInt,platt,False,0.376859,0.375956,-0.000903,0.069977,0.236752,0.002604,0.094288
3,MLP_BCE,platt,False,0.376499,0.375822,-0.000677,0.069887,0.236091,0.002118,0.094204
4,MLP_Focal,platt,False,0.375743,0.375245,-0.000498,0.070078,0.236858,0.006574,0.094208
5,MLP_Focal_FGSM,platt,False,0.374405,0.373583,-0.000821,0.070070,0.236682,0.004590,0.094219
6,DeepFM_v2,platt,False,0.374324,0.373844,-0.000480,0.070080,0.236999,0.001611,0.094199
7,LightGBM,platt,False,0.372779,0.372688,-0.000091,0.070177,0.237764,0.001273,0.094161
8,MLP_FGSM,platt,False,0.372766,0.370484,-0.002282,0.070205,0.237076,0.002198,0.094353
9,LogReg_ElasticNet,platt,False,0.371730,0.371663,-0.000066,0.070429,0.238359,0.009141,0.094147



FINAL CALIBRATION AUDIT


,Model,Method,Raw_PR_AUC,Raw_ROC_AUC,CrossFit_Cal_PR_AUC,CrossFit_Cal_ROC_AUC,Brier_raw,Brier_cal,NLL_raw,NLL_cal,ECE_raw,ECE_cal,Mean_raw,Mean_cal,DEV_Prevalence
0,CatBoost,platt,0.377535,0.849216,0.377444,0.849204,0.161784,0.069874,0.475348,0.236331,0.243946,0.001670,0.338129,0.094167,0.094184
1,HistGradientBoosting,raw,0.377012,0.849628,0.377012,0.849628,0.069876,0.069876,0.236194,0.236194,0.002378,0.002378,0.094016,0.094016,0.094184
2,AutoInt,platt,0.376859,0.849007,0.375956,0.848609,0.166093,0.069977,0.483137,0.236752,0.245961,0.002604,0.340145,0.094288,0.094184
3,MLP_BCE,platt,0.376499,0.850047,0.375822,0.849954,0.168696,0.069887,0.489990,0.236091,0.253454,0.002118,0.347637,0.094204,0.094184
4,MLP_Focal,platt,0.375743,0.849344,0.375245,0.849199,0.097017,0.070078,0.344413,0.236858,0.155949,0.006574,0.248018,0.094208,0.094184
5,MLP_Focal_FGSM,platt,0.374405,0.849344,0.373583,0.849244,0.096750,0.070070,0.343746,0.236682,0.155133,0.004590,0.247402,0.094219,0.094184
6,DeepFM_v2,platt,0.374324,0.848231,0.373844,0.848080,0.171079,0.070080,0.497199,0.236999,0.256397,0.001611,0.350580,0.094199,0.094184
7,LightGBM,platt,0.372779,0.846576,0.372688,0.846566,0.157540,0.070177,0.463084,0.237764,0.233591,0.001273,0.327775,0.094161,0.094184
8,MLP_FGSM,platt,0.372766,0.848833,0.370484,0.848250,0.177083,0.070205,0.512347,0.237076,0.269478,0.002198,0.363662,0.094353,0.094184
9,LogReg_ElasticNet,platt,0.371730,0.847375,0.371663,0.847367,0.164999,0.070429,0.495340,0.238359,0.256907,0.009141,0.351091,0.094147,0.094184



Observed DEV prevalence: 0.094184

Final calibrated means:
LogReg_L2                0.094147 | platt
LogReg_ElasticNet        0.094147 | platt
LogReg_Subset            0.094156 | platt
RandomForest             0.094151 | platt
ExtraTrees               0.094143 | platt
HistGradientBoosting     0.094016 | raw
XGBoost                  0.094158 | platt
LightGBM                 0.094161 | platt
CatBoost                 0.094167 | platt
MLP_BCE                  0.094204 | platt
MLP_Focal                0.094208 | platt
MLP_FGSM                 0.094353 | platt
MLP_Focal_FGSM           0.094219 | platt
AutoInt                  0.094288 | platt
DeepFM_v2                0.094199 | platt
TabNet                   0.094246 | platt

Probability-scale sanity check: PASSED

MODULE 9 PATCH COMPLETE.


In [13]:
# ============================================================
# MODULE 10 FAST — NESTED SIMPLE-ENSEMBLE CEILING
# ============================================================
#
# Fast version:
# - 7 serious fusion candidates
# - subset sizes 2–4
# - fixed 5-fold meta-CV
# - no HOLDOUT use
# - no exhaustive weak-model search
#
# ============================================================

import itertools
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import average_precision_score


# ------------------------------------------------------------
# 10F.1 Compact fusion pool
# ------------------------------------------------------------

FUSION_MODELS_FAST = [
    "CatBoost",
    "HistGradientBoosting",
    "AutoInt",
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
    "DeepFM_v2",
]

print("=" * 100)
print("MODULE 10 FAST — FUSION POOL")
print("=" * 100)

for i, m in enumerate(
    FUSION_MODELS_FAST,
    1
):
    print(f"{i:02d}. {m}")


# ------------------------------------------------------------
# 10F.2 Helpers
# ------------------------------------------------------------

def probability_mean(P):
    return clip_probs(
        P.mean(axis=1)
    )


def logit_mean(P):

    P = np.clip(
        P,
        EPS,
        1 - EPS
    )

    logits = np.log(
        P / (1 - P)
    )

    return clip_probs(
        sigmoid(
            logits.mean(axis=1)
        )
    )


def weighted_mean(
    P,
    weights
):

    w = np.asarray(
        weights,
        dtype=float
    )

    w = np.maximum(
        w,
        0
    )

    if w.sum() == 0:
        w = np.ones_like(w)

    w /= w.sum()

    return clip_probs(
        P @ w
    )


def fit_nll_weights(
    P,
    y
):

    n_models = P.shape[1]

    x0 = (
        np.ones(n_models)
        / n_models
    )

    def objective(w):

        p = weighted_mean(
            P,
            w
        )

        return -np.mean(
            y * np.log(p)
            +
            (1-y)
            * np.log(1-p)
        )

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=[
            (0, 1)
            for _ in range(
                n_models
            )
        ],
        constraints={
            "type": "eq",
            "fun":
                lambda w:
                    w.sum() - 1
        },
        options={
            "maxiter": 150,
            "ftol": 1e-8
        }
    )

    if not result.success:
        return x0

    w = np.maximum(
        result.x,
        0
    )

    return (
        w / w.sum()
    )


# ------------------------------------------------------------
# 10F.3 Cache calibrated matrix
# ------------------------------------------------------------

P_all = (
    CAL_OOF_BANK[
        FUSION_MODELS_FAST
    ]
    .to_numpy(
        dtype=np.float64
    )
)

y_all = (
    y_dev.to_numpy(
        dtype=int
    )
)

model_to_col = {
    m: i
    for i, m
    in enumerate(
        FUSION_MODELS_FAST
    )
}


# ------------------------------------------------------------
# 10F.4 Precompute combinations
# ------------------------------------------------------------

SUBSETS = []

for size in [
    2,
    3,
    4
]:

    SUBSETS.extend(
        list(
            itertools.combinations(
                FUSION_MODELS_FAST,
                size
            )
        )
    )

print(
    "\nNumber of subsets:",
    len(SUBSETS)
)

# only:
# C(7,2)+C(7,3)+C(7,4)
# = 21+35+35
# = 91 subsets total


# ------------------------------------------------------------
# 10F.5 Fast subset search
# ------------------------------------------------------------

def find_best_subset_fast(
    train_idx
):

    y = y_all[
        train_idx
    ]

    best = None

    for subset in SUBSETS:

        cols = [
            model_to_col[m]
            for m in subset
        ]

        P = (
            P_all[
                train_idx
            ][:, cols]
        )

        pred = (
            P.mean(axis=1)
        )

        pr = (
            average_precision_score(
                y,
                pred
            )
        )

        if (
            best is None
            or pr
            > best[
                "PR_AUC"
            ]
        ):

            best = {
                "Subset":
                    subset,

                "PR_AUC":
                    pr,
            }

    return best


# ------------------------------------------------------------
# 10F.6 Meta predictions
# ------------------------------------------------------------

METHODS = [
    "BestSingle",
    "BestSubsetMean",
    "BestSubsetLogit",
    "BestSubsetNLLWeighted",
    "Top4Mean",
    "All7Mean",
]

META_PRED = {
    m:
        np.zeros(
            len(y_all),
            dtype=float
        )
    for m in METHODS
}

selection_rows = []
weight_rows = []


# fixed strong top-4 control
TOP4 = [
    "CatBoost",
    "HistGradientBoosting",
    "AutoInt",
    "MLP_BCE",
]


# ------------------------------------------------------------
# 10F.7 Outer meta-CV
# ------------------------------------------------------------

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"META FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 90
    )

    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]

    y_train = y_all[
        train_idx
    ]


    # --------------------------------------------------------
    # Best single
    # --------------------------------------------------------

    best_single = None
    best_single_pr = -np.inf

    for j, model_name in enumerate(
        FUSION_MODELS_FAST
    ):

        pr = (
            average_precision_score(
                y_train,
                P_all[
                    train_idx,
                    j
                ]
            )
        )

        if pr > best_single_pr:

            best_single_pr = pr
            best_single = model_name


    single_col = (
        model_to_col[
            best_single
        ]
    )

    META_PRED[
        "BestSingle"
    ][valid_idx] = (
        P_all[
            valid_idx,
            single_col
        ]
    )


    # --------------------------------------------------------
    # Best subset
    # --------------------------------------------------------

    best = (
        find_best_subset_fast(
            train_idx
        )
    )

    subset = best[
        "Subset"
    ]

    cols = [
        model_to_col[m]
        for m in subset
    ]

    P_train_sub = (
        P_all[
            train_idx
        ][:, cols]
    )

    P_valid_sub = (
        P_all[
            valid_idx
        ][:, cols]
    )


    META_PRED[
        "BestSubsetMean"
    ][valid_idx] = (
        probability_mean(
            P_valid_sub
        )
    )

    META_PRED[
        "BestSubsetLogit"
    ][valid_idx] = (
        logit_mean(
            P_valid_sub
        )
    )


    # --------------------------------------------------------
    # NLL-weighted subset
    # --------------------------------------------------------

    weights = (
        fit_nll_weights(
            P_train_sub,
            y_train
        )
    )

    META_PRED[
        "BestSubsetNLLWeighted"
    ][valid_idx] = (
        weighted_mean(
            P_valid_sub,
            weights
        )
    )


    # --------------------------------------------------------
    # Fixed top-4 mean
    # --------------------------------------------------------

    top4_cols = [
        model_to_col[m]
        for m in TOP4
    ]

    META_PRED[
        "Top4Mean"
    ][valid_idx] = (
        probability_mean(
            P_all[
                valid_idx
            ][:, top4_cols]
        )
    )


    # --------------------------------------------------------
    # All-seven mean
    # --------------------------------------------------------

    META_PRED[
        "All7Mean"
    ][valid_idx] = (
        probability_mean(
            P_all[
                valid_idx
            ]
        )
    )


    # --------------------------------------------------------
    # Log selections
    # --------------------------------------------------------

    selection_rows.append({

        "Outer_Fold":
            outer_fold,

        "Best_Single":
            best_single,

        "Best_Single_Train_PR":
            best_single_pr,

        "Selected_Subset":
            " | ".join(
                subset
            ),

        "N_Experts":
            len(subset),

        "Subset_Train_PR":
            best[
                "PR_AUC"
            ],
    })


    for m, w in zip(
        subset,
        weights
    ):

        weight_rows.append({

            "Outer_Fold":
                outer_fold,

            "Model":
                m,

            "NLL_Weight":
                float(w),
        })


    print(
        "Best single:",
        best_single,
        f"PR={best_single_pr:.6f}"
    )

    print(
        "Best subset:",
        subset,
        f"PR={best['PR_AUC']:.6f}"
    )


# ------------------------------------------------------------
# 10F.8 Evaluate cross-fitted predictions
# ------------------------------------------------------------

result_rows = []

for method in METHODS:

    p = META_PRED[
        method
    ]

    assert np.isfinite(
        p
    ).all()

    threshold, _, _ = (
        select_best_threshold(
            y_all,
            p,
            objective="F1"
        )
    )

    m = (
        evaluate_binary_predictions(
            y_all,
            p,
            threshold=threshold
        )
    )

    result_rows.append({

        "Method":
            method,

        **m,
    })


# ------------------------------------------------------------
# 10F.9 Standalone references
# ------------------------------------------------------------

for ref_model in [
    "CatBoost",
    "HistGradientBoosting",
    "AutoInt",
    "MLP_BCE",
]:

    # use RAW predictions for authoritative ranking reference
    p = (
        OOF_BANK[
            ref_model
        ]
        .to_numpy()
    )

    threshold, _, _ = (
        select_best_threshold(
            y_all,
            p,
            objective="F1"
        )
    )

    m = (
        evaluate_binary_predictions(
            y_all,
            p,
            threshold=threshold
        )
    )

    result_rows.append({

        "Method":
            f"Standalone::{ref_model}",

        **m,
    })


results = (
    pd.DataFrame(
        result_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


BEST_STANDALONE_PR = (
    complete_leaderboard[
        "PR_AUC"
    ].max()
)

results[
    "Delta_PR_vs_BestStandalone"
] = (
    results["PR_AUC"]
    - BEST_STANDALONE_PR
)


print(
    "\n"
    + "=" * 110
)

print(
    "FAST NESTED SIMPLE-ENSEMBLE RESULTS"
)

print(
    "=" * 110
)

display(
    results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_BestStandalone",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ------------------------------------------------------------
# 10F.10 Selections
# ------------------------------------------------------------

selection_table = pd.DataFrame(
    selection_rows
)

weight_table = pd.DataFrame(
    weight_rows
)

print(
    "\n"
    + "=" * 110
)

print(
    "META-FOLD SELECTIONS"
)

print(
    "=" * 110
)

display(
    selection_table
)


# ------------------------------------------------------------
# 10F.11 Frequency
# ------------------------------------------------------------

freq_rows = []

for model_name in (
    FUSION_MODELS_FAST
):

    n = 0

    for s in (
        selection_table[
            "Selected_Subset"
        ]
    ):

        if model_name in (
            s.split(" | ")
        ):
            n += 1

    freq_rows.append({

        "Model":
            model_name,

        "Selected_Folds":
            n,

        "Selection_Rate":
            n / N_SPLITS,
    })


frequency_table = (
    pd.DataFrame(
        freq_rows
    )
    .sort_values(
        [
            "Selected_Folds",
            "Model"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 110
)

print(
    "EXPERT SELECTION FREQUENCY"
)

print(
    "=" * 110
)

display(
    frequency_table
)


# ------------------------------------------------------------
# 10F.12 Save
# ------------------------------------------------------------

save_csv(
    results,
    TAB_DIR
    / "module10_fast_ensemble_results.csv"
)

save_csv(
    selection_table,
    TAB_DIR
    / "module10_fast_meta_selections.csv"
)

save_csv(
    frequency_table,
    TAB_DIR
    / "module10_fast_selection_frequency.csv"
)

save_csv(
    weight_table,
    TAB_DIR
    / "module10_fast_weights.csv"
)

meta_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,
})

for method in METHODS:

    meta_pred_df[
        method
    ] = META_PRED[
        method
    ]

save_csv(
    meta_pred_df,
    PRED_DIR
    / "module10_fast_meta_oof_predictions.csv"
)

print(
    "\nMODULE 10 FAST COMPLETE."
)

MODULE 10 FAST — FUSION POOL
01. CatBoost
02. HistGradientBoosting
03. AutoInt
04. MLP_BCE
05. MLP_Focal
06. MLP_Focal_FGSM
07. DeepFM_v2

Number of subsets: 91

META FOLD 1/5
Best single: CatBoost PR=0.377785
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.380515

META FOLD 2/5
Best single: CatBoost PR=0.377145
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.379842

META FOLD 3/5
Best single: AutoInt PR=0.378197
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.380852

META FOLD 4/5
Best single: MLP_BCE PR=0.379927
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.382410

META FOLD 5/5
Best single: HistGradientBoosting PR=0.375007
Best subset: ('HistGradientBoosting', 'AutoInt', 'MLP_BCE', 'DeepFM_v2') PR=0.378270

FAST NESTED SIMPLE-ENSEMBLE RESULTS


,Method,PR_AUC,Delta_PR_vs_BestStandalone,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,BestSubsetMean,0.380129,0.002593,0.850561,0.069716,0.235601,0.002452,0.350761,0.525793,0.420801,0.356163,0.220
1,BestSubsetLogit,0.380095,0.002559,0.850525,0.069718,0.235611,0.002159,0.352149,0.523857,0.421174,0.356502,0.220
2,All7Mean,0.380079,0.002543,0.850832,0.069716,0.235516,0.002914,0.347531,0.536152,0.421711,0.357633,0.220
3,Top4Mean,0.380052,0.002517,0.850778,0.069699,0.235495,0.002115,0.346361,0.538767,0.421652,0.357702,0.215
4,BestSubsetNLLWeighted,0.380011,0.002476,0.850566,0.069719,0.235605,0.002153,0.351877,0.525165,0.421402,0.356804,0.220
5,Standalone::CatBoost,0.377535,0.000000,0.849216,0.161784,0.475348,0.243946,0.328529,0.581982,0.419980,0.359039,0.690
6,Standalone::HistGradientBoosting,0.377012,-0.000524,0.849628,0.069876,0.236194,0.002378,0.348113,0.532332,0.420950,0.356610,0.215
7,Standalone::AutoInt,0.376859,-0.000677,0.849007,0.166093,0.483137,0.245961,0.354659,0.512766,0.419303,0.354068,0.735
8,Standalone::MLP_BCE,0.376499,-0.001037,0.850047,0.168696,0.489990,0.253454,0.336808,0.559538,0.420500,0.357756,0.715
9,BestSingle,0.375170,-0.002366,0.849148,0.069986,0.236563,0.003737,0.336759,0.557131,0.419781,0.356802,0.215



META-FOLD SELECTIONS


,Outer_Fold,Best_Single,Best_Single_Train_PR,Selected_Subset,N_Experts,Subset_Train_PR
0,0,CatBoost,0.377785,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.380515
1,1,CatBoost,0.377145,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.379842
2,2,AutoInt,0.378197,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.380852
3,3,MLP_BCE,0.379927,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.382410
4,4,HistGradientBoosting,0.375007,HistGradientBoosting | AutoInt | MLP_BCE | Dee...,4,0.378270



EXPERT SELECTION FREQUENCY


,Model,Selected_Folds,Selection_Rate
0,AutoInt,5,1.0
1,DeepFM_v2,5,1.0
2,HistGradientBoosting,5,1.0
3,CatBoost,4,0.8
4,MLP_BCE,1,0.2
5,MLP_Focal,0,0.0
6,MLP_Focal_FGSM,0,0.0



MODULE 10 FAST COMPLETE.


In [14]:
# ============================================================
# MODULE 10R — STRICT NESTED CALIBRATION + ENSEMBLE SEARCH
# ============================================================
#
# Fixes subtle calibration/meta-CV leakage:
#
# Outer fold k:
#   1. take RAW OOF base predictions
#   2. fit calibrators on meta-training rows only
#   3. transform meta-train and meta-valid
#   4. select ensemble using meta-train only
#   5. predict meta-valid
#
# No base models retrained.
# HOLDOUT untouched.
# ============================================================

import itertools
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import average_precision_score


# ------------------------------------------------------------
# 10R.1 Expert pool
# ------------------------------------------------------------

NESTED_FUSION_MODELS = [
    "CatBoost",
    "HistGradientBoosting",
    "AutoInt",
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
    "DeepFM_v2",
]

TOP4 = [
    "CatBoost",
    "HistGradientBoosting",
    "AutoInt",
    "MLP_BCE",
]

y_all = y_dev.to_numpy(dtype=int)


# ------------------------------------------------------------
# 10R.2 Calibration inside outer fold
# ------------------------------------------------------------

def fit_apply_nested_calibration(
    model_name,
    train_idx,
    valid_idx
):
    """
    Fits calibration using ONLY meta-training rows.
    """

    method = (
        SELECTED_CALIBRATION_V2[
            model_name
        ]
    )

    raw = (
        OOF_BANK[
            model_name
        ].to_numpy()
    )

    p_train_raw = raw[
        train_idx
    ]

    p_valid_raw = raw[
        valid_idx
    ]

    y_train = y_all[
        train_idx
    ]

    if method == "raw":

        return (
            clip_probs(
                p_train_raw
            ),
            clip_probs(
                p_valid_raw
            )
        )

    calibrator = make_calibrator(
        method
    )

    calibrator.fit(
        p_train_raw,
        y_train
    )

    p_train_cal = (
        calibrator.predict(
            p_train_raw
        )
    )

    p_valid_cal = (
        calibrator.predict(
            p_valid_raw
        )
    )

    return (
        clip_probs(
            p_train_cal
        ),
        clip_probs(
            p_valid_cal
        )
    )


# ------------------------------------------------------------
# 10R.3 Fusion helpers
# ------------------------------------------------------------

def mean_prob(P):

    return clip_probs(
        P.mean(axis=1)
    )


def mean_logit(P):

    P = np.clip(
        P,
        EPS,
        1.0 - EPS
    )

    z = np.log(
        P / (1.0 - P)
    )

    return clip_probs(
        sigmoid(
            z.mean(axis=1)
        )
    )


def weighted_prob(
    P,
    w
):

    w = np.asarray(
        w,
        dtype=float
    )

    w = np.maximum(
        w,
        0
    )

    if w.sum() == 0:
        w = np.ones_like(w)

    w = (
        w / w.sum()
    )

    return clip_probs(
        P @ w
    )


def fit_nested_nll_weights(
    P,
    y
):

    k = P.shape[1]

    x0 = (
        np.ones(k)
        / k
    )

    def loss(w):

        p = weighted_prob(
            P,
            w
        )

        return -np.mean(
            y * np.log(p)
            +
            (1-y)
            * np.log(1-p)
        )

    res = minimize(
        loss,
        x0,
        method="SLSQP",
        bounds=[
            (0.0, 1.0)
            for _ in range(k)
        ],
        constraints={
            "type": "eq",
            "fun":
                lambda w:
                    np.sum(w) - 1.0,
        },
        options={
            "maxiter": 150,
            "ftol": 1e-8,
        },
    )

    if not res.success:
        return x0

    w = np.maximum(
        res.x,
        0
    )

    return (
        w / w.sum()
    )


# ------------------------------------------------------------
# 10R.4 Candidate subsets
# ------------------------------------------------------------

SUBSETS = []

for size in [
    2,
    3,
    4
]:

    SUBSETS.extend(
        itertools.combinations(
            NESTED_FUSION_MODELS,
            size
        )
    )

SUBSETS = list(
    SUBSETS
)

print(
    "Nested subsets:",
    len(SUBSETS)
)


# ------------------------------------------------------------
# 10R.5 Prediction buffers
# ------------------------------------------------------------

METHODS = [
    "BestSingle",
    "BestSubsetMean",
    "BestSubsetLogit",
    "BestSubsetNLLWeighted",
    "Top4Mean",
    "All7Mean",
]

NESTED_PRED = {
    method:
        np.zeros(
            len(y_all),
            dtype=float
        )
    for method in METHODS
}

selection_rows = []
weight_rows = []


# ============================================================
# 10R.6 Outer meta-CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 92
    )

    print(
        f"STRICT META FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 92
    )

    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]

    y_train = (
        y_all[
            train_idx
        ]
    )

    # --------------------------------------------------------
    # Build nested-calibrated matrices
    # --------------------------------------------------------

    P_train_dict = {}
    P_valid_dict = {}

    for model_name in (
        NESTED_FUSION_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        P_train_dict[
            model_name
        ] = p_tr

        P_valid_dict[
            model_name
        ] = p_va


    # --------------------------------------------------------
    # A. Best single
    # --------------------------------------------------------

    single_scores = {}

    for model_name in (
        NESTED_FUSION_MODELS
    ):

        single_scores[
            model_name
        ] = (
            average_precision_score(
                y_train,
                P_train_dict[
                    model_name
                ]
            )
        )

    best_single = max(
        single_scores,
        key=single_scores.get
    )

    NESTED_PRED[
        "BestSingle"
    ][valid_idx] = (
        P_valid_dict[
            best_single
        ]
    )


    # --------------------------------------------------------
    # B. Fast subset search
    # --------------------------------------------------------

    best_subset = None
    best_subset_pr = -np.inf

    for subset in SUBSETS:

        P_train_sub = (
            np.column_stack([
                P_train_dict[m]
                for m in subset
            ])
        )

        pred_train = (
            P_train_sub.mean(
                axis=1
            )
        )

        pr = (
            average_precision_score(
                y_train,
                pred_train
            )
        )

        if pr > best_subset_pr:

            best_subset_pr = pr
            best_subset = subset


    # --------------------------------------------------------
    # C. Held-out predictions from frozen subset
    # --------------------------------------------------------

    P_train_best = (
        np.column_stack([
            P_train_dict[m]
            for m in best_subset
        ])
    )

    P_valid_best = (
        np.column_stack([
            P_valid_dict[m]
            for m in best_subset
        ])
    )

    NESTED_PRED[
        "BestSubsetMean"
    ][valid_idx] = (
        mean_prob(
            P_valid_best
        )
    )

    NESTED_PRED[
        "BestSubsetLogit"
    ][valid_idx] = (
        mean_logit(
            P_valid_best
        )
    )


    # --------------------------------------------------------
    # D. NLL-optimized weights
    # --------------------------------------------------------

    w = fit_nested_nll_weights(
        P_train_best,
        y_train
    )

    NESTED_PRED[
        "BestSubsetNLLWeighted"
    ][valid_idx] = (
        weighted_prob(
            P_valid_best,
            w
        )
    )


    # --------------------------------------------------------
    # E. Fixed Top-4
    # --------------------------------------------------------

    P_valid_top4 = (
        np.column_stack([
            P_valid_dict[m]
            for m in TOP4
        ])
    )

    NESTED_PRED[
        "Top4Mean"
    ][valid_idx] = (
        mean_prob(
            P_valid_top4
        )
    )


    # --------------------------------------------------------
    # F. All-7
    # --------------------------------------------------------

    P_valid_all = (
        np.column_stack([
            P_valid_dict[m]
            for m in
            NESTED_FUSION_MODELS
        ])
    )

    NESTED_PRED[
        "All7Mean"
    ][valid_idx] = (
        mean_prob(
            P_valid_all
        )
    )


    # --------------------------------------------------------
    # Logs
    # --------------------------------------------------------

    selection_rows.append({

        "Outer_Fold":
            outer_fold,

        "Best_Single":
            best_single,

        "Best_Single_Train_PR":
            single_scores[
                best_single
            ],

        "Selected_Subset":
            " | ".join(
                best_subset
            ),

        "N_Experts":
            len(
                best_subset
            ),

        "Subset_Train_PR":
            best_subset_pr,
    })


    for model_name, weight in zip(
        best_subset,
        w
    ):

        weight_rows.append({

            "Outer_Fold":
                outer_fold,

            "Model":
                model_name,

            "NLL_Weight":
                float(weight),
        })


    print(
        "Best single:",
        best_single,
        f"PR={single_scores[best_single]:.6f}"
    )

    print(
        "Best subset:",
        best_subset,
        f"PR={best_subset_pr:.6f}"
    )


# ============================================================
# 10R.7 Evaluate genuinely nested predictions
# ============================================================

nested_rows = []

for method in METHODS:

    p = (
        NESTED_PRED[
            method
        ]
    )

    assert np.isfinite(
        p
    ).all()

    assert not np.any(
        p == 0
    )

    threshold, _, _ = (
        select_best_threshold(
            y_all,
            p,
            objective="F1"
        )
    )

    metrics = (
        evaluate_binary_predictions(
            y_all,
            p,
            threshold=threshold
        )
    )

    nested_rows.append({

        "Method":
            method,

        **metrics,
    })


strict_nested_results = (
    pd.DataFrame(
        nested_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


best_standalone_pr = (
    complete_leaderboard[
        "PR_AUC"
    ].max()
)

strict_nested_results[
    "Delta_PR_vs_BestStandalone"
] = (
    strict_nested_results[
        "PR_AUC"
    ]
    - best_standalone_pr
)


print(
    "\n"
    + "=" * 115
)

print(
    "STRICTLY NESTED SIMPLE-ENSEMBLE CEILING"
)

print(
    "=" * 115
)

display(
    strict_nested_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_BestStandalone",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 10R.8 Selection stability
# ============================================================

strict_selection_table = (
    pd.DataFrame(
        selection_rows
    )
)

strict_weight_table = (
    pd.DataFrame(
        weight_rows
    )
)

freq_rows = []

for model_name in (
    NESTED_FUSION_MODELS
):

    selected_count = 0

    for subset_string in (
        strict_selection_table[
            "Selected_Subset"
        ]
    ):

        members = (
            subset_string.split(
                " | "
            )
        )

        if model_name in members:
            selected_count += 1

    freq_rows.append({

        "Model":
            model_name,

        "Selected_Folds":
            selected_count,

        "Selection_Rate":
            selected_count
            / N_SPLITS,
    })


strict_frequency = (
    pd.DataFrame(
        freq_rows
    )
    .sort_values(
        [
            "Selected_Folds",
            "Model",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nSTRICT META-FOLD SELECTIONS"
)

display(
    strict_selection_table
)

print(
    "\nSTRICT EXPERT FREQUENCY"
)

display(
    strict_frequency
)


# ============================================================
# 10R.9 Save
# ============================================================

save_csv(
    strict_nested_results,
    TAB_DIR
    / "module10_strict_nested_results.csv"
)

save_csv(
    strict_selection_table,
    TAB_DIR
    / "module10_strict_nested_selections.csv"
)

save_csv(
    strict_frequency,
    TAB_DIR
    / "module10_strict_nested_frequency.csv"
)

save_csv(
    strict_weight_table,
    TAB_DIR
    / "module10_strict_nested_weights.csv"
)


strict_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,
})

for method in METHODS:

    strict_pred_df[
        method
    ] = (
        NESTED_PRED[
            method
        ]
    )

save_csv(
    strict_pred_df,
    PRED_DIR
    / "module10_strict_nested_oof_predictions.csv"
)

print(
    "\nMODULE 10R COMPLETE."
)

Nested subsets: 91

STRICT META FOLD 1/5
Best single: CatBoost PR=0.377899
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.380605

STRICT META FOLD 2/5
Best single: CatBoost PR=0.377255
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.379921

STRICT META FOLD 3/5
Best single: AutoInt PR=0.378440
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.380949

STRICT META FOLD 4/5
Best single: MLP_BCE PR=0.380001
Best subset: ('CatBoost', 'HistGradientBoosting', 'AutoInt', 'DeepFM_v2') PR=0.382485

STRICT META FOLD 5/5
Best single: HistGradientBoosting PR=0.375007
Best subset: ('HistGradientBoosting', 'AutoInt', 'MLP_BCE', 'DeepFM_v2') PR=0.378309

STRICTLY NESTED SIMPLE-ENSEMBLE CEILING


,Method,PR_AUC,Delta_PR_vs_BestStandalone,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,BestSubsetMean,0.380129,0.002593,0.850561,0.069716,0.235601,0.002452,0.350761,0.525793,0.420801,0.356163,0.220
1,BestSubsetLogit,0.380095,0.002559,0.850525,0.069718,0.235611,0.002159,0.352149,0.523857,0.421174,0.356502,0.220
2,All7Mean,0.380079,0.002543,0.850832,0.069716,0.235516,0.002914,0.347531,0.536152,0.421711,0.357633,0.220
3,Top4Mean,0.380052,0.002517,0.850778,0.069699,0.235495,0.002115,0.346361,0.538767,0.421652,0.357702,0.215
4,BestSubsetNLLWeighted,0.380004,0.002469,0.850581,0.069718,0.235595,0.002150,0.352067,0.524380,0.421285,0.356644,0.220
5,BestSingle,0.375170,-0.002366,0.849148,0.069986,0.236563,0.003737,0.336759,0.557131,0.419781,0.356802,0.215



STRICT META-FOLD SELECTIONS


,Outer_Fold,Best_Single,Best_Single_Train_PR,Selected_Subset,N_Experts,Subset_Train_PR
0,0,CatBoost,0.377899,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.380605
1,1,CatBoost,0.377255,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.379921
2,2,AutoInt,0.378440,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.380949
3,3,MLP_BCE,0.380001,CatBoost | HistGradientBoosting | AutoInt | De...,4,0.382485
4,4,HistGradientBoosting,0.375007,HistGradientBoosting | AutoInt | MLP_BCE | Dee...,4,0.378309



STRICT EXPERT FREQUENCY


,Model,Selected_Folds,Selection_Rate
0,AutoInt,5,1.0
1,DeepFM_v2,5,1.0
2,HistGradientBoosting,5,1.0
3,CatBoost,4,0.8
4,MLP_BCE,1,0.2
5,MLP_Focal,0,0.0
6,MLP_Focal_FGSM,0,0.0



MODULE 10R COMPLETE.


In [15]:
# ============================================================
# MODULE 11 — CONDITIONAL RESIDUAL / STACKING CEILING
# ============================================================
#
# QUESTION:
# Can conditional correction outperform the static
# simple-ensemble ceiling of ~0.38015 PR-AUC?
#
# Models:
# 1. Logistic meta-stacker
# 2. HistGB meta-stacker
#
# Inputs:
# - calibrated expert probabilities
# - consensus prediction
# - disagreement
# - uncertainty
# - expert deviations from consensus
#
# NO original clinical features yet.
# NO HOLDOUT.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 11.1 Core + specialist models
# ------------------------------------------------------------

CORE_MODELS = [
    "HistGradientBoosting",
    "AutoInt",
    "DeepFM_v2",
    "CatBoost",
]

SPECIALIST_MODELS = [
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
]

META_EXPERTS = (
    CORE_MODELS
    + SPECIALIST_MODELS
)

print("=" * 100)
print("MODULE 11 — CONDITIONAL RESIDUAL MODELS")
print("=" * 100)

print("Core experts:")
for m in CORE_MODELS:
    print(" -", m)

print("\nSpecialists:")
for m in SPECIALIST_MODELS:
    print(" -", m)


# ------------------------------------------------------------
# 11.2 Binary entropy
# ------------------------------------------------------------

def binary_entropy(p):

    p = np.clip(
        p,
        EPS,
        1.0 - EPS
    )

    return -(
        p * np.log(p)
        +
        (1.0 - p)
        * np.log(1.0 - p)
    )


# ------------------------------------------------------------
# 11.3 Build meta-features
# ------------------------------------------------------------

def build_meta_features(
    calibrated_dict
):
    """
    calibrated_dict:
        model_name -> probability vector

    Returns:
        DataFrame of prediction/disagreement features
    """

    df = pd.DataFrame()

    # --------------------------------------------------------
    # Individual calibrated predictions
    # --------------------------------------------------------

    for m in META_EXPERTS:

        df[f"p_{m}"] = (
            calibrated_dict[m]
        )

    # --------------------------------------------------------
    # Core consensus
    # --------------------------------------------------------

    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in CORE_MODELS
    ])

    all_matrix = np.column_stack([
        calibrated_dict[m]
        for m in META_EXPERTS
    ])

    core_mean = (
        core_matrix.mean(axis=1)
    )

    core_std = (
        core_matrix.std(axis=1)
    )

    core_min = (
        core_matrix.min(axis=1)
    )

    core_max = (
        core_matrix.max(axis=1)
    )

    core_range = (
        core_max - core_min
    )

    # --------------------------------------------------------
    # Main consensus features
    # --------------------------------------------------------

    df["core_mean"] = core_mean

    df["core_logit"] = safe_logit(
        core_mean
    )

    df["core_std"] = core_std

    df["core_range"] = core_range

    df["core_min"] = core_min

    df["core_max"] = core_max

    df["core_entropy"] = (
        binary_entropy(
            core_mean
        )
    )

    # --------------------------------------------------------
    # Wider disagreement
    # --------------------------------------------------------

    df["all_mean"] = (
        all_matrix.mean(axis=1)
    )

    df["all_std"] = (
        all_matrix.std(axis=1)
    )

    df["all_range"] = (
        all_matrix.max(axis=1)
        - all_matrix.min(axis=1)
    )

    # --------------------------------------------------------
    # Expert deviation from core consensus
    # --------------------------------------------------------

    for m in META_EXPERTS:

        p = calibrated_dict[m]

        df[
            f"delta_{m}"
        ] = (
            p - core_mean
        )

        df[
            f"abs_delta_{m}"
        ] = np.abs(
            p - core_mean
        )

    return df


# ------------------------------------------------------------
# 11.4 Strict outer-fold feature generation
# ------------------------------------------------------------

def build_nested_meta_data(
    train_idx,
    valid_idx
):
    """
    Calibration is fitted ONLY on outer-training data.
    """

    train_cal = {}
    valid_cal = {}

    for model_name in META_EXPERTS:

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va

    X_train_meta = (
        build_meta_features(
            train_cal
        )
    )

    X_valid_meta = (
        build_meta_features(
            valid_cal
        )
    )

    return (
        X_train_meta,
        X_valid_meta,
        train_cal,
        valid_cal,
    )


# ------------------------------------------------------------
# 11.5 Prediction buffers
# ------------------------------------------------------------

METHODS_11 = [
    "Core4Mean",
    "LogisticMeta",
    "HistGBMeta",
]

META11_PRED = {

    method:
        np.zeros(
            len(y_all),
            dtype=float
        )

    for method in METHODS_11
}

fold_rows = []


# ============================================================
# 11.6 Strict meta-CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"OUTER META FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 90
    )

    train_idx = np.where(
        fold_assignment != outer_fold
    )[0]

    valid_idx = np.where(
        fold_assignment == outer_fold
    )[0]

    y_train = (
        y_all[
            train_idx
        ]
    )

    y_valid = (
        y_all[
            valid_idx
        ]
    )

    (
        X_train_meta,
        X_valid_meta,
        train_cal,
        valid_cal,
    ) = build_nested_meta_data(
        train_idx,
        valid_idx
    )


    # ========================================================
    # A. Fixed Core-4 consensus
    # ========================================================

    p_core_valid = np.mean(
        np.column_stack([
            valid_cal[m]
            for m in CORE_MODELS
        ]),
        axis=1
    )

    META11_PRED[
        "Core4Mean"
    ][valid_idx] = (
        p_core_valid
    )


    # ========================================================
    # B. Logistic meta model
    # ========================================================

    scaler = StandardScaler()

    Xtr_lr = scaler.fit_transform(
        X_train_meta
    )

    Xva_lr = scaler.transform(
        X_valid_meta
    )

    logistic_meta = (
        LogisticRegression(
            penalty="l2",
            C=0.05,
            solver="lbfgs",
            max_iter=2000,
            random_state=SEED,
        )
    )

    logistic_meta.fit(
        Xtr_lr,
        y_train
    )

    p_lr = (
        logistic_meta.predict_proba(
            Xva_lr
        )[:, 1]
    )

    META11_PRED[
        "LogisticMeta"
    ][valid_idx] = clip_probs(
        p_lr
    )


    # ========================================================
    # C. Nonlinear residual meta model
    # ========================================================

    hist_meta = (
        HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_depth=3,
            max_iter=180,
            min_samples_leaf=100,
            l2_regularization=5.0,
            random_state=
                SEED + outer_fold,
        )
    )

    hist_meta.fit(
        X_train_meta,
        y_train
    )

    p_hist = (
        hist_meta.predict_proba(
            X_valid_meta
        )[:, 1]
    )

    META11_PRED[
        "HistGBMeta"
    ][valid_idx] = clip_probs(
        p_hist
    )


    # ========================================================
    # Fold diagnostics
    # ========================================================

    for method in METHODS_11:

        p_fold = (
            META11_PRED[
                method
            ][valid_idx]
        )

        m = probability_metrics(
            y_valid,
            p_fold
        )

        fold_rows.append({

            "Fold":
                outer_fold,

            "Method":
                method,

            "PR_AUC":
                m["PR_AUC"],

            "ROC_AUC":
                m["ROC_AUC"],

            "Brier":
                m["Brier"],

            "NLL":
                m["NLL"],

            "ECE":
                m["ECE"],
        })

    print(
        "Core4 PR :",
        f"{average_precision_score(y_valid, p_core_valid):.6f}"
    )

    print(
        "LogMeta PR:",
        f"{average_precision_score(y_valid, p_lr):.6f}"
    )

    print(
        "HistMeta PR:",
        f"{average_precision_score(y_valid, p_hist):.6f}"
    )


# ============================================================
# 11.7 Global cross-fitted evaluation
# ============================================================

result_rows = []

for method in METHODS_11:

    p = META11_PRED[
        method
    ]

    assert np.isfinite(
        p
    ).all()

    threshold, _, _ = (
        select_best_threshold(
            y_all,
            p,
            objective="F1"
        )
    )

    metrics = (
        evaluate_binary_predictions(
            y_all,
            p,
            threshold=threshold
        )
    )

    result_rows.append({

        "Method":
            method,

        **metrics,
    })


module11_results = (
    pd.DataFrame(
        result_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


SIMPLE_ENSEMBLE_CEILING = (
    0.380153
)

module11_results[
    "Delta_vs_Simple_Ceiling"
] = (
    module11_results[
        "PR_AUC"
    ]
    - SIMPLE_ENSEMBLE_CEILING
)


print(
    "\n"
    + "=" * 115
)

print(
    "MODULE 11 — CONDITIONAL MODEL RESULTS"
)

print(
    "=" * 115
)

display(
    module11_results[
        [
            "Method",
            "PR_AUC",
            "Delta_vs_Simple_Ceiling",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 11.8 Fold stability
# ============================================================

module11_fold_table = (
    pd.DataFrame(
        fold_rows
    )
)

module11_stability = (
    module11_fold_table
    .groupby(
        "Method"
    )
    .agg({

        "PR_AUC":
            ["mean", "std"],

        "ROC_AUC":
            ["mean", "std"],

        "Brier":
            ["mean", "std"],

        "ECE":
            ["mean", "std"],
    })
)

module11_stability.columns = [
    "_".join(c)
    for c in
    module11_stability.columns
]

module11_stability = (
    module11_stability
    .reset_index()
    .sort_values(
        "PR_AUC_mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "MODULE 11 — FOLD STABILITY"
)

print(
    "=" * 115
)

display(
    module11_stability
)


# ============================================================
# 11.9 Save
# ============================================================

save_csv(
    module11_results,
    TAB_DIR
    / "module11_conditional_meta_results.csv"
)

save_csv(
    module11_fold_table,
    TAB_DIR
    / "module11_conditional_meta_fold_metrics.csv"
)

save_csv(
    module11_stability,
    TAB_DIR
    / "module11_conditional_meta_stability.csv"
)


module11_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,
})

for method in METHODS_11:

    module11_pred_df[
        method
    ] = META11_PRED[
        method
    ]

save_csv(
    module11_pred_df,
    PRED_DIR
    / "module11_conditional_meta_oof_predictions.csv"
)


module11_manifest = {

    "core_models":
        CORE_MODELS,

    "specialist_models":
        SPECIALIST_MODELS,

    "meta_features":
        list(
            X_train_meta.columns
        ),

    "models_tested":
        METHODS_11,

    "simple_ensemble_reference_PR":
        SIMPLE_ENSEMBLE_CEILING,

    "outer_cv":
        "strict_fixed_5_fold",

    "holdout_used":
        False,
}

save_json(
    module11_manifest,
    LOG_DIR
    / "module11_manifest.json"
)

print(
    "\nMODULE 11 COMPLETE."
)

MODULE 11 — CONDITIONAL RESIDUAL MODELS
Core experts:
 - HistGradientBoosting
 - AutoInt
 - DeepFM_v2
 - CatBoost

Specialists:
 - MLP_BCE
 - MLP_Focal
 - MLP_Focal_FGSM

OUTER META FOLD 1/5
Core4 PR : 0.380069
LogMeta PR: 0.379655
HistMeta PR: 0.377976

OUTER META FOLD 2/5
Core4 PR : 0.382772
LogMeta PR: 0.382675
HistMeta PR: 0.381468

OUTER META FOLD 3/5
Core4 PR : 0.378349
LogMeta PR: 0.379219
HistMeta PR: 0.378582

OUTER META FOLD 4/5
Core4 PR : 0.372832
LogMeta PR: 0.372141
HistMeta PR: 0.371287

OUTER META FOLD 5/5
Core4 PR : 0.390070
LogMeta PR: 0.388350
HistMeta PR: 0.386620

MODULE 11 — CONDITIONAL MODEL RESULTS


,Method,PR_AUC,Delta_vs_Simple_Ceiling,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380303,0.000150,0.850539,0.069709,0.235595,0.002299,0.355461,0.514911,0.420580,0.355552,0.225
1,LogisticMeta,0.379501,-0.000652,0.850656,0.069723,0.235577,0.001991,0.342919,0.545883,0.421227,0.357638,0.215
2,HistGBMeta,0.378611,-0.001542,0.850374,0.069741,0.235682,0.001376,0.336465,0.560218,0.420424,0.357723,0.205



MODULE 11 — FOLD STABILITY


,Method,PR_AUC_mean,PR_AUC_std,ROC_AUC_mean,ROC_AUC_std,Brier_mean,Brier_std,ECE_mean,ECE_std
0,Core4Mean,0.380818,0.006322,0.850642,0.001633,0.069709,0.000286,0.004122,0.001178
2,LogisticMeta,0.380408,0.005885,0.850752,0.001524,0.069723,0.000286,0.003879,0.000573
1,HistGBMeta,0.379187,0.005583,0.850583,0.001539,0.069741,0.000248,0.004170,0.000608



MODULE 11 COMPLETE.


In [16]:
# ============================================================
# MODULE 12A — RA-MoE
# Routing-Aware Conditional Reliability MoE
# ============================================================
#
# Main contribution:
#
#      HGB --------\
#      CatBoost ----\
#      AutoInt ------> Reliability-supervised MoE Gate
#      DeepFM -------/               |
#                                    v
#                             weighted MoE risk
#                                    |
#                     -------------------------------
#                     |             |               |
#                 prediction   expert state     routing state
#                     |             |               |
#                     --------------------------------
#                                    |
#                                    v
#                      Conditional Reliability Head
#                                    |
#                                    v
#                         P(MoE error | patient)
#                                    |
#                              Trust = 1 - risk
#
#
# STRICT OUTER-CV PROTOCOL
# ------------------------------------------------------------
# For every outer fold:
#
#   OUTER TRAIN
#       |
#       +-- gate-fit subset
#       |       -> train routing gate
#       |
#       +-- threshold subset
#       |       -> choose disease decision threshold
#       |
#       +-- reliability-fit subset
#               -> train error-risk estimator
#
#   OUTER VALID
#       -> completely untouched
#
# HOLDOUT IS NEVER USED.
#
#
# Reliability benchmarks:
#
#   R1 Confidence:
#      MoE probability / entropy / decision margin
#
#   R2 Expert uncertainty:
#      R1 + expert disagreement information
#
#   R3 RA-MoE:
#      R2 + routing entropy / concentration /
#      weighted disagreement / dominant-expert behavior
#
#
# ============================================================


import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
)


# ============================================================
# 12A.1 EXPERTS / FEATURES
# ============================================================

RAMOE_EXPERTS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]


RAMOE_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


# ============================================================
# 12A.2 CONFIG
# ============================================================

RAMOE_CONFIG = {

    # --------------------------------------------------------
    # outer-training partition
    # --------------------------------------------------------

    # 70% initially allocated to gate-development pool.
    # This pool is subsequently split again into:
    # gate fit + threshold selection.
    "reliability_fraction":
        0.30,

    "threshold_fraction_within_gate_pool":
        0.15,


    # --------------------------------------------------------
    # reliability-supervised gate
    # --------------------------------------------------------

    "gate_hidden":
        32,

    "gate_dropout":
        0.10,

    "gate_epochs":
        4,

    "gate_batch_size":
        4096,

    "gate_lr":
        1e-3,

    "gate_weight_decay":
        1e-4,

    # soft oracle expert target
    "oracle_temperature":
        0.05,

    # small regularizer preventing pathological collapse
    "gate_uniform_reg":
        0.01,


    # --------------------------------------------------------
    # reliability head
    # --------------------------------------------------------

    "reliability_C":
        0.10,


    # --------------------------------------------------------
    # deferral evaluations
    # --------------------------------------------------------

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],
}


RAMOE_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


print(
    "RA-MoE device:",
    RAMOE_DEVICE
)


# ============================================================
# 12A.3 REPRODUCIBILITY
# ============================================================

def ramoe_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 12A.4 BASIC HELPERS
# ============================================================

def safe_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


def binary_entropy(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-8,
        1.0 - 1e-8
    )

    return (
        -p * np.log(p)
        -
        (1.0 - p)
        * np.log(1.0 - p)
    )


def normalized_binary_entropy(p):

    return (
        binary_entropy(p)
        /
        np.log(2.0)
    )


def routing_entropy(weights):

    w = np.clip(
        np.asarray(
            weights,
            dtype=float
        ),
        1e-12,
        1.0
    )

    ent = -np.sum(
        w * np.log(w),
        axis=1
    )

    return (
        ent
        /
        np.log(
            w.shape[1]
        )
    )


# ============================================================
# 12A.5 SOFT ORACLE EXPERT TARGETS
# ============================================================
#
# Training target for the MoE gate:
#
# For each sample and expert:
#
#       loss_j = binary log-loss of expert j
#
# convert losses into soft reliability targets:
#
#       q_j ∝ exp(-loss_j / T)
#
#
# This does NOT directly use the validation labels.
# q is created only for gate-training cases.
#
# ============================================================

def make_oracle_targets(
    expert_prob,
    y,
    temperature
):

    p = np.clip(
        np.asarray(
            expert_prob,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    y = np.asarray(
        y,
        dtype=float
    )[:, None]


    losses = (

        -y
        * np.log(p)

        -

        (1.0 - y)
        * np.log(
            1.0 - p
        )
    )


    logits = (

        -losses
        /
        temperature
    )


    logits = (

        logits
        -
        logits.max(
            axis=1,
            keepdims=True
        )
    )


    weights = np.exp(
        logits
    )


    weights /= (
        weights.sum(
            axis=1,
            keepdims=True
        )
        +
        1e-12
    )


    return weights.astype(
        np.float32
    )


# ============================================================
# 12A.6 GATE DATASET
# ============================================================

class RAMoEGateDataset(Dataset):

    def __init__(
        self,
        X,
        target_weights
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.target_weights = torch.tensor(
            target_weights,
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.X
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.target_weights[idx],
        )


# ============================================================
# 12A.7 ROUTING GATE
# ============================================================

class RAMoEGate(nn.Module):

    def __init__(
        self,
        input_dim,
        n_experts
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                RAMOE_CONFIG[
                    "gate_hidden"
                ]
            ),

            nn.GELU(),

            nn.LayerNorm(
                RAMOE_CONFIG[
                    "gate_hidden"
                ]
            ),

            nn.Dropout(
                RAMOE_CONFIG[
                    "gate_dropout"
                ]
            ),

            nn.Linear(
                RAMOE_CONFIG[
                    "gate_hidden"
                ],
                n_experts
            ),
        )


    def forward(
        self,
        x
    ):

        logits = self.network(
            x
        )

        weights = F.softmax(
            logits,
            dim=1
        )

        return (
            logits,
            weights
        )


# ============================================================
# 12A.8 TRAIN GATE
# ============================================================

def train_ramoe_gate(
    X,
    target_weights,
    seed
):

    ramoe_seed(
        seed
    )


    dataset = RAMoEGateDataset(

        X,
        target_weights
    )


    loader = DataLoader(

        dataset,

        batch_size=
            RAMOE_CONFIG[
                "gate_batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = RAMoEGate(

        input_dim=
            X.shape[1],

        n_experts=
            len(
                RAMOE_EXPERTS
            )

    ).to(
        RAMOE_DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            RAMOE_CONFIG[
                "gate_lr"
            ],

        weight_decay=
            RAMOE_CONFIG[
                "gate_weight_decay"
            ]
    )


    uniform_target = (
        1.0
        /
        len(
            RAMOE_EXPERTS
        )
    )


    for epoch in range(
        RAMOE_CONFIG[
            "gate_epochs"
        ]
    ):

        model.train()

        running = 0.0

        seen = 0


        for xb, target_w in loader:

            xb = xb.to(
                RAMOE_DEVICE
            )

            target_w = target_w.to(
                RAMOE_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, pred_w = model(
                xb
            )


            # ------------------------------------------------
            # soft-target cross entropy
            # ------------------------------------------------

            log_prob = F.log_softmax(
                logits,
                dim=1
            )


            routing_loss = (

                -(
                    target_w
                    *
                    log_prob
                ).sum(
                    dim=1
                )
            ).mean()


            # ------------------------------------------------
            # weak anti-collapse regularization
            # ------------------------------------------------

            mean_w = pred_w.mean(
                dim=0
            )


            uniform_loss = (

                (
                    mean_w
                    -
                    uniform_target
                )
                ** 2

            ).mean()


            loss = (

                routing_loss

                +

                RAMOE_CONFIG[
                    "gate_uniform_reg"
                ]
                *
                uniform_loss
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


            n = len(
                xb
            )

            running += (
                float(
                    loss.detach()
                    .cpu()
                )
                *
                n
            )

            seen += n


        print(

            f"    gate epoch "
            f"{epoch+1}/"
            f"{RAMOE_CONFIG['gate_epochs']} "

            f"loss="
            f"{running/seen:.6f}"
        )


    return model


# ============================================================
# 12A.9 GATE PREDICTION
# ============================================================

def predict_gate_weights(
    model,
    X,
    batch_size=8192
):

    model.eval()

    outputs = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X)
            )


            xb = torch.tensor(

                X[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    RAMOE_DEVICE
            )


            _, weights = model(
                xb
            )


            outputs.append(

                weights.cpu()
                .numpy()
            )


    return np.concatenate(
        outputs,
        axis=0
    )


# ============================================================
# 12A.10 BUILD MoE PROBABILITY
# ============================================================

def mixture_probability(
    expert_prob,
    weights
):

    return np.sum(

        expert_prob
        *
        weights,

        axis=1
    )


# ============================================================
# 12A.11 THRESHOLD SELECTION
# ============================================================

def select_f1_threshold_local(
    y,
    p
):

    thresholds = np.arange(
        0.05,
        0.501,
        0.005
    )


    best_t = None
    best_f1 = -np.inf


    for t in thresholds:

        pred = (
            p >= t
        ).astype(int)


        score = f1_score(
            y,
            pred,
            zero_division=0
        )


        if score > best_f1:

            best_f1 = score
            best_t = float(
                t
            )


    return (
        best_t,
        best_f1
    )


# ============================================================
# 12A.12 RELIABILITY FEATURE CONSTRUCTION
# ============================================================

def build_reliability_features(
    moe_prob,
    expert_prob,
    gate_weights
):

    moe_prob = np.asarray(
        moe_prob
    )

    expert_prob = np.asarray(
        expert_prob
    )

    gate_weights = np.asarray(
        gate_weights
    )


    # --------------------------------------------------------
    # Prediction confidence
    # --------------------------------------------------------

    pred_entropy = (
        normalized_binary_entropy(
            moe_prob
        )
    )


    margin = np.abs(
        moe_prob
        -
        0.5
    )


    # --------------------------------------------------------
    # Expert state
    # --------------------------------------------------------

    expert_std = np.std(
        expert_prob,
        axis=1
    )


    expert_range = (

        expert_prob.max(
            axis=1
        )

        -
        expert_prob.min(
            axis=1
        )
    )


    expert_mean = np.mean(
        expert_prob,
        axis=1
    )


    expert_median = np.median(
        expert_prob,
        axis=1
    )


    consensus_gap = np.abs(
        moe_prob
        -
        expert_mean
    )


    # --------------------------------------------------------
    # Routing state
    # --------------------------------------------------------

    gate_entropy = (
        routing_entropy(
            gate_weights
        )
    )


    max_gate = np.max(
        gate_weights,
        axis=1
    )


    sorted_weights = np.sort(
        gate_weights,
        axis=1
    )


    routing_margin = (

        sorted_weights[
            :,
            -1
        ]

        -
        sorted_weights[
            :,
            -2
        ]
    )


    dominant_expert = np.argmax(
        gate_weights,
        axis=1
    )


    dominant_probability = (
        expert_prob[
            np.arange(
                len(expert_prob)
            ),
            dominant_expert
        ]
    )


    dominant_gap = np.abs(

        dominant_probability
        -
        moe_prob
    )


    weighted_variance = np.sum(

        gate_weights

        *

        (
            expert_prob
            -
            moe_prob[
                :,
                None
            ]
        )
        ** 2,

        axis=1
    )


    effective_experts = np.exp(

        -np.sum(

            np.clip(
                gate_weights,
                1e-12,
                1.0
            )

            *

            np.log(
                np.clip(
                    gate_weights,
                    1e-12,
                    1.0
                )
            ),

            axis=1
        )
    )


    # ========================================================
    # Three nested feature sets
    # ========================================================

    confidence = np.column_stack([

        moe_prob,
        pred_entropy,
        margin,
    ])


    expert = np.column_stack([

        moe_prob,
        pred_entropy,
        margin,

        expert_std,
        expert_range,
        expert_mean,
        expert_median,
        consensus_gap,
    ])


    routing = np.column_stack([

        moe_prob,
        pred_entropy,
        margin,

        expert_std,
        expert_range,
        expert_mean,
        expert_median,
        consensus_gap,

        gate_entropy,
        max_gate,
        routing_margin,
        dominant_gap,
        weighted_variance,
        effective_experts,

        # individual gate state
        gate_weights,

        # individual expert state
        expert_prob,
    ])


    diagnostic = {

        "pred_entropy":
            pred_entropy,

        "margin":
            margin,

        "expert_std":
            expert_std,

        "expert_range":
            expert_range,

        "gate_entropy":
            gate_entropy,

        "max_gate":
            max_gate,

        "routing_margin":
            routing_margin,

        "weighted_variance":
            weighted_variance,

        "effective_experts":
            effective_experts,
    }


    return (
        confidence,
        expert,
        routing,
        diagnostic
    )


# ============================================================
# 12A.13 RELIABILITY MODEL
# ============================================================
#
# Same model class for all feature groups.
# Therefore gains reflect better trust information,
# not merely a stronger downstream classifier.
#
# ============================================================

def make_reliability_model():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    RAMOE_CONFIG[
                        "reliability_C"
                    ],

                solver="lbfgs",

                max_iter=3000
            )
        )
    ])


# ============================================================
# 12A.14 ERROR-DETECTION METRICS
# ============================================================

def error_detection_metrics(
    error,
    risk_score
):

    error = np.asarray(
        error,
        dtype=int
    )

    risk_score = np.asarray(
        risk_score,
        dtype=float
    )


    if len(
        np.unique(
            error
        )
    ) < 2:

        return {
            "Error_PR_AUC":
                np.nan,

            "Error_ROC_AUC":
                np.nan,
        }


    return {

        "Error_PR_AUC":
            average_precision_score(
                error,
                risk_score
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error,
                risk_score
            ),
    }


# ============================================================
# 12A.15 DEFER LOWEST-TRUST CASES
# ============================================================

def deferral_metrics(
    y,
    pred,
    risk_score,
    rates
):

    y = np.asarray(
        y,
        dtype=int
    )

    pred = np.asarray(
        pred,
        dtype=int
    )

    risk_score = np.asarray(
        risk_score,
        dtype=float
    )


    error = (
        pred != y
    ).astype(int)


    false_negative = (
        (y == 1)
        &
        (pred == 0)
    )


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    total_fn = max(
        int(
            false_negative.sum()
        ),
        1
    )


    base_error_rate = (
        error.mean()
    )


    order = np.argsort(
        -risk_score
    )


    rows = []


    for rate in rates:

        n_defer = max(

            1,

            int(
                round(
                    len(y)
                    *
                    rate
                )
            )
        )


        defer_idx = order[
            :n_defer
        ]


        keep_mask = np.ones(
            len(y),
            dtype=bool
        )

        keep_mask[
            defer_idx
        ] = False


        deferred_error_rate = (
            error[
                defer_idx
            ].mean()
        )


        error_enrichment = (

            deferred_error_rate
            /
            max(
                base_error_rate,
                1e-12
            )
        )


        errors_captured = (

            error[
                defer_idx
            ].sum()

            /
            total_errors
        )


        fn_captured = (

            false_negative[
                defer_idx
            ].sum()

            /
            total_fn
        )


        kept_y = y[
            keep_mask
        ]

        kept_pred = pred[
            keep_mask
        ]


        rows.append({

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0
                -
                rate,

            "Deferred_N":
                n_defer,

            "Deferred_Error_Rate":
                deferred_error_rate,

            "Error_Enrichment":
                error_enrichment,

            "Errors_Captured":
                errors_captured,

            "FalseNegatives_Captured":
                fn_captured,

            "Retained_Accuracy":
                accuracy_score(
                    kept_y,
                    kept_pred
                ),

            "Retained_F1":
                f1_score(
                    kept_y,
                    kept_pred,
                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(
                    kept_y,
                    kept_pred
                ),
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 12A.16 AURC
# ============================================================
#
# Sort high-trust -> low-trust.
#
# risk at coverage c =
# error rate among retained cases.
#
# ============================================================

def compute_aurc(
    error,
    risk_score
):

    error = np.asarray(
        error,
        dtype=float
    )

    risk_score = np.asarray(
        risk_score,
        dtype=float
    )


    # low estimated error risk first
    order = np.argsort(
        risk_score
    )


    sorted_error = error[
        order
    ]


    cumulative_errors = np.cumsum(
        sorted_error
    )


    n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_errors
        /
        n
    )


    coverage = (
        n
        /
        len(error)
    )


    return np.trapz(
        selective_risk,
        coverage
    )


# ============================================================
# 12A.17 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_gate_raw = (

    X_dev[
        RAMOE_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 12A.18 OOF BUFFERS
# ============================================================

N = len(
    y_all
)


RAMOE_PROB_OOF = np.zeros(
    N,
    dtype=float
)


RAMOE_CORE4_OOF = np.zeros(
    N,
    dtype=float
)


RAMOE_THRESHOLD_OOF = np.zeros(
    N,
    dtype=float
)


RAMOE_GATE_OOF = np.zeros(

    (
        N,
        len(
            RAMOE_EXPERTS
        )
    ),

    dtype=float
)


RAMOE_ERROR_OOF = np.zeros(
    N,
    dtype=int
)


# Estimated P(error)
RAMOE_RISK_CONF_OOF = np.zeros(
    N,
    dtype=float
)


RAMOE_RISK_EXPERT_OOF = np.zeros(
    N,
    dtype=float
)


RAMOE_RISK_FULL_OOF = np.zeros(
    N,
    dtype=float
)


fold_rows = []

routing_rows = []

deferral_fold_rows = []


# ============================================================
# 12A.19 OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 125
    )

    print(
        f"RA-MoE OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 125
    )


    # ========================================================
    # A. OUTER INDICES
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # B. STRICT FOLD-LOCAL EXPERT CALIBRATION
    # ========================================================

    train_cal = {}

    valid_cal = {}


    for model_name in RAMOE_EXPERTS:

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    P_outer_train = np.column_stack([

        train_cal[
            m
        ]

        for m
        in RAMOE_EXPERTS
    ])


    P_outer_valid = np.column_stack([

        valid_cal[
            m
        ]

        for m
        in RAMOE_EXPERTS
    ])


    # Equal consensus comparison
    core4_valid = (
        P_outer_valid.mean(
            axis=1
        )
    )


    RAMOE_CORE4_OOF[
        outer_valid_idx
    ] = core4_valid


    # ========================================================
    # C. OUTER TRAIN:
    #
    # 70% gate-development pool
    # 30% reliability-fit
    # ========================================================

    split_reliability = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                RAMOE_CONFIG[
                    "reliability_fraction"
                ],

            random_state=
                SEED
                + 60000
                + outer_fold
        )
    )


    (
        gate_pool_local,
        reliability_local

    ) = next(

        split_reliability.split(

            X_gate_raw[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    gate_pool_global = (

        outer_train_idx[
            gate_pool_local
        ]
    )


    reliability_global = (

        outer_train_idx[
            reliability_local
        ]
    )


    # ========================================================
    # D. SPLIT GATE POOL:
    #
    # gate-fit
    # threshold-selection
    # ========================================================

    y_gate_pool = (

        y_outer_train[
            gate_pool_local
        ]
    )


    split_threshold = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                RAMOE_CONFIG[
                    "threshold_fraction_within_gate_pool"
                ],

            random_state=
                SEED
                + 61000
                + outer_fold
        )
    )


    (
        gate_fit_inside,
        threshold_inside

    ) = next(

        split_threshold.split(

            X_gate_raw[
                gate_pool_global
            ],

            y_gate_pool
        )
    )


    gate_fit_global = (

        gate_pool_global[
            gate_fit_inside
        ]
    )


    threshold_global = (

        gate_pool_global[
            threshold_inside
        ]
    )


    # Need local positions relative to outer_train
    outer_position = {

        g:
            i

        for i, g
        in enumerate(
            outer_train_idx
        )
    }


    gate_fit_local = np.array([

        outer_position[g]

        for g in gate_fit_global

    ])


    threshold_local = np.array([

        outer_position[g]

        for g in threshold_global

    ])


    # reliability_local already indexes outer_train


    print(
        "\nOuter-training partitions:"
    )

    print(
        "  Gate fit       :",
        len(
            gate_fit_global
        )
    )

    print(
        "  Threshold fit  :",
        len(
            threshold_global
        )
    )

    print(
        "  Reliability fit:",
        len(
            reliability_global
        )
    )

    print(
        "  Outer valid    :",
        len(
            outer_valid_idx
        )
    )


    # ========================================================
    # E. GATE PREPROCESSING
    # ========================================================

    gate_scaler = StandardScaler()


    X_gate_fit = (

        gate_scaler.fit_transform(

            X_gate_raw[
                gate_fit_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_threshold = (

        gate_scaler.transform(

            X_gate_raw[
                threshold_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_reliability = (

        gate_scaler.transform(

            X_gate_raw[
                reliability_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_valid = (

        gate_scaler.transform(

            X_gate_raw[
                outer_valid_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # F. ORACLE RELIABILITY TARGETS FOR GATE
    # ========================================================

    P_gate_fit = (
        P_outer_train[
            gate_fit_local
        ]
    )


    y_gate_fit = (
        y_all[
            gate_fit_global
        ]
    )


    oracle_target = (
        make_oracle_targets(

            P_gate_fit,

            y_gate_fit,

            temperature=
                RAMOE_CONFIG[
                    "oracle_temperature"
                ]
        )
    )


    # ========================================================
    # G. TRAIN ROUTING GATE
    # ========================================================

    print(
        "\nTraining reliability-supervised MoE gate..."
    )


    gate_model = (
        train_ramoe_gate(

            X_gate_fit,

            oracle_target,

            seed=
                SEED
                + 62000
                + outer_fold
        )
    )


    # ========================================================
    # H. GATE WEIGHTS
    # ========================================================

    W_gate_fit = (
        predict_gate_weights(

            gate_model,

            X_gate_fit
        )
    )


    W_threshold = (
        predict_gate_weights(

            gate_model,

            X_threshold
        )
    )


    W_reliability = (
        predict_gate_weights(

            gate_model,

            X_reliability
        )
    )


    W_valid = (
        predict_gate_weights(

            gate_model,

            X_valid
        )
    )


    RAMOE_GATE_OOF[
        outer_valid_idx
    ] = W_valid


    # ========================================================
    # I. MoE PROBABILITIES
    # ========================================================

    P_threshold = (
        P_outer_train[
            threshold_local
        ]
    )


    P_reliability = (
        P_outer_train[
            reliability_local
        ]
    )


    moe_threshold_prob = (
        mixture_probability(

            P_threshold,

            W_threshold
        )
    )


    moe_reliability_prob = (
        mixture_probability(

            P_reliability,

            W_reliability
        )
    )


    moe_valid_prob = (
        mixture_probability(

            P_outer_valid,

            W_valid
        )
    )


    RAMOE_PROB_OOF[
        outer_valid_idx
    ] = moe_valid_prob


    # ========================================================
    # J. SELECT DECISION THRESHOLD
    #
    # threshold subset only
    # ========================================================

    y_threshold = (
        y_all[
            threshold_global
        ]
    )


    decision_threshold, threshold_f1 = (
        select_f1_threshold_local(

            y_threshold,

            moe_threshold_prob
        )
    )


    RAMOE_THRESHOLD_OOF[
        outer_valid_idx
    ] = decision_threshold


    print(
        "\nSelected disease threshold:",
        f"{decision_threshold:.3f}"
    )


    # ========================================================
    # K. DEFINE RELIABILITY TARGET
    #
    # e = 1 if MoE makes a classification error
    # ========================================================

    y_reliability = (
        y_all[
            reliability_global
        ]
    )


    pred_reliability = (

        moe_reliability_prob
        >=
        decision_threshold

    ).astype(int)


    error_reliability = (

        pred_reliability
        !=
        y_reliability

    ).astype(int)


    pred_valid = (

        moe_valid_prob
        >=
        decision_threshold

    ).astype(int)


    error_valid = (

        pred_valid
        !=
        y_outer_valid

    ).astype(int)


    RAMOE_ERROR_OOF[
        outer_valid_idx
    ] = error_valid


    print(
        "Reliability-fit error prevalence:",
        f"{error_reliability.mean():.4f}"
    )


    print(
        "Outer-valid error prevalence:",
        f"{error_valid.mean():.4f}"
    )


    # ========================================================
    # L. TRUST FEATURE SETS
    # ========================================================

    (
        F_conf_train,
        F_expert_train,
        F_full_train,
        diag_train

    ) = build_reliability_features(

        moe_reliability_prob,

        P_reliability,

        W_reliability
    )


    (
        F_conf_valid,
        F_expert_valid,
        F_full_valid,
        diag_valid

    ) = build_reliability_features(

        moe_valid_prob,

        P_outer_valid,

        W_valid
    )


    # ========================================================
    # M. TRAIN RELIABILITY BENCHMARKS
    # ========================================================

    confidence_model = (
        make_reliability_model()
    )


    expert_model = (
        make_reliability_model()
    )


    ramoe_model = (
        make_reliability_model()
    )


    confidence_model.fit(

        F_conf_train,

        error_reliability
    )


    expert_model.fit(

        F_expert_train,

        error_reliability
    )


    ramoe_model.fit(

        F_full_train,

        error_reliability
    )


    # ========================================================
    # N. PREDICT ERROR RISK
    # ========================================================

    risk_conf = (

        confidence_model
        .predict_proba(
            F_conf_valid
        )[
            :,
            1
        ]
    )


    risk_expert = (

        expert_model
        .predict_proba(
            F_expert_valid
        )[
            :,
            1
        ]
    )


    risk_full = (

        ramoe_model
        .predict_proba(
            F_full_valid
        )[
            :,
            1
        ]
    )


    RAMOE_RISK_CONF_OOF[
        outer_valid_idx
    ] = risk_conf


    RAMOE_RISK_EXPERT_OOF[
        outer_valid_idx
    ] = risk_expert


    RAMOE_RISK_FULL_OOF[
        outer_valid_idx
    ] = risk_full


    # ========================================================
    # O. ERROR-DETECTION PERFORMANCE
    # ========================================================

    conf_metrics = (
        error_detection_metrics(

            error_valid,

            risk_conf
        )
    )


    expert_metrics = (
        error_detection_metrics(

            error_valid,

            risk_expert
        )
    )


    full_metrics = (
        error_detection_metrics(

            error_valid,

            risk_full
        )
    )


    # ========================================================
    # P. DISEASE PERFORMANCE
    # ========================================================

    core_pr = (
        average_precision_score(

            y_outer_valid,

            core4_valid
        )
    )


    moe_pr = (
        average_precision_score(

            y_outer_valid,

            moe_valid_prob
        )
    )


    core_roc = (
        roc_auc_score(

            y_outer_valid,

            core4_valid
        )
    )


    moe_roc = (
        roc_auc_score(

            y_outer_valid,

            moe_valid_prob
        )
    )


    # ========================================================
    # Q. AURC
    # ========================================================

    aurc_conf = (
        compute_aurc(

            error_valid,

            risk_conf
        )
    )


    aurc_expert = (
        compute_aurc(

            error_valid,

            risk_expert
        )
    )


    aurc_full = (
        compute_aurc(

            error_valid,

            risk_full
        )
    )


    # ========================================================
    # R. ROUTING DIAGNOSTICS
    # ========================================================

    mean_weights = (
        W_valid.mean(
            axis=0
        )
    )


    mean_gate_entropy = (
        routing_entropy(
            W_valid
        ).mean()
    )


    routing_rows.append({

        "Fold":
            outer_fold,

        "GateEntropy":
            mean_gate_entropy,

        "EffectiveExperts":
            np.exp(
                mean_gate_entropy
                *
                np.log(
                    len(
                        RAMOE_EXPERTS
                    )
                )
            ),

        **{

            f"Weight_{RAMOE_EXPERTS[j]}":
                mean_weights[j]

            for j in range(
                len(
                    RAMOE_EXPERTS
                )
            )
        }
    })


    # ========================================================
    # S. FOLD SUMMARY
    # ========================================================

    fold_rows.append({

        "Fold":
            outer_fold,

        "Threshold":
            decision_threshold,

        "Core4_PR":
            core_pr,

        "RAMoE_PR":
            moe_pr,

        "Delta_PR":
            moe_pr
            -
            core_pr,

        "Core4_ROC":
            core_roc,

        "RAMoE_ROC":
            moe_roc,

        "Error_Prevalence":
            error_valid.mean(),

        "Confidence_Error_PR":
            conf_metrics[
                "Error_PR_AUC"
            ],

        "Expert_Error_PR":
            expert_metrics[
                "Error_PR_AUC"
            ],

        "RAMoE_Error_PR":
            full_metrics[
                "Error_PR_AUC"
            ],

        "Confidence_Error_ROC":
            conf_metrics[
                "Error_ROC_AUC"
            ],

        "Expert_Error_ROC":
            expert_metrics[
                "Error_ROC_AUC"
            ],

        "RAMoE_Error_ROC":
            full_metrics[
                "Error_ROC_AUC"
            ],

        "Confidence_AURC":
            aurc_conf,

        "Expert_AURC":
            aurc_expert,

        "RAMoE_AURC":
            aurc_full,

        "MeanGateEntropy":
            mean_gate_entropy,
    })


    # ========================================================
    # T. DEFERRAL ANALYSIS — FULL RA-MoE
    # ========================================================

    defer_df = (
        deferral_metrics(

            y_outer_valid,

            pred_valid,

            risk_full,

            RAMOE_CONFIG[
                "deferral_rates"
            ]
        )
    )


    defer_df[
        "Fold"
    ] = outer_fold


    deferral_fold_rows.append(
        defer_df
    )


    # ========================================================
    # U. PRINT
    # ========================================================

    print(
        "\nDisease prediction:"
    )


    print(
        f"  Core4 PR-AUC : "
        f"{core_pr:.6f}"
    )


    print(
        f"  RA-MoE PR    : "
        f"{moe_pr:.6f}"
    )


    print(
        f"  Delta        : "
        f"{moe_pr-core_pr:+.6f}"
    )


    print(
        "\nConditional error detection:"
    )


    print(
        f"  Confidence PR : "
        f"{conf_metrics['Error_PR_AUC']:.6f}"
    )


    print(
        f"  Expert PR     : "
        f"{expert_metrics['Error_PR_AUC']:.6f}"
    )


    print(
        f"  RA-MoE PR     : "
        f"{full_metrics['Error_PR_AUC']:.6f}"
    )


    print(
        "\nError-detection ROC:"
    )


    print(
        f"  Confidence ROC : "
        f"{conf_metrics['Error_ROC_AUC']:.6f}"
    )


    print(
        f"  Expert ROC     : "
        f"{expert_metrics['Error_ROC_AUC']:.6f}"
    )


    print(
        f"  RA-MoE ROC     : "
        f"{full_metrics['Error_ROC_AUC']:.6f}"
    )


    print(
        "\nAURC (lower is better):"
    )


    print(
        f"  Confidence : "
        f"{aurc_conf:.6f}"
    )


    print(
        f"  Expert     : "
        f"{aurc_expert:.6f}"
    )


    print(
        f"  RA-MoE     : "
        f"{aurc_full:.6f}"
    )


    print(
        "\nMean routing weights:"
    )


    for j, expert_name in enumerate(
        RAMOE_EXPERTS
    ):

        print(
            f"  {expert_name:<24}: "
            f"{mean_weights[j]:.4f}"
        )


    # ========================================================
    # V. CLEANUP
    # ========================================================

    del (
        gate_model,
        confidence_model,
        expert_model,
        ramoe_model,
    )

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 12A.20 GLOBAL DISEASE PERFORMANCE
# ============================================================

ramoe_global_pr = (
    average_precision_score(

        y_all,

        RAMOE_PROB_OOF
    )
)


core4_global_pr = (
    average_precision_score(

        y_all,

        RAMOE_CORE4_OOF
    )
)


ramoe_global_roc = (
    roc_auc_score(

        y_all,

        RAMOE_PROB_OOF
    )
)


core4_global_roc = (
    roc_auc_score(

        y_all,

        RAMOE_CORE4_OOF
    )
)


disease_results = pd.DataFrame([

    {
        "Model":
            "Core4Mean",

        "PR_AUC":
            core4_global_pr,

        "ROC_AUC":
            core4_global_roc,

        "Delta_PR":
            0.0,
    },

    {
        "Model":
            "RA_MoE",

        "PR_AUC":
            ramoe_global_pr,

        "ROC_AUC":
            ramoe_global_roc,

        "Delta_PR":
            ramoe_global_pr
            -
            core4_global_pr,
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 12A — DISEASE PREDICTION"
)

print(
    "=" * 130
)


display(
    disease_results
)


# ============================================================
# 12A.21 GLOBAL RELIABILITY BENCHMARK
# ============================================================

global_error = (
    RAMOE_ERROR_OOF
)


reliability_rows = []


for model_name, risk in [

    (
        "Confidence",
        RAMOE_RISK_CONF_OOF
    ),

    (
        "ExpertUncertainty",
        RAMOE_RISK_EXPERT_OOF
    ),

    (
        "RA_MoE",
        RAMOE_RISK_FULL_OOF
    ),
]:

    metrics = (
        error_detection_metrics(

            global_error,

            risk
        )
    )


    aurc = (
        compute_aurc(

            global_error,

            risk
        )
    )


    reliability_rows.append({

        "Reliability_Model":
            model_name,

        "Error_PR_AUC":
            metrics[
                "Error_PR_AUC"
            ],

        "Error_ROC_AUC":
            metrics[
                "Error_ROC_AUC"
            ],

        "AURC":
            aurc,
    })


reliability_results = pd.DataFrame(
    reliability_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 12A — CONDITIONAL RELIABILITY BENCHMARK"
)

print(
    "=" * 130
)


display(
    reliability_results
)


# ============================================================
# 12A.22 IMPROVEMENT OVER BASELINES
# ============================================================

confidence_row = (

    reliability_results
    .loc[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "Confidence"
    ]
    .iloc[0]
)


expert_row = (

    reliability_results
    .loc[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "ExpertUncertainty"
    ]
    .iloc[0]
)


ramoe_row = (

    reliability_results
    .loc[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "RA_MoE"
    ]
    .iloc[0]
)


reliability_gain = pd.DataFrame([

    {
        "Comparison":
            "RA-MoE vs Confidence",

        "Delta_Error_PR":
            ramoe_row[
                "Error_PR_AUC"
            ]
            -
            confidence_row[
                "Error_PR_AUC"
            ],

        "Delta_Error_ROC":
            ramoe_row[
                "Error_ROC_AUC"
            ]
            -
            confidence_row[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            ramoe_row[
                "AURC"
            ]
            -
            confidence_row[
                "AURC"
            ],
    },

    {
        "Comparison":
            "RA-MoE vs ExpertUncertainty",

        "Delta_Error_PR":
            ramoe_row[
                "Error_PR_AUC"
            ]
            -
            expert_row[
                "Error_PR_AUC"
            ],

        "Delta_Error_ROC":
            ramoe_row[
                "Error_ROC_AUC"
            ]
            -
            expert_row[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            ramoe_row[
                "AURC"
            ]
            -
            expert_row[
                "AURC"
            ],
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "ROUTING-AWARE RELIABILITY GAIN"
)

print(
    "=" * 130
)


display(
    reliability_gain
)


# ============================================================
# 12A.23 GLOBAL DEFERRAL ANALYSIS
# ============================================================

pred_global = (

    RAMOE_PROB_OOF
    >=
    RAMOE_THRESHOLD_OOF

).astype(int)


deferral_conf_global = (
    deferral_metrics(

        y_all,

        pred_global,

        RAMOE_RISK_CONF_OOF,

        RAMOE_CONFIG[
            "deferral_rates"
        ]
    )
)


deferral_conf_global[
    "Reliability_Model"
] = "Confidence"


deferral_expert_global = (
    deferral_metrics(

        y_all,

        pred_global,

        RAMOE_RISK_EXPERT_OOF,

        RAMOE_CONFIG[
            "deferral_rates"
        ]
    )
)


deferral_expert_global[
    "Reliability_Model"
] = "ExpertUncertainty"


deferral_ramoe_global = (
    deferral_metrics(

        y_all,

        pred_global,

        RAMOE_RISK_FULL_OOF,

        RAMOE_CONFIG[
            "deferral_rates"
        ]
    )
)


deferral_ramoe_global[
    "Reliability_Model"
] = "RA_MoE"


deferral_global = pd.concat(

    [
        deferral_conf_global,
        deferral_expert_global,
        deferral_ramoe_global,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 130
)

print(
    "GLOBAL SELECTIVE / DEFERRAL RESULTS"
)

print(
    "=" * 130
)


display(

    deferral_global[
        [
            "Reliability_Model",
            "Deferral_Rate",
            "Coverage",
            "Deferred_Error_Rate",
            "Error_Enrichment",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 12A.24 FOLD RESULTS
# ============================================================

fold_results = pd.DataFrame(
    fold_rows
)


routing_results = pd.DataFrame(
    routing_rows
)


deferral_fold_results = pd.concat(

    deferral_fold_rows,

    ignore_index=True
)


print(
    "\n"
    + "=" * 130
)

print(
    "RA-MoE FOLD RESULTS"
)

print(
    "=" * 130
)


display(
    fold_results
)


print(
    "\n"
    + "=" * 130
)

print(
    "ROUTING DIAGNOSTICS"
)

print(
    "=" * 130
)


display(
    routing_results
)


# ============================================================
# 12A.25 IMPORTANT SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 12A SUMMARY"
)

print(
    "=" * 130
)


print(
    "\nDisease PR:"
)

print(
    f"  Core4 : "
    f"{core4_global_pr:.6f}"
)

print(
    f"  RA-MoE: "
    f"{ramoe_global_pr:.6f}"
)

print(
    f"  Delta : "
    f"{ramoe_global_pr-core4_global_pr:+.6f}"
)


print(
    "\nError-detection PR-AUC:"
)


for _, row in reliability_results.iterrows():

    print(
        f"  "
        f"{row['Reliability_Model']:<20}: "
        f"{row['Error_PR_AUC']:.6f}"
    )


print(
    "\nError-detection ROC-AUC:"
)


for _, row in reliability_results.iterrows():

    print(
        f"  "
        f"{row['Reliability_Model']:<20}: "
        f"{row['Error_ROC_AUC']:.6f}"
    )


print(
    "\nAURC (lower is better):"
)


for _, row in reliability_results.iterrows():

    print(
        f"  "
        f"{row['Reliability_Model']:<20}: "
        f"{row['AURC']:.6f}"
    )


# ============================================================
# 12A.26 SAVE
# ============================================================

save_csv(

    disease_results,

    TAB_DIR
    / "module12a_ramoe_disease_results.csv"
)


save_csv(

    reliability_results,

    TAB_DIR
    / "module12a_ramoe_reliability_results.csv"
)


save_csv(

    reliability_gain,

    TAB_DIR
    / "module12a_ramoe_reliability_gain.csv"
)


save_csv(

    deferral_global,

    TAB_DIR
    / "module12a_ramoe_deferral_global.csv"
)


save_csv(

    fold_results,

    TAB_DIR
    / "module12a_ramoe_fold_results.csv"
)


save_csv(

    routing_results,

    TAB_DIR
    / "module12a_ramoe_routing_results.csv"
)


save_csv(

    deferral_fold_results,

    TAB_DIR
    / "module12a_ramoe_deferral_folds.csv"
)


ramoe_prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4":
        RAMOE_CORE4_OOF,

    "RA_MoE":
        RAMOE_PROB_OOF,

    "DecisionThreshold":
        RAMOE_THRESHOLD_OOF,

    "Prediction":
        pred_global,

    "Error":
        RAMOE_ERROR_OOF,

    "Confidence_ErrorRisk":
        RAMOE_RISK_CONF_OOF,

    "Expert_ErrorRisk":
        RAMOE_RISK_EXPERT_OOF,

    "RAMoE_ErrorRisk":
        RAMOE_RISK_FULL_OOF,

    "TrustScore":
        1.0
        -
        RAMOE_RISK_FULL_OOF,
})


for j, expert_name in enumerate(
    RAMOE_EXPERTS
):

    ramoe_prediction_df[
        f"GateWeight_{expert_name}"
    ] = (
        RAMOE_GATE_OOF[
            :,
            j
        ]
    )


save_csv(

    ramoe_prediction_df,

    PRED_DIR
    / "module12a_ramoe_oof.csv"
)


module12a_manifest = {

    "method":
        "Routing-Aware Conditional Reliability Mixture-of-Experts",

    "experts":
        RAMOE_EXPERTS,

    "gate_supervision":
        "soft oracle expert reliability from per-expert log loss",

    "reliability_target":
        "cross-fitted MoE classification error",

    "reliability_models": [
        "confidence",
        "expert uncertainty",
        "routing-aware RA-MoE",
    ],

    "deferral_rates":
        RAMOE_CONFIG[
            "deferral_rates"
        ],

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module12a_manifest,

    LOG_DIR
    / "module12a_manifest.json"
)


print(
    "\nMODULE 12A COMPLETE."
)

RA-MoE device: cuda

RA-MoE OUTER FOLD 1/5

Outer-training partitions:
  Gate fit       : 96600
  Threshold fit  : 17048
  Reliability fit: 48707
  Outer valid    : 40589

Training reliability-supervised MoE gate...
    gate epoch 1/4 loss=1.461497
    gate epoch 2/4 loss=1.408531
    gate epoch 3/4 loss=1.398000
    gate epoch 4/4 loss=1.393692

Selected disease threshold: 0.190
Reliability-fit error prevalence: 0.1565
Outer-valid error prevalence: 0.1547

Disease prediction:
  Core4 PR-AUC : 0.380069
  RA-MoE PR    : 0.379927
  Delta        : -0.000141

Conditional error detection:
  Confidence PR : 0.598835
  Expert PR     : 0.602326
  RA-MoE PR     : 0.602566

Error-detection ROC:
  Confidence ROC : 0.912353
  Expert ROC     : 0.912437
  RA-MoE ROC     : 0.912394

AURC (lower is better):
  Confidence : 0.028722
  Expert     : 0.028709
  RA-MoE     : 0.028711

Mean routing weights:
  HistGradientBoosting    : 0.2498
  CatBoost                : 0.2456
  AutoInt                 : 0.25

,Model,PR_AUC,ROC_AUC,Delta_PR
0,Core4Mean,0.380303,0.850539,0.000000
1,RA_MoE,0.380252,0.850527,-0.000051



MODULE 12A — CONDITIONAL RELIABILITY BENCHMARK


,Reliability_Model,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.577551,0.905571,0.027680
1,ExpertUncertainty,0.578517,0.905549,0.027682
2,RA_MoE,0.578399,0.905629,0.027653



ROUTING-AWARE RELIABILITY GAIN


...


RA-MoE FOLD RESULTS


,Fold,Threshold,Core4_PR,RAMoE_PR,Delta_PR,Core4_ROC,RAMoE_ROC,Error_Prevalence,Confidence_Error_PR,Expert_Error_PR,RAMoE_Error_PR,Confidence_Error_ROC,Expert_Error_ROC,RAMoE_Error_ROC,Confidence_AURC,Expert_AURC,RAMoE_AURC,MeanGateEntropy
0,0,0.190,0.380069,0.379927,-0.000141,0.852660,0.852639,0.154697,0.598835,0.602326,0.602566,0.912353,0.912437,0.912394,0.028722,0.028709,0.028711,0.997341
1,1,0.245,0.382772,0.382689,-0.000082,0.850766,0.850759,0.125108,0.514700,0.514832,0.522099,0.892952,0.892949,0.893065,0.024550,0.024551,0.024591,0.997795
2,2,0.195,0.378349,0.378173,-0.000176,0.848939,0.848897,0.151322,0.595208,0.593151,0.595052,0.909264,0.909078,0.909297,0.028712,0.028736,0.028696,0.996511
3,3,0.195,0.372832,0.372821,-0.000011,0.849083,0.849081,0.149006,0.578411,0.584448,0.583244,0.907097,0.907151,0.907155,0.028221,0.028214,0.028214,0.996691
4,4,0.215,0.390070,0.390083,0.000012,0.851760,0.851776,0.142136,0.560662,0.563865,0.566283,0.904109,0.904128,0.904241,0.027058,0.027058,0.027032,0.997417



ROUTING DIAGNOSTICS


,Fold,GateEntropy,EffectiveExperts,Weight_HistGradientBoosting,Weight_CatBoost,Weight_AutoInt,Weight_DeepFM_v2
0,0,0.997341,3.985281,0.249825,0.245648,0.253842,0.250686
1,1,0.997795,3.987794,0.249467,0.243584,0.253291,0.253658
2,2,0.996511,3.980697,0.249651,0.246953,0.248112,0.255283
3,3,0.996691,3.981693,0.248811,0.244857,0.251848,0.254483
4,4,0.997417,3.985703,0.249065,0.244847,0.251206,0.254883



MODULE 12A SUMMARY

Disease PR:
  Core4 : 0.380303
  RA-MoE: 0.380252
  Delta : -0.000051

Error-detection PR-AUC:
  Confidence          : 0.577551
  ExpertUncertainty   : 0.578517
  RA_MoE              : 0.578399

Error-detection ROC-AUC:
  Confidence          : 0.905571
  ExpertUncertainty   : 0.905549
  RA_MoE              : 0.905629

AURC (lower is better):
  Confidence          : 0.027680
  ExpertUncertainty   : 0.027682
  RA_MoE              : 0.027653

MODULE 12A COMPLETE.


In [17]:
# ============================================================
# MODULE 12B — DC-ETR-MoE
# Decision-Conditional Error-Type Reliability MoE
# ============================================================
#
# Core contribution:
#
#                        Existing MoE
#                            |
#                            v
#                  Disease decision y_hat
#                     /              \
#                    /                \
#              y_hat = 0           y_hat = 1
#                  |                    |
#                  v                    v
#             FN Safety Head       FP Safety Head
#                  |                    |
#                  v                    v
#            P(false negative)    P(false positive)
#                     \              /
#                      \            /
#                       v          v
#                    Cost-aware risk
#                          |
#                    Accept / Defer
#
#
# Important methodological idea:
#
# A false-negative model is trained ONLY on cases that the MoE
# predicts negative.
#
# A false-positive model is trained ONLY on cases that the MoE
# predicts positive.
#
# This avoids asking one generic uncertainty model to learn two
# fundamentally different error mechanisms.
#
#
# STRICT PROTOCOL
# ------------------------------------------------------------
# Uses only OOF predictions from Module 12A.
#
# For safety outer fold f:
#
#     train safety heads on folds != f
#     evaluate on fold == f
#
# Therefore every final safety prediction is itself OOF.
#
# HOLDOUT untouched.
#
#
# Primary safety endpoint:
#
#     False-negative capture at fixed review workload
#
# Secondary:
#
#     overall error capture
#     FP capture
#     error enrichment
#     AURC
#     retained accuracy / F1 / MCC
#
# ============================================================


import gc
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 12B.1 REQUIRE MODULE 12A
# ============================================================

required_12a = [
    "RAMOE_PROB_OOF",
    "RAMOE_GATE_OOF",
    "RAMOE_THRESHOLD_OOF",
    "RAMOE_ERROR_OOF",
    "RAMOE_RISK_CONF_OOF",
    "RAMOE_RISK_EXPERT_OOF",
    "RAMOE_RISK_FULL_OOF",
    "RAMOE_CORE4_OOF",
    "RAMOE_EXPERTS",
]

missing = [
    x for x in required_12a
    if x not in globals()
]

if len(missing) > 0:

    raise RuntimeError(
        "Run Module 12A first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 12B.2 CONFIG
# ============================================================

ETR_CONFIG = {

    # --------------------------------------------------------
    # Regularization of error-type heads
    # --------------------------------------------------------

    "head_C":
        0.05,

    # --------------------------------------------------------
    # False-negative cost sensitivity
    #
    # Primary setting = 5
    # others = sensitivity analysis
    # --------------------------------------------------------

    "primary_fn_cost":
        5.0,

    "fn_costs":
        [1.0, 2.0, 3.0, 5.0, 10.0],

    # --------------------------------------------------------
    # Selective workloads
    # --------------------------------------------------------

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],

    # --------------------------------------------------------
    # numerical
    # --------------------------------------------------------

    "eps":
        1e-8,
}


# ============================================================
# 12B.3 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N = len(
    y_all
)


moe_prob = np.asarray(
    RAMOE_PROB_OOF,
    dtype=float
)


gate_weights = np.asarray(
    RAMOE_GATE_OOF,
    dtype=float
)


decision_threshold = np.asarray(
    RAMOE_THRESHOLD_OOF,
    dtype=float
)


moe_prediction = (

    moe_prob
    >=
    decision_threshold

).astype(int)


# ============================================================
# 12B.4 RECONSTRUCT STRICT EXPERT OOF PROBABILITIES
# ============================================================
#
# Module 12A created fold-local calibrated expert predictions,
# but did not keep the complete OOF expert matrix.
#
# Recreate it using the SAME fold-local calibration helper.
#
# This does NOT retrain the main models.
# ============================================================

ETR_EXPERT_OOF = np.zeros(

    (
        N,
        len(
            RAMOE_EXPERTS
        )
    ),

    dtype=float
)


print(
    "=" * 120
)

print(
    "RECONSTRUCTING STRICT EXPERT OOF MATRIX"
)

print(
    "=" * 120
)


for outer_fold in range(
    N_SPLITS
):

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    for j, model_name in enumerate(
        RAMOE_EXPERTS
    ):

        _, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        ETR_EXPERT_OOF[
            outer_valid_idx,
            j
        ] = p_valid


print(
    "Expert OOF shape:",
    ETR_EXPERT_OOF.shape
)


print(
    "Expert probability range:",
    ETR_EXPERT_OOF.min(),
    ETR_EXPERT_OOF.max()
)


# ============================================================
# 12B.5 RAW PATIENT CONTEXT
# ============================================================
#
# No handcrafted clinical interactions.
#
# We use the original 21 variables only.
# ============================================================

ETR_CONTEXT_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


X_context = (

    X_dev[
        ETR_CONTEXT_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 12B.6 HELPERS
# ============================================================

def etr_binary_entropy(
    p
):

    p = np.clip(

        np.asarray(
            p,
            dtype=float
        ),

        ETR_CONFIG[
            "eps"
        ],

        1.0
        -
        ETR_CONFIG[
            "eps"
        ]
    )


    return (

        -p
        *
        np.log(
            p
        )

        -

        (
            1.0 - p
        )
        *
        np.log(
            1.0 - p
        )

    ) / np.log(
        2.0
    )


def etr_gate_entropy(
    w
):

    w = np.clip(

        np.asarray(
            w,
            dtype=float
        ),

        ETR_CONFIG[
            "eps"
        ],

        1.0
    )


    entropy = (

        -np.sum(

            w
            *
            np.log(
                w
            ),

            axis=1
        )

    )


    return (

        entropy

        /
        np.log(
            w.shape[1]
        )
    )


# ============================================================
# 12B.7 BUILD SAFETY FEATURES
# ============================================================

def build_etr_features(
    patient_features,
    moe_probability,
    threshold,
    expert_probability,
    routing_weights
):

    p = np.asarray(
        moe_probability,
        dtype=float
    )


    threshold = np.asarray(
        threshold,
        dtype=float
    )


    experts = np.asarray(
        expert_probability,
        dtype=float
    )


    weights = np.asarray(
        routing_weights,
        dtype=float
    )


    # --------------------------------------------------------
    # Prediction state
    # --------------------------------------------------------

    entropy = (
        etr_binary_entropy(
            p
        )
    )


    signed_threshold_margin = (

        p
        -
        threshold
    )


    absolute_threshold_margin = np.abs(
        signed_threshold_margin
    )


    # --------------------------------------------------------
    # Expert state
    # --------------------------------------------------------

    expert_mean = (
        experts.mean(
            axis=1
        )
    )


    expert_std = (
        experts.std(
            axis=1
        )
    )


    expert_min = (
        experts.min(
            axis=1
        )
    )


    expert_max = (
        experts.max(
            axis=1
        )
    )


    expert_range = (

        expert_max
        -
        expert_min
    )


    expert_median = np.median(
        experts,
        axis=1
    )


    consensus_gap = np.abs(

        p
        -
        expert_mean
    )


    # --------------------------------------------------------
    # Routing state
    # --------------------------------------------------------

    gate_entropy = (
        etr_gate_entropy(
            weights
        )
    )


    max_gate = (
        weights.max(
            axis=1
        )
    )


    sorted_w = np.sort(
        weights,
        axis=1
    )


    routing_margin = (

        sorted_w[
            :,
            -1
        ]

        -
        sorted_w[
            :,
            -2
        ]
    )


    effective_experts = np.exp(

        -np.sum(

            np.clip(
                weights,
                1e-12,
                1.0
            )

            *
            np.log(
                np.clip(
                    weights,
                    1e-12,
                    1.0
                )
            ),

            axis=1
        )
    )


    weighted_variance = np.sum(

        weights

        *
        (
            experts
            -
            p[
                :,
                None
            ]
        )
        ** 2,

        axis=1
    )


    dominant_expert = np.argmax(
        weights,
        axis=1
    )


    dominant_prob = experts[

        np.arange(
            len(experts)
        ),

        dominant_expert
    ]


    dominant_gap = np.abs(

        dominant_prob
        -
        p
    )


    # ========================================================
    # FEATURE GROUP A — ordinary confidence baseline
    # ========================================================

    confidence_features = np.column_stack([

        p,

        entropy,

        signed_threshold_margin,

        absolute_threshold_margin,
    ])


    # ========================================================
    # FEATURE GROUP B — MoE state only
    # ========================================================

    moe_state_features = np.column_stack([

        p,
        entropy,
        signed_threshold_margin,
        absolute_threshold_margin,

        expert_mean,
        expert_std,
        expert_min,
        expert_max,
        expert_range,
        expert_median,
        consensus_gap,

        gate_entropy,
        max_gate,
        routing_margin,
        effective_experts,
        weighted_variance,
        dominant_gap,

        experts,

        weights,
    ])


    # ========================================================
    # FEATURE GROUP C — full error-type model
    #
    # MoE state + raw patient context
    # ========================================================

    full_features = np.column_stack([

        moe_state_features,

        patient_features,
    ])


    return (
        confidence_features,
        moe_state_features,
        full_features
    )


# ============================================================
# 12B.8 BUILD GLOBAL CLEAN OOF FEATURE MATRICES
# ============================================================

(
    ETR_CONF_FEATURES,
    ETR_MOE_FEATURES,
    ETR_FULL_FEATURES

) = build_etr_features(

    X_context,

    moe_prob,

    decision_threshold,

    ETR_EXPERT_OOF,

    gate_weights
)


print(
    "\nFeature dimensions:"
)

print(
    "  Confidence:",
    ETR_CONF_FEATURES.shape
)

print(
    "  MoE state :",
    ETR_MOE_FEATURES.shape
)

print(
    "  Full      :",
    ETR_FULL_FEATURES.shape
)


# ============================================================
# 12B.9 ERROR-TYPE TARGETS
# ============================================================

FN_TARGET = (

    (y_all == 1)

    &
    (moe_prediction == 0)

).astype(int)


FP_TARGET = (

    (y_all == 0)

    &
    (moe_prediction == 1)

).astype(int)


ERROR_TARGET = (

    FN_TARGET
    |
    FP_TARGET

).astype(int)


print(
    "\nGlobal error composition:"
)

print(
    "  Total N:",
    N
)

print(
    "  Errors:",
    ERROR_TARGET.sum()
)

print(
    "  False negatives:",
    FN_TARGET.sum()
)

print(
    "  False positives:",
    FP_TARGET.sum()
)

print(
    "  Error prevalence:",
    ERROR_TARGET.mean()
)

print(
    "  FN prevalence:",
    FN_TARGET.mean()
)

print(
    "  FP prevalence:",
    FP_TARGET.mean()
)


# ============================================================
# 12B.10 MODEL CONSTRUCTOR
# ============================================================
#
# Logistic model intentionally used because:
#
# - extremely fast
# - stable
# - probabilities are interpretable
# - nonlinear predictive architecture remains the MoE
# - easier to defend as a safety head rather than another
#   disease classifier
#
# class_weight="balanced" is used only for rare error-type
# learning.
#
# ============================================================

def make_etr_head():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    ETR_CONFIG[
                        "head_C"
                    ],

                class_weight=
                    "balanced",

                solver=
                    "lbfgs",

                max_iter=
                    3000
            )
        )
    ])


# ============================================================
# 12B.11 OUTPUT BUFFERS
# ============================================================

# Confidence-only conditional heads

ETR_FN_CONF_OOF = np.zeros(
    N,
    dtype=float
)


ETR_FP_CONF_OOF = np.zeros(
    N,
    dtype=float
)


# MoE-state conditional heads

ETR_FN_MOE_OOF = np.zeros(
    N,
    dtype=float
)


ETR_FP_MOE_OOF = np.zeros(
    N,
    dtype=float
)


# Full context-aware error-type heads

ETR_FN_FULL_OOF = np.zeros(
    N,
    dtype=float
)


ETR_FP_FULL_OOF = np.zeros(
    N,
    dtype=float
)


fold_rows = []


# ============================================================
# 12B.12 STRICT SAFETY-HEAD CROSS-FITTING
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 125
    )

    print(
        f"DC-ETR-MoE SAFETY FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 125
    )


    train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    # --------------------------------------------------------
    # Decision-conditional training populations
    # --------------------------------------------------------

    train_negative = train_idx[

        moe_prediction[
            train_idx
        ]
        ==
        0
    ]


    train_positive = train_idx[

        moe_prediction[
            train_idx
        ]
        ==
        1
    ]


    valid_negative = valid_idx[

        moe_prediction[
            valid_idx
        ]
        ==
        0
    ]


    valid_positive = valid_idx[

        moe_prediction[
            valid_idx
        ]
        ==
        1
    ]


    print(
        "\nDecision-conditional populations:"
    )

    print(
        "  Train predicted negative:",
        len(
            train_negative
        ),
        "| FN:",
        FN_TARGET[
            train_negative
        ].sum()
    )

    print(
        "  Train predicted positive:",
        len(
            train_positive
        ),
        "| FP:",
        FP_TARGET[
            train_positive
        ].sum()
    )

    print(
        "  Valid predicted negative:",
        len(
            valid_negative
        ),
        "| FN:",
        FN_TARGET[
            valid_negative
        ].sum()
    )

    print(
        "  Valid predicted positive:",
        len(
            valid_positive
        ),
        "| FP:",
        FP_TARGET[
            valid_positive
        ].sum()
    )


    # ========================================================
    # A. CONFIDENCE-ONLY HEADS
    # ========================================================

    fn_conf_model = (
        make_etr_head()
    )


    fp_conf_model = (
        make_etr_head()
    )


    fn_conf_model.fit(

        ETR_CONF_FEATURES[
            train_negative
        ],

        FN_TARGET[
            train_negative
        ]
    )


    fp_conf_model.fit(

        ETR_CONF_FEATURES[
            train_positive
        ],

        FP_TARGET[
            train_positive
        ]
    )


    ETR_FN_CONF_OOF[
        valid_negative
    ] = (

        fn_conf_model
        .predict_proba(

            ETR_CONF_FEATURES[
                valid_negative
            ]

        )[
            :,
            1
        ]
    )


    ETR_FP_CONF_OOF[
        valid_positive
    ] = (

        fp_conf_model
        .predict_proba(

            ETR_CONF_FEATURES[
                valid_positive
            ]

        )[
            :,
            1
        ]
    )


    # ========================================================
    # B. MoE-STATE HEADS
    # ========================================================

    fn_moe_model = (
        make_etr_head()
    )


    fp_moe_model = (
        make_etr_head()
    )


    fn_moe_model.fit(

        ETR_MOE_FEATURES[
            train_negative
        ],

        FN_TARGET[
            train_negative
        ]
    )


    fp_moe_model.fit(

        ETR_MOE_FEATURES[
            train_positive
        ],

        FP_TARGET[
            train_positive
        ]
    )


    ETR_FN_MOE_OOF[
        valid_negative
    ] = (

        fn_moe_model
        .predict_proba(

            ETR_MOE_FEATURES[
                valid_negative
            ]

        )[
            :,
            1
        ]
    )


    ETR_FP_MOE_OOF[
        valid_positive
    ] = (

        fp_moe_model
        .predict_proba(

            ETR_MOE_FEATURES[
                valid_positive
            ]

        )[
            :,
            1
        ]
    )


    # ========================================================
    # C. FULL CONTEXT-AWARE ERROR-TYPE HEADS
    # ========================================================

    fn_full_model = (
        make_etr_head()
    )


    fp_full_model = (
        make_etr_head()
    )


    fn_full_model.fit(

        ETR_FULL_FEATURES[
            train_negative
        ],

        FN_TARGET[
            train_negative
        ]
    )


    fp_full_model.fit(

        ETR_FULL_FEATURES[
            train_positive
        ],

        FP_TARGET[
            train_positive
        ]
    )


    ETR_FN_FULL_OOF[
        valid_negative
    ] = (

        fn_full_model
        .predict_proba(

            ETR_FULL_FEATURES[
                valid_negative
            ]

        )[
            :,
            1
        ]
    )


    ETR_FP_FULL_OOF[
        valid_positive
    ] = (

        fp_full_model
        .predict_proba(

            ETR_FULL_FEATURES[
                valid_positive
            ]

        )[
            :,
            1
        ]
    )


    # ========================================================
    # D. FOLD FN PERFORMANCE
    # ========================================================

    if (
        len(
            np.unique(
                FN_TARGET[
                    valid_negative
                ]
            )
        )
        >
        1
    ):

        fn_conf_pr = (
            average_precision_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_CONF_OOF[
                    valid_negative
                ]
            )
        )


        fn_moe_pr = (
            average_precision_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_MOE_OOF[
                    valid_negative
                ]
            )
        )


        fn_full_pr = (
            average_precision_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_FULL_OOF[
                    valid_negative
                ]
            )
        )


        fn_conf_roc = (
            roc_auc_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_CONF_OOF[
                    valid_negative
                ]
            )
        )


        fn_moe_roc = (
            roc_auc_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_MOE_OOF[
                    valid_negative
                ]
            )
        )


        fn_full_roc = (
            roc_auc_score(

                FN_TARGET[
                    valid_negative
                ],

                ETR_FN_FULL_OOF[
                    valid_negative
                ]
            )
        )

    else:

        fn_conf_pr = np.nan
        fn_moe_pr = np.nan
        fn_full_pr = np.nan

        fn_conf_roc = np.nan
        fn_moe_roc = np.nan
        fn_full_roc = np.nan


    # ========================================================
    # E. FOLD FP PERFORMANCE
    # ========================================================

    if (
        len(
            np.unique(
                FP_TARGET[
                    valid_positive
                ]
            )
        )
        >
        1
    ):

        fp_conf_pr = (
            average_precision_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_CONF_OOF[
                    valid_positive
                ]
            )
        )


        fp_moe_pr = (
            average_precision_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_MOE_OOF[
                    valid_positive
                ]
            )
        )


        fp_full_pr = (
            average_precision_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_FULL_OOF[
                    valid_positive
                ]
            )
        )


        fp_conf_roc = (
            roc_auc_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_CONF_OOF[
                    valid_positive
                ]
            )
        )


        fp_moe_roc = (
            roc_auc_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_MOE_OOF[
                    valid_positive
                ]
            )
        )


        fp_full_roc = (
            roc_auc_score(

                FP_TARGET[
                    valid_positive
                ],

                ETR_FP_FULL_OOF[
                    valid_positive
                ]
            )
        )

    else:

        fp_conf_pr = np.nan
        fp_moe_pr = np.nan
        fp_full_pr = np.nan

        fp_conf_roc = np.nan
        fp_moe_roc = np.nan
        fp_full_roc = np.nan


    # ========================================================
    # F. SAVE FOLD SUMMARY
    # ========================================================

    fold_rows.append({

        "Fold":
            outer_fold,

        "FN_Prevalence_AmongPredNegative":
            FN_TARGET[
                valid_negative
            ].mean(),

        "FP_Prevalence_AmongPredPositive":
            FP_TARGET[
                valid_positive
            ].mean(),

        "FN_Confidence_PR":
            fn_conf_pr,

        "FN_MoEState_PR":
            fn_moe_pr,

        "FN_Full_PR":
            fn_full_pr,

        "FN_Confidence_ROC":
            fn_conf_roc,

        "FN_MoEState_ROC":
            fn_moe_roc,

        "FN_Full_ROC":
            fn_full_roc,

        "FP_Confidence_PR":
            fp_conf_pr,

        "FP_MoEState_PR":
            fp_moe_pr,

        "FP_Full_PR":
            fp_full_pr,

        "FP_Confidence_ROC":
            fp_conf_roc,

        "FP_MoEState_ROC":
            fp_moe_roc,

        "FP_Full_ROC":
            fp_full_roc,
    })


    print(
        "\nFalse-negative detection:"
    )

    print(
        f"  Confidence PR : "
        f"{fn_conf_pr:.6f}"
    )

    print(
        f"  MoE-state PR  : "
        f"{fn_moe_pr:.6f}"
    )

    print(
        f"  Full ETR PR   : "
        f"{fn_full_pr:.6f}"
    )

    print(
        f"  Full ETR ROC  : "
        f"{fn_full_roc:.6f}"
    )


    print(
        "\nFalse-positive detection:"
    )

    print(
        f"  Confidence PR : "
        f"{fp_conf_pr:.6f}"
    )

    print(
        f"  MoE-state PR  : "
        f"{fp_moe_pr:.6f}"
    )

    print(
        f"  Full ETR PR   : "
        f"{fp_full_pr:.6f}"
    )

    print(
        f"  Full ETR ROC  : "
        f"{fp_full_roc:.6f}"
    )


    del (
        fn_conf_model,
        fp_conf_model,
        fn_moe_model,
        fp_moe_model,
        fn_full_model,
        fp_full_model,
    )


    gc.collect()


# ============================================================
# 12B.13 GLOBAL ERROR-TYPE BENCHMARK
# ============================================================

negative_mask = (
    moe_prediction
    ==
    0
)


positive_mask = (
    moe_prediction
    ==
    1
)


def metric_pair(
    y,
    score
):

    return (
        average_precision_score(
            y,
            score
        ),
        roc_auc_score(
            y,
            score
        )
    )


# ------------------------------------------------------------
# FN
# ------------------------------------------------------------

fn_conf_pr, fn_conf_roc = (
    metric_pair(

        FN_TARGET[
            negative_mask
        ],

        ETR_FN_CONF_OOF[
            negative_mask
        ]
    )
)


fn_moe_pr, fn_moe_roc = (
    metric_pair(

        FN_TARGET[
            negative_mask
        ],

        ETR_FN_MOE_OOF[
            negative_mask
        ]
    )
)


fn_full_pr, fn_full_roc = (
    metric_pair(

        FN_TARGET[
            negative_mask
        ],

        ETR_FN_FULL_OOF[
            negative_mask
        ]
    )
)


# ------------------------------------------------------------
# FP
# ------------------------------------------------------------

fp_conf_pr, fp_conf_roc = (
    metric_pair(

        FP_TARGET[
            positive_mask
        ],

        ETR_FP_CONF_OOF[
            positive_mask
        ]
    )
)


fp_moe_pr, fp_moe_roc = (
    metric_pair(

        FP_TARGET[
            positive_mask
        ],

        ETR_FP_MOE_OOF[
            positive_mask
        ]
    )
)


fp_full_pr, fp_full_roc = (
    metric_pair(

        FP_TARGET[
            positive_mask
        ],

        ETR_FP_FULL_OOF[
            positive_mask
        ]
    )
)


error_type_results = pd.DataFrame([

    {
        "Error_Type":
            "FalseNegative",

        "Model":
            "Confidence",

        "PR_AUC":
            fn_conf_pr,

        "ROC_AUC":
            fn_conf_roc,
    },

    {
        "Error_Type":
            "FalseNegative",

        "Model":
            "MoE_State",

        "PR_AUC":
            fn_moe_pr,

        "ROC_AUC":
            fn_moe_roc,
    },

    {
        "Error_Type":
            "FalseNegative",

        "Model":
            "DC_ETR_MoE",

        "PR_AUC":
            fn_full_pr,

        "ROC_AUC":
            fn_full_roc,
    },


    {
        "Error_Type":
            "FalsePositive",

        "Model":
            "Confidence",

        "PR_AUC":
            fp_conf_pr,

        "ROC_AUC":
            fp_conf_roc,
    },

    {
        "Error_Type":
            "FalsePositive",

        "Model":
            "MoE_State",

        "PR_AUC":
            fp_moe_pr,

        "ROC_AUC":
            fp_moe_roc,
    },

    {
        "Error_Type":
            "FalsePositive",

        "Model":
            "DC_ETR_MoE",

        "PR_AUC":
            fp_full_pr,

        "ROC_AUC":
            fp_full_roc,
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 12B — ERROR-TYPE DETECTION"
)

print(
    "=" * 130
)


display(
    error_type_results
)


# ============================================================
# 12B.14 GAINS OVER CONFIDENCE
# ============================================================

error_type_gain = pd.DataFrame([

    {
        "Target":
            "FalseNegative",

        "Baseline_PR":
            fn_conf_pr,

        "ETR_PR":
            fn_full_pr,

        "Delta_PR":
            fn_full_pr
            -
            fn_conf_pr,

        "Baseline_ROC":
            fn_conf_roc,

        "ETR_ROC":
            fn_full_roc,

        "Delta_ROC":
            fn_full_roc
            -
            fn_conf_roc,
    },

    {
        "Target":
            "FalsePositive",

        "Baseline_PR":
            fp_conf_pr,

        "ETR_PR":
            fp_full_pr,

        "Delta_PR":
            fp_full_pr
            -
            fp_conf_pr,

        "Baseline_ROC":
            fp_conf_roc,

        "ETR_ROC":
            fp_full_roc,

        "Delta_ROC":
            fp_full_roc
            -
            fp_conf_roc,
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "ERROR-TYPE RELIABILITY GAIN"
)

print(
    "=" * 130
)


display(
    error_type_gain
)


# ============================================================
# 12B.15 COST-AWARE RISK
# ============================================================
#
# Predicted negative:
#
#     risk = lambda_FN * P(FN)
#
# Predicted positive:
#
#     risk = P(FP)
#
# ============================================================

def build_cost_aware_risk(
    fn_score,
    fp_score,
    prediction,
    fn_cost
):

    risk = np.zeros(
        len(
            prediction
        ),
        dtype=float
    )


    neg = (
        prediction
        ==
        0
    )


    pos = (
        prediction
        ==
        1
    )


    risk[
        neg
    ] = (

        fn_cost
        *
        fn_score[
            neg
        ]
    )


    risk[
        pos
    ] = (

        fp_score[
            pos
        ]
    )


    return risk


# ============================================================
# 12B.16 DEFERRAL EVALUATOR
# ============================================================

def evaluate_error_type_deferral(
    risk_score,
    method_name,
    fn_cost
):

    rows = []


    total_errors = max(
        ERROR_TARGET.sum(),
        1
    )


    total_fn = max(
        FN_TARGET.sum(),
        1
    )


    total_fp = max(
        FP_TARGET.sum(),
        1
    )


    base_error_rate = (
        ERROR_TARGET.mean()
    )


    order = np.argsort(
        -risk_score
    )


    for rate in (
        ETR_CONFIG[
            "deferral_rates"
        ]
    ):

        n_defer = max(

            1,

            int(
                round(
                    N
                    *
                    rate
                )
            )
        )


        defer_idx = (
            order[
                :n_defer
            ]
        )


        keep = np.ones(
            N,
            dtype=bool
        )


        keep[
            defer_idx
        ] = False


        deferred_error_rate = (

            ERROR_TARGET[
                defer_idx
            ].mean()
        )


        errors_captured = (

            ERROR_TARGET[
                defer_idx
            ].sum()

            /
            total_errors
        )


        fn_captured = (

            FN_TARGET[
                defer_idx
            ].sum()

            /
            total_fn
        )


        fp_captured = (

            FP_TARGET[
                defer_idx
            ].sum()

            /
            total_fp
        )


        rows.append({

            "Method":
                method_name,

            "FN_Cost":
                fn_cost,

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0
                -
                rate,

            "Deferred_N":
                n_defer,

            "Deferred_Error_Rate":
                deferred_error_rate,

            "Error_Enrichment":
                (
                    deferred_error_rate
                    /
                    max(
                        base_error_rate,
                        1e-12
                    )
                ),

            "Errors_Captured":
                errors_captured,

            "FalseNegatives_Captured":
                fn_captured,

            "FalsePositives_Captured":
                fp_captured,

            "Retained_Accuracy":
                accuracy_score(

                    y_all[
                        keep
                    ],

                    moe_prediction[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_all[
                        keep
                    ],

                    moe_prediction[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_all[
                        keep
                    ],

                    moe_prediction[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 12B.17 CONFIDENCE ERROR-TYPE BASELINE
# ============================================================

confidence_risk = np.zeros(
    N,
    dtype=float
)


# for predicted negatives:
# higher probability = more likely FN

confidence_risk[
    negative_mask
] = moe_prob[
    negative_mask
]


# for predicted positives:
# lower disease probability = more likely FP

confidence_risk[
    positive_mask
] = (

    1.0
    -
    moe_prob[
        positive_mask
    ]
)


confidence_deferral = (
    evaluate_error_type_deferral(

        confidence_risk,

        method_name=
            "Confidence",

        fn_cost=
            1.0
    )
)


# ============================================================
# 12B.18 GENERIC RA-MoE BASELINE FROM MODULE 12A
# ============================================================

generic_ramoe_deferral = (
    evaluate_error_type_deferral(

        RAMOE_RISK_FULL_OOF,

        method_name=
            "Generic_RA_MoE",

        fn_cost=
            1.0
    )
)


# ============================================================
# 12B.19 COST SENSITIVITY
# ============================================================

cost_deferral_frames = []


ETR_COST_RISK = {}


for fn_cost in (
    ETR_CONFIG[
        "fn_costs"
    ]
):

    risk = (
        build_cost_aware_risk(

            ETR_FN_FULL_OOF,

            ETR_FP_FULL_OOF,

            moe_prediction,

            fn_cost=
                fn_cost
        )
    )


    ETR_COST_RISK[
        fn_cost
    ] = risk


    df = (
        evaluate_error_type_deferral(

            risk,

            method_name=
                "DC_ETR_MoE",

            fn_cost=
                fn_cost
        )
    )


    cost_deferral_frames.append(
        df
    )


cost_deferral = pd.concat(

    cost_deferral_frames,

    ignore_index=True
)


# ============================================================
# 12B.20 PRIMARY SELECTIVE COMPARISON
# ============================================================

primary_fn_cost = (
    ETR_CONFIG[
        "primary_fn_cost"
    ]
)


primary_etr = (

    cost_deferral[
        cost_deferral[
            "FN_Cost"
        ]
        ==
        primary_fn_cost
    ]
    .copy()
)


primary_selective_comparison = pd.concat(

    [
        confidence_deferral,
        generic_ramoe_deferral,
        primary_etr,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 130
)

print(
    "PRIMARY ERROR-TYPE-AWARE SELECTIVE RESULTS"
)

print(
    f"DC-ETR-MoE primary FN cost = "
    f"{primary_fn_cost:g}"
)

print(
    "=" * 130
)


display(

    primary_selective_comparison[
        [
            "Method",
            "FN_Cost",
            "Deferral_Rate",
            "Coverage",
            "Deferred_Error_Rate",
            "Error_Enrichment",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "FalsePositives_Captured",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 12B.21 FN-COST SENSITIVITY
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FALSE-NEGATIVE COST SENSITIVITY"
)

print(
    "=" * 130
)


display(

    cost_deferral[
        [
            "FN_Cost",
            "Deferral_Rate",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "FalsePositives_Captured",
            "Error_Enrichment",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 12B.22 WORKLOAD EFFICIENCY
# ============================================================
#
# How much of the review workload is actually devoted
# to dangerous false negatives?
# ============================================================

workload_rows = []


for _, row in primary_selective_comparison.iterrows():

    deferred_n = int(
        round(
            N
            *
            row[
                "Deferral_Rate"
            ]
        )
    )


    if row[
        "Method"
    ] == "DC_ETR_MoE":

        risk = (
            ETR_COST_RISK[
                primary_fn_cost
            ]
        )

    elif row[
        "Method"
    ] == "Generic_RA_MoE":

        risk = (
            RAMOE_RISK_FULL_OOF
        )

    else:

        risk = (
            confidence_risk
        )


    idx = np.argsort(
        -risk
    )[
        :deferred_n
    ]


    fn_n = FN_TARGET[
        idx
    ].sum()


    fp_n = FP_TARGET[
        idx
    ].sum()


    workload_rows.append({

        "Method":
            row[
                "Method"
            ],

        "Deferral_Rate":
            row[
                "Deferral_Rate"
            ],

        "Deferred_N":
            deferred_n,

        "FN_Found":
            int(
                fn_n
            ),

        "FP_Found":
            int(
                fp_n
            ),

        "FN_Per_100_Reviews":
            (
                fn_n
                /
                deferred_n
                *
                100.0
            ),

        "Any_Error_Per_100_Reviews":
            (
                (
                    fn_n
                    +
                    fp_n
                )
                /
                deferred_n
                *
                100.0
            ),
    })


workload_efficiency = pd.DataFrame(
    workload_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "REVIEW WORKLOAD EFFICIENCY"
)

print(
    "=" * 130
)


display(
    workload_efficiency
)


# ============================================================
# 12B.23 FOLD STABILITY
# ============================================================

fold_results = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "DC-ETR-MoE FOLD RESULTS"
)

print(
    "=" * 130
)


display(
    fold_results
)


print(
    "\nMean FN PR-AUC:"
)

print(
    "  Confidence:",
    fold_results[
        "FN_Confidence_PR"
    ].mean()
)

print(
    "  MoE state :",
    fold_results[
        "FN_MoEState_PR"
    ].mean()
)

print(
    "  Full ETR  :",
    fold_results[
        "FN_Full_PR"
    ].mean()
)


print(
    "\nPositive FN-PR folds: Full ETR vs confidence:"
)

print(
    int(
        (
            fold_results[
                "FN_Full_PR"
            ]
            >
            fold_results[
                "FN_Confidence_PR"
            ]
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 12B.24 PRINCIPAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 12B SUMMARY"
)

print(
    "=" * 130
)


print(
    "\nFalse-negative detection:"
)

print(
    f"  Confidence PR-AUC : "
    f"{fn_conf_pr:.6f}"
)

print(
    f"  MoE-state PR-AUC  : "
    f"{fn_moe_pr:.6f}"
)

print(
    f"  DC-ETR-MoE PR-AUC : "
    f"{fn_full_pr:.6f}"
)


print(
    "\nFalse-negative ROC-AUC:"
)

print(
    f"  Confidence : "
    f"{fn_conf_roc:.6f}"
)

print(
    f"  MoE-state  : "
    f"{fn_moe_roc:.6f}"
)

print(
    f"  DC-ETR-MoE : "
    f"{fn_full_roc:.6f}"
)


print(
    "\nFalse-positive detection:"
)

print(
    f"  Confidence PR-AUC : "
    f"{fp_conf_pr:.6f}"
)

print(
    f"  MoE-state PR-AUC  : "
    f"{fp_moe_pr:.6f}"
)

print(
    f"  DC-ETR-MoE PR-AUC : "
    f"{fp_full_pr:.6f}"
)


# ============================================================
# 12B.25 SAVE
# ============================================================

save_csv(

    error_type_results,

    TAB_DIR
    / "module12b_error_type_detection.csv"
)


save_csv(

    error_type_gain,

    TAB_DIR
    / "module12b_error_type_gain.csv"
)


save_csv(

    primary_selective_comparison,

    TAB_DIR
    / "module12b_primary_selective_results.csv"
)


save_csv(

    cost_deferral,

    TAB_DIR
    / "module12b_fn_cost_sensitivity.csv"
)


save_csv(

    workload_efficiency,

    TAB_DIR
    / "module12b_workload_efficiency.csv"
)


save_csv(

    fold_results,

    TAB_DIR
    / "module12b_fold_results.csv"
)


etr_prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "MoE_Probability":
        moe_prob,

    "DecisionThreshold":
        decision_threshold,

    "MoE_Prediction":
        moe_prediction,

    "FalseNegative":
        FN_TARGET,

    "FalsePositive":
        FP_TARGET,

    "AnyError":
        ERROR_TARGET,

    "FN_Risk_Confidence":
        ETR_FN_CONF_OOF,

    "FN_Risk_MoEState":
        ETR_FN_MOE_OOF,

    "FN_Risk_DC_ETR":
        ETR_FN_FULL_OOF,

    "FP_Risk_Confidence":
        ETR_FP_CONF_OOF,

    "FP_Risk_MoEState":
        ETR_FP_MOE_OOF,

    "FP_Risk_DC_ETR":
        ETR_FP_FULL_OOF,

    "Generic_RA_MoE_ErrorRisk":
        RAMOE_RISK_FULL_OOF,

    "Primary_DC_ETR_Risk":
        ETR_COST_RISK[
            primary_fn_cost
        ],
})


save_csv(

    etr_prediction_df,

    PRED_DIR
    / "module12b_dc_etr_moe_oof.csv"
)


module12b_manifest = {

    "method":
        "Decision-Conditional Error-Type Reliability Mixture-of-Experts",

    "main_predictor":
        "RA-MoE",

    "safety_heads": [
        "false-negative conditional head",
        "false-positive conditional head",
    ],

    "conditioning":
        "MoE decision class",

    "full_safety_inputs": [
        "MoE probability",
        "decision margin",
        "predictive entropy",
        "expert probabilities",
        "expert disagreement",
        "routing weights",
        "routing entropy",
        "routing concentration",
        "raw patient context",
    ],

    "primary_fn_cost":
        primary_fn_cost,

    "fn_cost_sensitivity":
        ETR_CONFIG[
            "fn_costs"
        ],

    "deferral_rates":
        ETR_CONFIG[
            "deferral_rates"
        ],

    "safety_head_cross_fitted":
        True,

    "holdout_used":
        False,
}


save_json(

    module12b_manifest,

    LOG_DIR
    / "module12b_manifest.json"
)


print(
    "\nMODULE 12B COMPLETE."
)

RECONSTRUCTING STRICT EXPERT OOF MATRIX
Expert OOF shape: (202944, 4)
Expert probability range: 0.000102820928331898 0.9065921943035791

Feature dimensions:
  Confidence: (202944, 4)
  MoE state : (202944, 25)
  Full      : (202944, 46)

Global error composition:
  Total N: 202944
  Errors: 29316
  False negatives: 8505
  False positives: 20811
  Error prevalence: 0.14445364238410596
  FN prevalence: 0.04190811258278146
  FP prevalence: 0.1025455298013245

DC-ETR-MoE SAFETY FOLD 1/5

Decision-conditional populations:
  Train predicted negative: 137987 | FN: 6980
  Train predicted positive: 24368 | FP: 16057
  Valid predicted negative: 33537 | FN: 1525
  Valid predicted positive: 7052 | FP: 4754

False-negative detection:
  Confidence PR : 0.142713
  MoE-state PR  : 0.139891
  Full ETR PR   : 0.136171
  Full ETR ROC  : 0.802715

False-positive detection:
  Confidence PR : 0.755822
  MoE-state PR  : 0.753277
  Full ETR PR   : 0.749430
  Full ETR ROC  : 0.632331

DC-ETR-MoE SAFETY FOLD 2/

,Error_Type,Model,PR_AUC,ROC_AUC
0,FalseNegative,Confidence,0.147841,0.803959
1,FalseNegative,MoE_State,0.145306,0.803481
2,FalseNegative,DC_ETR_MoE,0.149525,0.804152
3,FalsePositive,Confidence,0.751431,0.640639
4,FalsePositive,MoE_State,0.750470,0.639994
5,FalsePositive,DC_ETR_MoE,0.749005,0.638522



ERROR-TYPE RELIABILITY GAIN


,Target,Baseline_PR,ETR_PR,Delta_PR,Baseline_ROC,ETR_ROC,Delta_ROC
0,FalseNegative,0.147841,0.149525,0.001684,0.803959,0.804152,0.000194
1,FalsePositive,0.751431,0.749005,-0.002426,0.640639,0.638522,-0.002116



PRIMARY ERROR-TYPE-AWARE SELECTIVE RESULTS
DC-ETR-MoE primary FN cost = 5


...


REVIEW WORKLOAD EFFICIENCY


,Method,Deferral_Rate,Deferred_N,FN_Found,FP_Found,FN_Per_100_Reviews,Any_Error_Per_100_Reviews
0,Confidence,0.05,10147,0,7871,0.000000,77.569725
1,Confidence,0.10,20294,0,14861,0.000000,73.228540
2,Confidence,0.15,30442,0,20520,0.000000,67.406872
3,Confidence,0.20,40589,1784,20811,4.395280,55.667792
4,Generic_RA_MoE,0.05,10147,0,6379,0.000000,62.865872
5,Generic_RA_MoE,0.10,20294,0,13155,0.000000,64.822115
6,Generic_RA_MoE,0.15,30442,131,20040,0.430327,66.260430
7,Generic_RA_MoE,0.20,40589,1799,20686,4.432235,55.396782
8,DC_ETR_MoE,0.05,10147,1836,0,18.094018,18.094018
9,DC_ETR_MoE,0.10,20294,3399,0,16.748793,16.748793



DC-ETR-MoE FOLD RESULTS


,Fold,FN_Prevalence_AmongPredNegative,FP_Prevalence_AmongPredPositive,FN_Confidence_PR,FN_MoEState_PR,FN_Full_PR,FN_Confidence_ROC,FN_MoEState_ROC,FN_Full_ROC,FP_Confidence_PR,FP_MoEState_PR,FP_Full_PR,FP_Confidence_ROC,FP_MoEState_ROC,FP_Full_ROC
0,0,0.045472,0.674135,0.142713,0.139891,0.136171,0.805552,0.804658,0.802715,0.755822,0.753277,0.749430,0.636601,0.636834,0.632331
1,1,0.056364,0.628718,0.163193,0.159514,0.170929,0.807477,0.806169,0.810140,0.705762,0.701111,0.704079,0.621011,0.619169,0.618855
2,2,0.047170,0.671144,0.140099,0.137599,0.135208,0.798629,0.799052,0.798097,0.756783,0.758701,0.757627,0.640922,0.641505,0.640277
3,3,0.048530,0.669356,0.149128,0.141864,0.146222,0.803038,0.801254,0.801359,0.759718,0.756830,0.756313,0.641133,0.639699,0.639232
4,4,0.049972,0.658318,0.149151,0.149141,0.152175,0.805740,0.804957,0.805579,0.761802,0.758339,0.758719,0.652691,0.652093,0.651873



Mean FN PR-AUC:
  Confidence: 0.14885676040170062
  MoE state : 0.14560177294296786
  Full ETR  : 0.14814100473291208

Positive FN-PR folds: Full ETR vs confidence:
2 / 5

MODULE 12B SUMMARY

False-negative detection:
  Confidence PR-AUC : 0.147841
  MoE-state PR-AUC  : 0.145306
  DC-ETR-MoE PR-AUC : 0.149525

False-negative ROC-AUC:
  Confidence : 0.803959
  MoE-state  : 0.803481
  DC-ETR-MoE : 0.804152

False-positive detection:
  Confidence PR-AUC : 0.751431
  MoE-state PR-AUC  : 0.750470
  DC-ETR-MoE PR-AUC : 0.749005

MODULE 12B COMPLETE.


In [20]:
# ============================================================
# MODULE 12C — CEH-MoE
# Calibrated Expected-Harm Selective Mixture-of-Experts
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Module 12B showed that decision-conditional FN/FP modeling
# is promising, but its FN and FP scores were trained using
# class_weight="balanced" and therefore were NOT calibrated
# onto a common probability scale.
#
# Module 12C fixes this.
#
#
# ARCHITECTURE
# ------------------------------------------------------------
#
#                         RA-MoE
#                           |
#                           v
#                 disease decision y_hat
#                     /             \
#                    /               \
#              predicted 0       predicted 1
#                  |                 |
#                  v                 v
#             FN risk head       FP risk head
#                  |                 |
#         held-out calibration held-out calibration
#                  |                 |
#                  v                 v
#          calibrated P(FN)   calibrated P(FP)
#                    \             /
#                     \           /
#                      v         v
#                     EXPECTED HARM
#
#         H(x) = C_FN * P(FN|x)      if y_hat=0
#                C_FP * P(FP|x)      if y_hat=1
#
#
# STRICT OUTER-CV
# ------------------------------------------------------------
# For each outer fold:
#
#     OUTER TRAIN
#        |
#        +-- 80% safety-head fit
#        |
#        +-- 20% probability calibration
#
#     OUTER VALID
#        -> untouched
#
#
# Two calibrated safety systems:
#
#   1. BranchConfidence
#      prediction probability / margin / entropy only
#
#   2. CEH-MoE
#      MoE state + routing + expert state + patient context
#
#
# Additional FN-specific benchmark:
#
# Among predicted-negative patients only:
#
#   raw MoE disease probability
#          vs
#   calibrated confidence FN risk
#          vs
#   calibrated CEH-MoE FN risk
#
#
# HOLDOUT IS NEVER USED.
# ============================================================


import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 12C.1 REQUIRE MODULES 12A + 12B
# ============================================================

required_12c = [

    "RAMOE_PROB_OOF",
    "RAMOE_THRESHOLD_OOF",
    "RAMOE_RISK_FULL_OOF",

    "ETR_CONF_FEATURES",
    "ETR_FULL_FEATURES",

    "FN_TARGET",
    "FP_TARGET",
    "ERROR_TARGET",

    "moe_prediction",
    "fold_assignment",
]


missing = [

    x
    for x in required_12c

    if x not in globals()
]


if len(missing) > 0:

    raise RuntimeError(

        "Run Modules 12A and 12B first. Missing: "
        +
        ", ".join(
            missing
        )
    )


# ============================================================
# 12C.2 CONFIG
# ============================================================

CEH_CONFIG = {

    # --------------------------------------------------------
    # 20% of each outer-training fold is reserved exclusively
    # for probability calibration.
    # --------------------------------------------------------

    "calibration_fraction":
        0.20,


    # --------------------------------------------------------
    # Discrimination head
    # --------------------------------------------------------

    "head_C":
        0.05,


    # --------------------------------------------------------
    # Platt probability calibration
    # --------------------------------------------------------

    "platt_C":
        1.0,


    # --------------------------------------------------------
    # Expected-harm cost ratios
    #
    # FP cost = 1.
    # --------------------------------------------------------

    "fn_costs":
        [1.0, 2.0, 3.0, 5.0, 10.0],


    # primary policy for later reporting
    "primary_fn_cost":
        5.0,


    # --------------------------------------------------------
    # total review workload
    # --------------------------------------------------------

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],


    # --------------------------------------------------------
    # FN-only review workload
    #
    # This means % of all predicted-negative cases reviewed.
    # --------------------------------------------------------

    "negative_review_rates":
        [0.05, 0.10, 0.15, 0.20],


    "ece_bins":
        15,
}


# ============================================================
# 12C.3 BASIC DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N = len(
    y_all
)


moe_prob = np.asarray(
    RAMOE_PROB_OOF,
    dtype=float
)


decision_threshold = np.asarray(
    RAMOE_THRESHOLD_OOF,
    dtype=float
)


prediction = np.asarray(
    moe_prediction,
    dtype=int
)


fn_target = np.asarray(
    FN_TARGET,
    dtype=int
)


fp_target = np.asarray(
    FP_TARGET,
    dtype=int
)


error_target = np.asarray(
    ERROR_TARGET,
    dtype=int
)


# ============================================================
# 12C.4 ERROR TYPE STRATIFICATION LABEL
# ============================================================
#
# 0 = correct
# 1 = false negative
# 2 = false positive
#
# This preserves both rare error types in the inner
# fit/calibration split.
# ============================================================

error_type_label = np.zeros(
    N,
    dtype=int
)


error_type_label[
    fn_target == 1
] = 1


error_type_label[
    fp_target == 1
] = 2


# ============================================================
# 12C.5 ECE
# ============================================================

def ceh_ece(
    y_true,
    prob,
    n_bins=15
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    prob = np.asarray(
        prob,
        dtype=float
    )


    bins = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    ece = 0.0


    for b in range(
        n_bins
    ):

        if b < (
            n_bins - 1
        ):

            mask = (

                (prob >= bins[b])

                &

                (prob < bins[b + 1])
            )

        else:

            mask = (

                (prob >= bins[b])

                &

                (prob <= bins[b + 1])
            )


        n_bin = int(
            mask.sum()
        )


        if n_bin == 0:

            continue


        mean_prob = (
            prob[
                mask
            ].mean()
        )


        empirical = (
            y_true[
                mask
            ].mean()
        )


        ece += (

            n_bin
            /
            len(
                y_true
            )

        ) * abs(
            mean_prob
            -
            empirical
        )


    return float(
        ece
    )


# ============================================================
# 12C.6 DISCRIMINATION HEAD
# ============================================================
#
# Balanced logistic regression remains useful for learning
# rare FN patterns.
#
# IMPORTANT:
# Its raw outputs are NOT treated as probabilities.
# They are calibrated afterward on independent inner data.
# ============================================================

def make_ceh_discrimination_head():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    CEH_CONFIG[
                        "head_C"
                    ],

                class_weight=
                    "balanced",

                solver=
                    "lbfgs",

                max_iter=
                    3000
            )
        )
    ])


# ============================================================
# 12C.7 PLATT CALIBRATOR
# ============================================================

def fit_platt_calibrator(
    raw_score,
    y
):

    raw_score = np.asarray(
        raw_score,
        dtype=float
    )


    y = np.asarray(
        y,
        dtype=int
    )


    calibrator = LogisticRegression(

        penalty="l2",

        C=
            CEH_CONFIG[
                "platt_C"
            ],

        solver="lbfgs",

        max_iter=2000
    )


    calibrator.fit(

        raw_score[
            :,
            None
        ],

        y
    )


    return calibrator


def apply_platt(
    calibrator,
    raw_score
):

    raw_score = np.asarray(
        raw_score,
        dtype=float
    )


    return (

        calibrator
        .predict_proba(

            raw_score[
                :,
                None
            ]

        )[
            :,
            1
        ]
    )


# ============================================================
# 12C.8 BUFFERS
# ============================================================

# ------------------------------------------------------------
# Branch-confidence calibrated error probabilities
# ------------------------------------------------------------

CEH_FN_CONF_OOF = np.zeros(
    N,
    dtype=float
)


CEH_FP_CONF_OOF = np.zeros(
    N,
    dtype=float
)


# ------------------------------------------------------------
# Full CEH-MoE calibrated error probabilities
# ------------------------------------------------------------

CEH_FN_FULL_OOF = np.zeros(
    N,
    dtype=float
)


CEH_FP_FULL_OOF = np.zeros(
    N,
    dtype=float
)


fold_rows = []


# ============================================================
# 12C.9 STRICT CROSS-FITTED CONDITIONAL CALIBRATION
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 130
    )

    print(
        f"CEH-MoE OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 130
    )


    # ========================================================
    # A. OUTER TRAIN / VALID
    # ========================================================

    train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    # ========================================================
    # B. STRATIFIED FIT / CALIBRATION SPLIT
    # ========================================================

    splitter = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                CEH_CONFIG[
                    "calibration_fraction"
                ],

            random_state=
                SEED
                + 70000
                + outer_fold
        )
    )


    (
        fit_local,
        calib_local

    ) = next(

        splitter.split(

            np.zeros(
                len(
                    train_idx
                )
            ),

            error_type_label[
                train_idx
            ]
        )
    )


    fit_idx = (
        train_idx[
            fit_local
        ]
    )


    calib_idx = (
        train_idx[
            calib_local
        ]
    )


    # ========================================================
    # C. DECISION-CONDITIONAL GROUPS
    # ========================================================

    fit_neg = fit_idx[
        prediction[
            fit_idx
        ]
        ==
        0
    ]


    fit_pos = fit_idx[
        prediction[
            fit_idx
        ]
        ==
        1
    ]


    calib_neg = calib_idx[
        prediction[
            calib_idx
        ]
        ==
        0
    ]


    calib_pos = calib_idx[
        prediction[
            calib_idx
        ]
        ==
        1
    ]


    valid_neg = valid_idx[
        prediction[
            valid_idx
        ]
        ==
        0
    ]


    valid_pos = valid_idx[
        prediction[
            valid_idx
        ]
        ==
        1
    ]


    print(
        "\nPredicted-negative branch:"
    )

    print(
        "  Fit N:",
        len(
            fit_neg
        ),
        "| FN:",
        fn_target[
            fit_neg
        ].sum()
    )


    print(
        "  Calibration N:",
        len(
            calib_neg
        ),
        "| FN:",
        fn_target[
            calib_neg
        ].sum()
    )


    print(
        "  Validation N:",
        len(
            valid_neg
        ),
        "| FN:",
        fn_target[
            valid_neg
        ].sum()
    )


    print(
        "\nPredicted-positive branch:"
    )

    print(
        "  Fit N:",
        len(
            fit_pos
        ),
        "| FP:",
        fp_target[
            fit_pos
        ].sum()
    )


    print(
        "  Calibration N:",
        len(
            calib_pos
        ),
        "| FP:",
        fp_target[
            calib_pos
        ].sum()
    )


    print(
        "  Validation N:",
        len(
            valid_pos
        ),
        "| FP:",
        fp_target[
            valid_pos
        ].sum()
    )


    # ========================================================
    # D. CONFIDENCE FN MODEL
    # ========================================================

    fn_conf_head = (
        make_ceh_discrimination_head()
    )


    fn_conf_head.fit(

        ETR_CONF_FEATURES[
            fit_neg
        ],

        fn_target[
            fit_neg
        ]
    )


    fn_conf_cal_raw = (
        fn_conf_head
        .decision_function(

            ETR_CONF_FEATURES[
                calib_neg
            ]
        )
    )


    fn_conf_valid_raw = (
        fn_conf_head
        .decision_function(

            ETR_CONF_FEATURES[
                valid_neg
            ]
        )
    )


    fn_conf_platt = (
        fit_platt_calibrator(

            fn_conf_cal_raw,

            fn_target[
                calib_neg
            ]
        )
    )


    CEH_FN_CONF_OOF[
        valid_neg
    ] = (
        apply_platt(

            fn_conf_platt,

            fn_conf_valid_raw
        )
    )


    # ========================================================
    # E. FULL FN MODEL
    # ========================================================

    fn_full_head = (
        make_ceh_discrimination_head()
    )


    fn_full_head.fit(

        ETR_FULL_FEATURES[
            fit_neg
        ],

        fn_target[
            fit_neg
        ]
    )


    fn_full_cal_raw = (
        fn_full_head
        .decision_function(

            ETR_FULL_FEATURES[
                calib_neg
            ]
        )
    )


    fn_full_valid_raw = (
        fn_full_head
        .decision_function(

            ETR_FULL_FEATURES[
                valid_neg
            ]
        )
    )


    fn_full_platt = (
        fit_platt_calibrator(

            fn_full_cal_raw,

            fn_target[
                calib_neg
            ]
        )
    )


    CEH_FN_FULL_OOF[
        valid_neg
    ] = (
        apply_platt(

            fn_full_platt,

            fn_full_valid_raw
        )
    )


    # ========================================================
    # F. CONFIDENCE FP MODEL
    # ========================================================

    fp_conf_head = (
        make_ceh_discrimination_head()
    )


    fp_conf_head.fit(

        ETR_CONF_FEATURES[
            fit_pos
        ],

        fp_target[
            fit_pos
        ]
    )


    fp_conf_cal_raw = (
        fp_conf_head
        .decision_function(

            ETR_CONF_FEATURES[
                calib_pos
            ]
        )
    )


    fp_conf_valid_raw = (
        fp_conf_head
        .decision_function(

            ETR_CONF_FEATURES[
                valid_pos
            ]
        )
    )


    fp_conf_platt = (
        fit_platt_calibrator(

            fp_conf_cal_raw,

            fp_target[
                calib_pos
            ]
        )
    )


    CEH_FP_CONF_OOF[
        valid_pos
    ] = (
        apply_platt(

            fp_conf_platt,

            fp_conf_valid_raw
        )
    )


    # ========================================================
    # G. FULL FP MODEL
    # ========================================================

    fp_full_head = (
        make_ceh_discrimination_head()
    )


    fp_full_head.fit(

        ETR_FULL_FEATURES[
            fit_pos
        ],

        fp_target[
            fit_pos
        ]
    )


    fp_full_cal_raw = (
        fp_full_head
        .decision_function(

            ETR_FULL_FEATURES[
                calib_pos
            ]
        )
    )


    fp_full_valid_raw = (
        fp_full_head
        .decision_function(

            ETR_FULL_FEATURES[
                valid_pos
            ]
        )
    )


    fp_full_platt = (
        fit_platt_calibrator(

            fp_full_cal_raw,

            fp_target[
                calib_pos
            ]
        )
    )


    CEH_FP_FULL_OOF[
        valid_pos
    ] = (
        apply_platt(

            fp_full_platt,

            fp_full_valid_raw
        )
    )


    # ========================================================
    # H. FOLD METRICS
    # ========================================================

    fn_conf_pr = (
        average_precision_score(

            fn_target[
                valid_neg
            ],

            CEH_FN_CONF_OOF[
                valid_neg
            ]
        )
    )


    fn_full_pr = (
        average_precision_score(

            fn_target[
                valid_neg
            ],

            CEH_FN_FULL_OOF[
                valid_neg
            ]
        )
    )


    fn_conf_roc = (
        roc_auc_score(

            fn_target[
                valid_neg
            ],

            CEH_FN_CONF_OOF[
                valid_neg
            ]
        )
    )


    fn_full_roc = (
        roc_auc_score(

            fn_target[
                valid_neg
            ],

            CEH_FN_FULL_OOF[
                valid_neg
            ]
        )
    )


    fp_conf_pr = (
        average_precision_score(

            fp_target[
                valid_pos
            ],

            CEH_FP_CONF_OOF[
                valid_pos
            ]
        )
    )


    fp_full_pr = (
        average_precision_score(

            fp_target[
                valid_pos
            ],

            CEH_FP_FULL_OOF[
                valid_pos
            ]
        )
    )


    fp_conf_roc = (
        roc_auc_score(

            fp_target[
                valid_pos
            ],

            CEH_FP_CONF_OOF[
                valid_pos
            ]
        )
    )


    fp_full_roc = (
        roc_auc_score(

            fp_target[
                valid_pos
            ],

            CEH_FP_FULL_OOF[
                valid_pos
            ]
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "FN_Prevalence":
            fn_target[
                valid_neg
            ].mean(),

        "FN_Confidence_PR":
            fn_conf_pr,

        "FN_CEH_PR":
            fn_full_pr,

        "FN_Delta_PR":
            fn_full_pr
            -
            fn_conf_pr,

        "FN_Confidence_ROC":
            fn_conf_roc,

        "FN_CEH_ROC":
            fn_full_roc,

        "FN_Delta_ROC":
            fn_full_roc
            -
            fn_conf_roc,

        "FP_Prevalence":
            fp_target[
                valid_pos
            ].mean(),

        "FP_Confidence_PR":
            fp_conf_pr,

        "FP_CEH_PR":
            fp_full_pr,

        "FP_Delta_PR":
            fp_full_pr
            -
            fp_conf_pr,

        "FP_Confidence_ROC":
            fp_conf_roc,

        "FP_CEH_ROC":
            fp_full_roc,

        "FP_Delta_ROC":
            fp_full_roc
            -
            fp_conf_roc,
    })


    print(
        "\nFN detection:"
    )

    print(
        f"  Confidence PR : "
        f"{fn_conf_pr:.6f}"
    )

    print(
        f"  CEH-MoE PR    : "
        f"{fn_full_pr:.6f}"
    )

    print(
        f"  Delta         : "
        f"{fn_full_pr-fn_conf_pr:+.6f}"
    )


    print(
        "\nFP detection:"
    )

    print(
        f"  Confidence PR : "
        f"{fp_conf_pr:.6f}"
    )

    print(
        f"  CEH-MoE PR    : "
        f"{fp_full_pr:.6f}"
    )

    print(
        f"  Delta         : "
        f"{fp_full_pr-fp_conf_pr:+.6f}"
    )


    del (
        fn_conf_head,
        fn_full_head,
        fp_conf_head,
        fp_full_head,

        fn_conf_platt,
        fn_full_platt,
        fp_conf_platt,
        fp_full_platt,
    )


    gc.collect()


# ============================================================
# 12C.10 GLOBAL CONDITIONAL PERFORMANCE
# ============================================================

negative_mask = (
    prediction
    ==
    0
)


positive_mask = (
    prediction
    ==
    1
)


# ============================================================
# FN metrics
# ============================================================

fn_prev = (
    fn_target[
        negative_mask
    ].mean()
)


fn_raw_prob_pr = (
    average_precision_score(

        fn_target[
            negative_mask
        ],

        moe_prob[
            negative_mask
        ]
    )
)


fn_raw_prob_roc = (
    roc_auc_score(

        fn_target[
            negative_mask
        ],

        moe_prob[
            negative_mask
        ]
    )
)


fn_conf_pr = (
    average_precision_score(

        fn_target[
            negative_mask
        ],

        CEH_FN_CONF_OOF[
            negative_mask
        ]
    )
)


fn_conf_roc = (
    roc_auc_score(

        fn_target[
            negative_mask
        ],

        CEH_FN_CONF_OOF[
            negative_mask
        ]
    )
)


fn_full_pr = (
    average_precision_score(

        fn_target[
            negative_mask
        ],

        CEH_FN_FULL_OOF[
            negative_mask
        ]
    )
)


fn_full_roc = (
    roc_auc_score(

        fn_target[
            negative_mask
        ],

        CEH_FN_FULL_OOF[
            negative_mask
        ]
    )
)


# ============================================================
# FP metrics
# ============================================================

fp_prev = (
    fp_target[
        positive_mask
    ].mean()
)


fp_raw_prob_pr = (
    average_precision_score(

        fp_target[
            positive_mask
        ],

        1.0
        -
        moe_prob[
            positive_mask
        ]
    )
)


fp_raw_prob_roc = (
    roc_auc_score(

        fp_target[
            positive_mask
        ],

        1.0
        -
        moe_prob[
            positive_mask
        ]
    )
)


fp_conf_pr = (
    average_precision_score(

        fp_target[
            positive_mask
        ],

        CEH_FP_CONF_OOF[
            positive_mask
        ]
    )
)


fp_conf_roc = (
    roc_auc_score(

        fp_target[
            positive_mask
        ],

        CEH_FP_CONF_OOF[
            positive_mask
        ]
    )
)


fp_full_pr = (
    average_precision_score(

        fp_target[
            positive_mask
        ],

        CEH_FP_FULL_OOF[
            positive_mask
        ]
    )
)


fp_full_roc = (
    roc_auc_score(

        fp_target[
            positive_mask
        ],

        CEH_FP_FULL_OOF[
            positive_mask
        ]
    )
)


conditional_detection = pd.DataFrame([

    {
        "Error_Type":
            "FalseNegative",

        "Method":
            "Raw_MoE_Probability",

        "Event_Prevalence":
            fn_prev,

        "PR_AUC":
            fn_raw_prob_pr,

        "ROC_AUC":
            fn_raw_prob_roc,
    },

    {
        "Error_Type":
            "FalseNegative",

        "Method":
            "Calibrated_BranchConfidence",

        "Event_Prevalence":
            fn_prev,

        "PR_AUC":
            fn_conf_pr,

        "ROC_AUC":
            fn_conf_roc,
    },

    {
        "Error_Type":
            "FalseNegative",

        "Method":
            "CEH_MoE",

        "Event_Prevalence":
            fn_prev,

        "PR_AUC":
            fn_full_pr,

        "ROC_AUC":
            fn_full_roc,
    },


    {
        "Error_Type":
            "FalsePositive",

        "Method":
            "Raw_MoE_Confidence",

        "Event_Prevalence":
            fp_prev,

        "PR_AUC":
            fp_raw_prob_pr,

        "ROC_AUC":
            fp_raw_prob_roc,
    },

    {
        "Error_Type":
            "FalsePositive",

        "Method":
            "Calibrated_BranchConfidence",

        "Event_Prevalence":
            fp_prev,

        "PR_AUC":
            fp_conf_pr,

        "ROC_AUC":
            fp_conf_roc,
    },

    {
        "Error_Type":
            "FalsePositive",

        "Method":
            "CEH_MoE",

        "Event_Prevalence":
            fp_prev,

        "PR_AUC":
            fp_full_pr,

        "ROC_AUC":
            fp_full_roc,
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 12C — CONDITIONAL ERROR DETECTION"
)

print(
    "=" * 135
)


display(
    conditional_detection
)


# ============================================================
# 12C.11 CALIBRATION PERFORMANCE
# ============================================================

calibration_rows = []


for (
    error_name,
    mask,
    target,
    method,
    prob
) in [

    (
        "FN",
        negative_mask,
        fn_target,
        "BranchConfidence",
        CEH_FN_CONF_OOF
    ),

    (
        "FN",
        negative_mask,
        fn_target,
        "CEH_MoE",
        CEH_FN_FULL_OOF
    ),

    (
        "FP",
        positive_mask,
        fp_target,
        "BranchConfidence",
        CEH_FP_CONF_OOF
    ),

    (
        "FP",
        positive_mask,
        fp_target,
        "CEH_MoE",
        CEH_FP_FULL_OOF
    ),
]:

    y_eval = (
        target[
            mask
        ]
    )


    p_eval = np.clip(

        prob[
            mask
        ],

        1e-8,
        1.0 - 1e-8
    )


    calibration_rows.append({

        "Error_Type":
            error_name,

        "Method":
            method,

        "Observed_Rate":
            float(
                y_eval.mean()
            ),

        "Mean_Predicted_Risk":
            float(
                p_eval.mean()
            ),

        "Brier":
            brier_score_loss(
                y_eval,
                p_eval
            ),

        "NLL":
            log_loss(
                y_eval,
                p_eval
            ),

        "ECE":
            ceh_ece(

                y_eval,

                p_eval,

                n_bins=
                    CEH_CONFIG[
                        "ece_bins"
                    ]
            ),
    })


calibration_results = pd.DataFrame(
    calibration_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "CONDITIONAL RISK CALIBRATION"
)

print(
    "=" * 135
)


display(
    calibration_results
)


# ============================================================
# 12C.12 EXPECTED HARM
# ============================================================

def build_expected_harm(
    prediction,
    fn_probability,
    fp_probability,
    fn_cost,
    fp_cost=1.0
):

    prediction = np.asarray(
        prediction,
        dtype=int
    )


    risk = np.zeros(
        len(
            prediction
        ),
        dtype=float
    )


    neg = (
        prediction
        ==
        0
    )


    pos = (
        prediction
        ==
        1
    )


    risk[
        neg
    ] = (

        float(
            fn_cost
        )

        *
        fn_probability[
            neg
        ]
    )


    risk[
        pos
    ] = (

        float(
            fp_cost
        )

        *
        fp_probability[
            pos
        ]
    )


    return risk


# ============================================================
# 12C.13 SELECTIVE EVALUATION
# ============================================================

def evaluate_ceh_deferral(
    risk_score,
    method,
    fn_cost
):

    rows = []


    total_errors = max(

        int(
            error_target.sum()
        ),

        1
    )


    total_fn = max(

        int(
            fn_target.sum()
        ),

        1
    )


    total_fp = max(

        int(
            fp_target.sum()
        ),

        1
    )


    base_error = (
        error_target.mean()
    )


    order = np.argsort(
        -risk_score
    )


    for rate in (
        CEH_CONFIG[
            "deferral_rates"
        ]
    ):

        n_defer = max(

            1,

            int(
                round(
                    N
                    *
                    rate
                )
            )
        )


        defer_idx = (
            order[
                :n_defer
            ]
        )


        keep = np.ones(
            N,
            dtype=bool
        )


        keep[
            defer_idx
        ] = False


        deferred_error = (

            error_target[
                defer_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "FN_Cost":
                fn_cost,

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0
                -
                rate,

            "Deferred_N":
                n_defer,

            "Deferred_Error_Rate":
                deferred_error,

            "Error_Enrichment":
                (
                    deferred_error
                    /
                    max(
                        base_error,
                        1e-12
                    )
                ),

            "Errors_Captured":
                (
                    error_target[
                        defer_idx
                    ].sum()
                    /
                    total_errors
                ),

            "FalseNegatives_Captured":
                (
                    fn_target[
                        defer_idx
                    ].sum()
                    /
                    total_fn
                ),

            "FalsePositives_Captured":
                (
                    fp_target[
                        defer_idx
                    ].sum()
                    /
                    total_fp
                ),

            "FN_Found":
                int(
                    fn_target[
                        defer_idx
                    ].sum()
                ),

            "FP_Found":
                int(
                    fp_target[
                        defer_idx
                    ].sum()
                ),

            "FN_Per_100_Reviews":
                (
                    fn_target[
                        defer_idx
                    ].sum()
                    /
                    n_defer
                    *
                    100.0
                ),

            "Error_Per_100_Reviews":
                (
                    error_target[
                        defer_idx
                    ].sum()
                    /
                    n_defer
                    *
                    100.0
                ),

            "Retained_Accuracy":
                accuracy_score(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


all_deferral_frames = []

CEH_HARM_SCORES = {}


for fn_cost in CEH_CONFIG["fn_costs"]:

    # ========================================================
    # Branch confidence expected harm
    # ========================================================

    confidence_harm = (
        build_expected_harm(
            prediction,
            CEH_FN_CONF_OOF,
            CEH_FP_CONF_OOF,
            fn_cost=fn_cost,
            fp_cost=1.0
        )
    )


    # ========================================================
    # Full CEH-MoE expected harm
    # ========================================================

    ceh_harm = (
        build_expected_harm(
            prediction,
            CEH_FN_FULL_OOF,
            CEH_FP_FULL_OOF,
            fn_cost=fn_cost,
            fp_cost=1.0
        )
    )


    CEH_HARM_SCORES[
        ("BranchConfidence", fn_cost)
    ] = confidence_harm


    CEH_HARM_SCORES[
        ("CEH_MoE", fn_cost)
    ] = ceh_harm


    conf_df = (
        evaluate_ceh_deferral(
            confidence_harm,
            method="Calibrated_BranchConfidence",
            fn_cost=fn_cost
        )
    )


    ceh_df = (
        evaluate_ceh_deferral(
            ceh_harm,
            method="CEH_MoE",
            fn_cost=fn_cost
        )
    )


    all_deferral_frames.extend(
        [
            conf_df,
            ceh_df,
        ]
    )


cost_sensitivity = pd.concat(
    all_deferral_frames,
    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "CALIBRATED EXPECTED-HARM COST SENSITIVITY"
)

print(
    "=" * 135
)


display(

    cost_sensitivity[
        [
            "Method",
            "FN_Cost",
            "Deferral_Rate",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "FalsePositives_Captured",
            "Error_Enrichment",
            "FN_Per_100_Reviews",
            "Error_Per_100_Reviews",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 12C.15 PRIMARY POLICY
# ============================================================

primary_cost = (
    CEH_CONFIG[
        "primary_fn_cost"
    ]
)


primary_selective = (

    cost_sensitivity[

        cost_sensitivity[
            "FN_Cost"
        ]
        ==
        primary_cost

    ]
    .copy()
)


# Add generic RA-MoE error-risk baseline from Module 12A

generic_df = (
    evaluate_ceh_deferral(

        RAMOE_RISK_FULL_OOF,

        method=
            "Generic_RA_MoE",

        fn_cost=
            np.nan
    )
)


primary_selective = pd.concat(

    [
        generic_df,
        primary_selective,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    f"PRIMARY SELECTIVE POLICY — FN COST = "
    f"{primary_cost:g}"
)

print(
    "=" * 135
)


display(

    primary_selective[
        [
            "Method",
            "FN_Cost",
            "Deferral_Rate",
            "Coverage",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "FalsePositives_Captured",
            "Error_Enrichment",
            "FN_Per_100_Reviews",
            "Error_Per_100_Reviews",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 12C.16 CRITICAL FN-ONLY BENCHMARK
# ============================================================
#
# This is crucial.
#
# Among patients ALREADY classified negative, can CEH-MoE
# find hidden false negatives better than simply sorting by
# disease probability?
#
# ============================================================

negative_indices = np.where(
    negative_mask
)[0]


n_negative = len(
    negative_indices
)


total_fn_negative = max(

    fn_target[
        negative_indices
    ].sum(),

    1
)


fn_review_rows = []


fn_review_methods = {

    "Raw_MoE_Probability":
        moe_prob,

    "Calibrated_BranchConfidence":
        CEH_FN_CONF_OOF,

    "CEH_MoE":
        CEH_FN_FULL_OOF,
}


for method_name, score in (
    fn_review_methods.items()
):

    ranked_negative = (

        negative_indices[
            np.argsort(
                -score[
                    negative_indices
                ]
            )
        ]
    )


    for rate in (
        CEH_CONFIG[
            "negative_review_rates"
        ]
    ):

        n_review = max(

            1,

            int(
                round(
                    n_negative
                    *
                    rate
                )
            )
        )


        review_idx = (
            ranked_negative[
                :n_review
            ]
        )


        fn_found = (

            fn_target[
                review_idx
            ].sum()
        )


        fn_review_rows.append({

            "Method":
                method_name,

            "Negative_Review_Rate":
                rate,

            "Reviewed_N":
                n_review,

            "FN_Found":
                int(
                    fn_found
                ),

            "FN_Capture":
                (
                    fn_found
                    /
                    total_fn_negative
                ),

            "FN_Per_100_Reviews":
                (
                    fn_found
                    /
                    n_review
                    *
                    100.0
                ),
        })


fn_only_review = pd.DataFrame(
    fn_review_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FALSE-NEGATIVE RECOVERY AMONG PREDICTED-NEGATIVE CASES"
)

print(
    "=" * 135
)


display(
    fn_only_review
)


# ============================================================
# 12C.17 FN LIFT OVER RAW MoE PROBABILITY
# ============================================================

fn_pivot_capture = (
    fn_only_review
    .pivot(

        index=
            "Negative_Review_Rate",

        columns=
            "Method",

        values=
            "FN_Capture"
    )
    .reset_index()
)


fn_pivot_capture[
    "CEH_Gain_vs_Raw"
] = (

    fn_pivot_capture[
        "CEH_MoE"
    ]

    -
    fn_pivot_capture[
        "Raw_MoE_Probability"
    ]
)


fn_pivot_capture[
    "CEH_Gain_vs_BranchConfidence"
] = (

    fn_pivot_capture[
        "CEH_MoE"
    ]

    -
    fn_pivot_capture[
        "Calibrated_BranchConfidence"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "FN RECOVERY GAIN"
)

print(
    "=" * 135
)


display(
    fn_pivot_capture
)


# ============================================================
# 12C.18 FOLD STABILITY
# ============================================================

fold_results = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "CEH-MoE FOLD RESULTS"
)

print(
    "=" * 135
)


display(
    fold_results
)


print(
    "\nFN fold stability:"
)


print(
    "  Mean Confidence PR:",
    fold_results[
        "FN_Confidence_PR"
    ].mean()
)


print(
    "  Mean CEH PR:",
    fold_results[
        "FN_CEH_PR"
    ].mean()
)


print(
    "  Mean Delta:",
    fold_results[
        "FN_Delta_PR"
    ].mean()
)


print(
    "  Positive folds:",
    int(
        (
            fold_results[
                "FN_Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "\nFP fold stability:"
)


print(
    "  Mean Confidence PR:",
    fold_results[
        "FP_Confidence_PR"
    ].mean()
)


print(
    "  Mean CEH PR:",
    fold_results[
        "FP_CEH_PR"
    ].mean()
)


print(
    "  Mean Delta:",
    fold_results[
        "FP_Delta_PR"
    ].mean()
)


print(
    "  Positive folds:",
    int(
        (
            fold_results[
                "FP_Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 12C.19 SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 12C SUMMARY"
)

print(
    "=" * 135
)


print(
    "\nFALSE-NEGATIVE DETECTION"
)


print(
    f"  Event prevalence       : "
    f"{fn_prev:.6f}"
)


print(
    f"  Raw MoE probability PR : "
    f"{fn_raw_prob_pr:.6f}"
)


print(
    f"  Branch confidence PR   : "
    f"{fn_conf_pr:.6f}"
)


print(
    f"  CEH-MoE PR             : "
    f"{fn_full_pr:.6f}"
)


print(
    f"  CEH Δ vs raw           : "
    f"{fn_full_pr-fn_raw_prob_pr:+.6f}"
)


print(
    f"  CEH Δ vs calibrated conf: "
    f"{fn_full_pr-fn_conf_pr:+.6f}"
)


print(
    "\nFALSE-POSITIVE DETECTION"
)


print(
    f"  Event prevalence       : "
    f"{fp_prev:.6f}"
)


print(
    f"  Raw MoE confidence PR  : "
    f"{fp_raw_prob_pr:.6f}"
)


print(
    f"  Branch confidence PR   : "
    f"{fp_conf_pr:.6f}"
)


print(
    f"  CEH-MoE PR             : "
    f"{fp_full_pr:.6f}"
)


print(
    "\nProbability calibration:"
)


display(
    calibration_results
)


# ============================================================
# 12C.20 SAVE
# ============================================================

save_csv(

    conditional_detection,

    TAB_DIR
    / "module12c_conditional_error_detection.csv"
)


save_csv(

    calibration_results,

    TAB_DIR
    / "module12c_conditional_calibration.csv"
)


save_csv(

    cost_sensitivity,

    TAB_DIR
    / "module12c_expected_harm_cost_sensitivity.csv"
)


save_csv(

    primary_selective,

    TAB_DIR
    / "module12c_primary_selective_policy.csv"
)


save_csv(

    fn_only_review,

    TAB_DIR
    / "module12c_fn_only_review.csv"
)


save_csv(

    fn_pivot_capture,

    TAB_DIR
    / "module12c_fn_recovery_gain.csv"
)


save_csv(

    fold_results,

    TAB_DIR
    / "module12c_fold_results.csv"
)


# ============================================================
# SAVE OOF OUTPUTS
# ============================================================

prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "MoE_Probability":
        moe_prob,

    "DecisionThreshold":
        decision_threshold,

    "Prediction":
        prediction,

    "FalseNegative":
        fn_target,

    "FalsePositive":
        fp_target,

    "AnyError":
        error_target,

    "FN_Probability_BranchConfidence":
        CEH_FN_CONF_OOF,

    "FN_Probability_CEH":
        CEH_FN_FULL_OOF,

    "FP_Probability_BranchConfidence":
        CEH_FP_CONF_OOF,

    "FP_Probability_CEH":
        CEH_FP_FULL_OOF,
})


for fn_cost in CEH_CONFIG["fn_costs"]:

    prediction_df[
        f"ExpectedHarm_CEH_FNcost_{fn_cost:g}"
    ] = CEH_HARM_SCORES[
        ("CEH_MoE", fn_cost)
    ]


save_csv(

    prediction_df,

    PRED_DIR
    / "module12c_ceh_moe_oof.csv"
)


# ============================================================
# MANIFEST
# ============================================================

module12c_manifest = {

    "method":
        "Calibrated Expected-Harm Selective Mixture-of-Experts",

    "main_predictor":
        "RA-MoE",

    "conditional_error_heads": [
        "false-negative",
        "false-positive",
    ],

    "head_training":
        "class-balanced logistic discrimination",

    "probability_calibration":
        "fold-local held-out Platt calibration",

    "calibration_fraction":
        CEH_CONFIG[
            "calibration_fraction"
        ],

    "expected_harm":
        "C_FN * calibrated P(FN) or C_FP * calibrated P(FP)",

    "fn_cost_ratios":
        CEH_CONFIG[
            "fn_costs"
        ],

    "primary_fn_cost":
        CEH_CONFIG[
            "primary_fn_cost"
        ],

    "critical_baseline":
        "raw MoE probability among predicted-negative cases",

    "safety_predictions_cross_fitted":
        True,

    "holdout_used":
        False,
}


save_json(

    module12c_manifest,

    LOG_DIR
    / "module12c_manifest.json"
)


print(
    "\nMODULE 12C COMPLETE."
)


CEH-MoE OUTER FOLD 1/5

Predicted-negative branch:
  Fit N: 110392 | FN: 5584
  Calibration N: 27595 | FN: 1396
  Validation N: 33537 | FN: 1525

Predicted-positive branch:
  Fit N: 19492 | FP: 12846
  Calibration N: 4876 | FP: 3211
  Validation N: 7052 | FP: 4754

FN detection:
  Confidence PR : 0.142713
  CEH-MoE PR    : 0.136139
  Delta         : -0.006574

FP detection:
  Confidence PR : 0.755822
  CEH-MoE PR    : 0.747982
  Delta         : -0.007840

CEH-MoE OUTER FOLD 2/5

Predicted-negative branch:
  Fit N: 108686 | FN: 5194
  Calibration N: 27124 | FN: 1298
  Validation N: 35714 | FN: 2013

Predicted-positive branch:
  Fit N: 21198 | FP: 14197
  Calibration N: 5347 | FP: 3549
  Validation N: 4875 | FP: 3065

FN detection:
  Confidence PR : 0.165690
  CEH-MoE PR    : 0.171840
  Delta         : +0.006150

FP detection:
  Confidence PR : 0.705762
  CEH-MoE PR    : 0.702614
  Delta         : -0.003148

CEH-MoE OUTER FOLD 3/5

Predicted-negative branch:
  Fit N: 110128 | FN: 5528
 

,Error_Type,Method,Event_Prevalence,PR_AUC,ROC_AUC
0,FalseNegative,Raw_MoE_Probability,0.049585,0.156575,0.805984
1,FalseNegative,Calibrated_BranchConfidence,0.049585,0.146945,0.803629
2,FalseNegative,CEH_MoE,0.049585,0.149055,0.804049
3,FalsePositive,Raw_MoE_Confidence,0.662349,0.751764,0.640813
4,FalsePositive,Calibrated_BranchConfidence,0.662349,0.751566,0.640531
5,FalsePositive,CEH_MoE,0.662349,0.748180,0.637387



CONDITIONAL RISK CALIBRATION


,Error_Type,Method,Observed_Rate,Mean_Predicted_Risk,Brier,NLL,ECE
0,FN,BranchConfidence,0.049585,0.048725,0.044108,0.167913,0.001465
1,FN,CEH_MoE,0.049585,0.049937,0.044118,0.167660,0.001327
2,FP,BranchConfidence,0.662349,0.663959,0.210265,0.610038,0.010114
3,FP,CEH_MoE,0.662349,0.662333,0.210789,0.611258,0.008687



CALIBRATED EXPECTED-HARM COST SENSITIVITY


...


CEH-MoE FOLD RESULTS


,Fold,FN_Prevalence,FN_Confidence_PR,FN_CEH_PR,FN_Delta_PR,FN_Confidence_ROC,FN_CEH_ROC,FN_Delta_ROC,FP_Prevalence,FP_Confidence_PR,FP_CEH_PR,FP_Delta_PR,FP_Confidence_ROC,FP_CEH_ROC,FP_Delta_ROC
0,0,0.045472,0.142713,0.136139,-0.006574,0.805552,0.802600,-0.002952,0.674135,0.755822,0.747982,-0.007840,0.636601,0.630401,-0.006200
1,1,0.056364,0.165690,0.171840,0.006150,0.808886,0.810535,0.001649,0.628718,0.705762,0.702614,-0.003148,0.621011,0.619165,-0.001846
2,2,0.047170,0.140099,0.135652,-0.004447,0.798629,0.797766,-0.000863,0.671144,0.756783,0.757047,0.000264,0.640922,0.639245,-0.001677
3,3,0.048530,0.149128,0.145975,-0.003152,0.803038,0.801020,-0.002019,0.669356,0.759718,0.755860,-0.003859,0.641133,0.638298,-0.002835
4,4,0.049972,0.149573,0.151363,0.001790,0.805722,0.805434,-0.000288,0.658318,0.761802,0.757521,-0.004281,0.652691,0.651421,-0.001270



FN fold stability:
  Mean Confidence PR: 0.14944053662218665
  Mean CEH PR: 0.14819388257785793
  Mean Delta: -0.0012466540443287378
  Positive folds: 2 / 5

FP fold stability:
  Mean Confidence PR: 0.7479773705718201
  Mean CEH PR: 0.7442046117856782
  Mean Delta: -0.0037727587861418055
  Positive folds: 1 / 5

MODULE 12C SUMMARY

FALSE-NEGATIVE DETECTION
  Event prevalence       : 0.049585
  Raw MoE probability PR : 0.156575
  Branch confidence PR   : 0.146945
  CEH-MoE PR             : 0.149055
  CEH Δ vs raw           : -0.007520
  CEH Δ vs calibrated conf: +0.002110

FALSE-POSITIVE DETECTION
  Event prevalence       : 0.662349
  Raw MoE confidence PR  : 0.751764
  Branch confidence PR   : 0.751566
  CEH-MoE PR             : 0.748180

Probability calibration:


,Error_Type,Method,Observed_Rate,Mean_Predicted_Risk,Brier,NLL,ECE
0,FN,BranchConfidence,0.049585,0.048725,0.044108,0.167913,0.001465
1,FN,CEH_MoE,0.049585,0.049937,0.044118,0.167660,0.001327
2,FP,BranchConfidence,0.662349,0.663959,0.210265,0.610038,0.010114
3,FP,CEH_MoE,0.662349,0.662333,0.210789,0.611258,0.008687



MODULE 12C COMPLETE.


In [21]:
# ============================================================
# MODULE 13A — PAESP
# Perturbation-Aware Expert Stability Profiling
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Before building a robustness-aware MoE gate, determine whether
# the four experts exhibit heterogeneous local stability.
#
# We DO NOT touch HOLDOUT.
#
# We use:
#
#   HGB
#   CatBoost
#   AutoInt
#   DeepFM_v2
#
# Existing strict OOF expert probabilities from Module 12B:
#
#   ETR_EXPERT_OOF
#
#
# WHY NEIGHBORHOOD PERTURBATION?
# ------------------------------------------------------------
# Instead of inventing medically impossible values, we construct
# local empirical perturbation neighborhoods from real DEV cases.
#
# For patient i:
#
#   1. standardize continuous/ordinal representation
#   2. retrieve nearby real patients
#   3. keep only sufficiently close neighbors
#   4. study expert prediction variation inside the neighborhood
#
#
# For each expert j, compute:
#
#   Local SD
#   Local MAD
#   Local range
#   Rank instability
#   Decision-flip rate
#   Mean absolute probability shift
#
#
# This is a SCREENING module.
#
# If experts show similar instability everywhere -> STOP.
#
# If instability is patient-specific and expert-specific ->
# proceed to robustness-aware routing.
#
# ============================================================


import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from scipy.stats import spearmanr


# ============================================================
# 13A.1 REQUIREMENTS
# ============================================================

required_13a = [
    "ETR_EXPERT_OOF",
    "RAMOE_EXPERTS",
    "RAMOE_PROB_OOF",
    "RAMOE_THRESHOLD_OOF",
    "fold_assignment",
    "y_dev",
    "X_dev",
]

missing = [
    x for x in required_13a
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Run Modules 12A and 12B first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 13A.2 CONFIG
# ============================================================

PAESP_CONFIG = {

    # number of empirical neighboring profiles examined
    "k_neighbors":
        40,

    # after retrieval, use top N closest
    "k_stability":
        20,

    # maximum standardized distance allowed
    "max_distance":
        3.0,

    # batch size for NN querying
    "query_batch":
        10000,

    # minimum number of valid neighbors required
    "min_neighbors":
        8,

    # thresholds used for robustness summaries
    "high_instability_quantile":
        0.90,
}


# ============================================================
# 13A.3 FEATURE GROUPS
# ============================================================

BINARY_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]


ORDINAL_FEATURES = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]


COUNT_FEATURES = [
    "MentHlth",
    "PhysHlth",
]


CONTINUOUS_FEATURES = [
    "BMI",
]


MULTICLASS_FEATURES = [
    "Diabetes",
]


ALL_FEATURES = (
    BINARY_FEATURES
    +
    ORDINAL_FEATURES
    +
    COUNT_FEATURES
    +
    CONTINUOUS_FEATURES
    +
    MULTICLASS_FEATURES
)


# ============================================================
# 13A.4 DATA
# ============================================================

X = (
    X_dev[
        ALL_FEATURES
    ]
    .copy()
)


y = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


expert_prob = np.asarray(
    ETR_EXPERT_OOF,
    dtype=float
)


moe_prob = np.asarray(
    RAMOE_PROB_OOF,
    dtype=float
)


threshold = np.asarray(
    RAMOE_THRESHOLD_OOF,
    dtype=float
)


N = len(X)


print(
    "N:",
    N
)

print(
    "Expert matrix:",
    expert_prob.shape
)


# ============================================================
# 13A.5 ROBUST DISTANCE REPRESENTATION
# ============================================================
#
# Continuous/count/ordinal:
#   standardized
#
# Binary/multiclass:
#   left as discrete indicator values
#
# We downweight binary dimensions slightly so Hamming-like
# mismatches do not overwhelm BMI / age / health-state distance.
#
# ============================================================

X_distance = X.copy()


scale_features = (
    CONTINUOUS_FEATURES
    +
    COUNT_FEATURES
    +
    ORDINAL_FEATURES
)


scaler = StandardScaler()


X_distance[
    scale_features
] = scaler.fit_transform(

    X_distance[
        scale_features
    ]
)


# moderate weighting for discrete variables
for c in (
    BINARY_FEATURES
    +
    MULTICLASS_FEATURES
):

    X_distance[
        c
    ] = (

        X_distance[
            c
        ].astype(float)

        *
        0.65
    )


X_distance_np = (
    X_distance
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 13A.6 FIT EMPIRICAL NEIGHBORHOOD MODEL
# ============================================================

print(
    "\nFitting nearest-neighbor index..."
)


nn = NearestNeighbors(

    n_neighbors=
        PAESP_CONFIG[
            "k_neighbors"
        ]
        + 1,

    algorithm="auto",

    metric="euclidean",

    n_jobs=-1,
)


nn.fit(
    X_distance_np
)


# ============================================================
# 13A.7 QUERY NEIGHBORS
# ============================================================

all_neighbor_idx = np.zeros(

    (
        N,
        PAESP_CONFIG[
            "k_neighbors"
        ]
    ),

    dtype=np.int32
)


all_neighbor_dist = np.zeros(

    (
        N,
        PAESP_CONFIG[
            "k_neighbors"
        ]
    ),

    dtype=np.float32
)


print(
    "Querying neighborhoods..."
)


batch_size = (
    PAESP_CONFIG[
        "query_batch"
    ]
)


for start in range(
    0,
    N,
    batch_size
):

    end = min(
        start + batch_size,
        N
    )


    distances, indices = (
        nn.kneighbors(
            X_distance_np[
                start:end
            ],
            return_distance=True
        )
    )


    # remove self-neighbor at position 0
    distances = distances[
        :,
        1:
    ]


    indices = indices[
        :,
        1:
    ]


    all_neighbor_idx[
        start:end
    ] = indices


    all_neighbor_dist[
        start:end
    ] = distances


    print(
        f"  processed {end:,}/{N:,}"
    )


# ============================================================
# 13A.8 LOCAL STABILITY BUFFERS
# ============================================================

n_experts = len(
    RAMOE_EXPERTS
)


LOCAL_SD = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


LOCAL_MAD = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


LOCAL_RANGE = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


LOCAL_SHIFT = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


LOCAL_FLIP = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


LOCAL_RANK_INSTABILITY = np.full(
    (N, n_experts),
    np.nan,
    dtype=float
)


VALID_NEIGHBOR_COUNT = np.zeros(
    N,
    dtype=int
)


# ============================================================
# 13A.9 LOCAL ROBUSTNESS PROFILING
# ============================================================

print(
    "\nComputing local expert stability..."
)


for i in range(
    N
):

    idx = all_neighbor_idx[
        i
    ]


    dist = all_neighbor_dist[
        i
    ]


    valid = (

        dist
        <=
        PAESP_CONFIG[
            "max_distance"
        ]
    )


    idx = idx[
        valid
    ]


    dist = dist[
        valid
    ]


    if len(idx) > (
        PAESP_CONFIG[
            "k_stability"
        ]
    ):

        idx = idx[
            :
            PAESP_CONFIG[
                "k_stability"
            ]
        ]


    VALID_NEIGHBOR_COUNT[
        i
    ] = len(
        idx
    )


    if len(idx) < (
        PAESP_CONFIG[
            "min_neighbors"
        ]
    ):

        continue


    center_prob = (
        expert_prob[
            i
        ]
    )


    neighbor_prob = (
        expert_prob[
            idx
        ]
    )


    # --------------------------------------------------------
    # Local SD
    # --------------------------------------------------------

    LOCAL_SD[
        i
    ] = np.std(

        neighbor_prob,

        axis=0
    )


    # --------------------------------------------------------
    # Local mean absolute deviation around center prediction
    # --------------------------------------------------------

    LOCAL_MAD[
        i
    ] = np.mean(

        np.abs(
            neighbor_prob
            -
            center_prob[
                None,
                :
            ]
        ),

        axis=0
    )


    # --------------------------------------------------------
    # Local range
    # --------------------------------------------------------

    LOCAL_RANGE[
        i
    ] = (

        np.max(
            neighbor_prob,
            axis=0
        )

        -
        np.min(
            neighbor_prob,
            axis=0
        )
    )


    # --------------------------------------------------------
    # Mean probability shift
    # --------------------------------------------------------

    LOCAL_SHIFT[
        i
    ] = np.mean(

        neighbor_prob
        -
        center_prob[
            None,
            :
        ],

        axis=0
    )


    # --------------------------------------------------------
    # Decision-flip probability
    #
    # Use patient's fold-specific MoE threshold as common
    # decision boundary for expert stability.
    # --------------------------------------------------------

    t = threshold[
        i
    ]


    center_decision = (

        center_prob
        >=
        t
    )


    neighbor_decision = (

        neighbor_prob
        >=
        t
    )


    LOCAL_FLIP[
        i
    ] = np.mean(

        neighbor_decision
        !=
        center_decision[
            None,
            :
        ],

        axis=0
    )


    # --------------------------------------------------------
    # Rank instability
    #
    # Compare expert ordering at patient i with expert ordering
    # across nearby patients.
    #
    # 0 = same expert ordering
    # 1 = very different
    # --------------------------------------------------------

    center_rank = (
        pd.Series(
            center_prob
        )
        .rank(
            method="average"
        )
        .to_numpy()
    )


    local_rank_instability = []


    for row in neighbor_prob:

        neighbor_rank = (
            pd.Series(
                row
            )
            .rank(
                method="average"
            )
            .to_numpy()
        )


        rho = spearmanr(

            center_rank,

            neighbor_rank
        ).statistic


        if np.isnan(
            rho
        ):

            rho = 1.0


        local_rank_instability.append(

            1.0
            -
            rho
        )


    # patient-level rank instability is common across experts,
    # but we distribute it according to each expert's local MAD
    # contribution so we obtain an expert-specific signal.

    global_rank_instability = np.mean(
        local_rank_instability
    )


    mad_sum = (
        LOCAL_MAD[
            i
        ].sum()
        +
        1e-12
    )


    contribution = (

        LOCAL_MAD[
            i
        ]

        /
        mad_sum
    )


    LOCAL_RANK_INSTABILITY[
        i
    ] = (

        global_rank_instability

        *
        contribution

        *
        n_experts
    )


    if (
        i + 1
    ) % 25000 == 0:

        print(
            f"  profiled {i+1:,}/{N:,}"
        )


# ============================================================
# 13A.10 VALID CASES
# ============================================================

valid_profile = (

    VALID_NEIGHBOR_COUNT
    >=
    PAESP_CONFIG[
        "min_neighbors"
    ]
)


print(
    "\nValid local profiles:",
    valid_profile.sum(),
    "/",
    N
)


print(
    "Coverage:",
    valid_profile.mean()
)


# ============================================================
# 13A.11 EXPERT-LEVEL ROBUSTNESS SUMMARY
# ============================================================

expert_rows = []


for j, name in enumerate(
    RAMOE_EXPERTS
):

    mask = (
        valid_profile

        &
        ~np.isnan(
            LOCAL_MAD[
                :,
                j
            ]
        )
    )


    expert_rows.append({

        "Expert":
            name,

        "N":
            int(
                mask.sum()
            ),

        "Mean_Local_SD":
            float(
                np.mean(
                    LOCAL_SD[
                        mask,
                        j
                    ]
                )
            ),

        "Median_Local_SD":
            float(
                np.median(
                    LOCAL_SD[
                        mask,
                        j
                    ]
                )
            ),

        "Mean_Local_MAD":
            float(
                np.mean(
                    LOCAL_MAD[
                        mask,
                        j
                    ]
                )
            ),

        "Median_Local_MAD":
            float(
                np.median(
                    LOCAL_MAD[
                        mask,
                        j
                    ]
                )
            ),

        "Mean_Local_Range":
            float(
                np.mean(
                    LOCAL_RANGE[
                        mask,
                        j
                    ]
                )
            ),

        "Mean_Flip_Rate":
            float(
                np.mean(
                    LOCAL_FLIP[
                        mask,
                        j
                    ]
                )
            ),

        "Mean_RankInstability":
            float(
                np.mean(
                    LOCAL_RANK_INSTABILITY[
                        mask,
                        j
                    ]
                )
            ),
    })


expert_stability_summary = (
    pd.DataFrame(
        expert_rows
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 13A — EXPERT LOCAL STABILITY"
)

print(
    "=" * 135
)


display(
    expert_stability_summary
)


# ============================================================
# 13A.12 PATIENT-SPECIFIC MOST STABLE EXPERT
# ============================================================

stability_matrix = (
    LOCAL_MAD.copy()
)


# Lower MAD = more stable
most_stable_expert_idx = np.full(
    N,
    -1,
    dtype=int
)


most_unstable_expert_idx = np.full(
    N,
    -1,
    dtype=int
)


for i in np.where(
    valid_profile
)[0]:

    most_stable_expert_idx[
        i
    ] = int(
        np.nanargmin(
            stability_matrix[
                i
            ]
        )
    )


    most_unstable_expert_idx[
        i
    ] = int(
        np.nanargmax(
            stability_matrix[
                i
            ]
        )
    )


stable_counts = []


for j, name in enumerate(
    RAMOE_EXPERTS
):

    stable_counts.append({

        "Expert":
            name,

        "MostStable_N":
            int(
                np.sum(
                    most_stable_expert_idx
                    ==
                    j
                )
            ),

        "MostStable_Fraction":
            float(
                np.mean(
                    most_stable_expert_idx[
                        valid_profile
                    ]
                    ==
                    j
                )
            ),

        "MostUnstable_N":
            int(
                np.sum(
                    most_unstable_expert_idx
                    ==
                    j
                )
            ),

        "MostUnstable_Fraction":
            float(
                np.mean(
                    most_unstable_expert_idx[
                        valid_profile
                    ]
                    ==
                    j
                )
            ),
    })


stable_expert_distribution = (
    pd.DataFrame(
        stable_counts
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "PATIENT-SPECIFIC STABILITY DISTRIBUTION"
)

print(
    "=" * 135
)


display(
    stable_expert_distribution
)


# ============================================================
# 13A.13 STABILITY HETEROGENEITY
# ============================================================
#
# For each patient:
#
#     max instability - min instability
#
# High values mean expert stability differs materially.
# ============================================================

patient_mean_mad = np.full(
    N,
    np.nan
)


patient_stability_gap = np.full(
    N,
    np.nan
)


patient_mad_cv = np.full(
    N,
    np.nan
)


for i in np.where(
    valid_profile
)[0]:

    v = LOCAL_MAD[
        i
    ]


    patient_mean_mad[
        i
    ] = np.mean(
        v
    )


    patient_stability_gap[
        i
    ] = (

        np.max(
            v
        )

        -
        np.min(
            v
        )
    )


    patient_mad_cv[
        i
    ] = (

        np.std(
            v
        )

        /
        (
            np.mean(
                v
            )
            +
            1e-12
        )
    )


heterogeneity_summary = pd.DataFrame({

    "Metric": [

        "Mean expert MAD",

        "Median expert MAD",

        "Mean stability gap",

        "Median stability gap",

        "Mean MAD coefficient of variation",

        "Median MAD coefficient of variation",
    ],

    "Value": [

        np.nanmean(
            patient_mean_mad
        ),

        np.nanmedian(
            patient_mean_mad
        ),

        np.nanmean(
            patient_stability_gap
        ),

        np.nanmedian(
            patient_stability_gap
        ),

        np.nanmean(
            patient_mad_cv
        ),

        np.nanmedian(
            patient_mad_cv
        ),
    ]
})


print(
    "\n"
    + "=" * 135
)

print(
    "EXPERT STABILITY HETEROGENEITY"
)

print(
    "=" * 135
)


display(
    heterogeneity_summary
)


# ============================================================
# 13A.14 RELATION WITH MoE ERROR
# ============================================================
#
# Does local instability predict actual model failure?
#
# ============================================================

moe_prediction = (

    moe_prob
    >=
    threshold

).astype(int)


moe_error = (

    moe_prediction
    !=
    y

).astype(int)


analysis_mask = (
    valid_profile
)


stability_vs_error_rows = []


for j, name in enumerate(
    RAMOE_EXPERTS
):

    score = (
        LOCAL_MAD[
            analysis_mask,
            j
        ]
    )


    err = (
        moe_error[
            analysis_mask
        ]
    )


    from sklearn.metrics import (
        average_precision_score,
        roc_auc_score,
    )


    stability_vs_error_rows.append({

        "Expert":
            name,

        "Error_PR_AUC_from_Instability":
            average_precision_score(
                err,
                score
            ),

        "Error_ROC_AUC_from_Instability":
            roc_auc_score(
                err,
                score
            ),
    })


# aggregate instability
aggregate_instability = (
    np.nanmean(
        LOCAL_MAD,
        axis=1
    )
)


max_instability = (
    np.nanmax(
        LOCAL_MAD,
        axis=1
    )
)


stability_vs_error_rows.extend([

    {
        "Expert":
            "MeanAcrossExperts",

        "Error_PR_AUC_from_Instability":
            average_precision_score(
                moe_error[
                    analysis_mask
                ],
                aggregate_instability[
                    analysis_mask
                ]
            ),

        "Error_ROC_AUC_from_Instability":
            roc_auc_score(
                moe_error[
                    analysis_mask
                ],
                aggregate_instability[
                    analysis_mask
                ]
            ),
    },

    {
        "Expert":
            "MaxExpertInstability",

        "Error_PR_AUC_from_Instability":
            average_precision_score(
                moe_error[
                    analysis_mask
                ],
                max_instability[
                    analysis_mask
                ]
            ),

        "Error_ROC_AUC_from_Instability":
            roc_auc_score(
                moe_error[
                    analysis_mask
                ],
                max_instability[
                    analysis_mask
                ]
            ),
    },
])


stability_error_results = pd.DataFrame(
    stability_vs_error_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "LOCAL INSTABILITY AS ERROR SIGNAL"
)

print(
    "=" * 135
)


display(
    stability_error_results
)


# ============================================================
# 13A.15 HIGH-INSTABILITY PATIENT ANALYSIS
# ============================================================

cutoff = np.nanquantile(

    aggregate_instability[
        valid_profile
    ],

    PAESP_CONFIG[
        "high_instability_quantile"
    ]
)


high_instability = (

    valid_profile

    &
    (
        aggregate_instability
        >=
        cutoff
    )
)


low_instability = (

    valid_profile

    &
    (
        aggregate_instability
        <
        cutoff
    )
)


instability_group_results = pd.DataFrame([

    {
        "Group":
            "HighInstability",

        "N":
            int(
                high_instability.sum()
            ),

        "ErrorRate":
            float(
                moe_error[
                    high_instability
                ].mean()
            ),

        "PositiveRate":
            float(
                y[
                    high_instability
                ].mean()
            ),

        "MeanMoEProbability":
            float(
                moe_prob[
                    high_instability
                ].mean()
            ),
    },

    {
        "Group":
            "Remaining90Percent",

        "N":
            int(
                low_instability.sum()
            ),

        "ErrorRate":
            float(
                moe_error[
                    low_instability
                ].mean()
            ),

        "PositiveRate":
            float(
                y[
                    low_instability
                ].mean()
            ),

        "MeanMoEProbability":
            float(
                moe_prob[
                    low_instability
                ].mean()
            ),
    },
])


baseline_error = (
    moe_error[
        valid_profile
    ].mean()
)


instability_group_results[
    "ErrorEnrichment"
] = (

    instability_group_results[
        "ErrorRate"
    ]

    /
    baseline_error
)


print(
    "\n"
    + "=" * 135
)

print(
    "HIGH-INSTABILITY PATIENTS"
)

print(
    "=" * 135
)


display(
    instability_group_results
)


# ============================================================
# 13A.16 EXPERT-STABILITY CORRELATION
# ============================================================
#
# Important:
#
# If all expert stability signals are ~1 correlated,
# stability-aware routing is unlikely to help.
# ============================================================

stability_df = pd.DataFrame(

    LOCAL_MAD[
        valid_profile
    ],

    columns=[
        f"MAD_{x}"
        for x in RAMOE_EXPERTS
    ]
)


stability_correlation = (
    stability_df
    .corr(
        method="spearman"
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "EXPERT LOCAL-STABILITY SPEARMAN CORRELATION"
)

print(
    "=" * 135
)


display(
    stability_correlation
)


# ============================================================
# 13A.17 ORACLE STABILITY ROUTING SCREEN
# ============================================================
#
# No learned gate yet.
#
# Ask a simple screening question:
#
# If we weight experts inversely by their empirical local MAD,
# does ranking improve at all?
#
# This is NOT a final model.
# It only tells us whether stability potentially contains
# useful routing information.
#
# ============================================================

EPS = 1e-4


stability_weight = np.zeros_like(
    expert_prob
)


stability_weight[
    valid_profile
] = (

    1.0

    /
    (
        LOCAL_MAD[
            valid_profile
        ]

        +
        EPS
    )
)


stability_weight[
    valid_profile
] /= (

    stability_weight[
        valid_profile
    ].sum(
        axis=1,
        keepdims=True
    )
)


# For invalid profiles, default to equal weights
stability_weight[
    ~valid_profile
] = (

    1.0
    /
    n_experts
)


stability_weighted_prob = np.sum(

    stability_weight
    *
    expert_prob,

    axis=1
)


equal_prob = (
    expert_prob.mean(
        axis=1
    )
)


from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


routing_screen = pd.DataFrame([

    {
        "Method":
            "EqualExpertMean",

        "PR_AUC":
            average_precision_score(
                y,
                equal_prob
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                equal_prob
            ),
    },

    {
        "Method":
            "InverseLocalMAD",

        "PR_AUC":
            average_precision_score(
                y,
                stability_weighted_prob
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                stability_weighted_prob
            ),
    },
])


routing_screen[
    "Delta_PR_vs_Equal"
] = (

    routing_screen[
        "PR_AUC"
    ]

    -
    routing_screen[
        "PR_AUC"
    ].iloc[0]
)


print(
    "\n"
    + "=" * 135
)

print(
    "STABILITY-ROUTING SCREEN"
)

print(
    "=" * 135
)


display(
    routing_screen
)


# ============================================================
# 13A.18 SAVE PATIENT-LEVEL OUTPUT
# ============================================================

patient_stability_df = pd.DataFrame({

    "target":
        y,

    "fold":
        fold_assignment,

    "MoE_probability":
        moe_prob,

    "MoE_error":
        moe_error,

    "valid_neighborhood":
        valid_profile,

    "neighbor_count":
        VALID_NEIGHBOR_COUNT,

    "mean_instability":
        aggregate_instability,

    "max_instability":
        max_instability,

    "stability_gap":
        patient_stability_gap,

    "stability_cv":
        patient_mad_cv,

    "most_stable_expert_idx":
        most_stable_expert_idx,

    "most_unstable_expert_idx":
        most_unstable_expert_idx,

    "stability_weighted_probability":
        stability_weighted_prob,
})


for j, name in enumerate(
    RAMOE_EXPERTS
):

    patient_stability_df[
        f"MAD_{name}"
    ] = LOCAL_MAD[
        :,
        j
    ]


    patient_stability_df[
        f"SD_{name}"
    ] = LOCAL_SD[
        :,
        j
    ]


    patient_stability_df[
        f"FlipRate_{name}"
    ] = LOCAL_FLIP[
        :,
        j
    ]


    patient_stability_df[
        f"StabilityWeight_{name}"
    ] = stability_weight[
        :,
        j
    ]


# ============================================================
# 13A.19 SAVE TABLES
# ============================================================

save_csv(

    expert_stability_summary,

    TAB_DIR
    / "module13a_expert_stability_summary.csv"
)


save_csv(

    stable_expert_distribution,

    TAB_DIR
    / "module13a_stable_expert_distribution.csv"
)


save_csv(

    heterogeneity_summary,

    TAB_DIR
    / "module13a_stability_heterogeneity.csv"
)


save_csv(

    stability_error_results,

    TAB_DIR
    / "module13a_stability_error_signal.csv"
)


save_csv(

    instability_group_results,

    TAB_DIR
    / "module13a_high_instability_groups.csv"
)


save_csv(

    stability_correlation.reset_index(),

    TAB_DIR
    / "module13a_stability_correlation.csv"
)


save_csv(

    routing_screen,

    TAB_DIR
    / "module13a_stability_routing_screen.csv"
)


save_csv(

    patient_stability_df,

    PRED_DIR
    / "module13a_patient_stability_oof.csv"
)


# ============================================================
# 13A.20 MANIFEST
# ============================================================

module13a_manifest = {

    "method":
        "Perturbation-Aware Expert Stability Profiling",

    "perturbation_type":
        "empirical local neighborhood using real DEV profiles",

    "experts":
        RAMOE_EXPERTS,

    "k_neighbors":
        PAESP_CONFIG[
            "k_neighbors"
        ],

    "k_stability":
        PAESP_CONFIG[
            "k_stability"
        ],

    "min_neighbors":
        PAESP_CONFIG[
            "min_neighbors"
        ],

    "stability_metrics": [
        "local standard deviation",
        "local mean absolute deviation",
        "local prediction range",
        "decision flip rate",
        "expert-rank instability",
    ],

    "primary_stability_measure":
        "local mean absolute deviation",

    "holdout_used":
        False,
}


save_json(

    module13a_manifest,

    LOG_DIR
    / "module13a_manifest.json"
)


print(
    "\nMODULE 13A COMPLETE."
)

N: 202944
Expert matrix: (202944, 4)

Fitting nearest-neighbor index...
Querying neighborhoods...
  processed 10,000/202,944
  processed 20,000/202,944
  processed 30,000/202,944
  processed 40,000/202,944
  processed 50,000/202,944
  processed 60,000/202,944
  processed 70,000/202,944
  processed 80,000/202,944
  processed 90,000/202,944
  processed 100,000/202,944
  processed 110,000/202,944
  processed 120,000/202,944
  processed 130,000/202,944
  processed 140,000/202,944
  processed 150,000/202,944
  processed 160,000/202,944
  processed 170,000/202,944
  processed 180,000/202,944
  processed 190,000/202,944
  processed 200,000/202,944
  processed 202,944/202,944

Computing local expert stability...
  profiled 25,000/202,944
  profiled 50,000/202,944
  profiled 75,000/202,944
  profiled 100,000/202,944
  profiled 125,000/202,944
  profiled 150,000/202,944
  profiled 175,000/202,944
  profiled 200,000/202,944

Valid local profiles: 202924 / 202944
Coverage: 0.9999014506464837

MODU

,Expert,N,Mean_Local_SD,Median_Local_SD,Mean_Local_MAD,Median_Local_MAD,Mean_Local_Range,Mean_Flip_Rate,Mean_RankInstability
0,HistGradientBoosting,202924,0.037350,0.021180,0.035673,0.017389,0.138810,0.085816,0.677816
1,CatBoost,202924,0.037361,0.021970,0.035964,0.018584,0.139666,0.086727,0.697411
2,AutoInt,202924,0.038552,0.025861,0.037706,0.022710,0.145606,0.089017,0.894060
3,DeepFM_v2,202924,0.037748,0.024689,0.037166,0.021395,0.142334,0.091425,0.781473



PATIENT-SPECIFIC STABILITY DISTRIBUTION


,Expert,MostStable_N,MostStable_Fraction,MostUnstable_N,MostUnstable_Fraction
0,HistGradientBoosting,78984,0.389229,27118,0.133636
1,CatBoost,49843,0.245624,22844,0.112574
2,AutoInt,31984,0.157616,94760,0.466973
3,DeepFM_v2,42113,0.207531,58202,0.286817



EXPERT STABILITY HETEROGENEITY


...


EXPERT LOCAL-STABILITY SPEARMAN CORRELATION


,MAD_HistGradientBoosting,MAD_CatBoost,MAD_AutoInt,MAD_DeepFM_v2
MAD_HistGradientBoosting,1.000000,0.991017,0.977296,0.981539
MAD_CatBoost,0.991017,1.000000,0.978105,0.984915
MAD_AutoInt,0.977296,0.978105,1.000000,0.975258
MAD_DeepFM_v2,0.981539,0.984915,0.975258,1.000000



STABILITY-ROUTING SCREEN


,Method,PR_AUC,ROC_AUC,Delta_PR_vs_Equal
0,EqualExpertMean,0.380303,0.850539,0.000000
1,InverseLocalMAD,0.380376,0.850661,0.000073



MODULE 13A COMPLETE.


In [22]:
# ============================================================
# MODULE 13B — LPNS-MoE
# Local Prediction Neighborhood Stability-Aware
# MoE Reliability Modeling
# ============================================================
#
# GOAL
# ------------------------------------------------------------
# Test whether local prediction-neighborhood instability
# improves conditional error detection beyond:
#
#   B0: confidence
#   B1: confidence + expert disagreement
#   B2: confidence + disagreement + LPNS
#   B3: confidence + disagreement + LPNS + routing state
#
#
# IMPORTANT
# ------------------------------------------------------------
# This is a DEVELOPMENT SCREENING module.
#
# LPNS features currently come from Module 13A's global DEV
# neighborhood graph.
#
# If LPNS shows meaningful gain, we will later rebuild the
# neighborhood features strictly fold-locally for the final
# manuscript.
#
# HOLDOUT remains untouched.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 13B.1 REQUIREMENTS
# ============================================================

required_13b = [
    "LOCAL_MAD",
    "LOCAL_SD",
    "LOCAL_FLIP",
    "VALID_NEIGHBOR_COUNT",
    "valid_profile",

    "ETR_EXPERT_OOF",
    "RAMOE_GATE_OOF",
    "RAMOE_PROB_OOF",
    "RAMOE_THRESHOLD_OOF",
    "RAMOE_EXPERTS",

    "fold_assignment",
    "y_dev",
]

missing = [
    x for x in required_13b
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Run Modules 12A/12B and 13A first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 13B.2 CONFIG
# ============================================================

LPNS_CONFIG = {

    "model_C":
        0.10,

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],

    "eps":
        1e-8,
}


# ============================================================
# 13B.3 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(dtype=int)
)


moe_prob = np.asarray(
    RAMOE_PROB_OOF,
    dtype=float
)


threshold = np.asarray(
    RAMOE_THRESHOLD_OOF,
    dtype=float
)


expert_prob = np.asarray(
    ETR_EXPERT_OOF,
    dtype=float
)


gate_weights = np.asarray(
    RAMOE_GATE_OOF,
    dtype=float
)


prediction = (
    moe_prob >= threshold
).astype(int)


error_target = (
    prediction != y_all
).astype(int)


N = len(y_all)


print(
    "N:",
    N
)

print(
    "Error prevalence:",
    error_target.mean()
)

print(
    "LPNS-valid coverage:",
    valid_profile.mean()
)


# ============================================================
# 13B.4 BASIC UNCERTAINTY FEATURES
# ============================================================

def lpns_binary_entropy(p):

    p = np.clip(
        p,
        1e-8,
        1.0 - 1e-8
    )

    return (
        -p * np.log(p)
        -
        (1.0 - p) * np.log(1.0 - p)
    ) / np.log(2.0)


pred_entropy = (
    lpns_binary_entropy(
        moe_prob
    )
)


signed_margin = (
    moe_prob
    -
    threshold
)


absolute_margin = np.abs(
    signed_margin
)


distance_to_half = np.abs(
    moe_prob
    -
    0.5
)


# ============================================================
# 13B.5 EXPERT DISAGREEMENT FEATURES
# ============================================================

expert_mean = np.mean(
    expert_prob,
    axis=1
)


expert_std = np.std(
    expert_prob,
    axis=1
)


expert_min = np.min(
    expert_prob,
    axis=1
)


expert_max = np.max(
    expert_prob,
    axis=1
)


expert_range = (
    expert_max
    -
    expert_min
)


expert_median = np.median(
    expert_prob,
    axis=1
)


consensus_gap = np.abs(
    moe_prob
    -
    expert_mean
)


expert_abs_dev = np.mean(
    np.abs(
        expert_prob
        -
        moe_prob[:, None]
    ),
    axis=1
)


# ============================================================
# 13B.6 ROUTING FEATURES
# ============================================================

gate_safe = np.clip(
    gate_weights,
    1e-12,
    1.0
)


gate_entropy = (
    -np.sum(
        gate_safe
        *
        np.log(gate_safe),
        axis=1
    )
    /
    np.log(
        gate_weights.shape[1]
    )
)


max_gate = np.max(
    gate_weights,
    axis=1
)


sorted_gate = np.sort(
    gate_weights,
    axis=1
)


gate_margin = (
    sorted_gate[:, -1]
    -
    sorted_gate[:, -2]
)


effective_experts = np.exp(

    -np.sum(

        gate_safe
        *
        np.log(gate_safe),

        axis=1
    )
)


weighted_expert_variance = np.sum(

    gate_weights
    *
    (
        expert_prob
        -
        moe_prob[:, None]
    ) ** 2,

    axis=1
)


# ============================================================
# 13B.7 LPNS FEATURES
# ============================================================

mad = np.asarray(
    LOCAL_MAD,
    dtype=float
)


sd = np.asarray(
    LOCAL_SD,
    dtype=float
)


flip = np.asarray(
    LOCAL_FLIP,
    dtype=float
)


mean_mad = np.nanmean(
    mad,
    axis=1
)


max_mad = np.nanmax(
    mad,
    axis=1
)


min_mad = np.nanmin(
    mad,
    axis=1
)


mad_gap = (
    max_mad
    -
    min_mad
)


mad_cv = (

    np.nanstd(
        mad,
        axis=1
    )

    /
    (
        np.nanmean(
            mad,
            axis=1
        )
        +
        LPNS_CONFIG["eps"]
    )
)


mean_sd = np.nanmean(
    sd,
    axis=1
)


max_sd = np.nanmax(
    sd,
    axis=1
)


mean_flip = np.nanmean(
    flip,
    axis=1
)


max_flip = np.nanmax(
    flip,
    axis=1
)


neighbor_count = (
    np.asarray(
        VALID_NEIGHBOR_COUNT,
        dtype=float
    )
)


# ============================================================
# 13B.8 HANDLE RARE INVALID NEIGHBORHOODS
# ============================================================
#
# Only ~20 cases were invalid in Module 13A.
# Replace NaN with development-wide medians.
# ============================================================

def median_impute_vector(x):

    x = np.asarray(
        x,
        dtype=float
    ).copy()

    median_value = np.nanmedian(
        x
    )

    x[
        np.isnan(x)
    ] = median_value

    return x


mean_mad = median_impute_vector(mean_mad)
max_mad = median_impute_vector(max_mad)
min_mad = median_impute_vector(min_mad)
mad_gap = median_impute_vector(mad_gap)
mad_cv = median_impute_vector(mad_cv)

mean_sd = median_impute_vector(mean_sd)
max_sd = median_impute_vector(max_sd)

mean_flip = median_impute_vector(mean_flip)
max_flip = median_impute_vector(max_flip)


for j in range(
    mad.shape[1]
):

    mad[
        :,
        j
    ] = median_impute_vector(
        mad[
            :,
            j
        ]
    )

    sd[
        :,
        j
    ] = median_impute_vector(
        sd[
            :,
            j
        ]
    )

    flip[
        :,
        j
    ] = median_impute_vector(
        flip[
            :,
            j
        ]
    )


# ============================================================
# 13B.9 FEATURE SETS
# ============================================================
#
# B0 — prediction confidence
# B1 — confidence + expert disagreement
# B2 — B1 + LPNS
# B3 — B2 + routing state
#
# Same downstream classifier for ALL comparisons.
# ============================================================


# ------------------------------------------------------------
# B0
# ------------------------------------------------------------

F_CONF = np.column_stack([

    moe_prob,
    pred_entropy,
    signed_margin,
    absolute_margin,
    distance_to_half,
])


# ------------------------------------------------------------
# B1
# ------------------------------------------------------------

F_DISAGREE = np.column_stack([

    F_CONF,

    expert_mean,
    expert_std,
    expert_min,
    expert_max,
    expert_range,
    expert_median,
    consensus_gap,
    expert_abs_dev,

    expert_prob,
])


# ------------------------------------------------------------
# B2 — LPNS
# ------------------------------------------------------------

F_LPNS = np.column_stack([

    F_DISAGREE,

    mean_mad,
    max_mad,
    min_mad,
    mad_gap,
    mad_cv,

    mean_sd,
    max_sd,

    mean_flip,
    max_flip,

    neighbor_count,

    mad,
    sd,
    flip,
])


# ------------------------------------------------------------
# B3 — routing-aware LPNS
# ------------------------------------------------------------

F_LPNS_ROUTING = np.column_stack([

    F_LPNS,

    gate_entropy,
    max_gate,
    gate_margin,
    effective_experts,
    weighted_expert_variance,

    gate_weights,
])


print(
    "\nFeature dimensions:"
)

print(
    "  Confidence:",
    F_CONF.shape
)

print(
    "  + Disagreement:",
    F_DISAGREE.shape
)

print(
    "  + LPNS:",
    F_LPNS.shape
)

print(
    "  + Routing:",
    F_LPNS_ROUTING.shape
)


# ============================================================
# 13B.10 RELIABILITY MODEL
# ============================================================

def make_lpns_model():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",
            LogisticRegression(

                penalty="l2",

                C=
                    LPNS_CONFIG[
                        "model_C"
                    ],

                class_weight=
                    "balanced",

                solver="lbfgs",

                max_iter=3000,
            )
        )
    ])


# ============================================================
# 13B.11 OOF BUFFERS
# ============================================================

LPNS_RISK_CONF = np.zeros(
    N,
    dtype=float
)


LPNS_RISK_DISAGREE = np.zeros(
    N,
    dtype=float
)


LPNS_RISK_STABILITY = np.zeros(
    N,
    dtype=float
)


LPNS_RISK_ROUTING = np.zeros(
    N,
    dtype=float
)


fold_rows = []


# ============================================================
# 13B.12 STRICT RELIABILITY-HEAD CROSS-FITTING
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 120
    )

    print(
        f"LPNS RELIABILITY FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 120
    )


    train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_error_train = (
        error_target[
            train_idx
        ]
    )


    y_error_valid = (
        error_target[
            valid_idx
        ]
    )


    # ========================================================
    # Train four matched models
    # ========================================================

    m_conf = (
        make_lpns_model()
    )

    m_disagree = (
        make_lpns_model()
    )

    m_lpns = (
        make_lpns_model()
    )

    m_routing = (
        make_lpns_model()
    )


    m_conf.fit(

        F_CONF[
            train_idx
        ],

        y_error_train
    )


    m_disagree.fit(

        F_DISAGREE[
            train_idx
        ],

        y_error_train
    )


    m_lpns.fit(

        F_LPNS[
            train_idx
        ],

        y_error_train
    )


    m_routing.fit(

        F_LPNS_ROUTING[
            train_idx
        ],

        y_error_train
    )


    # ========================================================
    # OOF error-risk predictions
    # ========================================================

    risk_conf = (

        m_conf
        .predict_proba(

            F_CONF[
                valid_idx
            ]

        )[:, 1]
    )


    risk_disagree = (

        m_disagree
        .predict_proba(

            F_DISAGREE[
                valid_idx
            ]

        )[:, 1]
    )


    risk_lpns = (

        m_lpns
        .predict_proba(

            F_LPNS[
                valid_idx
            ]

        )[:, 1]
    )


    risk_routing = (

        m_routing
        .predict_proba(

            F_LPNS_ROUTING[
                valid_idx
            ]

        )[:, 1]
    )


    LPNS_RISK_CONF[
        valid_idx
    ] = risk_conf


    LPNS_RISK_DISAGREE[
        valid_idx
    ] = risk_disagree


    LPNS_RISK_STABILITY[
        valid_idx
    ] = risk_lpns


    LPNS_RISK_ROUTING[
        valid_idx
    ] = risk_routing


    # ========================================================
    # Fold metrics
    # ========================================================

    fold_result = {

        "Fold":
            outer_fold,

        "Error_Prevalence":
            y_error_valid.mean(),
    }


    for (
        name,
        score
    ) in [

        (
            "Confidence",
            risk_conf
        ),

        (
            "Disagreement",
            risk_disagree
        ),

        (
            "LPNS",
            risk_lpns
        ),

        (
            "LPNS_Routing",
            risk_routing
        ),
    ]:

        fold_result[
            f"{name}_PR"
        ] = (
            average_precision_score(

                y_error_valid,

                score
            )
        )


        fold_result[
            f"{name}_ROC"
        ] = (
            roc_auc_score(

                y_error_valid,

                score
            )
        )


    fold_rows.append(
        fold_result
    )


    print(
        "\nError detection PR-AUC:"
    )

    print(
        f"  Confidence        : "
        f"{fold_result['Confidence_PR']:.6f}"
    )

    print(
        f"  + disagreement    : "
        f"{fold_result['Disagreement_PR']:.6f}"
    )

    print(
        f"  + LPNS            : "
        f"{fold_result['LPNS_PR']:.6f}"
    )

    print(
        f"  + LPNS + routing  : "
        f"{fold_result['LPNS_Routing_PR']:.6f}"
    )


# ============================================================
# 13B.13 AURC
# ============================================================

def lpns_aurc(
    error,
    risk_score
):

    order = np.argsort(
        risk_score
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(
        np.trapz(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 13B.14 GLOBAL RELIABILITY BENCHMARK
# ============================================================

reliability_rows = []


risk_models = {

    "Confidence":
        LPNS_RISK_CONF,

    "Confidence+Disagreement":
        LPNS_RISK_DISAGREE,

    "LPNS_MoE":
        LPNS_RISK_STABILITY,

    "LPNS_MoE+Routing":
        LPNS_RISK_ROUTING,
}


for name, score in (
    risk_models.items()
):

    reliability_rows.append({

        "Reliability_Model":
            name,

        "Error_PR_AUC":
            average_precision_score(

                error_target,

                score
            ),

        "Error_ROC_AUC":
            roc_auc_score(

                error_target,

                score
            ),

        "AURC":
            lpns_aurc(

                error_target,

                score
            ),
    })


reliability_results = pd.DataFrame(
    reliability_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 13B — LPNS RELIABILITY BENCHMARK"
)

print(
    "=" * 130
)


display(
    reliability_results
)


# ============================================================
# 13B.15 INCREMENTAL GAINS
# ============================================================

base_conf = (

    reliability_results[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "Confidence"
    ]
    .iloc[0]
)


base_disagree = (

    reliability_results[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "Confidence+Disagreement"
    ]
    .iloc[0]
)


lpns_row = (

    reliability_results[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "LPNS_MoE"
    ]
    .iloc[0]
)


routing_row = (

    reliability_results[
        reliability_results[
            "Reliability_Model"
        ]
        ==
        "LPNS_MoE+Routing"
    ]
    .iloc[0]
)


incremental_gain = pd.DataFrame([

    {
        "Comparison":
            "Disagreement vs Confidence",

        "Delta_PR":
            base_disagree[
                "Error_PR_AUC"
            ]
            -
            base_conf[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            base_disagree[
                "Error_ROC_AUC"
            ]
            -
            base_conf[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            base_disagree[
                "AURC"
            ]
            -
            base_conf[
                "AURC"
            ],
    },

    {
        "Comparison":
            "LPNS vs Disagreement",

        "Delta_PR":
            lpns_row[
                "Error_PR_AUC"
            ]
            -
            base_disagree[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            lpns_row[
                "Error_ROC_AUC"
            ]
            -
            base_disagree[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            lpns_row[
                "AURC"
            ]
            -
            base_disagree[
                "AURC"
            ],
    },

    {
        "Comparison":
            "Routing vs LPNS",

        "Delta_PR":
            routing_row[
                "Error_PR_AUC"
            ]
            -
            lpns_row[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            routing_row[
                "Error_ROC_AUC"
            ]
            -
            lpns_row[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            routing_row[
                "AURC"
            ]
            -
            lpns_row[
                "AURC"
            ],
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "LPNS INCREMENTAL VALUE"
)

print(
    "=" * 130
)


display(
    incremental_gain
)


# ============================================================
# 13B.16 DEFERRAL EVALUATION
# ============================================================

def evaluate_lpns_deferral(
    risk_score,
    method
):

    rows = []


    total_errors = max(
        error_target.sum(),
        1
    )


    fn_target = (

        (y_all == 1)

        &
        (prediction == 0)

    ).astype(int)


    total_fn = max(
        fn_target.sum(),
        1
    )


    base_error_rate = (
        error_target.mean()
    )


    order = np.argsort(
        -risk_score
    )


    for rate in (
        LPNS_CONFIG[
            "deferral_rates"
        ]
    ):

        n_defer = max(

            1,

            int(
                round(
                    N
                    *
                    rate
                )
            )
        )


        defer_idx = (
            order[
                :n_defer
            ]
        )


        keep = np.ones(
            N,
            dtype=bool
        )


        keep[
            defer_idx
        ] = False


        deferred_error_rate = (
            error_target[
                defer_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0
                -
                rate,

            "Deferred_Error_Rate":
                deferred_error_rate,

            "Error_Enrichment":
                (
                    deferred_error_rate
                    /
                    base_error_rate
                ),

            "Errors_Captured":
                (
                    error_target[
                        defer_idx
                    ].sum()
                    /
                    total_errors
                ),

            "FalseNegatives_Captured":
                (
                    fn_target[
                        defer_idx
                    ].sum()
                    /
                    total_fn
                ),

            "Error_Per_100_Reviews":
                (
                    error_target[
                        defer_idx
                    ].mean()
                    *
                    100.0
                ),

            "Retained_Accuracy":
                accuracy_score(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_all[
                        keep
                    ],

                    prediction[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


deferral_frames = []


for name, score in (
    risk_models.items()
):

    deferral_frames.append(

        evaluate_lpns_deferral(

            score,

            name
        )
    )


deferral_results = pd.concat(

    deferral_frames,

    ignore_index=True
)


print(
    "\n"
    + "=" * 130
)

print(
    "LPNS SELECTIVE / DEFERRAL RESULTS"
)

print(
    "=" * 130
)


display(

    deferral_results[
        [
            "Method",
            "Deferral_Rate",
            "Coverage",
            "Error_Enrichment",
            "Errors_Captured",
            "FalseNegatives_Captured",
            "Error_Per_100_Reviews",
            "Retained_Accuracy",
            "Retained_F1",
            "Retained_MCC",
        ]
    ]
)


# ============================================================
# 13B.17 HIGH-RISK DECILE
# ============================================================

high_risk_rows = []


for name, score in (
    risk_models.items()
):

    cutoff = np.quantile(
        score,
        0.90
    )


    mask = (
        score
        >=
        cutoff
    )


    high_risk_rows.append({

        "Method":
            name,

        "N":
            int(
                mask.sum()
            ),

        "ErrorRate":
            float(
                error_target[
                    mask
                ].mean()
            ),

        "ErrorEnrichment":
            float(

                error_target[
                    mask
                ].mean()

                /
                error_target.mean()
            ),
    })


high_risk_results = pd.DataFrame(
    high_risk_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "TOP-10% ESTIMATED-RISK ERROR ENRICHMENT"
)

print(
    "=" * 130
)


display(
    high_risk_results
)


# ============================================================
# 13B.18 FOLD STABILITY
# ============================================================

fold_results = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "LPNS FOLD RESULTS"
)

print(
    "=" * 130
)


display(
    fold_results
)


print(
    "\nFold consistency — LPNS vs disagreement:"
)


print(
    "PR improved folds:",
    int(
        (
            fold_results[
                "LPNS_PR"
            ]
            >
            fold_results[
                "Disagreement_PR"
            ]
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "ROC improved folds:",
    int(
        (
            fold_results[
                "LPNS_ROC"
            ]
            >
            fold_results[
                "Disagreement_ROC"
            ]
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13B.19 SAVE
# ============================================================

save_csv(

    reliability_results,

    TAB_DIR
    / "module13b_lpns_reliability_benchmark.csv"
)


save_csv(

    incremental_gain,

    TAB_DIR
    / "module13b_lpns_incremental_gain.csv"
)


save_csv(

    deferral_results,

    TAB_DIR
    / "module13b_lpns_deferral.csv"
)


save_csv(

    high_risk_results,

    TAB_DIR
    / "module13b_lpns_high_risk_decile.csv"
)


save_csv(

    fold_results,

    TAB_DIR
    / "module13b_lpns_fold_results.csv"
)


lpns_prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "MoE_probability":
        moe_prob,

    "MoE_prediction":
        prediction,

    "Error":
        error_target,

    "Risk_Confidence":
        LPNS_RISK_CONF,

    "Risk_Disagreement":
        LPNS_RISK_DISAGREE,

    "Risk_LPNS":
        LPNS_RISK_STABILITY,

    "Risk_LPNS_Routing":
        LPNS_RISK_ROUTING,

    "Mean_Local_MAD":
        mean_mad,

    "Max_Local_MAD":
        max_mad,

    "MAD_Gap":
        mad_gap,

    "MAD_CV":
        mad_cv,

    "Mean_Flip":
        mean_flip,

    "Max_Flip":
        max_flip,
})


save_csv(

    lpns_prediction_df,

    PRED_DIR
    / "module13b_lpns_moe_oof.csv"
)


module13b_manifest = {

    "method":
        "Local Prediction Neighborhood Stability-Aware MoE Reliability",

    "status":
        "development screening",

    "feature_ablation": [
        "confidence",
        "confidence + expert disagreement",
        "confidence + disagreement + LPNS",
        "confidence + disagreement + LPNS + routing",
    ],

    "primary_endpoint":
        "incremental error-detection PR-AUC and ROC-AUC",

    "secondary_endpoints": [
        "AURC",
        "error enrichment",
        "error capture",
        "false-negative capture",
        "retained F1",
        "retained MCC",
    ],

    "holdout_used":
        False,

    "final_fold_local_neighborhood_rebuild_required":
        True,
}


save_json(

    module13b_manifest,

    LOG_DIR
    / "module13b_manifest.json"
)


print(
    "\nMODULE 13B COMPLETE."
)

N: 202944
Error prevalence: 0.14445364238410596
LPNS-valid coverage: 0.9999014506464837

Feature dimensions:
  Confidence: (202944, 5)
  + Disagreement: (202944, 17)
  + LPNS: (202944, 39)
  + Routing: (202944, 48)

LPNS RELIABILITY FOLD 1/5

Error detection PR-AUC:
  Confidence        : 0.583412
  + disagreement    : 0.588377
  + LPNS            : 0.591742
  + LPNS + routing  : 0.589875

LPNS RELIABILITY FOLD 2/5

Error detection PR-AUC:
  Confidence        : 0.534120
  + disagreement    : 0.542383
  + LPNS            : 0.556828
  + LPNS + routing  : 0.556299

LPNS RELIABILITY FOLD 3/5

Error detection PR-AUC:
  Confidence        : 0.580069
  + disagreement    : 0.582067
  + LPNS            : 0.591681
  + LPNS + routing  : 0.592318

LPNS RELIABILITY FOLD 4/5

Error detection PR-AUC:
  Confidence        : 0.569571
  + disagreement    : 0.571416
  + LPNS            : 0.578357
  + LPNS + routing  : 0.578799

LPNS RELIABILITY FOLD 5/5

Error detection PR-AUC:
  Confidence        : 0.55710

,Reliability_Model,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.564642,0.903920,0.028074
1,Confidence+Disagreement,0.567798,0.904192,0.027987
2,LPNS_MoE,0.578102,0.904994,0.027681
3,LPNS_MoE+Routing,0.577708,0.904977,0.027665



LPNS INCREMENTAL VALUE


,Comparison,Delta_PR,Delta_ROC,Delta_AURC
0,Disagreement vs Confidence,0.003156,0.000272,-0.000087
1,LPNS vs Disagreement,0.010305,0.000802,-0.000306
2,Routing vs LPNS,-0.000394,-0.000018,-0.000016



LPNS SELECTIVE / DEFERRAL RESULTS


...


TOP-10% ESTIMATED-RISK ERROR ENRICHMENT


,Method,N,ErrorRate,ErrorEnrichment
0,Confidence,20295,0.635181,4.397127
1,Confidence+Disagreement,20295,0.635920,4.402244
2,LPNS_MoE,20295,0.646563,4.475922
3,LPNS_MoE+Routing,20295,0.645282,4.467053



LPNS FOLD RESULTS


,Fold,Error_Prevalence,Confidence_PR,Confidence_ROC,Disagreement_PR,Disagreement_ROC,LPNS_PR,LPNS_ROC,LPNS_Routing_PR,LPNS_Routing_ROC
0,0,0.154697,0.583412,0.910754,0.588377,0.911034,0.591742,0.910594,0.589875,0.910532
1,1,0.125108,0.534120,0.894109,0.542383,0.894591,0.556828,0.894701,0.556299,0.894684
2,2,0.151322,0.580069,0.907285,0.582067,0.907593,0.591681,0.907744,0.592318,0.907924
3,3,0.149006,0.569571,0.905924,0.571416,0.905972,0.578357,0.906010,0.578799,0.905927
4,4,0.142136,0.557109,0.903900,0.561989,0.904057,0.575631,0.904860,0.575967,0.904803



Fold consistency — LPNS vs disagreement:
PR improved folds: 5 / 5
ROC improved folds: 4 / 5

MODULE 13B COMPLETE.


In [24]:
# ============================================================
# FIX NAMESPACE COLLISION
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

print("nn restored to:", nn)

nn restored to: <module 'torch.nn' from '/usr/local/lib/python3.13/dist-packages/torch/nn/__init__.py'>


In [25]:
# ============================================================
# MODULE 13C — STRICT FOLD-LOCAL LPNS-MoE
# Local Prediction Neighborhood Stability-Aware
# Reliability Modeling
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Publication-oriented validation of Module 13B.
#
# Fixes the main leakage concern:
#
#   - outer-validation patients NEVER enter neighborhood bank
#   - reliability-fit patients NEVER enter neighborhood bank
#   - preprocessing fitted on outer training only
#   - gate trained inside outer training
#   - disease threshold selected inside outer training
#   - reliability model trained inside outer training
#
#
# Per outer fold:
#
# OUTER TRAIN
#     |
#     +------ gate-development pool (70%)
#     |           |
#     |           +--- gate fit
#     |           |
#     |           +--- threshold selection
#     |
#     +------ reliability-fit pool (30%)
#
# Neighborhood reference bank:
#
#     sampled only from gate-development pool
#
# Query:
#
#     reliability-fit ---> reference bank
#     outer-valid     ---> reference bank
#
#
# Feature ablation:
#
# B0 = confidence
# B1 = confidence + expert disagreement
# B2 = B1 + strict fold-local LPNS
# B3 = B2 + routing state
#
#
# HOLDOUT NEVER USED.
# ============================================================


import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 13C.1 REQUIREMENTS
# ============================================================

required_13c = [
    "fit_apply_nested_calibration",
    "make_oracle_targets",
    "train_ramoe_gate",
    "predict_gate_weights",
    "mixture_probability",
    "select_f1_threshold_local",
    "RAMOE_EXPERTS",
    "RAMOE_FEATURES",
    "RAMOE_CONFIG",
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",
    "SEED",
]

missing = [
    x
    for x in required_13c
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required earlier objects/functions: "
        + ", ".join(missing)
    )


print("torch.nn namespace:", nn)


# ============================================================
# 13C.2 CONFIG
# ============================================================

STRICT_LPNS_CONFIG = {

    # --------------------------------------------------------
    # Outer-training architecture
    # --------------------------------------------------------

    "reliability_fraction":
        0.30,

    "threshold_fraction_within_gate_pool":
        0.15,

    # --------------------------------------------------------
    # Neighborhood system
    # --------------------------------------------------------

    # Reference bank is sampled from outer-training only.
    "max_reference_size":
        80000,

    "k_neighbors":
        20,

    "max_distance":
        3.0,

    "min_neighbors":
        6,

    # weighting for discrete dimensions
    "discrete_weight":
        0.65,

    # --------------------------------------------------------
    # Reliability classifier
    # --------------------------------------------------------

    "reliability_C":
        0.10,

    # --------------------------------------------------------
    # Selective evaluation
    # --------------------------------------------------------

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],
}


# ============================================================
# 13C.3 FEATURE DEFINITIONS
# ============================================================

LPNS_BINARY = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]


LPNS_ORDINAL = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]


LPNS_COUNT = [
    "MentHlth",
    "PhysHlth",
]


LPNS_CONTINUOUS = [
    "BMI",
]


LPNS_MULTICLASS = [
    "Diabetes",
]


LPNS_ALL_FEATURES = (
    LPNS_BINARY
    +
    LPNS_ORDINAL
    +
    LPNS_COUNT
    +
    LPNS_CONTINUOUS
    +
    LPNS_MULTICLASS
)


LPNS_SCALE_FEATURES = (
    LPNS_CONTINUOUS
    +
    LPNS_COUNT
    +
    LPNS_ORDINAL
)


LPNS_DISCRETE_FEATURES = (
    LPNS_BINARY
    +
    LPNS_MULTICLASS
)


# ============================================================
# 13C.4 DATA MATRICES
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N = len(
    y_all
)


X_distance_raw = (
    X_dev[
        LPNS_ALL_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


X_gate_raw = (
    X_dev[
        RAMOE_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


distance_feature_position = {
    name: j
    for j, name
    in enumerate(
        LPNS_ALL_FEATURES
    )
}


scale_cols = np.array([
    distance_feature_position[x]
    for x in LPNS_SCALE_FEATURES
])


discrete_cols = np.array([
    distance_feature_position[x]
    for x in LPNS_DISCRETE_FEATURES
])


# ============================================================
# 13C.5 HELPERS
# ============================================================

def strict_binary_entropy(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-8,
        1.0 - 1e-8
    )

    return (
        -p * np.log(p)
        -
        (1.0 - p)
        * np.log(1.0 - p)
    ) / np.log(2.0)


def strict_gate_entropy(w):

    w = np.clip(
        np.asarray(
            w,
            dtype=float
        ),
        1e-12,
        1.0
    )

    return (
        -np.sum(
            w * np.log(w),
            axis=1
        )
        /
        np.log(
            w.shape[1]
        )
    )


# ============================================================
# 13C.6 DISTANCE REPRESENTATION
# ============================================================

def fit_distance_transform(
    X_train_raw
):

    X_train = (
        X_train_raw
        .astype(
            np.float32
        )
        .copy()
    )


    scaler = StandardScaler()


    X_train[
        :,
        scale_cols
    ] = (
        scaler
        .fit_transform(
            X_train[
                :,
                scale_cols
            ]
        )
        .astype(
            np.float32
        )
    )


    X_train[
        :,
        discrete_cols
    ] *= (
        STRICT_LPNS_CONFIG[
            "discrete_weight"
        ]
    )


    return (
        X_train,
        scaler
    )


def apply_distance_transform(
    X_raw,
    scaler
):

    X_out = (
        X_raw
        .astype(
            np.float32
        )
        .copy()
    )


    X_out[
        :,
        scale_cols
    ] = (
        scaler
        .transform(
            X_out[
                :,
                scale_cols
            ]
        )
        .astype(
            np.float32
        )
    )


    X_out[
        :,
        discrete_cols
    ] *= (
        STRICT_LPNS_CONFIG[
            "discrete_weight"
        ]
    )


    return X_out


# ============================================================
# 13C.7 LPNS FEATURE EXTRACTION
# ============================================================

def extract_strict_lpns_features(
    query_expert_prob,
    reference_expert_prob,
    neighbor_idx,
    neighbor_dist,
    decision_threshold
):

    query_expert_prob = np.asarray(
        query_expert_prob,
        dtype=np.float32
    )


    reference_expert_prob = np.asarray(
        reference_expert_prob,
        dtype=np.float32
    )


    neighbor_prob = (
        reference_expert_prob[
            neighbor_idx
        ]
    )


    valid_neighbor = (
        neighbor_dist
        <=
        STRICT_LPNS_CONFIG[
            "max_distance"
        ]
    )


    count = (
        valid_neighbor.sum(
            axis=1
        )
    )


    count_safe = np.maximum(
        count,
        1
    )[
        :,
        None
    ]


    mask3 = (
        valid_neighbor[
            :,
            :,
            None
        ]
    )


    center = (
        query_expert_prob[
            :,
            None,
            :
        ]
    )


    # --------------------------------------------------------
    # Local MAD
    # --------------------------------------------------------

    local_mad = (

        np.sum(

            np.abs(
                neighbor_prob
                -
                center
            )

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Local mean
    # --------------------------------------------------------

    local_mean = (

        np.sum(
            neighbor_prob
            *
            mask3,
            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Local SD
    # --------------------------------------------------------

    local_sd = np.sqrt(

        np.sum(

            (
                neighbor_prob
                -
                local_mean[
                    :,
                    None,
                    :
                ]
            ) ** 2

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Decision flip rate
    # --------------------------------------------------------

    threshold_value = float(
        decision_threshold
    )


    center_decision = (
        query_expert_prob
        >=
        threshold_value
    )


    neighbor_decision = (
        neighbor_prob
        >=
        threshold_value
    )


    flip = (

        np.sum(

            (
                neighbor_decision
                !=
                center_decision[
                    :,
                    None,
                    :
                ]
            )

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Aggregate features
    # --------------------------------------------------------

    mean_mad = np.mean(
        local_mad,
        axis=1
    )


    max_mad = np.max(
        local_mad,
        axis=1
    )


    min_mad = np.min(
        local_mad,
        axis=1
    )


    mad_gap = (
        max_mad
        -
        min_mad
    )


    mad_cv = (

        np.std(
            local_mad,
            axis=1
        )

        /
        (
            np.mean(
                local_mad,
                axis=1
            )
            +
            1e-8
        )
    )


    mean_sd = np.mean(
        local_sd,
        axis=1
    )


    max_sd = np.max(
        local_sd,
        axis=1
    )


    mean_flip = np.mean(
        flip,
        axis=1
    )


    max_flip = np.max(
        flip,
        axis=1
    )


    features = np.column_stack([

        mean_mad,
        max_mad,
        min_mad,
        mad_gap,
        mad_cv,

        mean_sd,
        max_sd,

        mean_flip,
        max_flip,

        count.astype(
            float
        ),

        local_mad,
        local_sd,
        flip,
    ])


    valid_profile = (
        count
        >=
        STRICT_LPNS_CONFIG[
            "min_neighbors"
        ]
    )


    features[
        ~valid_profile
    ] = np.nan


    return (
        features,
        valid_profile,
        local_mad,
        local_sd,
        flip,
        count,
    )


# ============================================================
# 13C.8 TRAIN-BASED NaN IMPUTATION
# ============================================================

def strict_impute(
    train_features,
    valid_features
):

    train_features = (
        train_features
        .astype(float)
        .copy()
    )


    valid_features = (
        valid_features
        .astype(float)
        .copy()
    )


    medians = np.nanmedian(
        train_features,
        axis=0
    )


    medians = np.where(
        np.isnan(medians),
        0.0,
        medians
    )


    train_nan = np.where(
        np.isnan(
            train_features
        )
    )


    train_features[
        train_nan
    ] = medians[
        train_nan[1]
    ]


    valid_nan = np.where(
        np.isnan(
            valid_features
        )
    )


    valid_features[
        valid_nan
    ] = medians[
        valid_nan[1]
    ]


    return (
        train_features,
        valid_features
    )


# ============================================================
# 13C.9 STANDARD TRUST FEATURES
# ============================================================

def build_strict_base_features(
    moe_probability,
    expert_probability,
    routing_weights,
    threshold_value
):

    p = np.asarray(
        moe_probability,
        dtype=float
    )


    experts = np.asarray(
        expert_probability,
        dtype=float
    )


    weights = np.asarray(
        routing_weights,
        dtype=float
    )


    entropy = (
        strict_binary_entropy(
            p
        )
    )


    signed_margin = (
        p
        -
        float(
            threshold_value
        )
    )


    abs_margin = np.abs(
        signed_margin
    )


    distance_half = np.abs(
        p
        -
        0.5
    )


    # --------------------------------------------------------
    # confidence features
    # --------------------------------------------------------

    F_conf = np.column_stack([

        p,
        entropy,
        signed_margin,
        abs_margin,
        distance_half,
    ])


    # --------------------------------------------------------
    # expert disagreement features
    # --------------------------------------------------------

    expert_mean = np.mean(
        experts,
        axis=1
    )


    expert_std = np.std(
        experts,
        axis=1
    )


    expert_min = np.min(
        experts,
        axis=1
    )


    expert_max = np.max(
        experts,
        axis=1
    )


    expert_range = (
        expert_max
        -
        expert_min
    )


    expert_median = np.median(
        experts,
        axis=1
    )


    consensus_gap = np.abs(
        p
        -
        expert_mean
    )


    expert_abs_dev = np.mean(

        np.abs(
            experts
            -
            p[:, None]
        ),

        axis=1
    )


    F_disagreement = np.column_stack([

        F_conf,

        expert_mean,
        expert_std,
        expert_min,
        expert_max,
        expert_range,
        expert_median,
        consensus_gap,
        expert_abs_dev,

        experts,
    ])


    # --------------------------------------------------------
    # routing state
    # --------------------------------------------------------

    gate_entropy = (
        strict_gate_entropy(
            weights
        )
    )


    max_gate = np.max(
        weights,
        axis=1
    )


    sorted_gate = np.sort(
        weights,
        axis=1
    )


    gate_margin = (

        sorted_gate[
            :,
            -1
        ]

        -
        sorted_gate[
            :,
            -2
        ]
    )


    effective_experts = np.exp(

        -np.sum(

            np.clip(
                weights,
                1e-12,
                1.0
            )

            *
            np.log(
                np.clip(
                    weights,
                    1e-12,
                    1.0
                )
            ),

            axis=1
        )
    )


    weighted_variance = np.sum(

        weights

        *
        (
            experts
            -
            p[:, None]
        ) ** 2,

        axis=1
    )


    F_routing = np.column_stack([

        gate_entropy,
        max_gate,
        gate_margin,
        effective_experts,
        weighted_variance,

        weights,
    ])


    return (
        F_conf,
        F_disagreement,
        F_routing
    )


# ============================================================
# 13C.10 RELIABILITY MODEL
# ============================================================

def make_strict_reliability_model():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    STRICT_LPNS_CONFIG[
                        "reliability_C"
                    ],

                class_weight="balanced",

                solver="lbfgs",

                max_iter=3000,
            )
        )
    ])


# ============================================================
# 13C.11 OUTPUT BUFFERS
# ============================================================

STRICT_MOE_OOF = np.zeros(
    N,
    dtype=float
)


STRICT_THRESHOLD_OOF = np.zeros(
    N,
    dtype=float
)


STRICT_ERROR_OOF = np.zeros(
    N,
    dtype=int
)


STRICT_RISK_CONF = np.zeros(
    N,
    dtype=float
)


STRICT_RISK_DISAGREE = np.zeros(
    N,
    dtype=float
)


STRICT_RISK_LPNS = np.zeros(
    N,
    dtype=float
)


STRICT_RISK_LPNS_ROUTING = np.zeros(
    N,
    dtype=float
)


STRICT_MEAN_MAD = np.full(
    N,
    np.nan
)


STRICT_MAX_MAD = np.full(
    N,
    np.nan
)


fold_rows = []

neighborhood_rows = []


# ============================================================
# 13C.12 OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"STRICT LPNS-MoE OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    # ========================================================
    # A. OUTER INDICES
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # B. STRICT EXPERT PROBABILITIES
    # ========================================================

    print(
        "\nBuilding strict fold-local expert probabilities..."
    )


    train_cal = {}
    valid_cal = {}


    for model_name in RAMOE_EXPERTS:

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    P_outer_train = np.column_stack([

        train_cal[
            x
        ]

        for x in RAMOE_EXPERTS
    ])


    P_outer_valid = np.column_stack([

        valid_cal[
            x
        ]

        for x in RAMOE_EXPERTS
    ])


    # ========================================================
    # C. SPLIT OUTER TRAIN
    #
    # 70% gate-development
    # 30% reliability-fit
    # ========================================================

    split_reliability = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                STRICT_LPNS_CONFIG[
                    "reliability_fraction"
                ],

            random_state=
                SEED
                + 80000
                + outer_fold
        )
    )


    (
        gate_pool_local,
        reliability_local

    ) = next(

        split_reliability.split(

            np.zeros(
                len(
                    outer_train_idx
                )
            ),

            y_outer_train
        )
    )


    gate_pool_global = (
        outer_train_idx[
            gate_pool_local
        ]
    )


    reliability_global = (
        outer_train_idx[
            reliability_local
        ]
    )


    y_gate_pool = (
        y_all[
            gate_pool_global
        ]
    )


    # ========================================================
    # D. GATE FIT / THRESHOLD SPLIT
    # ========================================================

    split_threshold = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                STRICT_LPNS_CONFIG[
                    "threshold_fraction_within_gate_pool"
                ],

            random_state=
                SEED
                + 81000
                + outer_fold
        )
    )


    (
        gate_fit_inside,
        threshold_inside

    ) = next(

        split_threshold.split(

            np.zeros(
                len(
                    gate_pool_global
                )
            ),

            y_gate_pool
        )
    )


    gate_fit_local = (
        gate_pool_local[
            gate_fit_inside
        ]
    )


    threshold_local = (
        gate_pool_local[
            threshold_inside
        ]
    )


    gate_fit_global = (
        outer_train_idx[
            gate_fit_local
        ]
    )


    threshold_global = (
        outer_train_idx[
            threshold_local
        ]
    )


    print(
        "\nOuter-training partitions:"
    )

    print(
        "  Gate fit:",
        len(
            gate_fit_global
        )
    )

    print(
        "  Threshold:",
        len(
            threshold_global
        )
    )

    print(
        "  Reliability:",
        len(
            reliability_global
        )
    )

    print(
        "  Outer validation:",
        len(
            outer_valid_idx
        )
    )


    # ========================================================
    # E. TRAIN STRICT MoE GATE
    # ========================================================

    gate_scaler = StandardScaler()


    X_gate_fit = (
        gate_scaler
        .fit_transform(

            X_gate_raw[
                gate_fit_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_threshold = (
        gate_scaler
        .transform(

            X_gate_raw[
                threshold_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_reliability_gate = (
        gate_scaler
        .transform(

            X_gate_raw[
                reliability_global
            ]
        )
        .astype(
            np.float32
        )
    )


    X_valid_gate = (
        gate_scaler
        .transform(

            X_gate_raw[
                outer_valid_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    P_gate_fit = (
        P_outer_train[
            gate_fit_local
        ]
    )


    y_gate_fit = (
        y_all[
            gate_fit_global
        ]
    )


    oracle_target = (
        make_oracle_targets(

            P_gate_fit,

            y_gate_fit,

            temperature=
                RAMOE_CONFIG[
                    "oracle_temperature"
                ]
        )
    )


    print(
        "\nTraining fold-local MoE gate..."
    )


    gate_model = (
        train_ramoe_gate(

            X_gate_fit,

            oracle_target,

            seed=
                SEED
                + 82000
                + outer_fold
        )
    )


    # ========================================================
    # F. ROUTING WEIGHTS
    # ========================================================

    W_threshold = (
        predict_gate_weights(

            gate_model,

            X_threshold
        )
    )


    W_reliability = (
        predict_gate_weights(

            gate_model,

            X_reliability_gate
        )
    )


    W_valid = (
        predict_gate_weights(

            gate_model,

            X_valid_gate
        )
    )


    # ========================================================
    # G. MoE PROBABILITIES
    # ========================================================

    P_threshold = (
        P_outer_train[
            threshold_local
        ]
    )


    P_reliability = (
        P_outer_train[
            reliability_local
        ]
    )


    moe_threshold = (
        mixture_probability(

            P_threshold,

            W_threshold
        )
    )


    moe_reliability = (
        mixture_probability(

            P_reliability,

            W_reliability
        )
    )


    moe_valid = (
        mixture_probability(

            P_outer_valid,

            W_valid
        )
    )


    # ========================================================
    # H. SELECT DISEASE THRESHOLD
    # ========================================================

    y_threshold = (
        y_all[
            threshold_global
        ]
    )


    decision_threshold, _ = (
        select_f1_threshold_local(

            y_threshold,

            moe_threshold
        )
    )


    print(
        "\nSelected disease threshold:",
        decision_threshold
    )


    # ========================================================
    # I. ERROR TARGETS
    # ========================================================

    y_reliability = (
        y_all[
            reliability_global
        ]
    )


    pred_reliability = (

        moe_reliability
        >=
        decision_threshold

    ).astype(int)


    pred_valid = (

        moe_valid
        >=
        decision_threshold

    ).astype(int)


    error_reliability = (

        pred_reliability
        !=
        y_reliability

    ).astype(int)


    error_valid = (

        pred_valid
        !=
        y_outer_valid

    ).astype(int)


    STRICT_MOE_OOF[
        outer_valid_idx
    ] = moe_valid


    STRICT_THRESHOLD_OOF[
        outer_valid_idx
    ] = decision_threshold


    STRICT_ERROR_OOF[
        outer_valid_idx
    ] = error_valid


    # ========================================================
    # J. FOLD-LOCAL DISTANCE REPRESENTATION
    # ========================================================

    print(
        "\nBuilding fold-local neighborhood representation..."
    )


    X_outer_distance, distance_scaler = (
        fit_distance_transform(

            X_distance_raw[
                outer_train_idx
            ]
        )
    )


    X_valid_distance = (
        apply_distance_transform(

            X_distance_raw[
                outer_valid_idx
            ],

            distance_scaler
        )
    )


    X_reliability_distance = (
        X_outer_distance[
            reliability_local
        ]
    )


    # ========================================================
    # K. REFERENCE BANK
    # ========================================================

    rng = np.random.default_rng(

        SEED
        +
        83000
        +
        outer_fold
    )


    reference_local = (
        gate_pool_local.copy()
    )


    max_reference = (
        STRICT_LPNS_CONFIG[
            "max_reference_size"
        ]
    )


    if len(
        reference_local
    ) > max_reference:

        reference_local = (
            rng.choice(

                reference_local,

                size=
                    max_reference,

                replace=False
            )
        )


    X_reference = (
        X_outer_distance[
            reference_local
        ]
    )


    P_reference = (
        P_outer_train[
            reference_local
        ]
    )


    print(
        "  Reference bank:",
        len(
            reference_local
        )
    )


    # ========================================================
    # L. NEAREST-NEIGHBOR INDEX
    #
    # IMPORTANT:
    # Do NOT call this variable "nn".
    #
    # nn is reserved for torch.nn.
    # ========================================================

    nbr_model = NearestNeighbors(

        n_neighbors=
            STRICT_LPNS_CONFIG[
                "k_neighbors"
            ],

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    nbr_model.fit(
        X_reference
    )


    # ========================================================
    # M. QUERY RELIABILITY-FIT
    # ========================================================

    print(
        "  Querying reliability-fit neighborhoods..."
    )


    rel_dist, rel_neighbor = (
        nbr_model.kneighbors(

            X_reliability_distance,

            return_distance=True
        )
    )


    # ========================================================
    # N. QUERY OUTER VALIDATION
    # ========================================================

    print(
        "  Querying outer-validation neighborhoods..."
    )


    valid_dist, valid_neighbor = (
        nbr_model.kneighbors(

            X_valid_distance,

            return_distance=True
        )
    )


    # ========================================================
    # O. LPNS FEATURES
    # ========================================================

    (
        LPNS_rel,
        valid_rel_profile,
        local_mad_rel,
        local_sd_rel,
        flip_rel,
        rel_count

    ) = extract_strict_lpns_features(

        P_reliability,

        P_reference,

        rel_neighbor,

        rel_dist,

        decision_threshold
    )


    (
        LPNS_valid,
        valid_valid_profile,
        local_mad_valid,
        local_sd_valid,
        flip_valid,
        valid_count

    ) = extract_strict_lpns_features(

        P_outer_valid,

        P_reference,

        valid_neighbor,

        valid_dist,

        decision_threshold
    )


    # ========================================================
    # P. IMPUTE USING RELIABILITY-FIT ONLY
    # ========================================================

    (
        LPNS_rel,
        LPNS_valid

    ) = strict_impute(

        LPNS_rel,

        LPNS_valid
    )


    # save interpretable LPNS values
    STRICT_MEAN_MAD[
        outer_valid_idx
    ] = np.nanmean(

        local_mad_valid,

        axis=1
    )


    STRICT_MAX_MAD[
        outer_valid_idx
    ] = np.nanmax(

        local_mad_valid,

        axis=1
    )


    neighborhood_rows.append({

        "Fold":
            outer_fold,

        "Reference_N":
            len(
                reference_local
            ),

        "Reliability_LPNS_Coverage":
            float(
                valid_rel_profile.mean()
            ),

        "Validation_LPNS_Coverage":
            float(
                valid_valid_profile.mean()
            ),

        "Reliability_MeanNeighbors":
            float(
                rel_count.mean()
            ),

        "Validation_MeanNeighbors":
            float(
                valid_count.mean()
            ),
    })


    print(
        "  Reliability LPNS coverage:",
        valid_rel_profile.mean()
    )


    print(
        "  Validation LPNS coverage:",
        valid_valid_profile.mean()
    )


    # ========================================================
    # Q. BASE FEATURE SETS
    # ========================================================

    (
        F_conf_rel,
        F_dis_rel,
        F_route_rel

    ) = build_strict_base_features(

        moe_reliability,

        P_reliability,

        W_reliability,

        decision_threshold
    )


    (
        F_conf_valid,
        F_dis_valid,
        F_route_valid

    ) = build_strict_base_features(

        moe_valid,

        P_outer_valid,

        W_valid,

        decision_threshold
    )


    # ========================================================
    # R. FINAL ABLATION MATRICES
    # ========================================================

    F_lpns_rel = np.column_stack([

        F_dis_rel,

        LPNS_rel,
    ])


    F_lpns_valid = np.column_stack([

        F_dis_valid,

        LPNS_valid,
    ])


    F_full_rel = np.column_stack([

        F_lpns_rel,

        F_route_rel,
    ])


    F_full_valid = np.column_stack([

        F_lpns_valid,

        F_route_valid,
    ])


    # ========================================================
    # S. MATCHED RELIABILITY MODELS
    # ========================================================

    m_conf = (
        make_strict_reliability_model()
    )


    m_dis = (
        make_strict_reliability_model()
    )


    m_lpns = (
        make_strict_reliability_model()
    )


    m_full = (
        make_strict_reliability_model()
    )


    m_conf.fit(
        F_conf_rel,
        error_reliability
    )


    m_dis.fit(
        F_dis_rel,
        error_reliability
    )


    m_lpns.fit(
        F_lpns_rel,
        error_reliability
    )


    m_full.fit(
        F_full_rel,
        error_reliability
    )


    # ========================================================
    # T. VALIDATION RISK
    # ========================================================

    risk_conf = (

        m_conf
        .predict_proba(
            F_conf_valid
        )[
            :,
            1
        ]
    )


    risk_dis = (

        m_dis
        .predict_proba(
            F_dis_valid
        )[
            :,
            1
        ]
    )


    risk_lpns = (

        m_lpns
        .predict_proba(
            F_lpns_valid
        )[
            :,
            1
        ]
    )


    risk_full = (

        m_full
        .predict_proba(
            F_full_valid
        )[
            :,
            1
        ]
    )


    STRICT_RISK_CONF[
        outer_valid_idx
    ] = risk_conf


    STRICT_RISK_DISAGREE[
        outer_valid_idx
    ] = risk_dis


    STRICT_RISK_LPNS[
        outer_valid_idx
    ] = risk_lpns


    STRICT_RISK_LPNS_ROUTING[
        outer_valid_idx
    ] = risk_full


    # ========================================================
    # U. FOLD METRICS
    # ========================================================

    fold_row = {

        "Fold":
            outer_fold,

        "Error_Prevalence":
            error_valid.mean(),

        "Confidence_PR":
            average_precision_score(
                error_valid,
                risk_conf
            ),

        "Disagreement_PR":
            average_precision_score(
                error_valid,
                risk_dis
            ),

        "LPNS_PR":
            average_precision_score(
                error_valid,
                risk_lpns
            ),

        "LPNS_Routing_PR":
            average_precision_score(
                error_valid,
                risk_full
            ),

        "Confidence_ROC":
            roc_auc_score(
                error_valid,
                risk_conf
            ),

        "Disagreement_ROC":
            roc_auc_score(
                error_valid,
                risk_dis
            ),

        "LPNS_ROC":
            roc_auc_score(
                error_valid,
                risk_lpns
            ),

        "LPNS_Routing_ROC":
            roc_auc_score(
                error_valid,
                risk_full
            ),
    }


    fold_row[
        "LPNS_Delta_PR_vs_Disagreement"
    ] = (

        fold_row[
            "LPNS_PR"
        ]

        -
        fold_row[
            "Disagreement_PR"
        ]
    )


    fold_row[
        "LPNS_Delta_ROC_vs_Disagreement"
    ] = (

        fold_row[
            "LPNS_ROC"
        ]

        -
        fold_row[
            "Disagreement_ROC"
        ]
    )


    fold_rows.append(
        fold_row
    )


    print(
        "\nFold error-detection performance:"
    )


    print(
        f"  Confidence          PR = "
        f"{fold_row['Confidence_PR']:.6f}"
    )


    print(
        f"  + disagreement      PR = "
        f"{fold_row['Disagreement_PR']:.6f}"
    )


    print(
        f"  + strict LPNS       PR = "
        f"{fold_row['LPNS_PR']:.6f}"
    )


    print(
        f"  + LPNS + routing    PR = "
        f"{fold_row['LPNS_Routing_PR']:.6f}"
    )


    print(
        f"  LPNS ΔPR            = "
        f"{fold_row['LPNS_Delta_PR_vs_Disagreement']:+.6f}"
    )


    # ========================================================
    # V. CLEANUP
    # ========================================================

    del (
        gate_model,
        nbr_model,
        m_conf,
        m_dis,
        m_lpns,
        m_full,
        X_outer_distance,
        X_valid_distance,
        X_reference,
        P_reference,
        rel_neighbor,
        valid_neighbor,
        rel_dist,
        valid_dist,
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13C.13 AURC
# ============================================================

def strict_aurc(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(

        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 13C.14 GLOBAL BENCHMARK
# ============================================================

strict_risk_models = {

    "Confidence":
        STRICT_RISK_CONF,

    "Confidence+Disagreement":
        STRICT_RISK_DISAGREE,

    "Strict_LPNS_MoE":
        STRICT_RISK_LPNS,

    "Strict_LPNS_MoE+Routing":
        STRICT_RISK_LPNS_ROUTING,
}


global_rows = []


for name, risk in (
    strict_risk_models.items()
):

    global_rows.append({

        "Reliability_Model":
            name,

        "Error_PR_AUC":
            average_precision_score(

                STRICT_ERROR_OOF,

                risk
            ),

        "Error_ROC_AUC":
            roc_auc_score(

                STRICT_ERROR_OOF,

                risk
            ),

        "AURC":
            strict_aurc(

                STRICT_ERROR_OOF,

                risk
            ),
    })


strict_results = pd.DataFrame(
    global_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 13C — STRICT FOLD-LOCAL LPNS RELIABILITY"
)

print(
    "=" * 135
)


display(
    strict_results
)


# ============================================================
# 13C.15 INCREMENTAL VALUE
# ============================================================

row_conf = (

    strict_results[
        strict_results[
            "Reliability_Model"
        ]
        ==
        "Confidence"
    ]
    .iloc[0]
)


row_dis = (

    strict_results[
        strict_results[
            "Reliability_Model"
        ]
        ==
        "Confidence+Disagreement"
    ]
    .iloc[0]
)


row_lpns = (

    strict_results[
        strict_results[
            "Reliability_Model"
        ]
        ==
        "Strict_LPNS_MoE"
    ]
    .iloc[0]
)


row_route = (

    strict_results[
        strict_results[
            "Reliability_Model"
        ]
        ==
        "Strict_LPNS_MoE+Routing"
    ]
    .iloc[0]
)


strict_gain = pd.DataFrame([

    {
        "Comparison":
            "Disagreement vs Confidence",

        "Delta_PR":
            row_dis[
                "Error_PR_AUC"
            ]
            -
            row_conf[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            row_dis[
                "Error_ROC_AUC"
            ]
            -
            row_conf[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            row_dis[
                "AURC"
            ]
            -
            row_conf[
                "AURC"
            ],
    },

    {
        "Comparison":
            "Strict LPNS vs Disagreement",

        "Delta_PR":
            row_lpns[
                "Error_PR_AUC"
            ]
            -
            row_dis[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            row_lpns[
                "Error_ROC_AUC"
            ]
            -
            row_dis[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            row_lpns[
                "AURC"
            ]
            -
            row_dis[
                "AURC"
            ],
    },

    {
        "Comparison":
            "Routing vs Strict LPNS",

        "Delta_PR":
            row_route[
                "Error_PR_AUC"
            ]
            -
            row_lpns[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            row_route[
                "Error_ROC_AUC"
            ]
            -
            row_lpns[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            row_route[
                "AURC"
            ]
            -
            row_lpns[
                "AURC"
            ],
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "STRICT LPNS INCREMENTAL VALUE"
)

print(
    "=" * 135
)


display(
    strict_gain
)


# ============================================================
# 13C.16 SELECTIVE / DEFERRAL
# ============================================================

STRICT_PREDICTION = (

    STRICT_MOE_OOF
    >=
    STRICT_THRESHOLD_OOF

).astype(int)


STRICT_FN = (

    (y_all == 1)

    &
    (STRICT_PREDICTION == 0)

).astype(int)


def evaluate_strict_deferral(
    risk,
    method
):

    rows = []


    total_error = max(
        STRICT_ERROR_OOF.sum(),
        1
    )


    total_fn = max(
        STRICT_FN.sum(),
        1
    )


    base_error = (
        STRICT_ERROR_OOF.mean()
    )


    order = np.argsort(
        -risk
    )


    for rate in (
        STRICT_LPNS_CONFIG[
            "deferral_rates"
        ]
    ):

        n_defer = max(

            1,

            int(
                round(
                    N
                    *
                    rate
                )
            )
        )


        defer_idx = (
            order[
                :n_defer
            ]
        )


        keep = np.ones(
            N,
            dtype=bool
        )


        keep[
            defer_idx
        ] = False


        deferred_error_rate = (
            STRICT_ERROR_OOF[
                defer_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0
                -
                rate,

            "Error_Enrichment":
                (
                    deferred_error_rate
                    /
                    base_error
                ),

            "Errors_Captured":
                (
                    STRICT_ERROR_OOF[
                        defer_idx
                    ].sum()
                    /
                    total_error
                ),

            "FalseNegatives_Captured":
                (
                    STRICT_FN[
                        defer_idx
                    ].sum()
                    /
                    total_fn
                ),

            "Error_Per_100_Reviews":
                (
                    deferred_error_rate
                    *
                    100.0
                ),

            "Retained_Accuracy":
                accuracy_score(

                    y_all[
                        keep
                    ],

                    STRICT_PREDICTION[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_all[
                        keep
                    ],

                    STRICT_PREDICTION[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_all[
                        keep
                    ],

                    STRICT_PREDICTION[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


strict_deferral_frames = []


for name, risk in (
    strict_risk_models.items()
):

    strict_deferral_frames.append(

        evaluate_strict_deferral(

            risk,

            name
        )
    )


strict_deferral = pd.concat(

    strict_deferral_frames,

    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "STRICT LPNS SELECTIVE / DEFERRAL RESULTS"
)

print(
    "=" * 135
)


display(
    strict_deferral
)


# ============================================================
# 13C.17 FOLD STABILITY
# ============================================================

strict_fold_results = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "STRICT LPNS FOLD RESULTS"
)

print(
    "=" * 135
)


display(
    strict_fold_results
)


print(
    "\nLPNS PR improved folds:",
    int(
        (
            strict_fold_results[
                "LPNS_Delta_PR_vs_Disagreement"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "LPNS ROC improved folds:",
    int(
        (
            strict_fold_results[
                "LPNS_Delta_ROC_vs_Disagreement"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "\nMean strict LPNS ΔPR:",
    strict_fold_results[
        "LPNS_Delta_PR_vs_Disagreement"
    ].mean()
)


print(
    "Mean strict LPNS ΔROC:",
    strict_fold_results[
        "LPNS_Delta_ROC_vs_Disagreement"
    ].mean()
)


# ============================================================
# 13C.18 NEIGHBORHOOD DIAGNOSTICS
# ============================================================

strict_neighborhood_results = (
    pd.DataFrame(
        neighborhood_rows
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "STRICT LPNS NEIGHBORHOOD DIAGNOSTICS"
)

print(
    "=" * 135
)


display(
    strict_neighborhood_results
)


# ============================================================
# 13C.19 DISEASE PERFORMANCE SANITY CHECK
# ============================================================

strict_disease_results = pd.DataFrame([

    {
        "Model":
            "Strict_RA_MoE",

        "PR_AUC":
            average_precision_score(

                y_all,

                STRICT_MOE_OOF
            ),

        "ROC_AUC":
            roc_auc_score(

                y_all,

                STRICT_MOE_OOF
            ),
    }
])


print(
    "\n"
    + "=" * 135
)

print(
    "STRICT MoE DISEASE PERFORMANCE"
)

print(
    "=" * 135
)


display(
    strict_disease_results
)


# ============================================================
# 13C.20 SAVE
# ============================================================

save_csv(

    strict_results,

    TAB_DIR
    / "module13c_strict_lpns_reliability.csv"
)


save_csv(

    strict_gain,

    TAB_DIR
    / "module13c_strict_lpns_incremental_gain.csv"
)


save_csv(

    strict_deferral,

    TAB_DIR
    / "module13c_strict_lpns_deferral.csv"
)


save_csv(

    strict_fold_results,

    TAB_DIR
    / "module13c_strict_lpns_folds.csv"
)


save_csv(

    strict_neighborhood_results,

    TAB_DIR
    / "module13c_neighborhood_diagnostics.csv"
)


save_csv(

    strict_disease_results,

    TAB_DIR
    / "module13c_strict_disease_performance.csv"
)


strict_prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "MoE_probability":
        STRICT_MOE_OOF,

    "Decision_threshold":
        STRICT_THRESHOLD_OOF,

    "Prediction":
        STRICT_PREDICTION,

    "Error":
        STRICT_ERROR_OOF,

    "FalseNegative":
        STRICT_FN,

    "Risk_Confidence":
        STRICT_RISK_CONF,

    "Risk_Disagreement":
        STRICT_RISK_DISAGREE,

    "Risk_Strict_LPNS":
        STRICT_RISK_LPNS,

    "Risk_Strict_LPNS_Routing":
        STRICT_RISK_LPNS_ROUTING,

    "Mean_Local_MAD":
        STRICT_MEAN_MAD,

    "Max_Local_MAD":
        STRICT_MAX_MAD,
})


save_csv(

    strict_prediction_df,

    PRED_DIR
    / "module13c_strict_lpns_oof.csv"
)


module13c_manifest = {

    "method":
        "Strict Fold-Local Local Prediction Neighborhood Stability-Aware MoE",

    "main_predictor":
        "RA-MoE",

    "neighborhood_reference":
        "outer-training gate-development pool only",

    "reference_sampling":
        "uniform random without label-based selection",

    "maximum_reference_size":
        STRICT_LPNS_CONFIG[
            "max_reference_size"
        ],

    "k_neighbors":
        STRICT_LPNS_CONFIG[
            "k_neighbors"
        ],

    "lpns_features": [
        "expert local MAD",
        "expert local SD",
        "expert decision-flip rate",
        "aggregate local instability",
        "cross-expert stability gap",
        "cross-expert stability CV",
    ],

    "ablation": [
        "confidence",
        "confidence + disagreement",
        "confidence + disagreement + LPNS",
        "confidence + disagreement + LPNS + routing",
    ],

    "outer_validation_in_neighborhood_bank":
        False,

    "reliability_fit_in_neighborhood_bank":
        False,

    "nearest_neighbor_variable":
        "nbr_model",

    "torch_nn_namespace_preserved":
        True,

    "holdout_used":
        False,
}


save_json(

    module13c_manifest,

    LOG_DIR
    / "module13c_manifest.json"
)


print(
    "\nMODULE 13C COMPLETE."
)

torch.nn namespace: <module 'torch.nn' from '/usr/local/lib/python3.13/dist-packages/torch/nn/__init__.py'>

STRICT LPNS-MoE OUTER FOLD 1/5

Building strict fold-local expert probabilities...

Outer-training partitions:
  Gate fit: 96600
  Threshold: 17048
  Reliability: 48707
  Outer validation: 40589

Training fold-local MoE gate...
    gate epoch 1/4 loss=1.465729
    gate epoch 2/4 loss=1.411385
    gate epoch 3/4 loss=1.399872
    gate epoch 4/4 loss=1.395049

Selected disease threshold: 0.2099999999999999

Building fold-local neighborhood representation...
  Reference bank: 80000
  Querying reliability-fit neighborhoods...
  Querying outer-validation neighborhoods...
  Reliability LPNS coverage: 0.9998357525612335
  Validation LPNS coverage: 0.9994579812264407

Fold error-detection performance:
  Confidence          PR = 0.557700
  + disagreement      PR = 0.565132
  + strict LPNS       PR = 0.573674
  + LPNS + routing    PR = 0.572918
  LPNS ΔPR            = +0.008542

STRICT LP

,Reliability_Model,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.566095,0.905832,0.028078
1,Confidence+Disagreement,0.571643,0.906044,0.028045
2,Strict_LPNS_MoE,0.583232,0.906469,0.027944
3,Strict_LPNS_MoE+Routing,0.583468,0.906375,0.027951



STRICT LPNS INCREMENTAL VALUE


,Comparison,Delta_PR,Delta_ROC,Delta_AURC
0,Disagreement vs Confidence,0.005548,0.000212,-0.000034
1,Strict LPNS vs Disagreement,0.011589,0.000426,-0.000100
2,Routing vs Strict LPNS,0.000236,-0.000095,0.000007



STRICT LPNS SELECTIVE / DEFERRAL RESULTS


...


LPNS PR improved folds: 5 / 5
LPNS ROC improved folds: 4 / 5

Mean strict LPNS ΔPR: 0.011688307100495333
Mean strict LPNS ΔROC: 0.00038082094147193766

STRICT LPNS NEIGHBORHOOD DIAGNOSTICS


,Fold,Reference_N,Reliability_LPNS_Coverage,Validation_LPNS_Coverage,Reliability_MeanNeighbors,Validation_MeanNeighbors
0,0,80000,0.999836,0.999458,19.993656,19.985144
1,1,80000,0.999754,0.999877,19.991952,19.994506
2,2,80000,0.999713,0.999901,19.990042,19.994087
3,3,80000,0.999713,0.999680,19.991459,19.989307
4,4,80000,0.999692,0.999803,19.990392,19.993594



STRICT MoE DISEASE PERFORMANCE


,Model,PR_AUC,ROC_AUC
0,Strict_RA_MoE,0.380301,0.850519



MODULE 13C COMPLETE.


In [26]:
# ============================================================
# MODULE 13D — LPNS-MoE STATISTICAL VALIDATION
# Paired bootstrap, confidence intervals, selective-review
# uncertainty, fold consistency, and locked architecture ablation
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Statistically validate the STRICT Module 13C result without
# changing the architecture or touching HOLDOUT.
#
#
# PRIMARY COMPARISON
# ------------------------------------------------------------
#
#   Confidence + Disagreement
#              vs
#   Strict LPNS-MoE
#
#
# PRIMARY ENDPOINT
# ------------------------------------------------------------
#
#   Error PR-AUC
#
#
# SECONDARY ENDPOINTS
# ------------------------------------------------------------
#
#   Error ROC-AUC
#   AURC
#   Error capture at 5%, 10%, 15%, 20% review
#   Error yield per 100 reviews
#   Retained F1 / MCC
#
#
# IMPORTANT
# ------------------------------------------------------------
# Architecture is LOCKED.
#
# This module DOES NOT:
#   - tune LPNS
#   - change features
#   - alter thresholds
#   - alter routing
#   - touch HOLDOUT
#
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 13D.1 REQUIREMENTS
# ============================================================

required_13d = [
    "STRICT_ERROR_OOF",
    "STRICT_RISK_CONF",
    "STRICT_RISK_DISAGREE",
    "STRICT_RISK_LPNS",
    "STRICT_RISK_LPNS_ROUTING",
    "STRICT_MOE_OOF",
    "STRICT_THRESHOLD_OOF",
    "STRICT_PREDICTION",
    "STRICT_FN",
    "strict_fold_results",
    "fold_assignment",
    "y_dev",
    "N_SPLITS",
    "SEED",
]


missing = [
    x
    for x in required_13d
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Run Module 13C first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 13D.2 CONFIG
# ============================================================

STAT_CONFIG = {

    # --------------------------------------------------------
    # Bootstrap
    #
    # 2000 is publication-grade while still practical.
    # --------------------------------------------------------

    "n_bootstrap":
        2000,

    "ci_lower":
        2.5,

    "ci_upper":
        97.5,

    "seed":
        SEED + 130000,

    # --------------------------------------------------------
    # Review workloads
    # --------------------------------------------------------

    "review_rates":
        [0.05, 0.10, 0.15, 0.20],

    # --------------------------------------------------------
    # Primary method comparison
    # --------------------------------------------------------

    "baseline":
        "Confidence+Disagreement",

    "proposed":
        "Strict_LPNS_MoE",
}


# ============================================================
# 13D.3 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


error_target = np.asarray(
    STRICT_ERROR_OOF,
    dtype=int
)


prediction = np.asarray(
    STRICT_PREDICTION,
    dtype=int
)


fn_target = np.asarray(
    STRICT_FN,
    dtype=int
)


N = len(
    y_all
)


risk_models = {

    "Confidence":
        np.asarray(
            STRICT_RISK_CONF,
            dtype=float
        ),

    "Confidence+Disagreement":
        np.asarray(
            STRICT_RISK_DISAGREE,
            dtype=float
        ),

    "Strict_LPNS_MoE":
        np.asarray(
            STRICT_RISK_LPNS,
            dtype=float
        ),

    "Strict_LPNS_MoE+Routing":
        np.asarray(
            STRICT_RISK_LPNS_ROUTING,
            dtype=float
        ),
}


baseline_risk = (
    risk_models[
        STAT_CONFIG[
            "baseline"
        ]
    ]
)


proposed_risk = (
    risk_models[
        STAT_CONFIG[
            "proposed"
        ]
    ]
)


print(
    "N:",
    N
)

print(
    "Error prevalence:",
    error_target.mean()
)

print(
    "False negatives:",
    fn_target.sum()
)


# ============================================================
# 13D.4 AURC
# ============================================================

def stat_aurc(
    error,
    risk
):

    error = np.asarray(
        error,
        dtype=float
    )


    risk = np.asarray(
        risk,
        dtype=float
    )


    # lowest predicted risk retained first
    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(
        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 13D.5 METRIC HELPERS
# ============================================================

def metric_pr(
    y,
    score
):

    return average_precision_score(
        y,
        score
    )


def metric_roc(
    y,
    score
):

    return roc_auc_score(
        y,
        score
    )


def metric_aurc(
    y,
    score
):

    return stat_aurc(
        y,
        score
    )


# ============================================================
# 13D.6 OBSERVED GLOBAL METRICS
# ============================================================

observed_rows = []


for name, score in (
    risk_models.items()
):

    observed_rows.append({

        "Method":
            name,

        "Error_PR_AUC":
            metric_pr(
                error_target,
                score
            ),

        "Error_ROC_AUC":
            metric_roc(
                error_target,
                score
            ),

        "AURC":
            metric_aurc(
                error_target,
                score
            ),
    })


observed_results = pd.DataFrame(
    observed_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 13D — OBSERVED LOCKED RELIABILITY PERFORMANCE"
)

print(
    "=" * 135
)


display(
    observed_results
)


# ============================================================
# 13D.7 STRATIFIED PAIRED BOOTSTRAP INDICES
# ============================================================
#
# Same bootstrap sample used for BOTH methods.
#
# Preserve error/non-error class proportions approximately by
# resampling within the two strata.
#
# ============================================================

error_idx = np.where(
    error_target == 1
)[0]


correct_idx = np.where(
    error_target == 0
)[0]


rng = np.random.default_rng(
    STAT_CONFIG[
        "seed"
    ]
)


def draw_stratified_bootstrap():

    sampled_error = (
        rng.choice(
            error_idx,
            size=len(error_idx),
            replace=True
        )
    )


    sampled_correct = (
        rng.choice(
            correct_idx,
            size=len(correct_idx),
            replace=True
        )
    )


    idx = np.concatenate([
        sampled_error,
        sampled_correct,
    ])


    return idx


# ============================================================
# 13D.8 BOOTSTRAP PRIMARY METRICS
# ============================================================

B = (
    STAT_CONFIG[
        "n_bootstrap"
    ]
)


boot_base_pr = np.zeros(
    B,
    dtype=float
)


boot_lpns_pr = np.zeros(
    B,
    dtype=float
)


boot_base_roc = np.zeros(
    B,
    dtype=float
)


boot_lpns_roc = np.zeros(
    B,
    dtype=float
)


boot_base_aurc = np.zeros(
    B,
    dtype=float
)


boot_lpns_aurc = np.zeros(
    B,
    dtype=float
)


print(
    "\nRunning paired stratified bootstrap..."
)


for b in range(
    B
):

    idx = (
        draw_stratified_bootstrap()
    )


    y_b = (
        error_target[
            idx
        ]
    )


    base_b = (
        baseline_risk[
            idx
        ]
    )


    lpns_b = (
        proposed_risk[
            idx
        ]
    )


    boot_base_pr[
        b
    ] = metric_pr(
        y_b,
        base_b
    )


    boot_lpns_pr[
        b
    ] = metric_pr(
        y_b,
        lpns_b
    )


    boot_base_roc[
        b
    ] = metric_roc(
        y_b,
        base_b
    )


    boot_lpns_roc[
        b
    ] = metric_roc(
        y_b,
        lpns_b
    )


    boot_base_aurc[
        b
    ] = metric_aurc(
        y_b,
        base_b
    )


    boot_lpns_aurc[
        b
    ] = metric_aurc(
        y_b,
        lpns_b
    )


    if (
        b + 1
    ) % 250 == 0:

        print(
            f"  bootstrap "
            f"{b+1:,}/{B:,}"
        )


# ============================================================
# 13D.9 CI HELPER
# ============================================================

def percentile_ci(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    return (

        float(
            np.percentile(
                values,
                STAT_CONFIG[
                    "ci_lower"
                ]
            )
        ),

        float(
            np.percentile(
                values,
                STAT_CONFIG[
                    "ci_upper"
                ]
            )
        ),
    )


# ============================================================
# 13D.10 PAIRED DELTAS
# ============================================================

delta_pr = (
    boot_lpns_pr
    -
    boot_base_pr
)


delta_roc = (
    boot_lpns_roc
    -
    boot_base_roc
)


# Lower AURC is better
delta_aurc = (
    boot_lpns_aurc
    -
    boot_base_aurc
)


# ============================================================
# 13D.11 BOOTSTRAP P-VALUE
# ============================================================
#
# Two-sided paired bootstrap probability around zero.
#
# ============================================================

def paired_bootstrap_p(
    delta
):

    delta = np.asarray(
        delta,
        dtype=float
    )


    p_lower = (
        np.mean(
            delta <= 0
        )
    )


    p_upper = (
        np.mean(
            delta >= 0
        )
    )


    p = (
        2.0
        *
        min(
            p_lower,
            p_upper
        )
    )


    return min(
        float(p),
        1.0
    )


# ============================================================
# 13D.12 PRIMARY STATISTICAL TABLE
# ============================================================

base_pr_obs = metric_pr(
    error_target,
    baseline_risk
)


lpns_pr_obs = metric_pr(
    error_target,
    proposed_risk
)


base_roc_obs = metric_roc(
    error_target,
    baseline_risk
)


lpns_roc_obs = metric_roc(
    error_target,
    proposed_risk
)


base_aurc_obs = metric_aurc(
    error_target,
    baseline_risk
)


lpns_aurc_obs = metric_aurc(
    error_target,
    proposed_risk
)


pr_ci = percentile_ci(
    delta_pr
)


roc_ci = percentile_ci(
    delta_roc
)


aurc_ci = percentile_ci(
    delta_aurc
)


statistical_results = pd.DataFrame([

    {
        "Metric":
            "Error PR-AUC",

        "Baseline":
            base_pr_obs,

        "Strict_LPNS":
            lpns_pr_obs,

        "Delta":
            lpns_pr_obs
            -
            base_pr_obs,

        "Delta_CI_Lower":
            pr_ci[0],

        "Delta_CI_Upper":
            pr_ci[1],

        "Paired_Bootstrap_P":
            paired_bootstrap_p(
                delta_pr
            ),

        "Higher_Is_Better":
            True,
    },

    {
        "Metric":
            "Error ROC-AUC",

        "Baseline":
            base_roc_obs,

        "Strict_LPNS":
            lpns_roc_obs,

        "Delta":
            lpns_roc_obs
            -
            base_roc_obs,

        "Delta_CI_Lower":
            roc_ci[0],

        "Delta_CI_Upper":
            roc_ci[1],

        "Paired_Bootstrap_P":
            paired_bootstrap_p(
                delta_roc
            ),

        "Higher_Is_Better":
            True,
    },

    {
        "Metric":
            "AURC",

        "Baseline":
            base_aurc_obs,

        "Strict_LPNS":
            lpns_aurc_obs,

        "Delta":
            lpns_aurc_obs
            -
            base_aurc_obs,

        "Delta_CI_Lower":
            aurc_ci[0],

        "Delta_CI_Upper":
            aurc_ci[1],

        "Paired_Bootstrap_P":
            paired_bootstrap_p(
                delta_aurc
            ),

        "Higher_Is_Better":
            False,
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "PAIRED BOOTSTRAP — STRICT LPNS vs CONFIDENCE+DISAGREEMENT"
)

print(
    "=" * 135
)


display(
    statistical_results
)


# ============================================================
# 13D.13 METHOD-SPECIFIC BOOTSTRAP CIs
# ============================================================

method_ci_rows = []


for (
    method,
    pr_values,
    roc_values,
    aurc_values
) in [

    (
        STAT_CONFIG[
            "baseline"
        ],
        boot_base_pr,
        boot_base_roc,
        boot_base_aurc
    ),

    (
        STAT_CONFIG[
            "proposed"
        ],
        boot_lpns_pr,
        boot_lpns_roc,
        boot_lpns_aurc
    ),
]:

    pr_method_ci = (
        percentile_ci(
            pr_values
        )
    )


    roc_method_ci = (
        percentile_ci(
            roc_values
        )
    )


    aurc_method_ci = (
        percentile_ci(
            aurc_values
        )
    )


    method_ci_rows.extend([

        {
            "Method":
                method,

            "Metric":
                "Error PR-AUC",

            "CI_Lower":
                pr_method_ci[0],

            "CI_Upper":
                pr_method_ci[1],
        },

        {
            "Method":
                method,

            "Metric":
                "Error ROC-AUC",

            "CI_Lower":
                roc_method_ci[0],

            "CI_Upper":
                roc_method_ci[1],
        },

        {
            "Method":
                method,

            "Metric":
                "AURC",

            "CI_Lower":
                aurc_method_ci[0],

            "CI_Upper":
                aurc_method_ci[1],
        },
    ])


method_metric_ci = pd.DataFrame(
    method_ci_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "BOOTSTRAP 95% CI — INDIVIDUAL METHODS"
)

print(
    "=" * 135
)


display(
    method_metric_ci
)


# ============================================================
# 13D.14 SELECTIVE REVIEW HELPER
# ============================================================

def selective_statistics(
    risk,
    rate
):

    n_review = max(

        1,

        int(
            round(
                N
                *
                rate
            )
        )
    )


    review_idx = (
        np.argsort(
            -risk
        )[
            :n_review
        ]
    )


    keep = np.ones(
        N,
        dtype=bool
    )


    keep[
        review_idx
    ] = False


    total_error = max(
        error_target.sum(),
        1
    )


    total_fn = max(
        fn_target.sum(),
        1
    )


    error_found = (
        error_target[
            review_idx
        ].sum()
    )


    fn_found = (
        fn_target[
            review_idx
        ].sum()
    )


    return {

        "Reviewed_N":
            n_review,

        "Errors_Captured":
            error_found
            /
            total_error,

        "FalseNegatives_Captured":
            fn_found
            /
            total_fn,

        "Error_Per_100_Reviews":
            error_found
            /
            n_review
            *
            100.0,

        "Retained_Accuracy":
            accuracy_score(

                y_all[
                    keep
                ],

                prediction[
                    keep
                ]
            ),

        "Retained_F1":
            f1_score(

                y_all[
                    keep
                ],

                prediction[
                    keep
                ],

                zero_division=0
            ),

        "Retained_MCC":
            matthews_corrcoef(

                y_all[
                    keep
                ],

                prediction[
                    keep
                ]
            ),
    }


# ============================================================
# 13D.15 OBSERVED SELECTIVE DIFFERENCES
# ============================================================

selective_difference_rows = []


for rate in (
    STAT_CONFIG[
        "review_rates"
    ]
):

    base_stats = (
        selective_statistics(
            baseline_risk,
            rate
        )
    )


    lpns_stats = (
        selective_statistics(
            proposed_risk,
            rate
        )
    )


    selective_difference_rows.append({

        "Review_Rate":
            rate,

        "Baseline_Error_Capture":
            base_stats[
                "Errors_Captured"
            ],

        "LPNS_Error_Capture":
            lpns_stats[
                "Errors_Captured"
            ],

        "Delta_Error_Capture":
            lpns_stats[
                "Errors_Captured"
            ]
            -
            base_stats[
                "Errors_Captured"
            ],

        "Baseline_Error_Per_100":
            base_stats[
                "Error_Per_100_Reviews"
            ],

        "LPNS_Error_Per_100":
            lpns_stats[
                "Error_Per_100_Reviews"
            ],

        "Delta_Error_Per_100":
            lpns_stats[
                "Error_Per_100_Reviews"
            ]
            -
            base_stats[
                "Error_Per_100_Reviews"
            ],

        "Baseline_FN_Capture":
            base_stats[
                "FalseNegatives_Captured"
            ],

        "LPNS_FN_Capture":
            lpns_stats[
                "FalseNegatives_Captured"
            ],

        "Delta_FN_Capture":
            lpns_stats[
                "FalseNegatives_Captured"
            ]
            -
            base_stats[
                "FalseNegatives_Captured"
            ],

        "Baseline_Retained_F1":
            base_stats[
                "Retained_F1"
            ],

        "LPNS_Retained_F1":
            lpns_stats[
                "Retained_F1"
            ],

        "Baseline_Retained_MCC":
            base_stats[
                "Retained_MCC"
            ],

        "LPNS_Retained_MCC":
            lpns_stats[
                "Retained_MCC"
            ],
    })


selective_difference = pd.DataFrame(
    selective_difference_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "OBSERVED SELECTIVE-REVIEW BENEFIT"
)

print(
    "=" * 135
)


display(
    selective_difference
)


# ============================================================
# 13D.16 BOOTSTRAP SELECTIVE REVIEW
# ============================================================
#
# Important:
#
# On each bootstrap sample:
#
#   - preserve paired risk scores
#   - recompute top X% review ranking
#   - recompute captured errors
#
# ============================================================

review_bootstrap = {

    rate: {
        "base_capture": [],
        "lpns_capture": [],
        "delta_capture": [],
        "base_yield": [],
        "lpns_yield": [],
        "delta_yield": [],
    }

    for rate in (
        STAT_CONFIG[
            "review_rates"
        ]
    )
}


rng_selective = np.random.default_rng(
    STAT_CONFIG[
        "seed"
    ]
    +
    555
)


print(
    "\nRunning selective-review bootstrap..."
)


for b in range(
    B
):

    sampled_error = (
        rng_selective.choice(
            error_idx,
            size=len(error_idx),
            replace=True
        )
    )


    sampled_correct = (
        rng_selective.choice(
            correct_idx,
            size=len(correct_idx),
            replace=True
        )
    )


    idx = np.concatenate([
        sampled_error,
        sampled_correct,
    ])


    err_b = (
        error_target[
            idx
        ]
    )


    base_b = (
        baseline_risk[
            idx
        ]
    )


    lpns_b = (
        proposed_risk[
            idx
        ]
    )


    total_err_b = max(
        err_b.sum(),
        1
    )


    n_b = len(
        idx
    )


    for rate in (
        STAT_CONFIG[
            "review_rates"
        ]
    ):

        n_review = max(

            1,

            int(
                round(
                    n_b
                    *
                    rate
                )
            )
        )


        base_review = (
            np.argsort(
                -base_b
            )[
                :n_review
            ]
        )


        lpns_review = (
            np.argsort(
                -lpns_b
            )[
                :n_review
            ]
        )


        base_errors = (
            err_b[
                base_review
            ].sum()
        )


        lpns_errors = (
            err_b[
                lpns_review
            ].sum()
        )


        base_capture = (
            base_errors
            /
            total_err_b
        )


        lpns_capture = (
            lpns_errors
            /
            total_err_b
        )


        base_yield = (
            base_errors
            /
            n_review
            *
            100.0
        )


        lpns_yield = (
            lpns_errors
            /
            n_review
            *
            100.0
        )


        review_bootstrap[
            rate
        ][
            "base_capture"
        ].append(
            base_capture
        )


        review_bootstrap[
            rate
        ][
            "lpns_capture"
        ].append(
            lpns_capture
        )


        review_bootstrap[
            rate
        ][
            "delta_capture"
        ].append(
            lpns_capture
            -
            base_capture
        )


        review_bootstrap[
            rate
        ][
            "base_yield"
        ].append(
            base_yield
        )


        review_bootstrap[
            rate
        ][
            "lpns_yield"
        ].append(
            lpns_yield
        )


        review_bootstrap[
            rate
        ][
            "delta_yield"
        ].append(
            lpns_yield
            -
            base_yield
        )


    if (
        b + 1
    ) % 250 == 0:

        print(
            f"  selective bootstrap "
            f"{b+1:,}/{B:,}"
        )


# ============================================================
# 13D.17 SELECTIVE BOOTSTRAP TABLE
# ============================================================

selective_ci_rows = []


for rate in (
    STAT_CONFIG[
        "review_rates"
    ]
):

    delta_capture_values = np.asarray(

        review_bootstrap[
            rate
        ][
            "delta_capture"
        ]
    )


    delta_yield_values = np.asarray(

        review_bootstrap[
            rate
        ][
            "delta_yield"
        ]
    )


    capture_ci = (
        percentile_ci(
            delta_capture_values
        )
    )


    yield_ci = (
        percentile_ci(
            delta_yield_values
        )
    )


    observed_row = (

        selective_difference[
            selective_difference[
                "Review_Rate"
            ]
            ==
            rate
        ]
        .iloc[0]
    )


    selective_ci_rows.append({

        "Review_Rate":
            rate,

        "Delta_Error_Capture":
            observed_row[
                "Delta_Error_Capture"
            ],

        "Capture_CI_Lower":
            capture_ci[0],

        "Capture_CI_Upper":
            capture_ci[1],

        "Capture_P":
            paired_bootstrap_p(
                delta_capture_values
            ),

        "Delta_Error_Per_100":
            observed_row[
                "Delta_Error_Per_100"
            ],

        "Yield_CI_Lower":
            yield_ci[0],

        "Yield_CI_Upper":
            yield_ci[1],

        "Yield_P":
            paired_bootstrap_p(
                delta_yield_values
            ),
    })


selective_bootstrap_results = pd.DataFrame(
    selective_ci_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "PAIRED BOOTSTRAP — SELECTIVE REVIEW BENEFIT"
)

print(
    "=" * 135
)


display(
    selective_bootstrap_results
)


# ============================================================
# 13D.18 FOLD-LEVEL EFFECT SUMMARY
# ============================================================

fold_effect = (
    strict_fold_results[
        [
            "Fold",
            "Error_Prevalence",
            "Disagreement_PR",
            "LPNS_PR",
            "LPNS_Delta_PR_vs_Disagreement",
            "Disagreement_ROC",
            "LPNS_ROC",
            "LPNS_Delta_ROC_vs_Disagreement",
        ]
    ]
    .copy()
)


fold_summary = pd.DataFrame([

    {
        "Metric":
            "PR delta",

        "Mean":
            fold_effect[
                "LPNS_Delta_PR_vs_Disagreement"
            ].mean(),

        "SD":
            fold_effect[
                "LPNS_Delta_PR_vs_Disagreement"
            ].std(
                ddof=1
            ),

        "Min":
            fold_effect[
                "LPNS_Delta_PR_vs_Disagreement"
            ].min(),

        "Max":
            fold_effect[
                "LPNS_Delta_PR_vs_Disagreement"
            ].max(),

        "Positive_Folds":
            int(
                (
                    fold_effect[
                        "LPNS_Delta_PR_vs_Disagreement"
                    ]
                    >
                    0
                ).sum()
            ),

        "Total_Folds":
            N_SPLITS,
    },

    {
        "Metric":
            "ROC delta",

        "Mean":
            fold_effect[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].mean(),

        "SD":
            fold_effect[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].std(
                ddof=1
            ),

        "Min":
            fold_effect[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].min(),

        "Max":
            fold_effect[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].max(),

        "Positive_Folds":
            int(
                (
                    fold_effect[
                        "LPNS_Delta_ROC_vs_Disagreement"
                    ]
                    >
                    0
                ).sum()
            ),

        "Total_Folds":
            N_SPLITS,
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FOLD-LEVEL LPNS EFFECT CONSISTENCY"
)

print(
    "=" * 135
)


display(
    fold_summary
)


# ============================================================
# 13D.19 ARCHITECTURE ABLATION
# ============================================================
#
# Existing locked variants:
#
#   Confidence
#   + disagreement
#   + LPNS
#   + routing
#
# This avoids further architecture tuning.
# ============================================================

ablation_table = (
    observed_results
    .copy()
)


confidence_row = (

    ablation_table[
        ablation_table[
            "Method"
        ]
        ==
        "Confidence"
    ]
    .iloc[0]
)


ablation_table[
    "Delta_PR_vs_Confidence"
] = (

    ablation_table[
        "Error_PR_AUC"
    ]

    -
    confidence_row[
        "Error_PR_AUC"
    ]
)


ablation_table[
    "Delta_ROC_vs_Confidence"
] = (

    ablation_table[
        "Error_ROC_AUC"
    ]

    -
    confidence_row[
        "Error_ROC_AUC"
    ]
)


ablation_table[
    "Delta_AURC_vs_Confidence"
] = (

    ablation_table[
        "AURC"
    ]

    -
    confidence_row[
        "AURC"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "LOCKED RELIABILITY ARCHITECTURE ABLATION"
)

print(
    "=" * 135
)


display(
    ablation_table
)


# ============================================================
# 13D.20 RELATIVE EFFECT SIZES
# ============================================================

effect_size_results = pd.DataFrame([

    {
        "Metric":
            "Error PR-AUC",

        "Absolute_Gain":
            lpns_pr_obs
            -
            base_pr_obs,

        "Relative_Gain_Percent":
            (
                (
                    lpns_pr_obs
                    -
                    base_pr_obs
                )
                /
                base_pr_obs
                *
                100.0
            ),
    },

    {
        "Metric":
            "Error ROC-AUC",

        "Absolute_Gain":
            lpns_roc_obs
            -
            base_roc_obs,

        "Relative_Gain_Percent":
            (
                (
                    lpns_roc_obs
                    -
                    base_roc_obs
                )
                /
                base_roc_obs
                *
                100.0
            ),
    },

    {
        "Metric":
            "AURC reduction",

        "Absolute_Gain":
            base_aurc_obs
            -
            lpns_aurc_obs,

        "Relative_Gain_Percent":
            (
                (
                    base_aurc_obs
                    -
                    lpns_aurc_obs
                )
                /
                base_aurc_obs
                *
                100.0
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "LPNS EFFECT SIZE"
)

print(
    "=" * 135
)


display(
    effect_size_results
)


# ============================================================
# 13D.21 DECISION SUMMARY
# ============================================================

primary_significant = (

    pr_ci[0]
    >
    0
)


all_pr_folds_positive = (

    (
        strict_fold_results[
            "LPNS_Delta_PR_vs_Disagreement"
        ]
        >
        0
    ).all()
)


decision_summary = pd.DataFrame([

    {
        "Criterion":
            "Strict pooled PR gain > 0",

        "Passed":
            (
                lpns_pr_obs
                >
                base_pr_obs
            ),

        "Value":
            (
                lpns_pr_obs
                -
                base_pr_obs
            ),
    },

    {
        "Criterion":
            "95% bootstrap CI for PR delta excludes 0",

        "Passed":
            primary_significant,

        "Value":
            f"[{pr_ci[0]:.6f}, {pr_ci[1]:.6f}]",
    },

    {
        "Criterion":
            "PR improved in every fold",

        "Passed":
            all_pr_folds_positive,

        "Value":
            (
                f"{int((strict_fold_results['LPNS_Delta_PR_vs_Disagreement'] > 0).sum())}"
                f"/{N_SPLITS}"
            ),
    },

    {
        "Criterion":
            "Routing materially improves LPNS",

        "Passed":
            False,

        "Value":
            (
                observed_results.loc[
                    observed_results[
                        "Method"
                    ]
                    ==
                    "Strict_LPNS_MoE+Routing",
                    "Error_PR_AUC"
                ].iloc[0]

                -

                lpns_pr_obs
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "LOCKED-ARCHITECTURE DECISION SUMMARY"
)

print(
    "=" * 135
)


display(
    decision_summary
)


# ============================================================
# 13D.22 SAVE
# ============================================================

save_csv(

    observed_results,

    TAB_DIR
    / "module13d_observed_reliability.csv"
)


save_csv(

    statistical_results,

    TAB_DIR
    / "module13d_paired_bootstrap_primary.csv"
)


save_csv(

    method_metric_ci,

    TAB_DIR
    / "module13d_method_metric_ci.csv"
)


save_csv(

    selective_difference,

    TAB_DIR
    / "module13d_selective_observed.csv"
)


save_csv(

    selective_bootstrap_results,

    TAB_DIR
    / "module13d_selective_bootstrap.csv"
)


save_csv(

    fold_effect,

    TAB_DIR
    / "module13d_fold_effects.csv"
)


save_csv(

    fold_summary,

    TAB_DIR
    / "module13d_fold_consistency_summary.csv"
)


save_csv(

    ablation_table,

    TAB_DIR
    / "module13d_locked_architecture_ablation.csv"
)


save_csv(

    effect_size_results,

    TAB_DIR
    / "module13d_effect_sizes.csv"
)


save_csv(

    decision_summary,

    TAB_DIR
    / "module13d_decision_summary.csv"
)


# ============================================================
# SAVE BOOTSTRAP DISTRIBUTIONS
# ============================================================

bootstrap_distribution = pd.DataFrame({

    "Bootstrap":
        np.arange(
            B
        ),

    "Baseline_PR":
        boot_base_pr,

    "LPNS_PR":
        boot_lpns_pr,

    "Delta_PR":
        delta_pr,

    "Baseline_ROC":
        boot_base_roc,

    "LPNS_ROC":
        boot_lpns_roc,

    "Delta_ROC":
        delta_roc,

    "Baseline_AURC":
        boot_base_aurc,

    "LPNS_AURC":
        boot_lpns_aurc,

    "Delta_AURC":
        delta_aurc,
})


save_csv(

    bootstrap_distribution,

    PRED_DIR
    / "module13d_bootstrap_distribution.csv"
)


# ============================================================
# MANIFEST
# ============================================================

module13d_manifest = {

    "method":
        "Statistical validation of Strict LPNS-MoE",

    "architecture_status":
        "locked",

    "primary_comparison":
        "Strict LPNS-MoE vs confidence + disagreement",

    "primary_metric":
        "error PR-AUC",

    "bootstrap_type":
        "paired stratified bootstrap",

    "bootstrap_iterations":
        STAT_CONFIG[
            "n_bootstrap"
        ],

    "confidence_interval":
        "percentile 95%",

    "review_rates":
        STAT_CONFIG[
            "review_rates"
        ],

    "architecture_variants": [
        "confidence",
        "confidence + disagreement",
        "confidence + disagreement + LPNS",
        "confidence + disagreement + LPNS + routing",
    ],

    "hyperparameter_tuning_performed":
        False,

    "holdout_used":
        False,
}


save_json(

    module13d_manifest,

    LOG_DIR
    / "module13d_manifest.json"
)


print(
    "\nMODULE 13D COMPLETE."
)

N: 202944
Error prevalence: 0.14649361400189215
False negatives: 8375

MODULE 13D — OBSERVED LOCKED RELIABILITY PERFORMANCE


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.566095,0.905832,0.028078
1,Confidence+Disagreement,0.571643,0.906044,0.028045
2,Strict_LPNS_MoE,0.583232,0.906469,0.027944
3,Strict_LPNS_MoE+Routing,0.583468,0.906375,0.027951



Running paired stratified bootstrap...
  bootstrap 250/2,000
  bootstrap 500/2,000
  bootstrap 750/2,000
  bootstrap 1,000/2,000
  bootstrap 1,250/2,000
  bootstrap 1,500/2,000
  bootstrap 1,750/2,000
  bootstrap 2,000/2,000

PAIRED BOOTSTRAP — STRICT LPNS vs CONFIDENCE+DISAGREEMENT


,Metric,Baseline,Strict_LPNS,Delta,Delta_CI_Lower,Delta_CI_Upper,Paired_Bootstrap_P,Higher_Is_Better
0,Error PR-AUC,0.571643,0.583232,0.011589,0.008763,0.014432,0.000,True
1,Error ROC-AUC,0.906044,0.906469,0.000426,0.000203,0.000648,0.000,True
2,AURC,0.028045,0.027944,-0.000100,-0.000194,-0.000008,0.037,False



BOOTSTRAP 95% CI — INDIVIDUAL METHODS


...


LPNS EFFECT SIZE


,Metric,Absolute_Gain,Relative_Gain_Percent
0,Error PR-AUC,0.011589,2.027255
1,Error ROC-AUC,0.000426,0.046970
2,AURC reduction,0.000100,0.358024



LOCKED-ARCHITECTURE DECISION SUMMARY


,Criterion,Passed,Value
0,Strict pooled PR gain > 0,True,0.011589
1,95% bootstrap CI for PR delta excludes 0,True,"[0.008763, 0.014432]"
2,PR improved in every fold,True,5/5
3,Routing materially improves LPNS,False,0.000236



MODULE 13D COMPLETE.


In [30]:
# ============================================================
# NEURAL REPAIR — ENVIRONMENT INSPECTION
# AutoInt + DeepFM_v2
# ============================================================

import inspect
import pandas as pd

keywords = [
    "autoint",
    "deepfm",
    "deep_fm",
    "tabular",
    "train",
    "fit",
    "predict",
    "dataset",
    "loader",
]

matches = []

# Freeze globals to avoid:
# RuntimeError: dictionary changed size during iteration
global_items = list(globals().items())

for name, obj in global_items:

    lname = name.lower()

    if any(k in lname for k in keywords):

        obj_type = type(obj).__name__

        module_name = getattr(
            obj,
            "__module__",
            None
        )

        signature = None

        try:
            if callable(obj):
                signature = str(
                    inspect.signature(obj)
                )
        except Exception:
            signature = None

        matches.append({
            "name": name,
            "type": obj_type,
            "module": module_name,
            "signature": signature,
        })


inspection_df = (
    pd.DataFrame(matches)
    .sort_values("name")
    .reset_index(drop=True)
)


print("=" * 120)
print("POTENTIAL AUTOINT / DEEPFM / TRAINING OBJECTS")
print("=" * 120)

mask = (
    inspection_df["name"]
    .str.lower()
    .str.contains(
        "autoint|deepfm|deep_fm|train|predict|loader|dataset",
        regex=True
    )
)

display(
    inspection_df.loc[mask]
)


print("\n" + "=" * 120)
print("LIKELY AUTOINT / DEEPFM OBJECTS")
print("=" * 120)

for _, row in inspection_df.iterrows():

    lname = row["name"].lower()

    if (
        "autoint" in lname
        or "deepfm" in lname
        or "deep_fm" in lname
    ):

        print(f"\n{row['name']}")
        print("  type      :", row["type"])
        print("  module    :", row["module"])
        print("  signature :", row["signature"])


print("\n" + "=" * 120)
print("IMPORTANT TRAINING CONFIG OBJECTS")
print("=" * 120)

for name in [
    "AUTOINT_CONFIG",
    "DEEPFM_CONFIG",
    "DEEPFM_V2_CONFIG",
    "NN_CONFIG",
    "DEEP_CONFIG",
    "DEVICE",
    "BATCH_SIZE",
    "SEED",
    "N_SPLITS",
    "fold_assignment",
]:

    if name in globals():

        obj = globals()[name]

        print(
            f"{name}:",
            obj
        )

POTENTIAL AUTOINT / DEEPFM / TRAINING OBJECTS


,name,type,module,signature
0,AutoIntModel,type,__main__,"(cardinalities, n_cont, embedding_dim=16, num_..."
1,DEEPFM_V2_CONFIG,dict,None,None
2,DataLoader,type,torch.utils.data.dataloader,"(dataset: 'Dataset[_T_co]', batch_size: 'int |..."
3,Dataset,type,torch.utils.data.dataset,()
4,DeepFMModel,type,__main__,"(cardinalities, n_cont, embedding_dim=16, hidd..."
5,DeepTabularDataset,type,__main__,"(X_cat, X_cont, y=None)"
6,F_conf_train,ndarray,None,None
7,F_expert_train,ndarray,None,None
8,F_full_train,ndarray,None,None
10,P_outer_train,ndarray,None,None



LIKELY AUTOINT / DEEPFM OBJECTS

AutoIntModel
  type      : type
  module    : __main__
  signature : (cardinalities, n_cont, embedding_dim=16, num_heads=2, num_layers=2, hidden_dims=(128, 64), dropout=0.2)

DEEPFM_V2_CONFIG
  type      : dict
  module    : None
  signature : None

DeepFMModel
  type      : type
  module    : __main__
  signature : (cardinalities, n_cont, embedding_dim=16, hidden_dims=(128, 64), dropout=0.2)

StableDeepFM
  type      : type
  module    : __main__
  signature : (cardinalities, n_cont, embedding_dim=8, hidden_dims=(256, 128), dropout=0.1, init_std=0.001)

deepfm_v2_folds
  type      : DataFrame
  module    : pandas.core.frame
  signature : None

train_deepfm_v2_fold
  type      : function
  module    : __main__
  signature : (fold_id, config)

IMPORTANT TRAINING CONFIG OBJECTS
DEEPFM_V2_CONFIG: {'embedding_dim': 8, 'hidden_dims': (256, 128), 'dropout': 0.1, 'init_std': 0.001, 'learning_rate': 0.0005, 'weight_decay': 1e-06, 'batch_size': 1024, 'max_epoch

In [31]:
# ============================================================
# SOURCE INSPECTION
# ============================================================

import inspect

targets = []

global_items = list(globals().items())

for name, obj in global_items:

    lname = name.lower()

    if (
        "autoint" in lname
        or "deepfm" in lname
        or "deep_fm" in lname
    ):

        if callable(obj):
            targets.append(name)


for name in sorted(set(targets)):

    obj = globals()[name]

    print(
        "\n"
        + "=" * 120
    )

    print(
        "SOURCE:",
        name
    )

    print(
        "=" * 120
    )

    try:

        src = inspect.getsource(obj)

        print(
            src[:12000]
        )

    except Exception as e:

        print(
            "Source unavailable:",
            repr(e)
        )


SOURCE: AutoIntModel
Source unavailable: OSError('source code not available')

SOURCE: DeepFMModel
Source unavailable: OSError('source code not available')

SOURCE: StableDeepFM
Source unavailable: OSError('source code not available')

SOURCE: train_deepfm_v2_fold
def train_deepfm_v2_fold(
    fold_id,
    config
):

    seed_everything(
        SEED + fold_id
    )

    data = prepare_deep_tabular_fold(
        fold_id
    )

    train_ds = DeepTabularDataset(
        data["Xtr_cat"],
        data["Xtr_cont"],
        data["ytr"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(),
    )

    model = StableDeepFM(
        cardinalities=
            CAT_CARDINALITIES,

        n_cont=
            len(CONT_COLS_DEEP),

        embedding_dim=
            config["embedding_dim"],

        hidden_dims=
  

In [33]:
# ============================================================
# AUTOINT / DEEP TABULAR CONFIG INSPECTION
# ============================================================

candidate_configs = []

for name, obj in list(globals().items()):

    lname = name.lower()

    if (
        isinstance(obj, dict)
        and
        (
            "autoint" in lname
            or
            "deep" in lname
            or
            "tabular" in lname
        )
    ):

        candidate_configs.append(
            name
        )


print(
    "Candidate configs:",
    candidate_configs
)


for name in candidate_configs:

    print(
        "\n",
        name,
        "=",
        globals()[name]
    )

Candidate configs: ['DEEP_CARDINALITIES', 'DEEP_CONFIGS', 'MODULE7_DEEP_RESULTS', 'DEEPFM_V2_CONFIG']

 DEEP_CARDINALITIES = {'HighBP': {'n_unique': 2, 'values': [0.0, 1.0]}, 'HighChol': {'n_unique': 2, 'values': [0.0, 1.0]}, 'CholCheck': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Smoker': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Stroke': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Diabetes': {'n_unique': 3, 'values': [0.0, 1.0, 2.0]}, 'PhysActivity': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Fruits': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Veggies': {'n_unique': 2, 'values': [0.0, 1.0]}, 'HvyAlcoholConsump': {'n_unique': 2, 'values': [0.0, 1.0]}, 'AnyHealthcare': {'n_unique': 2, 'values': [0.0, 1.0]}, 'NoDocbcCost': {'n_unique': 2, 'values': [0.0, 1.0]}, 'GenHlth': {'n_unique': 5, 'values': [1.0, 2.0, 3.0, 4.0, 5.0]}, 'DiffWalk': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Sex': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Age': {'n_unique': 13, 'values': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10

In [34]:
# ============================================================
# DEEP TABULAR PREPROCESSING OBJECTS
# ============================================================

wanted_patterns = [
    "cardinal",
    "cat_cols",
    "cont_cols",
    "deep_cat",
    "deep_cont",
    "preprocess",
    "category",
]

for name, obj in list(globals().items()):

    lname = name.lower()

    if any(
        p in lname
        for p in wanted_patterns
    ):

        print(
            name,
            ":",
            type(obj).__name__
        )

        try:

            if isinstance(
                obj,
                (
                    list,
                    tuple,
                    dict,
                    np.ndarray,
                    pd.Index,
                )
            ):

                print(
                    " ",
                    obj
                )

        except Exception:

            pass

build_linear_preprocessor : function
build_dense_nn_preprocessor : function
build_tree_preprocessor : function
build_subset_linear_preprocessor : function
DEEP_CATEGORICAL_COLS : list
  ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
DEEP_CONTINUOUS_COLS : list
  ['BMI', 'MentHlth', 'PhysHlth']
DEEP_CARDINALITIES : dict
  {'HighBP': {'n_unique': 2, 'values': [0.0, 1.0]}, 'HighChol': {'n_unique': 2, 'values': [0.0, 1.0]}, 'CholCheck': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Smoker': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Stroke': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Diabetes': {'n_unique': 3, 'values': [0.0, 1.0, 2.0]}, 'PhysActivity': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Fruits': {'n_unique': 2, 'values': [0.0, 1.0]}, 'Veggies': {'n_unique': 2, 'values': [0.0, 1.0]}, 'HvyAlcoholConsump': {'n_unique': 2, 'values'

In [35]:
# ============================================================
# FINAL SOURCE INSPECTION BEFORE NEURAL REPAIR
# ============================================================

import inspect

targets = [
    "AutoIntModel",
    "StableDeepFM",
    "train_deep_tabular_fold",
    "train_deepfm_v2_fold",
    "deep_predict_proba",
    "DeepTabularDataset",
]

for name in targets:

    print("\n" + "=" * 120)
    print("SOURCE:", name)
    print("=" * 120)

    obj = globals().get(name, None)

    if obj is None:
        print("NOT FOUND")
        continue

    try:
        print(inspect.getsource(obj))
    except Exception as e:
        print("SOURCE UNAVAILABLE:", repr(e))


SOURCE: AutoIntModel
SOURCE UNAVAILABLE: OSError('source code not available')

SOURCE: StableDeepFM
SOURCE UNAVAILABLE: OSError('source code not available')

SOURCE: train_deep_tabular_fold
def train_deep_tabular_fold(
    model_name,
    fold_id,
    config
):

    seed_everything(
        SEED + fold_id
    )

    data = prepare_deep_tabular_fold(
        fold_id
    )

    train_ds = DeepTabularDataset(
        data["Xtr_cat"],
        data["Xtr_cont"],
        data["ytr"]
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config[
            "batch_size"
        ],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(),
    )

    if model_name == "DeepFM":

        model = DeepFMModel(
            cardinalities=
                CAT_CARDINALITIES,
            n_cont=
                len(CONT_COLS_DEEP),
            embedding_dim=
                config["embedding_dim"],
  

In [36]:
# ============================================================
# MODULE 14 — NEURAL OOF REPAIR
# Strict Nested Early-Stopping Repair for:
#
#   1. AutoInt
#   2. DeepFM_v2 / StableDeepFM
#
# ============================================================
#
# WHY THIS MODULE EXISTS
# ------------------------------------------------------------
# Original neural OOF protocol:
#
#   outer-train -> train with outer-valid early stopping
#               -> report prediction on same outer-valid
#
# This makes OOF estimates mildly optimistic because the
# reported fold influenced epoch selection.
#
#
# REPAIRED PROTOCOL
# ------------------------------------------------------------
#
# For each outer fold:
#
# OUTER TRAIN
#     |
#     +---- INNER TRAIN
#     |
#     +---- INNER EARLY-STOP VALIDATION
#              |
#              -> choose epoch E*
#
# Then:
#
# FULL OUTER TRAIN
#     |
#     -> fresh model
#     -> train exactly E* epochs
#     -> predict untouched OUTER VALID
#
#
# HOLDOUT IS NEVER TOUCHED.
#
#
# PRESERVED FROM ORIGINAL IMPLEMENTATION
# ------------------------------------------------------------
# AutoInt:
#   architecture      = AutoIntModel
#   embedding_dim     = 16
#   heads             = 2
#   layers            = 2
#   hidden_dims       = (128, 64)
#   dropout           = 0.2
#   optimizer         = AdamW
#   lr                = 1e-3
#   weight_decay      = 1e-5
#   BCE pos_weight    = SCALE_POS_WEIGHT
#   AMP               = preserved
#
# DeepFM_v2:
#   architecture      = StableDeepFM
#   embedding_dim     = 8
#   hidden_dims       = (256, 128)
#   dropout           = 0.1
#   init_std          = 0.001
#   optimizer         = Adam
#   lr                = 5e-4
#   weight_decay      = 1e-6
#   gradient clipping = max_norm 5.0
#   BCE pos_weight    = SCALE_POS_WEIGHT
#   AMP               = preserved
#
# ============================================================


import gc
import copy
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# 14.1 SAFETY / REQUIREMENTS
# ============================================================

required_repair = [
    "AutoIntModel",
    "StableDeepFM",
    "DeepTabularDataset",
    "deep_predict_proba",

    "X_dev",
    "y_dev",
    "fold_assignment",

    "CAT_COLS",
    "CONT_COLS_DEEP",
    "CAT_CARDINALITIES",
    "DEEP_CARDINALITIES",

    "DEEP_CONFIGS",
    "DEEPFM_V2_CONFIG",

    "SCALE_POS_WEIGHT",
    "TORCH_DEVICE",
    "USE_AMP",
    "SEED",
    "N_SPLITS",

    "seed_everything",
    "seed_worker",
    "make_generator",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_repair
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
    )


# Important after earlier NearestNeighbors collision
import torch.nn as nn


# ============================================================
# 14.2 CONFIG
# ============================================================

NEURAL_REPAIR_CONFIG = {

    # --------------------------------------------------------
    # fraction of each OUTER TRAIN used exclusively for
    # choosing the epoch
    # --------------------------------------------------------

    "inner_es_fraction":
        0.15,

    # --------------------------------------------------------
    # minimum selected epoch
    # --------------------------------------------------------

    "minimum_epoch":
        1,

    # --------------------------------------------------------
    # prediction batch
    # --------------------------------------------------------

    "prediction_batch_size":
        4096,
}


AUTOINT_REPAIR_CONFIG = copy.deepcopy(
    DEEP_CONFIGS["AutoInt"]
)


DEEPFM_REPAIR_CONFIG = copy.deepcopy(
    DEEPFM_V2_CONFIG
)


print(
    "AutoInt config:",
    AUTOINT_REPAIR_CONFIG
)

print(
    "DeepFM_v2 config:",
    DEEPFM_REPAIR_CONFIG
)


# ============================================================
# 14.3 BASIC DATA
# ============================================================

y_np = (
    y_dev
    .to_numpy(
        dtype=np.float32
    )
)


N = len(
    y_np
)


# ============================================================
# 14.4 STRICT DEEP PREPROCESSOR
# ============================================================
#
# Categorical:
#   map the declared raw values -> contiguous integer IDs.
#
# Continuous:
#   StandardScaler fit ONLY on the relevant training partition.
#
# This avoids fitting normalization on outer validation.
# ============================================================

class StrictDeepPreprocessor:

    def __init__(self):

        self.category_maps = {}

        self.cont_scaler = StandardScaler()


    def fit(
        self,
        X
    ):

        # -----------------------------------------------
        # category mappings are based on declared
        # cardinality metadata
        # -----------------------------------------------

        for col in CAT_COLS:

            values = (
                DEEP_CARDINALITIES[
                    col
                ][
                    "values"
                ]
            )

            self.category_maps[
                col
            ] = {
                float(v): i
                for i, v in enumerate(
                    values
                )
            }


        # -----------------------------------------------
        # continuous scaler
        # -----------------------------------------------

        self.cont_scaler.fit(
            X[
                CONT_COLS_DEEP
            ].astype(float)
        )


        return self


    def transform(
        self,
        X
    ):

        n = len(
            X
        )


        # -----------------------------------------------
        # categorical
        # -----------------------------------------------

        X_cat = np.zeros(
            (
                n,
                len(
                    CAT_COLS
                )
            ),
            dtype=np.int64
        )


        for j, col in enumerate(
            CAT_COLS
        ):

            raw_values = (
                X[
                    col
                ]
                .to_numpy(
                    dtype=float
                )
            )


            mapping = (
                self.category_maps[
                    col
                ]
            )


            encoded = np.array(
                [
                    mapping.get(
                        float(v),
                        -1
                    )
                    for v in raw_values
                ],
                dtype=np.int64
            )


            if np.any(
                encoded < 0
            ):

                bad_values = np.unique(
                    raw_values[
                        encoded < 0
                    ]
                )

                raise ValueError(
                    f"Unknown category in {col}: "
                    f"{bad_values}"
                )


            X_cat[
                :,
                j
            ] = encoded


        # -----------------------------------------------
        # continuous
        # -----------------------------------------------

        X_cont = (
            self.cont_scaler
            .transform(
                X[
                    CONT_COLS_DEEP
                ].astype(float)
            )
            .astype(
                np.float32
            )
        )


        return (
            X_cat,
            X_cont
        )


    def fit_transform(
        self,
        X
    ):

        self.fit(
            X
        )

        return self.transform(
            X
        )


# ============================================================
# 14.5 MODEL FACTORY
# ============================================================

def make_repaired_neural_model(
    model_name,
    config
):

    if model_name == "AutoInt":

        model = AutoIntModel(

            cardinalities=
                CAT_CARDINALITIES,

            n_cont=
                len(
                    CONT_COLS_DEEP
                ),

            embedding_dim=
                config[
                    "embedding_dim"
                ],

            num_heads=
                config[
                    "num_heads"
                ],

            num_layers=
                config[
                    "num_layers"
                ],

            hidden_dims=
                config[
                    "hidden_dims"
                ],

            dropout=
                config[
                    "dropout"
                ],
        )


    elif model_name == "DeepFM_v2":

        model = StableDeepFM(

            cardinalities=
                CAT_CARDINALITIES,

            n_cont=
                len(
                    CONT_COLS_DEEP
                ),

            embedding_dim=
                config[
                    "embedding_dim"
                ],

            hidden_dims=
                config[
                    "hidden_dims"
                ],

            dropout=
                config[
                    "dropout"
                ],

            init_std=
                config[
                    "init_std"
                ],
        )


    else:

        raise ValueError(
            model_name
        )


    return model.to(
        TORCH_DEVICE
    )


# ============================================================
# 14.6 OPTIMIZER FACTORY
# ============================================================

def make_repaired_optimizer(
    model_name,
    model,
    config
):

    if model_name == "AutoInt":

        return torch.optim.AdamW(

            model.parameters(),

            lr=
                config[
                    "learning_rate"
                ],

            weight_decay=
                config[
                    "weight_decay"
                ],
        )


    elif model_name == "DeepFM_v2":

        return torch.optim.Adam(

            model.parameters(),

            lr=
                config[
                    "learning_rate"
                ],

            weight_decay=
                config[
                    "weight_decay"
                ],
        )


    else:

        raise ValueError(
            model_name
        )


# ============================================================
# 14.7 LOSS
# ============================================================

def make_repaired_loss():

    pos_weight = torch.tensor(
        [
            SCALE_POS_WEIGHT
        ],
        dtype=torch.float32,
        device=TORCH_DEVICE
    )


    return nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )


# ============================================================
# 14.8 TRAINING LOADER
# ============================================================

def make_repaired_train_loader(
    X_cat,
    X_cont,
    y,
    batch_size,
    seed
):

    ds = DeepTabularDataset(
        X_cat,
        X_cont,
        y
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    return DataLoader(

        ds,

        batch_size=
            batch_size,

        shuffle=True,

        num_workers=2,

        pin_memory=True,

        worker_init_fn=
            seed_worker,

        generator=
            generator,
    )


# ============================================================
# 14.9 TRAIN ONE EPOCH
# ============================================================

def train_repaired_epoch(
    model_name,
    model,
    train_loader,
    optimizer,
    loss_fn,
    amp_scaler
):

    model.train()


    total_loss = 0.0

    total_n = 0


    for (
        xcat,
        xcont,
        yb
    ) in train_loader:


        xcat = xcat.to(
            TORCH_DEVICE,
            non_blocking=True
        )


        xcont = xcont.to(
            TORCH_DEVICE,
            non_blocking=True
        )


        yb = yb.to(
            TORCH_DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type="cuda",

            dtype=torch.float16,

            enabled=USE_AMP

        ):

            logits = model(
                xcat,
                xcont
            )


            loss = loss_fn(
                logits,
                yb
            )


        amp_scaler.scale(
            loss
        ).backward()


        # -----------------------------------------------
        # Preserve DeepFM_v2 gradient clipping
        # -----------------------------------------------

        if (
            model_name
            ==
            "DeepFM_v2"
        ):

            amp_scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                max_norm=5.0
            )


        amp_scaler.step(
            optimizer
        )


        amp_scaler.update()


        batch_n = len(
            yb
        )


        total_loss += (

            float(
                loss
                .detach()
                .cpu()
            )

            *
            batch_n
        )


        total_n += (
            batch_n
        )


    return (

        total_loss
        /
        max(
            total_n,
            1
        )
    )


# ============================================================
# 14.10 INNER EARLY STOPPING
# ============================================================

def select_repaired_epoch(
    model_name,
    X_inner_train,
    y_inner_train,
    X_inner_es,
    y_inner_es,
    config,
    seed
):

    seed_everything(
        seed
    )


    # --------------------------------------------------------
    # preprocessing fitted only INNER TRAIN
    # --------------------------------------------------------

    preprocessor = (
        StrictDeepPreprocessor()
    )


    (
        Xtr_cat,
        Xtr_cont

    ) = (
        preprocessor
        .fit_transform(
            X_inner_train
        )
    )


    (
        Xes_cat,
        Xes_cont

    ) = (
        preprocessor
        .transform(
            X_inner_es
        )
    )


    train_loader = (
        make_repaired_train_loader(

            Xtr_cat,

            Xtr_cont,

            y_inner_train,

            batch_size=
                config[
                    "batch_size"
                ],

            seed=
                seed
                +
                10
        )
    )


    model = (
        make_repaired_neural_model(

            model_name,

            config
        )
    )


    optimizer = (
        make_repaired_optimizer(

            model_name,

            model,

            config
        )
    )


    loss_fn = (
        make_repaired_loss()
    )


    amp_scaler = (
        torch.amp.GradScaler(

            "cuda",

            enabled=
                USE_AMP
        )
    )


    best_pr = -np.inf

    best_epoch = -1

    patience_count = 0

    history = []


    for epoch in range(

        1,

        config[
            "max_epochs"
        ]
        +
        1
    ):


        epoch_loss = (
            train_repaired_epoch(

                model_name,

                model,

                train_loader,

                optimizer,

                loss_fn,

                amp_scaler
            )
        )


        p_es = (
            deep_predict_proba(

                model,

                Xes_cat,

                Xes_cont,

                batch_size=
                    NEURAL_REPAIR_CONFIG[
                        "prediction_batch_size"
                    ]
            )
        )


        es_pr = (
            average_precision_score(

                y_inner_es.astype(
                    int
                ),

                p_es
            )
        )


        history.append({

            "epoch":
                epoch,

            "train_loss":
                epoch_loss,

            "inner_ES_PR":
                es_pr,
        })


        if (
            es_pr
            >
            best_pr
            +
            config[
                "min_delta"
            ]
        ):

            best_pr = (
                es_pr
            )

            best_epoch = (
                epoch
            )

            patience_count = 0


        else:

            patience_count += (
                1
            )


        if (
            epoch == 1
            or
            epoch % 5 == 0
            or
            patience_count
            >=
            config[
                "patience"
            ]
        ):

            print(

                f"      ES epoch={epoch:02d} "
                f"loss={epoch_loss:.5f} "
                f"PR={es_pr:.6f} "
                f"best={best_pr:.6f} "
                f"best_epoch={best_epoch}"
            )


        if (
            patience_count
            >=
            config[
                "patience"
            ]
        ):

            break


    if (
        best_epoch
        <
        NEURAL_REPAIR_CONFIG[
            "minimum_epoch"
        ]
    ):

        raise RuntimeError(
            f"{model_name}: "
            "inner early stopping failed."
        )


    del (
        model,
        optimizer,
        train_loader,
        amp_scaler
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {
        "best_epoch":
            int(
                best_epoch
            ),

        "best_inner_PR":
            float(
                best_pr
            ),

        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# 14.11 RETRAIN FULL OUTER TRAIN FOR FIXED EPOCHS
# ============================================================

def fit_fixed_epoch_and_predict(
    model_name,
    X_outer_train,
    y_outer_train,
    X_outer_valid,
    config,
    selected_epoch,
    seed
):

    seed_everything(
        seed
    )


    # --------------------------------------------------------
    # NEW preprocessing fitted on FULL OUTER TRAIN
    # --------------------------------------------------------

    preprocessor = (
        StrictDeepPreprocessor()
    )


    (
        Xtr_cat,
        Xtr_cont

    ) = (
        preprocessor
        .fit_transform(
            X_outer_train
        )
    )


    (
        Xva_cat,
        Xva_cont

    ) = (
        preprocessor
        .transform(
            X_outer_valid
        )
    )


    train_loader = (
        make_repaired_train_loader(

            Xtr_cat,

            Xtr_cont,

            y_outer_train,

            batch_size=
                config[
                    "batch_size"
                ],

            seed=
                seed
                +
                10
        )
    )


    # --------------------------------------------------------
    # FRESH MODEL
    # --------------------------------------------------------

    model = (
        make_repaired_neural_model(

            model_name,

            config
        )
    )


    optimizer = (
        make_repaired_optimizer(

            model_name,

            model,

            config
        )
    )


    loss_fn = (
        make_repaired_loss()
    )


    amp_scaler = (
        torch.amp.GradScaler(

            "cuda",

            enabled=
                USE_AMP
        )
    )


    fixed_history = []


    # --------------------------------------------------------
    # NO VALIDATION / NO EARLY STOPPING HERE
    # --------------------------------------------------------

    for epoch in range(

        1,

        int(
            selected_epoch
        )
        +
        1
    ):


        epoch_loss = (
            train_repaired_epoch(

                model_name,

                model,

                train_loader,

                optimizer,

                loss_fn,

                amp_scaler
            )
        )


        fixed_history.append({

            "epoch":
                epoch,

            "train_loss":
                epoch_loss,
        })


        if (
            epoch == 1
            or
            epoch % 5 == 0
            or
            epoch
            ==
            int(
                selected_epoch
            )
        ):

            print(

                f"      Full-train epoch="
                f"{epoch:02d}/"
                f"{selected_epoch:02d} "
                f"loss={epoch_loss:.5f}"
            )


    # --------------------------------------------------------
    # OUTER VALIDATION PREDICTED ONCE
    # --------------------------------------------------------

    p_outer_valid = (
        deep_predict_proba(

            model,

            Xva_cat,

            Xva_cont,

            batch_size=
                NEURAL_REPAIR_CONFIG[
                    "prediction_batch_size"
                ]
        )
    )


    del (
        model,
        optimizer,
        train_loader,
        amp_scaler
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {
        "p_valid":
            p_outer_valid,

        "history":
            pd.DataFrame(
                fixed_history
            ),
    }


# ============================================================
# 14.12 REPAIR ONE MODEL
# ============================================================

def repair_neural_model_oof(
    model_name,
    config
):

    repaired_oof = np.zeros(
        N,
        dtype=float
    )


    fold_rows = []

    inner_histories = {}

    retrain_histories = {}


    for fold_id in range(
        N_SPLITS
    ):


        print(
            "\n"
            +
            "=" * 135
        )


        print(
            f"{model_name} — REPAIR OUTER FOLD "
            f"{fold_id+1}/{N_SPLITS}"
        )


        print(
            "=" * 135
        )


        start_time = (
            time.time()
        )


        outer_train_idx = np.where(

            fold_assignment
            !=
            fold_id

        )[0]


        outer_valid_idx = np.where(

            fold_assignment
            ==
            fold_id

        )[0]


        y_outer_train = (
            y_np[
                outer_train_idx
            ]
        )


        # ====================================================
        # INNER TRAIN / EARLY-STOP VALIDATION
        # ====================================================

        splitter = (
            StratifiedShuffleSplit(

                n_splits=1,

                test_size=
                    NEURAL_REPAIR_CONFIG[
                        "inner_es_fraction"
                    ],

                random_state=
                    SEED
                    +
                    14000
                    +
                    fold_id
            )
        )


        (
            inner_train_local,
            inner_es_local

        ) = next(

            splitter.split(

                np.zeros(
                    len(
                        outer_train_idx
                    )
                ),

                y_outer_train.astype(
                    int
                )
            )
        )


        inner_train_idx = (
            outer_train_idx[
                inner_train_local
            ]
        )


        inner_es_idx = (
            outer_train_idx[
                inner_es_local
            ]
        )


        print(
            "  Outer train:",
            len(
                outer_train_idx
            )
        )


        print(
            "  Inner train:",
            len(
                inner_train_idx
            )
        )


        print(
            "  Inner ES:",
            len(
                inner_es_idx
            )
        )


        print(
            "  Outer valid:",
            len(
                outer_valid_idx
            )
        )


        # ====================================================
        # STAGE 1 — SELECT EPOCH
        # ====================================================

        print(
            "\n  [Stage 1] Selecting epoch on INNER ES..."
        )


        epoch_result = (
            select_repaired_epoch(

                model_name=
                    model_name,

                X_inner_train=
                    X_dev.iloc[
                        inner_train_idx
                    ],

                y_inner_train=
                    y_np[
                        inner_train_idx
                    ],

                X_inner_es=
                    X_dev.iloc[
                        inner_es_idx
                    ],

                y_inner_es=
                    y_np[
                        inner_es_idx
                    ],

                config=
                    config,

                seed=
                    SEED
                    +
                    15000
                    +
                    fold_id
            )
        )


        selected_epoch = (
            epoch_result[
                "best_epoch"
            ]
        )


        inner_best_pr = (
            epoch_result[
                "best_inner_PR"
            ]
        )


        inner_histories[
            fold_id
        ] = (
            epoch_result[
                "history"
            ]
        )


        print(
            f"\n  Selected epoch = "
            f"{selected_epoch}"
        )


        print(
            f"  Inner best PR  = "
            f"{inner_best_pr:.6f}"
        )


        # ====================================================
        # STAGE 2 — FRESH RETRAIN ON FULL OUTER TRAIN
        # ====================================================

        print(
            "\n  [Stage 2] Fresh model on FULL OUTER TRAIN..."
        )


        final_result = (
            fit_fixed_epoch_and_predict(

                model_name=
                    model_name,

                X_outer_train=
                    X_dev.iloc[
                        outer_train_idx
                    ],

                y_outer_train=
                    y_np[
                        outer_train_idx
                    ],

                X_outer_valid=
                    X_dev.iloc[
                        outer_valid_idx
                    ],

                config=
                    config,

                selected_epoch=
                    selected_epoch,

                seed=
                    SEED
                    +
                    16000
                    +
                    fold_id
            )
        )


        p_valid = (
            final_result[
                "p_valid"
            ]
        )


        repaired_oof[
            outer_valid_idx
        ] = (
            p_valid
        )


        retrain_histories[
            fold_id
        ] = (
            final_result[
                "history"
            ]
        )


        # ====================================================
        # OUTER VALID METRICS
        # ====================================================

        y_valid = (
            y_np[
                outer_valid_idx
            ].astype(
                int
            )
        )


        fold_pr = (
            average_precision_score(

                y_valid,

                p_valid
            )
        )


        fold_roc = (
            roc_auc_score(

                y_valid,

                p_valid
            )
        )


        fold_brier = (
            brier_score_loss(

                y_valid,

                p_valid
            )
        )


        fold_nll = (
            log_loss(

                y_valid,

                np.clip(
                    p_valid,
                    1e-8,
                    1 - 1e-8
                )
            )
        )


        runtime = (
            time.time()
            -
            start_time
        )


        fold_rows.append({

            "Model":
                model_name,

            "Fold":
                fold_id,

            "Inner_Selected_Epoch":
                selected_epoch,

            "Inner_Best_PR":
                inner_best_pr,

            "Outer_PR_AUC":
                fold_pr,

            "Outer_ROC_AUC":
                fold_roc,

            "Outer_Brier":
                fold_brier,

            "Outer_NLL":
                fold_nll,

            "Runtime_seconds":
                runtime,
        })


        print(
            "\n  OUTER VALID RESULT"
        )


        print(
            f"    PR-AUC : "
            f"{fold_pr:.6f}"
        )


        print(
            f"    ROC-AUC: "
            f"{fold_roc:.6f}"
        )


        print(
            f"    Brier  : "
            f"{fold_brier:.6f}"
        )


        print(
            f"    NLL    : "
            f"{fold_nll:.6f}"
        )


        print(
            f"    runtime: "
            f"{runtime/60:.2f} min"
        )


    # ========================================================
    # GLOBAL OOF
    # ========================================================

    fold_df = pd.DataFrame(
        fold_rows
    )


    global_results = {

        "Model":
            model_name,

        "PR_AUC":
            average_precision_score(

                y_np.astype(
                    int
                ),

                repaired_oof
            ),

        "ROC_AUC":
            roc_auc_score(

                y_np.astype(
                    int
                ),

                repaired_oof
            ),

        "Brier":
            brier_score_loss(

                y_np.astype(
                    int
                ),

                repaired_oof
            ),

        "NLL":
            log_loss(

                y_np.astype(
                    int
                ),

                np.clip(
                    repaired_oof,
                    1e-8,
                    1 - 1e-8
                )
            ),

        "Mean_Selected_Epoch":
            float(
                fold_df[
                    "Inner_Selected_Epoch"
                ].mean()
            ),

        "SD_Selected_Epoch":
            float(
                fold_df[
                    "Inner_Selected_Epoch"
                ].std(
                    ddof=1
                )
            ),
    }


    return {

        "oof_pred":
            repaired_oof,

        "fold_metrics":
            fold_df,

        "global_metrics":
            global_results,

        "inner_histories":
            inner_histories,

        "retrain_histories":
            retrain_histories,
    }


# ============================================================
# 14.13 RUN AUTOINT REPAIR
# ============================================================

print(
    "\n"
    +
    "#" * 135
)

print(
    "REPAIRING AUTOINT"
)

print(
    "#" * 135
)


AUTOINT_REPAIRED = (
    repair_neural_model_oof(

        model_name=
            "AutoInt",

        config=
            AUTOINT_REPAIR_CONFIG
    )
)


AUTOINT_REPAIRED_OOF = (
    AUTOINT_REPAIRED[
        "oof_pred"
    ]
)


# ============================================================
# 14.14 RUN DEEPFM_v2 REPAIR
# ============================================================

print(
    "\n"
    +
    "#" * 135
)

print(
    "REPAIRING DEEPFM_v2"
)

print(
    "#" * 135
)


DEEPFM_V2_REPAIRED = (
    repair_neural_model_oof(

        model_name=
            "DeepFM_v2",

        config=
            DEEPFM_REPAIR_CONFIG
    )
)


DEEPFM_V2_REPAIRED_OOF = (
    DEEPFM_V2_REPAIRED[
        "oof_pred"
    ]
)


# ============================================================
# 14.15 GLOBAL RESULTS
# ============================================================

repair_summary = pd.DataFrame([

    AUTOINT_REPAIRED[
        "global_metrics"
    ],

    DEEPFM_V2_REPAIRED[
        "global_metrics"
    ],
])


print(
    "\n"
    +
    "=" * 135
)

print(
    "MODULE 14 — REPAIRED NEURAL OOF PERFORMANCE"
)

print(
    "=" * 135
)


display(
    repair_summary
)


# ============================================================
# 14.16 FOLD RESULTS
# ============================================================

repair_fold_results = pd.concat(

    [
        AUTOINT_REPAIRED[
            "fold_metrics"
        ],

        DEEPFM_V2_REPAIRED[
            "fold_metrics"
        ],
    ],

    ignore_index=True
)


print(
    "\n"
    +
    "=" * 135
)

print(
    "REPAIRED NEURAL FOLD RESULTS"
)

print(
    "=" * 135
)


display(
    repair_fold_results
)


# ============================================================
# 14.17 COMPARE AGAINST ORIGINAL PROVISIONAL OOF
# ============================================================

comparison_rows = []


# ------------------------------------------------------------
# AutoInt
# ------------------------------------------------------------

if (
    "MODULE7_DEEP_RESULTS"
    in globals()
    and
    "AutoInt"
    in MODULE7_DEEP_RESULTS
):

    old_auto = np.asarray(

        MODULE7_DEEP_RESULTS[
            "AutoInt"
        ][
            "oof_pred"
        ],

        dtype=float
    )


    comparison_rows.append({

        "Model":
            "AutoInt",

        "Original_PR":
            average_precision_score(
                y_np.astype(int),
                old_auto
            ),

        "Repaired_PR":
            average_precision_score(
                y_np.astype(int),
                AUTOINT_REPAIRED_OOF
            ),

        "Delta_PR":
            (
                average_precision_score(
                    y_np.astype(int),
                    AUTOINT_REPAIRED_OOF
                )
                -
                average_precision_score(
                    y_np.astype(int),
                    old_auto
                )
            ),

        "Original_ROC":
            roc_auc_score(
                y_np.astype(int),
                old_auto
            ),

        "Repaired_ROC":
            roc_auc_score(
                y_np.astype(int),
                AUTOINT_REPAIRED_OOF
            ),

        "Prediction_Correlation":
            np.corrcoef(
                old_auto,
                AUTOINT_REPAIRED_OOF
            )[0, 1],
    })


# ------------------------------------------------------------
# DeepFM_v2
# ------------------------------------------------------------
#
# Try common notebook locations.
# ------------------------------------------------------------

old_deepfm_v2 = None


if (
    "PREDICTION_BANK"
    in globals()
):

    for possible_name in [
        "DeepFM_v2",
        "DeepFMV2",
        "StableDeepFM",
    ]:

        if (
            possible_name
            in PREDICTION_BANK
        ):

            candidate = (
                PREDICTION_BANK[
                    possible_name
                ]
            )

            if isinstance(
                candidate,
                dict
            ):

                for key in [
                    "oof_pred",
                    "oof",
                    "OOF",
                ]:

                    if key in candidate:

                        old_deepfm_v2 = np.asarray(
                            candidate[
                                key
                            ],
                            dtype=float
                        )

                        break

            else:

                old_deepfm_v2 = np.asarray(
                    candidate,
                    dtype=float
                )


        if (
            old_deepfm_v2
            is not None
        ):

            break


# fallback: variables commonly used
if (
    old_deepfm_v2
    is None
):

    for possible_var in [
        "DEEPFM_V2_OOF",
        "deepfm_v2_oof",
        "DeepFM_v2_OOF",
    ]:

        if possible_var in globals():

            old_deepfm_v2 = np.asarray(
                globals()[
                    possible_var
                ],
                dtype=float
            )

            break


if (
    old_deepfm_v2
    is not None
    and
    len(
        old_deepfm_v2
    )
    ==
    N
):

    comparison_rows.append({

        "Model":
            "DeepFM_v2",

        "Original_PR":
            average_precision_score(
                y_np.astype(int),
                old_deepfm_v2
            ),

        "Repaired_PR":
            average_precision_score(
                y_np.astype(int),
                DEEPFM_V2_REPAIRED_OOF
            ),

        "Delta_PR":
            (
                average_precision_score(
                    y_np.astype(int),
                    DEEPFM_V2_REPAIRED_OOF
                )
                -
                average_precision_score(
                    y_np.astype(int),
                    old_deepfm_v2
                )
            ),

        "Original_ROC":
            roc_auc_score(
                y_np.astype(int),
                old_deepfm_v2
            ),

        "Repaired_ROC":
            roc_auc_score(
                y_np.astype(int),
                DEEPFM_V2_REPAIRED_OOF
            ),

        "Prediction_Correlation":
            np.corrcoef(
                old_deepfm_v2,
                DEEPFM_V2_REPAIRED_OOF
            )[0, 1],
    })


repair_comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\n"
    +
    "=" * 135
)

print(
    "ORIGINAL vs STRICTLY REPAIRED NEURAL OOF"
)

print(
    "=" * 135
)


if len(
    repair_comparison
) > 0:

    display(
        repair_comparison
    )

else:

    print(
        "Original OOF arrays were not automatically found."
    )


# ============================================================
# 14.18 SAVE REPAIRED OOF
# ============================================================

repaired_prediction_df = pd.DataFrame({

    "target":
        y_np.astype(
            int
        ),

    "fold":
        fold_assignment,

    "AutoInt_Repaired":
        AUTOINT_REPAIRED_OOF,

    "DeepFM_v2_Repaired":
        DEEPFM_V2_REPAIRED_OOF,
})


save_csv(

    repaired_prediction_df,

    PRED_DIR
    / "module14_repaired_neural_oof.csv"
)


save_csv(

    repair_summary,

    TAB_DIR
    / "module14_repaired_neural_summary.csv"
)


save_csv(

    repair_fold_results,

    TAB_DIR
    / "module14_repaired_neural_folds.csv"
)


if len(
    repair_comparison
) > 0:

    save_csv(

        repair_comparison,

        TAB_DIR
        / "module14_original_vs_repaired.csv"
    )


# ============================================================
# 14.19 SAVE EPOCH HISTORIES
# ============================================================

for fold_id, hist in (
    AUTOINT_REPAIRED[
        "inner_histories"
    ].items()
):

    save_csv(

        hist,

        TAB_DIR
        /
        f"module14_autoint_innerES_fold{fold_id}.csv"
    )


for fold_id, hist in (
    DEEPFM_V2_REPAIRED[
        "inner_histories"
    ].items()
):

    save_csv(

        hist,

        TAB_DIR
        /
        f"module14_deepfm_v2_innerES_fold{fold_id}.csv"
    )


# ============================================================
# 14.20 MANIFEST
# ============================================================

module14_manifest = {

    "module":
        "Neural OOF Repair",

    "models": [
        "AutoInt",
        "DeepFM_v2",
    ],

    "protocol":
        (
            "inner early stopping -> select epoch -> "
            "fresh full outer-train retraining -> "
            "untouched outer-validation prediction"
        ),

    "inner_es_fraction":
        NEURAL_REPAIR_CONFIG[
            "inner_es_fraction"
        ],

    "epoch_selection_metric":
        "PR-AUC",

    "autoint_optimizer":
        "AdamW",

    "deepfm_v2_optimizer":
        "Adam",

    "deepfm_v2_gradient_clip":
        5.0,

    "loss":
        "BCEWithLogitsLoss with original SCALE_POS_WEIGHT",

    "continuous_preprocessing":
        "StandardScaler fit inside each applicable training partition",

    "categorical_preprocessing":
        "declared DEEP_CARDINALITIES mapped to contiguous IDs",

    "outer_validation_used_for_epoch_selection":
        False,

    "holdout_used":
        False,

    "architecture_changed":
        False,
}


save_json(

    module14_manifest,

    LOG_DIR
    / "module14_manifest.json"
)


print(
    "\n"
    +
    "=" * 135
)

print(
    "MODULE 14 COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT overwrite the prediction bank yet."
)

print(
    "First inspect the repaired AutoInt and DeepFM_v2 results."
)

AutoInt config: {'family': 'deep_tabular', 'embedding_dim': 16, 'num_heads': 2, 'num_layers': 2, 'hidden_dims': (128, 64), 'dropout': 0.2, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 1024, 'max_epochs': 30, 'patience': 5, 'min_delta': 0.0001}
DeepFM_v2 config: {'embedding_dim': 8, 'hidden_dims': (256, 128), 'dropout': 0.1, 'init_std': 0.001, 'learning_rate': 0.0005, 'weight_decay': 1e-06, 'batch_size': 1024, 'max_epochs': 25, 'patience': 5, 'min_delta': 0.0001}

#######################################################################################################################################
REPAIRING AUTOINT
#######################################################################################################################################

AutoInt — REPAIR OUTER FOLD 1/5
  Outer train: 162355
  Inner train: 138001
  Inner ES: 24354
  Outer valid: 40589

  [Stage 1] Selecting epoch on INNER ES...
      ES epoch=01 loss=0.92614 PR=0.377615 best=0.377615 best_epoc

,Model,PR_AUC,ROC_AUC,Brier,NLL,Mean_Selected_Epoch,SD_Selected_Epoch
0,AutoInt,0.375464,0.848629,0.168368,0.491961,10.0,4.949747
1,DeepFM_v2,0.372787,0.848171,0.165097,0.483823,12.8,3.563706



REPAIRED NEURAL FOLD RESULTS


,Model,Fold,Inner_Selected_Epoch,Inner_Best_PR,Outer_PR_AUC,Outer_ROC_AUC,Outer_Brier,Outer_NLL,Runtime_seconds
0,AutoInt,0,8,0.387930,0.376410,0.850906,0.170187,0.490697,48.522444
1,AutoInt,1,10,0.389635,0.379039,0.849364,0.171607,0.499943,58.026431
2,AutoInt,2,13,0.378651,0.377862,0.847702,0.182044,0.525346,74.732848
3,AutoInt,3,16,0.378947,0.369394,0.848229,0.154441,0.458969,84.349223
4,AutoInt,4,3,0.371072,0.383650,0.850323,0.163564,0.484853,26.470233
5,DeepFM_v2,0,17,0.386072,0.373932,0.849902,0.171990,0.499089,80.676215
6,DeepFM_v2,1,9,0.388489,0.373103,0.848159,0.170838,0.497922,48.896532
7,DeepFM_v2,2,10,0.370431,0.372077,0.846274,0.163062,0.480193,52.639310
8,DeepFM_v2,3,16,0.372635,0.366264,0.846827,0.161320,0.476151,77.531953
9,DeepFM_v2,4,12,0.373517,0.384497,0.850747,0.158276,0.465758,61.220314



ORIGINAL vs STRICTLY REPAIRED NEURAL OOF


,Model,Original_PR,Repaired_PR,Delta_PR,Original_ROC,Repaired_ROC,Prediction_Correlation
0,AutoInt,0.376859,0.375464,-0.001394,0.849007,0.848629,0.988924



MODULE 14 COMPLETE

IMPORTANT:
Do NOT overwrite the prediction bank yet.
First inspect the repaired AutoInt and DeepFM_v2 results.


In [37]:
# ============================================================
# MODULE 15 — REPAIRED EXPERT INTEGRATION
# AND CORE4 REVALIDATION
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Integrate the strictly repaired:
#
#   AutoInt
#   DeepFM_v2
#
# into the existing predictive backbone without modifying the
# original prediction bank.
#
#
# PIPELINE
# ------------------------------------------------------------
#
# Repaired raw neural OOF
#       |
#       v
# Fold-cross-fitted Platt calibration
#       |
#       v
# HGB + CatBoost + repaired AutoInt + repaired DeepFM_v2
#       |
#       v
# Repaired Core4 equal mean
#
#
# COMPARISONS
# ------------------------------------------------------------
#
# 1. HGB
# 2. CatBoost
# 3. HGB + CatBoost
# 4. HGB + CatBoost + repaired AutoInt
# 5. HGB + CatBoost + repaired DeepFM_v2
# 6. Repaired Core4
# 7. Previous provisional Core4
#
#
# DEVELOPMENT ONLY.
# HOLDOUT REMAINS UNTOUCHED.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
)


# ============================================================
# 15.1 REQUIREMENTS
# ============================================================

required_15 = [
    "AUTOINT_REPAIRED_OOF",
    "DEEPFM_V2_REPAIRED_OOF",
    "ETR_EXPERT_OOF",
    "RAMOE_EXPERTS",
    "fold_assignment",
    "y_dev",
    "N_SPLITS",
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_15
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Run Modules 12B and 14 first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 15.2 BASIC DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N = len(
    y_all
)


autoint_raw = np.asarray(
    AUTOINT_REPAIRED_OOF,
    dtype=float
)


deepfm_raw = np.asarray(
    DEEPFM_V2_REPAIRED_OOF,
    dtype=float
)


existing_expert_matrix = np.asarray(
    ETR_EXPERT_OOF,
    dtype=float
)


print(
    "N:",
    N
)


print(
    "Existing expert order:",
    RAMOE_EXPERTS
)


print(
    "Existing expert matrix:",
    existing_expert_matrix.shape
)


# ============================================================
# 15.3 FIND EXISTING CLASSICAL EXPERT COLUMNS
# ============================================================

def find_expert_index(
    candidates
):

    names = [
        str(x).lower()
        for x in RAMOE_EXPERTS
    ]


    for candidate in candidates:

        c = candidate.lower()

        for i, name in enumerate(
            names
        ):

            if (
                c == name
                or
                c in name
            ):

                return i


    raise RuntimeError(
        "Could not find expert among candidates: "
        + str(candidates)
        + "\nAvailable: "
        + str(RAMOE_EXPERTS)
    )


hgb_idx = find_expert_index([
    "histgradientboosting",
    "histgb",
    "hgb",
])


cat_idx = find_expert_index([
    "catboost",
    "cat",
])


auto_old_idx = find_expert_index([
    "autoint",
])


deepfm_old_idx = find_expert_index([
    "deepfm_v2",
    "deepfm",
])


print(
    "\nResolved expert columns:"
)

print(
    "  HGB      :",
    hgb_idx,
    RAMOE_EXPERTS[
        hgb_idx
    ]
)

print(
    "  CatBoost :",
    cat_idx,
    RAMOE_EXPERTS[
        cat_idx
    ]
)

print(
    "  AutoInt  :",
    auto_old_idx,
    RAMOE_EXPERTS[
        auto_old_idx
    ]
)

print(
    "  DeepFM   :",
    deepfm_old_idx,
    RAMOE_EXPERTS[
        deepfm_old_idx
    ]
)


# ============================================================
# 15.4 CLASSICAL CALIBRATED OOF
# ============================================================
#
# These were already produced by the earlier strict
# calibration pipeline.
#
# No need to refit them.
# ============================================================

HGB_CAL_OOF = (
    existing_expert_matrix[
        :,
        hgb_idx
    ]
    .copy()
)


CATBOOST_CAL_OOF = (
    existing_expert_matrix[
        :,
        cat_idx
    ]
    .copy()
)


OLD_AUTOINT_CAL_OOF = (
    existing_expert_matrix[
        :,
        auto_old_idx
    ]
    .copy()
)


OLD_DEEPFM_CAL_OOF = (
    existing_expert_matrix[
        :,
        deepfm_old_idx
    ]
    .copy()
)


# ============================================================
# 15.5 LOGIT HELPER
# ============================================================

def probability_logit(
    p
):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )


    return np.log(
        p
        /
        (
            1.0
            -
            p
        )
    )


# ============================================================
# 15.6 STRICT CROSS-FITTED PLATT CALIBRATION
# ============================================================
#
# For each fold f:
#
#   calibrator fit:
#       repaired OOF predictions from folds != f
#
#   calibrator application:
#       repaired OOF predictions from fold f
#
#
# Therefore each target row:
#
#   1. was not used to train its neural prediction
#   2. was not used to fit its probability calibrator
#
# ============================================================

def crossfit_platt_calibration(
    raw_oof,
    model_name
):

    raw_oof = np.asarray(
        raw_oof,
        dtype=float
    )


    calibrated = np.zeros(
        len(
            raw_oof
        ),
        dtype=float
    )


    fold_rows = []


    raw_logit = probability_logit(
        raw_oof
    )


    for fold_id in range(
        N_SPLITS
    ):

        calibration_train = np.where(

            fold_assignment
            !=
            fold_id

        )[0]


        calibration_valid = np.where(

            fold_assignment
            ==
            fold_id

        )[0]


        calibrator = LogisticRegression(

            penalty="l2",

            C=1e6,

            solver="lbfgs",

            max_iter=2000,

            random_state=
                SEED
                +
                150000
                +
                fold_id
        )


        calibrator.fit(

            raw_logit[
                calibration_train,
                None
            ],

            y_all[
                calibration_train
            ]
        )


        calibrated_fold = (

            calibrator
            .predict_proba(

                raw_logit[
                    calibration_valid,
                    None
                ]

            )[
                :,
                1
            ]
        )


        calibrated[
            calibration_valid
        ] = calibrated_fold


        fold_rows.append({

            "Model":
                model_name,

            "Fold":
                fold_id,

            "Calibration_Train_N":
                len(
                    calibration_train
                ),

            "Calibration_Valid_N":
                len(
                    calibration_valid
                ),

            "Intercept":
                float(
                    calibrator.intercept_[
                        0
                    ]
                ),

            "Slope":
                float(
                    calibrator.coef_[
                        0,
                        0
                    ]
                ),

            "Mean_Raw":
                float(
                    raw_oof[
                        calibration_valid
                    ].mean()
                ),

            "Mean_Calibrated":
                float(
                    calibrated_fold.mean()
                ),

            "Observed_Prevalence":
                float(
                    y_all[
                        calibration_valid
                    ].mean()
                ),
        })


    return (
        calibrated,
        pd.DataFrame(
            fold_rows
        )
    )


# ============================================================
# 15.7 CALIBRATE REPAIRED NEURAL EXPERTS
# ============================================================

print(
    "\nCross-fitting repaired AutoInt calibration..."
)


(
    AUTOINT_REPAIRED_CAL_OOF,
    autoint_calibration_folds

) = crossfit_platt_calibration(

    autoint_raw,

    "AutoInt_Repaired"
)


print(
    "Cross-fitting repaired DeepFM_v2 calibration..."
)


(
    DEEPFM_V2_REPAIRED_CAL_OOF,
    deepfm_calibration_folds

) = crossfit_platt_calibration(

    deepfm_raw,

    "DeepFM_v2_Repaired"
)


repair_calibration_folds = pd.concat(

    [
        autoint_calibration_folds,
        deepfm_calibration_folds,
    ],

    ignore_index=True
)


# ============================================================
# 15.8 ECE
# ============================================================

def module15_ece(
    y_true,
    prob,
    n_bins=15
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    prob = np.asarray(
        prob,
        dtype=float
    )


    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )


    ece = 0.0


    for b in range(
        n_bins
    ):

        if b < (
            n_bins - 1
        ):

            mask = (

                (prob >= edges[b])

                &

                (prob < edges[b + 1])
            )

        else:

            mask = (

                (prob >= edges[b])

                &

                (prob <= edges[b + 1])
            )


        if not np.any(
            mask
        ):

            continue


        ece += (

            mask.mean()

            *
            abs(
                prob[
                    mask
                ].mean()
                -
                y_true[
                    mask
                ].mean()
            )
        )


    return float(
        ece
    )


# ============================================================
# 15.9 RAW vs CALIBRATED REPAIRED NEURAL
# ============================================================

neural_calibration_rows = []


for (
    model_name,
    raw_prob,
    cal_prob
) in [

    (
        "AutoInt_Repaired",
        autoint_raw,
        AUTOINT_REPAIRED_CAL_OOF
    ),

    (
        "DeepFM_v2_Repaired",
        deepfm_raw,
        DEEPFM_V2_REPAIRED_CAL_OOF
    ),
]:

    for (
        state,
        prob
    ) in [

        (
            "Raw",
            raw_prob
        ),

        (
            "CrossFit_Platt",
            cal_prob
        ),
    ]:

        neural_calibration_rows.append({

            "Model":
                model_name,

            "State":
                state,

            "PR_AUC":
                average_precision_score(
                    y_all,
                    prob
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_all,
                    prob
                ),

            "Brier":
                brier_score_loss(
                    y_all,
                    prob
                ),

            "NLL":
                log_loss(
                    y_all,
                    np.clip(
                        prob,
                        1e-8,
                        1 - 1e-8
                    )
                ),

            "ECE":
                module15_ece(
                    y_all,
                    prob
                ),

            "Mean_Probability":
                float(
                    prob.mean()
                ),
        })


neural_calibration_results = pd.DataFrame(
    neural_calibration_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 15 — REPAIRED NEURAL CALIBRATION"
)

print(
    "=" * 135
)


display(
    neural_calibration_results
)


# ============================================================
# 15.10 CREATE NEW REPAIRED EXPERT BANK
# ============================================================
#
# IMPORTANT:
#
# ETR_EXPERT_OOF is NOT modified.
#
# ============================================================

REPAIRED_EXPERT_OOF = np.column_stack([

    HGB_CAL_OOF,

    CATBOOST_CAL_OOF,

    AUTOINT_REPAIRED_CAL_OOF,

    DEEPFM_V2_REPAIRED_CAL_OOF,
])


REPAIRED_EXPERT_NAMES = [

    "HistGradientBoosting",

    "CatBoost",

    "AutoInt_Repaired",

    "DeepFM_v2_Repaired",
]


print(
    "\nRepaired expert bank:",
    REPAIRED_EXPERT_OOF.shape
)


# ============================================================
# 15.11 FIXED ENSEMBLE CANDIDATES
# ============================================================
#
# No optimization.
# No weight tuning.
#
# These are fixed diagnostic subsets.
# ============================================================

HGB_ONLY = (
    HGB_CAL_OOF
)


CAT_ONLY = (
    CATBOOST_CAL_OOF
)


HGB_CAT_MEAN = np.mean(

    np.column_stack([

        HGB_CAL_OOF,
        CATBOOST_CAL_OOF,

    ]),

    axis=1
)


HGB_CAT_AUTO_MEAN = np.mean(

    np.column_stack([

        HGB_CAL_OOF,
        CATBOOST_CAL_OOF,
        AUTOINT_REPAIRED_CAL_OOF,

    ]),

    axis=1
)


HGB_CAT_DEEPFM_MEAN = np.mean(

    np.column_stack([

        HGB_CAL_OOF,
        CATBOOST_CAL_OOF,
        DEEPFM_V2_REPAIRED_CAL_OOF,

    ]),

    axis=1
)


REPAIRED_CORE4_MEAN = np.mean(

    REPAIRED_EXPERT_OOF,

    axis=1
)


# Previous provisional Core4
PROVISIONAL_CORE4_MEAN = np.mean(

    existing_expert_matrix,

    axis=1
)


# ============================================================
# 15.12 GLOBAL PERFORMANCE
# ============================================================

candidate_predictions = {

    "HGB":
        HGB_ONLY,

    "CatBoost":
        CAT_ONLY,

    "HGB+CatBoost":
        HGB_CAT_MEAN,

    "HGB+CatBoost+AutoInt_Repaired":
        HGB_CAT_AUTO_MEAN,

    "HGB+CatBoost+DeepFM_Repaired":
        HGB_CAT_DEEPFM_MEAN,

    "Repaired_Core4":
        REPAIRED_CORE4_MEAN,

    "Provisional_Core4":
        PROVISIONAL_CORE4_MEAN,
}


core_revalidation_rows = []


for (
    model_name,
    prob
) in candidate_predictions.items():

    core_revalidation_rows.append({

        "Model":
            model_name,

        "PR_AUC":
            average_precision_score(
                y_all,
                prob
            ),

        "ROC_AUC":
            roc_auc_score(
                y_all,
                prob
            ),

        "Brier":
            brier_score_loss(
                y_all,
                prob
            ),

        "NLL":
            log_loss(

                y_all,

                np.clip(
                    prob,
                    1e-8,
                    1 - 1e-8
                )
            ),

        "ECE":
            module15_ece(
                y_all,
                prob
            ),

        "Mean_Probability":
            float(
                prob.mean()
            ),
    })


core_revalidation = pd.DataFrame(
    core_revalidation_rows
)


core_revalidation = (
    core_revalidation
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 15 — REPAIRED CORE BACKBONE REVALIDATION"
)

print(
    "=" * 135
)


display(
    core_revalidation
)


# ============================================================
# 15.13 DELTAS AGAINST KEY REFERENCES
# ============================================================

def get_candidate_metric(
    model,
    metric
):

    return float(

        core_revalidation.loc[

            core_revalidation[
                "Model"
            ]
            ==
            model,

            metric

        ].iloc[0]
    )


repaired_pr = (
    get_candidate_metric(
        "Repaired_Core4",
        "PR_AUC"
    )
)


provisional_pr = (
    get_candidate_metric(
        "Provisional_Core4",
        "PR_AUC"
    )
)


hgb_cat_pr = (
    get_candidate_metric(
        "HGB+CatBoost",
        "PR_AUC"
    )
)


hgb_pr = (
    get_candidate_metric(
        "HGB",
        "PR_AUC"
    )
)


core_delta_results = pd.DataFrame([

    {
        "Comparison":
            "Repaired Core4 vs Provisional Core4",

        "Delta_PR":
            repaired_pr
            -
            provisional_pr,

        "Delta_ROC":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "ROC_AUC"
                )
                -
                get_candidate_metric(
                    "Provisional_Core4",
                    "ROC_AUC"
                )
            ),

        "Delta_Brier":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "Brier"
                )
                -
                get_candidate_metric(
                    "Provisional_Core4",
                    "Brier"
                )
            ),
    },

    {
        "Comparison":
            "Repaired Core4 vs HGB+CatBoost",

        "Delta_PR":
            repaired_pr
            -
            hgb_cat_pr,

        "Delta_ROC":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "ROC_AUC"
                )
                -
                get_candidate_metric(
                    "HGB+CatBoost",
                    "ROC_AUC"
                )
            ),

        "Delta_Brier":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "Brier"
                )
                -
                get_candidate_metric(
                    "HGB+CatBoost",
                    "Brier"
                )
            ),
    },

    {
        "Comparison":
            "Repaired Core4 vs HGB",

        "Delta_PR":
            repaired_pr
            -
            hgb_pr,

        "Delta_ROC":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "ROC_AUC"
                )
                -
                get_candidate_metric(
                    "HGB",
                    "ROC_AUC"
                )
            ),

        "Delta_Brier":
            (
                get_candidate_metric(
                    "Repaired_Core4",
                    "Brier"
                )
                -
                get_candidate_metric(
                    "HGB",
                    "Brier"
                )
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "REPAIRED CORE4 DELTAS"
)

print(
    "=" * 135
)


display(
    core_delta_results
)


# ============================================================
# 15.14 FOLD-WISE PR-AUC
# ============================================================

fold_rows = []


for fold_id in range(
    N_SPLITS
):

    idx = np.where(

        fold_assignment
        ==
        fold_id

    )[0]


    y_fold = (
        y_all[
            idx
        ]
    )


    row = {

        "Fold":
            fold_id,

        "Prevalence":
            y_fold.mean(),
    }


    for (
        model_name,
        prob
    ) in candidate_predictions.items():

        row[
            f"{model_name}_PR"
        ] = (
            average_precision_score(

                y_fold,

                prob[
                    idx
                ]
            )
        )


    fold_rows.append(
        row
    )


core_fold_results = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "REPAIRED BACKBONE — FOLD PR-AUC"
)

print(
    "=" * 135
)


display(
    core_fold_results
)


# ============================================================
# 15.15 FOLD DELTA: CORE4 vs HGB+CATBOOST
# ============================================================

core_fold_delta = pd.DataFrame({

    "Fold":
        core_fold_results[
            "Fold"
        ],

    "Repaired_Core4_PR":
        core_fold_results[
            "Repaired_Core4_PR"
        ],

    "HGB+CatBoost_PR":
        core_fold_results[
            "HGB+CatBoost_PR"
        ],
})


core_fold_delta[
    "Delta_PR"
] = (

    core_fold_delta[
        "Repaired_Core4_PR"
    ]

    -
    core_fold_delta[
        "HGB+CatBoost_PR"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "CORE4 vs HGB+CATBOOST — FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    core_fold_delta
)


print(
    "\nMean ΔPR:",
    core_fold_delta[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            core_fold_delta[
                "Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 15.16 COMPARE OLD vs REPAIRED NEURAL CALIBRATED PREDICTIONS
# ============================================================

neural_replacement_comparison = pd.DataFrame([

    {
        "Model":
            "AutoInt",

        "Old_Calibrated_PR":
            average_precision_score(
                y_all,
                OLD_AUTOINT_CAL_OOF
            ),

        "Repaired_Calibrated_PR":
            average_precision_score(
                y_all,
                AUTOINT_REPAIRED_CAL_OOF
            ),

        "Prediction_Correlation":
            np.corrcoef(

                OLD_AUTOINT_CAL_OOF,

                AUTOINT_REPAIRED_CAL_OOF
            )[0, 1],
    },

    {
        "Model":
            "DeepFM_v2",

        "Old_Calibrated_PR":
            average_precision_score(
                y_all,
                OLD_DEEPFM_CAL_OOF
            ),

        "Repaired_Calibrated_PR":
            average_precision_score(
                y_all,
                DEEPFM_V2_REPAIRED_CAL_OOF
            ),

        "Prediction_Correlation":
            np.corrcoef(

                OLD_DEEPFM_CAL_OOF,

                DEEPFM_V2_REPAIRED_CAL_OOF
            )[0, 1],
    },
])


neural_replacement_comparison[
    "Delta_PR"
] = (

    neural_replacement_comparison[
        "Repaired_Calibrated_PR"
    ]

    -
    neural_replacement_comparison[
        "Old_Calibrated_PR"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "OLD vs REPAIRED CALIBRATED NEURAL EXPERTS"
)

print(
    "=" * 135
)


display(
    neural_replacement_comparison
)


# ============================================================
# 15.17 DECISION SUMMARY
# ============================================================

best_model = (

    core_revalidation
    .iloc[0][
        "Model"
    ]
)


decision_summary = pd.DataFrame([

    {
        "Question":
            "Does repaired Core4 outperform HGB alone?",

        "Answer":
            repaired_pr
            >
            hgb_pr,

        "Value":
            repaired_pr
            -
            hgb_pr,
    },

    {
        "Question":
            "Does repaired Core4 outperform HGB+CatBoost?",

        "Answer":
            repaired_pr
            >
            hgb_cat_pr,

        "Value":
            repaired_pr
            -
            hgb_cat_pr,
    },

    {
        "Question":
            "Does repair materially collapse previous Core4?",

        "Answer":
            (
                repaired_pr
                <
                provisional_pr
                -
                0.002
            ),

        "Value":
            repaired_pr
            -
            provisional_pr,
    },

    {
        "Question":
            "Is repaired Core4 the best tested fixed subset?",

        "Answer":
            (
                best_model
                ==
                "Repaired_Core4"
            ),

        "Value":
            best_model,
    },

    {
        "Question":
            "Core4 beats HGB+CatBoost in how many folds?",

        "Answer":
            int(
                (
                    core_fold_delta[
                        "Delta_PR"
                    ]
                    >
                    0
                ).sum()
            ),

        "Value":
            f"/ {N_SPLITS}",
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 15 — BACKBONE DECISION SUMMARY"
)

print(
    "=" * 135
)


display(
    decision_summary
)


# ============================================================
# 15.18 SAVE NEW REPAIRED BANK
# ============================================================

repaired_bank_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "HGB_Calibrated":
        HGB_CAL_OOF,

    "CatBoost_Calibrated":
        CATBOOST_CAL_OOF,

    "AutoInt_Repaired_Raw":
        autoint_raw,

    "AutoInt_Repaired_Calibrated":
        AUTOINT_REPAIRED_CAL_OOF,

    "DeepFM_v2_Repaired_Raw":
        deepfm_raw,

    "DeepFM_v2_Repaired_Calibrated":
        DEEPFM_V2_REPAIRED_CAL_OOF,

    "HGB_CatBoost_Mean":
        HGB_CAT_MEAN,

    "Repaired_Core4":
        REPAIRED_CORE4_MEAN,

    "Provisional_Core4":
        PROVISIONAL_CORE4_MEAN,
})


save_csv(

    repaired_bank_df,

    PRED_DIR
    / "module15_repaired_expert_bank.csv"
)


save_csv(

    neural_calibration_results,

    TAB_DIR
    / "module15_repaired_neural_calibration.csv"
)


save_csv(

    repair_calibration_folds,

    TAB_DIR
    / "module15_repaired_calibration_folds.csv"
)


save_csv(

    core_revalidation,

    TAB_DIR
    / "module15_core_revalidation.csv"
)


save_csv(

    core_delta_results,

    TAB_DIR
    / "module15_core_deltas.csv"
)


save_csv(

    core_fold_results,

    TAB_DIR
    / "module15_core_fold_results.csv"
)


save_csv(

    core_fold_delta,

    TAB_DIR
    / "module15_core_fold_delta.csv"
)


save_csv(

    neural_replacement_comparison,

    TAB_DIR
    / "module15_old_vs_repaired_neural.csv"
)


save_csv(

    decision_summary,

    TAB_DIR
    / "module15_backbone_decision.csv"
)


# ============================================================
# 15.19 MANIFEST
# ============================================================

module15_manifest = {

    "module":
        "Repaired Expert Integration and Core4 Revalidation",

    "development_only":
        True,

    "holdout_used":
        False,

    "original_prediction_bank_modified":
        False,

    "classical_experts": [
        "HistGradientBoosting",
        "CatBoost",
    ],

    "repaired_neural_experts": [
        "AutoInt",
        "DeepFM_v2",
    ],

    "neural_calibration":
        (
            "fold-cross-fitted Platt scaling "
            "using logit of repaired OOF probabilities"
        ),

    "candidate_backbones": [
        "HGB",
        "CatBoost",
        "HGB+CatBoost",
        "HGB+CatBoost+AutoInt_Repaired",
        "HGB+CatBoost+DeepFM_Repaired",
        "Repaired_Core4",
        "Provisional_Core4",
    ],

    "weight_optimization":
        False,

    "primary_metric":
        "PR-AUC",
}


save_json(

    module15_manifest,

    LOG_DIR
    / "module15_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 15 COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nIMPORTANT:"
)

print(
    "The original ETR_EXPERT_OOF matrix has NOT been overwritten."
)

print(
    "Use REPAIRED_EXPERT_OOF only after we inspect these results."
)

N: 202944
Existing expert order: ['HistGradientBoosting', 'CatBoost', 'AutoInt', 'DeepFM_v2']
Existing expert matrix: (202944, 4)

Resolved expert columns:
  HGB      : 0 HistGradientBoosting
  CatBoost : 1 CatBoost
  AutoInt  : 2 AutoInt
  DeepFM   : 3 DeepFM_v2

Cross-fitting repaired AutoInt calibration...
Cross-fitting repaired DeepFM_v2 calibration...

MODULE 15 — REPAIRED NEURAL CALIBRATION


,Model,State,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_Probability
0,AutoInt_Repaired,Raw,0.375464,0.848629,0.168368,0.491961,0.254629,0.348813
1,AutoInt_Repaired,CrossFit_Platt,0.374767,0.848331,0.070050,0.236921,0.002925,0.094252
2,DeepFM_v2_Repaired,Raw,0.372787,0.848171,0.165097,0.483823,0.247972,0.342156
3,DeepFM_v2_Repaired,CrossFit_Platt,0.372501,0.848072,0.070122,0.237043,0.001713,0.094187



Repaired expert bank: (202944, 4)

MODULE 15 — REPAIRED CORE BACKBONE REVALIDATION


,Model,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_Probability
0,Provisional_Core4,0.380303,0.850539,0.069709,0.235595,0.002299,0.094167
1,HGB+CatBoost+AutoInt_Repaired,0.379395,0.850319,0.069749,0.235746,0.002410,0.094145
2,Repaired_Core4,0.379284,0.850402,0.069748,0.235710,0.002527,0.094155
3,HGB+CatBoost+DeepFM_Repaired,0.379059,0.850359,0.069760,0.235738,0.002158,0.094123
4,HGB+CatBoost,0.378895,0.850161,0.069775,0.235840,0.001956,0.094091
5,CatBoost,0.377444,0.849204,0.069874,0.236331,0.001670,0.094167
6,HGB,0.377012,0.849628,0.069876,0.236194,0.002378,0.094016



REPAIRED CORE4 DELTAS


...


Mean ΔPR: 0.0005875815861387634
Positive folds: 4 / 5

OLD vs REPAIRED CALIBRATED NEURAL EXPERTS


,Model,Old_Calibrated_PR,Repaired_Calibrated_PR,Prediction_Correlation,Delta_PR
0,AutoInt,0.375956,0.374767,0.987665,-0.001189
1,DeepFM_v2,0.373844,0.372501,0.979907,-0.001342



MODULE 15 — BACKBONE DECISION SUMMARY


,Question,Answer,Value
0,Does repaired Core4 outperform HGB alone?,True,0.002272
1,Does repaired Core4 outperform HGB+CatBoost?,True,0.000389
2,Does repair materially collapse previous Core4?,False,-0.00102
3,Is repaired Core4 the best tested fixed subset?,False,Provisional_Core4
4,Core4 beats HGB+CatBoost in how many folds?,4,/ 5



MODULE 15 COMPLETE

IMPORTANT:
The original ETR_EXPERT_OOF matrix has NOT been overwritten.
Use REPAIRED_EXPERT_OOF only after we inspect these results.


In [38]:
# ============================================================
# MODULE 16A — STRICT REPAIRED EXPERT INTERFACE
#
# Produces fold-specific expert probabilities for:
#
#   HGB
#   CatBoost
#   AutoInt_Repaired
#   DeepFM_v2_Repaired
#
# for use by the final LPNS validation.
#
# IMPORTANT:
#   - HOLDOUT untouched
#   - repaired neural RAW OOF only
#   - calibration fitted within current outer training
#   - outer-valid labels never used to fit calibrators
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# 16A.1 REQUIREMENTS
# ============================================================

required_16a = [
    "AUTOINT_REPAIRED_OOF",
    "DEEPFM_V2_REPAIRED_OOF",
    "fit_apply_nested_calibration",
    "fold_assignment",
    "y_dev",
    "N_SPLITS",
    "SEED",
]


missing = [
    x
    for x in required_16a
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


y16 = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


AUTO_RAW_16 = np.asarray(
    AUTOINT_REPAIRED_OOF,
    dtype=float
)


DEEPFM_RAW_16 = np.asarray(
    DEEPFM_V2_REPAIRED_OOF,
    dtype=float
)


# ============================================================
# 16A.2 LOGIT
# ============================================================

def repair16_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


# ============================================================
# 16A.3 PLATT MODEL
# ============================================================

def repair16_fit_platt(
    p,
    y,
    seed
):

    model = LogisticRegression(
        penalty="l2",
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
        random_state=seed,
    )


    model.fit(
        repair16_logit(p)[:, None],
        y
    )


    return model


def repair16_apply_platt(
    model,
    p
):

    return (
        model
        .predict_proba(
            repair16_logit(p)[:, None]
        )[:, 1]
    )


# ============================================================
# 16A.4 STRICT NEURAL CALIBRATION INSIDE ONE OUTER FOLD
# ============================================================
#
# For OUTER VALID:
#
#   calibrator = all outer-training neural OOF predictions
#
# For OUTER TRAIN:
#
#   use cross-fitted calibration within outer-training only
#
# Therefore no outer-valid labels enter calibration.
#
# ============================================================

def repair16_nested_neural_calibration(
    raw_oof,
    outer_train_idx,
    outer_valid_idx,
    model_name,
    outer_fold
):

    raw_oof = np.asarray(
        raw_oof,
        dtype=float
    )


    # --------------------------------------------------------
    # A. OUTER-VALID CALIBRATION
    # --------------------------------------------------------

    outer_calibrator = (
        repair16_fit_platt(

            raw_oof[
                outer_train_idx
            ],

            y16[
                outer_train_idx
            ],

            seed=
                SEED
                + 160000
                + outer_fold
        )
    )


    p_outer_valid = (
        repair16_apply_platt(

            outer_calibrator,

            raw_oof[
                outer_valid_idx
            ]
        )
    )


    # --------------------------------------------------------
    # B. OUTER-TRAIN CROSS-FITTED CALIBRATION
    # --------------------------------------------------------

    p_outer_train = np.zeros(
        len(
            outer_train_idx
        ),
        dtype=float
    )


    train_original_folds = (
        fold_assignment[
            outer_train_idx
        ]
    )


    unique_train_folds = np.unique(
        train_original_folds
    )


    for inner_fold in unique_train_folds:

        local_valid = np.where(
            train_original_folds
            ==
            inner_fold
        )[0]


        local_fit = np.where(
            train_original_folds
            !=
            inner_fold
        )[0]


        global_fit = (
            outer_train_idx[
                local_fit
            ]
        )


        global_valid = (
            outer_train_idx[
                local_valid
            ]
        )


        calibrator = (
            repair16_fit_platt(

                raw_oof[
                    global_fit
                ],

                y16[
                    global_fit
                ],

                seed=
                    SEED
                    + 161000
                    + 100 * outer_fold
                    + int(inner_fold)
            )
        )


        p_outer_train[
            local_valid
        ] = (
            repair16_apply_platt(

                calibrator,

                raw_oof[
                    global_valid
                ]
            )
        )


    if np.any(
        p_outer_train <= 0
    ):

        raise RuntimeError(
            f"{model_name}: incomplete nested calibration."
        )


    return (
        p_outer_train,
        p_outer_valid
    )


# ============================================================
# 16A.5 FINAL STRICT EXPERT FUNCTION
# ============================================================

FINAL_REPAIRED_EXPERTS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt_Repaired",
    "DeepFM_v2_Repaired",
]


def fit_apply_final_repaired_expert(
    model_name,
    outer_train_idx,
    outer_valid_idx,
    outer_fold=None
):

    if outer_fold is None:

        detected = np.unique(
            fold_assignment[
                outer_valid_idx
            ]
        )

        if len(detected) != 1:

            raise ValueError(
                "Could not infer outer fold."
            )

        outer_fold = int(
            detected[0]
        )


    # --------------------------------------------------------
    # Classical experts:
    # reuse earlier strict nested-calibration machinery
    # --------------------------------------------------------

    if model_name == "HistGradientBoosting":

        return fit_apply_nested_calibration(
            "HistGradientBoosting",
            outer_train_idx,
            outer_valid_idx
        )


    if model_name == "CatBoost":

        return fit_apply_nested_calibration(
            "CatBoost",
            outer_train_idx,
            outer_valid_idx
        )


    # --------------------------------------------------------
    # Repaired AutoInt
    # --------------------------------------------------------

    if model_name == "AutoInt_Repaired":

        return repair16_nested_neural_calibration(

            AUTO_RAW_16,

            outer_train_idx,

            outer_valid_idx,

            "AutoInt_Repaired",

            outer_fold
        )


    # --------------------------------------------------------
    # Repaired DeepFM_v2
    # --------------------------------------------------------

    if model_name == "DeepFM_v2_Repaired":

        return repair16_nested_neural_calibration(

            DEEPFM_RAW_16,

            outer_train_idx,

            outer_valid_idx,

            "DeepFM_v2_Repaired",

            outer_fold
        )


    raise ValueError(
        f"Unknown expert: {model_name}"
    )


# ============================================================
# 16A.6 SANITY CHECK ALL OUTER FOLDS
# ============================================================

sanity_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 120
    )

    print(
        f"STRICT REPAIRED EXPERT CHECK — FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 120
    )


    outer_train_idx = np.where(
        fold_assignment
        !=
        outer_fold
    )[0]


    outer_valid_idx = np.where(
        fold_assignment
        ==
        outer_fold
    )[0]


    for model_name in (
        FINAL_REPAIRED_EXPERTS
    ):

        p_train, p_valid = (
            fit_apply_final_repaired_expert(

                model_name,

                outer_train_idx,

                outer_valid_idx,

                outer_fold=
                    outer_fold
            )
        )


        if len(p_train) != len(
            outer_train_idx
        ):

            raise RuntimeError(
                f"{model_name}: incorrect train length."
            )


        if len(p_valid) != len(
            outer_valid_idx
        ):

            raise RuntimeError(
                f"{model_name}: incorrect valid length."
            )


        if not np.all(
            np.isfinite(
                p_train
            )
        ):

            raise RuntimeError(
                f"{model_name}: nonfinite train predictions."
            )


        if not np.all(
            np.isfinite(
                p_valid
            )
        ):

            raise RuntimeError(
                f"{model_name}: nonfinite valid predictions."
            )


        fold_pr = (
            average_precision_score(

                y16[
                    outer_valid_idx
                ],

                p_valid
            )
        )


        fold_roc = (
            roc_auc_score(

                y16[
                    outer_valid_idx
                ],

                p_valid
            )
        )


        sanity_rows.append({

            "Fold":
                outer_fold,

            "Model":
                model_name,

            "Train_N":
                len(
                    p_train
                ),

            "Valid_N":
                len(
                    p_valid
                ),

            "Train_Mean":
                float(
                    np.mean(
                        p_train
                    )
                ),

            "Valid_Mean":
                float(
                    np.mean(
                        p_valid
                    )
                ),

            "Valid_PR":
                fold_pr,

            "Valid_ROC":
                fold_roc,
        })


        print(
            f"  {model_name:<24} "
            f"PR={fold_pr:.6f} "
            f"ROC={fold_roc:.6f} "
            f"mean={np.mean(p_valid):.6f}"
        )


FINAL_REPAIRED_EXPERT_SANITY = (
    pd.DataFrame(
        sanity_rows
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16A — STRICT REPAIRED EXPERT SANITY"
)

print(
    "=" * 135
)


display(
    FINAL_REPAIRED_EXPERT_SANITY
)


# ============================================================
# 16A.7 BUILD STRICT REPAIRED CORE4 OOF
# ============================================================

FINAL_REPAIRED_CORE4_OOF = np.zeros(
    len(
        y16
    ),
    dtype=float
)


FINAL_REPAIRED_EXPERT_OOF_STRICT = np.zeros(
    (
        len(
            y16
        ),
        4
    ),
    dtype=float
)


for outer_fold in range(
    N_SPLITS
):

    outer_train_idx = np.where(
        fold_assignment
        !=
        outer_fold
    )[0]


    outer_valid_idx = np.where(
        fold_assignment
        ==
        outer_fold
    )[0]


    valid_experts = []


    for model_name in (
        FINAL_REPAIRED_EXPERTS
    ):

        _, p_valid = (
            fit_apply_final_repaired_expert(

                model_name,

                outer_train_idx,

                outer_valid_idx,

                outer_fold=
                    outer_fold
            )
        )


        valid_experts.append(
            p_valid
        )


    valid_matrix = np.column_stack(
        valid_experts
    )


    FINAL_REPAIRED_EXPERT_OOF_STRICT[
        outer_valid_idx,
        :
    ] = valid_matrix


    FINAL_REPAIRED_CORE4_OOF[
        outer_valid_idx
    ] = np.mean(
        valid_matrix,
        axis=1
    )


# ============================================================
# 16A.8 GLOBAL STRICT BACKBONE RESULT
# ============================================================

final_core4_result = pd.DataFrame([{

    "Model":
        "Final_Repaired_Core4_Strict",

    "PR_AUC":
        average_precision_score(
            y16,
            FINAL_REPAIRED_CORE4_OOF
        ),

    "ROC_AUC":
        roc_auc_score(
            y16,
            FINAL_REPAIRED_CORE4_OOF
        ),

    "Brier":
        brier_score_loss(
            y16,
            FINAL_REPAIRED_CORE4_OOF
        ),

    "NLL":
        log_loss(
            y16,
            np.clip(
                FINAL_REPAIRED_CORE4_OOF,
                1e-8,
                1 - 1e-8
            )
        ),

    "Mean_Probability":
        float(
            FINAL_REPAIRED_CORE4_OOF.mean()
        ),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16A — FINAL STRICT REPAIRED CORE4"
)

print(
    "=" * 135
)


display(
    final_core4_result
)


# ============================================================
# 16A.9 SAVE
# ============================================================

save_csv(

    FINAL_REPAIRED_EXPERT_SANITY,

    TAB_DIR
    / "module16a_strict_repaired_expert_sanity.csv"
)


save_csv(

    final_core4_result,

    TAB_DIR
    / "module16a_final_repaired_core4.csv"
)


final16_pred_df = pd.DataFrame({

    "target":
        y16,

    "fold":
        fold_assignment,

    "HGB":
        FINAL_REPAIRED_EXPERT_OOF_STRICT[
            :,
            0
        ],

    "CatBoost":
        FINAL_REPAIRED_EXPERT_OOF_STRICT[
            :,
            1
        ],

    "AutoInt_Repaired":
        FINAL_REPAIRED_EXPERT_OOF_STRICT[
            :,
            2
        ],

    "DeepFM_v2_Repaired":
        FINAL_REPAIRED_EXPERT_OOF_STRICT[
            :,
            3
        ],

    "Core4":
        FINAL_REPAIRED_CORE4_OOF,
})


save_csv(

    final16_pred_df,

    PRED_DIR
    / "module16a_strict_repaired_core4_oof.csv"
)


module16a_manifest = {

    "module":
        "Strict repaired expert interface",

    "experts":
        FINAL_REPAIRED_EXPERTS,

    "neural_source":
        "Module 14 repaired raw OOF",

    "neural_calibration":
        (
            "nested within current outer-training set; "
            "outer-valid labels excluded"
        ),

    "classical_calibration":
        "existing strict nested calibration helper",

    "holdout_used":
        False,

    "architecture_changed":
        False,
}


save_json(

    module16a_manifest,

    LOG_DIR
    / "module16a_manifest.json"
)


print(
    "\nMODULE 16A COMPLETE."
)


STRICT REPAIRED EXPERT CHECK — FOLD 1/5
  HistGradientBoosting     PR=0.374987 ROC=0.850675 mean=0.094405
  CatBoost                 PR=0.376613 ROC=0.850823 mean=0.094742
  AutoInt_Repaired         PR=0.376410 ROC=0.850906 mean=0.091117
  DeepFM_v2_Repaired       PR=0.373932 ROC=0.849902 mean=0.097679

STRICT REPAIRED EXPERT CHECK — FOLD 2/5
  HistGradientBoosting     PR=0.382048 ROC=0.849930 mean=0.094629
  CatBoost                 PR=0.378998 ROC=0.849837 mean=0.094873
  AutoInt_Repaired         PR=0.379039 ROC=0.849364 mean=0.097745
  DeepFM_v2_Repaired       PR=0.373103 ROC=0.848159 mean=0.098539

STRICT REPAIRED EXPERT CHECK — FOLD 3/5
  HistGradientBoosting     PR=0.374669 ROC=0.848892 mean=0.093749
  CatBoost                 PR=0.376154 ROC=0.847471 mean=0.093871
  AutoInt_Repaired         PR=0.377862 ROC=0.847702 mean=0.105139
  DeepFM_v2_Repaired       PR=0.372077 ROC=0.846274 mean=0.093893

STRICT REPAIRED EXPERT CHECK — FOLD 4/5
  HistGradientBoosting     PR=0.370674 ROC=0

,Fold,Model,Train_N,Valid_N,Train_Mean,Valid_Mean,Valid_PR,Valid_ROC
0,0,HistGradientBoosting,162355,40589,0.093918,0.094405,0.374987,0.850675
1,0,CatBoost,162355,40589,0.094163,0.094742,0.376613,0.850823
2,0,AutoInt_Repaired,162355,40589,0.094291,0.091117,0.376410,0.850906
3,0,DeepFM_v2_Repaired,162355,40589,0.094197,0.097679,0.373932,0.849902
4,1,HistGradientBoosting,162355,40589,0.093862,0.094629,0.382048,0.849930
5,1,CatBoost,162355,40589,0.094163,0.094873,0.378998,0.849837
6,1,AutoInt_Repaired,162355,40589,0.094318,0.097745,0.379039,0.849364
7,1,DeepFM_v2_Repaired,162355,40589,0.094194,0.098539,0.373103,0.848159
8,2,HistGradientBoosting,162355,40589,0.094082,0.093749,0.374669,0.848892
9,2,CatBoost,162355,40589,0.094163,0.093871,0.376154,0.847471



MODULE 16A — FINAL STRICT REPAIRED CORE4


,Model,PR_AUC,ROC_AUC,Brier,NLL,Mean_Probability
0,Final_Repaired_Core4_Strict,0.379284,0.850402,0.069748,0.23571,0.094155



MODULE 16A COMPLETE.


In [39]:
# ============================================================
# MODULE 16B — FINAL STRICT REPAIRED LPNS-MoE
#
# Final development validation using:
#
#   HistGradientBoosting
#   CatBoost
#   AutoInt_Repaired
#   DeepFM_v2_Repaired
#
# Predictor:
#   strict repaired four-expert MoE
#
# Reliability head:
#   confidence
#   + expert disagreement
#   + Local Prediction Neighborhood Stability (LPNS)
#
# IMPORTANT:
#   - repaired neural experts only
#   - outer-valid labels excluded from calibration/training
#   - reliability-fit samples excluded from neighborhood bank
#   - outer-valid samples excluded from neighborhood bank
#   - distance preprocessing fitted on development/reference pool
#   - HOLDOUT untouched
#
# Architecture is LOCKED.
# ============================================================


import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
)


# ============================================================
# 16B.1 REQUIREMENTS
# ============================================================

required_16b = [
    "fit_apply_final_repaired_expert",
    "FINAL_REPAIRED_EXPERTS",
    "FINAL_REPAIRED_CORE4_OOF",
    "make_oracle_targets",
    "train_ramoe_gate",
    "predict_gate_weights",
    "mixture_probability",
    "select_f1_threshold_local",
    "RAMOE_FEATURES",
    "RAMOE_CONFIG",
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_16b
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


# Protect torch.nn namespace
import torch.nn as nn


# ============================================================
# 16B.2 CONFIG
# ============================================================

FINAL_LPNS_CONFIG = {

    "reliability_fraction":
        0.30,

    "threshold_fraction_within_gate_pool":
        0.15,

    "max_reference_size":
        80000,

    "k_neighbors":
        20,

    "max_distance":
        3.0,

    "min_neighbors":
        6,

    "discrete_weight":
        0.65,

    "reliability_C":
        0.10,

    "deferral_rates":
        [0.05, 0.10, 0.15, 0.20],
}


# ============================================================
# 16B.3 LPNS FEATURE DEFINITIONS
# ============================================================

LPNS_BINARY = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]

LPNS_ORDINAL = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]

LPNS_COUNT = [
    "MentHlth",
    "PhysHlth",
]

LPNS_CONTINUOUS = [
    "BMI",
]

LPNS_MULTICLASS = [
    "Diabetes",
]


LPNS_ALL_FEATURES = (
    LPNS_BINARY
    + LPNS_ORDINAL
    + LPNS_COUNT
    + LPNS_CONTINUOUS
    + LPNS_MULTICLASS
)


LPNS_SCALE_FEATURES = (
    LPNS_CONTINUOUS
    + LPNS_COUNT
    + LPNS_ORDINAL
)


LPNS_DISCRETE_FEATURES = (
    LPNS_BINARY
    + LPNS_MULTICLASS
)


# ============================================================
# 16B.4 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)

N = len(
    y_all
)


X_distance_raw = (
    X_dev[
        LPNS_ALL_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


X_gate_raw = (
    X_dev[
        RAMOE_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


distance_feature_position = {
    name: j
    for j, name in enumerate(
        LPNS_ALL_FEATURES
    )
}


scale_cols = np.array([
    distance_feature_position[x]
    for x in LPNS_SCALE_FEATURES
])


discrete_cols = np.array([
    distance_feature_position[x]
    for x in LPNS_DISCRETE_FEATURES
])


# ============================================================
# 16B.5 ENTROPY HELPERS
# ============================================================

def final_binary_entropy(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-8,
        1.0 - 1e-8
    )

    return (
        -p * np.log(p)
        -
        (1.0 - p) * np.log(1.0 - p)
    ) / np.log(2.0)


def final_gate_entropy(w):

    w = np.clip(
        np.asarray(
            w,
            dtype=float
        ),
        1e-12,
        1.0
    )

    return (
        -np.sum(
            w * np.log(w),
            axis=1
        )
        /
        np.log(
            w.shape[1]
        )
    )


# ============================================================
# 16B.6 DISTANCE TRANSFORM
#
# Fit ONLY on gate/reference-development pool.
# ============================================================

def fit_final_distance_transform(
    X_fit_raw
):

    X_fit = (
        X_fit_raw
        .astype(np.float32)
        .copy()
    )


    scaler = StandardScaler()


    X_fit[
        :,
        scale_cols
    ] = (
        scaler
        .fit_transform(
            X_fit[
                :,
                scale_cols
            ]
        )
        .astype(np.float32)
    )


    X_fit[
        :,
        discrete_cols
    ] *= (
        FINAL_LPNS_CONFIG[
            "discrete_weight"
        ]
    )


    return (
        X_fit,
        scaler
    )


def apply_final_distance_transform(
    X_raw,
    scaler
):

    X_out = (
        X_raw
        .astype(np.float32)
        .copy()
    )


    X_out[
        :,
        scale_cols
    ] = (
        scaler
        .transform(
            X_out[
                :,
                scale_cols
            ]
        )
        .astype(np.float32)
    )


    X_out[
        :,
        discrete_cols
    ] *= (
        FINAL_LPNS_CONFIG[
            "discrete_weight"
        ]
    )


    return X_out


# ============================================================
# 16B.7 LPNS EXTRACTION
# ============================================================

def extract_final_lpns_features(
    query_expert_prob,
    reference_expert_prob,
    neighbor_idx,
    neighbor_dist,
    decision_threshold
):

    query_expert_prob = np.asarray(
        query_expert_prob,
        dtype=np.float32
    )


    reference_expert_prob = np.asarray(
        reference_expert_prob,
        dtype=np.float32
    )


    neighbor_prob = (
        reference_expert_prob[
            neighbor_idx
        ]
    )


    valid_neighbor = (
        neighbor_dist
        <=
        FINAL_LPNS_CONFIG[
            "max_distance"
        ]
    )


    count = (
        valid_neighbor.sum(
            axis=1
        )
    )


    count_safe = np.maximum(
        count,
        1
    )[:, None]


    mask3 = (
        valid_neighbor[
            :,
            :,
            None
        ]
    )


    center = (
        query_expert_prob[
            :,
            None,
            :
        ]
    )


    # --------------------------------------------------------
    # Local MAD
    # --------------------------------------------------------

    local_mad = (

        np.sum(

            np.abs(
                neighbor_prob
                -
                center
            )

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Local mean
    # --------------------------------------------------------

    local_mean = (

        np.sum(
            neighbor_prob
            *
            mask3,
            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Local SD
    # --------------------------------------------------------

    local_sd = np.sqrt(

        np.sum(

            (
                neighbor_prob
                -
                local_mean[
                    :,
                    None,
                    :
                ]
            ) ** 2

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Decision flip
    # --------------------------------------------------------

    threshold_value = float(
        decision_threshold
    )


    center_decision = (
        query_expert_prob
        >=
        threshold_value
    )


    neighbor_decision = (
        neighbor_prob
        >=
        threshold_value
    )


    flip = (

        np.sum(

            (
                neighbor_decision
                !=
                center_decision[
                    :,
                    None,
                    :
                ]
            )

            *
            mask3,

            axis=1
        )

        /
        count_safe
    )


    # --------------------------------------------------------
    # Aggregates
    # --------------------------------------------------------

    mean_mad = np.mean(
        local_mad,
        axis=1
    )

    max_mad = np.max(
        local_mad,
        axis=1
    )

    min_mad = np.min(
        local_mad,
        axis=1
    )

    mad_gap = (
        max_mad
        -
        min_mad
    )

    mad_cv = (

        np.std(
            local_mad,
            axis=1
        )

        /
        (
            mean_mad
            +
            1e-8
        )
    )


    mean_sd = np.mean(
        local_sd,
        axis=1
    )

    max_sd = np.max(
        local_sd,
        axis=1
    )

    mean_flip = np.mean(
        flip,
        axis=1
    )

    max_flip = np.max(
        flip,
        axis=1
    )


    features = np.column_stack([

        mean_mad,
        max_mad,
        min_mad,
        mad_gap,
        mad_cv,

        mean_sd,
        max_sd,

        mean_flip,
        max_flip,

        count.astype(float),

        local_mad,
        local_sd,
        flip,
    ])


    valid_profile = (
        count
        >=
        FINAL_LPNS_CONFIG[
            "min_neighbors"
        ]
    )


    features[
        ~valid_profile
    ] = np.nan


    return (
        features,
        valid_profile,
        local_mad,
        local_sd,
        flip,
        count,
    )


# ============================================================
# 16B.8 TRAIN-ONLY IMPUTATION
# ============================================================

def final_impute(
    train_features,
    valid_features
):

    train_features = (
        train_features
        .astype(float)
        .copy()
    )


    valid_features = (
        valid_features
        .astype(float)
        .copy()
    )


    medians = np.nanmedian(
        train_features,
        axis=0
    )


    medians = np.where(
        np.isnan(medians),
        0.0,
        medians
    )


    train_nan = np.where(
        np.isnan(
            train_features
        )
    )


    train_features[
        train_nan
    ] = medians[
        train_nan[1]
    ]


    valid_nan = np.where(
        np.isnan(
            valid_features
        )
    )


    valid_features[
        valid_nan
    ] = medians[
        valid_nan[1]
    ]


    return (
        train_features,
        valid_features
    )


# ============================================================
# 16B.9 TRUST FEATURE SETS
# ============================================================

def build_final_base_features(
    moe_probability,
    expert_probability,
    routing_weights,
    threshold_value
):

    p = np.asarray(
        moe_probability,
        dtype=float
    )


    experts = np.asarray(
        expert_probability,
        dtype=float
    )


    weights = np.asarray(
        routing_weights,
        dtype=float
    )


    entropy = (
        final_binary_entropy(
            p
        )
    )


    signed_margin = (
        p
        -
        float(
            threshold_value
        )
    )


    abs_margin = np.abs(
        signed_margin
    )


    distance_half = np.abs(
        p
        -
        0.5
    )


    # confidence
    F_conf = np.column_stack([

        p,
        entropy,
        signed_margin,
        abs_margin,
        distance_half,
    ])


    # expert disagreement
    expert_mean = np.mean(
        experts,
        axis=1
    )

    expert_std = np.std(
        experts,
        axis=1
    )

    expert_min = np.min(
        experts,
        axis=1
    )

    expert_max = np.max(
        experts,
        axis=1
    )

    expert_range = (
        expert_max
        -
        expert_min
    )

    expert_median = np.median(
        experts,
        axis=1
    )

    consensus_gap = np.abs(
        p
        -
        expert_mean
    )

    expert_abs_dev = np.mean(

        np.abs(
            experts
            -
            p[:, None]
        ),

        axis=1
    )


    F_disagreement = np.column_stack([

        F_conf,

        expert_mean,
        expert_std,
        expert_min,
        expert_max,
        expert_range,
        expert_median,
        consensus_gap,
        expert_abs_dev,

        experts,
    ])


    # routing state
    gate_entropy = (
        final_gate_entropy(
            weights
        )
    )


    max_gate = np.max(
        weights,
        axis=1
    )


    sorted_gate = np.sort(
        weights,
        axis=1
    )


    gate_margin = (

        sorted_gate[
            :,
            -1
        ]

        -

        sorted_gate[
            :,
            -2
        ]
    )


    effective_experts = np.exp(

        -np.sum(

            np.clip(
                weights,
                1e-12,
                1.0
            )

            *
            np.log(
                np.clip(
                    weights,
                    1e-12,
                    1.0
                )
            ),

            axis=1
        )
    )


    weighted_variance = np.sum(

        weights

        *
        (
            experts
            -
            p[:, None]
        ) ** 2,

        axis=1
    )


    F_routing = np.column_stack([

        gate_entropy,
        max_gate,
        gate_margin,
        effective_experts,
        weighted_variance,

        weights,
    ])


    return (
        F_conf,
        F_disagreement,
        F_routing
    )


# ============================================================
# 16B.10 RELIABILITY MODEL
# ============================================================

def make_final_reliability_model():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    FINAL_LPNS_CONFIG[
                        "reliability_C"
                    ],

                class_weight="balanced",

                solver="lbfgs",

                max_iter=3000,
            )
        )
    ])


# ============================================================
# 16B.11 OUTPUT BUFFERS
# ============================================================

FINAL_MOE_OOF = np.zeros(
    N,
    dtype=float
)


FINAL_THRESHOLD_OOF = np.zeros(
    N,
    dtype=float
)


FINAL_ERROR_OOF = np.zeros(
    N,
    dtype=int
)


FINAL_RISK_CONF = np.zeros(
    N,
    dtype=float
)


FINAL_RISK_DISAGREE = np.zeros(
    N,
    dtype=float
)


FINAL_RISK_LPNS = np.zeros(
    N,
    dtype=float
)


FINAL_RISK_LPNS_ROUTING = np.zeros(
    N,
    dtype=float
)


FINAL_MEAN_MAD = np.full(
    N,
    np.nan
)


FINAL_MAX_MAD = np.full(
    N,
    np.nan
)


fold_rows = []

neighborhood_rows = []

routing_rows = []


# ============================================================
# 16B.12 OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"FINAL STRICT REPAIRED LPNS — FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    # ========================================================
    # A. OUTER INDICES
    # ========================================================

    outer_train_idx = np.where(
        fold_assignment
        !=
        outer_fold
    )[0]


    outer_valid_idx = np.where(
        fold_assignment
        ==
        outer_fold
    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # B. STRICT REPAIRED EXPERT PROBABILITIES
    # ========================================================

    print(
        "\nBuilding repaired strict expert probabilities..."
    )


    train_cal = {}

    valid_cal = {}


    for model_name in (
        FINAL_REPAIRED_EXPERTS
    ):

        p_train, p_valid = (
            fit_apply_final_repaired_expert(

                model_name,

                outer_train_idx,

                outer_valid_idx,

                outer_fold=
                    outer_fold
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    P_outer_train = np.column_stack([

        train_cal[x]

        for x in FINAL_REPAIRED_EXPERTS
    ])


    P_outer_valid = np.column_stack([

        valid_cal[x]

        for x in FINAL_REPAIRED_EXPERTS
    ])


    # ========================================================
    # C. GATE-DEVELOPMENT / RELIABILITY SPLIT
    # ========================================================

    split_reliability = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            FINAL_LPNS_CONFIG[
                "reliability_fraction"
            ],

        random_state=
            SEED
            + 180000
            + outer_fold
    )


    (
        gate_pool_local,
        reliability_local

    ) = next(

        split_reliability.split(

            np.zeros(
                len(
                    outer_train_idx
                )
            ),

            y_outer_train
        )
    )


    gate_pool_global = (
        outer_train_idx[
            gate_pool_local
        ]
    )


    reliability_global = (
        outer_train_idx[
            reliability_local
        ]
    )


    y_gate_pool = (
        y_all[
            gate_pool_global
        ]
    )


    # ========================================================
    # D. GATE FIT / THRESHOLD SPLIT
    # ========================================================

    split_threshold = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            FINAL_LPNS_CONFIG[
                "threshold_fraction_within_gate_pool"
            ],

        random_state=
            SEED
            + 181000
            + outer_fold
    )


    (
        gate_fit_inside,
        threshold_inside

    ) = next(

        split_threshold.split(

            np.zeros(
                len(
                    gate_pool_global
                )
            ),

            y_gate_pool
        )
    )


    gate_fit_local = (
        gate_pool_local[
            gate_fit_inside
        ]
    )


    threshold_local = (
        gate_pool_local[
            threshold_inside
        ]
    )


    gate_fit_global = (
        outer_train_idx[
            gate_fit_local
        ]
    )


    threshold_global = (
        outer_train_idx[
            threshold_local
        ]
    )


    print(
        "\nPartitions:"
    )

    print(
        "  Gate fit:",
        len(
            gate_fit_global
        )
    )

    print(
        "  Threshold:",
        len(
            threshold_global
        )
    )

    print(
        "  Reliability:",
        len(
            reliability_global
        )
    )

    print(
        "  Outer valid:",
        len(
            outer_valid_idx
        )
    )


    # ========================================================
    # E. TRAIN FIXED RA-MoE GATE
    # ========================================================

    gate_scaler = StandardScaler()


    X_gate_fit = (
        gate_scaler
        .fit_transform(
            X_gate_raw[
                gate_fit_global
            ]
        )
        .astype(np.float32)
    )


    X_threshold = (
        gate_scaler
        .transform(
            X_gate_raw[
                threshold_global
            ]
        )
        .astype(np.float32)
    )


    X_reliability_gate = (
        gate_scaler
        .transform(
            X_gate_raw[
                reliability_global
            ]
        )
        .astype(np.float32)
    )


    X_valid_gate = (
        gate_scaler
        .transform(
            X_gate_raw[
                outer_valid_idx
            ]
        )
        .astype(np.float32)
    )


    P_gate_fit = (
        P_outer_train[
            gate_fit_local
        ]
    )


    y_gate_fit = (
        y_all[
            gate_fit_global
        ]
    )


    oracle_target = (
        make_oracle_targets(

            P_gate_fit,

            y_gate_fit,

            temperature=
                RAMOE_CONFIG[
                    "oracle_temperature"
                ]
        )
    )


    gate_model = (
        train_ramoe_gate(

            X_gate_fit,

            oracle_target,

            seed=
                SEED
                + 182000
                + outer_fold
        )
    )


    # ========================================================
    # F. GATE WEIGHTS
    # ========================================================

    W_threshold = (
        predict_gate_weights(
            gate_model,
            X_threshold
        )
    )


    W_reliability = (
        predict_gate_weights(
            gate_model,
            X_reliability_gate
        )
    )


    W_valid = (
        predict_gate_weights(
            gate_model,
            X_valid_gate
        )
    )


    routing_rows.append({

        "Fold":
            outer_fold,

        "Mean_Gate_Entropy":
            float(
                np.mean(
                    final_gate_entropy(
                        W_valid
                    )
                )
            ),

        "Mean_Max_Weight":
            float(
                np.mean(
                    np.max(
                        W_valid,
                        axis=1
                    )
                )
            ),
    })


    # ========================================================
    # G. MoE PROBABILITIES
    # ========================================================

    P_threshold = (
        P_outer_train[
            threshold_local
        ]
    )


    P_reliability = (
        P_outer_train[
            reliability_local
        ]
    )


    moe_threshold = (
        mixture_probability(
            P_threshold,
            W_threshold
        )
    )


    moe_reliability = (
        mixture_probability(
            P_reliability,
            W_reliability
        )
    )


    moe_valid = (
        mixture_probability(
            P_outer_valid,
            W_valid
        )
    )


    # ========================================================
    # H. DISEASE THRESHOLD
    # ========================================================

    y_threshold = (
        y_all[
            threshold_global
        ]
    )


    decision_threshold, _ = (
        select_f1_threshold_local(

            y_threshold,

            moe_threshold
        )
    )


    print(
        "\nDecision threshold:",
        decision_threshold
    )


    # ========================================================
    # I. ERROR TARGET
    # ========================================================

    y_reliability = (
        y_all[
            reliability_global
        ]
    )


    pred_reliability = (
        moe_reliability
        >=
        decision_threshold
    ).astype(int)


    pred_valid = (
        moe_valid
        >=
        decision_threshold
    ).astype(int)


    error_reliability = (
        pred_reliability
        !=
        y_reliability
    ).astype(int)


    error_valid = (
        pred_valid
        !=
        y_outer_valid
    ).astype(int)


    FINAL_MOE_OOF[
        outer_valid_idx
    ] = moe_valid


    FINAL_THRESHOLD_OOF[
        outer_valid_idx
    ] = decision_threshold


    FINAL_ERROR_OOF[
        outer_valid_idx
    ] = error_valid


    # ========================================================
    # J. LPNS DISTANCE TRANSFORM
    #
    # FIT ONLY ON GATE/REFERENCE DEVELOPMENT POOL
    # ========================================================

    (
        X_gatepool_distance,
        distance_scaler

    ) = fit_final_distance_transform(

        X_distance_raw[
            gate_pool_global
        ]
    )


    X_reliability_distance = (
        apply_final_distance_transform(

            X_distance_raw[
                reliability_global
            ],

            distance_scaler
        )
    )


    X_valid_distance = (
        apply_final_distance_transform(

            X_distance_raw[
                outer_valid_idx
            ],

            distance_scaler
        )
    )


    # ========================================================
    # K. REFERENCE BANK
    # ========================================================

    rng = np.random.default_rng(

        SEED
        + 183000
        + outer_fold
    )


    reference_inside_gatepool = np.arange(
        len(
            gate_pool_global
        )
    )


    max_reference = (
        FINAL_LPNS_CONFIG[
            "max_reference_size"
        ]
    )


    if len(
        reference_inside_gatepool
    ) > max_reference:

        reference_inside_gatepool = (
            rng.choice(

                reference_inside_gatepool,

                size=
                    max_reference,

                replace=False
            )
        )


    reference_global = (
        gate_pool_global[
            reference_inside_gatepool
        ]
    )


    X_reference = (
        X_gatepool_distance[
            reference_inside_gatepool
        ]
    )


    # translate global reference rows into positions
    # inside outer_train_idx
    global_to_outer_local = {
        int(g): i
        for i, g in enumerate(
            outer_train_idx
        )
    }


    reference_outer_local = np.array([
        global_to_outer_local[
            int(g)
        ]
        for g in reference_global
    ])


    P_reference = (
        P_outer_train[
            reference_outer_local
        ]
    )


    print(
        "Reference bank:",
        len(
            reference_global
        )
    )


    # ========================================================
    # L. NEAREST NEIGHBOR MODEL
    # ========================================================

    nbr_model = NearestNeighbors(

        n_neighbors=
            FINAL_LPNS_CONFIG[
                "k_neighbors"
            ],

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    nbr_model.fit(
        X_reference
    )


    rel_dist, rel_neighbor = (
        nbr_model.kneighbors(

            X_reliability_distance,

            return_distance=True
        )
    )


    valid_dist, valid_neighbor = (
        nbr_model.kneighbors(

            X_valid_distance,

            return_distance=True
        )
    )


    # ========================================================
    # M. LPNS FEATURES
    # ========================================================

    (
        LPNS_rel,
        valid_rel_profile,
        local_mad_rel,
        local_sd_rel,
        flip_rel,
        rel_count

    ) = extract_final_lpns_features(

        P_reliability,

        P_reference,

        rel_neighbor,

        rel_dist,

        decision_threshold
    )


    (
        LPNS_valid,
        valid_valid_profile,
        local_mad_valid,
        local_sd_valid,
        flip_valid,
        valid_count

    ) = extract_final_lpns_features(

        P_outer_valid,

        P_reference,

        valid_neighbor,

        valid_dist,

        decision_threshold
    )


    # ========================================================
    # N. TRAIN-ONLY IMPUTATION
    # ========================================================

    (
        LPNS_rel,
        LPNS_valid

    ) = final_impute(

        LPNS_rel,

        LPNS_valid
    )


    FINAL_MEAN_MAD[
        outer_valid_idx
    ] = np.nanmean(

        local_mad_valid,

        axis=1
    )


    FINAL_MAX_MAD[
        outer_valid_idx
    ] = np.nanmax(

        local_mad_valid,

        axis=1
    )


    neighborhood_rows.append({

        "Fold":
            outer_fold,

        "Reference_N":
            len(
                reference_global
            ),

        "Reliability_LPNS_Coverage":
            float(
                valid_rel_profile.mean()
            ),

        "Validation_LPNS_Coverage":
            float(
                valid_valid_profile.mean()
            ),

        "Reliability_MeanNeighbors":
            float(
                rel_count.mean()
            ),

        "Validation_MeanNeighbors":
            float(
                valid_count.mean()
            ),
    })


    # ========================================================
    # O. BASE TRUST FEATURES
    # ========================================================

    (
        F_conf_rel,
        F_dis_rel,
        F_route_rel

    ) = build_final_base_features(

        moe_reliability,

        P_reliability,

        W_reliability,

        decision_threshold
    )


    (
        F_conf_valid,
        F_dis_valid,
        F_route_valid

    ) = build_final_base_features(

        moe_valid,

        P_outer_valid,

        W_valid,

        decision_threshold
    )


    # ========================================================
    # P. LOCKED ABLATION SETS
    # ========================================================

    F_lpns_rel = np.column_stack([
        F_dis_rel,
        LPNS_rel,
    ])


    F_lpns_valid = np.column_stack([
        F_dis_valid,
        LPNS_valid,
    ])


    F_full_rel = np.column_stack([
        F_lpns_rel,
        F_route_rel,
    ])


    F_full_valid = np.column_stack([
        F_lpns_valid,
        F_route_valid,
    ])


    # ========================================================
    # Q. RELIABILITY MODELS
    # ========================================================

    m_conf = (
        make_final_reliability_model()
    )


    m_dis = (
        make_final_reliability_model()
    )


    m_lpns = (
        make_final_reliability_model()
    )


    m_full = (
        make_final_reliability_model()
    )


    m_conf.fit(
        F_conf_rel,
        error_reliability
    )


    m_dis.fit(
        F_dis_rel,
        error_reliability
    )


    m_lpns.fit(
        F_lpns_rel,
        error_reliability
    )


    m_full.fit(
        F_full_rel,
        error_reliability
    )


    # ========================================================
    # R. OUTER-VALID RELIABILITY RISK
    # ========================================================

    risk_conf = (
        m_conf
        .predict_proba(
            F_conf_valid
        )[:, 1]
    )


    risk_dis = (
        m_dis
        .predict_proba(
            F_dis_valid
        )[:, 1]
    )


    risk_lpns = (
        m_lpns
        .predict_proba(
            F_lpns_valid
        )[:, 1]
    )


    risk_full = (
        m_full
        .predict_proba(
            F_full_valid
        )[:, 1]
    )


    FINAL_RISK_CONF[
        outer_valid_idx
    ] = risk_conf


    FINAL_RISK_DISAGREE[
        outer_valid_idx
    ] = risk_dis


    FINAL_RISK_LPNS[
        outer_valid_idx
    ] = risk_lpns


    FINAL_RISK_LPNS_ROUTING[
        outer_valid_idx
    ] = risk_full


    # ========================================================
    # S. FOLD METRICS
    # ========================================================

    fold_row = {

        "Fold":
            outer_fold,

        "Error_Prevalence":
            error_valid.mean(),

        "Confidence_PR":
            average_precision_score(
                error_valid,
                risk_conf
            ),

        "Disagreement_PR":
            average_precision_score(
                error_valid,
                risk_dis
            ),

        "LPNS_PR":
            average_precision_score(
                error_valid,
                risk_lpns
            ),

        "LPNS_Routing_PR":
            average_precision_score(
                error_valid,
                risk_full
            ),

        "Confidence_ROC":
            roc_auc_score(
                error_valid,
                risk_conf
            ),

        "Disagreement_ROC":
            roc_auc_score(
                error_valid,
                risk_dis
            ),

        "LPNS_ROC":
            roc_auc_score(
                error_valid,
                risk_lpns
            ),

        "LPNS_Routing_ROC":
            roc_auc_score(
                error_valid,
                risk_full
            ),
    }


    fold_row[
        "LPNS_Delta_PR_vs_Disagreement"
    ] = (

        fold_row[
            "LPNS_PR"
        ]

        -

        fold_row[
            "Disagreement_PR"
        ]
    )


    fold_row[
        "LPNS_Delta_ROC_vs_Disagreement"
    ] = (

        fold_row[
            "LPNS_ROC"
        ]

        -

        fold_row[
            "Disagreement_ROC"
        ]
    )


    fold_rows.append(
        fold_row
    )


    print(
        "\nFold reliability:"
    )

    print(
        f"  Confidence       PR={fold_row['Confidence_PR']:.6f}"
    )

    print(
        f"  +Disagreement    PR={fold_row['Disagreement_PR']:.6f}"
    )

    print(
        f"  +LPNS            PR={fold_row['LPNS_PR']:.6f}"
    )

    print(
        f"  +LPNS+Routing    PR={fold_row['LPNS_Routing_PR']:.6f}"
    )

    print(
        f"  LPNS ΔPR         "
        f"{fold_row['LPNS_Delta_PR_vs_Disagreement']:+.6f}"
    )


    # ========================================================
    # T. CLEANUP
    # ========================================================

    del (
        gate_model,
        nbr_model,
        m_conf,
        m_dis,
        m_lpns,
        m_full,
        X_reference,
        P_reference,
        rel_neighbor,
        valid_neighbor,
        rel_dist,
        valid_dist,
    )


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 16B.13 AURC
# ============================================================

def final_aurc(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(

        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 16B.14 GLOBAL RELIABILITY RESULTS
# ============================================================

final_risk_models = {

    "Confidence":
        FINAL_RISK_CONF,

    "Confidence+Disagreement":
        FINAL_RISK_DISAGREE,

    "Final_Strict_LPNS":
        FINAL_RISK_LPNS,

    "Final_Strict_LPNS+Routing":
        FINAL_RISK_LPNS_ROUTING,
}


global_rows = []


for name, risk in (
    final_risk_models.items()
):

    global_rows.append({

        "Reliability_Model":
            name,

        "Error_PR_AUC":
            average_precision_score(
                FINAL_ERROR_OOF,
                risk
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                FINAL_ERROR_OOF,
                risk
            ),

        "AURC":
            final_aurc(
                FINAL_ERROR_OOF,
                risk
            ),
    })


FINAL_LPNS_RESULTS = pd.DataFrame(
    global_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16B — FINAL STRICT REPAIRED LPNS RELIABILITY"
)

print(
    "=" * 135
)


display(
    FINAL_LPNS_RESULTS
)


# ============================================================
# 16B.15 INCREMENTAL VALUE
# ============================================================

def final_result_row(name):

    return (
        FINAL_LPNS_RESULTS[
            FINAL_LPNS_RESULTS[
                "Reliability_Model"
            ]
            ==
            name
        ]
        .iloc[0]
    )


r_conf = final_result_row(
    "Confidence"
)

r_dis = final_result_row(
    "Confidence+Disagreement"
)

r_lpns = final_result_row(
    "Final_Strict_LPNS"
)

r_route = final_result_row(
    "Final_Strict_LPNS+Routing"
)


FINAL_LPNS_GAIN = pd.DataFrame([

    {
        "Comparison":
            "Disagreement vs Confidence",

        "Delta_PR":
            r_dis[
                "Error_PR_AUC"
            ]
            -
            r_conf[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            r_dis[
                "Error_ROC_AUC"
            ]
            -
            r_conf[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            r_dis[
                "AURC"
            ]
            -
            r_conf[
                "AURC"
            ],
    },

    {
        "Comparison":
            "LPNS vs Disagreement",

        "Delta_PR":
            r_lpns[
                "Error_PR_AUC"
            ]
            -
            r_dis[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            r_lpns[
                "Error_ROC_AUC"
            ]
            -
            r_dis[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            r_lpns[
                "AURC"
            ]
            -
            r_dis[
                "AURC"
            ],
    },

    {
        "Comparison":
            "Routing vs LPNS",

        "Delta_PR":
            r_route[
                "Error_PR_AUC"
            ]
            -
            r_lpns[
                "Error_PR_AUC"
            ],

        "Delta_ROC":
            r_route[
                "Error_ROC_AUC"
            ]
            -
            r_lpns[
                "Error_ROC_AUC"
            ],

        "Delta_AURC":
            r_route[
                "AURC"
            ]
            -
            r_lpns[
                "AURC"
            ],
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL LPNS INCREMENTAL VALUE"
)

print(
    "=" * 135
)


display(
    FINAL_LPNS_GAIN
)


# ============================================================
# 16B.16 FINAL PREDICTIONS
# ============================================================

FINAL_PREDICTION = (
    FINAL_MOE_OOF
    >=
    FINAL_THRESHOLD_OOF
).astype(int)


FINAL_FN = (

    (y_all == 1)

    &

    (FINAL_PREDICTION == 0)

).astype(int)


# ============================================================
# 16B.17 DEFERRAL EVALUATION
# ============================================================

def final_deferral(
    risk,
    method
):

    rows = []


    total_error = max(
        FINAL_ERROR_OOF.sum(),
        1
    )


    total_fn = max(
        FINAL_FN.sum(),
        1
    )


    base_error = (
        FINAL_ERROR_OOF.mean()
    )


    order = np.argsort(
        -risk
    )


    for rate in (
        FINAL_LPNS_CONFIG[
            "deferral_rates"
        ]
    ):

        n_defer = max(
            1,
            int(
                round(
                    N * rate
                )
            )
        )


        defer_idx = (
            order[
                :n_defer
            ]
        )


        keep = np.ones(
            N,
            dtype=bool
        )


        keep[
            defer_idx
        ] = False


        deferred_error_rate = (
            FINAL_ERROR_OOF[
                defer_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Deferral_Rate":
                rate,

            "Coverage":
                1.0 - rate,

            "Error_Enrichment":
                (
                    deferred_error_rate
                    /
                    base_error
                ),

            "Errors_Captured":
                (
                    FINAL_ERROR_OOF[
                        defer_idx
                    ].sum()
                    /
                    total_error
                ),

            "FalseNegatives_Captured":
                (
                    FINAL_FN[
                        defer_idx
                    ].sum()
                    /
                    total_fn
                ),

            "Error_Per_100_Reviews":
                (
                    deferred_error_rate
                    * 100.0
                ),

            "Retained_Accuracy":
                accuracy_score(

                    y_all[
                        keep
                    ],

                    FINAL_PREDICTION[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_all[
                        keep
                    ],

                    FINAL_PREDICTION[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_all[
                        keep
                    ],

                    FINAL_PREDICTION[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


final_deferral_frames = []


for name, risk in (
    final_risk_models.items()
):

    final_deferral_frames.append(

        final_deferral(
            risk,
            name
        )
    )


FINAL_DEFERRAL_RESULTS = pd.concat(
    final_deferral_frames,
    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL STRICT LPNS SELECTIVE / DEFERRAL RESULTS"
)

print(
    "=" * 135
)


display(
    FINAL_DEFERRAL_RESULTS
)


# ============================================================
# 16B.18 FOLD RESULTS
# ============================================================

FINAL_LPNS_FOLD_RESULTS = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL STRICT LPNS FOLD RESULTS"
)

print(
    "=" * 135
)


display(
    FINAL_LPNS_FOLD_RESULTS
)


print(
    "\nLPNS PR positive folds:",
    int(
        (
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_PR_vs_Disagreement"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean LPNS ΔPR:",
    FINAL_LPNS_FOLD_RESULTS[
        "LPNS_Delta_PR_vs_Disagreement"
    ].mean()
)


print(
    "LPNS ROC positive folds:",
    int(
        (
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_ROC_vs_Disagreement"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean LPNS ΔROC:",
    FINAL_LPNS_FOLD_RESULTS[
        "LPNS_Delta_ROC_vs_Disagreement"
    ].mean()
)


# ============================================================
# 16B.19 NEIGHBORHOOD DIAGNOSTICS
# ============================================================

FINAL_NEIGHBORHOOD_RESULTS = pd.DataFrame(
    neighborhood_rows
)


FINAL_ROUTING_RESULTS = pd.DataFrame(
    routing_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL LPNS NEIGHBORHOOD DIAGNOSTICS"
)

print(
    "=" * 135
)


display(
    FINAL_NEIGHBORHOOD_RESULTS
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL ROUTING DIAGNOSTICS"
)

print(
    "=" * 135
)


display(
    FINAL_ROUTING_RESULTS
)


# ============================================================
# 16B.20 DISEASE PERFORMANCE
# ============================================================

FINAL_DISEASE_RESULTS = pd.DataFrame([{

    "Model":
        "Final_Repaired_RA_MoE",

    "PR_AUC":
        average_precision_score(
            y_all,
            FINAL_MOE_OOF
        ),

    "ROC_AUC":
        roc_auc_score(
            y_all,
            FINAL_MOE_OOF
        ),

    "Brier":
        brier_score_loss(
            y_all,
            FINAL_MOE_OOF
        ),

    "NLL":
        log_loss(
            y_all,
            np.clip(
                FINAL_MOE_OOF,
                1e-8,
                1 - 1e-8
            )
        ),

    "Mean_Probability":
        float(
            FINAL_MOE_OOF.mean()
        ),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL REPAIRED MoE DISEASE PERFORMANCE"
)

print(
    "=" * 135
)


display(
    FINAL_DISEASE_RESULTS
)


# ============================================================
# 16B.21 SAVE
# ============================================================

save_csv(

    FINAL_LPNS_RESULTS,

    TAB_DIR
    / "module16b_final_lpns_reliability.csv"
)


save_csv(

    FINAL_LPNS_GAIN,

    TAB_DIR
    / "module16b_final_lpns_incremental_gain.csv"
)


save_csv(

    FINAL_DEFERRAL_RESULTS,

    TAB_DIR
    / "module16b_final_lpns_deferral.csv"
)


save_csv(

    FINAL_LPNS_FOLD_RESULTS,

    TAB_DIR
    / "module16b_final_lpns_folds.csv"
)


save_csv(

    FINAL_NEIGHBORHOOD_RESULTS,

    TAB_DIR
    / "module16b_final_lpns_neighborhood.csv"
)


save_csv(

    FINAL_ROUTING_RESULTS,

    TAB_DIR
    / "module16b_final_routing_diagnostics.csv"
)


save_csv(

    FINAL_DISEASE_RESULTS,

    TAB_DIR
    / "module16b_final_disease_performance.csv"
)


final_prediction_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "MoE_probability":
        FINAL_MOE_OOF,

    "Threshold":
        FINAL_THRESHOLD_OOF,

    "Prediction":
        FINAL_PREDICTION,

    "Error":
        FINAL_ERROR_OOF,

    "FalseNegative":
        FINAL_FN,

    "Risk_Confidence":
        FINAL_RISK_CONF,

    "Risk_Disagreement":
        FINAL_RISK_DISAGREE,

    "Risk_LPNS":
        FINAL_RISK_LPNS,

    "Risk_LPNS_Routing":
        FINAL_RISK_LPNS_ROUTING,

    "Mean_Local_MAD":
        FINAL_MEAN_MAD,

    "Max_Local_MAD":
        FINAL_MAX_MAD,
})


save_csv(

    final_prediction_df,

    PRED_DIR
    / "module16b_final_lpns_oof.csv"
)


module16b_manifest = {

    "module":
        "Final strict repaired LPNS-MoE",

    "experts":
        FINAL_REPAIRED_EXPERTS,

    "neural_models_repaired":
        True,

    "expert_calibration":
        "strict nested outer-fold interface",

    "reliability_architecture":
        (
            "confidence + expert disagreement + LPNS"
        ),

    "routing_in_final_reliability_head":
        False,

    "routing_variant_evaluated_as_ablation":
        True,

    "distance_scaler_fit":
        "gate/reference-development pool only",

    "reference_bank":
        "gate-development pool only",

    "reliability_fit_in_reference_bank":
        False,

    "outer_validation_in_reference_bank":
        False,

    "k_neighbors":
        FINAL_LPNS_CONFIG[
            "k_neighbors"
        ],

    "max_reference_size":
        FINAL_LPNS_CONFIG[
            "max_reference_size"
        ],

    "holdout_used":
        False,

    "architecture_locked":
        True,
}


save_json(

    module16b_manifest,

    LOG_DIR
    / "module16b_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nDevelopment architecture remains locked."
)

print(
    "Do not touch HOLDOUT yet."
)


FINAL STRICT REPAIRED LPNS — FOLD 1/5

Building repaired strict expert probabilities...

Partitions:
  Gate fit: 96600
  Threshold: 17048
  Reliability: 48707
  Outer valid: 40589
    gate epoch 1/4 loss=1.461870
    gate epoch 2/4 loss=1.410360
    gate epoch 3/4 loss=1.399336
    gate epoch 4/4 loss=1.394687

Decision threshold: 0.24499999999999988
Reference bank: 80000

Fold reliability:
  Confidence       PR=0.501174
  +Disagreement    PR=0.501098
  +LPNS            PR=0.513369
  +LPNS+Routing    PR=0.512548
  LPNS ΔPR         +0.012271

FINAL STRICT REPAIRED LPNS — FOLD 2/5

Building repaired strict expert probabilities...

Partitions:
  Gate fit: 96600
  Threshold: 17048
  Reliability: 48707
  Outer valid: 40589
    gate epoch 1/4 loss=1.493308
    gate epoch 2/4 loss=1.419055
    gate epoch 3/4 loss=1.402525
    gate epoch 4/4 loss=1.396744

Decision threshold: 0.22499999999999992
Reference bank: 80000

Fold reliability:
  Confidence       PR=0.546909
  +Disagreement    PR=0.55

,Reliability_Model,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.545862,0.900056,0.026931
1,Confidence+Disagreement,0.547198,0.900208,0.026892
2,Final_Strict_LPNS,0.559891,0.900525,0.026829
3,Final_Strict_LPNS+Routing,0.560273,0.900499,0.026830



FINAL LPNS INCREMENTAL VALUE


,Comparison,Delta_PR,Delta_ROC,Delta_AURC
0,Disagreement vs Confidence,0.001336,0.000152,-3.931210e-05
1,LPNS vs Disagreement,0.012693,0.000316,-6.266164e-05
2,Routing vs LPNS,0.000382,-0.000025,6.749020e-07



FINAL STRICT LPNS SELECTIVE / DEFERRAL RESULTS


...


FINAL ROUTING DIAGNOSTICS


,Fold,Mean_Gate_Entropy,Mean_Max_Weight
0,0,0.997133,0.275582
1,1,0.996416,0.279565
2,2,0.997871,0.272462
3,3,0.997673,0.273198
4,4,0.997111,0.276670



FINAL REPAIRED MoE DISEASE PERFORMANCE


,Model,PR_AUC,ROC_AUC,Brier,NLL,Mean_Probability
0,Final_Repaired_RA_MoE,0.379249,0.850396,0.06975,0.235714,0.094123



MODULE 16B COMPLETE

Development architecture remains locked.
Do not touch HOLDOUT yet.


In [40]:
# ============================================================
# MODULE 16C — FINAL STATISTICAL VALIDATION
# Repaired Core4 + Strict LPNS
#
# Primary comparison:
#   Confidence + Disagreement
#            vs
#   Final Strict LPNS
#
# Primary endpoint:
#   Error PR-AUC
#
# Secondary:
#   Error ROC-AUC
#   AURC
#   Selective-review benefit
#
# HOLDOUT REMAINS UNTOUCHED.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 16C.1 REQUIREMENTS
# ============================================================

required_16c = [
    "FINAL_ERROR_OOF",
    "FINAL_RISK_CONF",
    "FINAL_RISK_DISAGREE",
    "FINAL_RISK_LPNS",
    "FINAL_RISK_LPNS_ROUTING",
    "FINAL_FN",
    "FINAL_LPNS_FOLD_RESULTS",
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_16c
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Run Module 16B first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 16C.2 CONFIG
# ============================================================

FINAL_STAT_CONFIG = {

    "n_bootstrap":
        2000,

    "ci_lower":
        2.5,

    "ci_upper":
        97.5,

    "review_rates":
        [0.05, 0.10, 0.15, 0.20],

    "seed":
        SEED + 190000,
}


# ============================================================
# 16C.3 DATA
# ============================================================

error_target = np.asarray(
    FINAL_ERROR_OOF,
    dtype=int
)


fn_target = np.asarray(
    FINAL_FN,
    dtype=int
)


risk_conf = np.asarray(
    FINAL_RISK_CONF,
    dtype=float
)


risk_dis = np.asarray(
    FINAL_RISK_DISAGREE,
    dtype=float
)


risk_lpns = np.asarray(
    FINAL_RISK_LPNS,
    dtype=float
)


risk_route = np.asarray(
    FINAL_RISK_LPNS_ROUTING,
    dtype=float
)


N = len(
    error_target
)


print(
    "N:",
    N
)

print(
    "Error prevalence:",
    error_target.mean()
)

print(
    "False negatives:",
    fn_target.sum()
)


# ============================================================
# 16C.4 AURC
# ============================================================

def final_stat_aurc(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(
        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 16C.5 OBSERVED METRICS
# ============================================================

FINAL_OBSERVED_STATS = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            average_precision_score(
                error_target,
                risk_conf
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error_target,
                risk_conf
            ),

        "AURC":
            final_stat_aurc(
                error_target,
                risk_conf
            ),
    },

    {
        "Method":
            "Confidence+Disagreement",

        "Error_PR_AUC":
            average_precision_score(
                error_target,
                risk_dis
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error_target,
                risk_dis
            ),

        "AURC":
            final_stat_aurc(
                error_target,
                risk_dis
            ),
    },

    {
        "Method":
            "Final_Strict_LPNS",

        "Error_PR_AUC":
            average_precision_score(
                error_target,
                risk_lpns
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error_target,
                risk_lpns
            ),

        "AURC":
            final_stat_aurc(
                error_target,
                risk_lpns
            ),
    },

    {
        "Method":
            "Final_Strict_LPNS+Routing",

        "Error_PR_AUC":
            average_precision_score(
                error_target,
                risk_route
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error_target,
                risk_route
            ),

        "AURC":
            final_stat_aurc(
                error_target,
                risk_route
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16C — FINAL OBSERVED RELIABILITY PERFORMANCE"
)

print(
    "=" * 135
)


display(
    FINAL_OBSERVED_STATS
)


# ============================================================
# 16C.6 BOOTSTRAP HELPERS
# ============================================================

error_idx = np.where(
    error_target == 1
)[0]


correct_idx = np.where(
    error_target == 0
)[0]


rng = np.random.default_rng(
    FINAL_STAT_CONFIG[
        "seed"
    ]
)


def final_draw_bootstrap():

    sampled_error = rng.choice(
        error_idx,
        size=len(error_idx),
        replace=True
    )


    sampled_correct = rng.choice(
        correct_idx,
        size=len(correct_idx),
        replace=True
    )


    return np.concatenate([
        sampled_error,
        sampled_correct,
    ])


def final_ci(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    return (
        float(
            np.percentile(
                values,
                FINAL_STAT_CONFIG[
                    "ci_lower"
                ]
            )
        ),

        float(
            np.percentile(
                values,
                FINAL_STAT_CONFIG[
                    "ci_upper"
                ]
            )
        ),
    )


def final_bootstrap_p(
    delta
):

    delta = np.asarray(
        delta,
        dtype=float
    )


    lower = np.mean(
        delta <= 0
    )


    upper = np.mean(
        delta >= 0
    )


    return min(
        1.0,
        float(
            2.0
            *
            min(
                lower,
                upper
            )
        )
    )


# ============================================================
# 16C.7 PRIMARY PAIRED BOOTSTRAP
# ============================================================

B = (
    FINAL_STAT_CONFIG[
        "n_bootstrap"
    ]
)


base_pr = np.zeros(
    B
)

lpns_pr = np.zeros(
    B
)

base_roc = np.zeros(
    B
)

lpns_roc = np.zeros(
    B
)

base_aurc = np.zeros(
    B
)

lpns_aurc = np.zeros(
    B
)


print(
    "\nRunning final paired bootstrap..."
)


for b in range(
    B
):

    idx = (
        final_draw_bootstrap()
    )


    yb = (
        error_target[
            idx
        ]
    )


    rb = (
        risk_dis[
            idx
        ]
    )


    rl = (
        risk_lpns[
            idx
        ]
    )


    base_pr[
        b
    ] = average_precision_score(
        yb,
        rb
    )


    lpns_pr[
        b
    ] = average_precision_score(
        yb,
        rl
    )


    base_roc[
        b
    ] = roc_auc_score(
        yb,
        rb
    )


    lpns_roc[
        b
    ] = roc_auc_score(
        yb,
        rl
    )


    base_aurc[
        b
    ] = final_stat_aurc(
        yb,
        rb
    )


    lpns_aurc[
        b
    ] = final_stat_aurc(
        yb,
        rl
    )


    if (
        b + 1
    ) % 250 == 0:

        print(
            f"  bootstrap "
            f"{b+1:,}/{B:,}"
        )


# ============================================================
# 16C.8 DELTAS
# ============================================================

delta_pr = (
    lpns_pr
    -
    base_pr
)


delta_roc = (
    lpns_roc
    -
    base_roc
)


delta_aurc = (
    lpns_aurc
    -
    base_aurc
)


pr_ci = final_ci(
    delta_pr
)


roc_ci = final_ci(
    delta_roc
)


aurc_ci = final_ci(
    delta_aurc
)


FINAL_BOOTSTRAP_PRIMARY = pd.DataFrame([

    {
        "Metric":
            "Error PR-AUC",

        "Baseline":
            average_precision_score(
                error_target,
                risk_dis
            ),

        "LPNS":
            average_precision_score(
                error_target,
                risk_lpns
            ),

        "Delta":
            average_precision_score(
                error_target,
                risk_lpns
            )
            -
            average_precision_score(
                error_target,
                risk_dis
            ),

        "CI_Lower":
            pr_ci[0],

        "CI_Upper":
            pr_ci[1],

        "P":
            final_bootstrap_p(
                delta_pr
            ),

        "Higher_Is_Better":
            True,
    },

    {
        "Metric":
            "Error ROC-AUC",

        "Baseline":
            roc_auc_score(
                error_target,
                risk_dis
            ),

        "LPNS":
            roc_auc_score(
                error_target,
                risk_lpns
            ),

        "Delta":
            roc_auc_score(
                error_target,
                risk_lpns
            )
            -
            roc_auc_score(
                error_target,
                risk_dis
            ),

        "CI_Lower":
            roc_ci[0],

        "CI_Upper":
            roc_ci[1],

        "P":
            final_bootstrap_p(
                delta_roc
            ),

        "Higher_Is_Better":
            True,
    },

    {
        "Metric":
            "AURC",

        "Baseline":
            final_stat_aurc(
                error_target,
                risk_dis
            ),

        "LPNS":
            final_stat_aurc(
                error_target,
                risk_lpns
            ),

        "Delta":
            final_stat_aurc(
                error_target,
                risk_lpns
            )
            -
            final_stat_aurc(
                error_target,
                risk_dis
            ),

        "CI_Lower":
            aurc_ci[0],

        "CI_Upper":
            aurc_ci[1],

        "P":
            final_bootstrap_p(
                delta_aurc
            ),

        "Higher_Is_Better":
            False,
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL PAIRED BOOTSTRAP — LPNS vs CONFIDENCE+DISAGREEMENT"
)

print(
    "=" * 135
)


display(
    FINAL_BOOTSTRAP_PRIMARY
)


# ============================================================
# 16C.9 SELECTIVE REVIEW OBSERVED
# ============================================================

def final_review_metrics(
    risk,
    review_rate
):

    n_review = max(
        1,
        int(
            round(
                N
                *
                review_rate
            )
        )
    )


    review_idx = np.argsort(
        -risk
    )[:n_review]


    errors_found = (
        error_target[
            review_idx
        ].sum()
    )


    total_errors = max(
        error_target.sum(),
        1
    )


    return {

        "Errors_Captured":
            errors_found
            /
            total_errors,

        "Error_Per_100":
            (
                errors_found
                /
                n_review
                *
                100.0
            ),
    }


selective_rows = []


for rate in (
    FINAL_STAT_CONFIG[
        "review_rates"
    ]
):

    base = final_review_metrics(
        risk_dis,
        rate
    )


    lpns = final_review_metrics(
        risk_lpns,
        rate
    )


    selective_rows.append({

        "Review_Rate":
            rate,

        "Baseline_Error_Capture":
            base[
                "Errors_Captured"
            ],

        "LPNS_Error_Capture":
            lpns[
                "Errors_Captured"
            ],

        "Delta_Error_Capture":
            lpns[
                "Errors_Captured"
            ]
            -
            base[
                "Errors_Captured"
            ],

        "Baseline_Error_Per_100":
            base[
                "Error_Per_100"
            ],

        "LPNS_Error_Per_100":
            lpns[
                "Error_Per_100"
            ],

        "Delta_Error_Per_100":
            lpns[
                "Error_Per_100"
            ]
            -
            base[
                "Error_Per_100"
            ],
    })


FINAL_SELECTIVE_OBSERVED = pd.DataFrame(
    selective_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL OBSERVED SELECTIVE REVIEW BENEFIT"
)

print(
    "=" * 135
)


display(
    FINAL_SELECTIVE_OBSERVED
)


# ============================================================
# 16C.10 SELECTIVE REVIEW BOOTSTRAP
# ============================================================

selective_boot = {

    rate: {
        "capture_delta": [],
        "yield_delta": [],
    }

    for rate in (
        FINAL_STAT_CONFIG[
            "review_rates"
        ]
    )
}


rng_selective = np.random.default_rng(
    FINAL_STAT_CONFIG[
        "seed"
    ]
    +
    1000
)


print(
    "\nRunning final selective-review bootstrap..."
)


for b in range(
    B
):

    sampled_error = (
        rng_selective.choice(
            error_idx,
            size=len(error_idx),
            replace=True
        )
    )


    sampled_correct = (
        rng_selective.choice(
            correct_idx,
            size=len(correct_idx),
            replace=True
        )
    )


    idx = np.concatenate([
        sampled_error,
        sampled_correct,
    ])


    err_b = (
        error_target[
            idx
        ]
    )


    base_b = (
        risk_dis[
            idx
        ]
    )


    lpns_b = (
        risk_lpns[
            idx
        ]
    )


    total_error_b = max(
        err_b.sum(),
        1
    )


    n_b = len(
        idx
    )


    for rate in (
        FINAL_STAT_CONFIG[
            "review_rates"
        ]
    ):

        n_review = max(
            1,
            int(
                round(
                    n_b
                    *
                    rate
                )
            )
        )


        base_idx = np.argsort(
            -base_b
        )[:n_review]


        lpns_idx = np.argsort(
            -lpns_b
        )[:n_review]


        base_errors = (
            err_b[
                base_idx
            ].sum()
        )


        lpns_errors = (
            err_b[
                lpns_idx
            ].sum()
        )


        base_capture = (
            base_errors
            /
            total_error_b
        )


        lpns_capture = (
            lpns_errors
            /
            total_error_b
        )


        base_yield = (
            base_errors
            /
            n_review
            *
            100.0
        )


        lpns_yield = (
            lpns_errors
            /
            n_review
            *
            100.0
        )


        selective_boot[
            rate
        ][
            "capture_delta"
        ].append(
            lpns_capture
            -
            base_capture
        )


        selective_boot[
            rate
        ][
            "yield_delta"
        ].append(
            lpns_yield
            -
            base_yield
        )


    if (
        b + 1
    ) % 250 == 0:

        print(
            f"  selective "
            f"{b+1:,}/{B:,}"
        )


# ============================================================
# 16C.11 SELECTIVE BOOTSTRAP SUMMARY
# ============================================================

selective_stat_rows = []


for rate in (
    FINAL_STAT_CONFIG[
        "review_rates"
    ]
):

    capture_delta = np.asarray(
        selective_boot[
            rate
        ][
            "capture_delta"
        ]
    )


    yield_delta = np.asarray(
        selective_boot[
            rate
        ][
            "yield_delta"
        ]
    )


    capture_ci = final_ci(
        capture_delta
    )


    yield_ci = final_ci(
        yield_delta
    )


    observed = (
        FINAL_SELECTIVE_OBSERVED[
            FINAL_SELECTIVE_OBSERVED[
                "Review_Rate"
            ]
            ==
            rate
        ]
        .iloc[0]
    )


    selective_stat_rows.append({

        "Review_Rate":
            rate,

        "Delta_Error_Capture":
            observed[
                "Delta_Error_Capture"
            ],

        "Capture_CI_Lower":
            capture_ci[0],

        "Capture_CI_Upper":
            capture_ci[1],

        "Capture_P":
            final_bootstrap_p(
                capture_delta
            ),

        "Delta_Error_Per_100":
            observed[
                "Delta_Error_Per_100"
            ],

        "Yield_CI_Lower":
            yield_ci[0],

        "Yield_CI_Upper":
            yield_ci[1],

        "Yield_P":
            final_bootstrap_p(
                yield_delta
            ),
    })


FINAL_SELECTIVE_BOOTSTRAP = pd.DataFrame(
    selective_stat_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL PAIRED BOOTSTRAP — SELECTIVE REVIEW"
)

print(
    "=" * 135
)


display(
    FINAL_SELECTIVE_BOOTSTRAP
)


# ============================================================
# 16C.12 FOLD CONSISTENCY
# ============================================================

FINAL_FOLD_STAT_SUMMARY = pd.DataFrame([

    {
        "Metric":
            "PR delta",

        "Mean":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_PR_vs_Disagreement"
            ].mean(),

        "SD":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_PR_vs_Disagreement"
            ].std(
                ddof=1
            ),

        "Min":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_PR_vs_Disagreement"
            ].min(),

        "Max":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_PR_vs_Disagreement"
            ].max(),

        "Positive_Folds":
            int(
                (
                    FINAL_LPNS_FOLD_RESULTS[
                        "LPNS_Delta_PR_vs_Disagreement"
                    ]
                    >
                    0
                ).sum()
            ),

        "Total_Folds":
            len(
                FINAL_LPNS_FOLD_RESULTS
            ),
    },

    {
        "Metric":
            "ROC delta",

        "Mean":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].mean(),

        "SD":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].std(
                ddof=1
            ),

        "Min":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].min(),

        "Max":
            FINAL_LPNS_FOLD_RESULTS[
                "LPNS_Delta_ROC_vs_Disagreement"
            ].max(),

        "Positive_Folds":
            int(
                (
                    FINAL_LPNS_FOLD_RESULTS[
                        "LPNS_Delta_ROC_vs_Disagreement"
                    ]
                    >
                    0
                ).sum()
            ),

        "Total_Folds":
            len(
                FINAL_LPNS_FOLD_RESULTS
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FINAL_FOLD_STAT_SUMMARY
)


# ============================================================
# 16C.13 FINAL LOCK DECISION
# ============================================================

FINAL_LOCK_DECISION = pd.DataFrame([

    {
        "Criterion":
            "LPNS pooled PR gain > 0",

        "Passed":
            (
                FINAL_BOOTSTRAP_PRIMARY
                .iloc[0][
                    "Delta"
                ]
                >
                0
            ),

        "Value":
            FINAL_BOOTSTRAP_PRIMARY
            .iloc[0][
                "Delta"
            ],
    },

    {
        "Criterion":
            "95% CI for PR delta excludes 0",

        "Passed":
            (
                pr_ci[0]
                >
                0
            ),

        "Value":
            f"[{pr_ci[0]:.6f}, {pr_ci[1]:.6f}]",
    },

    {
        "Criterion":
            "PR improved in every outer fold",

        "Passed":
            (
                FINAL_LPNS_FOLD_RESULTS[
                    "LPNS_Delta_PR_vs_Disagreement"
                ]
                >
                0
            ).all(),

        "Value":
            (
                f"{int((FINAL_LPNS_FOLD_RESULTS['LPNS_Delta_PR_vs_Disagreement'] > 0).sum())}"
                f"/{len(FINAL_LPNS_FOLD_RESULTS)}"
            ),
    },

    {
        "Criterion":
            "Routing materially improves LPNS",

        "Passed":
            False,

        "Value":
            (
                FINAL_OBSERVED_STATS.loc[
                    FINAL_OBSERVED_STATS[
                        "Method"
                    ]
                    ==
                    "Final_Strict_LPNS+Routing",
                    "Error_PR_AUC"
                ].iloc[0]

                -

                FINAL_OBSERVED_STATS.loc[
                    FINAL_OBSERVED_STATS[
                        "Method"
                    ]
                    ==
                    "Final_Strict_LPNS",
                    "Error_PR_AUC"
                ].iloc[0]
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL DEVELOPMENT LOCK DECISION"
)

print(
    "=" * 135
)


display(
    FINAL_LOCK_DECISION
)


# ============================================================
# 16C.14 SAVE
# ============================================================

save_csv(
    FINAL_OBSERVED_STATS,
    TAB_DIR
    / "module16c_final_observed_reliability.csv"
)


save_csv(
    FINAL_BOOTSTRAP_PRIMARY,
    TAB_DIR
    / "module16c_final_bootstrap_primary.csv"
)


save_csv(
    FINAL_SELECTIVE_OBSERVED,
    TAB_DIR
    / "module16c_final_selective_observed.csv"
)


save_csv(
    FINAL_SELECTIVE_BOOTSTRAP,
    TAB_DIR
    / "module16c_final_selective_bootstrap.csv"
)


save_csv(
    FINAL_FOLD_STAT_SUMMARY,
    TAB_DIR
    / "module16c_final_fold_consistency.csv"
)


save_csv(
    FINAL_LOCK_DECISION,
    TAB_DIR
    / "module16c_final_lock_decision.csv"
)


bootstrap_df = pd.DataFrame({

    "Bootstrap":
        np.arange(
            B
        ),

    "Baseline_PR":
        base_pr,

    "LPNS_PR":
        lpns_pr,

    "Delta_PR":
        delta_pr,

    "Baseline_ROC":
        base_roc,

    "LPNS_ROC":
        lpns_roc,

    "Delta_ROC":
        delta_roc,

    "Baseline_AURC":
        base_aurc,

    "LPNS_AURC":
        lpns_aurc,

    "Delta_AURC":
        delta_aurc,
})


save_csv(
    bootstrap_df,
    PRED_DIR
    / "module16c_final_bootstrap_distribution.csv"
)


module16c_manifest = {

    "module":
        "Final paired bootstrap validation",

    "primary_comparison":
        "Final Strict LPNS vs Confidence+Disagreement",

    "primary_endpoint":
        "Error PR-AUC",

    "bootstrap":
        "paired stratified",

    "bootstrap_iterations":
        B,

    "confidence_interval":
        "95% percentile",

    "review_rates":
        FINAL_STAT_CONFIG[
            "review_rates"
        ],

    "architecture_locked":
        True,

    "holdout_used":
        False,
}


save_json(
    module16c_manifest,
    LOG_DIR
    / "module16c_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 16C COMPLETE"
)

print(
    "=" * 135
)

print(
    "\nDEV development is now statistically validated."
)

print(
    "Do not modify the architecture after this point."
)

N: 202944
Error prevalence: 0.13753055029959002
False negatives: 9051

MODULE 16C — FINAL OBSERVED RELIABILITY PERFORMANCE


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.545862,0.900056,0.026931
1,Confidence+Disagreement,0.547198,0.900208,0.026892
2,Final_Strict_LPNS,0.559891,0.900525,0.026829
3,Final_Strict_LPNS+Routing,0.560273,0.900499,0.026830



Running final paired bootstrap...
  bootstrap 250/2,000
  bootstrap 500/2,000
  bootstrap 750/2,000
  bootstrap 1,000/2,000
  bootstrap 1,250/2,000
  bootstrap 1,500/2,000
  bootstrap 1,750/2,000
  bootstrap 2,000/2,000

FINAL PAIRED BOOTSTRAP — LPNS vs CONFIDENCE+DISAGREEMENT


,Metric,Baseline,LPNS,Delta,CI_Lower,CI_Upper,P,Higher_Is_Better
0,Error PR-AUC,0.547198,0.559891,0.012693,0.009757,0.015606,0.000,True
1,Error ROC-AUC,0.900208,0.900525,0.000316,0.000056,0.000586,0.017,True
2,AURC,0.026892,0.026829,-0.000063,-0.000179,0.000058,0.312,False



FINAL OBSERVED SELECTIVE REVIEW BENEFIT


...


FINAL FOLD CONSISTENCY


,Metric,Mean,SD,Min,Max,Positive_Folds,Total_Folds
0,PR delta,0.012936,0.001971,0.010707,0.015704,5,5
1,ROC delta,0.000237,0.000491,-0.000381,0.000723,3,5



FINAL DEVELOPMENT LOCK DECISION


,Criterion,Passed,Value
0,LPNS pooled PR gain > 0,True,0.012693
1,95% CI for PR delta excludes 0,True,"[0.009757, 0.015606]"
2,PR improved in every outer fold,True,5/5
3,Routing materially improves LPNS,False,0.000382



MODULE 16C COMPLETE

DEV development is now statistically validated.
Do not modify the architecture after this point.


In [42]:
# ============================================================
# MODULE 17 PRE-FLIGHT — RECOVER LOCKED HGB/CATBOOST SETTINGS
# ============================================================

import inspect
import pandas as pd

records = []

keywords = [
    "hgb",
    "hist",
    "gradient",
    "catboost",
    "cat_",
    "catmodel",
    "classical",
    "baseline",
    "tree",
    "train",
]


for name, obj in list(globals().items()):

    lname = name.lower()

    if not any(
        k in lname
        for k in keywords
    ):
        continue

    obj_type = type(obj).__name__

    signature = None

    try:
        if callable(obj):
            signature = str(
                inspect.signature(obj)
            )
    except Exception:
        pass

    records.append({
        "name": name,
        "type": obj_type,
        "signature": signature,
    })


inspection = pd.DataFrame(
    records
).sort_values(
    "name"
).reset_index(
    drop=True
)


print(
    "=" * 130
)
print(
    "POTENTIAL HGB / CATBOOST OBJECTS"
)
print(
    "=" * 130
)

display(
    inspection
)


# ============================================================
# DICTIONARY CONFIGS
# ============================================================

print(
    "\n"
    + "=" * 130
)
print(
    "POTENTIAL CONFIG DICTIONARIES"
)
print(
    "=" * 130
)


for name, obj in list(globals().items()):

    if not isinstance(
        obj,
        dict
    ):
        continue

    lname = name.lower()

    if any(
        k in lname
        for k in [
            "hgb",
            "hist",
            "cat",
            "classical",
            "baseline",
            "model",
            "config",
        ]
    ):

        print(
            f"\n{name}"
        )

        try:
            print(obj)
        except Exception as e:
            print(
                "Could not print:",
                repr(e)
            )


# ============================================================
# EXISTING SKLEARN/CATBOOST MODEL OBJECTS
# ============================================================

print(
    "\n"
    + "=" * 130
)
print(
    "EXISTING FITTED / UNFITTED MODEL OBJECTS"
)
print(
    "=" * 130
)


for name, obj in list(globals().items()):

    cls_name = (
        type(obj).__name__.lower()
    )

    if (
        "histgradientboosting" in cls_name
        or
        "catboost" in cls_name
    ):

        print(
            f"\n{name}"
        )

        print(
            "class:",
            type(obj).__name__
        )

        try:
            print(
                "params:",
                obj.get_params()
            )
        except Exception as e:
            print(
                "get_params unavailable:",
                repr(e)
            )


# ============================================================
# LIKELY TRAINING FUNCTIONS
# ============================================================

print(
    "\n"
    + "=" * 130
)
print(
    "SOURCE FOR LIKELY HGB/CATBOOST TRAINING FUNCTIONS"
)
print(
    "=" * 130
)


candidate_functions = []

for name, obj in list(globals().items()):

    if not callable(obj):
        continue

    lname = name.lower()

    if (
        (
            "hgb" in lname
            or
            "hist" in lname
            or
            "catboost" in lname
            or
            "classical" in lname
        )
        and
        (
            "train" in lname
            or
            "fit" in lname
            or
            "fold" in lname
        )
    ):

        candidate_functions.append(
            name
        )


for name in sorted(
    set(
        candidate_functions
    )
):

    print(
        "\n"
        + "-" * 130
    )

    print(
        "SOURCE:",
        name
    )

    print(
        "-" * 130
    )

    try:

        print(
            inspect.getsource(
                globals()[name]
            )[:20000]
        )

    except Exception as e:

        print(
            "Source unavailable:",
            repr(e)
        )


# ============================================================
# SEARCH RAW OOF ARRAYS
# ============================================================

print(
    "\n"
    + "=" * 130
)
print(
    "POTENTIAL HGB / CATBOOST OOF ARRAYS"
)
print(
    "=" * 130
)


for name, obj in list(globals().items()):

    lname = name.lower()

    if not (
        "hgb" in lname
        or
        "hist" in lname
        or
        "cat" in lname
    ):
        continue

    try:

        arr = np.asarray(
            obj
        )

        if (
            arr.ndim == 1
            and
            len(arr) == len(
                y_dev
            )
        ):

            print(
                name,
                arr.shape,
                "mean=",
                float(
                    np.mean(
                        arr.astype(float)
                    )
                )
            )

    except Exception:
        pass

POTENTIAL HGB / CATBOOST OBJECTS


,name,type,signature
0,CATBOOST_CAL_OOF,ndarray,None
1,CAT_CARDINALITIES,list,None
2,CAT_COLS,list,None
3,CAT_FINAL_CONFIG,NoneType,None
4,CAT_ONLY,ndarray,None
5,CAT_VALUE_MAPS,dict,None
6,CatBoostClassifier,type,"(iterations=None, learning_rate=None, depth=No..."
7,ExtraTreesClassifier,ABCMeta,"(n_estimators=100, *, criterion='gini', max_de..."
8,F_conf_train,ndarray,None
9,F_expert_train,ndarray,None



POTENTIAL CONFIG DICTIONARIES

CONFIG
{'project_name': 'TA_MoE_CVD_Rebuild', 'seed': 42, 'target': 'HeartDiseaseorAttack', 'test_size': 0.2, 'n_splits': 5, 'primary_metric': 'PR_AUC', 'secondary_metrics': ['ROC_AUC', 'Brier', 'NLL', 'ECE', 'Precision', 'Recall', 'F1', 'MCC', 'Balanced_Accuracy'], 'task': 'cardiovascular_disease_status_classification', 'holdout_policy': 'final_evaluation_only'}

evaluation_config
{'ece_bins': 15, 'threshold_grid': {'start': 0.005, 'stop': 0.995, 'step': 0.005}, 'primary_metric': 'PR_AUC', 'threshold_objectives': ['F1', 'MCC', 'Balanced_Accuracy'], 'threshold_selection_source': 'DEV_OOF_ONLY', 'holdout_threshold_tuning': False}

MODEL_FACTORIES
{'LogReg_L2': {'family': 'linear', 'preprocessor': 'linear', 'factory': <function get_module6_model_factories.<locals>.<lambda> at 0x7a5c0f5d3100>}, 'LogReg_ElasticNet': {'family': 'linear', 'preprocessor': 'linear', 'factory': <function get_module6_model_factories.<locals>.<lambda> at 0x7a5c0f5dc360>}, 'RandomFo

In [44]:
# ============================================================
# MODULE 17A — FINAL CLASSICAL EXPERT RECONSTRUCTION
#
# PURPOSE
# ------------------------------------------------------------
# Recover the EXACT original Module-6 HGB and CatBoost
# definitions using MODEL_FACTORIES.
#
# Then regenerate:
#   - raw CatBoost DEV OOF
#   - final HGB fit on full DEV
#   - final CatBoost fit on full DEV
#
# IMPORTANT:
#   HOLDOUT metrics are NOT evaluated here.
# ============================================================


import copy
import gc
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# 17A.1 REQUIREMENTS
# ============================================================

required_17a = [
    "MODEL_FACTORIES",
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",
    "SEED",
    "preprocessing_manifest",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_17a
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


# ============================================================
# 17A.2 VERIFY ORIGINAL FACTORIES
# ============================================================

for model_name in [
    "HistGradientBoosting",
    "CatBoost",
]:

    if model_name not in MODEL_FACTORIES:

        raise RuntimeError(
            f"{model_name} missing from MODEL_FACTORIES."
        )

    if "factory" not in MODEL_FACTORIES[model_name]:

        raise RuntimeError(
            f"{model_name} factory missing."
        )


print(
    "Using ORIGINAL Module-6 factories:"
)

print(
    "  HGB factory:",
    MODEL_FACTORIES[
        "HistGradientBoosting"
    ][
        "factory"
    ]
)

print(
    "  CatBoost factory:",
    MODEL_FACTORIES[
        "CatBoost"
    ][
        "factory"
    ]
)


# ============================================================
# 17A.3 INSPECT EXACT FACTORY MODELS
# ============================================================

probe_hgb = (
    MODEL_FACTORIES[
        "HistGradientBoosting"
    ][
        "factory"
    ]()
)


probe_cat = (
    MODEL_FACTORIES[
        "CatBoost"
    ][
        "factory"
    ]()
)


print(
    "\nExact HGB parameters:"
)

print(
    probe_hgb.get_params()
)


print(
    "\nExact CatBoost parameters:"
)

try:
    print(
        probe_cat.get_params()
    )
except Exception:
    print(
        probe_cat
    )


HGB_LOCKED_PARAMS = copy.deepcopy(
    probe_hgb.get_params()
)


try:

    CATBOOST_LOCKED_PARAMS = copy.deepcopy(
        probe_cat.get_params()
    )

except Exception:

    CATBOOST_LOCKED_PARAMS = {}


del probe_hgb, probe_cat

gc.collect()


# ============================================================
# 17A.4 TREE FEATURES
# ============================================================

FINAL_TREE_COLS = list(
    preprocessing_manifest[
        "tree_cols"
    ]
)


print(
    "\nTree feature count:",
    len(
        FINAL_TREE_COLS
    )
)


print(
    "Tree features:",
    FINAL_TREE_COLS
)


# ============================================================
# 17A.5 DATA
# ============================================================

y_dev_np = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_tree_dev = (
    X_dev[
        FINAL_TREE_COLS
    ]
)


N_DEV = len(
    y_dev_np
)


# ============================================================
# 17A.6 REGENERATE RAW HGB + CATBOOST DEV OOF
#
# Uses exact locked outer folds and exact Module-6 factories.
# No HOLDOUT involved.
# ============================================================

FINAL_HGB_RAW_OOF = np.zeros(
    N_DEV,
    dtype=float
)


FINAL_CAT_RAW_OOF = np.zeros(
    N_DEV,
    dtype=float
)


classical_fold_rows = []


for fold_id in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 125
    )

    print(
        f"RECONSTRUCTING CLASSICAL OOF — "
        f"FOLD {fold_id+1}/{N_SPLITS}"
    )

    print(
        "=" * 125
    )


    train_idx = np.where(
        fold_assignment
        !=
        fold_id
    )[0]


    valid_idx = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    Xtr = (
        X_tree_dev
        .iloc[
            train_idx
        ]
    )


    Xva = (
        X_tree_dev
        .iloc[
            valid_idx
        ]
    )


    ytr = (
        y_dev_np[
            train_idx
        ]
    )


    yva = (
        y_dev_np[
            valid_idx
        ]
    )


    # --------------------------------------------------------
    # HGB
    # --------------------------------------------------------

    hgb_model = (
        MODEL_FACTORIES[
            "HistGradientBoosting"
        ][
            "factory"
        ]()
    )


    hgb_model.fit(
        Xtr,
        ytr
    )


    p_hgb = (
        hgb_model
        .predict_proba(
            Xva
        )[:, 1]
    )


    FINAL_HGB_RAW_OOF[
        valid_idx
    ] = p_hgb


    # --------------------------------------------------------
    # CatBoost
    # --------------------------------------------------------

    cat_model = (
        MODEL_FACTORIES[
            "CatBoost"
        ][
            "factory"
        ]()
    )


    cat_model.fit(
        Xtr,
        ytr
    )


    p_cat = (
        cat_model
        .predict_proba(
            Xva
        )[:, 1]
    )


    FINAL_CAT_RAW_OOF[
        valid_idx
    ] = p_cat


    hgb_pr = (
        average_precision_score(
            yva,
            p_hgb
        )
    )


    cat_pr = (
        average_precision_score(
            yva,
            p_cat
        )
    )


    classical_fold_rows.append({

        "Fold":
            fold_id,

        "HGB_PR":
            hgb_pr,

        "HGB_ROC":
            roc_auc_score(
                yva,
                p_hgb
            ),

        "CatBoost_PR":
            cat_pr,

        "CatBoost_ROC":
            roc_auc_score(
                yva,
                p_cat
            ),
    })


    print(
        f"HGB      PR={hgb_pr:.6f}"
    )

    print(
        f"CatBoost PR={cat_pr:.6f}"
    )


    del (
        hgb_model,
        cat_model
    )

    gc.collect()


# ============================================================
# 17A.7 GLOBAL RECONSTRUCTION CHECK
# ============================================================

FINAL_CLASSICAL_RECONSTRUCTION = pd.DataFrame([

    {
        "Model":
            "HistGradientBoosting",

        "PR_AUC":
            average_precision_score(
                y_dev_np,
                FINAL_HGB_RAW_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y_dev_np,
                FINAL_HGB_RAW_OOF
            ),

        "Brier":
            brier_score_loss(
                y_dev_np,
                FINAL_HGB_RAW_OOF
            ),

        "NLL":
            log_loss(
                y_dev_np,
                np.clip(
                    FINAL_HGB_RAW_OOF,
                    1e-8,
                    1 - 1e-8
                )
            ),
    },

    {
        "Model":
            "CatBoost",

        "PR_AUC":
            average_precision_score(
                y_dev_np,
                FINAL_CAT_RAW_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y_dev_np,
                FINAL_CAT_RAW_OOF
            ),

        "Brier":
            brier_score_loss(
                y_dev_np,
                FINAL_CAT_RAW_OOF
            ),

        "NLL":
            log_loss(
                y_dev_np,
                np.clip(
                    FINAL_CAT_RAW_OOF,
                    1e-8,
                    1 - 1e-8
                )
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 17A — RECONSTRUCTED CLASSICAL DEV OOF"
)

print(
    "=" * 135
)


display(
    FINAL_CLASSICAL_RECONSTRUCTION
)


# ============================================================
# 17A.8 COMPARE AGAINST ORIGINAL KNOWN RESULTS
# ============================================================

reconstruction_check = []


if "HGB_CAL_OOF" in globals():

    reconstruction_check.append({

        "Model":
            "HGB",

        "Existing_PR":
            average_precision_score(
                y_dev_np,
                HGB_CAL_OOF
            ),

        "Reconstructed_Raw_PR":
            average_precision_score(
                y_dev_np,
                FINAL_HGB_RAW_OOF
            ),

        "Correlation":
            np.corrcoef(
                HGB_CAL_OOF,
                FINAL_HGB_RAW_OOF
            )[0, 1],
    })


if "CATBOOST_CAL_OOF" in globals():

    reconstruction_check.append({

        "Model":
            "CatBoost",

        "Existing_Calibrated_PR":
            average_precision_score(
                y_dev_np,
                CATBOOST_CAL_OOF
            ),

        "Reconstructed_Raw_PR":
            average_precision_score(
                y_dev_np,
                FINAL_CAT_RAW_OOF
            ),

        "Correlation":
            np.corrcoef(
                CATBOOST_CAL_OOF,
                FINAL_CAT_RAW_OOF
            )[0, 1],
    })


FINAL_CLASSICAL_CHECK = pd.DataFrame(
    reconstruction_check
)


print(
    "\n"
    + "=" * 135
)

print(
    "CLASSICAL RECONSTRUCTION CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FINAL_CLASSICAL_CHECK
)


# ============================================================
# 17A.9 SAVE
# ============================================================

classical_oof_df = pd.DataFrame({

    "target":
        y_dev_np,

    "fold":
        fold_assignment,

    "HGB_Raw":
        FINAL_HGB_RAW_OOF,

    "CatBoost_Raw":
        FINAL_CAT_RAW_OOF,
})


save_csv(

    classical_oof_df,

    PRED_DIR
    / "module17a_reconstructed_classical_oof.csv"
)


save_csv(

    FINAL_CLASSICAL_RECONSTRUCTION,

    TAB_DIR
    / "module17a_classical_reconstruction.csv"
)


save_csv(

    pd.DataFrame(
        classical_fold_rows
    ),

    TAB_DIR
    / "module17a_classical_folds.csv"
)


save_csv(

    FINAL_CLASSICAL_CHECK,

    TAB_DIR
    / "module17a_classical_consistency.csv"
)


module17a_manifest = {

    "module":
        "Final classical expert reconstruction",

    "HGB_source":
        "MODEL_FACTORIES['HistGradientBoosting']['factory']",

    "CatBoost_source":
        "MODEL_FACTORIES['CatBoost']['factory']",

    "folds":
        "original locked fold_assignment",

    "holdout_used":
        False,

    "hyperparameters_changed":
        False,

    "purpose":
        (
            "recover exact raw DEV OOF needed for "
            "final calibration and HOLDOUT fitting"
        ),
}


save_json(

    module17a_manifest,

    LOG_DIR
    / "module17a_manifest.json"
)


print(
    "\nMODULE 17A COMPLETE."
)

print(
    "HOLDOUT still has NOT been evaluated."
)

Using ORIGINAL Module-6 factories:
  HGB factory: <function get_module6_model_factories.<locals>.<lambda> at 0x7a5c0f5dc720>
  CatBoost factory: <function get_module6_model_factories.<locals>.<lambda> at 0x7a5c0f5dc900>

Exact HGB parameters:
{'categorical_features': 'from_dtype', 'class_weight': None, 'early_stopping': 'auto', 'interaction_cst': None, 'l2_regularization': 1.0, 'learning_rate': 0.05, 'loss': 'log_loss', 'max_bins': 255, 'max_depth': 6, 'max_features': 1.0, 'max_iter': 400, 'max_leaf_nodes': 31, 'min_samples_leaf': 40, 'monotonic_cst': None, 'n_iter_no_change': 10, 'random_state': 42, 'scoring': 'loss', 'tol': 1e-07, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}

Exact CatBoost parameters:
{'iterations': 500, 'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 3.0, 'loss_function': 'Logloss', 'random_seed': 42, 'verbose': False, 'auto_class_weights': 'Balanced', 'eval_metric': 'Logloss', 'bagging_temperature': 0.5, 'task_type': 'GPU', 'devices': '0'}

Tre

,Model,PR_AUC,ROC_AUC,Brier,NLL
0,HistGradientBoosting,0.377012,0.849628,0.069876,0.236194
1,CatBoost,0.377537,0.849216,0.161784,0.475348



CLASSICAL RECONSTRUCTION CONSISTENCY


,Model,Existing_PR,Reconstructed_Raw_PR,Correlation,Existing_Calibrated_PR
0,HGB,0.377012,0.377012,1.00000,NaN
1,CatBoost,NaN,0.377537,0.90402,0.377444



MODULE 17A COMPLETE.
HOLDOUT still has NOT been evaluated.


In [45]:
# ============================================================
# MODULE 17B — SINGLE FINAL LOCKED HOLDOUT EVALUATION
#
# Uses:
#   exact reconstructed HGB / CatBoost factories
#   repaired AutoInt / DeepFM_v2
#   locked RA-MoE gate
#   locked LPNS reliability architecture
#
# THIS OPENS HOLDOUT.
#
# After this cell finishes:
#   NO architecture changes
#   NO threshold changes
#   NO calibration changes
#   NO expert changes
#   NO LPNS tuning
# ============================================================


import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    confusion_matrix,
)


# ============================================================
# 17B.1 REQUIREMENTS
# ============================================================

required = [
    "X_dev",
    "y_dev",

    "MODEL_FACTORIES",
    "FINAL_TREE_COLS",

    "FINAL_HGB_RAW_OOF",
    "FINAL_CAT_RAW_OOF",

    "AUTOINT_REPAIRED_OOF",
    "DEEPFM_V2_REPAIRED_OOF",

    "AUTOINT_REPAIRED",
    "DEEPFM_V2_REPAIRED",

    "FINAL_REPAIRED_EXPERT_OOF_STRICT",

    "AutoIntModel",
    "StableDeepFM",
    "DeepTabularDataset",
    "deep_predict_proba",

    "CAT_COLS",
    "CONT_COLS_DEEP",
    "CAT_CARDINALITIES",
    "DEEP_CARDINALITIES",

    "DEEP_CONFIGS",
    "DEEPFM_V2_CONFIG",

    "SCALE_POS_WEIGHT",
    "TORCH_DEVICE",
    "USE_AMP",

    "seed_everything",
    "seed_worker",

    "make_oracle_targets",
    "train_ramoe_gate",
    "predict_gate_weights",
    "mixture_probability",
    "select_f1_threshold_local",

    "RAMOE_FEATURES",
    "RAMOE_CONFIG",

    "FINAL_LPNS_CONFIG",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


import torch.nn as nn


# ============================================================
# 17B.2 RESOLVE LOCKED HOLDOUT
# ============================================================

def find_first(names):

    for name in names:

        if name in globals():
            return name, globals()[name]

    return None, None


x_name, X_holdout_final = find_first([
    "X_holdout",
    "X_hold",
    "X_ho",
    "X_HOLDOUT",
    "X_test_holdout",
    "X_holdout_raw",
])


y_name, y_holdout_final = find_first([
    "y_holdout",
    "y_hold",
    "y_ho",
    "y_HOLDOUT",
    "y_test_holdout",
])


if X_holdout_final is None:
    raise RuntimeError(
        "HOLDOUT features not found."
    )


if y_holdout_final is None:
    raise RuntimeError(
        "HOLDOUT labels not found."
    )


if not isinstance(
    X_holdout_final,
    pd.DataFrame
):
    X_holdout_final = pd.DataFrame(
        X_holdout_final,
        columns=X_dev.columns
    )


if isinstance(
    y_holdout_final,
    pd.Series
):
    y_hold = y_holdout_final.to_numpy(
        dtype=int
    )
else:
    y_hold = np.asarray(
        y_holdout_final,
        dtype=int
    )


y_dev_np = y_dev.to_numpy(
    dtype=int
)


N_DEV = len(
    y_dev_np
)


N_HOLD = len(
    y_hold
)


print(
    "HOLDOUT resolved:"
)

print(
    " X:",
    x_name,
    X_holdout_final.shape
)

print(
    " y:",
    y_name,
    y_hold.shape
)


if N_HOLD != 50736:
    raise RuntimeError(
        f"Expected locked HOLDOUT N=50736, "
        f"found {N_HOLD}."
    )


# ============================================================
# 17B.3 FINAL LOCKED CONFIG
# ============================================================

FINAL17_CONFIG = {

    "reference_size": 80000,

    "k_neighbors": 20,

    "max_distance": 3.0,

    "min_neighbors": 6,

    "discrete_weight": 0.65,

    "reliability_fraction": 0.30,

    "threshold_fraction": 0.15,

    "reliability_C": 0.10,

    "review_rates": [
        0.05,
        0.10,
        0.15,
        0.20
    ],
}


# ============================================================
# 17B.4 FINAL NEURAL EPOCHS
#
# Completely determined before HOLDOUT.
# ============================================================

autoint_epochs = (
    AUTOINT_REPAIRED[
        "fold_metrics"
    ][
        "Inner_Selected_Epoch"
    ]
    .to_numpy(
        dtype=int
    )
)


deepfm_epochs = (
    DEEPFM_V2_REPAIRED[
        "fold_metrics"
    ][
        "Inner_Selected_Epoch"
    ]
    .to_numpy(
        dtype=int
    )
)


FINAL_AUTOINT_EPOCH = int(
    np.median(
        autoint_epochs
    )
)


FINAL_DEEPFM_EPOCH = int(
    np.median(
        deepfm_epochs
    )
)


print(
    "\nLocked neural epochs:"
)

print(
    " AutoInt:",
    FINAL_AUTOINT_EPOCH,
    autoint_epochs.tolist()
)

print(
    " DeepFM:",
    FINAL_DEEPFM_EPOCH,
    deepfm_epochs.tolist()
)


# ============================================================
# 17B.5 PLATT HELPERS
# ============================================================

def final_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p / (1 - p)
    )


def fit_final_platt(
    p,
    y,
    seed
):

    model = LogisticRegression(
        penalty="l2",
        C=1e6,
        solver="lbfgs",
        max_iter=3000,
        random_state=seed,
    )


    model.fit(
        final_logit(p)[:, None],
        y
    )


    return model


def apply_final_platt(
    model,
    p
):

    return (
        model
        .predict_proba(
            final_logit(p)[:, None]
        )[:, 1]
    )


# ============================================================
# 17B.6 FIT FINAL HGB
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FINAL HGB — FULL DEV FIT"
)

print(
    "=" * 130
)


FINAL_HGB_MODEL = (
    MODEL_FACTORIES[
        "HistGradientBoosting"
    ][
        "factory"
    ]()
)


FINAL_HGB_MODEL.fit(

    X_dev[
        FINAL_TREE_COLS
    ],

    y_dev_np
)


HOLD_HGB = (
    FINAL_HGB_MODEL
    .predict_proba(

        X_holdout_final[
            FINAL_TREE_COLS
        ]

    )[:, 1]
)


# HGB remained uncalibrated in locked development.
HOLD_HGB_CAL = (
    HOLD_HGB.copy()
)


# ============================================================
# 17B.7 FIT FINAL CATBOOST
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FINAL CATBOOST — FULL DEV FIT"
)

print(
    "=" * 130
)


FINAL_CAT_MODEL = (
    MODEL_FACTORIES[
        "CatBoost"
    ][
        "factory"
    ]()
)


FINAL_CAT_MODEL.fit(

    X_dev[
        FINAL_TREE_COLS
    ],

    y_dev_np
)


HOLD_CAT_RAW = (
    FINAL_CAT_MODEL
    .predict_proba(

        X_holdout_final[
            FINAL_TREE_COLS
        ]

    )[:, 1]
)


# ============================================================
# 17B.8 FINAL CATBOOST CALIBRATION
#
# Calibrator fit using raw DEV OOF only.
# ============================================================

FINAL_CAT_CALIBRATOR = (
    fit_final_platt(

        FINAL_CAT_RAW_OOF,

        y_dev_np,

        SEED + 170100
    )
)


HOLD_CAT_CAL = (
    apply_final_platt(

        FINAL_CAT_CALIBRATOR,

        HOLD_CAT_RAW
    )
)


# ============================================================
# 17B.9 DEEP PREPROCESSOR
# ============================================================

class FinalDeepPreprocessor:

    def __init__(self):

        self.maps = {}

        self.scaler = (
            StandardScaler()
        )


    def fit(
        self,
        X
    ):

        for col in CAT_COLS:

            vals = (
                DEEP_CARDINALITIES[
                    col
                ][
                    "values"
                ]
            )


            self.maps[
                col
            ] = {
                float(v): i
                for i, v
                in enumerate(vals)
            }


        self.scaler.fit(

            X[
                CONT_COLS_DEEP
            ].astype(float)
        )


        return self


    def transform(
        self,
        X
    ):

        Xcat = np.zeros(

            (
                len(X),
                len(CAT_COLS)
            ),

            dtype=np.int64
        )


        for j, col in enumerate(
            CAT_COLS
        ):

            raw = (
                X[col]
                .to_numpy(
                    dtype=float
                )
            )


            mapping = (
                self.maps[
                    col
                ]
            )


            encoded = np.array(
                [
                    mapping.get(
                        float(v),
                        -1
                    )
                    for v in raw
                ],
                dtype=np.int64
            )


            if np.any(
                encoded < 0
            ):

                raise RuntimeError(
                    f"Unknown category in HOLDOUT "
                    f"for {col}: "
                    f"{np.unique(raw[encoded < 0])}"
                )


            Xcat[:, j] = encoded


        Xcont = (
            self.scaler
            .transform(

                X[
                    CONT_COLS_DEEP
                ].astype(float)
            )
            .astype(
                np.float32
            )
        )


        return (
            Xcat,
            Xcont
        )


# ============================================================
# 17B.10 NEURAL FACTORY
# ============================================================

def make_final_neural(
    model_name
):

    if model_name == "AutoInt":

        cfg = (
            DEEP_CONFIGS[
                "AutoInt"
            ]
        )


        model = AutoIntModel(

            cardinalities=
                CAT_CARDINALITIES,

            n_cont=
                len(
                    CONT_COLS_DEEP
                ),

            embedding_dim=
                cfg[
                    "embedding_dim"
                ],

            num_heads=
                cfg[
                    "num_heads"
                ],

            num_layers=
                cfg[
                    "num_layers"
                ],

            hidden_dims=
                cfg[
                    "hidden_dims"
                ],

            dropout=
                cfg[
                    "dropout"
                ],
        )


    elif model_name == "DeepFM_v2":

        cfg = (
            DEEPFM_V2_CONFIG
        )


        model = StableDeepFM(

            cardinalities=
                CAT_CARDINALITIES,

            n_cont=
                len(
                    CONT_COLS_DEEP
                ),

            embedding_dim=
                cfg[
                    "embedding_dim"
                ],

            hidden_dims=
                cfg[
                    "hidden_dims"
                ],

            dropout=
                cfg[
                    "dropout"
                ],

            init_std=
                cfg[
                    "init_std"
                ],
        )


    else:

        raise ValueError(
            model_name
        )


    return model.to(
        TORCH_DEVICE
    )


# ============================================================
# 17B.11 FINAL NEURAL TRAINING
# ============================================================

def train_final_neural(
    model_name,
    epochs,
    seed
):

    seed_everything(
        seed
    )


    prep = (
        FinalDeepPreprocessor()
    )


    prep.fit(
        X_dev
    )


    (
        Xdev_cat,
        Xdev_cont

    ) = prep.transform(
        X_dev
    )


    (
        Xhold_cat,
        Xhold_cont

    ) = prep.transform(
        X_holdout_final
    )


    train_ds = DeepTabularDataset(

        Xdev_cat,

        Xdev_cont,

        y_dev_np.astype(
            np.float32
        )
    )


    g = torch.Generator()

    g.manual_seed(
        seed
    )


    cfg = (

        DEEP_CONFIGS[
            "AutoInt"
        ]

        if model_name
        ==
        "AutoInt"

        else

        DEEPFM_V2_CONFIG
    )


    train_loader = DataLoader(

        train_ds,

        batch_size=
            cfg[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,

        worker_init_fn=
            seed_worker,

        generator=
            g,
    )


    model = (
        make_final_neural(
            model_name
        )
    )


    pos_weight = torch.tensor(

        [
            SCALE_POS_WEIGHT
        ],

        dtype=torch.float32,

        device=TORCH_DEVICE
    )


    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )


    if model_name == "AutoInt":

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=
                cfg[
                    "learning_rate"
                ],

            weight_decay=
                cfg[
                    "weight_decay"
                ],
        )


    else:

        optimizer = torch.optim.Adam(

            model.parameters(),

            lr=
                cfg[
                    "learning_rate"
                ],

            weight_decay=
                cfg[
                    "weight_decay"
                ],
        )


    amp_scaler = torch.amp.GradScaler(

        "cuda",

        enabled=
            USE_AMP
    )


    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        total_loss = 0.0

        total_n = 0


        for (
            xcat,
            xcont,
            yb
        ) in train_loader:

            xcat = xcat.to(
                TORCH_DEVICE,
                non_blocking=True
            )


            xcont = xcont.to(
                TORCH_DEVICE,
                non_blocking=True
            )


            yb = yb.to(
                TORCH_DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(

                device_type="cuda",

                dtype=torch.float16,

                enabled=USE_AMP

            ):

                logits = model(
                    xcat,
                    xcont
                )


                loss = loss_fn(
                    logits,
                    yb
                )


            amp_scaler.scale(
                loss
            ).backward()


            if model_name == "DeepFM_v2":

                amp_scaler.unscale_(
                    optimizer
                )


                torch.nn.utils.clip_grad_norm_(

                    model.parameters(),

                    max_norm=5.0
                )


            amp_scaler.step(
                optimizer
            )


            amp_scaler.update()


            n = len(
                yb
            )


            total_loss += (

                float(
                    loss
                    .detach()
                    .cpu()
                )

                *
                n
            )


            total_n += n


        print(

            f"{model_name} "
            f"epoch {epoch:02d}/{epochs:02d} "
            f"loss="
            f"{total_loss/total_n:.6f}"
        )


    p_hold = deep_predict_proba(

        model,

        Xhold_cat,

        Xhold_cont,

        batch_size=4096
    )


    del (
        model,
        optimizer,
        amp_scaler,
        train_loader,
    )


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return p_hold


# ============================================================
# 17B.12 FINAL AUTOINT
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FINAL AUTOINT — FULL DEV FIT"
)

print(
    "=" * 130
)


HOLD_AUTO_RAW = (
    train_final_neural(

        "AutoInt",

        FINAL_AUTOINT_EPOCH,

        SEED + 171000
    )
)


# ============================================================
# 17B.13 FINAL DEEPFM
# ============================================================

print(
    "\n"
    + "=" * 130
)

print(
    "FINAL DEEPFM_v2 — FULL DEV FIT"
)

print(
    "=" * 130
)


HOLD_DEEPFM_RAW = (
    train_final_neural(

        "DeepFM_v2",

        FINAL_DEEPFM_EPOCH,

        SEED + 172000
    )
)


# ============================================================
# 17B.14 FINAL NEURAL CALIBRATION
# ============================================================

FINAL_AUTO_CAL = (
    fit_final_platt(

        np.asarray(
            AUTOINT_REPAIRED_OOF,
            dtype=float
        ),

        y_dev_np,

        SEED + 173000
    )
)


FINAL_DEEPFM_CAL = (
    fit_final_platt(

        np.asarray(
            DEEPFM_V2_REPAIRED_OOF,
            dtype=float
        ),

        y_dev_np,

        SEED + 174000
    )
)


HOLD_AUTO_CAL = (
    apply_final_platt(

        FINAL_AUTO_CAL,

        HOLD_AUTO_RAW
    )
)


HOLD_DEEPFM_CAL = (
    apply_final_platt(

        FINAL_DEEPFM_CAL,

        HOLD_DEEPFM_RAW
    )
)


# ============================================================
# 17B.15 FINAL EXPERT MATRICES
# ============================================================

P_DEV = np.asarray(

    FINAL_REPAIRED_EXPERT_OOF_STRICT,

    dtype=float
)


P_HOLD = np.column_stack([

    HOLD_HGB_CAL,

    HOLD_CAT_CAL,

    HOLD_AUTO_CAL,

    HOLD_DEEPFM_CAL,
])


print(
    "\nDEV expert matrix:",
    P_DEV.shape
)

print(
    "HOLD expert matrix:",
    P_HOLD.shape
)


# ============================================================
# 17B.16 LOCK META TRAINING SPLITS
# ============================================================

split_meta = StratifiedShuffleSplit(

    n_splits=1,

    test_size=
        FINAL17_CONFIG[
            "reliability_fraction"
        ],

    random_state=
        SEED
        + 175000
)


(
    gate_pool_idx,
    reliability_idx

) = next(

    split_meta.split(

        np.zeros(
            N_DEV
        ),

        y_dev_np
    )
)


split_threshold = StratifiedShuffleSplit(

    n_splits=1,

    test_size=
        FINAL17_CONFIG[
            "threshold_fraction"
        ],

    random_state=
        SEED
        + 176000
)


(
    gate_fit_inside,
    threshold_inside

) = next(

    split_threshold.split(

        np.zeros(
            len(
                gate_pool_idx
            )
        ),

        y_dev_np[
            gate_pool_idx
        ]
    )
)


gate_fit_idx = (
    gate_pool_idx[
        gate_fit_inside
    ]
)


threshold_idx = (
    gate_pool_idx[
        threshold_inside
    ]
)


print(
    "\nMeta partitions:"
)

print(
    " gate fit:",
    len(
        gate_fit_idx
    )
)

print(
    " threshold:",
    len(
        threshold_idx
    )
)

print(
    " reliability:",
    len(
        reliability_idx
    )
)


# ============================================================
# 17B.17 TRAIN FINAL GATE
# ============================================================

gate_scaler = StandardScaler()


X_gate_fit = (
    gate_scaler
    .fit_transform(

        X_dev[
            RAMOE_FEATURES
        ]
        .iloc[
            gate_fit_idx
        ]
    )
    .astype(
        np.float32
    )
)


X_threshold_gate = (
    gate_scaler
    .transform(

        X_dev[
            RAMOE_FEATURES
        ]
        .iloc[
            threshold_idx
        ]
    )
    .astype(
        np.float32
    )
)


X_reliability_gate = (
    gate_scaler
    .transform(

        X_dev[
            RAMOE_FEATURES
        ]
        .iloc[
            reliability_idx
        ]
    )
    .astype(
        np.float32
    )
)


X_hold_gate = (
    gate_scaler
    .transform(

        X_holdout_final[
            RAMOE_FEATURES
        ]
    )
    .astype(
        np.float32
    )
)


oracle = make_oracle_targets(

    P_DEV[
        gate_fit_idx
    ],

    y_dev_np[
        gate_fit_idx
    ],

    temperature=
        RAMOE_CONFIG[
            "oracle_temperature"
        ]
)


FINAL_GATE = train_ramoe_gate(

    X_gate_fit,

    oracle,

    seed=
        SEED
        + 177000
)


W_THRESHOLD = predict_gate_weights(

    FINAL_GATE,

    X_threshold_gate
)


W_RELIABILITY = predict_gate_weights(

    FINAL_GATE,

    X_reliability_gate
)


W_HOLD = predict_gate_weights(

    FINAL_GATE,

    X_hold_gate
)


# ============================================================
# 17B.18 MoE PROBABILITIES
# ============================================================

P_THRESHOLD = mixture_probability(

    P_DEV[
        threshold_idx
    ],

    W_THRESHOLD
)


P_RELIABILITY = mixture_probability(

    P_DEV[
        reliability_idx
    ],

    W_RELIABILITY
)


P_HOLD_MOE = mixture_probability(

    P_HOLD,

    W_HOLD
)


# ============================================================
# 17B.19 LOCK DISEASE THRESHOLD
# ============================================================

FINAL_THRESHOLD, _ = (
    select_f1_threshold_local(

        y_dev_np[
            threshold_idx
        ],

        P_THRESHOLD
    )
)


print(
    "\nFINAL LOCKED THRESHOLD:",
    FINAL_THRESHOLD
)


# ============================================================
# 17B.20 RELIABILITY TRAINING TARGET
# ============================================================

reliability_pred = (

    P_RELIABILITY
    >=
    FINAL_THRESHOLD

).astype(int)


reliability_error = (

    reliability_pred
    !=
    y_dev_np[
        reliability_idx
    ]

).astype(int)


# ============================================================
# 17B.21 LPNS FEATURES / DISTANCE SETUP
# ============================================================

LPNS_BINARY = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]


LPNS_ORDINAL = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]


LPNS_COUNT = [
    "MentHlth",
    "PhysHlth",
]


LPNS_CONTINUOUS = [
    "BMI",
]


LPNS_MULTICLASS = [
    "Diabetes",
]


LPNS_ALL = (
    LPNS_BINARY
    +
    LPNS_ORDINAL
    +
    LPNS_COUNT
    +
    LPNS_CONTINUOUS
    +
    LPNS_MULTICLASS
)


LPNS_SCALE = (
    LPNS_CONTINUOUS
    +
    LPNS_COUNT
    +
    LPNS_ORDINAL
)


LPNS_DISCRETE = (
    LPNS_BINARY
    +
    LPNS_MULTICLASS
)


pos = {
    c: i
    for i, c
    in enumerate(
        LPNS_ALL
    )
}


scale_idx = np.array([
    pos[c]
    for c in LPNS_SCALE
])


disc_idx = np.array([
    pos[c]
    for c in LPNS_DISCRETE
])


DEV_DIST_RAW = (

    X_dev[
        LPNS_ALL
    ]
    .to_numpy(
        dtype=np.float32
    )
)


HOLD_DIST_RAW = (

    X_holdout_final[
        LPNS_ALL
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 17B.22 DISTANCE SCALER
# ============================================================

distance_scaler = StandardScaler()


X_GATEPOOL_DIST = (
    DEV_DIST_RAW[
        gate_pool_idx
    ]
    .copy()
)


X_GATEPOOL_DIST[
    :,
    scale_idx
] = (
    distance_scaler
    .fit_transform(

        X_GATEPOOL_DIST[
            :,
            scale_idx
        ]
    )
    .astype(
        np.float32
    )
)


X_GATEPOOL_DIST[
    :,
    disc_idx
] *= (
    FINAL17_CONFIG[
        "discrete_weight"
    ]
)


def transform_distance(X):

    X = (
        X.astype(
            np.float32
        )
        .copy()
    )


    X[
        :,
        scale_idx
    ] = (
        distance_scaler
        .transform(

            X[
                :,
                scale_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X[
        :,
        disc_idx
    ] *= (
        FINAL17_CONFIG[
            "discrete_weight"
        ]
    )


    return X


X_REL_DIST = transform_distance(

    DEV_DIST_RAW[
        reliability_idx
    ]
)


X_HOLD_DIST = transform_distance(
    HOLD_DIST_RAW
)


# ============================================================
# 17B.23 REFERENCE BANK
# ============================================================

rng = np.random.default_rng(
    SEED
    + 178000
)


reference_inside = np.arange(
    len(
        gate_pool_idx
    )
)


if len(
    reference_inside
) > FINAL17_CONFIG[
    "reference_size"
]:

    reference_inside = rng.choice(

        reference_inside,

        size=
            FINAL17_CONFIG[
                "reference_size"
            ],

        replace=False
    )


reference_global = (
    gate_pool_idx[
        reference_inside
    ]
)


X_REFERENCE = (
    X_GATEPOOL_DIST[
        reference_inside
    ]
)


P_REFERENCE = (
    P_DEV[
        reference_global
    ]
)


print(
    "\nReference bank:",
    len(
        reference_global
    )
)


# ============================================================
# 17B.24 NEAREST NEIGHBORS
# ============================================================

nbr_model = NearestNeighbors(

    n_neighbors=
        FINAL17_CONFIG[
            "k_neighbors"
        ],

    metric="euclidean",

    n_jobs=-1,
)


nbr_model.fit(
    X_REFERENCE
)


REL_DIST, REL_NBR = (
    nbr_model.kneighbors(

        X_REL_DIST,

        return_distance=True
    )
)


HOLD_DIST, HOLD_NBR = (
    nbr_model.kneighbors(

        X_HOLD_DIST,

        return_distance=True
    )
)


# ============================================================
# 17B.25 LPNS EXTRACTOR
# ============================================================

def extract_lpns_final(
    query_experts,
    neighbor_idx,
    neighbor_dist
):

    neighbor_prob = (
        P_REFERENCE[
            neighbor_idx
        ]
    )


    valid = (
        neighbor_dist
        <=
        FINAL17_CONFIG[
            "max_distance"
        ]
    )


    count = (
        valid.sum(
            axis=1
        )
    )


    count_safe = np.maximum(
        count,
        1
    )[:, None]


    mask = valid[
        :,
        :,
        None
    ]


    center = (
        query_experts[
            :,
            None,
            :
        ]
    )


    local_mad = (

        np.sum(

            np.abs(
                neighbor_prob
                -
                center
            )

            *
            mask,

            axis=1
        )

        /
        count_safe
    )


    local_mean = (

        np.sum(
            neighbor_prob
            *
            mask,
            axis=1
        )

        /
        count_safe
    )


    local_sd = np.sqrt(

        np.sum(

            (
                neighbor_prob
                -
                local_mean[
                    :,
                    None,
                    :
                ]
            ) ** 2

            *
            mask,

            axis=1
        )

        /
        count_safe
    )


    center_decision = (
        query_experts
        >=
        FINAL_THRESHOLD
    )


    neighbor_decision = (
        neighbor_prob
        >=
        FINAL_THRESHOLD
    )


    flip = (

        np.sum(

            (
                neighbor_decision
                !=
                center_decision[
                    :,
                    None,
                    :
                ]
            )

            *
            mask,

            axis=1
        )

        /
        count_safe
    )


    mean_mad = np.mean(
        local_mad,
        axis=1
    )


    max_mad = np.max(
        local_mad,
        axis=1
    )


    min_mad = np.min(
        local_mad,
        axis=1
    )


    mad_gap = (
        max_mad
        -
        min_mad
    )


    mad_cv = (

        np.std(
            local_mad,
            axis=1
        )

        /
        (
            mean_mad
            +
            1e-8
        )
    )


    mean_sd = np.mean(
        local_sd,
        axis=1
    )


    max_sd = np.max(
        local_sd,
        axis=1
    )


    mean_flip = np.mean(
        flip,
        axis=1
    )


    max_flip = np.max(
        flip,
        axis=1
    )


    features = np.column_stack([

        mean_mad,
        max_mad,
        min_mad,
        mad_gap,
        mad_cv,

        mean_sd,
        max_sd,

        mean_flip,
        max_flip,

        count.astype(float),

        local_mad,
        local_sd,
        flip,
    ])


    valid_profile = (
        count
        >=
        FINAL17_CONFIG[
            "min_neighbors"
        ]
    )


    features[
        ~valid_profile
    ] = np.nan


    return (
        features,
        valid_profile,
        local_mad,
        count
    )


(
    LPNS_REL,
    REL_VALID_PROFILE,
    REL_LOCAL_MAD,
    REL_COUNT

) = extract_lpns_final(

    P_DEV[
        reliability_idx
    ],

    REL_NBR,

    REL_DIST
)


(
    LPNS_HOLD,
    HOLD_VALID_PROFILE,
    HOLD_LOCAL_MAD,
    HOLD_COUNT

) = extract_lpns_final(

    P_HOLD,

    HOLD_NBR,

    HOLD_DIST
)


print(
    "\nLPNS coverage:"
)

print(
    " DEV reliability:",
    REL_VALID_PROFILE.mean()
)

print(
    " HOLDOUT:",
    HOLD_VALID_PROFILE.mean()
)


# ============================================================
# 17B.26 TRAIN-ONLY IMPUTATION
# ============================================================

lpns_medians = np.nanmedian(

    LPNS_REL,

    axis=0
)


lpns_medians = np.where(

    np.isnan(
        lpns_medians
    ),

    0.0,

    lpns_medians
)


def impute_lpns(X):

    X = (
        X.copy()
        .astype(float)
    )


    idx = np.where(
        np.isnan(X)
    )


    X[idx] = (
        lpns_medians[
            idx[1]
        ]
    )


    return X


LPNS_REL = impute_lpns(
    LPNS_REL
)


LPNS_HOLD = impute_lpns(
    LPNS_HOLD
)


# ============================================================
# 17B.27 TRUST FEATURE BUILDER
# ============================================================

def binary_entropy(p):

    p = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )


    return (

        -p * np.log(p)

        -

        (1 - p)
        *
        np.log(
            1 - p
        )

    ) / np.log(2)


def build_trust(
    moe_p,
    expert_p
):

    entropy = binary_entropy(
        moe_p
    )


    signed_margin = (
        moe_p
        -
        FINAL_THRESHOLD
    )


    abs_margin = np.abs(
        signed_margin
    )


    distance_half = np.abs(
        moe_p
        -
        0.5
    )


    F_conf = np.column_stack([

        moe_p,
        entropy,
        signed_margin,
        abs_margin,
        distance_half,
    ])


    mean = np.mean(
        expert_p,
        axis=1
    )


    std = np.std(
        expert_p,
        axis=1
    )


    mn = np.min(
        expert_p,
        axis=1
    )


    mx = np.max(
        expert_p,
        axis=1
    )


    rng = (
        mx - mn
    )


    median = np.median(
        expert_p,
        axis=1
    )


    consensus_gap = np.abs(
        moe_p
        -
        mean
    )


    abs_dev = np.mean(

        np.abs(
            expert_p
            -
            moe_p[:, None]
        ),

        axis=1
    )


    F_dis = np.column_stack([

        F_conf,

        mean,
        std,
        mn,
        mx,
        rng,
        median,
        consensus_gap,
        abs_dev,

        expert_p,
    ])


    return (
        F_conf,
        F_dis
    )


(
    CONF_REL,
    DIS_REL

) = build_trust(

    P_RELIABILITY,

    P_DEV[
        reliability_idx
    ]
)


(
    CONF_HOLD,
    DIS_HOLD

) = build_trust(

    P_HOLD_MOE,

    P_HOLD
)


LPNS_FEATURE_REL = np.column_stack([

    DIS_REL,

    LPNS_REL,
])


LPNS_FEATURE_HOLD = np.column_stack([

    DIS_HOLD,

    LPNS_HOLD,
])


# ============================================================
# 17B.28 RELIABILITY MODELS
# ============================================================

def make_rel_model():

    return Pipeline([

        (
            "scale",
            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    FINAL17_CONFIG[
                        "reliability_C"
                    ],

                class_weight=
                    "balanced",

                solver=
                    "lbfgs",

                max_iter=
                    3000,
            )
        )
    ])


MODEL_CONF = make_rel_model()

MODEL_DIS = make_rel_model()

MODEL_LPNS = make_rel_model()


MODEL_CONF.fit(

    CONF_REL,

    reliability_error
)


MODEL_DIS.fit(

    DIS_REL,

    reliability_error
)


MODEL_LPNS.fit(

    LPNS_FEATURE_REL,

    reliability_error
)


RISK_CONF_HOLD = (

    MODEL_CONF
    .predict_proba(
        CONF_HOLD
    )[:, 1]
)


RISK_DIS_HOLD = (

    MODEL_DIS
    .predict_proba(
        DIS_HOLD
    )[:, 1]
)


RISK_LPNS_HOLD = (

    MODEL_LPNS
    .predict_proba(
        LPNS_FEATURE_HOLD
    )[:, 1]
)


# ============================================================
# 17B.29 FINAL HOLDOUT CLASSIFICATION
# ============================================================

HOLD_PRED = (

    P_HOLD_MOE
    >=
    FINAL_THRESHOLD

).astype(int)


HOLD_ERROR = (

    HOLD_PRED
    !=
    y_hold

).astype(int)


HOLD_FN = (

    (y_hold == 1)

    &

    (HOLD_PRED == 0)

).astype(int)


# ============================================================
# 17B.30 ECE
# ============================================================

def ece_score(
    y,
    p,
    n_bins=15
):

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )


    result = 0.0


    for i in range(
        n_bins
    ):

        if i < (
            n_bins - 1
        ):

            mask = (
                (p >= edges[i])
                &
                (p < edges[i + 1])
            )

        else:

            mask = (
                (p >= edges[i])
                &
                (p <= edges[i + 1])
            )


        if not np.any(
            mask
        ):
            continue


        result += (

            mask.mean()

            *
            abs(
                p[
                    mask
                ].mean()

                -

                y[
                    mask
                ].mean()
            )
        )


    return float(
        result
    )


# ============================================================
# 17B.31 DISEASE PERFORMANCE
# ============================================================

tn, fp, fn, tp = confusion_matrix(

    y_hold,

    HOLD_PRED

).ravel()


FINAL_HOLDOUT_DISEASE = pd.DataFrame([{

    "N":
        N_HOLD,

    "Prevalence":
        y_hold.mean(),

    "PR_AUC":
        average_precision_score(
            y_hold,
            P_HOLD_MOE
        ),

    "ROC_AUC":
        roc_auc_score(
            y_hold,
            P_HOLD_MOE
        ),

    "Brier":
        brier_score_loss(
            y_hold,
            P_HOLD_MOE
        ),

    "NLL":
        log_loss(
            y_hold,
            P_HOLD_MOE
        ),

    "ECE":
        ece_score(
            y_hold,
            P_HOLD_MOE
        ),

    "Threshold":
        FINAL_THRESHOLD,

    "TP":
        tp,

    "FP":
        fp,

    "TN":
        tn,

    "FN":
        fn,

    "Precision":
        precision_score(
            y_hold,
            HOLD_PRED,
            zero_division=0
        ),

    "Recall":
        recall_score(
            y_hold,
            HOLD_PRED,
            zero_division=0
        ),

    "Specificity":
        (
            tn / (tn + fp)
        ),

    "F1":
        f1_score(
            y_hold,
            HOLD_PRED,
            zero_division=0
        ),

    "MCC":
        matthews_corrcoef(
            y_hold,
            HOLD_PRED
        ),

    "Balanced_Accuracy":
        balanced_accuracy_score(
            y_hold,
            HOLD_PRED
        ),

    "Accuracy":
        accuracy_score(
            y_hold,
            HOLD_PRED
        ),

    "Mean_Probability":
        P_HOLD_MOE.mean(),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 17B — FINAL HOLDOUT DISEASE PERFORMANCE"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_DISEASE
)


# ============================================================
# 17B.32 AURC
# ============================================================

def aurc(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    error_sorted = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative = np.cumsum(
        error_sorted
    )


    n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative / n
    )


    coverage = (
        n / len(error)
    )


    return float(
        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 17B.33 RELIABILITY PERFORMANCE
# ============================================================

FINAL_HOLDOUT_RELIABILITY = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            average_precision_score(
                HOLD_ERROR,
                RISK_CONF_HOLD
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                HOLD_ERROR,
                RISK_CONF_HOLD
            ),

        "AURC":
            aurc(
                HOLD_ERROR,
                RISK_CONF_HOLD
            ),
    },

    {
        "Method":
            "Confidence+Disagreement",

        "Error_PR_AUC":
            average_precision_score(
                HOLD_ERROR,
                RISK_DIS_HOLD
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                HOLD_ERROR,
                RISK_DIS_HOLD
            ),

        "AURC":
            aurc(
                HOLD_ERROR,
                RISK_DIS_HOLD
            ),
    },

    {
        "Method":
            "LPNS",

        "Error_PR_AUC":
            average_precision_score(
                HOLD_ERROR,
                RISK_LPNS_HOLD
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                HOLD_ERROR,
                RISK_LPNS_HOLD
            ),

        "AURC":
            aurc(
                HOLD_ERROR,
                RISK_LPNS_HOLD
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT RELIABILITY PERFORMANCE"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_RELIABILITY
)


# ============================================================
# 17B.34 LPNS INCREMENTAL VALUE
# ============================================================

base = (
    FINAL_HOLDOUT_RELIABILITY[
        FINAL_HOLDOUT_RELIABILITY[
            "Method"
        ]
        ==
        "Confidence+Disagreement"
    ]
    .iloc[0]
)


lpns = (
    FINAL_HOLDOUT_RELIABILITY[
        FINAL_HOLDOUT_RELIABILITY[
            "Method"
        ]
        ==
        "LPNS"
    ]
    .iloc[0]
)


FINAL_HOLDOUT_LPNS_GAIN = pd.DataFrame([{

    "Delta_Error_PR_AUC":
        lpns[
            "Error_PR_AUC"
        ]
        -
        base[
            "Error_PR_AUC"
        ],

    "Delta_Error_ROC_AUC":
        lpns[
            "Error_ROC_AUC"
        ]
        -
        base[
            "Error_ROC_AUC"
        ],

    "Delta_AURC":
        lpns[
            "AURC"
        ]
        -
        base[
            "AURC"
        ],
}])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT LPNS INCREMENTAL VALUE"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_LPNS_GAIN
)


# ============================================================
# 17B.35 SELECTIVE REVIEW
# ============================================================

def selective_eval(
    risk,
    method
):

    rows = []


    total_errors = max(
        HOLD_ERROR.sum(),
        1
    )


    total_fn = max(
        HOLD_FN.sum(),
        1
    )


    base_error_rate = (
        HOLD_ERROR.mean()
    )


    order = np.argsort(
        -risk
    )


    for rate in (
        FINAL17_CONFIG[
            "review_rates"
        ]
    ):

        n_review = max(
            1,
            int(
                round(
                    N_HOLD
                    *
                    rate
                )
            )
        )


        review = (
            order[
                :n_review
            ]
        )


        keep = np.ones(
            N_HOLD,
            dtype=bool
        )


        keep[
            review
        ] = False


        reviewed_error_rate = (
            HOLD_ERROR[
                review
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Review_Rate":
                rate,

            "Coverage":
                1 - rate,

            "Error_Enrichment":
                reviewed_error_rate
                /
                base_error_rate,

            "Errors_Captured":
                HOLD_ERROR[
                    review
                ].sum()
                /
                total_errors,

            "FalseNegatives_Captured":
                HOLD_FN[
                    review
                ].sum()
                /
                total_fn,

            "Error_Per_100_Reviews":
                reviewed_error_rate
                *
                100.0,

            "Retained_Accuracy":
                accuracy_score(

                    y_hold[
                        keep
                    ],

                    HOLD_PRED[
                        keep
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y_hold[
                        keep
                    ],

                    HOLD_PRED[
                        keep
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y_hold[
                        keep
                    ],

                    HOLD_PRED[
                        keep
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


FINAL_HOLDOUT_SELECTIVE = pd.concat(

    [

        selective_eval(
            RISK_CONF_HOLD,
            "Confidence"
        ),

        selective_eval(
            RISK_DIS_HOLD,
            "Confidence+Disagreement"
        ),

        selective_eval(
            RISK_LPNS_HOLD,
            "LPNS"
        ),
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT SELECTIVE REVIEW"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_SELECTIVE
)


# ============================================================
# 17B.36 EXPERT PERFORMANCE
# ============================================================

expert_names = [

    "HistGradientBoosting",

    "CatBoost",

    "AutoInt_Repaired",

    "DeepFM_v2_Repaired",
]


expert_rows = []


for j, name in enumerate(
    expert_names
):

    p = (
        P_HOLD[
            :,
            j
        ]
    )


    expert_rows.append({

        "Expert":
            name,

        "PR_AUC":
            average_precision_score(
                y_hold,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y_hold,
                p
            ),

        "Brier":
            brier_score_loss(
                y_hold,
                p
            ),

        "NLL":
            log_loss(
                y_hold,
                p
            ),

        "ECE":
            ece_score(
                y_hold,
                p
            ),

        "Mean_Probability":
            p.mean(),
    })


FINAL_HOLDOUT_EXPERTS = pd.DataFrame(
    expert_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT EXPERT PERFORMANCE"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_EXPERTS
)


# ============================================================
# 17B.37 GATE DIAGNOSTICS
# ============================================================

W_SAFE = np.clip(
    W_HOLD,
    1e-12,
    1.0
)


gate_entropy = (

    -np.sum(

        W_SAFE
        *
        np.log(
            W_SAFE
        ),

        axis=1
    )

    /
    np.log(
        W_HOLD.shape[1]
    )
)


FINAL_HOLDOUT_ROUTING = pd.DataFrame([{

    "Mean_Gate_Entropy":
        gate_entropy.mean(),

    "Median_Gate_Entropy":
        np.median(
            gate_entropy
        ),

    "Mean_Max_Weight":
        np.max(
            W_HOLD,
            axis=1
        ).mean(),

    "Mean_HGB_Weight":
        W_HOLD[
            :,
            0
        ].mean(),

    "Mean_CatBoost_Weight":
        W_HOLD[
            :,
            1
        ].mean(),

    "Mean_AutoInt_Weight":
        W_HOLD[
            :,
            2
        ].mean(),

    "Mean_DeepFM_Weight":
        W_HOLD[
            :,
            3
        ].mean(),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT ROUTING DIAGNOSTICS"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_ROUTING
)


# ============================================================
# 17B.38 LPNS DIAGNOSTICS
# ============================================================

FINAL_HOLDOUT_LPNS_DIAG = pd.DataFrame([{

    "Reference_N":
        len(
            reference_global
        ),

    "Coverage":
        HOLD_VALID_PROFILE.mean(),

    "Mean_Neighbors":
        HOLD_COUNT.mean(),

    "Median_Neighbors":
        np.median(
            HOLD_COUNT
        ),

    "Mean_Local_MAD":
        np.nanmean(
            HOLD_LOCAL_MAD
        ),

    "Median_Local_MAD":
        np.nanmedian(
            HOLD_LOCAL_MAD
        ),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL HOLDOUT LPNS DIAGNOSTICS"
)

print(
    "=" * 135
)


display(
    FINAL_HOLDOUT_LPNS_DIAG
)


# ============================================================
# 17B.39 SAVE ALL FINAL RESULTS
# ============================================================

FINAL_HOLDOUT_PREDICTIONS = pd.DataFrame({

    "target":
        y_hold,

    "HGB":
        HOLD_HGB_CAL,

    "CatBoost":
        HOLD_CAT_CAL,

    "AutoInt":
        HOLD_AUTO_CAL,

    "DeepFM_v2":
        HOLD_DEEPFM_CAL,

    "MoE":
        P_HOLD_MOE,

    "Threshold":
        FINAL_THRESHOLD,

    "Prediction":
        HOLD_PRED,

    "Error":
        HOLD_ERROR,

    "FalseNegative":
        HOLD_FN,

    "Risk_Confidence":
        RISK_CONF_HOLD,

    "Risk_Disagreement":
        RISK_DIS_HOLD,

    "Risk_LPNS":
        RISK_LPNS_HOLD,

    "Mean_Local_MAD":
        np.nanmean(
            HOLD_LOCAL_MAD,
            axis=1
        ),

    "Neighbor_Count":
        HOLD_COUNT,
})


save_csv(
    FINAL_HOLDOUT_PREDICTIONS,
    PRED_DIR
    / "module17b_final_holdout_predictions.csv"
)


save_csv(
    FINAL_HOLDOUT_DISEASE,
    TAB_DIR
    / "module17b_holdout_disease.csv"
)


save_csv(
    FINAL_HOLDOUT_RELIABILITY,
    TAB_DIR
    / "module17b_holdout_reliability.csv"
)


save_csv(
    FINAL_HOLDOUT_LPNS_GAIN,
    TAB_DIR
    / "module17b_holdout_lpns_gain.csv"
)


save_csv(
    FINAL_HOLDOUT_SELECTIVE,
    TAB_DIR
    / "module17b_holdout_selective.csv"
)


save_csv(
    FINAL_HOLDOUT_EXPERTS,
    TAB_DIR
    / "module17b_holdout_experts.csv"
)


save_csv(
    FINAL_HOLDOUT_ROUTING,
    TAB_DIR
    / "module17b_holdout_routing.csv"
)


save_csv(
    FINAL_HOLDOUT_LPNS_DIAG,
    TAB_DIR
    / "module17b_holdout_lpns_diagnostics.csv"
)


module17b_manifest = {

    "module":
        "Single Final Locked HOLDOUT Evaluation",

    "holdout_opened":
        True,

    "architecture_locked_before_holdout":
        True,

    "experts": [
        "HistGradientBoosting",
        "CatBoost",
        "AutoInt_Repaired",
        "DeepFM_v2_Repaired",
    ],

    "HGB_source":
        "original Module-6 factory",

    "CatBoost_source":
        "original Module-6 factory",

    "AutoInt_epoch":
        FINAL_AUTOINT_EPOCH,

    "DeepFM_epoch":
        FINAL_DEEPFM_EPOCH,

    "neural_epoch_rule":
        "median repaired inner-selected epoch",

    "calibration": {
        "HGB":
            "raw",

        "CatBoost":
            "Platt fitted on reconstructed raw DEV OOF",

        "AutoInt":
            "Platt fitted on repaired raw DEV OOF",

        "DeepFM_v2":
            "Platt fitted on repaired raw DEV OOF",
    },

    "threshold_source":
        "DEV only",

    "gate_source":
        "DEV only",

    "reliability_source":
        "DEV only",

    "LPNS_reference_source":
        "DEV only",

    "LPNS_k":
        20,

    "LPNS_reference_size":
        80000,

    "routing_in_final_reliability":
        False,

    "post_holdout_tuning_allowed":
        False,
}


save_json(
    module17b_manifest,
    LOG_DIR
    / "module17b_final_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 17B COMPLETE — HOLDOUT HAS NOW BEEN OPENED"
)

print(
    "=" * 135
)


print(
    "\nNO FURTHER MODEL OR ARCHITECTURE TUNING IS PERMITTED."
)

HOLDOUT resolved:
 X: X_holdout (50736, 21)
 y: y_holdout (50736,)

Locked neural epochs:
 AutoInt: 10 [8, 10, 13, 16, 3]
 DeepFM: 12 [17, 9, 10, 16, 12]

FINAL HGB — FULL DEV FIT

FINAL CATBOOST — FULL DEV FIT

FINAL AUTOINT — FULL DEV FIT
AutoInt epoch 01/10 loss=0.909748
AutoInt epoch 02/10 loss=0.879546
AutoInt epoch 03/10 loss=0.875315
AutoInt epoch 04/10 loss=0.871993
AutoInt epoch 05/10 loss=0.870831
AutoInt epoch 06/10 loss=0.870750
AutoInt epoch 07/10 loss=0.867948
AutoInt epoch 08/10 loss=0.867768
AutoInt epoch 09/10 loss=0.865994
AutoInt epoch 10/10 loss=0.865736

FINAL DEEPFM_v2 — FULL DEV FIT
DeepFM_v2 epoch 01/12 loss=0.977912
DeepFM_v2 epoch 02/12 loss=0.903864
DeepFM_v2 epoch 03/12 loss=0.892231
DeepFM_v2 epoch 04/12 loss=0.886352
DeepFM_v2 epoch 05/12 loss=0.880230
DeepFM_v2 epoch 06/12 loss=0.873385
DeepFM_v2 epoch 07/12 loss=0.871311
DeepFM_v2 epoch 08/12 loss=0.867946
DeepFM_v2 epoch 09/12 loss=0.865706
DeepFM_v2 epoch 10/12 loss=0.863878
DeepFM_v2 epoch 11/12 loss=

,N,Prevalence,PR_AUC,ROC_AUC,Brier,NLL,ECE,Threshold,TP,FP,TN,FN,Precision,Recall,Specificity,F1,MCC,Balanced_Accuracy,Accuracy,Mean_Probability
0,50736,0.094193,0.372075,0.851104,0.069944,0.235925,0.004022,0.23,2336,4048,41909,2443,0.365915,0.488805,0.911918,0.418525,0.352927,0.700361,0.872063,0.092268



FINAL HOLDOUT RELIABILITY PERFORMANCE


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.519181,0.894671,0.025144
1,Confidence+Disagreement,0.517617,0.894580,0.025154
2,LPNS,0.530683,0.894701,0.025383



FINAL HOLDOUT LPNS INCREMENTAL VALUE


...


FINAL HOLDOUT ROUTING DIAGNOSTICS


,Mean_Gate_Entropy,Median_Gate_Entropy,Mean_Max_Weight,Mean_HGB_Weight,Mean_CatBoost_Weight,Mean_AutoInt_Weight,Mean_DeepFM_Weight
0,0.998532,0.999065,0.2682,0.249751,0.24639,0.251905,0.251954



FINAL HOLDOUT LPNS DIAGNOSTICS


,Reference_N,Coverage,Mean_Neighbors,Median_Neighbors,Mean_Local_MAD,Median_Local_MAD
0,80000,0.999527,19.987149,20.0,0.039885,0.02281



MODULE 17B COMPLETE — HOLDOUT HAS NOW BEEN OPENED

NO FURTHER MODEL OR ARCHITECTURE TUNING IS PERMITTED.


In [46]:
# ============================================================
# MODULE 18A — DISCRIMINATION RESCUE
#
# GOAL
# ------------------------------------------------------------
# Improve disease-status discrimination BEFORE rebuilding the
# final ensemble.
#
# Search:
#   - HistGradientBoosting
#   - CatBoost
#   - LightGBM
#   - XGBoost
#
# Improvements tested:
#   - clinically sensible interaction features
#   - multimorbidity / burden features
#   - class-weight sensitivity
#   - stronger boosting configurations
#
#
# PROTOCOL
# ------------------------------------------------------------
# Stage 1:
#   coarse screening on locked folds 0,1,2
#
# Stage 2:
#   best TWO configurations per family
#   confirmed on ALL 5 locked folds
#
# Selection metric:
#   PR-AUC ONLY
#
# HOLDOUT:
#   NEVER TOUCHED
# ============================================================


import gc
import time
import copy
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# ============================================================
# 18A.1 REQUIREMENTS
# ============================================================

required_18a = [
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_18a
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


y18 = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 18A.2 ORIGINAL FEATURE SET
# ============================================================

ORIGINAL_FEATURES_18 = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


# ============================================================
# 18A.3 FEATURE ENGINEERING
#
# No target information is used.
#
# All features derive only from existing BRFSS predictors.
# ============================================================

def build_rescue_features(X):

    Z = (
        X[
            ORIGINAL_FEATURES_18
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Binary / interpretable helper indicators
    # --------------------------------------------------------

    Z[
        "DiabetesAny"
    ] = (
        Z[
            "Diabetes"
        ]
        >
        0
    ).astype(float)


    Z[
        "PoorGenHealth"
    ] = (
        Z[
            "GenHlth"
        ]
        >=
        4
    ).astype(float)


    Z[
        "OlderAge"
    ] = (
        Z[
            "Age"
        ]
        >=
        9
    ).astype(float)


    Z[
        "ObesityProxy"
    ] = (
        Z[
            "BMI"
        ]
        >=
        30
    ).astype(float)


    # --------------------------------------------------------
    # Cardiometabolic burden
    # --------------------------------------------------------

    Z[
        "CardiometabolicBurden"
    ] = (

        Z[
            "HighBP"
        ]

        +

        Z[
            "HighChol"
        ]

        +

        Z[
            "DiabetesAny"
        ]

        +

        Z[
            "ObesityProxy"
        ]
    )


    # --------------------------------------------------------
    # Cardiovascular-history burden
    # --------------------------------------------------------

    Z[
        "CVHistoryBurden"
    ] = (

        Z[
            "Stroke"
        ]

        +

        Z[
            "DiffWalk"
        ]

        +

        Z[
            "HighBP"
        ]

        +

        Z[
            "HighChol"
        ]
    )


    # --------------------------------------------------------
    # Lifestyle-risk burden
    # --------------------------------------------------------

    Z[
        "LifestyleRiskBurden"
    ] = (

        Z[
            "Smoker"
        ]

        +

        (
            1.0
            -
            Z[
                "PhysActivity"
            ]
        )

        +

        (
            1.0
            -
            Z[
                "Fruits"
            ]
        )

        +

        (
            1.0
            -
            Z[
                "Veggies"
            ]
        )

        +

        Z[
            "HvyAlcoholConsump"
        ]
    )


    # --------------------------------------------------------
    # Healthcare-access burden
    # --------------------------------------------------------

    Z[
        "AccessBurden"
    ] = (

        Z[
            "NoDocbcCost"
        ]

        +

        (
            1.0
            -
            Z[
                "AnyHealthcare"
            ]
        )
    )


    # --------------------------------------------------------
    # Functional / health burden
    # --------------------------------------------------------

    Z[
        "PhysicalBurden"
    ] = (

        Z[
            "PhysHlth"
        ]
        /
        30.0

        +

        Z[
            "DiffWalk"
        ]

        +

        (
            Z[
                "GenHlth"
            ]
            -
            1.0
        )
        /
        4.0
    )


    Z[
        "MentalPhysicalBurden"
    ] = (

        Z[
            "MentHlth"
        ]

        +

        Z[
            "PhysHlth"
        ]
    )


    # --------------------------------------------------------
    # Multimorbidity
    # --------------------------------------------------------

    Z[
        "MultimorbidityCount"
    ] = (

        Z[
            "HighBP"
        ]

        +

        Z[
            "HighChol"
        ]

        +

        Z[
            "Stroke"
        ]

        +

        Z[
            "DiabetesAny"
        ]

        +

        Z[
            "DiffWalk"
        ]
    )


    # --------------------------------------------------------
    # Clinically plausible interactions
    # --------------------------------------------------------

    Z[
        "Age_x_GenHlth"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "GenHlth"
        ]
    )


    Z[
        "Age_x_HighBP"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "HighBP"
        ]
    )


    Z[
        "Age_x_HighChol"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "HighChol"
        ]
    )


    Z[
        "Age_x_Diabetes"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "Diabetes"
        ]
    )


    Z[
        "Age_x_Stroke"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "Stroke"
        ]
    )


    Z[
        "Age_x_DiffWalk"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "DiffWalk"
        ]
    )


    Z[
        "Age_x_Sex"
    ] = (
        Z[
            "Age"
        ]
        *
        Z[
            "Sex"
        ]
    )


    Z[
        "BP_x_Chol"
    ] = (
        Z[
            "HighBP"
        ]
        *
        Z[
            "HighChol"
        ]
    )


    Z[
        "BP_x_Diabetes"
    ] = (
        Z[
            "HighBP"
        ]
        *
        Z[
            "DiabetesAny"
        ]
    )


    Z[
        "Chol_x_Diabetes"
    ] = (
        Z[
            "HighChol"
        ]
        *
        Z[
            "DiabetesAny"
        ]
    )


    Z[
        "Stroke_x_DiffWalk"
    ] = (
        Z[
            "Stroke"
        ]
        *
        Z[
            "DiffWalk"
        ]
    )


    Z[
        "BMI_x_Inactivity"
    ] = (

        Z[
            "BMI"
        ]

        *
        (
            1.0
            -
            Z[
                "PhysActivity"
            ]
        )
    )


    Z[
        "GenHlth_x_PhysHlth"
    ] = (
        Z[
            "GenHlth"
        ]
        *
        Z[
            "PhysHlth"
        ]
    )


    # --------------------------------------------------------
    # Mild nonlinear terms
    # --------------------------------------------------------

    Z[
        "Age_sq"
    ] = (
        Z[
            "Age"
        ] ** 2
    )


    Z[
        "BMI_sq"
    ] = (
        Z[
            "BMI"
        ] ** 2
    )


    return Z.astype(
        np.float32
    )


X18 = build_rescue_features(
    X_dev
)


print(
    "Original features:",
    len(
        ORIGINAL_FEATURES_18
    )
)

print(
    "Rescue features:",
    X18.shape[1]
)


# ============================================================
# 18A.4 CLASS-WEIGHT HELPERS
# ============================================================

def hgb_sample_weight(
    y,
    pos_weight
):

    w = np.ones(
        len(y),
        dtype=float
    )

    w[
        y == 1
    ] = float(
        pos_weight
    )

    return w


# ============================================================
# 18A.5 MODEL FACTORY
# ============================================================

def make_rescue_model(
    family,
    cfg,
    seed
):

    cfg = copy.deepcopy(
        cfg
    )


    pos_weight = cfg.pop(
        "pos_weight"
    )


    if family == "HGB":

        model = (
            HistGradientBoostingClassifier(

                loss="log_loss",

                learning_rate=
                    cfg[
                        "learning_rate"
                    ],

                max_iter=
                    cfg[
                        "max_iter"
                    ],

                max_depth=
                    cfg[
                        "max_depth"
                    ],

                min_samples_leaf=
                    cfg[
                        "min_samples_leaf"
                    ],

                l2_regularization=
                    cfg[
                        "l2_regularization"
                    ],

                random_state=
                    seed,

                early_stopping=
                    False,
            )
        )


    elif family == "CatBoost":

        model = (
            CatBoostClassifier(

                iterations=
                    cfg[
                        "iterations"
                    ],

                learning_rate=
                    cfg[
                        "learning_rate"
                    ],

                depth=
                    cfg[
                        "depth"
                    ],

                l2_leaf_reg=
                    cfg[
                        "l2_leaf_reg"
                    ],

                random_strength=
                    cfg[
                        "random_strength"
                    ],

                bagging_temperature=
                    cfg[
                        "bagging_temperature"
                    ],

                loss_function=
                    "Logloss",

                eval_metric=
                    "PRAUC",

                class_weights=[
                    1.0,
                    float(
                        pos_weight
                    )
                ],

                random_seed=
                    seed,

                verbose=
                    False,

                task_type=
                    "GPU",

                devices=
                    "0",
            )
        )


    elif family == "LightGBM":

        model = (
            LGBMClassifier(

                objective=
                    "binary",

                n_estimators=
                    cfg[
                        "n_estimators"
                    ],

                learning_rate=
                    cfg[
                        "learning_rate"
                    ],

                num_leaves=
                    cfg[
                        "num_leaves"
                    ],

                max_depth=
                    cfg[
                        "max_depth"
                    ],

                min_child_samples=
                    cfg[
                        "min_child_samples"
                    ],

                subsample=
                    cfg[
                        "subsample"
                    ],

                colsample_bytree=
                    cfg[
                        "colsample_bytree"
                    ],

                reg_alpha=
                    cfg[
                        "reg_alpha"
                    ],

                reg_lambda=
                    cfg[
                        "reg_lambda"
                    ],

                scale_pos_weight=
                    float(
                        pos_weight
                    ),

                random_state=
                    seed,

                n_jobs=
                    -1,

                verbosity=
                    -1,
            )
        )


    elif family == "XGBoost":

        model = (
            XGBClassifier(

                objective=
                    "binary:logistic",

                n_estimators=
                    cfg[
                        "n_estimators"
                    ],

                learning_rate=
                    cfg[
                        "learning_rate"
                    ],

                max_depth=
                    cfg[
                        "max_depth"
                    ],

                min_child_weight=
                    cfg[
                        "min_child_weight"
                    ],

                subsample=
                    cfg[
                        "subsample"
                    ],

                colsample_bytree=
                    cfg[
                        "colsample_bytree"
                    ],

                gamma=
                    cfg[
                        "gamma"
                    ],

                reg_alpha=
                    cfg[
                        "reg_alpha"
                    ],

                reg_lambda=
                    cfg[
                        "reg_lambda"
                    ],

                scale_pos_weight=
                    float(
                        pos_weight
                    ),

                eval_metric=
                    "aucpr",

                tree_method=
                    "hist",

                device=
                    "cuda",

                random_state=
                    seed,

                n_jobs=
                    -1,
            )
        )


    else:

        raise ValueError(
            family
        )


    return (
        model,
        pos_weight
    )


# ============================================================
# 18A.6 COARSE CONFIGURATION BANK
#
# Intentionally emphasizes:
#   - lower/intermediate class weights
#   - moderate tree depth
#   - regularization
# ============================================================

SEARCH_CONFIGS = []


def add_cfg(
    family,
    name,
    **kwargs
):

    SEARCH_CONFIGS.append({

        "family":
            family,

        "name":
            name,

        "config":
            kwargs,
    })


# ------------------------------------------------------------
# HGB
# ------------------------------------------------------------

add_cfg(
    "HGB",
    "HGB_A",
    learning_rate=0.04,
    max_iter=450,
    max_depth=5,
    min_samples_leaf=40,
    l2_regularization=1.0,
    pos_weight=1.0,
)


add_cfg(
    "HGB",
    "HGB_B",
    learning_rate=0.04,
    max_iter=500,
    max_depth=6,
    min_samples_leaf=40,
    l2_regularization=2.0,
    pos_weight=2.0,
)


add_cfg(
    "HGB",
    "HGB_C",
    learning_rate=0.03,
    max_iter=650,
    max_depth=6,
    min_samples_leaf=60,
    l2_regularization=3.0,
    pos_weight=3.0,
)


add_cfg(
    "HGB",
    "HGB_D",
    learning_rate=0.05,
    max_iter=400,
    max_depth=7,
    min_samples_leaf=50,
    l2_regularization=5.0,
    pos_weight=4.0,
)


add_cfg(
    "HGB",
    "HGB_E",
    learning_rate=0.03,
    max_iter=650,
    max_depth=5,
    min_samples_leaf=80,
    l2_regularization=5.0,
    pos_weight=5.0,
)


# ------------------------------------------------------------
# CATBOOST
# ------------------------------------------------------------

add_cfg(
    "CatBoost",
    "CAT_A",
    iterations=700,
    learning_rate=0.04,
    depth=5,
    l2_leaf_reg=5.0,
    random_strength=0.5,
    bagging_temperature=0.5,
    pos_weight=1.0,
)


add_cfg(
    "CatBoost",
    "CAT_B",
    iterations=800,
    learning_rate=0.04,
    depth=6,
    l2_leaf_reg=5.0,
    random_strength=0.5,
    bagging_temperature=0.5,
    pos_weight=2.0,
)


add_cfg(
    "CatBoost",
    "CAT_C",
    iterations=900,
    learning_rate=0.035,
    depth=6,
    l2_leaf_reg=8.0,
    random_strength=0.8,
    bagging_temperature=0.7,
    pos_weight=3.0,
)


add_cfg(
    "CatBoost",
    "CAT_D",
    iterations=800,
    learning_rate=0.04,
    depth=7,
    l2_leaf_reg=8.0,
    random_strength=1.0,
    bagging_temperature=0.7,
    pos_weight=4.0,
)


add_cfg(
    "CatBoost",
    "CAT_E",
    iterations=1000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=12.0,
    random_strength=0.5,
    bagging_temperature=1.0,
    pos_weight=5.0,
)


add_cfg(
    "CatBoost",
    "CAT_F",
    iterations=900,
    learning_rate=0.035,
    depth=8,
    l2_leaf_reg=12.0,
    random_strength=1.0,
    bagging_temperature=1.0,
    pos_weight=3.0,
)


# ------------------------------------------------------------
# LIGHTGBM
# ------------------------------------------------------------

add_cfg(
    "LightGBM",
    "LGB_A",
    n_estimators=600,
    learning_rate=0.035,
    num_leaves=24,
    max_depth=5,
    min_child_samples=60,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.2,
    reg_lambda=2.0,
    pos_weight=1.0,
)


add_cfg(
    "LightGBM",
    "LGB_B",
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    min_child_samples=60,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.3,
    reg_lambda=3.0,
    pos_weight=2.0,
)


add_cfg(
    "LightGBM",
    "LGB_C",
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=40,
    max_depth=7,
    min_child_samples=80,
    subsample=0.90,
    colsample_bytree=0.85,
    reg_alpha=0.5,
    reg_lambda=4.0,
    pos_weight=3.0,
)


add_cfg(
    "LightGBM",
    "LGB_D",
    n_estimators=700,
    learning_rate=0.035,
    num_leaves=48,
    max_depth=7,
    min_child_samples=100,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=5.0,
    pos_weight=4.0,
)


add_cfg(
    "LightGBM",
    "LGB_E",
    n_estimators=900,
    learning_rate=0.025,
    num_leaves=31,
    max_depth=6,
    min_child_samples=100,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.5,
    reg_lambda=8.0,
    pos_weight=5.0,
)


add_cfg(
    "LightGBM",
    "LGB_F",
    n_estimators=750,
    learning_rate=0.03,
    num_leaves=20,
    max_depth=5,
    min_child_samples=120,
    subsample=0.90,
    colsample_bytree=0.80,
    reg_alpha=1.0,
    reg_lambda=6.0,
    pos_weight=2.0,
)


# ------------------------------------------------------------
# XGBOOST
# ------------------------------------------------------------

add_cfg(
    "XGBoost",
    "XGB_A",
    n_estimators=600,
    learning_rate=0.035,
    max_depth=4,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.90,
    gamma=0.0,
    reg_alpha=0.1,
    reg_lambda=2.0,
    pos_weight=1.0,
)


add_cfg(
    "XGBoost",
    "XGB_B",
    n_estimators=700,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=8,
    subsample=0.85,
    colsample_bytree=0.90,
    gamma=0.0,
    reg_alpha=0.2,
    reg_lambda=3.0,
    pos_weight=2.0,
)


add_cfg(
    "XGBoost",
    "XGB_C",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0.05,
    reg_alpha=0.3,
    reg_lambda=4.0,
    pos_weight=3.0,
)


add_cfg(
    "XGBoost",
    "XGB_D",
    n_estimators=700,
    learning_rate=0.035,
    max_depth=6,
    min_child_weight=12,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.10,
    reg_alpha=0.5,
    reg_lambda=5.0,
    pos_weight=4.0,
)


add_cfg(
    "XGBoost",
    "XGB_E",
    n_estimators=900,
    learning_rate=0.025,
    max_depth=6,
    min_child_weight=15,
    subsample=0.90,
    colsample_bytree=0.90,
    gamma=0.10,
    reg_alpha=0.5,
    reg_lambda=8.0,
    pos_weight=5.0,
)


add_cfg(
    "XGBoost",
    "XGB_F",
    n_estimators=750,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    subsample=0.90,
    colsample_bytree=0.80,
    gamma=0.05,
    reg_alpha=1.0,
    reg_lambda=5.0,
    pos_weight=2.0,
)


print(
    "\nTotal coarse configs:",
    len(
        SEARCH_CONFIGS
    )
)


# ============================================================
# 18A.7 FIT ONE CONFIG ON SPECIFIED FOLDS
# ============================================================

def evaluate_rescue_config(
    family,
    cfg,
    folds,
    return_oof=False,
    verbose=True,
):

    fold_results = []

    if return_oof:

        oof = np.full(
            len(
                y18
            ),
            np.nan,
            dtype=float
        )

    else:

        oof = None


    for fold_id in folds:

        train_idx = np.where(
            fold_assignment
            !=
            fold_id
        )[0]


        valid_idx = np.where(
            fold_assignment
            ==
            fold_id
        )[0]


        Xtr = (
            X18
            .iloc[
                train_idx
            ]
        )


        Xva = (
            X18
            .iloc[
                valid_idx
            ]
        )


        ytr = (
            y18[
                train_idx
            ]
        )


        yva = (
            y18[
                valid_idx
            ]
        )


        model, pos_weight = (
            make_rescue_model(

                family,

                cfg,

                seed=
                    SEED
                    +
                    18000
                    +
                    fold_id
            )
        )


        fit_start = time.time()


        if family == "HGB":

            sw = hgb_sample_weight(
                ytr,
                pos_weight
            )


            model.fit(
                Xtr,
                ytr,
                sample_weight=sw
            )


        else:

            model.fit(
                Xtr,
                ytr
            )


        p = (
            model
            .predict_proba(
                Xva
            )[:, 1]
        )


        pr = (
            average_precision_score(
                yva,
                p
            )
        )


        roc = (
            roc_auc_score(
                yva,
                p
            )
        )


        runtime = (
            time.time()
            -
            fit_start
        )


        fold_results.append({

            "Fold":
                fold_id,

            "PR_AUC":
                pr,

            "ROC_AUC":
                roc,

            "Runtime_seconds":
                runtime,
        })


        if return_oof:

            oof[
                valid_idx
            ] = p


        if verbose:

            print(
                f"      fold={fold_id} "
                f"PR={pr:.6f} "
                f"ROC={roc:.6f} "
                f"time={runtime:.1f}s"
            )


        del model

        gc.collect()


    df = pd.DataFrame(
        fold_results
    )


    return {

        "mean_PR":
            float(
                df[
                    "PR_AUC"
                ].mean()
            ),

        "sd_PR":
            float(
                df[
                    "PR_AUC"
                ].std(
                    ddof=1
                )
            )
            if len(df) > 1
            else 0.0,

        "mean_ROC":
            float(
                df[
                    "ROC_AUC"
                ].mean()
            ),

        "fold_results":
            df,

        "oof":
            oof,
    }


# ============================================================
# 18A.8 STAGE 1 — COARSE SCREENING
# ============================================================

COARSE_FOLDS = [
    0,
    1,
    2,
]


coarse_rows = []


print(
    "\n"
    + "#" * 135
)

print(
    "STAGE 1 — COARSE PR-AUC SCREENING"
)

print(
    "#" * 135
)


for i, item in enumerate(
    SEARCH_CONFIGS,
    start=1
):

    family = item[
        "family"
    ]

    name = item[
        "name"
    ]

    cfg = item[
        "config"
    ]


    print(
        "\n"
        + "=" * 120
    )

    print(
        f"[{i}/{len(SEARCH_CONFIGS)}] "
        f"{name}"
    )

    print(
        "=" * 120
    )


    result = (
        evaluate_rescue_config(

            family,

            cfg,

            folds=
                COARSE_FOLDS,

            return_oof=
                False,

            verbose=
                True,
        )
    )


    coarse_rows.append({

        "Family":
            family,

        "Config":
            name,

        "Mean_PR_3Fold":
            result[
                "mean_PR"
            ],

        "SD_PR_3Fold":
            result[
                "sd_PR"
            ],

        "Mean_ROC_3Fold":
            result[
                "mean_ROC"
            ],

        "Config_Object":
            copy.deepcopy(
                cfg
            ),
    })


COARSE_RESULTS = pd.DataFrame(
    coarse_rows
)


COARSE_RESULTS = (
    COARSE_RESULTS
    .sort_values(
        "Mean_PR_3Fold",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18A — COARSE SEARCH RANKING"
)

print(
    "=" * 135
)


display(
    COARSE_RESULTS[
        [
            "Family",
            "Config",
            "Mean_PR_3Fold",
            "SD_PR_3Fold",
            "Mean_ROC_3Fold",
        ]
    ]
)


# ============================================================
# 18A.9 SELECT TOP TWO PER FAMILY
# ============================================================

TOP_CONFIGS = []


for family in [
    "HGB",
    "CatBoost",
    "LightGBM",
    "XGBoost",
]:

    family_df = (

        COARSE_RESULTS[
            COARSE_RESULTS[
                "Family"
            ]
            ==
            family
        ]
        .head(2)
    )


    for _, row in (
        family_df.iterrows()
    ):

        TOP_CONFIGS.append({

            "family":
                family,

            "name":
                row[
                    "Config"
                ],

            "config":
                copy.deepcopy(
                    row[
                        "Config_Object"
                    ]
                ),
        })


print(
    "\nSelected for full 5-fold confirmation:"
)


for x in TOP_CONFIGS:

    print(
        " ",
        x[
            "name"
        ]
    )


# ============================================================
# 18A.10 STAGE 2 — FULL 5-FOLD CONFIRMATION
# ============================================================

confirm_rows = []

RESCUE_OOF_BANK = {}

RESCUE_FOLD_BANK = {}


print(
    "\n"
    + "#" * 135
)

print(
    "STAGE 2 — FULL 5-FOLD CONFIRMATION"
)

print(
    "#" * 135
)


for i, item in enumerate(
    TOP_CONFIGS,
    start=1
):

    family = item[
        "family"
    ]

    name = item[
        "name"
    ]

    cfg = item[
        "config"
    ]


    print(
        "\n"
        + "=" * 125
    )

    print(
        f"[{i}/{len(TOP_CONFIGS)}] "
        f"{name} — FULL 5 FOLDS"
    )

    print(
        "=" * 125
    )


    result = (
        evaluate_rescue_config(

            family,

            cfg,

            folds=
                list(
                    range(
                        N_SPLITS
                    )
                ),

            return_oof=
                True,

            verbose=
                True,
        )
    )


    oof = (
        result[
            "oof"
        ]
    )


    if np.any(
        np.isnan(
            oof
        )
    ):

        raise RuntimeError(
            f"Incomplete OOF for {name}"
        )


    pooled_pr = (
        average_precision_score(
            y18,
            oof
        )
    )


    pooled_roc = (
        roc_auc_score(
            y18,
            oof
        )
    )


    RESCUE_OOF_BANK[
        name
    ] = oof


    RESCUE_FOLD_BANK[
        name
    ] = (
        result[
            "fold_results"
        ]
    )


    confirm_rows.append({

        "Family":
            family,

        "Config":
            name,

        "Pooled_PR_AUC":
            pooled_pr,

        "Mean_Fold_PR":
            result[
                "mean_PR"
            ],

        "SD_Fold_PR":
            result[
                "sd_PR"
            ],

        "Pooled_ROC_AUC":
            pooled_roc,

        "Mean_Fold_ROC":
            result[
                "mean_ROC"
            ],
    })


CONFIRMED_RESULTS = pd.DataFrame(
    confirm_rows
)


CONFIRMED_RESULTS = (
    CONFIRMED_RESULTS
    .sort_values(
        "Pooled_PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18A — FULL 5-FOLD DISCRIMINATION RESULTS"
)

print(
    "=" * 135
)


display(
    CONFIRMED_RESULTS
)


# ============================================================
# 18A.11 COMPARE AGAINST CURRENT BACKBONE
# ============================================================

reference_rows = []


if (
    "FINAL_REPAIRED_CORE4_OOF"
    in globals()
):

    reference_rows.append({

        "Reference":
            "Repaired_Core4",

        "PR_AUC":
            average_precision_score(

                y18,

                FINAL_REPAIRED_CORE4_OOF
            ),

        "ROC_AUC":
            roc_auc_score(

                y18,

                FINAL_REPAIRED_CORE4_OOF
            ),
    })


if (
    "FINAL_HGB_RAW_OOF"
    in globals()
):

    reference_rows.append({

        "Reference":
            "Original_HGB",

        "PR_AUC":
            average_precision_score(

                y18,

                FINAL_HGB_RAW_OOF
            ),

        "ROC_AUC":
            roc_auc_score(

                y18,

                FINAL_HGB_RAW_OOF
            ),
    })


if (
    "FINAL_CAT_RAW_OOF"
    in globals()
):

    reference_rows.append({

        "Reference":
            "Original_CatBoost",

        "PR_AUC":
            average_precision_score(

                y18,

                FINAL_CAT_RAW_OOF
            ),

        "ROC_AUC":
            roc_auc_score(

                y18,

                FINAL_CAT_RAW_OOF
            ),
    })


REFERENCE_RESULTS = pd.DataFrame(
    reference_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "CURRENT REFERENCE MODELS"
)

print(
    "=" * 135
)


display(
    REFERENCE_RESULTS
)


# ============================================================
# 18A.12 BEST RESCUE MODEL
# ============================================================

BEST_RESCUE_NAME = (
    CONFIRMED_RESULTS
    .iloc[0][
        "Config"
    ]
)


BEST_RESCUE_PR = float(

    CONFIRMED_RESULTS
    .iloc[0][
        "Pooled_PR_AUC"
    ]
)


BEST_RESCUE_ROC = float(

    CONFIRMED_RESULTS
    .iloc[0][
        "Pooled_ROC_AUC"
    ]
)


BEST_RESCUE_OOF = (
    RESCUE_OOF_BANK[
        BEST_RESCUE_NAME
    ]
)


CURRENT_BEST_REFERENCE = (

    REFERENCE_RESULTS[
        "PR_AUC"
    ].max()

    if len(
        REFERENCE_RESULTS
    ) > 0

    else np.nan
)


print(
    "\n"
    + "=" * 135
)

print(
    "DISCRIMINATION RESCUE SUMMARY"
)

print(
    "=" * 135
)


print(
    "Best rescue model:",
    BEST_RESCUE_NAME
)


print(
    "PR-AUC:",
    BEST_RESCUE_PR
)


print(
    "ROC-AUC:",
    BEST_RESCUE_ROC
)


if np.isfinite(
    CURRENT_BEST_REFERENCE
):

    print(
        "Gain over current best reference:",
        BEST_RESCUE_PR
        -
        CURRENT_BEST_REFERENCE
    )


print(
    "Reached PR >= 0.385:",
    BEST_RESCUE_PR
    >=
    0.385
)


print(
    "Reached PR >= 0.390:",
    BEST_RESCUE_PR
    >=
    0.390
)


# ============================================================
# 18A.13 FOLD-WISE COMPARISON TO OLD HGB
# ============================================================

if (
    "FINAL_HGB_RAW_OOF"
    in globals()
):

    fold_compare_rows = []


    for fold_id in range(
        N_SPLITS
    ):

        idx = np.where(
            fold_assignment
            ==
            fold_id
        )[0]


        rescue_pr = (
            average_precision_score(

                y18[
                    idx
                ],

                BEST_RESCUE_OOF[
                    idx
                ]
            )
        )


        old_pr = (
            average_precision_score(

                y18[
                    idx
                ],

                FINAL_HGB_RAW_OOF[
                    idx
                ]
            )
        )


        fold_compare_rows.append({

            "Fold":
                fold_id,

            "Best_Rescue_PR":
                rescue_pr,

            "Original_HGB_PR":
                old_pr,

            "Delta_PR":
                rescue_pr
                -
                old_pr,
        })


    RESCUE_FOLD_COMPARISON = (
        pd.DataFrame(
            fold_compare_rows
        )
    )


    print(
        "\n"
        + "=" * 135
    )

    print(
        "BEST RESCUE vs ORIGINAL HGB — FOLD CONSISTENCY"
    )

    print(
        "=" * 135
    )


    display(
        RESCUE_FOLD_COMPARISON
    )


    print(
        "\nPositive folds:",
        int(
            (
                RESCUE_FOLD_COMPARISON[
                    "Delta_PR"
                ]
                >
                0
            ).sum()
        ),
        "/",
        N_SPLITS
    )


    print(
        "Mean fold ΔPR:",
        RESCUE_FOLD_COMPARISON[
            "Delta_PR"
        ].mean()
    )


# ============================================================
# 18A.14 SAVE OOF BANK
# ============================================================

oof_save = pd.DataFrame({

    "target":
        y18,

    "fold":
        fold_assignment,
})


for name, pred in (
    RESCUE_OOF_BANK.items()
):

    oof_save[
        name
    ] = pred


save_csv(

    oof_save,

    PRED_DIR
    / "module18a_rescue_oof_bank.csv"
)


save_csv(

    COARSE_RESULTS[
        [
            "Family",
            "Config",
            "Mean_PR_3Fold",
            "SD_PR_3Fold",
            "Mean_ROC_3Fold",
        ]
    ],

    TAB_DIR
    / "module18a_coarse_search.csv"
)


save_csv(

    CONFIRMED_RESULTS,

    TAB_DIR
    / "module18a_confirmed_models.csv"
)


save_csv(

    REFERENCE_RESULTS,

    TAB_DIR
    / "module18a_reference_models.csv"
)


if (
    "RESCUE_FOLD_COMPARISON"
    in globals()
):

    save_csv(

        RESCUE_FOLD_COMPARISON,

        TAB_DIR
        / "module18a_best_vs_hgb_folds.csv"
    )


module18a_manifest = {

    "module":
        "Discrimination Rescue Boosting Search",

    "selection_metric":
        "PR-AUC",

    "coarse_folds":
        COARSE_FOLDS,

    "confirmation_folds":
        5,

    "families": [
        "HistGradientBoosting",
        "CatBoost",
        "LightGBM",
        "XGBoost",
    ],

    "feature_engineering":
        True,

    "class_weight_search":
        True,

    "holdout_used":
        False,

    "minimum_desired_PR":
        0.385,

    "stretch_target_PR":
        0.390,
}


save_json(

    module18a_manifest,

    LOG_DIR
    / "module18a_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nHOLDOUT WAS NOT USED."
)

print(
    "Do not select an ensemble until Module 18B."
)

Original features: 21
Rescue features: 47

Total coarse configs: 23

#######################################################################################################################################
STAGE 1 — COARSE PR-AUC SCREENING
#######################################################################################################################################

[1/23] HGB_A
      fold=0 PR=0.375309 ROC=0.851084 time=20.0s
      fold=1 PR=0.375929 ROC=0.849028 time=20.6s
      fold=2 PR=0.371666 ROC=0.847761 time=20.3s

[2/23] HGB_B
      fold=0 PR=0.374296 ROC=0.850516 time=22.7s
      fold=1 PR=0.376505 ROC=0.848947 time=23.2s
      fold=2 PR=0.371011 ROC=0.847273 time=22.8s

[3/23] HGB_C
      fold=0 PR=0.373893 ROC=0.850701 time=30.0s
      fold=1 PR=0.377317 ROC=0.848953 time=30.3s
      fold=2 PR=0.372855 ROC=0.847500 time=30.0s

[4/23] HGB_D
      fold=0 PR=0.373273 ROC=0.850240 time=18.1s
      fold=1 PR=0.375597 ROC=0.848838 time=18.6s
      fold=2 PR=0.372157 ROC=0

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


      fold=0 PR=0.373020 ROC=0.851740 time=6.0s


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


      fold=1 PR=0.379073 ROC=0.849484 time=5.7s


...


CURRENT REFERENCE MODELS


,Reference,PR_AUC,ROC_AUC
0,Repaired_Core4,0.379284,0.850402
1,Original_HGB,0.377012,0.849628
2,Original_CatBoost,0.377537,0.849216



DISCRIMINATION RESCUE SUMMARY
Best rescue model: XGB_F
PR-AUC: 0.3780283118341419
ROC-AUC: 0.8500316604027663
Gain over current best reference: -0.0012555091760814152
Reached PR >= 0.385: False
Reached PR >= 0.390: False

BEST RESCUE vs ORIGINAL HGB — FOLD CONSISTENCY


,Fold,Best_Rescue_PR,Original_HGB_PR,Delta_PR
0,0,0.376310,0.374987,0.001323
1,1,0.379776,0.382048,-0.002272
2,2,0.376699,0.374669,0.002030
3,3,0.372945,0.370674,0.002270
4,4,0.386901,0.385669,0.001232



Positive folds: 4 / 5
Mean fold ΔPR: 0.0009166407834867663

MODULE 18A COMPLETE

HOLDOUT WAS NOT USED.
Do not select an ensemble until Module 18B.


In [47]:
# ============================================================
# MODULE 18B — NESTED DISCRIMINATION ENSEMBLE RESCUE
#
# PURPOSE
# ------------------------------------------------------------
# Exploit complementary ranking information among:
#
#   HGB
#   CatBoost
#   AutoInt
#   DeepFM
#   XGBoost rescue models
#   LightGBM rescue models
#   CatBoost rescue models
#
#
# STRICT PROTOCOL
# ------------------------------------------------------------
# For each outer fold:
#
#   1. Use the OTHER 4 folds as meta-training data.
#   2. Compare equal-weight subsets.
#   3. Compare:
#          a) calibrated probability blending
#          b) ECDF rank blending
#   4. Refine non-negative weights on meta-training only.
#   5. Apply frozen subset/mode/weights to untouched fold.
#
# Primary metric:
#   PR-AUC
#
# HOLDOUT:
#   NOT TOUCHED.
# ============================================================


import copy
import itertools
import numpy as np
import pandas as pd

from scipy.stats import rankdata

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 18B.1 REQUIREMENTS
# ============================================================

required_18b = [
    "y_dev",
    "fold_assignment",
    "N_SPLITS",

    "FINAL_REPAIRED_EXPERT_OOF_STRICT",

    "RESCUE_OOF_BANK",
    "CONFIRMED_RESULTS",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_18b
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required Module 18B objects: "
        + ", ".join(missing)
    )


y18b = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N18B = len(
    y18b
)


# ============================================================
# 18B.2 BUILD CANDIDATE BANK
#
# Strict repaired matrix order:
#   0 HGB
#   1 CatBoost
#   2 AutoInt
#   3 DeepFM_v2
# ============================================================

strict_core = np.asarray(
    FINAL_REPAIRED_EXPERT_OOF_STRICT,
    dtype=float
)


if strict_core.shape != (
    N18B,
    4
):
    raise RuntimeError(
        "Unexpected strict expert matrix shape: "
        + str(
            strict_core.shape
        )
    )


CANDIDATE_BANK = {

    "HGB":
        strict_core[:, 0],

    "CatBoost":
        strict_core[:, 1],

    "AutoInt":
        strict_core[:, 2],

    "DeepFM":
        strict_core[:, 3],
}


# ------------------------------------------------------------
# Add confirmed rescue models.
#
# Keep the strongest useful models from each family.
# ------------------------------------------------------------

confirmed_sorted = (
    CONFIRMED_RESULTS
    .sort_values(
        "Pooled_PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# top 2 XGB
for name in (
    confirmed_sorted[
        confirmed_sorted[
            "Family"
        ]
        ==
        "XGBoost"
    ][
        "Config"
    ]
    .head(2)
    .tolist()
):

    CANDIDATE_BANK[
        name
    ] = np.asarray(
        RESCUE_OOF_BANK[
            name
        ],
        dtype=float
    )


# top LightGBM
lgb_names = (
    confirmed_sorted[
        confirmed_sorted[
            "Family"
        ]
        ==
        "LightGBM"
    ][
        "Config"
    ]
    .head(1)
    .tolist()
)


for name in lgb_names:

    CANDIDATE_BANK[
        name
    ] = np.asarray(
        RESCUE_OOF_BANK[
            name
        ],
        dtype=float
    )


# top rescue CatBoost
rescue_cat_names = (
    confirmed_sorted[
        confirmed_sorted[
            "Family"
        ]
        ==
        "CatBoost"
    ][
        "Config"
    ]
    .head(1)
    .tolist()
)


for name in rescue_cat_names:

    CANDIDATE_BANK[
        name
    ] = np.asarray(
        RESCUE_OOF_BANK[
            name
        ],
        dtype=float
    )


MODEL_NAMES = list(
    CANDIDATE_BANK.keys()
)


P_ALL = np.column_stack([
    CANDIDATE_BANK[
        name
    ]
    for name in MODEL_NAMES
])


print(
    "Candidate models:"
)

for i, name in enumerate(
    MODEL_NAMES
):
    print(
        f"  {i}: {name}"
    )


print(
    "\nCandidate matrix:",
    P_ALL.shape
)


# ============================================================
# 18B.3 INDIVIDUAL MODEL PERFORMANCE
# ============================================================

individual_rows = []


for j, name in enumerate(
    MODEL_NAMES
):

    individual_rows.append({

        "Model":
            name,

        "PR_AUC":
            average_precision_score(
                y18b,
                P_ALL[:, j]
            ),

        "ROC_AUC":
            roc_auc_score(
                y18b,
                P_ALL[:, j]
            ),
    })


INDIVIDUAL_RESULTS_18B = (
    pd.DataFrame(
        individual_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B — CANDIDATE INDIVIDUAL PERFORMANCE"
)

print(
    "=" * 135
)


display(
    INDIVIDUAL_RESULTS_18B
)


# ============================================================
# 18B.4 PLATT CALIBRATION
#
# Needed because raw rescue models have different score scales.
#
# Calibration is fitted inside each meta-training split only.
# ============================================================

def logit18b(p):

    p = np.clip(
        p,
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p / (1 - p)
    )


def meta_platt_calibrate(
    train_scores,
    valid_scores,
    y_train,
    seed,
):

    train_cal = np.zeros_like(
        train_scores,
        dtype=float
    )

    valid_cal = np.zeros_like(
        valid_scores,
        dtype=float
    )


    for j in range(
        train_scores.shape[1]
    ):

        model = LogisticRegression(
            penalty="l2",
            C=1e6,
            solver="lbfgs",
            max_iter=2000,
            random_state=
                seed + j,
        )


        model.fit(

            logit18b(
                train_scores[:, j]
            )[:, None],

            y_train
        )


        train_cal[:, j] = (
            model
            .predict_proba(

                logit18b(
                    train_scores[:, j]
                )[:, None]

            )[:, 1]
        )


        valid_cal[:, j] = (
            model
            .predict_proba(

                logit18b(
                    valid_scores[:, j]
                )[:, None]

            )[:, 1]
        )


    return (
        train_cal,
        valid_cal
    )


# ============================================================
# 18B.5 STRICT ECDF RANK TRANSFORM
#
# Valid scores are transformed using training-score CDF.
# No valid labels used.
# ============================================================

def ecdf_train_valid(
    train_scores,
    valid_scores
):

    train_rank = np.zeros_like(
        train_scores,
        dtype=float
    )


    valid_rank = np.zeros_like(
        valid_scores,
        dtype=float
    )


    n = len(
        train_scores
    )


    for j in range(
        train_scores.shape[1]
    ):

        # training percentile rank
        train_rank[:, j] = (
            rankdata(
                train_scores[:, j],
                method="average"
            )
            /
            n
        )


        sorted_train = np.sort(
            train_scores[:, j]
        )


        valid_rank[:, j] = (

            np.searchsorted(
                sorted_train,
                valid_scores[:, j],
                side="right"
            )

            /
            n
        )


    return (
        train_rank,
        valid_rank
    )


# ============================================================
# 18B.6 BLEND
# ============================================================

def blend_scores(
    P,
    cols,
    weights
):

    cols = list(
        cols
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    weights = (
        weights
        /
        weights.sum()
    )


    return np.dot(
        P[:, cols],
        weights
    )


# ============================================================
# 18B.7 EQUAL-WEIGHT SUBSET SEARCH
#
# Search subset sizes 2 through 5.
# ============================================================

def search_equal_subsets(
    P_train,
    y_train,
    max_subset_size=5,
):

    M = (
        P_train.shape[1]
    )


    rows = []


    for size in range(
        2,
        min(
            max_subset_size,
            M
        )
        + 1
    ):

        for cols in itertools.combinations(
            range(M),
            size
        ):

            score = np.mean(
                P_train[
                    :,
                    cols
                ],
                axis=1
            )


            pr = (
                average_precision_score(
                    y_train,
                    score
                )
            )


            rows.append({

                "cols":
                    cols,

                "size":
                    size,

                "PR":
                    pr,
            })


    rows = sorted(
        rows,
        key=lambda x: x[
            "PR"
        ],
        reverse=True
    )


    return rows


# ============================================================
# 18B.8 DIRECT PR-AUC WEIGHT REFINEMENT
#
# Coordinate-transfer hill climbing.
#
# Starts at equal weights and moves probability mass from one
# model to another only when meta-training PR-AUC improves.
#
# Non-negative weights and sum(weights)=1 always.
# ============================================================

def refine_ap_weights(
    P,
    y,
    cols,
    initial_weights=None,
    max_rounds=8,
):

    K = len(
        cols
    )


    if initial_weights is None:

        weights = np.ones(
            K,
            dtype=float
        ) / K

    else:

        weights = np.asarray(
            initial_weights,
            dtype=float
        )

        weights = (
            weights
            /
            weights.sum()
        )


    best_score = (
        average_precision_score(

            y,

            blend_scores(
                P,
                cols,
                weights
            )
        )
    )


    step_schedule = [
        0.15,
        0.10,
        0.05,
        0.025,
        0.01,
    ]


    for _ in range(
        max_rounds
    ):

        changed = False


        for step in step_schedule:

            local_best = (
                best_score
            )


            local_weights = (
                weights.copy()
            )


            for donor in range(
                K
            ):

                if weights[
                    donor
                ] < step:

                    continue


                for receiver in range(
                    K
                ):

                    if receiver == donor:
                        continue


                    candidate = (
                        weights.copy()
                    )


                    candidate[
                        donor
                    ] -= step


                    candidate[
                        receiver
                    ] += step


                    score = (
                        average_precision_score(

                            y,

                            blend_scores(
                                P,
                                cols,
                                candidate
                            )
                        )
                    )


                    if (
                        score
                        >
                        local_best
                        +
                        1e-8
                    ):

                        local_best = (
                            score
                        )

                        local_weights = (
                            candidate
                        )


            if (
                local_best
                >
                best_score
                +
                1e-8
            ):

                best_score = (
                    local_best
                )

                weights = (
                    local_weights
                )

                changed = True


        if not changed:
            break


    return (
        weights,
        best_score
    )


# ============================================================
# 18B.9 NESTED OUTER META-EVALUATION
# ============================================================

NESTED_BLEND_OOF = np.zeros(
    N18B,
    dtype=float
)


NESTED_EQUAL_OOF = np.zeros(
    N18B,
    dtype=float
)


fold_rows = []

selection_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"NESTED ENSEMBLE — OUTER FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    train_idx = np.where(
        fold_assignment
        !=
        outer_fold
    )[0]


    valid_idx = np.where(
        fold_assignment
        ==
        outer_fold
    )[0]


    y_train = (
        y18b[
            train_idx
        ]
    )


    y_valid = (
        y18b[
            valid_idx
        ]
    )


    P_train_raw = (
        P_ALL[
            train_idx
        ]
    )


    P_valid_raw = (
        P_ALL[
            valid_idx
        ]
    )


    # --------------------------------------------------------
    # probability representation
    # --------------------------------------------------------

    (
        P_train_prob,
        P_valid_prob

    ) = meta_platt_calibrate(

        P_train_raw,

        P_valid_raw,

        y_train,

        seed=
            SEED
            +
            181000
            +
            outer_fold
    )


    # --------------------------------------------------------
    # rank representation
    # --------------------------------------------------------

    (
        P_train_rank,
        P_valid_rank

    ) = ecdf_train_valid(

        P_train_raw,

        P_valid_raw
    )


    representation_bank = {

        "Probability":
            (
                P_train_prob,
                P_valid_prob
            ),

        "Rank":
            (
                P_train_rank,
                P_valid_rank
            ),
    }


    best_meta = None


    # --------------------------------------------------------
    # search both representations
    # --------------------------------------------------------

    for mode, (
        Ptrain_mode,
        Pvalid_mode

    ) in representation_bank.items():

        subset_results = (
            search_equal_subsets(

                Ptrain_mode,

                y_train,

                max_subset_size=5,
            )
        )


        # Refine the top few subsets only.
        top_subsets = (
            subset_results[
                :8
            ]
        )


        for subset_info in (
            top_subsets
        ):

            cols = (
                subset_info[
                    "cols"
                ]
            )


            equal_weights = (
                np.ones(
                    len(cols)
                )
                /
                len(cols)
            )


            refined_weights, train_pr = (
                refine_ap_weights(

                    Ptrain_mode,

                    y_train,

                    cols,

                    initial_weights=
                        equal_weights,
                )
            )


            if (
                best_meta is None
                or
                train_pr
                >
                best_meta[
                    "train_PR"
                ]
            ):

                best_meta = {

                    "mode":
                        mode,

                    "cols":
                        cols,

                    "weights":
                        refined_weights,

                    "train_PR":
                        train_pr,

                    "equal_train_PR":
                        subset_info[
                            "PR"
                        ],

                    "Pvalid":
                        Pvalid_mode,
                }


    # --------------------------------------------------------
    # frozen selected blend -> outer validation
    # --------------------------------------------------------

    cols = (
        best_meta[
            "cols"
        ]
    )


    weights = (
        best_meta[
            "weights"
        ]
    )


    Pvalid_selected = (
        best_meta[
            "Pvalid"
        ]
    )


    p_valid_blend = (
        blend_scores(

            Pvalid_selected,

            cols,

            weights
        )
    )


    p_valid_equal = (
        np.mean(

            Pvalid_selected[
                :,
                cols
            ],

            axis=1
        )
    )


    NESTED_BLEND_OOF[
        valid_idx
    ] = (
        p_valid_blend
    )


    NESTED_EQUAL_OOF[
        valid_idx
    ] = (
        p_valid_equal
    )


    # --------------------------------------------------------
    # comparisons
    # --------------------------------------------------------

    core4_valid = np.mean(
        strict_core[
            valid_idx
        ],
        axis=1
    )


    hgb_valid = (
        strict_core[
            valid_idx,
            0
        ]
    )


    blend_pr = (
        average_precision_score(
            y_valid,
            p_valid_blend
        )
    )


    equal_pr = (
        average_precision_score(
            y_valid,
            p_valid_equal
        )
    )


    core_pr = (
        average_precision_score(
            y_valid,
            core4_valid
        )
    )


    hgb_pr = (
        average_precision_score(
            y_valid,
            hgb_valid
        )
    )


    blend_roc = (
        roc_auc_score(
            y_valid,
            p_valid_blend
        )
    )


    core_roc = (
        roc_auc_score(
            y_valid,
            core4_valid
        )
    )


    selected_names = [
        MODEL_NAMES[j]
        for j in cols
    ]


    print(
        "Mode:",
        best_meta[
            "mode"
        ]
    )


    print(
        "Subset:",
        selected_names
    )


    print(
        "Weights:",
        np.round(
            weights,
            4
        ).tolist()
    )


    print(
        f"Meta-train PR: "
        f"{best_meta['train_PR']:.6f}"
    )


    print(
        f"Outer blend PR: "
        f"{blend_pr:.6f}"
    )


    print(
        f"Core4 PR: "
        f"{core_pr:.6f}"
    )


    print(
        f"Δ vs Core4: "
        f"{blend_pr - core_pr:+.6f}"
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Mode":
            best_meta[
                "mode"
            ],

        "Meta_Train_PR":
            best_meta[
                "train_PR"
            ],

        "Equal_Outer_PR":
            equal_pr,

        "Optimized_Outer_PR":
            blend_pr,

        "Core4_PR":
            core_pr,

        "HGB_PR":
            hgb_pr,

        "Delta_vs_Core4":
            blend_pr
            -
            core_pr,

        "Delta_vs_HGB":
            blend_pr
            -
            hgb_pr,

        "Optimized_ROC":
            blend_roc,

        "Core4_ROC":
            core_roc,

        "Delta_ROC_vs_Core4":
            blend_roc
            -
            core_roc,
    })


    selection_rows.append({

        "Fold":
            outer_fold,

        "Mode":
            best_meta[
                "mode"
            ],

        "Models":
            " + ".join(
                selected_names
            ),

        "Weights":
            str(
                {
                    name:
                        round(
                            float(w),
                            5
                        )

                    for name, w
                    in zip(
                        selected_names,
                        weights
                    )
                }
            ),
    })


# ============================================================
# 18B.10 GLOBAL STRICT PERFORMANCE
# ============================================================

FOLD_RESULTS_18B = pd.DataFrame(
    fold_rows
)


SELECTION_RESULTS_18B = (
    pd.DataFrame(
        selection_rows
    )
)


CORE4_OOF_18B = np.mean(
    strict_core,
    axis=1
)


FINAL_18B_RESULTS = pd.DataFrame([

    {
        "Model":
            "HGB",

        "PR_AUC":
            average_precision_score(
                y18b,
                strict_core[:, 0]
            ),

        "ROC_AUC":
            roc_auc_score(
                y18b,
                strict_core[:, 0]
            ),
    },

    {
        "Model":
            "Repaired_Core4",

        "PR_AUC":
            average_precision_score(
                y18b,
                CORE4_OOF_18B
            ),

        "ROC_AUC":
            roc_auc_score(
                y18b,
                CORE4_OOF_18B
            ),
    },

    {
        "Model":
            "Nested_Selected_Equal",

        "PR_AUC":
            average_precision_score(
                y18b,
                NESTED_EQUAL_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y18b,
                NESTED_EQUAL_OOF
            ),
    },

    {
        "Model":
            "Nested_Optimized_Blend",

        "PR_AUC":
            average_precision_score(
                y18b,
                NESTED_BLEND_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y18b,
                NESTED_BLEND_OOF
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B — STRICT NESTED ENSEMBLE RESULTS"
)

print(
    "=" * 135
)


display(
    FINAL_18B_RESULTS
)


# ============================================================
# 18B.11 FOLD CONSISTENCY
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B — FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FOLD_RESULTS_18B
)


print(
    "\nPositive vs Core4:"
)

print(
    int(
        (
            FOLD_RESULTS_18B[
                "Delta_vs_Core4"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean ΔPR vs Core4:",
    FOLD_RESULTS_18B[
        "Delta_vs_Core4"
    ].mean()
)


print(
    "\nPositive vs HGB:"
)

print(
    int(
        (
            FOLD_RESULTS_18B[
                "Delta_vs_HGB"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean ΔPR vs HGB:",
    FOLD_RESULTS_18B[
        "Delta_vs_HGB"
    ].mean()
)


# ============================================================
# 18B.12 SELECTION STABILITY
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B — SELECTED SUBSETS / WEIGHTS"
)

print(
    "=" * 135
)


display(
    SELECTION_RESULTS_18B
)


# ============================================================
# 18B.13 MODEL SELECTION FREQUENCY
# ============================================================

freq_rows = []


for name in (
    MODEL_NAMES
):

    count = (
        SELECTION_RESULTS_18B[
            "Models"
        ]
        .str.contains(
            name,
            regex=False
        )
        .sum()
    )


    freq_rows.append({

        "Model":
            name,

        "Selected_Folds":
            int(
                count
            ),

        "Selection_Frequency":
            count
            /
            N_SPLITS,
    })


SELECTION_FREQUENCY_18B = (
    pd.DataFrame(
        freq_rows
    )
    .sort_values(
        [
            "Selected_Folds",
            "Model",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODEL SELECTION FREQUENCY"
)

print(
    "=" * 135
)


display(
    SELECTION_FREQUENCY_18B
)


# ============================================================
# 18B.14 SUCCESS ASSESSMENT
# ============================================================

blend_pr = (
    average_precision_score(
        y18b,
        NESTED_BLEND_OOF
    )
)


core_pr = (
    average_precision_score(
        y18b,
        CORE4_OOF_18B
    )
)


hgb_pr = (
    average_precision_score(
        y18b,
        strict_core[:, 0]
    )
)


ASSESSMENT_18B = pd.DataFrame([{

    "Nested_Blend_PR":
        blend_pr,

    "Core4_PR":
        core_pr,

    "HGB_PR":
        hgb_pr,

    "Delta_vs_Core4":
        blend_pr
        -
        core_pr,

    "Delta_vs_HGB":
        blend_pr
        -
        hgb_pr,

    "Positive_Folds_vs_Core4":
        int(
            (
                FOLD_RESULTS_18B[
                    "Delta_vs_Core4"
                ]
                >
                0
            ).sum()
        ),

    "Positive_Folds_vs_HGB":
        int(
            (
                FOLD_RESULTS_18B[
                    "Delta_vs_HGB"
                ]
                >
                0
            ).sum()
        ),

    "Reached_0.385":
        blend_pr
        >=
        0.385,

    "Reached_0.390":
        blend_pr
        >=
        0.390,
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B — DISCRIMINATION ASSESSMENT"
)

print(
    "=" * 135
)


display(
    ASSESSMENT_18B
)


# ============================================================
# 18B.15 SAVE
# ============================================================

save_csv(
    FINAL_18B_RESULTS,
    TAB_DIR
    / "module18b_nested_results.csv"
)


save_csv(
    FOLD_RESULTS_18B,
    TAB_DIR
    / "module18b_fold_consistency.csv"
)


save_csv(
    SELECTION_RESULTS_18B,
    TAB_DIR
    / "module18b_selected_subsets.csv"
)


save_csv(
    SELECTION_FREQUENCY_18B,
    TAB_DIR
    / "module18b_selection_frequency.csv"
)


save_csv(
    ASSESSMENT_18B,
    TAB_DIR
    / "module18b_assessment.csv"
)


pred18b = pd.DataFrame({

    "target":
        y18b,

    "fold":
        fold_assignment,

    "HGB":
        strict_core[:, 0],

    "Core4":
        CORE4_OOF_18B,

    "Nested_Selected_Equal":
        NESTED_EQUAL_OOF,

    "Nested_Optimized_Blend":
        NESTED_BLEND_OOF,
})


save_csv(
    pred18b,
    PRED_DIR
    / "module18b_nested_blend_oof.csv"
)


manifest18b = {

    "module":
        "Nested discriminative ensemble optimization",

    "primary_metric":
        "PR-AUC",

    "selection":
        "strict outer-fold nested",

    "representations": [
        "Platt-calibrated probability",
        "ECDF rank",
    ],

    "subset_sizes":
        "2 through 5",

    "weight_constraints":
        "nonnegative, sum-to-one",

    "weight_objective":
        "meta-training PR-AUC",

    "holdout_used":
        False,
}


save_json(
    manifest18b,
    LOG_DIR
    / "module18b_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nHOLDOUT WAS NOT USED."
)

Candidate models:
  0: HGB
  1: CatBoost
  2: AutoInt
  3: DeepFM
  4: XGB_F
  5: XGB_A
  6: LGB_F
  7: CAT_A

Candidate matrix: (202944, 8)

MODULE 18B — CANDIDATE INDIVIDUAL PERFORMANCE


,Model,PR_AUC,ROC_AUC
0,XGB_F,0.378028,0.850032
1,CatBoost,0.377444,0.849204
2,XGB_A,0.377148,0.849911
3,HGB,0.377012,0.849628
4,CAT_A,0.376405,0.850013
5,LGB_F,0.376187,0.849501
6,AutoInt,0.374767,0.848331
7,DeepFM,0.372501,0.848072



NESTED ENSEMBLE — OUTER FOLD 1/5
Mode: Probability
Subset: ['HGB', 'CatBoost', 'AutoInt', 'XGB_F', 'CAT_A']
Weights: [0.2, 0.2, 0.21, 0.265, 0.125]
Meta-train PR: 0.380379
Outer blend PR: 0.378282
Core4 PR: 0.378693
Δ vs Core4: -0.000411

NESTED ENSEMBLE — OUTER FOLD 2/5
Mode: Probability
Subset: ['HGB', 'CatBoost', 'AutoInt', 'DeepFM', 'XGB_F']
Weights: [0.15, 0.2, 0.185, 0.065, 0.4]
Meta-train PR: 0.379448
Outer blend PR: 0.382077
Core4 PR: 0.381922
Δ vs Core4: +0.000156

NESTED ENSEMBLE — OUTER FOLD 3/5
Mode: Probability
Subset: ['HGB', 'CatBoost', 'AutoInt', 'DeepFM', 'XGB_F']
Weights: [0.255, 0.175, 0.155, 0.095, 0.32]
Meta-train PR: 0.380342
Outer blend PR: 0.378721
Core4 PR: 0.378879
Δ vs Core4: -0.000158

NESTED ENSEMBLE — OUTER FOLD 4/5
Mode: Probability
Subset: ['HGB', 'CatBoost', 'AutoInt', 'DeepFM', 'XGB_F']
Weights: [0.26, 0.285, 0.15, 0.115, 0.19]
Meta-train PR: 0.381929
Outer blend PR: 0.372766
Core4 PR: 0.372029
Δ vs Core4: +0.000737

NESTED ENSEMBLE — OUTER FOLD 5/5
M

,Model,PR_AUC,ROC_AUC
0,HGB,0.377012,0.849628
1,Repaired_Core4,0.379284,0.850402
2,Nested_Selected_Equal,0.379370,0.850549
3,Nested_Optimized_Blend,0.379486,0.850582



MODULE 18B — FOLD CONSISTENCY


...


MODEL SELECTION FREQUENCY


,Model,Selected_Folds,Selection_Frequency
0,AutoInt,5,1.0
1,CatBoost,5,1.0
2,HGB,5,1.0
3,XGB_F,5,1.0
4,DeepFM,4,0.8
5,CAT_A,1,0.2
6,LGB_F,0,0.0
7,XGB_A,0,0.0



MODULE 18B — DISCRIMINATION ASSESSMENT


,Nested_Blend_PR,Core4_PR,HGB_PR,Delta_vs_Core4,Delta_vs_HGB,Positive_Folds_vs_Core4,Positive_Folds_vs_HGB,Reached_0.385,Reached_0.390
0,0.379486,0.379284,0.377012,0.000202,0.002474,3,5,False,False



MODULE 18B COMPLETE

HOLDOUT WAS NOT USED.


In [48]:
# ============================================================
# MODULE 18C — STRICT RESIDUAL STACKING / HARD-POSITIVE RESCUE
#
# OBJECTIVE
# ------------------------------------------------------------
# Break through the ~0.379 ensemble ceiling by learning
# systematic residual structure missed by the first-stage
# experts.
#
# Inputs:
#   strict OOF expert probabilities
#   best rescue XGB_F
#   original BRFSS predictors
#   engineered Module-18A predictors
#
# Meta learners:
#   XGBoost
#   CatBoost
#
# Variants:
#   standard
#   hard-positive weighted
#
# STRICTNESS:
#   for every outer fold:
#       meta model trained on other 4 folds only
#       untouched outer fold used only for evaluation
#
# HOLDOUT NOT USED.
# ============================================================


import gc
import copy
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# ============================================================
# 18C.1 REQUIREMENTS
# ============================================================

required_18c = [
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",

    "FINAL_REPAIRED_EXPERT_OOF_STRICT",

    "RESCUE_OOF_BANK",

    "build_rescue_features",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_18c
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing required Module 18C objects: "
        + ", ".join(missing)
    )


y18c = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N18C = len(
    y18c
)


# ============================================================
# 18C.2 BASE EXPERT MATRIX
# ============================================================

core = np.asarray(
    FINAL_REPAIRED_EXPERT_OOF_STRICT,
    dtype=float
)


P_HGB = core[:, 0]
P_CAT = core[:, 1]
P_AUTO = core[:, 2]
P_DEEP = core[:, 3]


if "XGB_F" not in RESCUE_OOF_BANK:
    raise RuntimeError(
        "XGB_F predictions missing."
    )


P_XGB = np.asarray(
    RESCUE_OOF_BANK[
        "XGB_F"
    ],
    dtype=float
)


P_BASE = np.column_stack([
    P_HGB,
    P_CAT,
    P_AUTO,
    P_DEEP,
    P_XGB,
])


BASE_NAMES = [
    "HGB",
    "CatBoost",
    "AutoInt",
    "DeepFM",
    "XGB_F",
]


# ============================================================
# 18C.3 FIRST-STAGE SUMMARY FEATURES
# ============================================================

BASE_MEAN = np.mean(
    P_BASE,
    axis=1
)


BASE_STD = np.std(
    P_BASE,
    axis=1
)


BASE_MIN = np.min(
    P_BASE,
    axis=1
)


BASE_MAX = np.max(
    P_BASE,
    axis=1
)


BASE_RANGE = (
    BASE_MAX
    -
    BASE_MIN
)


BASE_MEDIAN = np.median(
    P_BASE,
    axis=1
)


# ============================================================
# 18C.4 ENGINEERED FEATURES
# ============================================================

X_ENGINEERED = (
    build_rescue_features(
        X_dev
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 18C.5 META FEATURE MATRIX
# ============================================================

META_FEATURES = (
    X_ENGINEERED
    .copy()
)


for j, name in enumerate(
    BASE_NAMES
):

    META_FEATURES[
        f"Pred_{name}"
    ] = (
        P_BASE[:, j]
    )


META_FEATURES[
    "Pred_Mean"
] = BASE_MEAN


META_FEATURES[
    "Pred_STD"
] = BASE_STD


META_FEATURES[
    "Pred_Min"
] = BASE_MIN


META_FEATURES[
    "Pred_Max"
] = BASE_MAX


META_FEATURES[
    "Pred_Range"
] = BASE_RANGE


META_FEATURES[
    "Pred_Median"
] = BASE_MEDIAN


# Pairwise disagreement features

META_FEATURES[
    "HGB_minus_CAT"
] = (
    P_HGB
    -
    P_CAT
)


META_FEATURES[
    "HGB_minus_AUTO"
] = (
    P_HGB
    -
    P_AUTO
)


META_FEATURES[
    "CAT_minus_AUTO"
] = (
    P_CAT
    -
    P_AUTO
)


META_FEATURES[
    "XGB_minus_HGB"
] = (
    P_XGB
    -
    P_HGB
)


META_FEATURES[
    "XGB_minus_CAT"
] = (
    P_XGB
    -
    P_CAT
)


META_FEATURES = (
    META_FEATURES
    .astype(
        np.float32
    )
)


print(
    "Meta feature matrix:",
    META_FEATURES.shape
)


# ============================================================
# 18C.6 HARD-POSITIVE WEIGHTS
#
# Low-scored positives receive greater training weight.
#
# NO label information from outer validation is used.
# ============================================================

def hard_case_weights(
    y,
    base_score,
    strength=2.0,
):

    w = np.ones(
        len(y),
        dtype=np.float32
    )


    positive = (
        y == 1
    )


    # Hard positive:
    # lower predicted score -> larger weight
    hardness = (
        1.0
        -
        np.clip(
            base_score,
            0.0,
            1.0
        )
    )


    w[
        positive
    ] = (

        1.0
        +
        strength
        *
        hardness[
            positive
        ]
    )


    return w


# ============================================================
# 18C.7 META MODEL FACTORY
# ============================================================

def make_meta_model(
    family,
    variant,
    seed,
):

    if family == "XGBoost":

        if variant == "A":

            return XGBClassifier(

                objective="binary:logistic",

                n_estimators=700,

                learning_rate=0.025,

                max_depth=3,

                min_child_weight=12,

                subsample=0.90,

                colsample_bytree=0.85,

                gamma=0.05,

                reg_alpha=0.5,

                reg_lambda=8.0,

                eval_metric="aucpr",

                tree_method="hist",

                device="cuda",

                random_state=seed,

                n_jobs=-1,
            )


        elif variant == "B":

            return XGBClassifier(

                objective="binary:logistic",

                n_estimators=900,

                learning_rate=0.02,

                max_depth=4,

                min_child_weight=15,

                subsample=0.90,

                colsample_bytree=0.80,

                gamma=0.10,

                reg_alpha=1.0,

                reg_lambda=10.0,

                eval_metric="aucpr",

                tree_method="hist",

                device="cuda",

                random_state=seed,

                n_jobs=-1,
            )


    elif family == "CatBoost":

        if variant == "A":

            return CatBoostClassifier(

                iterations=700,

                learning_rate=0.035,

                depth=5,

                l2_leaf_reg=8.0,

                random_strength=0.5,

                bagging_temperature=0.5,

                loss_function="Logloss",

                eval_metric="PRAUC",

                random_seed=seed,

                verbose=False,

                task_type="GPU",

                devices="0",
            )


        elif variant == "B":

            return CatBoostClassifier(

                iterations=900,

                learning_rate=0.025,

                depth=6,

                l2_leaf_reg=12.0,

                random_strength=0.8,

                bagging_temperature=0.7,

                loss_function="Logloss",

                eval_metric="PRAUC",

                random_seed=seed,

                verbose=False,

                task_type="GPU",

                devices="0",
            )


    raise ValueError(
        family,
        variant
    )


# ============================================================
# 18C.8 VARIANTS
# ============================================================

META_VARIANTS = [

    {
        "name":
            "XGB_Meta_A",

        "family":
            "XGBoost",

        "variant":
            "A",

        "hard_weight":
            False,

        "strength":
            0.0,
    },

    {
        "name":
            "XGB_Meta_A_Hard",

        "family":
            "XGBoost",

        "variant":
            "A",

        "hard_weight":
            True,

        "strength":
            1.0,
    },

    {
        "name":
            "XGB_Meta_B",

        "family":
            "XGBoost",

        "variant":
            "B",

        "hard_weight":
            False,

        "strength":
            0.0,
    },

    {
        "name":
            "XGB_Meta_B_Hard",

        "family":
            "XGBoost",

        "variant":
            "B",

        "hard_weight":
            True,

        "strength":
            1.5,
    },

    {
        "name":
            "CAT_Meta_A",

        "family":
            "CatBoost",

        "variant":
            "A",

        "hard_weight":
            False,

        "strength":
            0.0,
    },

    {
        "name":
            "CAT_Meta_A_Hard",

        "family":
            "CatBoost",

        "variant":
            "A",

        "hard_weight":
            True,

        "strength":
            1.0,
    },

    {
        "name":
            "CAT_Meta_B",

        "family":
            "CatBoost",

        "variant":
            "B",

        "hard_weight":
            False,

        "strength":
            0.0,
    },

    {
        "name":
            "CAT_Meta_B_Hard",

        "family":
            "CatBoost",

        "variant":
            "B",

        "hard_weight":
            True,

        "strength":
            1.5,
    },
]


# ============================================================
# 18C.9 OUTER-FOLD STRICT META EVALUATION
# ============================================================

META_OOF_BANK = {

    item[
        "name"
    ]:
        np.zeros(
            N18C,
            dtype=float
        )

    for item in META_VARIANTS
}


fold_rows = []


for fold_id in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"MODULE 18C — OUTER FOLD "
        f"{fold_id + 1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    train_idx = np.where(
        fold_assignment
        !=
        fold_id
    )[0]


    valid_idx = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    Xtr = (
        META_FEATURES
        .iloc[
            train_idx
        ]
    )


    Xva = (
        META_FEATURES
        .iloc[
            valid_idx
        ]
    )


    ytr = (
        y18c[
            train_idx
        ]
    )


    yva = (
        y18c[
            valid_idx
        ]
    )


    base_train = (
        BASE_MEAN[
            train_idx
        ]
    )


    for k, item in enumerate(
        META_VARIANTS
    ):

        name = item[
            "name"
        ]


        print(
            "\n",
            name
        )


        model = (
            make_meta_model(

                item[
                    "family"
                ],

                item[
                    "variant"
                ],

                seed=
                    SEED
                    +
                    182000
                    +
                    fold_id * 100
                    +
                    k
            )
        )


        if item[
            "hard_weight"
        ]:

            weights = hard_case_weights(

                ytr,

                base_train,

                strength=
                    item[
                        "strength"
                    ]
            )


            model.fit(

                Xtr,

                ytr,

                sample_weight=
                    weights
            )


        else:

            model.fit(
                Xtr,
                ytr
            )


        p = (
            model
            .predict_proba(
                Xva
            )[:, 1]
        )


        META_OOF_BANK[
            name
        ][
            valid_idx
        ] = p


        pr = (
            average_precision_score(
                yva,
                p
            )
        )


        roc = (
            roc_auc_score(
                yva,
                p
            )
        )


        core_pr = (
            average_precision_score(

                yva,

                BASE_MEAN[
                    valid_idx
                ]
            )
        )


        print(
            f"  PR={pr:.6f} "
            f"Δbase={pr-core_pr:+.6f}"
        )


        fold_rows.append({

            "Fold":
                fold_id,

            "Model":
                name,

            "PR_AUC":
                pr,

            "ROC_AUC":
                roc,

            "Base_PR":
                core_pr,

            "Delta_vs_Base":
                pr
                -
                core_pr,
        })


        del model

        gc.collect()


# ============================================================
# 18C.10 GLOBAL RESULTS
# ============================================================

results = []


# Original repaired Core4
CORE4_18C = np.mean(
    core,
    axis=1
)


results.append({

    "Model":
        "Repaired_Core4",

    "PR_AUC":
        average_precision_score(
            y18c,
            CORE4_18C
        ),

    "ROC_AUC":
        roc_auc_score(
            y18c,
            CORE4_18C
        ),
})


# Core4 + XGB_F equal baseline
CORE5_18C = np.mean(
    P_BASE,
    axis=1
)


results.append({

    "Model":
        "Core5_Equal",

    "PR_AUC":
        average_precision_score(
            y18c,
            CORE5_18C
        ),

    "ROC_AUC":
        roc_auc_score(
            y18c,
            CORE5_18C
        ),
})


for name, p in (
    META_OOF_BANK.items()
):

    results.append({

        "Model":
            name,

        "PR_AUC":
            average_precision_score(
                y18c,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y18c,
                p
            ),
    })


RESULTS_18C = (
    pd.DataFrame(
        results
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18C — STRICT RESIDUAL STACKING RESULTS"
)

print(
    "=" * 135
)


display(
    RESULTS_18C
)


# ============================================================
# 18C.11 BEST MODEL
# ============================================================

BEST_18C_NAME = (
    RESULTS_18C
    .iloc[0][
        "Model"
    ]
)


BEST_18C_PR = float(

    RESULTS_18C
    .iloc[0][
        "PR_AUC"
    ]
)


print(
    "\nBest model:",
    BEST_18C_NAME
)


print(
    "Best PR-AUC:",
    BEST_18C_PR
)


print(
    "Gain vs repaired Core4:",
    BEST_18C_PR
    -
    average_precision_score(
        y18c,
        CORE4_18C
    )
)


# ============================================================
# 18C.12 FOLD CONSISTENCY FOR BEST META MODEL
# ============================================================

if BEST_18C_NAME in (
    META_OOF_BANK
):

    BEST_18C_OOF = (
        META_OOF_BANK[
            BEST_18C_NAME
        ]
    )


else:

    if (
        BEST_18C_NAME
        ==
        "Core5_Equal"
    ):

        BEST_18C_OOF = (
            CORE5_18C
        )

    else:

        BEST_18C_OOF = (
            CORE4_18C
        )


fold_compare = []


for fold_id in range(
    N_SPLITS
):

    idx = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    best_pr = (
        average_precision_score(

            y18c[
                idx
            ],

            BEST_18C_OOF[
                idx
            ]
        )
    )


    core_pr = (
        average_precision_score(

            y18c[
                idx
            ],

            CORE4_18C[
                idx
            ]
        )
    )


    hgb_pr = (
        average_precision_score(

            y18c[
                idx
            ],

            P_HGB[
                idx
            ]
        )
    )


    fold_compare.append({

        "Fold":
            fold_id,

        "Best_PR":
            best_pr,

        "Core4_PR":
            core_pr,

        "HGB_PR":
            hgb_pr,

        "Delta_vs_Core4":
            best_pr
            -
            core_pr,

        "Delta_vs_HGB":
            best_pr
            -
            hgb_pr,
    })


FOLD_COMPARE_18C = pd.DataFrame(
    fold_compare
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18C — BEST MODEL FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FOLD_COMPARE_18C
)


print(
    "\nPositive vs Core4:",
    int(
        (
            FOLD_COMPARE_18C[
                "Delta_vs_Core4"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Positive vs HGB:",
    int(
        (
            FOLD_COMPARE_18C[
                "Delta_vs_HGB"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 18C.13 SUMMARY
# ============================================================

core4_pr = (
    average_precision_score(
        y18c,
        CORE4_18C
    )
)


SUMMARY_18C = pd.DataFrame([{

    "Best_Model":
        BEST_18C_NAME,

    "Best_PR":
        BEST_18C_PR,

    "Core4_PR":
        core4_pr,

    "Delta_vs_Core4":
        BEST_18C_PR
        -
        core4_pr,

    "Positive_Folds_vs_Core4":
        int(
            (
                FOLD_COMPARE_18C[
                    "Delta_vs_Core4"
                ]
                >
                0
            ).sum()
        ),

    "Reached_0.385":
        BEST_18C_PR
        >=
        0.385,

    "Reached_0.390":
        BEST_18C_PR
        >=
        0.390,
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18C — DISCRIMINATION SUMMARY"
)

print(
    "=" * 135
)


display(
    SUMMARY_18C
)


# ============================================================
# 18C.14 SAVE
# ============================================================

save_csv(
    RESULTS_18C,
    TAB_DIR
    / "module18c_residual_results.csv"
)


save_csv(
    pd.DataFrame(
        fold_rows
    ),
    TAB_DIR
    / "module18c_all_fold_results.csv"
)


save_csv(
    FOLD_COMPARE_18C,
    TAB_DIR
    / "module18c_best_fold_consistency.csv"
)


save_csv(
    SUMMARY_18C,
    TAB_DIR
    / "module18c_summary.csv"
)


pred_df = pd.DataFrame({

    "target":
        y18c,

    "fold":
        fold_assignment,

    "Core4":
        CORE4_18C,

    "Core5":
        CORE5_18C,

    "Best_18C":
        BEST_18C_OOF,
})


for name, p in (
    META_OOF_BANK.items()
):

    pred_df[
        name
    ] = p


save_csv(
    pred_df,
    PRED_DIR
    / "module18c_residual_oof.csv"
)


manifest18c = {

    "module":
        "Strict residual stacking and hard-positive rescue",

    "meta_features":
        "clinical engineered features + strict OOF expert predictions",

    "meta_models": [
        "XGBoost",
        "CatBoost",
    ],

    "hard_positive_weighting":
        True,

    "selection_metric":
        "PR-AUC",

    "validation":
        "strict outer-fold",

    "holdout_used":
        False,
}


save_json(
    manifest18c,
    LOG_DIR
    / "module18c_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 18C COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nHOLDOUT WAS NOT USED."
)

Meta feature matrix: (202944, 63)

MODULE 18C — OUTER FOLD 1/5

 XGB_Meta_A
  PR=0.373188 Δbase=-0.005884

 XGB_Meta_A_Hard
  PR=0.372905 Δbase=-0.006167

 XGB_Meta_B
  PR=0.371921 Δbase=-0.007151

 XGB_Meta_B_Hard
  PR=0.372372 Δbase=-0.006700

 CAT_Meta_A


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


  PR=0.371035 Δbase=-0.008036

 CAT_Meta_A_Hard


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


  PR=0.370065 Δbase=-0.009006

 CAT_Meta_B


...


Best model: Core5_Equal
Best PR-AUC: 0.3797507550749909
Gain vs repaired Core4: 0.0004669340647676101

MODULE 18C — BEST MODEL FOLD CONSISTENCY


,Fold,Best_PR,Core4_PR,HGB_PR,Delta_vs_Core4,Delta_vs_HGB
0,0,0.379071,0.378693,0.374987,0.000379,0.004084
1,1,0.382118,0.381922,0.382048,0.000196,0.000071
2,2,0.379130,0.378879,0.374669,0.000251,0.004461
3,3,0.372684,0.372029,0.370674,0.000655,0.002009
4,4,0.389362,0.388951,0.385669,0.000412,0.003693



Positive vs Core4: 5 / 5
Positive vs HGB: 5 / 5

MODULE 18C — DISCRIMINATION SUMMARY


,Best_Model,Best_PR,Core4_PR,Delta_vs_Core4,Positive_Folds_vs_Core4,Reached_0.385,Reached_0.390
0,Core5_Equal,0.379751,0.379284,0.000467,5,False,False



MODULE 18C COMPLETE

HOLDOUT WAS NOT USED.


In [49]:
# ============================================================
# MODULE 19A — FINAL LSA-HE DEVELOPMENT VALIDATION
#
# LSA-HE:
#   Local-Stability-Aware Heterogeneous Ensemble
#
# PREDICTIVE BACKBONE:
#   HGB
#   CatBoost
#   AutoInt
#   DeepFM_v2
#   XGB_F
#
# Aggregation:
#   equal heterogeneous consensus
#
# RELIABILITY:
#   Confidence
#   + expert disagreement
#   + LPNS
#
# Adaptive routing:
#   REMOVED from the primary architecture.
#
# PRIMARY DISEASE ENDPOINT:
#   PR-AUC
#
# PRIMARY RELIABILITY ENDPOINT:
#   Error PR-AUC
#
# HOLDOUT:
#   NOT USED
# ============================================================


import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 19A.1 REQUIREMENTS
# ============================================================

required_19a = [
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",

    "FINAL_REPAIRED_EXPERT_OOF_STRICT",
    "RESCUE_OOF_BANK",

    "FINAL_LPNS_CONFIG",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_19a
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )


if "XGB_F" not in RESCUE_OOF_BANK:

    raise RuntimeError(
        "XGB_F OOF predictions are required."
    )


y19 = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N19 = len(
    y19
)


# ============================================================
# 19A.2 FINAL FIVE-EXPERT MATRIX
# ============================================================

core4 = np.asarray(
    FINAL_REPAIRED_EXPERT_OOF_STRICT,
    dtype=float
)


xgb_f = np.asarray(
    RESCUE_OOF_BANK[
        "XGB_F"
    ],
    dtype=float
)


P5 = np.column_stack([

    core4[:, 0],      # HGB
    core4[:, 1],      # CatBoost
    core4[:, 2],      # AutoInt
    core4[:, 3],      # DeepFM_v2
    xgb_f,            # XGB_F
])


EXPERT_NAMES_19 = [
    "HGB",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
    "XGB_F",
]


if P5.shape != (
    N19,
    5
):

    raise RuntimeError(
        "Unexpected expert matrix shape: "
        + str(
            P5.shape
        )
    )


# ============================================================
# 19A.3 FINAL CONSENSUS PREDICTION
# ============================================================

LSA_HE_OOF = np.mean(
    P5,
    axis=1
)


print(
    "=" * 135
)

print(
    "LSA-HE PREDICTIVE BACKBONE"
)

print(
    "=" * 135
)


print(
    "PR-AUC:",
    average_precision_score(
        y19,
        LSA_HE_OOF
    )
)


print(
    "ROC-AUC:",
    roc_auc_score(
        y19,
        LSA_HE_OOF
    )
)


print(
    "Brier:",
    brier_score_loss(
        y19,
        LSA_HE_OOF
    )
)


print(
    "NLL:",
    log_loss(
        y19,
        LSA_HE_OOF
    )
)


# ============================================================
# 19A.4 CONFIG
# ============================================================

LSA_HE_CONFIG = {

    "reliability_fraction":
        0.30,

    "threshold_fraction":
        0.15,

    "max_reference_size":
        80000,

    "k_neighbors":
        20,

    "max_distance":
        3.0,

    "min_neighbors":
        6,

    "discrete_weight":
        0.65,

    "reliability_C":
        0.10,

    "review_rates": [
        0.05,
        0.10,
        0.15,
        0.20,
    ],
}


# ============================================================
# 19A.5 LPNS VARIABLE GROUPS
# ============================================================

LPNS_BINARY = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]


LPNS_ORDINAL = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]


LPNS_COUNT = [
    "MentHlth",
    "PhysHlth",
]


LPNS_CONTINUOUS = [
    "BMI",
]


LPNS_MULTICLASS = [
    "Diabetes",
]


LPNS_ALL = (
    LPNS_BINARY
    +
    LPNS_ORDINAL
    +
    LPNS_COUNT
    +
    LPNS_CONTINUOUS
    +
    LPNS_MULTICLASS
)


LPNS_SCALE = (
    LPNS_CONTINUOUS
    +
    LPNS_COUNT
    +
    LPNS_ORDINAL
)


LPNS_DISCRETE = (
    LPNS_BINARY
    +
    LPNS_MULTICLASS
)


feature_pos = {
    name: i
    for i, name
    in enumerate(
        LPNS_ALL
    )
}


scale_cols = np.array([
    feature_pos[x]
    for x in LPNS_SCALE
])


discrete_cols = np.array([
    feature_pos[x]
    for x in LPNS_DISCRETE
])


X_LPNS_RAW = (

    X_dev[
        LPNS_ALL
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 19A.6 THRESHOLD SELECTION
# ============================================================

def select_f1_threshold_19(
    y,
    p
):

    thresholds = np.arange(
        0.02,
        0.501,
        0.005
    )


    best_threshold = None
    best_f1 = -1.0


    for threshold in thresholds:

        pred = (
            p
            >=
            threshold
        ).astype(int)


        value = f1_score(
            y,
            pred,
            zero_division=0
        )


        if value > best_f1:

            best_f1 = value

            best_threshold = (
                threshold
            )


    return (
        float(
            best_threshold
        ),
        float(
            best_f1
        ),
    )


# ============================================================
# 19A.7 ENTROPY + TRUST FEATURES
# ============================================================

def binary_entropy_19(
    p
):

    p = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )


    return (

        -p
        *
        np.log(p)

        -

        (1 - p)
        *
        np.log(
            1 - p
        )

    ) / np.log(2)


def build_trust_features_19(
    ensemble_probability,
    experts,
    threshold,
):

    p = (
        ensemble_probability
    )


    entropy = (
        binary_entropy_19(
            p
        )
    )


    signed_margin = (
        p
        -
        threshold
    )


    absolute_margin = np.abs(
        signed_margin
    )


    distance_half = np.abs(
        p - 0.5
    )


    confidence = np.column_stack([

        p,
        entropy,
        signed_margin,
        absolute_margin,
        distance_half,
    ])


    expert_mean = np.mean(
        experts,
        axis=1
    )


    expert_std = np.std(
        experts,
        axis=1
    )


    expert_min = np.min(
        experts,
        axis=1
    )


    expert_max = np.max(
        experts,
        axis=1
    )


    expert_range = (
        expert_max
        -
        expert_min
    )


    expert_median = np.median(
        experts,
        axis=1
    )


    consensus_gap = np.abs(
        p
        -
        expert_mean
    )


    mean_abs_dev = np.mean(

        np.abs(
            experts
            -
            p[:, None]
        ),

        axis=1
    )


    disagreement = np.column_stack([

        confidence,

        expert_mean,
        expert_std,
        expert_min,
        expert_max,
        expert_range,
        expert_median,
        consensus_gap,
        mean_abs_dev,

        experts,
    ])


    return (
        confidence,
        disagreement,
    )


# ============================================================
# 19A.8 LPNS EXTRACTOR
# ============================================================

def extract_lpns_19(
    query_experts,
    reference_experts,
    neighbor_index,
    neighbor_distance,
    threshold,
):

    neighbor_probability = (
        reference_experts[
            neighbor_index
        ]
    )


    valid_neighbor = (
        neighbor_distance
        <=
        LSA_HE_CONFIG[
            "max_distance"
        ]
    )


    neighbor_count = (
        valid_neighbor.sum(
            axis=1
        )
    )


    safe_count = np.maximum(
        neighbor_count,
        1
    )[:, None]


    mask = (
        valid_neighbor[
            :,
            :,
            None
        ]
    )


    query_center = (
        query_experts[
            :,
            None,
            :
        ]
    )


    # --------------------------------------------------------
    # Local absolute deviation
    # --------------------------------------------------------

    local_mad = (

        np.sum(

            np.abs(
                neighbor_probability
                -
                query_center
            )

            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # Local SD
    # --------------------------------------------------------

    local_mean = (

        np.sum(

            neighbor_probability
            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    local_sd = np.sqrt(

        np.sum(

            (
                neighbor_probability
                -
                local_mean[
                    :,
                    None,
                    :
                ]
            ) ** 2

            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # Local decision flips
    # --------------------------------------------------------

    query_decision = (
        query_experts
        >=
        threshold
    )


    neighbor_decision = (
        neighbor_probability
        >=
        threshold
    )


    local_flip = (

        np.sum(

            (
                neighbor_decision
                !=
                query_decision[
                    :,
                    None,
                    :
                ]
            )

            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # Aggregates
    # --------------------------------------------------------

    mean_mad = np.mean(
        local_mad,
        axis=1
    )


    max_mad = np.max(
        local_mad,
        axis=1
    )


    min_mad = np.min(
        local_mad,
        axis=1
    )


    mad_gap = (
        max_mad
        -
        min_mad
    )


    mad_cv = (

        np.std(
            local_mad,
            axis=1
        )

        /
        (
            mean_mad
            +
            1e-8
        )
    )


    mean_sd = np.mean(
        local_sd,
        axis=1
    )


    max_sd = np.max(
        local_sd,
        axis=1
    )


    mean_flip = np.mean(
        local_flip,
        axis=1
    )


    max_flip = np.max(
        local_flip,
        axis=1
    )


    features = np.column_stack([

        mean_mad,
        max_mad,
        min_mad,
        mad_gap,
        mad_cv,

        mean_sd,
        max_sd,

        mean_flip,
        max_flip,

        neighbor_count.astype(
            float
        ),

        local_mad,
        local_sd,
        local_flip,
    ])


    valid_profile = (
        neighbor_count
        >=
        LSA_HE_CONFIG[
            "min_neighbors"
        ]
    )


    features[
        ~valid_profile
    ] = np.nan


    return (
        features,
        valid_profile,
        neighbor_count,
        local_mad,
    )


# ============================================================
# 19A.9 RELIABILITY MODEL
# ============================================================

def make_reliability_model_19():

    return Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "lr",
            LogisticRegression(

                penalty="l2",

                C=
                    LSA_HE_CONFIG[
                        "reliability_C"
                    ],

                class_weight=
                    "balanced",

                solver=
                    "lbfgs",

                max_iter=
                    3000,
            )
        ),
    ])


# ============================================================
# 19A.10 STORAGE
# ============================================================

RISK_CONF_19 = np.zeros(
    N19,
    dtype=float
)


RISK_DIS_19 = np.zeros(
    N19,
    dtype=float
)


RISK_LPNS_19 = np.zeros(
    N19,
    dtype=float
)


ERROR_19 = np.zeros(
    N19,
    dtype=int
)


PRED_19 = np.zeros(
    N19,
    dtype=int
)


THRESHOLD_19 = np.zeros(
    N19,
    dtype=float
)


LPNS_VALID_19 = np.zeros(
    N19,
    dtype=bool
)


LPNS_NEIGHBOR_COUNT_19 = np.zeros(
    N19,
    dtype=int
)


fold_rows = []


# ============================================================
# 19A.11 OUTER-FOLD RELIABILITY VALIDATION
# ============================================================

for fold_id in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"LSA-HE STRICT RELIABILITY — "
        f"FOLD {fold_id + 1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    outer_train = np.where(
        fold_assignment
        !=
        fold_id
    )[0]


    outer_valid = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    # --------------------------------------------------------
    # Split outer training into:
    #
    # gate/reference-development pool
    # reliability fitting pool
    # --------------------------------------------------------

    split_rel = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                LSA_HE_CONFIG[
                    "reliability_fraction"
                ],

            random_state=
                SEED
                +
                190000
                +
                fold_id,
        )
    )


    (
        dev_inside,
        reliability_inside

    ) = next(

        split_rel.split(

            np.zeros(
                len(
                    outer_train
                )
            ),

            y19[
                outer_train
            ]
        )
    )


    development_idx = (
        outer_train[
            dev_inside
        ]
    )


    reliability_idx = (
        outer_train[
            reliability_inside
        ]
    )


    # --------------------------------------------------------
    # Threshold subset
    # --------------------------------------------------------

    split_threshold = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                LSA_HE_CONFIG[
                    "threshold_fraction"
                ],

            random_state=
                SEED
                +
                191000
                +
                fold_id,
        )
    )


    (
        reference_inside,
        threshold_inside

    ) = next(

        split_threshold.split(

            np.zeros(
                len(
                    development_idx
                )
            ),

            y19[
                development_idx
            ]
        )
    )


    reference_pool_idx = (
        development_idx[
            reference_inside
        ]
    )


    threshold_idx = (
        development_idx[
            threshold_inside
        ]
    )


    # --------------------------------------------------------
    # Disease threshold from DEV only
    # --------------------------------------------------------

    fold_threshold, _ = (
        select_f1_threshold_19(

            y19[
                threshold_idx
            ],

            LSA_HE_OOF[
                threshold_idx
            ]
        )
    )


    THRESHOLD_19[
        outer_valid
    ] = (
        fold_threshold
    )


    # --------------------------------------------------------
    # Validation disease prediction
    # --------------------------------------------------------

    validation_prediction = (

        LSA_HE_OOF[
            outer_valid
        ]

        >=

        fold_threshold

    ).astype(int)


    validation_error = (

        validation_prediction

        !=

        y19[
            outer_valid
        ]

    ).astype(int)


    PRED_19[
        outer_valid
    ] = (
        validation_prediction
    )


    ERROR_19[
        outer_valid
    ] = (
        validation_error
    )


    # --------------------------------------------------------
    # Reliability training error target
    # --------------------------------------------------------

    reliability_prediction = (

        LSA_HE_OOF[
            reliability_idx
        ]

        >=

        fold_threshold

    ).astype(int)


    reliability_error = (

        reliability_prediction

        !=

        y19[
            reliability_idx
        ]

    ).astype(int)


    # ========================================================
    # LPNS DISTANCE PREPROCESSOR
    # ========================================================

    distance_scaler = (
        StandardScaler()
    )


    X_reference_distance = (

        X_LPNS_RAW[
            reference_pool_idx
        ]
        .copy()
    )


    X_reference_distance[
        :,
        scale_cols
    ] = (

        distance_scaler
        .fit_transform(

            X_reference_distance[
                :,
                scale_cols
            ]
        )
        .astype(
            np.float32
        )
    )


    X_reference_distance[
        :,
        discrete_cols
    ] *= (
        LSA_HE_CONFIG[
            "discrete_weight"
        ]
    )


    def transform_distance_19(
        X
    ):

        X = (
            X.copy()
            .astype(
                np.float32
            )
        )


        X[
            :,
            scale_cols
        ] = (

            distance_scaler
            .transform(

                X[
                    :,
                    scale_cols
                ]
            )
            .astype(
                np.float32
            )
        )


        X[
            :,
            discrete_cols
        ] *= (
            LSA_HE_CONFIG[
                "discrete_weight"
            ]
        )


        return X


    X_rel_distance = (
        transform_distance_19(

            X_LPNS_RAW[
                reliability_idx
            ]
        )
    )


    X_valid_distance = (
        transform_distance_19(

            X_LPNS_RAW[
                outer_valid
            ]
        )
    )


    # ========================================================
    # REFERENCE BANK
    # ========================================================

    rng = np.random.default_rng(
        SEED
        +
        192000
        +
        fold_id
    )


    reference_selection = np.arange(
        len(
            reference_pool_idx
        )
    )


    if len(
        reference_selection
    ) > LSA_HE_CONFIG[
        "max_reference_size"
    ]:

        reference_selection = (
            rng.choice(

                reference_selection,

                size=
                    LSA_HE_CONFIG[
                        "max_reference_size"
                    ],

                replace=False
            )
        )


    final_reference_idx = (
        reference_pool_idx[
            reference_selection
        ]
    )


    X_reference_final = (
        X_reference_distance[
            reference_selection
        ]
    )


    P_reference_final = (
        P5[
            final_reference_idx
        ]
    )


    # ========================================================
    # NEIGHBORS
    # ========================================================

    nbr_model = (
        NearestNeighbors(

            n_neighbors=
                LSA_HE_CONFIG[
                    "k_neighbors"
                ],

            metric="euclidean",

            n_jobs=-1,
        )
    )


    nbr_model.fit(
        X_reference_final
    )


    (
        rel_distance,
        rel_neighbor

    ) = nbr_model.kneighbors(

        X_rel_distance,

        return_distance=True
    )


    (
        valid_distance,
        valid_neighbor

    ) = nbr_model.kneighbors(

        X_valid_distance,

        return_distance=True
    )


    # ========================================================
    # LPNS FEATURES
    # ========================================================

    (
        lpns_rel,
        lpns_rel_valid,
        rel_neighbor_count,
        rel_local_mad,

    ) = extract_lpns_19(

        P5[
            reliability_idx
        ],

        P_reference_final,

        rel_neighbor,

        rel_distance,

        fold_threshold,
    )


    (
        lpns_valid,
        lpns_valid_profile,
        valid_neighbor_count,
        valid_local_mad,

    ) = extract_lpns_19(

        P5[
            outer_valid
        ],

        P_reference_final,

        valid_neighbor,

        valid_distance,

        fold_threshold,
    )


    LPNS_VALID_19[
        outer_valid
    ] = (
        lpns_valid_profile
    )


    LPNS_NEIGHBOR_COUNT_19[
        outer_valid
    ] = (
        valid_neighbor_count
    )


    # ========================================================
    # TRAIN-ONLY IMPUTATION
    # ========================================================

    medians = np.nanmedian(
        lpns_rel,
        axis=0
    )


    medians = np.where(
        np.isnan(
            medians
        ),
        0.0,
        medians
    )


    def impute_19(
        X
    ):

        X = (
            X.copy()
            .astype(float)
        )


        missing_idx = np.where(
            np.isnan(X)
        )


        X[
            missing_idx
        ] = medians[
            missing_idx[1]
        ]


        return X


    lpns_rel = (
        impute_19(
            lpns_rel
        )
    )


    lpns_valid = (
        impute_19(
            lpns_valid
        )
    )


    # ========================================================
    # CONFIDENCE / DISAGREEMENT FEATURES
    # ========================================================

    (
        confidence_rel,
        disagreement_rel

    ) = build_trust_features_19(

        LSA_HE_OOF[
            reliability_idx
        ],

        P5[
            reliability_idx
        ],

        fold_threshold,
    )


    (
        confidence_valid,
        disagreement_valid

    ) = build_trust_features_19(

        LSA_HE_OOF[
            outer_valid
        ],

        P5[
            outer_valid
        ],

        fold_threshold,
    )


    lpns_features_rel = np.column_stack([

        disagreement_rel,

        lpns_rel,
    ])


    lpns_features_valid = np.column_stack([

        disagreement_valid,

        lpns_valid,
    ])


    # ========================================================
    # RELIABILITY HEADS
    # ========================================================

    confidence_model = (
        make_reliability_model_19()
    )


    disagreement_model = (
        make_reliability_model_19()
    )


    lpns_model = (
        make_reliability_model_19()
    )


    confidence_model.fit(

        confidence_rel,

        reliability_error
    )


    disagreement_model.fit(

        disagreement_rel,

        reliability_error
    )


    lpns_model.fit(

        lpns_features_rel,

        reliability_error
    )


    risk_conf = (

        confidence_model
        .predict_proba(
            confidence_valid
        )[:, 1]
    )


    risk_dis = (

        disagreement_model
        .predict_proba(
            disagreement_valid
        )[:, 1]
    )


    risk_lpns = (

        lpns_model
        .predict_proba(
            lpns_features_valid
        )[:, 1]
    )


    RISK_CONF_19[
        outer_valid
    ] = risk_conf


    RISK_DIS_19[
        outer_valid
    ] = risk_dis


    RISK_LPNS_19[
        outer_valid
    ] = risk_lpns


    # ========================================================
    # FOLD METRICS
    # ========================================================

    conf_pr = (
        average_precision_score(
            validation_error,
            risk_conf
        )
    )


    dis_pr = (
        average_precision_score(
            validation_error,
            risk_dis
        )
    )


    lpns_pr = (
        average_precision_score(
            validation_error,
            risk_lpns
        )
    )


    conf_roc = (
        roc_auc_score(
            validation_error,
            risk_conf
        )
    )


    dis_roc = (
        roc_auc_score(
            validation_error,
            risk_dis
        )
    )


    lpns_roc = (
        roc_auc_score(
            validation_error,
            risk_lpns
        )
    )


    fold_disease_pr = (
        average_precision_score(

            y19[
                outer_valid
            ],

            LSA_HE_OOF[
                outer_valid
            ]
        )
    )


    fold_disease_roc = (
        roc_auc_score(

            y19[
                outer_valid
            ],

            LSA_HE_OOF[
                outer_valid
            ]
        )
    )


    fold_rows.append({

        "Fold":
            fold_id,

        "Disease_PR":
            fold_disease_pr,

        "Disease_ROC":
            fold_disease_roc,

        "Threshold":
            fold_threshold,

        "Error_Prevalence":
            validation_error.mean(),

        "Confidence_PR":
            conf_pr,

        "Disagreement_PR":
            dis_pr,

        "LPNS_PR":
            lpns_pr,

        "LPNS_Delta_PR_vs_Disagreement":
            lpns_pr
            -
            dis_pr,

        "Confidence_ROC":
            conf_roc,

        "Disagreement_ROC":
            dis_roc,

        "LPNS_ROC":
            lpns_roc,

        "LPNS_Delta_ROC_vs_Disagreement":
            lpns_roc
            -
            dis_roc,

        "LPNS_Coverage":
            lpns_valid_profile.mean(),

        "Mean_Neighbors":
            valid_neighbor_count.mean(),
    })


    print(
        f"Disease PR={fold_disease_pr:.6f}"
    )


    print(
        f"Confidence error PR={conf_pr:.6f}"
    )


    print(
        f"Disagreement error PR={dis_pr:.6f}"
    )


    print(
        f"LPNS error PR={lpns_pr:.6f}"
    )


    print(
        "LPNS ΔPR vs disagreement="
        f"{lpns_pr-dis_pr:+.6f}"
    )


# ============================================================
# 19A.12 GLOBAL DISEASE PERFORMANCE
# ============================================================

DISEASE_RESULTS_19 = pd.DataFrame([

    {
        "Model":
            "HGB",

        "PR_AUC":
            average_precision_score(
                y19,
                P5[:, 0]
            ),

        "ROC_AUC":
            roc_auc_score(
                y19,
                P5[:, 0]
            ),
    },

    {
        "Model":
            "Repaired_Core4",

        "PR_AUC":
            average_precision_score(
                y19,
                np.mean(
                    P5[:, :4],
                    axis=1
                )
            ),

        "ROC_AUC":
            roc_auc_score(
                y19,
                np.mean(
                    P5[:, :4],
                    axis=1
                )
            ),
    },

    {
        "Model":
            "LSA_HE_Core5",

        "PR_AUC":
            average_precision_score(
                y19,
                LSA_HE_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y19,
                LSA_HE_OOF
            ),

        "Brier":
            brier_score_loss(
                y19,
                LSA_HE_OOF
            ),

        "NLL":
            log_loss(
                y19,
                LSA_HE_OOF
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A — FINAL LSA-HE DISEASE PERFORMANCE"
)

print(
    "=" * 135
)


display(
    DISEASE_RESULTS_19
)


# ============================================================
# 19A.13 GLOBAL RELIABILITY PERFORMANCE
# ============================================================

def aurc_19(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(
        np.trapezoid(
            selective_risk,
            coverage
        )
    )


RELIABILITY_RESULTS_19 = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            average_precision_score(
                ERROR_19,
                RISK_CONF_19
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                ERROR_19,
                RISK_CONF_19
            ),

        "AURC":
            aurc_19(
                ERROR_19,
                RISK_CONF_19
            ),
    },

    {
        "Method":
            "Confidence+Disagreement",

        "Error_PR_AUC":
            average_precision_score(
                ERROR_19,
                RISK_DIS_19
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                ERROR_19,
                RISK_DIS_19
            ),

        "AURC":
            aurc_19(
                ERROR_19,
                RISK_DIS_19
            ),
    },

    {
        "Method":
            "LSA_HE_LPNS",

        "Error_PR_AUC":
            average_precision_score(
                ERROR_19,
                RISK_LPNS_19
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                ERROR_19,
                RISK_LPNS_19
            ),

        "AURC":
            aurc_19(
                ERROR_19,
                RISK_LPNS_19
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A — FINAL LSA-HE RELIABILITY PERFORMANCE"
)

print(
    "=" * 135
)


display(
    RELIABILITY_RESULTS_19
)


# ============================================================
# 19A.14 INCREMENTAL LPNS VALUE
# ============================================================

baseline_row = (

    RELIABILITY_RESULTS_19[
        RELIABILITY_RESULTS_19[
            "Method"
        ]
        ==
        "Confidence+Disagreement"
    ]
    .iloc[0]
)


lpns_row = (

    RELIABILITY_RESULTS_19[
        RELIABILITY_RESULTS_19[
            "Method"
        ]
        ==
        "LSA_HE_LPNS"
    ]
    .iloc[0]
)


LPNS_GAIN_19 = pd.DataFrame([{

    "Delta_Error_PR_AUC":
        lpns_row[
            "Error_PR_AUC"
        ]
        -
        baseline_row[
            "Error_PR_AUC"
        ],

    "Delta_Error_ROC_AUC":
        lpns_row[
            "Error_ROC_AUC"
        ]
        -
        baseline_row[
            "Error_ROC_AUC"
        ],

    "Delta_AURC":
        lpns_row[
            "AURC"
        ]
        -
        baseline_row[
            "AURC"
        ],
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A — LPNS INCREMENTAL VALUE"
)

print(
    "=" * 135
)


display(
    LPNS_GAIN_19
)


# ============================================================
# 19A.15 FOLD CONSISTENCY
# ============================================================

FOLD_RESULTS_19 = (
    pd.DataFrame(
        fold_rows
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A — FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FOLD_RESULTS_19
)


print(
    "\nLPNS positive PR folds:",
    int(
        (
            FOLD_RESULTS_19[
                "LPNS_Delta_PR_vs_Disagreement"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean LPNS ΔPR:",
    FOLD_RESULTS_19[
        "LPNS_Delta_PR_vs_Disagreement"
    ].mean()
)


# ============================================================
# 19A.16 SELECTIVE REVIEW
# ============================================================

def selective_review_19(
    risk,
    method
):

    rows = []


    total_error = max(
        ERROR_19.sum(),
        1
    )


    base_error_rate = (
        ERROR_19.mean()
    )


    order = np.argsort(
        -risk
    )


    for rate in (
        LSA_HE_CONFIG[
            "review_rates"
        ]
    ):

        n_review = max(
            1,
            int(
                round(
                    N19
                    *
                    rate
                )
            )
        )


        review_idx = (
            order[
                :n_review
            ]
        )


        retained = np.ones(
            N19,
            dtype=bool
        )


        retained[
            review_idx
        ] = False


        reviewed_error_rate = (
            ERROR_19[
                review_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Review_Rate":
                rate,

            "Coverage":
                1 - rate,

            "Error_Enrichment":
                reviewed_error_rate
                /
                base_error_rate,

            "Errors_Captured":
                ERROR_19[
                    review_idx
                ].sum()
                /
                total_error,

            "Error_Per_100_Reviews":
                reviewed_error_rate
                *
                100.0,

            "Retained_Accuracy":
                accuracy_score(

                    y19[
                        retained
                    ],

                    PRED_19[
                        retained
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y19[
                        retained
                    ],

                    PRED_19[
                        retained
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y19[
                        retained
                    ],

                    PRED_19[
                        retained
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


SELECTIVE_RESULTS_19 = pd.concat(

    [

        selective_review_19(
            RISK_CONF_19,
            "Confidence"
        ),

        selective_review_19(
            RISK_DIS_19,
            "Confidence+Disagreement"
        ),

        selective_review_19(
            RISK_LPNS_19,
            "LSA_HE_LPNS"
        ),
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A — SELECTIVE REVIEW"
)

print(
    "=" * 135
)


display(
    SELECTIVE_RESULTS_19
)


# ============================================================
# 19A.17 ARCHITECTURE SUMMARY
# ============================================================

ARCHITECTURE_19 = pd.DataFrame([{

    "Framework":
        "LSA-HE",

    "Predictive_Backbone":
        (
            "HGB + CatBoost + AutoInt + "
            "DeepFM_v2 + XGB_F"
        ),

    "Aggregation":
        "Equal heterogeneous consensus",

    "Reliability_Base":
        "Confidence + Expert Disagreement",

    "Proposed_Reliability":
        (
            "Confidence + Expert Disagreement "
            "+ LPNS"
        ),

    "Adaptive_Routing":
        "Ablation only",

    "Primary_Disease_Metric":
        "PR-AUC",

    "Primary_Reliability_Metric":
        "Error PR-AUC",
}])


print(
    "\n"
    + "=" * 135
)

print(
    "FINAL LSA-HE ARCHITECTURE"
)

print(
    "=" * 135
)


display(
    ARCHITECTURE_19
)


# ============================================================
# 19A.18 SAVE
# ============================================================

save_csv(
    DISEASE_RESULTS_19,
    TAB_DIR
    / "module19a_lsahe_disease.csv"
)


save_csv(
    RELIABILITY_RESULTS_19,
    TAB_DIR
    / "module19a_lsahe_reliability.csv"
)


save_csv(
    LPNS_GAIN_19,
    TAB_DIR
    / "module19a_lsahe_lpns_gain.csv"
)


save_csv(
    FOLD_RESULTS_19,
    TAB_DIR
    / "module19a_lsahe_folds.csv"
)


save_csv(
    SELECTIVE_RESULTS_19,
    TAB_DIR
    / "module19a_lsahe_selective.csv"
)


save_csv(
    ARCHITECTURE_19,
    TAB_DIR
    / "module19a_lsahe_architecture.csv"
)


predictions_19 = pd.DataFrame({

    "target":
        y19,

    "fold":
        fold_assignment,

    "HGB":
        P5[:, 0],

    "CatBoost":
        P5[:, 1],

    "AutoInt":
        P5[:, 2],

    "DeepFM_v2":
        P5[:, 3],

    "XGB_F":
        P5[:, 4],

    "LSA_HE":
        LSA_HE_OOF,

    "Threshold":
        THRESHOLD_19,

    "Prediction":
        PRED_19,

    "Error":
        ERROR_19,

    "Risk_Confidence":
        RISK_CONF_19,

    "Risk_Disagreement":
        RISK_DIS_19,

    "Risk_LPNS":
        RISK_LPNS_19,

    "LPNS_Valid":
        LPNS_VALID_19,

    "LPNS_Neighbor_Count":
        LPNS_NEIGHBOR_COUNT_19,
})


save_csv(
    predictions_19,
    PRED_DIR
    / "module19a_lsahe_oof_predictions.csv"
)


manifest_19 = {

    "framework":
        "LSA-HE",

    "full_name":
        "Local-Stability-Aware Heterogeneous Ensemble",

    "experts": [
        "HistGradientBoosting",
        "CatBoost",
        "AutoInt",
        "DeepFM_v2",
        "XGB_F",
    ],

    "aggregation":
        "equal consensus",

    "adaptive_routing":
        "ablation only",

    "primary_reliability_innovation":
        "Local Prediction Neighborhood Stability",

    "reliability_features": [
        "confidence",
        "expert disagreement",
        "LPNS",
    ],

    "holdout_used":
        False,

    "development_architecture":
        "candidate final",
}


save_json(
    manifest_19,
    LOG_DIR
    / "module19a_lsahe_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 19A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nLSA-HE architecture evaluated on DEV only."
)

print(
    "No HOLDOUT was used."
)

LSA-HE PREDICTIVE BACKBONE
PR-AUC: 0.3797507550749909
ROC-AUC: 0.8506080823100575
Brier: 0.06994018519120405
NLL: 0.23646766541277803

LSA-HE STRICT RELIABILITY — FOLD 1/5
Disease PR=0.379071
Confidence error PR=0.540535
Disagreement error PR=0.540187
LPNS error PR=0.568847
LPNS ΔPR vs disagreement=+0.028660

LSA-HE STRICT RELIABILITY — FOLD 2/5
Disease PR=0.382118
Confidence error PR=0.511542
Disagreement error PR=0.511964
LPNS error PR=0.539251
LPNS ΔPR vs disagreement=+0.027286

LSA-HE STRICT RELIABILITY — FOLD 3/5
Disease PR=0.379130
Confidence error PR=0.538822
Disagreement error PR=0.538431
LPNS error PR=0.568332
LPNS ΔPR vs disagreement=+0.029901

LSA-HE STRICT RELIABILITY — FOLD 4/5
Disease PR=0.372684
Confidence error PR=0.536849
Disagreement error PR=0.536817
LPNS error PR=0.565484
LPNS ΔPR vs disagreement=+0.028667

LSA-HE STRICT RELIABILITY — FOLD 5/5
Disease PR=0.389362
Confidence error PR=0.583341
Disagreement error PR=0.590002
LPNS error PR=0.637605
LPNS ΔPR vs disagreem

,Model,PR_AUC,ROC_AUC,Brier,NLL
0,HGB,0.377012,0.849628,NaN,NaN
1,Repaired_Core4,0.379284,0.850402,NaN,NaN
2,LSA_HE_Core5,0.379751,0.850608,0.06994,0.236468



MODULE 19A — FINAL LSA-HE RELIABILITY PERFORMANCE


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.545534,0.899878,0.026851
1,Confidence+Disagreement,0.545689,0.899927,0.026833
2,LSA_HE_LPNS,0.577716,0.901827,0.026424



MODULE 19A — LPNS INCREMENTAL VALUE


...


LPNS positive PR folds: 5 / 5
Mean LPNS ΔPR: 0.03242342243134884

MODULE 19A — SELECTIVE REVIEW


,Method,Review_Rate,Coverage,Error_Enrichment,Errors_Captured,Error_Per_100_Reviews,Retained_Accuracy,Retained_F1,Retained_MCC
0,Confidence,0.05,0.95,4.294806,0.214736,58.736572,0.886954,0.347670,0.288514
1,Confidence,0.10,0.90,4.636733,0.463664,63.412831,0.918500,0.256666,0.218267
2,Confidence,0.15,0.85,4.558158,0.683733,62.338217,0.949114,0.117434,0.202127
3,Confidence,0.20,0.80,3.787947,0.757593,51.804676,0.958560,0.072512,0.164045
4,Confidence+Disagreement,0.05,0.95,4.325072,0.216249,59.150488,0.887171,0.349744,0.290643
5,Confidence+Disagreement,0.10,0.90,4.658712,0.465862,63.713413,0.918834,0.262034,0.224037
6,Confidence+Disagreement,0.15,0.85,4.548310,0.682255,62.203535,0.948876,0.120825,0.198926
7,Confidence+Disagreement,0.20,0.80,3.786506,0.757305,51.784966,0.958511,0.072432,0.164195
8,LSA_HE_LPNS,0.05,0.95,4.877055,0.243848,66.699517,0.891145,0.389043,0.331052
9,LSA_HE_LPNS,0.10,0.90,4.781215,0.478112,65.388785,0.920695,0.315939,0.279537



FINAL LSA-HE ARCHITECTURE


,Framework,Predictive_Backbone,Aggregation,Reliability_Base,Proposed_Reliability,Adaptive_Routing,Primary_Disease_Metric,Primary_Reliability_Metric
0,LSA-HE,HGB + CatBoost + AutoInt + DeepFM_v2 + XGB_F,Equal heterogeneous consensus,Confidence + Expert Disagreement,Confidence + Expert Disagreement + LPNS,Ablation only,PR-AUC,Error PR-AUC



MODULE 19A COMPLETE

LSA-HE architecture evaluated on DEV only.
No HOLDOUT was used.


In [54]:
# ============================================================
# MODULE 20A — BRFSS-2020 EXTERNAL SCHEMA AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# 1. Load external BRFSS-2020 dataset.
# 2. Inspect target / predictors.
# 3. Compare against BRFSS-2015 predictors.
# 4. Identify:
#       - exact/direct overlaps
#       - semantically mappable variables
#       - 2015-only variables
#       - 2020-only variables
# 5. Build the external-validation feature-alignment registry.
#
# NO MODEL TRAINING.
# NO HOLDOUT / EXTERNAL METRIC EVALUATION.
# ============================================================


import os
import numpy as np
import pandas as pd


# ============================================================
# 20A.1 EXTERNAL PATH
# ============================================================

# ============================================================
# 20A.1 EXTERNAL PATH — ROBUST KAGGLE DISCOVERY
# ============================================================

import os
import glob


TARGET_FILENAME = "heart_2020_cleaned.csv"


# ------------------------------------------------------------
# First try likely Kaggle mount locations
# ------------------------------------------------------------

EXTERNAL_PATH_CANDIDATES = [

    "/kaggle/input/heart-2020-cleaned/heart_2020_cleaned.csv",

    "/kaggle/input/datasets/luyezhang/heart-2020-cleaned/heart_2020_cleaned.csv",

    "/kaggle/input/personal-key-indicators-of-heart-disease/heart_2020_cleaned.csv",

    "/kaggle/input/heart-disease-health-indicators-dataset/heart_2020_cleaned.csv",
]


EXTERNAL_PATH = None


for path in EXTERNAL_PATH_CANDIDATES:

    if os.path.isfile(path):

        EXTERNAL_PATH = path
        break


# ------------------------------------------------------------
# If not found, recursively search ALL mounted Kaggle inputs
# ------------------------------------------------------------

if EXTERNAL_PATH is None:

    matches = glob.glob(
        f"/kaggle/input/**/{TARGET_FILENAME}",
        recursive=True
    )


    if len(matches) > 0:

        print(
            "Found candidate file(s):"
        )

        for i, path in enumerate(matches):

            print(
                f"{i}: {path}"
            )


        # use first exact filename match
        EXTERNAL_PATH = matches[0]


# ------------------------------------------------------------
# Still missing -> inspect mounted inputs
# ------------------------------------------------------------

if EXTERNAL_PATH is None:

    print(
        "\nCould not locate:",
        TARGET_FILENAME
    )


    print(
        "\nMounted Kaggle input directories:"
    )


    if os.path.exists(
        "/kaggle/input"
    ):

        for item in sorted(
            os.listdir(
                "/kaggle/input"
            )
        ):

            print(
                " ",
                item
            )


    raise FileNotFoundError(
        "heart_2020_cleaned.csv is not currently mounted "
        "in this Kaggle notebook."
    )


print(
    "\nExternal dataset found:"
)

print(
    EXTERNAL_PATH
)


print(
    "\nFile exists:",
    os.path.isfile(
        EXTERNAL_PATH
    )
)


print(
    "File size:",
    os.path.getsize(
        EXTERNAL_PATH
    ),
    "bytes"
)

# ============================================================
# 20A.2 LOAD
# ============================================================

EXT20_RAW = pd.read_csv(
    EXTERNAL_PATH
)


print(
    "\nShape:",
    EXT20_RAW.shape
)


print(
    "\nColumns:"
)

for i, col in enumerate(
    EXT20_RAW.columns
):

    print(
        f"{i:02d}: {col}"
    )


# ============================================================
# 20A.3 TARGET
# ============================================================

EXT_TARGET = "HeartDisease"


if EXT_TARGET not in EXT20_RAW.columns:

    raise RuntimeError(
        f"Expected target '{EXT_TARGET}' not found."
    )


print(
    "\nTarget distribution:"
)


display(
    EXT20_RAW[
        EXT_TARGET
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "Count"
    )
    .to_frame()
)


print(
    "\nTarget proportions:"
)


display(
    EXT20_RAW[
        EXT_TARGET
    ]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .rename(
        "Proportion"
    )
    .to_frame()
)


# ============================================================
# 20A.4 EXTERNAL FEATURE LIST
# ============================================================

EXT20_FEATURES = [

    c
    for c in EXT20_RAW.columns
    if c != EXT_TARGET
]


print(
    "\nNumber of external predictors:",
    len(
        EXT20_FEATURES
    )
)


# ============================================================
# 20A.5 INTERNAL 2015 FEATURES
# ============================================================

INT15_FEATURES = [

    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


print(
    "\nInternal predictors:",
    len(
        INT15_FEATURES
    )
)


# ============================================================
# 20A.6 EXPECTED EXTERNAL SCHEMA
# ============================================================

EXPECTED_EXT_FEATURES = [

    "BMI",
    "Smoking",
    "AlcoholDrinking",
    "Stroke",
    "PhysicalHealth",
    "MentalHealth",
    "DiffWalking",
    "Sex",
    "AgeCategory",
    "Race",
    "Diabetic",
    "PhysicalActivity",
    "GenHealth",
    "SleepTime",
    "Asthma",
    "KidneyDisease",
    "SkinCancer",
]


missing_expected = [
    c
    for c in EXPECTED_EXT_FEATURES
    if c not in EXT20_RAW.columns
]


unexpected_external = [
    c
    for c in EXT20_FEATURES
    if c not in EXPECTED_EXT_FEATURES
]


print(
    "\nMissing expected external columns:",
    missing_expected
)


print(
    "Unexpected external columns:",
    unexpected_external
)


# ============================================================
# 20A.7 SEMANTIC ALIGNMENT REGISTRY
#
# Important:
# This does NOT yet claim that all mappings are identical.
#
# Each mapping will be audited for:
#     coding
#     units
#     categories
#     meaning
# ============================================================

ALIGNMENT_REGISTRY = pd.DataFrame([

    {
        "Internal_2015":
            "BMI",

        "External_2020":
            "BMI",

        "Alignment":
            "direct",

        "Status":
            "audit coding",
    },

    {
        "Internal_2015":
            "Smoker",

        "External_2020":
            "Smoking",

        "Alignment":
            "semantic",

        "Status":
            "audit yes/no coding",
    },

    {
        "Internal_2015":
            "Stroke",

        "External_2020":
            "Stroke",

        "Alignment":
            "direct",

        "Status":
            "audit coding",
    },

    {
        "Internal_2015":
            "Diabetes",

        "External_2020":
            "Diabetic",

        "Alignment":
            "semantic",

        "Status":
            "audit category mapping",
    },

    {
        "Internal_2015":
            "PhysActivity",

        "External_2020":
            "PhysicalActivity",

        "Alignment":
            "semantic",

        "Status":
            "audit yes/no coding",
    },

    {
        "Internal_2015":
            "HvyAlcoholConsump",

        "External_2020":
            "AlcoholDrinking",

        "Alignment":
            "partial_semantic",

        "Status":
            "NEEDS CAREFUL DEFINITION CHECK",
    },

    {
        "Internal_2015":
            "GenHlth",

        "External_2020":
            "GenHealth",

        "Alignment":
            "semantic",

        "Status":
            "audit ordinal mapping",
    },

    {
        "Internal_2015":
            "MentHlth",

        "External_2020":
            "MentalHealth",

        "Alignment":
            "semantic",

        "Status":
            "audit units/range",
    },

    {
        "Internal_2015":
            "PhysHlth",

        "External_2020":
            "PhysicalHealth",

        "Alignment":
            "semantic",

        "Status":
            "audit units/range",
    },

    {
        "Internal_2015":
            "DiffWalk",

        "External_2020":
            "DiffWalking",

        "Alignment":
            "semantic",

        "Status":
            "audit yes/no coding",
    },

    {
        "Internal_2015":
            "Sex",

        "External_2020":
            "Sex",

        "Alignment":
            "direct_name",

        "Status":
            "audit coding",
    },

    {
        "Internal_2015":
            "Age",

        "External_2020":
            "AgeCategory",

        "Alignment":
            "semantic",

        "Status":
            "requires category harmonization",
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "PROVISIONAL CROSS-DATASET ALIGNMENT"
)

print(
    "=" * 135
)


display(
    ALIGNMENT_REGISTRY
)


# ============================================================
# 20A.8 INTERNAL-ONLY VARIABLES
# ============================================================

aligned_internal = set(
    ALIGNMENT_REGISTRY[
        "Internal_2015"
    ]
)


INTERNAL_ONLY = [

    c
    for c in INT15_FEATURES
    if c not in aligned_internal
]


# ============================================================
# 20A.9 EXTERNAL-ONLY VARIABLES
# ============================================================

aligned_external = set(
    ALIGNMENT_REGISTRY[
        "External_2020"
    ]
)


EXTERNAL_ONLY = [

    c
    for c in EXT20_FEATURES
    if c not in aligned_external
]


print(
    "\nInternal-only variables:"
)

print(
    INTERNAL_ONLY
)


print(
    "\nExternal-only variables:"
)

print(
    EXTERNAL_ONLY
)


# ============================================================
# 20A.10 DATA TYPES
# ============================================================

dtype_table = pd.DataFrame({

    "Column":
        EXT20_RAW.columns,

    "DType":
        [
            str(
                EXT20_RAW[
                    c
                ].dtype
            )
            for c in EXT20_RAW.columns
        ],

    "Missing":
        [
            int(
                EXT20_RAW[
                    c
                ].isna().sum()
            )
            for c in EXT20_RAW.columns
        ],

    "NUnique":
        [
            int(
                EXT20_RAW[
                    c
                ].nunique(
                    dropna=True
                )
            )
            for c in EXT20_RAW.columns
        ],
})


print(
    "\n"
    + "=" * 135
)

print(
    "EXTERNAL DATA TYPES"
)

print(
    "=" * 135
)


display(
    dtype_table
)


# ============================================================
# 20A.11 CATEGORY AUDIT
# ============================================================

categorical_audit_cols = [

    "HeartDisease",

    "Smoking",
    "AlcoholDrinking",
    "Stroke",
    "DiffWalking",
    "Sex",
    "AgeCategory",
    "Race",
    "Diabetic",
    "PhysicalActivity",
    "GenHealth",
    "Asthma",
    "KidneyDisease",
    "SkinCancer",
]


CATEGORY_AUDIT = {}


for col in categorical_audit_cols:

    if col not in EXT20_RAW.columns:
        continue

    CATEGORY_AUDIT[
        col
    ] = (
        EXT20_RAW[
            col
        ]
        .value_counts(
            dropna=False
        )
        .to_dict()
    )


print(
    "\n"
    + "=" * 135
)

print(
    "EXTERNAL CATEGORY AUDIT"
)

print(
    "=" * 135
)


for col, values in (
    CATEGORY_AUDIT.items()
):

    print(
        "\n",
        col
    )

    print(
        values
    )


# ============================================================
# 20A.12 CONTINUOUS RANGE AUDIT
# ============================================================

continuous_external = [

    "BMI",
    "PhysicalHealth",
    "MentalHealth",
    "SleepTime",
]


continuous_rows = []


for col in continuous_external:

    if col not in EXT20_RAW.columns:
        continue

    s = pd.to_numeric(
        EXT20_RAW[
            col
        ],
        errors="coerce"
    )


    continuous_rows.append({

        "Feature":
            col,

        "N":
            s.notna().sum(),

        "Missing":
            s.isna().sum(),

        "Min":
            s.min(),

        "P01":
            s.quantile(
                0.01
            ),

        "Median":
            s.median(),

        "Mean":
            s.mean(),

        "P99":
            s.quantile(
                0.99
            ),

        "Max":
            s.max(),
    })


CONTINUOUS_AUDIT = pd.DataFrame(
    continuous_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "EXTERNAL CONTINUOUS/RANGE AUDIT"
)

print(
    "=" * 135
)


display(
    CONTINUOUS_AUDIT
)


# ============================================================
# 20A.13 INTERNAL VS EXTERNAL SUMMARY
# ============================================================

ALIGNMENT_SUMMARY = pd.DataFrame([{

    "Internal_N":
        len(
            X_dev
        )
        if "X_dev" in globals()
        else np.nan,

    "Internal_Features":
        len(
            INT15_FEATURES
        ),

    "External_N":
        len(
            EXT20_RAW
        ),

    "External_Features":
        len(
            EXT20_FEATURES
        ),

    "Provisional_Aligned_Features":
        len(
            ALIGNMENT_REGISTRY
        ),

    "Internal_Only_Features":
        len(
            INTERNAL_ONLY
        ),

    "External_Only_Features":
        len(
            EXTERNAL_ONLY
        ),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20A — ALIGNMENT SUMMARY"
)

print(
    "=" * 135
)


display(
    ALIGNMENT_SUMMARY
)


# ============================================================
# 20A.14 SAVE
# ============================================================

if "save_csv" in globals():

    save_csv(
        ALIGNMENT_REGISTRY,
        TAB_DIR
        / "module20a_alignment_registry.csv"
    )


    save_csv(
        dtype_table,
        TAB_DIR
        / "module20a_external_dtypes.csv"
    )


    save_csv(
        CONTINUOUS_AUDIT,
        TAB_DIR
        / "module20a_external_continuous_audit.csv"
    )


    save_csv(
        ALIGNMENT_SUMMARY,
        TAB_DIR
        / "module20a_alignment_summary.csv"
    )


if "save_json" in globals():

    save_json(

        {
            "module":
                "External BRFSS-2020 schema audit",

            "external_path":
                EXTERNAL_PATH,

            "internal_feature_count":
                len(
                    INT15_FEATURES
                ),

            "external_feature_count":
                len(
                    EXT20_FEATURES
                ),

            "provisional_aligned_count":
                len(
                    ALIGNMENT_REGISTRY
                ),

            "internal_only":
                INTERNAL_ONLY,

            "external_only":
                EXTERNAL_ONLY,

            "model_training_performed":
                False,

            "external_metrics_evaluated":
                False,
        },

        LOG_DIR
        / "module20a_manifest.json"
    )


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nNo model was fitted to external data."
)

print(
    "No external performance metric was inspected."
)

Found candidate file(s):
0: /kaggle/input/datasets/youssefislamelrefaie/heart-2020-cleaned/heart_2020_cleaned.csv

External dataset found:
/kaggle/input/datasets/youssefislamelrefaie/heart-2020-cleaned/heart_2020_cleaned.csv

File exists: True
File size: 25189554 bytes

Shape: (319795, 18)

Columns:
00: HeartDisease
01: BMI
02: Smoking
03: AlcoholDrinking
04: Stroke
05: PhysicalHealth
06: MentalHealth
07: DiffWalking
08: Sex
09: AgeCategory
10: Race
11: Diabetic
12: PhysicalActivity
13: GenHealth
14: SleepTime
15: Asthma
16: KidneyDisease
17: SkinCancer

Target distribution:


,Count
HeartDisease,
No,292422
Yes,27373



Target proportions:


,Proportion
HeartDisease,
No,0.914405
Yes,0.085595



Number of external predictors: 17

Internal predictors: 21

Missing expected external columns: []
Unexpected external columns: []

PROVISIONAL CROSS-DATASET ALIGNMENT


...


EXTERNAL CATEGORY AUDIT

 HeartDisease
{'No': 292422, 'Yes': 27373}

 Smoking
{'No': 187887, 'Yes': 131908}

 AlcoholDrinking
{'No': 298018, 'Yes': 21777}

 Stroke
{'No': 307726, 'Yes': 12069}

 DiffWalking
{'No': 275385, 'Yes': 44410}

 Sex
{'Female': 167805, 'Male': 151990}

 AgeCategory
{'65-69': 34151, '60-64': 33686, '70-74': 31065, '55-59': 29757, '50-54': 25382, '80 or older': 24153, '45-49': 21791, '75-79': 21482, '18-24': 21064, '40-44': 21006, '35-39': 20550, '30-34': 18753, '25-29': 16955}

 Race
{'White': 245212, 'Hispanic': 27446, 'Black': 22939, 'Other': 10928, 'Asian': 8068, 'American Indian/Alaskan Native': 5202}

 Diabetic
{'No': 269653, 'Yes': 40802, 'No, borderline diabetes': 6781, 'Yes (during pregnancy)': 2559}

 PhysicalActivity
{'Yes': 247957, 'No': 71838}

 GenHealth
{'Very good': 113858, 'Good': 93129, 'Excellent': 66842, 'Fair': 34677, 'Poor': 11289}

 Asthma
{'No': 276923, 'Yes': 42872}

 KidneyDisease
{'No': 308016, 'Yes': 11779}

 SkinCancer
{'No': 289976,

,Feature,N,Missing,Min,P01,Median,Mean,P99,Max
0,BMI,319795,0,12.02,17.92,27.34,28.325399,48.66,94.85
1,PhysicalHealth,319795,0,0.00,0.00,0.00,3.371710,30.00,30.00
2,MentalHealth,319795,0,0.00,0.00,0.00,3.898366,30.00,30.00
3,SleepTime,319795,0,1.00,3.00,7.00,7.097075,12.00,24.00



MODULE 20A — ALIGNMENT SUMMARY


,Internal_N,Internal_Features,External_N,External_Features,Provisional_Aligned_Features,Internal_Only_Features,External_Only_Features
0,202944,21,319795,17,12,9,5



MODULE 20A COMPLETE

No model was fitted to external data.
No external performance metric was inspected.


In [55]:
# ============================================================
# MODULE 20B — MATCHED-FEATURE INTERNAL CONTROL
#
# PURPOSE
# ------------------------------------------------------------
# Build an internal BRFSS-2015 model using ONLY predictors
# that can be harmonized with BRFSS-2020.
#
# Common feature set (12):
#
#   BMI
#   Smoker
#   Stroke
#   Diabetes
#   PhysActivity
#   HvyAlcoholConsump
#   GenHlth
#   MentHlth
#   PhysHlth
#   DiffWalk
#   Sex
#   Age
#
#
# This experiment answers:
#
#   "How much predictive performance is lost solely because
#    external validation requires feature harmonization?"
#
#
# MODELS
# ------------------------------------------------------------
# HGB
# CatBoost
# XGBoost
#
# We intentionally begin with the strongest reproducible
# classical components because the same representations can
# be transferred cleanly to 2020.
#
# Evaluation:
#   locked 5-fold DEV OOF
#
# HOLDOUT:
#   NOT USED
#
# EXTERNAL:
#   NOT EVALUATED
# ============================================================


import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)

from catboost import CatBoostClassifier

from xgboost import XGBClassifier


# ============================================================
# 20B.1 REQUIREMENTS
# ============================================================

required_20b = [
    "X_dev",
    "y_dev",
    "fold_assignment",
    "N_SPLITS",
    "SEED",

    "LSA_HE_OOF",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_20b
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing required Module 20B objects: "
        + ", ".join(missing)
    )


y20b = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


N20B = len(
    y20b
)


# ============================================================
# 20B.2 COMMON FEATURE SET
# ============================================================

COMMON12 = [

    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "HvyAlcoholConsump",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
]


missing_common = [
    c
    for c in COMMON12
    if c not in X_dev.columns
]


if missing_common:

    raise RuntimeError(
        "Missing common features: "
        + ", ".join(
            missing_common
        )
    )


X20B = (
    X_dev[
        COMMON12
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Matched feature set:",
    COMMON12
)


print(
    "Matched feature count:",
    X20B.shape[1]
)


print(
    "DEV shape:",
    X20B.shape
)


# ============================================================
# 20B.3 FEATURE TYPES
# ============================================================

COMMON12_CATEGORICAL = [

    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "HvyAlcoholConsump",
    "GenHlth",
    "DiffWalk",
    "Sex",
    "Age",
]


COMMON12_NUMERIC = [

    "BMI",
    "MentHlth",
    "PhysHlth",
]


# ============================================================
# 20B.4 MODEL FACTORIES
#
# Use conservative configurations derived from our earlier
# experiments rather than starting another large tuning loop.
# ============================================================

def make_hgb20b(
    fold_id
):

    return HistGradientBoostingClassifier(

        loss="log_loss",

        learning_rate=0.05,

        max_iter=400,

        max_depth=6,

        max_leaf_nodes=31,

        min_samples_leaf=40,

        l2_regularization=1.0,

        early_stopping=False,

        random_state=
            SEED
            +
            200000
            +
            fold_id,
    )


def make_cat20b(
    fold_id
):

    return CatBoostClassifier(

        iterations=700,

        learning_rate=0.04,

        depth=6,

        l2_leaf_reg=5.0,

        random_strength=0.5,

        bagging_temperature=0.5,

        loss_function="Logloss",

        eval_metric="PRAUC",

        random_seed=
            SEED
            +
            201000
            +
            fold_id,

        verbose=False,

        allow_writing_files=False,

        task_type="GPU",

        devices="0",
    )


def make_xgb20b(
    fold_id
):

    return XGBClassifier(

        objective=
            "binary:logistic",

        n_estimators=
            750,

        learning_rate=
            0.03,

        max_depth=
            4,

        min_child_weight=
            10,

        subsample=
            0.90,

        colsample_bytree=
            0.80,

        gamma=
            0.05,

        reg_alpha=
            1.0,

        reg_lambda=
            5.0,

        scale_pos_weight=
            2.0,

        eval_metric=
            "aucpr",

        tree_method=
            "hist",

        device=
            "cuda",

        random_state=
            SEED
            +
            202000
            +
            fold_id,

        n_jobs=-1,
    )


# ============================================================
# 20B.5 STORAGE
# ============================================================

OOF_HGB_12 = np.zeros(
    N20B,
    dtype=float
)


OOF_CAT_12 = np.zeros(
    N20B,
    dtype=float
)


OOF_XGB_12 = np.zeros(
    N20B,
    dtype=float
)


fold_rows = []


# ============================================================
# 20B.6 LOCKED 5-FOLD TRAINING
# ============================================================

for fold_id in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        f"COMMON-12 INTERNAL CONTROL — "
        f"FOLD {fold_id + 1}/{N_SPLITS}"
    )

    print(
        "=" * 135
    )


    train_idx = np.where(
        fold_assignment
        !=
        fold_id
    )[0]


    valid_idx = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    Xtr = (
        X20B
        .iloc[
            train_idx
        ]
    )


    Xva = (
        X20B
        .iloc[
            valid_idx
        ]
    )


    ytr = (
        y20b[
            train_idx
        ]
    )


    yva = (
        y20b[
            valid_idx
        ]
    )


    # ========================================================
    # HGB
    # ========================================================

    hgb = make_hgb20b(
        fold_id
    )


    hgb.fit(
        Xtr,
        ytr
    )


    p_hgb = (
        hgb
        .predict_proba(
            Xva
        )[:, 1]
    )


    OOF_HGB_12[
        valid_idx
    ] = p_hgb


    # ========================================================
    # CATBOOST
    # ========================================================

    cat = make_cat20b(
        fold_id
    )


    cat.fit(
        Xtr,
        ytr
    )


    p_cat = (
        cat
        .predict_proba(
            Xva
        )[:, 1]
    )


    OOF_CAT_12[
        valid_idx
    ] = p_cat


    # ========================================================
    # XGBOOST
    # ========================================================

    xgb = make_xgb20b(
        fold_id
    )


    xgb.fit(
        Xtr,
        ytr
    )


    p_xgb = (
        xgb
        .predict_proba(
            Xva
        )[:, 1]
    )


    OOF_XGB_12[
        valid_idx
    ] = p_xgb


    # ========================================================
    # FOLD METRICS
    # ========================================================

    consensus = np.mean(

        np.column_stack([

            p_hgb,
            p_cat,
            p_xgb,
        ]),

        axis=1
    )


    fold_rows.append({

        "Fold":
            fold_id,

        "HGB_PR":
            average_precision_score(
                yva,
                p_hgb
            ),

        "CatBoost_PR":
            average_precision_score(
                yva,
                p_cat
            ),

        "XGBoost_PR":
            average_precision_score(
                yva,
                p_xgb
            ),

        "Consensus_PR":
            average_precision_score(
                yva,
                consensus
            ),

        "Consensus_ROC":
            roc_auc_score(
                yva,
                consensus
            ),
    })


    print(
        "HGB PR:",
        average_precision_score(
            yva,
            p_hgb
        )
    )


    print(
        "CatBoost PR:",
        average_precision_score(
            yva,
            p_cat
        )
    )


    print(
        "XGBoost PR:",
        average_precision_score(
            yva,
            p_xgb
        )
    )


    print(
        "Consensus PR:",
        average_precision_score(
            yva,
            consensus
        )
    )


    del (
        hgb,
        cat,
        xgb
    )

    gc.collect()


# ============================================================
# 20B.7 COMMON-12 CONSENSUS
# ============================================================

COMMON12_CONSENSUS = np.mean(

    np.column_stack([

        OOF_HGB_12,
        OOF_CAT_12,
        OOF_XGB_12,
    ]),

    axis=1
)


# ============================================================
# 20B.8 GLOBAL RESULTS
# ============================================================

RESULTS_20B = pd.DataFrame([

    {
        "Model":
            "HGB_Common12",

        "Features":
            12,

        "PR_AUC":
            average_precision_score(
                y20b,
                OOF_HGB_12
            ),

        "ROC_AUC":
            roc_auc_score(
                y20b,
                OOF_HGB_12
            ),

        "Brier":
            brier_score_loss(
                y20b,
                OOF_HGB_12
            ),

        "NLL":
            log_loss(
                y20b,
                OOF_HGB_12
            ),
    },

    {
        "Model":
            "CatBoost_Common12",

        "Features":
            12,

        "PR_AUC":
            average_precision_score(
                y20b,
                OOF_CAT_12
            ),

        "ROC_AUC":
            roc_auc_score(
                y20b,
                OOF_CAT_12
            ),

        "Brier":
            brier_score_loss(
                y20b,
                OOF_CAT_12
            ),

        "NLL":
            log_loss(
                y20b,
                OOF_CAT_12
            ),
    },

    {
        "Model":
            "XGBoost_Common12",

        "Features":
            12,

        "PR_AUC":
            average_precision_score(
                y20b,
                OOF_XGB_12
            ),

        "ROC_AUC":
            roc_auc_score(
                y20b,
                OOF_XGB_12
            ),

        "Brier":
            brier_score_loss(
                y20b,
                OOF_XGB_12
            ),

        "NLL":
            log_loss(
                y20b,
                OOF_XGB_12
            ),
    },

    {
        "Model":
            "Common12_Consensus",

        "Features":
            12,

        "PR_AUC":
            average_precision_score(
                y20b,
                COMMON12_CONSENSUS
            ),

        "ROC_AUC":
            roc_auc_score(
                y20b,
                COMMON12_CONSENSUS
            ),

        "Brier":
            brier_score_loss(
                y20b,
                COMMON12_CONSENSUS
            ),

        "NLL":
            log_loss(
                y20b,
                COMMON12_CONSENSUS
            ),
    },

    {
        "Model":
            "Full21_LSA_HE",

        "Features":
            21,

        "PR_AUC":
            average_precision_score(
                y20b,
                LSA_HE_OOF
            ),

        "ROC_AUC":
            roc_auc_score(
                y20b,
                LSA_HE_OOF
            ),

        "Brier":
            brier_score_loss(
                y20b,
                LSA_HE_OOF
            ),

        "NLL":
            log_loss(
                y20b,
                LSA_HE_OOF
            ),
    },
])


RESULTS_20B = (

    RESULTS_20B
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20B — MATCHED-FEATURE INTERNAL CONTROL"
)

print(
    "=" * 135
)


display(
    RESULTS_20B
)


# ============================================================
# 20B.9 FEATURE-ALIGNMENT PENALTY
# ============================================================

FULL21_PR = (
    average_precision_score(
        y20b,
        LSA_HE_OOF
    )
)


COMMON12_PR = (
    average_precision_score(
        y20b,
        COMMON12_CONSENSUS
    )
)


FULL21_ROC = (
    roc_auc_score(
        y20b,
        LSA_HE_OOF
    )
)


COMMON12_ROC = (
    roc_auc_score(
        y20b,
        COMMON12_CONSENSUS
    )
)


ALIGNMENT_PENALTY_20B = pd.DataFrame([{

    "Full21_PR":
        FULL21_PR,

    "Common12_PR":
        COMMON12_PR,

    "Delta_PR":
        COMMON12_PR
        -
        FULL21_PR,

    "Relative_PR_Change_pct":
        100.0
        *
        (
            COMMON12_PR
            -
            FULL21_PR
        )
        /
        FULL21_PR,

    "Full21_ROC":
        FULL21_ROC,

    "Common12_ROC":
        COMMON12_ROC,

    "Delta_ROC":
        COMMON12_ROC
        -
        FULL21_ROC,
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20B — FEATURE ALIGNMENT PENALTY"
)

print(
    "=" * 135
)


display(
    ALIGNMENT_PENALTY_20B
)


# ============================================================
# 20B.10 FOLD CONSISTENCY
# ============================================================

FOLD_RESULTS_20B = pd.DataFrame(
    fold_rows
)


# Full LSA-HE fold performance

full_fold_pr = []


for fold_id in range(
    N_SPLITS
):

    idx = np.where(
        fold_assignment
        ==
        fold_id
    )[0]


    full_fold_pr.append(

        average_precision_score(

            y20b[
                idx
            ],

            LSA_HE_OOF[
                idx
            ]
        )
    )


FOLD_RESULTS_20B[
    "Full21_LSA_HE_PR"
] = (
    full_fold_pr
)


FOLD_RESULTS_20B[
    "Delta_Common12_vs_Full21"
] = (

    FOLD_RESULTS_20B[
        "Consensus_PR"
    ]

    -

    FOLD_RESULTS_20B[
        "Full21_LSA_HE_PR"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20B — FOLD CONSISTENCY"
)

print(
    "=" * 135
)


display(
    FOLD_RESULTS_20B
)


print(
    "\nMean common-12 PR:",
    FOLD_RESULTS_20B[
        "Consensus_PR"
    ].mean()
)


print(
    "Mean full-21 PR:",
    FOLD_RESULTS_20B[
        "Full21_LSA_HE_PR"
    ].mean()
)


print(
    "Mean feature-alignment ΔPR:",
    FOLD_RESULTS_20B[
        "Delta_Common12_vs_Full21"
    ].mean()
)


# ============================================================
# 20B.11 SAVE
# ============================================================

save_csv(
    RESULTS_20B,
    TAB_DIR
    / "module20b_internal_matched_results.csv"
)


save_csv(
    ALIGNMENT_PENALTY_20B,
    TAB_DIR
    / "module20b_alignment_penalty.csv"
)


save_csv(
    FOLD_RESULTS_20B,
    TAB_DIR
    / "module20b_fold_results.csv"
)


pred20b = pd.DataFrame({

    "target":
        y20b,

    "fold":
        fold_assignment,

    "HGB_Common12":
        OOF_HGB_12,

    "CatBoost_Common12":
        OOF_CAT_12,

    "XGBoost_Common12":
        OOF_XGB_12,

    "Common12_Consensus":
        COMMON12_CONSENSUS,

    "Full21_LSA_HE":
        LSA_HE_OOF,
})


save_csv(
    pred20b,
    PRED_DIR
    / "module20b_common12_oof.csv"
)


manifest20b = {

    "module":
        "Matched-feature internal control",

    "internal_dataset":
        "BRFSS 2015",

    "full_feature_count":
        21,

    "matched_feature_count":
        12,

    "matched_features":
        COMMON12,

    "matched_models": [
        "HGB",
        "CatBoost",
        "XGBoost",
    ],

    "matched_aggregation":
        "equal consensus",

    "selection_metric":
        "PR-AUC",

    "external_metrics_used":
        False,

    "holdout_used":
        False,
}


save_json(
    manifest20b,
    LOG_DIR
    / "module20b_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nExternal BRFSS-2020 was NOT evaluated."
)

print(
    "Next: harmonize 2020 and perform frozen external validation."
)

Matched feature set: ['BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age']
Matched feature count: 12
DEV shape: (202944, 12)

COMMON-12 INTERNAL CONTROL — FOLD 1/5


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


HGB PR: 0.34842895469182433
CatBoost PR: 0.35007916650541737
XGBoost PR: 0.3535583620679694
Consensus PR: 0.3514565623856733

COMMON-12 INTERNAL CONTROL — FOLD 2/5


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


HGB PR: 0.354141484354946
CatBoost PR: 0.3555619667456696
XGBoost PR: 0.3590577335316014
Consensus PR: 0.3573991437779913

COMMON-12 INTERNAL CONTROL — FOLD 3/5


...


MODULE 20B — FEATURE ALIGNMENT PENALTY


,Full21_PR,Common12_PR,Delta_PR,Relative_PR_Change_pct,Full21_ROC,Common12_ROC,Delta_ROC
0,0.379751,0.356374,-0.023376,-6.155712,0.850608,0.837299,-0.013309



MODULE 20B — FOLD CONSISTENCY


,Fold,HGB_PR,CatBoost_PR,XGBoost_PR,Consensus_PR,Consensus_ROC,Full21_LSA_HE_PR,Delta_Common12_vs_Full21
0,0,0.348429,0.350079,0.353558,0.351457,0.837703,0.379071,-0.027615
1,1,0.354141,0.355562,0.359058,0.357399,0.837285,0.382118,-0.024719
2,2,0.351926,0.353431,0.357596,0.355711,0.837037,0.379130,-0.023419
3,3,0.350709,0.352626,0.355039,0.353612,0.836300,0.372684,-0.019072
4,4,0.363857,0.365456,0.366338,0.366372,0.838307,0.389362,-0.022990



Mean common-12 PR: 0.3569100769487334
Mean full-21 PR: 0.38047317312861134
Mean feature-alignment ΔPR: -0.023563096179877917

MODULE 20B COMPLETE

External BRFSS-2020 was NOT evaluated.
Next: harmonize 2020 and perform frozen external validation.


In [56]:
# ============================================================
# MODULE 20C — FROZEN BRFSS-2020 EXTERNAL VALIDATION
#
# PURPOSE
# ------------------------------------------------------------
# Train the previously defined Common12 models on ALL 2015 DEV
# and evaluate them ONCE on harmonized BRFSS-2020.
#
# TRAINING:
#   BRFSS 2015 DEV only
#
# MODEL SELECTION:
#   already completed in Module 20B
#
# CALIBRATION:
#   fitted on 2015 OOF predictions only
#
# THRESHOLD:
#   selected on 2015 OOF predictions only
#
# EXTERNAL LABELS:
#   evaluation only
#
# NO tuning is permitted after inspecting the 2020 results.
# ============================================================


import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)

from catboost import CatBoostClassifier
from xgboost import XGBClassifier


# ============================================================
# 20C.1 REQUIREMENTS
# ============================================================

required_20c = [
    "X_dev",
    "y_dev",

    "EXT20_RAW",

    "COMMON12",

    "OOF_HGB_12",
    "OOF_CAT_12",
    "OOF_XGB_12",
    "COMMON12_CONSENSUS",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_20c
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing required Module 20C objects: "
        + ", ".join(missing)
    )


y15 = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 20C.2 HARMONIZATION MAP
#
# BRFSS-2015 coding assumed from the processed 2015 dataset:
#
# Sex:
#   Female=0
#   Male=1
#
# GenHlth:
#   Excellent=1
#   Very good=2
#   Good=3
#   Fair=4
#   Poor=5
#
# Age:
#   1 = 18-24
#   ...
#   13 = 80+
#
# Diabetes:
#   0 = No
#   1 = borderline / pregnancy
#   2 = Yes
# ============================================================

YES_NO = {
    "No": 0.0,
    "Yes": 1.0,
}


SEX_MAP = {
    "Female": 0.0,
    "Male": 1.0,
}


GEN_HEALTH_MAP = {
    "Excellent": 1.0,
    "Very good": 2.0,
    "Good": 3.0,
    "Fair": 4.0,
    "Poor": 5.0,
}


AGE_MAP = {
    "18-24": 1.0,
    "25-29": 2.0,
    "30-34": 3.0,
    "35-39": 4.0,
    "40-44": 5.0,
    "45-49": 6.0,
    "50-54": 7.0,
    "55-59": 8.0,
    "60-64": 9.0,
    "65-69": 10.0,
    "70-74": 11.0,
    "75-79": 12.0,
    "80 or older": 13.0,
}


DIABETES_MAP = {
    "No": 0.0,
    "No, borderline diabetes": 1.0,
    "Yes (during pregnancy)": 1.0,
    "Yes": 2.0,
}


# ============================================================
# 20C.3 BUILD HARMONIZED 2020 MATRIX
# ============================================================

X20 = pd.DataFrame(
    index=EXT20_RAW.index
)


X20["BMI"] = (
    pd.to_numeric(
        EXT20_RAW["BMI"],
        errors="coerce"
    )
)


X20["Smoker"] = (
    EXT20_RAW["Smoking"]
    .map(YES_NO)
)


X20["Stroke"] = (
    EXT20_RAW["Stroke"]
    .map(YES_NO)
)


X20["Diabetes"] = (
    EXT20_RAW["Diabetic"]
    .map(DIABETES_MAP)
)


X20["PhysActivity"] = (
    EXT20_RAW["PhysicalActivity"]
    .map(YES_NO)
)


X20["HvyAlcoholConsump"] = (
    EXT20_RAW["AlcoholDrinking"]
    .map(YES_NO)
)


X20["GenHlth"] = (
    EXT20_RAW["GenHealth"]
    .map(GEN_HEALTH_MAP)
)


X20["MentHlth"] = (
    pd.to_numeric(
        EXT20_RAW["MentalHealth"],
        errors="coerce"
    )
)


X20["PhysHlth"] = (
    pd.to_numeric(
        EXT20_RAW["PhysicalHealth"],
        errors="coerce"
    )
)


X20["DiffWalk"] = (
    EXT20_RAW["DiffWalking"]
    .map(YES_NO)
)


X20["Sex"] = (
    EXT20_RAW["Sex"]
    .map(SEX_MAP)
)


X20["Age"] = (
    EXT20_RAW["AgeCategory"]
    .map(AGE_MAP)
)


# enforce exact order
X20 = (
    X20[
        COMMON12
    ]
    .astype(
        np.float32
    )
)


y20 = (
    EXT20_RAW["HeartDisease"]
    .map({
        "No": 0,
        "Yes": 1,
    })
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 20C.4 HARMONIZATION VALIDATION
# ============================================================

if X20.isna().any().any():

    missing_table = (
        X20
        .isna()
        .sum()
    )

    missing_table = (
        missing_table[
            missing_table > 0
        ]
    )

    print(
        missing_table
    )

    raise RuntimeError(
        "Harmonization produced missing values."
    )


if np.isnan(y20).any():

    raise RuntimeError(
        "External target mapping failed."
    )


print(
    "=" * 135
)

print(
    "MODULE 20C — HARMONIZED EXTERNAL DATA"
)

print(
    "=" * 135
)


print(
    "External shape:",
    X20.shape
)


print(
    "External prevalence:",
    y20.mean()
)


print(
    "\nFeature order:"
)

print(
    list(
        X20.columns
    )
)


# ============================================================
# 20C.5 INTERNAL / EXTERNAL DISTRIBUTION AUDIT
# ============================================================

distribution_rows = []


for feature in COMMON12:

    internal = pd.to_numeric(
        X_dev[feature],
        errors="coerce"
    )


    external = pd.to_numeric(
        X20[feature],
        errors="coerce"
    )


    distribution_rows.append({

        "Feature":
            feature,

        "Internal_Mean":
            internal.mean(),

        "External_Mean":
            external.mean(),

        "Internal_SD":
            internal.std(),

        "External_SD":
            external.std(),

        "Internal_Min":
            internal.min(),

        "External_Min":
            external.min(),

        "Internal_Max":
            internal.max(),

        "External_Max":
            external.max(),
    })


DISTRIBUTION_20C = pd.DataFrame(
    distribution_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C — HARMONIZED FEATURE DISTRIBUTIONS"
)

print(
    "=" * 135
)


display(
    DISTRIBUTION_20C
)


# ============================================================
# 20C.6 LOCK INTERNAL OOF CALIBRATOR
#
# Fit one Platt calibrator to the Common12 consensus using only
# BRFSS-2015 OOF predictions.
# ============================================================

def logit20c(
    p
):

    p = np.clip(
        p,
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p
        /
        (1 - p)
    )


COMMON12_PLATT = LogisticRegression(

    penalty="l2",

    C=1e6,

    solver="lbfgs",

    max_iter=2000,

    random_state=
        SEED + 203000,
)


COMMON12_PLATT.fit(

    logit20c(
        COMMON12_CONSENSUS
    )[:, None],

    y15
)


COMMON12_INTERNAL_CAL = (
    COMMON12_PLATT
    .predict_proba(

        logit20c(
            COMMON12_CONSENSUS
        )[:, None]

    )[:, 1]
)


print(
    "\nInternal calibrated mean probability:",
    COMMON12_INTERNAL_CAL.mean()
)


print(
    "Internal prevalence:",
    y15.mean()
)


# ============================================================
# 20C.7 LOCK DISEASE THRESHOLD ON 2015 OOF
# ============================================================

threshold_grid = np.arange(
    0.02,
    0.501,
    0.005
)


best_threshold = None
best_f1 = -np.inf


for threshold in threshold_grid:

    pred = (
        COMMON12_INTERNAL_CAL
        >=
        threshold
    ).astype(int)


    value = (
        f1_score(
            y15,
            pred,
            zero_division=0
        )
    )


    if value > best_f1:

        best_f1 = value

        best_threshold = float(
            threshold
        )


LOCKED_COMMON12_THRESHOLD = (
    best_threshold
)


print(
    "\nLocked Common12 threshold:",
    LOCKED_COMMON12_THRESHOLD
)


print(
    "Internal threshold F1:",
    best_f1
)


# ============================================================
# 20C.8 TRAIN FROZEN MODELS ON ALL 2015 DEV
# ============================================================

X15 = (
    X_dev[
        COMMON12
    ]
    .copy()
    .astype(
        np.float32
    )
)


# ------------------------------------------------------------
# HGB
# ------------------------------------------------------------

HGB_COMMON12_FINAL = (
    HistGradientBoostingClassifier(

        loss=
            "log_loss",

        learning_rate=
            0.05,

        max_iter=
            400,

        max_depth=
            6,

        max_leaf_nodes=
            31,

        min_samples_leaf=
            40,

        l2_regularization=
            1.0,

        early_stopping=
            False,

        random_state=
            SEED + 204000,
    )
)


HGB_COMMON12_FINAL.fit(
    X15,
    y15
)


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

CAT_COMMON12_FINAL = (
    CatBoostClassifier(

        iterations=
            700,

        learning_rate=
            0.04,

        depth=
            6,

        l2_leaf_reg=
            5.0,

        random_strength=
            0.5,

        bagging_temperature=
            0.5,

        loss_function=
            "Logloss",

        eval_metric=
            "PRAUC",

        random_seed=
            SEED + 205000,

        verbose=
            False,

        allow_writing_files=
            False,

        task_type=
            "GPU",

        devices=
            "0",
    )
)


CAT_COMMON12_FINAL.fit(
    X15,
    y15
)


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

XGB_COMMON12_FINAL = (
    XGBClassifier(

        objective=
            "binary:logistic",

        n_estimators=
            750,

        learning_rate=
            0.03,

        max_depth=
            4,

        min_child_weight=
            10,

        subsample=
            0.90,

        colsample_bytree=
            0.80,

        gamma=
            0.05,

        reg_alpha=
            1.0,

        reg_lambda=
            5.0,

        scale_pos_weight=
            2.0,

        eval_metric=
            "aucpr",

        tree_method=
            "hist",

        device=
            "cuda",

        random_state=
            SEED + 206000,

        n_jobs=
            -1,
    )
)


XGB_COMMON12_FINAL.fit(
    X15,
    y15
)


# ============================================================
# 20C.9 EXTERNAL PREDICTIONS — FIRST OPENING
# ============================================================

P20_HGB = (
    HGB_COMMON12_FINAL
    .predict_proba(
        X20
    )[:, 1]
)


P20_CAT = (
    CAT_COMMON12_FINAL
    .predict_proba(
        X20
    )[:, 1]
)


P20_XGB = (
    XGB_COMMON12_FINAL
    .predict_proba(
        X20
    )[:, 1]
)


P20_CONSENSUS_RAW = np.mean(

    np.column_stack([

        P20_HGB,
        P20_CAT,
        P20_XGB,
    ]),

    axis=1
)


# Apply 2015-locked Platt transform
P20_CONSENSUS_CAL = (
    COMMON12_PLATT
    .predict_proba(

        logit20c(
            P20_CONSENSUS_RAW
        )[:, None]

    )[:, 1]
)


# ============================================================
# 20C.10 ECE
# ============================================================

def ece20c(
    y,
    p,
    n_bins=15
):

    y = np.asarray(
        y
    )


    p = np.asarray(
        p
    )


    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    total = len(
        y
    )


    value = 0.0


    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (p >= edges[i])
                &
                (p <= edges[i + 1])
            )

        else:

            mask = (
                (p >= edges[i])
                &
                (p < edges[i + 1])
            )


        if not np.any(
            mask
        ):
            continue


        value += (

            mask.mean()

            *

            abs(
                p[mask].mean()
                -
                y[mask].mean()
            )
        )


    return float(
        value
    )


# ============================================================
# 20C.11 EXTERNAL DISCRIMINATION
# ============================================================

EXTERNAL_RESULTS_20C = pd.DataFrame([

    {
        "Model":
            "HGB_Common12",

        "PR_AUC":
            average_precision_score(
                y20,
                P20_HGB
            ),

        "ROC_AUC":
            roc_auc_score(
                y20,
                P20_HGB
            ),
    },

    {
        "Model":
            "CatBoost_Common12",

        "PR_AUC":
            average_precision_score(
                y20,
                P20_CAT
            ),

        "ROC_AUC":
            roc_auc_score(
                y20,
                P20_CAT
            ),
    },

    {
        "Model":
            "XGBoost_Common12",

        "PR_AUC":
            average_precision_score(
                y20,
                P20_XGB
            ),

        "ROC_AUC":
            roc_auc_score(
                y20,
                P20_XGB
            ),
    },

    {
        "Model":
            "Common12_Consensus_Raw",

        "PR_AUC":
            average_precision_score(
                y20,
                P20_CONSENSUS_RAW
            ),

        "ROC_AUC":
            roc_auc_score(
                y20,
                P20_CONSENSUS_RAW
            ),

        "Brier":
            brier_score_loss(
                y20,
                P20_CONSENSUS_RAW
            ),

        "NLL":
            log_loss(
                y20,
                P20_CONSENSUS_RAW
            ),

        "ECE":
            ece20c(
                y20,
                P20_CONSENSUS_RAW
            ),

        "Mean_Probability":
            P20_CONSENSUS_RAW.mean(),
    },

    {
        "Model":
            "Common12_Consensus_2015Platt",

        "PR_AUC":
            average_precision_score(
                y20,
                P20_CONSENSUS_CAL
            ),

        "ROC_AUC":
            roc_auc_score(
                y20,
                P20_CONSENSUS_CAL
            ),

        "Brier":
            brier_score_loss(
                y20,
                P20_CONSENSUS_CAL
            ),

        "NLL":
            log_loss(
                y20,
                P20_CONSENSUS_CAL
            ),

        "ECE":
            ece20c(
                y20,
                P20_CONSENSUS_CAL
            ),

        "Mean_Probability":
            P20_CONSENSUS_CAL.mean(),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C — BRFSS-2020 EXTERNAL PERFORMANCE"
)

print(
    "=" * 135
)


display(
    EXTERNAL_RESULTS_20C
)


# ============================================================
# 20C.12 INTERNAL VS EXTERNAL COMPARISON
# ============================================================

internal_pr = (
    average_precision_score(
        y15,
        COMMON12_CONSENSUS
    )
)


internal_roc = (
    roc_auc_score(
        y15,
        COMMON12_CONSENSUS
    )
)


external_pr = (
    average_precision_score(
        y20,
        P20_CONSENSUS_RAW
    )
)


external_roc = (
    roc_auc_score(
        y20,
        P20_CONSENSUS_RAW
    )
)


TRANSFER_RESULTS_20C = pd.DataFrame([{

    "Internal_2015_N":
        len(
            y15
        ),

    "External_2020_N":
        len(
            y20
        ),

    "Internal_Prevalence":
        y15.mean(),

    "External_Prevalence":
        y20.mean(),

    "Internal_PR_AUC":
        internal_pr,

    "External_PR_AUC":
        external_pr,

    "Delta_PR":
        external_pr
        -
        internal_pr,

    "Internal_ROC_AUC":
        internal_roc,

    "External_ROC_AUC":
        external_roc,

    "Delta_ROC":
        external_roc
        -
        internal_roc,
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C — INTERNAL TO EXTERNAL TRANSFER"
)

print(
    "=" * 135
)


display(
    TRANSFER_RESULTS_20C
)


# ============================================================
# 20C.13 LOCKED THRESHOLD EXTERNAL METRICS
# ============================================================

PRED20 = (

    P20_CONSENSUS_CAL

    >=

    LOCKED_COMMON12_THRESHOLD

).astype(int)


tn, fp, fn, tp = (
    confusion_matrix(
        y20,
        PRED20
    ).ravel()
)


specificity = (
    tn
    /
    max(
        tn + fp,
        1
    )
)


THRESHOLD_RESULTS_20C = pd.DataFrame([{

    "Threshold":
        LOCKED_COMMON12_THRESHOLD,

    "TP":
        int(tp),

    "FP":
        int(fp),

    "TN":
        int(tn),

    "FN":
        int(fn),

    "Precision":
        precision_score(
            y20,
            PRED20,
            zero_division=0
        ),

    "Recall":
        recall_score(
            y20,
            PRED20,
            zero_division=0
        ),

    "Specificity":
        specificity,

    "F1":
        f1_score(
            y20,
            PRED20,
            zero_division=0
        ),

    "MCC":
        matthews_corrcoef(
            y20,
            PRED20
        ),

    "Balanced_Accuracy":
        balanced_accuracy_score(
            y20,
            PRED20
        ),

    "Accuracy":
        accuracy_score(
            y20,
            PRED20
        ),
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C — EXTERNAL LOCKED-THRESHOLD PERFORMANCE"
)

print(
    "=" * 135
)


display(
    THRESHOLD_RESULTS_20C
)


# ============================================================
# 20C.14 PREVALENCE-NORMALIZED PR
#
# Simple diagnostic:
# PR-AUC divided by class prevalence.
#
# This is NOT a replacement for ordinary PR-AUC.
# ============================================================

PREVALENCE_CONTEXT_20C = pd.DataFrame([

    {
        "Dataset":
            "BRFSS-2015 Common12",

        "Prevalence":
            y15.mean(),

        "PR_AUC":
            internal_pr,

        "PR_to_Prevalence":
            internal_pr
            /
            y15.mean(),
    },

    {
        "Dataset":
            "BRFSS-2020 External",

        "Prevalence":
            y20.mean(),

        "PR_AUC":
            external_pr,

        "PR_to_Prevalence":
            external_pr
            /
            y20.mean(),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C — PR-AUC PREVALENCE CONTEXT"
)

print(
    "=" * 135
)


display(
    PREVALENCE_CONTEXT_20C
)


# ============================================================
# 20C.15 SAVE
# ============================================================

save_csv(
    DISTRIBUTION_20C,
    TAB_DIR
    / "module20c_feature_distribution_shift.csv"
)


save_csv(
    EXTERNAL_RESULTS_20C,
    TAB_DIR
    / "module20c_external_results.csv"
)


save_csv(
    TRANSFER_RESULTS_20C,
    TAB_DIR
    / "module20c_transfer_results.csv"
)


save_csv(
    THRESHOLD_RESULTS_20C,
    TAB_DIR
    / "module20c_threshold_results.csv"
)


save_csv(
    PREVALENCE_CONTEXT_20C,
    TAB_DIR
    / "module20c_prevalence_context.csv"
)


external_predictions_20c = pd.DataFrame({

    "target":
        y20,

    "HGB_Common12":
        P20_HGB,

    "CatBoost_Common12":
        P20_CAT,

    "XGBoost_Common12":
        P20_XGB,

    "Consensus_Raw":
        P20_CONSENSUS_RAW,

    "Consensus_2015Platt":
        P20_CONSENSUS_CAL,

    "Locked_Prediction":
        PRED20,
})


save_csv(
    external_predictions_20c,
    PRED_DIR
    / "module20c_external_predictions.csv"
)


manifest20c = {

    "module":
        "Frozen BRFSS-2020 external validation",

    "training_dataset":
        "BRFSS-2015 DEV",

    "external_dataset":
        "BRFSS-2020 heart_2020_cleaned",

    "features":
        COMMON12,

    "feature_count":
        12,

    "models": [
        "HGB",
        "CatBoost",
        "XGBoost",
    ],

    "aggregation":
        "equal consensus",

    "calibration":
        "Platt fitted only on 2015 OOF predictions",

    "threshold":
        "F1 threshold selected only on 2015 OOF predictions",

    "external_labels_used_for":
        "evaluation only",

    "post_external_tuning_allowed":
        False,
}


save_json(
    manifest20c,
    LOG_DIR
    / "module20c_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20C COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nBRFSS-2020 has now been opened for final external evaluation."
)

print(
    "Do NOT tune models, mappings, weights, calibration, or threshold "
    "using the 2020 results."
)

MODULE 20C — HARMONIZED EXTERNAL DATA
External shape: (319795, 12)
External prevalence: 0.08559545959130067

Feature order:
['BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age']

MODULE 20C — HARMONIZED FEATURE DISTRIBUTIONS


,Feature,Internal_Mean,External_Mean,Internal_SD,External_SD,Internal_Min,External_Min,Internal_Max,External_Max
0,BMI,28.381056,28.325396,6.610957,6.355871,12.0,12.02,98.0,94.849998
1,Smoker,0.443122,0.412477,0.496756,0.492524,0.0,0.00,1.0,1.000000
2,Stroke,0.040346,0.037740,0.196770,0.190279,0.0,0.00,1.0,1.000000
3,Diabetes,0.297353,0.284382,0.698492,0.676461,0.0,0.00,2.0,2.000000
4,PhysActivity,0.756815,0.775362,0.429007,0.417667,0.0,0.00,1.0,1.000000
5,HvyAlcoholConsump,0.056011,0.068097,0.229943,0.251730,0.0,0.00,1.0,1.000000
6,GenHlth,2.511476,2.404972,1.068834,1.041522,1.0,1.00,5.0,5.000000
7,MentHlth,3.185982,3.898366,7.415479,7.956418,0.0,0.00,30.0,30.000000
8,PhysHlth,4.233202,3.371710,8.702652,7.950258,0.0,0.00,30.0,30.000000
9,DiffWalk,0.167884,0.138870,0.373764,0.346088,0.0,0.00,1.0,1.000000



Internal calibrated mean probability: 0.09418217857349395
Internal prevalence: 0.09418361715547147

Locked Common12 threshold: 0.20000000000000004
Internal threshold F1: 0.4050950913622893


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time



MODULE 20C — BRFSS-2020 EXTERNAL PERFORMANCE


...


MODULE 20C — EXTERNAL LOCKED-THRESHOLD PERFORMANCE


,Threshold,TP,FP,TN,FN,Precision,Recall,Specificity,F1,MCC,Balanced_Accuracy,Accuracy
0,0.2,14170,30953,261469,13203,0.314031,0.517663,0.89415,0.390918,0.330948,0.705906,0.861924



MODULE 20C — PR-AUC PREVALENCE CONTEXT


,Dataset,Prevalence,PR_AUC,PR_to_Prevalence
0,BRFSS-2015 Common12,0.094184,0.356374,3.783826
1,BRFSS-2020 External,0.085595,0.339058,3.961165



MODULE 20C COMPLETE

BRFSS-2020 has now been opened for final external evaluation.
Do NOT tune models, mappings, weights, calibration, or threshold using the 2020 results.


In [57]:
# ============================================================
# MODULE 20D — FROZEN COMMON12 LPNS EXTERNAL VALIDATION
#
# PURPOSE
# ------------------------------------------------------------
# External validation of the principal reliability contribution:
#
#   confidence
#       +
#   expert disagreement
#       +
#   Local Prediction Neighborhood Stability (LPNS)
#
# MATCHED SYSTEM:
#   HGB + CatBoost + XGBoost
#
# REFERENCE DATA:
#   BRFSS-2015 only
#
# RELIABILITY MODEL:
#   fitted using BRFSS-2015 only
#
# EXTERNAL:
#   BRFSS-2020 used only for evaluation.
#
# IMPORTANT:
#   No parameter changes after viewing Module 20C.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 20D.1 REQUIREMENTS
# ============================================================

required_20d = [

    "X_dev",
    "y_dev",

    "COMMON12",

    "OOF_HGB_12",
    "OOF_CAT_12",
    "OOF_XGB_12",
    "COMMON12_INTERNAL_CAL",
    "LOCKED_COMMON12_THRESHOLD",

    "X20",
    "y20",

    "P20_HGB",
    "P20_CAT",
    "P20_XGB",
    "P20_CONSENSUS_CAL",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "PRED_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_20d
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 20D objects: "
        + ", ".join(missing)
    )


y15_20d = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


y20_20d = np.asarray(
    y20,
    dtype=int
)


# ============================================================
# 20D.2 MATCHED EXPERT MATRICES
# ============================================================

P15_3 = np.column_stack([

    OOF_HGB_12,
    OOF_CAT_12,
    OOF_XGB_12,
])


P20_3 = np.column_stack([

    P20_HGB,
    P20_CAT,
    P20_XGB,
])


# ============================================================
# 20D.3 FIXED CONFIGURATION
# ============================================================

CONFIG20D = {

    "max_reference_size":
        80000,

    "k_neighbors":
        20,

    "max_distance":
        3.0,

    "min_neighbors":
        6,

    "discrete_weight":
        0.65,

    "reliability_C":
        0.10,

    "review_rates": [
        0.05,
        0.10,
        0.15,
        0.20,
    ],
}


# ============================================================
# 20D.4 COMMON12 LPNS FEATURE TYPES
# ============================================================

LPNS12_BINARY = [

    "Smoker",
    "Stroke",
    "PhysActivity",
    "HvyAlcoholConsump",
    "DiffWalk",
    "Sex",
]


LPNS12_ORDINAL = [

    "GenHlth",
    "Age",
]


LPNS12_COUNT = [

    "MentHlth",
    "PhysHlth",
]


LPNS12_CONTINUOUS = [

    "BMI",
]


LPNS12_MULTICLASS = [

    "Diabetes",
]


LPNS12_ALL = (

    LPNS12_BINARY
    +
    LPNS12_ORDINAL
    +
    LPNS12_COUNT
    +
    LPNS12_CONTINUOUS
    +
    LPNS12_MULTICLASS
)


LPNS12_SCALE = (

    LPNS12_CONTINUOUS
    +
    LPNS12_COUNT
    +
    LPNS12_ORDINAL
)


LPNS12_DISCRETE = (

    LPNS12_BINARY
    +
    LPNS12_MULTICLASS
)


position12 = {

    name: i

    for i, name
    in enumerate(
        LPNS12_ALL
    )
}


scale_idx12 = np.array([

    position12[x]
    for x in LPNS12_SCALE
])


discrete_idx12 = np.array([

    position12[x]
    for x in LPNS12_DISCRETE
])


X15_lpns_raw = (

    X_dev[
        LPNS12_ALL
    ]
    .to_numpy(
        dtype=np.float32
    )
)


X20_lpns_raw = (

    X20[
        LPNS12_ALL
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 20D.5 DISEASE ERROR TARGETS
# ============================================================

pred15 = (

    COMMON12_INTERNAL_CAL
    >=
    LOCKED_COMMON12_THRESHOLD

).astype(int)


error15 = (

    pred15
    !=
    y15_20d

).astype(int)


pred20 = (

    P20_CONSENSUS_CAL
    >=
    LOCKED_COMMON12_THRESHOLD

).astype(int)


error20 = (

    pred20
    !=
    y20_20d

).astype(int)


print(
    "2015 error prevalence:",
    error15.mean()
)


print(
    "2020 error prevalence:",
    error20.mean()
)


# ============================================================
# 20D.6 CONFIDENCE / DISAGREEMENT FEATURES
# ============================================================

def entropy20d(
    p
):

    p = np.clip(
        p,
        1e-8,
        1 - 1e-8
    )


    return (

        -p * np.log(p)

        -

        (1-p)
        * np.log(1-p)

    ) / np.log(2)


def build_base_reliability20d(
    consensus,
    experts,
    threshold,
):

    entropy = entropy20d(
        consensus
    )


    signed_margin = (
        consensus
        -
        threshold
    )


    abs_margin = np.abs(
        signed_margin
    )


    distance_half = np.abs(
        consensus
        -
        0.5
    )


    confidence = np.column_stack([

        consensus,
        entropy,
        signed_margin,
        abs_margin,
        distance_half,
    ])


    expert_mean = np.mean(
        experts,
        axis=1
    )


    expert_std = np.std(
        experts,
        axis=1
    )


    expert_min = np.min(
        experts,
        axis=1
    )


    expert_max = np.max(
        experts,
        axis=1
    )


    expert_range = (
        expert_max
        -
        expert_min
    )


    expert_median = np.median(
        experts,
        axis=1
    )


    mean_abs_dev = np.mean(

        np.abs(
            experts
            -
            expert_mean[:, None]
        ),

        axis=1
    )


    disagreement = np.column_stack([

        confidence,

        expert_mean,
        expert_std,
        expert_min,
        expert_max,
        expert_range,
        expert_median,
        mean_abs_dev,

        experts,
    ])


    return (
        confidence,
        disagreement
    )


CONF15, DIS15 = (
    build_base_reliability20d(

        COMMON12_INTERNAL_CAL,

        P15_3,

        LOCKED_COMMON12_THRESHOLD
    )
)


CONF20, DIS20 = (
    build_base_reliability20d(

        P20_CONSENSUS_CAL,

        P20_3,

        LOCKED_COMMON12_THRESHOLD
    )
)


# ============================================================
# 20D.7 BUILD 2015 REFERENCE REPRESENTATION
# ============================================================

scaler20d = StandardScaler()


X15_distance = (
    X15_lpns_raw
    .copy()
)


X20_distance = (
    X20_lpns_raw
    .copy()
)


X15_distance[
    :,
    scale_idx12
] = (

    scaler20d
    .fit_transform(

        X15_distance[
            :,
            scale_idx12
        ]
    )
)


X20_distance[
    :,
    scale_idx12
] = (

    scaler20d
    .transform(

        X20_distance[
            :,
            scale_idx12
        ]
    )
)


X15_distance[
    :,
    discrete_idx12
] *= (
    CONFIG20D[
        "discrete_weight"
    ]
)


X20_distance[
    :,
    discrete_idx12
] *= (
    CONFIG20D[
        "discrete_weight"
    ]
)


# ============================================================
# 20D.8 FIXED REFERENCE SUBSET
# ============================================================

rng20d = np.random.default_rng(
    SEED + 207000
)


reference_indices = np.arange(
    len(
        X15_distance
    )
)


if len(
    reference_indices
) > CONFIG20D[
    "max_reference_size"
]:

    reference_indices = (
        rng20d.choice(

            reference_indices,

            size=
                CONFIG20D[
                    "max_reference_size"
                ],

            replace=False
        )
    )


X_reference20d = (
    X15_distance[
        reference_indices
    ]
)


P_reference20d = (
    P15_3[
        reference_indices
    ]
)


print(
    "Reference size:",
    len(
        reference_indices
    )
)


# ============================================================
# 20D.9 FIT NEAREST-NEIGHBOR INDEX
# ============================================================

nbr20d = NearestNeighbors(

    n_neighbors=
        CONFIG20D[
            "k_neighbors"
        ],

    metric=
        "euclidean",

    n_jobs=
        -1,
)


nbr20d.fit(
    X_reference20d
)


# ============================================================
# 20D.10 QUERY NEIGHBORS
# ============================================================

dist15, idx15 = (
    nbr20d.kneighbors(

        X15_distance,

        return_distance=True
    )
)


dist20, idx20 = (
    nbr20d.kneighbors(

        X20_distance,

        return_distance=True
    )
)


# ============================================================
# 20D.11 LPNS FEATURE EXTRACTION
# ============================================================

def lpns_features20d(
    query_experts,
    reference_experts,
    neighbor_idx,
    neighbor_dist,
    threshold,
):

    neigh_pred = (
        reference_experts[
            neighbor_idx
        ]
    )


    valid = (
        neighbor_dist
        <=
        CONFIG20D[
            "max_distance"
        ]
    )


    count = valid.sum(
        axis=1
    )


    safe_count = np.maximum(
        count,
        1
    )[:, None]


    mask = valid[
        :,
        :,
        None
    ]


    query = (
        query_experts[
            :,
            None,
            :
        ]
    )


    # --------------------------------------------------------
    # MAD
    # --------------------------------------------------------

    local_mad = (

        np.sum(

            np.abs(
                neigh_pred
                -
                query
            )
            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # local mean / SD
    # --------------------------------------------------------

    local_mean = (

        np.sum(
            neigh_pred
            *
            mask,
            axis=1
        )

        /
        safe_count
    )


    local_sd = np.sqrt(

        np.sum(

            (
                neigh_pred
                -
                local_mean[
                    :,
                    None,
                    :
                ]
            ) ** 2

            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # decision flips
    # --------------------------------------------------------

    query_decision = (
        query_experts
        >=
        threshold
    )


    neigh_decision = (
        neigh_pred
        >=
        threshold
    )


    local_flip = (

        np.sum(

            (
                neigh_decision
                !=
                query_decision[
                    :,
                    None,
                    :
                ]
            )

            *
            mask,

            axis=1
        )

        /
        safe_count
    )


    # --------------------------------------------------------
    # aggregates
    # --------------------------------------------------------

    mean_mad = np.mean(
        local_mad,
        axis=1
    )


    max_mad = np.max(
        local_mad,
        axis=1
    )


    min_mad = np.min(
        local_mad,
        axis=1
    )


    mad_gap = (
        max_mad
        -
        min_mad
    )


    mad_cv = (

        np.std(
            local_mad,
            axis=1
        )

        /
        (
            mean_mad
            +
            1e-8
        )
    )


    mean_sd = np.mean(
        local_sd,
        axis=1
    )


    max_sd = np.max(
        local_sd,
        axis=1
    )


    mean_flip = np.mean(
        local_flip,
        axis=1
    )


    max_flip = np.max(
        local_flip,
        axis=1
    )


    features = np.column_stack([

        mean_mad,
        max_mad,
        min_mad,
        mad_gap,
        mad_cv,

        mean_sd,
        max_sd,

        mean_flip,
        max_flip,

        count,

        local_mad,
        local_sd,
        local_flip,
    ])


    valid_profile = (
        count
        >=
        CONFIG20D[
            "min_neighbors"
        ]
    )


    features[
        ~valid_profile
    ] = np.nan


    return (
        features,
        valid_profile,
        count
    )


LPNS15, VALID15, NCOUNT15 = (
    lpns_features20d(

        P15_3,
        P_reference20d,

        idx15,
        dist15,

        LOCKED_COMMON12_THRESHOLD
    )
)


LPNS20, VALID20, NCOUNT20 = (
    lpns_features20d(

        P20_3,
        P_reference20d,

        idx20,
        dist20,

        LOCKED_COMMON12_THRESHOLD
    )
)


print(
    "\n2015 LPNS coverage:",
    VALID15.mean()
)


print(
    "2020 LPNS coverage:",
    VALID20.mean()
)


print(
    "2020 mean neighbors:",
    NCOUNT20.mean()
)


# ============================================================
# 20D.12 TRAIN-ONLY IMPUTATION
# ============================================================

lpns_medians20d = np.nanmedian(
    LPNS15,
    axis=0
)


lpns_medians20d = np.where(
    np.isnan(
        lpns_medians20d
    ),
    0.0,
    lpns_medians20d
)


def impute_lpns20d(
    X
):

    X = X.copy()


    missing = np.where(
        np.isnan(
            X
        )
    )


    X[
        missing
    ] = (
        lpns_medians20d[
            missing[1]
        ]
    )


    return X


LPNS15_I = (
    impute_lpns20d(
        LPNS15
    )
)


LPNS20_I = (
    impute_lpns20d(
        LPNS20
    )
)


# ============================================================
# 20D.13 FINAL RELIABILITY FEATURE MATRICES
# ============================================================

FULL15 = np.column_stack([

    DIS15,
    LPNS15_I,
])


FULL20 = np.column_stack([

    DIS20,
    LPNS20_I,
])


# ============================================================
# 20D.14 FIT FROZEN 2015 RELIABILITY HEADS
# ============================================================

def make_rel20d():

    return LogisticRegression(

        penalty=
            "l2",

        C=
            CONFIG20D[
                "reliability_C"
            ],

        class_weight=
            "balanced",

        solver=
            "lbfgs",

        max_iter=
            3000,

        random_state=
            SEED + 208000,
    )


scale_conf = StandardScaler()
scale_dis = StandardScaler()
scale_full = StandardScaler()


CONF15_S = (
    scale_conf
    .fit_transform(
        CONF15
    )
)


CONF20_S = (
    scale_conf
    .transform(
        CONF20
    )
)


DIS15_S = (
    scale_dis
    .fit_transform(
        DIS15
    )
)


DIS20_S = (
    scale_dis
    .transform(
        DIS20
    )
)


FULL15_S = (
    scale_full
    .fit_transform(
        FULL15
    )
)


FULL20_S = (
    scale_full
    .transform(
        FULL20
    )
)


REL_CONF20D = make_rel20d()
REL_DIS20D = make_rel20d()
REL_LPNS20D = make_rel20d()


REL_CONF20D.fit(
    CONF15_S,
    error15
)


REL_DIS20D.fit(
    DIS15_S,
    error15
)


REL_LPNS20D.fit(
    FULL15_S,
    error15
)


# ============================================================
# 20D.15 EXTERNAL ERROR-RISK SCORES
# ============================================================

RISK20_CONF = (
    REL_CONF20D
    .predict_proba(
        CONF20_S
    )[:, 1]
)


RISK20_DIS = (
    REL_DIS20D
    .predict_proba(
        DIS20_S
    )[:, 1]
)


RISK20_LPNS = (
    REL_LPNS20D
    .predict_proba(
        FULL20_S
    )[:, 1]
)


# ============================================================
# 20D.16 AURC
# ============================================================

def aurc20d(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    sorted_error = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = np.cumsum(
        sorted_error
    )


    retained_n = np.arange(
        1,
        len(error) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(error)
    )


    return float(
        np.trapezoid(
            selective_risk,
            coverage
        )
    )


# ============================================================
# 20D.17 EXTERNAL RELIABILITY RESULTS
# ============================================================

RESULTS_20D = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            average_precision_score(
                error20,
                RISK20_CONF
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error20,
                RISK20_CONF
            ),

        "AURC":
            aurc20d(
                error20,
                RISK20_CONF
            ),
    },

    {
        "Method":
            "Confidence+Disagreement",

        "Error_PR_AUC":
            average_precision_score(
                error20,
                RISK20_DIS
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error20,
                RISK20_DIS
            ),

        "AURC":
            aurc20d(
                error20,
                RISK20_DIS
            ),
    },

    {
        "Method":
            "Common12_LPNS",

        "Error_PR_AUC":
            average_precision_score(
                error20,
                RISK20_LPNS
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error20,
                RISK20_LPNS
            ),

        "AURC":
            aurc20d(
                error20,
                RISK20_LPNS
            ),
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20D — EXTERNAL LPNS RELIABILITY"
)

print(
    "=" * 135
)


display(
    RESULTS_20D
)


# ============================================================
# 20D.18 LPNS EXTERNAL GAIN
# ============================================================

baseline20d = (

    RESULTS_20D[
        RESULTS_20D[
            "Method"
        ]
        ==
        "Confidence+Disagreement"
    ]
    .iloc[0]
)


lpns20d = (

    RESULTS_20D[
        RESULTS_20D[
            "Method"
        ]
        ==
        "Common12_LPNS"
    ]
    .iloc[0]
)


GAIN_20D = pd.DataFrame([{

    "Delta_Error_PR":
        lpns20d[
            "Error_PR_AUC"
        ]
        -
        baseline20d[
            "Error_PR_AUC"
        ],

    "Delta_Error_ROC":
        lpns20d[
            "Error_ROC_AUC"
        ]
        -
        baseline20d[
            "Error_ROC_AUC"
        ],

    "Delta_AURC":
        lpns20d[
            "AURC"
        ]
        -
        baseline20d[
            "AURC"
        ],
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20D — EXTERNAL LPNS INCREMENTAL VALUE"
)

print(
    "=" * 135
)


display(
    GAIN_20D
)


# ============================================================
# 20D.19 SELECTIVE REVIEW
# ============================================================

def selective20d(
    risk,
    method
):

    rows = []


    total_errors = max(
        error20.sum(),
        1
    )


    base_error_rate = (
        error20.mean()
    )


    order = np.argsort(
        -risk
    )


    for rate in CONFIG20D[
        "review_rates"
    ]:

        n_review = max(
            1,
            int(
                round(
                    len(
                        error20
                    )
                    *
                    rate
                )
            )
        )


        review_idx = (
            order[
                :n_review
            ]
        )


        retained = np.ones(
            len(
                error20
            ),
            dtype=bool
        )


        retained[
            review_idx
        ] = False


        reviewed_error_rate = (
            error20[
                review_idx
            ].mean()
        )


        rows.append({

            "Method":
                method,

            "Review_Rate":
                rate,

            "Review_N":
                n_review,

            "Error_Enrichment":
                reviewed_error_rate
                /
                base_error_rate,

            "Errors_Captured":
                error20[
                    review_idx
                ].sum()
                /
                total_errors,

            "Errors_Per_100_Reviews":
                reviewed_error_rate
                *
                100.0,

            "Retained_Accuracy":
                accuracy_score(

                    y20_20d[
                        retained
                    ],

                    pred20[
                        retained
                    ]
                ),

            "Retained_F1":
                f1_score(

                    y20_20d[
                        retained
                    ],

                    pred20[
                        retained
                    ],

                    zero_division=0
                ),

            "Retained_MCC":
                matthews_corrcoef(

                    y20_20d[
                        retained
                    ],

                    pred20[
                        retained
                    ]
                ),
        })


    return pd.DataFrame(
        rows
    )


SELECTIVE_20D = pd.concat(

    [

        selective20d(
            RISK20_CONF,
            "Confidence"
        ),

        selective20d(
            RISK20_DIS,
            "Confidence+Disagreement"
        ),

        selective20d(
            RISK20_LPNS,
            "Common12_LPNS"
        ),
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20D — EXTERNAL SELECTIVE REVIEW"
)

print(
    "=" * 135
)


display(
    SELECTIVE_20D
)


# ============================================================
# 20D.20 SAVE
# ============================================================

save_csv(
    RESULTS_20D,
    TAB_DIR
    / "module20d_external_lpns_reliability.csv"
)


save_csv(
    GAIN_20D,
    TAB_DIR
    / "module20d_external_lpns_gain.csv"
)


save_csv(
    SELECTIVE_20D,
    TAB_DIR
    / "module20d_external_selective_review.csv"
)


pred20d = pd.DataFrame({

    "target":
        y20_20d,

    "prediction":
        pred20,

    "error":
        error20,

    "risk_confidence":
        RISK20_CONF,

    "risk_disagreement":
        RISK20_DIS,

    "risk_lpns":
        RISK20_LPNS,

    "lpns_valid":
        VALID20,

    "lpns_neighbors":
        NCOUNT20,
})


save_csv(
    pred20d,
    PRED_DIR
    / "module20d_external_reliability_predictions.csv"
)


manifest20d = {

    "module":
        "Frozen Common12 LPNS external reliability validation",

    "training_dataset":
        "BRFSS-2015",

    "external_dataset":
        "BRFSS-2020",

    "matched_experts": [
        "HGB",
        "CatBoost",
        "XGBoost",
    ],

    "primary_reliability_comparison":
        "Confidence+Disagreement vs LPNS",

    "reference_source":
        "2015 only",

    "reliability_training_source":
        "2015 only",

    "external_parameters_tuned":
        False,

    "status":
        "secondary frozen external evaluation",
}


save_json(
    manifest20d,
    LOG_DIR
    / "module20d_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20D COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nNo reliability parameter was tuned using BRFSS-2020."
)

2015 error prevalence: 0.14935647272153896
2020 error prevalence: 0.13807595490861332
Reference size: 80000

2015 LPNS coverage: 0.9999704351939451
2020 LPNS coverage: 0.9999906189902906
2020 mean neighbors: 19.99938398036242

MODULE 20D — EXTERNAL LPNS RELIABILITY


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.571734,0.905746,0.026288
1,Confidence+Disagreement,0.575812,0.906016,0.026255
2,Common12_LPNS,0.587949,0.905927,0.026337



MODULE 20D — EXTERNAL LPNS INCREMENTAL VALUE


,Delta_Error_PR,Delta_Error_ROC,Delta_AURC
0,0.012137,-0.000088,0.000082



MODULE 20D — EXTERNAL SELECTIVE REVIEW


,Method,Review_Rate,Review_N,Error_Enrichment,Errors_Captured,Errors_Per_100_Reviews,Retained_Accuracy,Retained_F1,Retained_MCC
0,Confidence,0.05,15990,4.408845,0.220446,60.875547,0.886697,0.314985,0.258862
1,Confidence,0.10,31980,4.793838,0.479391,66.191370,0.920129,0.226097,0.185948
2,Confidence,0.15,47969,4.741849,0.711274,65.473535,0.953099,0.071517,0.149702
3,Confidence,0.20,63959,3.878975,0.775795,53.559311,0.961303,0.041812,0.118685
4,Confidence+Disagreement,0.05,15990,4.469085,0.223458,61.707317,0.887135,0.319433,0.263381
5,Confidence+Disagreement,0.10,31980,4.798367,0.479844,66.253909,0.920199,0.227291,0.187206
6,Confidence+Disagreement,0.15,47969,4.743661,0.711545,65.498551,0.953143,0.069137,0.147445
7,Confidence+Disagreement,0.20,63959,3.879541,0.775908,53.567129,0.961323,0.039227,0.114873
8,Common12_LPNS,0.05,15990,4.733145,0.236661,65.353346,0.889054,0.338994,0.283269
9,Common12_LPNS,0.10,31980,4.840037,0.484011,66.829268,0.920838,0.254889,0.215161



MODULE 20D COMPLETE

No reliability parameter was tuned using BRFSS-2020.


In [58]:
# ============================================================
# MODULE 20E — EXTERNAL LPNS STATISTICAL CONFIRMATION
#
# DATA:
#   Frozen BRFSS-2020 predictions from Module 20D
#
# PRIMARY COMPARISONS:
#   LPNS vs Confidence
#   LPNS vs Confidence + Disagreement
#
# ENDPOINTS:
#   Error PR-AUC       [PRIMARY]
#   Error ROC-AUC
#   AURC
#
# SELECTIVE:
#   5%, 10%, 15%, 20%
#
# BOOTSTRAP:
#   paired, stratified by error status
#
# IMPORTANT:
#   No model fitting or parameter tuning occurs here.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 20E.1 REQUIREMENTS
# ============================================================

required_20e = [

    "error20",

    "RISK20_CONF",
    "RISK20_DIS",
    "RISK20_LPNS",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_20e
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 20E objects: "
        + ", ".join(missing)
    )


ERR = np.asarray(
    error20,
    dtype=int
)


R_CONF = np.asarray(
    RISK20_CONF,
    dtype=float
)


R_DIS = np.asarray(
    RISK20_DIS,
    dtype=float
)


R_LPNS = np.asarray(
    RISK20_LPNS,
    dtype=float
)


# ============================================================
# 20E.2 BASIC FUNCTIONS
# ============================================================

def aurc20e(
    error,
    risk
):

    order = np.argsort(
        risk
    )


    e = (
        error[
            order
        ]
        .astype(float)
    )


    cumulative_error = (
        np.cumsum(e)
    )


    retained_n = np.arange(
        1,
        len(e) + 1
    )


    selective_risk = (
        cumulative_error
        /
        retained_n
    )


    coverage = (
        retained_n
        /
        len(e)
    )


    return float(

        np.trapezoid(
            selective_risk,
            coverage
        )
    )


def error_capture20e(
    error,
    risk,
    review_rate
):

    n = len(
        error
    )


    n_review = max(
        1,
        int(
            round(
                review_rate
                *
                n
            )
        )
    )


    order = np.argsort(
        -risk
    )


    idx = order[
        :n_review
    ]


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    reviewed_errors = int(
        error[
            idx
        ].sum()
    )


    capture = (
        reviewed_errors
        /
        total_errors
    )


    errors_per_100 = (

        100.0
        *
        reviewed_errors
        /
        n_review
    )


    return (
        capture,
        errors_per_100,
        reviewed_errors,
    )


# ============================================================
# 20E.3 OBSERVED RESULTS
# ============================================================

OBSERVED_20E = []


for name, risk in [

    (
        "Confidence",
        R_CONF
    ),

    (
        "Confidence+Disagreement",
        R_DIS
    ),

    (
        "LPNS",
        R_LPNS
    ),
]:

    OBSERVED_20E.append({

        "Method":
            name,

        "Error_PR_AUC":
            average_precision_score(
                ERR,
                risk
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                ERR,
                risk
            ),

        "AURC":
            aurc20e(
                ERR,
                risk
            ),
    })


OBSERVED_20E = pd.DataFrame(
    OBSERVED_20E
)


print(
    "=" * 135
)

print(
    "MODULE 20E — OBSERVED EXTERNAL RELIABILITY"
)

print(
    "=" * 135
)


display(
    OBSERVED_20E
)


# ============================================================
# 20E.4 OBSERVED SELECTIVE RESULTS
# ============================================================

review_rates = [
    0.05,
    0.10,
    0.15,
    0.20,
]


selective_rows = []


for rate in review_rates:

    for name, risk in [

        (
            "Confidence",
            R_CONF
        ),

        (
            "Confidence+Disagreement",
            R_DIS
        ),

        (
            "LPNS",
            R_LPNS
        ),
    ]:

        (
            capture,
            errors100,
            n_errors

        ) = error_capture20e(

            ERR,
            risk,
            rate
        )


        selective_rows.append({

            "Method":
                name,

            "Review_Rate":
                rate,

            "Error_Capture":
                capture,

            "Errors_Per_100":
                errors100,

            "Errors_Found":
                n_errors,
        })


OBS_SELECTIVE_20E = pd.DataFrame(
    selective_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E — OBSERVED SELECTIVE PERFORMANCE"
)

print(
    "=" * 135
)


display(
    OBS_SELECTIVE_20E
)


# ============================================================
# 20E.5 STRATIFIED BOOTSTRAP INDICES
# ============================================================

positive_idx = np.where(
    ERR == 1
)[0]


negative_idx = np.where(
    ERR == 0
)[0]


print(
    "\nErrors:",
    len(
        positive_idx
    )
)


print(
    "Correct:",
    len(
        negative_idx
    )
)


# 1000 is adequate for this large external cohort.
# Increase to 2000 for final manuscript run if runtime allows.

B = 1000


rng = np.random.default_rng(
    SEED + 209000
)


# ============================================================
# 20E.6 STORAGE
# ============================================================

comparison_names = [

    "LPNS_vs_Confidence",

    "LPNS_vs_Disagreement",
]


boot = {

    name: {

        "PR":
            [],

        "ROC":
            [],

        "AURC":
            [],

        "Capture_5":
            [],

        "Errors100_5":
            [],

        "Capture_10":
            [],

        "Errors100_10":
            [],

        "Capture_15":
            [],

        "Errors100_15":
            [],

        "Capture_20":
            [],

        "Errors100_20":
            [],
    }

    for name in comparison_names
}


# ============================================================
# 20E.7 BOOTSTRAP
# ============================================================

for b in range(
    B
):

    pos_sample = rng.choice(

        positive_idx,

        size=len(
            positive_idx
        ),

        replace=True
    )


    neg_sample = rng.choice(

        negative_idx,

        size=len(
            negative_idx
        ),

        replace=True
    )


    idx = np.concatenate([

        pos_sample,
        neg_sample,
    ])


    yb = ERR[
        idx
    ]


    conf_b = R_CONF[
        idx
    ]


    dis_b = R_DIS[
        idx
    ]


    lpns_b = R_LPNS[
        idx
    ]


    # --------------------------------------------------------
    # LPNS vs confidence
    # --------------------------------------------------------

    boot[
        "LPNS_vs_Confidence"
    ][
        "PR"
    ].append(

        average_precision_score(
            yb,
            lpns_b
        )

        -

        average_precision_score(
            yb,
            conf_b
        )
    )


    boot[
        "LPNS_vs_Confidence"
    ][
        "ROC"
    ].append(

        roc_auc_score(
            yb,
            lpns_b
        )

        -

        roc_auc_score(
            yb,
            conf_b
        )
    )


    boot[
        "LPNS_vs_Confidence"
    ][
        "AURC"
    ].append(

        aurc20e(
            yb,
            lpns_b
        )

        -

        aurc20e(
            yb,
            conf_b
        )
    )


    # --------------------------------------------------------
    # LPNS vs disagreement
    # --------------------------------------------------------

    boot[
        "LPNS_vs_Disagreement"
    ][
        "PR"
    ].append(

        average_precision_score(
            yb,
            lpns_b
        )

        -

        average_precision_score(
            yb,
            dis_b
        )
    )


    boot[
        "LPNS_vs_Disagreement"
    ][
        "ROC"
    ].append(

        roc_auc_score(
            yb,
            lpns_b
        )

        -

        roc_auc_score(
            yb,
            dis_b
        )
    )


    boot[
        "LPNS_vs_Disagreement"
    ][
        "AURC"
    ].append(

        aurc20e(
            yb,
            lpns_b
        )

        -

        aurc20e(
            yb,
            dis_b
        )
    )


    # --------------------------------------------------------
    # selective review
    # --------------------------------------------------------

    for rate in review_rates:

        rate_pct = int(
            rate * 100
        )


        (
            cap_conf,
            e100_conf,
            _

        ) = error_capture20e(

            yb,
            conf_b,
            rate
        )


        (
            cap_dis,
            e100_dis,
            _

        ) = error_capture20e(

            yb,
            dis_b,
            rate
        )


        (
            cap_lpns,
            e100_lpns,
            _

        ) = error_capture20e(

            yb,
            lpns_b,
            rate
        )


        boot[
            "LPNS_vs_Confidence"
        ][
            f"Capture_{rate_pct}"
        ].append(

            cap_lpns
            -
            cap_conf
        )


        boot[
            "LPNS_vs_Confidence"
        ][
            f"Errors100_{rate_pct}"
        ].append(

            e100_lpns
            -
            e100_conf
        )


        boot[
            "LPNS_vs_Disagreement"
        ][
            f"Capture_{rate_pct}"
        ].append(

            cap_lpns
            -
            cap_dis
        )


        boot[
            "LPNS_vs_Disagreement"
        ][
            f"Errors100_{rate_pct}"
        ].append(

            e100_lpns
            -
            e100_dis
        )


    if (
        (b + 1)
        %
        100
        ==
        0
    ):

        print(
            f"Bootstrap {b+1}/{B}"
        )


# ============================================================
# 20E.8 PLUS-ONE P-VALUE
# ============================================================

def bootstrap_summary20e(
    observed,
    samples,
    beneficial_direction="positive",
):

    samples = np.asarray(
        samples,
        dtype=float
    )


    low, high = np.percentile(
        samples,
        [
            2.5,
            97.5
        ]
    )


    if beneficial_direction == "positive":

        opposing = np.sum(
            samples <= 0
        )


    else:

        opposing = np.sum(
            samples >= 0
        )


    # plus-one correction:
    p_one = (
        opposing
        +
        1
    ) / (
        len(
            samples
        )
        +
        1
    )


    p_two = min(
        1.0,
        2.0
        *
        p_one
    )


    return {

        "Observed_Delta":
            observed,

        "CI_Low":
            low,

        "CI_High":
            high,

        "P_two_sided":
            p_two,
    }


# ============================================================
# 20E.9 OBSERVED DELTAS
# ============================================================

observed_lookup = {

    row[
        "Method"
    ]:
        row

    for _, row in (
        OBSERVED_20E
        .iterrows()
    )
}


OBS_LPNS = observed_lookup[
    "LPNS"
]


OBS_CONF = observed_lookup[
    "Confidence"
]


OBS_DIS = observed_lookup[
    "Confidence+Disagreement"
]


# ============================================================
# 20E.10 GLOBAL STATISTICAL TABLE
# ============================================================

stat_rows = []


for comparison, baseline in [

    (
        "LPNS_vs_Confidence",
        OBS_CONF
    ),

    (
        "LPNS_vs_Disagreement",
        OBS_DIS
    ),
]:

    # PR
    result = bootstrap_summary20e(

        OBS_LPNS[
            "Error_PR_AUC"
        ]
        -
        baseline[
            "Error_PR_AUC"
        ],

        boot[
            comparison
        ][
            "PR"
        ],

        "positive"
    )


    stat_rows.append({

        "Comparison":
            comparison,

        "Metric":
            "Error_PR_AUC",

        **result
    })


    # ROC
    result = bootstrap_summary20e(

        OBS_LPNS[
            "Error_ROC_AUC"
        ]
        -
        baseline[
            "Error_ROC_AUC"
        ],

        boot[
            comparison
        ][
            "ROC"
        ],

        "positive"
    )


    stat_rows.append({

        "Comparison":
            comparison,

        "Metric":
            "Error_ROC_AUC",

        **result
    })


    # AURC: negative is better
    result = bootstrap_summary20e(

        OBS_LPNS[
            "AURC"
        ]
        -
        baseline[
            "AURC"
        ],

        boot[
            comparison
        ][
            "AURC"
        ],

        "negative"
    )


    stat_rows.append({

        "Comparison":
            comparison,

        "Metric":
            "AURC",

        **result
    })


STATS_20E = pd.DataFrame(
    stat_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E — EXTERNAL PAIRED BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    STATS_20E
)


# ============================================================
# 20E.11 SELECTIVE STATISTICS
# ============================================================

selective_stat_rows = []


for comparison, baseline_risk in [

    (
        "LPNS_vs_Confidence",
        R_CONF
    ),

    (
        "LPNS_vs_Disagreement",
        R_DIS
    ),
]:

    for rate in review_rates:

        pct = int(
            rate * 100
        )


        (
            cap_base,
            e100_base,
            _

        ) = error_capture20e(

            ERR,
            baseline_risk,
            rate
        )


        (
            cap_lpns,
            e100_lpns,
            _

        ) = error_capture20e(

            ERR,
            R_LPNS,
            rate
        )


        # capture
        result = bootstrap_summary20e(

            cap_lpns
            -
            cap_base,

            boot[
                comparison
            ][
                f"Capture_{pct}"
            ],

            "positive"
        )


        selective_stat_rows.append({

            "Comparison":
                comparison,

            "Review_Rate":
                rate,

            "Metric":
                "Error_Capture",

            **result
        })


        # errors / 100
        result = bootstrap_summary20e(

            e100_lpns
            -
            e100_base,

            boot[
                comparison
            ][
                f"Errors100_{pct}"
            ],

            "positive"
        )


        selective_stat_rows.append({

            "Comparison":
                comparison,

            "Review_Rate":
                rate,

            "Metric":
                "Errors_Per_100",

            **result
        })


SELECTIVE_STATS_20E = pd.DataFrame(
    selective_stat_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E — EXTERNAL SELECTIVE BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    SELECTIVE_STATS_20E
)


# ============================================================
# 20E.12 PRACTICAL 5% EFFECT
# ============================================================

cap_dis_5, e100_dis_5, errors_dis_5 = (
    error_capture20e(

        ERR,
        R_DIS,
        0.05
    )
)


cap_lpns_5, e100_lpns_5, errors_lpns_5 = (
    error_capture20e(

        ERR,
        R_LPNS,
        0.05
    )
)


PRACTICAL_20E = pd.DataFrame([{

    "External_N":
        len(
            ERR
        ),

    "Total_Errors":
        int(
            ERR.sum()
        ),

    "Review_Rate":
        0.05,

    "Review_N":
        int(
            round(
                0.05
                *
                len(
                    ERR
                )
            )
        ),

    "Baseline_Errors_Found":
        errors_dis_5,

    "LPNS_Errors_Found":
        errors_lpns_5,

    "Additional_Errors_Found":
        errors_lpns_5
        -
        errors_dis_5,

    "Baseline_Errors_Per_100":
        e100_dis_5,

    "LPNS_Errors_Per_100":
        e100_lpns_5,
}])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E — PRACTICAL EXTERNAL EFFECT"
)

print(
    "=" * 135
)


display(
    PRACTICAL_20E
)


# ============================================================
# 20E.13 SAVE
# ============================================================

save_csv(
    STATS_20E,
    TAB_DIR
    / "module20e_external_bootstrap.csv"
)


save_csv(
    SELECTIVE_STATS_20E,
    TAB_DIR
    / "module20e_external_selective_bootstrap.csv"
)


save_csv(
    PRACTICAL_20E,
    TAB_DIR
    / "module20e_practical_effect.csv"
)


save_json(

    {
        "module":
            "External LPNS statistical confirmation",

        "bootstrap_repetitions":
            B,

        "bootstrap":
            "paired stratified",

        "primary_metric":
            "Error PR-AUC",

        "comparisons": [
            "LPNS vs Confidence",
            "LPNS vs Confidence+Disagreement",
        ],

        "selective_review_rates":
            review_rates,

        "model_tuning":
            False,
    },

    LOG_DIR
    / "module20e_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nStatistical evaluation only. "
    "No model or parameter was changed."
)

MODULE 20E — OBSERVED EXTERNAL RELIABILITY


,Method,Error_PR_AUC,Error_ROC_AUC,AURC
0,Confidence,0.571734,0.905746,0.026288
1,Confidence+Disagreement,0.575812,0.906016,0.026255
2,LPNS,0.587949,0.905927,0.026337



MODULE 20E — OBSERVED SELECTIVE PERFORMANCE


,Method,Review_Rate,Error_Capture,Errors_Per_100,Errors_Found
0,Confidence,0.05,0.220446,60.875547,9734
1,Confidence+Disagreement,0.05,0.223458,61.707317,9867
2,LPNS,0.05,0.236661,65.353346,10450
3,Confidence,0.10,0.479391,66.191370,21168
4,Confidence+Disagreement,0.10,0.479844,66.253909,21188
5,LPNS,0.10,0.484011,66.829268,21372
6,Confidence,0.15,0.711274,65.473535,31407
7,Confidence+Disagreement,0.15,0.711545,65.498551,31419
8,LPNS,0.15,0.699995,64.435365,30909
9,Confidence,0.20,0.775795,53.559311,34256



Errors: 44156
Correct: 275639
Bootstrap 100/1000
Bootstrap 200/1000
Bootstrap 300/1000
Bootstrap 400/1000
Bootstrap 500/1000
Bootstrap 600/1000
Bootstrap 700/1000
Bootstrap 800/1000
Bootstrap 900/1000
Bootstrap 1000/1000

MODULE 20E — EXTERNAL PAIRED BOOTSTRAP


...


MODULE 20E — EXTERNAL SELECTIVE BOOTSTRAP


,Comparison,Review_Rate,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided
0,LPNS_vs_Confidence,0.05,Error_Capture,0.016215,0.014200,0.018141,0.001998
1,LPNS_vs_Confidence,0.05,Errors_Per_100,4.477799,3.921201,5.009537,0.001998
2,LPNS_vs_Confidence,0.10,Error_Capture,0.004620,0.002536,0.006546,0.001998
3,LPNS_vs_Confidence,0.10,Errors_Per_100,0.637899,0.350219,0.903768,0.001998
4,LPNS_vs_Confidence,0.15,Error_Capture,-0.011278,-0.013362,-0.009081,1.000000
5,LPNS_vs_Confidence,0.15,Errors_Per_100,-1.038170,-1.230013,-0.835957,1.000000
6,LPNS_vs_Confidence,0.20,Error_Capture,-0.000498,-0.001585,0.000317,1.000000
7,LPNS_vs_Confidence,0.20,Errors_Per_100,-0.034397,-0.109445,0.021889,1.000000
8,LPNS_vs_Disagreement,0.05,Error_Capture,0.013203,0.011346,0.015128,0.001998
9,LPNS_vs_Disagreement,0.05,Errors_Per_100,3.646029,3.133208,4.177611,0.001998



MODULE 20E — PRACTICAL EXTERNAL EFFECT


,External_N,Total_Errors,Review_Rate,Review_N,Baseline_Errors_Found,LPNS_Errors_Found,Additional_Errors_Found,Baseline_Errors_Per_100,LPNS_Errors_Per_100
0,319795,44156,0.05,15990,9867,10450,583,61.707317,65.353346



MODULE 20E COMPLETE

Statistical evaluation only. No model or parameter was changed.


In [59]:
# ============================================================
# MODULE 20E-FIX — CORRECT TWO-SIDED BOOTSTRAP SIGN P-VALUES
#
# Statistical reporting correction only.
# No models, predictions, thresholds, or parameters change.
# ============================================================

import numpy as np
import pandas as pd


def correct_bootstrap_pvalue(samples):

    samples = np.asarray(
        samples,
        dtype=float
    )

    n = len(samples)

    # plus-one corrected empirical tail probabilities
    p_le_zero = (
        np.sum(samples <= 0) + 1
    ) / (
        n + 1
    )

    p_ge_zero = (
        np.sum(samples >= 0) + 1
    ) / (
        n + 1
    )

    # genuine two-sided sign-based empirical p-value
    p_two = min(
        1.0,
        2.0 * min(
            p_le_zero,
            p_ge_zero
        )
    )

    return p_two


# ------------------------------------------------------------
# Rebuild global table
# ------------------------------------------------------------

corrected_rows = []


for comparison in [
    "LPNS_vs_Confidence",
    "LPNS_vs_Disagreement",
]:

    for metric in [
        "PR",
        "ROC",
        "AURC",
    ]:

        samples = np.asarray(
            boot[comparison][metric]
        )

        low, high = np.percentile(
            samples,
            [2.5, 97.5]
        )

        observed_row = (
            STATS_20E[
                (
                    STATS_20E["Comparison"]
                    ==
                    comparison
                )
                &
                (
                    STATS_20E["Metric"]
                    ==
                    {
                        "PR":
                            "Error_PR_AUC",
                        "ROC":
                            "Error_ROC_AUC",
                        "AURC":
                            "AURC",
                    }[metric]
                )
            ]
            .iloc[0]
        )

        corrected_rows.append({

            "Comparison":
                comparison,

            "Metric":
                {
                    "PR":
                        "Error_PR_AUC",
                    "ROC":
                        "Error_ROC_AUC",
                    "AURC":
                        "AURC",
                }[metric],

            "Observed_Delta":
                observed_row[
                    "Observed_Delta"
                ],

            "CI_Low":
                low,

            "CI_High":
                high,

            "P_two_sided":
                correct_bootstrap_pvalue(
                    samples
                ),
        })


STATS_20E_CORRECTED = pd.DataFrame(
    corrected_rows
)


print(
    "=" * 135
)

print(
    "MODULE 20E — CORRECTED EXTERNAL PAIRED BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    STATS_20E_CORRECTED
)


# ------------------------------------------------------------
# Correct selective table
# ------------------------------------------------------------

corrected_selective_rows = []


for comparison in [
    "LPNS_vs_Confidence",
    "LPNS_vs_Disagreement",
]:

    for rate in [
        0.05,
        0.10,
        0.15,
        0.20,
    ]:

        pct = int(
            rate * 100
        )

        for metric_name, boot_key in [

            (
                "Error_Capture",
                f"Capture_{pct}"
            ),

            (
                "Errors_Per_100",
                f"Errors100_{pct}"
            ),
        ]:

            samples = np.asarray(
                boot[
                    comparison
                ][
                    boot_key
                ]
            )


            old_row = (
                SELECTIVE_STATS_20E[
                    (
                        SELECTIVE_STATS_20E[
                            "Comparison"
                        ]
                        ==
                        comparison
                    )
                    &
                    (
                        SELECTIVE_STATS_20E[
                            "Review_Rate"
                        ]
                        ==
                        rate
                    )
                    &
                    (
                        SELECTIVE_STATS_20E[
                            "Metric"
                        ]
                        ==
                        metric_name
                    )
                ]
                .iloc[0]
            )


            low, high = np.percentile(
                samples,
                [2.5, 97.5]
            )


            corrected_selective_rows.append({

                "Comparison":
                    comparison,

                "Review_Rate":
                    rate,

                "Metric":
                    metric_name,

                "Observed_Delta":
                    old_row[
                        "Observed_Delta"
                    ],

                "CI_Low":
                    low,

                "CI_High":
                    high,

                "P_two_sided":
                    correct_bootstrap_pvalue(
                        samples
                    ),
            })


SELECTIVE_STATS_20E_CORRECTED = pd.DataFrame(
    corrected_selective_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 20E — CORRECTED EXTERNAL SELECTIVE BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    SELECTIVE_STATS_20E_CORRECTED
)


# Save corrected reporting tables
save_csv(
    STATS_20E_CORRECTED,
    TAB_DIR
    / "module20e_external_bootstrap_corrected.csv"
)


save_csv(
    SELECTIVE_STATS_20E_CORRECTED,
    TAB_DIR
    / "module20e_external_selective_bootstrap_corrected.csv"
)


print(
    "\nStatistical reporting corrected."
)

print(
    "No model/result was changed."
)

MODULE 20E — CORRECTED EXTERNAL PAIRED BOOTSTRAP


,Comparison,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided
0,LPNS_vs_Confidence,Error_PR_AUC,0.016215,0.013412,0.019050,0.001998
1,LPNS_vs_Confidence,Error_ROC_AUC,0.000182,-0.000023,0.000384,0.077922
2,LPNS_vs_Confidence,AURC,0.000048,-0.000059,0.000150,0.379620
3,LPNS_vs_Disagreement,Error_PR_AUC,0.012137,0.009551,0.014591,0.001998
4,LPNS_vs_Disagreement,Error_ROC_AUC,-0.000088,-0.000285,0.000108,0.389610
5,LPNS_vs_Disagreement,AURC,0.000082,-0.000027,0.000184,0.125874



MODULE 20E — CORRECTED EXTERNAL SELECTIVE BOOTSTRAP


,Comparison,Review_Rate,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided
0,LPNS_vs_Confidence,0.05,Error_Capture,0.016215,0.014200,0.018141,0.001998
1,LPNS_vs_Confidence,0.05,Errors_Per_100,4.477799,3.921201,5.009537,0.001998
2,LPNS_vs_Confidence,0.10,Error_Capture,0.004620,0.002536,0.006546,0.001998
3,LPNS_vs_Confidence,0.10,Errors_Per_100,0.637899,0.350219,0.903768,0.001998
4,LPNS_vs_Confidence,0.15,Error_Capture,-0.011278,-0.013362,-0.009081,0.001998
5,LPNS_vs_Confidence,0.15,Errors_Per_100,-1.038170,-1.230013,-0.835957,0.001998
6,LPNS_vs_Confidence,0.20,Error_Capture,-0.000498,-0.001585,0.000317,0.231768
7,LPNS_vs_Confidence,0.20,Errors_Per_100,-0.034397,-0.109445,0.021889,0.231768
8,LPNS_vs_Disagreement,0.05,Error_Capture,0.013203,0.011346,0.015128,0.001998
9,LPNS_vs_Disagreement,0.05,Errors_Per_100,3.646029,3.133208,4.177611,0.001998



Statistical reporting corrected.
No model/result was changed.


In [60]:
# ============================================================
# MODULE 21A — FROZEN FAIRNESS / SUBGROUP AUDIT
#
# INTERNAL:
#   BRFSS-2015
#   strict cross-fitted full LSA-HE
#
# EXTERNAL:
#   BRFSS-2020
#   frozen Common12 predictor
#
# SUBGROUPS:
#   Sex
#   Age group
#   Sex x Age group
#
# NO subgroup-specific thresholding.
# NO fairness retraining.
# NO post-hoc adjustment.
#
# Main fairness quantities:
#   Sensitivity / TPR
#   Specificity / TNR
#   FPR
#   FNR
#   Precision / PPV
#   NPV
#   F1
#   MCC
#   Balanced Accuracy
#   Positive Prediction Rate
#   Brier
#   ECE
#
# Also:
#   group counts
#   positive counts
#   disparity ranges
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
)


# ============================================================
# 21A.1 REQUIREMENTS
# ============================================================

required_21a = [

    # internal
    "X_dev",
    "y_dev",
    "LSA_HE_OOF",
    "PRED_19",

    # external
    "X20",
    "y20",
    "P20_CONSENSUS_CAL",
    "PRED20",

    # saving
    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_21a
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 21A objects: "
        + ", ".join(missing)
    )


# ============================================================
# 21A.2 ARRAYS
# ============================================================

y15_fair = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


p15_fair = np.asarray(
    LSA_HE_OOF,
    dtype=float
)


pred15_fair = np.asarray(
    PRED_19,
    dtype=int
)


y20_fair = np.asarray(
    y20,
    dtype=int
)


p20_fair = np.asarray(
    P20_CONSENSUS_CAL,
    dtype=float
)


pred20_fair = np.asarray(
    PRED20,
    dtype=int
)


# ============================================================
# 21A.3 DEMOGRAPHIC MAPPING
#
# Sex:
#   0 = Female
#   1 = Male
#
# Age BRFSS coding:
#   1  = 18-24
#   2  = 25-29
#   3  = 30-34
#   4  = 35-39
#   5  = 40-44
#   6  = 45-49
#   7  = 50-54
#   8  = 55-59
#   9  = 60-64
#   10 = 65-69
#   11 = 70-74
#   12 = 75-79
#   13 = 80+
# ============================================================

SEX_LABEL = {
    0: "Female",
    1: "Male",
}


def age_group_21a(
    age_code
):

    age_code = np.asarray(
        age_code,
        dtype=float
    )


    out = np.empty(
        len(age_code),
        dtype=object
    )


    out[
        age_code <= 5
    ] = "18-44"


    out[
        (
            age_code >= 6
        )
        &
        (
            age_code <= 9
        )
    ] = "45-64"


    out[
        age_code >= 10
    ] = "65+"


    return out


# ============================================================
# 21A.4 BUILD GROUP REGISTRIES
# ============================================================

sex15 = (
    X_dev[
        "Sex"
    ]
    .astype(int)
    .map(
        SEX_LABEL
    )
    .to_numpy()
)


age15 = age_group_21a(
    X_dev[
        "Age"
    ].to_numpy()
)


sex20 = (
    X20[
        "Sex"
    ]
    .astype(int)
    .map(
        SEX_LABEL
    )
    .to_numpy()
)


age20 = age_group_21a(
    X20[
        "Age"
    ].to_numpy()
)


intersection15 = np.array(
    [
        f"{s} | {a}"
        for s, a
        in zip(
            sex15,
            age15
        )
    ],
    dtype=object
)


intersection20 = np.array(
    [
        f"{s} | {a}"
        for s, a
        in zip(
            sex20,
            age20
        )
    ],
    dtype=object
)


# ============================================================
# 21A.5 ECE
# ============================================================

def ece21a(
    y,
    p,
    n_bins=10,
):

    y = np.asarray(
        y
    )


    p = np.asarray(
        p
    )


    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    value = 0.0


    for i in range(
        n_bins
    ):

        if i == (
            n_bins - 1
        ):

            mask = (
                (p >= edges[i])
                &
                (p <= edges[i + 1])
            )

        else:

            mask = (
                (p >= edges[i])
                &
                (p < edges[i + 1])
            )


        if not np.any(
            mask
        ):

            continue


        value += (

            mask.mean()

            *

            abs(
                p[
                    mask
                ].mean()

                -

                y[
                    mask
                ].mean()
            )
        )


    return float(
        value
    )


# ============================================================
# 21A.6 GROUP METRICS
# ============================================================

def subgroup_metrics_21a(
    y,
    p,
    pred,
    group_values,
    subgroup_type,
    dataset_name,
):

    rows = []


    unique_groups = sorted(
        pd.Series(
            group_values
        )
        .dropna()
        .unique()
    )


    for group in unique_groups:

        mask = (
            group_values
            ==
            group
        )


        yg = y[
            mask
        ]


        pg = p[
            mask
        ]


        predg = pred[
            mask
        ]


        if len(
            np.unique(
                yg
            )
        ) < 2:

            continue


        tn, fp, fn, tp = (
            confusion_matrix(
                yg,
                predg,
                labels=[
                    0,
                    1
                ]
            )
            .ravel()
        )


        sensitivity = (
            tp
            /
            max(
                tp + fn,
                1
            )
        )


        specificity = (
            tn
            /
            max(
                tn + fp,
                1
            )
        )


        fpr = (
            fp
            /
            max(
                fp + tn,
                1
            )
        )


        fnr = (
            fn
            /
            max(
                fn + tp,
                1
            )
        )


        precision = (
            tp
            /
            max(
                tp + fp,
                1
            )
        )


        npv = (
            tn
            /
            max(
                tn + fn,
                1
            )
        )


        rows.append({

            "Dataset":
                dataset_name,

            "Subgroup_Type":
                subgroup_type,

            "Group":
                group,

            "N":
                int(
                    mask.sum()
                ),

            "Positive_N":
                int(
                    yg.sum()
                ),

            "Negative_N":
                int(
                    len(
                        yg
                    )
                    -
                    yg.sum()
                ),

            "Observed_Prevalence":
                float(
                    yg.mean()
                ),

            "Predicted_Positive_N":
                int(
                    predg.sum()
                ),

            "Positive_Prediction_Rate":
                float(
                    predg.mean()
                ),

            "TP":
                int(tp),

            "FP":
                int(fp),

            "TN":
                int(tn),

            "FN":
                int(fn),

            "Sensitivity_TPR":
                sensitivity,

            "Specificity_TNR":
                specificity,

            "FPR":
                fpr,

            "FNR":
                fnr,

            "Precision_PPV":
                precision,

            "NPV":
                npv,

            "F1":
                f1_score(
                    yg,
                    predg,
                    zero_division=0
                ),

            "MCC":
                matthews_corrcoef(
                    yg,
                    predg
                ),

            "Balanced_Accuracy":
                balanced_accuracy_score(
                    yg,
                    predg
                ),

            "Accuracy":
                accuracy_score(
                    yg,
                    predg
                ),

            "Brier":
                brier_score_loss(
                    yg,
                    pg
                ),

            "ECE":
                ece21a(
                    yg,
                    pg,
                    n_bins=10
                ),

            "Mean_Probability":
                float(
                    pg.mean()
                ),
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 21A.7 INTERNAL FAIRNESS TABLES
# ============================================================

FAIR15_SEX = (
    subgroup_metrics_21a(

        y15_fair,
        p15_fair,
        pred15_fair,

        sex15,

        "Sex",

        "BRFSS-2015"
    )
)


FAIR15_AGE = (
    subgroup_metrics_21a(

        y15_fair,
        p15_fair,
        pred15_fair,

        age15,

        "Age",

        "BRFSS-2015"
    )
)


FAIR15_INTERSECTION = (
    subgroup_metrics_21a(

        y15_fair,
        p15_fair,
        pred15_fair,

        intersection15,

        "Sex×Age",

        "BRFSS-2015"
    )
)


FAIR15_ALL = pd.concat(

    [
        FAIR15_SEX,
        FAIR15_AGE,
        FAIR15_INTERSECTION,
    ],

    ignore_index=True
)


# ============================================================
# 21A.8 EXTERNAL FAIRNESS TABLES
# ============================================================

FAIR20_SEX = (
    subgroup_metrics_21a(

        y20_fair,
        p20_fair,
        pred20_fair,

        sex20,

        "Sex",

        "BRFSS-2020"
    )
)


FAIR20_AGE = (
    subgroup_metrics_21a(

        y20_fair,
        p20_fair,
        pred20_fair,

        age20,

        "Age",

        "BRFSS-2020"
    )
)


FAIR20_INTERSECTION = (
    subgroup_metrics_21a(

        y20_fair,
        p20_fair,
        pred20_fair,

        intersection20,

        "Sex×Age",

        "BRFSS-2020"
    )
)


FAIR20_ALL = pd.concat(

    [
        FAIR20_SEX,
        FAIR20_AGE,
        FAIR20_INTERSECTION,
    ],

    ignore_index=True
)


# ============================================================
# 21A.9 PRINT INTERNAL
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — INTERNAL FAIRNESS: SEX"
)

print(
    "=" * 135
)


display(
    FAIR15_SEX
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — INTERNAL FAIRNESS: AGE"
)

print(
    "=" * 135
)


display(
    FAIR15_AGE
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — INTERNAL FAIRNESS: SEX × AGE"
)

print(
    "=" * 135
)


display(
    FAIR15_INTERSECTION
)


# ============================================================
# 21A.10 PRINT EXTERNAL
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — EXTERNAL FAIRNESS: SEX"
)

print(
    "=" * 135
)


display(
    FAIR20_SEX
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — EXTERNAL FAIRNESS: AGE"
)

print(
    "=" * 135
)


display(
    FAIR20_AGE
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — EXTERNAL FAIRNESS: SEX × AGE"
)

print(
    "=" * 135
)


display(
    FAIR20_INTERSECTION
)


# ============================================================
# 21A.11 DISPARITY SUMMARY
#
# max-min range across groups
#
# We deliberately report gaps instead of changing thresholds.
# ============================================================

FAIRNESS_METRICS_21A = [

    "Sensitivity_TPR",
    "Specificity_TNR",
    "FPR",
    "FNR",
    "Precision_PPV",
    "NPV",
    "Positive_Prediction_Rate",
    "Balanced_Accuracy",
    "Brier",
    "ECE",
]


def disparity_summary_21a(
    table
):

    rows = []


    for (
        dataset,
        subgroup_type

    ), group_df in (
        table
        .groupby(
            [
                "Dataset",
                "Subgroup_Type",
            ]
        )
    ):

        for metric in (
            FAIRNESS_METRICS_21A
        ):

            values = (
                group_df[
                    metric
                ]
                .to_numpy(
                    dtype=float
                )
            )


            max_idx = np.argmax(
                values
            )


            min_idx = np.argmin(
                values
            )


            rows.append({

                "Dataset":
                    dataset,

                "Subgroup_Type":
                    subgroup_type,

                "Metric":
                    metric,

                "Minimum":
                    values[
                        min_idx
                    ],

                "Minimum_Group":
                    group_df
                    .iloc[
                        min_idx
                    ][
                        "Group"
                    ],

                "Maximum":
                    values[
                        max_idx
                    ],

                "Maximum_Group":
                    group_df
                    .iloc[
                        max_idx
                    ][
                        "Group"
                    ],

                "Max_Min_Gap":
                    values[
                        max_idx
                    ]
                    -
                    values[
                        min_idx
                    ],
            })


    return pd.DataFrame(
        rows
    )


FAIR_ALL = pd.concat(

    [
        FAIR15_ALL,
        FAIR20_ALL,
    ],

    ignore_index=True
)


DISPARITY_21A = (
    disparity_summary_21a(
        FAIR_ALL
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — FAIRNESS DISPARITY SUMMARY"
)

print(
    "=" * 135
)


display(
    DISPARITY_21A
)


# ============================================================
# 21A.12 KEY FAIRNESS SUMMARY
#
# Focus reviewer-facing metrics:
#   TPR gap
#   FPR gap
#   PPV gap
#   Brier gap
#   ECE gap
# ============================================================

KEY_METRICS = [

    "Sensitivity_TPR",
    "FPR",
    "Precision_PPV",
    "Brier",
    "ECE",
]


KEY_DISPARITY_21A = (

    DISPARITY_21A[
        DISPARITY_21A[
            "Metric"
        ]
        .isin(
            KEY_METRICS
        )
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — KEY FAIRNESS GAPS"
)

print(
    "=" * 135
)


display(
    KEY_DISPARITY_21A
)


# ============================================================
# 21A.13 GROUP SUPPORT CHECK
#
# Reviewer should see whether any intersection is too small.
# ============================================================

SUPPORT_21A = (

    FAIR_ALL[
        [
            "Dataset",
            "Subgroup_Type",
            "Group",
            "N",
            "Positive_N",
            "Negative_N",
        ]
    ]
    .copy()
)


SUPPORT_21A[
    "Adequate_N_500"
] = (
    SUPPORT_21A[
        "N"
    ]
    >=
    500
)


SUPPORT_21A[
    "Adequate_Positive_50"
] = (
    SUPPORT_21A[
        "Positive_N"
    ]
    >=
    50
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A — SUBGROUP SUPPORT"
)

print(
    "=" * 135
)


display(
    SUPPORT_21A
)


# ============================================================
# 21A.14 SAVE
# ============================================================

save_csv(
    FAIR15_ALL,
    TAB_DIR
    / "module21a_internal_fairness.csv"
)


save_csv(
    FAIR20_ALL,
    TAB_DIR
    / "module21a_external_fairness.csv"
)


save_csv(
    DISPARITY_21A,
    TAB_DIR
    / "module21a_disparity_summary.csv"
)


save_csv(
    KEY_DISPARITY_21A,
    TAB_DIR
    / "module21a_key_fairness_gaps.csv"
)


save_csv(
    SUPPORT_21A,
    TAB_DIR
    / "module21a_subgroup_support.csv"
)


save_json(

    {
        "module":
            "Frozen demographic fairness audit",

        "internal_model":
            "Strict cross-fitted full LSA-HE",

        "external_model":
            "Frozen Common12 external predictor",

        "demographics": [
            "Sex",
            "Age",
            "Sex x Age",
        ],

        "age_groups": [
            "18-44",
            "45-64",
            "65+",
        ],

        "threshold_adjustment":
            False,

        "subgroup_specific_training":
            False,

        "external_posthoc_tuning":
            False,
    },

    LOG_DIR
    / "module21a_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nNo subgroup threshold was changed."
)

print(
    "No fairness-aware training was performed."
)


MODULE 21A — INTERNAL FAIRNESS: SEX


,Dataset,Subgroup_Type,Group,N,Positive_N,Negative_N,Observed_Prevalence,Predicted_Positive_N,Positive_Prediction_Rate,TP,...,FNR,Precision_PPV,NPV,F1,MCC,Balanced_Accuracy,Accuracy,Brier,ECE,Mean_Probability
0,BRFSS-2015,Sex,Female,113674,8154,105520,0.071731,10222,0.089924,3385,...,0.584866,0.331149,0.953901,0.368415,0.316013,0.675170,0.897901,0.056104,0.009740,0.081351
1,BRFSS-2015,Sex,Male,89270,10960,78310,0.122774,18409,0.206217,6610,...,0.396898,0.359064,0.938612,0.450134,0.366985,0.726216,0.819099,0.087559,0.013525,0.136299



MODULE 21A — INTERNAL FAIRNESS: AGE


,Dataset,Subgroup_Type,Group,N,Positive_N,Negative_N,Observed_Prevalence,Predicted_Positive_N,Positive_Prediction_Rate,TP,...,FNR,Precision_PPV,NPV,F1,MCC,Balanced_Accuracy,Accuracy,Brier,ECE,Mean_Probability
0,BRFSS-2015,Age,18-44,43458,608,42850,0.013991,244,0.005615,80,...,0.868421,0.327869,0.987782,0.187793,0.200811,0.563876,0.984077,0.012739,0.002734,0.016630
1,BRFSS-2015,Age,45-64,88079,6175,81904,0.070108,7158,0.081268,2363,...,0.617328,0.330120,0.952892,0.354459,0.302873,0.662064,0.902281,0.055608,0.009952,0.080006
2,BRFSS-2015,Age,65+,71407,12331,59076,0.172686,21229,0.297296,7552,...,0.387560,0.355740,0.904759,0.450060,0.315009,0.690462,0.741538,0.122431,0.018483,0.191093



MODULE 21A — INTERNAL FAIRNESS: SEX × AGE


...


MODULE 21A — KEY FAIRNESS GAPS


,Dataset,Subgroup_Type,Metric,Minimum,Minimum_Group,Maximum,Maximum_Group,Max_Min_Gap
0,BRFSS-2015,Age,Sensitivity_TPR,0.131579,18-44,0.612440,65+,0.480861
1,BRFSS-2015,Age,FPR,0.003827,18-44,0.231515,65+,0.227688
2,BRFSS-2015,Age,Precision_PPV,0.327869,18-44,0.355740,65+,0.027871
3,BRFSS-2015,Age,Brier,0.012739,18-44,0.122431,65+,0.109692
4,BRFSS-2015,Age,ECE,0.002734,18-44,0.018483,65+,0.015749
5,BRFSS-2015,Sex,Sensitivity_TPR,0.415134,Female,0.603102,Male,0.187969
6,BRFSS-2015,Sex,FPR,0.064793,Female,0.150670,Male,0.085877
7,BRFSS-2015,Sex,Precision_PPV,0.331149,Female,0.359064,Male,0.027915
8,BRFSS-2015,Sex,Brier,0.056104,Female,0.087559,Male,0.031455
9,BRFSS-2015,Sex,ECE,0.009740,Female,0.013525,Male,0.003785



MODULE 21A — SUBGROUP SUPPORT


,Dataset,Subgroup_Type,Group,N,Positive_N,Negative_N,Adequate_N_500,Adequate_Positive_50
0,BRFSS-2015,Sex,Female,113674,8154,105520,True,True
1,BRFSS-2015,Sex,Male,89270,10960,78310,True,True
2,BRFSS-2015,Age,18-44,43458,608,42850,True,True
3,BRFSS-2015,Age,45-64,88079,6175,81904,True,True
4,BRFSS-2015,Age,65+,71407,12331,59076,True,True
5,BRFSS-2015,Sex×Age,Female | 18-44,23686,293,23393,True,True
6,BRFSS-2015,Sex×Age,Female | 45-64,49183,2646,46537,True,True
7,BRFSS-2015,Sex×Age,Female | 65+,40805,5215,35590,True,True
8,BRFSS-2015,Sex×Age,Male | 18-44,19772,315,19457,True,True
9,BRFSS-2015,Sex×Age,Male | 45-64,38896,3529,35367,True,True



MODULE 21A COMPLETE

No subgroup threshold was changed.
No fairness-aware training was performed.


In [61]:
# ============================================================
# MODULE 21B — BOOTSTRAP CONFIDENCE INTERVALS FOR FAIRNESS GAPS
#
# PURPOSE
# ------------------------------------------------------------
# Quantify uncertainty around subgroup disparities observed in
# Module 21A.
#
# DATA:
#   BRFSS-2015 frozen/cross-fitted LSA-HE
#   BRFSS-2020 frozen Common12 external model
#
# PRIMARY DISPARITY METRICS:
#   Sensitivity / TPR
#   FPR
#   PPV
#   Brier
#
# SECONDARY:
#   ECE
#
# Comparisons:
#   Sex: Male - Female
#
#   Age:
#       45-64 - 18-44
#       65+   - 18-44
#       65+   - 45-64
#
# Bootstrap:
#   stratified within subgroup by outcome
#
# No model fitting.
# No threshold changes.
# ============================================================


import numpy as np
import pandas as pd


# ============================================================
# 21B.1 REQUIREMENTS
# ============================================================

required_21b = [

    "y15_fair",
    "p15_fair",
    "pred15_fair",
    "sex15",
    "age15",

    "y20_fair",
    "p20_fair",
    "pred20_fair",
    "sex20",
    "age20",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_21b
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 21B objects: "
        + ", ".join(missing)
    )


# ============================================================
# 21B.2 METRIC FUNCTIONS
# ============================================================

def ece21b(
    y,
    p,
    n_bins=10,
):

    y = np.asarray(y)
    p = np.asarray(p)


    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    result = 0.0


    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (p >= edges[i])
                &
                (p <= edges[i+1])
            )

        else:

            mask = (
                (p >= edges[i])
                &
                (p < edges[i+1])
            )


        if not np.any(mask):
            continue


        result += (

            mask.mean()

            *

            abs(
                p[mask].mean()
                -
                y[mask].mean()
            )
        )


    return float(result)


def metrics21b(
    y,
    p,
    pred,
):

    y = np.asarray(y)
    p = np.asarray(p)
    pred = np.asarray(pred)


    tp = np.sum(
        (y == 1)
        &
        (pred == 1)
    )


    fn = np.sum(
        (y == 1)
        &
        (pred == 0)
    )


    fp = np.sum(
        (y == 0)
        &
        (pred == 1)
    )


    tn = np.sum(
        (y == 0)
        &
        (pred == 0)
    )


    sensitivity = (
        tp
        /
        max(
            tp + fn,
            1
        )
    )


    fpr = (
        fp
        /
        max(
            fp + tn,
            1
        )
    )


    ppv = (
        tp
        /
        max(
            tp + fp,
            1
        )
    )


    brier = np.mean(
        (
            p - y
        ) ** 2
    )


    return {

        "Sensitivity_TPR":
            sensitivity,

        "FPR":
            fpr,

        "Precision_PPV":
            ppv,

        "Brier":
            brier,

        "ECE":
            ece21b(
                y,
                p
            ),
    }


# ============================================================
# 21B.3 STRATIFIED SUBGROUP BOOTSTRAP
# ============================================================

def stratified_sample_group21b(
    y,
    group_idx,
    rng,
):

    idx = np.where(
        group_idx
    )[0]


    y_group = y[
        idx
    ]


    pos = idx[
        y_group == 1
    ]


    neg = idx[
        y_group == 0
    ]


    pos_sample = rng.choice(

        pos,

        size=len(pos),

        replace=True
    )


    neg_sample = rng.choice(

        neg,

        size=len(neg),

        replace=True
    )


    return np.concatenate([
        pos_sample,
        neg_sample
    ])


# ============================================================
# 21B.4 GAP COMPUTATION
# ============================================================

def observed_gap21b(
    y,
    p,
    pred,
    group_values,
    group_a,
    group_b,
    metric,
):

    mask_a = (
        group_values
        ==
        group_a
    )


    mask_b = (
        group_values
        ==
        group_b
    )


    m_a = metrics21b(

        y[
            mask_a
        ],

        p[
            mask_a
        ],

        pred[
            mask_a
        ],
    )[metric]


    m_b = metrics21b(

        y[
            mask_b
        ],

        p[
            mask_b
        ],

        pred[
            mask_b
        ],
    )[metric]


    return (
        m_b - m_a,
        m_a,
        m_b,
    )


# ============================================================
# 21B.5 COMPARISON REGISTRY
# ============================================================

comparisons = [

    {
        "Subgroup_Type":
            "Sex",

        "Group_A":
            "Female",

        "Group_B":
            "Male",
    },

    {
        "Subgroup_Type":
            "Age",

        "Group_A":
            "18-44",

        "Group_B":
            "45-64",
    },

    {
        "Subgroup_Type":
            "Age",

        "Group_A":
            "18-44",

        "Group_B":
            "65+",
    },

    {
        "Subgroup_Type":
            "Age",

        "Group_A":
            "45-64",

        "Group_B":
            "65+",
    },
]


metrics_to_test = [

    "Sensitivity_TPR",
    "FPR",
    "Precision_PPV",
    "Brier",
    "ECE",
]


# ============================================================
# 21B.6 DATASETS
# ============================================================

datasets21b = {

    "BRFSS-2015": {

        "y":
            y15_fair,

        "p":
            p15_fair,

        "pred":
            pred15_fair,

        "Sex":
            sex15,

        "Age":
            age15,
    },

    "BRFSS-2020": {

        "y":
            y20_fair,

        "p":
            p20_fair,

        "pred":
            pred20_fair,

        "Sex":
            sex20,

        "Age":
            age20,
    },
}


# ============================================================
# 21B.7 BOOTSTRAP
# ============================================================

B = 1000


rng = np.random.default_rng(
    SEED + 211000
)


summary_rows = []


for dataset_name, D in (
    datasets21b.items()
):

    print(
        "\n"
        + "=" * 135
    )

    print(
        dataset_name
    )

    print(
        "=" * 135
    )


    y = np.asarray(
        D["y"]
    )


    p = np.asarray(
        D["p"]
    )


    pred = np.asarray(
        D["pred"]
    )


    for comp in comparisons:

        subgroup_type = (
            comp[
                "Subgroup_Type"
            ]
        )


        group_values = np.asarray(
            D[
                subgroup_type
            ]
        )


        group_a = (
            comp[
                "Group_A"
            ]
        )


        group_b = (
            comp[
                "Group_B"
            ]
        )


        mask_a = (
            group_values
            ==
            group_a
        )


        mask_b = (
            group_values
            ==
            group_b
        )


        for metric in (
            metrics_to_test
        ):

            (
                observed,
                value_a,
                value_b

            ) = observed_gap21b(

                y,
                p,
                pred,

                group_values,

                group_a,
                group_b,

                metric,
            )


            bootstrap_delta = []


            for b in range(
                B
            ):

                idx_a = (
                    stratified_sample_group21b(

                        y,

                        mask_a,

                        rng
                    )
                )


                idx_b = (
                    stratified_sample_group21b(

                        y,

                        mask_b,

                        rng
                    )
                )


                metric_a = (
                    metrics21b(

                        y[
                            idx_a
                        ],

                        p[
                            idx_a
                        ],

                        pred[
                            idx_a
                        ],

                    )[metric]
                )


                metric_b = (
                    metrics21b(

                        y[
                            idx_b
                        ],

                        p[
                            idx_b
                        ],

                        pred[
                            idx_b
                        ],

                    )[metric]
                )


                bootstrap_delta.append(

                    metric_b
                    -
                    metric_a
                )


            bootstrap_delta = np.asarray(
                bootstrap_delta
            )


            low, high = np.percentile(

                bootstrap_delta,

                [
                    2.5,
                    97.5
                ]
            )


            # genuine two-sided empirical sign p-value
            p_left = (

                np.sum(
                    bootstrap_delta
                    <=
                    0
                )

                +
                1

            ) / (
                B + 1
            )


            p_right = (

                np.sum(
                    bootstrap_delta
                    >=
                    0
                )

                +
                1

            ) / (
                B + 1
            )


            p_two = min(

                1.0,

                2.0
                *
                min(
                    p_left,
                    p_right
                )
            )


            summary_rows.append({

                "Dataset":
                    dataset_name,

                "Subgroup_Type":
                    subgroup_type,

                "Group_A":
                    group_a,

                "Group_B":
                    group_b,

                "Metric":
                    metric,

                "Value_A":
                    value_a,

                "Value_B":
                    value_b,

                "Delta_B_minus_A":
                    observed,

                "CI_Low":
                    low,

                "CI_High":
                    high,

                "P_two_sided":
                    p_two,
            })


RESULTS_21B = pd.DataFrame(
    summary_rows
)


# ============================================================
# 21B.8 PRIMARY TABLE
# ============================================================

PRIMARY_21B = (

    RESULTS_21B[
        RESULTS_21B[
            "Metric"
        ]
        .isin(
            [
                "Sensitivity_TPR",
                "FPR",
                "Precision_PPV",
            ]
        )
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21B — PRIMARY FAIRNESS GAP BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    PRIMARY_21B
)


# ============================================================
# 21B.9 CALIBRATION TABLE
# ============================================================

CALIBRATION_21B = (

    RESULTS_21B[
        RESULTS_21B[
            "Metric"
        ]
        .isin(
            [
                "Brier",
                "ECE",
            ]
        )
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21B — CALIBRATION GAP BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    CALIBRATION_21B
)


# ============================================================
# 21B.10 REPLICATION TABLE
#
# Compare direction of gaps between 2015 and 2020.
# ============================================================

replication_rows = []


for comp in comparisons:

    subgroup_type = (
        comp[
            "Subgroup_Type"
        ]
    )


    group_a = (
        comp[
            "Group_A"
        ]
    )


    group_b = (
        comp[
            "Group_B"
        ]
    )


    for metric in [
        "Sensitivity_TPR",
        "FPR",
        "Precision_PPV",
        "Brier",
        "ECE",
    ]:

        r15 = (

            RESULTS_21B[
                (
                    RESULTS_21B[
                        "Dataset"
                    ]
                    ==
                    "BRFSS-2015"
                )
                &
                (
                    RESULTS_21B[
                        "Subgroup_Type"
                    ]
                    ==
                    subgroup_type
                )
                &
                (
                    RESULTS_21B[
                        "Group_A"
                    ]
                    ==
                    group_a
                )
                &
                (
                    RESULTS_21B[
                        "Group_B"
                    ]
                    ==
                    group_b
                )
                &
                (
                    RESULTS_21B[
                        "Metric"
                    ]
                    ==
                    metric
                )
            ]
            .iloc[0]
        )


        r20 = (

            RESULTS_21B[
                (
                    RESULTS_21B[
                        "Dataset"
                    ]
                    ==
                    "BRFSS-2020"
                )
                &
                (
                    RESULTS_21B[
                        "Subgroup_Type"
                    ]
                    ==
                    subgroup_type
                )
                &
                (
                    RESULTS_21B[
                        "Group_A"
                    ]
                    ==
                    group_a
                )
                &
                (
                    RESULTS_21B[
                        "Group_B"
                    ]
                    ==
                    group_b
                )
                &
                (
                    RESULTS_21B[
                        "Metric"
                    ]
                    ==
                    metric
                )
            ]
            .iloc[0]
        )


        delta15 = (
            r15[
                "Delta_B_minus_A"
            ]
        )


        delta20 = (
            r20[
                "Delta_B_minus_A"
            ]
        )


        replication_rows.append({

            "Subgroup_Type":
                subgroup_type,

            "Comparison":
                f"{group_b} - {group_a}",

            "Metric":
                metric,

            "Delta_2015":
                delta15,

            "Delta_2020":
                delta20,

            "Same_Direction":
                (
                    np.sign(
                        delta15
                    )
                    ==
                    np.sign(
                        delta20
                    )
                ),
        })


REPLICATION_21B = pd.DataFrame(
    replication_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21B — FAIRNESS GAP REPLICATION"
)

print(
    "=" * 135
)


display(
    REPLICATION_21B
)


# ============================================================
# 21B.11 SAVE
# ============================================================

save_csv(
    RESULTS_21B,
    TAB_DIR
    / "module21b_all_gap_bootstrap.csv"
)


save_csv(
    PRIMARY_21B,
    TAB_DIR
    / "module21b_primary_fairness_bootstrap.csv"
)


save_csv(
    CALIBRATION_21B,
    TAB_DIR
    / "module21b_calibration_gap_bootstrap.csv"
)


save_csv(
    REPLICATION_21B,
    TAB_DIR
    / "module21b_gap_replication.csv"
)


save_json(

    {
        "module":
            "Fairness gap bootstrap",

        "bootstrap_repetitions":
            B,

        "datasets": [
            "BRFSS-2015",
            "BRFSS-2020",
        ],

        "comparisons": [
            "Male vs Female",
            "45-64 vs 18-44",
            "65+ vs 18-44",
            "65+ vs 45-64",
        ],

        "primary_metrics": [
            "Sensitivity",
            "FPR",
            "PPV",
        ],

        "calibration_metrics": [
            "Brier",
            "ECE",
        ],

        "threshold_adjustment":
            False,

        "model_retraining":
            False,
    },

    LOG_DIR
    / "module21b_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 21B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nStatistical audit only."
)

print(
    "No subgroup-specific threshold or model was changed."
)


BRFSS-2015

BRFSS-2020

MODULE 21B — PRIMARY FAIRNESS GAP BOOTSTRAP


,Dataset,Subgroup_Type,Group_A,Group_B,Metric,Value_A,Value_B,Delta_B_minus_A,CI_Low,CI_High,P_two_sided
0,BRFSS-2015,Sex,Female,Male,Sensitivity_TPR,0.415134,0.603102,0.187969,0.173758,0.202292,0.001998
1,BRFSS-2015,Sex,Female,Male,FPR,0.064793,0.150670,0.085877,0.083094,0.088903,0.001998
2,BRFSS-2015,Sex,Female,Male,Precision_PPV,0.331149,0.359064,0.027915,0.018385,0.037406,0.001998
3,BRFSS-2015,Age,18-44,45-64,Sensitivity_TPR,0.131579,0.382672,0.251093,0.219563,0.282501,0.001998
4,BRFSS-2015,Age,18-44,45-64,FPR,0.003827,0.058544,0.054717,0.052932,0.056423,0.001998
5,BRFSS-2015,Age,18-44,45-64,Precision_PPV,0.327869,0.330120,0.002251,-0.057667,0.058189,0.941059
6,BRFSS-2015,Age,18-44,65+,Sensitivity_TPR,0.131579,0.612440,0.480861,0.453824,0.508098,0.001998
7,BRFSS-2015,Age,18-44,65+,FPR,0.003827,0.231515,0.227688,0.224107,0.231049,0.001998
8,BRFSS-2015,Age,18-44,65+,Precision_PPV,0.327869,0.355740,0.027871,-0.026513,0.083607,0.333666
9,BRFSS-2015,Age,45-64,65+,Sensitivity_TPR,0.382672,0.612440,0.229768,0.214945,0.244921,0.001998



MODULE 21B — CALIBRATION GAP BOOTSTRAP


,Dataset,Subgroup_Type,Group_A,Group_B,Metric,Value_A,Value_B,Delta_B_minus_A,CI_Low,CI_High,P_two_sided
0,BRFSS-2015,Sex,Female,Male,Brier,0.056104,0.087559,0.031455,0.030711,0.032218,0.001998
1,BRFSS-2015,Sex,Female,Male,ECE,0.009740,0.013525,0.003785,0.002794,0.004896,0.001998
2,BRFSS-2015,Age,18-44,45-64,Brier,0.012739,0.055608,0.042869,0.042355,0.043406,0.001998
3,BRFSS-2015,Age,18-44,45-64,ECE,0.002734,0.009952,0.007218,0.006436,0.007975,0.001998
4,BRFSS-2015,Age,18-44,65+,Brier,0.012739,0.122431,0.109692,0.108858,0.110524,0.001998
5,BRFSS-2015,Age,18-44,65+,ECE,0.002734,0.018483,0.015749,0.014646,0.016917,0.001998
6,BRFSS-2015,Age,45-64,65+,Brier,0.055608,0.122431,0.066823,0.065881,0.067775,0.001998
7,BRFSS-2015,Age,45-64,65+,ECE,0.009952,0.018483,0.008531,0.007321,0.009746,0.001998
8,BRFSS-2020,Sex,Female,Male,Brier,0.054407,0.078464,0.024057,0.023538,0.024612,0.001998
9,BRFSS-2020,Sex,Female,Male,ECE,0.002855,0.001923,-0.000932,-0.001959,0.000985,0.521479



MODULE 21B — FAIRNESS GAP REPLICATION


,Subgroup_Type,Comparison,Metric,Delta_2015,Delta_2020,Same_Direction
0,Sex,Male - Female,Sensitivity_TPR,0.187969,0.213849,True
1,Sex,Male - Female,FPR,0.085877,0.087919,True
2,Sex,Male - Female,Precision_PPV,0.027915,0.018101,True
3,Sex,Male - Female,Brier,0.031455,0.024057,True
4,Sex,Male - Female,ECE,0.003785,-0.000932,False
5,Age,45-64 - 18-44,Sensitivity_TPR,0.251093,0.251465,True
6,Age,45-64 - 18-44,FPR,0.054717,0.052312,True
7,Age,45-64 - 18-44,Precision_PPV,0.002251,0.050360,True
8,Age,45-64 - 18-44,Brier,0.042869,0.044471,True
9,Age,45-64 - 18-44,ECE,0.007218,0.001642,True



MODULE 21B COMPLETE

Statistical audit only.
No subgroup-specific threshold or model was changed.


In [62]:
# ============================================================
# MODULE 22A — FROZEN EXTERNAL ROBUSTNESS AUDIT
#
# DATA:
#   BRFSS-2020 external Common12 representation
#
# FROZEN MODELS:
#   HGB_Common12
#   CatBoost_Common12
#   XGBoost_Common12
#   2015 Platt calibrator
#   locked threshold
#
# CORRUPTIONS:
#
#   1. Continuous Gaussian noise
#      levels = 0.05, 0.10, 0.20 SD
#
#   2. Random feature masking
#      levels = 5%, 10%, 20%
#
#   3. Random categorical corruption
#      levels = 5%, 10%, 20%
#
# Metrics:
#   PR-AUC
#   ROC-AUC
#   Brier
#   NLL
#   ECE
#   F1
#   MCC
#
# No retraining.
# No recalibration.
# No threshold adjustment.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    f1_score,
    matthews_corrcoef,
)


# ============================================================
# 22A.1 REQUIREMENTS
# ============================================================

required_22a = [

    "X20",
    "y20",

    "HGB_COMMON12_FINAL",
    "CAT_COMMON12_FINAL",
    "XGB_COMMON12_FINAL",

    "COMMON12_PLATT",
    "LOCKED_COMMON12_THRESHOLD",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_22a
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 22A objects: "
        + ", ".join(missing)
    )


X22 = (
    X20
    .copy()
    .reset_index(
        drop=True
    )
)


y22 = np.asarray(
    y20,
    dtype=int
)


# ============================================================
# 22A.2 FEATURE TYPES
# ============================================================

CONTINUOUS_22 = [

    "BMI",
    "MentHlth",
    "PhysHlth",
]


BINARY_22 = [

    "Smoker",
    "Stroke",
    "PhysActivity",
    "HvyAlcoholConsump",
    "DiffWalk",
    "Sex",
]


ORDINAL_22 = [

    "GenHlth",
    "Age",
]


MULTICLASS_22 = [

    "Diabetes",
]


ALL_FEATURES_22 = list(
    X22.columns
)


# ============================================================
# 22A.3 BASELINE STATISTICS
# ============================================================

TRAIN_MEDIANS_22 = (

    X_dev[
        COMMON12
    ]
    .median()
)


TRAIN_MEANS_22 = (

    X_dev[
        CONTINUOUS_22
    ]
    .mean()
)


TRAIN_SDS_22 = (

    X_dev[
        CONTINUOUS_22
    ]
    .std()
)


# ============================================================
# 22A.4 HELPERS
# ============================================================

def logit22(
    p
):

    p = np.clip(
        p,
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p / (1 - p)
    )


def ece22(
    y,
    p,
    n_bins=15,
):

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    value = 0.0


    for i in range(
        n_bins
    ):

        if i == n_bins - 1:

            mask = (
                (p >= edges[i])
                &
                (p <= edges[i+1])
            )

        else:

            mask = (
                (p >= edges[i])
                &
                (p < edges[i+1])
            )


        if not np.any(mask):
            continue


        value += (

            mask.mean()

            *

            abs(
                p[mask].mean()
                -
                y[mask].mean()
            )
        )


    return float(
        value
    )


def predict_frozen22(
    X
):

    p_hgb = (
        HGB_COMMON12_FINAL
        .predict_proba(
            X
        )[:, 1]
    )


    p_cat = (
        CAT_COMMON12_FINAL
        .predict_proba(
            X
        )[:, 1]
    )


    p_xgb = (
        XGB_COMMON12_FINAL
        .predict_proba(
            X
        )[:, 1]
    )


    raw = np.mean(

        np.column_stack([

            p_hgb,
            p_cat,
            p_xgb,
        ]),

        axis=1
    )


    calibrated = (

        COMMON12_PLATT
        .predict_proba(

            logit22(
                raw
            )[:, None]

        )[:, 1]
    )


    return (
        calibrated,
        p_hgb,
        p_cat,
        p_xgb,
    )


def metrics22(
    y,
    p,
):

    pred = (
        p
        >=
        LOCKED_COMMON12_THRESHOLD
    ).astype(int)


    return {

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                p
            ),

        "ECE":
            ece22(
                y,
                p
            ),

        "F1":
            f1_score(
                y,
                pred,
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                y,
                pred
            ),

        "Mean_Probability":
            p.mean(),
    }


# ============================================================
# 22A.5 CLEAN BASELINE
# ============================================================

P_CLEAN_22, _, _, _ = (
    predict_frozen22(
        X22
    )
)


BASELINE_22 = metrics22(
    y22,
    P_CLEAN_22
)


rows22 = [

    {
        "Corruption":
            "Clean",

        "Level":
            0.0,

        **BASELINE_22
    }
]


print(
    "Clean PR:",
    BASELINE_22[
        "PR_AUC"
    ]
)


print(
    "Clean ROC:",
    BASELINE_22[
        "ROC_AUC"
    ]
)


# ============================================================
# 22A.6 CONTINUOUS NOISE
#
# Add Gaussian noise relative to 2015 SD.
# Clip medically / structurally valid ranges afterward.
# ============================================================

rng22 = np.random.default_rng(
    SEED + 220000
)


noise_levels = [
    0.05,
    0.10,
    0.20,
]


for level in noise_levels:

    Xc = X22.copy()


    for feature in CONTINUOUS_22:

        noise = rng22.normal(

            loc=0.0,

            scale=
                level
                *
                TRAIN_SDS_22[
                    feature
                ],

            size=len(
                Xc
            )
        )


        Xc[
            feature
        ] = (

            Xc[
                feature
            ]
            +
            noise
        )


    # valid ranges
    Xc["BMI"] = (
        Xc["BMI"]
        .clip(
            10,
            100
        )
    )


    Xc["MentHlth"] = (
        Xc["MentHlth"]
        .clip(
            0,
            30
        )
    )


    Xc["PhysHlth"] = (
        Xc["PhysHlth"]
        .clip(
            0,
            30
        )
    )


    p, _, _, _ = (
        predict_frozen22(
            Xc
        )
    )


    rows22.append({

        "Corruption":
            "Continuous_Noise",

        "Level":
            level,

        **metrics22(
            y22,
            p
        )
    })


# ============================================================
# 22A.7 RANDOM FEATURE MASKING
#
# Replace selected values with 2015 training medians.
#
# This simulates unavailable / missing questionnaire answers.
# ============================================================

mask_levels = [
    0.05,
    0.10,
    0.20,
]


for level in mask_levels:

    Xc = X22.copy()


    mask = (

        rng22.random(
            Xc.shape
        )

        <
        level
    )


    values = (
        Xc
        .to_numpy(
            dtype=float
        )
    )


    medians = (

        TRAIN_MEDIANS_22[
            Xc.columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    values = np.where(

        mask,

        medians[
            None,
            :
        ],

        values
    )


    Xc = pd.DataFrame(

        values,

        columns=X22.columns
    )


    p, _, _, _ = (
        predict_frozen22(
            Xc
        )
    )


    rows22.append({

        "Corruption":
            "Feature_Masking",

        "Level":
            level,

        **metrics22(
            y22,
            p
        )
    })


# ============================================================
# 22A.8 CATEGORICAL CORRUPTION
#
# Binary:
#   0 <-> 1
#
# Ordinal:
#   randomly +/-1, clipped
#
# Diabetes:
#   change to a different valid state {0,1,2}
# ============================================================

cat_levels = [
    0.05,
    0.10,
    0.20,
]


for level in cat_levels:

    Xc = X22.copy()


    # --------------------------------------------------------
    # Binary
    # --------------------------------------------------------

    for feature in BINARY_22:

        corrupt = (
            rng22.random(
                len(
                    Xc
                )
            )
            <
            level
        )


        Xc.loc[
            corrupt,
            feature
        ] = (

            1
            -
            Xc.loc[
                corrupt,
                feature
            ]
        )


    # --------------------------------------------------------
    # Ordinal
    # --------------------------------------------------------

    ordinal_limits = {

        "GenHlth":
            (
                1,
                5
            ),

        "Age":
            (
                1,
                13
            ),
    }


    for feature in ORDINAL_22:

        corrupt = (
            rng22.random(
                len(
                    Xc
                )
            )
            <
            level
        )


        n_corrupt = int(
            corrupt.sum()
        )


        direction = rng22.choice(

            [
                -1,
                1
            ],

            size=n_corrupt
        )


        Xc.loc[
            corrupt,
            feature
        ] = (

            Xc.loc[
                corrupt,
                feature
            ]
            .to_numpy()

            +
            direction
        )


        lo, hi = (
            ordinal_limits[
                feature
            ]
        )


        Xc[
            feature
        ] = (

            Xc[
                feature
            ]
            .clip(
                lo,
                hi
            )
        )


    # --------------------------------------------------------
    # Diabetes
    # --------------------------------------------------------

    corrupt = (
        rng22.random(
            len(
                Xc
            )
        )
        <
        level
    )


    current = (

        Xc.loc[
            corrupt,
            "Diabetes"
        ]
        .astype(int)
        .to_numpy()
    )


    replacement = []


    for value in current:

        options = [
            x
            for x in [
                0,
                1,
                2
            ]
            if x != value
        ]


        replacement.append(

            rng22.choice(
                options
            )
        )


    Xc.loc[
        corrupt,
        "Diabetes"
    ] = replacement


    p, _, _, _ = (
        predict_frozen22(
            Xc
        )
    )


    rows22.append({

        "Corruption":
            "Categorical_Corruption",

        "Level":
            level,

        **metrics22(
            y22,
            p
        )
    })


# ============================================================
# 22A.9 RESULTS
# ============================================================

ROBUSTNESS_22A = pd.DataFrame(
    rows22
)


for metric in [

    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",
    "F1",
    "MCC",
]:

    ROBUSTNESS_22A[
        f"Delta_{metric}"
    ] = (

        ROBUSTNESS_22A[
            metric
        ]

        -

        BASELINE_22[
            metric
        ]
    )


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22A — EXTERNAL ROBUSTNESS RESULTS"
)

print(
    "=" * 135
)


display(
    ROBUSTNESS_22A
)


# ============================================================
# 22A.10 RELATIVE PERFORMANCE RETENTION
# ============================================================

ROBUSTNESS_RETENTION_22A = (

    ROBUSTNESS_22A[
        [
            "Corruption",
            "Level",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "ECE",
        ]
    ]
    .copy()
)


ROBUSTNESS_RETENTION_22A[
    "PR_Retention_pct"
] = (

    100

    *
    ROBUSTNESS_RETENTION_22A[
        "PR_AUC"
    ]

    /
    BASELINE_22[
        "PR_AUC"
    ]
)


ROBUSTNESS_RETENTION_22A[
    "ROC_Retention_pct"
] = (

    100

    *
    ROBUSTNESS_RETENTION_22A[
        "ROC_AUC"
    ]

    /
    BASELINE_22[
        "ROC_AUC"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22A — ROBUSTNESS RETENTION"
)

print(
    "=" * 135
)


display(
    ROBUSTNESS_RETENTION_22A
)


# ============================================================
# 22A.11 WORST-CASE SUMMARY
# ============================================================

nonclean = (

    ROBUSTNESS_22A[
        ROBUSTNESS_22A[
            "Corruption"
        ]
        !=
        "Clean"
    ]
)


worst_pr = (
    nonclean
    .sort_values(
        "PR_AUC"
    )
    .iloc[0]
)


worst_roc = (
    nonclean
    .sort_values(
        "ROC_AUC"
    )
    .iloc[0]
)


WORST_22A = pd.DataFrame([

    {
        "Metric":
            "PR_AUC",

        "Corruption":
            worst_pr[
                "Corruption"
            ],

        "Level":
            worst_pr[
                "Level"
            ],

        "Value":
            worst_pr[
                "PR_AUC"
            ],

        "Clean_Value":
            BASELINE_22[
                "PR_AUC"
            ],

        "Delta":
            worst_pr[
                "PR_AUC"
            ]
            -
            BASELINE_22[
                "PR_AUC"
            ],
    },

    {
        "Metric":
            "ROC_AUC",

        "Corruption":
            worst_roc[
                "Corruption"
            ],

        "Level":
            worst_roc[
                "Level"
            ],

        "Value":
            worst_roc[
                "ROC_AUC"
            ],

        "Clean_Value":
            BASELINE_22[
                "ROC_AUC"
            ],

        "Delta":
            worst_roc[
                "ROC_AUC"
            ]
            -
            BASELINE_22[
                "ROC_AUC"
            ],
    },
])


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22A — WORST-CASE ROBUSTNESS"
)

print(
    "=" * 135
)


display(
    WORST_22A
)


# ============================================================
# 22A.12 SAVE
# ============================================================

save_csv(
    ROBUSTNESS_22A,
    TAB_DIR
    / "module22a_external_robustness.csv"
)


save_csv(
    ROBUSTNESS_RETENTION_22A,
    TAB_DIR
    / "module22a_robustness_retention.csv"
)


save_csv(
    WORST_22A,
    TAB_DIR
    / "module22a_worst_case.csv"
)


save_json(

    {
        "module":
            "Frozen external robustness audit",

        "dataset":
            "BRFSS-2020",

        "corruptions": {

            "continuous_noise": [
                0.05,
                0.10,
                0.20
            ],

            "feature_masking": [
                0.05,
                0.10,
                0.20
            ],

            "categorical_corruption": [
                0.05,
                0.10,
                0.20
            ],
        },

        "retraining":
            False,

        "recalibration":
            False,

        "threshold_adjustment":
            False,
    },

    LOG_DIR
    / "module22a_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nFrozen-model robustness evaluation only."
)

Clean PR: 0.3390577333635292
Clean ROC: 0.8405537613853831

MODULE 22A — EXTERNAL ROBUSTNESS RESULTS


,Corruption,Level,PR_AUC,ROC_AUC,Brier,NLL,ECE,F1,MCC,Mean_Probability,Delta_PR_AUC,Delta_ROC_AUC,Delta_Brier,Delta_NLL,Delta_ECE,Delta_F1,Delta_MCC
0,Clean,0.00,0.339058,0.840554,0.065841,0.226781,0.002159,0.390918,0.330948,0.085322,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Continuous_Noise,0.05,0.339116,0.840603,0.065837,0.226766,0.002298,0.390926,0.331047,0.085550,0.000058,0.000049,-0.000004,-0.000016,0.000139,0.000008,0.000099
2,Continuous_Noise,0.10,0.338739,0.840499,0.065856,0.226830,0.002208,0.392334,0.332835,0.085913,-0.000318,-0.000055,0.000015,0.000049,0.000049,0.001415,0.001887
3,Continuous_Noise,0.20,0.338002,0.840186,0.065894,0.226983,0.002468,0.391912,0.332696,0.086684,-0.001055,-0.000368,0.000053,0.000201,0.000309,0.000994,0.001748
4,Feature_Masking,0.05,0.330815,0.834725,0.066408,0.229684,0.005921,0.383033,0.320329,0.080224,-0.008243,-0.005829,0.000567,0.002903,0.003762,-0.007885,-0.010620
5,Feature_Masking,0.10,0.318934,0.827307,0.067220,0.233744,0.010760,0.370105,0.305755,0.075285,-0.020124,-0.013247,0.001379,0.006963,0.008602,-0.020813,-0.025193
6,Feature_Masking,0.20,0.299758,0.811033,0.068891,0.243374,0.020351,0.339669,0.277704,0.065406,-0.039299,-0.029520,0.003050,0.016593,0.018192,-0.051249,-0.053244
7,Categorical_Corruption,0.05,0.313452,0.825786,0.067592,0.233972,0.008732,0.370198,0.308965,0.092690,-0.025606,-0.014768,0.001751,0.007191,0.006573,-0.020720,-0.021984
8,Categorical_Corruption,0.10,0.289029,0.811180,0.069509,0.241444,0.015898,0.349786,0.287574,0.100595,-0.050028,-0.029373,0.003668,0.014663,0.013739,-0.041132,-0.043374
9,Categorical_Corruption,0.20,0.252702,0.779832,0.073840,0.258080,0.033264,0.310983,0.246750,0.117868,-0.086355,-0.060721,0.007999,0.031299,0.031105,-0.079936,-0.084198



MODULE 22A — ROBUSTNESS RETENTION


,Corruption,Level,PR_AUC,ROC_AUC,Brier,ECE,PR_Retention_pct,ROC_Retention_pct
0,Clean,0.00,0.339058,0.840554,0.065841,0.002159,100.000000,100.000000
1,Continuous_Noise,0.05,0.339116,0.840603,0.065837,0.002298,100.017134,100.005844
2,Continuous_Noise,0.10,0.338739,0.840499,0.065856,0.002208,99.906082,99.993454
3,Continuous_Noise,0.20,0.338002,0.840186,0.065894,0.002468,99.688761,99.956206
4,Feature_Masking,0.05,0.330815,0.834725,0.066408,0.005921,97.568799,99.306543
5,Feature_Masking,0.10,0.318934,0.827307,0.067220,0.010760,94.064867,98.424069
6,Feature_Masking,0.20,0.299758,0.811033,0.068891,0.020351,88.409258,96.487987
7,Categorical_Corruption,0.05,0.313452,0.825786,0.067592,0.008732,92.448012,98.243067
8,Categorical_Corruption,0.10,0.289029,0.811180,0.069509,0.015898,85.244906,96.505464
9,Categorical_Corruption,0.20,0.252702,0.779832,0.073840,0.033264,74.530788,92.776035



MODULE 22A — WORST-CASE ROBUSTNESS


,Metric,Corruption,Level,Value,Clean_Value,Delta
0,PR_AUC,Categorical_Corruption,0.2,0.252702,0.339058,-0.086355
1,ROC_AUC,Categorical_Corruption,0.2,0.779832,0.840554,-0.060721



MODULE 22A COMPLETE

Frozen-model robustness evaluation only.


In [63]:
# ============================================================
# MODULE 22B — LPNS RELIABILITY ROBUSTNESS UNDER CORRUPTION
#
# PURPOSE
# ------------------------------------------------------------
# Test whether the frozen LPNS trust mechanism remains useful
# when the external inputs are corrupted.
#
# Uses:
#   frozen 2015 reference bank
#   frozen 2015 LPNS scaler
#   frozen 2015 reliability heads
#   frozen disease threshold
#
# NO retraining.
# NO recalibration.
# NO threshold adjustment.
#
# Conditions:
#   Clean
#   Continuous noise: 0.05 / 0.10 / 0.20
#   Feature masking:   0.05 / 0.10 / 0.20
#   Categorical corruption: 0.05 / 0.10 / 0.20
#
# Reliability metrics:
#   Error PR-AUC
#   Error ROC-AUC
#   AURC
#
# Selective review:
#   5%
#   10%
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 22B.1 REQUIREMENTS
# ============================================================

required_22b = [

    # external
    "X20",
    "y20",

    # frozen disease system
    "predict_frozen22",
    "LOCKED_COMMON12_THRESHOLD",

    # LPNS frozen objects
    "X_reference20d",
    "P_reference20d",
    "nbr20d",
    "scaler20d",
    "scale_idx12",
    "discrete_idx12",
    "LPNS12_ALL",
    "CONFIG20D",
    "lpns_features20d",
    "impute_lpns20d",

    # frozen reliability scalers/heads
    "scale_conf",
    "scale_dis",
    "scale_full",

    "REL_CONF20D",
    "REL_DIS20D",
    "REL_LPNS20D",

    # helpers
    "build_base_reliability20d",
    "aurc20d",

    # corruption config
    "TRAIN_MEDIANS_22",
    "TRAIN_SDS_22",
    "CONTINUOUS_22",
    "BINARY_22",
    "ORDINAL_22",

    # misc
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_22b
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 22B objects: "
        + ", ".join(missing)
    )


X22B_BASE = (
    X20
    .copy()
    .reset_index(drop=True)
)


y22b = np.asarray(
    y20,
    dtype=int
)


# ============================================================
# 22B.2 RELIABILITY SCORING FUNCTION
# ============================================================

def evaluate_reliability22b(
    X_input,
    condition,
    level,
):

    # --------------------------------------------------------
    # Disease predictions
    # --------------------------------------------------------

    (
        p_consensus,
        p_hgb,
        p_cat,
        p_xgb,

    ) = predict_frozen22(
        X_input
    )


    expert_matrix = np.column_stack([

        p_hgb,
        p_cat,
        p_xgb,
    ])


    pred = (
        p_consensus
        >=
        LOCKED_COMMON12_THRESHOLD
    ).astype(int)


    error = (
        pred
        !=
        y22b
    ).astype(int)


    # --------------------------------------------------------
    # Confidence + disagreement
    # --------------------------------------------------------

    CONF, DIS = (
        build_base_reliability20d(

            p_consensus,

            expert_matrix,

            LOCKED_COMMON12_THRESHOLD
        )
    )


    # --------------------------------------------------------
    # LPNS distance representation
    # --------------------------------------------------------

    X_lpns_raw = (

        X_input[
            LPNS12_ALL
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    X_distance = (
        X_lpns_raw
        .copy()
    )


    X_distance[
        :,
        scale_idx12
    ] = (

        scaler20d
        .transform(

            X_distance[
                :,
                scale_idx12
            ]
        )
    )


    X_distance[
        :,
        discrete_idx12
    ] *= (
        CONFIG20D[
            "discrete_weight"
        ]
    )


    # --------------------------------------------------------
    # Query frozen 2015 reference
    # --------------------------------------------------------

    dist, idx = (
        nbr20d.kneighbors(

            X_distance,

            return_distance=True
        )
    )


    LPNS, VALID, NCOUNT = (
        lpns_features20d(

            expert_matrix,
            P_reference20d,

            idx,
            dist,

            LOCKED_COMMON12_THRESHOLD
        )
    )


    LPNS_I = (
        impute_lpns20d(
            LPNS
        )
    )


    FULL = np.column_stack([

        DIS,
        LPNS_I,
    ])


    # --------------------------------------------------------
    # Frozen feature scaling
    # --------------------------------------------------------

    CONF_S = (
        scale_conf
        .transform(
            CONF
        )
    )


    DIS_S = (
        scale_dis
        .transform(
            DIS
        )
    )


    FULL_S = (
        scale_full
        .transform(
            FULL
        )
    )


    # --------------------------------------------------------
    # Frozen reliability heads
    # --------------------------------------------------------

    risk_conf = (

        REL_CONF20D
        .predict_proba(
            CONF_S
        )[:, 1]
    )


    risk_dis = (

        REL_DIS20D
        .predict_proba(
            DIS_S
        )[:, 1]
    )


    risk_lpns = (

        REL_LPNS20D
        .predict_proba(
            FULL_S
        )[:, 1]
    )


    # --------------------------------------------------------
    # Reliability metrics
    # --------------------------------------------------------

    rows = []


    for method, risk in [

        (
            "Confidence",
            risk_conf
        ),

        (
            "Confidence+Disagreement",
            risk_dis
        ),

        (
            "LPNS",
            risk_lpns
        ),
    ]:

        rows.append({

            "Corruption":
                condition,

            "Level":
                level,

            "Method":
                method,

            "Error_Prevalence":
                error.mean(),

            "Error_PR_AUC":
                average_precision_score(
                    error,
                    risk
                ),

            "Error_ROC_AUC":
                roc_auc_score(
                    error,
                    risk
                ),

            "AURC":
                aurc20d(
                    error,
                    risk
                ),

            "LPNS_Coverage":
                VALID.mean(),

            "Mean_Neighbors":
                NCOUNT.mean(),
        })


    # --------------------------------------------------------
    # Selective review
    # --------------------------------------------------------

    selective_rows = []


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    for review_rate in [
        0.05,
        0.10,
    ]:

        n_review = max(
            1,
            int(
                round(
                    review_rate
                    *
                    len(error)
                )
            )
        )


        for method, risk in [

            (
                "Confidence",
                risk_conf
            ),

            (
                "Confidence+Disagreement",
                risk_dis
            ),

            (
                "LPNS",
                risk_lpns
            ),
        ]:

            order = np.argsort(
                -risk
            )


            review_idx = order[
                :n_review
            ]


            errors_found = int(
                error[
                    review_idx
                ].sum()
            )


            selective_rows.append({

                "Corruption":
                    condition,

                "Level":
                    level,

                "Method":
                    method,

                "Review_Rate":
                    review_rate,

                "Review_N":
                    n_review,

                "Errors_Captured":
                    errors_found
                    /
                    total_errors,

                "Errors_Per_100_Reviews":
                    100.0
                    *
                    errors_found
                    /
                    n_review,
            })


    return (
        pd.DataFrame(rows),
        pd.DataFrame(selective_rows),
    )


# ============================================================
# 22B.3 CORRUPTION GENERATOR
# ============================================================

rng22b = np.random.default_rng(
    SEED + 221000
)


def make_corruption22b(
    corruption,
    level,
):

    Xc = X22B_BASE.copy()


    # --------------------------------------------------------
    # CLEAN
    # --------------------------------------------------------

    if corruption == "Clean":

        return Xc


    # --------------------------------------------------------
    # CONTINUOUS NOISE
    # --------------------------------------------------------

    if corruption == "Continuous_Noise":

        for feature in CONTINUOUS_22:

            noise = rng22b.normal(

                0.0,

                level
                *
                TRAIN_SDS_22[
                    feature
                ],

                size=len(Xc)
            )


            Xc[
                feature
            ] = (

                Xc[
                    feature
                ]
                +
                noise
            )


        Xc["BMI"] = (
            Xc["BMI"]
            .clip(
                10,
                100
            )
        )


        Xc["MentHlth"] = (
            Xc["MentHlth"]
            .clip(
                0,
                30
            )
        )


        Xc["PhysHlth"] = (
            Xc["PhysHlth"]
            .clip(
                0,
                30
            )
        )


        return Xc


    # --------------------------------------------------------
    # FEATURE MASKING
    # --------------------------------------------------------

    if corruption == "Feature_Masking":

        mask = (

            rng22b.random(
                Xc.shape
            )

            <
            level
        )


        arr = (
            Xc
            .to_numpy(
                dtype=float
            )
        )


        med = (

            TRAIN_MEDIANS_22[
                Xc.columns
            ]
            .to_numpy(
                dtype=float
            )
        )


        arr = np.where(

            mask,

            med[
                None,
                :
            ],

            arr
        )


        return pd.DataFrame(

            arr,

            columns=Xc.columns
        )


    # --------------------------------------------------------
    # CATEGORICAL CORRUPTION
    # --------------------------------------------------------

    if corruption == "Categorical_Corruption":

        # binary flips

        for feature in BINARY_22:

            corrupt = (

                rng22b.random(
                    len(Xc)
                )

                <
                level
            )


            Xc.loc[
                corrupt,
                feature
            ] = (

                1

                -

                Xc.loc[
                    corrupt,
                    feature
                ]
            )


        # ordinal +/-1

        limits = {

            "GenHlth":
                (
                    1,
                    5
                ),

            "Age":
                (
                    1,
                    13
                ),
        }


        for feature in ORDINAL_22:

            corrupt = (

                rng22b.random(
                    len(Xc)
                )

                <
                level
            )


            n_corrupt = int(
                corrupt.sum()
            )


            direction = rng22b.choice(

                [
                    -1,
                    1
                ],

                size=n_corrupt
            )


            Xc.loc[
                corrupt,
                feature
            ] = (

                Xc.loc[
                    corrupt,
                    feature
                ]
                .to_numpy()

                +
                direction
            )


            lo, hi = limits[
                feature
            ]


            Xc[
                feature
            ] = (

                Xc[
                    feature
                ]
                .clip(
                    lo,
                    hi
                )
            )


        # diabetes -> different valid state

        corrupt = (

            rng22b.random(
                len(Xc)
            )

            <
            level
        )


        current = (

            Xc.loc[
                corrupt,
                "Diabetes"
            ]
            .astype(int)
            .to_numpy()
        )


        replacement = []


        for value in current:

            candidates = [

                x
                for x
                in [
                    0,
                    1,
                    2
                ]

                if x != value
            ]


            replacement.append(

                rng22b.choice(
                    candidates
                )
            )


        Xc.loc[
            corrupt,
            "Diabetes"
        ] = replacement


        return Xc


    raise ValueError(
        f"Unknown corruption: {corruption}"
    )


# ============================================================
# 22B.4 RUN ALL CONDITIONS
# ============================================================

conditions22b = [

    (
        "Clean",
        0.0
    ),

    (
        "Continuous_Noise",
        0.05
    ),

    (
        "Continuous_Noise",
        0.10
    ),

    (
        "Continuous_Noise",
        0.20
    ),

    (
        "Feature_Masking",
        0.05
    ),

    (
        "Feature_Masking",
        0.10
    ),

    (
        "Feature_Masking",
        0.20
    ),

    (
        "Categorical_Corruption",
        0.05
    ),

    (
        "Categorical_Corruption",
        0.10
    ),

    (
        "Categorical_Corruption",
        0.20
    ),
]


reliability_frames = []
selective_frames = []


for corruption, level in conditions22b:

    print(
        f"Running: {corruption} | {level}"
    )


    Xc = make_corruption22b(
        corruption,
        level
    )


    rel_df, sel_df = (
        evaluate_reliability22b(

            Xc,

            corruption,

            level,
        )
    )


    reliability_frames.append(
        rel_df
    )


    selective_frames.append(
        sel_df
    )


ROBUST_REL_22B = pd.concat(

    reliability_frames,

    ignore_index=True
)


ROBUST_SEL_22B = pd.concat(

    selective_frames,

    ignore_index=True
)


# ============================================================
# 22B.5 LPNS INCREMENTAL VALUE
# ============================================================

gain_rows = []


for (
    corruption,
    level

), group in (
    ROBUST_REL_22B
    .groupby(
        [
            "Corruption",
            "Level",
        ]
    )
):

    baseline = (

        group[
            group[
                "Method"
            ]
            ==
            "Confidence+Disagreement"
        ]
        .iloc[0]
    )


    lpns = (

        group[
            group[
                "Method"
            ]
            ==
            "LPNS"
        ]
        .iloc[0]
    )


    gain_rows.append({

        "Corruption":
            corruption,

        "Level":
            level,

        "Error_Prevalence":
            lpns[
                "Error_Prevalence"
            ],

        "Baseline_Error_PR":
            baseline[
                "Error_PR_AUC"
            ],

        "LPNS_Error_PR":
            lpns[
                "Error_PR_AUC"
            ],

        "Delta_Error_PR":
            lpns[
                "Error_PR_AUC"
            ]
            -
            baseline[
                "Error_PR_AUC"
            ],

        "Baseline_Error_ROC":
            baseline[
                "Error_ROC_AUC"
            ],

        "LPNS_Error_ROC":
            lpns[
                "Error_ROC_AUC"
            ],

        "Delta_Error_ROC":
            lpns[
                "Error_ROC_AUC"
            ]
            -
            baseline[
                "Error_ROC_AUC"
            ],

        "Baseline_AURC":
            baseline[
                "AURC"
            ],

        "LPNS_AURC":
            lpns[
                "AURC"
            ],

        "Delta_AURC":
            lpns[
                "AURC"
            ]
            -
            baseline[
                "AURC"
            ],

        "LPNS_Coverage":
            lpns[
                "LPNS_Coverage"
            ],

        "Mean_Neighbors":
            lpns[
                "Mean_Neighbors"
            ],
    })


GAIN_22B = pd.DataFrame(
    gain_rows
)


# ============================================================
# 22B.6 SELECTIVE INCREMENT
# ============================================================

selective_gain_rows = []


for (
    corruption,
    level,
    review_rate

), group in (
    ROBUST_SEL_22B
    .groupby(
        [
            "Corruption",
            "Level",
            "Review_Rate",
        ]
    )
):

    base = (

        group[
            group[
                "Method"
            ]
            ==
            "Confidence+Disagreement"
        ]
        .iloc[0]
    )


    lpns = (

        group[
            group[
                "Method"
            ]
            ==
            "LPNS"
        ]
        .iloc[0]
    )


    selective_gain_rows.append({

        "Corruption":
            corruption,

        "Level":
            level,

        "Review_Rate":
            review_rate,

        "Baseline_Error_Capture":
            base[
                "Errors_Captured"
            ],

        "LPNS_Error_Capture":
            lpns[
                "Errors_Captured"
            ],

        "Delta_Error_Capture":
            lpns[
                "Errors_Captured"
            ]
            -
            base[
                "Errors_Captured"
            ],

        "Baseline_Errors_Per_100":
            base[
                "Errors_Per_100_Reviews"
            ],

        "LPNS_Errors_Per_100":
            lpns[
                "Errors_Per_100_Reviews"
            ],

        "Delta_Errors_Per_100":
            lpns[
                "Errors_Per_100_Reviews"
            ]
            -
            base[
                "Errors_Per_100_Reviews"
            ],
    })


SELECTIVE_GAIN_22B = pd.DataFrame(
    selective_gain_rows
)


# ============================================================
# 22B.7 PRINT RESULTS
# ============================================================

print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22B — RELIABILITY UNDER CORRUPTION"
)

print(
    "=" * 135
)


display(
    ROBUST_REL_22B
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22B — LPNS INCREMENTAL VALUE UNDER CORRUPTION"
)

print(
    "=" * 135
)


display(
    GAIN_22B
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22B — LOW-BUDGET SELECTIVE REVIEW UNDER CORRUPTION"
)

print(
    "=" * 135
)


display(
    SELECTIVE_GAIN_22B
)


# ============================================================
# 22B.8 SUMMARY
# ============================================================

SUMMARY_22B = (

    GAIN_22B[
        [
            "Corruption",
            "Level",
            "Error_Prevalence",
            "Delta_Error_PR",
            "Delta_Error_ROC",
            "Delta_AURC",
            "LPNS_Coverage",
        ]
    ]
    .copy()
)


SUMMARY_22B[
    "LPNS_PR_Better"
] = (
    SUMMARY_22B[
        "Delta_Error_PR"
    ]
    >
    0
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22B — LPNS ROBUSTNESS SUMMARY"
)

print(
    "=" * 135
)


display(
    SUMMARY_22B
)


# ============================================================
# 22B.9 SAVE
# ============================================================

save_csv(
    ROBUST_REL_22B,
    TAB_DIR
    / "module22b_reliability_under_corruption.csv"
)


save_csv(
    GAIN_22B,
    TAB_DIR
    / "module22b_lpns_gain_under_corruption.csv"
)


save_csv(
    ROBUST_SEL_22B,
    TAB_DIR
    / "module22b_selective_under_corruption.csv"
)


save_csv(
    SELECTIVE_GAIN_22B,
    TAB_DIR
    / "module22b_selective_gain_under_corruption.csv"
)


save_csv(
    SUMMARY_22B,
    TAB_DIR
    / "module22b_lpns_robustness_summary.csv"
)


save_json(

    {
        "module":
            "LPNS external reliability robustness",

        "dataset":
            "BRFSS-2020",

        "reliability_model":
            "Frozen BRFSS-2015 LPNS reliability head",

        "corruptions": [
            "Continuous noise",
            "Feature masking",
            "Categorical corruption",
        ],

        "primary_metric":
            "Error PR-AUC",

        "review_rates": [
            0.05,
            0.10
        ],

        "retraining":
            False,

        "recalibration":
            False,

        "threshold_adjustment":
            False,
    },

    LOG_DIR
    / "module22b_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nFrozen LPNS robustness evaluation only."
)

Running: Clean | 0.0
Running: Continuous_Noise | 0.05
Running: Continuous_Noise | 0.1
Running: Continuous_Noise | 0.2
Running: Feature_Masking | 0.05
Running: Feature_Masking | 0.1
Running: Feature_Masking | 0.2
Running: Categorical_Corruption | 0.05
Running: Categorical_Corruption | 0.1
Running: Categorical_Corruption | 0.2

MODULE 22B — RELIABILITY UNDER CORRUPTION


,Corruption,Level,Method,Error_Prevalence,Error_PR_AUC,Error_ROC_AUC,AURC,LPNS_Coverage,Mean_Neighbors
0,Clean,0.00,Confidence,0.138076,0.571734,0.905746,0.026288,0.999991,19.999384
1,Clean,0.00,Confidence+Disagreement,0.138076,0.575812,0.906016,0.026255,0.999991,19.999384
2,Clean,0.00,LPNS,0.138076,0.587949,0.905927,0.026337,0.999991,19.999384
3,Continuous_Noise,0.05,Confidence,0.138542,0.572690,0.906026,0.026361,0.999994,19.999393
4,Continuous_Noise,0.05,Confidence+Disagreement,0.138542,0.577527,0.906302,0.026327,0.999994,19.999393
5,Continuous_Noise,0.05,LPNS,0.138542,0.589167,0.906164,0.026371,0.999994,19.999393
6,Continuous_Noise,0.10,Confidence,0.138861,0.573810,0.906183,0.026416,0.999994,19.999378
7,Continuous_Noise,0.10,Confidence+Disagreement,0.138861,0.578939,0.906471,0.026380,0.999994,19.999378
8,Continuous_Noise,0.10,LPNS,0.138861,0.590441,0.906282,0.026472,0.999994,19.999378
9,Continuous_Noise,0.20,Confidence,0.140377,0.577559,0.907012,0.026678,0.999994,19.999334



MODULE 22B — LPNS INCREMENTAL VALUE UNDER CORRUPTION


,Corruption,Level,Error_Prevalence,Baseline_Error_PR,LPNS_Error_PR,Delta_Error_PR,Baseline_Error_ROC,LPNS_Error_ROC,Delta_Error_ROC,Baseline_AURC,LPNS_AURC,Delta_AURC,LPNS_Coverage,Mean_Neighbors
0,Categorical_Corruption,0.05,0.151941,0.607465,0.626293,0.018828,0.907399,0.907735,0.000336,0.029800,0.029773,-0.000026,0.999991,19.999384
1,Categorical_Corruption,0.10,0.168042,0.644587,0.662871,0.018285,0.909919,0.910383,0.000464,0.033903,0.033863,-0.000040,0.999994,19.999328
2,Categorical_Corruption,0.20,0.207586,0.708617,0.724909,0.016292,0.914756,0.915299,0.000543,0.045196,0.045140,-0.000057,0.999984,19.999243
3,Clean,0.00,0.138076,0.575812,0.587949,0.012137,0.906016,0.905927,-0.000088,0.026255,0.026337,0.000082,0.999991,19.999384
4,Continuous_Noise,0.05,0.138542,0.577527,0.589167,0.011639,0.906302,0.906164,-0.000138,0.026327,0.026371,0.000044,0.999994,19.999393
5,Continuous_Noise,0.10,0.138861,0.578939,0.590441,0.011502,0.906471,0.906282,-0.000188,0.026380,0.026472,0.000092,0.999994,19.999378
6,Continuous_Noise,0.20,0.140377,0.582252,0.595312,0.013060,0.907295,0.907348,0.000053,0.026643,0.026678,0.000036,0.999994,19.999334
7,Feature_Masking,0.05,0.131112,0.556269,0.566196,0.009927,0.897932,0.897679,-0.000253,0.026056,0.026172,0.000117,0.999991,19.999346
8,Feature_Masking,0.10,0.124864,0.533938,0.542035,0.008097,0.888647,0.888126,-0.000521,0.026103,0.026212,0.000109,0.999991,19.999387
9,Feature_Masking,0.20,0.113751,0.487127,0.490264,0.003137,0.867253,0.866402,-0.000851,0.026733,0.026996,0.000263,0.999991,19.999450



MODULE 22B — LOW-BUDGET SELECTIVE REVIEW UNDER CORRUPTION


,Corruption,Level,Review_Rate,Baseline_Error_Capture,LPNS_Error_Capture,Delta_Error_Capture,Baseline_Errors_Per_100,LPNS_Errors_Per_100,Delta_Errors_Per_100
0,Categorical_Corruption,0.05,0.05,0.212040,0.227413,0.015374,64.434021,69.105691,4.671670
1,Categorical_Corruption,0.05,0.10,0.448446,0.457296,0.008850,68.136335,69.480926,1.344590
2,Categorical_Corruption,0.10,0.05,0.203279,0.217682,0.014403,68.317699,73.158224,4.840525
3,Categorical_Corruption,0.10,0.10,0.421072,0.431158,0.010086,70.756723,72.451532,1.694809
4,Categorical_Corruption,0.20,0.05,0.176802,0.191941,0.015139,73.402126,79.687305,6.285178
5,Categorical_Corruption,0.20,0.10,0.359795,0.371198,0.011403,74.687305,77.054409,2.367104
6,Clean,0.00,0.05,0.223458,0.236661,0.013203,61.707317,65.353346,3.646029
7,Clean,0.00,0.10,0.479844,0.484011,0.004167,66.253909,66.829268,0.575360
8,Continuous_Noise,0.05,0.05,0.223045,0.236294,0.013249,61.801126,65.472170,3.671044
9,Continuous_Noise,0.05,0.10,0.478569,0.482090,0.003521,66.300813,66.788618,0.487805



MODULE 22B — LPNS ROBUSTNESS SUMMARY


,Corruption,Level,Error_Prevalence,Delta_Error_PR,Delta_Error_ROC,Delta_AURC,LPNS_Coverage,LPNS_PR_Better
0,Categorical_Corruption,0.05,0.151941,0.018828,0.000336,-0.000026,0.999991,True
1,Categorical_Corruption,0.10,0.168042,0.018285,0.000464,-0.000040,0.999994,True
2,Categorical_Corruption,0.20,0.207586,0.016292,0.000543,-0.000057,0.999984,True
3,Clean,0.00,0.138076,0.012137,-0.000088,0.000082,0.999991,True
4,Continuous_Noise,0.05,0.138542,0.011639,-0.000138,0.000044,0.999994,True
5,Continuous_Noise,0.10,0.138861,0.011502,-0.000188,0.000092,0.999994,True
6,Continuous_Noise,0.20,0.140377,0.013060,0.000053,0.000036,0.999994,True
7,Feature_Masking,0.05,0.131112,0.009927,-0.000253,0.000117,0.999991,True
8,Feature_Masking,0.10,0.124864,0.008097,-0.000521,0.000109,0.999991,True
9,Feature_Masking,0.20,0.113751,0.003137,-0.000851,0.000263,0.999991,True



MODULE 22B COMPLETE

Frozen LPNS robustness evaluation only.


In [64]:
# ============================================================
# MODULE 22C — FINAL LPNS ROBUSTNESS STATISTICAL CONFIRMATION
#
# PURPOSE
# ------------------------------------------------------------
# Statistically confirm the most important Module 22B findings.
#
# Representative conditions:
#
#   1. Clean
#   2. Feature masking = 20%
#   3. Categorical corruption = 20%
#
# Comparisons:
#
#   LPNS vs Confidence + Disagreement
#
# Primary:
#   Error PR-AUC
#
# Secondary:
#   Error ROC-AUC
#   AURC
#
# Operational:
#   5% review:
#       error capture
#       errors per 100 reviews
#
# Bootstrap:
#   paired
#   stratified by current classification-error status
#   1000 repetitions
#
# IMPORTANT
# ------------------------------------------------------------
# We reset the Module 22B RNG and replay ALL corruption
# conditions in the original order so selected conditions are
# identical to Module 22B.
#
# No model fitting.
# No tuning.
# No recalibration.
# No threshold change.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 22C.1 REQUIREMENTS
# ============================================================

required_22c = [

    "X22B_BASE",
    "y22b",

    "predict_frozen22",
    "LOCKED_COMMON12_THRESHOLD",

    "build_base_reliability20d",

    "LPNS12_ALL",
    "scale_idx12",
    "discrete_idx12",
    "scaler20d",
    "CONFIG20D",

    "nbr20d",
    "P_reference20d",

    "lpns_features20d",
    "impute_lpns20d",

    "scale_dis",
    "scale_full",

    "REL_DIS20D",
    "REL_LPNS20D",

    "aurc20d",

    "make_corruption22b",
    "conditions22b",

    "GAIN_22B",
    "SELECTIVE_GAIN_22B",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_22c
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 22C objects: "
        + ", ".join(missing)
    )


# ============================================================
# 22C.2 ROW-LEVEL RELIABILITY OUTPUT
# ============================================================

def get_outputs22c(
    X_input
):

    (
        p_consensus,
        p_hgb,
        p_cat,
        p_xgb,

    ) = predict_frozen22(
        X_input
    )


    experts = np.column_stack([

        p_hgb,
        p_cat,
        p_xgb,
    ])


    pred = (

        p_consensus
        >=
        LOCKED_COMMON12_THRESHOLD

    ).astype(int)


    error = (

        pred
        !=
        y22b

    ).astype(int)


    # --------------------------------------------------------
    # Confidence + disagreement
    # --------------------------------------------------------

    _, DIS = (
        build_base_reliability20d(

            p_consensus,

            experts,

            LOCKED_COMMON12_THRESHOLD
        )
    )


    # --------------------------------------------------------
    # LPNS representation
    # --------------------------------------------------------

    X_lpns = (

        X_input[
            LPNS12_ALL
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    X_distance = (
        X_lpns.copy()
    )


    X_distance[
        :,
        scale_idx12
    ] = (

        scaler20d
        .transform(

            X_distance[
                :,
                scale_idx12
            ]
        )
    )


    X_distance[
        :,
        discrete_idx12
    ] *= (
        CONFIG20D[
            "discrete_weight"
        ]
    )


    dist, idx = (
        nbr20d.kneighbors(

            X_distance,

            return_distance=True
        )
    )


    LPNS, VALID, NCOUNT = (
        lpns_features20d(

            experts,
            P_reference20d,

            idx,
            dist,

            LOCKED_COMMON12_THRESHOLD
        )
    )


    LPNS_I = (
        impute_lpns20d(
            LPNS
        )
    )


    FULL = np.column_stack([

        DIS,
        LPNS_I,
    ])


    # --------------------------------------------------------
    # Frozen reliability models
    # --------------------------------------------------------

    DIS_S = (
        scale_dis
        .transform(
            DIS
        )
    )


    FULL_S = (
        scale_full
        .transform(
            FULL
        )
    )


    risk_dis = (

        REL_DIS20D
        .predict_proba(
            DIS_S
        )[:, 1]
    )


    risk_lpns = (

        REL_LPNS20D
        .predict_proba(
            FULL_S
        )[:, 1]
    )


    return {

        "error":
            error,

        "risk_dis":
            risk_dis,

        "risk_lpns":
            risk_lpns,

        "coverage":
            VALID.mean(),

        "mean_neighbors":
            NCOUNT.mean(),
    }


# ============================================================
# 22C.3 EXACT REPLAY OF MODULE 22B CORRUPTIONS
#
# Reset global RNG used by make_corruption22b.
# ============================================================

rng22b = np.random.default_rng(
    SEED + 221000
)


SELECTED_KEYS_22C = {

    (
        "Clean",
        0.0
    ),

    (
        "Feature_Masking",
        0.20
    ),

    (
        "Categorical_Corruption",
        0.20
    ),
}


ROW_OUTPUTS_22C = {}


for corruption, level in conditions22b:

    print(
        f"Replaying: {corruption} | {level}"
    )


    Xc = make_corruption22b(
        corruption,
        level
    )


    key = (
        corruption,
        float(level)
    )


    if key in SELECTED_KEYS_22C:

        ROW_OUTPUTS_22C[
            key
        ] = get_outputs22c(
            Xc
        )


# ============================================================
# 22C.4 VERIFY POINT ESTIMATES AGAINST MODULE 22B
#
# Must reproduce the existing results before bootstrapping.
# ============================================================

verification_rows = []


for (
    corruption,
    level

), out in (
    ROW_OUTPUTS_22C.items()
):

    error = out[
        "error"
    ]


    risk_dis = out[
        "risk_dis"
    ]


    risk_lpns = out[
        "risk_lpns"
    ]


    current_delta_pr = (

        average_precision_score(
            error,
            risk_lpns
        )

        -

        average_precision_score(
            error,
            risk_dis
        )
    )


    old_row = (

        GAIN_22B[
            (
                GAIN_22B[
                    "Corruption"
                ]
                ==
                corruption
            )
            &
            (
                np.isclose(
                    GAIN_22B[
                        "Level"
                    ],
                    level
                )
            )
        ]
        .iloc[0]
    )


    old_delta_pr = float(
        old_row[
            "Delta_Error_PR"
        ]
    )


    verification_rows.append({

        "Corruption":
            corruption,

        "Level":
            level,

        "Module22B_Delta_PR":
            old_delta_pr,

        "Replayed_Delta_PR":
            current_delta_pr,

        "Absolute_Difference":
            abs(
                current_delta_pr
                -
                old_delta_pr
            ),

        "Reproduced":
            bool(
                np.isclose(
                    current_delta_pr,
                    old_delta_pr,
                    atol=1e-10
                )
            ),
    })


VERIFY_22C = pd.DataFrame(
    verification_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C — REPRODUCTION CHECK"
)

print(
    "=" * 135
)


display(
    VERIFY_22C
)


if not VERIFY_22C[
    "Reproduced"
].all():

    raise RuntimeError(

        "Module 22B corruption replay did not reproduce "
        "the original point estimates. Stop before bootstrap."
    )


# ============================================================
# 22C.5 SELECTIVE REVIEW FUNCTION
# ============================================================

def selective_metric22c(
    error,
    risk,
    review_rate=0.05,
):

    n_review = max(

        1,

        int(
            round(
                len(error)
                *
                review_rate
            )
        )
    )


    idx = np.argsort(
        -risk
    )[
        :n_review
    ]


    errors_found = int(
        error[
            idx
        ].sum()
    )


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    capture = (
        errors_found
        /
        total_errors
    )


    errors100 = (

        100.0
        *
        errors_found
        /
        n_review
    )


    return (
        capture,
        errors100,
        errors_found,
    )


# ============================================================
# 22C.6 OBSERVED RESULTS
# ============================================================

observed_rows = []


for (
    corruption,
    level

), out in (
    ROW_OUTPUTS_22C.items()
):

    error = out[
        "error"
    ]


    risk_dis = out[
        "risk_dis"
    ]


    risk_lpns = out[
        "risk_lpns"
    ]


    baseline_pr = (
        average_precision_score(
            error,
            risk_dis
        )
    )


    lpns_pr = (
        average_precision_score(
            error,
            risk_lpns
        )
    )


    baseline_roc = (
        roc_auc_score(
            error,
            risk_dis
        )
    )


    lpns_roc = (
        roc_auc_score(
            error,
            risk_lpns
        )
    )


    baseline_aurc = (
        aurc20d(
            error,
            risk_dis
        )
    )


    lpns_aurc = (
        aurc20d(
            error,
            risk_lpns
        )
    )


    (
        cap_base,
        e100_base,
        errors_base

    ) = selective_metric22c(

        error,
        risk_dis,
        0.05
    )


    (
        cap_lpns,
        e100_lpns,
        errors_lpns

    ) = selective_metric22c(

        error,
        risk_lpns,
        0.05
    )


    observed_rows.append({

        "Corruption":
            corruption,

        "Level":
            level,

        "N":
            len(error),

        "Error_Prevalence":
            error.mean(),

        "Baseline_Error_PR":
            baseline_pr,

        "LPNS_Error_PR":
            lpns_pr,

        "Delta_Error_PR":
            lpns_pr
            -
            baseline_pr,

        "Baseline_Error_ROC":
            baseline_roc,

        "LPNS_Error_ROC":
            lpns_roc,

        "Delta_Error_ROC":
            lpns_roc
            -
            baseline_roc,

        "Baseline_AURC":
            baseline_aurc,

        "LPNS_AURC":
            lpns_aurc,

        "Delta_AURC":
            lpns_aurc
            -
            baseline_aurc,

        "Baseline_5pct_Capture":
            cap_base,

        "LPNS_5pct_Capture":
            cap_lpns,

        "Delta_5pct_Capture":
            cap_lpns
            -
            cap_base,

        "Baseline_5pct_Errors100":
            e100_base,

        "LPNS_5pct_Errors100":
            e100_lpns,

        "Delta_5pct_Errors100":
            e100_lpns
            -
            e100_base,

        "Additional_Errors_Found":
            errors_lpns
            -
            errors_base,

        "LPNS_Coverage":
            out[
                "coverage"
            ],
    })


OBSERVED_22C = pd.DataFrame(
    observed_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C — OBSERVED REPRESENTATIVE ROBUSTNESS"
)

print(
    "=" * 135
)


display(
    OBSERVED_22C
)


# ============================================================
# 22C.7 BOOTSTRAP
# ============================================================

B22C = 1000


rng_boot22c = np.random.default_rng(
    SEED + 222000
)


bootstrap_results22c = {}


for (
    corruption,
    level

), out in (
    ROW_OUTPUTS_22C.items()
):

    print(
        "\nBootstrapping:",
        corruption,
        level
    )


    error = out[
        "error"
    ]


    risk_dis = out[
        "risk_dis"
    ]


    risk_lpns = out[
        "risk_lpns"
    ]


    err_idx = np.where(
        error == 1
    )[0]


    correct_idx = np.where(
        error == 0
    )[0]


    boot_pr = []
    boot_roc = []
    boot_aurc = []

    boot_capture5 = []
    boot_errors100_5 = []


    for b in range(
        B22C
    ):

        idx_err = (
            rng_boot22c.choice(

                err_idx,

                size=len(
                    err_idx
                ),

                replace=True
            )
        )


        idx_correct = (
            rng_boot22c.choice(

                correct_idx,

                size=len(
                    correct_idx
                ),

                replace=True
            )
        )


        idx = np.concatenate([

            idx_err,
            idx_correct,
        ])


        yb = error[
            idx
        ]


        dis_b = risk_dis[
            idx
        ]


        lpns_b = risk_lpns[
            idx
        ]


        # ----------------------------------------------------
        # Error PR
        # ----------------------------------------------------

        boot_pr.append(

            average_precision_score(
                yb,
                lpns_b
            )

            -

            average_precision_score(
                yb,
                dis_b
            )
        )


        # ----------------------------------------------------
        # Error ROC
        # ----------------------------------------------------

        boot_roc.append(

            roc_auc_score(
                yb,
                lpns_b
            )

            -

            roc_auc_score(
                yb,
                dis_b
            )
        )


        # ----------------------------------------------------
        # AURC
        # ----------------------------------------------------

        boot_aurc.append(

            aurc20d(
                yb,
                lpns_b
            )

            -

            aurc20d(
                yb,
                dis_b
            )
        )


        # ----------------------------------------------------
        # 5% selective review
        # ----------------------------------------------------

        (
            cap_dis,
            e100_dis,
            _

        ) = selective_metric22c(

            yb,
            dis_b,
            0.05
        )


        (
            cap_lpns,
            e100_lpns,
            _

        ) = selective_metric22c(

            yb,
            lpns_b,
            0.05
        )


        boot_capture5.append(

            cap_lpns
            -
            cap_dis
        )


        boot_errors100_5.append(

            e100_lpns
            -
            e100_dis
        )


    bootstrap_results22c[
        (
            corruption,
            level
        )
    ] = {

        "Error_PR":
            np.asarray(
                boot_pr
            ),

        "Error_ROC":
            np.asarray(
                boot_roc
            ),

        "AURC":
            np.asarray(
                boot_aurc
            ),

        "Capture_5":
            np.asarray(
                boot_capture5
            ),

        "Errors100_5":
            np.asarray(
                boot_errors100_5
            ),
    }


# ============================================================
# 22C.8 CORRECT TWO-SIDED EMPIRICAL P-VALUE
# ============================================================

def summarize_bootstrap22c(
    observed,
    samples,
):

    samples = np.asarray(
        samples,
        dtype=float
    )


    low, high = np.percentile(

        samples,

        [
            2.5,
            97.5
        ]
    )


    n = len(
        samples
    )


    p_le_zero = (

        np.sum(
            samples <= 0
        )

        +
        1

    ) / (
        n + 1
    )


    p_ge_zero = (

        np.sum(
            samples >= 0
        )

        +
        1

    ) / (
        n + 1
    )


    p_two = min(

        1.0,

        2.0
        *
        min(
            p_le_zero,
            p_ge_zero
        )
    )


    return {

        "Observed_Delta":
            observed,

        "CI_Low":
            low,

        "CI_High":
            high,

        "P_two_sided":
            p_two,
    }


# ============================================================
# 22C.9 FINAL STATISTICAL TABLE
# ============================================================

stat_rows22c = []


for _, observed in (
    OBSERVED_22C
    .iterrows()
):

    key = (

        observed[
            "Corruption"
        ],

        float(
            observed[
                "Level"
            ]
        ),
    )


    samples = (
        bootstrap_results22c[
            key
        ]
    )


    metric_registry = [

        (
            "Error_PR_AUC",
            "Delta_Error_PR",
            "Error_PR"
        ),

        (
            "Error_ROC_AUC",
            "Delta_Error_ROC",
            "Error_ROC"
        ),

        (
            "AURC",
            "Delta_AURC",
            "AURC"
        ),

        (
            "5pct_Error_Capture",
            "Delta_5pct_Capture",
            "Capture_5"
        ),

        (
            "5pct_Errors_Per_100",
            "Delta_5pct_Errors100",
            "Errors100_5"
        ),
    ]


    for (
        metric_name,
        observed_column,
        sample_key

    ) in metric_registry:

        result = (
            summarize_bootstrap22c(

                observed[
                    observed_column
                ],

                samples[
                    sample_key
                ]
            )
        )


        stat_rows22c.append({

            "Corruption":
                observed[
                    "Corruption"
                ],

            "Level":
                observed[
                    "Level"
                ],

            "Metric":
                metric_name,

            **result
        })


STATS_22C = pd.DataFrame(
    stat_rows22c
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C — FINAL ROBUSTNESS BOOTSTRAP"
)

print(
    "=" * 135
)


display(
    STATS_22C
)


# ============================================================
# 22C.10 PRIMARY CLAIM TABLE
# ============================================================

PRIMARY_22C = (

    STATS_22C[
        STATS_22C[
            "Metric"
        ]
        .isin([
            "Error_PR_AUC",
            "5pct_Error_Capture",
            "5pct_Errors_Per_100",
        ])
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C — PRIMARY ROBUSTNESS CLAIMS"
)

print(
    "=" * 135
)


display(
    PRIMARY_22C
)


# ============================================================
# 22C.11 COMPACT MANUSCRIPT SUMMARY
# ============================================================

MANUSCRIPT_22C = []


for corruption, level in [

    (
        "Clean",
        0.0
    ),

    (
        "Feature_Masking",
        0.20
    ),

    (
        "Categorical_Corruption",
        0.20
    ),
]:

    pr_row = (

        STATS_22C[
            (
                STATS_22C[
                    "Corruption"
                ]
                ==
                corruption
            )
            &
            (
                np.isclose(
                    STATS_22C[
                        "Level"
                    ],
                    level
                )
            )
            &
            (
                STATS_22C[
                    "Metric"
                ]
                ==
                "Error_PR_AUC"
            )
        ]
        .iloc[0]
    )


    capture_row = (

        STATS_22C[
            (
                STATS_22C[
                    "Corruption"
                ]
                ==
                corruption
            )
            &
            (
                np.isclose(
                    STATS_22C[
                        "Level"
                    ],
                    level
                )
            )
            &
            (
                STATS_22C[
                    "Metric"
                ]
                ==
                "5pct_Error_Capture"
            )
        ]
        .iloc[0]
    )


    yield_row = (

        STATS_22C[
            (
                STATS_22C[
                    "Corruption"
                ]
                ==
                corruption
            )
            &
            (
                np.isclose(
                    STATS_22C[
                        "Level"
                    ],
                    level
                )
            )
            &
            (
                STATS_22C[
                    "Metric"
                ]
                ==
                "5pct_Errors_Per_100"
            )
        ]
        .iloc[0]
    )


    MANUSCRIPT_22C.append({

        "Condition":
            corruption,

        "Level":
            level,

        "Delta_Error_PR":
            pr_row[
                "Observed_Delta"
            ],

        "PR_CI_Low":
            pr_row[
                "CI_Low"
            ],

        "PR_CI_High":
            pr_row[
                "CI_High"
            ],

        "PR_p":
            pr_row[
                "P_two_sided"
            ],

        "Delta_5pct_Capture":
            capture_row[
                "Observed_Delta"
            ],

        "Capture_CI_Low":
            capture_row[
                "CI_Low"
            ],

        "Capture_CI_High":
            capture_row[
                "CI_High"
            ],

        "Delta_Errors_Per_100":
            yield_row[
                "Observed_Delta"
            ],

        "Yield_CI_Low":
            yield_row[
                "CI_Low"
            ],

        "Yield_CI_High":
            yield_row[
                "CI_High"
            ],
    })


MANUSCRIPT_22C = pd.DataFrame(
    MANUSCRIPT_22C
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C — MANUSCRIPT ROBUSTNESS SUMMARY"
)

print(
    "=" * 135
)


display(
    MANUSCRIPT_22C
)


# ============================================================
# 22C.12 SAVE
# ============================================================

save_csv(
    VERIFY_22C,
    TAB_DIR
    / "module22c_reproduction_check.csv"
)


save_csv(
    OBSERVED_22C,
    TAB_DIR
    / "module22c_observed_representative_robustness.csv"
)


save_csv(
    STATS_22C,
    TAB_DIR
    / "module22c_final_robustness_bootstrap.csv"
)


save_csv(
    PRIMARY_22C,
    TAB_DIR
    / "module22c_primary_robustness_claims.csv"
)


save_csv(
    MANUSCRIPT_22C,
    TAB_DIR
    / "module22c_manuscript_robustness_summary.csv"
)


save_json(

    {
        "module":
            "Final LPNS robustness statistical confirmation",

        "conditions": [
            "Clean",
            "20% feature masking",
            "20% categorical corruption",
        ],

        "comparison":
            "LPNS vs Confidence+Disagreement",

        "primary_reliability_metric":
            "Error PR-AUC",

        "operational_review_rate":
            0.05,

        "bootstrap_repetitions":
            B22C,

        "bootstrap_design":
            "paired stratified by classification-error status",

        "model_changes":
            False,

        "parameter_tuning":
            False,

        "recalibration":
            False,

        "threshold_adjustment":
            False,

        "experimental_status_after_module":
            "CLOSED",
    },

    LOG_DIR
    / "module22c_manifest.json"
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 22C COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nFINAL ROBUSTNESS STATISTICAL CONFIRMATION COMPLETE."
)

print(
    "No models or parameters were changed."
)

print(
    "After successful completion, experimental development is CLOSED."
)

Replaying: Clean | 0.0
Replaying: Continuous_Noise | 0.05
Replaying: Continuous_Noise | 0.1
Replaying: Continuous_Noise | 0.2
Replaying: Feature_Masking | 0.05
Replaying: Feature_Masking | 0.1
Replaying: Feature_Masking | 0.2
Replaying: Categorical_Corruption | 0.05
Replaying: Categorical_Corruption | 0.1
Replaying: Categorical_Corruption | 0.2

MODULE 22C — REPRODUCTION CHECK


,Corruption,Level,Module22B_Delta_PR,Replayed_Delta_PR,Absolute_Difference,Reproduced
0,Clean,0.0,0.012137,0.012137,0.0,True
1,Feature_Masking,0.2,0.003137,0.003137,0.0,True
2,Categorical_Corruption,0.2,0.016292,0.016292,0.0,True



MODULE 22C — OBSERVED REPRESENTATIVE ROBUSTNESS


,Corruption,Level,N,Error_Prevalence,Baseline_Error_PR,LPNS_Error_PR,Delta_Error_PR,Baseline_Error_ROC,LPNS_Error_ROC,Delta_Error_ROC,...,LPNS_AURC,Delta_AURC,Baseline_5pct_Capture,LPNS_5pct_Capture,Delta_5pct_Capture,Baseline_5pct_Errors100,LPNS_5pct_Errors100,Delta_5pct_Errors100,Additional_Errors_Found,LPNS_Coverage
0,Clean,0.0,319795,0.138076,0.575812,0.587949,0.012137,0.906016,0.905927,-0.000088,...,0.026337,0.000082,0.223458,0.236661,0.013203,61.707317,65.353346,3.646029,583,0.999991
1,Feature_Masking,0.2,319795,0.113751,0.487127,0.490264,0.003137,0.867253,0.866402,-0.000851,...,0.026996,0.000263,0.273937,0.279462,0.005525,62.320200,63.577236,1.257036,201,0.999991
2,Categorical_Corruption,0.2,319795,0.207586,0.708617,0.724909,0.016292,0.914756,0.915299,0.000543,...,0.045140,-0.000057,0.176802,0.191941,0.015139,73.402126,79.687305,6.285178,1005,0.999984



Bootstrapping: Clean 0.0

Bootstrapping: Feature_Masking 0.2

Bootstrapping: Categorical_Corruption 0.2

MODULE 22C — FINAL ROBUSTNESS BOOTSTRAP


...


MODULE 22C — PRIMARY ROBUSTNESS CLAIMS


,Corruption,Level,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided
0,Clean,0.0,Error_PR_AUC,0.012137,0.009678,0.014779,0.001998
1,Clean,0.0,5pct_Error_Capture,0.013203,0.011368,0.014992,0.001998
2,Clean,0.0,5pct_Errors_Per_100,3.646029,3.139306,4.140088,0.001998
3,Feature_Masking,0.2,Error_PR_AUC,0.003137,0.000699,0.005572,0.017982
4,Feature_Masking,0.2,5pct_Error_Capture,0.005525,0.003628,0.007258,0.001998
5,Feature_Masking,0.2,5pct_Errors_Per_100,1.257036,0.825360,1.651188,0.001998
6,Categorical_Corruption,0.2,Error_PR_AUC,0.016292,0.014042,0.018542,0.001998
7,Categorical_Corruption,0.2,5pct_Error_Capture,0.015139,0.013572,0.016660,0.001998
8,Categorical_Corruption,0.2,5pct_Errors_Per_100,6.285178,5.634615,6.916823,0.001998



MODULE 22C — MANUSCRIPT ROBUSTNESS SUMMARY


,Condition,Level,Delta_Error_PR,PR_CI_Low,PR_CI_High,PR_p,Delta_5pct_Capture,Capture_CI_Low,Capture_CI_High,Delta_Errors_Per_100,Yield_CI_Low,Yield_CI_High
0,Clean,0.0,0.012137,0.009678,0.014779,0.001998,0.013203,0.011368,0.014992,3.646029,3.139306,4.140088
1,Feature_Masking,0.2,0.003137,0.000699,0.005572,0.017982,0.005525,0.003628,0.007258,1.257036,0.825360,1.651188
2,Categorical_Corruption,0.2,0.016292,0.014042,0.018542,0.001998,0.015139,0.013572,0.016660,6.285178,5.634615,6.916823



MODULE 22C COMPLETE

FINAL ROBUSTNESS STATISTICAL CONFIRMATION COMPLETE.
No models or parameters were changed.
After successful completion, experimental development is CLOSED.


In [65]:
# ============================================================
# MODULE 23A — FINAL MASTER RESULTS REGISTRY
#
# PURPOSE
# ------------------------------------------------------------
# Freeze the manuscript-facing experimental record.
#
# NO model fitting.
# NO tuning.
# NO metric recomputation.
#
# Categories:
#   PRIMARY
#   SUPPORTING
#   ABLATION
#   LIMITATION
#   EXCLUDE_FROM_PRIMARY_CLAIMS
#
# After this module:
#   manuscript numbers should come from this registry.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 23A.1 PROJECT IDENTITY
# ============================================================

PROJECT_23A = {

    "Framework":
        "LSA-HE",

    "Expanded_Name":
        "Local-Stability-Aware Heterogeneous Ensemble",

    "Task":
        "Cardiovascular disease-status classification",

    "Internal_Dataset":
        "BRFSS 2015",

    "External_Dataset":
        "BRFSS 2020",

    "Primary_Prediction_Metric":
        "PR-AUC",

    "Primary_Reliability_Metric":
        "Error PR-AUC",

    "Main_Reliability_Method":
        "Local Prediction Neighborhood Stability (LPNS)",

    "Primary_Operational_Setting":
        "Low-budget selective review (5% primary; 10% secondary)",

    "Adaptive_Routing_Status":
        "Ablation only",

    "Experimental_Development_Status":
        "CLOSED",
}


# ============================================================
# 23A.2 FINAL ARCHITECTURE
# ============================================================

ARCHITECTURE_23A = pd.DataFrame([

    {
        "Component":
            "Full internal predictive backbone",

        "Definition":
            "Equal heterogeneous consensus",

        "Models":
            "HGB + CatBoost + AutoInt + DeepFM_v2 + XGB_F",

        "Feature_Set":
            "21 BRFSS-2015 predictors",

        "Role":
            "Primary internal disease-status predictor",
    },

    {
        "Component":
            "External matched predictor",

        "Definition":
            "Equal heterogeneous consensus",

        "Models":
            "HGB + CatBoost + XGBoost",

        "Feature_Set":
            "12 harmonized BRFSS predictors",

        "Role":
            "Temporal/external transportability evaluation",
    },

    {
        "Component":
            "Reliability baseline",

        "Definition":
            "Confidence + inter-expert disagreement",

        "Models":
            "-",

        "Feature_Set":
            "Prediction-space reliability features",

        "Role":
            "Primary trust baseline",
    },

    {
        "Component":
            "Proposed reliability layer",

        "Definition":
            "Confidence + disagreement + LPNS",

        "Models":
            "Logistic reliability head",

        "Feature_Set":
            "Prediction + neighborhood-stability features",

        "Role":
            "Primary methodological contribution",
    },

    {
        "Component":
            "Adaptive routing",

        "Definition":
            "Reliability-aware mixture routing",

        "Models":
            "RA-MoE",

        "Feature_Set":
            "Expert probabilities + trust features",

        "Role":
            "Negative/diagnostic ablation only",
    },
])


# ============================================================
# 23A.3 PRIMARY INTERNAL DISEASE RESULTS
# ============================================================

INTERNAL_DISEASE_23A = pd.DataFrame([

    {
        "Model":
            "HGB",

        "PR_AUC":
            0.377012,

        "ROC_AUC":
            0.849628,

        "Status":
            "Strong single-model baseline",
    },

    {
        "Model":
            "Repaired Core4",

        "PR_AUC":
            0.379284,

        "ROC_AUC":
            0.850402,

        "Status":
            "Supporting ensemble baseline",
    },

    {
        "Model":
            "LSA-HE Core5",

        "PR_AUC":
            0.379751,

        "ROC_AUC":
            0.850608,

        "Status":
            "FINAL internal predictive backbone",
    },
])


# ============================================================
# 23A.4 CORE5 CONSISTENCY
# ============================================================

CORE5_FOLD_23A = pd.DataFrame([

    {
        "Fold":
            1,

        "Core4_PR":
            0.378693,

        "Core5_PR":
            0.379071,

        "Delta":
            0.000379,
    },

    {
        "Fold":
            2,

        "Core4_PR":
            0.381922,

        "Core5_PR":
            0.382118,

        "Delta":
            0.000196,
    },

    {
        "Fold":
            3,

        "Core4_PR":
            0.378879,

        "Core5_PR":
            0.379130,

        "Delta":
            0.000251,
    },

    {
        "Fold":
            4,

        "Core4_PR":
            0.372029,

        "Core5_PR":
            0.372684,

        "Delta":
            0.000655,
    },

    {
        "Fold":
            5,

        "Core4_PR":
            0.388951,

        "Core5_PR":
            0.389362,

        "Delta":
            0.000412,
    },
])


# ============================================================
# 23A.5 FEATURE-HARMONIZATION CONTROL
# ============================================================

HARMONIZATION_23A = pd.DataFrame([

    {
        "Representation":
            "Full21 LSA-HE",

        "Feature_Count":
            21,

        "PR_AUC":
            0.379751,

        "ROC_AUC":
            0.850608,
    },

    {
        "Representation":
            "Common12 internal consensus",

        "Feature_Count":
            12,

        "PR_AUC":
            0.356374,

        "ROC_AUC":
            0.837299,
    },
])


HARMONIZATION_PENALTY_23A = {

    "Delta_PR":
        -0.023376,

    "Relative_PR_Change_pct":
        -6.155712,

    "Delta_ROC":
        -0.013309,
}


# ============================================================
# 23A.6 EXTERNAL PREDICTION RESULTS
# ============================================================

EXTERNAL_DISEASE_23A = pd.DataFrame([

    {
        "Dataset":
            "BRFSS-2015 Common12",

        "N":
            202944,

        "Prevalence":
            0.094184,

        "PR_AUC":
            0.356374,

        "ROC_AUC":
            0.837299,

        "Role":
            "Matched internal reference",
    },

    {
        "Dataset":
            "BRFSS-2020 Common12",

        "N":
            319795,

        "Prevalence":
            0.085595,

        "PR_AUC":
            0.339058,

        "ROC_AUC":
            0.840554,

        "Role":
            "Frozen external evaluation",
    },
])


EXTERNAL_CALIBRATION_23A = pd.DataFrame([

    {
        "Calibration":
            "Raw consensus",

        "Brier":
            0.066587,

        "NLL":
            0.228933,

        "ECE":
            0.017560,

        "Mean_Probability":
            0.103156,
    },

    {
        "Calibration":
            "2015-fitted Platt",

        "Brier":
            0.065841,

        "NLL":
            0.226781,

        "ECE":
            0.002159,

        "Mean_Probability":
            0.085322,
    },
])


EXTERNAL_THRESHOLD_23A = pd.DataFrame([{

    "Threshold":
        0.20,

    "Precision":
        0.314031,

    "Recall":
        0.517663,

    "Specificity":
        0.894150,

    "F1":
        0.390918,

    "MCC":
        0.330948,

    "Balanced_Accuracy":
        0.705906,

    "Accuracy":
        0.861924,
}])


# ============================================================
# 23A.7 STRICT INTERNAL LPNS EVIDENCE
#
# IMPORTANT:
# Use strict Module 16B/16C evidence for inferential internal
# reliability claims.
#
# Do NOT use Module 19A +0.032 as the main internal claim.
# ============================================================

INTERNAL_LPNS_23A = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            0.545862,

        "Error_ROC_AUC":
            0.900056,

        "AURC":
            0.026931,
    },

    {
        "Method":
            "Confidence + Disagreement",

        "Error_PR_AUC":
            0.547198,

        "Error_ROC_AUC":
            0.900208,

        "AURC":
            0.026892,
    },

    {
        "Method":
            "Strict LPNS",

        "Error_PR_AUC":
            0.559891,

        "Error_ROC_AUC":
            0.900525,

        "AURC":
            0.026829,
    },
])


INTERNAL_LPNS_EFFECT_23A = pd.DataFrame([{

    "Comparison":
        "LPNS vs Confidence+Disagreement",

    "Delta_Error_PR":
        0.012693,

    "CI_Low":
        0.009757,

    "CI_High":
        0.015606,

    "Delta_Error_ROC":
        0.000316,

    "ROC_CI_Low":
        0.000056,

    "ROC_CI_High":
        0.000586,

    "Delta_AURC":
        -0.000063,

    "AURC_CI_Low":
        -0.000179,

    "AURC_CI_High":
        0.000058,

    "Positive_PR_Folds":
        "5/5",
}])


# ============================================================
# 23A.8 EXTERNAL LPNS RESULTS
# ============================================================

EXTERNAL_LPNS_23A = pd.DataFrame([

    {
        "Method":
            "Confidence",

        "Error_PR_AUC":
            0.571734,

        "Error_ROC_AUC":
            0.905746,

        "AURC":
            0.026288,
    },

    {
        "Method":
            "Confidence + Disagreement",

        "Error_PR_AUC":
            0.575812,

        "Error_ROC_AUC":
            0.906016,

        "AURC":
            0.026255,
    },

    {
        "Method":
            "LPNS",

        "Error_PR_AUC":
            0.587949,

        "Error_ROC_AUC":
            0.905927,

        "AURC":
            0.026337,
    },
])


EXTERNAL_LPNS_EFFECT_23A = pd.DataFrame([

    {
        "Comparison":
            "LPNS vs Confidence",

        "Delta_Error_PR":
            0.016215,

        "CI_Low":
            0.013412,

        "CI_High":
            0.019050,

        "P_two_sided":
            0.001998,
    },

    {
        "Comparison":
            "LPNS vs Confidence+Disagreement",

        "Delta_Error_PR":
            0.012137,

        "CI_Low":
            0.009551,

        "CI_High":
            0.014591,

        "P_two_sided":
            0.001998,
    },
])


# ============================================================
# 23A.9 EXTERNAL SELECTIVE REVIEW
# ============================================================

SELECTIVE_23A = pd.DataFrame([

    {
        "Review_Rate":
            0.05,

        "Baseline_Error_Capture":
            0.223458,

        "LPNS_Error_Capture":
            0.236661,

        "Delta_Error_Capture":
            0.013203,

        "Capture_CI_Low":
            0.011346,

        "Capture_CI_High":
            0.015128,

        "Baseline_Errors_Per_100":
            61.707317,

        "LPNS_Errors_Per_100":
            65.353346,

        "Delta_Errors_Per_100":
            3.646029,

        "Yield_CI_Low":
            3.133208,

        "Yield_CI_High":
            4.177611,

        "Additional_Errors_Found":
            583,

        "Interpretation":
            "PRIMARY operational result",
    },

    {
        "Review_Rate":
            0.10,

        "Baseline_Error_Capture":
            0.479844,

        "LPNS_Error_Capture":
            0.484011,

        "Delta_Error_Capture":
            0.004167,

        "Capture_CI_Low":
            0.002060,

        "Capture_CI_High":
            0.006047,

        "Baseline_Errors_Per_100":
            66.253909,

        "LPNS_Errors_Per_100":
            66.829268,

        "Delta_Errors_Per_100":
            0.575360,

        "Yield_CI_Low":
            0.284475,

        "Yield_CI_High":
            0.834975,

        "Additional_Errors_Found":
            np.nan,

        "Interpretation":
            "SECONDARY operational result",
    },

    {
        "Review_Rate":
            0.15,

        "Baseline_Error_Capture":
            0.711545,

        "LPNS_Error_Capture":
            0.699995,

        "Delta_Error_Capture":
            -0.011550,

        "Capture_CI_Low":
            -0.013589,

        "Capture_CI_High":
            -0.009421,

        "Baseline_Errors_Per_100":
            65.498551,

        "LPNS_Errors_Per_100":
            64.435365,

        "Delta_Errors_Per_100":
            -1.063187,

        "Yield_CI_Low":
            -1.250912,

        "Yield_CI_High":
            -0.867227,

        "Additional_Errors_Found":
            np.nan,

        "Interpretation":
            "LIMITATION — LPNS worse",
    },

    {
        "Review_Rate":
            0.20,

        "Baseline_Error_Capture":
            0.775908,

        "LPNS_Error_Capture":
            0.775297,

        "Delta_Error_Capture":
            -0.000611,

        "Capture_CI_Low":
            -0.001608,

        "Capture_CI_High":
            0.000385,

        "Baseline_Errors_Per_100":
            53.567129,

        "LPNS_Errors_Per_100":
            53.524914,

        "Delta_Errors_Per_100":
            -0.042215,

        "Yield_CI_Low":
            -0.111009,

        "Yield_CI_High":
            0.026580,

        "Additional_Errors_Found":
            np.nan,

        "Interpretation":
            "No meaningful difference",
    },
])


# ============================================================
# 23A.10 FAIRNESS — KEY REPLICATED RESULTS
# ============================================================

FAIRNESS_23A = pd.DataFrame([

    {
        "Comparison":
            "Male - Female",

        "Metric":
            "Sensitivity",

        "Delta_2015":
            0.187969,

        "Delta_2020":
            0.213849,

        "Replicated_Direction":
            True,
    },

    {
        "Comparison":
            "Male - Female",

        "Metric":
            "FPR",

        "Delta_2015":
            0.085877,

        "Delta_2020":
            0.087919,

        "Replicated_Direction":
            True,
    },

    {
        "Comparison":
            "65+ - 18-44",

        "Metric":
            "Sensitivity",

        "Delta_2015":
            0.480861,

        "Delta_2020":
            0.526047,

        "Replicated_Direction":
            True,
    },

    {
        "Comparison":
            "65+ - 18-44",

        "Metric":
            "FPR",

        "Delta_2015":
            0.227688,

        "Delta_2020":
            0.265989,

        "Replicated_Direction":
            True,
    },
])


FAIRNESS_INTERPRETATION_23A = {

    "Claim":
        "Residual threshold-dependent demographic disparities exist.",

    "Do_Not_Claim":
        "The system is demographically fair.",

    "Important_Nuance":
        (
            "External subgroup calibration errors are generally small, "
            "but a common operating threshold produces unequal TPR/FPR."
        ),
}


# ============================================================
# 23A.11 CLASSIFIER ROBUSTNESS
# ============================================================

ROBUSTNESS_CLASSIFIER_23A = pd.DataFrame([

    {
        "Condition":
            "Clean",

        "Level":
            0.00,

        "PR_AUC":
            0.339058,

        "ROC_AUC":
            0.840554,

        "PR_Retention_pct":
            100.000,
    },

    {
        "Condition":
            "Continuous noise",

        "Level":
            0.20,

        "PR_AUC":
            0.338002,

        "ROC_AUC":
            0.840186,

        "PR_Retention_pct":
            99.689,
    },

    {
        "Condition":
            "Feature masking",

        "Level":
            0.20,

        "PR_AUC":
            0.299758,

        "ROC_AUC":
            0.811033,

        "PR_Retention_pct":
            88.409,
    },

    {
        "Condition":
            "Categorical corruption",

        "Level":
            0.20,

        "PR_AUC":
            0.252702,

        "ROC_AUC":
            0.779832,

        "PR_Retention_pct":
            74.531,
    },
])


# ============================================================
# 23A.12 LPNS ROBUSTNESS — FINAL BOOTSTRAPPED CLAIMS
# ============================================================

ROBUSTNESS_LPNS_23A = pd.DataFrame([

    {
        "Condition":
            "Clean",

        "Level":
            0.00,

        "Delta_Error_PR":
            0.012137,

        "PR_CI_Low":
            0.009678,

        "PR_CI_High":
            0.014779,

        "P":
            0.001998,

        "Delta_5pct_Capture":
            0.013203,

        "Delta_Errors_Per_100":
            3.646029,

        "Additional_Errors_Found":
            583,
    },

    {
        "Condition":
            "Feature masking",

        "Level":
            0.20,

        "Delta_Error_PR":
            0.003137,

        "PR_CI_Low":
            0.000699,

        "PR_CI_High":
            0.005572,

        "P":
            0.017982,

        "Delta_5pct_Capture":
            0.005525,

        "Delta_Errors_Per_100":
            1.257036,

        "Additional_Errors_Found":
            201,
    },

    {
        "Condition":
            "Categorical corruption",

        "Level":
            0.20,

        "Delta_Error_PR":
            0.016292,

        "PR_CI_Low":
            0.014042,

        "PR_CI_High":
            0.018542,

        "P":
            0.001998,

        "Delta_5pct_Capture":
            0.015139,

        "Delta_Errors_Per_100":
            6.285178,

        "Additional_Errors_Found":
            1005,
    },
])


# ============================================================
# 23A.13 MAJOR ABLATIONS / NEGATIVE FINDINGS
# ============================================================

ABLATIONS_23A = pd.DataFrame([

    {
        "Experiment":
            "Adaptive RA-MoE disease prediction",

        "Finding":
            (
                "Did not outperform stable equal heterogeneous "
                "consensus."
            ),

        "Evidence":
            "Routing weights approximately uniform",

        "Use":
            "Main ablation / negative finding",
    },

    {
        "Experiment":
            "Routing entropy",

        "Finding":
            "Effective routing remained close to all experts",

        "Evidence":
            "Entropy approximately 0.996–0.998",

        "Use":
            "Supports consensus interpretation",
    },

    {
        "Experiment":
            "Nonlinear stacking / meta-models",

        "Finding":
            "No stable material improvement over equal consensus",

        "Evidence":
            "Multiple meta-models underperformed",

        "Use":
            "Supplement / ablation",
    },

    {
        "Experiment":
            "Residual/hard-positive refinements",

        "Finding":
            "Only very small or unstable gains",

        "Evidence":
            "HPSC/HPRR gains not fold-stable",

        "Use":
            "Supplement / exploratory",
    },

    {
        "Experiment":
            "Module 19A LPNS",

        "Finding":
            "Large apparent LPNS gain but not fully nested",

        "Evidence":
            "Global OOF reuse in meta-stage",

        "Use":
            "DO NOT use as primary inferential evidence",
    },
])


# ============================================================
# 23A.14 EXCLUDED / HISTORICAL RESULTS
# ============================================================

EXCLUDED_23A = pd.DataFrame([

    {
        "Result":
            "Old 2015 holdout Core4/MoE",

        "Reason":
            (
                "Holdout was opened before later Core5/LSA-HE "
                "architecture changes."
            ),

        "Allowed_Use":
            "Historical/supporting only",
    },

    {
        "Result":
            "Old adaptive MoE superiority claims",

        "Reason":
            "Not supported by final experiments",

        "Allowed_Use":
            "Ablation discussion only",
    },

    {
        "Result":
            "Module 19A reliability +0.032 Error PR",

        "Reason":
            "Meta-level OOF provenance not fully nested",

        "Allowed_Use":
            "Exploratory/descriptive only",
    },

    {
        "Result":
            "Original neural OOF values",

        "Reason":
            "Outer validation used for early stopping",

        "Allowed_Use":
            "Do not use final; use repaired neural results",
    },

    {
        "Result":
            "Original custom DeepFM",

        "Reason":
            "Broken / poor implementation",

        "Allowed_Use":
            "Exclude",
    },
])


# ============================================================
# 23A.15 CLAIM REGISTRY
# ============================================================

CLAIMS_23A = pd.DataFrame([

    {
        "Claim_ID":
            "C1",

        "Claim":
            (
                "A heterogeneous consensus provides a stable "
                "predictive backbone for BRFSS cardiovascular "
                "disease-status classification."
            ),

        "Strength":
            "MODERATE",

        "Primary_Evidence":
            "Internal Core5 5/5 fold improvement",

        "Caveat":
            "Absolute gain over strong baselines is small.",
    },

    {
        "Claim_ID":
            "C2",

        "Claim":
            (
                "LPNS adds complementary error-risk information "
                "beyond confidence and expert disagreement."
            ),

        "Strength":
            "STRONG",

        "Primary_Evidence":
            (
                "Strict internal +0.012693 Error PR; "
                "external +0.012137 Error PR"
            ),

        "Caveat":
            "Not universally superior on ROC/AURC.",
    },

    {
        "Claim_ID":
            "C3",

        "Claim":
            (
                "LPNS improves selective review when review "
                "capacity is highly constrained."
            ),

        "Strength":
            "STRONG",

        "Primary_Evidence":
            (
                "5% external review: +583 errors; "
                "+3.646 errors/100 reviews"
            ),

        "Caveat":
            (
                "Benefit diminishes at 10%; LPNS is worse at "
                "15% and neutral near 20%."
            ),
    },

    {
        "Claim_ID":
            "C4",

        "Claim":
            (
                "The reliability benefit transfers temporally "
                "from BRFSS-2015 to BRFSS-2020."
            ),

        "Strength":
            "STRONG",

        "Primary_Evidence":
            "Frozen external reliability evaluation",

        "Caveat":
            "External validation uses 12 harmonized predictors.",
    },

    {
        "Claim_ID":
            "C5",

        "Claim":
            (
                "LPNS retains PR-based error-prioritization value "
                "under realistic feature corruption."
            ),

        "Strength":
            "STRONG",

        "Primary_Evidence":
            "Module 22C bootstrap",

        "Caveat":
            (
                "Under severe masking, ROC/AURC can worsen even "
                "while Error PR and 5% review improve."
            ),
    },

    {
        "Claim_ID":
            "C6",

        "Claim":
            (
                "Threshold-dependent demographic disparities "
                "remain and replicate across cohorts."
            ),

        "Strength":
            "STRONG",

        "Primary_Evidence":
            "Sex and age bootstrap comparisons",

        "Caveat":
            (
                "This is an audit finding, not a claim of "
                "fairness."
            ),
    },
])


# ============================================================
# 23A.16 SAVE ALL REGISTRIES
# ============================================================

tables23a = {

    "architecture":
        ARCHITECTURE_23A,

    "internal_disease":
        INTERNAL_DISEASE_23A,

    "core5_folds":
        CORE5_FOLD_23A,

    "harmonization":
        HARMONIZATION_23A,

    "external_disease":
        EXTERNAL_DISEASE_23A,

    "external_calibration":
        EXTERNAL_CALIBRATION_23A,

    "external_threshold":
        EXTERNAL_THRESHOLD_23A,

    "internal_lpns":
        INTERNAL_LPNS_23A,

    "internal_lpns_effect":
        INTERNAL_LPNS_EFFECT_23A,

    "external_lpns":
        EXTERNAL_LPNS_23A,

    "external_lpns_effect":
        EXTERNAL_LPNS_EFFECT_23A,

    "selective_review":
        SELECTIVE_23A,

    "fairness":
        FAIRNESS_23A,

    "classifier_robustness":
        ROBUSTNESS_CLASSIFIER_23A,

    "lpns_robustness":
        ROBUSTNESS_LPNS_23A,

    "ablations":
        ABLATIONS_23A,

    "excluded_results":
        EXCLUDED_23A,

    "claim_registry":
        CLAIMS_23A,
}


for name, table in (
    tables23a.items()
):

    save_csv(

        table,

        TAB_DIR
        /
        f"module23a_{name}.csv"
    )


save_json(

    {

        "project":
            PROJECT_23A,

        "harmonization_penalty":
            HARMONIZATION_PENALTY_23A,

        "fairness_interpretation":
            FAIRNESS_INTERPRETATION_23A,

        "experimental_development":
            "CLOSED",

        "future_model_tuning":
            "NOT PERMITTED for manuscript results",

        "primary_innovation":
            "LPNS",

        "adaptive_routing":
            "Ablation only",
    },

    LOG_DIR
    /
    "module23a_master_registry.json"
)


print(
    "=" * 135
)

print(
    "MODULE 23A — FINAL MASTER RESULTS REGISTRY"
)

print(
    "=" * 135
)


print(
    "\nFramework:",
    PROJECT_23A[
        "Framework"
    ]
)


print(
    "Primary innovation:",
    PROJECT_23A[
        "Main_Reliability_Method"
    ]
)


print(
    "Experimental status:",
    PROJECT_23A[
        "Experimental_Development_Status"
    ]
)


print(
    "\nFinal internal predictor:"
)

display(
    INTERNAL_DISEASE_23A
)


print(
    "\nPrimary claims:"
)

display(
    CLAIMS_23A
)


print(
    "\nResults explicitly excluded from primary claims:"
)

display(
    EXCLUDED_23A
)


print(
    "\n"
    + "=" * 135
)

print(
    "MASTER REGISTRY LOCKED"
)

print(
    "=" * 135
)


print(
    "\nFrom this point onward:"
)

print(
    "1. No further architecture search."
)

print(
    "2. No threshold optimization."
)

print(
    "3. No external recalibration."
)

print(
    "4. No result should enter the manuscript unless it is "
    "consistent with this registry."
)

MODULE 23A — FINAL MASTER RESULTS REGISTRY

Framework: LSA-HE
Primary innovation: Local Prediction Neighborhood Stability (LPNS)
Experimental status: CLOSED

Final internal predictor:


,Model,PR_AUC,ROC_AUC,Status
0,HGB,0.377012,0.849628,Strong single-model baseline
1,Repaired Core4,0.379284,0.850402,Supporting ensemble baseline
2,LSA-HE Core5,0.379751,0.850608,FINAL internal predictive backbone



Primary claims:


,Claim_ID,Claim,Strength,Primary_Evidence,Caveat
0,C1,A heterogeneous consensus provides a stable pr...,MODERATE,Internal Core5 5/5 fold improvement,Absolute gain over strong baselines is small.
1,C2,LPNS adds complementary error-risk information...,STRONG,Strict internal +0.012693 Error PR; external +...,Not universally superior on ROC/AURC.
2,C3,LPNS improves selective review when review cap...,STRONG,5% external review: +583 errors; +3.646 errors...,Benefit diminishes at 10%; LPNS is worse at 15...
3,C4,The reliability benefit transfers temporally f...,STRONG,Frozen external reliability evaluation,External validation uses 12 harmonized predict...
4,C5,LPNS retains PR-based error-prioritization val...,STRONG,Module 22C bootstrap,"Under severe masking, ROC/AURC can worsen even..."
5,C6,Threshold-dependent demographic disparities re...,STRONG,Sex and age bootstrap comparisons,"This is an audit finding, not a claim of fairn..."



Results explicitly excluded from primary claims:


,Result,Reason,Allowed_Use
0,Old 2015 holdout Core4/MoE,Holdout was opened before later Core5/LSA-HE a...,Historical/supporting only
1,Old adaptive MoE superiority claims,Not supported by final experiments,Ablation discussion only
2,Module 19A reliability +0.032 Error PR,Meta-level OOF provenance not fully nested,Exploratory/descriptive only
3,Original neural OOF values,Outer validation used for early stopping,Do not use final; use repaired neural results
4,Original custom DeepFM,Broken / poor implementation,Exclude



MASTER REGISTRY LOCKED

From this point onward:
1. No further architecture search.
2. No threshold optimization.
3. No external recalibration.
4. No result should enter the manuscript unless it is consistent with this registry.


In [68]:
# ============================================================
# MODULE 23C — FINAL MANUSCRIPT ARCHITECTURE
#
# PURPOSE
# ------------------------------------------------------------
# Freeze:
#   - paper structure
#   - main tables
#   - main figures
#   - supplementary material
#   - claim-to-evidence mapping
#
# NO modelling.
# NO statistical analysis.
# ============================================================

import pandas as pd


# ============================================================
# 23C.1 WORKING TITLE
# ============================================================

TITLE_23C = (
    "LSA-HE: Local-Stability-Aware Heterogeneous Ensemble "
    "for Reliable Cardiovascular Disease-Status Classification"
)


SHORT_TITLE_23C = (
    "Local-Stability-Aware Reliable Cardiovascular Classification"
)


# ============================================================
# 23C.2 ONE-SENTENCE PAPER MESSAGE
# ============================================================

CORE_MESSAGE_23C = (
    "LSA-HE combines a stable heterogeneous predictive consensus "
    "with Local Prediction Neighborhood Stability (LPNS), which "
    "quantifies how consistently multiple predictive experts behave "
    "among similar real-world profiles to identify predictions that "
    "are more likely to be unreliable."
)


# ============================================================
# 23C.3 PAPER CONTRIBUTIONS
# ============================================================

CONTRIBUTIONS_23C = pd.DataFrame([

    {
        "ID": "P1",

        "Contribution":
            (
                "A heterogeneous consensus framework for "
                "cardiovascular disease-status classification "
                "from population-health survey data."
            ),

        "Strength":
            "Supporting",

        "Evidence":
            (
                "Final Core5 PR-AUC 0.379751; "
                "5/5 fold improvement over repaired Core4."
            ),
    },

    {
        "ID": "P2",

        "Contribution":
            (
                "Local Prediction Neighborhood Stability (LPNS), "
                "a reliability representation based on variation "
                "in heterogeneous model predictions over nearby "
                "real-world profiles."
            ),

        "Strength":
            "Primary methodological contribution",

        "Evidence":
            (
                "Strict internal and frozen external "
                "error-risk evaluation."
            ),
    },

    {
        "ID": "P3",

        "Contribution":
            (
                "Integration of confidence, inter-expert disagreement, "
                "and LPNS for prediction-error risk estimation."
            ),

        "Strength":
            "Primary",

        "Evidence":
            (
                "Internal Error PR gain +0.012693; "
                "external gain +0.012137."
            ),
    },

    {
        "ID": "P4",

        "Contribution":
            (
                "Temporal external evaluation using a harmonized "
                "BRFSS-2015 to BRFSS-2020 transfer protocol."
            ),

        "Strength":
            "Primary validation contribution",

        "Evidence":
            (
                "319,795 external participants; "
                "PR-AUC 0.339058, ROC-AUC 0.840554."
            ),
    },

    {
        "ID": "P5",

        "Contribution":
            (
                "Operational evaluation showing that LPNS improves "
                "error discovery under constrained review budgets."
            ),

        "Strength":
            "Primary",

        "Evidence":
            (
                "At 5% review: +583 external errors found and "
                "+3.646 errors per 100 reviews."
            ),
    },

    {
        "ID": "P6",

        "Contribution":
            (
                "Reliability evaluation under demographic subgroup "
                "variation and realistic input corruption."
            ),

        "Strength":
            "Supporting validation",

        "Evidence":
            (
                "Fairness audit and robustness Modules 21–22."
            ),
    },
])


# ============================================================
# 23C.4 MANUSCRIPT SECTIONS
# ============================================================

SECTIONS_23C = pd.DataFrame([

    {
        "Section":
            "1. Introduction",

        "Purpose":
            (
                "Motivate reliable survey-based cardiovascular "
                "classification and the need to identify unreliable "
                "predictions."
            ),

        "Key_Content":
            (
                "Clinical motivation; limitations of accuracy-only "
                "systems; limitations of confidence/disagreement; "
                "motivation for local predictive stability; "
                "contributions."
            ),
    },

    {
        "Section":
            "2. Related Work",

        "Purpose":
            "Position LPNS relative to existing literature.",

        "Key_Content":
            (
                "CVD prediction; tabular ensembles; calibration; "
                "uncertainty; selective prediction; neighborhood-based "
                "trust; robustness/fairness."
            ),
    },

    {
        "Section":
            "3. Materials and Methods",

        "Purpose":
            "Fully define LSA-HE and the evaluation protocol.",

        "Key_Content":
            (
                "Datasets; preprocessing; experts; consensus; LPNS; "
                "reliability head; selective review; external "
                "harmonization; fairness; robustness; statistics."
            ),
    },

    {
        "Section":
            "4. Results",

        "Purpose":
            "Present results in a fixed evidence hierarchy.",

        "Key_Content":
            (
                "Prediction; external transfer; reliability; "
                "selective review; fairness; robustness; ablations."
            ),
    },

    {
        "Section":
            "5. Discussion",

        "Purpose":
            "Interpret what the results mean and do not mean.",

        "Key_Content":
            (
                "LPNS value; low-budget review; routing collapse; "
                "transportability; demographic disparities; "
                "robustness; deployment implications."
            ),
    },

    {
        "Section":
            "6. Limitations",

        "Purpose":
            "Explicitly delimit claims.",

        "Key_Content":
            (
                "Self-reported disease status; cross-sectional data; "
                "survey-only features; harmonized external subset; "
                "threshold disparities; no prospective deployment."
            ),
    },

    {
        "Section":
            "7. Conclusion",

        "Purpose":
            "Concise synthesis.",

        "Key_Content":
            (
                "Reliable prediction requires both disease estimates "
                "and prediction-specific reliability assessment."
            ),
    },
])


# ============================================================
# 23C.5 METHODS SUBSECTIONS
# ============================================================

METHODS_23C = pd.DataFrame([

    {
        "Subsection":
            "3.1 Data sources and cohorts",

        "Content":
            (
                "BRFSS-2015 development cohort and BRFSS-2020 "
                "external cohort; prevalence; target definition."
            ),
    },

    {
        "Subsection":
            "3.2 Outcome and predictor definitions",

        "Content":
            (
                "Existing self-reported cardiovascular disease/"
                "heart attack status; 21 internal predictors."
            ),
    },

    {
        "Subsection":
            "3.3 Development protocol",

        "Content":
            (
                "DEV split; fixed folds; OOF prediction; "
                "holdout chronology; strict leakage controls."
            ),
    },

    {
        "Subsection":
            "3.4 Heterogeneous predictive experts",

        "Content":
            (
                "HGB, CatBoost, AutoInt, DeepFM_v2, XGB_F."
            ),
    },

    {
        "Subsection":
            "3.5 Consensus disease predictor",

        "Content":
            (
                "Equal heterogeneous consensus; rationale from "
                "adaptive-routing collapse."
            ),
    },

    {
        "Subsection":
            "3.6 Local Prediction Neighborhood Stability",

        "Content":
            (
                "Neighborhood construction; distance representation; "
                "local prediction MAD/SD/decision-flip features; "
                "coverage."
            ),
    },

    {
        "Subsection":
            "3.7 Reliability-risk model",

        "Content":
            (
                "Confidence + disagreement baseline; "
                "confidence + disagreement + LPNS proposed model."
            ),
    },

    {
        "Subsection":
            "3.8 Selective review policy",

        "Content":
            (
                "Rank by error-risk score; review top 5/10/15/20%; "
                "no claim of automated correction."
            ),
    },

    {
        "Subsection":
            "3.9 External feature harmonization",

        "Content":
            (
                "12 aligned variables; internal Common12 control; "
                "2015-only calibration; frozen 2020 evaluation."
            ),
    },

    {
        "Subsection":
            "3.10 Fairness evaluation",

        "Content":
            (
                "Sex, age, Sex×Age; shared threshold; "
                "TPR/FPR/PPV/calibration."
            ),
    },

    {
        "Subsection":
            "3.11 Robustness evaluation",

        "Content":
            (
                "Continuous noise, feature masking, "
                "categorical corruption."
            ),
    },

    {
        "Subsection":
            "3.12 Statistical analysis",

        "Content":
            (
                "PR-AUC primary disease metric; Error PR-AUC primary "
                "reliability metric; paired stratified bootstrap; "
                "95% CIs."
            ),
    },
])


# ============================================================
# 23C.6 RESULTS ORDER
# ============================================================

RESULTS_SECTIONS_23C = pd.DataFrame([

    {
        "Subsection":
            "4.1 Predictive performance",

        "Primary_Result":
            (
                "LSA-HE Core5 PR-AUC 0.379751, "
                "ROC-AUC 0.850608."
            ),

        "Message":
            (
                "Heterogeneous consensus gives a small but consistent "
                "gain over strong individual and ensemble baselines."
            ),
    },

    {
        "Subsection":
            "4.2 Feature harmonization and temporal transfer",

        "Primary_Result":
            (
                "Common12 internal PR 0.356374; "
                "external PR 0.339058; external ROC 0.840554."
            ),

        "Message":
            (
                "Feature reduction explains a substantial portion "
                "of the apparent external PR difference."
            ),
    },

    {
        "Subsection":
            "4.3 Calibration transfer",

        "Primary_Result":
            (
                "External ECE 0.002159 after 2015-only Platt."
            ),

        "Message":
            (
                "Probability calibration transfers strongly across "
                "the temporal cohort."
            ),
    },

    {
        "Subsection":
            "4.4 Internal LPNS reliability",

        "Primary_Result":
            (
                "Error PR +0.012693, "
                "95% CI 0.009757–0.015606; 5/5 folds."
            ),

        "Message":
            (
                "LPNS adds information beyond confidence and "
                "expert disagreement."
            ),
    },

    {
        "Subsection":
            "4.5 External LPNS replication",

        "Primary_Result":
            (
                "0.575812 → 0.587949 Error PR; "
                "delta +0.012137."
            ),

        "Message":
            (
                "The reliability benefit persists under temporal "
                "external transfer."
            ),
    },

    {
        "Subsection":
            "4.6 Selective review",

        "Primary_Result":
            (
                "5% review: +583 errors and "
                "+3.646 errors/100 reviews."
            ),

        "Message":
            (
                "Operational advantage is concentrated at low "
                "review budgets."
            ),
    },

    {
        "Subsection":
            "4.7 Demographic subgroup audit",

        "Primary_Result":
            (
                "Sex and age TPR/FPR gaps replicate across cohorts."
            ),

        "Message":
            (
                "Calibration can remain good while a common threshold "
                "produces unequal group error rates."
            ),
    },

    {
        "Subsection":
            "4.8 Robustness",

        "Primary_Result":
            (
                "Classifier stable to continuous noise; categorical "
                "corruption most damaging; LPNS Error-PR benefit "
                "remains positive across all tested conditions."
            ),

        "Message":
            (
                "LPNS remains useful when predictive reliability "
                "degrades."
            ),
    },

    {
        "Subsection":
            "4.9 Ablation analysis",

        "Primary_Result":
            (
                "Adaptive routing converges toward nearly uniform "
                "weights and does not outperform equal consensus."
            ),

        "Message":
            (
                "Stable consensus is preferable to unnecessary "
                "routing complexity for this dataset."
            ),
    },
])


# ============================================================
# 23C.7 MAIN TABLE PLAN
# ============================================================

TABLE_PLAN_23C = pd.DataFrame([

    {
        "Table":
            "Table 1",

        "Title":
            (
                "Cohort characteristics and cross-cohort "
                "feature harmonization"
            ),

        "Main_Content":
            (
                "2015 N/prevalence/features; 2020 N/prevalence/"
                "features; Common12 mapping."
            ),

        "Location":
            "Main manuscript",
    },

    {
        "Table":
            "Table 2",

        "Title":
            (
                "Internal predictive performance of baseline "
                "and heterogeneous models"
            ),

        "Main_Content":
            (
                "HGB, CatBoost, neural experts, Core4, Core5; "
                "PR-AUC/ROC-AUC/calibration where appropriate."
            ),

        "Location":
            "Main manuscript",
    },

    {
        "Table":
            "Table 3",

        "Title":
            (
                "Internal and external reliability performance"
            ),

        "Main_Content":
            (
                "Confidence, confidence+disagreement, LPNS; "
                "Error PR/ROC/AURC; CIs."
            ),

        "Location":
            "Main manuscript",
    },

    {
        "Table":
            "Table 4",

        "Title":
            (
                "Selective review performance under constrained "
                "review budgets"
            ),

        "Main_Content":
            (
                "5% and 10% only in main paper; "
                "capture, errors/100, additional errors."
            ),

        "Location":
            "Main manuscript",
    },

    {
        "Table":
            "Table 5",

        "Title":
            (
                "Fairness and robustness summary"
            ),

        "Main_Content":
            (
                "Key sex/age gaps + representative clean/masking/"
                "categorical corruption results."
            ),

        "Location":
            "Main manuscript",
    },
])


# ============================================================
# 23C.8 MAIN FIGURE PLAN
# ============================================================

FIGURE_PLAN_23C = pd.DataFrame([

    {
        "Figure":
            "Figure 1",

        "Title":
            "Overview of the LSA-HE framework",

        "Panels":
            (
                "A: heterogeneous experts; "
                "B: consensus prediction; "
                "C: LPNS neighborhood; "
                "D: reliability-risk scoring; "
                "E: selective review."
            ),

        "Purpose":
            "Main methods figure",
    },

    {
        "Figure":
            "Figure 2",

        "Title":
            "Conceptual illustration of LPNS",

        "Panels":
            (
                "A: locally stable prediction; "
                "B: locally unstable prediction; "
                "C: expert prediction spread/decision flips."
            ),

        "Purpose":
            "Explain novelty visually",
    },

    {
        "Figure":
            "Figure 3",

        "Title":
            (
                "External error-risk ranking and selective review"
            ),

        "Panels":
            (
                "A: Error PR comparison; "
                "B: error capture vs review budget; "
                "C: errors per 100 reviews."
            ),

        "Purpose":
            "Primary empirical contribution",
    },

    {
        "Figure":
            "Figure 4",

        "Title":
            (
                "Robustness of prediction and LPNS reliability"
            ),

        "Panels":
            (
                "A: disease PR retention under corruption; "
                "B: LPNS Error-PR gain; "
                "C: 5% review yield gain."
            ),

        "Purpose":
            "Reliability under degraded inputs",
    },
])


# ============================================================
# 23C.9 SUPPLEMENT PLAN
# ============================================================

SUPPLEMENT_23C = pd.DataFrame([

    {
        "Item":
            "Supplementary Table S1",

        "Content":
            "Full predictor definitions and coding.",
    },

    {
        "Item":
            "Supplementary Table S2",

        "Content":
            "All base-model hyperparameters.",
    },

    {
        "Item":
            "Supplementary Table S3",

        "Content":
            "Fold-wise disease-model performance.",
    },

    {
        "Item":
            "Supplementary Table S4",

        "Content":
            "Full selective-review results at 5/10/15/20%.",
    },

    {
        "Item":
            "Supplementary Table S5",

        "Content":
            "Full sex, age, and Sex×Age subgroup results.",
    },

    {
        "Item":
            "Supplementary Table S6",

        "Content":
            "Fairness bootstrap confidence intervals.",
    },

    {
        "Item":
            "Supplementary Table S7",

        "Content":
            "All corruption levels and disease robustness metrics.",
    },

    {
        "Item":
            "Supplementary Table S8",

        "Content":
            "All LPNS reliability robustness conditions.",
    },

    {
        "Item":
            "Supplementary Table S9",

        "Content":
            "Adaptive routing and meta-model ablations.",
    },

    {
        "Item":
            "Supplementary Figure S1",

        "Content":
            "Routing-weight distributions / entropy.",
    },

    {
        "Item":
            "Supplementary Figure S2",

        "Content":
            "Subgroup calibration plots.",
    },

    {
        "Item":
            "Supplementary Figure S3",

        "Content":
            "Full risk-coverage curves.",
    },
])


# ============================================================
# 23C.10 CLAIM → EVIDENCE MAP
# ============================================================

EVIDENCE_MAP_23C = pd.DataFrame([

    {
        "Claim":
            "C1",

        "Evidence":
            "Table 2 + fold consistency",

        "Section":
            "4.1",
    },

    {
        "Claim":
            "C2",

        "Evidence":
            "Table 3",

        "Section":
            "4.4–4.5",
    },

    {
        "Claim":
            "C3",

        "Evidence":
            "Table 4 + Figure 3",

        "Section":
            "4.6",
    },

    {
        "Claim":
            "C4",

        "Evidence":
            "Table 1 + Table 3",

        "Section":
            "4.2–4.5",
    },

    {
        "Claim":
            "C5",

        "Evidence":
            "Figure 4 + Supplement S7/S8",

        "Section":
            "4.8",
    },

    {
        "Claim":
            "C6",

        "Evidence":
            "Table 5 + Supplement S5/S6",

        "Section":
            "4.7",
    },
])


# ============================================================
# 23C.11 WORD BUDGET
# ============================================================

WORD_BUDGET_23C = pd.DataFrame([

    {
        "Section":
            "Abstract",
        "Approx_Words":
            250,
    },

    {
        "Section":
            "Introduction",
        "Approx_Words":
            900,
    },

    {
        "Section":
            "Related Work",
        "Approx_Words":
            900,
    },

    {
        "Section":
            "Methods",
        "Approx_Words":
            2200,
    },

    {
        "Section":
            "Results",
        "Approx_Words":
            2200,
    },

    {
        "Section":
            "Discussion",
        "Approx_Words":
            1300,
    },

    {
        "Section":
            "Limitations",
        "Approx_Words":
            500,
    },

    {
        "Section":
            "Conclusion",
        "Approx_Words":
            250,
    },
])


# ============================================================
# 23C.12 WRITING ORDER
#
# Important:
# Do not write abstract/introduction first.
# ============================================================

WRITING_ORDER_23C = pd.DataFrame([

    {
        "Order":
            1,

        "Section":
            "Methods",

        "Reason":
            "Experimental protocol is now fully locked.",
    },

    {
        "Order":
            2,

        "Section":
            "Results",

        "Reason":
            "All claims must follow the master registry.",
    },

    {
        "Order":
            3,

        "Section":
            "Figures and tables",

        "Reason":
            "Align visual evidence with final Results.",
    },

    {
        "Order":
            4,

        "Section":
            "Discussion + Limitations",

        "Reason":
            "Interpret only established findings.",
    },

    {
        "Order":
            5,

        "Section":
            "Related Work",

        "Reason":
            "Final novelty positioning after method is fixed.",
    },

    {
        "Order":
            6,

        "Section":
            "Introduction",

        "Reason":
            "Frame the exact contribution actually demonstrated.",
    },

    {
        "Order":
            7,

        "Section":
            "Conclusion",

        "Reason":
            "Summarize established contributions.",
    },

    {
        "Order":
            8,

        "Section":
            "Abstract",

        "Reason":
            "Write last to avoid overclaiming.",
    },
])


# ============================================================
# 23C.13 SAVE
# ============================================================

tables23c = {

    "contributions":
        CONTRIBUTIONS_23C,

    "sections":
        SECTIONS_23C,

    "methods_structure":
        METHODS_23C,

    "results_structure":
        RESULTS_SECTIONS_23C,

    "table_plan":
        TABLE_PLAN_23C,

    "figure_plan":
        FIGURE_PLAN_23C,

    "supplement_plan":
        SUPPLEMENT_23C,

    "evidence_map":
        EVIDENCE_MAP_23C,

    "word_budget":
        WORD_BUDGET_23C,

    "writing_order":
        WRITING_ORDER_23C,
}


for name, table in tables23c.items():

    save_csv(
        table,
        TAB_DIR / f"module23c_{name}.csv"
    )


save_json(

    {
        "working_title":
            TITLE_23C,

        "short_title":
            SHORT_TITLE_23C,

        "core_message":
            CORE_MESSAGE_23C,

        "primary_innovation":
            "Local Prediction Neighborhood Stability (LPNS)",

        "main_figures":
            4,

        "main_tables":
            5,

        "experimental_status":
            "CLOSED",

        "writing_status":
            "READY",
    },

    LOG_DIR / "module23c_manuscript_architecture.json"
)


print(
    "=" * 135
)

print(
    "MODULE 23C — FINAL MANUSCRIPT ARCHITECTURE"
)

print(
    "=" * 135
)


print(
    "\nWorking title:"
)

print(
    TITLE_23C
)


print(
    "\nCore paper message:"
)

print(
    CORE_MESSAGE_23C
)


print(
    "\nContributions:"
)

display(
    CONTRIBUTIONS_23C
)


print(
    "\nResults structure:"
)

display(
    RESULTS_SECTIONS_23C
)


print(
    "\nMain tables:"
)

display(
    TABLE_PLAN_23C
)


print(
    "\nMain figures:"
)

display(
    FIGURE_PLAN_23C
)


print(
    "\nWriting order:"
)

display(
    WRITING_ORDER_23C
)


print(
    "\n"
    + "=" * 135
)

print(
    "MANUSCRIPT ARCHITECTURE LOCKED"
)

print(
    "=" * 135
)

MODULE 23C — FINAL MANUSCRIPT ARCHITECTURE

Working title:
LSA-HE: Local-Stability-Aware Heterogeneous Ensemble for Reliable Cardiovascular Disease-Status Classification

Core paper message:
LSA-HE combines a stable heterogeneous predictive consensus with Local Prediction Neighborhood Stability (LPNS), which quantifies how consistently multiple predictive experts behave among similar real-world profiles to identify predictions that are more likely to be unreliable.

Contributions:


,ID,Contribution,Strength,Evidence
0,P1,A heterogeneous consensus framework for cardio...,Supporting,Final Core5 PR-AUC 0.379751; 5/5 fold improvem...
1,P2,Local Prediction Neighborhood Stability (LPNS)...,Primary methodological contribution,Strict internal and frozen external error-risk...
2,P3,"Integration of confidence, inter-expert disagr...",Primary,Internal Error PR gain +0.012693; external gai...
3,P4,Temporal external evaluation using a harmonize...,Primary validation contribution,"319,795 external participants; PR-AUC 0.339058..."
4,P5,Operational evaluation showing that LPNS impro...,Primary,At 5% review: +583 external errors found and +...
5,P6,Reliability evaluation under demographic subgr...,Supporting validation,Fairness audit and robustness Modules 21–22.



Results structure:


,Subsection,Primary_Result,Message
0,4.1 Predictive performance,"LSA-HE Core5 PR-AUC 0.379751, ROC-AUC 0.850608.",Heterogeneous consensus gives a small but cons...
1,4.2 Feature harmonization and temporal transfer,Common12 internal PR 0.356374; external PR 0.3...,Feature reduction explains a substantial porti...
2,4.3 Calibration transfer,External ECE 0.002159 after 2015-only Platt.,Probability calibration transfers strongly acr...
3,4.4 Internal LPNS reliability,"Error PR +0.012693, 95% CI 0.009757–0.015606; ...",LPNS adds information beyond confidence and ex...
4,4.5 External LPNS replication,0.575812 → 0.587949 Error PR; delta +0.012137.,The reliability benefit persists under tempora...
5,4.6 Selective review,5% review: +583 errors and +3.646 errors/100 r...,Operational advantage is concentrated at low r...
6,4.7 Demographic subgroup audit,Sex and age TPR/FPR gaps replicate across coho...,Calibration can remain good while a common thr...
7,4.8 Robustness,Classifier stable to continuous noise; categor...,LPNS remains useful when predictive reliabilit...
8,4.9 Ablation analysis,Adaptive routing converges toward nearly unifo...,Stable consensus is preferable to unnecessary ...



Main tables:


...


Main figures:


,Figure,Title,Panels,Purpose
0,Figure 1,Overview of the LSA-HE framework,A: heterogeneous experts; B: consensus predict...,Main methods figure
1,Figure 2,Conceptual illustration of LPNS,A: locally stable prediction; B: locally unsta...,Explain novelty visually
2,Figure 3,External error-risk ranking and selective review,A: Error PR comparison; B: error capture vs re...,Primary empirical contribution
3,Figure 4,Robustness of prediction and LPNS reliability,A: disease PR retention under corruption; B: L...,Reliability under degraded inputs



Writing order:


,Order,Section,Reason
0,1,Methods,Experimental protocol is now fully locked.
1,2,Results,All claims must follow the master registry.
2,3,Figures and tables,Align visual evidence with final Results.
3,4,Discussion + Limitations,Interpret only established findings.
4,5,Related Work,Final novelty positioning after method is fixed.
5,6,Introduction,Frame the exact contribution actually demonstr...
6,7,Conclusion,Summarize established contributions.
7,8,Abstract,Write last to avoid overclaiming.



MANUSCRIPT ARCHITECTURE LOCKED


In [69]:
# ============================================================
# MODULE 24A0 — LPNS ABLATION OBJECT AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Identify the exact frozen objects currently available for:
#   - 2015 reliability-head development
#   - 2020 frozen external evaluation
#   - LPNS feature construction
#
# NO fitting.
# NO tuning.
# NO evaluation.
#
# We use this before the final ablation so we do not guess
# variable names or accidentally change the strict protocol.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 24A0.1 TARGET KEYWORDS
# ============================================================

keywords_24a0 = [

    "lpns",
    "rel_",
    "reliab",
    "error",
    "conf",
    "dis",
    "full",
    "scale",
    "reference",
    "neighbor",
    "nbr",
    "pred",
    "oof",
    "20d",
    "16b",
    "16c",
]


# ============================================================
# 24A0.2 SCAN GLOBALS
# ============================================================

rows_24a0 = []


for name, obj in globals().items():

    lname = name.lower()

    if not any(
        key in lname
        for key in keywords_24a0
    ):
        continue


    obj_type = type(obj).__name__


    shape = None

    try:
        shape = obj.shape
    except Exception:
        pass


    length = None

    try:
        length = len(obj)
    except Exception:
        pass


    columns = None

    if isinstance(
        obj,
        pd.DataFrame
    ):

        columns = list(
            obj.columns
        )


    rows_24a0.append({

        "Object":
            name,

        "Type":
            obj_type,

        "Shape":
            str(shape),

        "Length":
            length,

        "Columns":
            (
                ", ".join(
                    map(
                        str,
                        columns[:20]
                    )
                )
                if columns is not None
                else ""
            ),
    })


AUDIT_24A0 = (

    pd.DataFrame(
        rows_24a0
    )
    .sort_values(
        "Object"
    )
    .reset_index(
        drop=True
    )
)


print(
    "=" * 140
)

print(
    "MODULE 24A0 — AVAILABLE LPNS / RELIABILITY OBJECTS"
)

print(
    "=" * 140
)


display(
    AUDIT_24A0
)


# ============================================================
# 24A0.3 SPECIFIC IMPORTANT OBJECT CHECK
# ============================================================

important_24a0 = [

    # LPNS construction
    "LPNS12_ALL",
    "lpns_features20d",
    "impute_lpns20d",
    "nbr20d",
    "P_reference20d",
    "X_reference20d",
    "scaler20d",
    "scale_idx12",
    "discrete_idx12",
    "CONFIG20D",

    # external evaluation
    "X20",
    "y20",
    "predict_frozen22",

    # reliability heads
    "REL_CONF20D",
    "REL_DIS20D",
    "REL_LPNS20D",
    "scale_conf",
    "scale_dis",
    "scale_full",

    # strict internal results / objects
    "LSA_HE_OOF",
    "PRED_19",
]


CHECK_24A0 = pd.DataFrame([

    {
        "Object":
            name,

        "Available":
            name in globals(),

        "Type":
            (
                type(
                    globals()[name]
                ).__name__
                if name in globals()
                else ""
            ),

        "Shape":
            (
                str(
                    getattr(
                        globals()[name],
                        "shape",
                        ""
                    )
                )
                if name in globals()
                else ""
            ),
    }

    for name in important_24a0
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A0 — IMPORTANT OBJECT CHECK"
)

print(
    "=" * 140
)


display(
    CHECK_24A0
)


# ============================================================
# 24A0.4 INSPECT FROZEN RELIABILITY-HEAD DIMENSIONS
# ============================================================

head_rows = []


for head_name in [

    "REL_CONF20D",
    "REL_DIS20D",
    "REL_LPNS20D",

]:

    if head_name not in globals():
        continue


    head = globals()[
        head_name
    ]


    n_features = None

    coef_shape = None


    if hasattr(
        head,
        "n_features_in_"
    ):

        n_features = (
            head.n_features_in_
        )


    if hasattr(
        head,
        "coef_"
    ):

        coef_shape = str(
            head.coef_.shape
        )


    head_rows.append({

        "Head":
            head_name,

        "N_Input_Features":
            n_features,

        "Coefficient_Shape":
            coef_shape,
    })


HEAD_DIMENSIONS_24A0 = (
    pd.DataFrame(
        head_rows
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A0 — FROZEN RELIABILITY HEAD DIMENSIONS"
)

print(
    "=" * 140
)


display(
    HEAD_DIMENSIONS_24A0
)


# ============================================================
# 24A0.5 INSPECT SCALER DIMENSIONS
# ============================================================

scaler_rows = []


for scaler_name in [

    "scale_conf",
    "scale_dis",
    "scale_full",

]:

    if scaler_name not in globals():
        continue


    scaler = globals()[
        scaler_name
    ]


    scaler_rows.append({

        "Scaler":
            scaler_name,

        "N_Input_Features":
            getattr(
                scaler,
                "n_features_in_",
                None
            ),

        "Mean_Length":
            (
                len(
                    scaler.mean_
                )
                if hasattr(
                    scaler,
                    "mean_"
                )
                else None
            ),
    })


SCALER_DIMENSIONS_24A0 = (
    pd.DataFrame(
        scaler_rows
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A0 — SCALER DIMENSIONS"
)

print(
    "=" * 140
)


display(
    SCALER_DIMENSIONS_24A0
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A0 COMPLETE"
)

print(
    "=" * 140
)


print(
    "\nNo models were fitted and no results were changed."
)

MODULE 24A0 — AVAILABLE LPNS / RELIABILITY OBJECTS


,Object,Type,Shape,Length,Columns
0,AUTOINT_REPAIRED_CAL_OOF,ndarray,"(202944,)",202944.0,
1,AUTOINT_REPAIRED_OOF,ndarray,"(202944,)",202944.0,
2,AUTOINT_REPAIR_CONFIG,dict,None,12.0,
3,BEST_18C_OOF,ndarray,"(202944,)",202944.0,
4,BEST_RESCUE_OOF,ndarray,"(202944,)",202944.0,
...,...,...,...,...,...
568,y15_20d,ndarray,"(202944,)",202944.0,
569,y20_20d,ndarray,"(319795,)",319795.0,
570,y_error_train,ndarray,"(162356,)",162356.0,
571,y_error_valid,ndarray,"(40588,)",40588.0,



MODULE 24A0 — IMPORTANT OBJECT CHECK


,Object,Available,Type,Shape
0,LPNS12_ALL,True,list,
1,lpns_features20d,True,function,
2,impute_lpns20d,True,function,
3,nbr20d,True,NearestNeighbors,
4,P_reference20d,True,ndarray,"(80000, 3)"
5,X_reference20d,True,ndarray,"(80000, 12)"
6,scaler20d,True,StandardScaler,
7,scale_idx12,True,ndarray,"(5,)"
8,discrete_idx12,True,ndarray,"(7,)"
9,CONFIG20D,True,dict,



MODULE 24A0 — FROZEN RELIABILITY HEAD DIMENSIONS


,Head,N_Input_Features,Coefficient_Shape
0,REL_CONF20D,5,"(1, 5)"
1,REL_DIS20D,15,"(1, 15)"
2,REL_LPNS20D,34,"(1, 34)"



MODULE 24A0 — SCALER DIMENSIONS


,Scaler,N_Input_Features,Mean_Length
0,scale_conf,5,5
1,scale_dis,15,15
2,scale_full,34,34



MODULE 24A0 COMPLETE

No models were fitted and no results were changed.


In [73]:
# ============================================================
# MODULE 24A1 — TARGETED RELIABILITY TRAINING OBJECT AUDIT
# CORRECTED ROBUST VERSION
#
# PURPOSE
# ------------------------------------------------------------
# Find exact 2015 reliability-development feature matrices
# and labels used for the frozen reliability pipeline.
#
# NO training.
# NO tuning.
# NO evaluation.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 24A1.0 SAFE HELPERS
# ============================================================

def safe_shape_24a1(obj):
    """
    Return a normal tuple shape only when obj.shape is genuinely
    array/dataframe-like. Otherwise return None.
    """

    try:
        shape = getattr(obj, "shape", None)

        if shape is None:
            return None

        # Normal ndarray / dataframe shape
        if isinstance(shape, tuple):
            return shape

        # Occasionally shape may be list-like
        if isinstance(shape, list):
            return tuple(shape)

        # Do NOT try len() on functions, dispatchers, methods, etc.
        return None

    except Exception:
        return None


def safe_dtype_24a1(obj):

    try:
        return str(
            getattr(obj, "dtype", None)
        )

    except Exception:
        return ""


# ============================================================
# 24A1.1 SEARCH FOR RELEVANT OBJECTS
# ============================================================

target_lengths_24a1 = {
    202944,   # BRFSS-2015 DEV
    162356,   # typical outer-train
    40588,    # typical outer-valid
    48707,    # reliability subset observed earlier
    80000,    # LPNS reference bank
}


name_tokens_24a1 = [

    "conf",
    "dis",
    "lpns",
    "full",
    "rel",
    "error",
    "train",
    "valid",
    "20d",
    "base",
    "feature",
    "meta",
    "risk",
]


rows_24a1 = []


for name, obj in list(globals().items()):

    lname = name.lower()

    if not any(
        token in lname
        for token in name_tokens_24a1
    ):
        continue


    shape = safe_shape_24a1(
        obj
    )


    first_dim = None

    if (
        shape is not None
        and
        len(shape) >= 1
    ):
        first_dim = shape[0]


    strongly_relevant = any(

        token in lname

        for token in [

            "conf",
            "dis",
            "lpns",
            "reliab",
            "error",
            "20d",
        ]
    )


    if (
        first_dim not in target_lengths_24a1
        and
        not strongly_relevant
    ):
        continue


    columns = ""

    if isinstance(
        obj,
        pd.DataFrame
    ):

        try:

            columns = ", ".join(
                map(
                    str,
                    list(obj.columns)[:50]
                )
            )

        except Exception:

            columns = ""


    rows_24a1.append({

        "Object":
            name,

        "Type":
            type(obj).__name__,

        "Shape":
            str(shape),

        "DType":
            safe_dtype_24a1(
                obj
            ),

        "Columns":
            columns,
    })


AUDIT_24A1 = pd.DataFrame(
    rows_24a1
)


if len(
    AUDIT_24A1
) > 0:

    AUDIT_24A1 = (

        AUDIT_24A1
        .sort_values(
            [
                "Shape",
                "Object"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "=" * 150
)

print(
    "MODULE 24A1 — TARGETED RELIABILITY OBJECTS"
)

print(
    "=" * 150
)


display(
    AUDIT_24A1
)


# ============================================================
# 24A1.2 MATRICES MATCHING FROZEN RELIABILITY HEAD WIDTHS
#
# Confidence = 5
# Conf+Dis   = 15
# Full LPNS  = 34
# ============================================================

feature_widths_24a1 = {
    5,
    15,
    34,
}


matrix_rows_24a1 = []


for name, obj in list(globals().items()):

    shape = safe_shape_24a1(
        obj
    )


    if shape is None:
        continue


    if len(shape) != 2:
        continue


    if shape[1] not in feature_widths_24a1:
        continue


    matrix_rows_24a1.append({

        "Object":
            name,

        "Type":
            type(obj).__name__,

        "Shape":
            str(shape),

        "Rows":
            shape[0],

        "Features":
            shape[1],

        "DType":
            safe_dtype_24a1(
                obj
            ),
    })


MATRICES_24A1 = pd.DataFrame(
    matrix_rows_24a1
)


if len(
    MATRICES_24A1
) > 0:

    MATRICES_24A1 = (

        MATRICES_24A1
        .sort_values(
            [
                "Features",
                "Rows",
                "Object"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 24A1 — MATRICES MATCHING RELIABILITY HEAD WIDTHS"
)

print(
    "=" * 150
)


display(
    MATRICES_24A1
)


# ============================================================
# 24A1.3 CANDIDATE LABEL ARRAYS
# ============================================================

label_rows_24a1 = []


for name, obj in list(globals().items()):

    shape = safe_shape_24a1(
        obj
    )


    if shape is None:
        continue


    if len(shape) != 1:
        continue


    if shape[0] not in target_lengths_24a1:
        continue


    lname = name.lower()


    if not any(

        token in lname

        for token in [

            "y_",
            "error",
            "label",
            "target",
            "reliab",
        ]
    ):
        continue


    try:

        arr = np.asarray(
            obj
        )

    except Exception:

        continue


    # Avoid object/function-like arrays
    if arr.ndim != 1:
        continue


    unique_preview = ""

    try:

        u = np.unique(
            arr
        )

        unique_preview = str(
            u[:10]
        )

    except Exception:

        pass


    mean_value = np.nan

    try:

        if np.issubdtype(
            arr.dtype,
            np.number
        ):

            mean_value = float(
                np.mean(
                    arr.astype(float)
                )
            )

    except Exception:

        pass


    label_rows_24a1.append({

        "Object":
            name,

        "Shape":
            str(shape),

        "DType":
            str(
                arr.dtype
            ),

        "Mean":
            mean_value,

        "Unique_Preview":
            unique_preview,
    })


LABELS_24A1 = pd.DataFrame(
    label_rows_24a1
)


if len(
    LABELS_24A1
) > 0:

    LABELS_24A1 = (

        LABELS_24A1
        .sort_values(
            "Object"
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 24A1 — CANDIDATE RELIABILITY LABEL ARRAYS"
)

print(
    "=" * 150
)


display(
    LABELS_24A1
)


# ============================================================
# 24A1.4 MODEL / SCALER METADATA
# ============================================================

objects_to_check_24a1 = [

    "REL_CONF20D",
    "REL_DIS20D",
    "REL_LPNS20D",

    "scale_conf",
    "scale_dis",
    "scale_full",
]


provenance_rows_24a1 = []


for name in objects_to_check_24a1:

    if name not in globals():
        continue


    obj = globals()[
        name
    ]


    try:

        feature_names = getattr(
            obj,
            "feature_names_in_",
            None
        )

        if feature_names is not None:

            feature_names = list(
                feature_names
            )

    except Exception:

        feature_names = None


    provenance_rows_24a1.append({

        "Object":
            name,

        "Type":
            type(obj).__name__,

        "N_features_in":
            getattr(
                obj,
                "n_features_in_",
                None
            ),

        "Feature_names":
            str(
                feature_names
            ),
    })


PROVENANCE_24A1 = pd.DataFrame(
    provenance_rows_24a1
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 24A1 — MODEL/SCALER FEATURE METADATA"
)

print(
    "=" * 150
)


display(
    PROVENANCE_24A1
)


# ============================================================
# 24A1.5 HIGH-PRIORITY NAME SEARCH
#
# This makes it easier to locate likely feature matrices even
# if their dimensions do not exactly match the expected groups.
# ============================================================

priority_tokens_24a1 = [

    "feat",
    "matrix",
    "train",
    "valid",
    "reliability",
    "lpns",
    "dis",
    "conf",
]


priority_rows_24a1 = []


for name, obj in list(globals().items()):

    lname = name.lower()


    if not any(
        token in lname
        for token in priority_tokens_24a1
    ):
        continue


    shape = safe_shape_24a1(
        obj
    )


    if shape is None:
        continue


    if len(shape) not in [
        1,
        2
    ]:
        continue


    priority_rows_24a1.append({

        "Object":
            name,

        "Type":
            type(obj).__name__,

        "Shape":
            str(shape),

        "DType":
            safe_dtype_24a1(
                obj
            ),
    })


PRIORITY_24A1 = pd.DataFrame(
    priority_rows_24a1
)


if len(
    PRIORITY_24A1
) > 0:

    PRIORITY_24A1 = (

        PRIORITY_24A1
        .sort_values(
            [
                "Shape",
                "Object"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 24A1 — HIGH-PRIORITY ARRAY/MATRIX OBJECTS"
)

print(
    "=" * 150
)


display(
    PRIORITY_24A1
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 24A1 COMPLETE"
)

print(
    "=" * 150
)


print(
    "\nNo model was fitted, changed, or evaluated."
)

MODULE 24A1 — TARGETED RELIABILITY OBJECTS


,Object,Type,Shape,DType,Columns
0,aurc_conf,float64,(),float64,
1,base_errors,int64,(),int64,
2,baseline_error,float64,(),float64,
3,conf_pr,float64,(),float64,
4,conf_roc,float64,(),float64,
...,...,...,...,...,...
478,selective20d,function,None,None,
479,split_reliability,StratifiedShuffleSplit,None,None,
480,stability_vs_error_rows,list,None,None,
481,transform_distance,function,None,None,



MODULE 24A1 — MATRICES MATCHING RELIABILITY HEAD WIDTHS


,Object,Type,Shape,Rows,Features,DType
0,EXTERNAL_CALIBRATION_23A,DataFrame,"(2, 5)",2,5,None
1,EXTERNAL_LPNS_EFFECT_23A,DataFrame,"(2, 5)",2,5,None
2,FINAL_CLASSICAL_CHECK,DataFrame,"(2, 5)",2,5,None
3,FINAL_CLASSICAL_RECONSTRUCTION,DataFrame,"(2, 5)",2,5,None
4,neural_replacement_comparison,DataFrame,"(2, 5)",2,5,None
5,DISEASE_RESULTS_19,DataFrame,"(3, 5)",3,5,None
6,split_summary,DataFrame,"(3, 5)",3,5,None
7,FAIRNESS_23A,DataFrame,"(4, 5)",4,5,None
8,ROBUSTNESS_CLASSIFIER_23A,DataFrame,"(4, 5)",4,5,None
9,stable_expert_distribution,DataFrame,"(4, 5)",4,5,None



MODULE 24A1 — CANDIDATE RELIABILITY LABEL ARRAYS


...


MODULE 24A1 — MODEL/SCALER FEATURE METADATA


,Object,Type,N_features_in,Feature_names
0,REL_CONF20D,LogisticRegression,5,None
1,REL_DIS20D,LogisticRegression,15,None
2,REL_LPNS20D,LogisticRegression,34,None
3,scale_conf,StandardScaler,5,None
4,scale_dis,StandardScaler,15,None
5,scale_full,StandardScaler,34,None



MODULE 24A1 — HIGH-PRIORITY ARRAY/MATRIX OBJECTS


,Object,Type,Shape,DType
0,INTERNAL_LPNS_EFFECT_23A,DataFrame,"(1, 11)",None
1,FINAL_HOLDOUT_DISEASE,DataFrame,"(1, 20)",None
2,FINAL_HOLDOUT_LPNS_GAIN,DataFrame,"(1, 3)",None
3,LPNS_GAIN_19,DataFrame,"(1, 3)",None
4,strict_disease_results,DataFrame,"(1, 3)",None
...,...,...,...,...
279,discrete_idx12,ndarray,"(7,)",int64
280,lpns,Series,"(7,)",object
281,CONFIRMED_RESULTS,DataFrame,"(8, 7)",None
282,confirmed_sorted,DataFrame,"(8, 7)",None



MODULE 24A1 COMPLETE

No model was fitted, changed, or evaluated.


In [74]:
# ============================================================
# MODULE 24A — LPNS COMPONENT ABLATION
#
# PURPOSE
# ------------------------------------------------------------
# Quantify the independent and complementary value of:
#
#   C = Confidence
#   D = Inter-expert disagreement
#   L = LPNS
#
# External ablation variants:
#
#   1. Confidence only
#   2. Disagreement only
#   3. LPNS only
#   4. Confidence + Disagreement
#   5. Confidence + LPNS
#   6. Disagreement + LPNS
#   7. Confidence + Disagreement + LPNS
#
# TRAIN:
#   BRFSS-2015 only
#
# TEST:
#   Frozen BRFSS-2020
#
# Primary metric:
#   Error PR-AUC
#
# Secondary:
#   Error ROC-AUC
#   AURC
#   5% error capture
#   Errors / 100 reviews
#
# NO disease-model changes.
# NO threshold changes.
# NO use of 2020 labels for fitting.
# ============================================================

import numpy as np
import pandas as pd

from copy import deepcopy

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 24A.1 REQUIREMENTS
# ============================================================

required_24a = [

    "CONF15",
    "DIS15",
    "FULL15",

    "CONF20",
    "DIS20",
    "FULL20",

    "error15",
    "y20",

    "P20_CONSENSUS_CAL",
    "LOCKED_COMMON12_THRESHOLD",

    "REL_CONF20D",
    "REL_DIS20D",
    "REL_LPNS20D",

    "scale_conf",
    "scale_dis",
    "scale_full",

    "aurc20d",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",

    "SEED",
]


missing = [
    name
    for name in required_24a
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 24A objects: "
        + ", ".join(missing)
    )


# ============================================================
# 24A.2 BASIC CHECKS
# ============================================================

CONF15 = np.asarray(
    CONF15,
    dtype=float
)

DIS15 = np.asarray(
    DIS15,
    dtype=float
)

FULL15 = np.asarray(
    FULL15,
    dtype=float
)

CONF20 = np.asarray(
    CONF20,
    dtype=float
)

DIS20 = np.asarray(
    DIS20,
    dtype=float
)

FULL20 = np.asarray(
    FULL20,
    dtype=float
)

error15 = np.asarray(
    error15,
    dtype=int
)

y20_24a = np.asarray(
    y20,
    dtype=int
)


assert CONF15.shape == (
    len(error15),
    5
)

assert DIS15.shape == (
    len(error15),
    15
)

assert FULL15.shape == (
    len(error15),
    34
)


assert CONF20.shape[1] == 5
assert DIS20.shape[1] == 15
assert FULL20.shape[1] == 34


# ============================================================
# 24A.3 VERIFY NESTED FEATURE CONSTRUCTION
#
# We expect:
#   DIS[:, :5]  == CONF
#   FULL[:, :15] == DIS
#
# This is critical before splitting feature blocks.
# ============================================================

nested_checks_24a = pd.DataFrame([

    {
        "Check":
            "2015 DIS first 5 == CONF",

        "Max_Absolute_Difference":
            float(
                np.max(
                    np.abs(
                        DIS15[:, :5]
                        -
                        CONF15
                    )
                )
            ),

        "Match":
            bool(
                np.allclose(
                    DIS15[:, :5],
                    CONF15,
                    atol=1e-12,
                    equal_nan=True
                )
            ),
    },

    {
        "Check":
            "2015 FULL first 15 == DIS",

        "Max_Absolute_Difference":
            float(
                np.max(
                    np.abs(
                        FULL15[:, :15]
                        -
                        DIS15
                    )
                )
            ),

        "Match":
            bool(
                np.allclose(
                    FULL15[:, :15],
                    DIS15,
                    atol=1e-12,
                    equal_nan=True
                )
            ),
    },

    {
        "Check":
            "2020 DIS first 5 == CONF",

        "Max_Absolute_Difference":
            float(
                np.max(
                    np.abs(
                        DIS20[:, :5]
                        -
                        CONF20
                    )
                )
            ),

        "Match":
            bool(
                np.allclose(
                    DIS20[:, :5],
                    CONF20,
                    atol=1e-12,
                    equal_nan=True
                )
            ),
    },

    {
        "Check":
            "2020 FULL first 15 == DIS",

        "Max_Absolute_Difference":
            float(
                np.max(
                    np.abs(
                        FULL20[:, :15]
                        -
                        DIS20
                    )
                )
            ),

        "Match":
            bool(
                np.allclose(
                    FULL20[:, :15],
                    DIS20,
                    atol=1e-12,
                    equal_nan=True
                )
            ),
    },
])


print(
    "=" * 140
)

print(
    "MODULE 24A — FEATURE BLOCK VERIFICATION"
)

print(
    "=" * 140
)

display(
    nested_checks_24a
)


if not nested_checks_24a[
    "Match"
].all():

    raise RuntimeError(
        "Nested feature construction does not match the expected "
        "CONF -> DIS -> FULL structure. Stop before ablation."
    )


# ============================================================
# 24A.4 DEFINE COMPONENT BLOCKS
# ============================================================

# Confidence block
C15 = CONF15
C20 = CONF20


# Disagreement-only block
#
# DIS = [Confidence | Disagreement]
D15 = DIS15[
    :,
    5:15
]

D20 = DIS20[
    :,
    5:15
]


# LPNS-only block
#
# FULL = [Confidence | Disagreement | LPNS]
L15 = FULL15[
    :,
    15:34
]

L20 = FULL20[
    :,
    15:34
]


BLOCKS_24A = pd.DataFrame([

    {
        "Component":
            "Confidence",

        "N_Features":
            C15.shape[1],

        "Column_Range":
            "0:5",
    },

    {
        "Component":
            "Disagreement",

        "N_Features":
            D15.shape[1],

        "Column_Range":
            "5:15",
    },

    {
        "Component":
            "LPNS",

        "N_Features":
            L15.shape[1],

        "Column_Range":
            "15:34",
    },
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A — COMPONENT BLOCKS"
)

print(
    "=" * 140
)

display(
    BLOCKS_24A
)


# ============================================================
# 24A.5 EXTERNAL ERROR TARGET
# ============================================================

P20_24A = np.asarray(
    P20_CONSENSUS_CAL,
    dtype=float
)


pred20_24a = (

    P20_24A
    >=
    LOCKED_COMMON12_THRESHOLD

).astype(int)


error20_24a = (

    pred20_24a
    !=
    y20_24a

).astype(int)


print(
    "\n2015 reliability target prevalence:",
    error15.mean()
)

print(
    "2020 external error prevalence:",
    error20_24a.mean()
)


# ============================================================
# 24A.6 RELIABILITY HEAD TEMPLATE
#
# Use exactly the same logistic-regression hyperparameters as
# the frozen full LPNS reliability head.
# ============================================================

MODEL_TEMPLATE_24A = clone(
    REL_LPNS20D
)


print(
    "\nReliability-head configuration:"
)

print(
    MODEL_TEMPLATE_24A.get_params()
)


# ============================================================
# 24A.7 DEFINE ABLATION VARIANTS
# ============================================================

VARIANTS_24A = {

    "Confidence":
        (
            C15,
            C20
        ),

    "Disagreement":
        (
            D15,
            D20
        ),

    "LPNS":
        (
            L15,
            L20
        ),

    "Confidence+Disagreement":
        (
            np.column_stack([
                C15,
                D15,
            ]),

            np.column_stack([
                C20,
                D20,
            ])
        ),

    "Confidence+LPNS":
        (
            np.column_stack([
                C15,
                L15,
            ]),

            np.column_stack([
                C20,
                L20,
            ])
        ),

    "Disagreement+LPNS":
        (
            np.column_stack([
                D15,
                L15,
            ]),

            np.column_stack([
                D20,
                L20,
            ])
        ),

    "Confidence+Disagreement+LPNS":
        (
            np.column_stack([
                C15,
                D15,
                L15,
            ]),

            np.column_stack([
                C20,
                D20,
                L20,
            ])
        ),
}


# ============================================================
# 24A.8 SELECTIVE REVIEW HELPER
# ============================================================

def selective24a(
    error,
    risk,
    review_rate=0.05,
):

    n_review = max(
        1,
        int(
            round(
                review_rate
                *
                len(error)
            )
        )
    )


    order = np.argsort(
        -risk
    )


    idx = order[
        :n_review
    ]


    found = int(
        error[
            idx
        ].sum()
    )


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    capture = (
        found
        /
        total_errors
    )


    errors100 = (

        100.0
        *
        found
        /
        n_review
    )


    return {

        "Review_N":
            n_review,

        "Errors_Found":
            found,

        "Error_Capture":
            capture,

        "Errors_Per_100":
            errors100,
    }


# ============================================================
# 24A.9 TRAIN ON 2015, TEST ON FROZEN 2020
# ============================================================

rows_24a = []

risk_predictions_24a = {}

models_24a = {}

scalers_24a = {}


for name, (
    X15_variant,
    X20_variant

) in VARIANTS_24A.items():

    print(
        "Fitting:",
        name,
        "| features =",
        X15_variant.shape[1]
    )


    scaler = StandardScaler()


    X15_s = scaler.fit_transform(
        X15_variant
    )


    X20_s = scaler.transform(
        X20_variant
    )


    model = clone(
        MODEL_TEMPLATE_24A
    )


    model.fit(
        X15_s,
        error15
    )


    risk20 = (

        model
        .predict_proba(
            X20_s
        )[:, 1]
    )


    risk_predictions_24a[
        name
    ] = risk20


    models_24a[
        name
    ] = model


    scalers_24a[
        name
    ] = scaler


    sel5 = selective24a(
        error20_24a,
        risk20,
        0.05
    )


    sel10 = selective24a(
        error20_24a,
        risk20,
        0.10
    )


    rows_24a.append({

        "Method":
            name,

        "N_Features":
            X15_variant.shape[1],

        "Error_PR_AUC":
            average_precision_score(
                error20_24a,
                risk20
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error20_24a,
                risk20
            ),

        "AURC":
            aurc20d(
                error20_24a,
                risk20
            ),

        "Review5_Error_Capture":
            sel5[
                "Error_Capture"
            ],

        "Review5_Errors_Per_100":
            sel5[
                "Errors_Per_100"
            ],

        "Review5_Errors_Found":
            sel5[
                "Errors_Found"
            ],

        "Review10_Error_Capture":
            sel10[
                "Error_Capture"
            ],

        "Review10_Errors_Per_100":
            sel10[
                "Errors_Per_100"
            ],
    })


ABLATION_24A = (

    pd.DataFrame(
        rows_24a
    )
    .sort_values(
        "Error_PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A — EXTERNAL LPNS COMPONENT ABLATION"
)

print(
    "=" * 140
)

display(
    ABLATION_24A
)


# ============================================================
# 24A.10 RECONSTRUCTION CHECK AGAINST FROZEN MODULE 20D
#
# We do not require bit-for-bit equality because sklearn
# fitting details may differ slightly across re-instantiation.
#
# But Confidence, C+D and Full should reproduce very closely.
# ============================================================

known20d_24a = {

    "Confidence":
        {
            "Error_PR_AUC":
                0.571734,

            "Error_ROC_AUC":
                0.905746,

            "AURC":
                0.026288,
        },

    "Confidence+Disagreement":
        {
            "Error_PR_AUC":
                0.575812,

            "Error_ROC_AUC":
                0.906016,

            "AURC":
                0.026255,
        },

    "Confidence+Disagreement+LPNS":
        {
            "Error_PR_AUC":
                0.587949,

            "Error_ROC_AUC":
                0.905927,

            "AURC":
                0.026337,
        },
}


reconstruction_rows_24a = []


for method, reference in (
    known20d_24a.items()
):

    current = (

        ABLATION_24A[
            ABLATION_24A[
                "Method"
            ]
            ==
            method
        ]
        .iloc[0]
    )


    reconstruction_rows_24a.append({

        "Method":
            method,

        "Old_Error_PR":
            reference[
                "Error_PR_AUC"
            ],

        "Recreated_Error_PR":
            current[
                "Error_PR_AUC"
            ],

        "Delta_PR":
            current[
                "Error_PR_AUC"
            ]
            -
            reference[
                "Error_PR_AUC"
            ],

        "Old_Error_ROC":
            reference[
                "Error_ROC_AUC"
            ],

        "Recreated_Error_ROC":
            current[
                "Error_ROC_AUC"
            ],

        "Delta_ROC":
            current[
                "Error_ROC_AUC"
            ]
            -
            reference[
                "Error_ROC_AUC"
            ],

        "Old_AURC":
            reference[
                "AURC"
            ],

        "Recreated_AURC":
            current[
                "AURC"
            ],

        "Delta_AURC":
            current[
                "AURC"
            ]
            -
            reference[
                "AURC"
            ],
    })


RECONSTRUCTION_24A = pd.DataFrame(
    reconstruction_rows_24a
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A — RECONSTRUCTION CHECK"
)

print(
    "=" * 140
)

display(
    RECONSTRUCTION_24A
)


# ============================================================
# 24A.11 INCREMENTAL CONTRIBUTION TABLE
# ============================================================

def value24a(
    method,
    metric
):

    return float(

        ABLATION_24A.loc[
            ABLATION_24A[
                "Method"
            ]
            ==
            method,
            metric
        ]
        .iloc[0]
    )


increment_rows_24a = [

    {
        "Comparison":
            "C+D vs C",

        "Interpretation":
            "Incremental value of disagreement given confidence",

        "Delta_Error_PR":
            (
                value24a(
                    "Confidence+Disagreement",
                    "Error_PR_AUC"
                )
                -
                value24a(
                    "Confidence",
                    "Error_PR_AUC"
                )
            ),
    },

    {
        "Comparison":
            "C+L vs C",

        "Interpretation":
            "Incremental value of LPNS given confidence",

        "Delta_Error_PR":
            (
                value24a(
                    "Confidence+LPNS",
                    "Error_PR_AUC"
                )
                -
                value24a(
                    "Confidence",
                    "Error_PR_AUC"
                )
            ),
    },

    {
        "Comparison":
            "D+L vs D",

        "Interpretation":
            "Incremental value of LPNS given disagreement",

        "Delta_Error_PR":
            (
                value24a(
                    "Disagreement+LPNS",
                    "Error_PR_AUC"
                )
                -
                value24a(
                    "Disagreement",
                    "Error_PR_AUC"
                )
            ),
    },

    {
        "Comparison":
            "C+D+L vs C+D",

        "Interpretation":
            (
                "PRIMARY incremental value of LPNS beyond "
                "confidence and disagreement"
            ),

        "Delta_Error_PR":
            (
                value24a(
                    "Confidence+Disagreement+LPNS",
                    "Error_PR_AUC"
                )
                -
                value24a(
                    "Confidence+Disagreement",
                    "Error_PR_AUC"
                )
            ),
    },

    {
        "Comparison":
            "C+D+L vs C+L",

        "Interpretation":
            (
                "Incremental value of disagreement once LPNS "
                "is already available"
            ),

        "Delta_Error_PR":
            (
                value24a(
                    "Confidence+Disagreement+LPNS",
                    "Error_PR_AUC"
                )
                -
                value24a(
                    "Confidence+LPNS",
                    "Error_PR_AUC"
                )
            ),
    },
]


INCREMENTAL_24A = pd.DataFrame(
    increment_rows_24a
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A — INCREMENTAL COMPONENT VALUE"
)

print(
    "=" * 140
)

display(
    INCREMENTAL_24A
)


# ============================================================
# 24A.12 FIVE-PERCENT REVIEW COMPARISON
# ============================================================

REVIEW5_24A = (

    ABLATION_24A[
        [
            "Method",
            "Error_PR_AUC",
            "Review5_Error_Capture",
            "Review5_Errors_Per_100",
            "Review5_Errors_Found",
        ]
    ]
    .copy()
    .sort_values(
        "Review5_Errors_Per_100",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


baseline_method_24a = (
    "Confidence+Disagreement"
)


baseline_capture_24a = (
    value24a(
        baseline_method_24a,
        "Review5_Error_Capture"
    )
)


baseline_yield_24a = (
    value24a(
        baseline_method_24a,
        "Review5_Errors_Per_100"
    )
)


baseline_found_24a = (
    value24a(
        baseline_method_24a,
        "Review5_Errors_Found"
    )
)


REVIEW5_24A[
    "Delta_Capture_vs_C+D"
] = (

    REVIEW5_24A[
        "Review5_Error_Capture"
    ]
    -
    baseline_capture_24a
)


REVIEW5_24A[
    "Delta_Errors100_vs_C+D"
] = (

    REVIEW5_24A[
        "Review5_Errors_Per_100"
    ]
    -
    baseline_yield_24a
)


REVIEW5_24A[
    "Additional_Errors_vs_C+D"
] = (

    REVIEW5_24A[
        "Review5_Errors_Found"
    ]
    -
    baseline_found_24a
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A — 5% REVIEW COMPONENT ABLATION"
)

print(
    "=" * 140
)

display(
    REVIEW5_24A
)


# ============================================================
# 24A.13 STORE OBJECTS FOR LATER FIGURES
# ============================================================

ABLATION_RISK_24A = (
    risk_predictions_24a
)


ABLATION_MODELS_24A = (
    models_24a
)


ABLATION_SCALERS_24A = (
    scalers_24a
)


# ============================================================
# 24A.14 SAVE
# ============================================================

save_csv(
    nested_checks_24a,
    TAB_DIR
    /
    "module24a_feature_block_verification.csv"
)


save_csv(
    BLOCKS_24A,
    TAB_DIR
    /
    "module24a_component_blocks.csv"
)


save_csv(
    ABLATION_24A,
    TAB_DIR
    /
    "module24a_external_component_ablation.csv"
)


save_csv(
    RECONSTRUCTION_24A,
    TAB_DIR
    /
    "module24a_reconstruction_check.csv"
)


save_csv(
    INCREMENTAL_24A,
    TAB_DIR
    /
    "module24a_incremental_component_value.csv"
)


save_csv(
    REVIEW5_24A,
    TAB_DIR
    /
    "module24a_5pct_review_ablation.csv"
)


save_json(

    {
        "module":
            "LPNS component ablation",

        "training_dataset":
            "BRFSS-2015 Common12",

        "evaluation_dataset":
            "BRFSS-2020 Common12",

        "components": {

            "confidence_features":
                5,

            "disagreement_features":
                10,

            "lpns_features":
                19,
        },

        "variants": [
            "Confidence",
            "Disagreement",
            "LPNS",
            "Confidence+Disagreement",
            "Confidence+LPNS",
            "Disagreement+LPNS",
            "Confidence+Disagreement+LPNS",
        ],

        "primary_metric":
            "Error PR-AUC",

        "operational_endpoint":
            "5% selective review",

        "external_labels_used_for_training":
            False,

        "disease_model_changed":
            False,

        "threshold_changed":
            False,
    },

    LOG_DIR
    /
    "module24a_manifest.json"
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24A COMPLETE"
)

print(
    "=" * 140
)


print(
    "\nLPNS component ablation completed."
)

print(
    "No disease model, calibration model, or disease threshold was changed."
)

MODULE 24A — FEATURE BLOCK VERIFICATION


,Check,Max_Absolute_Difference,Match
0,2015 DIS first 5 == CONF,0.0,True
1,2015 FULL first 15 == DIS,0.0,True
2,2020 DIS first 5 == CONF,0.0,True
3,2020 FULL first 15 == DIS,0.0,True



MODULE 24A — COMPONENT BLOCKS


,Component,N_Features,Column_Range
0,Confidence,5,0:5
1,Disagreement,10,5:15
2,LPNS,19,15:34



2015 reliability target prevalence: 0.14935647272153896
2020 external error prevalence: 0.13807595490861332

Reliability-head configuration:
{'C': 0.1, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 3000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 208042, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
Fitting: Confidence | features = 5
Fitting: Disagreement | features = 10
Fitting: LPNS | features = 19
Fitting: Confidence+Disagreement | features = 15
Fitting: Confidence+LPNS | features = 24
Fitting: Disagreement+LPNS | features = 29
Fitting: Confidence+Disagreement+LPNS | features = 34

MODULE 24A — EXTERNAL LPNS COMPONENT ABLATION


...


MODULE 24A — INCREMENTAL COMPONENT VALUE


,Comparison,Interpretation,Delta_Error_PR
0,C+D vs C,Incremental value of disagreement given confid...,0.004078
1,C+L vs C,Incremental value of LPNS given confidence,0.015227
2,D+L vs D,Incremental value of LPNS given disagreement,0.014252
3,C+D+L vs C+D,PRIMARY incremental value of LPNS beyond confi...,0.012137
4,C+D+L vs C+L,Incremental value of disagreement once LPNS is...,0.000987



MODULE 24A — 5% REVIEW COMPONENT ABLATION


,Method,Error_PR_AUC,Review5_Error_Capture,Review5_Errors_Per_100,Review5_Errors_Found,Delta_Capture_vs_C+D,Delta_Errors100_vs_C+D,Additional_Errors_vs_C+D
0,Confidence+LPNS,0.586962,0.236774,65.384615,10455,0.013316,3.677298,588.0
1,Confidence+Disagreement+LPNS,0.587949,0.236661,65.353346,10450,0.013203,3.646029,583.0
2,Disagreement+LPNS,0.563057,0.225949,62.395247,9977,0.002491,0.687930,110.0
3,Confidence+Disagreement,0.575812,0.223458,61.707317,9867,0.000000,0.000000,0.0
4,Confidence,0.571734,0.220446,60.875547,9734,-0.003012,-0.831770,-133.0
5,Disagreement,0.548805,0.213787,59.036898,9440,-0.009670,-2.670419,-427.0
6,LPNS,0.491287,0.212995,58.818011,9405,-0.010463,-2.889306,-462.0



MODULE 24A COMPLETE

LPNS component ablation completed.
No disease model, calibration model, or disease threshold was changed.


In [78]:
# ============================================================
# MODULE 24B0 — LOCATE 2015 COMMON12 EXPERT PREDICTIONS
#
# Finds:
#   1. all (202944, 3) matrices
#   2. likely 202944-length HGB/CatBoost/XGB prediction arrays
#
# NO fitting.
# NO evaluation.
# ============================================================

import numpy as np
import pandas as pd


N15_24B0 = 202944


# ============================================================
# 24B0.1 SAFE SHAPE
# ============================================================

def safe_shape_24b0(obj):

    try:

        shape = getattr(
            obj,
            "shape",
            None
        )

        if isinstance(
            shape,
            tuple
        ):
            return shape

        if isinstance(
            shape,
            list
        ):
            return tuple(
                shape
            )

        return None

    except Exception:

        return None


# ============================================================
# 24B0.2 FIND ALL (202944, 3) MATRICES
# ============================================================

matrix_rows_24b0 = []


for name, obj in list(
    globals().items()
):

    shape = safe_shape_24b0(
        obj
    )

    if shape != (
        N15_24B0,
        3
    ):
        continue


    arr = None

    try:

        arr = np.asarray(
            obj
        )

    except Exception:

        pass


    matrix_rows_24b0.append({

        "Object":
            name,

        "Type":
            type(
                obj
            ).__name__,

        "Shape":
            str(
                shape
            ),

        "DType":
            (
                str(
                    arr.dtype
                )
                if arr is not None
                else ""
            ),

        "Min":
            (
                float(
                    np.nanmin(
                        arr
                    )
                )
                if arr is not None
                and np.issubdtype(
                    arr.dtype,
                    np.number
                )
                else np.nan
            ),

        "Max":
            (
                float(
                    np.nanmax(
                        arr
                    )
                )
                if arr is not None
                and np.issubdtype(
                    arr.dtype,
                    np.number
                )
                else np.nan
            ),

        "Mean":
            (
                float(
                    np.nanmean(
                        arr
                    )
                )
                if arr is not None
                and np.issubdtype(
                    arr.dtype,
                    np.number
                )
                else np.nan
            ),
    })


MATRICES3_24B0 = pd.DataFrame(
    matrix_rows_24b0
)


if len(
    MATRICES3_24B0
) > 0:

    MATRICES3_24B0 = (

        MATRICES3_24B0
        .sort_values(
            "Object"
        )
        .reset_index(
            drop=True
        )
    )


print(
    "=" * 140
)

print(
    "MODULE 24B0 — ALL (202944, 3) MATRICES"
)

print(
    "=" * 140
)

display(
    MATRICES3_24B0
)


# ============================================================
# 24B0.3 FIND LIKELY INDIVIDUAL COMMON12 EXPERT ARRAYS
#
# Look for one-dimensional 202944-length arrays associated with:
#   HGB
#   CatBoost
#   XGB
#   Common12 / 20D
# ============================================================

expert_tokens_24b0 = [

    "hgb",
    "hist",
    "cat",
    "xgb",
    "common12",
    "common_12",
    "20d",
]


prediction_tokens_24b0 = [

    "pred",
    "prob",
    "oof",
    "p15",
    "cal",
]


array_rows_24b0 = []


for name, obj in list(
    globals().items()
):

    shape = safe_shape_24b0(
        obj
    )

    if shape != (
        N15_24B0,
    ):
        continue


    lname = name.lower()


    if not any(
        token in lname
        for token in expert_tokens_24b0
    ):
        continue


    # Prefer prediction-like objects,
    # but include expert-named arrays even if no "pred" token.
    pred_like = any(
        token in lname
        for token in prediction_tokens_24b0
    )


    try:

        arr = np.asarray(
            obj
        )

    except Exception:

        continue


    if not np.issubdtype(
        arr.dtype,
        np.number
    ):
        continue


    array_rows_24b0.append({

        "Object":
            name,

        "Shape":
            str(
                shape
            ),

        "DType":
            str(
                arr.dtype
            ),

        "Prediction_Like_Name":
            pred_like,

        "Min":
            float(
                np.nanmin(
                    arr
                )
            ),

        "Max":
            float(
                np.nanmax(
                    arr
                )
            ),

        "Mean":
            float(
                np.nanmean(
                    arr
                )
            ),
    })


EXPERT_ARRAYS_24B0 = pd.DataFrame(
    array_rows_24b0
)


if len(
    EXPERT_ARRAYS_24B0
) > 0:

    EXPERT_ARRAYS_24B0 = (

        EXPERT_ARRAYS_24B0
        .sort_values(
            [
                "Prediction_Like_Name",
                "Object",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B0 — LIKELY 2015 COMMON12 EXPERT ARRAYS"
)

print(
    "=" * 140
)

display(
    EXPERT_ARRAYS_24B0
)


# ============================================================
# 24B0.4 SPECIFIC NAME SEARCH
# ============================================================

specific_rows_24b0 = []


for name, obj in list(
    globals().items()
):

    lname = name.lower()


    if not any(
        token in lname
        for token in [

            "hgb_common",
            "cat_common",
            "xgb_common",

            "common12_hgb",
            "common12_cat",
            "common12_xgb",

            "hgb15",
            "cat15",
            "xgb15",

            "p_hgb",
            "p_cat",
            "p_xgb",
        ]
    ):

        continue


    shape = safe_shape_24b0(
        obj
    )


    specific_rows_24b0.append({

        "Object":
            name,

        "Type":
            type(
                obj
            ).__name__,

        "Shape":
            str(
                shape
            ),
    })


SPECIFIC_24B0 = pd.DataFrame(
    specific_rows_24b0
)


if len(
    SPECIFIC_24B0
) > 0:

    SPECIFIC_24B0 = (

        SPECIFIC_24B0
        .sort_values(
            "Object"
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B0 — SPECIFIC COMMON12/HGB/CAT/XGB OBJECTS"
)

print(
    "=" * 140
)

display(
    SPECIFIC_24B0
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B0 COMPLETE"
)

print(
    "=" * 140
)

print(
    "\nNo models were fitted or modified."
)

MODULE 24B0 — ALL (202944, 3) MATRICES


,Object,Type,Shape,DType,Min,Max,Mean
0,P15_3,ndarray,"(202944, 3)",float64,0.000172,0.907156,0.113689
1,difficulty_df,DataFrame,"(202944, 3)",float64,0.000000,1.000000,0.344899



MODULE 24B0 — LIKELY 2015 COMMON12 EXPERT ARRAYS


,Object,Shape,DType,Prediction_Like_Name,Min,Max,Mean
0,CATBOOST_CAL_OOF,"(202944,)",float64,True,0.000103,0.836934,0.094167
1,COMMON12_INTERNAL_CAL,"(202944,)",float64,True,0.000678,0.822230,0.094182
2,FINAL_CAT_RAW_OOF,"(202944,)",float64,True,0.000805,0.980286,0.338129
3,FINAL_HGB_RAW_OOF,"(202944,)",float64,True,0.001151,0.796138,0.094016
4,HGB_CAL_OOF,"(202944,)",float64,True,0.001151,0.796138,0.094016
5,OOF_CAT_12,"(202944,)",float64,True,0.000569,0.884460,0.094042
6,OOF_HGB_12,"(202944,)",float64,True,0.000172,0.852445,0.094047
7,OOF_XGB_12,"(202944,)",float64,True,0.000894,0.907156,0.152978
8,CAT_ONLY,"(202944,)",float64,False,0.000103,0.836934,0.094167
9,COMMON12_CONSENSUS,"(202944,)",float64,False,0.000873,0.856659,0.113689



MODULE 24B0 — SPECIFIC COMMON12/HGB/CAT/XGB OBJECTS


,Object,Type,Shape
0,CAT_COMMON12_FINAL,CatBoostClassifier,None
1,COMMON12_CATEGORICAL,list,None
2,DEEP_CATEGORICAL_COLS,list,None
3,HGB_COMMON12_FINAL,HistGradientBoostingClassifier,None
4,P_CAT,ndarray,"(202944,)"
5,P_HGB,ndarray,"(202944,)"
6,P_XGB,ndarray,"(202944,)"
7,XGB_COMMON12_FINAL,XGBClassifier,None
8,p_cat,ndarray,"(40588,)"
9,p_hgb,ndarray,"(40588,)"



MODULE 24B0 COMPLETE

No models were fitted or modified.


In [79]:
# ============================================================
# MODULE 24B0.1 — VERIFY P15_3 COLUMN ORDER
# ============================================================

import numpy as np
import pandas as pd


P15_candidate = np.asarray(
    P15_3,
    dtype=float
)


known_predictions = {

    "HGB":
        np.asarray(
            OOF_HGB_12,
            dtype=float
        ),

    "CatBoost":
        np.asarray(
            OOF_CAT_12,
            dtype=float
        ),

    "XGBoost":
        np.asarray(
            OOF_XGB_12,
            dtype=float
        ),
}


rows = []


for col in range(3):

    for model_name, p in known_predictions.items():

        rows.append({

            "P15_3_Column":
                col,

            "Candidate_Model":
                model_name,

            "Max_Absolute_Difference":
                float(
                    np.max(
                        np.abs(
                            P15_candidate[:, col]
                            -
                            p
                        )
                    )
                ),

            "Mean_Absolute_Difference":
                float(
                    np.mean(
                        np.abs(
                            P15_candidate[:, col]
                            -
                            p
                        )
                    )
                ),

            "Correlation":
                float(
                    np.corrcoef(
                        P15_candidate[:, col],
                        p
                    )[0, 1]
                ),
        })


VERIFY_P15_24B0 = pd.DataFrame(
    rows
)


print(
    "=" * 120
)

print(
    "P15_3 COLUMN IDENTIFICATION"
)

print(
    "=" * 120
)


display(
    VERIFY_P15_24B0
    .sort_values(
        [
            "P15_3_Column",
            "Mean_Absolute_Difference"
        ]
    )
)


# Best match for each column
BEST_MATCH_P15_24B0 = (

    VERIFY_P15_24B0
    .sort_values(
        [
            "P15_3_Column",
            "Mean_Absolute_Difference"
        ]
    )
    .groupby(
        "P15_3_Column",
        as_index=False
    )
    .first()
)


print(
    "\nBest match per column:"
)

display(
    BEST_MATCH_P15_24B0
)

P15_3 COLUMN IDENTIFICATION


,P15_3_Column,Candidate_Model,Max_Absolute_Difference,Mean_Absolute_Difference,Correlation
0,0,HGB,0.000000,0.000000,1.000000
1,0,CatBoost,0.514727,0.008144,0.990869
2,0,XGBoost,0.469650,0.058986,0.984703
4,1,CatBoost,0.000000,0.000000,1.000000
3,1,HGB,0.514727,0.008144,0.990869
5,1,XGBoost,0.450543,0.059001,0.983497
8,2,XGBoost,0.000000,0.000000,1.000000
6,2,HGB,0.469650,0.058986,0.984703
7,2,CatBoost,0.450543,0.059001,0.983497



Best match per column:


,P15_3_Column,Candidate_Model,Max_Absolute_Difference,Mean_Absolute_Difference,Correlation
0,0,HGB,0.0,0.0,1.0
1,1,CatBoost,0.0,0.0,1.0
2,2,XGBoost,0.0,0.0,1.0


In [80]:
# ============================================================
# 24B.3 2015 COMMON12 EXPERT PREDICTIONS
# ============================================================

P15_EXPERT_24B = np.asarray(
    P15_3,
    dtype=float
)


assert P15_EXPERT_24B.shape == (
    202944,
    3
)


# Strict verification against known OOF Common12 experts
assert np.allclose(
    P15_EXPERT_24B[:, 0],
    OOF_HGB_12,
    atol=1e-12
)

assert np.allclose(
    P15_EXPERT_24B[:, 1],
    OOF_CAT_12,
    atol=1e-12
)

assert np.allclose(
    P15_EXPERT_24B[:, 2],
    OOF_XGB_12,
    atol=1e-12
)


P15_NAME_24B = "P15_3"


print(
    "Resolved 2015 expert matrix:",
    P15_NAME_24B,
    P15_EXPERT_24B.shape
)

print(
    "Column order verified: HGB, CatBoost, XGBoost"
)

Resolved 2015 expert matrix: P15_3 (202944, 3)
Column order verified: HGB, CatBoost, XGBoost


In [81]:
# ============================================================
# MODULE 24B — FINAL LPNS NEIGHBORHOOD / REFERENCE SENSITIVITY
#
# PURPOSE
# ------------------------------------------------------------
# Test whether the frozen LPNS result is sensitive to:
#
#   A. Neighborhood size:
#      k = 5, 10, 20, 40, 80
#
#   B. Reference-bank size:
#      N = 10k, 20k, 40k, 80k
#
# IMPORTANT
# ------------------------------------------------------------
# Final manuscript configuration REMAINS:
#      k = 20
#      reference bank = 80,000
#
# This is post-hoc sensitivity characterization only.
# No architecture selection or retuning is permitted.
#
# TRAIN RELIABILITY HEAD:
#      BRFSS-2015 Common12 only
#
# EVALUATE:
#      Frozen BRFSS-2020
#
# Primary:
#      Error PR-AUC
#
# Secondary:
#      Error ROC-AUC
#      AURC
#      5% Error Capture
#      Errors per 100 Reviews
#
# ============================================================

import numpy as np
import pandas as pd
import time

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 24B.1 REQUIRED OBJECTS
# ============================================================

required_24b = [

    "P15_3",
    "OOF_HGB_12",
    "OOF_CAT_12",
    "OOF_XGB_12",

    "X20",
    "y20",

    "CONF15",
    "DIS15",
    "CONF20",
    "DIS20",

    "error15",

    "X_reference20d",
    "P_reference20d",

    "scaler20d",
    "scale_idx12",
    "discrete_idx12",
    "CONFIG20D",
    "LPNS12_ALL",

    "lpns_features20d",
    "impute_lpns20d",

    "REL_LPNS20D",
    "aurc20d",

    "predict_frozen22",
    "P20_CONSENSUS_CAL",
    "LOCKED_COMMON12_THRESHOLD",

    "SEED",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_24b
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 24B objects: "
        + ", ".join(missing)
    )


# ============================================================
# 24B.2 RESOLVE BRFSS-2015 COMMON12 DATAFRAME
# ============================================================

candidate_names_24b = [

    "X15_20d",
    "X15_COMMON12",
    "X15_common12",
    "X15",
]


X15_RAW_24B = None
X15_NAME_24B = None


# First try likely names
for name in candidate_names_24b:

    if name not in globals():
        continue

    obj = globals()[name]

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    if obj.shape[0] != 202944:
        continue

    if not all(
        col in obj.columns
        for col in LPNS12_ALL
    ):
        continue

    X15_RAW_24B = (
        obj
        .copy()
        .reset_index(drop=True)
    )

    X15_NAME_24B = name

    break


# If not found, safely search all DataFrames
if X15_RAW_24B is None:

    candidates = []

    for name, obj in list(
        globals().items()
    ):

        if not isinstance(
            obj,
            pd.DataFrame
        ):
            continue

        if obj.shape[0] != 202944:
            continue

        if not all(
            col in obj.columns
            for col in LPNS12_ALL
        ):
            continue

        candidates.append(
            name
        )


    if len(candidates) == 1:

        X15_NAME_24B = (
            candidates[0]
        )

        X15_RAW_24B = (
            globals()[
                candidates[0]
            ]
            .copy()
            .reset_index(drop=True)
        )


    elif len(candidates) > 1:

        print(
            "Candidate 2015 Common12 DataFrames:",
            candidates
        )

        # Prefer exact 12-column representation
        exact12 = [

            name
            for name in candidates

            if globals()[
                name
            ].shape[1] == 12
        ]


        if len(exact12) == 1:

            X15_NAME_24B = (
                exact12[0]
            )

            X15_RAW_24B = (
                globals()[
                    exact12[0]
                ]
                .copy()
                .reset_index(drop=True)
            )


if X15_RAW_24B is None:

    raise RuntimeError(
        "Unable to uniquely resolve BRFSS-2015 "
        "Common12 DataFrame."
    )


print(
    "Resolved 2015 Common12 matrix:",
    X15_NAME_24B,
    X15_RAW_24B.shape
)


# ============================================================
# 24B.3 VERIFIED 2015 EXPERT MATRIX
#
# Exact column ordering established in Module 24B0.1:
#
#   0 = HGB
#   1 = CatBoost
#   2 = XGBoost
# ============================================================

P15_EXPERT_24B = np.asarray(
    P15_3,
    dtype=float
)


assert (
    P15_EXPERT_24B.shape
    ==
    (202944, 3)
)


assert np.allclose(
    P15_EXPERT_24B[:, 0],
    OOF_HGB_12,
    atol=1e-12
)


assert np.allclose(
    P15_EXPERT_24B[:, 1],
    OOF_CAT_12,
    atol=1e-12
)


assert np.allclose(
    P15_EXPERT_24B[:, 2],
    OOF_XGB_12,
    atol=1e-12
)


print(
    "2015 expert matrix verified:",
    P15_EXPERT_24B.shape
)

print(
    "Order = HGB | CatBoost | XGBoost"
)


# ============================================================
# 24B.4 FROZEN 2020 EXPERT MATRIX
# ============================================================

(
    p20_consensus_check,
    p20_hgb_24b,
    p20_cat_24b,
    p20_xgb_24b,

) = predict_frozen22(
    X20
)


P20_EXPERT_24B = np.column_stack([

    p20_hgb_24b,
    p20_cat_24b,
    p20_xgb_24b,
])


assert (
    P20_EXPERT_24B.shape
    ==
    (len(X20), 3)
)


print(
    "2020 expert matrix:",
    P20_EXPERT_24B.shape
)


# ============================================================
# 24B.5 EXTERNAL ERROR TARGET
# ============================================================

p20_final_24b = np.asarray(
    P20_CONSENSUS_CAL,
    dtype=float
)


pred20_24b = (

    p20_final_24b
    >=
    LOCKED_COMMON12_THRESHOLD

).astype(int)


error20_24b = (

    pred20_24b
    !=
    np.asarray(
        y20,
        dtype=int
    )

).astype(int)


error15_24b = np.asarray(
    error15,
    dtype=int
)


print(
    "\n2015 error prevalence:",
    error15_24b.mean()
)

print(
    "2020 error prevalence:",
    error20_24b.mean()
)


# ============================================================
# 24B.6 DISTANCE-SPACE TRANSFORMATION
# ============================================================

def transform_lpns_distance_24b(
    Xdf
):

    Xraw = (

        Xdf[
            LPNS12_ALL
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    Z = Xraw.copy()


    Z[
        :,
        scale_idx12
    ] = (

        scaler20d
        .transform(

            Z[
                :,
                scale_idx12
            ]
        )
    )


    Z[
        :,
        discrete_idx12
    ] *= (
        CONFIG20D[
            "discrete_weight"
        ]
    )


    return Z


X15_DISTANCE_24B = (
    transform_lpns_distance_24b(
        X15_RAW_24B
    )
)


X20_DISTANCE_24B = (
    transform_lpns_distance_24b(
        X20
    )
)


print(
    "\nDistance-space shapes:"
)

print(
    "2015:",
    X15_DISTANCE_24B.shape
)

print(
    "2020:",
    X20_DISTANCE_24B.shape
)

print(
    "Reference:",
    X_reference20d.shape
)


# ============================================================
# 24B.7 VERIFY BASE RELIABILITY MATRICES
# ============================================================

DIS15_24B = np.asarray(
    DIS15,
    dtype=float
)


DIS20_24B = np.asarray(
    DIS20,
    dtype=float
)


assert (
    DIS15_24B.shape
    ==
    (202944, 15)
)


assert (
    DIS20_24B.shape
    ==
    (319795, 15)
)


# ============================================================
# 24B.8 SELECTIVE REVIEW HELPER
# ============================================================

def selective24b(
    error,
    risk,
    rate=0.05,
):

    n = max(
        1,
        int(
            round(
                len(error)
                *
                rate
            )
        )
    )


    idx = np.argsort(
        -risk
    )[:n]


    found = int(
        error[
            idx
        ].sum()
    )


    total_errors = max(
        int(
            error.sum()
        ),
        1
    )


    return {

        "Review_N":
            n,

        "Errors_Found":
            found,

        "Error_Capture":
            found
            /
            total_errors,

        "Errors_Per_100":
            100.0
            *
            found
            /
            n,
    }


# ============================================================
# 24B.9 FIT FROZEN-STYLE RELIABILITY HEAD
#
# For each sensitivity configuration:
#
#   [Confidence + Disagreement + LPNS]
#
# Fit using 2015 only.
# Evaluate 2020.
# ============================================================

MODEL_TEMPLATE_24B = clone(
    REL_LPNS20D
)


def evaluate_lpns_configuration_24b(

    lpns15,
    lpns20,

    valid15,
    valid20,

    ncount15,
    ncount20,
):

    # --------------------------------------------
    # Same LPNS imputation logic as Module 20D
    # --------------------------------------------

    L15 = impute_lpns20d(
        lpns15
    )


    L20 = impute_lpns20d(
        lpns20
    )


    F15 = np.column_stack([

        DIS15_24B,
        L15,
    ])


    F20 = np.column_stack([

        DIS20_24B,
        L20,
    ])


    assert F15.shape[1] == 34
    assert F20.shape[1] == 34


    scaler = StandardScaler()


    F15_s = scaler.fit_transform(
        F15
    )


    F20_s = scaler.transform(
        F20
    )


    model = clone(
        MODEL_TEMPLATE_24B
    )


    model.fit(
        F15_s,
        error15_24b
    )


    risk20 = (

        model
        .predict_proba(
            F20_s
        )[:, 1]
    )


    sel5 = selective24b(

        error20_24b,
        risk20,
        0.05
    )


    return {

        "Error_PR_AUC":
            average_precision_score(
                error20_24b,
                risk20
            ),

        "Error_ROC_AUC":
            roc_auc_score(
                error20_24b,
                risk20
            ),

        "AURC":
            aurc20d(
                error20_24b,
                risk20
            ),

        "Review5_Error_Capture":
            sel5[
                "Error_Capture"
            ],

        "Review5_Errors_Per_100":
            sel5[
                "Errors_Per_100"
            ],

        "Review5_Errors_Found":
            sel5[
                "Errors_Found"
            ],

        "Coverage_2015":
            float(
                np.mean(
                    valid15
                )
            ),

        "Coverage_2020":
            float(
                np.mean(
                    valid20
                )
            ),

        "Mean_Neighbors_2015":
            float(
                np.mean(
                    ncount15
                )
            ),

        "Mean_Neighbors_2020":
            float(
                np.mean(
                    ncount20
                )
            ),
    }


# ============================================================
# 24B.10 K-SENSITIVITY
#
# Query maximum K ONCE.
# Then slice the identical ordered neighbor list.
# ============================================================

K_VALUES_24B = [

    5,
    10,
    20,
    40,
    80,
]


MAX_K_24B = max(
    K_VALUES_24B
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — COMPUTING MAX-K NEIGHBORS ONCE"
)

print(
    "=" * 140
)


nbr_maxk_24b = NearestNeighbors(

    n_neighbors=
        MAX_K_24B,

    metric="euclidean",

    algorithm="auto",

    n_jobs=-1,
)


nbr_maxk_24b.fit(
    X_reference20d
)


t_query_start = (
    time.perf_counter()
)


dist15_max, idx15_max = (
    nbr_maxk_24b
    .kneighbors(

        X15_DISTANCE_24B,

        return_distance=True,
    )
)


dist20_max, idx20_max = (
    nbr_maxk_24b
    .kneighbors(

        X20_DISTANCE_24B,

        return_distance=True,
    )
)


maxk_query_seconds = (

    time.perf_counter()
    -
    t_query_start
)


print(
    "Max-K neighbor search completed in",
    round(
        maxk_query_seconds,
        2
    ),
    "seconds."
)


# ============================================================
# 24B.11 EVALUATE EACH K
# ============================================================

k_rows_24b = []


for k in K_VALUES_24B:

    print(
        f"\nEvaluating k = {k}"
    )


    t0 = time.perf_counter()


    dist15 = (
        dist15_max[
            :,
            :k
        ]
    )


    idx15 = (
        idx15_max[
            :,
            :k
        ]
    )


    dist20 = (
        dist20_max[
            :,
            :k
        ]
    )


    idx20 = (
        idx20_max[
            :,
            :k
        ]
    )


    (
        lpns15,
        valid15,
        ncount15,

    ) = lpns_features20d(

        P15_EXPERT_24B,
        P_reference20d,

        idx15,
        dist15,

        LOCKED_COMMON12_THRESHOLD
    )


    (
        lpns20,
        valid20,
        ncount20,

    ) = lpns_features20d(

        P20_EXPERT_24B,
        P_reference20d,

        idx20,
        dist20,

        LOCKED_COMMON12_THRESHOLD
    )


    result = (
        evaluate_lpns_configuration_24b(

            lpns15,
            lpns20,

            valid15,
            valid20,

            ncount15,
            ncount20,
        )
    )


    elapsed = (

        time.perf_counter()
        -
        t0
    )


    k_rows_24b.append({

        "K":
            k,

        "Reference_N":
            80000,

        **result,

        "Evaluation_Seconds":
            elapsed,
    })


K_RESULTS_24B = pd.DataFrame(
    k_rows_24b
)


# ============================================================
# 24B.12 DELTAS RELATIVE TO LOCKED K=20
# ============================================================

ref_k20_24b = (

    K_RESULTS_24B[
        K_RESULTS_24B[
            "K"
        ]
        ==
        20
    ]
    .iloc[0]
)


K_RESULTS_24B[
    "Delta_Error_PR_vs_K20"
] = (

    K_RESULTS_24B[
        "Error_PR_AUC"
    ]

    -
    ref_k20_24b[
        "Error_PR_AUC"
    ]
)


K_RESULTS_24B[
    "Delta_Error_ROC_vs_K20"
] = (

    K_RESULTS_24B[
        "Error_ROC_AUC"
    ]

    -
    ref_k20_24b[
        "Error_ROC_AUC"
    ]
)


K_RESULTS_24B[
    "Delta_Yield_vs_K20"
] = (

    K_RESULTS_24B[
        "Review5_Errors_Per_100"
    ]

    -
    ref_k20_24b[
        "Review5_Errors_Per_100"
    ]
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — K-SENSITIVITY RESULTS"
)

print(
    "=" * 140
)


display(
    K_RESULTS_24B
)


# ============================================================
# 24B.13 REFERENCE-BANK-SIZE SENSITIVITY
#
# IMPORTANT:
# Deterministic nested subsets.
#
# Same single seeded permutation:
#
# 10k subset ⊂ 20k subset ⊂ 40k subset ⊂ 80k subset
#
# k fixed at 20.
# ============================================================

REFERENCE_SIZES_24B = [

    10000,
    20000,
    40000,
    80000,
]


rng_ref_24b = (
    np.random.default_rng(
        SEED + 240200
    )
)


reference_order_24b = (

    rng_ref_24b
    .permutation(
        len(
            X_reference20d
        )
    )
)


ref_rows_24b = []


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — REFERENCE-BANK-SIZE SENSITIVITY"
)

print(
    "=" * 140
)


for ref_n in REFERENCE_SIZES_24B:

    print(
        f"\nReference N = {ref_n}"
    )


    selected = (
        reference_order_24b[
            :ref_n
        ]
    )


    X_ref = (
        X_reference20d[
            selected
        ]
    )


    P_ref = (
        P_reference20d[
            selected
        ]
    )


    nbr_ref = NearestNeighbors(

        n_neighbors=20,

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    t0 = time.perf_counter()


    nbr_ref.fit(
        X_ref
    )


    dist15_ref, idx15_ref = (
        nbr_ref
        .kneighbors(

            X15_DISTANCE_24B,

            return_distance=True,
        )
    )


    dist20_ref, idx20_ref = (
        nbr_ref
        .kneighbors(

            X20_DISTANCE_24B,

            return_distance=True,
        )
    )


    (
        lpns15_ref,
        valid15_ref,
        ncount15_ref,

    ) = lpns_features20d(

        P15_EXPERT_24B,
        P_ref,

        idx15_ref,
        dist15_ref,

        LOCKED_COMMON12_THRESHOLD
    )


    (
        lpns20_ref,
        valid20_ref,
        ncount20_ref,

    ) = lpns_features20d(

        P20_EXPERT_24B,
        P_ref,

        idx20_ref,
        dist20_ref,

        LOCKED_COMMON12_THRESHOLD
    )


    result = (
        evaluate_lpns_configuration_24b(

            lpns15_ref,
            lpns20_ref,

            valid15_ref,
            valid20_ref,

            ncount15_ref,
            ncount20_ref,
        )
    )


    elapsed = (

        time.perf_counter()
        -
        t0
    )


    ref_rows_24b.append({

        "Reference_N":
            ref_n,

        "K":
            20,

        **result,

        "Elapsed_Seconds":
            elapsed,
    })


REF_RESULTS_24B = pd.DataFrame(
    ref_rows_24b
)


# ============================================================
# 24B.14 DELTAS RELATIVE TO 80K
# ============================================================

ref80_24b = (

    REF_RESULTS_24B[
        REF_RESULTS_24B[
            "Reference_N"
        ]
        ==
        80000
    ]
    .iloc[0]
)


REF_RESULTS_24B[
    "Delta_Error_PR_vs_80k"
] = (

    REF_RESULTS_24B[
        "Error_PR_AUC"
    ]

    -
    ref80_24b[
        "Error_PR_AUC"
    ]
)


REF_RESULTS_24B[
    "Delta_Error_ROC_vs_80k"
] = (

    REF_RESULTS_24B[
        "Error_ROC_AUC"
    ]

    -
    ref80_24b[
        "Error_ROC_AUC"
    ]
)


REF_RESULTS_24B[
    "Delta_Yield_vs_80k"
] = (

    REF_RESULTS_24B[
        "Review5_Errors_Per_100"
    ]

    -
    ref80_24b[
        "Review5_Errors_Per_100"
    ]
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — REFERENCE-SIZE RESULTS"
)

print(
    "=" * 140
)


display(
    REF_RESULTS_24B
)


# ============================================================
# 24B.15 REPRODUCTION CHECK
#
# k=20 / 80k should be close to the locked Module 20D value.
# ============================================================

LOCKED_ERROR_PR_24B = (
    0.587949
)


LOCKED_REVIEW_YIELD_24B = (
    65.353346
)


observed_k20_pr = float(

    ref_k20_24b[
        "Error_PR_AUC"
    ]
)


observed_k20_yield = float(

    ref_k20_24b[
        "Review5_Errors_Per_100"
    ]
)


REPRODUCTION_24B = pd.DataFrame([

    {
        "Metric":
            "Error PR-AUC",

        "Locked_Module20D":
            LOCKED_ERROR_PR_24B,

        "Sensitivity_K20":
            observed_k20_pr,

        "Absolute_Difference":
            abs(
                observed_k20_pr
                -
                LOCKED_ERROR_PR_24B
            ),
    },

    {
        "Metric":
            "5% errors / 100",

        "Locked_Module20D":
            LOCKED_REVIEW_YIELD_24B,

        "Sensitivity_K20":
            observed_k20_yield,

        "Absolute_Difference":
            abs(
                observed_k20_yield
                -
                LOCKED_REVIEW_YIELD_24B
            ),
    },
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — LOCKED CONFIGURATION CHECK"
)

print(
    "=" * 140
)


display(
    REPRODUCTION_24B
)


# ============================================================
# 24B.16 COMPACT SENSITIVITY SUMMARY
# ============================================================

SUMMARY_24B = pd.DataFrame([

    {
        "Sensitivity":
            "Neighborhood size K",

        "N_Configurations":
            len(
                K_RESULTS_24B
            ),

        "Min_Error_PR":
            K_RESULTS_24B[
                "Error_PR_AUC"
            ].min(),

        "Max_Error_PR":
            K_RESULTS_24B[
                "Error_PR_AUC"
            ].max(),

        "Error_PR_Range":
            (
                K_RESULTS_24B[
                    "Error_PR_AUC"
                ].max()

                -

                K_RESULTS_24B[
                    "Error_PR_AUC"
                ].min()
            ),

        "Min_5pct_Yield":
            K_RESULTS_24B[
                "Review5_Errors_Per_100"
            ].min(),

        "Max_5pct_Yield":
            K_RESULTS_24B[
                "Review5_Errors_Per_100"
            ].max(),

        "Yield_Range":
            (
                K_RESULTS_24B[
                    "Review5_Errors_Per_100"
                ].max()

                -

                K_RESULTS_24B[
                    "Review5_Errors_Per_100"
                ].min()
            ),
    },

    {
        "Sensitivity":
            "Reference-bank size",

        "N_Configurations":
            len(
                REF_RESULTS_24B
            ),

        "Min_Error_PR":
            REF_RESULTS_24B[
                "Error_PR_AUC"
            ].min(),

        "Max_Error_PR":
            REF_RESULTS_24B[
                "Error_PR_AUC"
            ].max(),

        "Error_PR_Range":
            (
                REF_RESULTS_24B[
                    "Error_PR_AUC"
                ].max()

                -

                REF_RESULTS_24B[
                    "Error_PR_AUC"
                ].min()
            ),

        "Min_5pct_Yield":
            REF_RESULTS_24B[
                "Review5_Errors_Per_100"
            ].min(),

        "Max_5pct_Yield":
            REF_RESULTS_24B[
                "Review5_Errors_Per_100"
            ].max(),

        "Yield_Range":
            (
                REF_RESULTS_24B[
                    "Review5_Errors_Per_100"
                ].max()

                -

                REF_RESULTS_24B[
                    "Review5_Errors_Per_100"
                ].min()
            ),
    },
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B — COMPACT SENSITIVITY SUMMARY"
)

print(
    "=" * 140
)


display(
    SUMMARY_24B
)


# ============================================================
# 24B.17 DESCRIPTIVE BEST CONFIGURATIONS
#
# IMPORTANT:
# These DO NOT alter the frozen configuration.
# ============================================================

BEST_K_24B = (

    K_RESULTS_24B
    .sort_values(
        "Error_PR_AUC",
        ascending=False
    )
    .head(1)
    .copy()
)


BEST_REF_24B = (

    REF_RESULTS_24B
    .sort_values(
        "Error_PR_AUC",
        ascending=False
    )
    .head(1)
    .copy()
)


print(
    "\nDescriptively highest Error PR across K:"
)

display(
    BEST_K_24B
)


print(
    "\nDescriptively highest Error PR across reference sizes:"
)

display(
    BEST_REF_24B
)


print(
    "\nIMPORTANT:"
)

print(
    "These are descriptive sensitivity results only."
)

print(
    "The frozen final configuration remains:"
)

print(
    "k = 20"
)

print(
    "reference N = 80,000"
)


# ============================================================
# 24B.18 SAVE
# ============================================================

save_csv(
    K_RESULTS_24B,
    TAB_DIR
    /
    "module24b_k_sensitivity.csv"
)


save_csv(
    REF_RESULTS_24B,
    TAB_DIR
    /
    "module24b_reference_size_sensitivity.csv"
)


save_csv(
    REPRODUCTION_24B,
    TAB_DIR
    /
    "module24b_locked_configuration_check.csv"
)


save_csv(
    SUMMARY_24B,
    TAB_DIR
    /
    "module24b_sensitivity_summary.csv"
)


save_json(

    {
        "module":
            (
                "LPNS neighborhood and "
                "reference-bank sensitivity"
            ),

        "k_values":
            K_VALUES_24B,

        "reference_sizes":
            REFERENCE_SIZES_24B,

        "locked_k":
            20,

        "locked_reference_size":
            80000,

        "configuration_changed":
            False,

        "external_labels_used_for_selection":
            False,

        "purpose":
            (
                "Post-hoc sensitivity "
                "characterization only"
            ),
    },

    LOG_DIR
    /
    "module24b_manifest.json"
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 24B COMPLETE"
)

print(
    "=" * 140
)


print(
    "\nFinal LPNS configuration remains frozen:"
)

print(
    "k = 20 | reference N = 80,000"
)

Resolved 2015 Common12 matrix: X15 (202944, 12)
2015 expert matrix verified: (202944, 3)
Order = HGB | CatBoost | XGBoost
2020 expert matrix: (319795, 3)

2015 error prevalence: 0.14935647272153896
2020 error prevalence: 0.13807595490861332

Distance-space shapes:
2015: (202944, 12)
2020: (319795, 12)
Reference: (80000, 12)

MODULE 24B — COMPUTING MAX-K NEIGHBORS ONCE
Max-K neighbor search completed in 67.53 seconds.

Evaluating k = 5

Evaluating k = 10

Evaluating k = 20

Evaluating k = 40

Evaluating k = 80

MODULE 24B — K-SENSITIVITY RESULTS


,K,Reference_N,Error_PR_AUC,Error_ROC_AUC,AURC,Review5_Error_Capture,Review5_Errors_Per_100,Review5_Errors_Found,Coverage_2015,Coverage_2020,Mean_Neighbors_2015,Mean_Neighbors_2020,Evaluation_Seconds,Delta_Error_PR_vs_K20,Delta_Error_ROC_vs_K20,Delta_Yield_vs_K20
0,5,80000,0.575812,0.906016,0.026255,0.223458,61.707317,9867,0.00000,0.000000,4.999951,4.999994,4.918832,-0.012611,0.000037,-3.689806
1,10,80000,0.585980,0.905855,0.026371,0.235008,64.896811,10377,0.99997,0.999991,9.999685,9.999916,8.479174,-0.002443,-0.000124,-0.500313
2,20,80000,0.588423,0.905979,0.026320,0.236819,65.397123,10457,0.99997,0.999991,19.998384,19.999384,10.296878,0.000000,0.000000,0.000000
3,40,80000,0.590717,0.906150,0.026316,0.237725,65.647280,10497,0.99997,0.999991,39.993225,39.996807,10.439883,0.002294,0.000171,0.250156
4,80,80000,0.594287,0.906399,0.026298,0.240239,66.341463,10608,0.99997,0.999991,79.974574,79.986466,16.002335,0.005864,0.000420,0.944340



MODULE 24B — REFERENCE-BANK-SIZE SENSITIVITY

Reference N = 10000

Reference N = 20000

Reference N = 40000

Reference N = 80000

MODULE 24B — REFERENCE-SIZE RESULTS


,Reference_N,K,Error_PR_AUC,Error_ROC_AUC,AURC,Review5_Error_Capture,Review5_Errors_Per_100,Review5_Errors_Found,Coverage_2015,Coverage_2020,Mean_Neighbors_2015,Mean_Neighbors_2020,Elapsed_Seconds,Delta_Error_PR_vs_80k,Delta_Error_ROC_vs_80k,Delta_Yield_vs_80k
0,10000,20,0.595760,0.906656,0.026233,0.242504,66.966854,10708,0.999616,0.999800,19.985173,19.992145,26.953957,0.008177,0.000749,1.757348
1,20000,20,0.594481,0.906509,0.026249,0.240149,66.316448,10604,0.999808,0.999903,19.993102,19.996485,36.385804,0.006898,0.000603,1.106942
2,40000,20,0.591072,0.906225,0.026299,0.238858,65.959975,10547,0.999936,0.999975,19.996955,19.998608,57.558424,0.003488,0.000319,0.750469
3,80000,20,0.587584,0.905906,0.026339,0.236140,65.209506,10427,0.999970,0.999991,19.998384,19.999384,63.699199,0.000000,0.000000,0.000000



MODULE 24B — LOCKED CONFIGURATION CHECK


...


Descriptively highest Error PR across K:


,K,Reference_N,Error_PR_AUC,Error_ROC_AUC,AURC,Review5_Error_Capture,Review5_Errors_Per_100,Review5_Errors_Found,Coverage_2015,Coverage_2020,Mean_Neighbors_2015,Mean_Neighbors_2020,Evaluation_Seconds,Delta_Error_PR_vs_K20,Delta_Error_ROC_vs_K20,Delta_Yield_vs_K20
4,80,80000,0.594287,0.906399,0.026298,0.240239,66.341463,10608,0.99997,0.999991,79.974574,79.986466,16.002335,0.005864,0.00042,0.94434



Descriptively highest Error PR across reference sizes:


,Reference_N,K,Error_PR_AUC,Error_ROC_AUC,AURC,Review5_Error_Capture,Review5_Errors_Per_100,Review5_Errors_Found,Coverage_2015,Coverage_2020,Mean_Neighbors_2015,Mean_Neighbors_2020,Elapsed_Seconds,Delta_Error_PR_vs_80k,Delta_Error_ROC_vs_80k,Delta_Yield_vs_80k
0,10000,20,0.59576,0.906656,0.026233,0.242504,66.966854,10708,0.999616,0.9998,19.985173,19.992145,26.953957,0.008177,0.000749,1.757348



IMPORTANT:
These are descriptive sensitivity results only.
The frozen final configuration remains:
k = 20
reference N = 80,000

MODULE 24B COMPLETE

Final LPNS configuration remains frozen:
k = 20 | reference N = 80,000


In [82]:
# ============================================================
# MODULE 25A — FINAL PREDICTIVE BACKBONE STATISTICAL AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Quantify paired uncertainty around:
#
#   LSA-HE Core5 vs HGB
#   LSA-HE Core5 vs repaired Core4
#
# Metrics:
#   PR-AUC  [PRIMARY]
#   ROC-AUC
#
# Uses fixed OOF predictions only.
#
# NO model fitting.
# NO tuning.
# NO threshold selection.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 25A.1 RESOLVE FINAL OOF ARRAYS
# ============================================================

required_25a = [
    "y_dev",
    "LSA_HE_OOF",
    "HGB_CAL_OOF",
    "SEED",
    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_25a
    if x not in globals()
]


if missing:
    raise RuntimeError(
        "Missing Module 25A objects: "
        + ", ".join(missing)
    )


y25 = np.asarray(
    y_dev,
    dtype=int
)


core5_25 = np.asarray(
    LSA_HE_OOF,
    dtype=float
)


hgb_25 = np.asarray(
    HGB_CAL_OOF,
    dtype=float
)


# ------------------------------------------------------------
# Resolve repaired Core4
# ------------------------------------------------------------

core4_candidates_25 = [
    "CORE4_REPAIRED_OOF",
    "REPAIRED_CORE4_OOF",
    "CORE4_OOF_REPAIRED",
    "CORE4_REPAIRED",
]


core4_25 = None
core4_name_25 = None


for name in core4_candidates_25:

    if name not in globals():
        continue

    arr = np.asarray(
        globals()[name]
    )

    if arr.shape == (
        len(y25),
    ):

        core4_25 = arr.astype(
            float
        )

        core4_name_25 = name

        break


# Fallback: reconstruct from repaired experts if present
if core4_25 is None:

    reconstruction_sets = [

        [
            "HGB_CAL_OOF",
            "CATBOOST_CAL_OOF",
            "AUTOINT_REPAIRED_CAL_OOF",
            "DEEPFM_V2_REPAIRED_CAL_OOF",
        ],

        [
            "HGB_CAL_OOF",
            "CATBOOST_CAL_OOF",
            "AUTOINT_REPAIRED_CAL_OOF",
            "DEEPFM_REPAIRED_CAL_OOF",
        ],
    ]


    for names in reconstruction_sets:

        if all(
            name in globals()
            for name in names
        ):

            arrays = [
                np.asarray(
                    globals()[name],
                    dtype=float
                )
                for name in names
            ]


            if all(
                arr.shape
                ==
                (len(y25),)
                for arr in arrays
            ):

                core4_25 = np.mean(
                    np.column_stack(
                        arrays
                    ),
                    axis=1
                )

                core4_name_25 = (
                    "Reconstructed: "
                    + " + ".join(names)
                )

                break


if core4_25 is None:

    raise RuntimeError(
        "Could not resolve repaired Core4 OOF probabilities."
    )


print(
    "Resolved Core4:",
    core4_name_25
)


# ============================================================
# 25A.2 VERIFY POINT ESTIMATES
# ============================================================

def discrimination25(
    y,
    p
):

    return {
        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),
    }


point_rows_25 = []


for name, p in [

    (
        "HGB",
        hgb_25
    ),

    (
        "Repaired Core4",
        core4_25
    ),

    (
        "LSA-HE Core5",
        core5_25
    ),

]:

    m = discrimination25(
        y25,
        p
    )


    point_rows_25.append({

        "Model":
            name,

        **m
    })


POINT_25A = pd.DataFrame(
    point_rows_25
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 25A — POINT ESTIMATE CHECK"
)

print(
    "=" * 120
)


display(
    POINT_25A
)


# Expected approximately:
#
# HGB:
#   PR  0.377012
#   ROC 0.849628
#
# Core4:
#   PR  0.379284
#   ROC 0.850402
#
# Core5:
#   PR  0.379751
#   ROC 0.850608


# ============================================================
# 25A.3 STRATIFIED PAIRED BOOTSTRAP
# ============================================================

B25 = 1000


rng25 = np.random.default_rng(
    SEED + 250100
)


pos25 = np.where(
    y25 == 1
)[0]


neg25 = np.where(
    y25 == 0
)[0]


comparisons25 = {

    "Core5_vs_HGB":
        (
            core5_25,
            hgb_25
        ),

    "Core5_vs_Core4":
        (
            core5_25,
            core4_25
        ),
}


boot25 = {

    name: {
        "PR": [],
        "ROC": [],
    }

    for name
    in comparisons25
}


for b in range(
    B25
):

    pos_b = rng25.choice(
        pos25,
        size=len(pos25),
        replace=True
    )


    neg_b = rng25.choice(
        neg25,
        size=len(neg25),
        replace=True
    )


    idx = np.concatenate([
        pos_b,
        neg_b
    ])


    yb = y25[
        idx
    ]


    for name, (
        p_new,
        p_base

    ) in comparisons25.items():

        new_b = p_new[
            idx
        ]


        base_b = p_base[
            idx
        ]


        boot25[
            name
        ][
            "PR"
        ].append(

            average_precision_score(
                yb,
                new_b
            )
            -
            average_precision_score(
                yb,
                base_b
            )
        )


        boot25[
            name
        ][
            "ROC"
        ].append(

            roc_auc_score(
                yb,
                new_b
            )
            -
            roc_auc_score(
                yb,
                base_b
            )
        )


# ============================================================
# 25A.4 BOOTSTRAP SUMMARY
# ============================================================

def empirical_summary25(
    observed,
    samples
):

    samples = np.asarray(
        samples,
        dtype=float
    )


    lo, hi = np.percentile(
        samples,
        [
            2.5,
            97.5
        ]
    )


    n = len(
        samples
    )


    p_le = (
        np.sum(
            samples <= 0
        )
        +
        1
    ) / (
        n + 1
    )


    p_ge = (
        np.sum(
            samples >= 0
        )
        +
        1
    ) / (
        n + 1
    )


    p_two = min(
        1.0,
        2.0
        *
        min(
            p_le,
            p_ge
        )
    )


    return (
        observed,
        lo,
        hi,
        p_two,
    )


stat_rows_25 = []


for name, (
    p_new,
    p_base

) in comparisons25.items():

    obs_pr = (
        average_precision_score(
            y25,
            p_new
        )
        -
        average_precision_score(
            y25,
            p_base
        )
    )


    obs_roc = (
        roc_auc_score(
            y25,
            p_new
        )
        -
        roc_auc_score(
            y25,
            p_base
        )
    )


    for metric, observed in [

        (
            "PR_AUC",
            obs_pr
        ),

        (
            "ROC_AUC",
            obs_roc
        ),

    ]:

        key = (
            "PR"
            if metric
            ==
            "PR_AUC"
            else
            "ROC"
        )


        obs, lo, hi, p = (
            empirical_summary25(

                observed,

                boot25[
                    name
                ][
                    key
                ]
            )
        )


        stat_rows_25.append({

            "Comparison":
                name,

            "Metric":
                metric,

            "Observed_Delta":
                obs,

            "CI_Low":
                lo,

            "CI_High":
                hi,

            "P_two_sided":
                p,
        })


STATS_25A = pd.DataFrame(
    stat_rows_25
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 25A — PAIRED BOOTSTRAP RESULTS"
)

print(
    "=" * 120
)


display(
    STATS_25A
)


# ============================================================
# 25A.5 MANUSCRIPT-FACING SUMMARY
# ============================================================

SUMMARY_25A = (

    STATS_25A[
        STATS_25A[
            "Metric"
        ]
        ==
        "PR_AUC"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


SUMMARY_25A[
    "Statistically_Distinguishable"
] = (

    (
        SUMMARY_25A[
            "CI_Low"
        ]
        >
        0
    )

    |

    (
        SUMMARY_25A[
            "CI_High"
        ]
        <
        0
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 25A — PRIMARY PR-AUC COMPARISONS"
)

print(
    "=" * 120
)


display(
    SUMMARY_25A
)


# ============================================================
# 25A.6 SAVE
# ============================================================

save_csv(
    POINT_25A,
    TAB_DIR
    /
    "module25a_predictive_point_estimates.csv"
)


save_csv(
    STATS_25A,
    TAB_DIR
    /
    "module25a_predictive_bootstrap.csv"
)


save_csv(
    SUMMARY_25A,
    TAB_DIR
    /
    "module25a_primary_pr_comparisons.csv"
)


save_json(

    {
        "module":
            (
                "Final predictive backbone "
                "statistical audit"
            ),

        "bootstrap_repetitions":
            B25,

        "bootstrap":
            (
                "paired stratified "
                "by disease status"
            ),

        "primary_metric":
            "PR-AUC",

        "comparisons": [
            "Core5 vs HGB",
            "Core5 vs repaired Core4",
        ],

        "model_changes":
            False,
    },

    LOG_DIR
    /
    "module25a_manifest.json"
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 25A COMPLETE"
)

print(
    "=" * 120
)

Resolved Core4: Reconstructed: HGB_CAL_OOF + CATBOOST_CAL_OOF + AUTOINT_REPAIRED_CAL_OOF + DEEPFM_V2_REPAIRED_CAL_OOF

MODULE 25A — POINT ESTIMATE CHECK


,Model,PR_AUC,ROC_AUC
0,HGB,0.377012,0.849628
1,Repaired Core4,0.379284,0.850402
2,LSA-HE Core5,0.379751,0.850608



MODULE 25A — PAIRED BOOTSTRAP RESULTS


,Comparison,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided
0,Core5_vs_HGB,PR_AUC,0.002739,0.001613,0.004054,0.001998
1,Core5_vs_HGB,ROC_AUC,0.000980,0.000676,0.001297,0.001998
2,Core5_vs_Core4,PR_AUC,0.000467,0.000197,0.000745,0.001998
3,Core5_vs_Core4,ROC_AUC,0.000206,0.000112,0.000301,0.001998



MODULE 25A — PRIMARY PR-AUC COMPARISONS


,Comparison,Metric,Observed_Delta,CI_Low,CI_High,P_two_sided,Statistically_Distinguishable
0,Core5_vs_HGB,PR_AUC,0.002739,0.001613,0.004054,0.001998,True
1,Core5_vs_Core4,PR_AUC,0.000467,0.000197,0.000745,0.001998,True



MODULE 25A COMPLETE


In [83]:
# ============================================================
# MODULE 25B — CALIBRATION TRANSPORTABILITY AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Characterize calibration for the Common12 consensus:
#
#   2015 raw consensus
#   2015 calibrated consensus
#   2020 raw consensus
#   2020 frozen 2015-Platt consensus
#
# Metrics:
#   Brier
#   NLL
#   ECE
#   Calibration-in-the-large
#   Calibration intercept
#   Calibration slope
#
# Also generates reliability-diagram data.
#
# NO model fitting except calibration-diagnostic logistic
# regression used to estimate intercept/slope.
#
# NO recalibration of 2020.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
)

from sklearn.linear_model import LogisticRegression


# ============================================================
# 25B.1 REQUIRED OBJECTS
# ============================================================

required_25b = [

    "y15_20d",
    "y20",

    "COMMON12_CONSENSUS",
    "COMMON12_INTERNAL_CAL",

    "P20_CONSENSUS_RAW",
    "P20_CONSENSUS_CAL",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [

    name
    for name in required_25b
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 25B objects: "
        + ", ".join(missing)
    )


# ============================================================
# 25B.2 ARRAYS
# ============================================================

y15_25b = np.asarray(
    y15_20d,
    dtype=int
)


y20_25b = np.asarray(
    y20,
    dtype=int
)


p15_raw_25b = np.asarray(
    COMMON12_CONSENSUS,
    dtype=float
)


p15_cal_25b = np.asarray(
    COMMON12_INTERNAL_CAL,
    dtype=float
)


p20_raw_25b = np.asarray(
    P20_CONSENSUS_RAW,
    dtype=float
)


p20_cal_25b = np.asarray(
    P20_CONSENSUS_CAL,
    dtype=float
)


# ============================================================
# 25B.3 SANITY CHECKS
# ============================================================

for name, y, p in [

    (
        "2015 Raw",
        y15_25b,
        p15_raw_25b
    ),

    (
        "2015 Calibrated",
        y15_25b,
        p15_cal_25b
    ),

    (
        "2020 Raw",
        y20_25b,
        p20_raw_25b
    ),

    (
        "2020 Frozen Platt",
        y20_25b,
        p20_cal_25b
    ),
]:

    assert len(y) == len(p)

    assert np.all(
        np.isfinite(p)
    )

    assert np.min(p) >= 0

    assert np.max(p) <= 1


# ============================================================
# 25B.4 ECE
# ============================================================

def ece_25b(
    y,
    p,
    n_bins=15
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )


    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )


    ids = np.digitize(
        p,
        edges[1:-1],
        right=False
    )


    ece = 0.0


    for b in range(
        n_bins
    ):

        mask = (
            ids == b
        )


        if not np.any(
            mask
        ):
            continue


        frac = np.mean(
            mask
        )


        obs = np.mean(
            y[
                mask
            ]
        )


        pred = np.mean(
            p[
                mask
            ]
        )


        ece += (
            frac
            *
            abs(
                obs
                -
                pred
            )
        )


    return float(
        ece
    )


# ============================================================
# 25B.5 LOGIT TRANSFORM
# ============================================================

def logit_25b(
    p
):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-7,
        1 - 1e-7
    )


    return np.log(
        p
        /
        (
            1 - p
        )
    )


# ============================================================
# 25B.6 CALIBRATION INTERCEPT + SLOPE
#
# Logistic recalibration diagnostic:
#
# logit P(Y=1) = intercept + slope * logit(predicted probability)
#
# Ideal:
#   intercept = 0
#   slope     = 1
# ============================================================

def calibration_intercept_slope_25b(
    y,
    p
):

    x = logit_25b(
        p
    ).reshape(
        -1,
        1
    )


    # Very large C approximates unpenalized logistic regression.
    model = LogisticRegression(

        penalty="l2",
        C=1e6,

        solver="lbfgs",
        max_iter=5000,
    )


    model.fit(
        x,
        y
    )


    return (
        float(
            model.intercept_[0]
        ),
        float(
            model.coef_[0, 0]
        ),
    )


# ============================================================
# 25B.7 CALIBRATION-IN-THE-LARGE
#
# Simple probability-scale difference:
#
# mean predicted probability - observed prevalence
#
# 0 is ideal.
# ============================================================

def citl_probability_25b(
    y,
    p
):

    return float(
        np.mean(p)
        -
        np.mean(y)
    )


# ============================================================
# 25B.8 CALIBRATION SUMMARY
# ============================================================

calibration_sets_25b = {

    "2015 Raw Common12":
        (
            y15_25b,
            p15_raw_25b
        ),

    "2015 Calibrated Common12":
        (
            y15_25b,
            p15_cal_25b
        ),

    "2020 Raw Common12":
        (
            y20_25b,
            p20_raw_25b
        ),

    "2020 Frozen 2015-Platt":
        (
            y20_25b,
            p20_cal_25b
        ),
}


rows_25b = []


for name, (
    y,
    p

) in calibration_sets_25b.items():

    intercept, slope = (
        calibration_intercept_slope_25b(
            y,
            p
        )
    )


    rows_25b.append({

        "Model":
            name,

        "N":
            len(y),

        "Observed_Prevalence":
            np.mean(y),

        "Mean_Predicted":
            np.mean(p),

        "Calibration_In_Large_Prob":
            citl_probability_25b(
                y,
                p
            ),

        "Calibration_Intercept":
            intercept,

        "Calibration_Slope":
            slope,

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                p,
                labels=[0, 1]
            ),

        "ECE_15":
            ece_25b(
                y,
                p,
                n_bins=15
            ),
    })


CALIBRATION_25B = pd.DataFrame(
    rows_25b
)


print(
    "=" * 150
)

print(
    "MODULE 25B — CALIBRATION SUMMARY"
)

print(
    "=" * 150
)


display(
    CALIBRATION_25B
)


# ============================================================
# 25B.9 RELIABILITY DIAGRAM DATA
#
# Quantile bins are used because the outcome is uncommon
# and probabilities are concentrated near zero.
# ============================================================

def reliability_table_25b(
    y,
    p,
    label,
    n_bins=10
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )


    # Quantile bin edges
    quantiles = np.linspace(
        0,
        1,
        n_bins + 1
    )


    edges = np.quantile(
        p,
        quantiles
    )


    # Remove duplicate edges
    edges = np.unique(
        edges
    )


    if len(edges) < 3:

        raise RuntimeError(
            f"Too few unique probability bins for {label}"
        )


    # Ensure numerical inclusion
    edges[0] = -np.inf
    edges[-1] = np.inf


    bin_ids = np.digitize(
        p,
        edges[1:-1],
        right=True
    )


    rows = []


    for b in range(
        len(edges) - 1
    ):

        mask = (
            bin_ids == b
        )


        if not np.any(
            mask
        ):
            continue


        n = int(
            mask.sum()
        )


        mean_pred = float(
            np.mean(
                p[
                    mask
                ]
            )
        )


        observed = float(
            np.mean(
                y[
                    mask
                ]
            )
        )


        se = np.sqrt(
            max(
                observed
                *
                (
                    1 - observed
                )
                /
                n,
                0
            )
        )


        rows.append({

            "Model":
                label,

            "Bin":
                b + 1,

            "N":
                n,

            "Mean_Predicted":
                mean_pred,

            "Observed_Rate":
                observed,

            "Observed_CI_Low":
                max(
                    0.0,
                    observed
                    -
                    1.96 * se
                ),

            "Observed_CI_High":
                min(
                    1.0,
                    observed
                    +
                    1.96 * se
                ),

            "Absolute_Gap":
                abs(
                    mean_pred
                    -
                    observed
                ),
        })


    return pd.DataFrame(
        rows
    )


reliability_tables_25b = []


for name, (
    y,
    p

) in calibration_sets_25b.items():

    reliability_tables_25b.append(

        reliability_table_25b(
            y,
            p,
            name,
            n_bins=10
        )
    )


RELIABILITY_25B = pd.concat(

    reliability_tables_25b,

    ignore_index=True
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 25B — RELIABILITY-DIAGRAM DATA"
)

print(
    "=" * 150
)


display(
    RELIABILITY_25B
)


# ============================================================
# 25B.10 TEMPORAL CALIBRATION TRANSFER SUMMARY
# ============================================================

TRANSFER_25B = pd.DataFrame([

    {
        "Metric":
            "Mean predicted probability",

        "2015_Calibrated":
            np.mean(
                p15_cal_25b
            ),

        "2020_Frozen_Platt":
            np.mean(
                p20_cal_25b
            ),

        "Observed_2015":
            np.mean(
                y15_25b
            ),

        "Observed_2020":
            np.mean(
                y20_25b
            ),
    },

    {
        "Metric":
            "Brier",

        "2015_Calibrated":
            brier_score_loss(
                y15_25b,
                p15_cal_25b
            ),

        "2020_Frozen_Platt":
            brier_score_loss(
                y20_25b,
                p20_cal_25b
            ),

        "Observed_2015":
            np.nan,

        "Observed_2020":
            np.nan,
    },

    {
        "Metric":
            "ECE",

        "2015_Calibrated":
            ece_25b(
                y15_25b,
                p15_cal_25b
            ),

        "2020_Frozen_Platt":
            ece_25b(
                y20_25b,
                p20_cal_25b
            ),

        "Observed_2015":
            np.nan,

        "Observed_2020":
            np.nan,
    },
])


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 25B — TEMPORAL CALIBRATION TRANSFER"
)

print(
    "=" * 150
)


display(
    TRANSFER_25B
)


# ============================================================
# 25B.11 SAVE
# ============================================================

save_csv(
    CALIBRATION_25B,
    TAB_DIR
    /
    "module25b_calibration_summary.csv"
)


save_csv(
    RELIABILITY_25B,
    TAB_DIR
    /
    "module25b_reliability_diagram_data.csv"
)


save_csv(
    TRANSFER_25B,
    TAB_DIR
    /
    "module25b_temporal_calibration_transfer.csv"
)


save_json(

    {
        "module":
            "Calibration transportability audit",

        "datasets": [
            "BRFSS-2015 Common12",
            "BRFSS-2020 Common12",
        ],

        "external_recalibration":
            False,

        "reliability_diagram_bins":
            10,

        "reliability_diagram_binning":
            "quantile",

        "ece_bins":
            15,

        "ideal_calibration_intercept":
            0.0,

        "ideal_calibration_slope":
            1.0,
    },

    LOG_DIR
    /
    "module25b_manifest.json"
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 25B COMPLETE"
)

print(
    "=" * 150
)

MODULE 25B — CALIBRATION SUMMARY


,Model,N,Observed_Prevalence,Mean_Predicted,Calibration_In_Large_Prob,Calibration_Intercept,Calibration_Slope,Brier,NLL,ECE_15
0,2015 Raw Common12,202944,0.094184,0.113689,0.019505,-0.255603,0.999619,0.072023,0.244842,0.019505
1,2015 Calibrated Common12,202944,0.094184,0.094182,-0.000001,0.000084,1.000035,0.071276,0.242413,0.001209
2,2020 Raw Common12,319795,0.085595,0.103156,0.017560,-0.309396,0.959415,0.066587,0.228933,0.017560
3,2020 Frozen 2015-Platt,319795,0.085595,0.085322,-0.000273,-0.063736,0.959771,0.065841,0.226781,0.002159



MODULE 25B — RELIABILITY-DIAGRAM DATA


,Model,Bin,N,Mean_Predicted,Observed_Rate,Observed_CI_Low,Observed_CI_High,Absolute_Gap
0,2015 Raw Common12,1,20295,0.003619,0.002611,0.001909,0.003314,0.001008
1,2015 Raw Common12,2,20295,0.008053,0.006553,0.005443,0.007663,0.001499
2,2015 Raw Common12,3,20294,0.015179,0.011580,0.010108,0.013052,0.003599
3,2015 Raw Common12,4,20294,0.026160,0.018971,0.017094,0.020848,0.007189
4,2015 Raw Common12,5,20294,0.043506,0.033360,0.030889,0.035830,0.010147
5,2015 Raw Common12,6,20294,0.069948,0.055484,0.052335,0.058634,0.014464
6,2015 Raw Common12,7,20295,0.110385,0.087657,0.083766,0.091548,0.022728
7,2015 Raw Common12,8,20294,0.168195,0.138415,0.133664,0.143167,0.029780
8,2015 Raw Common12,9,20294,0.253824,0.210210,0.204604,0.215816,0.043614
9,2015 Raw Common12,10,20295,0.438015,0.376989,0.370322,0.383657,0.061025



MODULE 25B — TEMPORAL CALIBRATION TRANSFER


,Metric,2015_Calibrated,2020_Frozen_Platt,Observed_2015,Observed_2020
0,Mean predicted probability,0.094182,0.085322,0.094184,0.085595
1,Brier,0.071276,0.065841,NaN,NaN
2,ECE,0.001209,0.002159,NaN,NaN



MODULE 25B COMPLETE


In [84]:
# ============================================================
# MODULE 26 — FINAL SUBGROUP PERFORMANCE + CALIBRATION AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Complete the fairness characterization by evaluating:
#
#   - subgroup prevalence
#   - PR-AUC
#   - ROC-AUC
#   - Brier score
#   - ECE
#   - calibration-in-the-large
#   - calibration intercept
#   - calibration slope
#
# Groups:
#   Sex
#   Age:
#       18–44
#       45–64
#       65+
#
# INTERNAL:
#   frozen LSA-HE Core5 OOF
#
# EXTERNAL:
#   frozen Common12 2015-Platt probabilities
#
# IMPORTANT:
#   No subgroup thresholds
#   No subgroup calibration
#   No retraining
#   No fairness optimization
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

from sklearn.linear_model import LogisticRegression


# ============================================================
# 26.1 REQUIRED OBJECTS
# ============================================================

required_26 = [

    "y_dev",
    "LSA_HE_OOF",

    "X20",
    "y20",
    "P20_CONSENSUS_CAL",

    "save_csv",
    "save_json",
    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x for x in required_26
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 26 objects: "
        + ", ".join(missing)
    )


# ============================================================
# 26.2 RESOLVE INTERNAL FEATURE DATAFRAME
# ============================================================

candidate_internal_26 = [

    "X_dev",
    "X_DEV",
    "X15_FULL",
    "X15",
]


X_INTERNAL_26 = None
X_INTERNAL_NAME_26 = None


for name in candidate_internal_26:

    if name not in globals():
        continue

    obj = globals()[name]

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    if len(obj) != len(
        y_dev
    ):
        continue

    if (
        "Sex" not in obj.columns
        or
        "Age" not in obj.columns
    ):
        continue

    X_INTERNAL_26 = (
        obj
        .copy()
        .reset_index(drop=True)
    )

    X_INTERNAL_NAME_26 = name

    break


if X_INTERNAL_26 is None:

    # Safe fallback search
    candidates = []

    for name, obj in list(
        globals().items()
    ):

        if not isinstance(
            obj,
            pd.DataFrame
        ):
            continue

        if len(obj) != len(
            y_dev
        ):
            continue

        if (
            "Sex" in obj.columns
            and
            "Age" in obj.columns
        ):

            candidates.append(
                name
            )


    if len(candidates) == 1:

        X_INTERNAL_NAME_26 = (
            candidates[0]
        )

        X_INTERNAL_26 = (
            globals()[
                candidates[0]
            ]
            .copy()
            .reset_index(drop=True)
        )


if X_INTERNAL_26 is None:

    raise RuntimeError(
        "Could not uniquely resolve the internal BRFSS-2015 "
        "feature DataFrame containing Sex and Age."
    )


print(
    "Internal feature matrix:",
    X_INTERNAL_NAME_26,
    X_INTERNAL_26.shape
)


# ============================================================
# 26.3 ARRAYS
# ============================================================

y15_26 = np.asarray(
    y_dev,
    dtype=int
)


p15_26 = np.asarray(
    LSA_HE_OOF,
    dtype=float
)


y20_26 = np.asarray(
    y20,
    dtype=int
)


p20_26 = np.asarray(
    P20_CONSENSUS_CAL,
    dtype=float
)


assert len(
    X_INTERNAL_26
) == len(
    y15_26
) == len(
    p15_26
)


assert len(
    X20
) == len(
    y20_26
) == len(
    p20_26
)


# ============================================================
# 26.4 ECE
# ============================================================

def ece26(
    y,
    p,
    n_bins=15,
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )


    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )


    ids = np.digitize(
        p,
        edges[1:-1],
        right=False
    )


    out = 0.0


    for b in range(
        n_bins
    ):

        mask = (
            ids == b
        )


        if not np.any(
            mask
        ):
            continue


        out += (

            np.mean(
                mask
            )

            *

            abs(
                np.mean(
                    y[
                        mask
                    ]
                )
                -
                np.mean(
                    p[
                        mask
                    ]
                )
            )
        )


    return float(
        out
    )


# ============================================================
# 26.5 CALIBRATION INTERCEPT / SLOPE
# ============================================================

def calibration26(
    y,
    p,
):

    y = np.asarray(
        y,
        dtype=int
    )


    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-7,
        1 - 1e-7
    )


    if len(
        np.unique(
            y
        )
    ) < 2:

        return (
            np.nan,
            np.nan
        )


    x = np.log(
        p
        /
        (
            1 - p
        )
    ).reshape(
        -1,
        1
    )


    model = LogisticRegression(

        C=1e6,
        penalty="l2",

        solver="lbfgs",
        max_iter=5000,
    )


    model.fit(
        x,
        y
    )


    return (

        float(
            model.intercept_[0]
        ),

        float(
            model.coef_[0, 0]
        ),
    )


# ============================================================
# 26.6 MAP SEX
#
# BRFSS convention:
#
# 0 = Female
# 1 = Male
#
# Verify values before proceeding.
# ============================================================

print(
    "\nInternal Sex values:",
    sorted(
        pd.unique(
            X_INTERNAL_26[
                "Sex"
            ]
        )
    )
)


print(
    "External Sex values:",
    sorted(
        pd.unique(
            X20[
                "Sex"
            ]
        )
    )
)


def sex_label26(
    s
):

    s = np.asarray(
        s
    )


    labels = np.full(
        len(s),
        "Unknown",
        dtype=object
    )


    labels[
        s == 0
    ] = "Female"


    labels[
        s == 1
    ] = "Male"


    return labels


# ============================================================
# 26.7 AGE GROUPS
#
# BRFSS coded age:
#
#   1: 18–24
#   2: 25–29
#   3: 30–34
#   4: 35–39
#   5: 40–44
#
#   6: 45–49
#   7: 50–54
#   8: 55–59
#   9: 60–64
#
#   10: 65–69
#   11: 70–74
#   12: 75–79
#   13: 80+
# ============================================================

def age_group26(
    age
):

    age = np.asarray(
        age
    )


    out = np.full(
        len(age),
        "Unknown",
        dtype=object
    )


    out[
        (
            age >= 1
        )
        &
        (
            age <= 5
        )
    ] = "18-44"


    out[
        (
            age >= 6
        )
        &
        (
            age <= 9
        )
    ] = "45-64"


    out[
        age >= 10
    ] = "65+"


    return out


# ============================================================
# 26.8 CREATE GROUP LABELS
# ============================================================

sex15_26 = sex_label26(
    X_INTERNAL_26[
        "Sex"
    ].to_numpy()
)


sex20_26 = sex_label26(
    X20[
        "Sex"
    ].to_numpy()
)


age15_26 = age_group26(
    X_INTERNAL_26[
        "Age"
    ].to_numpy()
)


age20_26 = age_group26(
    X20[
        "Age"
    ].to_numpy()
)


# ============================================================
# 26.9 GROUP METRIC HELPER
# ============================================================

def subgroup_metric26(
    dataset_name,
    grouping_name,
    labels,
    y,
    p,
):

    rows = []


    for group in pd.unique(
        labels
    ):

        if group == "Unknown":
            continue


        mask = (
            labels == group
        )


        yg = y[
            mask
        ]


        pg = p[
            mask
        ]


        n = int(
            mask.sum()
        )


        positives = int(
            yg.sum()
        )


        prevalence = float(
            yg.mean()
        )


        mean_pred = float(
            pg.mean()
        )


        intercept, slope = (
            calibration26(
                yg,
                pg
            )
        )


        rows.append({

            "Dataset":
                dataset_name,

            "Grouping":
                grouping_name,

            "Group":
                group,

            "N":
                n,

            "Positives":
                positives,

            "Prevalence":
                prevalence,

            "Mean_Predicted":
                mean_pred,

            "Calibration_In_Large_Prob":
                mean_pred
                -
                prevalence,

            "PR_AUC":
                average_precision_score(
                    yg,
                    pg
                ),

            "ROC_AUC":
                roc_auc_score(
                    yg,
                    pg
                ),

            "Brier":
                brier_score_loss(
                    yg,
                    pg
                ),

            "ECE_15":
                ece26(
                    yg,
                    pg,
                    15
                ),

            "Calibration_Intercept":
                intercept,

            "Calibration_Slope":
                slope,
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 26.10 SEX RESULTS
# ============================================================

SEX_INTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2015",

        "Sex",

        sex15_26,

        y15_26,
        p15_26,
    )
)


SEX_EXTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2020",

        "Sex",

        sex20_26,

        y20_26,
        p20_26,
    )
)


SEX_26 = pd.concat(

    [
        SEX_INTERNAL_26,
        SEX_EXTERNAL_26,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 — SEX-SPECIFIC DISCRIMINATION / CALIBRATION"
)

print(
    "=" * 150
)


display(
    SEX_26
)


# ============================================================
# 26.11 AGE RESULTS
# ============================================================

AGE_INTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2015",

        "Age",

        age15_26,

        y15_26,
        p15_26,
    )
)


AGE_EXTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2020",

        "Age",

        age20_26,

        y20_26,
        p20_26,
    )
)


AGE_26 = pd.concat(

    [
        AGE_INTERNAL_26,
        AGE_EXTERNAL_26,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 — AGE-SPECIFIC DISCRIMINATION / CALIBRATION"
)

print(
    "=" * 150
)


display(
    AGE_26
)


# ============================================================
# 26.12 INTERSECTIONAL SEX × AGE
#
# Mainly supplementary characterization.
# ============================================================

intersection15_26 = np.array(
    [
        f"{s}_{a}"
        for s, a in zip(
            sex15_26,
            age15_26
        )
    ],
    dtype=object
)


intersection20_26 = np.array(
    [
        f"{s}_{a}"
        for s, a in zip(
            sex20_26,
            age20_26
        )
    ],
    dtype=object
)


INTERSECTION_INTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2015",

        "Sex×Age",

        intersection15_26,

        y15_26,
        p15_26,
    )
)


INTERSECTION_EXTERNAL_26 = (
    subgroup_metric26(

        "BRFSS-2020",

        "Sex×Age",

        intersection20_26,

        y20_26,
        p20_26,
    )
)


INTERSECTION_26 = pd.concat(

    [
        INTERSECTION_INTERNAL_26,
        INTERSECTION_EXTERNAL_26,
    ],

    ignore_index=True
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 — INTERSECTIONAL SEX × AGE AUDIT"
)

print(
    "=" * 150
)


display(
    INTERSECTION_26
)


# ============================================================
# 26.13 RANGE / GAP SUMMARY
#
# Descriptive ranges only.
# We already have bootstrap inference for TPR/FPR gaps
# from Module 21B.
# ============================================================

gap_rows_26 = []


for source_df, grouping in [

    (
        SEX_26,
        "Sex"
    ),

    (
        AGE_26,
        "Age"
    ),

]:

    for dataset in [

        "BRFSS-2015",
        "BRFSS-2020",
    ]:

        sub = source_df[
            source_df[
                "Dataset"
            ]
            ==
            dataset
        ]


        for metric in [

            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "ECE_15",
            "Calibration_In_Large_Prob",
            "Calibration_Slope",
        ]:

            vals = (
                sub[
                    metric
                ]
                .to_numpy(
                    dtype=float
                )
            )


            gap_rows_26.append({

                "Dataset":
                    dataset,

                "Grouping":
                    grouping,

                "Metric":
                    metric,

                "Minimum":
                    np.nanmin(
                        vals
                    ),

                "Maximum":
                    np.nanmax(
                        vals
                    ),

                "Range":
                    np.nanmax(
                        vals
                    )
                    -
                    np.nanmin(
                        vals
                    ),
            })


GAPS_26 = pd.DataFrame(
    gap_rows_26
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 — SUBGROUP METRIC RANGES"
)

print(
    "=" * 150
)


display(
    GAPS_26
)


# ============================================================
# 26.14 EXTERNAL CALIBRATION-IN-LARGE CHECK
#
# Useful concise table for manuscript.
# ============================================================

EXTERNAL_CALIBRATION_GROUPS_26 = pd.concat(

    [
        SEX_EXTERNAL_26,
        AGE_EXTERNAL_26,
    ],

    ignore_index=True
)[
    [
        "Grouping",
        "Group",
        "N",
        "Prevalence",
        "Mean_Predicted",
        "Calibration_In_Large_Prob",
        "ECE_15",
        "Calibration_Intercept",
        "Calibration_Slope",
    ]
]


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 — EXTERNAL GROUP CALIBRATION"
)

print(
    "=" * 150
)


display(
    EXTERNAL_CALIBRATION_GROUPS_26
)


# ============================================================
# 26.15 SAVE
# ============================================================

save_csv(
    SEX_26,
    TAB_DIR
    /
    "module26_sex_discrimination_calibration.csv"
)


save_csv(
    AGE_26,
    TAB_DIR
    /
    "module26_age_discrimination_calibration.csv"
)


save_csv(
    INTERSECTION_26,
    TAB_DIR
    /
    "module26_intersectional_discrimination_calibration.csv"
)


save_csv(
    GAPS_26,
    TAB_DIR
    /
    "module26_subgroup_metric_ranges.csv"
)


save_csv(
    EXTERNAL_CALIBRATION_GROUPS_26,
    TAB_DIR
    /
    "module26_external_group_calibration.csv"
)


save_json(

    {
        "module":
            "Subgroup discrimination and calibration audit",

        "groups": [
            "Sex",
            "Age",
            "Sex×Age",
        ],

        "age_groups": [
            "18-44",
            "45-64",
            "65+",
        ],

        "group_specific_thresholds":
            False,

        "group_specific_recalibration":
            False,

        "fairness_optimization":
            False,

        "interpretation":
            (
                "Audit of subgroup discrimination and "
                "calibration; not a claim of algorithmic fairness."
            ),
    },

    LOG_DIR
    /
    "module26_manifest.json"
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 26 COMPLETE"
)

print(
    "=" * 150
)

Internal feature matrix: X_dev (202944, 21)

Internal Sex values: [np.float64(0.0), np.float64(1.0)]
External Sex values: [np.float32(0.0), np.float32(1.0)]

MODULE 26 — SEX-SPECIFIC DISCRIMINATION / CALIBRATION


,Dataset,Grouping,Group,N,Positives,Prevalence,Mean_Predicted,Calibration_In_Large_Prob,PR_AUC,ROC_AUC,Brier,ECE_15,Calibration_Intercept,Calibration_Slope
0,BRFSS-2015,Sex,Female,113674,8154,0.071731,0.081351,0.009620,0.330173,0.852706,0.056104,0.009756,-0.104329,1.034553
1,BRFSS-2015,Sex,Male,89270,10960,0.122774,0.136299,0.013525,0.413560,0.837459,0.087559,0.013538,-0.122135,1.021361
2,BRFSS-2020,Sex,Female,167805,11234,0.066947,0.066436,-0.000511,0.280079,0.833816,0.054407,0.002826,-0.108074,0.940027
3,BRFSS-2020,Sex,Male,151990,16139,0.106185,0.106173,-0.000011,0.376786,0.837462,0.078464,0.002354,-0.036108,0.975600



MODULE 26 — AGE-SPECIFIC DISCRIMINATION / CALIBRATION


,Dataset,Grouping,Group,N,Positives,Prevalence,Mean_Predicted,Calibration_In_Large_Prob,PR_AUC,ROC_AUC,Brier,ECE_15,Calibration_Intercept,Calibration_Slope
0,BRFSS-2015,Age,45-64,88079,6175,0.070108,0.080006,0.009899,0.318253,0.839444,0.055608,0.010037,-0.116292,1.029410
1,BRFSS-2015,Age,65+,71407,12331,0.172686,0.191093,0.018407,0.415426,0.771288,0.122431,0.018504,-0.111441,1.029380
2,BRFSS-2015,Age,18-44,43458,608,0.013991,0.016630,0.002639,0.155446,0.853437,0.012739,0.002748,-0.153773,1.012467
3,BRFSS-2020,Age,45-64,110616,7656,0.069212,0.066682,-0.002530,0.280789,0.805768,0.056617,0.003384,-0.033197,0.962469
4,BRFSS-2020,Age,65+,110851,18446,0.166404,0.169062,0.002659,0.371609,0.746292,0.122675,0.003022,-0.046021,0.981776
5,BRFSS-2020,Age,18-44,98328,1271,0.012926,0.011887,-0.001039,0.103895,0.788943,0.012145,0.001599,-0.178640,0.925218



MODULE 26 — INTERSECTIONAL SEX × AGE AUDIT


...


MODULE 26 — SUBGROUP METRIC RANGES


,Dataset,Grouping,Metric,Minimum,Maximum,Range
0,BRFSS-2015,Sex,PR_AUC,0.330173,0.413560,0.083387
1,BRFSS-2015,Sex,ROC_AUC,0.837459,0.852706,0.015247
2,BRFSS-2015,Sex,Brier,0.056104,0.087559,0.031455
3,BRFSS-2015,Sex,ECE_15,0.009756,0.013538,0.003782
4,BRFSS-2015,Sex,Calibration_In_Large_Prob,0.009620,0.013525,0.003905
5,BRFSS-2015,Sex,Calibration_Slope,1.021361,1.034553,0.013192
6,BRFSS-2020,Sex,PR_AUC,0.280079,0.376786,0.096706
7,BRFSS-2020,Sex,ROC_AUC,0.833816,0.837462,0.003645
8,BRFSS-2020,Sex,Brier,0.054407,0.078464,0.024057
9,BRFSS-2020,Sex,ECE_15,0.002354,0.002826,0.000472



MODULE 26 — EXTERNAL GROUP CALIBRATION


,Grouping,Group,N,Prevalence,Mean_Predicted,Calibration_In_Large_Prob,ECE_15,Calibration_Intercept,Calibration_Slope
0,Sex,Female,167805,0.066947,0.066436,-0.000511,0.002826,-0.108074,0.940027
1,Sex,Male,151990,0.106185,0.106173,-0.000011,0.002354,-0.036108,0.975600
2,Age,45-64,110616,0.069212,0.066682,-0.002530,0.003384,-0.033197,0.962469
3,Age,65+,110851,0.166404,0.169062,0.002659,0.003022,-0.046021,0.981776
4,Age,18-44,98328,0.012926,0.011887,-0.001039,0.001599,-0.178640,0.925218



MODULE 26 COMPLETE


In [85]:
# ============================================================
# MODULE 27 — DENSE SELECTIVE-REVIEW CHARACTERIZATION
#
# PURPOSE
# ------------------------------------------------------------
# Compare:
#
#   1. Confidence
#   2. Confidence + Disagreement
#   3. LPNS reliability
#
# over review budgets from 1% to 25%.
#
# Metrics:
#   - Error capture
#   - Errors found per 100 reviews
#   - Residual error rate among unreviewed cases
#   - LPNS gain vs Confidence+Disagreement
#
# Uses frozen BRFSS-2020 reliability scores.
#
# NO fitting.
# NO tuning.
# NO threshold changes.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 27.1 REQUIRED OBJECTS
# ============================================================

required_27 = [

    "error20_24a",
    "ABLATION_RISK_24A",

    "save_csv",
    "save_json",

    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    name
    for name in required_27
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 27 objects: "
        + ", ".join(missing)
    )


# ============================================================
# 27.2 ARRAYS
# ============================================================

error27 = np.asarray(
    error20_24a,
    dtype=int
)


risk_conf27 = np.asarray(

    ABLATION_RISK_24A[
        "Confidence"
    ],

    dtype=float
)


risk_dis27 = np.asarray(

    ABLATION_RISK_24A[
        "Confidence+Disagreement"
    ],

    dtype=float
)


risk_lpns27 = np.asarray(

    ABLATION_RISK_24A[
        "Confidence+Disagreement+LPNS"
    ],

    dtype=float
)


assert len(
    error27
) == len(
    risk_conf27
) == len(
    risk_dis27
) == len(
    risk_lpns27
)


# ============================================================
# 27.3 REVIEW BUDGETS
#
# 1% to 25%, step 1%.
# ============================================================

BUDGETS_27 = np.arange(
    0.01,
    0.251,
    0.01
)


# ============================================================
# 27.4 CURVE HELPER
# ============================================================

def review_curve27(
    error,
    risk,
    method,
):

    order = np.argsort(
        -risk
    )


    total_errors = int(
        error.sum()
    )


    n_total = len(
        error
    )


    cumulative_errors = np.cumsum(
        error[
            order
        ]
    )


    rows = []


    for rate in BUDGETS_27:

        n_review = max(
            1,
            int(
                round(
                    rate
                    *
                    n_total
                )
            )
        )


        found = int(
            cumulative_errors[
                n_review - 1
            ]
        )


        remaining_errors = (
            total_errors
            -
            found
        )


        remaining_n = (
            n_total
            -
            n_review
        )


        capture = (
            found
            /
            total_errors
        )


        errors_per_100 = (

            100.0
            *
            found
            /
            n_review
        )


        residual_error_rate = (

            remaining_errors
            /
            remaining_n

            if remaining_n > 0
            else np.nan
        )


        rows.append({

            "Method":
                method,

            "Review_Rate":
                rate,

            "Review_Percent":
                100 * rate,

            "Review_N":
                n_review,

            "Errors_Found":
                found,

            "Error_Capture":
                capture,

            "Errors_Per_100_Reviews":
                errors_per_100,

            "Remaining_Errors":
                remaining_errors,

            "Residual_Error_Rate":
                residual_error_rate,
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 27.5 GENERATE CURVES
# ============================================================

CURVE_CONF_27 = review_curve27(

    error27,
    risk_conf27,
    "Confidence"
)


CURVE_DIS_27 = review_curve27(

    error27,
    risk_dis27,
    "Confidence+Disagreement"
)


CURVE_LPNS_27 = review_curve27(

    error27,
    risk_lpns27,
    "LPNS"
)


SELECTIVE_CURVES_27 = pd.concat(

    [
        CURVE_CONF_27,
        CURVE_DIS_27,
        CURVE_LPNS_27,
    ],

    ignore_index=True
)


print(
    "=" * 150
)

print(
    "MODULE 27 — DENSE SELECTIVE-REVIEW CURVES"
)

print(
    "=" * 150
)


display(
    SELECTIVE_CURVES_27
)


# ============================================================
# 27.6 LPNS ADVANTAGE VS CONF+DIS
# ============================================================

lpns27 = (

    CURVE_LPNS_27
    .set_index(
        "Review_Rate"
    )
)


dis27 = (

    CURVE_DIS_27
    .set_index(
        "Review_Rate"
    )
)


ADVANTAGE_27 = pd.DataFrame({

    "Review_Rate":
        BUDGETS_27,

    "Review_Percent":
        100 * BUDGETS_27,

    "Delta_Error_Capture":
        (
            lpns27[
                "Error_Capture"
            ].to_numpy()
            -
            dis27[
                "Error_Capture"
            ].to_numpy()
        ),

    "Delta_Errors_Per_100":
        (
            lpns27[
                "Errors_Per_100_Reviews"
            ].to_numpy()
            -
            dis27[
                "Errors_Per_100_Reviews"
            ].to_numpy()
        ),

    "Additional_Errors_Found":
        (
            lpns27[
                "Errors_Found"
            ].to_numpy()
            -
            dis27[
                "Errors_Found"
            ].to_numpy()
        ),

    "Delta_Residual_Error_Rate":
        (
            lpns27[
                "Residual_Error_Rate"
            ].to_numpy()
            -
            dis27[
                "Residual_Error_Rate"
            ].to_numpy()
        ),
})


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 27 — LPNS ADVANTAGE VS CONFIDENCE+DISAGREEMENT"
)

print(
    "=" * 150
)


display(
    ADVANTAGE_27
)


# ============================================================
# 27.7 KEY OPERATING POINTS
# ============================================================

KEY_RATES_27 = [

    0.01,
    0.02,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
]


KEY_POINTS_27 = (

    SELECTIVE_CURVES_27[
        SELECTIVE_CURVES_27[
            "Review_Rate"
        ].isin(
            KEY_RATES_27
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 27 — KEY REVIEW BUDGETS"
)

print(
    "=" * 150
)


display(
    KEY_POINTS_27
)


# ============================================================
# 27.8 CROSSOVER ANALYSIS
#
# Find review budgets where LPNS beats / loses to Conf+Dis.
# ============================================================

ADVANTAGE_27[
    "LPNS_Better_Capture"
] = (

    ADVANTAGE_27[
        "Delta_Error_Capture"
    ]
    >
    0
)


ADVANTAGE_27[
    "LPNS_Better_Yield"
] = (

    ADVANTAGE_27[
        "Delta_Errors_Per_100"
    ]
    >
    0
)


positive_capture = (

    ADVANTAGE_27[
        ADVANTAGE_27[
            "LPNS_Better_Capture"
        ]
    ]
)


negative_capture = (

    ADVANTAGE_27[
        ~ADVANTAGE_27[
            "LPNS_Better_Capture"
        ]
    ]
)


CROSSOVER_SUMMARY_27 = pd.DataFrame([

    {
        "Quantity":
            "Highest budget with positive LPNS capture advantage",

        "Value":
            (
                positive_capture[
                    "Review_Percent"
                ].max()

                if len(
                    positive_capture
                ) > 0

                else np.nan
            )
    },

    {
        "Quantity":
            "First budget with non-positive LPNS capture advantage",

        "Value":
            (
                negative_capture[
                    "Review_Percent"
                ].min()

                if len(
                    negative_capture
                ) > 0

                else np.nan
            )
    },

    {
        "Quantity":
            "Maximum additional errors found",

        "Value":
            ADVANTAGE_27[
                "Additional_Errors_Found"
            ].max()
    },

    {
        "Quantity":
            "Budget at maximum additional errors",

        "Value":
            ADVANTAGE_27.loc[
                ADVANTAGE_27[
                    "Additional_Errors_Found"
                ].idxmax(),
                "Review_Percent"
            ]
    },

])


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 27 — CROSSOVER SUMMARY"
)

print(
    "=" * 150
)


display(
    CROSSOVER_SUMMARY_27
)


# ============================================================
# 27.9 AREA UNDER ERROR-CAPTURE CURVE
#
# Descriptive summary over the fixed 1%-25% range.
# Higher = better.
# ============================================================

curve_area_rows_27 = []


for method, df in [

    (
        "Confidence",
        CURVE_CONF_27
    ),

    (
        "Confidence+Disagreement",
        CURVE_DIS_27
    ),

    (
        "LPNS",
        CURVE_LPNS_27
    ),
]:

    area = np.trapz(

        df[
            "Error_Capture"
        ].to_numpy(),

        df[
            "Review_Rate"
        ].to_numpy(),
    )


    curve_area_rows_27.append({

        "Method":
            method,

        "Capture_AUC_1to25pct":
            area,
    })


CAPTURE_AREA_27 = pd.DataFrame(
    curve_area_rows_27
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 27 — ERROR-CAPTURE AREA, 1%-25%"
)

print(
    "=" * 150
)


display(
    CAPTURE_AREA_27
)


# ============================================================
# 27.10 FIGURE-READY TABLE
# ============================================================

FIGURE_DATA_27 = (

    SELECTIVE_CURVES_27[
        [
            "Method",
            "Review_Percent",
            "Error_Capture",
            "Errors_Per_100_Reviews",
            "Residual_Error_Rate",
        ]
    ]
    .copy()
)


# ============================================================
# 27.11 SAVE
# ============================================================

save_csv(

    SELECTIVE_CURVES_27,

    TAB_DIR
    /
    "module27_dense_selective_curves.csv"
)


save_csv(

    ADVANTAGE_27,

    TAB_DIR
    /
    "module27_lpns_advantage_curve.csv"
)


save_csv(

    KEY_POINTS_27,

    TAB_DIR
    /
    "module27_key_review_budgets.csv"
)


save_csv(

    CROSSOVER_SUMMARY_27,

    TAB_DIR
    /
    "module27_crossover_summary.csv"
)


save_csv(

    CAPTURE_AREA_27,

    TAB_DIR
    /
    "module27_capture_area_1to25pct.csv"
)


save_csv(

    FIGURE_DATA_27,

    TAB_DIR
    /
    "module27_selective_review_figure_data.csv"
)


save_json(

    {
        "module":
            "Dense selective-review characterization",

        "review_budget_min":
            0.01,

        "review_budget_max":
            0.25,

        "review_budget_step":
            0.01,

        "methods": [
            "Confidence",
            "Confidence+Disagreement",
            "LPNS",
        ],

        "primary_operational_region":
            "5%-10% review",

        "interpretation":
            (
                "Selective review prioritization only; "
                "not automated correction."
            ),

        "model_changes":
            False,

        "threshold_changes":
            False,
    },

    LOG_DIR
    /
    "module27_manifest.json"
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE 27 COMPLETE"
)

print(
    "=" * 150
)

MODULE 27 — DENSE SELECTIVE-REVIEW CURVES


,Method,Review_Rate,Review_Percent,Review_N,Errors_Found,Error_Capture,Errors_Per_100_Reviews,Remaining_Errors,Residual_Error_Rate
0,Confidence,0.01,1.0,3198,1850,0.041897,57.848655,42306,0.133627
1,Confidence,0.02,2.0,6396,3707,0.083952,57.958099,40449,0.129066
2,Confidence,0.03,3.0,9594,5624,0.127367,58.619971,38532,0.124216
3,Confidence,0.04,4.0,12792,7643,0.173091,59.748280,36513,0.118934
4,Confidence,0.05,5.0,15990,9734,0.220446,60.875547,34422,0.113303
...,...,...,...,...,...,...,...,...,...
70,LPNS,0.21,21.0,67157,34668,0.785125,51.622318,9488,0.037556
71,LPNS,0.22,22.0,70355,35145,0.795928,49.953806,9011,0.036125
72,LPNS,0.23,23.0,73553,35606,0.806368,48.408631,8550,0.034722
73,LPNS,0.24,24.0,76751,36020,0.815744,46.930985,8136,0.033475



MODULE 27 — LPNS ADVANTAGE VS CONFIDENCE+DISAGREEMENT


,Review_Rate,Review_Percent,Delta_Error_Capture,Delta_Errors_Per_100,Additional_Errors_Found,Delta_Residual_Error_Rate
0,0.01,1.0,0.004439,6.128831,196,-0.000619
1,0.02,2.0,0.007383,5.096936,326,-0.001040
2,0.03,3.0,0.009512,4.377736,420,-0.001354
3,0.04,4.0,0.011346,3.916510,501,-0.001632
4,0.05,5.0,0.013203,3.646029,583,-0.001919
5,0.06,6.0,0.013769,3.168647,608,-0.002023
6,0.07,7.0,0.013203,2.604306,583,-0.001960
7,0.08,8.0,0.010531,1.817542,465,-0.001580
8,0.09,9.0,0.009376,1.438399,414,-0.001423
9,0.10,10.0,0.004144,0.572251,183,-0.000636



MODULE 27 — KEY REVIEW BUDGETS


...


MODULE 27 — CROSSOVER SUMMARY


,Quantity,Value
0,Highest budget with positive LPNS capture adva...,18.0
1,First budget with non-positive LPNS capture ad...,11.0
2,Maximum additional errors found,608.0
3,Budget at maximum additional errors,6.0



MODULE 27 — ERROR-CAPTURE AREA, 1%-25%


,Method,Capture_AUC_1to25pct
0,Confidence,0.130434
1,Confidence+Disagreement,0.130653
2,LPNS,0.130783



MODULE 27 COMPLETE


In [86]:
# ============================================================
# MODULE 28 — LPNS COMPUTATIONAL COST / OVERHEAD AUDIT
#
# PURPOSE
# ------------------------------------------------------------
# Quantify computational overhead of the frozen LPNS pipeline.
#
# Reports:
#   - reference-bank memory
#   - query time
#   - LPNS feature-construction time
#   - reliability-head inference time
#   - total processing time
#   - approximate milliseconds per record
#
# IMPORTANT
# ------------------------------------------------------------
# These timings characterize the current experimental
# environment only.
#
# They are NOT production latency claims.
#
# No model fitting.
# No tuning.
# No architecture changes.
# ============================================================

import numpy as np
import pandas as pd
import time


# ============================================================
# 28.1 REQUIRED OBJECTS
# ============================================================

required_28 = [

    "X20_DISTANCE_24B",
    "P20_EXPERT_24B",

    "X_reference20d",
    "P_reference20d",

    "nbr20d",

    "lpns_features20d",
    "impute_lpns20d",

    "DIS20",

    "scale_full",
    "REL_LPNS20D",

    "LOCKED_COMMON12_THRESHOLD",

    "REF_RESULTS_24B",

    "save_csv",
    "save_json",

    "TAB_DIR",
    "LOG_DIR",
]


missing = [
    x
    for x in required_28
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing Module 28 objects: "
        + ", ".join(missing)
    )


# ============================================================
# 28.2 BASIC DIMENSIONS
# ============================================================

N28 = len(
    X20_DISTANCE_24B
)


K28 = 20


print(
    "External records:",
    N28
)

print(
    "Frozen reference bank:",
    len(
        X_reference20d
    )
)

print(
    "Frozen k:",
    K28
)


# ============================================================
# 28.3 MEMORY HELPER
# ============================================================

def mb28(
    arr
):

    return (
        np.asarray(
            arr
        ).nbytes
        /
        (
            1024 ** 2
        )
    )


MEMORY_28 = pd.DataFrame([

    {
        "Object":
            "Reference profiles",

        "Shape":
            str(
                X_reference20d.shape
            ),

        "DType":
            str(
                X_reference20d.dtype
            ),

        "Memory_MB":
            mb28(
                X_reference20d
            ),
    },

    {
        "Object":
            "Reference expert predictions",

        "Shape":
            str(
                P_reference20d.shape
            ),

        "DType":
            str(
                P_reference20d.dtype
            ),

        "Memory_MB":
            mb28(
                P_reference20d
            ),
    },

    {
        "Object":
            "External distance matrix",

        "Shape":
            str(
                X20_DISTANCE_24B.shape
            ),

        "DType":
            str(
                X20_DISTANCE_24B.dtype
            ),

        "Memory_MB":
            mb28(
                X20_DISTANCE_24B
            ),
    },

    {
        "Object":
            "External expert probabilities",

        "Shape":
            str(
                P20_EXPERT_24B.shape
            ),

        "DType":
            str(
                P20_EXPERT_24B.dtype
            ),

        "Memory_MB":
            mb28(
                P20_EXPERT_24B
            ),
    },
])


MEMORY_28[
    "Memory_MB"
] = MEMORY_28[
    "Memory_MB"
].astype(
    float
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — MEMORY FOOTPRINT"
)

print(
    "=" * 140
)


display(
    MEMORY_28
)


REFERENCE_MEMORY_28 = (

    mb28(
        X_reference20d
    )

    +

    mb28(
        P_reference20d
    )
)


print(
    "\nCombined frozen reference-bank memory:"
)

print(
    f"{REFERENCE_MEMORY_28:.3f} MB"
)


# ============================================================
# 28.4 WARM-UP
#
# Run on small subset to reduce first-call overhead.
# ============================================================

warm_n28 = min(
    1000,
    N28
)


_ = nbr20d.kneighbors(

    X20_DISTANCE_24B[
        :warm_n28
    ],

    n_neighbors=K28,

    return_distance=True,
)


# ============================================================
# 28.5 REPEATED TIMING
#
# Three repetitions.
# Median used for summary.
# ============================================================

REPS_28 = 3


timing_rows_28 = []


for rep in range(
    REPS_28
):

    # --------------------------------------------------------
    # A. Neighbor query
    # --------------------------------------------------------

    t0 = time.perf_counter()


    dist28, idx28 = (
        nbr20d.kneighbors(

            X20_DISTANCE_24B,

            n_neighbors=K28,

            return_distance=True,
        )
    )


    t1 = time.perf_counter()


    # --------------------------------------------------------
    # B. LPNS feature construction
    # --------------------------------------------------------

    (
        lpns28,
        valid28,
        ncount28,

    ) = lpns_features20d(

        P20_EXPERT_24B,

        P_reference20d,

        idx28,
        dist28,

        LOCKED_COMMON12_THRESHOLD
    )


    lpns28_imp = (
        impute_lpns20d(
            lpns28
        )
    )


    t2 = time.perf_counter()


    # --------------------------------------------------------
    # C. Reliability-head inference
    # --------------------------------------------------------

    F28 = np.column_stack([

        np.asarray(
            DIS20,
            dtype=float
        ),

        lpns28_imp,
    ])


    F28_s = (
        scale_full
        .transform(
            F28
        )
    )


    risk28 = (

        REL_LPNS20D
        .predict_proba(
            F28_s
        )[:, 1]
    )


    t3 = time.perf_counter()


    query_s = (
        t1 - t0
    )


    feature_s = (
        t2 - t1
    )


    inference_s = (
        t3 - t2
    )


    total_s = (
        t3 - t0
    )


    timing_rows_28.append({

        "Repeat":
            rep + 1,

        "Neighbor_Query_Seconds":
            query_s,

        "LPNS_Feature_Seconds":
            feature_s,

        "Reliability_Inference_Seconds":
            inference_s,

        "Total_Seconds":
            total_s,

        "Total_ms_per_Record":
            (
                1000
                *
                total_s
                /
                N28
            ),

        "Query_ms_per_Record":
            (
                1000
                *
                query_s
                /
                N28
            ),

        "LPNS_Feature_ms_per_Record":
            (
                1000
                *
                feature_s
                /
                N28
            ),

        "Inference_ms_per_Record":
            (
                1000
                *
                inference_s
                /
                N28
            ),

        "Coverage":
            float(
                np.mean(
                    valid28
                )
            ),

        "Mean_Neighbors":
            float(
                np.mean(
                    ncount28
                )
            ),
    })


TIMING_28 = pd.DataFrame(
    timing_rows_28
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — REPEATED LPNS TIMING"
)

print(
    "=" * 140
)


display(
    TIMING_28
)


# ============================================================
# 28.6 MEDIAN TIMING SUMMARY
# ============================================================

timing_metrics_28 = [

    "Neighbor_Query_Seconds",
    "LPNS_Feature_Seconds",
    "Reliability_Inference_Seconds",
    "Total_Seconds",

    "Total_ms_per_Record",
    "Query_ms_per_Record",
    "LPNS_Feature_ms_per_Record",
    "Inference_ms_per_Record",
]


summary_rows_28 = []


for metric in timing_metrics_28:

    vals = TIMING_28[
        metric
    ].to_numpy(
        dtype=float
    )


    summary_rows_28.append({

        "Metric":
            metric,

        "Median":
            np.median(
                vals
            ),

        "Minimum":
            np.min(
                vals
            ),

        "Maximum":
            np.max(
                vals
            ),
    })


TIMING_SUMMARY_28 = pd.DataFrame(
    summary_rows_28
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — TIMING SUMMARY"
)

print(
    "=" * 140
)


display(
    TIMING_SUMMARY_28
)


# ============================================================
# 28.7 THROUGHPUT
# ============================================================

median_total28 = float(

    TIMING_28[
        "Total_Seconds"
    ].median()
)


THROUGHPUT_28 = pd.DataFrame([

    {
        "N_Records":
            N28,

        "Median_Total_Seconds":
            median_total28,

        "Records_Per_Second":
            (
                N28
                /
                median_total28
            ),

        "Milliseconds_Per_Record":
            (
                1000
                *
                median_total28
                /
                N28
            ),

        "Reference_Bank_N":
            len(
                X_reference20d
            ),

        "K":
            K28,

        "Reference_Memory_MB":
            REFERENCE_MEMORY_28,
    }
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — THROUGHPUT SUMMARY"
)

print(
    "=" * 140
)


display(
    THROUGHPUT_28
)


# ============================================================
# 28.8 REFERENCE-SIZE COMPUTATIONAL SCALING
#
# Reuse Module 24B timing values.
# Do not rerun.
# ============================================================

REFERENCE_SCALING_28 = (

    REF_RESULTS_24B[
        [
            "Reference_N",
            "Elapsed_Seconds",
            "Error_PR_AUC",
            "Review5_Errors_Per_100",
            "Coverage_2020",
        ]
    ]
    .copy()
)


REFERENCE_SCALING_28[
    "Relative_Time_vs_10k"
] = (

    REFERENCE_SCALING_28[
        "Elapsed_Seconds"
    ]

    /
    REFERENCE_SCALING_28[
        "Elapsed_Seconds"
    ].iloc[0]
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — REFERENCE-SIZE COMPUTATIONAL SCALING"
)

print(
    "=" * 140
)


display(
    REFERENCE_SCALING_28
)


# ============================================================
# 28.9 STORAGE ESTIMATE ACROSS REFERENCE SIZES
#
# Simple array-memory estimate based on actual bytes/profile.
# ============================================================

bytes_per_profile28 = (

    X_reference20d.nbytes
    /
    len(
        X_reference20d
    )
)


bytes_per_predrow28 = (

    P_reference20d.nbytes
    /
    len(
        P_reference20d
    )
)


storage_rows_28 = []


for n_ref in [

    10000,
    20000,
    40000,
    80000,

]:

    total_bytes = (

        n_ref
        *
        (
            bytes_per_profile28
            +
            bytes_per_predrow28
        )
    )


    storage_rows_28.append({

        "Reference_N":
            n_ref,

        "Estimated_Array_Memory_MB":
            total_bytes
            /
            (
                1024 ** 2
            ),
    })


STORAGE_SCALING_28 = pd.DataFrame(
    storage_rows_28
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — REFERENCE STORAGE SCALING"
)

print(
    "=" * 140
)


display(
    STORAGE_SCALING_28
)


# ============================================================
# 28.10 MANUSCRIPT-FACING COST SUMMARY
# ============================================================

query_med28 = float(

    TIMING_28[
        "Neighbor_Query_Seconds"
    ].median()
)


feature_med28 = float(

    TIMING_28[
        "LPNS_Feature_Seconds"
    ].median()
)


infer_med28 = float(

    TIMING_28[
        "Reliability_Inference_Seconds"
    ].median()
)


COST_SUMMARY_28 = pd.DataFrame([

    {
        "Component":
            "Neighbor retrieval",

        "Median_Seconds":
            query_med28,

        "Share_of_Total":
            query_med28
            /
            median_total28,
    },

    {
        "Component":
            "LPNS feature computation",

        "Median_Seconds":
            feature_med28,

        "Share_of_Total":
            feature_med28
            /
            median_total28,
    },

    {
        "Component":
            "Reliability-head inference",

        "Median_Seconds":
            infer_med28,

        "Share_of_Total":
            infer_med28
            /
            median_total28,
    },
])


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 — COST BREAKDOWN"
)

print(
    "=" * 140
)


display(
    COST_SUMMARY_28
)


# ============================================================
# 28.11 SAVE
# ============================================================

save_csv(

    MEMORY_28,

    TAB_DIR
    /
    "module28_memory_footprint.csv"
)


save_csv(

    TIMING_28,

    TAB_DIR
    /
    "module28_repeated_timing.csv"
)


save_csv(

    TIMING_SUMMARY_28,

    TAB_DIR
    /
    "module28_timing_summary.csv"
)


save_csv(

    THROUGHPUT_28,

    TAB_DIR
    /
    "module28_throughput_summary.csv"
)


save_csv(

    REFERENCE_SCALING_28,

    TAB_DIR
    /
    "module28_reference_scaling.csv"
)


save_csv(

    STORAGE_SCALING_28,

    TAB_DIR
    /
    "module28_storage_scaling.csv"
)


save_csv(

    COST_SUMMARY_28,

    TAB_DIR
    /
    "module28_cost_breakdown.csv"
)


save_json(

    {
        "module":
            "LPNS computational-cost audit",

        "timing_repetitions":
            REPS_28,

        "reported_summary":
            "median",

        "external_records":
            N28,

        "reference_bank":
            len(
                X_reference20d
            ),

        "k":
            K28,

        "interpretation":
            (
                "Empirical computational overhead in the "
                "study environment; not production latency."
            ),

        "model_changes":
            False,
    },

    LOG_DIR
    /
    "module28_manifest.json"
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE 28 COMPLETE"
)

print(
    "=" * 140
)

External records: 319795
Frozen reference bank: 80000
Frozen k: 20

MODULE 28 — MEMORY FOOTPRINT


,Object,Shape,DType,Memory_MB
0,Reference profiles,"(80000, 12)",float32,3.662109
1,Reference expert predictions,"(80000, 3)",float64,1.831055
2,External distance matrix,"(319795, 12)",float32,14.639053
3,External expert probabilities,"(319795, 3)",float64,7.319527



Combined frozen reference-bank memory:
5.493 MB

MODULE 28 — REPEATED LPNS TIMING


,Repeat,Neighbor_Query_Seconds,LPNS_Feature_Seconds,Reliability_Inference_Seconds,Total_Seconds,Total_ms_per_Record,Query_ms_per_Record,LPNS_Feature_ms_per_Record,Inference_ms_per_Record,Coverage,Mean_Neighbors
0,1,30.265984,1.554037,0.116406,31.936427,0.099865,0.094642,0.004859,0.000364,0.999991,19.999384
1,2,30.043332,1.536568,0.118034,31.697934,0.099120,0.093946,0.004805,0.000369,0.999991,19.999384
2,3,29.918795,1.577757,0.118787,31.615339,0.098861,0.093556,0.004934,0.000371,0.999991,19.999384



MODULE 28 — TIMING SUMMARY


...


MODULE 28 — REFERENCE STORAGE SCALING


,Reference_N,Estimated_Array_Memory_MB
0,10000,0.686646
1,20000,1.373291
2,40000,2.746582
3,80000,5.493164



MODULE 28 — COST BREAKDOWN


,Component,Median_Seconds,Share_of_Total
0,Neighbor retrieval,30.043332,0.947801
1,LPNS feature computation,1.554037,0.049026
2,Reliability-head inference,0.118034,0.003724



MODULE 28 COMPLETE


In [17]:
# ============================================================
# MODULE 12 — CLINICAL-CONTEXT RESIDUAL CORRECTION
# ============================================================
#
# Backbone:
#   Core4 calibrated probability mean
#
# Residual specialist:
#   clinical/context features
#   + expert predictions
#   + disagreement signals
#
# Strict outer-fold CV.
# No HOLDOUT.
#
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    average_precision_score
)


# ============================================================
# 12.1 Core architecture
# ============================================================

CORE_MODELS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

SPECIALIST_MODELS = [
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
]

ALL_META_MODELS = (
    CORE_MODELS
    + SPECIALIST_MODELS
)


# ============================================================
# 12.2 Clinical/context feature set
# ============================================================

CLINICAL_CONTEXT_FEATURES = [

    # cardiometabolic
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Diabetes",

    # prior vascular disease
    "Stroke",

    # global health / functional status
    "GenHlth",
    "PhysHlth",
    "MentHlth",
    "DiffWalk",

    # demographic
    "Age",
    "Sex",

    # behavior
    "Smoker",
    "PhysActivity",

    # access / socioeconomic
    "AnyHealthcare",
    "NoDocbcCost",
    "Education",
    "Income",
]

print(
    "Clinical/context features:",
    len(CLINICAL_CONTEXT_FEATURES)
)


# ============================================================
# 12.3 Meta + clinical feature builder
# ============================================================

def build_context_residual_features(
    X_original,
    calibrated_dict
):

    out = pd.DataFrame(
        index=np.arange(
            len(X_original)
        )
    )

    # --------------------------------------------------------
    # Original context features
    # --------------------------------------------------------

    for c in CLINICAL_CONTEXT_FEATURES:

        out[
            f"x_{c}"
        ] = (
            X_original[c]
            .to_numpy(
                dtype=float
            )
        )

    # --------------------------------------------------------
    # Expert probabilities
    # --------------------------------------------------------

    for model_name in ALL_META_MODELS:

        out[
            f"p_{model_name}"
        ] = (
            calibrated_dict[
                model_name
            ]
        )

    # --------------------------------------------------------
    # Core consensus
    # --------------------------------------------------------

    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in CORE_MODELS
    ])

    core_mean = (
        core_matrix.mean(
            axis=1
        )
    )

    core_std = (
        core_matrix.std(
            axis=1
        )
    )

    core_min = (
        core_matrix.min(
            axis=1
        )
    )

    core_max = (
        core_matrix.max(
            axis=1
        )
    )

    out[
        "core_mean"
    ] = core_mean

    out[
        "core_logit"
    ] = safe_logit(
        core_mean
    )

    out[
        "core_std"
    ] = core_std

    out[
        "core_range"
    ] = (
        core_max
        - core_min
    )

    out[
        "core_entropy"
    ] = binary_entropy(
        core_mean
    )

    # --------------------------------------------------------
    # Expert residual signals
    # --------------------------------------------------------

    for m in ALL_META_MODELS:

        delta = (
            calibrated_dict[m]
            - core_mean
        )

        out[
            f"delta_{m}"
        ] = delta

        out[
            f"abs_delta_{m}"
        ] = np.abs(
            delta
        )

    return out


# ============================================================
# 12.4 Lambda candidates
# ============================================================
#
# Correction strength.
#
# 0 = no correction
# 1 = full residual model correction
#
# Conservative values are intentional.
# ============================================================

LAMBDA_GRID = [
    0.00,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]


# ============================================================
# 12.5 Buffers
# ============================================================

y_all = y_dev.to_numpy(
    dtype=int
)

RESIDUAL_PRED = np.zeros(
    len(y_all),
    dtype=float
)

CORE12_PRED = np.zeros(
    len(y_all),
    dtype=float
)

CORRECTION_MAG = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []
lambda_rows = []


# ============================================================
# 12.6 Strict outer-fold loop
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 95
    )

    print(
        f"MODULE 12 FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 95
    )

    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]

    y_train = (
        y_all[
            train_idx
        ]
    )

    y_valid = (
        y_all[
            valid_idx
        ]
    )


    # ========================================================
    # Nested calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}

    for model_name in (
        ALL_META_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va


    # ========================================================
    # Core probabilities
    # ========================================================

    core_train = np.mean(
        np.column_stack([
            train_cal[m]
            for m in CORE_MODELS
        ]),
        axis=1
    )

    core_valid = np.mean(
        np.column_stack([
            valid_cal[m]
            for m in CORE_MODELS
        ]),
        axis=1
    )

    CORE12_PRED[
        valid_idx
    ] = core_valid


    # ========================================================
    # Build clinical-context residual features
    # ========================================================

    Xtr_meta = (
        build_context_residual_features(
            X_dev.iloc[
                train_idx
            ].reset_index(
                drop=True
            ),
            train_cal
        )
    )

    Xva_meta = (
        build_context_residual_features(
            X_dev.iloc[
                valid_idx
            ].reset_index(
                drop=True
            ),
            valid_cal
        )
    )


    # ========================================================
    # Residual target
    # ========================================================
    #
    # Positive residual:
    #   true outcome > core probability
    #
    # Negative residual:
    #   true outcome < core probability
    #
    # ========================================================

    residual_target = (
        y_train
        - core_train
    )


    # ========================================================
    # Conservative residual model
    # ========================================================

    residual_model = (
        HistGradientBoostingRegressor(

            learning_rate=0.035,

            max_depth=2,

            max_iter=120,

            min_samples_leaf=200,

            l2_regularization=10.0,

            loss="squared_error",

            random_state=
                SEED + outer_fold,
        )
    )

    residual_model.fit(
        Xtr_meta,
        residual_target
    )


    # ========================================================
    # Predict residuals
    # ========================================================

    delta_train = (
        residual_model.predict(
            Xtr_meta
        )
    )

    delta_valid = (
        residual_model.predict(
            Xva_meta
        )
    )


    # ========================================================
    # Bound corrections
    # ========================================================
    #
    # Prevent aggressive probability modification.
    # ========================================================

    delta_train = np.clip(
        delta_train,
        -0.15,
        0.15
    )

    delta_valid = np.clip(
        delta_valid,
        -0.15,
        0.15
    )


    # ========================================================
    # Lambda selection — TRAIN ONLY
    # ========================================================

    lambda_candidates = []

    for lam in LAMBDA_GRID:

        p_train_candidate = (
            clip_probs(
                core_train
                + lam
                * delta_train
            )
        )

        pr = (
            average_precision_score(
                y_train,
                p_train_candidate
            )
        )

        pm = probability_metrics(
            y_train,
            p_train_candidate
        )

        lambda_candidates.append({

            "lambda":
                lam,

            "PR_AUC":
                pr,

            "Brier":
                pm["Brier"],

            "NLL":
                pm["NLL"],
        })


    lambda_df = (
        pd.DataFrame(
            lambda_candidates
        )
        .sort_values(
            [
                "PR_AUC",
                "Brier"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    best_lambda = float(
        lambda_df.iloc[0][
            "lambda"
        ]
    )


    # ========================================================
    # Frozen validation prediction
    # ========================================================

    p_final = clip_probs(
        core_valid
        + best_lambda
        * delta_valid
    )

    RESIDUAL_PRED[
        valid_idx
    ] = p_final

    CORRECTION_MAG[
        valid_idx
    ] = (
        best_lambda
        * delta_valid
    )


    # ========================================================
    # Fold metrics
    # ========================================================

    core_pm = probability_metrics(
        y_valid,
        core_valid
    )

    final_pm = probability_metrics(
        y_valid,
        p_final
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Selected_Lambda":
            best_lambda,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "Residual_PR":
            final_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            final_pm[
                "PR_AUC"
            ]
            - core_pm[
                "PR_AUC"
            ],

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "Residual_ROC":
            final_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "Residual_Brier":
            final_pm[
                "Brier"
            ],

        "Mean_Abs_Correction":
            np.mean(
                np.abs(
                    best_lambda
                    * delta_valid
                )
            ),

        "Max_Abs_Correction":
            np.max(
                np.abs(
                    best_lambda
                    * delta_valid
                )
            ),
    })


    for _, row in (
        lambda_df.iterrows()
    ):

        lambda_rows.append({

            "Fold":
                outer_fold,

            **row.to_dict()
        })


    print(
        f"Lambda       : "
        f"{best_lambda:.2f}"
    )

    print(
        f"Core PR      : "
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        f"Residual PR  : "
        f"{final_pm['PR_AUC']:.6f}"
    )

    print(
        f"Delta        : "
        f"{final_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )

    print(
        f"Mean |corr|  : "
        f"{np.mean(np.abs(best_lambda * delta_valid)):.6f}"
    )


# ============================================================
# 12.7 Global evaluation
# ============================================================

core_threshold, _, _ = (
    select_best_threshold(
        y_all,
        CORE12_PRED,
        objective="F1"
    )
)

res_threshold, _, _ = (
    select_best_threshold(
        y_all,
        RESIDUAL_PRED,
        objective="F1"
    )
)

core_metrics = (
    evaluate_binary_predictions(
        y_all,
        CORE12_PRED,
        threshold=
            core_threshold
    )
)

residual_metrics = (
    evaluate_binary_predictions(
        y_all,
        RESIDUAL_PRED,
        threshold=
            res_threshold
    )
)


module12_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "ClinicalContextResidual",

        **residual_metrics
    },
])


module12_results[
    "Delta_PR_vs_Core"
] = (
    module12_results[
        "PR_AUC"
    ]
    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 12 — GLOBAL RESULTS"
)

print(
    "=" * 120
)

display(
    module12_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 12.8 Fold results
# ============================================================

module12_fold_table = (
    pd.DataFrame(
        fold_rows
    )
)

print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 12 — FOLD RESULTS"
)

print(
    "=" * 120
)

display(
    module12_fold_table
)


# ============================================================
# 12.9 Lambda stability
# ============================================================

lambda_summary = (
    module12_fold_table[
        "Selected_Lambda"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "Lambda"
    )
    .reset_index(
        name="Selected_Folds"
    )
)

print(
    "\n"
    + "=" * 120
)

print(
    "LAMBDA SELECTION STABILITY"
)

print(
    "=" * 120
)

display(
    lambda_summary
)


# ============================================================
# 12.10 Correction analysis
# ============================================================

correction_summary = pd.DataFrame({

    "Metric": [
        "Mean correction",
        "Mean absolute correction",
        "Median absolute correction",
        "95th percentile abs correction",
        "Max absolute correction",
        "Fraction abs correction > 0.01",
        "Fraction abs correction > 0.02",
        "Fraction abs correction > 0.05",
    ],

    "Value": [

        float(
            CORRECTION_MAG.mean()
        ),

        float(
            np.abs(
                CORRECTION_MAG
            ).mean()
        ),

        float(
            np.median(
                np.abs(
                    CORRECTION_MAG
                )
            )
        ),

        float(
            np.quantile(
                np.abs(
                    CORRECTION_MAG
                ),
                0.95
            )
        ),

        float(
            np.abs(
                CORRECTION_MAG
            ).max()
        ),

        float(
            np.mean(
                np.abs(
                    CORRECTION_MAG
                ) > 0.01
            )
        ),

        float(
            np.mean(
                np.abs(
                    CORRECTION_MAG
                ) > 0.02
            )
        ),

        float(
            np.mean(
                np.abs(
                    CORRECTION_MAG
                ) > 0.05
            )
        ),
    ]
})


print(
    "\n"
    + "=" * 120
)

print(
    "RESIDUAL CORRECTION MAGNITUDE"
)

print(
    "=" * 120
)

display(
    correction_summary
)


# ============================================================
# 12.11 Save
# ============================================================

save_csv(
    module12_results,
    TAB_DIR
    / "module12_clinical_context_residual_results.csv"
)

save_csv(
    module12_fold_table,
    TAB_DIR
    / "module12_clinical_context_residual_folds.csv"
)

save_csv(
    pd.DataFrame(
        lambda_rows
    ),
    TAB_DIR
    / "module12_lambda_search.csv"
)

save_csv(
    correction_summary,
    TAB_DIR
    / "module12_correction_summary.csv"
)


module12_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        CORE12_PRED,

    "ClinicalContextResidual":
        RESIDUAL_PRED,

    "ResidualCorrection":
        CORRECTION_MAG,
})

save_csv(
    module12_pred_df,
    PRED_DIR
    / "module12_residual_oof_predictions.csv"
)


print(
    "\nMODULE 12 COMPLETE."
)

Clinical/context features: 18

MODULE 12 FOLD 1/5
Lambda       : 0.50
Core PR      : 0.380070
Residual PR  : 0.379975
Delta        : -0.000095
Mean |corr|  : 0.000423

MODULE 12 FOLD 2/5
Lambda       : 0.50
Core PR      : 0.382772
Residual PR  : 0.382961
Delta        : +0.000190
Mean |corr|  : 0.000701

MODULE 12 FOLD 3/5
Lambda       : 0.50
Core PR      : 0.378467
Residual PR  : 0.378545
Delta        : +0.000078
Mean |corr|  : 0.001102

MODULE 12 FOLD 4/5
Lambda       : 0.50
Core PR      : 0.372833
Residual PR  : 0.372827
Delta        : -0.000006
Mean |corr|  : 0.000658

MODULE 12 FOLD 5/5
Lambda       : 0.50
Core PR      : 0.390072
Residual PR  : 0.390100
Delta        : +0.000028
Mean |corr|  : 0.001215

MODULE 12 — GLOBAL RESULTS


,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,ClinicalContextResidual,0.380350,0.000023,0.850614,0.069699,0.235545,0.001809,0.350476,0.526263,0.420747,0.356121,0.22



MODULE 12 — FOLD RESULTS


,Fold,Selected_Lambda,Core_PR,Residual_PR,Delta_PR,Core_ROC,Residual_ROC,Core_Brier,Residual_Brier,Mean_Abs_Correction,Max_Abs_Correction
0,0,0.5,0.380070,0.379975,-0.000095,0.852660,0.852678,0.069637,0.069635,0.000423,0.006994
1,1,0.5,0.382772,0.382961,0.000190,0.850766,0.850822,0.069684,0.069676,0.000701,0.014398
2,2,0.5,0.378467,0.378545,0.000078,0.849030,0.849083,0.069827,0.069825,0.001102,0.022798
3,3,0.5,0.372833,0.372827,-0.000006,0.849083,0.849115,0.070081,0.070072,0.000658,0.013094
4,4,0.5,0.390072,0.390100,0.000028,0.851760,0.851863,0.069301,0.069287,0.001215,0.027594



LAMBDA SELECTION STABILITY


,Lambda,Selected_Folds
0,0.5,5



RESIDUAL CORRECTION MAGNITUDE


,Metric,Value
0,Mean correction,-0.000086
1,Mean absolute correction,0.000820
2,Median absolute correction,0.000281
3,95th percentile abs correction,0.002973
4,Max absolute correction,0.027594
5,Fraction abs correction > 0.01,0.002523
6,Fraction abs correction > 0.02,0.000054
7,Fraction abs correction > 0.05,0.000000



MODULE 12 COMPLETE.


In [18]:
# ============================================================
# MODULE 13A — TRUST-AWARE HARD-POSITIVE RESIDUAL RE-RANKER
# ============================================================
#
# THRR:
# Trust-Aware Hard-Positive Residual Re-Ranker
#
# Backbone:
#   calibrated Core4 consensus
#
# Learns:
#   a bounded correction to the core logit
#
# Training:
#   BCE + pairwise ranking loss + correction regularization
#
# Emphasis:
#   hard positives + hard negatives
#
# DEVELOPMENT SCREENING ONLY
# HOLDOUT untouched
# ============================================================

import copy
import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score


# ============================================================
# 13A.1 Configuration
# ============================================================

THRR_CORE_MODELS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

THRR_SPECIALISTS = [
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
]

THRR_MODELS = (
    THRR_CORE_MODELS
    + THRR_SPECIALISTS
)

THRR_CONTEXT_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]

THRR_CONFIG = {

    "hidden":
        (64, 32),

    "dropout":
        0.15,

    "batch_size":
        1024,

    "epochs":
        20,

    "lr":
        5e-4,

    "weight_decay":
        1e-4,

    # ranking-loss strength
    "alpha_rank":
        0.50,

    # residual magnitude penalty
    "beta_residual":
        0.05,

    # maximum logit correction
    "max_logit_correction":
        0.75,

    # percentage used for hard mining
    "hard_fraction":
        0.30,

    # additional hard-case weight
    "hard_weight":
        2.0,

    "patience":
        4,
}


# ============================================================
# 13A.2 Helper functions
# ============================================================

def torch_logit(p):

    p = torch.clamp(
        p,
        1e-6,
        1.0 - 1e-6
    )

    return torch.log(
        p / (1.0 - p)
    )


def np_sigmoid(z):

    return 1.0 / (
        1.0 + np.exp(-z)
    )


def percentile_rank(x):

    order = np.argsort(
        np.argsort(x)
    )

    return (
        order.astype(float)
        / max(
            len(x) - 1,
            1
        )
    )


# ============================================================
# 13A.3 Feature builder
# ============================================================

def build_thrr_features(
    X_original,
    calibrated_dict
):

    out = pd.DataFrame(
        index=np.arange(
            len(X_original)
        )
    )

    # --------------------------------------------------------
    # Clinical/context features
    # --------------------------------------------------------

    for c in THRR_CONTEXT_FEATURES:

        out[
            f"x_{c}"
        ] = (
            X_original[c]
            .to_numpy(
                dtype=np.float32
            )
        )

    # --------------------------------------------------------
    # Expert probabilities
    # --------------------------------------------------------

    for m in THRR_MODELS:

        p = np.asarray(
            calibrated_dict[m],
            dtype=float
        )

        out[
            f"p_{m}"
        ] = p

        out[
            f"logit_{m}"
        ] = safe_logit(
            p
        )

        out[
            f"rank_{m}"
        ] = percentile_rank(
            p
        )

    # --------------------------------------------------------
    # Core consensus
    # --------------------------------------------------------

    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in THRR_CORE_MODELS
    ])

    core_mean = (
        core_matrix.mean(
            axis=1
        )
    )

    core_std = (
        core_matrix.std(
            axis=1
        )
    )

    core_min = (
        core_matrix.min(
            axis=1
        )
    )

    core_max = (
        core_matrix.max(
            axis=1
        )
    )

    out["core_mean"] = (
        core_mean
    )

    out["core_logit"] = (
        safe_logit(
            core_mean
        )
    )

    out["core_rank"] = (
        percentile_rank(
            core_mean
        )
    )

    out["core_std"] = (
        core_std
    )

    out["core_range"] = (
        core_max
        - core_min
    )

    out["core_entropy"] = (
        binary_entropy(
            core_mean
        )
    )

    # --------------------------------------------------------
    # Consensus disagreement features
    # --------------------------------------------------------

    for m in THRR_MODELS:

        p = np.asarray(
            calibrated_dict[m]
        )

        out[
            f"delta_{m}"
        ] = (
            p - core_mean
        )

        out[
            f"abs_delta_{m}"
        ] = np.abs(
            p - core_mean
        )

    # --------------------------------------------------------
    # Pairwise core differences
    # --------------------------------------------------------

    for i in range(
        len(THRR_CORE_MODELS)
    ):

        for j in range(
            i + 1,
            len(THRR_CORE_MODELS)
        ):

            m1 = (
                THRR_CORE_MODELS[i]
            )

            m2 = (
                THRR_CORE_MODELS[j]
            )

            out[
                f"pairdiff_{m1}_{m2}"
            ] = (
                calibrated_dict[m1]
                - calibrated_dict[m2]
            )

    return out


# ============================================================
# 13A.4 Hard-case weights
# ============================================================

def build_hard_weights(
    y,
    core_prob,
    hard_fraction=0.30,
    hard_weight=2.0
):

    y = np.asarray(
        y,
        dtype=int
    )

    core_prob = np.asarray(
        core_prob
    )

    weights = np.ones(
        len(y),
        dtype=np.float32
    )

    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]

    # hard positive:
    # lowest-scored positives
    n_pos_hard = max(
        1,
        int(
            len(pos_idx)
            * hard_fraction
        )
    )

    hard_pos = pos_idx[
        np.argsort(
            core_prob[
                pos_idx
            ]
        )[
            :n_pos_hard
        ]
    ]

    # hard negative:
    # highest-scored negatives
    n_neg_hard = max(
        1,
        int(
            len(neg_idx)
            * hard_fraction
        )
    )

    hard_neg = neg_idx[
        np.argsort(
            core_prob[
                neg_idx
            ]
        )[
            -n_neg_hard:
        ]
    ]

    weights[
        hard_pos
    ] = hard_weight

    weights[
        hard_neg
    ] = hard_weight

    hard_mask = np.zeros(
        len(y),
        dtype=bool
    )

    hard_mask[
        hard_pos
    ] = True

    hard_mask[
        hard_neg
    ] = True

    return (
        weights,
        hard_mask
    )


# ============================================================
# 13A.5 Dataset
# ============================================================

class THRRDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        core_prob,
        sample_weight,
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.core_prob = torch.tensor(
            core_prob,
            dtype=torch.float32
        )

        self.sample_weight = (
            torch.tensor(
                sample_weight,
                dtype=torch.float32
            )
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx],
            self.core_prob[idx],
            self.sample_weight[idx],
        )


# ============================================================
# 13A.6 Residual ranking network
# ============================================================

class THRRNet(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden=(64, 32),
        dropout=0.15,
        max_correction=0.75,
    ):

        super().__init__()

        layers = []

        prev = input_dim

        for h in hidden:

            layers.extend([
                nn.Linear(
                    prev,
                    h
                ),
                nn.ReLU(),
                nn.Dropout(
                    dropout
                ),
            ])

            prev = h

        self.net = nn.Sequential(
            *layers
        )

        self.out = nn.Linear(
            prev,
            1
        )

        self.max_correction = (
            max_correction
        )

    def forward(
        self,
        X
    ):

        raw = (
            self.out(
                self.net(X)
            )
            .squeeze(1)
        )

        # bounded logit correction
        correction = (
            self.max_correction
            * torch.tanh(
                raw
            )
        )

        return correction


# ============================================================
# 13A.7 Pairwise ranking loss
# ============================================================

def pairwise_rank_loss(
    final_logits,
    y
):
    """
    Rank positives above negatives.

    Uses all pos-neg pairs inside minibatch.
    """

    pos_scores = (
        final_logits[
            y > 0.5
        ]
    )

    neg_scores = (
        final_logits[
            y <= 0.5
        ]
    )

    if (
        len(pos_scores) == 0
        or len(neg_scores) == 0
    ):

        return torch.tensor(
            0.0,
            device=
                final_logits.device
        )

    # To prevent enormous matrices
    max_pairs_side = 128

    if (
        len(pos_scores)
        > max_pairs_side
    ):

        idx = torch.randperm(
            len(pos_scores),
            device=
                final_logits.device
        )[
            :max_pairs_side
        ]

        pos_scores = (
            pos_scores[idx]
        )

    if (
        len(neg_scores)
        > max_pairs_side
    ):

        idx = torch.randperm(
            len(neg_scores),
            device=
                final_logits.device
        )[
            :max_pairs_side
        ]

        neg_scores = (
            neg_scores[idx]
        )

    differences = (
        pos_scores[:, None]
        - neg_scores[None, :]
    )

    # logistic ranking loss
    return (
        torch.nn.functional
        .softplus(
            -differences
        )
        .mean()
    )


# ============================================================
# 13A.8 Training fold
# ============================================================

def train_thrr_fold(
    outer_fold
):

    seed_everything(
        SEED + 500
        + outer_fold
    )

    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]

    y_train = (
        y_all[
            train_idx
        ]
    )

    y_valid = (
        y_all[
            valid_idx
        ]
    )

    # --------------------------------------------------------
    # Nested calibration
    # --------------------------------------------------------

    train_cal = {}
    valid_cal = {}

    for model_name in (
        THRR_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va

    # --------------------------------------------------------
    # Core consensus
    # --------------------------------------------------------

    core_train = np.mean(
        np.column_stack([
            train_cal[m]
            for m in THRR_CORE_MODELS
        ]),
        axis=1
    )

    core_valid = np.mean(
        np.column_stack([
            valid_cal[m]
            for m in THRR_CORE_MODELS
        ]),
        axis=1
    )

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    X_train_df = (
        build_thrr_features(
            X_dev.iloc[
                train_idx
            ].reset_index(
                drop=True
            ),
            train_cal
        )
    )

    X_valid_df = (
        build_thrr_features(
            X_dev.iloc[
                valid_idx
            ].reset_index(
                drop=True
            ),
            valid_cal
        )
    )

    scaler = StandardScaler()

    X_train = (
        scaler.fit_transform(
            X_train_df
        )
        .astype(
            np.float32
        )
    )

    X_valid = (
        scaler.transform(
            X_valid_df
        )
        .astype(
            np.float32
        )
    )

    # --------------------------------------------------------
    # Hard-case mining
    # --------------------------------------------------------

    sample_weight, hard_mask = (
        build_hard_weights(
            y_train,
            core_train,
            hard_fraction=
                THRR_CONFIG[
                    "hard_fraction"
                ],
            hard_weight=
                THRR_CONFIG[
                    "hard_weight"
                ],
        )
    )

    print(
        f"Hard training cases: "
        f"{hard_mask.sum()} / "
        f"{len(hard_mask)}"
    )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    train_ds = THRRDataset(
        X_train,
        y_train,
        core_train,
        sample_weight
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            THRR_CONFIG[
                "batch_size"
            ],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = THRRNet(

        input_dim=
            X_train.shape[1],

        hidden=
            THRR_CONFIG[
                "hidden"
            ],

        dropout=
            THRR_CONFIG[
                "dropout"
            ],

        max_correction=
            THRR_CONFIG[
                "max_logit_correction"
            ],

    ).to(TORCH_DEVICE)

    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                THRR_CONFIG[
                    "lr"
                ],

            weight_decay=
                THRR_CONFIG[
                    "weight_decay"
                ],
        )
    )

    best_pr = -np.inf
    best_state = None
    best_epoch = None
    patience_count = 0

    history = []

    # validation tensors
    X_valid_tensor = torch.tensor(
        X_valid,
        dtype=torch.float32,
        device=TORCH_DEVICE
    )

    core_valid_tensor = torch.tensor(
        core_valid,
        dtype=torch.float32,
        device=TORCH_DEVICE
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        1,
        THRR_CONFIG[
            "epochs"
        ] + 1
    ):

        model.train()

        loss_total = 0.0
        n_total = 0

        for (
            xb,
            yb,
            core_b,
            weight_b
        ) in train_loader:

            xb = xb.to(
                TORCH_DEVICE
            )

            yb = yb.to(
                TORCH_DEVICE
            )

            core_b = core_b.to(
                TORCH_DEVICE
            )

            weight_b = weight_b.to(
                TORCH_DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            correction = model(
                xb
            )

            core_logit = (
                torch_logit(
                    core_b
                )
            )

            final_logit = (
                core_logit
                + correction
            )

            # -----------------------------------------------
            # weighted BCE
            # -----------------------------------------------

            bce_each = (
                torch.nn.functional
                .binary_cross_entropy_with_logits(
                    final_logit,
                    yb,
                    reduction="none"
                )
            )

            bce_loss = (
                (
                    bce_each
                    * weight_b
                )
                .mean()
            )

            # -----------------------------------------------
            # ranking loss
            # -----------------------------------------------

            rank_loss = (
                pairwise_rank_loss(
                    final_logit,
                    yb
                )
            )

            # -----------------------------------------------
            # correction regularization
            # -----------------------------------------------

            residual_penalty = (
                correction
                .pow(2)
                .mean()
            )

            loss = (
                bce_loss
                +
                THRR_CONFIG[
                    "alpha_rank"
                ]
                * rank_loss
                +
                THRR_CONFIG[
                    "beta_residual"
                ]
                * residual_penalty
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()

            n = len(yb)

            loss_total += (
                float(
                    loss.detach().cpu()
                )
                * n
            )

            n_total += n

        # ----------------------------------------------------
        # Development screening:
        # check outer-fold PR for early stopping
        #
        # IMPORTANT:
        # if THRR wins, we will rerun with fully nested
        # inner early stopping before reporting it.
        # ----------------------------------------------------

        model.eval()

        with torch.no_grad():

            correction_valid = (
                model(
                    X_valid_tensor
                )
                .cpu()
                .numpy()
            )

        final_valid = (
            np_sigmoid(
                safe_logit(
                    core_valid
                )
                + correction_valid
            )
        )

        valid_pr = (
            average_precision_score(
                y_valid,
                final_valid
            )
        )

        history.append({

            "epoch":
                epoch,

            "loss":
                loss_total
                / n_total,

            "valid_PR":
                valid_pr,
        })

        if valid_pr > (
            best_pr + 1e-5
        ):

            best_pr = valid_pr

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_count = 0

        else:

            patience_count += 1

        print(
            f"epoch={epoch:02d} "
            f"loss="
            f"{loss_total/n_total:.5f} "
            f"PR={valid_pr:.6f} "
            f"best={best_pr:.6f}"
        )

        if (
            patience_count
            >= THRR_CONFIG[
                "patience"
            ]
        ):

            break

    # --------------------------------------------------------
    # Restore best
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    model.eval()

    with torch.no_grad():

        correction_valid = (
            model(
                X_valid_tensor
            )
            .cpu()
            .numpy()
        )

    final_valid = clip_probs(
        np_sigmoid(
            safe_logit(
                core_valid
            )
            + correction_valid
        )
    )

    return {

        "valid_idx":
            valid_idx,

        "core_valid":
            core_valid,

        "final_valid":
            final_valid,

        "correction":
            correction_valid,

        "best_epoch":
            best_epoch,

        "best_PR":
            best_pr,

        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# 13A.9 Run 5-fold development screen
# ============================================================

THRR_OOF = np.zeros(
    len(y_all),
    dtype=float
)

THRR_CORE = np.zeros(
    len(y_all),
    dtype=float
)

THRR_CORRECTION = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []

for fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"THRR FOLD "
        f"{fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 100
    )

    result = train_thrr_fold(
        fold
    )

    idx = result[
        "valid_idx"
    ]

    THRR_OOF[
        idx
    ] = (
        result[
            "final_valid"
        ]
    )

    THRR_CORE[
        idx
    ] = (
        result[
            "core_valid"
        ]
    )

    THRR_CORRECTION[
        idx
    ] = (
        result[
            "correction"
        ]
    )

    core_pr = (
        average_precision_score(
            y_all[idx],
            result[
                "core_valid"
            ]
        )
    )

    thrr_pr = (
        average_precision_score(
            y_all[idx],
            result[
                "final_valid"
            ]
        )
    )

    fold_rows.append({

        "Fold":
            fold,

        "Best_Epoch":
            result[
                "best_epoch"
            ],

        "Core_PR":
            core_pr,

        "THRR_PR":
            thrr_pr,

        "Delta_PR":
            thrr_pr
            - core_pr,

        "Mean_Abs_Logit_Correction":
            float(
                np.mean(
                    np.abs(
                        result[
                            "correction"
                        ]
                    )
                )
            ),

        "Max_Abs_Logit_Correction":
            float(
                np.max(
                    np.abs(
                        result[
                            "correction"
                        ]
                    )
                )
            ),
    })

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 13A.10 Global results
# ============================================================

core_t, _, _ = (
    select_best_threshold(
        y_all,
        THRR_CORE,
        objective="F1"
    )
)

thrr_t, _, _ = (
    select_best_threshold(
        y_all,
        THRR_OOF,
        objective="F1"
    )
)

core_metrics = (
    evaluate_binary_predictions(
        y_all,
        THRR_CORE,
        threshold=core_t
    )
)

thrr_metrics = (
    evaluate_binary_predictions(
        y_all,
        THRR_OOF,
        threshold=thrr_t
    )
)


module13a_results = (
    pd.DataFrame([
        {
            "Method":
                "Core4Mean",

            **core_metrics
        },

        {
            "Method":
                "THRR",

            **thrr_metrics
        },
    ])
)

module13a_results[
    "Delta_PR_vs_Core"
] = (
    module13a_results[
        "PR_AUC"
    ]
    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13A — THRR DEVELOPMENT RESULTS"
)

print(
    "=" * 120
)

display(
    module13a_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13A.11 Fold behavior
# ============================================================

module13a_folds = (
    pd.DataFrame(
        fold_rows
    )
)

print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13A — FOLD RESULTS"
)

print(
    "=" * 120
)

display(
    module13a_folds
)


print(
    "\nMean ΔPR:",
    module13a_folds[
        "Delta_PR"
    ].mean()
)

print(
    "Positive-gain folds:",
    int(
        (
            module13a_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13A.12 Correction distribution
# ============================================================

correction_abs = (
    np.abs(
        THRR_CORRECTION
    )
)

correction_table = pd.DataFrame({

    "Metric": [
        "Mean abs logit correction",
        "Median abs logit correction",
        "P90 abs logit correction",
        "P95 abs logit correction",
        "P99 abs logit correction",
        "Maximum abs logit correction",
    ],

    "Value": [

        correction_abs.mean(),

        np.median(
            correction_abs
        ),

        np.quantile(
            correction_abs,
            0.90
        ),

        np.quantile(
            correction_abs,
            0.95
        ),

        np.quantile(
            correction_abs,
            0.99
        ),

        correction_abs.max(),
    ],
})

print(
    "\n"
    + "=" * 120
)

print(
    "THRR CORRECTION DISTRIBUTION"
)

print(
    "=" * 120
)

display(
    correction_table
)


# ============================================================
# 13A.13 Save
# ============================================================

save_csv(
    module13a_results,
    TAB_DIR
    / "module13a_thrr_results.csv"
)

save_csv(
    module13a_folds,
    TAB_DIR
    / "module13a_thrr_fold_results.csv"
)

save_csv(
    correction_table,
    TAB_DIR
    / "module13a_thrr_correction_distribution.csv"
)


module13a_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        THRR_CORE,

    "THRR":
        THRR_OOF,

    "THRR_Logit_Correction":
        THRR_CORRECTION,
})

save_csv(
    module13a_pred_df,
    PRED_DIR
    / "module13a_thrr_oof_predictions.csv"
)

print(
    "\nMODULE 13A COMPLETE."
)


THRR FOLD 1/5
Hard training cases: 48706 / 162355
epoch=01 loss=0.53919 PR=0.379573 best=0.379573
epoch=02 loss=0.54029 PR=0.379386 best=0.379573
epoch=03 loss=0.54078 PR=0.379392 best=0.379573
epoch=04 loss=0.53812 PR=0.379678 best=0.379678
epoch=05 loss=0.53944 PR=0.378829 best=0.379678
epoch=06 loss=0.53874 PR=0.379283 best=0.379678
epoch=07 loss=0.53895 PR=0.378964 best=0.379678
epoch=08 loss=0.53912 PR=0.379352 best=0.379678

THRR FOLD 2/5
Hard training cases: 48706 / 162355
epoch=01 loss=0.53912 PR=0.382508 best=0.382508
epoch=02 loss=0.53740 PR=0.382574 best=0.382574
epoch=03 loss=0.53903 PR=0.382355 best=0.382574
epoch=04 loss=0.53911 PR=0.382264 best=0.382574
epoch=05 loss=0.53890 PR=0.382037 best=0.382574
epoch=06 loss=0.53723 PR=0.381839 best=0.382574

THRR FOLD 3/5
Hard training cases: 48706 / 162355
epoch=01 loss=0.54001 PR=0.377619 best=0.377619
epoch=02 loss=0.53973 PR=0.377730 best=0.377730
epoch=03 loss=0.53780 PR=0.377768 best=0.377768
epoch=04 loss=0.53786 PR=0.3777

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,THRR,0.379755,-0.000572,0.850413,0.070897,0.238651,0.016749,0.344330,0.541697,0.421031,0.357181,0.17



MODULE 13A — FOLD RESULTS


,Fold,Best_Epoch,Core_PR,THRR_PR,Delta_PR,Mean_Abs_Logit_Correction,Max_Abs_Logit_Correction
0,0,4,0.380070,0.379678,-0.000392,0.141799,0.677741
1,1,2,0.382772,0.382574,-0.000198,0.133283,0.555083
2,2,6,0.378467,0.378029,-0.000438,0.130231,0.623315
3,3,8,0.372833,0.373396,0.000563,0.097620,0.533917
4,4,2,0.390072,0.389952,-0.000121,0.123842,0.568007



Mean ΔPR: -0.00011702187165387734
Positive-gain folds: 1 / 5

THRR CORRECTION DISTRIBUTION


,Metric,Value
0,Mean abs logit correction,0.125355
1,Median abs logit correction,0.075006
2,P90 abs logit correction,0.343519
3,P95 abs logit correction,0.410338
4,P99 abs logit correction,0.499812
5,Maximum abs logit correction,0.677741



MODULE 13A COMPLETE.


In [19]:
# ============================================================
# MODULE 13B — SARR
# Sparse Ambiguity-aware Rank Repair
# ============================================================
#
# Key differences from THRR:
#
# 1. Core4 remains unchanged for most samples
# 2. Corrections only occur in ambiguous/high-disagreement cases
# 3. Very small bounded logit corrections
# 4. Pairwise loss focuses on confused positive-negative pairs
# 5. Strong preservation penalty keeps ranking close to Core4
#
# DEV screening only.
# HOLDOUT untouched.
# ============================================================

import copy
import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score


# ============================================================
# 13B.1 Configuration
# ============================================================

SARR_CORE_MODELS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

SARR_SPECIALISTS = [
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
]

SARR_MODELS = (
    SARR_CORE_MODELS
    + SARR_SPECIALISTS
)

SARR_CONTEXT_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]

SARR_CONFIG = {

    "hidden":
        (48, 24),

    "dropout":
        0.20,

    "batch_size":
        1024,

    "epochs":
        15,

    "lr":
        3e-4,

    "weight_decay":
        3e-4,

    # much smaller than THRR
    "max_logit_correction":
        0.15,

    # only most ambiguous fraction can receive
    # substantial correction
    "ambiguity_fraction":
        0.30,

    # ranking loss strength
    "alpha_rank":
        0.30,

    # preserve original core scores
    "beta_preserve":
        0.50,

    # encourage sparse correction
    "gamma_sparse":
        0.10,

    # emphasize confused pairs only
    "pair_margin":
        0.35,

    "patience":
        3,
}


# ============================================================
# 13B.2 Helpers
# ============================================================

def torch_safe_logit(p):

    p = torch.clamp(
        p,
        1e-6,
        1.0 - 1e-6
    )

    return torch.log(
        p / (1.0 - p)
    )


def sigmoid_np(z):

    return (
        1.0
        / (
            1.0
            + np.exp(-z)
        )
    )


def rank01(x):

    x = np.asarray(x)

    order = np.argsort(
        np.argsort(x)
    )

    return (
        order.astype(
            np.float32
        )
        / max(
            len(x) - 1,
            1
        )
    )


# ============================================================
# 13B.3 Feature builder
# ============================================================

def build_sarr_features(
    X_original,
    calibrated_dict
):

    out = pd.DataFrame(
        index=np.arange(
            len(X_original)
        )
    )

    # clinical/context
    for c in SARR_CONTEXT_FEATURES:

        out[
            f"x_{c}"
        ] = (
            X_original[c]
            .to_numpy(
                dtype=np.float32
            )
        )

    # core probabilities
    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in SARR_CORE_MODELS
    ])

    core_mean = (
        core_matrix.mean(
            axis=1
        )
    )

    core_std = (
        core_matrix.std(
            axis=1
        )
    )

    core_range = (
        core_matrix.max(axis=1)
        - core_matrix.min(axis=1)
    )

    out["core_mean"] = core_mean

    out["core_logit"] = (
        safe_logit(
            core_mean
        )
    )

    out["core_rank"] = (
        rank01(
            core_mean
        )
    )

    out["core_std"] = (
        core_std
    )

    out["core_range"] = (
        core_range
    )

    out["core_entropy"] = (
        binary_entropy(
            core_mean
        )
    )

    # expert information
    for m in SARR_MODELS:

        p = np.asarray(
            calibrated_dict[m]
        )

        out[
            f"p_{m}"
        ] = p

        out[
            f"delta_{m}"
        ] = (
            p - core_mean
        )

        out[
            f"absdelta_{m}"
        ] = np.abs(
            p - core_mean
        )

    return out


# ============================================================
# 13B.4 Train-derived ambiguity gate
# ============================================================

def build_ambiguity_gate(
    core_train,
    train_cal,
    core_target,
    core_apply,
    apply_cal,
    fraction=0.30,
):
    """
    Gate is derived WITHOUT labels.

    Ambiguity combines:
      - core predictive entropy
      - inter-core disagreement

    Threshold determined from training portion only.
    """

    train_matrix = np.column_stack([
        train_cal[m]
        for m in SARR_CORE_MODELS
    ])

    apply_matrix = np.column_stack([
        apply_cal[m]
        for m in SARR_CORE_MODELS
    ])

    entropy_train = (
        binary_entropy(
            core_train
        )
        / np.log(2.0)
    )

    entropy_apply = (
        binary_entropy(
            core_apply
        )
        / np.log(2.0)
    )

    std_train = (
        train_matrix.std(
            axis=1
        )
    )

    std_apply = (
        apply_matrix.std(
            axis=1
        )
    )

    # normalize disagreement using TRAIN only
    scale = np.quantile(
        std_train,
        0.95
    )

    scale = max(
        scale,
        1e-6
    )

    disagree_train = np.clip(
        std_train / scale,
        0,
        1
    )

    disagree_apply = np.clip(
        std_apply / scale,
        0,
        1
    )

    ambiguity_train = (
        0.65 * entropy_train
        +
        0.35 * disagree_train
    )

    ambiguity_apply = (
        0.65 * entropy_apply
        +
        0.35 * disagree_apply
    )

    threshold = np.quantile(
        ambiguity_train,
        1.0 - fraction
    )

    # soft gate rather than binary gate
    width = max(
        np.std(
            ambiguity_train
        ) * 0.20,
        1e-4
    )

    gate_train = (
        1.0
        / (
            1.0
            + np.exp(
                -(
                    ambiguity_train
                    - threshold
                )
                / width
            )
        )
    )

    gate_apply = (
        1.0
        / (
            1.0
            + np.exp(
                -(
                    ambiguity_apply
                    - threshold
                )
                / width
            )
        )
    )

    return (
        gate_train.astype(
            np.float32
        ),
        gate_apply.astype(
            np.float32
        ),
        ambiguity_train,
        ambiguity_apply,
        threshold,
    )


# ============================================================
# 13B.5 Dataset
# ============================================================

class SARRDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        core,
        gate
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.core = torch.tensor(
            core,
            dtype=torch.float32
        )

        self.gate = torch.tensor(
            gate,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx],
            self.core[idx],
            self.gate[idx],
        )


# ============================================================
# 13B.6 Small repair network
# ============================================================

class SARRNet(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden=(48, 24),
        dropout=0.20,
        max_correction=0.15
    ):

        super().__init__()

        layers = []

        previous = input_dim

        for h in hidden:

            layers.extend([
                nn.Linear(
                    previous,
                    h
                ),
                nn.GELU(),
                nn.Dropout(
                    dropout
                ),
            ])

            previous = h

        self.backbone = (
            nn.Sequential(
                *layers
            )
        )

        self.output = nn.Linear(
            previous,
            1
        )

        # start near zero correction
        nn.init.zeros_(
            self.output.weight
        )

        nn.init.zeros_(
            self.output.bias
        )

        self.max_correction = (
            max_correction
        )

    def forward(
        self,
        x
    ):

        raw = (
            self.output(
                self.backbone(x)
            )
            .squeeze(1)
        )

        return (
            self.max_correction
            * torch.tanh(
                raw
            )
        )


# ============================================================
# 13B.7 Local pairwise ranking loss
# ============================================================

def local_pairwise_rank_loss(
    final_logit,
    core_logit,
    y,
    gate,
    margin=0.35,
    max_side=96
):
    """
    Only pairs that were already close/confused according
    to Core4 receive strong ranking pressure.
    """

    pos = (
        y > 0.5
    )

    neg = (
        y <= 0.5
    )

    if (
        pos.sum() == 0
        or neg.sum() == 0
    ):

        return torch.tensor(
            0.0,
            device=
                final_logit.device
        )

    pos_final = (
        final_logit[pos]
    )

    neg_final = (
        final_logit[neg]
    )

    pos_core = (
        core_logit[pos]
    )

    neg_core = (
        core_logit[neg]
    )

    pos_gate = (
        gate[pos]
    )

    neg_gate = (
        gate[neg]
    )

    # sample sides
    if len(pos_final) > max_side:

        ids = torch.randperm(
            len(pos_final),
            device=
                final_logit.device
        )[:max_side]

        pos_final = (
            pos_final[ids]
        )

        pos_core = (
            pos_core[ids]
        )

        pos_gate = (
            pos_gate[ids]
        )

    if len(neg_final) > max_side:

        ids = torch.randperm(
            len(neg_final),
            device=
                final_logit.device
        )[:max_side]

        neg_final = (
            neg_final[ids]
        )

        neg_core = (
            neg_core[ids]
        )

        neg_gate = (
            neg_gate[ids]
        )

    # core pair difference
    core_diff = (
        pos_core[:, None]
        - neg_core[None, :]
    )

    # Focus on pairs where positive and negative
    # are already close or incorrectly ordered.
    closeness = torch.exp(
        -torch.abs(
            core_diff
        )
        / margin
    )

    pair_gate = (
        0.5
        * (
            pos_gate[:, None]
            + neg_gate[None, :]
        )
    )

    pair_weight = (
        closeness
        * pair_gate
    )

    final_diff = (
        pos_final[:, None]
        - neg_final[None, :]
    )

    loss_matrix = (
        F.softplus(
            -final_diff
        )
    )

    return (
        (
            loss_matrix
            * pair_weight
        ).sum()
        /
        (
            pair_weight.sum()
            + 1e-8
        )
    )


# ============================================================
# 13B.8 Fold trainer
# ============================================================

def run_sarr_fold(
    fold
):

    seed_everything(
        SEED
        + 800
        + fold
    )

    train_idx = np.where(
        fold_assignment
        != fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == fold
    )[0]

    y_train = (
        y_all[
            train_idx
        ]
    )

    y_valid = (
        y_all[
            valid_idx
        ]
    )

    # --------------------------------------------------------
    # nested calibration relative to outer fold
    # --------------------------------------------------------

    train_cal = {}
    valid_cal = {}

    for model_name in (
        SARR_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va


    # --------------------------------------------------------
    # Core consensus
    # --------------------------------------------------------

    core_train = np.mean(
        np.column_stack([
            train_cal[m]
            for m in SARR_CORE_MODELS
        ]),
        axis=1
    )

    core_valid = np.mean(
        np.column_stack([
            valid_cal[m]
            for m in SARR_CORE_MODELS
        ]),
        axis=1
    )


    # --------------------------------------------------------
    # Build features
    # --------------------------------------------------------

    Xtr_df = (
        build_sarr_features(
            X_dev.iloc[
                train_idx
            ].reset_index(
                drop=True
            ),
            train_cal
        )
    )

    Xva_df = (
        build_sarr_features(
            X_dev.iloc[
                valid_idx
            ].reset_index(
                drop=True
            ),
            valid_cal
        )
    )

    scaler = (
        StandardScaler()
    )

    Xtr = (
        scaler.fit_transform(
            Xtr_df
        )
        .astype(
            np.float32
        )
    )

    Xva = (
        scaler.transform(
            Xva_df
        )
        .astype(
            np.float32
        )
    )


    # --------------------------------------------------------
    # Ambiguity gate
    # --------------------------------------------------------

    (
        gate_train,
        gate_valid,
        ambiguity_train,
        ambiguity_valid,
        gate_threshold,
    ) = build_ambiguity_gate(

        core_train,
        train_cal,
        y_train,

        core_valid,
        valid_cal,

        fraction=
            SARR_CONFIG[
                "ambiguity_fraction"
            ],
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    ds = SARRDataset(
        Xtr,
        y_train,
        core_train,
        gate_train
    )

    loader = DataLoader(
        ds,
        batch_size=
            SARR_CONFIG[
                "batch_size"
            ],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = SARRNet(

        input_dim=
            Xtr.shape[1],

        hidden=
            SARR_CONFIG[
                "hidden"
            ],

        dropout=
            SARR_CONFIG[
                "dropout"
            ],

        max_correction=
            SARR_CONFIG[
                "max_logit_correction"
            ],

    ).to(
        TORCH_DEVICE
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            SARR_CONFIG[
                "lr"
            ],

        weight_decay=
            SARR_CONFIG[
                "weight_decay"
            ],
    )


    # validation tensors
    Xva_t = torch.tensor(
        Xva,
        dtype=torch.float32,
        device=TORCH_DEVICE
    )

    gate_va_t = torch.tensor(
        gate_valid,
        dtype=torch.float32,
        device=TORCH_DEVICE
    )


    best_state = None
    best_pr = -np.inf
    best_epoch = -1

    wait = 0

    history = []


    # ========================================================
    # training
    # ========================================================

    for epoch in range(
        1,
        SARR_CONFIG[
            "epochs"
        ] + 1
    ):

        model.train()

        running = 0.0
        seen = 0

        for (
            xb,
            yb,
            core_b,
            gate_b
        ) in loader:

            xb = xb.to(
                TORCH_DEVICE
            )

            yb = yb.to(
                TORCH_DEVICE
            )

            core_b = core_b.to(
                TORCH_DEVICE
            )

            gate_b = gate_b.to(
                TORCH_DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            raw_correction = (
                model(xb)
            )

            # sparse/local repair
            correction = (
                gate_b
                * raw_correction
            )

            core_logit = (
                torch_safe_logit(
                    core_b
                )
            )

            final_logit = (
                core_logit
                + correction
            )


            # ------------------------------------------------
            # probability preservation
            # ------------------------------------------------

            bce = (
                F.binary_cross_entropy_with_logits(
                    final_logit,
                    yb
                )
            )


            # ------------------------------------------------
            # local ranking
            # ------------------------------------------------

            rank_loss = (
                local_pairwise_rank_loss(

                    final_logit,
                    core_logit,
                    yb,
                    gate_b,

                    margin=
                        SARR_CONFIG[
                            "pair_margin"
                        ],
                )
            )


            # ------------------------------------------------
            # preserve core ordering / predictions
            # ------------------------------------------------

            preserve_loss = (
                correction
                .pow(2)
                .mean()
            )


            # ------------------------------------------------
            # sparse correction penalty
            # ------------------------------------------------

            sparse_loss = (
                torch.abs(
                    correction
                )
                .mean()
            )


            loss = (

                bce

                + SARR_CONFIG[
                    "alpha_rank"
                ]
                * rank_loss

                + SARR_CONFIG[
                    "beta_preserve"
                ]
                * preserve_loss

                + SARR_CONFIG[
                    "gamma_sparse"
                ]
                * sparse_loss
            )


            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                2.0
            )

            optimizer.step()

            n = len(yb)

            running += (
                float(
                    loss.detach()
                    .cpu()
                )
                * n
            )

            seen += n


        # ----------------------------------------------------
        # Screening validation
        # ----------------------------------------------------

        model.eval()

        with torch.no_grad():

            raw_corr = (
                model(
                    Xva_t
                )
            )

            corr = (
                gate_va_t
                * raw_corr
            )

            corr_np = (
                corr.cpu()
                .numpy()
            )

        p_valid = clip_probs(

            sigmoid_np(

                safe_logit(
                    core_valid
                )

                + corr_np
            )
        )

        pr = (
            average_precision_score(
                y_valid,
                p_valid
            )
        )


        history.append({

            "epoch":
                epoch,

            "loss":
                running
                / seen,

            "valid_PR":
                pr,

            "mean_abs_correction":
                float(
                    np.mean(
                        np.abs(
                            corr_np
                        )
                    )
                ),
        })


        if pr > (
            best_pr
            + 1e-5
        ):

            best_pr = pr
            best_epoch = epoch

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            wait = 0

        else:

            wait += 1


        print(

            f"epoch={epoch:02d} "
            f"loss={running/seen:.5f} "
            f"PR={pr:.6f} "
            f"best={best_pr:.6f} "
            f"|corr|="
            f"{np.mean(np.abs(corr_np)):.5f}"
        )


        if wait >= (
            SARR_CONFIG[
                "patience"
            ]
        ):
            break


    # --------------------------------------------------------
    # restore
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    model.eval()

    with torch.no_grad():

        raw_corr = (
            model(
                Xva_t
            )
        )

        corr = (
            gate_va_t
            * raw_corr
        )

        corr = (
            corr.cpu()
            .numpy()
        )


    p_valid = clip_probs(

        sigmoid_np(

            safe_logit(
                core_valid
            )

            + corr
        )
    )


    return {

        "valid_idx":
            valid_idx,

        "core":
            core_valid,

        "prediction":
            p_valid,

        "correction":
            corr,

        "gate":
            gate_valid,

        "ambiguity":
            ambiguity_valid,

        "gate_threshold":
            gate_threshold,

        "best_epoch":
            best_epoch,

        "best_PR":
            best_pr,

        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# 13B.9 Run
# ============================================================

SARR_OOF = np.zeros(
    len(y_all),
    dtype=float
)

SARR_CORE = np.zeros(
    len(y_all),
    dtype=float
)

SARR_CORRECTION = np.zeros(
    len(y_all),
    dtype=float
)

SARR_GATE = np.zeros(
    len(y_all),
    dtype=float
)

SARR_AMBIGUITY = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []


for fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"SARR FOLD "
        f"{fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 100
    )

    result = (
        run_sarr_fold(
            fold
        )
    )

    idx = result[
        "valid_idx"
    ]

    SARR_OOF[
        idx
    ] = (
        result[
            "prediction"
        ]
    )

    SARR_CORE[
        idx
    ] = (
        result[
            "core"
        ]
    )

    SARR_CORRECTION[
        idx
    ] = (
        result[
            "correction"
        ]
    )

    SARR_GATE[
        idx
    ] = (
        result[
            "gate"
        ]
    )

    SARR_AMBIGUITY[
        idx
    ] = (
        result[
            "ambiguity"
        ]
    )


    core_pr = (
        average_precision_score(
            y_all[idx],
            result[
                "core"
            ]
        )
    )

    sarr_pr = (
        average_precision_score(
            y_all[idx],
            result[
                "prediction"
            ]
        )
    )


    fold_rows.append({

        "Fold":
            fold,

        "Best_Epoch":
            result[
                "best_epoch"
            ],

        "Core_PR":
            core_pr,

        "SARR_PR":
            sarr_pr,

        "Delta_PR":
            sarr_pr
            - core_pr,

        "Mean_Gate":
            float(
                np.mean(
                    result[
                        "gate"
                    ]
                )
            ),

        "Mean_Abs_Correction":
            float(
                np.mean(
                    np.abs(
                        result[
                            "correction"
                        ]
                    )
                )
            ),

        "P95_Abs_Correction":
            float(
                np.quantile(
                    np.abs(
                        result[
                            "correction"
                        ]
                    ),
                    0.95
                )
            ),
    })


    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13B.10 Global evaluation
# ============================================================

core_threshold, _, _ = (
    select_best_threshold(
        y_all,
        SARR_CORE,
        objective="F1"
    )
)

sarr_threshold, _, _ = (
    select_best_threshold(
        y_all,
        SARR_OOF,
        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(
        y_all,
        SARR_CORE,
        threshold=
            core_threshold
    )
)

sarr_metrics = (
    evaluate_binary_predictions(
        y_all,
        SARR_OOF,
        threshold=
            sarr_threshold
    )
)


module13b_results = (
    pd.DataFrame([

        {
            "Method":
                "Core4Mean",

            **core_metrics
        },

        {
            "Method":
                "SARR",

            **sarr_metrics
        },
    ])
)


module13b_results[
    "Delta_PR_vs_Core"
] = (
    module13b_results[
        "PR_AUC"
    ]
    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13B — SARR DEVELOPMENT RESULTS"
)

print(
    "=" * 120
)

display(
    module13b_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13B.11 Fold results
# ============================================================

module13b_folds = (
    pd.DataFrame(
        fold_rows
    )
)

print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13B — FOLD RESULTS"
)

print(
    "=" * 120
)

display(
    module13b_folds
)


print(
    "\nMean ΔPR:",
    float(
        module13b_folds[
            "Delta_PR"
        ].mean()
    )
)

print(
    "Positive folds:",
    int(
        (
            module13b_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13B.12 Sparsity diagnostics
# ============================================================

abs_corr = (
    np.abs(
        SARR_CORRECTION
    )
)

sparsity_table = pd.DataFrame({

    "Metric": [

        "Mean gate",

        "Median gate",

        "Fraction gate > 0.50",

        "Mean abs logit correction",

        "Median abs logit correction",

        "P95 abs logit correction",

        "P99 abs logit correction",

        "Max abs logit correction",

        "Fraction |correction| > 0.01",

        "Fraction |correction| > 0.05",
    ],

    "Value": [

        float(
            SARR_GATE.mean()
        ),

        float(
            np.median(
                SARR_GATE
            )
        ),

        float(
            np.mean(
                SARR_GATE
                > 0.50
            )
        ),

        float(
            abs_corr.mean()
        ),

        float(
            np.median(
                abs_corr
            )
        ),

        float(
            np.quantile(
                abs_corr,
                0.95
            )
        ),

        float(
            np.quantile(
                abs_corr,
                0.99
            )
        ),

        float(
            abs_corr.max()
        ),

        float(
            np.mean(
                abs_corr
                > 0.01
            )
        ),

        float(
            np.mean(
                abs_corr
                > 0.05
            )
        ),
    ],
})


print(
    "\n"
    + "=" * 120
)

print(
    "SARR SPARSITY / CORRECTION DIAGNOSTICS"
)

print(
    "=" * 120
)

display(
    sparsity_table
)


# ============================================================
# 13B.13 Save
# ============================================================

save_csv(
    module13b_results,
    TAB_DIR
    / "module13b_sarr_results.csv"
)

save_csv(
    module13b_folds,
    TAB_DIR
    / "module13b_sarr_fold_results.csv"
)

save_csv(
    sparsity_table,
    TAB_DIR
    / "module13b_sarr_sparsity.csv"
)


sarr_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        SARR_CORE,

    "SARR":
        SARR_OOF,

    "Ambiguity":
        SARR_AMBIGUITY,

    "Gate":
        SARR_GATE,

    "LogitCorrection":
        SARR_CORRECTION,
})


save_csv(
    sarr_pred_df,
    PRED_DIR
    / "module13b_sarr_oof_predictions.csv"
)


print(
    "\nMODULE 13B COMPLETE."
)


SARR FOLD 1/5
epoch=01 loss=0.43626 PR=0.380069 best=0.380069 |corr|=0.00002
epoch=02 loss=0.43623 PR=0.380069 best=0.380069 |corr|=0.00002
epoch=03 loss=0.43587 PR=0.380072 best=0.380069 |corr|=0.00002
epoch=04 loss=0.43633 PR=0.380077 best=0.380069 |corr|=0.00004

SARR FOLD 2/5
epoch=01 loss=0.43590 PR=0.382771 best=0.382771 |corr|=0.00003
epoch=02 loss=0.43587 PR=0.382772 best=0.382771 |corr|=0.00001
epoch=03 loss=0.43568 PR=0.382771 best=0.382771 |corr|=0.00003
epoch=04 loss=0.43603 PR=0.382774 best=0.382771 |corr|=0.00003

SARR FOLD 3/5
epoch=01 loss=0.43605 PR=0.378449 best=0.378449 |corr|=0.00003
epoch=02 loss=0.43526 PR=0.378462 best=0.378462 |corr|=0.00003
epoch=03 loss=0.43609 PR=0.378463 best=0.378462 |corr|=0.00002
epoch=04 loss=0.43551 PR=0.378469 best=0.378462 |corr|=0.00001
epoch=05 loss=0.43580 PR=0.378466 best=0.378462 |corr|=0.00003

SARR FOLD 4/5
epoch=01 loss=0.43558 PR=0.372833 best=0.372833 |corr|=0.00003
epoch=02 loss=0.43554 PR=0.372828 best=0.372833 |corr|=0.0

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,SARR,0.380330,0.000002,0.850558,0.069706,0.235584,0.002375,0.350554,0.526107,0.420753,0.356122,0.22



MODULE 13B — FOLD RESULTS


,Fold,Best_Epoch,Core_PR,SARR_PR,Delta_PR,Mean_Gate,Mean_Abs_Correction,P95_Abs_Correction
0,0,1,0.380070,0.380069,-1.112812e-06,0.310251,0.000015,0.000079
1,1,1,0.382772,0.382771,-3.243640e-07,0.297955,0.000032,0.000153
2,2,2,0.378467,0.378462,-4.634431e-06,0.325827,0.000028,0.000138
3,3,1,0.372833,0.372833,-1.431523e-07,0.293320,0.000030,0.000164
4,4,1,0.390072,0.390073,4.782263e-07,0.314491,0.000036,0.000189



Mean ΔPR: -1.147306766757783e-06
Positive folds: 1 / 5

SARR SPARSITY / CORRECTION DIAGNOSTICS


,Metric,Value
0,Mean gate,3.083687e-01
1,Median gate,1.698742e-02
2,Fraction gate > 0.50,3.036552e-01
3,Mean abs logit correction,2.827701e-05
4,Median abs logit correction,7.352263e-07
5,P95 abs logit correction,1.508608e-04
6,P99 abs logit correction,2.636346e-04
7,Max abs logit correction,1.024034e-03
8,Fraction |correction| > 0.01,0.000000e+00
9,Fraction |correction| > 0.05,0.000000e+00



MODULE 13B COMPLETE.


In [20]:
# ============================================================
# MODULE 13C — TRUST-REGULARIZED RANK FUSION (TRRF)
# ============================================================
#
# Goal:
# Learn static Core4 weights specifically for ranking.
#
# Core experts:
#   HistGB, CatBoost, AutoInt, DeepFM_v2
#
# Objective:
#   pairwise ranking loss
#   + probability/BCE preservation
#   + regularization toward equal weights
#
# Strict outer-fold evaluation.
# NO HOLDOUT.
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import average_precision_score


# ============================================================
# 13C.1 Core
# ============================================================

TRRF_MODELS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

N_EXPERTS = len(TRRF_MODELS)

TRRF_CONFIG = {

    "epochs":
        300,

    "lr":
        0.03,

    # pairwise ranking objective
    "alpha_rank":
        1.0,

    # keep probabilities sensible
    "beta_bce":
        0.20,

    # keep weights close to equal
    "gamma_equal":
        0.20,

    # number sampled per class
    "pair_sample_size":
        1024,

    "patience":
        30,

    "min_delta":
        1e-6,
}


# ============================================================
# 13C.2 Weighted logit fusion
# ============================================================

def torch_logit_clip(p):

    return torch.log(
        torch.clamp(
            p,
            1e-6,
            1 - 1e-6
        )
        /
        (
            1 -
            torch.clamp(
                p,
                1e-6,
                1 - 1e-6
            )
        )
    )


def weighted_logit_prediction(
    P,
    weights
):

    P = np.clip(
        P,
        1e-6,
        1 - 1e-6
    )

    logits = np.log(
        P / (1-P)
    )

    z = logits @ weights

    return clip_probs(
        1 / (1 + np.exp(-z))
    )


# ============================================================
# 13C.3 Rank optimizer
# ============================================================

def fit_trrf(
    P_train,
    y_train,
    seed
):

    torch.manual_seed(
        seed
    )

    device = TORCH_DEVICE

    P_t = torch.tensor(
        P_train,
        dtype=torch.float32,
        device=device
    )

    y_t = torch.tensor(
        y_train,
        dtype=torch.float32,
        device=device
    )

    expert_logits = (
        torch_logit_clip(
            P_t
        )
    )

    # trainable unconstrained parameters
    raw_weights = torch.zeros(
        N_EXPERTS,
        dtype=torch.float32,
        device=device,
        requires_grad=True
    )

    optimizer = torch.optim.Adam(
        [raw_weights],
        lr=TRRF_CONFIG["lr"]
    )

    equal_weights = torch.full(
        (N_EXPERTS,),
        1.0 / N_EXPERTS,
        device=device
    )

    pos_idx_all = torch.where(
        y_t > 0.5
    )[0]

    neg_idx_all = torch.where(
        y_t <= 0.5
    )[0]

    best_loss = np.inf
    best_weights = None
    best_epoch = 0
    wait = 0

    history = []

    rng = np.random.default_rng(
        seed
    )

    for epoch in range(
        1,
        TRRF_CONFIG["epochs"] + 1
    ):

        optimizer.zero_grad()

        # -----------------------------------------------
        # nonnegative normalized weights
        # -----------------------------------------------

        weights = torch.softmax(
            raw_weights,
            dim=0
        )

        fused_logit = (
            expert_logits
            @ weights
        )

        # -----------------------------------------------
        # sample pos/neg cases
        # -----------------------------------------------

        n_pos = min(
            TRRF_CONFIG[
                "pair_sample_size"
            ],
            len(pos_idx_all)
        )

        n_neg = min(
            TRRF_CONFIG[
                "pair_sample_size"
            ],
            len(neg_idx_all)
        )

        pos_sel_np = rng.choice(
            len(pos_idx_all),
            size=n_pos,
            replace=False
        )

        neg_sel_np = rng.choice(
            len(neg_idx_all),
            size=n_neg,
            replace=False
        )

        pos_sel = pos_idx_all[
            torch.tensor(
                pos_sel_np,
                device=device
            )
        ]

        neg_sel = neg_idx_all[
            torch.tensor(
                neg_sel_np,
                device=device
            )
        ]

        pos_scores = fused_logit[
            pos_sel
        ]

        neg_scores = fused_logit[
            neg_sel
        ]

        # pairwise rank loss
        pair_diff = (
            pos_scores[:, None]
            - neg_scores[None, :]
        )

        rank_loss = (
            F.softplus(
                -pair_diff
            )
            .mean()
        )

        # probability preservation
        bce_loss = (
            F.binary_cross_entropy_with_logits(
                fused_logit,
                y_t
            )
        )

        # trust regularization:
        # discourage unstable extreme weights
        equal_penalty = (
            (
                weights
                - equal_weights
            )
            .pow(2)
            .mean()
        )

        loss = (
            TRRF_CONFIG[
                "alpha_rank"
            ]
            * rank_loss

            + TRRF_CONFIG[
                "beta_bce"
            ]
            * bce_loss

            + TRRF_CONFIG[
                "gamma_equal"
            ]
            * equal_penalty
        )

        loss.backward()

        optimizer.step()

        loss_value = float(
            loss.detach().cpu()
        )

        w_np = (
            weights.detach()
            .cpu()
            .numpy()
        )

        history.append({
            "epoch":
                epoch,

            "loss":
                loss_value,

            **{
                f"w_{m}":
                    float(w)
                for m, w in zip(
                    TRRF_MODELS,
                    w_np
                )
            }
        })

        if (
            loss_value
            < best_loss
            - TRRF_CONFIG[
                "min_delta"
            ]
        ):

            best_loss = (
                loss_value
            )

            best_weights = (
                w_np.copy()
            )

            best_epoch = epoch

            wait = 0

        else:

            wait += 1

        if wait >= (
            TRRF_CONFIG[
                "patience"
            ]
        ):
            break


    if best_weights is None:

        best_weights = (
            np.ones(
                N_EXPERTS
            )
            / N_EXPERTS
        )

    return (
        best_weights,
        best_epoch,
        pd.DataFrame(
            history
        )
    )


# ============================================================
# 13C.4 Strict outer CV
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)

TRRF_OOF = np.zeros(
    len(y_all),
    dtype=float
)

TRRF_EQUAL = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []
weight_rows = []


for fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"TRRF FOLD "
        f"{fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 100
    )

    train_idx = np.where(
        fold_assignment
        != fold
    )[0]

    valid_idx = np.where(
        fold_assignment
        == fold
    )[0]

    y_train = y_all[
        train_idx
    ]

    y_valid = y_all[
        valid_idx
    ]


    # ========================================================
    # Strict fold-local calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}

    for model_name in (
        TRRF_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(
                model_name,
                train_idx,
                valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va


    P_train = np.column_stack([
        train_cal[m]
        for m in TRRF_MODELS
    ])

    P_valid = np.column_stack([
        valid_cal[m]
        for m in TRRF_MODELS
    ])


    # ========================================================
    # Equal-weight reference
    # ========================================================

    p_equal = (
        P_valid.mean(
            axis=1
        )
    )

    TRRF_EQUAL[
        valid_idx
    ] = p_equal


    # ========================================================
    # Learn ranking weights
    # ========================================================

    weights, best_epoch, history = (
        fit_trrf(
            P_train,
            y_train,
            seed=
                SEED
                + 1300
                + fold
        )
    )


    # ========================================================
    # Frozen validation prediction
    # ========================================================

    p_trrf = (
        weighted_logit_prediction(
            P_valid,
            weights
        )
    )

    TRRF_OOF[
        valid_idx
    ] = p_trrf


    equal_pr = (
        average_precision_score(
            y_valid,
            p_equal
        )
    )

    trrf_pr = (
        average_precision_score(
            y_valid,
            p_trrf
        )
    )


    fold_rows.append({

        "Fold":
            fold,

        "Best_Epoch":
            best_epoch,

        "Equal_PR":
            equal_pr,

        "TRRF_PR":
            trrf_pr,

        "Delta_PR":
            trrf_pr
            - equal_pr,

        "Weight_Entropy":
            float(
                -np.sum(
                    weights
                    * np.log(
                        weights
                        + 1e-12
                    )
                )
            ),
    })


    for m, w in zip(
        TRRF_MODELS,
        weights
    ):

        weight_rows.append({

            "Fold":
                fold,

            "Model":
                m,

            "Weight":
                float(w),
        })


    print(
        "Weights:"
    )

    for m, w in zip(
        TRRF_MODELS,
        weights
    ):

        print(
            f"  {m:<24} "
            f"{w:.4f}"
        )


    print(
        f"Equal PR : "
        f"{equal_pr:.6f}"
    )

    print(
        f"TRRF PR  : "
        f"{trrf_pr:.6f}"
    )

    print(
        f"Delta    : "
        f"{trrf_pr - equal_pr:+.6f}"
    )


# ============================================================
# 13C.5 Global evaluation
# ============================================================

equal_t, _, _ = (
    select_best_threshold(
        y_all,
        TRRF_EQUAL,
        objective="F1"
    )
)

trrf_t, _, _ = (
    select_best_threshold(
        y_all,
        TRRF_OOF,
        objective="F1"
    )
)


equal_metrics = (
    evaluate_binary_predictions(
        y_all,
        TRRF_EQUAL,
        threshold=equal_t
    )
)

trrf_metrics = (
    evaluate_binary_predictions(
        y_all,
        TRRF_OOF,
        threshold=trrf_t
    )
)


module13c_results = (
    pd.DataFrame([
        {
            "Method":
                "Core4Mean",
            **equal_metrics
        },
        {
            "Method":
                "TRRF",
            **trrf_metrics
        },
    ])
)


module13c_results[
    "Delta_PR_vs_Core"
] = (
    module13c_results[
        "PR_AUC"
    ]
    - equal_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13C — TRRF RESULTS"
)

print(
    "=" * 120
)

display(
    module13c_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13C.6 Fold stability
# ============================================================

module13c_folds = (
    pd.DataFrame(
        fold_rows
    )
)

module13c_weights = (
    pd.DataFrame(
        weight_rows
    )
)

print(
    "\n"
    + "=" * 120
)

print(
    "MODULE 13C — FOLD RESULTS"
)

print(
    "=" * 120
)

display(
    module13c_folds
)


print(
    "\n"
    + "=" * 120
)

print(
    "TRRF WEIGHTS")
print(
    "=" * 120
)

display(
    module13c_weights
)


# ============================================================
# 13C.7 Weight stability summary
# ============================================================

weight_summary = (
    module13c_weights
    .groupby(
        "Model"
    )[
        "Weight"
    ]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
    .reset_index()
    .sort_values(
        "mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "TRRF WEIGHT STABILITY"
)

print(
    "=" * 120
)

display(
    weight_summary
)


print(
    "\nMean fold Delta PR:",
    module13c_folds[
        "Delta_PR"
    ].mean()
)

print(
    "Positive folds:",
    int(
        (
            module13c_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13C.8 Save
# ============================================================

save_csv(
    module13c_results,
    TAB_DIR
    / "module13c_trrf_results.csv"
)

save_csv(
    module13c_folds,
    TAB_DIR
    / "module13c_trrf_fold_results.csv"
)

save_csv(
    module13c_weights,
    TAB_DIR
    / "module13c_trrf_weights.csv"
)

save_csv(
    weight_summary,
    TAB_DIR
    / "module13c_trrf_weight_stability.csv"
)


trrf_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        TRRF_EQUAL,

    "TRRF":
        TRRF_OOF,
})

save_csv(
    trrf_pred_df,
    PRED_DIR
    / "module13c_trrf_oof_predictions.csv"
)


print(
    "\nMODULE 13C COMPLETE."
)


TRRF FOLD 1/5
Weights:
  HistGradientBoosting     0.2537
  CatBoost                 0.2537
  AutoInt                  0.2389
  DeepFM_v2                0.2537
Equal PR : 0.380070
TRRF PR  : 0.379970
Delta    : -0.000100

TRRF FOLD 2/5
Weights:
  HistGradientBoosting     0.2490
  CatBoost                 0.2438
  AutoInt                  0.2647
  DeepFM_v2                0.2424
Equal PR : 0.382772
TRRF PR  : 0.382794
Delta    : +0.000022

TRRF FOLD 3/5
Weights:
  HistGradientBoosting     0.2444
  CatBoost                 0.2589
  AutoInt                  0.2484
  DeepFM_v2                0.2482
Equal PR : 0.378467
TRRF PR  : 0.378368
Delta    : -0.000099

TRRF FOLD 4/5
Weights:
  HistGradientBoosting     0.2536
  CatBoost                 0.2519
  AutoInt                  0.2389
  DeepFM_v2                0.2556
Equal PR : 0.372833
TRRF PR  : 0.372760
Delta    : -0.000072

TRRF FOLD 5/5
Weights:
  HistGradientBoosting     0.2496
  CatBoost                 0.2467
  AutoInt               

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,TRRF,0.380334,0.000007,0.850517,0.069709,0.235597,0.001972,0.351496,0.523805,0.420690,0.355963,0.22



MODULE 13C — FOLD RESULTS


,Fold,Best_Epoch,Equal_PR,TRRF_PR,Delta_PR,Weight_Entropy
0,0,2,0.380070,0.379970,-0.000100,1.385964
1,1,30,0.382772,0.382794,0.000022,1.385674
2,2,6,0.378467,0.378368,-0.000099,1.386063
3,3,43,0.372833,0.372760,-0.000072,1.385951
4,4,11,0.390072,0.390093,0.000021,1.386241



TRRF WEIGHTS


,Fold,Model,Weight
0,0,HistGradientBoosting,0.253692
1,0,CatBoost,0.253694
2,0,AutoInt,0.238920
3,0,DeepFM_v2,0.253694
4,1,HistGradientBoosting,0.249006
5,1,CatBoost,0.243850
6,1,AutoInt,0.264733
7,1,DeepFM_v2,0.242411
8,2,HistGradientBoosting,0.244449
9,2,CatBoost,0.258925



TRRF WEIGHT STABILITY


,Model,mean,std,min,max
1,CatBoost,0.251003,0.005933,0.243850,0.258925
3,HistGradientBoosting,0.250075,0.003833,0.244449,0.253692
2,DeepFM_v2,0.249944,0.005129,0.242411,0.255552
0,AutoInt,0.248978,0.010898,0.238920,0.264733



Mean fold Delta PR: -4.573981991184972e-05
Positive folds: 2 / 5

MODULE 13C COMPLETE.


In [21]:
# ============================================================
# MODULE 13D — HIDDEN SPECIALIST REGION DISCOVERY (HSRD)
# ============================================================
#
# IDEA:
#
# 1. Core4 is the default prediction.
# 2. For each candidate expert, learn where that expert tends
#    to move the score in the correct direction vs Core4.
# 3. Advantage models are INNER-CROSS-FITTED on outer-train.
# 4. Choose routing threshold + blend strength using ONLY
#    inner-OOF predictions from outer-train.
# 5. Refit region models on all outer-train.
# 6. Apply frozen routing policy to outer validation.
#
# HOLDOUT untouched.
# ============================================================

import numpy as np
import pandas as pd
import itertools

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score


# ============================================================
# 13D.1 Architecture
# ============================================================

HSRD_CORE = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

# Include strong models plus models with potentially different
# behavior. Do not include duplicate logistic variants.
HSRD_SPECIALISTS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
    "MLP_BCE",
    "MLP_Focal",
    "MLP_Focal_FGSM",
    "MLP_FGSM",
    "LightGBM",
    "TabNet",
]

HSRD_CONTEXT = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]

HSRD_CONFIG = {

    "inner_splits":
        3,

    # deliberately shallow/interpretable
    "tree_max_depth":
        3,

    "tree_min_samples_leaf":
        1200,

    # cap extreme advantage targets
    "advantage_clip":
        1.0,

    # candidate routing thresholds are quantiles of
    # predicted advantage
    "gate_quantiles": [
        0.50,
        0.60,
        0.70,
        0.80,
        0.85,
        0.90,
        0.95,
    ],

    # how far to move toward selected specialist
    "lambda_grid": [
        0.25,
        0.50,
        0.75,
        1.00,
    ],

    # avoid microscopic regions
    "min_route_fraction":
        0.02,

    # avoid replacing core for huge fractions
    "max_route_fraction":
        0.40,
}


# ============================================================
# 13D.2 Helpers
# ============================================================

def logit_np(p):

    p = np.clip(
        np.asarray(p, dtype=float),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


def sigmoid_np(z):

    z = np.clip(
        z,
        -30,
        30
    )

    return 1.0 / (
        1.0 + np.exp(-z)
    )


# ============================================================
# 13D.3 Region-discovery features
# ============================================================

def build_hsrd_features(
    X_original,
    calibrated_dict
):

    out = pd.DataFrame(
        index=np.arange(
            len(X_original)
        )
    )

    # --------------------------------------------------------
    # clinical/context
    # --------------------------------------------------------

    for c in HSRD_CONTEXT:

        out[
            f"x_{c}"
        ] = (
            X_original[c]
            .to_numpy(
                dtype=np.float32
            )
        )

    # --------------------------------------------------------
    # Core statistics
    # --------------------------------------------------------

    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in HSRD_CORE
    ])

    core_mean = (
        core_matrix.mean(axis=1)
    )

    out["core_prob"] = core_mean
    out["core_logit"] = logit_np(core_mean)

    out["core_std"] = (
        core_matrix.std(axis=1)
    )

    out["core_range"] = (
        core_matrix.max(axis=1)
        -
        core_matrix.min(axis=1)
    )

    out["core_min"] = (
        core_matrix.min(axis=1)
    )

    out["core_max"] = (
        core_matrix.max(axis=1)
    )

    out["core_entropy"] = (
        binary_entropy(
            core_mean
        )
    )

    # --------------------------------------------------------
    # Expert scores + deviations
    # --------------------------------------------------------

    for m in HSRD_SPECIALISTS:

        p = np.asarray(
            calibrated_dict[m],
            dtype=float
        )

        out[f"p_{m}"] = p

        out[
            f"delta_{m}"
        ] = (
            p - core_mean
        )

        out[
            f"absdelta_{m}"
        ] = np.abs(
            p - core_mean
        )

    return out


# ============================================================
# 13D.4 Signed expert advantage
# ============================================================

def expert_advantage_target(
    y,
    core_p,
    expert_p
):

    y = np.asarray(
        y,
        dtype=float
    )

    sign = (
        2.0 * y - 1.0
    )

    advantage = (
        sign
        * (
            logit_np(expert_p)
            -
            logit_np(core_p)
        )
    )

    return np.clip(
        advantage,
        -HSRD_CONFIG[
            "advantage_clip"
        ],
        HSRD_CONFIG[
            "advantage_clip"
        ]
    )


# ============================================================
# 13D.5 Region model
# ============================================================

def make_region_model(
    seed
):

    return DecisionTreeRegressor(

        max_depth=
            HSRD_CONFIG[
                "tree_max_depth"
            ],

        min_samples_leaf=
            HSRD_CONFIG[
                "tree_min_samples_leaf"
            ],

        criterion=
            "squared_error",

        random_state=
            seed,
    )


# ============================================================
# 13D.6 Inner cross-fitted advantage prediction
# ============================================================

def inner_oof_advantage_predictions(
    X_meta,
    y_train,
    core_train,
    train_cal,
    outer_fold
):

    n = len(
        y_train
    )

    predicted_advantage = {
        m:
            np.zeros(
                n,
                dtype=np.float64
            )
        for m in HSRD_SPECIALISTS
    }

    inner_cv = StratifiedKFold(

        n_splits=
            HSRD_CONFIG[
                "inner_splits"
            ],

        shuffle=True,

        random_state=
            SEED + 1700
            + outer_fold,
    )


    for inner_fold, (
        inner_train_idx,
        inner_valid_idx
    ) in enumerate(
        inner_cv.split(
            X_meta,
            y_train
        )
    ):

        for specialist in (
            HSRD_SPECIALISTS
        ):

            target = (
                expert_advantage_target(

                    y_train[
                        inner_train_idx
                    ],

                    core_train[
                        inner_train_idx
                    ],

                    train_cal[
                        specialist
                    ][
                        inner_train_idx
                    ]
                )
            )

            region_model = (
                make_region_model(

                    SEED
                    + outer_fold * 100
                    + inner_fold * 10
                    + HSRD_SPECIALISTS.index(
                        specialist
                    )
                )
            )

            region_model.fit(

                X_meta.iloc[
                    inner_train_idx
                ],

                target
            )

            predicted_advantage[
                specialist
            ][
                inner_valid_idx
            ] = (
                region_model.predict(

                    X_meta.iloc[
                        inner_valid_idx
                    ]
                )
            )

    return predicted_advantage


# ============================================================
# 13D.7 Convert specialist predictions to routing policy
# ============================================================

def construct_routed_prediction(
    core_p,
    specialist_prob_dict,
    predicted_advantage_dict,
    threshold,
    lam
):

    specialists = list(
        predicted_advantage_dict.keys()
    )

    advantage_matrix = np.column_stack([
        predicted_advantage_dict[m]
        for m in specialists
    ])

    # specialist with highest predicted benefit
    best_expert_idx = (
        np.argmax(
            advantage_matrix,
            axis=1
        )
    )

    best_advantage = (
        np.max(
            advantage_matrix,
            axis=1
        )
    )

    route_mask = (
        best_advantage
        >= threshold
    )

    core_logit = (
        logit_np(
            core_p
        )
    )

    final_logit = (
        core_logit.copy()
    )

    selected_expert = np.array(
        ["CORE"] * len(core_p),
        dtype=object
    )

    for j, specialist in enumerate(
        specialists
    ):

        m = (
            route_mask
            &
            (
                best_expert_idx
                == j
            )
        )

        if not np.any(m):
            continue

        expert_logit = (
            logit_np(
                specialist_prob_dict[
                    specialist
                ][m]
            )
        )

        final_logit[m] = (

            core_logit[m]

            + lam
            * (
                expert_logit
                - core_logit[m]
            )
        )

        selected_expert[m] = (
            specialist
        )

    p_final = clip_probs(
        sigmoid_np(
            final_logit
        )
    )

    return (
        p_final,
        route_mask,
        selected_expert,
        best_advantage,
    )


# ============================================================
# 13D.8 Select threshold + lambda on INNER OOF only
# ============================================================

def select_routing_policy(
    y_train,
    core_train,
    train_cal,
    predicted_advantage
):

    advantage_matrix = np.column_stack([
        predicted_advantage[m]
        for m in HSRD_SPECIALISTS
    ])

    best_advantage = (
        advantage_matrix.max(
            axis=1
        )
    )

    rows = []

    for q in (
        HSRD_CONFIG[
            "gate_quantiles"
        ]
    ):

        threshold = float(
            np.quantile(
                best_advantage,
                q
            )
        )

        for lam in (
            HSRD_CONFIG[
                "lambda_grid"
            ]
        ):

            (
                p_candidate,
                route_mask,
                selected,
                _
            ) = construct_routed_prediction(

                core_train,
                train_cal,
                predicted_advantage,
                threshold,
                lam
            )

            route_fraction = float(
                route_mask.mean()
            )

            if (
                route_fraction
                <
                HSRD_CONFIG[
                    "min_route_fraction"
                ]
                or
                route_fraction
                >
                HSRD_CONFIG[
                    "max_route_fraction"
                ]
            ):
                continue

            pr = (
                average_precision_score(
                    y_train,
                    p_candidate
                )
            )

            pm = probability_metrics(
                y_train,
                p_candidate
            )

            rows.append({

                "Quantile":
                    q,

                "Threshold":
                    threshold,

                "Lambda":
                    lam,

                "Route_Fraction":
                    route_fraction,

                "PR_AUC":
                    pr,

                "ROC_AUC":
                    pm[
                        "ROC_AUC"
                    ],

                "Brier":
                    pm[
                        "Brier"
                    ],

                "NLL":
                    pm[
                        "NLL"
                    ],

                "ECE":
                    pm[
                        "ECE"
                    ],
            })


    search_df = (
        pd.DataFrame(
            rows
        )
    )

    if len(
        search_df
    ) == 0:

        raise RuntimeError(
            "No eligible HSRD routing policies."
        )


    # --------------------------------------------------------
    # PR first; Brier second
    # --------------------------------------------------------

    search_df = (
        search_df
        .sort_values(

            [
                "PR_AUC",
                "Brier",
                "Route_Fraction",
            ],

            ascending=[
                False,
                True,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return (
        search_df.iloc[0],
        search_df
    )


# ============================================================
# 13D.9 Outer CV buffers
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)

HSRD_CORE_OOF = np.zeros(
    len(y_all),
    dtype=float
)

HSRD_OOF = np.zeros(
    len(y_all),
    dtype=float
)

HSRD_ROUTE = np.zeros(
    len(y_all),
    dtype=bool
)

HSRD_ADVANTAGE = np.zeros(
    len(y_all),
    dtype=float
)

HSRD_EXPERT = np.array(
    [""] * len(y_all),
    dtype=object
)

outer_rows = []
policy_rows = []
expert_rows = []


# ============================================================
# 13D.10 Strict outer folds
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 105
    )

    print(
        f"HSRD OUTER FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 105
    )

    outer_train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    outer_valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    y_train = (
        y_all[
            outer_train_idx
        ]
    )

    y_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # Fold-local calibrations
    # ========================================================

    train_cal = {}
    valid_cal = {}

    for model_name in (
        HSRD_SPECIALISTS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(

                model_name,
                outer_train_idx,
                outer_valid_idx
            )
        )

        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va


    # ========================================================
    # Core consensus
    # ========================================================

    core_train = np.mean(
        np.column_stack([
            train_cal[m]
            for m in HSRD_CORE
        ]),
        axis=1
    )

    core_valid = np.mean(
        np.column_stack([
            valid_cal[m]
            for m in HSRD_CORE
        ]),
        axis=1
    )


    # ========================================================
    # Region-discovery features
    # ========================================================

    X_train_meta = (
        build_hsrd_features(

            X_dev.iloc[
                outer_train_idx
            ].reset_index(
                drop=True
            ),

            train_cal
        )
    )

    X_valid_meta = (
        build_hsrd_features(

            X_dev.iloc[
                outer_valid_idx
            ].reset_index(
                drop=True
            ),

            valid_cal
        )
    )


    # ========================================================
    # INNER OOF advantage prediction
    # ========================================================

    predicted_adv_train = (
        inner_oof_advantage_predictions(

            X_train_meta,
            y_train,
            core_train,
            train_cal,
            outer_fold
        )
    )


    # ========================================================
    # Select gate + lambda using INNER OOF only
    # ========================================================

    best_policy, search_df = (
        select_routing_policy(

            y_train,
            core_train,
            train_cal,
            predicted_adv_train
        )
    )

    selected_threshold = float(
        best_policy[
            "Threshold"
        ]
    )

    selected_lambda = float(
        best_policy[
            "Lambda"
        ]
    )


    # ========================================================
    # Refit advantage models on COMPLETE OUTER TRAIN
    # ========================================================

    predicted_adv_valid = {}

    for specialist in (
        HSRD_SPECIALISTS
    ):

        target = (
            expert_advantage_target(

                y_train,
                core_train,
                train_cal[
                    specialist
                ]
            )
        )

        model = make_region_model(

            SEED
            + 2500
            + outer_fold * 20
            + HSRD_SPECIALISTS.index(
                specialist
            )
        )

        model.fit(
            X_train_meta,
            target
        )

        predicted_adv_valid[
            specialist
        ] = (
            model.predict(
                X_valid_meta
            )
        )


    # ========================================================
    # Apply frozen policy to OUTER VALID
    # ========================================================

    (
        p_valid_hsrd,
        route_mask,
        selected_expert,
        best_advantage
    ) = construct_routed_prediction(

        core_valid,
        valid_cal,
        predicted_adv_valid,
        selected_threshold,
        selected_lambda
    )


    HSRD_CORE_OOF[
        outer_valid_idx
    ] = core_valid

    HSRD_OOF[
        outer_valid_idx
    ] = p_valid_hsrd

    HSRD_ROUTE[
        outer_valid_idx
    ] = route_mask

    HSRD_ADVANTAGE[
        outer_valid_idx
    ] = best_advantage

    HSRD_EXPERT[
        outer_valid_idx
    ] = selected_expert


    # ========================================================
    # Fold results
    # ========================================================

    core_pm = probability_metrics(
        y_valid,
        core_valid
    )

    hsrd_pm = probability_metrics(
        y_valid,
        p_valid_hsrd
    )


    outer_rows.append({

        "Fold":
            outer_fold,

        "Selected_Threshold":
            selected_threshold,

        "Selected_Lambda":
            selected_lambda,

        "Train_Route_Fraction":
            float(
                best_policy[
                    "Route_Fraction"
                ]
            ),

        "Valid_Route_Fraction":
            float(
                route_mask.mean()
            ),

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "HSRD_PR":
            hsrd_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            hsrd_pm[
                "PR_AUC"
            ]
            - core_pm[
                "PR_AUC"
            ],

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "HSRD_ROC":
            hsrd_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "HSRD_Brier":
            hsrd_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "HSRD_ECE":
            hsrd_pm[
                "ECE"
            ],
    })


    # --------------------------------------------------------
    # Routing frequency
    # --------------------------------------------------------

    for specialist in (
        HSRD_SPECIALISTS
    ):

        expert_rows.append({

            "Fold":
                outer_fold,

            "Expert":
                specialist,

            "Routed_N":
                int(
                    np.sum(
                        selected_expert
                        == specialist
                    )
                ),

            "Routed_Fraction":
                float(
                    np.mean(
                        selected_expert
                        == specialist
                    )
                ),
        })


    # save policy search
    tmp = (
        search_df.copy()
    )

    tmp[
        "Outer_Fold"
    ] = outer_fold

    policy_rows.append(
        tmp
    )


    print(
        f"Threshold      : "
        f"{selected_threshold:.6f}"
    )

    print(
        f"Lambda         : "
        f"{selected_lambda:.2f}"
    )

    print(
        f"Valid routed   : "
        f"{route_mask.mean():.3f}"
    )

    print(
        f"Core PR        : "
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        f"HSRD PR        : "
        f"{hsrd_pm['PR_AUC']:.6f}"
    )

    print(
        f"Delta          : "
        f"{hsrd_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )


# ============================================================
# 13D.11 Global evaluation
# ============================================================

core_t, _, _ = (
    select_best_threshold(
        y_all,
        HSRD_CORE_OOF,
        objective="F1"
    )
)

hsrd_t, _, _ = (
    select_best_threshold(
        y_all,
        HSRD_OOF,
        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(
        y_all,
        HSRD_CORE_OOF,
        threshold=core_t
    )
)

hsrd_metrics = (
    evaluate_binary_predictions(
        y_all,
        HSRD_OOF,
        threshold=hsrd_t
    )
)


module13d_results = (
    pd.DataFrame([
        {
            "Method":
                "Core4Mean",
            **core_metrics
        },
        {
            "Method":
                "HSRD",
            **hsrd_metrics
        },
    ])
)


module13d_results[
    "Delta_PR_vs_Core"
] = (
    module13d_results[
        "PR_AUC"
    ]
    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 13D — HIDDEN SPECIALIST REGION DISCOVERY"
)

print(
    "=" * 125
)

display(
    module13d_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13D.12 Fold behavior
# ============================================================

module13d_folds = (
    pd.DataFrame(
        outer_rows
    )
)

print(
    "\n"
    + "=" * 125
)

print(
    "HSRD FOLD RESULTS"
)

print(
    "=" * 125
)

display(
    module13d_folds
)


print(
    "\nMean fold ΔPR:",
    float(
        module13d_folds[
            "Delta_PR"
        ].mean()
    )
)

print(
    "Positive folds:",
    int(
        (
            module13d_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13D.13 Expert-routing behavior
# ============================================================

routing_table = (
    pd.DataFrame(
        expert_rows
    )
)

routing_summary = (
    routing_table
    .groupby(
        "Expert"
    )
    .agg({

        "Routed_N":
            "sum",

        "Routed_Fraction":
            "mean",
    })
    .reset_index()
    .sort_values(
        "Routed_N",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "DISCOVERED SPECIALIST ROUTING"
)

print(
    "=" * 125
)

display(
    routing_summary
)


# ============================================================
# 13D.14 Routed vs non-routed diagnostics
# ============================================================

route_diagnostics = []

for routed_value, name in [
    (True, "Routed"),
    (False, "Core_kept"),
]:

    m = (
        HSRD_ROUTE
        == routed_value
    )

    if m.sum() == 0:
        continue

    route_diagnostics.append({

        "Region":
            name,

        "N":
            int(
                m.sum()
            ),

        "Fraction":
            float(
                m.mean()
            ),

        "Prevalence":
            float(
                y_all[m].mean()
            ),

        "Core_PR":
            (
                average_precision_score(
                    y_all[m],
                    HSRD_CORE_OOF[m]
                )
                if len(
                    np.unique(
                        y_all[m]
                    )
                ) > 1
                else np.nan
            ),

        "HSRD_PR":
            (
                average_precision_score(
                    y_all[m],
                    HSRD_OOF[m]
                )
                if len(
                    np.unique(
                        y_all[m]
                    )
                ) > 1
                else np.nan
            ),
    })


route_diagnostic_df = (
    pd.DataFrame(
        route_diagnostics
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "ROUTED REGION DIAGNOSTICS"
)

print(
    "=" * 125
)

display(
    route_diagnostic_df
)


# ============================================================
# 13D.15 Save
# ============================================================

save_csv(
    module13d_results,
    TAB_DIR
    / "module13d_hsrd_results.csv"
)

save_csv(
    module13d_folds,
    TAB_DIR
    / "module13d_hsrd_fold_results.csv"
)

save_csv(
    routing_summary,
    TAB_DIR
    / "module13d_hsrd_routing_summary.csv"
)

save_csv(
    route_diagnostic_df,
    TAB_DIR
    / "module13d_hsrd_region_diagnostics.csv"
)

save_csv(
    pd.concat(
        policy_rows,
        ignore_index=True
    ),
    TAB_DIR
    / "module13d_hsrd_policy_search.csv"
)


hsrd_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        HSRD_CORE_OOF,

    "HSRD":
        HSRD_OOF,

    "Routed":
        HSRD_ROUTE.astype(int),

    "PredictedAdvantage":
        HSRD_ADVANTAGE,

    "SelectedExpert":
        HSRD_EXPERT,
})


save_csv(
    hsrd_pred_df,
    PRED_DIR
    / "module13d_hsrd_oof_predictions.csv"
)


print(
    "\nMODULE 13D COMPLETE."
)


HSRD OUTER FOLD 1/5
Threshold      : 0.798942
Lambda         : 0.25
Valid routed   : 0.000
Core PR        : 0.380070
HSRD PR        : 0.380070
Delta          : +0.000000

HSRD OUTER FOLD 2/5
Threshold      : 0.761598
Lambda         : 0.25
Valid routed   : 0.035
Core PR        : 0.382772
HSRD PR        : 0.382788
Delta          : +0.000016

HSRD OUTER FOLD 3/5
Threshold      : 0.776002
Lambda         : 0.25
Valid routed   : 0.015
Core PR        : 0.378467
HSRD PR        : 0.378464
Delta          : -0.000003

HSRD OUTER FOLD 4/5
Threshold      : 0.705000
Lambda         : 0.25
Valid routed   : 0.029
Core PR        : 0.372833
HSRD PR        : 0.372833
Delta          : -0.000000

HSRD OUTER FOLD 5/5
Threshold      : 0.674907
Lambda         : 0.25
Valid routed   : 0.000
Core PR        : 0.390072
HSRD PR        : 0.390072
Delta          : +0.000000

MODULE 13D — HIDDEN SPECIALIST REGION DISCOVERY


,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,HSRD,0.380330,0.000003,0.850563,0.069706,0.235581,0.002363,0.350507,0.526054,0.420702,0.356064,0.22



HSRD FOLD RESULTS


,Fold,Selected_Threshold,Selected_Lambda,Train_Route_Fraction,Valid_Route_Fraction,Core_PR,HSRD_PR,Delta_PR,Core_ROC,HSRD_ROC,Core_Brier,HSRD_Brier,Core_ECE,HSRD_ECE
0,0,0.798942,0.25,0.061027,0.000000,0.380070,0.380070,0.000000e+00,0.852660,0.852660,0.069637,0.069637,0.003499,0.003499
1,1,0.761598,0.25,0.075587,0.035034,0.382772,0.382788,1.585696e-05,0.850766,0.850822,0.069684,0.069682,0.004297,0.004258
2,2,0.776002,0.25,0.057276,0.014758,0.378467,0.378464,-2.788540e-06,0.849030,0.849001,0.069827,0.069827,0.002486,0.002499
3,3,0.705000,0.25,0.074128,0.028801,0.372833,0.372833,-5.921238e-08,0.849083,0.849082,0.070081,0.070081,0.005053,0.005058
4,4,0.674907,0.25,0.059437,0.000000,0.390072,0.390072,0.000000e+00,0.851760,0.851760,0.069301,0.069301,0.005318,0.005318



Mean fold ΔPR: 2.601840820282142e-06
Positive folds: 1 / 5

DISCOVERED SPECIALIST ROUTING


,Expert,Routed_N,Routed_Fraction
9,TabNet,2021,0.009958
7,MLP_Focal,1169,0.005760
1,CatBoost,0,0.000000
0,AutoInt,0,0.000000
2,DeepFM_v2,0,0.000000
3,HistGradientBoosting,0,0.000000
5,MLP_BCE,0,0.000000
4,LightGBM,0,0.000000
6,MLP_FGSM,0,0.000000
8,MLP_Focal_FGSM,0,0.000000



ROUTED REGION DIAGNOSTICS


,Region,N,Fraction,Prevalence,Core_PR,HSRD_PR
0,Routed,3190,0.015719,0.002821,0.006855,0.007553
1,Core_kept,199754,0.984281,0.095643,0.380462,0.380462



MODULE 13D COMPLETE.


In [22]:
# ============================================================
# MODULE 13E — PCEA
# Patient-Conditioned Cross-Expert Attention
# ============================================================
#
# Architecture:
#
# Clinical feature groups
#       ↓
# Group-attention patient representation
#       ↓
# Patient query  →  Cross-expert attention
#                        ↓
#       HGB / CatBoost / AutoInt / DeepFM
#                        ↓
#          bounded residual fusion
#                        ↓
#                 final probability
#
# Attention model:
# - inner split selects epoch
# - fresh refit on complete outer-training data
# - untouched outer fold evaluated once
#
# HOLDOUT IS NOT USED.
# ============================================================


import gc
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import average_precision_score


# ============================================================
# 13E.1 Configuration
# ============================================================

PCEA_EXPERTS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]


PCEA_GROUPS = {

    "cardiometabolic": [
        "HighBP",
        "HighChol",
        "CholCheck",
        "BMI",
        "Diabetes",
    ],

    "vascular_history": [
        "Stroke",
    ],

    "general_functional": [
        "GenHlth",
        "MentHlth",
        "PhysHlth",
        "DiffWalk",
    ],

    "lifestyle": [
        "Smoker",
        "PhysActivity",
        "Fruits",
        "Veggies",
        "HvyAlcoholConsump",
    ],

    "demographic_access": [
        "Sex",
        "Age",
        "Education",
        "Income",
        "AnyHealthcare",
        "NoDocbcCost",
    ],
}


# verify all 21 predictors covered exactly once
_flat_group_features = (
    sum(
        PCEA_GROUPS.values(),
        []
    )
)

assert len(
    _flat_group_features
) == 21

assert len(
    set(
        _flat_group_features
    )
) == 21


PCEA_CONFIG = {

    "d_model":
        32,

    "group_hidden":
        32,

    "expert_hidden":
        32,

    "dropout":
        0.15,

    "batch_size":
        2048,

    "max_epochs":
        25,

    "patience":
        4,

    "inner_es_fraction":
        0.15,

    "lr":
        5e-4,

    "weight_decay":
        2e-4,

    # maximum logit movement away from Core4
    "max_logit_correction":
        0.25,

    # penalize unnecessary deviation
    "residual_penalty":
        0.05,

    # mild entropy floor discourages immediate
    # one-hot expert collapse
    "expert_entropy_penalty":
        0.005,
}


DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print(
    "PCEA device:",
    DEVICE
)


# ============================================================
# 13E.2 Reproducibility
# ============================================================

def pcea_seed_everything(seed):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 13E.3 Numerical helpers
# ============================================================

def pcea_logit_np(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=np.float64
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


def pcea_sigmoid_np(z):

    z = np.clip(
        z,
        -30,
        30
    )

    return (
        1.0
        /
        (
            1.0
            + np.exp(-z)
        )
    )


# ============================================================
# 13E.4 Expert descriptor builder
# ============================================================
#
# Each expert token contains:
#
# p_k
# logit(p_k)
# deviation from Core4
# absolute deviation
# Core4 probability
# Core disagreement
# Core entropy
#
# Model identity is separately represented by a learned
# expert embedding.
# ============================================================

def build_pcea_expert_features(
    calibrated_dict
):

    P = np.column_stack([
        calibrated_dict[m]
        for m in PCEA_EXPERTS
    ])

    core = (
        P.mean(
            axis=1
        )
    )

    core_std = (
        P.std(
            axis=1
        )
    )

    core_entropy = (
        binary_entropy(
            core
        )
    )

    features = []

    for j, model_name in enumerate(
        PCEA_EXPERTS
    ):

        p = (
            P[:, j]
        )

        delta = (
            p - core
        )

        expert_features = np.column_stack([

            p,

            pcea_logit_np(
                p
            ),

            delta,

            np.abs(
                delta
            ),

            core,

            core_std,

            core_entropy,
        ])

        features.append(
            expert_features
        )

    # N x 4 x 7
    expert_tensor = np.stack(
        features,
        axis=1
    ).astype(
        np.float32
    )

    return (
        expert_tensor,
        core.astype(
            np.float32
        ),
    )


# ============================================================
# 13E.5 Clinical matrix builder
# ============================================================

ALL_CLINICAL_COLUMNS = list(
    X_dev.columns
)

assert set(
    _flat_group_features
).issubset(
    set(
        ALL_CLINICAL_COLUMNS
    )
)


def clinical_matrix_from_indices(
    row_idx
):

    return (
        X_dev
        .iloc[
            row_idx
        ][
            _flat_group_features
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


# ============================================================
# 13E.6 Group indices
# ============================================================

GROUP_NAMES = list(
    PCEA_GROUPS.keys()
)

GROUP_COLUMN_INDICES = {}

start = 0

for group_name in GROUP_NAMES:

    width = len(
        PCEA_GROUPS[
            group_name
        ]
    )

    GROUP_COLUMN_INDICES[
        group_name
    ] = list(
        range(
            start,
            start + width
        )
    )

    start += width


print(
    "PCEA clinical groups:"
)

for name in GROUP_NAMES:

    print(
        f"  {name:<24}",
        PCEA_GROUPS[
            name
        ]
    )


# ============================================================
# 13E.7 Dataset
# ============================================================

class PCEADataset(
    Dataset
):

    def __init__(
        self,
        X_clinical,
        X_expert,
        core_prob,
        y
    ):

        self.X_clinical = torch.tensor(
            X_clinical,
            dtype=torch.float32
        )

        self.X_expert = torch.tensor(
            X_expert,
            dtype=torch.float32
        )

        self.core_prob = torch.tensor(
            core_prob,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(
            self.y
        )

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X_clinical[
                idx
            ],

            self.X_expert[
                idx
            ],

            self.core_prob[
                idx
            ],

            self.y[
                idx
            ],
        )


# ============================================================
# 13E.8 PCEA architecture
# ============================================================

class PCEANet(
    nn.Module
):

    def __init__(
        self,
        group_indices,
        expert_feature_dim,
        n_experts,
        d_model=32,
        dropout=0.15,
        max_correction=0.25,
    ):

        super().__init__()

        self.group_names = list(
            group_indices.keys()
        )

        self.group_indices = (
            group_indices
        )

        self.d_model = (
            d_model
        )

        self.max_correction = (
            max_correction
        )

        # ----------------------------------------------------
        # A. One encoder per clinical feature group
        # ----------------------------------------------------

        self.group_encoders = (
            nn.ModuleDict()
        )

        for group_name in (
            self.group_names
        ):

            input_dim = len(
                group_indices[
                    group_name
                ]
            )

            self.group_encoders[
                group_name
            ] = nn.Sequential(

                nn.Linear(
                    input_dim,
                    PCEA_CONFIG[
                        "group_hidden"
                    ]
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                ),

                nn.Linear(
                    PCEA_CONFIG[
                        "group_hidden"
                    ],
                    d_model
                ),
            )


        # learned identity for each clinical group
        self.group_identity = (
            nn.Parameter(
                torch.randn(
                    len(
                        self.group_names
                    ),
                    d_model
                )
                * 0.02
            )
        )


        # ----------------------------------------------------
        # B. Group-attention scorer
        # ----------------------------------------------------

        self.group_score = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    d_model
                ),

                nn.Tanh(),

                nn.Linear(
                    d_model,
                    1
                ),
            )
        )


        # ----------------------------------------------------
        # C. Patient-query projection
        # ----------------------------------------------------

        self.patient_query = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    d_model
                ),

                nn.GELU(),

                nn.LayerNorm(
                    d_model
                ),
            )
        )


        # ----------------------------------------------------
        # D. Expert token encoder
        # ----------------------------------------------------

        self.expert_encoder = (
            nn.Sequential(

                nn.Linear(
                    expert_feature_dim,
                    PCEA_CONFIG[
                        "expert_hidden"
                    ]
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                ),

                nn.Linear(
                    PCEA_CONFIG[
                        "expert_hidden"
                    ],
                    d_model
                ),
            )
        )


        # learned expert identity
        self.expert_identity = (
            nn.Parameter(
                torch.randn(
                    n_experts,
                    d_model
                )
                * 0.02
            )
        )


        # ----------------------------------------------------
        # E. Keys and values
        # ----------------------------------------------------

        self.key_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.value_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )


        # ----------------------------------------------------
        # F. Patient-specific correction gate
        # ----------------------------------------------------

        self.correction_gate = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    16
                ),

                nn.GELU(),

                nn.Linear(
                    16,
                    1
                ),

                nn.Sigmoid(),
            )
        )


    def forward(
        self,
        clinical_x,
        expert_x,
        core_prob
    ):

        batch_size = (
            clinical_x.shape[0]
        )

        # ====================================================
        # Group attention
        # ====================================================

        group_tokens = []

        for g_idx, group_name in enumerate(
            self.group_names
        ):

            cols = (
                self.group_indices[
                    group_name
                ]
            )

            xg = (
                clinical_x[
                    :,
                    cols
                ]
            )

            token = (
                self.group_encoders[
                    group_name
                ](
                    xg
                )
            )

            token = (
                token
                +
                self.group_identity[
                    g_idx
                ]
            )

            group_tokens.append(
                token
            )


        # B x G x D
        group_tokens = (
            torch.stack(
                group_tokens,
                dim=1
            )
        )


        group_logits = (
            self.group_score(
                group_tokens
            )
            .squeeze(-1)
        )


        group_attention = (
            torch.softmax(
                group_logits,
                dim=1
            )
        )


        patient_repr = (
            group_tokens
            *
            group_attention.unsqueeze(
                -1
            )
        ).sum(
            dim=1
        )


        query = (
            self.patient_query(
                patient_repr
            )
        )


        # ====================================================
        # Expert attention
        # ====================================================

        expert_tokens = (
            self.expert_encoder(
                expert_x
            )
        )


        expert_tokens = (
            expert_tokens
            +
            self.expert_identity[
                None,
                :,
                :
            ]
        )


        keys = (
            self.key_proj(
                expert_tokens
            )
        )

        values = (
            self.value_proj(
                expert_tokens
            )
        )


        # dot product patient query -> expert keys
        attention_logits = (
            (
                keys
                *
                query.unsqueeze(1)
            )
            .sum(
                dim=-1
            )
            /
            np.sqrt(
                self.d_model
            )
        )


        expert_attention = (
            torch.softmax(
                attention_logits,
                dim=1
            )
        )


        # ====================================================
        # Attention-weighted expert logits
        # ====================================================

        expert_prob = (
            expert_x[
                :,
                :,
                0
            ]
        )

        expert_prob = torch.clamp(
            expert_prob,
            1e-6,
            1.0 - 1e-6
        )


        expert_logit = torch.log(
            expert_prob
            /
            (
                1.0
                - expert_prob
            )
        )


        core_prob_safe = torch.clamp(
            core_prob,
            1e-6,
            1.0 - 1e-6
        )


        core_logit = torch.log(
            core_prob_safe
            /
            (
                1.0
                - core_prob_safe
            )
        )


        attended_logit = (
            expert_attention
            *
            expert_logit
        ).sum(
            dim=1
        )


        raw_delta = (
            attended_logit
            -
            core_logit
        )


        # bounded move toward attention-weighted expert score
        bounded_delta = (
            self.max_correction
            *
            torch.tanh(
                raw_delta
                /
                self.max_correction
            )
        )


        correction_strength = (
            self.correction_gate(
                patient_repr
            )
            .squeeze(-1)
        )


        correction = (
            correction_strength
            *
            bounded_delta
        )


        final_logit = (
            core_logit
            +
            correction
        )


        return {

            "logit":
                final_logit,

            "correction":
                correction,

            "expert_attention":
                expert_attention,

            "group_attention":
                group_attention,

            "correction_strength":
                correction_strength,

            "patient_repr":
                patient_repr,
        }


# ============================================================
# 13E.9 Training for fixed number of epochs
# ============================================================

def train_pcea_fixed_epochs(
    X_clinical,
    X_expert,
    core_prob,
    y,
    epochs,
    seed,
    verbose=False,
):

    pcea_seed_everything(
        seed
    )

    dataset = PCEADataset(

        X_clinical,
        X_expert,
        core_prob,
        y
    )

    loader = DataLoader(

        dataset,

        batch_size=
            PCEA_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = PCEANet(

        group_indices=
            GROUP_COLUMN_INDICES,

        expert_feature_dim=
            X_expert.shape[2],

        n_experts=
            len(
                PCEA_EXPERTS
            ),

        d_model=
            PCEA_CONFIG[
                "d_model"
            ],

        dropout=
            PCEA_CONFIG[
                "dropout"
            ],

        max_correction=
            PCEA_CONFIG[
                "max_logit_correction"
            ],

    ).to(
        DEVICE
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                PCEA_CONFIG[
                    "lr"
                ],

            weight_decay=
                PCEA_CONFIG[
                    "weight_decay"
                ],
        )
    )


    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        epoch_loss = 0.0
        seen = 0


        for (
            clinical_b,
            expert_b,
            core_b,
            y_b
        ) in loader:

            clinical_b = clinical_b.to(
                DEVICE
            )

            expert_b = expert_b.to(
                DEVICE
            )

            core_b = core_b.to(
                DEVICE
            )

            y_b = y_b.to(
                DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            output = model(

                clinical_b,
                expert_b,
                core_b
            )


            # ------------------------------------------------
            # Primary BCE
            # ------------------------------------------------

            bce_loss = (
                F.binary_cross_entropy_with_logits(

                    output[
                        "logit"
                    ],

                    y_b
                )
            )


            # ------------------------------------------------
            # Preserve strong Core4
            # ------------------------------------------------

            residual_loss = (
                output[
                    "correction"
                ]
                .pow(2)
                .mean()
            )


            # ------------------------------------------------
            # Prevent premature one-hot attention collapse
            # ------------------------------------------------

            attn = (
                output[
                    "expert_attention"
                ]
            )

            entropy = -(
                attn
                *
                torch.log(
                    attn
                    + 1e-8
                )
            ).sum(
                dim=1
            ).mean()


            # maximizing entropy => subtract entropy
            loss = (

                bce_loss

                + PCEA_CONFIG[
                    "residual_penalty"
                ]
                * residual_loss

                - PCEA_CONFIG[
                    "expert_entropy_penalty"
                ]
                * entropy
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                3.0
            )


            optimizer.step()


            n = len(
                y_b
            )

            epoch_loss += (
                float(
                    loss.detach()
                    .cpu()
                )
                * n
            )

            seen += n


        if verbose:

            print(
                f"epoch={epoch:02d} "
                f"loss="
                f"{epoch_loss/seen:.6f}"
            )


    return model


# ============================================================
# 13E.10 Prediction helper
# ============================================================

def predict_pcea(
    model,
    X_clinical,
    X_expert,
    core_prob,
    batch_size=4096,
):

    model.eval()

    n = len(
        core_prob
    )

    predictions = []
    corrections = []
    expert_attentions = []
    group_attentions = []
    correction_strengths = []


    with torch.no_grad():

        for start in range(
            0,
            n,
            batch_size
        ):

            end = min(
                start + batch_size,
                n
            )


            clinical_b = torch.tensor(

                X_clinical[
                    start:end
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            expert_b = torch.tensor(

                X_expert[
                    start:end
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            core_b = torch.tensor(

                core_prob[
                    start:end
                ],

                dtype=torch.float32,

                device=DEVICE
            )


            out = model(

                clinical_b,
                expert_b,
                core_b
            )


            p = torch.sigmoid(
                out[
                    "logit"
                ]
            )


            predictions.append(

                p.cpu()
                .numpy()
            )

            corrections.append(

                out[
                    "correction"
                ]
                .cpu()
                .numpy()
            )

            expert_attentions.append(

                out[
                    "expert_attention"
                ]
                .cpu()
                .numpy()
            )

            group_attentions.append(

                out[
                    "group_attention"
                ]
                .cpu()
                .numpy()
            )

            correction_strengths.append(

                out[
                    "correction_strength"
                ]
                .cpu()
                .numpy()
            )


    return {

        "prediction":
            np.concatenate(
                predictions
            ),

        "correction":
            np.concatenate(
                corrections
            ),

        "expert_attention":
            np.concatenate(
                expert_attentions
            ),

        "group_attention":
            np.concatenate(
                group_attentions
            ),

        "correction_strength":
            np.concatenate(
                correction_strengths
            ),
    }


# ============================================================
# 13E.11 Inner epoch selection
# ============================================================

def select_pcea_epoch(
    X_clinical,
    X_expert,
    core_prob,
    y,
    seed,
):

    splitter = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                PCEA_CONFIG[
                    "inner_es_fraction"
                ],

            random_state=
                seed
        )
    )


    inner_train_idx, inner_es_idx = next(

        splitter.split(
            X_clinical,
            y
        )
    )


    # --------------------------------------------------------
    # scale clinical variables using INNER TRAIN only
    # --------------------------------------------------------

    scaler = (
        StandardScaler()
    )


    X_inner_train = (
        scaler.fit_transform(

            X_clinical[
                inner_train_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_inner_es = (
        scaler.transform(

            X_clinical[
                inner_es_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    pcea_seed_everything(
        seed
    )


    train_ds = PCEADataset(

        X_inner_train,

        X_expert[
            inner_train_idx
        ],

        core_prob[
            inner_train_idx
        ],

        y[
            inner_train_idx
        ]
    )


    train_loader = DataLoader(

        train_ds,

        batch_size=
            PCEA_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = PCEANet(

        group_indices=
            GROUP_COLUMN_INDICES,

        expert_feature_dim=
            X_expert.shape[2],

        n_experts=
            len(
                PCEA_EXPERTS
            ),

        d_model=
            PCEA_CONFIG[
                "d_model"
            ],

        dropout=
            PCEA_CONFIG[
                "dropout"
            ],

        max_correction=
            PCEA_CONFIG[
                "max_logit_correction"
            ],

    ).to(
        DEVICE
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                PCEA_CONFIG[
                    "lr"
                ],

            weight_decay=
                PCEA_CONFIG[
                    "weight_decay"
                ],
        )
    )


    best_pr = -np.inf
    best_epoch = 1

    wait = 0

    history = []


    for epoch in range(
        1,
        PCEA_CONFIG[
            "max_epochs"
        ] + 1
    ):

        model.train()

        running = 0.0
        seen = 0


        for (
            clinical_b,
            expert_b,
            core_b,
            y_b
        ) in train_loader:

            clinical_b = (
                clinical_b.to(
                    DEVICE
                )
            )

            expert_b = (
                expert_b.to(
                    DEVICE
                )
            )

            core_b = (
                core_b.to(
                    DEVICE
                )
            )

            y_b = (
                y_b.to(
                    DEVICE
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            output = model(

                clinical_b,
                expert_b,
                core_b
            )


            bce_loss = (
                F.binary_cross_entropy_with_logits(

                    output[
                        "logit"
                    ],

                    y_b
                )
            )


            residual_loss = (

                output[
                    "correction"
                ]
                .pow(2)
                .mean()
            )


            attn = (
                output[
                    "expert_attention"
                ]
            )


            entropy = -(

                attn

                * torch.log(
                    attn
                    + 1e-8
                )

            ).sum(
                dim=1
            ).mean()


            loss = (

                bce_loss

                + PCEA_CONFIG[
                    "residual_penalty"
                ]
                * residual_loss

                - PCEA_CONFIG[
                    "expert_entropy_penalty"
                ]
                * entropy
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                3.0
            )


            optimizer.step()


            n = len(
                y_b
            )

            running += (
                float(
                    loss.detach()
                    .cpu()
                )
                * n
            )

            seen += n


        # ----------------------------------------------------
        # INNER ES evaluation only
        # ----------------------------------------------------

        es_out = predict_pcea(

            model,

            X_inner_es,

            X_expert[
                inner_es_idx
            ],

            core_prob[
                inner_es_idx
            ]
        )


        es_pr = (
            average_precision_score(

                y[
                    inner_es_idx
                ],

                es_out[
                    "prediction"
                ]
            )
        )


        history.append({

            "Epoch":
                epoch,

            "Train_Loss":
                running / seen,

            "Inner_ES_PR":
                es_pr,
        })


        print(
            f"    epoch={epoch:02d} "
            f"ES_PR={es_pr:.6f} "
            f"best={max(best_pr, es_pr):.6f}"
        )


        if es_pr > (
            best_pr + 1e-5
        ):

            best_pr = (
                es_pr
            )

            best_epoch = (
                epoch
            )

            wait = 0

        else:

            wait += 1


        if wait >= (
            PCEA_CONFIG[
                "patience"
            ]
        ):

            break


    return (
        best_epoch,
        best_pr,
        pd.DataFrame(
            history
        ),
    )


# ============================================================
# 13E.12 Outer CV
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


PCEA_OOF = np.zeros(
    len(
        y_all
    ),
    dtype=np.float64
)

PCEA_CORE_OOF = np.zeros(
    len(
        y_all
    ),
    dtype=np.float64
)

PCEA_CORRECTION = np.zeros(
    len(
        y_all
    ),
    dtype=np.float64
)

PCEA_GATE = np.zeros(
    len(
        y_all
    ),
    dtype=np.float64
)

PCEA_EXPERT_ATTENTION = np.zeros(
    (
        len(
            y_all
        ),
        len(
            PCEA_EXPERTS
        )
    ),
    dtype=np.float32
)

PCEA_GROUP_ATTENTION = np.zeros(
    (
        len(
            y_all
        ),
        len(
            GROUP_NAMES
        )
    ),
    dtype=np.float32
)


fold_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"PCEA OUTER FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    outer_train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    outer_valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    y_train = (
        y_all[
            outer_train_idx
        ]
    )

    y_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # Strict outer-fold calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for expert_name in (
        PCEA_EXPERTS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(

                expert_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            expert_name
        ] = p_tr

        valid_cal[
            expert_name
        ] = p_va


    # ========================================================
    # Expert descriptors
    # ========================================================

    (
        expert_train,
        core_train
    ) = build_pcea_expert_features(
        train_cal
    )


    (
        expert_valid,
        core_valid
    ) = build_pcea_expert_features(
        valid_cal
    )


    # ========================================================
    # Clinical matrices
    # ========================================================

    clinical_train_raw = (
        clinical_matrix_from_indices(
            outer_train_idx
        )
    )


    clinical_valid_raw = (
        clinical_matrix_from_indices(
            outer_valid_idx
        )
    )


    # ========================================================
    # A. INNER epoch selection
    # ========================================================

    (
        selected_epoch,
        inner_best_pr,
        inner_history
    ) = select_pcea_epoch(

        clinical_train_raw,

        expert_train,

        core_train,

        y_train,

        seed=
            SEED
            + 3100
            + outer_fold
    )


    print(
        "\nSelected epoch:",
        selected_epoch
    )

    print(
        "Inner ES best PR:",
        f"{inner_best_pr:.6f}"
    )


    # ========================================================
    # B. Refit preprocessing on ALL OUTER TRAIN
    # ========================================================

    final_scaler = (
        StandardScaler()
    )


    clinical_train = (
        final_scaler.fit_transform(
            clinical_train_raw
        )
        .astype(
            np.float32
        )
    )


    clinical_valid = (
        final_scaler.transform(
            clinical_valid_raw
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # C. Fresh model trained exact selected epochs
    # ========================================================

    final_model = (
        train_pcea_fixed_epochs(

            clinical_train,

            expert_train,

            core_train,

            y_train,

            epochs=
                selected_epoch,

            seed=
                SEED
                + 4100
                + outer_fold,

            verbose=False,
        )
    )


    # ========================================================
    # D. Outer validation prediction — ONCE
    # ========================================================

    valid_out = (
        predict_pcea(

            final_model,

            clinical_valid,

            expert_valid,

            core_valid
        )
    )


    p_valid = (
        valid_out[
            "prediction"
        ]
    )


    PCEA_OOF[
        outer_valid_idx
    ] = p_valid


    PCEA_CORE_OOF[
        outer_valid_idx
    ] = core_valid


    PCEA_CORRECTION[
        outer_valid_idx
    ] = (
        valid_out[
            "correction"
        ]
    )


    PCEA_GATE[
        outer_valid_idx
    ] = (
        valid_out[
            "correction_strength"
        ]
    )


    PCEA_EXPERT_ATTENTION[
        outer_valid_idx,
        :
    ] = (
        valid_out[
            "expert_attention"
        ]
    )


    PCEA_GROUP_ATTENTION[
        outer_valid_idx,
        :
    ] = (
        valid_out[
            "group_attention"
        ]
    )


    # ========================================================
    # E. Fold evaluation
    # ========================================================

    core_pm = (
        probability_metrics(
            y_valid,
            core_valid
        )
    )


    pcea_pm = (
        probability_metrics(
            y_valid,
            p_valid
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            selected_epoch,

        "Inner_ES_PR":
            inner_best_pr,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "PCEA_PR":
            pcea_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            pcea_pm[
                "PR_AUC"
            ]
            -
            core_pm[
                "PR_AUC"
            ],

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "PCEA_ROC":
            pcea_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "PCEA_Brier":
            pcea_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "PCEA_ECE":
            pcea_pm[
                "ECE"
            ],

        "Mean_Abs_Correction":
            float(
                np.mean(
                    np.abs(
                        valid_out[
                            "correction"
                        ]
                    )
                )
            ),

        "Mean_Correction_Gate":
            float(
                np.mean(
                    valid_out[
                        "correction_strength"
                    ]
                )
            ),
    })


    print(
        "\nCore PR :",
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        "PCEA PR :",
        f"{pcea_pm['PR_AUC']:.6f}"
    )

    print(
        "Delta   :",
        f"{pcea_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )

    print(
        "Mean |correction|:",
        f"{np.mean(np.abs(valid_out['correction'])):.6f}"
    )


    del final_model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13E.13 Global evaluation
# ============================================================

core_threshold, _, _ = (
    select_best_threshold(

        y_all,

        PCEA_CORE_OOF,

        objective="F1"
    )
)


pcea_threshold, _, _ = (
    select_best_threshold(

        y_all,

        PCEA_OOF,

        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(

        y_all,

        PCEA_CORE_OOF,

        threshold=
            core_threshold
    )
)


pcea_metrics = (
    evaluate_binary_predictions(

        y_all,

        PCEA_OOF,

        threshold=
            pcea_threshold
    )
)


module13e_results = (
    pd.DataFrame([

        {
            "Method":
                "Core4Mean",

            **core_metrics
        },

        {
            "Method":
                "PCEA",

            **pcea_metrics
        },
    ])
)


module13e_results[
    "Delta_PR_vs_Core"
] = (
    module13e_results[
        "PR_AUC"
    ]
    -
    core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 13E — PCEA RESULTS"
)

print(
    "=" * 125
)


display(
    module13e_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13E.14 Fold results
# ============================================================

module13e_folds = (
    pd.DataFrame(
        fold_rows
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "PCEA FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module13e_folds
)


print(
    "\nMean fold Delta PR:",
    module13e_folds[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            module13e_folds[
                "Delta_PR"
            ]
            > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13E.15 Expert-attention summary
# ============================================================

expert_attention_rows = []


for j, expert_name in enumerate(
    PCEA_EXPERTS
):

    a = (
        PCEA_EXPERT_ATTENTION[
            :,
            j
        ]
    )


    expert_attention_rows.append({

        "Expert":
            expert_name,

        "Mean_Attention":
            float(
                np.mean(a)
            ),

        "Std_Attention":
            float(
                np.std(a)
            ),

        "Median_Attention":
            float(
                np.median(a)
            ),

        "P10":
            float(
                np.quantile(
                    a,
                    0.10
                )
            ),

        "P90":
            float(
                np.quantile(
                    a,
                    0.90
                )
            ),
    })


expert_attention_summary = (
    pd.DataFrame(
        expert_attention_rows
    )
    .sort_values(
        "Mean_Attention",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "PCEA EXPERT ATTENTION SUMMARY"
)

print(
    "=" * 125
)


display(
    expert_attention_summary
)


# ============================================================
# 13E.16 Clinical-group attention summary
# ============================================================

group_attention_rows = []


for j, group_name in enumerate(
    GROUP_NAMES
):

    a = (
        PCEA_GROUP_ATTENTION[
            :,
            j
        ]
    )


    group_attention_rows.append({

        "Clinical_Group":
            group_name,

        "Mean_Attention":
            float(
                np.mean(a)
            ),

        "Std_Attention":
            float(
                np.std(a)
            ),

        "Median_Attention":
            float(
                np.median(a)
            ),

        "P10":
            float(
                np.quantile(
                    a,
                    0.10
                )
            ),

        "P90":
            float(
                np.quantile(
                    a,
                    0.90
                )
            ),
    })


group_attention_summary = (
    pd.DataFrame(
        group_attention_rows
    )
    .sort_values(
        "Mean_Attention",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "PCEA CLINICAL-GROUP ATTENTION"
)

print(
    "=" * 125
)


display(
    group_attention_summary
)


# ============================================================
# 13E.17 Correction diagnostics
# ============================================================

abs_corr = (
    np.abs(
        PCEA_CORRECTION
    )
)


correction_summary = pd.DataFrame({

    "Metric": [

        "Mean correction",

        "Mean absolute correction",

        "Median absolute correction",

        "P90 absolute correction",

        "P95 absolute correction",

        "P99 absolute correction",

        "Maximum absolute correction",

        "Mean correction gate",

        "Fraction |correction| > 0.01",

        "Fraction |correction| > 0.05",

        "Fraction |correction| > 0.10",
    ],

    "Value": [

        float(
            PCEA_CORRECTION.mean()
        ),

        float(
            abs_corr.mean()
        ),

        float(
            np.median(
                abs_corr
            )
        ),

        float(
            np.quantile(
                abs_corr,
                0.90
            )
        ),

        float(
            np.quantile(
                abs_corr,
                0.95
            )
        ),

        float(
            np.quantile(
                abs_corr,
                0.99
            )
        ),

        float(
            abs_corr.max()
        ),

        float(
            PCEA_GATE.mean()
        ),

        float(
            np.mean(
                abs_corr
                > 0.01
            )
        ),

        float(
            np.mean(
                abs_corr
                > 0.05
            )
        ),

        float(
            np.mean(
                abs_corr
                > 0.10
            )
        ),
    ]
})


print(
    "\n"
    + "=" * 125
)

print(
    "PCEA CORRECTION DIAGNOSTICS"
)

print(
    "=" * 125
)


display(
    correction_summary
)


# ============================================================
# 13E.18 Attention entropy
# ============================================================

expert_entropy = -np.sum(

    PCEA_EXPERT_ATTENTION

    * np.log(
        PCEA_EXPERT_ATTENTION
        + 1e-12
    ),

    axis=1
)


group_entropy = -np.sum(

    PCEA_GROUP_ATTENTION

    * np.log(
        PCEA_GROUP_ATTENTION
        + 1e-12
    ),

    axis=1
)


attention_entropy_summary = (
    pd.DataFrame({

        "Metric": [

            "Mean expert-attention entropy",

            "Maximum possible expert entropy",

            "Mean group-attention entropy",

            "Maximum possible group entropy",
        ],

        "Value": [

            float(
                expert_entropy.mean()
            ),

            float(
                np.log(
                    len(
                        PCEA_EXPERTS
                    )
                )
            ),

            float(
                group_entropy.mean()
            ),

            float(
                np.log(
                    len(
                        GROUP_NAMES
                    )
                )
            ),
        ]
    })
)


print(
    "\n"
    + "=" * 125
)

print(
    "PCEA ATTENTION ENTROPY"
)

print(
    "=" * 125
)


display(
    attention_entropy_summary
)


# ============================================================
# 13E.19 Save
# ============================================================

save_csv(

    module13e_results,

    TAB_DIR
    / "module13e_pcea_results.csv"
)


save_csv(

    module13e_folds,

    TAB_DIR
    / "module13e_pcea_fold_results.csv"
)


save_csv(

    expert_attention_summary,

    TAB_DIR
    / "module13e_pcea_expert_attention.csv"
)


save_csv(

    group_attention_summary,

    TAB_DIR
    / "module13e_pcea_group_attention.csv"
)


save_csv(

    correction_summary,

    TAB_DIR
    / "module13e_pcea_correction_summary.csv"
)


save_csv(

    attention_entropy_summary,

    TAB_DIR
    / "module13e_pcea_attention_entropy.csv"
)


# row-level output
pcea_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        PCEA_CORE_OOF,

    "PCEA":
        PCEA_OOF,

    "LogitCorrection":
        PCEA_CORRECTION,

    "CorrectionGate":
        PCEA_GATE,
})


for j, expert_name in enumerate(
    PCEA_EXPERTS
):

    pcea_pred_df[
        f"attn_expert_{expert_name}"
    ] = (
        PCEA_EXPERT_ATTENTION[
            :,
            j
        ]
    )


for j, group_name in enumerate(
    GROUP_NAMES
):

    pcea_pred_df[
        f"attn_group_{group_name}"
    ] = (
        PCEA_GROUP_ATTENTION[
            :,
            j
        ]
    )


save_csv(

    pcea_pred_df,

    PRED_DIR
    / "module13e_pcea_oof_predictions.csv"
)


module13e_manifest = {

    "architecture":
        "Patient-Conditioned Cross-Expert Attention",

    "experts":
        PCEA_EXPERTS,

    "clinical_groups":
        PCEA_GROUPS,

    "inner_epoch_selection":
        True,

    "fresh_outer_train_refit":
        True,

    "max_logit_correction":
        PCEA_CONFIG[
            "max_logit_correction"
        ],

    "holdout_used":
        False,
}


save_json(

    module13e_manifest,

    LOG_DIR
    / "module13e_manifest.json"
)


print(
    "\nMODULE 13E COMPLETE."
)

PCEA device: cuda
PCEA clinical groups:
  cardiometabolic          ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Diabetes']
  vascular_history         ['Stroke']
  general_functional       ['GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk']
  lifestyle                ['Smoker', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump']
  demographic_access       ['Sex', 'Age', 'Education', 'Income', 'AnyHealthcare', 'NoDocbcCost']

PCEA OUTER FOLD 1/5
    epoch=01 ES_PR=0.374227 best=0.374227
    epoch=02 ES_PR=0.374246 best=0.374246
    epoch=03 ES_PR=0.374252 best=0.374252
    epoch=04 ES_PR=0.374213 best=0.374246
    epoch=05 ES_PR=0.374225 best=0.374246
    epoch=06 ES_PR=0.374225 best=0.374246

Selected epoch: 2
Inner ES best PR: 0.374246

Core PR : 0.380069
PCEA PR : 0.379972
Delta   : -0.000097
Mean |correction|: 0.006898

PCEA OUTER FOLD 2/5
    epoch=01 ES_PR=0.373468 best=0.373468
    epoch=02 ES_PR=0.373476 best=0.373476
    epoch=03 ES_PR=0.373481 best=0.373481
    epoch=04 ES_PR=

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,PCEA,0.380277,-0.000051,0.850541,0.069707,0.235588,0.002167,0.350968,0.525479,0.420850,0.356204,0.22



PCEA FOLD RESULTS


,Fold,Selected_Epoch,Inner_ES_PR,Core_PR,PCEA_PR,Delta_PR,Core_ROC,PCEA_ROC,Core_Brier,PCEA_Brier,Core_ECE,PCEA_ECE,Mean_Abs_Correction,Mean_Correction_Gate
0,0,2,0.374246,0.380069,0.379972,-0.000097,0.852660,0.852629,0.069637,0.069641,0.003499,0.003585,0.006898,0.453898
1,1,5,0.373494,0.382772,0.382770,-0.000001,0.850766,0.850772,0.069684,0.069683,0.004297,0.004322,0.003235,0.218337
2,2,19,0.374906,0.378467,0.378422,-0.000045,0.849030,0.848976,0.069827,0.069830,0.002486,0.002304,0.007884,0.404876
3,3,1,0.390810,0.372833,0.372803,-0.000029,0.849083,0.849081,0.070081,0.070084,0.005053,0.005259,0.012113,0.504455
4,4,1,0.398008,0.390072,0.390116,0.000043,0.851760,0.851762,0.069301,0.069298,0.005318,0.005026,0.011365,0.488289



Mean fold Delta PR: -2.5929322756557126e-05
Positive folds: 1 / 5

PCEA EXPERT ATTENTION SUMMARY


...


PCEA CORRECTION DIAGNOSTICS


,Metric,Value
0,Mean correction,-0.007675
1,Mean absolute correction,0.008299
2,Median absolute correction,0.004016
3,P90 absolute correction,0.021386
4,P95 absolute correction,0.031168
5,P99 absolute correction,0.058459
6,Maximum absolute correction,0.218769
7,Mean correction gate,0.413971
8,Fraction |correction| > 0.01,0.253479
9,Fraction |correction| > 0.05,0.016034



PCEA ATTENTION ENTROPY


,Metric,Value
0,Mean expert-attention entropy,1.385599
1,Maximum possible expert entropy,1.386294
2,Mean group-attention entropy,1.593229
3,Maximum possible group entropy,1.609438



MODULE 13E COMPLETE.


In [23]:
# ============================================================
# MODULE 13F — RSAF
# Reliability-Supervised Attention Fusion
# ============================================================
#
# Key innovation:
#
# Attention is explicitly supervised using per-sample
# expert reliability targets derived ONLY from training labels.
#
# Loss:
#   BCE
# + pairwise ranking
# + reliability-supervised attention
# + bounded correction regularization
#
# Inner split selects epoch.
# Fresh refit on full outer train.
# Outer validation untouched.
#
# HOLDOUT untouched.
# ============================================================

import gc
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score


# ============================================================
# 13F.1 Configuration
# ============================================================

RSAF_EXPERTS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

RSAF_CONFIG = {

    "d_model":
        32,

    "hidden":
        32,

    "dropout":
        0.10,

    "batch_size":
        2048,

    "max_epochs":
        25,

    "patience":
        4,

    "inner_es_fraction":
        0.15,

    "lr":
        5e-4,

    "weight_decay":
        2e-4,

    # reliability target temperature
    "reliability_temperature":
        0.20,

    # final-loss weights
    "alpha_rank":
        0.20,

    "beta_attention":
        0.30,

    "gamma_residual":
        0.05,

    # keep correction modest
    "max_logit_correction":
        0.20,

    # max positives / negatives used in pairwise loss
    "rank_side":
        96,
}


# ============================================================
# 13F.2 Reliability target
# ============================================================

def make_reliability_targets(
    expert_prob,
    y,
    temperature=0.20
):
    """
    expert_prob:
        N x K calibrated probabilities

    y:
        N binary labels

    Output:
        N x K soft reliability targets
    """

    P = np.clip(
        expert_prob,
        1e-6,
        1.0 - 1e-6
    )

    y = np.asarray(
        y,
        dtype=float
    )[:, None]

    losses = -(
        y * np.log(P)
        +
        (1.0 - y)
        * np.log(1.0 - P)
    )

    logits = (
        -losses
        / temperature
    )

    logits = (
        logits
        - logits.max(
            axis=1,
            keepdims=True
        )
    )

    target = np.exp(
        logits
    )

    target = (
        target
        /
        target.sum(
            axis=1,
            keepdims=True
        )
    )

    return target.astype(
        np.float32
    )


# ============================================================
# 13F.3 Dataset
# ============================================================

class RSAFDataset(Dataset):

    def __init__(
        self,
        clinical_x,
        expert_x,
        core_prob,
        y,
        reliability_target
    ):

        self.clinical_x = torch.tensor(
            clinical_x,
            dtype=torch.float32
        )

        self.expert_x = torch.tensor(
            expert_x,
            dtype=torch.float32
        )

        self.core_prob = torch.tensor(
            core_prob,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.reliability_target = (
            torch.tensor(
                reliability_target,
                dtype=torch.float32
            )
        )

    def __len__(self):

        return len(
            self.y
        )

    def __getitem__(
        self,
        idx
    ):

        return (
            self.clinical_x[idx],
            self.expert_x[idx],
            self.core_prob[idx],
            self.y[idx],
            self.reliability_target[idx],
        )


# ============================================================
# 13F.4 Reliability-supervised attention network
# ============================================================

class RSAFNet(nn.Module):

    def __init__(
        self,
        group_indices,
        expert_feature_dim,
        n_experts,
        d_model=32,
        dropout=0.10,
        max_correction=0.20
    ):

        super().__init__()

        self.group_names = list(
            group_indices.keys()
        )

        self.group_indices = (
            group_indices
        )

        self.d_model = (
            d_model
        )

        self.max_correction = (
            max_correction
        )


        # ----------------------------------------------------
        # Clinical group encoders
        # ----------------------------------------------------

        self.group_encoders = nn.ModuleDict()

        for group_name in (
            self.group_names
        ):

            n_features = len(
                group_indices[
                    group_name
                ]
            )

            self.group_encoders[
                group_name
            ] = nn.Sequential(

                nn.Linear(
                    n_features,
                    RSAF_CONFIG[
                        "hidden"
                    ]
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                ),

                nn.Linear(
                    RSAF_CONFIG[
                        "hidden"
                    ],
                    d_model
                ),
            )


        self.group_identity = (
            nn.Parameter(
                torch.randn(
                    len(
                        self.group_names
                    ),
                    d_model
                )
                * 0.02
            )
        )


        # ----------------------------------------------------
        # Learnable group attention
        # ----------------------------------------------------

        self.group_score = nn.Sequential(

            nn.Linear(
                d_model,
                16
            ),

            nn.Tanh(),

            nn.Linear(
                16,
                1
            )
        )


        # ----------------------------------------------------
        # Patient representation
        # ----------------------------------------------------

        self.patient_projection = nn.Sequential(

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        # ----------------------------------------------------
        # Expert encoding
        # ----------------------------------------------------

        self.expert_encoder = nn.Sequential(

            nn.Linear(
                expert_feature_dim,
                32
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                32,
                d_model
            )
        )


        self.expert_identity = (
            nn.Parameter(
                torch.randn(
                    n_experts,
                    d_model
                )
                * 0.02
            )
        )


        # ----------------------------------------------------
        # Attention projections
        # ----------------------------------------------------

        self.query = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.key = nn.Linear(
            d_model,
            d_model,
            bias=False
        )


        # ----------------------------------------------------
        # Trust / correction-strength gate
        # ----------------------------------------------------

        self.trust_gate = nn.Sequential(

            nn.Linear(
                d_model,
                16
            ),

            nn.GELU(),

            nn.Linear(
                16,
                1
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        clinical_x,
        expert_x,
        core_prob
    ):

        # ====================================================
        # A. Clinical-group attention
        # ====================================================

        group_tokens = []

        for i, group_name in enumerate(
            self.group_names
        ):

            cols = (
                self.group_indices[
                    group_name
                ]
            )

            token = (
                self.group_encoders[
                    group_name
                ](
                    clinical_x[
                        :,
                        cols
                    ]
                )
            )

            token = (
                token
                +
                self.group_identity[
                    i
                ]
            )

            group_tokens.append(
                token
            )


        group_tokens = torch.stack(
            group_tokens,
            dim=1
        )


        group_logits = (
            self.group_score(
                group_tokens
            )
            .squeeze(-1)
        )


        group_attention = (
            torch.softmax(
                group_logits,
                dim=1
            )
        )


        patient = (
            group_tokens
            *
            group_attention.unsqueeze(
                -1
            )
        ).sum(
            dim=1
        )


        patient = (
            self.patient_projection(
                patient
            )
        )


        # ====================================================
        # B. Expert attention
        # ====================================================

        expert_tokens = (
            self.expert_encoder(
                expert_x
            )
        )


        expert_tokens = (
            expert_tokens
            +
            self.expert_identity[
                None,
                :,
                :
            ]
        )


        q = (
            self.query(
                patient
            )
            .unsqueeze(1)
        )


        k = (
            self.key(
                expert_tokens
            )
        )


        attention_logits = (
            (
                q
                * k
            )
            .sum(
                dim=-1
            )
            /
            np.sqrt(
                self.d_model
            )
        )


        attention = (
            torch.softmax(
                attention_logits,
                dim=1
            )
        )


        # ====================================================
        # C. Expert logits
        # ====================================================

        expert_prob = (
            expert_x[
                :,
                :,
                0
            ]
        )


        expert_prob = torch.clamp(
            expert_prob,
            1e-6,
            1.0 - 1e-6
        )


        expert_logit = torch.log(
            expert_prob
            /
            (
                1.0
                -
                expert_prob
            )
        )


        core_prob = torch.clamp(
            core_prob,
            1e-6,
            1.0 - 1e-6
        )


        core_logit = torch.log(
            core_prob
            /
            (
                1.0
                -
                core_prob
            )
        )


        attention_logit = (
            attention
            *
            expert_logit
        ).sum(
            dim=1
        )


        raw_delta = (
            attention_logit
            -
            core_logit
        )


        # ----------------------------------------------------
        # bounded correction
        # ----------------------------------------------------

        bounded_delta = (

            self.max_correction

            * torch.tanh(

                raw_delta
                /
                self.max_correction
            )
        )


        trust = (
            self.trust_gate(
                patient
            )
            .squeeze(-1)
        )


        correction = (
            trust
            * bounded_delta
        )


        final_logit = (
            core_logit
            +
            correction
        )


        return {

            "logit":
                final_logit,

            "attention_logits":
                attention_logits,

            "attention":
                attention,

            "group_attention":
                group_attention,

            "trust":
                trust,

            "correction":
                correction,
        }


# ============================================================
# 13F.5 Hard-pair ranking loss
# ============================================================

def rsaf_rank_loss(
    logits,
    y,
    max_side=96
):

    pos = logits[
        y > 0.5
    ]

    neg = logits[
        y <= 0.5
    ]


    if (
        len(pos) == 0
        or len(neg) == 0
    ):

        return torch.tensor(
            0.0,
            device=
                logits.device
        )


    if len(pos) > max_side:

        ids = torch.randperm(
            len(pos),
            device=
                logits.device
        )[
            :max_side
        ]

        pos = pos[
            ids
        ]


    if len(neg) > max_side:

        ids = torch.randperm(
            len(neg),
            device=
                logits.device
        )[
            :max_side
        ]

        neg = neg[
            ids
        ]


    diff = (
        pos[:, None]
        -
        neg[None, :]
    )


    return (
        F.softplus(
            -diff
        )
        .mean()
    )


# ============================================================
# 13F.6 Combined loss
# ============================================================

def rsaf_loss(
    output,
    y,
    reliability_target
):

    # --------------------------------------------------------
    # predictive loss
    # --------------------------------------------------------

    bce = (
        F.binary_cross_entropy_with_logits(
            output[
                "logit"
            ],
            y
        )
    )


    # --------------------------------------------------------
    # ranking
    # --------------------------------------------------------

    rank = rsaf_rank_loss(

        output[
            "logit"
        ],

        y,

        max_side=
            RSAF_CONFIG[
                "rank_side"
            ]
    )


    # --------------------------------------------------------
    # reliability-supervised attention
    # --------------------------------------------------------

    log_attention = (
        F.log_softmax(
            output[
                "attention_logits"
            ],
            dim=1
        )
    )


    attn_loss = -(
        reliability_target
        *
        log_attention
    ).sum(
        dim=1
    ).mean()


    # --------------------------------------------------------
    # preserve strong Core4
    # --------------------------------------------------------

    residual = (
        output[
            "correction"
        ]
        .pow(2)
        .mean()
    )


    total = (

        bce

        + RSAF_CONFIG[
            "alpha_rank"
        ]
        * rank

        + RSAF_CONFIG[
            "beta_attention"
        ]
        * attn_loss

        + RSAF_CONFIG[
            "gamma_residual"
        ]
        * residual
    )


    return (
        total,
        bce,
        rank,
        attn_loss,
        residual,
    )


# ============================================================
# 13F.7 Prediction
# ============================================================

def predict_rsaf(
    model,
    clinical_x,
    expert_x,
    core_prob,
    batch_size=4096
):

    model.eval()

    pred = []
    attention = []
    group_attention = []
    correction = []
    trust = []


    with torch.no_grad():

        for start in range(
            0,
            len(core_prob),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(core_prob)
            )


            xc = torch.tensor(
                clinical_x[
                    start:end
                ],
                dtype=torch.float32,
                device=DEVICE
            )


            xe = torch.tensor(
                expert_x[
                    start:end
                ],
                dtype=torch.float32,
                device=DEVICE
            )


            cp = torch.tensor(
                core_prob[
                    start:end
                ],
                dtype=torch.float32,
                device=DEVICE
            )


            out = model(
                xc,
                xe,
                cp
            )


            pred.append(
                torch.sigmoid(
                    out["logit"]
                )
                .cpu()
                .numpy()
            )


            attention.append(
                out[
                    "attention"
                ]
                .cpu()
                .numpy()
            )


            group_attention.append(
                out[
                    "group_attention"
                ]
                .cpu()
                .numpy()
            )


            correction.append(
                out[
                    "correction"
                ]
                .cpu()
                .numpy()
            )


            trust.append(
                out[
                    "trust"
                ]
                .cpu()
                .numpy()
            )


    return {

        "prediction":
            np.concatenate(
                pred
            ),

        "attention":
            np.concatenate(
                attention
            ),

        "group_attention":
            np.concatenate(
                group_attention
            ),

        "correction":
            np.concatenate(
                correction
            ),

        "trust":
            np.concatenate(
                trust
            ),
    }


# ============================================================
# 13F.8 Model factory
# ============================================================

def make_rsaf_model(
    expert_dim
):

    return RSAFNet(

        group_indices=
            GROUP_COLUMN_INDICES,

        expert_feature_dim=
            expert_dim,

        n_experts=
            len(
                RSAF_EXPERTS
            ),

        d_model=
            RSAF_CONFIG[
                "d_model"
            ],

        dropout=
            RSAF_CONFIG[
                "dropout"
            ],

        max_correction=
            RSAF_CONFIG[
                "max_logit_correction"
            ],
    ).to(
        DEVICE
    )


# ============================================================
# 13F.9 Epoch-selection trainer
# ============================================================

def select_rsaf_epoch(
    clinical_raw,
    expert_features,
    expert_probs,
    core_prob,
    y,
    seed
):

    split = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            RSAF_CONFIG[
                "inner_es_fraction"
            ],

        random_state=
            seed,
    )


    train_idx, es_idx = next(

        split.split(
            clinical_raw,
            y
        )
    )


    scaler = (
        StandardScaler()
    )


    Xtr = (
        scaler.fit_transform(
            clinical_raw[
                train_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    Xes = (
        scaler.transform(
            clinical_raw[
                es_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    reliability = (
        make_reliability_targets(

            expert_probs[
                train_idx
            ],

            y[
                train_idx
            ],

            temperature=
                RSAF_CONFIG[
                    "reliability_temperature"
                ]
        )
    )


    dataset = RSAFDataset(

        Xtr,

        expert_features[
            train_idx
        ],

        core_prob[
            train_idx
        ],

        y[
            train_idx
        ],

        reliability,
    )


    loader = DataLoader(

        dataset,

        batch_size=
            RSAF_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    pcea_seed_everything(
        seed
    )


    model = make_rsaf_model(
        expert_features.shape[
            2
        ]
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                RSAF_CONFIG[
                    "lr"
                ],

            weight_decay=
                RSAF_CONFIG[
                    "weight_decay"
                ],
        )
    )


    best_pr = -np.inf
    best_epoch = 1

    wait = 0


    for epoch in range(
        1,
        RSAF_CONFIG[
            "max_epochs"
        ] + 1
    ):

        model.train()


        for (
            xc,
            xe,
            cp,
            yb,
            rt
        ) in loader:

            xc = xc.to(
                DEVICE
            )

            xe = xe.to(
                DEVICE
            )

            cp = cp.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            rt = rt.to(
                DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            out = model(
                xc,
                xe,
                cp
            )


            (
                loss,
                _,
                _,
                _,
                _
            ) = rsaf_loss(
                out,
                yb,
                rt
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                3.0
            )


            optimizer.step()


        # ----------------------------------------------------
        # inner early-stop only
        # ----------------------------------------------------

        es_out = predict_rsaf(

            model,

            Xes,

            expert_features[
                es_idx
            ],

            core_prob[
                es_idx
            ]
        )


        pr = (
            average_precision_score(

                y[
                    es_idx
                ],

                es_out[
                    "prediction"
                ]
            )
        )


        print(
            f"    epoch={epoch:02d} "
            f"ES_PR={pr:.6f} "
            f"best={max(best_pr,pr):.6f}"
        )


        if pr > (
            best_pr + 1e-5
        ):

            best_pr = pr
            best_epoch = epoch
            wait = 0

        else:

            wait += 1


        if wait >= (
            RSAF_CONFIG[
                "patience"
            ]
        ):

            break


    return (
        best_epoch,
        best_pr
    )


# ============================================================
# 13F.10 Full outer-training refit
# ============================================================

def train_rsaf_fixed(
    clinical_x,
    expert_features,
    expert_probs,
    core_prob,
    y,
    epochs,
    seed
):

    reliability = (
        make_reliability_targets(

            expert_probs,

            y,

            temperature=
                RSAF_CONFIG[
                    "reliability_temperature"
                ]
        )
    )


    dataset = RSAFDataset(

        clinical_x,
        expert_features,
        core_prob,
        y,
        reliability
    )


    loader = DataLoader(

        dataset,

        batch_size=
            RSAF_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    pcea_seed_everything(
        seed
    )


    model = make_rsaf_model(
        expert_features.shape[
            2
        ]
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                RSAF_CONFIG[
                    "lr"
                ],

            weight_decay=
                RSAF_CONFIG[
                    "weight_decay"
                ],
        )
    )


    for epoch in range(
        epochs
    ):

        model.train()


        for (
            xc,
            xe,
            cp,
            yb,
            rt
        ) in loader:

            xc = xc.to(
                DEVICE
            )

            xe = xe.to(
                DEVICE
            )

            cp = cp.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            rt = rt.to(
                DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            out = model(
                xc,
                xe,
                cp
            )


            loss, *_ = rsaf_loss(
                out,
                yb,
                rt
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                3.0
            )


            optimizer.step()


    return model


# ============================================================
# 13F.11 Outer CV
# ============================================================

y_all = (
    y_dev.to_numpy(
        dtype=int
    )
)


RSAF_CORE = np.zeros(
    len(y_all),
    dtype=float
)

RSAF_OOF = np.zeros(
    len(y_all),
    dtype=float
)

RSAF_CORRECTION = np.zeros(
    len(y_all),
    dtype=float
)

RSAF_TRUST = np.zeros(
    len(y_all),
    dtype=float
)

RSAF_ATTENTION = np.zeros(
    (
        len(y_all),
        len(RSAF_EXPERTS)
    ),
    dtype=np.float32
)

RSAF_GROUP_ATTENTION = np.zeros(
    (
        len(y_all),
        len(GROUP_NAMES)
    ),
    dtype=np.float32
)


fold_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"RSAF OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]


    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    y_train = y_all[
        train_idx
    ]

    y_valid = y_all[
        valid_idx
    ]


    # ========================================================
    # fold-local calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for expert in (
        RSAF_EXPERTS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(

                expert,
                train_idx,
                valid_idx
            )
        )


        train_cal[
            expert
        ] = p_tr

        valid_cal[
            expert
        ] = p_va


    # ========================================================
    # descriptors
    # ========================================================

    (
        expert_train,
        core_train
    ) = build_pcea_expert_features(
        train_cal
    )


    (
        expert_valid,
        core_valid
    ) = build_pcea_expert_features(
        valid_cal
    )


    expert_prob_train = (
        np.column_stack([
            train_cal[m]
            for m in
            RSAF_EXPERTS
        ])
        .astype(
            np.float32
        )
    )


    expert_prob_valid = (
        np.column_stack([
            valid_cal[m]
            for m in
            RSAF_EXPERTS
        ])
        .astype(
            np.float32
        )
    )


    clinical_train_raw = (
        clinical_matrix_from_indices(
            train_idx
        )
    )


    clinical_valid_raw = (
        clinical_matrix_from_indices(
            valid_idx
        )
    )


    # ========================================================
    # inner epoch selection
    # ========================================================

    selected_epoch, inner_pr = (
        select_rsaf_epoch(

            clinical_train_raw,

            expert_train,

            expert_prob_train,

            core_train,

            y_train,

            seed=
                SEED
                + 5100
                + outer_fold
        )
    )


    print(
        "Selected epoch:",
        selected_epoch
    )


    # ========================================================
    # refit scaler on complete outer train
    # ========================================================

    scaler = (
        StandardScaler()
    )


    clinical_train = (
        scaler.fit_transform(
            clinical_train_raw
        )
        .astype(
            np.float32
        )
    )


    clinical_valid = (
        scaler.transform(
            clinical_valid_raw
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # fresh refit
    # ========================================================

    model = train_rsaf_fixed(

        clinical_train,

        expert_train,

        expert_prob_train,

        core_train,

        y_train,

        epochs=
            selected_epoch,

        seed=
            SEED
            + 6100
            + outer_fold
    )


    # ========================================================
    # untouched outer prediction
    # ========================================================

    out = predict_rsaf(

        model,

        clinical_valid,

        expert_valid,

        core_valid
    )


    RSAF_CORE[
        valid_idx
    ] = core_valid


    RSAF_OOF[
        valid_idx
    ] = out[
        "prediction"
    ]


    RSAF_CORRECTION[
        valid_idx
    ] = out[
        "correction"
    ]


    RSAF_TRUST[
        valid_idx
    ] = out[
        "trust"
    ]


    RSAF_ATTENTION[
        valid_idx
    ] = out[
        "attention"
    ]


    RSAF_GROUP_ATTENTION[
        valid_idx
    ] = out[
        "group_attention"
    ]


    # ========================================================
    # metrics
    # ========================================================

    core_pm = probability_metrics(
        y_valid,
        core_valid
    )


    rsaf_pm = probability_metrics(
        y_valid,
        out[
            "prediction"
        ]
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            selected_epoch,

        "Inner_ES_PR":
            inner_pr,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "RSAF_PR":
            rsaf_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            rsaf_pm[
                "PR_AUC"
            ]
            -
            core_pm[
                "PR_AUC"
            ],

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "RSAF_ROC":
            rsaf_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "RSAF_Brier":
            rsaf_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "RSAF_ECE":
            rsaf_pm[
                "ECE"
            ],

        "Mean_Abs_Correction":
            float(
                np.mean(
                    np.abs(
                        out[
                            "correction"
                        ]
                    )
                )
            ),

        "Mean_Trust":
            float(
                np.mean(
                    out[
                        "trust"
                    ]
                )
            ),
    })


    print(
        f"Core PR : "
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        f"RSAF PR : "
        f"{rsaf_pm['PR_AUC']:.6f}"
    )

    print(
        f"Delta   : "
        f"{rsaf_pm['PR_AUC']-core_pm['PR_AUC']:+.6f}"
    )


    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13F.12 Global results
# ============================================================

core_t, _, _ = (
    select_best_threshold(
        y_all,
        RSAF_CORE,
        objective="F1"
    )
)


rsaf_t, _, _ = (
    select_best_threshold(
        y_all,
        RSAF_OOF,
        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(
        y_all,
        RSAF_CORE,
        threshold=core_t
    )
)


rsaf_metrics = (
    evaluate_binary_predictions(
        y_all,
        RSAF_OOF,
        threshold=rsaf_t
    )
)


module13f_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "RSAF",

        **rsaf_metrics
    },
])


module13f_results[
    "Delta_PR_vs_Core"
] = (

    module13f_results[
        "PR_AUC"
    ]

    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 13F — RSAF RESULTS"
)

print(
    "=" * 125
)


display(
    module13f_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13F.13 Fold results
# ============================================================

module13f_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "RSAF FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module13f_folds
)


print(
    "\nMean fold ΔPR:",
    module13f_folds[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            module13f_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13F.14 Attention summary
# ============================================================

attention_rows = []


for j, expert in enumerate(
    RSAF_EXPERTS
):

    a = RSAF_ATTENTION[
        :,
        j
    ]


    attention_rows.append({

        "Expert":
            expert,

        "Mean_Attention":
            float(
                a.mean()
            ),

        "Std_Attention":
            float(
                a.std()
            ),

        "Median":
            float(
                np.median(a)
            ),

        "P10":
            float(
                np.quantile(
                    a,
                    .10
                )
            ),

        "P90":
            float(
                np.quantile(
                    a,
                    .90
                )
            ),
    })


rsaf_attention_summary = (
    pd.DataFrame(
        attention_rows
    )
    .sort_values(
        "Mean_Attention",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "RSAF EXPERT ATTENTION"
)

print(
    "=" * 125
)


display(
    rsaf_attention_summary
)


# ============================================================
# 13F.15 Attention entropy
# ============================================================

att_entropy = -np.sum(

    RSAF_ATTENTION

    * np.log(
        RSAF_ATTENTION
        + 1e-12
    ),

    axis=1
)


print(
    "\nMean expert-attention entropy:",
    float(
        att_entropy.mean()
    )
)

print(
    "Maximum entropy:",
    float(
        np.log(
            len(
                RSAF_EXPERTS
            )
        )
    )
)


# ============================================================
# 13F.16 Save
# ============================================================

save_csv(
    module13f_results,
    TAB_DIR
    / "module13f_rsaf_results.csv"
)


save_csv(
    module13f_folds,
    TAB_DIR
    / "module13f_rsaf_fold_results.csv"
)


save_csv(
    rsaf_attention_summary,
    TAB_DIR
    / "module13f_rsaf_attention.csv"
)


rsaf_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        RSAF_CORE,

    "RSAF":
        RSAF_OOF,

    "Correction":
        RSAF_CORRECTION,

    "Trust":
        RSAF_TRUST,
})


for j, expert in enumerate(
    RSAF_EXPERTS
):

    rsaf_pred_df[
        f"Attention_{expert}"
    ] = RSAF_ATTENTION[
        :,
        j
    ]


save_csv(
    rsaf_pred_df,
    PRED_DIR
    / "module13f_rsaf_oof_predictions.csv"
)


print(
    "\nMODULE 13F COMPLETE."
)


RSAF OUTER FOLD 1/5
    epoch=01 ES_PR=0.375783 best=0.375783
    epoch=02 ES_PR=0.375840 best=0.375840
    epoch=03 ES_PR=0.375771 best=0.375840
    epoch=04 ES_PR=0.375768 best=0.375840
    epoch=05 ES_PR=0.375771 best=0.375840
    epoch=06 ES_PR=0.375778 best=0.375840
Selected epoch: 2
Core PR : 0.380069
RSAF PR : 0.380059
Delta   : -0.000010

RSAF OUTER FOLD 2/5
    epoch=01 ES_PR=0.377661 best=0.377661
    epoch=02 ES_PR=0.377666 best=0.377666
    epoch=03 ES_PR=0.377652 best=0.377661
    epoch=04 ES_PR=0.377651 best=0.377661
    epoch=05 ES_PR=0.377652 best=0.377661
Selected epoch: 1
Core PR : 0.382772
RSAF PR : 0.382745
Delta   : -0.000027

RSAF OUTER FOLD 3/5
    epoch=01 ES_PR=0.384441 best=0.384441
    epoch=02 ES_PR=0.384447 best=0.384447
    epoch=03 ES_PR=0.384581 best=0.384581
    epoch=04 ES_PR=0.384561 best=0.384581
    epoch=05 ES_PR=0.384557 best=0.384581
    epoch=06 ES_PR=0.384541 best=0.384581
    epoch=07 ES_PR=0.384532 best=0.384581
Selected epoch: 3
Core PR : 0

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,RSAF,0.380314,-0.000014,0.850550,0.069707,0.235585,0.002238,0.351126,0.525531,0.420980,0.356351,0.22



RSAF FOLD RESULTS


,Fold,Selected_Epoch,Inner_ES_PR,Core_PR,RSAF_PR,Delta_PR,Core_ROC,RSAF_ROC,Core_Brier,RSAF_Brier,Core_ECE,RSAF_ECE,Mean_Abs_Correction,Mean_Trust
0,0,2,0.375840,0.380069,0.380059,-0.000010,0.852660,0.852641,0.069637,0.069639,0.003499,0.003595,0.006210,0.322789
1,1,1,0.377661,0.382772,0.382745,-0.000027,0.850766,0.850777,0.069684,0.069683,0.004297,0.004302,0.005901,0.360768
2,2,3,0.384581,0.378467,0.378431,-0.000036,0.849030,0.849014,0.069827,0.069828,0.002486,0.002515,0.006579,0.251948
3,3,14,0.379865,0.372833,0.372817,-0.000015,0.849083,0.849074,0.070081,0.070086,0.005053,0.005209,0.006444,0.324329
4,4,1,0.380777,0.390072,0.390092,0.000019,0.851760,0.851759,0.069301,0.069299,0.005318,0.005059,0.009334,0.336686



Mean fold ΔPR: -1.3853900084415117e-05
Positive folds: 1 / 5

RSAF EXPERT ATTENTION


,Expert,Mean_Attention,Std_Attention,Median,P10,P90
3,DeepFM_v2,0.250406,0.002676,0.250254,0.248891,0.251819
2,AutoInt,0.250306,0.002737,0.250178,0.248699,0.251986
0,HistGradientBoosting,0.249883,0.002372,0.250078,0.248286,0.251223
1,CatBoost,0.249405,0.002594,0.249376,0.248217,0.250315



Mean expert-attention entropy: 1.3862395286560059
Maximum entropy: 1.3862943611198906

MODULE 13F COMPLETE.


In [24]:
# ============================================================
# MODULE 13G — GAFT-RANK
# Group-Aware Feature Transformer with Ranking-Aware Training
# ============================================================
#
# NEW SOURCE OF REPRESENTATION:
#
# Raw 21 BRFSS features
#      ↓
# feature tokens
# + clinical-group embeddings
#      ↓
# Transformer self-attention
#      ↓
# CLS patient representation
#      ↓
# disease score
#
# Training:
#   BCE + pairwise ranking loss
#
# Validation protocol:
#   outer 5-fold CV
#   inner train/early-stop split
#   fresh refit on complete outer-train
#   outer-valid predicted ONCE
#
# HOLDOUT untouched.
# ============================================================

import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import average_precision_score


# ============================================================
# 13G.1 Feature definition
# ============================================================

GAFT_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


GAFT_GROUPS = {

    "cardiometabolic": [
        "HighBP",
        "HighChol",
        "CholCheck",
        "BMI",
        "Diabetes",
    ],

    "vascular_history": [
        "Stroke",
    ],

    "general_functional": [
        "GenHlth",
        "MentHlth",
        "PhysHlth",
        "DiffWalk",
    ],

    "lifestyle": [
        "Smoker",
        "PhysActivity",
        "Fruits",
        "Veggies",
        "HvyAlcoholConsump",
    ],

    "demographic_access": [
        "Sex",
        "Age",
        "Education",
        "Income",
        "AnyHealthcare",
        "NoDocbcCost",
    ],
}


assert len(GAFT_FEATURES) == 21

_flat = sum(
    GAFT_GROUPS.values(),
    []
)

assert set(_flat) == set(
    GAFT_FEATURES
)


# ============================================================
# 13G.2 Feature → group ID
# ============================================================

GROUP_NAMES_GAFT = list(
    GAFT_GROUPS.keys()
)

GROUP_ID = {}

for group_id, group_name in enumerate(
    GROUP_NAMES_GAFT
):

    for feature in GAFT_GROUPS[
        group_name
    ]:

        GROUP_ID[
            feature
        ] = group_id


GAFT_GROUP_IDS = np.array(
    [
        GROUP_ID[f]
        for f in GAFT_FEATURES
    ],
    dtype=np.int64
)


print("GAFT feature groups:")

for name in GROUP_NAMES_GAFT:

    print(
        f"{name:<24}",
        GAFT_GROUPS[name]
    )


# ============================================================
# 13G.3 Configuration
# ============================================================

GAFT_CONFIG = {

    "d_model":
        48,

    "n_heads":
        4,

    "n_layers":
        2,

    "ff_dim":
        96,

    "dropout":
        0.15,

    "batch_size":
        2048,

    "max_epochs":
        30,

    "inner_es_fraction":
        0.15,

    "patience":
        5,

    "lr":
        4e-4,

    "weight_decay":
        2e-4,

    # ranking objective
    "rank_weight":
        0.25,

    # moderate imbalance correction
    # actual value calculated fold-wise
    "use_sqrt_pos_weight":
        True,

    # pairwise subsampling
    "max_rank_side":
        128,
}


GAFT_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


print(
    "\nGAFT device:",
    GAFT_DEVICE
)


# ============================================================
# 13G.4 Reproducibility
# ============================================================

def gaft_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 13G.5 Dataset
# ============================================================

class GAFTDataset(Dataset):

    def __init__(
        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )


# ============================================================
# 13G.6 Group-aware numerical feature tokenizer
# ============================================================
#
# Each scalar feature gets:
#
#     token_j = x_j * W_j + B_j
#               + feature identity
#               + clinical-group identity
#
# Therefore the transformer knows:
#   - which variable it is
#   - which clinical group it belongs to
# ============================================================

class GroupAwareFeatureTokenizer(nn.Module):

    def __init__(
        self,
        n_features,
        n_groups,
        d_model
    ):

        super().__init__()

        self.weight = nn.Parameter(

            torch.randn(
                n_features,
                d_model
            )
            * 0.02
        )

        self.bias = nn.Parameter(

            torch.zeros(
                n_features,
                d_model
            )
        )

        self.feature_embedding = (
            nn.Parameter(

                torch.randn(
                    n_features,
                    d_model
                )
                * 0.02
            )
        )

        self.group_embedding = (
            nn.Embedding(
                n_groups,
                d_model
            )
        )


        self.register_buffer(

            "group_ids",

            torch.tensor(
                GAFT_GROUP_IDS,
                dtype=torch.long
            )
        )


    def forward(
        self,
        x
    ):

        # x:
        # B x F

        tokens = (

            x.unsqueeze(-1)
            * self.weight.unsqueeze(0)

            + self.bias.unsqueeze(0)

            + self.feature_embedding.unsqueeze(0)

            + self.group_embedding(
                self.group_ids
            ).unsqueeze(0)
        )

        return tokens


# ============================================================
# 13G.7 GAFT model
# ============================================================

class GAFTRankNet(nn.Module):

    def __init__(
        self,
        n_features,
        n_groups,
        d_model=48,
        n_heads=4,
        n_layers=2,
        ff_dim=96,
        dropout=0.15
    ):

        super().__init__()


        self.tokenizer = (
            GroupAwareFeatureTokenizer(

                n_features=
                    n_features,

                n_groups=
                    n_groups,

                d_model=
                    d_model
            )
        )


        self.cls_token = nn.Parameter(

            torch.randn(
                1,
                1,
                d_model
            )
            * 0.02
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=
                    d_model,

                nhead=
                    n_heads,

                dim_feedforward=
                    ff_dim,

                dropout=
                    dropout,

                activation=
                    "gelu",

                batch_first=True,

                norm_first=True,
            )
        )


        self.transformer = (
            nn.TransformerEncoder(

                encoder_layer,

                num_layers=
                    n_layers
            )
        )


        self.norm = (
            nn.LayerNorm(
                d_model
            )
        )


        self.head = (
            nn.Sequential(

                nn.Linear(
                    d_model,
                    32
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                ),

                nn.Linear(
                    32,
                    1
                )
            )
        )


    def forward(
        self,
        x
    ):

        feature_tokens = (
            self.tokenizer(
                x
            )
        )


        cls = self.cls_token.expand(

            x.shape[0],
            -1,
            -1
        )


        tokens = torch.cat(

            [
                cls,
                feature_tokens
            ],

            dim=1
        )


        encoded = (
            self.transformer(
                tokens
            )
        )


        cls_repr = (
            self.norm(
                encoded[
                    :,
                    0
                ]
            )
        )


        logit = (
            self.head(
                cls_repr
            )
            .squeeze(-1)
        )


        return (
            logit,
            cls_repr
        )


# ============================================================
# 13G.8 Pairwise ranking loss
# ============================================================

def gaft_pairwise_rank_loss(
    logits,
    y,
    max_side=128
):

    pos = logits[
        y > 0.5
    ]

    neg = logits[
        y <= 0.5
    ]


    if (
        len(pos) == 0
        or len(neg) == 0
    ):

        return torch.tensor(

            0.0,

            device=
                logits.device
        )


    if len(pos) > max_side:

        ids = torch.randperm(

            len(pos),

            device=
                logits.device
        )[
            :max_side
        ]

        pos = pos[
            ids
        ]


    if len(neg) > max_side:

        ids = torch.randperm(

            len(neg),

            device=
                logits.device
        )[
            :max_side
        ]

        neg = neg[
            ids
        ]


    difference = (

        pos[:, None]
        -
        neg[None, :]
    )


    return (
        F.softplus(
            -difference
        )
        .mean()
    )


# ============================================================
# 13G.9 Combined objective
# ============================================================

def gaft_loss(
    logits,
    y,
    pos_weight
):

    bce = (
        F.binary_cross_entropy_with_logits(

            logits,

            y,

            pos_weight=
                pos_weight
        )
    )


    ranking = (
        gaft_pairwise_rank_loss(

            logits,

            y,

            max_side=
                GAFT_CONFIG[
                    "max_rank_side"
                ]
        )
    )


    total = (

        bce

        + GAFT_CONFIG[
            "rank_weight"
        ]
        * ranking
    )


    return (
        total,
        bce,
        ranking
    )


# ============================================================
# 13G.10 Prediction helper
# ============================================================

def predict_gaft(
    model,
    X,
    batch_size=4096
):

    model.eval()

    pred = []
    reprs = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(

                start + batch_size,

                len(X)
            )


            xb = torch.tensor(

                X[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    GAFT_DEVICE
            )


            logits, representation = (
                model(
                    xb
                )
            )


            pred.append(

                torch.sigmoid(
                    logits
                )
                .cpu()
                .numpy()
            )


            reprs.append(

                representation
                .cpu()
                .numpy()
            )


    return (
        np.concatenate(
            pred
        ),

        np.concatenate(
            reprs
        )
    )


# ============================================================
# 13G.11 Model factory
# ============================================================

def make_gaft_model():

    return GAFTRankNet(

        n_features=
            len(
                GAFT_FEATURES
            ),

        n_groups=
            len(
                GROUP_NAMES_GAFT
            ),

        d_model=
            GAFT_CONFIG[
                "d_model"
            ],

        n_heads=
            GAFT_CONFIG[
                "n_heads"
            ],

        n_layers=
            GAFT_CONFIG[
                "n_layers"
            ],

        ff_dim=
            GAFT_CONFIG[
                "ff_dim"
            ],

        dropout=
            GAFT_CONFIG[
                "dropout"
            ],

    ).to(
        GAFT_DEVICE
    )


# ============================================================
# 13G.12 Epoch-selection stage
# ============================================================

def select_gaft_epoch(
    X_outer_train_raw,
    y_outer_train,
    seed
):

    splitter = (
        StratifiedShuffleSplit(

            n_splits=1,

            test_size=
                GAFT_CONFIG[
                    "inner_es_fraction"
                ],

            random_state=
                seed
        )
    )


    inner_train_idx, inner_es_idx = next(

        splitter.split(

            X_outer_train_raw,

            y_outer_train
        )
    )


    # --------------------------------------------------------
    # Preprocessing INNER TRAIN ONLY
    # --------------------------------------------------------

    scaler = (
        StandardScaler()
    )


    X_inner_train = (
        scaler.fit_transform(

            X_outer_train_raw[
                inner_train_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_inner_es = (
        scaler.transform(

            X_outer_train_raw[
                inner_es_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    y_inner_train = (
        y_outer_train[
            inner_train_idx
        ]
    )


    y_inner_es = (
        y_outer_train[
            inner_es_idx
        ]
    )


    # --------------------------------------------------------
    # moderate class imbalance weighting
    # --------------------------------------------------------

    n_pos = np.sum(
        y_inner_train == 1
    )

    n_neg = np.sum(
        y_inner_train == 0
    )


    ratio = (
        n_neg / n_pos
    )


    if GAFT_CONFIG[
        "use_sqrt_pos_weight"
    ]:

        weight_value = (
            np.sqrt(
                ratio
            )
        )

    else:

        weight_value = ratio


    pos_weight = torch.tensor(

        weight_value,

        dtype=torch.float32,

        device=
            GAFT_DEVICE
    )


    # --------------------------------------------------------
    # Loader
    # --------------------------------------------------------

    dataset = GAFTDataset(

        X_inner_train,

        y_inner_train
    )


    loader = DataLoader(

        dataset,

        batch_size=
            GAFT_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    gaft_seed(
        seed
    )


    model = (
        make_gaft_model()
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                GAFT_CONFIG[
                    "lr"
                ],

            weight_decay=
                GAFT_CONFIG[
                    "weight_decay"
                ]
        )
    )


    best_pr = -np.inf
    best_epoch = 1

    wait = 0

    history = []


    # ========================================================
    # Train / inner ES
    # ========================================================

    for epoch in range(
        1,
        GAFT_CONFIG[
            "max_epochs"
        ] + 1
    ):

        model.train()

        running_loss = 0.0
        running_bce = 0.0
        running_rank = 0.0
        seen = 0


        for xb, yb in loader:

            xb = xb.to(
                GAFT_DEVICE
            )

            yb = yb.to(
                GAFT_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, _ = (
                model(
                    xb
                )
            )


            (
                loss,
                bce,
                ranking
            ) = gaft_loss(

                logits,
                yb,
                pos_weight
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


            n = len(yb)

            running_loss += (
                float(
                    loss.detach()
                    .cpu()
                )
                * n
            )

            running_bce += (
                float(
                    bce.detach()
                    .cpu()
                )
                * n
            )

            running_rank += (
                float(
                    ranking.detach()
                    .cpu()
                )
                * n
            )

            seen += n


        # ----------------------------------------------------
        # inner ES performance
        # ----------------------------------------------------

        p_es, _ = predict_gaft(

            model,

            X_inner_es
        )


        es_pr = (
            average_precision_score(

                y_inner_es,

                p_es
            )
        )


        history.append({

            "Epoch":
                epoch,

            "Loss":
                running_loss
                / seen,

            "BCE":
                running_bce
                / seen,

            "RankLoss":
                running_rank
                / seen,

            "Inner_ES_PR":
                es_pr,
        })


        print(

            f"    epoch={epoch:02d} "
            f"PR={es_pr:.6f} "
            f"best={max(best_pr, es_pr):.6f}"
        )


        if es_pr > (
            best_pr + 1e-5
        ):

            best_pr = es_pr
            best_epoch = epoch
            wait = 0

        else:

            wait += 1


        if wait >= (
            GAFT_CONFIG[
                "patience"
            ]
        ):

            break


    return (
        best_epoch,
        best_pr,
        pd.DataFrame(
            history
        )
    )


# ============================================================
# 13G.13 Fixed-epoch complete outer-train refit
# ============================================================

def train_gaft_fixed(
    X_train,
    y_train,
    epochs,
    seed
):

    gaft_seed(
        seed
    )


    n_pos = np.sum(
        y_train == 1
    )

    n_neg = np.sum(
        y_train == 0
    )


    ratio = (
        n_neg / n_pos
    )


    pos_weight_value = (

        np.sqrt(
            ratio
        )

        if GAFT_CONFIG[
            "use_sqrt_pos_weight"
        ]

        else ratio
    )


    pos_weight = torch.tensor(

        pos_weight_value,

        dtype=torch.float32,

        device=
            GAFT_DEVICE
    )


    dataset = GAFTDataset(
        X_train,
        y_train
    )


    loader = DataLoader(

        dataset,

        batch_size=
            GAFT_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = (
        make_gaft_model()
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                GAFT_CONFIG[
                    "lr"
                ],

            weight_decay=
                GAFT_CONFIG[
                    "weight_decay"
                ]
        )
    )


    for epoch in range(
        epochs
    ):

        model.train()


        for xb, yb in loader:

            xb = xb.to(
                GAFT_DEVICE
            )

            yb = yb.to(
                GAFT_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, _ = (
                model(
                    xb
                )
            )


            loss, _, _ = (
                gaft_loss(

                    logits,
                    yb,
                    pos_weight
                )
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


    return model


# ============================================================
# 13G.14 Strict outer CV
# ============================================================

y_all = (
    y_dev.to_numpy(
        dtype=int
    )
)


X_gaft_raw = (

    X_dev[
        GAFT_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


GAFT_OOF = np.zeros(
    len(y_all),
    dtype=np.float64
)

GAFT_REPRESENTATION = np.zeros(
    (
        len(y_all),
        GAFT_CONFIG[
            "d_model"
        ]
    ),
    dtype=np.float32
)


fold_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"GAFT OUTER FOLD "
        f"{outer_fold + 1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    outer_train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]


    outer_valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    X_outer_train_raw = (
        X_gaft_raw[
            outer_train_idx
        ]
    )


    X_outer_valid_raw = (
        X_gaft_raw[
            outer_valid_idx
        ]
    )


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # Stage A: INNER epoch selection
    # ========================================================

    (
        selected_epoch,
        inner_best_pr,
        inner_history
    ) = select_gaft_epoch(

        X_outer_train_raw,

        y_outer_train,

        seed=
            SEED
            + 7100
            + outer_fold
    )


    print(
        "\nSelected epoch:",
        selected_epoch
    )

    print(
        "Inner best PR:",
        f"{inner_best_pr:.6f}"
    )


    # ========================================================
    # Stage B: preprocess complete outer training set
    # ========================================================

    outer_scaler = (
        StandardScaler()
    )


    X_outer_train = (
        outer_scaler.fit_transform(
            X_outer_train_raw
        )
        .astype(
            np.float32
        )
    )


    X_outer_valid = (
        outer_scaler.transform(
            X_outer_valid_raw
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # Stage C: fresh model, fixed selected epochs
    # ========================================================

    model = train_gaft_fixed(

        X_outer_train,

        y_outer_train,

        epochs=
            selected_epoch,

        seed=
            SEED
            + 8100
            + outer_fold
    )


    # ========================================================
    # Stage D: untouched outer validation
    # ========================================================

    p_valid, repr_valid = (
        predict_gaft(

            model,

            X_outer_valid
        )
    )


    GAFT_OOF[
        outer_valid_idx
    ] = p_valid


    GAFT_REPRESENTATION[
        outer_valid_idx
    ] = repr_valid


    pm = probability_metrics(

        y_outer_valid,

        p_valid
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            selected_epoch,

        "Inner_ES_PR":
            inner_best_pr,

        "PR_AUC":
            pm[
                "PR_AUC"
            ],

        "ROC_AUC":
            pm[
                "ROC_AUC"
            ],

        "Brier":
            pm[
                "Brier"
            ],

        "NLL":
            pm[
                "NLL"
            ],

        "ECE":
            pm[
                "ECE"
            ],
    })


    print(
        "Outer PR :",
        f"{pm['PR_AUC']:.6f}"
    )

    print(
        "Outer ROC:",
        f"{pm['ROC_AUC']:.6f}"
    )


    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13G.15 Global performance
# ============================================================

gaft_threshold, _, _ = (
    select_best_threshold(

        y_all,

        GAFT_OOF,

        objective="F1"
    )
)


gaft_metrics = (
    evaluate_binary_predictions(

        y_all,

        GAFT_OOF,

        threshold=
            gaft_threshold
    )
)


module13g_results = pd.DataFrame([

    {
        "Method":
            "GAFT_Rank",

        **gaft_metrics
    }
])


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 13G — GAFT-RANK RESULTS"
)

print(
    "=" * 125
)


display(
    module13g_results[
        [
            "Method",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13G.16 Fold stability
# ============================================================

module13g_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "GAFT-RANK FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module13g_folds
)


print(
    "\nPR mean:",
    module13g_folds[
        "PR_AUC"
    ].mean()
)

print(
    "PR std:",
    module13g_folds[
        "PR_AUC"
    ].std()
)


# ============================================================
# 13G.17 Compare to current candidates
# ============================================================

comparison_rows = [

    {
        "Model":
            "CatBoost",

        "PR_AUC":
            0.377657,
    },

    {
        "Model":
            "HistGradientBoosting",

        "PR_AUC":
            0.377012,
    },

    {
        "Model":
            "AutoInt",

        "PR_AUC":
            0.376859,
    },

    {
        "Model":
            "MLP_BCE",

        "PR_AUC":
            0.376499,
    },

    {
        "Model":
            "DeepFM_v2",

        "PR_AUC":
            0.374324,
    },

    {
        "Model":
            "GAFT_Rank",

        "PR_AUC":
            gaft_metrics[
                "PR_AUC"
            ],
    },
]


gaft_comparison = (
    pd.DataFrame(
        comparison_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "GAFT vs CURRENT STRONG EXPERTS"
)

print(
    "=" * 125
)


display(
    gaft_comparison
)


# ============================================================
# 13G.18 Diversity relative to Core4 experts
# ============================================================

diversity_rows = []


for model_name in [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]:

    raw_other = (
        OOF_BANK[
            model_name
        ]
        .to_numpy()
    )


    pearson = np.corrcoef(
        GAFT_OOF,
        raw_other
    )[0, 1]


    rank_gaft = pd.Series(
        GAFT_OOF
    ).rank(
        method="average"
    )


    rank_other = pd.Series(
        raw_other
    ).rank(
        method="average"
    )


    spearman = np.corrcoef(
        rank_gaft,
        rank_other
    )[0, 1]


    diversity_rows.append({

        "Compared_With":
            model_name,

        "Pearson":
            pearson,

        "Spearman":
            spearman,

        "Rank_Diversity":
            1.0 - spearman,
    })


gaft_diversity = pd.DataFrame(
    diversity_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "GAFT RANK DIVERSITY"
)

print(
    "=" * 125
)


display(
    gaft_diversity
)


# ============================================================
# 13G.19 Save
# ============================================================

save_csv(

    module13g_results,

    TAB_DIR
    / "module13g_gaft_rank_results.csv"
)


save_csv(

    module13g_folds,

    TAB_DIR
    / "module13g_gaft_rank_folds.csv"
)


save_csv(

    gaft_comparison,

    TAB_DIR
    / "module13g_gaft_comparison.csv"
)


save_csv(

    gaft_diversity,

    TAB_DIR
    / "module13g_gaft_diversity.csv"
)


gaft_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "GAFT_Rank":
        GAFT_OOF,
})


save_csv(

    gaft_pred_df,

    PRED_DIR
    / "module13g_gaft_rank_oof.csv"
)


np.save(

    PRED_DIR
    / "module13g_gaft_representations.npy",

    GAFT_REPRESENTATION
)


module13g_manifest = {

    "architecture":
        "Group-Aware Feature Transformer",

    "objective":
        "weighted_BCE_plus_pairwise_ranking",

    "clinical_groups":
        GAFT_GROUPS,

    "n_features":
        21,

    "nested_inner_epoch_selection":
        True,

    "fresh_outer_refit":
        True,

    "holdout_used":
        False,
}


save_json(

    module13g_manifest,

    LOG_DIR
    / "module13g_manifest.json"
)


print(
    "\nMODULE 13G COMPLETE."
)

GAFT feature groups:
cardiometabolic          ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Diabetes']
vascular_history         ['Stroke']
general_functional       ['GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk']
lifestyle                ['Smoker', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump']
demographic_access       ['Sex', 'Age', 'Education', 'Income', 'AnyHealthcare', 'NoDocbcCost']

GAFT device: cuda

GAFT OUTER FOLD 1/5
    epoch=01 PR=0.290289 best=0.290289
    epoch=02 PR=0.323654 best=0.323654
    epoch=03 PR=0.350096 best=0.350096
    epoch=04 PR=0.354643 best=0.354643
    epoch=05 PR=0.358113 best=0.358113
    epoch=06 PR=0.357648 best=0.358113
    epoch=07 PR=0.356834 best=0.358113
    epoch=08 PR=0.357983 best=0.358113
    epoch=09 PR=0.358668 best=0.358668
    epoch=10 PR=0.358281 best=0.358668
    epoch=11 PR=0.359104 best=0.359104
    epoch=12 PR=0.358733 best=0.359104
    epoch=13 PR=0.358816 best=0.359104
    epoch=14 PR=0.358864 best=0.359104
    epoch=15 PR

,Method,PR_AUC,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,GAFT_Rank,0.373355,0.848574,0.088031,0.284481,0.096311,0.333753,0.554881,0.416804,0.353409,0.44



GAFT-RANK FOLD RESULTS


,Fold,Selected_Epoch,Inner_ES_PR,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,0,30,0.361822,0.373166,0.850402,0.089122,0.287334,0.100780
1,1,26,0.369894,0.376424,0.849005,0.088436,0.286059,0.098561
2,2,29,0.378683,0.372062,0.846937,0.087017,0.280945,0.090332
3,3,18,0.360373,0.365370,0.847420,0.087629,0.282888,0.092952
4,4,16,0.385433,0.382421,0.849897,0.087949,0.285177,0.098929



PR mean: 0.3738886714982105
PR std: 0.006238043563895476

GAFT vs CURRENT STRONG EXPERTS


,Model,PR_AUC
0,CatBoost,0.377657
1,HistGradientBoosting,0.377012
2,AutoInt,0.376859
3,MLP_BCE,0.376499
4,DeepFM_v2,0.374324
5,GAFT_Rank,0.373355



GAFT RANK DIVERSITY


,Compared_With,Pearson,Spearman,Rank_Diversity
0,HistGradientBoosting,0.956238,0.985232,0.014768
1,CatBoost,0.957232,0.986083,0.013917
2,AutoInt,0.954840,0.986162,0.013838
3,DeepFM_v2,0.949229,0.983406,0.016594



MODULE 13G COMPLETE.


In [25]:
# ============================================================
# MODULE 13H — STRICT CORE4 + GAFT COMPLEMENTARITY TEST
# ============================================================
#
# Question:
# Does GAFT contain enough complementary signal to improve
# Core4 after proper probability calibration?
#
# Strict meta-CV:
# - calibrate GAFT on meta-train only
# - select blend weight on meta-train only
# - evaluate frozen policy on meta-valid
#
# HOLDOUT untouched.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score


# ============================================================
# 13H.1 Inputs
# ============================================================

y_all = y_dev.to_numpy(dtype=int)

core_all = (
    PCEA_CORE_OOF.copy()
)

gaft_raw = (
    GAFT_OOF.copy()
)

assert len(core_all) == len(y_all)
assert len(gaft_raw) == len(y_all)


# ============================================================
# 13H.2 Platt helper
# ============================================================

def fit_platt_1d(
    p_train,
    y_train,
    p_apply
):
    """
    Platt scaling using logit(probability) as single feature.
    """

    p_train = np.clip(
        p_train,
        1e-6,
        1.0 - 1e-6
    )

    p_apply = np.clip(
        p_apply,
        1e-6,
        1.0 - 1e-6
    )

    z_train = np.log(
        p_train / (1.0 - p_train)
    ).reshape(-1, 1)

    z_apply = np.log(
        p_apply / (1.0 - p_apply)
    ).reshape(-1, 1)

    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    model.fit(
        z_train,
        y_train
    )

    return (
        model.predict_proba(
            z_apply
        )[:, 1]
    )


# ============================================================
# 13H.3 Blend grid
# ============================================================
#
# w = GAFT contribution.
# Core contribution = 1-w.
#
# Keep GAFT contribution deliberately modest.
# ============================================================

WEIGHT_GRID = np.array([
    0.00,
    0.025,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
])


# ============================================================
# 13H.4 OOF buffers
# ============================================================

H_GAFT_CAL = np.zeros(
    len(y_all),
    dtype=float
)

H_BLEND = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []


# ============================================================
# 13H.5 Strict meta-CV
# ============================================================

for fold in range(N_SPLITS):

    print(
        "\n"
        + "=" * 95
    )

    print(
        f"CORE4 + GAFT META FOLD "
        f"{fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 95
    )

    train_idx = np.where(
        fold_assignment != fold
    )[0]

    valid_idx = np.where(
        fold_assignment == fold
    )[0]

    y_train = y_all[
        train_idx
    ]

    y_valid = y_all[
        valid_idx
    ]


    # --------------------------------------------------------
    # Meta-train-only GAFT calibration
    # --------------------------------------------------------

    gaft_train_cal = fit_platt_1d(
        gaft_raw[train_idx],
        y_train,
        gaft_raw[train_idx]
    )

    gaft_valid_cal = fit_platt_1d(
        gaft_raw[train_idx],
        y_train,
        gaft_raw[valid_idx]
    )


    H_GAFT_CAL[
        valid_idx
    ] = gaft_valid_cal


    # --------------------------------------------------------
    # Select blend using META TRAIN ONLY
    # --------------------------------------------------------

    candidate_rows = []

    for w in WEIGHT_GRID:

        p_train = (
            (1.0 - w)
            * core_all[
                train_idx
            ]
            +
            w
            * gaft_train_cal
        )

        pr = (
            average_precision_score(
                y_train,
                p_train
            )
        )

        pm = probability_metrics(
            y_train,
            p_train
        )

        candidate_rows.append({

            "GAFT_Weight":
                float(w),

            "PR_AUC":
                pr,

            "Brier":
                pm["Brier"],

            "NLL":
                pm["NLL"],
        })


    candidate_df = (
        pd.DataFrame(
            candidate_rows
        )
        .sort_values(
            [
                "PR_AUC",
                "Brier",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    best_w = float(
        candidate_df.iloc[0][
            "GAFT_Weight"
        ]
    )


    # --------------------------------------------------------
    # Frozen validation blend
    # --------------------------------------------------------

    p_valid = (
        (1.0 - best_w)
        * core_all[
            valid_idx
        ]
        +
        best_w
        * gaft_valid_cal
    )


    H_BLEND[
        valid_idx
    ] = p_valid


    core_pr = (
        average_precision_score(
            y_valid,
            core_all[
                valid_idx
            ]
        )
    )

    blend_pr = (
        average_precision_score(
            y_valid,
            p_valid
        )
    )


    fold_rows.append({

        "Fold":
            fold,

        "Selected_GAFT_Weight":
            best_w,

        "Core_PR":
            core_pr,

        "Blend_PR":
            blend_pr,

        "Delta_PR":
            blend_pr
            - core_pr,
    })


    print(
        "Selected GAFT weight:",
        best_w
    )

    print(
        f"Core PR : {core_pr:.6f}"
    )

    print(
        f"Blend PR: {blend_pr:.6f}"
    )

    print(
        f"Delta   : {blend_pr-core_pr:+.6f}"
    )


# ============================================================
# 13H.6 Global results
# ============================================================

core_t, _, _ = (
    select_best_threshold(
        y_all,
        core_all,
        objective="F1"
    )
)

blend_t, _, _ = (
    select_best_threshold(
        y_all,
        H_BLEND,
        objective="F1"
    )
)

gaft_t, _, _ = (
    select_best_threshold(
        y_all,
        H_GAFT_CAL,
        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(
        y_all,
        core_all,
        threshold=core_t
    )
)

gaft_metrics_cal = (
    evaluate_binary_predictions(
        y_all,
        H_GAFT_CAL,
        threshold=gaft_t
    )
)

blend_metrics = (
    evaluate_binary_predictions(
        y_all,
        H_BLEND,
        threshold=blend_t
    )
)


module13h_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",
        **core_metrics
    },

    {
        "Method":
            "GAFT_Platt",
        **gaft_metrics_cal
    },

    {
        "Method":
            "Core4_GAFT_Blend",
        **blend_metrics
    },
])


module13h_results[
    "Delta_PR_vs_Core"
] = (
    module13h_results[
        "PR_AUC"
    ]
    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 115
)

print(
    "MODULE 13H — CORE4 + GAFT RESULTS"
)

print(
    "=" * 115
)

display(
    module13h_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


module13h_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 115
)

print(
    "MODULE 13H — FOLD RESULTS"
)

print(
    "=" * 115
)

display(
    module13h_folds
)


print(
    "\nPositive folds:",
    int(
        (
            module13h_folds[
                "Delta_PR"
            ] > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "Mean delta:",
    module13h_folds[
        "Delta_PR"
    ].mean()
)


# ============================================================
# Save
# ============================================================

save_csv(
    module13h_results,
    TAB_DIR
    / "module13h_core4_gaft_results.csv"
)

save_csv(
    module13h_folds,
    TAB_DIR
    / "module13h_core4_gaft_folds.csv"
)


module13h_pred = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        core_all,

    "GAFT_Platt":
        H_GAFT_CAL,

    "Core4_GAFT_Blend":
        H_BLEND,
})


save_csv(
    module13h_pred,
    PRED_DIR
    / "module13h_core4_gaft_oof.csv"
)


print(
    "\nMODULE 13H COMPLETE."
)


CORE4 + GAFT META FOLD 1/5
Selected GAFT weight: 0.05
Core PR : 0.380069
Blend PR: 0.380000
Delta   : -0.000069

CORE4 + GAFT META FOLD 2/5
Selected GAFT weight: 0.0
Core PR : 0.382772
Blend PR: 0.382772
Delta   : +0.000000

CORE4 + GAFT META FOLD 3/5
Selected GAFT weight: 0.0
Core PR : 0.378467
Blend PR: 0.378467
Delta   : +0.000000

CORE4 + GAFT META FOLD 4/5
Selected GAFT weight: 0.0
Core PR : 0.372833
Blend PR: 0.372833
Delta   : +0.000000

CORE4 + GAFT META FOLD 5/5
Selected GAFT weight: 0.0
Core PR : 0.390072
Blend PR: 0.390072
Delta   : +0.000000

MODULE 13H — CORE4 + GAFT RESULTS


,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.22
1,GAFT_Platt,0.373208,-0.007119,0.848507,0.070100,0.236828,0.001543,0.333427,0.556608,0.417036,0.353785,0.21
2,Core4_GAFT_Blend,0.380355,0.000027,0.850557,0.069707,0.235586,0.002340,0.350354,0.526263,0.420659,0.356024,0.22



MODULE 13H — FOLD RESULTS


,Fold,Selected_GAFT_Weight,Core_PR,Blend_PR,Delta_PR
0,0,0.05,0.380069,0.380000,-0.000069
1,1,0.00,0.382772,0.382772,0.000000
2,2,0.00,0.378467,0.378467,0.000000
3,3,0.00,0.372833,0.372833,0.000000
4,4,0.00,0.390072,0.390072,0.000000



Positive folds: 0 / 5
Mean delta: -1.3822296702326042e-05

MODULE 13H COMPLETE.


In [26]:
# ============================================================
# MODULE 13I — CARE
# Complementarity-Aware Recovery Expert
# ============================================================
#
# Core idea:
#
#   Core4 score
#        +
#   bounded CARE residual
#
# CARE is NOT trained as an ordinary classifier.
#
# It emphasizes:
#   1. low-ranked positives
#   2. high-ranked negatives
#   3. Core4-misordered positive/negative pairs
#   4. residual complementarity / decorrelation
#
# DEVELOPMENT SCREENING.
# HOLDOUT untouched.
# ============================================================

import copy
import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import average_precision_score


# ============================================================
# 13I.1 Configuration
# ============================================================

CARE_CORE_MODELS = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]

CARE_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


CARE_CONFIG = {

    "hidden":
        (64, 32),

    "dropout":
        0.20,

    "batch_size":
        2048,

    "max_epochs":
        25,

    "patience":
        4,

    "inner_es_fraction":
        0.15,

    "lr":
        4e-4,

    "weight_decay":
        3e-4,

    # hard cases
    "hard_fraction":
        0.20,

    "hard_weight":
        3.0,

    # correction bound
    "max_logit_correction":
        0.20,

    # losses
    "alpha_rank":
        0.40,

    "beta_residual":
        0.08,

    "gamma_orthogonal":
        0.10,

    # local ranking-pair width
    "pair_margin":
        0.50,

    "max_pair_side":
        96,
}


CARE_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print("CARE device:", CARE_DEVICE)


# ============================================================
# 13I.2 Helpers
# ============================================================

def care_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def care_logit_np(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


# ============================================================
# 13I.3 CARE feature builder
# ============================================================

def build_care_features(
    row_idx,
    calibrated_dict
):

    Xclinical = (
        X_dev
        .iloc[
            row_idx
        ][
            CARE_FEATURES
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    core_matrix = np.column_stack([
        calibrated_dict[m]
        for m in CARE_CORE_MODELS
    ])

    core_mean = (
        core_matrix.mean(
            axis=1
        )
    )

    core_std = (
        core_matrix.std(
            axis=1
        )
    )

    core_range = (
        core_matrix.max(
            axis=1
        )
        -
        core_matrix.min(
            axis=1
        )
    )

    core_entropy = (
        binary_entropy(
            core_mean
        )
    )

    # expert deviation signals
    extra = [

        core_mean,
        care_logit_np(
            core_mean
        ),
        core_std,
        core_range,
        core_entropy,
    ]

    for j in range(
        core_matrix.shape[1]
    ):

        extra.append(
            core_matrix[:, j]
        )

        extra.append(
            core_matrix[:, j]
            - core_mean
        )

        extra.append(
            np.abs(
                core_matrix[:, j]
                - core_mean
            )
        )


    Xextra = np.column_stack(
        extra
    ).astype(
        np.float32
    )


    Xall = np.column_stack([
        Xclinical,
        Xextra
    ]).astype(
        np.float32
    )


    return (
        Xall,
        core_mean.astype(
            np.float32
        ),
        core_matrix.astype(
            np.float32
        ),
    )


# ============================================================
# 13I.4 Hard-case weights
# ============================================================

def care_hard_weights(
    y,
    core_prob
):

    y = np.asarray(
        y,
        dtype=int
    )

    core_prob = np.asarray(
        core_prob
    )


    weights = np.ones(
        len(y),
        dtype=np.float32
    )


    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]


    n_hard_pos = max(

        1,

        int(
            len(pos_idx)
            * CARE_CONFIG[
                "hard_fraction"
            ]
        )
    )


    n_hard_neg = max(

        1,

        int(
            len(neg_idx)
            * CARE_CONFIG[
                "hard_fraction"
            ]
        )
    )


    # lowest-scoring positives
    hard_pos = pos_idx[
        np.argsort(
            core_prob[
                pos_idx
            ]
        )[
            :n_hard_pos
        ]
    ]


    # highest-scoring negatives
    hard_neg = neg_idx[
        np.argsort(
            core_prob[
                neg_idx
            ]
        )[
            -n_hard_neg:
        ]
    ]


    weights[
        hard_pos
    ] = CARE_CONFIG[
        "hard_weight"
    ]

    weights[
        hard_neg
    ] = CARE_CONFIG[
        "hard_weight"
    ]


    hard_mask = np.zeros(
        len(y),
        dtype=bool
    )

    hard_mask[
        hard_pos
    ] = True

    hard_mask[
        hard_neg
    ] = True


    return (
        weights,
        hard_mask
    )


# ============================================================
# 13I.5 Dataset
# ============================================================

class CAREDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        core_prob,
        sample_weight
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.core_prob = torch.tensor(
            core_prob,
            dtype=torch.float32
        )

        self.sample_weight = torch.tensor(
            sample_weight,
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx],
            self.core_prob[idx],
            self.sample_weight[idx],
        )


# ============================================================
# 13I.6 CARE network
# ============================================================

class CARENet(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()


        layers = []

        previous = (
            input_dim
        )


        for hidden in CARE_CONFIG[
            "hidden"
        ]:

            layers.extend([

                nn.Linear(
                    previous,
                    hidden
                ),

                nn.GELU(),

                nn.LayerNorm(
                    hidden
                ),

                nn.Dropout(
                    CARE_CONFIG[
                        "dropout"
                    ]
                ),
            ])

            previous = hidden


        self.backbone = (
            nn.Sequential(
                *layers
            )
        )


        self.residual_head = (
            nn.Linear(
                previous,
                1
            )
        )


        # start from exactly Core4
        nn.init.zeros_(
            self.residual_head.weight
        )

        nn.init.zeros_(
            self.residual_head.bias
        )


    def forward(
        self,
        x
    ):

        h = self.backbone(
            x
        )


        raw = (
            self.residual_head(
                h
            )
            .squeeze(-1)
        )


        residual = (

            CARE_CONFIG[
                "max_logit_correction"
            ]

            * torch.tanh(
                raw
            )
        )


        return residual


# ============================================================
# 13I.7 Core-aware ranking loss
# ============================================================

def care_pairwise_loss(
    final_logit,
    core_logit,
    y
):

    pos_mask = (
        y > 0.5
    )

    neg_mask = (
        y <= 0.5
    )


    if (
        pos_mask.sum() == 0
        or neg_mask.sum() == 0
    ):

        return torch.tensor(
            0.0,
            device=
                final_logit.device
        )


    fp = final_logit[
        pos_mask
    ]

    fn = final_logit[
        neg_mask
    ]


    cp = core_logit[
        pos_mask
    ]

    cn = core_logit[
        neg_mask
    ]


    max_side = CARE_CONFIG[
        "max_pair_side"
    ]


    if len(fp) > max_side:

        ids = torch.randperm(
            len(fp),
            device=
                fp.device
        )[
            :max_side
        ]

        fp = fp[ids]
        cp = cp[ids]


    if len(fn) > max_side:

        ids = torch.randperm(
            len(fn),
            device=
                fn.device
        )[
            :max_side
        ]

        fn = fn[ids]
        cn = cn[ids]


    # --------------------------------------------------------
    # Core pair difference:
    # positive - negative
    #
    # Negative/near-zero = exactly the ranking mistakes
    # that CARE should focus on.
    # --------------------------------------------------------

    core_diff = (

        cp[:, None]
        -
        cn[None, :]
    )


    # Larger weight for misordered / close pairs
    pair_weight = torch.sigmoid(

        -core_diff
        /
        CARE_CONFIG[
            "pair_margin"
        ]
    )


    final_diff = (

        fp[:, None]
        -
        fn[None, :]
    )


    pair_loss = (
        F.softplus(
            -final_diff
        )
    )


    return (

        (
            pair_weight
            * pair_loss
        ).sum()

        /
        (
            pair_weight.sum()
            + 1e-8
        )
    )


# ============================================================
# 13I.8 Residual orthogonality loss
# ============================================================

def care_orthogonality_loss(
    residual,
    core_logit
):
    """
    Discourages CARE from learning merely a monotonic
    transformation of Core4.
    """

    r = (
        residual
        - residual.mean()
    )

    c = (
        core_logit
        - core_logit.mean()
    )


    covariance = (
        r * c
    ).mean()


    denominator = (

        torch.sqrt(
            r.pow(2).mean()
            + 1e-8
        )

        *

        torch.sqrt(
            c.pow(2).mean()
            + 1e-8
        )
    )


    correlation = (
        covariance
        /
        denominator
    )


    return (
        correlation.pow(2)
    )


# ============================================================
# 13I.9 Combined CARE loss
# ============================================================

def care_loss(
    residual,
    core_prob,
    y,
    sample_weight
):

    core_prob = torch.clamp(
        core_prob,
        1e-6,
        1.0 - 1e-6
    )


    core_logit = torch.log(

        core_prob
        /
        (
            1.0
            -
            core_prob
        )
    )


    final_logit = (
        core_logit
        + residual
    )


    # --------------------------------------------------------
    # Hard-case weighted classification
    # --------------------------------------------------------

    bce_each = (
        F.binary_cross_entropy_with_logits(

            final_logit,
            y,

            reduction="none"
        )
    )


    bce = (

        bce_each
        * sample_weight

    ).mean()


    # --------------------------------------------------------
    # Core-error ranking repair
    # --------------------------------------------------------

    rank = (
        care_pairwise_loss(

            final_logit,
            core_logit,
            y
        )
    )


    # --------------------------------------------------------
    # Small residual
    # --------------------------------------------------------

    residual_penalty = (
        residual
        .pow(2)
        .mean()
    )


    # --------------------------------------------------------
    # Explicit complementarity
    # --------------------------------------------------------

    orthogonal = (
        care_orthogonality_loss(

            residual,
            core_logit
        )
    )


    total = (

        bce

        + CARE_CONFIG[
            "alpha_rank"
        ]
        * rank

        + CARE_CONFIG[
            "beta_residual"
        ]
        * residual_penalty

        + CARE_CONFIG[
            "gamma_orthogonal"
        ]
        * orthogonal
    )


    return (
        total,
        final_logit,
        bce,
        rank,
        residual_penalty,
        orthogonal,
    )


# ============================================================
# 13I.10 Prediction
# ============================================================

def predict_care(
    model,
    X,
    core_prob,
    batch_size=4096
):

    model.eval()

    predictions = []
    residuals = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(
                start
                + batch_size,
                len(X)
            )


            xb = torch.tensor(

                X[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    CARE_DEVICE
            )


            cp = torch.tensor(

                core_prob[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    CARE_DEVICE
            )


            residual = (
                model(
                    xb
                )
            )


            cp = torch.clamp(
                cp,
                1e-6,
                1.0 - 1e-6
            )


            core_logit = torch.log(

                cp
                /
                (
                    1.0
                    -
                    cp
                )
            )


            final_logit = (
                core_logit
                + residual
            )


            probability = torch.sigmoid(
                final_logit
            )


            predictions.append(

                probability
                .cpu()
                .numpy()
            )


            residuals.append(

                residual
                .cpu()
                .numpy()
            )


    return (

        np.concatenate(
            predictions
        ),

        np.concatenate(
            residuals
        )
    )


# ============================================================
# 13I.11 Train model for fixed epochs
# ============================================================

def train_care_fixed(
    X_train,
    y_train,
    core_train,
    epochs,
    seed
):

    care_seed(
        seed
    )


    sample_weight, _ = (
        care_hard_weights(

            y_train,
            core_train
        )
    )


    dataset = CAREDataset(

        X_train,
        y_train,
        core_train,
        sample_weight
    )


    loader = DataLoader(

        dataset,

        batch_size=
            CARE_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = CARENet(

        input_dim=
            X_train.shape[1]

    ).to(
        CARE_DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            CARE_CONFIG[
                "lr"
            ],

        weight_decay=
            CARE_CONFIG[
                "weight_decay"
            ]
    )


    for epoch in range(
        epochs
    ):

        model.train()


        for (
            xb,
            yb,
            cp,
            sw
        ) in loader:

            xb = xb.to(
                CARE_DEVICE
            )

            yb = yb.to(
                CARE_DEVICE
            )

            cp = cp.to(
                CARE_DEVICE
            )

            sw = sw.to(
                CARE_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            residual = model(
                xb
            )


            loss, *_ = care_loss(

                residual,
                cp,
                yb,
                sw
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


    return model


# ============================================================
# 13I.12 Inner epoch selection
# ============================================================

def select_care_epoch(
    X_raw,
    y,
    core_prob,
    seed
):

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            CARE_CONFIG[
                "inner_es_fraction"
            ],

        random_state=
            seed
    )


    train_idx, es_idx = next(

        splitter.split(
            X_raw,
            y
        )
    )


    scaler = StandardScaler()


    Xtr = (
        scaler.fit_transform(
            X_raw[
                train_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    Xes = (
        scaler.transform(
            X_raw[
                es_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    ytr = y[
        train_idx
    ]

    yes = y[
        es_idx
    ]


    ctr = core_prob[
        train_idx
    ]

    ces = core_prob[
        es_idx
    ]


    care_seed(
        seed
    )


    sample_weight, _ = (
        care_hard_weights(
            ytr,
            ctr
        )
    )


    dataset = CAREDataset(

        Xtr,
        ytr,
        ctr,
        sample_weight
    )


    loader = DataLoader(

        dataset,

        batch_size=
            CARE_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True
    )


    model = CARENet(

        input_dim=
            Xtr.shape[1]

    ).to(
        CARE_DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            CARE_CONFIG[
                "lr"
            ],

        weight_decay=
            CARE_CONFIG[
                "weight_decay"
            ]
    )


    best_pr = -np.inf
    best_epoch = 1
    wait = 0


    for epoch in range(
        1,
        CARE_CONFIG[
            "max_epochs"
        ] + 1
    ):

        model.train()


        for (
            xb,
            yb,
            cp,
            sw
        ) in loader:

            xb = xb.to(
                CARE_DEVICE
            )

            yb = yb.to(
                CARE_DEVICE
            )

            cp = cp.to(
                CARE_DEVICE
            )

            sw = sw.to(
                CARE_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            residual = (
                model(
                    xb
                )
            )


            loss, *_ = (
                care_loss(

                    residual,
                    cp,
                    yb,
                    sw
                )
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


        # ----------------------------------------------------
        # inner validation only
        # ----------------------------------------------------

        p_es, residual_es = (
            predict_care(

                model,
                Xes,
                ces
            )
        )


        es_pr = (
            average_precision_score(

                yes,
                p_es
            )
        )


        print(

            f"    epoch={epoch:02d} "
            f"ES_PR={es_pr:.6f} "
            f"best={max(best_pr, es_pr):.6f} "
            f"|r|={np.mean(np.abs(residual_es)):.5f}"
        )


        if es_pr > (
            best_pr + 1e-5
        ):

            best_pr = (
                es_pr
            )

            best_epoch = (
                epoch
            )

            wait = 0

        else:

            wait += 1


        if wait >= (
            CARE_CONFIG[
                "patience"
            ]
        ):

            break


    return (
        best_epoch,
        best_pr
    )


# ============================================================
# 13I.13 Outer CV
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


CARE_CORE = np.zeros(
    len(y_all),
    dtype=float
)

CARE_OOF = np.zeros(
    len(y_all),
    dtype=float
)

CARE_RESIDUAL = np.zeros(
    len(y_all),
    dtype=float
)


fold_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"CARE OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]


    valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    y_train = y_all[
        train_idx
    ]

    y_valid = y_all[
        valid_idx
    ]


    # ========================================================
    # Fold-local calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for model_name in (
        CARE_CORE_MODELS
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(

                model_name,
                train_idx,
                valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_tr

        valid_cal[
            model_name
        ] = p_va


    # ========================================================
    # Build features
    # ========================================================

    (
        X_train_raw,
        core_train,
        _
    ) = build_care_features(

        train_idx,
        train_cal
    )


    (
        X_valid_raw,
        core_valid,
        _
    ) = build_care_features(

        valid_idx,
        valid_cal
    )


    # ========================================================
    # Inner epoch selection
    # ========================================================

    (
        selected_epoch,
        inner_pr
    ) = select_care_epoch(

        X_train_raw,

        y_train,

        core_train,

        seed=
            SEED
            + 9100
            + outer_fold
    )


    print(
        "Selected epoch:",
        selected_epoch
    )


    # ========================================================
    # Outer-train preprocessing
    # ========================================================

    scaler = StandardScaler()


    X_train = (
        scaler.fit_transform(
            X_train_raw
        )
        .astype(
            np.float32
        )
    )


    X_valid = (
        scaler.transform(
            X_valid_raw
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # Fresh refit
    # ========================================================

    model = train_care_fixed(

        X_train,
        y_train,
        core_train,

        epochs=
            selected_epoch,

        seed=
            SEED
            + 10100
            + outer_fold
    )


    # ========================================================
    # Untouched outer validation
    # ========================================================

    p_valid, residual_valid = (
        predict_care(

            model,

            X_valid,

            core_valid
        )
    )


    CARE_CORE[
        valid_idx
    ] = core_valid


    CARE_OOF[
        valid_idx
    ] = p_valid


    CARE_RESIDUAL[
        valid_idx
    ] = residual_valid


    core_pm = (
        probability_metrics(
            y_valid,
            core_valid
        )
    )


    care_pm = (
        probability_metrics(
            y_valid,
            p_valid
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            selected_epoch,

        "Inner_ES_PR":
            inner_pr,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "CARE_PR":
            care_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            care_pm[
                "PR_AUC"
            ]
            -
            core_pm[
                "PR_AUC"
            ],

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "CARE_ROC":
            care_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "CARE_Brier":
            care_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "CARE_ECE":
            care_pm[
                "ECE"
            ],

        "Mean_Abs_Residual":
            float(
                np.mean(
                    np.abs(
                        residual_valid
                    )
                )
            ),

        "P95_Abs_Residual":
            float(
                np.quantile(
                    np.abs(
                        residual_valid
                    ),
                    0.95
                )
            ),
    })


    print(
        f"Core PR : "
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        f"CARE PR : "
        f"{care_pm['PR_AUC']:.6f}"
    )

    print(
        f"Delta   : "
        f"{care_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )


    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 13I.14 Global evaluation
# ============================================================

core_t, _, _ = (
    select_best_threshold(
        y_all,
        CARE_CORE,
        objective="F1"
    )
)


care_t, _, _ = (
    select_best_threshold(
        y_all,
        CARE_OOF,
        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(

        y_all,
        CARE_CORE,
        threshold=core_t
    )
)


care_metrics = (
    evaluate_binary_predictions(

        y_all,
        CARE_OOF,
        threshold=care_t
    )
)


module13i_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "CARE",

        **care_metrics
    }
])


module13i_results[
    "Delta_PR_vs_Core"
] = (

    module13i_results[
        "PR_AUC"
    ]

    - core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 13I — CARE RESULTS"
)

print(
    "=" * 125
)


display(
    module13i_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 13I.15 Fold results
# ============================================================

module13i_folds = (
    pd.DataFrame(
        fold_rows
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "CARE FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module13i_folds
)


print(
    "\nMean fold Delta:",
    module13i_folds[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            module13i_folds[
                "Delta_PR"
            ]
            > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 13I.16 Complementarity diagnostics
# ============================================================

core_logit_all = (
    care_logit_np(
        CARE_CORE
    )
)


residual_core_corr = (
    np.corrcoef(

        CARE_RESIDUAL,

        core_logit_all
    )[0, 1]
)


abs_residual = (
    np.abs(
        CARE_RESIDUAL
    )
)


care_diagnostics = pd.DataFrame({

    "Metric": [

        "Residual/Core-logit correlation",

        "Mean abs residual",

        "Median abs residual",

        "P90 abs residual",

        "P95 abs residual",

        "P99 abs residual",

        "Max abs residual",

        "Fraction abs residual > 0.01",

        "Fraction abs residual > 0.05",

        "Fraction abs residual > 0.10",
    ],

    "Value": [

        float(
            residual_core_corr
        ),

        float(
            abs_residual.mean()
        ),

        float(
            np.median(
                abs_residual
            )
        ),

        float(
            np.quantile(
                abs_residual,
                .90
            )
        ),

        float(
            np.quantile(
                abs_residual,
                .95
            )
        ),

        float(
            np.quantile(
                abs_residual,
                .99
            )
        ),

        float(
            abs_residual.max()
        ),

        float(
            np.mean(
                abs_residual > .01
            )
        ),

        float(
            np.mean(
                abs_residual > .05
            )
        ),

        float(
            np.mean(
                abs_residual > .10
            )
        ),
    ],
})


print(
    "\n"
    + "=" * 125
)

print(
    "CARE COMPLEMENTARITY DIAGNOSTICS"
)

print(
    "=" * 125
)


display(
    care_diagnostics
)


# ============================================================
# 13I.17 Hard-region evaluation
# ============================================================

# Label-free inference difficulty:
# high disagreement + probability near decision/ranking middle

core_matrix_full = np.column_stack([

    CAL_OOF_BANK[
        m
    ].to_numpy()

    for m in CARE_CORE_MODELS
])


core_disagreement = (
    core_matrix_full.std(
        axis=1
    )
)


disagreement_threshold = (
    np.quantile(
        core_disagreement,
        0.80
    )
)


difficult_mask = (
    core_disagreement
    >= disagreement_threshold
)


hard_region_rows = []


for mask, region_name in [

    (
        difficult_mask,
        "HighDisagreement"
    ),

    (
        ~difficult_mask,
        "LowerDisagreement"
    ),
]:

    if (
        mask.sum() == 0
        or len(
            np.unique(
                y_all[
                    mask
                ]
            )
        ) < 2
    ):

        continue


    hard_region_rows.append({

        "Region":
            region_name,

        "N":
            int(
                mask.sum()
            ),

        "Fraction":
            float(
                mask.mean()
            ),

        "Prevalence":
            float(
                y_all[
                    mask
                ].mean()
            ),

        "Core_PR":
            average_precision_score(

                y_all[
                    mask
                ],

                CARE_CORE[
                    mask
                ]
            ),

        "CARE_PR":
            average_precision_score(

                y_all[
                    mask
                ],

                CARE_OOF[
                    mask
                ]
            ),
    })


care_region_results = (
    pd.DataFrame(
        hard_region_rows
    )
)


care_region_results[
    "Delta_PR"
] = (

    care_region_results[
        "CARE_PR"
    ]

    - care_region_results[
        "Core_PR"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "CARE REGION RESULTS"
)

print(
    "=" * 125
)


display(
    care_region_results
)


# ============================================================
# 13I.18 Save
# ============================================================

save_csv(

    module13i_results,

    TAB_DIR
    / "module13i_care_results.csv"
)


save_csv(

    module13i_folds,

    TAB_DIR
    / "module13i_care_folds.csv"
)


save_csv(

    care_diagnostics,

    TAB_DIR
    / "module13i_care_diagnostics.csv"
)


save_csv(

    care_region_results,

    TAB_DIR
    / "module13i_care_regions.csv"
)


care_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        CARE_CORE,

    "CARE":
        CARE_OOF,

    "CARE_Residual":
        CARE_RESIDUAL,
})


save_csv(

    care_pred_df,

    PRED_DIR
    / "module13i_care_oof.csv"
)


print(
    "\nMODULE 13I COMPLETE."
)

CARE device: cuda

CARE OUTER FOLD 1/5
    epoch=01 ES_PR=0.378281 best=0.378281 |r|=0.00992
    epoch=02 ES_PR=0.378196 best=0.378281 |r|=0.02619
    epoch=03 ES_PR=0.378213 best=0.378281 |r|=0.06030
    epoch=04 ES_PR=0.378214 best=0.378281 |r|=0.10730
    epoch=05 ES_PR=0.378108 best=0.378281 |r|=0.14286
Selected epoch: 1
Core PR : 0.380069
CARE PR : 0.380156
Delta   : +0.000087

CARE OUTER FOLD 2/5
    epoch=01 ES_PR=0.376982 best=0.376982 |r|=0.00691
    epoch=02 ES_PR=0.376886 best=0.376982 |r|=0.01552
    epoch=03 ES_PR=0.376707 best=0.376982 |r|=0.03618
    epoch=04 ES_PR=0.376664 best=0.376982 |r|=0.08484
    epoch=05 ES_PR=0.376704 best=0.376982 |r|=0.12829
Selected epoch: 1
Core PR : 0.382772
CARE PR : 0.382805
Delta   : +0.000034

CARE OUTER FOLD 3/5
    epoch=01 ES_PR=0.372148 best=0.372148 |r|=0.01049
    epoch=02 ES_PR=0.372102 best=0.372148 |r|=0.02503
    epoch=03 ES_PR=0.371969 best=0.372148 |r|=0.05324
    epoch=04 ES_PR=0.371866 best=0.372148 |r|=0.09068
    epoch=0

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.220
1,CARE,0.380022,-0.000305,0.850489,0.069741,0.235676,0.002952,0.351185,0.526944,0.421475,0.356956,0.215



CARE FOLD RESULTS


,Fold,Selected_Epoch,Inner_ES_PR,Core_PR,CARE_PR,Delta_PR,Core_ROC,CARE_ROC,Core_Brier,CARE_Brier,Core_ECE,CARE_ECE,Mean_Abs_Residual,P95_Abs_Residual
0,0,1,0.378281,0.380069,0.380156,0.000087,0.852660,0.852677,0.069637,0.069643,0.003499,0.004303,0.010355,0.019600
1,1,1,0.376982,0.382772,0.382805,0.000034,0.850766,0.850749,0.069684,0.069684,0.004297,0.004573,0.010832,0.023321
2,2,1,0.372148,0.378467,0.378394,-0.000072,0.849030,0.849031,0.069827,0.069830,0.002486,0.002123,0.008898,0.020593
3,3,2,0.389556,0.372833,0.372818,-0.000015,0.849083,0.849085,0.070081,0.070125,0.005053,0.006343,0.023633,0.045597
4,4,2,0.377431,0.390072,0.389787,-0.000286,0.851760,0.851548,0.069301,0.069422,0.005318,0.007214,0.047145,0.095646



Mean fold Delta: -5.0485581977977036e-05
Positive folds: 2 / 5

CARE COMPLEMENTARITY DIAGNOSTICS


,Metric,Value
0,Residual/Core-logit correlation,-0.025287
1,Mean abs residual,0.020172
2,Median abs residual,0.013408
3,P90 abs residual,0.045072
4,P95 abs residual,0.077914
5,P99 abs residual,0.095646
6,Max abs residual,0.099653
7,Fraction abs residual > 0.01,0.613741
8,Fraction abs residual > 0.05,0.078238
9,Fraction abs residual > 0.10,0.000000



CARE REGION RESULTS


,Region,N,Fraction,Prevalence,Core_PR,CARE_PR,Delta_PR
0,HighDisagreement,40589,0.200001,0.264579,0.459715,0.459195,-0.00052
1,LowerDisagreement,162355,0.799999,0.051584,0.244686,0.244695,0.00001



MODULE 13I COMPLETE.


In [27]:
# ============================================================
# MODULE 14A — TARGETED CATBOOST OPTIMIZATION
# ============================================================
#
# Goal:
# Improve the strongest base learner directly.
#
# Protocol:
# - outer 5-fold CV
# - each outer fold:
#       inner train / inner ES split
#       evaluate a restrained set of CatBoost configs
#       select by PR-AUC
#       retrain selected config on full outer-train
#       predict untouched outer-valid
#
# HOLDOUT untouched.
# ============================================================

import gc
import json
import random
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import average_precision_score


# ============================================================
# 14A.1 Configuration
# ============================================================

CATOPT_CONFIG = {

    "n_candidates":
        24,

    "inner_es_fraction":
        0.15,

    "max_iterations":
        2500,

    "early_stopping_rounds":
        120,

    "verbose":
        False,
}


# ============================================================
# 14A.2 Search space
# ============================================================

CATOPT_SPACE = {

    "depth":
        [4, 5, 6, 7, 8],

    "learning_rate":
        [0.015, 0.025, 0.035, 0.05, 0.07],

    "l2_leaf_reg":
        [2, 4, 6, 8, 12, 20],

    "random_strength":
        [0.0, 0.25, 0.5, 1.0, 2.0],

    "bagging_temperature":
        [0.0, 0.25, 0.5, 1.0, 2.0],

    "border_count":
        [32, 64, 128, 254],

    # positive-class multiplier
    "positive_weight":
        [1.0, 1.25, 1.5, 2.0],
}


# ============================================================
# 14A.3 Candidate sampler
# ============================================================

def sample_catboost_configs(
    n_candidates,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    configs = []

    seen = set()

    while len(configs) < n_candidates:

        cfg = {

            k:
                rng.choice(v).item()

            for k, v
            in CATOPT_SPACE.items()
        }


        key = tuple(
            sorted(
                cfg.items()
            )
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        configs.append(
            cfg
        )


    return configs


# ============================================================
# 14A.4 Build model
# ============================================================

def make_catopt_model(
    cfg,
    seed,
    iterations=None
):

    if iterations is None:

        iterations = (
            CATOPT_CONFIG[
                "max_iterations"
            ]
        )


    return CatBoostClassifier(

        iterations=
            iterations,

        depth=
            int(
                cfg[
                    "depth"
                ]
            ),

        learning_rate=
            float(
                cfg[
                    "learning_rate"
                ]
            ),

        l2_leaf_reg=
            float(
                cfg[
                    "l2_leaf_reg"
                ]
            ),

        random_strength=
            float(
                cfg[
                    "random_strength"
                ]
            ),

        bagging_temperature=
            float(
                cfg[
                    "bagging_temperature"
                ]
            ),

        border_count=
            int(
                cfg[
                    "border_count"
                ]
            ),

        class_weights=[
            1.0,
            float(
                cfg[
                    "positive_weight"
                ]
            )
        ],

        loss_function=
            "Logloss",

        eval_metric=
            "PRAUC",

        random_seed=
            seed,

        task_type=
            "GPU"
            if torch.cuda.is_available()
            else "CPU",

        devices=
            "0",

        verbose=
            False,

        allow_writing_files=
            False,
    )


# ============================================================
# 14A.5 Data
# ============================================================

X_cat = (
    X_dev[
        [
            "HighBP",
            "HighChol",
            "CholCheck",
            "BMI",
            "Smoker",
            "Stroke",
            "Diabetes",
            "PhysActivity",
            "Fruits",
            "Veggies",
            "HvyAlcoholConsump",
            "AnyHealthcare",
            "NoDocbcCost",
            "GenHlth",
            "MentHlth",
            "PhysHlth",
            "DiffWalk",
            "Sex",
            "Age",
            "Education",
            "Income",
        ]
    ]
    .to_numpy()
)

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 14A.6 Outer CV buffers
# ============================================================

CATOPT_OOF = np.zeros(
    len(y_all),
    dtype=float
)

fold_rows = []
config_rows = []


# ============================================================
# 14A.7 Outer folds
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"CATBOOST OPT OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    outer_train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]

    outer_valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    X_outer_train = X_cat[
        outer_train_idx
    ]

    y_outer_train = y_all[
        outer_train_idx
    ]


    X_outer_valid = X_cat[
        outer_valid_idx
    ]

    y_outer_valid = y_all[
        outer_valid_idx
    ]


    # --------------------------------------------------------
    # inner split
    # --------------------------------------------------------

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            CATOPT_CONFIG[
                "inner_es_fraction"
            ],

        random_state=
            SEED
            + 14000
            + outer_fold
    )


    inner_train_idx, inner_es_idx = next(

        splitter.split(
            X_outer_train,
            y_outer_train
        )
    )


    X_inner_train = X_outer_train[
        inner_train_idx
    ]

    y_inner_train = y_outer_train[
        inner_train_idx
    ]


    X_inner_es = X_outer_train[
        inner_es_idx
    ]

    y_inner_es = y_outer_train[
        inner_es_idx
    ]


    # --------------------------------------------------------
    # candidate configs
    # --------------------------------------------------------

    candidates = (
        sample_catboost_configs(

            CATOPT_CONFIG[
                "n_candidates"
            ],

            seed=
                SEED
                + 15000
                + outer_fold
        )
    )


    candidate_results = []


    for candidate_id, cfg in enumerate(
        candidates
    ):

        model = make_catopt_model(

            cfg,

            seed=
                SEED
                + 16000
                + outer_fold * 100
                + candidate_id
        )


        model.fit(

            X_inner_train,
            y_inner_train,

            eval_set=(
                X_inner_es,
                y_inner_es
            ),

            early_stopping_rounds=
                CATOPT_CONFIG[
                    "early_stopping_rounds"
                ],

            verbose=False
        )


        p_es = (
            model.predict_proba(
                X_inner_es
            )[:, 1]
        )


        pr = (
            average_precision_score(

                y_inner_es,
                p_es
            )
        )


        best_iteration = (
            model.get_best_iteration()
        )


        candidate_results.append({

            "Candidate":
                candidate_id,

            "PR_AUC":
                pr,

            "Best_Iteration":
                int(
                    best_iteration
                ),

            **cfg
        })


        print(

            f"  candidate={candidate_id:02d} "
            f"PR={pr:.6f} "
            f"iter={best_iteration}"
        )


        del model

        gc.collect()


    candidate_df = (
        pd.DataFrame(
            candidate_results
        )
        .sort_values(
            "PR_AUC",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    best_row = (
        candidate_df.iloc[
            0
        ]
    )


    best_cfg = {

        k:
            best_row[k]

        for k in CATOPT_SPACE.keys()
    }


    best_iteration = max(

        50,

        int(
            best_row[
                "Best_Iteration"
            ]
        )
        + 1
    )


    print(
        "\nBest config:"
    )

    print(
        best_cfg
    )

    print(
        "Best inner PR:",
        f"{best_row['PR_AUC']:.6f}"
    )

    print(
        "Best iteration:",
        best_iteration
    )


    # --------------------------------------------------------
    # fresh refit on complete outer training
    # --------------------------------------------------------

    final_model = make_catopt_model(

        best_cfg,

        seed=
            SEED
            + 17000
            + outer_fold,

        iterations=
            best_iteration
    )


    final_model.fit(

        X_outer_train,
        y_outer_train,

        verbose=False
    )


    # --------------------------------------------------------
    # untouched outer validation
    # --------------------------------------------------------

    p_valid = (
        final_model.predict_proba(
            X_outer_valid
        )[:, 1]
    )


    CATOPT_OOF[
        outer_valid_idx
    ] = p_valid


    pm = (
        probability_metrics(
            y_outer_valid,
            p_valid
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "PR_AUC":
            pm[
                "PR_AUC"
            ],

        "ROC_AUC":
            pm[
                "ROC_AUC"
            ],

        "Brier":
            pm[
                "Brier"
            ],

        "NLL":
            pm[
                "NLL"
            ],

        "ECE":
            pm[
                "ECE"
            ],

        "Best_Inner_PR":
            float(
                best_row[
                    "PR_AUC"
                ]
            ),

        "Best_Iteration":
            best_iteration,

        **best_cfg
    })


    candidate_df[
        "Outer_Fold"
    ] = outer_fold


    config_rows.append(
        candidate_df
    )


    print(
        "\nOuter PR:",
        f"{pm['PR_AUC']:.6f}"
    )


    del final_model

    gc.collect()


# ============================================================
# 14A.8 Overall metrics
# ============================================================

catopt_threshold, _, _ = (
    select_best_threshold(

        y_all,

        CATOPT_OOF,

        objective="F1"
    )
)


catopt_metrics = (
    evaluate_binary_predictions(

        y_all,

        CATOPT_OOF,

        threshold=
            catopt_threshold
    )
)


module14a_results = pd.DataFrame([

    {
        "Method":
            "CatBoost_Optimized",

        **catopt_metrics
    }
])


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 14A — OPTIMIZED CATBOOST"
)

print(
    "=" * 125
)


display(
    module14a_results
)


module14a_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "OPTIMIZED CATBOOST FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module14a_folds
)


print(
    "\nMean PR:",
    module14a_folds[
        "PR_AUC"
    ].mean()
)

print(
    "Std PR:",
    module14a_folds[
        "PR_AUC"
    ].std()
)


# ============================================================
# 14A.9 Compare with original CatBoost
# ============================================================

original_catboost = (
    OOF_BANK[
        "CatBoost"
    ]
    .to_numpy()
)


orig_pr = (
    average_precision_score(
        y_all,
        original_catboost
    )
)


opt_pr = (
    average_precision_score(
        y_all,
        CATOPT_OOF
    )
)


print(
    "\nOriginal CatBoost PR:",
    f"{orig_pr:.6f}"
)

print(
    "Optimized CatBoost PR:",
    f"{opt_pr:.6f}"
)

print(
    "Delta:",
    f"{opt_pr-orig_pr:+.6f}"
)


# ============================================================
# 14A.10 Diversity vs Core4
# ============================================================

diversity_rows = []


for model_name in (
    CARE_CORE_MODELS
):

    p_other = (
        OOF_BANK[
            model_name
        ]
        .to_numpy()
    )


    pearson = np.corrcoef(
        CATOPT_OOF,
        p_other
    )[0, 1]


    rank_new = (
        pd.Series(
            CATOPT_OOF
        )
        .rank()
    )


    rank_other = (
        pd.Series(
            p_other
        )
        .rank()
    )


    spearman = np.corrcoef(
        rank_new,
        rank_other
    )[0, 1]


    diversity_rows.append({

        "Compared_With":
            model_name,

        "Pearson":
            pearson,

        "Spearman":
            spearman,
    })


catopt_diversity = pd.DataFrame(
    diversity_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "OPTIMIZED CATBOOST DIVERSITY"
)

print(
    "=" * 125
)


display(
    catopt_diversity
)


# ============================================================
# 14A.11 Save
# ============================================================

save_csv(

    module14a_results,

    TAB_DIR
    / "module14a_catboost_optimized_results.csv"
)


save_csv(

    module14a_folds,

    TAB_DIR
    / "module14a_catboost_optimized_folds.csv"
)


save_csv(

    catopt_diversity,

    TAB_DIR
    / "module14a_catboost_optimized_diversity.csv"
)


save_csv(

    pd.concat(
        config_rows,
        ignore_index=True
    ),

    TAB_DIR
    / "module14a_catboost_search_all.csv"
)


save_csv(

    pd.DataFrame({

        "target":
            y_all,

        "fold":
            fold_assignment,

        "CatBoost_Optimized":
            CATOPT_OOF,
    }),

    PRED_DIR
    / "module14a_catboost_optimized_oof.csv"
)


print(
    "\nMODULE 14A COMPLETE."
)


CATBOOST OPT OUTER FOLD 1/5


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


  candidate=00 PR=0.390906 iter=167


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


  candidate=01 PR=0.390758 iter=2266


...


OPTIMIZED CATBOOST FOLD RESULTS


,Fold,PR_AUC,ROC_AUC,Brier,NLL,ECE,Best_Inner_PR,Best_Iteration,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature,border_count,positive_weight
0,0,0.376553,0.852003,0.075987,0.254065,0.057925,0.391614,763,6.0,0.025,6.0,2.00,2.00,128.0,2.00
1,1,0.379477,0.850026,0.075776,0.255050,0.058677,0.392936,163,7.0,0.035,2.0,2.00,1.00,128.0,2.00
2,2,0.377856,0.849214,0.071657,0.242039,0.030563,0.378089,150,7.0,0.050,8.0,0.25,0.25,254.0,1.50
3,3,0.372333,0.848800,0.070711,0.238523,0.016152,0.382294,495,7.0,0.025,12.0,0.25,0.25,128.0,1.25
4,4,0.389281,0.852033,0.075134,0.251983,0.055435,0.375485,1079,6.0,0.015,4.0,1.00,1.00,32.0,2.00



Mean PR: 0.3791000082503511
Std PR: 0.00627768953143518

Original CatBoost PR: 0.377657
Optimized CatBoost PR: 0.373886
Delta: -0.003771

OPTIMIZED CATBOOST DIVERSITY


,Compared_With,Pearson,Spearman
0,HistGradientBoosting,0.975778,0.984255
1,CatBoost,0.925252,0.984699
2,AutoInt,0.917631,0.985537
3,DeepFM_v2,0.913761,0.979998



MODULE 14A COMPLETE.


In [28]:
# ============================================================
# MODULE 14B — FAST HPRR
# Hard-Positive Retrieval-Augmented Repair
# ============================================================
#
# FAST DEVELOPMENT VERSION
#
# Core idea:
#
# Raw BRFSS features
#      ↓
# small hard-case-aware embedding network
#      ↓
# retrieve similar positive + negative training patients
#      ↓
# positive-vs-negative similarity evidence
#      ↓
# fuse retrieval evidence with Core4
#
# IMPORTANT:
# - only outer TRAIN cases are used as retrieval memory
# - outer VALID never retrieves another validation case
# - no HOLDOUT
# - fixed training epochs -> no outer-fold early-stopping leak
#
# Designed to run quickly on T4 GPU.
# ============================================================

import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score


# ============================================================
# 14B.1 Configuration
# ============================================================

HPRR_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


HPRR_CORE = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]


HPRR_CONFIG = {

    # --------------------------------------------------------
    # very small embedding network
    # --------------------------------------------------------

    "hidden_dim":
        64,

    "embedding_dim":
        24,

    "dropout":
        0.15,

    "epochs":
        3,

    "batch_size":
        2048,

    "lr":
        7e-4,

    "weight_decay":
        2e-4,


    # --------------------------------------------------------
    # hard-case training
    # --------------------------------------------------------

    "hard_fraction":
        0.25,

    "hard_weight":
        2.5,


    # --------------------------------------------------------
    # outer-train split
    #
    # 82% embedding/memory
    # 18% retrieval-fusion calibration
    # --------------------------------------------------------

    "fusion_fraction":
        0.18,


    # --------------------------------------------------------
    # retrieval memory sizes
    # --------------------------------------------------------

    "max_positive_memory":
        6000,

    "max_negative_memory":
        12000,


    # nearest neighbors PER CLASS
    # --------------------------------------------------------

    "k":
        32,


    # retrieval batch
    # --------------------------------------------------------

    "query_chunk":
        2048,
}


HPRR_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


print(
    "HPRR device:",
    HPRR_DEVICE
)


# ============================================================
# 14B.2 Reproducibility
# ============================================================

def hprr_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 14B.3 Helpers
# ============================================================

def hprr_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


# ============================================================
# 14B.4 Hard-case training weights
# ============================================================
#
# Low-scoring positives and high-scoring negatives
# receive extra importance during embedding learning.
# ============================================================

def make_hprr_weights(
    y,
    core_prob
):

    y = np.asarray(
        y,
        dtype=int
    )

    core_prob = np.asarray(
        core_prob,
        dtype=float
    )


    weights = np.ones(
        len(y),
        dtype=np.float32
    )


    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]


    n_pos_hard = max(
        1,
        int(
            len(pos_idx)
            * HPRR_CONFIG[
                "hard_fraction"
            ]
        )
    )

    n_neg_hard = max(
        1,
        int(
            len(neg_idx)
            * HPRR_CONFIG[
                "hard_fraction"
            ]
        )
    )


    # hardest positives = lowest Core4 scores
    hard_pos = pos_idx[
        np.argsort(
            core_prob[
                pos_idx
            ]
        )[:n_pos_hard]
    ]


    # hardest negatives = highest Core4 scores
    hard_neg = neg_idx[
        np.argsort(
            core_prob[
                neg_idx
            ]
        )[-n_neg_hard:]
    ]


    weights[
        hard_pos
    ] = HPRR_CONFIG[
        "hard_weight"
    ]

    weights[
        hard_neg
    ] = HPRR_CONFIG[
        "hard_weight"
    ]


    return weights


# ============================================================
# 14B.5 Dataset
# ============================================================

class HPRRDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        weights
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.weights = torch.tensor(
            weights,
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx],
            self.weights[idx],
        )


# ============================================================
# 14B.6 Lightweight retrieval embedding
# ============================================================

class HPRREmbeddingNet(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()


        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                HPRR_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.GELU(),

            nn.LayerNorm(
                HPRR_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.Dropout(
                HPRR_CONFIG[
                    "dropout"
                ]
            ),

            nn.Linear(
                HPRR_CONFIG[
                    "hidden_dim"
                ],
                HPRR_CONFIG[
                    "embedding_dim"
                ]
            ),
        )


        self.classifier = nn.Linear(

            HPRR_CONFIG[
                "embedding_dim"
            ],

            1
        )


    def forward(
        self,
        x
    ):

        embedding = (
            self.encoder(
                x
            )
        )


        # normalized embedding for cosine retrieval
        embedding_norm = (
            F.normalize(
                embedding,
                p=2,
                dim=1
            )
        )


        logit = (
            self.classifier(
                embedding
            )
            .squeeze(-1)
        )


        return (
            logit,
            embedding_norm
        )


# ============================================================
# 14B.7 Train embedding
# ============================================================

def train_hprr_embedding(
    X,
    y,
    core_prob,
    seed
):

    hprr_seed(
        seed
    )


    weights = (
        make_hprr_weights(
            y,
            core_prob
        )
    )


    dataset = HPRRDataset(
        X,
        y,
        weights
    )


    loader = DataLoader(

        dataset,

        batch_size=
            HPRR_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = HPRREmbeddingNet(

        input_dim=
            X.shape[1]

    ).to(
        HPRR_DEVICE
    )


    optimizer = (
        torch.optim.AdamW(

            model.parameters(),

            lr=
                HPRR_CONFIG[
                    "lr"
                ],

            weight_decay=
                HPRR_CONFIG[
                    "weight_decay"
                ],
        )
    )


    # mild imbalance correction only
    n_pos = np.sum(
        y == 1
    )

    n_neg = np.sum(
        y == 0
    )


    pos_weight_value = np.sqrt(
        n_neg / n_pos
    )


    pos_weight = torch.tensor(

        pos_weight_value,

        dtype=torch.float32,

        device=
            HPRR_DEVICE
    )


    for epoch in range(
        HPRR_CONFIG[
            "epochs"
        ]
    ):

        model.train()

        total_loss = 0.0
        seen = 0


        for xb, yb, wb in loader:

            xb = xb.to(
                HPRR_DEVICE
            )

            yb = yb.to(
                HPRR_DEVICE
            )

            wb = wb.to(
                HPRR_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, _ = (
                model(
                    xb
                )
            )


            raw_loss = (
                F.binary_cross_entropy_with_logits(

                    logits,
                    yb,

                    pos_weight=
                        pos_weight,

                    reduction="none"
                )
            )


            loss = (
                raw_loss
                * wb
            ).mean()


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


            total_loss += (
                float(
                    loss.detach()
                    .cpu()
                )
                * len(yb)
            )

            seen += len(yb)


        print(

            f"    embedding epoch "
            f"{epoch+1}/"
            f"{HPRR_CONFIG['epochs']} "
            f"loss="
            f"{total_loss/seen:.6f}"
        )


    return model


# ============================================================
# 14B.8 Get embeddings
# ============================================================

def get_hprr_embeddings(
    model,
    X,
    batch_size=4096
):

    model.eval()

    embeddings = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X)
            )


            xb = torch.tensor(

                X[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    HPRR_DEVICE
            )


            _, emb = model(
                xb
            )


            embeddings.append(

                emb.cpu()
                .numpy()
            )


    return np.concatenate(
        embeddings,
        axis=0
    )


# ============================================================
# 14B.9 Hard-positive-aware memory sampling
# ============================================================
#
# We maintain SEPARATE positive and negative memory banks.
#
# Therefore oversampling hard positives does NOT distort
# a class-frequency estimate.
# ============================================================

def select_retrieval_memory(
    y,
    core_prob,
    seed
):

    rng = np.random.default_rng(
        seed
    )


    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]


    # --------------------------------------------------------
    # positive memory:
    # prioritize low-scoring / hard positives
    # --------------------------------------------------------

    pos_hardness = (
        1.0
        - core_prob[
            pos_idx
        ]
    )

    pos_sampling_weight = (
        0.25
        +
        pos_hardness
    )

    pos_sampling_weight /= (
        pos_sampling_weight.sum()
    )


    n_pos = min(

        HPRR_CONFIG[
            "max_positive_memory"
        ],

        len(pos_idx)
    )


    selected_pos = rng.choice(

        pos_idx,

        size=
            n_pos,

        replace=False,

        p=
            pos_sampling_weight
    )


    # --------------------------------------------------------
    # negative memory:
    # prioritize high-scoring / confusing negatives
    # --------------------------------------------------------

    neg_hardness = (
        core_prob[
            neg_idx
        ]
    )

    neg_sampling_weight = (
        0.05
        +
        neg_hardness
    )

    neg_sampling_weight /= (
        neg_sampling_weight.sum()
    )


    n_neg = min(

        HPRR_CONFIG[
            "max_negative_memory"
        ],

        len(neg_idx)
    )


    selected_neg = rng.choice(

        neg_idx,

        size=
            n_neg,

        replace=False,

        p=
            neg_sampling_weight
    )


    return (
        selected_pos,
        selected_neg
    )


# ============================================================
# 14B.10 Fast GPU retrieval
# ============================================================
#
# Output per query:
#
# - positive nearest-neighbor mean similarity
# - negative nearest-neighbor mean similarity
# - best positive similarity
# - best negative similarity
# - retrieval margin
# - support / confidence
# ============================================================

def retrieve_hprr_features(
    query_embeddings,
    positive_memory,
    negative_memory,
    k=32
):

    pos_memory = torch.tensor(

        positive_memory,

        dtype=torch.float32,

        device=
            HPRR_DEVICE
    )


    neg_memory = torch.tensor(

        negative_memory,

        dtype=torch.float32,

        device=
            HPRR_DEVICE
    )


    output_rows = []


    with torch.no_grad():

        for start in range(
            0,
            len(query_embeddings),
            HPRR_CONFIG[
                "query_chunk"
            ]
        ):

            end = min(

                start
                +
                HPRR_CONFIG[
                    "query_chunk"
                ],

                len(
                    query_embeddings
                )
            )


            q = torch.tensor(

                query_embeddings[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    HPRR_DEVICE
            )


            # embeddings already L2 normalized
            sim_pos = (
                q
                @
                pos_memory.T
            )


            sim_neg = (
                q
                @
                neg_memory.T
            )


            k_pos = min(
                k,
                sim_pos.shape[1]
            )

            k_neg = min(
                k,
                sim_neg.shape[1]
            )


            top_pos = torch.topk(

                sim_pos,

                k=
                    k_pos,

                dim=1,

                largest=True
            ).values


            top_neg = torch.topk(

                sim_neg,

                k=
                    k_neg,

                dim=1,

                largest=True
            ).values


            mean_pos = (
                top_pos.mean(
                    dim=1
                )
            )

            mean_neg = (
                top_neg.mean(
                    dim=1
                )
            )


            max_pos = (
                top_pos[
                    :,
                    0
                ]
            )

            max_neg = (
                top_neg[
                    :,
                    0
                ]
            )


            margin = (
                mean_pos
                -
                mean_neg
            )


            # similarity support:
            # higher means retrieved neighbors are genuinely close
            support = (
                0.5
                *
                (
                    mean_pos
                    +
                    mean_neg
                )
            )


            block = torch.stack(

                [
                    mean_pos,
                    mean_neg,
                    max_pos,
                    max_neg,
                    margin,
                    support,
                ],

                dim=1
            )


            output_rows.append(

                block
                .cpu()
                .numpy()
            )


            del (
                sim_pos,
                sim_neg,
                top_pos,
                top_neg
            )


    return np.concatenate(
        output_rows,
        axis=0
    )


# ============================================================
# 14B.11 Outer CV
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_all_raw = (
    X_dev[
        HPRR_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


HPRR_CORE_OOF = np.zeros(
    len(y_all),
    dtype=float
)

HPRR_OOF = np.zeros(
    len(y_all),
    dtype=float
)

HPRR_RETRIEVAL_SCORE = np.zeros(
    len(y_all),
    dtype=float
)


fold_rows = []


for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"HPRR OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 110
    )


    outer_train_idx = np.where(
        fold_assignment
        != outer_fold
    )[0]


    outer_valid_idx = np.where(
        fold_assignment
        == outer_fold
    )[0]


    y_outer_train = y_all[
        outer_train_idx
    ]


    y_outer_valid = y_all[
        outer_valid_idx
    ]


    # ========================================================
    # A. Core4 fold-local calibrated probabilities
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for model_name in (
        HPRR_CORE
    ):

        p_tr, p_va = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_tr


        valid_cal[
            model_name
        ] = p_va


    core_outer_train = np.mean(

        np.column_stack([

            train_cal[m]

            for m in
            HPRR_CORE
        ]),

        axis=1
    )


    core_outer_valid = np.mean(

        np.column_stack([

            valid_cal[m]

            for m in
            HPRR_CORE
        ]),

        axis=1
    )


    HPRR_CORE_OOF[
        outer_valid_idx
    ] = core_outer_valid


    # ========================================================
    # B. Split outer training into:
    #
    # embedding/memory train
    # fusion-calibration train
    # ========================================================

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            HPRR_CONFIG[
                "fusion_fraction"
            ],

        random_state=
            SEED
            + 20000
            + outer_fold
    )


    memory_local_idx, fusion_local_idx = next(

        splitter.split(

            X_all_raw[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    memory_global_idx = (
        outer_train_idx[
            memory_local_idx
        ]
    )


    fusion_global_idx = (
        outer_train_idx[
            fusion_local_idx
        ]
    )


    y_memory = (
        y_all[
            memory_global_idx
        ]
    )


    y_fusion = (
        y_all[
            fusion_global_idx
        ]
    )


    core_memory = (
        core_outer_train[
            memory_local_idx
        ]
    )


    core_fusion = (
        core_outer_train[
            fusion_local_idx
        ]
    )


    # ========================================================
    # C. Preprocessing memory train only
    # ========================================================

    scaler = StandardScaler()


    X_memory = (
        scaler.fit_transform(

            X_all_raw[
                memory_global_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_fusion = (
        scaler.transform(

            X_all_raw[
                fusion_global_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_valid = (
        scaler.transform(

            X_all_raw[
                outer_valid_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # D. Train fast hard-aware embedding
    # ========================================================

    embedding_model = (
        train_hprr_embedding(

            X_memory,

            y_memory,

            core_memory,

            seed=
                SEED
                + 21000
                + outer_fold
        )
    )


    # ========================================================
    # E. Embeddings
    # ========================================================

    E_memory = get_hprr_embeddings(
        embedding_model,
        X_memory
    )


    E_fusion = get_hprr_embeddings(
        embedding_model,
        X_fusion
    )


    E_valid = get_hprr_embeddings(
        embedding_model,
        X_valid
    )


    # ========================================================
    # F. Hard-focused retrieval memory
    # ========================================================

    pos_mem_idx, neg_mem_idx = (
        select_retrieval_memory(

            y_memory,

            core_memory,

            seed=
                SEED
                + 22000
                + outer_fold
        )
    )


    E_pos_memory = (
        E_memory[
            pos_mem_idx
        ]
    )


    E_neg_memory = (
        E_memory[
            neg_mem_idx
        ]
    )


    print(
        "Positive memory:",
        len(
            E_pos_memory
        )
    )

    print(
        "Negative memory:",
        len(
            E_neg_memory
        )
    )


    # ========================================================
    # G. Retrieval features
    # ========================================================

    retrieval_fusion = (
        retrieve_hprr_features(

            E_fusion,

            E_pos_memory,

            E_neg_memory,

            k=
                HPRR_CONFIG[
                    "k"
                ]
        )
    )


    retrieval_valid = (
        retrieve_hprr_features(

            E_valid,

            E_pos_memory,

            E_neg_memory,

            k=
                HPRR_CONFIG[
                    "k"
                ]
        )
    )


    # ========================================================
    # H. Small fusion model
    #
    # IMPORTANT:
    # fit only using OUTER-TRAIN fusion subset
    # ========================================================

    # Core logit
    core_logit_fusion = (
        hprr_logit(
            core_fusion
        )
    )


    core_logit_valid = (
        hprr_logit(
            core_outer_valid
        )
    )


    # Expert disagreement
    fusion_expert_matrix = np.column_stack([

        train_cal[m][
            fusion_local_idx
        ]

        for m in HPRR_CORE
    ])


    valid_expert_matrix = np.column_stack([

        valid_cal[m]

        for m in HPRR_CORE
    ])


    fusion_disagreement = (
        fusion_expert_matrix.std(
            axis=1
        )
    )


    valid_disagreement = (
        valid_expert_matrix.std(
            axis=1
        )
    )


    # retrieval-enhanced meta-features
    Z_fusion = np.column_stack([

        core_logit_fusion,

        retrieval_fusion,

        fusion_disagreement,
    ])


    Z_valid = np.column_stack([

        core_logit_valid,

        retrieval_valid,

        valid_disagreement,
    ])


    # --------------------------------------------------------
    # strongly regularized logistic fusion
    # --------------------------------------------------------

    fusion_model = LogisticRegression(

        penalty="l2",

        C=0.10,

        solver="lbfgs",

        max_iter=2000,
    )


    fusion_model.fit(

        Z_fusion,

        y_fusion
    )


    p_valid = (
        fusion_model.predict_proba(
            Z_valid
        )[:, 1]
    )


    HPRR_OOF[
        outer_valid_idx
    ] = p_valid


    # Save raw retrieval margin
    HPRR_RETRIEVAL_SCORE[
        outer_valid_idx
    ] = retrieval_valid[
        :,
        4
    ]


    # ========================================================
    # I. Fold evaluation
    # ========================================================

    core_pm = probability_metrics(

        y_outer_valid,

        core_outer_valid
    )


    hprr_pm = probability_metrics(

        y_outer_valid,

        p_valid
    )


    retrieval_only_pr = (
        average_precision_score(

            y_outer_valid,

            retrieval_valid[
                :,
                4
            ]
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "HPRR_PR":
            hprr_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            hprr_pm[
                "PR_AUC"
            ]
            -
            core_pm[
                "PR_AUC"
            ],

        "Retrieval_Only_PR":
            retrieval_only_pr,

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "HPRR_ROC":
            hprr_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "HPRR_Brier":
            hprr_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "HPRR_ECE":
            hprr_pm[
                "ECE"
            ],
    })


    print(
        "\nCore PR      :",
        f"{core_pm['PR_AUC']:.6f}"
    )

    print(
        "Retrieval PR :",
        f"{retrieval_only_pr:.6f}"
    )

    print(
        "HPRR PR      :",
        f"{hprr_pm['PR_AUC']:.6f}"
    )

    print(
        "Delta        :",
        f"{hprr_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )


    # cleanup
    del (
        embedding_model,
        E_memory,
        E_fusion,
        E_valid,
        E_pos_memory,
        E_neg_memory,
        retrieval_fusion,
        retrieval_valid,
    )

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 14B.12 Global evaluation
# ============================================================

core_t, _, _ = (
    select_best_threshold(

        y_all,

        HPRR_CORE_OOF,

        objective="F1"
    )
)


hprr_t, _, _ = (
    select_best_threshold(

        y_all,

        HPRR_OOF,

        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPRR_CORE_OOF,

        threshold=
            core_t
    )
)


hprr_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPRR_OOF,

        threshold=
            hprr_t
    )
)


module14b_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "Fast_HPRR",

        **hprr_metrics
    },
])


module14b_results[
    "Delta_PR_vs_Core"
] = (

    module14b_results[
        "PR_AUC"
    ]

    -
    core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 14B — FAST HPRR RESULTS"
)

print(
    "=" * 125
)


display(
    module14b_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 14B.13 Fold results
# ============================================================

module14b_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "FAST HPRR FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module14b_folds
)


print(
    "\nMean fold Delta:",
    module14b_folds[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            module14b_folds[
                "Delta_PR"
            ]
            > 0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 14B.14 Retrieval complementarity
# ============================================================

core_rank = (
    pd.Series(
        HPRR_CORE_OOF
    )
    .rank()
)


retrieval_rank = (
    pd.Series(
        HPRR_RETRIEVAL_SCORE
    )
    .rank()
)


retrieval_spearman = (
    np.corrcoef(
        core_rank,
        retrieval_rank
    )[0, 1]
)


retrieval_pr_global = (
    average_precision_score(

        y_all,

        HPRR_RETRIEVAL_SCORE
    )
)


retrieval_summary = pd.DataFrame({

    "Metric": [

        "Retrieval-only PR-AUC",

        "Core/Retrieval Spearman",

        "Rank diversity",
    ],

    "Value": [

        retrieval_pr_global,

        retrieval_spearman,

        1.0
        -
        retrieval_spearman,
    ]
})


print(
    "\n"
    + "=" * 125
)

print(
    "HPRR RETRIEVAL COMPLEMENTARITY"
)

print(
    "=" * 125
)


display(
    retrieval_summary
)


# ============================================================
# 14B.15 Hard-positive analysis
# ============================================================
#
# Define difficult positives by Core4 score:
# bottom 25% of positive cases.
# This is evaluation only.
# ============================================================

positive_idx = np.where(
    y_all == 1
)[0]


hard_positive_threshold = np.quantile(

    HPRR_CORE_OOF[
        positive_idx
    ],

    0.25
)


hard_positive_mask = (

    (y_all == 1)

    &
    (
        HPRR_CORE_OOF
        <=
        hard_positive_threshold
    )
)


hard_positive_df = pd.DataFrame({

    "Metric": [

        "Hard positive N",

        "Mean Core score",

        "Mean HPRR score",

        "Mean retrieval evidence",
    ],

    "Value": [

        int(
            hard_positive_mask.sum()
        ),

        float(
            HPRR_CORE_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPRR_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPRR_RETRIEVAL_SCORE[
                hard_positive_mask
            ].mean()
        ),
    ]
})


print(
    "\n"
    + "=" * 125
)

print(
    "HPRR HARD-POSITIVE DIAGNOSTICS"
)

print(
    "=" * 125
)


display(
    hard_positive_df
)


# ============================================================
# 14B.16 Save
# ============================================================

save_csv(

    module14b_results,

    TAB_DIR
    / "module14b_fast_hprr_results.csv"
)


save_csv(

    module14b_folds,

    TAB_DIR
    / "module14b_fast_hprr_folds.csv"
)


save_csv(

    retrieval_summary,

    TAB_DIR
    / "module14b_fast_hprr_retrieval_summary.csv"
)


save_csv(

    hard_positive_df,

    TAB_DIR
    / "module14b_fast_hprr_hard_positive.csv"
)


hprr_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        HPRR_CORE_OOF,

    "Fast_HPRR":
        HPRR_OOF,

    "RetrievalEvidence":
        HPRR_RETRIEVAL_SCORE,
})


save_csv(

    hprr_pred_df,

    PRED_DIR
    / "module14b_fast_hprr_oof.csv"
)


module14b_manifest = {

    "method":
        "Fast Hard-Positive Retrieval-Augmented Repair",

    "embedding_epochs":
        HPRR_CONFIG[
            "epochs"
        ],

    "embedding_dim":
        HPRR_CONFIG[
            "embedding_dim"
        ],

    "positive_memory":
        HPRR_CONFIG[
            "max_positive_memory"
        ],

    "negative_memory":
        HPRR_CONFIG[
            "max_negative_memory"
        ],

    "neighbors_per_class":
        HPRR_CONFIG[
            "k"
        ],

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module14b_manifest,

    LOG_DIR
    / "module14b_manifest.json"
)


print(
    "\nMODULE 14B COMPLETE."
)

HPRR device: cuda

HPRR OUTER FOLD 1/5
    embedding epoch 1/3 loss=0.965354
    embedding epoch 2/3 loss=0.858812
    embedding epoch 3/3 loss=0.852947
Positive memory: 6000
Negative memory: 12000

Core PR      : 0.380070
Retrieval PR : 0.251981
HPRR PR      : 0.380062
Delta        : -0.000008

HPRR OUTER FOLD 2/5
    embedding epoch 1/3 loss=0.950004
    embedding epoch 2/3 loss=0.856234
    embedding epoch 3/3 loss=0.850831
Positive memory: 6000
Negative memory: 12000

Core PR      : 0.382772
Retrieval PR : 0.273072
HPRR PR      : 0.382752
Delta        : -0.000019

HPRR OUTER FOLD 3/5
    embedding epoch 1/3 loss=0.904084
    embedding epoch 2/3 loss=0.850969
    embedding epoch 3/3 loss=0.846256
Positive memory: 6000
Negative memory: 12000

Core PR      : 0.378467
Retrieval PR : 0.277167
HPRR PR      : 0.378471
Delta        : +0.000004

HPRR OUTER FOLD 4/5
    embedding epoch 1/3 loss=0.892771
    embedding epoch 2/3 loss=0.852692
    embedding epoch 3/3 loss=0.848945
Positive memo

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.220
1,Fast_HPRR,0.380384,0.000057,0.850552,0.069699,0.235563,0.001413,0.335292,0.564351,0.420661,0.358290,0.205



FAST HPRR FOLD RESULTS


,Fold,Core_PR,HPRR_PR,Delta_PR,Retrieval_Only_PR,Core_ROC,HPRR_ROC,Core_Brier,HPRR_Brier,Core_ECE,HPRR_ECE
0,0,0.380070,0.380062,-0.000008,0.251981,0.852660,0.852660,0.069637,0.069626,0.003499,0.002385
1,1,0.382772,0.382752,-0.000019,0.273072,0.850766,0.850765,0.069684,0.069685,0.004297,0.004424
2,2,0.378467,0.378471,0.000004,0.277167,0.849030,0.849031,0.069827,0.069827,0.002486,0.002532
3,3,0.372833,0.372843,0.000010,0.277059,0.849083,0.849079,0.070081,0.070082,0.005053,0.005842
4,4,0.390072,0.390099,0.000027,0.272772,0.851760,0.851762,0.069301,0.069273,0.005318,0.003927



Mean fold Delta: 2.668276946315995e-06
Positive folds: 3 / 5

HPRR RETRIEVAL COMPLEMENTARITY


,Metric,Value
0,Retrieval-only PR-AUC,0.269942
1,Core/Retrieval Spearman,0.798839
2,Rank diversity,0.201161



HPRR HARD-POSITIVE DIAGNOSTICS


,Metric,Value
0,Hard positive N,4779.000000
1,Mean Core score,0.068849
2,Mean HPRR score,0.067911
3,Mean retrieval evidence,-0.033532



MODULE 14B COMPLETE.


In [32]:
# ============================================================
# MODULE 14C — HPSC-HPRR
# Hard-Positive Supervised Contrastive Retrieval Repair
# ============================================================
#
# Purpose:
# Improve the retrieval geometry from Module 14B.
#
# Main change vs 14B:
# - embedding is trained with:
#       weighted BCE
#       +
#       supervised contrastive loss
#
# - difficult positives and confusing negatives receive
#   extra anchor weight
#
# Protocol:
# - strict outer 5-fold CV
# - outer validation NEVER used to train embedding
# - retrieval memory contains ONLY outer-training cases
# - fusion model fitted ONLY on outer-training fusion subset
# - no HOLDOUT
#
# Fast settings:
# - 3 epochs
# - 24-dimensional embedding
# - max 6k positive + 12k negative retrieval memory
# - K=32 per class
# ============================================================


import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score


# ============================================================
# 14C.1 FEATURES / CORE EXPERTS
# ============================================================

HPSC_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


HPSC_CORE = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]


# ============================================================
# 14C.2 CONFIGURATION
# ============================================================

HPSC_CONFIG = {

    # --------------------------------------------------------
    # embedding
    # --------------------------------------------------------

    "hidden_dim":
        64,

    "embedding_dim":
        24,

    "dropout":
        0.15,

    "epochs":
        3,

    "batch_size":
        1024,

    "lr":
        7e-4,

    "weight_decay":
        2e-4,


    # --------------------------------------------------------
    # supervised contrastive learning
    # --------------------------------------------------------

    "temperature":
        0.15,

    "bce_weight":
        0.50,

    "supcon_weight":
        1.00,


    # --------------------------------------------------------
    # hard case emphasis
    # --------------------------------------------------------

    "hard_fraction":
        0.25,

    "hard_multiplier":
        2.5,


    # --------------------------------------------------------
    # outer-train split
    # --------------------------------------------------------

    "fusion_fraction":
        0.18,


    # --------------------------------------------------------
    # retrieval memory
    # --------------------------------------------------------

    "max_positive_memory":
        6000,

    "max_negative_memory":
        12000,


    # --------------------------------------------------------
    # nearest neighbors PER CLASS
    # --------------------------------------------------------

    "k":
        32,


    # --------------------------------------------------------
    # GPU retrieval chunk
    # --------------------------------------------------------

    "query_chunk":
        2048,
}


HPSC_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


print(
    "HPSC device:",
    HPSC_DEVICE
)


# ============================================================
# 14C.3 REPRODUCIBILITY
# ============================================================

def hpsc_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 14C.4 NUMERICAL HELPERS
# ============================================================

def hpsc_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p / (1.0 - p)
    )


# ============================================================
# 14C.5 HARD-CASE ANCHOR WEIGHTS
# ============================================================
#
# Difficult positives:
#     lowest Core4 scores among positive cases
#
# Difficult negatives:
#     highest Core4 scores among negative cases
# ============================================================

def make_hpsc_weights(
    y,
    core_prob
):

    y = np.asarray(
        y,
        dtype=int
    )

    core_prob = np.asarray(
        core_prob,
        dtype=float
    )


    weights = np.ones(
        len(y),
        dtype=np.float32
    )


    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]


    n_hard_pos = max(

        1,

        int(
            len(pos_idx)
            * HPSC_CONFIG[
                "hard_fraction"
            ]
        )
    )


    n_hard_neg = max(

        1,

        int(
            len(neg_idx)
            * HPSC_CONFIG[
                "hard_fraction"
            ]
        )
    )


    # --------------------------------------------------------
    # lowest-scoring positives
    # --------------------------------------------------------

    hard_pos = pos_idx[
        np.argsort(
            core_prob[
                pos_idx
            ]
        )[
            :n_hard_pos
        ]
    ]


    # --------------------------------------------------------
    # highest-scoring negatives
    # --------------------------------------------------------

    hard_neg = neg_idx[
        np.argsort(
            core_prob[
                neg_idx
            ]
        )[
            -n_hard_neg:
        ]
    ]


    weights[
        hard_pos
    ] = HPSC_CONFIG[
        "hard_multiplier"
    ]


    weights[
        hard_neg
    ] = HPSC_CONFIG[
        "hard_multiplier"
    ]


    hard_mask = np.zeros(
        len(y),
        dtype=bool
    )

    hard_mask[
        hard_pos
    ] = True

    hard_mask[
        hard_neg
    ] = True


    return (
        weights,
        hard_mask
    )


# ============================================================
# 14C.6 DATASET
# ============================================================

class HPSCDataset(Dataset):

    def __init__(
        self,
        X,
        y,
        anchor_weight
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.anchor_weight = torch.tensor(
            anchor_weight,
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[
                idx
            ],

            self.y[
                idx
            ],

            self.anchor_weight[
                idx
            ],
        )


# ============================================================
# 14C.7 EMBEDDING NETWORK
# ============================================================

class HPSCEmbeddingNet(nn.Module):

    def __init__(
        self,
        input_dim
    ):

        super().__init__()


        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                HPSC_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.GELU(),

            nn.LayerNorm(
                HPSC_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.Dropout(
                HPSC_CONFIG[
                    "dropout"
                ]
            ),

            nn.Linear(
                HPSC_CONFIG[
                    "hidden_dim"
                ],
                HPSC_CONFIG[
                    "embedding_dim"
                ]
            ),
        )


        self.classifier = nn.Linear(

            HPSC_CONFIG[
                "embedding_dim"
            ],

            1
        )


    def forward(
        self,
        x
    ):

        z_raw = self.encoder(
            x
        )


        z = F.normalize(

            z_raw,

            p=2,

            dim=1
        )


        logit = (

            self.classifier(
                z_raw
            )
            .squeeze(-1)
        )


        return (
            logit,
            z
        )


# ============================================================
# 14C.8 WEIGHTED SUPERVISED CONTRASTIVE LOSS
# ============================================================

def weighted_supcon_loss(
    embeddings,
    labels,
    anchor_weights,
    temperature=0.15
):

    batch_size = (
        embeddings.shape[0]
    )


    similarity = (

        embeddings
        @
        embeddings.T

    ) / temperature


    # --------------------------------------------------------
    # numerical stability
    # --------------------------------------------------------

    similarity = (

        similarity

        -
        similarity.max(
            dim=1,
            keepdim=True
        ).values.detach()
    )


    labels = labels.view(
        -1,
        1
    )


    same_class = (
        labels
        ==
        labels.T
    )


    eye = torch.eye(

        batch_size,

        dtype=torch.bool,

        device=
            embeddings.device
    )


    positive_mask = (
        same_class
        &
        ~eye
    )


    valid_mask = (
        ~eye
    )


    exp_similarity = (

        torch.exp(
            similarity
        )

        *
        valid_mask.float()
    )


    denominator = (

        exp_similarity.sum(
            dim=1,
            keepdim=True
        )

        +
        1e-12
    )


    log_prob = (

        similarity

        -
        torch.log(
            denominator
        )
    )


    positive_count = (

        positive_mask.sum(
            dim=1
        )
    )


    usable = (

        positive_count
        >
        0
    )


    mean_positive_log_prob = (

        (
            log_prob

            *
            positive_mask.float()
        ).sum(
            dim=1
        )

        /

        positive_count.clamp(
            min=1
        )
    )


    losses = (

        -mean_positive_log_prob[
            usable
        ]
    )


    weights = (

        anchor_weights[
            usable
        ]
    )


    return (

        (
            losses
            *
            weights
        ).sum()

        /

        (
            weights.sum()
            +
            1e-12
        )
    )


# ============================================================
# 14C.9 TRAIN HPSC EMBEDDING
# ============================================================

def train_hpsc_embedding(
    X,
    y,
    core_prob,
    seed
):

    hpsc_seed(
        seed
    )


    (
        anchor_weights,
        _
    ) = make_hpsc_weights(

        y,
        core_prob
    )


    dataset = HPSCDataset(

        X,
        y,
        anchor_weights
    )


    loader = DataLoader(

        dataset,

        batch_size=
            HPSC_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,

        drop_last=True,
    )


    model = HPSCEmbeddingNet(

        input_dim=
            X.shape[1]

    ).to(
        HPSC_DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            HPSC_CONFIG[
                "lr"
            ],

        weight_decay=
            HPSC_CONFIG[
                "weight_decay"
            ]
    )


    # --------------------------------------------------------
    # mild class imbalance correction
    # --------------------------------------------------------

    n_pos = np.sum(
        y == 1
    )

    n_neg = np.sum(
        y == 0
    )


    pos_weight = torch.tensor(

        np.sqrt(
            n_neg / n_pos
        ),

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    for epoch in range(
        HPSC_CONFIG[
            "epochs"
        ]
    ):

        model.train()


        running_total = 0.0
        running_bce = 0.0
        running_supcon = 0.0
        seen = 0


        for (
            xb,
            yb,
            wb
        ) in loader:

            xb = xb.to(
                HPSC_DEVICE
            )

            yb = yb.to(
                HPSC_DEVICE
            )

            wb = wb.to(
                HPSC_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, embeddings = (
                model(
                    xb
                )
            )


            # =================================================
            # classification stabilization
            # =================================================

            bce_raw = (
                F.binary_cross_entropy_with_logits(

                    logits,

                    yb,

                    pos_weight=
                        pos_weight,

                    reduction="none"
                )
            )


            bce = (

                bce_raw
                *
                wb

            ).mean()


            # =================================================
            # retrieval-space objective
            # =================================================

            supcon = (
                weighted_supcon_loss(

                    embeddings,

                    yb.long(),

                    wb,

                    temperature=
                        HPSC_CONFIG[
                            "temperature"
                        ]
                )
            )


            total_loss = (

                HPSC_CONFIG[
                    "bce_weight"
                ]
                *
                bce

                +

                HPSC_CONFIG[
                    "supcon_weight"
                ]
                *
                supcon
            )


            total_loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


            n = len(
                yb
            )


            running_total += (

                float(
                    total_loss
                    .detach()
                    .cpu()
                )

                *
                n
            )


            running_bce += (

                float(
                    bce
                    .detach()
                    .cpu()
                )

                *
                n
            )


            running_supcon += (

                float(
                    supcon
                    .detach()
                    .cpu()
                )

                *
                n
            )


            seen += n


        print(

            f"    epoch "
            f"{epoch+1}/"
            f"{HPSC_CONFIG['epochs']} "

            f"loss="
            f"{running_total/seen:.5f} "

            f"BCE="
            f"{running_bce/seen:.5f} "

            f"SupCon="
            f"{running_supcon/seen:.5f}"
        )


    return model


# ============================================================
# 14C.10 EMBEDDING EXTRACTION
# ============================================================

def get_hpsc_embeddings(
    model,
    X,
    batch_size=4096
):

    model.eval()

    embeddings = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(

                start
                +
                batch_size,

                len(X)
            )


            xb = torch.tensor(

                X[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    HPSC_DEVICE
            )


            _, z = model(
                xb
            )


            embeddings.append(

                z
                .cpu()
                .numpy()
            )


    return np.concatenate(
        embeddings,
        axis=0
    )


# ============================================================
# 14C.11 RETRIEVAL MEMORY SAMPLING
# ============================================================

def select_hpsc_memory(
    y,
    core_prob,
    seed
):

    rng = np.random.default_rng(
        seed
    )


    pos_idx = np.where(
        y == 1
    )[0]

    neg_idx = np.where(
        y == 0
    )[0]


    # --------------------------------------------------------
    # prioritize difficult positives
    # --------------------------------------------------------

    pos_hardness = (

        1.0
        -
        core_prob[
            pos_idx
        ]
    )


    pos_sampling_weight = (

        0.25
        +
        pos_hardness
    )


    pos_sampling_weight /= (

        pos_sampling_weight.sum()
    )


    n_pos = min(

        HPSC_CONFIG[
            "max_positive_memory"
        ],

        len(
            pos_idx
        )
    )


    selected_pos = rng.choice(

        pos_idx,

        size=
            n_pos,

        replace=False,

        p=
            pos_sampling_weight
    )


    # --------------------------------------------------------
    # prioritize confusing negatives
    # --------------------------------------------------------

    neg_hardness = (

        core_prob[
            neg_idx
        ]
    )


    neg_sampling_weight = (

        0.05
        +
        neg_hardness
    )


    neg_sampling_weight /= (

        neg_sampling_weight.sum()
    )


    n_neg = min(

        HPSC_CONFIG[
            "max_negative_memory"
        ],

        len(
            neg_idx
        )
    )


    selected_neg = rng.choice(

        neg_idx,

        size=
            n_neg,

        replace=False,

        p=
            neg_sampling_weight
    )


    return (
        selected_pos,
        selected_neg
    )


# ============================================================
# 14C.12 FAST GPU RETRIEVAL
# ============================================================
#
# Returns:
#
# col 0 = mean top-K positive similarity
# col 1 = mean top-K negative similarity
# col 2 = max positive similarity
# col 3 = max negative similarity
# col 4 = retrieval margin
# col 5 = neighborhood support
# ============================================================

def retrieve_hpsc_features(
    query_embeddings,
    positive_memory,
    negative_memory,
    k=32
):

    pos_memory = torch.tensor(

        positive_memory,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    neg_memory = torch.tensor(

        negative_memory,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    output_rows = []


    with torch.no_grad():

        for start in range(

            0,

            len(
                query_embeddings
            ),

            HPSC_CONFIG[
                "query_chunk"
            ]
        ):

            end = min(

                start
                +
                HPSC_CONFIG[
                    "query_chunk"
                ],

                len(
                    query_embeddings
                )
            )


            query = torch.tensor(

                query_embeddings[
                    start:end
                ],

                dtype=torch.float32,

                device=
                    HPSC_DEVICE
            )


            # ------------------------------------------------
            # cosine similarities
            # embeddings already normalized
            # ------------------------------------------------

            sim_pos = (

                query
                @
                pos_memory.T
            )


            sim_neg = (

                query
                @
                neg_memory.T
            )


            k_pos = min(

                k,

                sim_pos.shape[1]
            )


            k_neg = min(

                k,

                sim_neg.shape[1]
            )


            top_pos = torch.topk(

                sim_pos,

                k=
                    k_pos,

                dim=1,

                largest=True

            ).values


            top_neg = torch.topk(

                sim_neg,

                k=
                    k_neg,

                dim=1,

                largest=True

            ).values


            mean_pos = (
                top_pos.mean(
                    dim=1
                )
            )


            mean_neg = (
                top_neg.mean(
                    dim=1
                )
            )


            max_pos = (
                top_pos[
                    :,
                    0
                ]
            )


            max_neg = (
                top_neg[
                    :,
                    0
                ]
            )


            retrieval_margin = (

                mean_pos
                -
                mean_neg
            )


            support = (

                0.5

                *
                (
                    mean_pos
                    +
                    mean_neg
                )
            )


            block = torch.stack(

                [
                    mean_pos,
                    mean_neg,
                    max_pos,
                    max_neg,
                    retrieval_margin,
                    support,
                ],

                dim=1
            )


            output_rows.append(

                block
                .cpu()
                .numpy()
            )


            del (
                sim_pos,
                sim_neg,
                top_pos,
                top_neg
            )


    return np.concatenate(
        output_rows,
        axis=0
    )


# ============================================================
# 14C.13 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_all_raw = (

    X_dev[
        HPSC_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 14C.14 OUTPUT BUFFERS
# ============================================================

HPSC_CORE_OOF = np.zeros(
    len(
        y_all
    ),
    dtype=float
)


HPSC_OOF = np.zeros(
    len(
        y_all
    ),
    dtype=float
)


HPSC_RETRIEVAL_SCORE = np.zeros(
    len(
        y_all
    ),
    dtype=float
)


HPSC_RETRIEVAL_SUPPORT = np.zeros(
    len(
        y_all
    ),
    dtype=float
)


fold_rows = []


# ============================================================
# 14C.15 STRICT OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 115
    )

    print(
        f"HPSC-HPRR OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 115
    )


    # ========================================================
    # Outer indices
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (

        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (

        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # A. Fold-local Core4 calibration
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for model_name in (
        HPSC_CORE
    ):

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    core_outer_train = np.mean(

        np.column_stack([

            train_cal[
                m
            ]

            for m
            in HPSC_CORE
        ]),

        axis=1
    )


    core_outer_valid = np.mean(

        np.column_stack([

            valid_cal[
                m
            ]

            for m
            in HPSC_CORE
        ]),

        axis=1
    )


    HPSC_CORE_OOF[
        outer_valid_idx
    ] = core_outer_valid


    # ========================================================
    # B. Split OUTER TRAIN
    #
    # memory/embedding subset
    # fusion-calibration subset
    # ========================================================

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            HPSC_CONFIG[
                "fusion_fraction"
            ],

        random_state=
            SEED
            + 23000
            + outer_fold
    )


    (
        memory_local_idx,
        fusion_local_idx

    ) = next(

        splitter.split(

            X_all_raw[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    memory_global_idx = (

        outer_train_idx[
            memory_local_idx
        ]
    )


    fusion_global_idx = (

        outer_train_idx[
            fusion_local_idx
        ]
    )


    y_memory = (

        y_all[
            memory_global_idx
        ]
    )


    y_fusion = (

        y_all[
            fusion_global_idx
        ]
    )


    core_memory = (

        core_outer_train[
            memory_local_idx
        ]
    )


    core_fusion = (

        core_outer_train[
            fusion_local_idx
        ]
    )


    # ========================================================
    # C. Preprocessing
    #
    # fit only on embedding/memory subset
    # ========================================================

    scaler = StandardScaler()


    X_memory = (

        scaler.fit_transform(

            X_all_raw[
                memory_global_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    X_fusion = (

        scaler.transform(

            X_all_raw[
                fusion_global_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    X_valid = (

        scaler.transform(

            X_all_raw[
                outer_valid_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    # ========================================================
    # D. TRAIN SUPERVISED-CONTRASTIVE EMBEDDING
    # ========================================================

    embedding_model = (
        train_hpsc_embedding(

            X_memory,

            y_memory,

            core_memory,

            seed=
                SEED
                + 24000
                + outer_fold
        )
    )


    # ========================================================
    # E. EXTRACT EMBEDDINGS
    # ========================================================

    E_memory = (
        get_hpsc_embeddings(

            embedding_model,

            X_memory
        )
    )


    E_fusion = (
        get_hpsc_embeddings(

            embedding_model,

            X_fusion
        )
    )


    E_valid = (
        get_hpsc_embeddings(

            embedding_model,

            X_valid
        )
    )


    # ========================================================
    # F. SELECT HARD-FOCUSED RETRIEVAL MEMORY
    # ========================================================

    (
        pos_mem_idx,
        neg_mem_idx

    ) = select_hpsc_memory(

        y_memory,

        core_memory,

        seed=
            SEED
            + 25000
            + outer_fold
    )


    E_pos_memory = (

        E_memory[
            pos_mem_idx
        ]
    )


    E_neg_memory = (

        E_memory[
            neg_mem_idx
        ]
    )


    print(
        "\nPositive memory:",
        len(
            E_pos_memory
        )
    )


    print(
        "Negative memory:",
        len(
            E_neg_memory
        )
    )


    # ========================================================
    # G. RETRIEVAL FEATURES
    # ========================================================

    retrieval_fusion = (
        retrieve_hpsc_features(

            E_fusion,

            E_pos_memory,

            E_neg_memory,

            k=
                HPSC_CONFIG[
                    "k"
                ]
        )
    )


    retrieval_valid = (
        retrieve_hpsc_features(

            E_valid,

            E_pos_memory,

            E_neg_memory,

            k=
                HPSC_CONFIG[
                    "k"
                ]
        )
    )


    # ========================================================
    # G2. HARD-POSITIVE RETRIEVAL DIAGNOSTICS
    # ========================================================

    positive_mask = (

        y_outer_valid
        ==
        1
    )


    positive_q25 = np.quantile(

        core_outer_valid[
            positive_mask
        ],

        0.25
    )


    hard_positive_local = (

        positive_mask

        &

        (
            core_outer_valid
            <=
            positive_q25
        )
    )


    retrieval_evidence_all = float(

        retrieval_valid[
            :,
            4
        ].mean()
    )


    retrieval_evidence_positive = float(

        retrieval_valid[
            positive_mask,
            4
        ].mean()
    )


    retrieval_evidence_hard_positive = float(

        retrieval_valid[
            hard_positive_local,
            4
        ].mean()
    )


    print(
        "\nRetrieval evidence diagnostics:"
    )


    print(
        "  All cases          :",
        f"{retrieval_evidence_all:.6f}"
    )


    print(
        "  Positive cases     :",
        f"{retrieval_evidence_positive:.6f}"
    )


    print(
        "  Hard positives     :",
        f"{retrieval_evidence_hard_positive:.6f}"
    )


    # ========================================================
    # H. BUILD FUSION META FEATURES
    # ========================================================

    core_logit_fusion = (
        hpsc_logit(
            core_fusion
        )
    )


    core_logit_valid = (
        hpsc_logit(
            core_outer_valid
        )
    )


    # --------------------------------------------------------
    # expert disagreement
    # --------------------------------------------------------

    fusion_expert_matrix = np.column_stack([

        train_cal[
            m
        ][
            fusion_local_idx
        ]

        for m
        in HPSC_CORE
    ])


    valid_expert_matrix = np.column_stack([

        valid_cal[
            m
        ]

        for m
        in HPSC_CORE
    ])


    fusion_disagreement = (

        fusion_expert_matrix.std(
            axis=1
        )
    )


    valid_disagreement = (

        valid_expert_matrix.std(
            axis=1
        )
    )


    # --------------------------------------------------------
    # final fusion features
    # --------------------------------------------------------

    Z_fusion = np.column_stack([

        core_logit_fusion,

        retrieval_fusion,

        fusion_disagreement,
    ])


    Z_valid = np.column_stack([

        core_logit_valid,

        retrieval_valid,

        valid_disagreement,
    ])


    # ========================================================
    # I. REGULARIZED FUSION MODEL
    # ========================================================

    fusion_model = LogisticRegression(

        penalty="l2",

        C=0.10,

        solver="lbfgs",

        max_iter=2000
    )


    fusion_model.fit(

        Z_fusion,

        y_fusion
    )


    p_valid_hpsc = (

        fusion_model
        .predict_proba(
            Z_valid
        )[
            :,
            1
        ]
    )


    # ========================================================
    # J. STORE OUTER OOF
    # ========================================================

    HPSC_OOF[
        outer_valid_idx
    ] = p_valid_hpsc


    HPSC_RETRIEVAL_SCORE[
        outer_valid_idx
    ] = retrieval_valid[
        :,
        4
    ]


    HPSC_RETRIEVAL_SUPPORT[
        outer_valid_idx
    ] = retrieval_valid[
        :,
        5
    ]


    # ========================================================
    # K. OUTER-FOLD METRICS
    # ========================================================

    core_pm = (
        probability_metrics(

            y_outer_valid,

            core_outer_valid
        )
    )


    hpsc_pm = (
        probability_metrics(

            y_outer_valid,

            p_valid_hpsc
        )
    )


    retrieval_only_pr = (
        average_precision_score(

            y_outer_valid,

            retrieval_valid[
                :,
                4
            ]
        )
    )


    fold_rows.append({

        "Fold":
            outer_fold,

        "Core_PR":
            core_pm[
                "PR_AUC"
            ],

        "HPSC_PR":
            hpsc_pm[
                "PR_AUC"
            ],

        "Delta_PR":
            hpsc_pm[
                "PR_AUC"
            ]
            -
            core_pm[
                "PR_AUC"
            ],

        "Retrieval_Only_PR":
            retrieval_only_pr,

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "HPSC_ROC":
            hpsc_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "HPSC_Brier":
            hpsc_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "HPSC_ECE":
            hpsc_pm[
                "ECE"
            ],

        "RetrievalEvidence_All":
            retrieval_evidence_all,

        "RetrievalEvidence_Positive":
            retrieval_evidence_positive,

        "RetrievalEvidence_HardPositive":
            retrieval_evidence_hard_positive,
    })


    print(
        "\nCore PR      :",
        f"{core_pm['PR_AUC']:.6f}"
    )


    print(
        "Retrieval PR :",
        f"{retrieval_only_pr:.6f}"
    )


    print(
        "HPSC PR      :",
        f"{hpsc_pm['PR_AUC']:.6f}"
    )


    print(
        "Delta        :",
        f"{hpsc_pm['PR_AUC'] - core_pm['PR_AUC']:+.6f}"
    )


    # ========================================================
    # L. CLEANUP
    # ========================================================

    del (
        embedding_model,
        E_memory,
        E_fusion,
        E_valid,
        E_pos_memory,
        E_neg_memory,
        retrieval_fusion,
        retrieval_valid,
        fusion_model,
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 14C.16 GLOBAL EVALUATION
# ============================================================

core_t, _, _ = (
    select_best_threshold(

        y_all,

        HPSC_CORE_OOF,

        objective="F1"
    )
)


hpsc_t, _, _ = (
    select_best_threshold(

        y_all,

        HPSC_OOF,

        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPSC_CORE_OOF,

        threshold=
            core_t
    )
)


hpsc_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPSC_OOF,

        threshold=
            hpsc_t
    )
)


module14c_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "HPSC_HPRR",

        **hpsc_metrics
    },
])


module14c_results[
    "Delta_PR_vs_Core"
] = (

    module14c_results[
        "PR_AUC"
    ]

    -
    core_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "MODULE 14C — HPSC-HPRR RESULTS"
)

print(
    "=" * 125
)


display(

    module14c_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 14C.17 FOLD RESULTS
# ============================================================

module14c_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "HPSC-HPRR FOLD RESULTS"
)

print(
    "=" * 125
)


display(
    module14c_folds
)


print(
    "\nMean fold Delta:",
    module14c_folds[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            module14c_folds[
                "Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 14C.18 RETRIEVAL COMPLEMENTARITY
# ============================================================

core_rank = (
    pd.Series(
        HPSC_CORE_OOF
    )
    .rank(
        method="average"
    )
)


retrieval_rank = (
    pd.Series(
        HPSC_RETRIEVAL_SCORE
    )
    .rank(
        method="average"
    )
)


retrieval_spearman = (

    np.corrcoef(

        core_rank,

        retrieval_rank

    )[0, 1]
)


retrieval_pr_global = (
    average_precision_score(

        y_all,

        HPSC_RETRIEVAL_SCORE
    )
)


retrieval_summary = pd.DataFrame({

    "Metric": [

        "Retrieval-only PR-AUC",

        "Core/Retrieval Spearman",

        "Rank diversity",

        "Mean retrieval support",
    ],

    "Value": [

        retrieval_pr_global,

        retrieval_spearman,

        1.0
        -
        retrieval_spearman,

        float(
            HPSC_RETRIEVAL_SUPPORT.mean()
        ),
    ],
})


print(
    "\n"
    + "=" * 125
)

print(
    "HPSC RETRIEVAL COMPLEMENTARITY"
)

print(
    "=" * 125
)


display(
    retrieval_summary
)


# ============================================================
# 14C.19 HARD-POSITIVE GLOBAL DIAGNOSTICS
# ============================================================

positive_idx = np.where(
    y_all == 1
)[0]


hard_positive_threshold = np.quantile(

    HPSC_CORE_OOF[
        positive_idx
    ],

    0.25
)


hard_positive_mask = (

    (y_all == 1)

    &

    (
        HPSC_CORE_OOF
        <=
        hard_positive_threshold
    )
)


hard_positive_df = pd.DataFrame({

    "Metric": [

        "Hard positive N",

        "Mean Core score",

        "Mean HPSC score",

        "Mean retrieval evidence — all",

        "Mean retrieval evidence — positives",

        "Mean retrieval evidence — hard positives",

        "Mean retrieval support — hard positives",
    ],

    "Value": [

        int(
            hard_positive_mask.sum()
        ),

        float(
            HPSC_CORE_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPSC_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPSC_RETRIEVAL_SCORE.mean()
        ),

        float(
            HPSC_RETRIEVAL_SCORE[
                y_all == 1
            ].mean()
        ),

        float(
            HPSC_RETRIEVAL_SCORE[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPSC_RETRIEVAL_SUPPORT[
                hard_positive_mask
            ].mean()
        ),
    ]
})


print(
    "\n"
    + "=" * 125
)

print(
    "HPSC HARD-POSITIVE DIAGNOSTICS"
)

print(
    "=" * 125
)


display(
    hard_positive_df
)


# ============================================================
# 14C.20 COMPARE MODULE 14B VS 14C
# ============================================================
#
# This block works if Module 14B variables are still present.
# If not, it safely skips the comparison.
# ============================================================

if (
    "HPRR_OOF"
    in globals()
    and
    "HPRR_CORE_OOF"
    in globals()
):

    comparison_14b_14c = pd.DataFrame([

        {
            "Method":
                "Core4Mean",

            "PR_AUC":
                average_precision_score(
                    y_all,
                    HPSC_CORE_OOF
                )
        },

        {
            "Method":
                "Fast_HPRR_14B",

            "PR_AUC":
                average_precision_score(
                    y_all,
                    HPRR_OOF
                )
        },

        {
            "Method":
                "HPSC_HPRR_14C",

            "PR_AUC":
                average_precision_score(
                    y_all,
                    HPSC_OOF
                )
        },
    ])


    comparison_14b_14c[
        "Delta_vs_Core"
    ] = (

        comparison_14b_14c[
            "PR_AUC"
        ]

        -
        comparison_14b_14c.loc[
            comparison_14b_14c[
                "Method"
            ]
            ==
            "Core4Mean",

            "PR_AUC"
        ].iloc[0]
    )


    print(
        "\n"
        + "=" * 125
    )

    print(
        "MODULE 14B vs 14C COMPARISON"
    )

    print(
        "=" * 125
    )


    display(
        comparison_14b_14c
    )


# ============================================================
# 14C.21 SAVE
# ============================================================

save_csv(

    module14c_results,

    TAB_DIR
    / "module14c_hpsc_hprr_results.csv"
)


save_csv(

    module14c_folds,

    TAB_DIR
    / "module14c_hpsc_hprr_folds.csv"
)


save_csv(

    retrieval_summary,

    TAB_DIR
    / "module14c_hpsc_retrieval_summary.csv"
)


save_csv(

    hard_positive_df,

    TAB_DIR
    / "module14c_hpsc_hard_positive.csv"
)


if (
    "comparison_14b_14c"
    in locals()
):

    save_csv(

        comparison_14b_14c,

        TAB_DIR
        / "module14c_vs_14b_comparison.csv"
    )


hpsc_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        HPSC_CORE_OOF,

    "HPSC_HPRR":
        HPSC_OOF,

    "RetrievalEvidence":
        HPSC_RETRIEVAL_SCORE,

    "RetrievalSupport":
        HPSC_RETRIEVAL_SUPPORT,
})


save_csv(

    hpsc_pred_df,

    PRED_DIR
    / "module14c_hpsc_hprr_oof.csv"
)


module14c_manifest = {

    "method":
        "Hard-Positive Supervised Contrastive Retrieval Repair",

    "embedding_epochs":
        HPSC_CONFIG[
            "epochs"
        ],

    "embedding_dim":
        HPSC_CONFIG[
            "embedding_dim"
        ],

    "supcon_temperature":
        HPSC_CONFIG[
            "temperature"
        ],

    "bce_weight":
        HPSC_CONFIG[
            "bce_weight"
        ],

    "supcon_weight":
        HPSC_CONFIG[
            "supcon_weight"
        ],

    "hard_fraction":
        HPSC_CONFIG[
            "hard_fraction"
        ],

    "hard_multiplier":
        HPSC_CONFIG[
            "hard_multiplier"
        ],

    "positive_memory":
        HPSC_CONFIG[
            "max_positive_memory"
        ],

    "negative_memory":
        HPSC_CONFIG[
            "max_negative_memory"
        ],

    "neighbors_per_class":
        HPSC_CONFIG[
            "k"
        ],

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module14c_manifest,

    LOG_DIR
    / "module14c_manifest.json"
)


print(
    "\nMODULE 14C COMPLETE."
)

HPSC device: cuda

HPSC-HPRR OUTER FOLD 1/5
    epoch 1/3 loss=7.73163 BCE=0.93394 SupCon=7.26466
    epoch 2/3 loss=7.37024 BCE=0.85662 SupCon=6.94193
    epoch 3/3 loss=7.35915 BCE=0.85011 SupCon=6.93409

Positive memory: 6000
Negative memory: 12000

Retrieval evidence diagnostics:
  All cases          : -0.002268
  Positive cases     : -0.000715
  Hard positives     : -0.001743

Core PR      : 0.380070
Retrieval PR : 0.284685
HPSC PR      : 0.380073
Delta        : +0.000004

HPSC-HPRR OUTER FOLD 2/5
    epoch 1/3 loss=7.69391 BCE=0.92224 SupCon=7.23279
    epoch 2/3 loss=7.37124 BCE=0.85886 SupCon=6.94181
    epoch 3/3 loss=7.35971 BCE=0.85099 SupCon=6.93422

Positive memory: 6000
Negative memory: 12000

Retrieval evidence diagnostics:
  All cases          : -0.002023
  Positive cases     : -0.000656
  Hard positives     : -0.001538

Core PR      : 0.382772
Retrieval PR : 0.281993
HPSC PR      : 0.382779
Delta        : +0.000007

HPSC-HPRR OUTER FOLD 3/5
    epoch 1/3 loss=7.74295 B

,Method,PR_AUC,Delta_PR_vs_Core,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.220
1,HPSC_HPRR,0.380429,0.000101,0.850579,0.069702,0.235558,0.001474,0.352712,0.522130,0.421017,0.356265,0.225



HPSC-HPRR FOLD RESULTS


,Fold,Core_PR,HPSC_PR,Delta_PR,Retrieval_Only_PR,Core_ROC,HPSC_ROC,Core_Brier,HPSC_Brier,Core_ECE,HPSC_ECE,RetrievalEvidence_All,RetrievalEvidence_Positive,RetrievalEvidence_HardPositive
0,0,0.380070,0.380073,3.679420e-06,0.284685,0.852660,0.852660,0.069637,0.069625,0.003499,0.002212,-0.002268,-0.000715,-0.001743
1,1,0.382772,0.382779,7.488288e-06,0.281993,0.850766,0.850766,0.069684,0.069702,0.004297,0.003285,-0.002023,-0.000656,-0.001538
2,2,0.378467,0.378453,-1.425415e-05,0.285060,0.849030,0.849029,0.069827,0.069837,0.002486,0.003441,-0.002319,-0.000729,-0.001810
3,3,0.372833,0.372832,-6.922440e-07,0.285095,0.849083,0.849083,0.070081,0.070074,0.005053,0.004770,-0.002237,-0.000709,-0.001776
4,4,0.390072,0.390072,-2.740461e-07,0.268943,0.851760,0.851760,0.069301,0.069272,0.005318,0.003616,-0.001968,-0.000655,-0.001512



Mean fold Delta: -8.105459944673932e-07
Positive folds: 2 / 5

HPSC RETRIEVAL COMPLEMENTARITY


...


HPSC HARD-POSITIVE DIAGNOSTICS


,Metric,Value
0,Hard positive N,4779.000000
1,Mean Core score,0.068849
2,Mean HPSC score,0.067319
3,Mean retrieval evidence — all,-0.002163
4,Mean retrieval evidence — positives,-0.000693
5,Mean retrieval evidence — hard positives,-0.001676
6,Mean retrieval support — hard positives,0.993398



MODULE 14B vs 14C COMPARISON


,Method,PR_AUC,Delta_vs_Core
0,Core4Mean,0.380327,0.000000
1,Fast_HPRR_14B,0.380384,0.000057
2,HPSC_HPRR_14C,0.380429,0.000101



MODULE 14C COMPLETE.


In [33]:
# ============================================================
# MODULE 14D — MSCR
# Multi-Scale Contrastive Retrieval Fusion
# ============================================================
#
# Goal:
# Exploit HPSC retrieval evidence at several neighborhood scales.
#
# K = 8, 16, 32, 64, 128
#
# Improvements over 14C:
# 1. Multi-scale retrieval instead of one arbitrary K
# 2. Retrieve top-128 only ONCE -> fast
# 3. Neighborhood stability features
# 4. Core-only meta baseline fitted on SAME fusion subset
#    to isolate genuine retrieval contribution
# 5. Strongly regularized fusion
#
# Protocol:
# - outer 5-fold CV
# - HPSC embedding trained only on outer-train memory subset
# - fusion model fitted only on outer-train fusion subset
# - outer validation untouched
# - HOLDOUT untouched
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score


# ============================================================
# 14D.1 REQUIRED FUNCTIONS FROM MODULE 14C
# ============================================================

required_objects = [
    "train_hpsc_embedding",
    "get_hpsc_embeddings",
    "select_hpsc_memory",
    "hpsc_logit",
    "HPSC_FEATURES",
    "HPSC_CORE",
    "HPSC_DEVICE",
]

missing = [
    x
    for x in required_objects
    if x not in globals()
]

if len(missing) > 0:
    raise RuntimeError(
        "Run Module 14C definitions first. Missing: "
        + ", ".join(missing)
    )


# ============================================================
# 14D.2 CONFIG
# ============================================================

MSCR_CONFIG = {

    "k_values":
        [8, 16, 32, 64, 128],

    # same fast embedding setup as 14C
    "fusion_fraction":
        0.18,

    "max_positive_memory":
        6000,

    "max_negative_memory":
        12000,

    "query_chunk":
        2048,

    # strong regularization because retrieval features
    # are correlated across K
    "fusion_C":
        0.03,

    # core-only comparison model
    "core_meta_C":
        0.10,
}


K_VALUES = MSCR_CONFIG[
    "k_values"
]

MAX_K = max(
    K_VALUES
)


print(
    "MSCR neighborhood scales:",
    K_VALUES
)


# ============================================================
# 14D.3 FAST MULTI-SCALE RETRIEVAL
# ============================================================
#
# We calculate top-MAX_K positive/negative similarities ONCE.
#
# For every K:
#   mean positive similarity
#   mean negative similarity
#   margin
#   max positive similarity
#   max negative similarity
#   max contrast
#   support
#
# Then derive:
#   cross-scale margin changes
#   margin std
#   margin range
#
# ============================================================

def retrieve_multiscale_features(
    query_embeddings,
    positive_memory,
    negative_memory,
    k_values
):

    positive_memory_t = torch.tensor(

        positive_memory,

        dtype=torch.float32,

        device=HPSC_DEVICE
    )


    negative_memory_t = torch.tensor(

        negative_memory,

        dtype=torch.float32,

        device=HPSC_DEVICE
    )


    max_k = max(
        k_values
    )


    max_k_pos = min(
        max_k,
        positive_memory_t.shape[0]
    )

    max_k_neg = min(
        max_k,
        negative_memory_t.shape[0]
    )


    output_blocks = []


    with torch.no_grad():

        for start in range(
            0,
            len(query_embeddings),
            MSCR_CONFIG[
                "query_chunk"
            ]
        ):

            end = min(

                start
                +
                MSCR_CONFIG[
                    "query_chunk"
                ],

                len(query_embeddings)
            )


            q = torch.tensor(

                query_embeddings[
                    start:end
                ],

                dtype=torch.float32,

                device=HPSC_DEVICE
            )


            # =================================================
            # Similarity calculated once
            # =================================================

            sim_pos = (
                q
                @
                positive_memory_t.T
            )


            sim_neg = (
                q
                @
                negative_memory_t.T
            )


            # =================================================
            # Top MAX_K calculated once
            # =================================================

            top_pos = torch.topk(

                sim_pos,

                k=max_k_pos,

                dim=1,

                largest=True,

                sorted=True

            ).values


            top_neg = torch.topk(

                sim_neg,

                k=max_k_neg,

                dim=1,

                largest=True,

                sorted=True

            ).values


            # cumulative sum makes mean@K cheap
            pos_cumsum = torch.cumsum(
                top_pos,
                dim=1
            )


            neg_cumsum = torch.cumsum(
                top_neg,
                dim=1
            )


            block_features = []

            margin_list = []


            for k in k_values:

                kp = min(
                    k,
                    max_k_pos
                )

                kn = min(
                    k,
                    max_k_neg
                )


                mean_pos = (
                    pos_cumsum[
                        :,
                        kp - 1
                    ]
                    /
                    float(kp)
                )


                mean_neg = (
                    neg_cumsum[
                        :,
                        kn - 1
                    ]
                    /
                    float(kn)
                )


                margin = (
                    mean_pos
                    -
                    mean_neg
                )


                max_pos = (
                    top_pos[
                        :,
                        0
                    ]
                )


                max_neg = (
                    top_neg[
                        :,
                        0
                    ]
                )


                max_contrast = (
                    max_pos
                    -
                    max_neg
                )


                support = (

                    0.5
                    *
                    (
                        mean_pos
                        +
                        mean_neg
                    )
                )


                block_features.extend([

                    mean_pos,
                    mean_neg,
                    margin,
                    max_contrast,
                    support,
                ])


                margin_list.append(
                    margin
                )


            # =================================================
            # Multi-scale behavior
            # =================================================

            margins = torch.stack(
                margin_list,
                dim=1
            )


            # change between consecutive scales
            margin_deltas = (

                margins[
                    :,
                    :-1
                ]
                -
                margins[
                    :,
                    1:
                ]
            )


            margin_std = torch.std(

                margins,

                dim=1,

                unbiased=False
            )


            margin_range = (

                margins.max(
                    dim=1
                ).values

                -
                margins.min(
                    dim=1
                ).values
            )


            # fine-vs-broad neighborhood contrast
            fine_broad_delta = (

                margins[
                    :,
                    0
                ]

                -
                margins[
                    :,
                    -1
                ]
            )


            base_matrix = torch.stack(

                block_features,

                dim=1
            )


            block = torch.cat(

                [
                    base_matrix,
                    margin_deltas,
                    margin_std[
                        :,
                        None
                    ],
                    margin_range[
                        :,
                        None
                    ],
                    fine_broad_delta[
                        :,
                        None
                    ],
                ],

                dim=1
            )


            output_blocks.append(

                block
                .cpu()
                .numpy()
            )


            del (
                sim_pos,
                sim_neg,
                top_pos,
                top_neg,
                pos_cumsum,
                neg_cumsum,
                margins,
                margin_deltas,
                base_matrix,
                block,
            )


    return np.concatenate(
        output_blocks,
        axis=0
    )


# ============================================================
# 14D.4 FEATURE NAMES
# ============================================================

MSCR_FEATURE_NAMES = []


for k in K_VALUES:

    MSCR_FEATURE_NAMES.extend([

        f"mean_pos_k{k}",
        f"mean_neg_k{k}",
        f"margin_k{k}",
        f"max_contrast_k{k}",
        f"support_k{k}",
    ])


for k1, k2 in zip(
    K_VALUES[:-1],
    K_VALUES[1:]
):

    MSCR_FEATURE_NAMES.append(
        f"margin_delta_k{k1}_k{k2}"
    )


MSCR_FEATURE_NAMES.extend([

    "margin_std",
    "margin_range",
    "fine_broad_delta",
])


print(
    "Retrieval feature count:",
    len(
        MSCR_FEATURE_NAMES
    )
)


# ============================================================
# 14D.5 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_all_raw = (

    X_dev[
        HPSC_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 14D.6 OUTPUT BUFFERS
# ============================================================

MSCR_CORE_OOF = np.zeros(
    len(y_all),
    dtype=float
)


# Core logit refitted/recalibrated on same fusion subset
MSCR_CORE_META_OOF = np.zeros(
    len(y_all),
    dtype=float
)


MSCR_OOF = np.zeros(
    len(y_all),
    dtype=float
)


# save important K-specific margins
MSCR_MARGIN_K8 = np.zeros(
    len(y_all),
    dtype=float
)

MSCR_MARGIN_K16 = np.zeros(
    len(y_all),
    dtype=float
)

MSCR_MARGIN_K32 = np.zeros(
    len(y_all),
    dtype=float
)

MSCR_MARGIN_K64 = np.zeros(
    len(y_all),
    dtype=float
)

MSCR_MARGIN_K128 = np.zeros(
    len(y_all),
    dtype=float
)


MSCR_MARGIN_STD = np.zeros(
    len(y_all),
    dtype=float
)


fold_rows = []

coefficient_rows = []


# ============================================================
# 14D.7 STRICT OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 118
    )

    print(
        f"MSCR OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 118
    )


    # ========================================================
    # Outer indices
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # A. CORE4 FOLD-LOCAL CALIBRATION
    # ========================================================

    train_cal = {}

    valid_cal = {}


    for model_name in HPSC_CORE:

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    core_outer_train = np.mean(

        np.column_stack([

            train_cal[m]

            for m in HPSC_CORE
        ]),

        axis=1
    )


    core_outer_valid = np.mean(

        np.column_stack([

            valid_cal[m]

            for m in HPSC_CORE
        ]),

        axis=1
    )


    MSCR_CORE_OOF[
        outer_valid_idx
    ] = core_outer_valid


    # ========================================================
    # B. OUTER-TRAIN SPLIT
    # ========================================================

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            MSCR_CONFIG[
                "fusion_fraction"
            ],

        random_state=
            SEED
            + 30000
            + outer_fold
    )


    (
        memory_local_idx,
        fusion_local_idx

    ) = next(

        splitter.split(

            X_all_raw[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    memory_global_idx = (

        outer_train_idx[
            memory_local_idx
        ]
    )


    fusion_global_idx = (

        outer_train_idx[
            fusion_local_idx
        ]
    )


    y_memory = (
        y_all[
            memory_global_idx
        ]
    )


    y_fusion = (
        y_all[
            fusion_global_idx
        ]
    )


    core_memory = (

        core_outer_train[
            memory_local_idx
        ]
    )


    core_fusion = (

        core_outer_train[
            fusion_local_idx
        ]
    )


    # ========================================================
    # C. PREPROCESSING — MEMORY ONLY
    # ========================================================

    feature_scaler = (
        StandardScaler()
    )


    X_memory = (

        feature_scaler
        .fit_transform(

            X_all_raw[
                memory_global_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_fusion = (

        feature_scaler
        .transform(

            X_all_raw[
                fusion_global_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    X_valid = (

        feature_scaler
        .transform(

            X_all_raw[
                outer_valid_idx
            ]
        )
        .astype(
            np.float32
        )
    )


    # ========================================================
    # D. FAST HPSC EMBEDDING
    # ========================================================

    embedding_model = (
        train_hpsc_embedding(

            X_memory,

            y_memory,

            core_memory,

            seed=
                SEED
                + 31000
                + outer_fold
        )
    )


    # ========================================================
    # E. EMBEDDINGS
    # ========================================================

    E_memory = (
        get_hpsc_embeddings(

            embedding_model,

            X_memory
        )
    )


    E_fusion = (
        get_hpsc_embeddings(

            embedding_model,

            X_fusion
        )
    )


    E_valid = (
        get_hpsc_embeddings(

            embedding_model,

            X_valid
        )
    )


    # ========================================================
    # F. RETRIEVAL MEMORY
    # ========================================================

    (
        pos_mem_idx,
        neg_mem_idx

    ) = select_hpsc_memory(

        y_memory,

        core_memory,

        seed=
            SEED
            + 32000
            + outer_fold
    )


    # respect 14D memory caps
    if len(pos_mem_idx) > MSCR_CONFIG[
        "max_positive_memory"
    ]:

        pos_mem_idx = pos_mem_idx[
            :
            MSCR_CONFIG[
                "max_positive_memory"
            ]
        ]


    if len(neg_mem_idx) > MSCR_CONFIG[
        "max_negative_memory"
    ]:

        neg_mem_idx = neg_mem_idx[
            :
            MSCR_CONFIG[
                "max_negative_memory"
            ]
        ]


    E_pos_memory = (
        E_memory[
            pos_mem_idx
        ]
    )


    E_neg_memory = (
        E_memory[
            neg_mem_idx
        ]
    )


    print(
        "\nRetrieval memory:"
    )

    print(
        "  Positive:",
        len(
            E_pos_memory
        )
    )

    print(
        "  Negative:",
        len(
            E_neg_memory
        )
    )


    # ========================================================
    # G. MULTI-SCALE RETRIEVAL
    # ========================================================

    R_fusion = (
        retrieve_multiscale_features(

            E_fusion,

            E_pos_memory,

            E_neg_memory,

            K_VALUES
        )
    )


    R_valid = (
        retrieve_multiscale_features(

            E_valid,

            E_pos_memory,

            E_neg_memory,

            K_VALUES
        )
    )


    assert (
        R_fusion.shape[1]
        ==
        len(
            MSCR_FEATURE_NAMES
        )
    )


    # ========================================================
    # H. RETRIEVAL COLUMN LOOKUP
    # ========================================================

    col = {

        name:
            i

        for i, name
        in enumerate(
            MSCR_FEATURE_NAMES
        )
    }


    # ========================================================
    # I. EXPERT DISAGREEMENT
    # ========================================================

    fusion_expert_matrix = np.column_stack([

        train_cal[
            m
        ][
            fusion_local_idx
        ]

        for m in HPSC_CORE
    ])


    valid_expert_matrix = np.column_stack([

        valid_cal[m]

        for m in HPSC_CORE
    ])


    fusion_disagreement = (

        fusion_expert_matrix.std(
            axis=1
        )
    )


    valid_disagreement = (

        valid_expert_matrix.std(
            axis=1
        )
    )


    # ========================================================
    # J. CORE LOGITS
    # ========================================================

    core_logit_fusion = (
        hpsc_logit(
            core_fusion
        )
    )


    core_logit_valid = (
        hpsc_logit(
            core_outer_valid
        )
    )


    # ========================================================
    # K. CORE-ONLY META BASELINE
    #
    # Important control:
    # Same training subset as retrieval fusion.
    # ========================================================

    core_meta_model = (
        LogisticRegression(

            penalty="l2",

            C=
                MSCR_CONFIG[
                    "core_meta_C"
                ],

            solver="lbfgs",

            max_iter=2000
        )
    )


    core_meta_model.fit(

        core_logit_fusion[
            :,
            None
        ],

        y_fusion
    )


    p_core_meta_valid = (

        core_meta_model
        .predict_proba(

            core_logit_valid[
                :,
                None
            ]

        )[
            :,
            1
        ]
    )


    MSCR_CORE_META_OOF[
        outer_valid_idx
    ] = p_core_meta_valid


    # ========================================================
    # L. MULTI-SCALE RETRIEVAL META FEATURES
    # ========================================================

    Z_fusion = np.column_stack([

        core_logit_fusion,

        fusion_disagreement,

        R_fusion,
    ])


    Z_valid = np.column_stack([

        core_logit_valid,

        valid_disagreement,

        R_valid,
    ])


    meta_feature_names = (

        [
            "core_logit",
            "expert_disagreement",
        ]

        +
        MSCR_FEATURE_NAMES
    )


    # ========================================================
    # M. STRONGLY REGULARIZED FUSION
    # ========================================================

    fusion_model = Pipeline([

        (
            "scaler",

            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    MSCR_CONFIG[
                        "fusion_C"
                    ],

                solver="lbfgs",

                max_iter=3000
            )
        ),
    ])


    fusion_model.fit(

        Z_fusion,

        y_fusion
    )


    p_valid_mscr = (

        fusion_model
        .predict_proba(
            Z_valid
        )[
            :,
            1
        ]
    )


    MSCR_OOF[
        outer_valid_idx
    ] = p_valid_mscr


    # ========================================================
    # N. STORE KEY RETRIEVAL SIGNALS
    # ========================================================

    MSCR_MARGIN_K8[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_k8"
        ]
    ]


    MSCR_MARGIN_K16[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_k16"
        ]
    ]


    MSCR_MARGIN_K32[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_k32"
        ]
    ]


    MSCR_MARGIN_K64[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_k64"
        ]
    ]


    MSCR_MARGIN_K128[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_k128"
        ]
    ]


    MSCR_MARGIN_STD[
        outer_valid_idx
    ] = R_valid[
        :,
        col[
            "margin_std"
        ]
    ]


    # ========================================================
    # O. METRICS
    # ========================================================

    core_pr = (
        average_precision_score(

            y_outer_valid,

            core_outer_valid
        )
    )


    core_meta_pr = (
        average_precision_score(

            y_outer_valid,

            p_core_meta_valid
        )
    )


    mscr_pr = (
        average_precision_score(

            y_outer_valid,

            p_valid_mscr
        )
    )


    # retrieval-only PR at every K
    retrieval_pr_by_k = {}


    for k in K_VALUES:

        retrieval_pr_by_k[
            k
        ] = (

            average_precision_score(

                y_outer_valid,

                R_valid[
                    :,
                    col[
                        f"margin_k{k}"
                    ]
                ]
            )
        )


    core_pm = (
        probability_metrics(

            y_outer_valid,

            core_outer_valid
        )
    )


    core_meta_pm = (
        probability_metrics(

            y_outer_valid,

            p_core_meta_valid
        )
    )


    mscr_pm = (
        probability_metrics(

            y_outer_valid,

            p_valid_mscr
        )
    )


    # ========================================================
    # P. HARD-POSITIVE RETRIEVAL DIAGNOSTICS
    # ========================================================

    positive_mask = (
        y_outer_valid
        ==
        1
    )


    q25 = np.quantile(

        core_outer_valid[
            positive_mask
        ],

        0.25
    )


    hard_positive_mask = (

        positive_mask

        &

        (
            core_outer_valid
            <=
            q25
        )
    )


    hard_positive_margin_k8 = float(

        R_valid[
            hard_positive_mask,
            col[
                "margin_k8"
            ]
        ].mean()
    )


    hard_positive_margin_k32 = float(

        R_valid[
            hard_positive_mask,
            col[
                "margin_k32"
            ]
        ].mean()
    )


    hard_positive_margin_k128 = float(

        R_valid[
            hard_positive_mask,
            col[
                "margin_k128"
            ]
        ].mean()
    )


    # ========================================================
    # Q. COEFFICIENTS
    # ========================================================

    coefficients = (

        fusion_model
        .named_steps[
            "lr"
        ]
        .coef_[0]
    )


    for feature_name, coefficient in zip(

        meta_feature_names,

        coefficients
    ):

        coefficient_rows.append({

            "Fold":
                outer_fold,

            "Feature":
                feature_name,

            "Coefficient":
                float(
                    coefficient
                )
        })


    # ========================================================
    # R. FOLD TABLE
    # ========================================================

    fold_record = {

        "Fold":
            outer_fold,

        "Core_PR":
            core_pr,

        "CoreMeta_PR":
            core_meta_pr,

        "MSCR_PR":
            mscr_pr,

        "Delta_vs_Core":
            mscr_pr
            -
            core_pr,

        "Delta_vs_CoreMeta":
            mscr_pr
            -
            core_meta_pr,

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "MSCR_ROC":
            mscr_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "CoreMeta_Brier":
            core_meta_pm[
                "Brier"
            ],

        "MSCR_Brier":
            mscr_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "CoreMeta_ECE":
            core_meta_pm[
                "ECE"
            ],

        "MSCR_ECE":
            mscr_pm[
                "ECE"
            ],

        "HardPositive_Margin_K8":
            hard_positive_margin_k8,

        "HardPositive_Margin_K32":
            hard_positive_margin_k32,

        "HardPositive_Margin_K128":
            hard_positive_margin_k128,
    }


    for k in K_VALUES:

        fold_record[
            f"Retrieval_PR_K{k}"
        ] = (
            retrieval_pr_by_k[
                k
            ]
        )


    fold_rows.append(
        fold_record
    )


    print(
        "\nPerformance:"
    )


    print(
        f"  Core PR       : "
        f"{core_pr:.6f}"
    )


    print(
        f"  CoreMeta PR   : "
        f"{core_meta_pr:.6f}"
    )


    print(
        f"  MSCR PR       : "
        f"{mscr_pr:.6f}"
    )


    print(
        f"  Δ vs Core     : "
        f"{mscr_pr-core_pr:+.6f}"
    )


    print(
        f"  Δ vs CoreMeta : "
        f"{mscr_pr-core_meta_pr:+.6f}"
    )


    print(
        "\nRetrieval-only PR:"
    )


    for k in K_VALUES:

        print(
            f"  K={k:<3}: "
            f"{retrieval_pr_by_k[k]:.6f}"
        )


    print(
        "\nHard-positive retrieval margin:"
    )


    print(
        "  K=8   :",
        f"{hard_positive_margin_k8:.6f}"
    )


    print(
        "  K=32  :",
        f"{hard_positive_margin_k32:.6f}"
    )


    print(
        "  K=128 :",
        f"{hard_positive_margin_k128:.6f}"
    )


    # ========================================================
    # S. CLEANUP
    # ========================================================

    del (
        embedding_model,
        E_memory,
        E_fusion,
        E_valid,
        E_pos_memory,
        E_neg_memory,
        R_fusion,
        R_valid,
        fusion_model,
        core_meta_model,
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 14D.8 GLOBAL THRESHOLDS
# ============================================================

core_t, _, _ = (
    select_best_threshold(

        y_all,

        MSCR_CORE_OOF,

        objective="F1"
    )
)


core_meta_t, _, _ = (
    select_best_threshold(

        y_all,

        MSCR_CORE_META_OOF,

        objective="F1"
    )
)


mscr_t, _, _ = (
    select_best_threshold(

        y_all,

        MSCR_OOF,

        objective="F1"
    )
)


# ============================================================
# 14D.9 GLOBAL METRICS
# ============================================================

core_metrics = (
    evaluate_binary_predictions(

        y_all,

        MSCR_CORE_OOF,

        threshold=
            core_t
    )
)


core_meta_metrics = (
    evaluate_binary_predictions(

        y_all,

        MSCR_CORE_META_OOF,

        threshold=
            core_meta_t
    )
)


mscr_metrics = (
    evaluate_binary_predictions(

        y_all,

        MSCR_OOF,

        threshold=
            mscr_t
    )
)


module14d_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "Core4_MetaBaseline",

        **core_meta_metrics
    },

    {
        "Method":
            "MSCR",

        **mscr_metrics
    },
])


module14d_results[
    "Delta_PR_vs_Core"
] = (

    module14d_results[
        "PR_AUC"
    ]

    -
    core_metrics[
        "PR_AUC"
    ]
)


module14d_results[
    "Delta_PR_vs_CoreMeta"
] = (

    module14d_results[
        "PR_AUC"
    ]

    -
    core_meta_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 14D — MULTI-SCALE CONTRASTIVE RETRIEVAL"
)

print(
    "=" * 130
)


display(

    module14d_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "Delta_PR_vs_CoreMeta",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 14D.10 FOLD RESULTS
# ============================================================

module14d_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "MSCR FOLD RESULTS"
)

print(
    "=" * 130
)


display(
    module14d_folds
)


print(
    "\nMean ΔPR vs Core:",
    module14d_folds[
        "Delta_vs_Core"
    ].mean()
)


print(
    "Positive folds vs Core:",
    int(
        (
            module14d_folds[
                "Delta_vs_Core"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "\nMean ΔPR vs CoreMeta:",
    module14d_folds[
        "Delta_vs_CoreMeta"
    ].mean()
)


print(
    "Positive folds vs CoreMeta:",
    int(
        (
            module14d_folds[
                "Delta_vs_CoreMeta"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 14D.11 GLOBAL RETRIEVAL PERFORMANCE BY SCALE
# ============================================================

retrieval_scale_rows = []


margin_buffers = {

    8:
        MSCR_MARGIN_K8,

    16:
        MSCR_MARGIN_K16,

    32:
        MSCR_MARGIN_K32,

    64:
        MSCR_MARGIN_K64,

    128:
        MSCR_MARGIN_K128,
}


for k in K_VALUES:

    score = (
        margin_buffers[
            k
        ]
    )


    retrieval_pr = (
        average_precision_score(

            y_all,

            score
        )
    )


    core_rank = (
        pd.Series(
            MSCR_CORE_OOF
        )
        .rank(
            method="average"
        )
    )


    retrieval_rank = (
        pd.Series(
            score
        )
        .rank(
            method="average"
        )
    )


    spearman = (

        np.corrcoef(

            core_rank,

            retrieval_rank

        )[0, 1]
    )


    retrieval_scale_rows.append({

        "K":
            k,

        "Retrieval_PR":
            retrieval_pr,

        "Core_Retrieval_Spearman":
            spearman,

        "Rank_Diversity":
            1.0
            -
            spearman,
    })


retrieval_scale_summary = (
    pd.DataFrame(
        retrieval_scale_rows
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "MULTI-SCALE RETRIEVAL SUMMARY"
)

print(
    "=" * 130
)


display(
    retrieval_scale_summary
)


# ============================================================
# 14D.12 COEFFICIENT STABILITY
# ============================================================

coefficient_df = pd.DataFrame(
    coefficient_rows
)


coefficient_summary = (

    coefficient_df
    .groupby(
        "Feature"
    )[
        "Coefficient"
    ]
    .agg([
        "mean",
        "std",
        "min",
        "max",
    ])
    .reset_index()
)


coefficient_summary[
    "Mean_Abs"
] = (

    coefficient_df
    .assign(
        Abs=lambda x:
            np.abs(
                x[
                    "Coefficient"
                ]
            )
    )
    .groupby(
        "Feature"
    )[
        "Abs"
    ]
    .mean()
    .values
)


coefficient_summary = (
    coefficient_summary
    .sort_values(
        "Mean_Abs",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "MSCR META-FEATURE COEFFICIENTS"
)

print(
    "=" * 130
)


display(
    coefficient_summary.head(
        20
    )
)


# ============================================================
# 14D.13 HARD-POSITIVE GLOBAL ANALYSIS
# ============================================================

positive_idx = np.where(
    y_all == 1
)[0]


hard_positive_cut = np.quantile(

    MSCR_CORE_OOF[
        positive_idx
    ],

    0.25
)


hard_positive_mask = (

    (y_all == 1)

    &

    (
        MSCR_CORE_OOF
        <=
        hard_positive_cut
    )
)


hard_positive_summary = []


for k in K_VALUES:

    margin = (
        margin_buffers[
            k
        ]
    )


    hard_positive_summary.append({

        "K":
            k,

        "Mean_All":
            float(
                margin.mean()
            ),

        "Mean_Positive":
            float(
                margin[
                    y_all == 1
                ].mean()
            ),

        "Mean_HardPositive":
            float(
                margin[
                    hard_positive_mask
                ].mean()
            ),
    })


hard_positive_scale_df = pd.DataFrame(
    hard_positive_summary
)


print(
    "\n"
    + "=" * 130
)

print(
    "MULTI-SCALE HARD-POSITIVE RETRIEVAL"
)

print(
    "=" * 130
)


display(
    hard_positive_scale_df
)


# ============================================================
# 14D.14 COMPARE 14B / 14C / 14D
# ============================================================

comparison_rows = [

    {
        "Method":
            "Core4Mean",

        "PR_AUC":
            average_precision_score(

                y_all,

                MSCR_CORE_OOF
            )
    },
]


if "HPRR_OOF" in globals():

    comparison_rows.append({

        "Method":
            "Fast_HPRR_14B",

        "PR_AUC":
            average_precision_score(

                y_all,

                HPRR_OOF
            )
    })


if "HPSC_OOF" in globals():

    comparison_rows.append({

        "Method":
            "HPSC_HPRR_14C",

        "PR_AUC":
            average_precision_score(

                y_all,

                HPSC_OOF
            )
    })


comparison_rows.append({

    "Method":
        "MSCR_14D",

    "PR_AUC":
        average_precision_score(

            y_all,

            MSCR_OOF
        )
})


retrieval_progression = (
    pd.DataFrame(
        comparison_rows
    )
)


core_pr_reference = (

    retrieval_progression
    .loc[
        retrieval_progression[
            "Method"
        ]
        ==
        "Core4Mean",

        "PR_AUC"
    ]
    .iloc[0]
)


retrieval_progression[
    "Delta_vs_Core"
] = (

    retrieval_progression[
        "PR_AUC"
    ]

    -
    core_pr_reference
)


print(
    "\n"
    + "=" * 130
)

print(
    "RETRIEVAL DEVELOPMENT PROGRESSION"
)

print(
    "=" * 130
)


display(
    retrieval_progression
)


# ============================================================
# 14D.15 SAVE
# ============================================================

save_csv(

    module14d_results,

    TAB_DIR
    / "module14d_mscr_results.csv"
)


save_csv(

    module14d_folds,

    TAB_DIR
    / "module14d_mscr_folds.csv"
)


save_csv(

    retrieval_scale_summary,

    TAB_DIR
    / "module14d_mscr_retrieval_scales.csv"
)


save_csv(

    coefficient_summary,

    TAB_DIR
    / "module14d_mscr_coefficients.csv"
)


save_csv(

    hard_positive_scale_df,

    TAB_DIR
    / "module14d_mscr_hard_positive_scales.csv"
)


save_csv(

    retrieval_progression,

    TAB_DIR
    / "module14d_retrieval_progression.csv"
)


mscr_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        MSCR_CORE_OOF,

    "Core4_MetaBaseline":
        MSCR_CORE_META_OOF,

    "MSCR":
        MSCR_OOF,

    "Margin_K8":
        MSCR_MARGIN_K8,

    "Margin_K16":
        MSCR_MARGIN_K16,

    "Margin_K32":
        MSCR_MARGIN_K32,

    "Margin_K64":
        MSCR_MARGIN_K64,

    "Margin_K128":
        MSCR_MARGIN_K128,

    "Margin_Stability":
        MSCR_MARGIN_STD,
})


save_csv(

    mscr_pred_df,

    PRED_DIR
    / "module14d_mscr_oof.csv"
)


module14d_manifest = {

    "method":
        "Multi-Scale Contrastive Retrieval Fusion",

    "k_values":
        K_VALUES,

    "embedding":
        "HPSC supervised contrastive",

    "embedding_epochs":
        HPSC_CONFIG[
            "epochs"
        ],

    "retrieval_computed_once_at_max_k":
        True,

    "core_only_meta_control":
        True,

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module14d_manifest,

    LOG_DIR
    / "module14d_manifest.json"
)


print(
    "\nMODULE 14D COMPLETE."
)

MSCR neighborhood scales: [8, 16, 32, 64, 128]
Retrieval feature count: 32

MSCR OUTER FOLD 1/5
    epoch 1/3 loss=7.71098 BCE=0.93737 SupCon=7.24229
    epoch 2/3 loss=7.37133 BCE=0.85905 SupCon=6.94180
    epoch 3/3 loss=7.36066 BCE=0.85322 SupCon=6.93405

Retrieval memory:
  Positive: 6000
  Negative: 12000

Performance:
  Core PR       : 0.380070
  CoreMeta PR   : 0.380070
  MSCR PR       : 0.376732
  Δ vs Core     : -0.003338
  Δ vs CoreMeta : -0.003338

Retrieval-only PR:
  K=8  : 0.225152
  K=16 : 0.254417
  K=32 : 0.282027
  K=64 : 0.305494
  K=128: 0.322714

Hard-positive retrieval margin:
  K=8   : -0.001374
  K=32  : -0.001746
  K=128 : -0.002241

MSCR OUTER FOLD 2/5
    epoch 1/3 loss=7.72662 BCE=0.95920 SupCon=7.24702
    epoch 2/3 loss=7.36863 BCE=0.85611 SupCon=6.94057
    epoch 3/3 loss=7.35883 BCE=0.85041 SupCon=6.93362

Retrieval memory:
  Positive: 6000
  Negative: 12000

Performance:
  Core PR       : 0.382772
  CoreMeta PR   : 0.382772
  MSCR PR       : 0.380966
  

,Method,PR_AUC,Delta_PR_vs_Core,Delta_PR_vs_CoreMeta,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.000074,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.220
1,Core4_MetaBaseline,0.380253,-0.000074,0.000000,0.850553,0.069703,0.235574,0.001200,0.353200,0.520561,0.420852,0.356028,0.225
2,MSCR,0.378104,-0.002224,-0.002149,0.849960,0.069830,0.235966,0.002068,0.351632,0.521450,0.420026,0.355140,0.225



MSCR FOLD RESULTS


,Fold,Core_PR,CoreMeta_PR,MSCR_PR,Delta_vs_Core,Delta_vs_CoreMeta,Core_ROC,MSCR_ROC,Core_Brier,CoreMeta_Brier,...,CoreMeta_ECE,MSCR_ECE,HardPositive_Margin_K8,HardPositive_Margin_K32,HardPositive_Margin_K128,Retrieval_PR_K8,Retrieval_PR_K16,Retrieval_PR_K32,Retrieval_PR_K64,Retrieval_PR_K128
0,0,0.380070,0.380070,0.376732,-0.003338,-0.003338,0.852660,0.852048,0.069637,0.069631,...,0.002773,0.004259,-0.001374,-0.001746,-0.002241,0.225152,0.254417,0.282027,0.305494,0.322714
1,1,0.382772,0.382772,0.380966,-0.001806,-0.001806,0.850766,0.849895,0.069684,0.069692,...,0.003345,0.004022,-0.001242,-0.001609,-0.002071,0.222614,0.256328,0.288499,0.314727,0.331610
2,2,0.378467,0.378467,0.377594,-0.000873,-0.000873,0.849030,0.848443,0.069827,0.069837,...,0.003364,0.003938,-0.001367,-0.001741,-0.002288,0.220521,0.244631,0.269419,0.293930,0.312191
3,3,0.372833,0.372833,0.371648,-0.001184,-0.001184,0.849083,0.848923,0.070081,0.070080,...,0.005765,0.004151,-0.001277,-0.001647,-0.002125,0.223065,0.257341,0.290692,0.315890,0.331814
4,4,0.390072,0.390072,0.387002,-0.003070,-0.003070,0.851760,0.850957,0.069301,0.069274,...,0.003709,0.004983,-0.001336,-0.001690,-0.002131,0.218250,0.248928,0.278717,0.306027,0.324585



Mean ΔPR vs Core: -0.0020541488810824405
Positive folds vs Core: 0 / 5

Mean ΔPR vs CoreMeta: -0.0020541488810824405
Positive folds vs CoreMeta: 0 / 5

MULTI-SCALE RETRIEVAL SUMMARY


...


MULTI-SCALE HARD-POSITIVE RETRIEVAL


,K,Mean_All,Mean_Positive,Mean_HardPositive
0,8,-0.001643,-0.000549,-0.001320
1,16,-0.001898,-0.000620,-0.001492
2,32,-0.002195,-0.000702,-0.001687
3,64,-0.002557,-0.000796,-0.001906
4,128,-0.003031,-0.000906,-0.002171



RETRIEVAL DEVELOPMENT PROGRESSION


,Method,PR_AUC,Delta_vs_Core
0,Core4Mean,0.380327,0.000000
1,Fast_HPRR_14B,0.380384,0.000057
2,HPSC_HPRR_14C,0.380429,0.000101
3,MSCR_14D,0.378104,-0.002224



MODULE 14D COMPLETE.


In [34]:
# ============================================================
# MODULE 14E — HPPM
# Hard-Positive Prototype Memory
# ============================================================
#
# Idea:
#
# 1. Train HPSC embedding on outer-training memory subset
# 2. Build class-specific prototypes:
#
#       regular positives
#       hard positives
#       negatives
#
# 3. For each query patient, calculate similarity to
#    closest / top prototype groups
# 4. Fuse a SMALL number of prototype-memory signals with Core4
#
# Fast:
# - same 3-epoch embedding as Module 14C
# - MiniBatchKMeans in 24-D space
# - ~64 + 32 + 128 prototypes
# - no expensive retrieval bank
#
# Strict:
# - outer validation never used to learn prototypes
# - fusion model fitted only on outer-training fusion subset
# - HOLDOUT untouched
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score


# ============================================================
# 14E.1 REQUIRE MODULE 14C DEFINITIONS
# ============================================================

required_objects = [
    "train_hpsc_embedding",
    "get_hpsc_embeddings",
    "HPSC_FEATURES",
    "HPSC_CORE",
    "HPSC_DEVICE",
    "hpsc_logit",
]

missing = [
    x
    for x in required_objects
    if x not in globals()
]

if len(missing) > 0:

    raise RuntimeError(

        "Run Module 14C definitions first. Missing: "
        + ", ".join(
            missing
        )
    )


# ============================================================
# 14E.2 CONFIG
# ============================================================

HPPM_CONFIG = {

    # outer train split
    "fusion_fraction":
        0.18,

    # prototype counts
    "regular_positive_prototypes":
        64,

    "hard_positive_prototypes":
        32,

    "negative_prototypes":
        128,

    # hard positives:
    # lowest Core4-scored positive training cases
    "hard_positive_fraction":
        0.25,

    # prototype clustering
    "kmeans_batch_size":
        2048,

    "kmeans_max_iter":
        100,

    "kmeans_n_init":
        3,

    # prototype neighborhood summaries
    "top_proto_k":
        5,

    # strongly regularized fusion
    "fusion_C":
        0.03,

    # same-subset Core-only meta control
    "core_meta_C":
        0.10,
}


print(
    "HPPM configuration:"
)

for k, v in HPPM_CONFIG.items():

    print(
        f"  {k:<32}: {v}"
    )


# ============================================================
# 14E.3 DATA
# ============================================================

y_all = (
    y_dev
    .to_numpy(
        dtype=int
    )
)


X_all_raw = (

    X_dev[
        HPSC_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 14E.4 HELPERS
# ============================================================

def normalize_rows(
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    norm = np.linalg.norm(
        X,
        axis=1,
        keepdims=True
    )

    norm = np.maximum(
        norm,
        1e-8
    )

    return (
        X / norm
    )


# ============================================================
# 14E.5 BUILD PROTOTYPES
# ============================================================

def fit_prototypes(
    embeddings,
    n_clusters,
    seed
):

    n_clusters = min(
        int(n_clusters),
        len(embeddings)
    )


    model = MiniBatchKMeans(

        n_clusters=
            n_clusters,

        random_state=
            seed,

        batch_size=
            HPPM_CONFIG[
                "kmeans_batch_size"
            ],

        max_iter=
            HPPM_CONFIG[
                "kmeans_max_iter"
            ],

        n_init=
            HPPM_CONFIG[
                "kmeans_n_init"
            ],

        reassignment_ratio=
            0.01,
    )


    model.fit(
        embeddings
    )


    centers = (
        model.cluster_centers_
        .astype(
            np.float32
        )
    )


    # cosine-based scoring later
    centers = normalize_rows(
        centers
    )


    return (
        centers,
        model
    )


# ============================================================
# 14E.6 PROTOTYPE SIMILARITY FEATURES
# ============================================================

def prototype_similarity_features(
    query_embeddings,
    regular_pos_proto,
    hard_pos_proto,
    negative_proto,
    top_k=5
):

    q = torch.tensor(

        query_embeddings,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    rp = torch.tensor(

        regular_pos_proto,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    hp = torch.tensor(

        hard_pos_proto,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    ng = torch.tensor(

        negative_proto,

        dtype=torch.float32,

        device=
            HPSC_DEVICE
    )


    with torch.no_grad():

        sim_regular = (
            q
            @
            rp.T
        )


        sim_hard = (
            q
            @
            hp.T
        )


        sim_negative = (
            q
            @
            ng.T
        )


        # ----------------------------------------------------
        # closest prototype
        # ----------------------------------------------------

        max_regular = (
            sim_regular.max(
                dim=1
            ).values
        )


        max_hard = (
            sim_hard.max(
                dim=1
            ).values
        )


        max_negative = (
            sim_negative.max(
                dim=1
            ).values
        )


        # ----------------------------------------------------
        # mean top-K prototype similarity
        # ----------------------------------------------------

        k_reg = min(
            top_k,
            sim_regular.shape[1]
        )

        k_hard = min(
            top_k,
            sim_hard.shape[1]
        )

        k_neg = min(
            top_k,
            sim_negative.shape[1]
        )


        top_regular = torch.topk(

            sim_regular,

            k=
                k_reg,

            dim=1,

            largest=True

        ).values.mean(
            dim=1
        )


        top_hard = torch.topk(

            sim_hard,

            k=
                k_hard,

            dim=1,

            largest=True

        ).values.mean(
            dim=1
        )


        top_negative = torch.topk(

            sim_negative,

            k=
                k_neg,

            dim=1,

            largest=True

        ).values.mean(
            dim=1
        )


        # ----------------------------------------------------
        # margins
        # ----------------------------------------------------

        regular_margin = (
            top_regular
            -
            top_negative
        )


        hard_margin = (
            top_hard
            -
            top_negative
        )


        best_positive = torch.maximum(

            top_regular,

            top_hard
        )


        best_positive_margin = (

            best_positive
            -
            top_negative
        )


        hard_vs_regular = (

            top_hard
            -
            top_regular
        )


        # ----------------------------------------------------
        # local prototype support
        # ----------------------------------------------------

        support = (

            (
                top_regular
                +
                top_hard
                +
                top_negative
            )
            /
            3.0
        )


        features = torch.stack(

            [
                max_regular,
                max_hard,
                max_negative,

                top_regular,
                top_hard,
                top_negative,

                regular_margin,
                hard_margin,
                best_positive_margin,

                hard_vs_regular,
                support,
            ],

            dim=1
        )


    return (
        features
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


HPPM_PROTO_FEATURES = [

    "max_regular_positive",
    "max_hard_positive",
    "max_negative",

    "top_regular_positive",
    "top_hard_positive",
    "top_negative",

    "regular_positive_margin",
    "hard_positive_margin",
    "best_positive_margin",

    "hard_vs_regular",
    "prototype_support",
]


# ============================================================
# 14E.7 BUFFERS
# ============================================================

HPPM_CORE_OOF = np.zeros(
    len(y_all),
    dtype=float
)


HPPM_CORE_META_OOF = np.zeros(
    len(y_all),
    dtype=float
)


HPPM_OOF = np.zeros(
    len(y_all),
    dtype=float
)


HPPM_HARD_MARGIN = np.zeros(
    len(y_all),
    dtype=float
)


HPPM_REGULAR_MARGIN = np.zeros(
    len(y_all),
    dtype=float
)


HPPM_BEST_POS_MARGIN = np.zeros(
    len(y_all),
    dtype=float
)


fold_rows = []

coefficient_rows = []

prototype_rows = []


# ============================================================
# 14E.8 STRICT OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 120
    )

    print(
        f"HPPM OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 120
    )


    # ========================================================
    # Outer indices
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # A. Fold-local calibrated Core4
    # ========================================================

    train_cal = {}
    valid_cal = {}


    for model_name in HPSC_CORE:

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    core_outer_train = np.mean(

        np.column_stack([

            train_cal[m]

            for m in HPSC_CORE
        ]),

        axis=1
    )


    core_outer_valid = np.mean(

        np.column_stack([

            valid_cal[m]

            for m in HPSC_CORE
        ]),

        axis=1
    )


    HPPM_CORE_OOF[
        outer_valid_idx
    ] = core_outer_valid


    # ========================================================
    # B. Outer-train split
    # ========================================================

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            HPPM_CONFIG[
                "fusion_fraction"
            ],

        random_state=
            SEED
            + 40000
            + outer_fold
    )


    (
        memory_local_idx,
        fusion_local_idx

    ) = next(

        splitter.split(

            X_all_raw[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    memory_global_idx = (

        outer_train_idx[
            memory_local_idx
        ]
    )


    fusion_global_idx = (

        outer_train_idx[
            fusion_local_idx
        ]
    )


    y_memory = (
        y_all[
            memory_global_idx
        ]
    )


    y_fusion = (
        y_all[
            fusion_global_idx
        ]
    )


    core_memory = (

        core_outer_train[
            memory_local_idx
        ]
    )


    core_fusion = (

        core_outer_train[
            fusion_local_idx
        ]
    )


    # ========================================================
    # C. Preprocessing
    # ========================================================

    scaler = StandardScaler()


    X_memory = (

        scaler.fit_transform(

            X_all_raw[
                memory_global_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    X_fusion = (

        scaler.transform(

            X_all_raw[
                fusion_global_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    X_valid = (

        scaler.transform(

            X_all_raw[
                outer_valid_idx
            ]
        )

        .astype(
            np.float32
        )
    )


    # ========================================================
    # D. Train HPSC embedding
    # ========================================================

    embedding_model = (
        train_hpsc_embedding(

            X_memory,

            y_memory,

            core_memory,

            seed=
                SEED
                + 41000
                + outer_fold
        )
    )


    # ========================================================
    # E. Embeddings
    # ========================================================

    E_memory = (
        get_hpsc_embeddings(

            embedding_model,

            X_memory
        )
    )


    E_fusion = (
        get_hpsc_embeddings(

            embedding_model,

            X_fusion
        )
    )


    E_valid = (
        get_hpsc_embeddings(

            embedding_model,

            X_valid
        )
    )


    # ========================================================
    # F. Define prototype populations
    # ========================================================

    positive_memory_idx = np.where(
        y_memory == 1
    )[0]


    negative_memory_idx = np.where(
        y_memory == 0
    )[0]


    positive_scores = (

        core_memory[
            positive_memory_idx
        ]
    )


    hard_positive_cut = np.quantile(

        positive_scores,

        HPPM_CONFIG[
            "hard_positive_fraction"
        ]
    )


    hard_positive_idx = positive_memory_idx[

        positive_scores
        <=
        hard_positive_cut
    ]


    regular_positive_idx = positive_memory_idx[

        positive_scores
        >
        hard_positive_cut
    ]


    print(
        "\nPrototype populations:"
    )

    print(
        "  Regular positives:",
        len(
            regular_positive_idx
        )
    )

    print(
        "  Hard positives   :",
        len(
            hard_positive_idx
        )
    )

    print(
        "  Negatives        :",
        len(
            negative_memory_idx
        )
    )


    # ========================================================
    # G. Fit class-specific prototypes
    # ========================================================

    regular_pos_proto, reg_km = (
        fit_prototypes(

            E_memory[
                regular_positive_idx
            ],

            HPPM_CONFIG[
                "regular_positive_prototypes"
            ],

            seed=
                SEED
                + 42000
                + outer_fold
        )
    )


    hard_pos_proto, hard_km = (
        fit_prototypes(

            E_memory[
                hard_positive_idx
            ],

            HPPM_CONFIG[
                "hard_positive_prototypes"
            ],

            seed=
                SEED
                + 43000
                + outer_fold
        )
    )


    negative_proto, neg_km = (
        fit_prototypes(

            E_memory[
                negative_memory_idx
            ],

            HPPM_CONFIG[
                "negative_prototypes"
            ],

            seed=
                SEED
                + 44000
                + outer_fold
        )
    )


    prototype_rows.append({

        "Fold":
            outer_fold,

        "RegularPositive_Prototypes":
            len(
                regular_pos_proto
            ),

        "HardPositive_Prototypes":
            len(
                hard_pos_proto
            ),

        "Negative_Prototypes":
            len(
                negative_proto
            ),

        "RegularPositive_N":
            len(
                regular_positive_idx
            ),

        "HardPositive_N":
            len(
                hard_positive_idx
            ),

        "Negative_N":
            len(
                negative_memory_idx
            ),
    })


    # ========================================================
    # H. Prototype-memory features
    # ========================================================

    P_fusion = (
        prototype_similarity_features(

            E_fusion,

            regular_pos_proto,

            hard_pos_proto,

            negative_proto,

            top_k=
                HPPM_CONFIG[
                    "top_proto_k"
                ]
        )
    )


    P_valid = (
        prototype_similarity_features(

            E_valid,

            regular_pos_proto,

            hard_pos_proto,

            negative_proto,

            top_k=
                HPPM_CONFIG[
                    "top_proto_k"
                ]
        )
    )


    proto_col = {

        name:
            i

        for i, name in enumerate(
            HPPM_PROTO_FEATURES
        )
    }


    # ========================================================
    # I. Expert disagreement
    # ========================================================

    fusion_expert_matrix = np.column_stack([

        train_cal[m][
            fusion_local_idx
        ]

        for m in HPSC_CORE
    ])


    valid_expert_matrix = np.column_stack([

        valid_cal[m]

        for m in HPSC_CORE
    ])


    fusion_disagreement = (

        fusion_expert_matrix.std(
            axis=1
        )
    )


    valid_disagreement = (

        valid_expert_matrix.std(
            axis=1
        )
    )


    # ========================================================
    # J. Core logits
    # ========================================================

    core_logit_fusion = (
        hpsc_logit(
            core_fusion
        )
    )


    core_logit_valid = (
        hpsc_logit(
            core_outer_valid
        )
    )


    # ========================================================
    # K. Core-only meta control
    # ========================================================

    core_meta_model = LogisticRegression(

        penalty="l2",

        C=
            HPPM_CONFIG[
                "core_meta_C"
            ],

        solver="lbfgs",

        max_iter=2000
    )


    core_meta_model.fit(

        core_logit_fusion[
            :,
            None
        ],

        y_fusion
    )


    p_core_meta = (

        core_meta_model
        .predict_proba(

            core_logit_valid[
                :,
                None
            ]

        )[
            :,
            1
        ]
    )


    HPPM_CORE_META_OOF[
        outer_valid_idx
    ] = p_core_meta


    # ========================================================
    # L. Small fusion representation
    # ========================================================
    #
    # Keep intentionally compact.
    #
    # ========================================================

    selected_proto_features = [

        "top_regular_positive",
        "top_hard_positive",
        "top_negative",

        "regular_positive_margin",
        "hard_positive_margin",
        "best_positive_margin",

        "hard_vs_regular",
        "prototype_support",
    ]


    selected_proto_idx = [

        proto_col[x]

        for x in selected_proto_features
    ]


    Z_fusion = np.column_stack([

        core_logit_fusion,

        fusion_disagreement,

        P_fusion[
            :,
            selected_proto_idx
        ],
    ])


    Z_valid = np.column_stack([

        core_logit_valid,

        valid_disagreement,

        P_valid[
            :,
            selected_proto_idx
        ],
    ])


    meta_feature_names = [

        "core_logit",
        "expert_disagreement",

    ] + selected_proto_features


    # ========================================================
    # M. Fusion
    # ========================================================

    fusion_model = Pipeline([

        (
            "scaler",

            StandardScaler()
        ),

        (
            "lr",

            LogisticRegression(

                penalty="l2",

                C=
                    HPPM_CONFIG[
                        "fusion_C"
                    ],

                solver="lbfgs",

                max_iter=3000
            )
        ),
    ])


    fusion_model.fit(

        Z_fusion,

        y_fusion
    )


    p_valid_hppm = (

        fusion_model
        .predict_proba(
            Z_valid
        )[
            :,
            1
        ]
    )


    HPPM_OOF[
        outer_valid_idx
    ] = p_valid_hppm


    # ========================================================
    # N. Save main prototype margins
    # ========================================================

    HPPM_HARD_MARGIN[
        outer_valid_idx
    ] = P_valid[
        :,
        proto_col[
            "hard_positive_margin"
        ]
    ]


    HPPM_REGULAR_MARGIN[
        outer_valid_idx
    ] = P_valid[
        :,
        proto_col[
            "regular_positive_margin"
        ]
    ]


    HPPM_BEST_POS_MARGIN[
        outer_valid_idx
    ] = P_valid[
        :,
        proto_col[
            "best_positive_margin"
        ]
    ]


    # ========================================================
    # O. Evaluation
    # ========================================================

    core_pr = (
        average_precision_score(

            y_outer_valid,

            core_outer_valid
        )
    )


    core_meta_pr = (
        average_precision_score(

            y_outer_valid,

            p_core_meta
        )
    )


    hppm_pr = (
        average_precision_score(

            y_outer_valid,

            p_valid_hppm
        )
    )


    hard_proto_pr = (
        average_precision_score(

            y_outer_valid,

            HPPM_HARD_MARGIN[
                outer_valid_idx
            ]
        )
    )


    regular_proto_pr = (
        average_precision_score(

            y_outer_valid,

            HPPM_REGULAR_MARGIN[
                outer_valid_idx
            ]
        )
    )


    best_proto_pr = (
        average_precision_score(

            y_outer_valid,

            HPPM_BEST_POS_MARGIN[
                outer_valid_idx
            ]
        )
    )


    core_pm = probability_metrics(

        y_outer_valid,

        core_outer_valid
    )


    hppm_pm = probability_metrics(

        y_outer_valid,

        p_valid_hppm
    )


    # ========================================================
    # P. Hard-positive diagnostic
    # ========================================================

    positive_mask = (
        y_outer_valid == 1
    )


    hp_q25 = np.quantile(

        core_outer_valid[
            positive_mask
        ],

        0.25
    )


    hard_positive_valid = (

        positive_mask

        &

        (
            core_outer_valid
            <=
            hp_q25
        )
    )


    hard_margin_hp = float(

        HPPM_HARD_MARGIN[
            outer_valid_idx
        ][
            hard_positive_valid
        ].mean()
    )


    regular_margin_hp = float(

        HPPM_REGULAR_MARGIN[
            outer_valid_idx
        ][
            hard_positive_valid
        ].mean()
    )


    # ========================================================
    # Q. Coefficients
    # ========================================================

    coefficients = (

        fusion_model
        .named_steps[
            "lr"
        ]
        .coef_[0]
    )


    for name, value in zip(

        meta_feature_names,

        coefficients
    ):

        coefficient_rows.append({

            "Fold":
                outer_fold,

            "Feature":
                name,

            "Coefficient":
                float(
                    value
                ),
        })


    # ========================================================
    # R. Fold row
    # ========================================================

    fold_rows.append({

        "Fold":
            outer_fold,

        "Core_PR":
            core_pr,

        "CoreMeta_PR":
            core_meta_pr,

        "HPPM_PR":
            hppm_pr,

        "Delta_vs_Core":
            hppm_pr
            -
            core_pr,

        "Delta_vs_CoreMeta":
            hppm_pr
            -
            core_meta_pr,

        "HardPrototype_PR":
            hard_proto_pr,

        "RegularPrototype_PR":
            regular_proto_pr,

        "BestPositivePrototype_PR":
            best_proto_pr,

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "HPPM_ROC":
            hppm_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "HPPM_Brier":
            hppm_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "HPPM_ECE":
            hppm_pm[
                "ECE"
            ],

        "HardPositive_HardProtoMargin":
            hard_margin_hp,

        "HardPositive_RegularProtoMargin":
            regular_margin_hp,
    })


    print(
        "\nPerformance:"
    )

    print(
        f"  Core PR       : "
        f"{core_pr:.6f}"
    )

    print(
        f"  CoreMeta PR   : "
        f"{core_meta_pr:.6f}"
    )

    print(
        f"  HPPM PR       : "
        f"{hppm_pr:.6f}"
    )

    print(
        f"  Δ vs Core     : "
        f"{hppm_pr-core_pr:+.6f}"
    )

    print(
        f"  Δ vs CoreMeta : "
        f"{hppm_pr-core_meta_pr:+.6f}"
    )


    print(
        "\nPrototype-only PR:"
    )

    print(
        f"  Hard-positive : "
        f"{hard_proto_pr:.6f}"
    )

    print(
        f"  Regular-pos   : "
        f"{regular_proto_pr:.6f}"
    )

    print(
        f"  Best-positive : "
        f"{best_proto_pr:.6f}"
    )


    print(
        "\nHard-positive prototype margins:"
    )

    print(
        f"  Hard prototype    : "
        f"{hard_margin_hp:.6f}"
    )

    print(
        f"  Regular prototype : "
        f"{regular_margin_hp:.6f}"
    )


    # ========================================================
    # S. Cleanup
    # ========================================================

    del (
        embedding_model,
        E_memory,
        E_fusion,
        E_valid,
        regular_pos_proto,
        hard_pos_proto,
        negative_proto,
        reg_km,
        hard_km,
        neg_km,
        P_fusion,
        P_valid,
        fusion_model,
        core_meta_model,
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 14E.9 GLOBAL EVALUATION
# ============================================================

core_t, _, _ = (
    select_best_threshold(

        y_all,

        HPPM_CORE_OOF,

        objective="F1"
    )
)


core_meta_t, _, _ = (
    select_best_threshold(

        y_all,

        HPPM_CORE_META_OOF,

        objective="F1"
    )
)


hppm_t, _, _ = (
    select_best_threshold(

        y_all,

        HPPM_OOF,

        objective="F1"
    )
)


core_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPPM_CORE_OOF,

        threshold=
            core_t
    )
)


core_meta_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPPM_CORE_META_OOF,

        threshold=
            core_meta_t
    )
)


hppm_metrics = (
    evaluate_binary_predictions(

        y_all,

        HPPM_OOF,

        threshold=
            hppm_t
    )
)


module14e_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "Core4_MetaBaseline",

        **core_meta_metrics
    },

    {
        "Method":
            "HPPM",

        **hppm_metrics
    },
])


module14e_results[
    "Delta_PR_vs_Core"
] = (

    module14e_results[
        "PR_AUC"
    ]

    -
    core_metrics[
        "PR_AUC"
    ]
)


module14e_results[
    "Delta_PR_vs_CoreMeta"
] = (

    module14e_results[
        "PR_AUC"
    ]

    -
    core_meta_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE 14E — HPPM RESULTS"
)

print(
    "=" * 130
)


display(

    module14e_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "Delta_PR_vs_CoreMeta",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 14E.10 FOLD RESULTS
# ============================================================

module14e_folds = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "HPPM FOLD RESULTS"
)

print(
    "=" * 130
)


display(
    module14e_folds
)


print(
    "\nMean ΔPR vs Core:",
    module14e_folds[
        "Delta_vs_Core"
    ].mean()
)


print(
    "Positive folds vs Core:",
    int(
        (
            module14e_folds[
                "Delta_vs_Core"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "\nMean ΔPR vs CoreMeta:",
    module14e_folds[
        "Delta_vs_CoreMeta"
    ].mean()
)


print(
    "Positive folds vs CoreMeta:",
    int(
        (
            module14e_folds[
                "Delta_vs_CoreMeta"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 14E.11 PROTOTYPE-ONLY GLOBAL PERFORMANCE
# ============================================================

prototype_global = pd.DataFrame([

    {
        "Signal":
            "HardPositiveMargin",

        "PR_AUC":
            average_precision_score(

                y_all,

                HPPM_HARD_MARGIN
            )
    },

    {
        "Signal":
            "RegularPositiveMargin",

        "PR_AUC":
            average_precision_score(

                y_all,

                HPPM_REGULAR_MARGIN
            )
    },

    {
        "Signal":
            "BestPositiveMargin",

        "PR_AUC":
            average_precision_score(

                y_all,

                HPPM_BEST_POS_MARGIN
            )
    },
])


print(
    "\n"
    + "=" * 130
)

print(
    "PROTOTYPE-MEMORY SIGNAL PERFORMANCE"
)

print(
    "=" * 130
)


display(
    prototype_global
)


# ============================================================
# 14E.12 DIVERSITY
# ============================================================

core_rank = (
    pd.Series(
        HPPM_CORE_OOF
    )
    .rank(
        method="average"
    )
)


diversity_rows = []


for name, score in [

    (
        "HardPositiveMargin",
        HPPM_HARD_MARGIN
    ),

    (
        "RegularPositiveMargin",
        HPPM_REGULAR_MARGIN
    ),

    (
        "BestPositiveMargin",
        HPPM_BEST_POS_MARGIN
    ),
]:

    score_rank = (
        pd.Series(
            score
        )
        .rank(
            method="average"
        )
    )


    spearman = np.corrcoef(

        core_rank,

        score_rank

    )[0, 1]


    diversity_rows.append({

        "Signal":
            name,

        "Core_Spearman":
            spearman,

        "Rank_Diversity":
            1.0
            -
            spearman,
    })


prototype_diversity = pd.DataFrame(
    diversity_rows
)


print(
    "\n"
    + "=" * 130
)

print(
    "PROTOTYPE-MEMORY DIVERSITY"
)

print(
    "=" * 130
)


display(
    prototype_diversity
)


# ============================================================
# 14E.13 COEFFICIENT STABILITY
# ============================================================

coefficient_df = pd.DataFrame(
    coefficient_rows
)


coef_summary = (

    coefficient_df
    .groupby(
        "Feature"
    )[
        "Coefficient"
    ]
    .agg([
        "mean",
        "std",
        "min",
        "max",
    ])
    .reset_index()
)


mean_abs = (

    coefficient_df
    .assign(
        AbsCoeff=
            np.abs(
                coefficient_df[
                    "Coefficient"
                ]
            )
    )
    .groupby(
        "Feature"
    )[
        "AbsCoeff"
    ]
    .mean()
)


coef_summary[
    "Mean_Abs"
] = (

    coef_summary[
        "Feature"
    ]
    .map(
        mean_abs
    )
)


coef_summary = (

    coef_summary
    .sort_values(
        "Mean_Abs",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "HPPM COEFFICIENT STABILITY"
)

print(
    "=" * 130
)


display(
    coef_summary
)


# ============================================================
# 14E.14 GLOBAL HARD-POSITIVE ANALYSIS
# ============================================================

positive_idx = np.where(
    y_all == 1
)[0]


hard_positive_cut = np.quantile(

    HPPM_CORE_OOF[
        positive_idx
    ],

    0.25
)


hard_positive_mask = (

    (y_all == 1)

    &

    (
        HPPM_CORE_OOF
        <=
        hard_positive_cut
    )
)


hard_positive_summary = pd.DataFrame({

    "Metric": [

        "Hard positive N",

        "Mean Core score",

        "Mean HPPM score",

        "Hard-positive prototype margin",

        "Regular-positive prototype margin",

        "Best-positive prototype margin",
    ],

    "Value": [

        int(
            hard_positive_mask.sum()
        ),

        float(
            HPPM_CORE_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPPM_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPPM_HARD_MARGIN[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPPM_REGULAR_MARGIN[
                hard_positive_mask
            ].mean()
        ),

        float(
            HPPM_BEST_POS_MARGIN[
                hard_positive_mask
            ].mean()
        ),
    ],
})


print(
    "\n"
    + "=" * 130
)

print(
    "HPPM HARD-POSITIVE ANALYSIS"
)

print(
    "=" * 130
)


display(
    hard_positive_summary
)


# ============================================================
# 14E.15 RETRIEVAL/PROTOTYPE DEVELOPMENT PROGRESSION
# ============================================================

progression = [

    {
        "Method":
            "Core4Mean",

        "PR_AUC":
            average_precision_score(
                y_all,
                HPPM_CORE_OOF
            )
    },
]


if "HPRR_OOF" in globals():

    progression.append({

        "Method":
            "Fast_HPRR_14B",

        "PR_AUC":
            average_precision_score(
                y_all,
                HPRR_OOF
            )
    })


if "HPSC_OOF" in globals():

    progression.append({

        "Method":
            "HPSC_HPRR_14C",

        "PR_AUC":
            average_precision_score(
                y_all,
                HPSC_OOF
            )
    })


if "MSCR_OOF" in globals():

    progression.append({

        "Method":
            "MSCR_14D",

        "PR_AUC":
            average_precision_score(
                y_all,
                MSCR_OOF
            )
    })


progression.append({

    "Method":
        "HPPM_14E",

    "PR_AUC":
        average_precision_score(
            y_all,
            HPPM_OOF
        )
})


retrieval_progression_14e = (
    pd.DataFrame(
        progression
    )
)


core_reference = (

    retrieval_progression_14e
    .loc[
        retrieval_progression_14e[
            "Method"
        ]
        ==
        "Core4Mean",

        "PR_AUC"
    ]
    .iloc[0]
)


retrieval_progression_14e[
    "Delta_vs_Core"
] = (

    retrieval_progression_14e[
        "PR_AUC"
    ]

    -
    core_reference
)


print(
    "\n"
    + "=" * 130
)

print(
    "RETRIEVAL / PROTOTYPE DEVELOPMENT PROGRESSION"
)

print(
    "=" * 130
)


display(
    retrieval_progression_14e
)


# ============================================================
# 14E.16 SAVE
# ============================================================

save_csv(

    module14e_results,

    TAB_DIR
    / "module14e_hppm_results.csv"
)


save_csv(

    module14e_folds,

    TAB_DIR
    / "module14e_hppm_folds.csv"
)


save_csv(

    prototype_global,

    TAB_DIR
    / "module14e_hppm_prototype_signals.csv"
)


save_csv(

    prototype_diversity,

    TAB_DIR
    / "module14e_hppm_diversity.csv"
)


save_csv(

    coef_summary,

    TAB_DIR
    / "module14e_hppm_coefficients.csv"
)


save_csv(

    pd.DataFrame(
        prototype_rows
    ),

    TAB_DIR
    / "module14e_hppm_prototype_counts.csv"
)


save_csv(

    hard_positive_summary,

    TAB_DIR
    / "module14e_hppm_hard_positive.csv"
)


save_csv(

    retrieval_progression_14e,

    TAB_DIR
    / "module14e_retrieval_progression.csv"
)


hppm_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        HPPM_CORE_OOF,

    "Core4_MetaBaseline":
        HPPM_CORE_META_OOF,

    "HPPM":
        HPPM_OOF,

    "HardPositiveMargin":
        HPPM_HARD_MARGIN,

    "RegularPositiveMargin":
        HPPM_REGULAR_MARGIN,

    "BestPositiveMargin":
        HPPM_BEST_POS_MARGIN,
})


save_csv(

    hppm_pred_df,

    PRED_DIR
    / "module14e_hppm_oof.csv"
)


module14e_manifest = {

    "method":
        "Hard-Positive Prototype Memory",

    "embedding":
        "HPSC supervised contrastive embedding",

    "regular_positive_prototypes":
        HPPM_CONFIG[
            "regular_positive_prototypes"
        ],

    "hard_positive_prototypes":
        HPPM_CONFIG[
            "hard_positive_prototypes"
        ],

    "negative_prototypes":
        HPPM_CONFIG[
            "negative_prototypes"
        ],

    "hard_positive_fraction":
        HPPM_CONFIG[
            "hard_positive_fraction"
        ],

    "top_proto_k":
        HPPM_CONFIG[
            "top_proto_k"
        ],

    "core_meta_control":
        True,

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module14e_manifest,

    LOG_DIR
    / "module14e_manifest.json"
)


print(
    "\nMODULE 14E COMPLETE."
)

HPPM configuration:
  fusion_fraction                 : 0.18
  regular_positive_prototypes     : 64
  hard_positive_prototypes        : 32
  negative_prototypes             : 128
  hard_positive_fraction          : 0.25
  kmeans_batch_size               : 2048
  kmeans_max_iter                 : 100
  kmeans_n_init                   : 3
  top_proto_k                     : 5
  fusion_C                        : 0.03
  core_meta_C                     : 0.1

HPPM OUTER FOLD 1/5
    epoch 1/3 loss=7.68836 BCE=0.93489 SupCon=7.22092
    epoch 2/3 loss=7.37301 BCE=0.86195 SupCon=6.94204
    epoch 3/3 loss=7.36022 BCE=0.85151 SupCon=6.93447

Prototype populations:
  Regular positives: 9404
  Hard positives   : 3135
  Negatives        : 120592

Performance:
  Core PR       : 0.380070
  CoreMeta PR   : 0.380070
  HPPM PR       : 0.378584
  Δ vs Core     : -0.001486
  Δ vs CoreMeta : -0.001486

Prototype-only PR:
  Hard-positive : 0.074372
  Regular-pos   : 0.350458
  Best-positive : 0.336406

Ha

,Method,PR_AUC,Delta_PR_vs_Core,Delta_PR_vs_CoreMeta,ROC_AUC,Brier,NLL,ECE,Precision,Recall,F1,MCC,Threshold
0,Core4Mean,0.380327,0.000000,0.000134,0.850558,0.069706,0.235584,0.002374,0.350507,0.526054,0.420702,0.356064,0.220
1,Core4_MetaBaseline,0.380194,-0.000134,0.000000,0.850534,0.069709,0.235585,0.001387,0.353899,0.520979,0.421485,0.356745,0.225
2,HPPM,0.377337,-0.002990,-0.002856,0.850113,0.069843,0.235966,0.002553,0.342480,0.545255,0.420708,0.357034,0.210



HPPM FOLD RESULTS


,Fold,Core_PR,CoreMeta_PR,HPPM_PR,Delta_vs_Core,Delta_vs_CoreMeta,HardPrototype_PR,RegularPrototype_PR,BestPositivePrototype_PR,Core_ROC,HPPM_ROC,Core_Brier,HPPM_Brier,Core_ECE,HPPM_ECE,HardPositive_HardProtoMargin,HardPositive_RegularProtoMargin
0,0,0.380070,0.380070,0.378584,-0.001486,-0.001486,0.074372,0.350458,0.336406,0.852660,0.851856,0.069637,0.069755,0.003499,0.003469,-0.001650,-0.006616
1,1,0.382772,0.382772,0.378173,-0.004598,-0.004598,0.075245,0.346402,0.335030,0.850766,0.849988,0.069684,0.069878,0.004297,0.002716,-0.001756,-0.006630
2,2,0.378467,0.378467,0.375973,-0.002494,-0.002494,0.077906,0.332545,0.320997,0.849030,0.848973,0.069827,0.069945,0.002486,0.002088,-0.001795,-0.006512
3,3,0.372833,0.372833,0.370186,-0.002647,-0.002647,0.078967,0.338297,0.329694,0.849083,0.848811,0.070081,0.070189,0.005053,0.005344,-0.001689,-0.006506
4,4,0.390072,0.390072,0.386298,-0.003775,-0.003775,0.078725,0.348509,0.339591,0.851760,0.851429,0.069301,0.069446,0.005318,0.005640,-0.001586,-0.006290



Mean ΔPR vs Core: -0.0029998967449173853
Positive folds vs Core: 0 / 5

Mean ΔPR vs CoreMeta: -0.0029998967449173853
Positive folds vs CoreMeta: 0 / 5

PROTOTYPE-MEMORY SIGNAL PERFORMANCE


...


HPPM HARD-POSITIVE ANALYSIS


,Metric,Value
0,Hard positive N,4779.000000
1,Mean Core score,0.068849
2,Mean HPPM score,0.069264
3,Hard-positive prototype margin,-0.001700
4,Regular-positive prototype margin,-0.006512
5,Best-positive prototype margin,-0.001527



RETRIEVAL / PROTOTYPE DEVELOPMENT PROGRESSION


,Method,PR_AUC,Delta_vs_Core
0,Core4Mean,0.380327,0.000000
1,Fast_HPRR_14B,0.380384,0.000057
2,HPSC_HPRR_14C,0.380429,0.000101
3,MSCR_14D,0.378104,-0.002224
4,HPPM_14E,0.377337,-0.002990



MODULE 14E COMPLETE.


In [36]:
# ============================================================
# MODULE 15A — CLER
# Cross-Model Leaf Embedding Ranker
# ============================================================
#
# Novel representation:
#
# Raw tabular features
#       |
#       +------> fold-local CatBoost ------> leaf IDs
#       |
#       +------> fold-local LightGBM ------> leaf IDs
#                                           |
#                              trainable leaf embeddings
#                                           |
#                        cross-model structural representation
#                                           |
#                    BCE + hard-pair ranking objective
#                                           |
#                                      CLER score
#
# Then:
#
#       Core4 + CLER -> small inner-trained fusion
#
#
# STRICT PROTOCOL
# ------------------------------------------------------------
# - Outer validation is untouched.
# - Inside every outer fold:
#
#       outer train
#          |
#          +---- 85% structural training
#          |
#          +---- 15% calibration/fusion
#
# - Trees are trained ONLY on structural-training subset.
# - Leaf network trained ONLY on structural-training subset.
# - Platt/fusion fitted ONLY on calibration subset.
# - HOLDOUT untouched.
#
#
# FAST SETTINGS
# ------------------------------------------------------------
# - CatBoost: 180 trees
# - LightGBM: 220 trees
# - embedding dim: 6
# - leaf network: tiny
# - epochs: 2
# - batch size: 4096
#
# ============================================================


import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier


# ============================================================
# 15A.1 CONFIG
# ============================================================

CLER_FEATURES = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


CLER_CORE = [
    "HistGradientBoosting",
    "CatBoost",
    "AutoInt",
    "DeepFM_v2",
]


CLER_CONFIG = {

    # --------------------------------------------------------
    # internal calibration/fusion split
    # --------------------------------------------------------

    "calibration_fraction":
        0.15,


    # --------------------------------------------------------
    # CatBoost structural model
    # --------------------------------------------------------

    "cat_iterations":
        180,

    "cat_depth":
        6,

    "cat_learning_rate":
        0.05,

    "cat_l2":
        6.0,


    # --------------------------------------------------------
    # LightGBM structural model
    # --------------------------------------------------------

    "lgb_iterations":
        220,

    "lgb_num_leaves":
        31,

    "lgb_learning_rate":
        0.05,


    # --------------------------------------------------------
    # leaf embedding network
    # --------------------------------------------------------

    "embedding_dim":
        6,

    "hidden_dim":
        32,

    "dropout":
        0.15,

    "epochs":
        2,

    "batch_size":
        4096,

    "lr":
        8e-4,

    "weight_decay":
        2e-4,


    # --------------------------------------------------------
    # ranking objective
    # --------------------------------------------------------

    "bce_weight":
        0.65,

    "rank_weight":
        0.35,

    "rank_temperature":
        0.08,

    "max_pairs_per_batch":
        192,


    # --------------------------------------------------------
    # final fusion
    # --------------------------------------------------------

    "fusion_C":
        0.05,

    "core_meta_C":
        0.10,
}


CLER_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)


print(
    "CLER device:",
    CLER_DEVICE
)


# ============================================================
# 15A.2 REPRODUCIBILITY
# ============================================================

def cler_seed(seed):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 15A.3 HELPERS
# ============================================================

def cler_logit(p):

    p = np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

    return np.log(
        p
        /
        (
            1.0 - p
        )
    )


# ============================================================
# 15A.4 STRUCTURAL TREE MODELS
# ============================================================

def make_cler_catboost(
    seed
):

    return CatBoostClassifier(

        iterations=
            CLER_CONFIG[
                "cat_iterations"
            ],

        depth=
            CLER_CONFIG[
                "cat_depth"
            ],

        learning_rate=
            CLER_CONFIG[
                "cat_learning_rate"
            ],

        l2_leaf_reg=
            CLER_CONFIG[
                "cat_l2"
            ],

        loss_function=
            "Logloss",

        random_seed=
            seed,

        task_type=
            "GPU"
            if torch.cuda.is_available()
            else "CPU",

        devices=
            "0",

        verbose=
            False,

        allow_writing_files=
            False,
    )


def make_cler_lgbm(
    seed
):

    return LGBMClassifier(

        n_estimators=
            CLER_CONFIG[
                "lgb_iterations"
            ],

        num_leaves=
            CLER_CONFIG[
                "lgb_num_leaves"
            ],

        learning_rate=
            CLER_CONFIG[
                "lgb_learning_rate"
            ],

        max_depth=
            -1,

        subsample=
            0.90,

        colsample_bytree=
            0.90,

        reg_lambda=
            1.0,

        random_state=
            seed,

        n_jobs=
            -1,

        verbosity=
            -1,
    )


# ============================================================
# 15A.5 EXTRACT LEAF INDICES
# ============================================================

def extract_leaf_indices(
    cat_model,
    lgb_model,
    X
):

    # --------------------------------------------------------
    # CatBoost
    # shape:
    # N x number_of_trees
    # --------------------------------------------------------

    cat_leaf = (
        cat_model
        .get_leaf_indexes(
            X
        )
        .astype(
            np.int64
        )
    )


    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    lgb_leaf = (

        lgb_model
        .booster_
        .predict(
            X,
            pred_leaf=True
        )
        .astype(
            np.int64
        )
    )


    if lgb_leaf.ndim == 1:

        lgb_leaf = (
            lgb_leaf[
                :,
                None
            ]
        )


    return (
        cat_leaf,
        lgb_leaf
    )


# ============================================================
# 15A.6 CONVERT LEAVES TO GLOBAL EMBEDDING IDS
# ============================================================
#
# Every tree receives its own ID region.
#
# CatBoost:
# depth = 6 -> at most 64 leaves/tree
#
# LightGBM:
# at most num_leaves/tree
#
# ============================================================

def make_global_leaf_ids(
    cat_leaf,
    lgb_leaf
):

    n_samples = (
        cat_leaf.shape[0]
    )


    n_cat_trees = (
        cat_leaf.shape[1]
    )


    n_lgb_trees = (
        lgb_leaf.shape[1]
    )


    cat_capacity = (
        2
        **
        CLER_CONFIG[
            "cat_depth"
        ]
    )


    lgb_capacity = (
        CLER_CONFIG[
            "lgb_num_leaves"
        ]
    )


    # --------------------------------------------------------
    # CatBoost offsets
    # --------------------------------------------------------

    cat_offsets = (

        np.arange(
            n_cat_trees,
            dtype=np.int64
        )

        *
        cat_capacity
    )


    cat_ids = (

        cat_leaf

        +
        cat_offsets[
            None,
            :
        ]
    )


    # --------------------------------------------------------
    # LightGBM offsets
    # --------------------------------------------------------

    lgb_base = (
        n_cat_trees
        *
        cat_capacity
    )


    lgb_offsets = (

        lgb_base

        +

        np.arange(
            n_lgb_trees,
            dtype=np.int64
        )
        *
        lgb_capacity
    )


    lgb_ids = (

        lgb_leaf

        +
        lgb_offsets[
            None,
            :
        ]
    )


    total_vocab = (

        lgb_base

        +
        n_lgb_trees
        *
        lgb_capacity
    )


    return (
        cat_ids,
        lgb_ids,
        int(
            total_vocab
        )
    )


# ============================================================
# 15A.7 DATASET
# ============================================================

class CLERDataset(Dataset):

    def __init__(
        self,
        cat_ids,
        lgb_ids,
        y,
        core_score
    ):

        self.cat_ids = torch.tensor(
            cat_ids,
            dtype=torch.long
        )

        self.lgb_ids = torch.tensor(
            lgb_ids,
            dtype=torch.long
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

        self.core_score = torch.tensor(
            core_score,
            dtype=torch.float32
        )


    def __len__(self):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.cat_ids[
                idx
            ],

            self.lgb_ids[
                idx
            ],

            self.y[
                idx
            ],

            self.core_score[
                idx
            ],
        )


# ============================================================
# 15A.8 LEAF EMBEDDING NETWORK
# ============================================================

class CLERNet(nn.Module):

    def __init__(
        self,
        total_vocab
    ):

        super().__init__()


        emb_dim = (
            CLER_CONFIG[
                "embedding_dim"
            ]
        )


        self.embedding = nn.Embedding(

            total_vocab,

            emb_dim
        )


        self.head = nn.Sequential(

            nn.Linear(
                emb_dim * 2,
                CLER_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.GELU(),

            nn.LayerNorm(
                CLER_CONFIG[
                    "hidden_dim"
                ]
            ),

            nn.Dropout(
                CLER_CONFIG[
                    "dropout"
                ]
            ),

            nn.Linear(
                CLER_CONFIG[
                    "hidden_dim"
                ],
                1
            ),
        )


        nn.init.normal_(

            self.embedding.weight,

            mean=0.0,

            std=0.02
        )


    def forward(
        self,
        cat_ids,
        lgb_ids
    ):

        # ----------------------------------------------------
        # Tree-specific leaf embeddings
        # ----------------------------------------------------

        cat_emb = (
            self.embedding(
                cat_ids
            )
        )


        lgb_emb = (
            self.embedding(
                lgb_ids
            )
        )


        # ----------------------------------------------------
        # Pool across trees separately
        # ----------------------------------------------------

        cat_repr = (
            cat_emb.mean(
                dim=1
            )
        )


        lgb_repr = (
            lgb_emb.mean(
                dim=1
            )
        )


        z = torch.cat(

            [
                cat_repr,
                lgb_repr,
            ],

            dim=1
        )


        logit = (

            self.head(
                z
            )
            .squeeze(-1)
        )


        return (
            logit,
            z
        )


# ============================================================
# 15A.9 HARD-PAIR RANKING LOSS
# ============================================================
#
# Pair difficult positives with difficult negatives.
#
# Extra emphasis when Core4 scores are close:
#
#     positive core score ≈ negative core score
#
# These are precisely the pairs where ranking repair can
# potentially improve PR-AUC.
#
# ============================================================

def cler_pairwise_rank_loss(
    logits,
    labels,
    core_scores
):

    pos_idx = torch.where(
        labels > 0.5
    )[0]


    neg_idx = torch.where(
        labels <= 0.5
    )[0]


    if (
        len(pos_idx) == 0
        or
        len(neg_idx) == 0
    ):

        return torch.tensor(

            0.0,

            device=
                logits.device
        )


    # --------------------------------------------------------
    # Hard positives:
    # low Core4 scores
    # --------------------------------------------------------

    pos_core = (
        core_scores[
            pos_idx
        ]
    )


    pos_order = torch.argsort(
        pos_core,
        descending=False
    )


    pos_idx = (
        pos_idx[
            pos_order
        ]
    )


    # --------------------------------------------------------
    # Hard negatives:
    # high Core4 scores
    # --------------------------------------------------------

    neg_core = (
        core_scores[
            neg_idx
        ]
    )


    neg_order = torch.argsort(
        neg_core,
        descending=True
    )


    neg_idx = (
        neg_idx[
            neg_order
        ]
    )


    n_pairs = min(

        len(
            pos_idx
        ),

        len(
            neg_idx
        ),

        CLER_CONFIG[
            "max_pairs_per_batch"
        ]
    )


    if n_pairs < 1:

        return torch.tensor(

            0.0,

            device=
                logits.device
        )


    pos_idx = pos_idx[
        :n_pairs
    ]


    neg_idx = neg_idx[
        :n_pairs
    ]


    pos_logit = (
        logits[
            pos_idx
        ]
    )


    neg_logit = (
        logits[
            neg_idx
        ]
    )


    pos_core = (
        core_scores[
            pos_idx
        ]
    )


    neg_core = (
        core_scores[
            neg_idx
        ]
    )


    # --------------------------------------------------------
    # emphasize Core4-near-tie pairs
    # --------------------------------------------------------

    core_gap = torch.abs(

        pos_core
        -
        neg_core
    )


    closeness_weight = torch.exp(

        -core_gap

        /
        CLER_CONFIG[
            "rank_temperature"
        ]
    )


    # --------------------------------------------------------
    # additional hard-case weighting
    # --------------------------------------------------------

    positive_hardness = (

        1.0
        +
        (
            1.0
            -
            pos_core
        )
    )


    negative_hardness = (

        1.0
        +
        neg_core
    )


    pair_weight = (

        closeness_weight

        *
        positive_hardness

        *
        negative_hardness
    )


    pair_loss = F.softplus(

        -(
            pos_logit
            -
            neg_logit
        )
    )


    loss = (

        (
            pair_loss
            *
            pair_weight
        ).sum()

        /

        (
            pair_weight.sum()
            +
            1e-8
        )
    )


    return loss


# ============================================================
# 15A.10 TRAIN CLER
# ============================================================

def train_cler(
    cat_ids,
    lgb_ids,
    y,
    core_score,
    total_vocab,
    seed
):

    cler_seed(
        seed
    )


    dataset = CLERDataset(

        cat_ids,

        lgb_ids,

        y,

        core_score
    )


    loader = DataLoader(

        dataset,

        batch_size=
            CLER_CONFIG[
                "batch_size"
            ],

        shuffle=True,

        num_workers=2,

        pin_memory=True,
    )


    model = CLERNet(

        total_vocab=
            total_vocab

    ).to(
        CLER_DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            CLER_CONFIG[
                "lr"
            ],

        weight_decay=
            CLER_CONFIG[
                "weight_decay"
            ]
    )


    # --------------------------------------------------------
    # mild imbalance correction
    # --------------------------------------------------------

    n_pos = np.sum(
        y == 1
    )


    n_neg = np.sum(
        y == 0
    )


    pos_weight = torch.tensor(

        np.sqrt(
            n_neg / n_pos
        ),

        dtype=torch.float32,

        device=
            CLER_DEVICE
    )


    for epoch in range(
        CLER_CONFIG[
            "epochs"
        ]
    ):

        model.train()


        total_running = 0.0
        bce_running = 0.0
        rank_running = 0.0

        seen = 0


        for (
            cat_batch,
            lgb_batch,
            y_batch,
            core_batch
        ) in loader:

            cat_batch = cat_batch.to(
                CLER_DEVICE
            )

            lgb_batch = lgb_batch.to(
                CLER_DEVICE
            )

            y_batch = y_batch.to(
                CLER_DEVICE
            )

            core_batch = core_batch.to(
                CLER_DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits, _ = model(

                cat_batch,

                lgb_batch
            )


            # =================================================
            # Classification
            # =================================================

            bce = (
                F.binary_cross_entropy_with_logits(

                    logits,

                    y_batch,

                    pos_weight=
                        pos_weight
                )
            )


            # =================================================
            # Ranking repair
            # =================================================

            rank_loss = (
                cler_pairwise_rank_loss(

                    logits,

                    y_batch,

                    core_batch
                )
            )


            loss = (

                CLER_CONFIG[
                    "bce_weight"
                ]
                *
                bce

                +

                CLER_CONFIG[
                    "rank_weight"
                ]
                *
                rank_loss
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                3.0
            )


            optimizer.step()


            n = len(
                y_batch
            )


            total_running += (
                float(
                    loss.detach().cpu()
                )
                *
                n
            )


            bce_running += (
                float(
                    bce.detach().cpu()
                )
                *
                n
            )


            rank_running += (
                float(
                    rank_loss.detach().cpu()
                )
                *
                n
            )


            seen += n


        print(

            f"    epoch "
            f"{epoch+1}/"
            f"{CLER_CONFIG['epochs']} "

            f"loss="
            f"{total_running/seen:.5f} "

            f"BCE="
            f"{bce_running/seen:.5f} "

            f"Rank="
            f"{rank_running/seen:.5f}"
        )


    return model


# ============================================================
# 15A.11 PREDICT CLER LOGITS
# ============================================================

def predict_cler_logits(
    model,
    cat_ids,
    lgb_ids,
    batch_size=8192
):

    model.eval()

    outputs = []


    with torch.no_grad():

        for start in range(
            0,
            len(cat_ids),
            batch_size
        ):

            end = min(

                start
                +
                batch_size,

                len(
                    cat_ids
                )
            )


            cat_batch = torch.tensor(

                cat_ids[
                    start:end
                ],

                dtype=torch.long,

                device=
                    CLER_DEVICE
            )


            lgb_batch = torch.tensor(

                lgb_ids[
                    start:end
                ],

                dtype=torch.long,

                device=
                    CLER_DEVICE
            )


            logits, _ = model(

                cat_batch,

                lgb_batch
            )


            outputs.append(

                logits
                .cpu()
                .numpy()
            )


    return np.concatenate(
        outputs
    )


# ============================================================
# 15A.12 DATA
# ============================================================

X_all = (

    X_dev[
        CLER_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)


y_all = (

    y_dev
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 15A.13 OOF BUFFERS
# ============================================================

CLER_CORE_OOF = np.zeros(
    len(y_all),
    dtype=float
)


CLER_CORE_META_OOF = np.zeros(
    len(y_all),
    dtype=float
)


CLER_STANDALONE_OOF = np.zeros(
    len(y_all),
    dtype=float
)


CLER_FUSION_OOF = np.zeros(
    len(y_all),
    dtype=float
)


CLER_RAW_SCORE = np.zeros(
    len(y_all),
    dtype=float
)


fold_rows = []

fusion_coef_rows = []


# ============================================================
# 15A.14 STRICT OUTER CV
# ============================================================

for outer_fold in range(
    N_SPLITS
):

    print(
        "\n"
        + "=" * 120
    )

    print(
        f"CLER OUTER FOLD "
        f"{outer_fold+1}/{N_SPLITS}"
    )

    print(
        "=" * 120
    )


    # ========================================================
    # A. OUTER INDICES
    # ========================================================

    outer_train_idx = np.where(

        fold_assignment
        !=
        outer_fold

    )[0]


    outer_valid_idx = np.where(

        fold_assignment
        ==
        outer_fold

    )[0]


    y_outer_train = (
        y_all[
            outer_train_idx
        ]
    )


    y_outer_valid = (
        y_all[
            outer_valid_idx
        ]
    )


    # ========================================================
    # B. EXISTING CORE4 — FOLD-LOCAL CALIBRATION
    # ========================================================

    train_cal = {}

    valid_cal = {}


    for model_name in CLER_CORE:

        p_train, p_valid = (
            fit_apply_nested_calibration(

                model_name,

                outer_train_idx,

                outer_valid_idx
            )
        )


        train_cal[
            model_name
        ] = p_train


        valid_cal[
            model_name
        ] = p_valid


    core_outer_train = np.mean(

        np.column_stack([

            train_cal[m]

            for m
            in CLER_CORE
        ]),

        axis=1
    )


    core_outer_valid = np.mean(

        np.column_stack([

            valid_cal[m]

            for m
            in CLER_CORE
        ]),

        axis=1
    )


    CLER_CORE_OOF[
        outer_valid_idx
    ] = core_outer_valid


    # ========================================================
    # C. SPLIT OUTER TRAIN
    #
    # structural fit = 85%
    # calibration/fusion = 15%
    # ========================================================

    splitter = StratifiedShuffleSplit(

        n_splits=1,

        test_size=
            CLER_CONFIG[
                "calibration_fraction"
            ],

        random_state=
            SEED
            + 50000
            + outer_fold
    )


    (
        fit_local_idx,
        calib_local_idx

    ) = next(

        splitter.split(

            X_all[
                outer_train_idx
            ],

            y_outer_train
        )
    )


    fit_global_idx = (

        outer_train_idx[
            fit_local_idx
        ]
    )


    calib_global_idx = (

        outer_train_idx[
            calib_local_idx
        ]
    )


    X_fit = (
        X_all[
            fit_global_idx
        ]
    )


    y_fit = (
        y_all[
            fit_global_idx
        ]
    )


    X_calib = (
        X_all[
            calib_global_idx
        ]
    )


    y_calib = (
        y_all[
            calib_global_idx
        ]
    )


    X_valid = (
        X_all[
            outer_valid_idx
        ]
    )


    core_fit = (

        core_outer_train[
            fit_local_idx
        ]
    )


    core_calib = (

        core_outer_train[
            calib_local_idx
        ]
    )


    print(
        "\nStructural fit N:",
        len(
            y_fit
        )
    )

    print(
        "Calibration/fusion N:",
        len(
            y_calib
        )
    )

    print(
        "Outer validation N:",
        len(
            y_outer_valid
        )
    )


    # ========================================================
    # D. TRAIN FOLD-LOCAL STRUCTURAL TREES
    # ========================================================

    print(
        "\nTraining structural CatBoost..."
    )


    structural_cat = (
        make_cler_catboost(

            seed=
                SEED
                + 51000
                + outer_fold
        )
    )


    structural_cat.fit(

        X_fit,

        y_fit,

        verbose=False
    )


    print(
        "Training structural LightGBM..."
    )


    structural_lgb = (
        make_cler_lgbm(

            seed=
                SEED
                + 52000
                + outer_fold
        )
    )


    structural_lgb.fit(

        X_fit,

        y_fit
    )


    # ========================================================
    # E. BASE STRUCTURAL PREDICTIONS — DIAGNOSTIC ONLY
    # ========================================================

    structural_cat_valid = (

        structural_cat
        .predict_proba(
            X_valid
        )[
            :,
            1
        ]
    )


    structural_lgb_valid = (

        structural_lgb
        .predict_proba(
            X_valid
        )[
            :,
            1
        ]
    )


    # ========================================================
    # F. EXTRACT LEAF IDS
    # ========================================================

    print(
        "Extracting leaf representations..."
    )


    cat_leaf_fit, lgb_leaf_fit = (
        extract_leaf_indices(

            structural_cat,

            structural_lgb,

            X_fit
        )
    )


    cat_leaf_calib, lgb_leaf_calib = (
        extract_leaf_indices(

            structural_cat,

            structural_lgb,

            X_calib
        )
    )


    cat_leaf_valid, lgb_leaf_valid = (
        extract_leaf_indices(

            structural_cat,

            structural_lgb,

            X_valid
        )
    )


    # ========================================================
    # G. GLOBAL LEAF IDS
    # ========================================================

    (
        cat_ids_fit,
        lgb_ids_fit,
        total_vocab

    ) = make_global_leaf_ids(

        cat_leaf_fit,

        lgb_leaf_fit
    )


    (
        cat_ids_calib,
        lgb_ids_calib,
        total_vocab_calib

    ) = make_global_leaf_ids(

        cat_leaf_calib,

        lgb_leaf_calib
    )


    (
        cat_ids_valid,
        lgb_ids_valid,
        total_vocab_valid

    ) = make_global_leaf_ids(

        cat_leaf_valid,

        lgb_leaf_valid
    )


    assert (
        total_vocab
        ==
        total_vocab_calib
        ==
        total_vocab_valid
    )


    print(
        "  CatBoost trees:",
        cat_ids_fit.shape[1]
    )

    print(
        "  LightGBM trees:",
        lgb_ids_fit.shape[1]
    )

    print(
        "  Embedding vocabulary:",
        total_vocab
    )


    # ========================================================
    # H. TRAIN CLER
    # ========================================================

    print(
        "\nTraining CLER leaf ranker..."
    )


    cler_model = (
        train_cler(

            cat_ids_fit,

            lgb_ids_fit,

            y_fit,

            core_fit,

            total_vocab,

            seed=
                SEED
                + 53000
                + outer_fold
        )
    )


    # ========================================================
    # I. RAW CLER LOGITS
    # ========================================================

    cler_logit_calib = (
        predict_cler_logits(

            cler_model,

            cat_ids_calib,

            lgb_ids_calib
        )
    )


    cler_logit_valid = (
        predict_cler_logits(

            cler_model,

            cat_ids_valid,

            lgb_ids_valid
        )
    )


    CLER_RAW_SCORE[
        outer_valid_idx
    ] = cler_logit_valid


    # ========================================================
    # J. PLATT CALIBRATE CLER
    #
    # calibration subset ONLY
    # ========================================================

    cler_platt = (
        LogisticRegression(

            penalty="l2",

            C=1.0,

            solver="lbfgs",

            max_iter=2000
        )
    )


    cler_platt.fit(

        cler_logit_calib[
            :,
            None
        ],

        y_calib
    )


    cler_prob_valid = (

        cler_platt
        .predict_proba(

            cler_logit_valid[
                :,
                None
            ]

        )[
            :,
            1
        ]
    )


    CLER_STANDALONE_OOF[
        outer_valid_idx
    ] = cler_prob_valid


    # ========================================================
    # K. CORE-ONLY META BASELINE
    #
    # Exactly same calibration subset.
    # ========================================================

    core_meta_model = (
        LogisticRegression(

            penalty="l2",

            C=
                CLER_CONFIG[
                    "core_meta_C"
                ],

            solver="lbfgs",

            max_iter=2000
        )
    )


    core_meta_model.fit(

        cler_logit(
            core_calib
        )[
            :,
            None
        ],

        y_calib
    )


    core_meta_valid = (

        core_meta_model
        .predict_proba(

            cler_logit(
                core_outer_valid
            )[
                :,
                None
            ]

        )[
            :,
            1
        ]
    )


    CLER_CORE_META_OOF[
        outer_valid_idx
    ] = core_meta_valid


    # ========================================================
    # L. CORE4 + CLER FUSION
    #
    # Only TWO signals:
    #
    #     Core4 logit
    #     CLER leaf-representation logit
    #
    # Avoid another high-dimensional meta-model.
    # ========================================================

    Z_calib = np.column_stack([

        cler_logit(
            core_calib
        ),

        cler_logit_calib,
    ])


    Z_valid = np.column_stack([

        cler_logit(
            core_outer_valid
        ),

        cler_logit_valid,
    ])


    fusion_scaler = (
        StandardScaler()
    )


    Z_calib_scaled = (
        fusion_scaler
        .fit_transform(
            Z_calib
        )
    )


    Z_valid_scaled = (
        fusion_scaler
        .transform(
            Z_valid
        )
    )


    fusion_model = (
        LogisticRegression(

            penalty="l2",

            C=
                CLER_CONFIG[
                    "fusion_C"
                ],

            solver="lbfgs",

            max_iter=2000
        )
    )


    fusion_model.fit(

        Z_calib_scaled,

        y_calib
    )


    cler_fusion_valid = (

        fusion_model
        .predict_proba(
            Z_valid_scaled
        )[
            :,
            1
        ]
    )


    CLER_FUSION_OOF[
        outer_valid_idx
    ] = cler_fusion_valid


    # ========================================================
    # M. FOLD METRICS
    # ========================================================

    core_pr = (
        average_precision_score(

            y_outer_valid,

            core_outer_valid
        )
    )


    core_meta_pr = (
        average_precision_score(

            y_outer_valid,

            core_meta_valid
        )
    )


    cler_pr = (
        average_precision_score(

            y_outer_valid,

            cler_prob_valid
        )
    )


    fusion_pr = (
        average_precision_score(

            y_outer_valid,

            cler_fusion_valid
        )
    )


    cat_pr = (
        average_precision_score(

            y_outer_valid,

            structural_cat_valid
        )
    )


    lgb_pr = (
        average_precision_score(

            y_outer_valid,

            structural_lgb_valid
        )
    )


    # ========================================================
    # N. DIVERSITY
    # ========================================================

    core_rank = (
        pd.Series(
            core_outer_valid
        )
        .rank()
    )


    cler_rank = (
        pd.Series(
            cler_logit_valid
        )
        .rank()
    )


    cler_spearman = (

        np.corrcoef(

            core_rank,

            cler_rank

        )[0, 1]
    )


    # ========================================================
    # O. HARD-POSITIVE ANALYSIS
    # ========================================================

    positive_mask = (
        y_outer_valid
        ==
        1
    )


    hard_positive_cut = np.quantile(

        core_outer_valid[
            positive_mask
        ],

        0.25
    )


    hard_positive_mask = (

        positive_mask

        &

        (
            core_outer_valid
            <=
            hard_positive_cut
        )
    )


    hp_core_mean = float(

        core_outer_valid[
            hard_positive_mask
        ].mean()
    )


    hp_cler_mean = float(

        cler_prob_valid[
            hard_positive_mask
        ].mean()
    )


    hp_fusion_mean = float(

        cler_fusion_valid[
            hard_positive_mask
        ].mean()
    )


    # ========================================================
    # P. GENERAL METRICS
    # ========================================================

    core_pm = (
        probability_metrics(

            y_outer_valid,

            core_outer_valid
        )
    )


    cler_pm = (
        probability_metrics(

            y_outer_valid,

            cler_prob_valid
        )
    )


    fusion_pm = (
        probability_metrics(

            y_outer_valid,

            cler_fusion_valid
        )
    )


    # ========================================================
    # Q. FUSION COEFFICIENTS
    # ========================================================

    fusion_coef_rows.append({

        "Fold":
            outer_fold,

        "Core_Coefficient":
            float(
                fusion_model
                .coef_[
                    0,
                    0
                ]
            ),

        "CLER_Coefficient":
            float(
                fusion_model
                .coef_[
                    0,
                    1
                ]
            ),
    })


    # ========================================================
    # R. FOLD ROW
    # ========================================================

    fold_rows.append({

        "Fold":
            outer_fold,

        "Core_PR":
            core_pr,

        "CoreMeta_PR":
            core_meta_pr,

        "CLER_PR":
            cler_pr,

        "Fusion_PR":
            fusion_pr,

        "Delta_CLER_vs_Core":
            cler_pr
            -
            core_pr,

        "Delta_Fusion_vs_Core":
            fusion_pr
            -
            core_pr,

        "Delta_Fusion_vs_CoreMeta":
            fusion_pr
            -
            core_meta_pr,

        "StructuralCat_PR":
            cat_pr,

        "StructuralLGB_PR":
            lgb_pr,

        "Core_CLER_Spearman":
            cler_spearman,

        "Rank_Diversity":
            1.0
            -
            cler_spearman,

        "Core_ROC":
            core_pm[
                "ROC_AUC"
            ],

        "CLER_ROC":
            cler_pm[
                "ROC_AUC"
            ],

        "Fusion_ROC":
            fusion_pm[
                "ROC_AUC"
            ],

        "Core_Brier":
            core_pm[
                "Brier"
            ],

        "CLER_Brier":
            cler_pm[
                "Brier"
            ],

        "Fusion_Brier":
            fusion_pm[
                "Brier"
            ],

        "Core_ECE":
            core_pm[
                "ECE"
            ],

        "CLER_ECE":
            cler_pm[
                "ECE"
            ],

        "Fusion_ECE":
            fusion_pm[
                "ECE"
            ],

        "HardPositive_CoreMean":
            hp_core_mean,

        "HardPositive_CLERMean":
            hp_cler_mean,

        "HardPositive_FusionMean":
            hp_fusion_mean,
    })


    # ========================================================
    # S. PRINT FOLD SUMMARY
    # ========================================================

    print(
        "\nFold performance:"
    )


    print(
        f"  Structural CatBoost : "
        f"{cat_pr:.6f}"
    )


    print(
        f"  Structural LGBM     : "
        f"{lgb_pr:.6f}"
    )


    print(
        f"  Core4               : "
        f"{core_pr:.6f}"
    )


    print(
        f"  CoreMeta            : "
        f"{core_meta_pr:.6f}"
    )


    print(
        f"  CLER standalone     : "
        f"{cler_pr:.6f}"
    )


    print(
        f"  Core4 + CLER        : "
        f"{fusion_pr:.6f}"
    )


    print(
        f"  Δ fusion vs Core    : "
        f"{fusion_pr-core_pr:+.6f}"
    )


    print(
        f"  CLER/Core Spearman  : "
        f"{cler_spearman:.6f}"
    )


    print(
        f"  Rank diversity      : "
        f"{1.0-cler_spearman:.6f}"
    )


    print(
        "\nHard positives:"
    )


    print(
        f"  Core mean   : "
        f"{hp_core_mean:.6f}"
    )


    print(
        f"  CLER mean   : "
        f"{hp_cler_mean:.6f}"
    )


    print(
        f"  Fusion mean : "
        f"{hp_fusion_mean:.6f}"
    )


    # ========================================================
    # T. CLEANUP
    # ========================================================

    del (
        structural_cat,
        structural_lgb,
        cler_model,
        cler_platt,
        core_meta_model,
        fusion_model,
        fusion_scaler,

        cat_leaf_fit,
        cat_leaf_calib,
        cat_leaf_valid,

        lgb_leaf_fit,
        lgb_leaf_calib,
        lgb_leaf_valid,

        cat_ids_fit,
        cat_ids_calib,
        cat_ids_valid,

        lgb_ids_fit,
        lgb_ids_calib,
        lgb_ids_valid,
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 15A.15 GLOBAL THRESHOLDS
# ============================================================

core_t, _, _ = (
    select_best_threshold(

        y_all,

        CLER_CORE_OOF,

        objective="F1"
    )
)


core_meta_t, _, _ = (
    select_best_threshold(

        y_all,

        CLER_CORE_META_OOF,

        objective="F1"
    )
)


cler_t, _, _ = (
    select_best_threshold(

        y_all,

        CLER_STANDALONE_OOF,

        objective="F1"
    )
)


fusion_t, _, _ = (
    select_best_threshold(

        y_all,

        CLER_FUSION_OOF,

        objective="F1"
    )
)


# ============================================================
# 15A.16 GLOBAL METRICS
# ============================================================

core_metrics = (
    evaluate_binary_predictions(

        y_all,

        CLER_CORE_OOF,

        threshold=
            core_t
    )
)


core_meta_metrics = (
    evaluate_binary_predictions(

        y_all,

        CLER_CORE_META_OOF,

        threshold=
            core_meta_t
    )
)


cler_metrics = (
    evaluate_binary_predictions(

        y_all,

        CLER_STANDALONE_OOF,

        threshold=
            cler_t
    )
)


fusion_metrics = (
    evaluate_binary_predictions(

        y_all,

        CLER_FUSION_OOF,

        threshold=
            fusion_t
    )
)


module15a_results = pd.DataFrame([

    {
        "Method":
            "Core4Mean",

        **core_metrics
    },

    {
        "Method":
            "Core4_MetaBaseline",

        **core_meta_metrics
    },

    {
        "Method":
            "CLER",

        **cler_metrics
    },

    {
        "Method":
            "Core4_CLER_Fusion",

        **fusion_metrics
    },
])


module15a_results[
    "Delta_PR_vs_Core"
] = (

    module15a_results[
        "PR_AUC"
    ]

    -
    core_metrics[
        "PR_AUC"
    ]
)


module15a_results[
    "Delta_PR_vs_CoreMeta"
] = (

    module15a_results[
        "PR_AUC"
    ]

    -
    core_meta_metrics[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE 15A — CLER RESULTS"
)

print(
    "=" * 135
)


display(

    module15a_results[
        [
            "Method",
            "PR_AUC",
            "Delta_PR_vs_Core",
            "Delta_PR_vs_CoreMeta",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Threshold",
        ]
    ]
)


# ============================================================
# 15A.17 FOLD RESULTS
# ============================================================

module15a_folds = (
    pd.DataFrame(
        fold_rows
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "CLER FOLD RESULTS"
)

print(
    "=" * 135
)


display(
    module15a_folds
)


print(
    "\nMean Fusion ΔPR vs Core:",
    module15a_folds[
        "Delta_Fusion_vs_Core"
    ].mean()
)


print(
    "Positive folds vs Core:",
    int(
        (
            module15a_folds[
                "Delta_Fusion_vs_Core"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


print(
    "\nMean Fusion ΔPR vs CoreMeta:",
    module15a_folds[
        "Delta_Fusion_vs_CoreMeta"
    ].mean()
)


print(
    "Positive folds vs CoreMeta:",
    int(
        (
            module15a_folds[
                "Delta_Fusion_vs_CoreMeta"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_SPLITS
)


# ============================================================
# 15A.18 GLOBAL COMPLEMENTARITY
# ============================================================

core_rank = (
    pd.Series(
        CLER_CORE_OOF
    )
    .rank(
        method="average"
    )
)


cler_rank = (
    pd.Series(
        CLER_RAW_SCORE
    )
    .rank(
        method="average"
    )
)


global_spearman = (

    np.corrcoef(

        core_rank,

        cler_rank

    )[0, 1]
)


complementarity_df = pd.DataFrame({

    "Metric": [

        "CLER standalone PR-AUC",

        "Core/CLER Spearman",

        "Rank diversity",

        "Fusion PR-AUC",

        "Fusion gain vs Core4",
    ],

    "Value": [

        average_precision_score(
            y_all,
            CLER_STANDALONE_OOF
        ),

        global_spearman,

        1.0
        -
        global_spearman,

        average_precision_score(
            y_all,
            CLER_FUSION_OOF
        ),

        (
            average_precision_score(
                y_all,
                CLER_FUSION_OOF
            )

            -

            average_precision_score(
                y_all,
                CLER_CORE_OOF
            )
        ),
    ]
})


print(
    "\n"
    + "=" * 135
)

print(
    "CLER COMPLEMENTARITY"
)

print(
    "=" * 135
)


display(
    complementarity_df
)


# ============================================================
# 15A.19 FUSION COEFFICIENTS
# ============================================================

fusion_coef_df = pd.DataFrame(
    fusion_coef_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "CLER FUSION COEFFICIENTS"
)

print(
    "=" * 135
)


display(
    fusion_coef_df
)


print(
    "\nMean Core coefficient:",
    fusion_coef_df[
        "Core_Coefficient"
    ].mean()
)


print(
    "Mean CLER coefficient:",
    fusion_coef_df[
        "CLER_Coefficient"
    ].mean()
)


# ============================================================
# 15A.20 GLOBAL HARD-POSITIVE ANALYSIS
# ============================================================

positive_idx = np.where(
    y_all == 1
)[0]


hard_positive_cut = np.quantile(

    CLER_CORE_OOF[
        positive_idx
    ],

    0.25
)


hard_positive_mask = (

    (y_all == 1)

    &

    (
        CLER_CORE_OOF
        <=
        hard_positive_cut
    )
)


hard_positive_df = pd.DataFrame({

    "Metric": [

        "Hard positive N",

        "Mean Core4 score",

        "Mean CLER score",

        "Mean Fusion score",
    ],

    "Value": [

        int(
            hard_positive_mask.sum()
        ),

        float(
            CLER_CORE_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            CLER_STANDALONE_OOF[
                hard_positive_mask
            ].mean()
        ),

        float(
            CLER_FUSION_OOF[
                hard_positive_mask
            ].mean()
        ),
    ]
})


print(
    "\n"
    + "=" * 135
)

print(
    "CLER HARD-POSITIVE ANALYSIS"
)

print(
    "=" * 135
)


display(
    hard_positive_df
)


# ============================================================
# 15A.21 DEVELOPMENT PROGRESSION
# ============================================================

progression = [

    {
        "Method":
            "Core4Mean",

        "PR_AUC":
            average_precision_score(
                y_all,
                CLER_CORE_OOF
            )
    },
]


if "HPSC_OOF" in globals():

    progression.append({

        "Method":
            "HPSC_HPRR_14C",

        "PR_AUC":
            average_precision_score(
                y_all,
                HPSC_OOF
            )
    })


progression.extend([

    {
        "Method":
            "CLER_Standalone",

        "PR_AUC":
            average_precision_score(
                y_all,
                CLER_STANDALONE_OOF
            )
    },

    {
        "Method":
            "Core4_CLER_Fusion",

        "PR_AUC":
            average_precision_score(
                y_all,
                CLER_FUSION_OOF
            )
    },
])


module15a_progression = (
    pd.DataFrame(
        progression
    )
)


core_reference = (

    module15a_progression
    .loc[
        module15a_progression[
            "Method"
        ]
        ==
        "Core4Mean",

        "PR_AUC"
    ]
    .iloc[0]
)


module15a_progression[
    "Delta_vs_Core"
] = (

    module15a_progression[
        "PR_AUC"
    ]

    -
    core_reference
)


print(
    "\n"
    + "=" * 135
)

print(
    "PERFORMANCE DEVELOPMENT PROGRESSION"
)

print(
    "=" * 135
)


display(
    module15a_progression
)


# ============================================================
# 15A.22 SAVE RESULTS
# ============================================================

save_csv(

    module15a_results,

    TAB_DIR
    / "module15a_cler_results.csv"
)


save_csv(

    module15a_folds,

    TAB_DIR
    / "module15a_cler_folds.csv"
)


save_csv(

    complementarity_df,

    TAB_DIR
    / "module15a_cler_complementarity.csv"
)


save_csv(

    fusion_coef_df,

    TAB_DIR
    / "module15a_cler_fusion_coefficients.csv"
)


save_csv(

    hard_positive_df,

    TAB_DIR
    / "module15a_cler_hard_positive.csv"
)


save_csv(

    module15a_progression,

    TAB_DIR
    / "module15a_performance_progression.csv"
)


cler_pred_df = pd.DataFrame({

    "target":
        y_all,

    "fold":
        fold_assignment,

    "Core4Mean":
        CLER_CORE_OOF,

    "Core4_MetaBaseline":
        CLER_CORE_META_OOF,

    "CLER":
        CLER_STANDALONE_OOF,

    "CLER_RawScore":
        CLER_RAW_SCORE,

    "Core4_CLER_Fusion":
        CLER_FUSION_OOF,
})


save_csv(

    cler_pred_df,

    PRED_DIR
    / "module15a_cler_oof.csv"
)


module15a_manifest = {

    "method":
        "Cross-Model Leaf Embedding Ranker",

    "structural_models": [
        "CatBoost",
        "LightGBM",
    ],

    "catboost_trees":
        CLER_CONFIG[
            "cat_iterations"
        ],

    "lightgbm_trees":
        CLER_CONFIG[
            "lgb_iterations"
        ],

    "embedding_dim":
        CLER_CONFIG[
            "embedding_dim"
        ],

    "epochs":
        CLER_CONFIG[
            "epochs"
        ],

    "objective":
        "weighted BCE + hard-pair ranking",

    "ranking_target":
        "Core4 difficult / near-tie positive-negative pairs",

    "inner_calibration_fraction":
        CLER_CONFIG[
            "calibration_fraction"
        ],

    "outer_validation_used_for_training":
        False,

    "holdout_used":
        False,
}


save_json(

    module15a_manifest,

    LOG_DIR
    / "module15a_manifest.json"
)


print(
    "\nMODULE 15A COMPLETE."
)

CLER device: cuda

CLER OUTER FOLD 1/5

Structural fit N: 138001
Calibration/fusion N: 24354
Outer validation N: 40589

Training structural CatBoost...
Training structural LightGBM...
Extracting leaf representations...


AttributeError: 'CatBoostClassifier' object has no attribute 'get_leaf_indexes'